# FTS France — **Préparation + Enrichissement TVA** (pipeline complet)

À partir du fichier **brut France**, le notebook enchaîne automatiquement :

### Étape A — Nettoyage géographique
1. **Adresses** remises à la norme française « numéro puis voie » (« RUE BISCORNET 9 » → « 9 RUE BISCORNET »).
   v5_40 : nouvelle colonne **« Adresse réordonnée »** juste après l'adresse — la colonne de la
   Commission n'est **plus jamais modifiée**. La nouvelle colonne contient l'adresse réordonnée
   quand une correction s'applique, sinon l'adresse d'origine telle quelle ; les cellules
   réellement réordonnées y sont **surlignées**.
2. **Code postal corrigé** : nouvelle colonne (espaces retirés, zéro initial rétabli…), **colonne verte** (ajoutée).
3. **NUTS2** corrigé sur place (« Ile de France » → « Île-de-France »), cellules modifiées **en vert** ;
   **NUTS3 FR** : nouvelle colonne (département), **colonne verte** (ajoutée).
   Tout est régénéré depuis le code postal corrigé (table officielle département → région/département).

### Étape B — Enrichissement TVA (moteur v3/v4)
Recherche des TVA manquantes + colonnes **SIRET / Forme juridique / Code NAF-APE / État**,
en utilisant l'adresse réordonnée et le code postal corrigé pour de meilleurs résultats.

> Méthode **locale** (pas d'API externe) pour le nettoyage : instantané, sans quota.
> Le code postal corrigé sert ensuite de filtre département à la recherche TVA.


















### v5_40 — Nom API pour les TVA existantes, « Adresse réordonnée », « AUTRE » / « A CHERCHER » généralisés
- **« Nom API » massivement complétée.** Cause du vide signalé par l'utilisateur : la colonne n'était
  alimentée que par le RAPPORT (passes 0, mémoire et 1 — les TVA *manquantes*). La **passe 2** (TVA déjà
  fournies par la Commission — la majorité du fichier) documentait SIRET/forme juridique mais ne remontait
  jamais le nom officiel. Elle alimente désormais un dictionnaire dédié (`NOM_API_PAR_FTS`, lecture seule
  de la fiche SIREN déjà obtenue — aucun appel API supplémentaire), fusionné dans `ajouter_nom_api` avec
  trois priorités : 1) rapport avec TVA retenue, 2) passe 2, 3) autres lignes du rapport (candidat écarté).
- **Nouvelle colonne « Adresse réordonnée »** placée juste après l'adresse : la colonne Address de la
  Commission n'est plus modifiée (alignement sur la règle « aucune colonne d'origine touchée »). La
  nouvelle colonne porte l'adresse réordonnée quand une correction s'applique, sinon l'adresse d'origine
  telle quelle — les traitements d'affichage en aval (export, coloriage) s'appuient désormais sur elle.
  Le moteur de recherche continue, lui, d'utiliser l'adresse BRUTE (règle intangible : matching sur données
  brutes) — et il tournait de toute façon AVANT le réordonnancement, donc zéro impact sur les TVA.
- **Finition « AUTRE » / « A CHERCHER » généralisée** (fichier seulement, le rapport garde tout) :
  - « A CHERCHER » (bénéficiaire ≥ 300 000 € sans info) s'applique désormais aussi à SIREN, SIRET,
    Forme_juridique, Niveau_I/II/III, Code_NAF_APE, Activite_principale, Section_NAF, Etat_entreprise
    et Référentiel SGAE — plus seulement à la TVA.
  - Toutes les AUTRES colonnes ajoutées par le pipeline (Nom API, Adresse réordonnée, géographie, CFP,
    opérateurs…) reçoivent « AUTRE » quand la case est vide.
  - « République française » et les personnes physiques anonymisées (NATURAL PERSON, PERSONNE PRIVÉE,
    Art. 38(7)…) ne reçoivent JAMAIS « A CHERCHER » : toutes leurs cases vides passent à « AUTRE »
    (lecture seule de `_est_exclu`, fonction figée non modifiée).
- Moteur de recherche : strictement inchangé.

### v5_39 — Nom API remonté aussi pour les Passes 0 et mémoire (MAIN_REGISTRATION)
- Correctif signalé par l'utilisateur : les fiches obtenues par SIREN (`infos_par_siren`, utilisées par la Passe 0, la Passe mémoire et la passe 2) ne contenaient pas le nom de l'entreprise — le champ `Nom_API` du rapport restait donc vide pour les stratégies MAIN_REGISTRATION et MEMOIRE_*, et la colonne « Nom API » aussi.
- Une enveloppe lecture seule (`_fiche_avec_nom`) ajoute le nom officiel (« nom_complet », sinon « nom_raison_sociale » — même extraction que le moteur) aux fiches par SIREN. `_infos_entreprise` n'est pas modifiée car elle alimente aussi `_appeler_api` (moteur figé), où ce champ écraserait le nom du candidat du moteur.
- Effet secondaire voulu : « Bénéficiaire corrigé » reçoit désormais aussi le nom officiel pour les lignes Passe 0/mémoire (le code le prévoyait déjà via NOM_CORRIGE_PAR_FTS mais ne se déclenchait jamais, faute de nom dans la fiche).
- Moteur de recherche : strictement inchangé.

### v5_38 — Colonne « Nom API » (écriture officielle de l'annuaire)
- Nouvelle colonne « Nom API » juste après « Bénéficiaire corrigé », coloriée en bleu BDD7EE : l'écriture exacte du bénéficiaire telle que retournée par recherche-entreprises.api.gouv.fr (champ Nom_API du rapport — passes 0, mémoire et 1). Vide quand aucune fiche API n'a été obtenue.
- « Bénéficiaire corrigé » reste inchangée (nom FTS nettoyé, éventuellement remplacé par le nom officiel via les corrections) ; la nouvelle colonne permet de COMPARER les deux écritures.
- Moteur de recherche : strictement inchangé.

### v5_37 — Mémoire de correspondance : socle embarqué en dur (mécanisme hybride)
- Les 338 correspondances fiables du millésime 2025 (SIREN Commission + TVA fournies) sont intégrées EN DUR dans le notebook, comme les autres tables du pipeline : plus besoin de déposer `correspondance_siren.json` pour en bénéficier.
- Si le fichier `correspondance_siren.json` est déposé malgré tout, il se fusionne par-dessus le socle (ses entrées gagnent en cas de doublon — il est plus récent).
- Moteur de recherche : strictement inchangé.

### v5_36 — Mémoire de correspondance : clé vide bannie + repli « nom seul »
- Les clés au nom vide (bénéficiaires masqués) ne sont plus ni construites, ni rechargées (le fichier `correspondance_siren.json` existant est purgé au chargement — la clé `'||'` observée disparaît d'elle-même).
- Repli « nom seul » : si (nom, département) ne matche pas mais que le nom correspond à UN SEUL SIREN dans toute la mémoire, il est réutilisé (déménagements entre millésimes, ex. CARE FRANCE 93→75) ; étiquette d'audit dédiée `MEMOIRE_NOM_SEUL` dans le rapport. Plusieurs SIREN pour un même nom → on ne tranche pas, recherche normale.
- Moteur de recherche : strictement inchangé.

### v5_35 — Correctif : plantage pandas sur sélection vide (mask_memoire)
Trouvé en testant le scénario cross-années bout en bout (fichier 2025 puis
2024 dans la même session) : `mask_memoire.loc[idx_vide] = []` lève
`TypeError: Invalid value '[]' for dtype 'bool'` sous la version de pandas
utilisée ici, alors que l'opération devrait être un no-op. Corrigé par une
garde sur la longueur avant affectation -- reproduit et vérifié isolément
avant correction, puis sur le scénario complet qui l'avait révélé.

### v5_34 — 6 alias pays supplémentaires + correctif de propagation intra-fichier
Îles Falkland (FK), Pitcairn (PN), Sainte-Hélène (SH), Vatican (VA), et les
**deux** Îles Vierges (VG britanniques, VI américaines -- l'utilisateur n'a
donné que VI, l'ambiguïté est signalée plutôt que tranchée seule). Clés basées
sur les noms officiels ISO ; **non vérifiées empiriquement** (aucune occurrence
dans le fichier FTS 2025 disponible pour ces territoires rares), contrairement
à la campagne précédente (v5_33) -- à confirmer si l'utilisateur dispose d'un
fichier où ces pays apparaissent réellement.

**Correctif important, trouvé en testant la question de l'utilisateur sur les
« SIREN présent une année, vide les autres »** : ce cas se posait aussi *au
sein d'un même fichier* -- un bénéficiaire avec plusieurs lignes 2025 (projets
différents), dont une seule porte un Main Registration valide, voyait ses
AUTRES lignes repartir inutilement en recherche classique (la mémoire ne se
construisait qu'À LA FIN du traitement, pour un usage sur un fichier
ultérieur). Corrigé : la mémoire est désormais alimentée dès la lecture du
Main Registration, AVANT le calcul du périmètre de recherche -- les lignes
sœurs d'un même fichier bénéficient donc, elles aussi, de la Passe mémoire.
Aucun appel API supplémentaire (simple lecture de colonne).

### v5_33 — 13 nouveaux programmes CFP 21-27 + 7 alias pays manquants
**13 programmes** ajoutés à `SOUS_CATEGORIES` (fichier PROG25.xlsx, tous
CFP 21-27) : Single Market Programme, ESF+, European Recovery and Resilience
Facility, Pericles IV, Rights and Values, AMIF, Nuclear Safety
(Bulgarie/Slovaquie), Short-term Defence instrument, NDICI - Global Europe,
INSC, Ukraine Loan Cooperation Mechanism, DA/JU/EUI, et un item PPPA
supplémentaire -- aucun doublon avec les 185 déjà présents (vérifié). Total :
**198 programmes**.

**7 alias pays** ajoutés à `PAYS_ALIAS.iso_alias`, chacun vérifié contre le
vrai fichier FTS 2025 avant intégration (volumes réels trouvés entre
parenthèses) : Côte d'Ivoire "Cote d'Ivoire" (15 lignes) -> CI ; République
démocratique du Congo "Democratic Republic of the Congo" (20 lignes) -> CD ;
Kosovo "Kosovo (under UNSCR 1244/99)" (108 lignes) -> XK ; Macédoine du Nord
"Macedonia" seul seul, distinct de "North Macedonia" déjà géré (137 lignes)
-> MK ; Sainte-Lucie "St. Lucia" (2 lignes) -> LC ; et deux troncatures de
l'export Commission, confirmées par l'adresse des lignes concernées :
**"Republ"** (383 lignes !) -> RS (Serbie) et **"Republic o"** (138 lignes)
-> ME (Monténégro).

**Vérifications faites avant d'intégrer**, pour ne rien casser ni doublonner :
« Congo (Democratic Republic of) » (13 lignes) et « Kosovo » seul étaient
déjà correctement résolus par un alias existant -- non retouchés. « Congo »
et « Republic of the Congo » se résolvent déjà, via `pycountry`, en **CG**
(Congo-Brazzaville) -- un pays *différent* de la RDC : volontairement laissés
tels quels pour ne pas les confondre avec les nouveaux alias ci-dessus.
« North Macedonia » se résolvait déjà correctement (MK, via `pycountry`).

### v5_32 — Colonne SIREN repositionnée + désambiguïsation d'établissement (best-effort)
**Colonne SIREN déplacée** juste après « Main registration number of
beneficiary » à l'export (comparaison directe du SIREN Commission et du
SIREN retenu par le pipeline) ; ignoré proprement sur un fichier sans cette
colonne.

**Désambiguïsation du SIRET par code postal**, pour les bénéficiaires
multi-établissements (Passe 0 et Passe mémoire) : quand le code postal de la
ligne FTS diffère de celui du siège, une recherche par nom + département est
tentée pour retrouver l'établissement correspondant à CE code postal, dont le
SIRET remplace celui du siège.

**Limite honnête, vérifiée avant d'implémenter** : l'API
`recherche-entreprises.api.gouv.fr` ne permet PAS de lister tous les
établissements d'un SIREN -- documentation officielle : un filtre
géographique combiné à une requête SIREN/SIRET brute est **silencieusement
ignoré** ; et le champ `matching_etablissements` reste souvent vide même
quand plusieurs établissements existent réellement (confirmé par un
témoignage d'utilisateur rencontrant exactement ce besoin). Le contournement
implémenté (recherche textuelle nom + département, filtrée au SIREN déjà
connu) est légitime et documenté, mais **best-effort** : en l'absence de
résultat probant, repli systématique et sûr sur le SIRET du siège
(comportement inchangé) -- jamais d'établissement deviné. Une désambiguïsation
garantie à 100 % supposerait l'API Sirene authentifiée de l'INSEE (clé
requise) ou les fichiers stock complets, hors du périmètre actuel (API
publique sans clé).

### v5_31 — Mémoire de correspondance inter-millésimes (réutiliser 2025 sur les autres années)
Un export FTS ne contient qu'**un seul millésime** (vérifié : le fichier 2025
ne contient que Year=2025). Les SIREN certains obtenus via Main Registration
(Passe 0, v5_30) ou via une TVA déjà fournie (Passe 2) sont donc désormais
**mémorisés** (clé : nom normalisé + département déduit du code postal --
jamais le nom seul, pour ne pas confondre deux implantations réellement
distinctes d'un même nom, ex. une chambre de commerce régionale) et
**exportés** en fin de traitement (`correspondance_siren.json`).

Au traitement d'un **autre millésime** (2014-2024, sans Main Registration) :
redéposer ce fichier au même endroit -> une nouvelle **Passe mémoire**,
intercalée entre la Passe 0 et la Passe 1, documente directement ces
bénéficiaires depuis leur SIREN déjà connu (score 100, stratégie
`MEMOIRE_CORRESPONDANCE`) -- **zéro appel de recherche**. Seuls les
bénéficiaires jamais rencontrés dans un millésime à Main Registration
repartent en Passe 1 (recherche classique, inchangée). La mémoire s'enrichit
à chaque exécution et se reporte automatiquement d'un traitement à l'autre.

Ordre des priorités dans `enrichir_france`, désormais à trois niveaux :
**Passe 0** (Main Registration de la ligne elle-même) -> **Passe mémoire**
(SIREN connu d'un autre millésime) -> **Passe 1** (recherche, en dernier
recours seulement).

### v5_30 — Passe 0 : SIREN déjà fourni par la Commission (Main Registration)
L'export FTS 2025 introduit la colonne **« Main registration number of
beneficiary »**, qui contient souvent le SIREN du bénéficiaire directement
(vérifié : 5 466/5 813 lignes françaises, 100 % Luhn-valides ; recoupement
indépendant concluant sur des SIREN bien connus : CNRS 180089013, Airbus
383474814, Thales 552059024).

**C'est un champ générique européen** : chaque pays y inscrit son propre
numéro de registre national dans son propre format (Allemagne : « VR7795 »,
Belgique : numéro d'entreprise, Italie/Espagne : codes provinciaux) -- donc
exploité **uniquement pour les lignes France**, et seulement si la valeur
ressemble à un SIREN (9 chiffres) et passe la clé de Luhn.

**Nouvelle Passe 0**, insérée avant la Passe 1 (recherche) : pour les lignes
françaises sans TVA mais avec un Main Registration valide, le SIREN est
utilisé directement (documentation via `fiche_siren`, comme la Passe 2) --
**zéro appel au moteur de recherche** pour ces lignes. Score 100 (fourni
directement par la Commission, validé par Luhn), stratégie
`MAIN_REGISTRATION` visible au rapport, `Bénéficiaire corrigé` renseigné
avec le nom officiel de la fiche SIRENE. La Passe 1 ne porte plus que sur
les lignes qui n'ont NI TVA NI Main Registration exploitable -- réduction
attendue substantielle du volume d'appels réseau sur les exports récents.

Garde-fous : colonne absente (exports antérieurs à 2025) -> Passe 0
ignorée proprement, comportement inchangé ; valeurs « N/A - Not applicable »,
« - », vides traitées comme absentes ; SIRET (14 chiffres) fourni par erreur
-> 9 premiers chiffres repris comme SIREN ; **les bénéficiaires « sans TVA
possible » (République française) n'entrent jamais dans cette passe**, même
si Main Registration contenait une valeur exploitable pour cette ligne.

### v5_29 — Décalage N-1 sur la borne de début de chaque période
Un opérateur reconnu à compter de l'année N dans le référentiel OPETAT peut
déjà apparaître dans le FTS dès l'année **N-1**, en raison du décalage entre
l'engagement (comptabilisé dans le FTS) et sa reconnaissance officielle
comme opérateur de l'État (compilée l'année suivante). Chaque période
`[début, fin]` est donc désormais évaluée comme `[début-1, fin]` : une ligne
FTS de l'année (début-1) est acceptée, comme une ligne de n'importe quelle
année de la période déclarée. **Seule la borne de début est assouplie** ; la
borne de fin reste stricte (aucune indication contraire).

Exemple concret : Pass Culture, reconnu à partir de 2026 seulement
(SIREN 853318459, période [2026, 2026]) -> une ligne FTS de **2025** est
désormais « oui » (2025 = 2026-1), une ligne de 2024 reste « non ».

### v5_28 — Référentiel de périodes étendu (438 SIREN) + support multi-périodes
Nouveau fichier OPETAT.xlsx officiel intégré : 438 SIREN avec période(s) de
validité (contre 21 en v5_27), **443 SIREN au total** dans le référentiel
opérateurs (+7 nouveaux). Ce fichier est destiné à être **remis à jour
périodiquement** par l'utilisateur ; la procédure de nettoyage/intégration
ci-dessous est reproductible à l'identique sur une future version.

**Découverte structurante : un SIREN peut avoir PLUSIEURS périodes
successives sous des noms différents.** Le format de `OPERATEURS_ETAT.periodes`
passe donc de `SIREN -> [début, fin]` à `SIREN -> [[début1,fin1], [début2,fin2],
...]` (rétrocompatible avec l'ancien format à couple unique). Trois cas réels
confirmés dans les données : **Pôle emploi (2000-2024) -> France Travail
(2025-2026)**, même SIREN 130005481 — confirmation en conditions réelles du
mécanisme de renommage déjà géré par `_est_france_travail` au niveau de la
recherche TVA ; ENSTA Paris (->2025) -> ENSTA (2026->) ; Établissement public
du Mobilier National (->2025) -> Manufactures nationales, Mobilier-Sèvres
(2026->). Une ligne FTS est « oui » si son année tombe dans **au moins une**
des périodes du SIREN.

**Nettoyage nécessaire** (mêmes anomalies récurrentes qu'en v5_27, plus
deux nouvelles) :
- ligne d'en-tête dupliquée et agrégat GLOBAL retirés, comme avant ;
- **2 SIREN Luhn-invalides connus** (EPMQB 185552001, EPPGHV 775684014) :
  corrigés directement avec les valeurs déjà vérifiées officiellement en
  v5_23 (180092140, 391406956), sans nouvelle recherche ;
- **1 opérateur sans SIREN** (« Associations de coordination technique
  agricole... ») rapproché du SIREN ACTIA déjà résolu en v5_24 (338840564) ;
- **CROUS Clermont-Ferrand/Corte et Paris VIII/Université de Paris**
  réapparaissent avec le MÊME SIREN partagé à tort que celui déjà corrigé en
  v5_25 -- ré-appliqué à l'identique (182020107 = Corte seul, 199318270 =
  Paris VIII seul ; la date 2000-2026 associée à tort à Clermont-Ferrand est
  restituée à son vrai SIREN 186306973) ;
- **EPAURIF** : doublon avec deux dates de fin différentes (2024 vs 2026)
  pour la MÊME ligne -- résolu par la valeur la plus permissive (2026),
  **à confirmer par l'utilisateur**, ce n'est pas un cas de renommage comme
  Pôle emploi (le nom ne change pas entre les deux lignes).

**7 nouveaux SIREN ajoutés** (fusions/renommages d'universités constatés :
Besançon, Brest, Toulouse, Montpellier ; + Pass Culture et Solidéo Alpes
2030, ces deux derniers actifs uniquement à partir de 2026 -- restriction
réelle et significative, contrairement à la quasi-totalité des autres
entrées qui couvrent 2000-2026 par défaut). **5 anciens SIREN d'universités
correspondantes restent dans le référentiel SANS période** (faute de date de
fin officielle dans ce fichier) : Université de Besançon, Université Brest
Bretagne Occidentale, Université Paul Sabatier Toulouse III, Université
Montpellier III Paul Valéry -- probablement supersédées par les 4 nouvelles
ci-dessus, mais laissées sans restriction plutôt que de deviner une date de
bascule.

### v5_27 — Operateur_Etat dépend désormais de la période de validité + oui/non
La colonne `Operateur_Etat` passe de 1/0 à **« oui »/« non »**, et son calcul
tient maintenant compte de la **période de validité officielle** de chaque
opérateur (fichier OPETAT.xlsx : Opérateur, SIREN, Date de début, Date de
fin), comparée à la colonne FTS **« Year »** (déjà exploitée par
`enrichir_global`). Un SIREN opérateur de 2013 à 2024 donne « oui » pour les
lignes FTS de 2013 à 2024 inclus, « non » en dehors.

**Nettoyage nécessaire du fichier source**, plusieurs anomalies trouvées et
corrigées avant intégration :
- une ligne d'en-tête dupliquée au milieu du fichier (retirée) ;
- une ligne d'agrégat « ARS- GLOBAL (x18) » (retirée, jamais une entité) ;
- une **corruption en série du SIREN par glissement Excel** touchant 5
  opérateurs (ACMOSS, ADEME, AFR, ANDRA, ANSC), chacun dupliqué sur 20 lignes
  avec un SIREN incrémenté de 1 à chaque ligne (ex. ADEME : 385290309,
  385290310, 385290311…) alors que nom et dates restaient identiques ;
  SIREN canonique repris du référentiel déjà validé (436 SIREN), dates
  prises au mode des lignes cohérentes plutôt qu'à la première ligne venue.

**Constat honnête sur la portée réelle de cette mise à jour** : après
nettoyage, le fichier ne couvre que **21 opérateurs distincts** sur les 436
du référentiel (pas les 460 lignes brutes, qui n'étaient que des doublons).
Les 21 partagent tous exactement la même période **2000-01-01 / 2026-12-31**
— aucune restriction réelle n'apparaît donc dans ce fichier ; l'exemple
« 2013 à 2024 » de la demande était illustratif. Le mécanisme a été validé
sur un cas synthétique injecté pour cette raison (voir tests). Les 415
autres SIREN du référentiel restent **sans restriction de date**, faute de
donnée — comportement identique à avant v5_27, non silencieux (compté et
affiché à l'exécution). Année FTS illisible -> pas de restriction, par
prudence (compté et affiché).

### v5_26 — Dernier opérateur résolu : INSHEA -> INSEI (130000383)
L'utilisateur a fourni le SIREN de l'INSHEA, confirmé par recoupement officiel
indépendant (Wikipédia, annuaire-entreprises.data.gouv.fr, societe.com) :
**130000383**. L'établissement a changé de nom en 2023 (décret du 3 mars
2023) — l'Institut national supérieur de formation et de recherche pour
l'éducation des jeunes handicapés et les enseignements adaptés (INSHEA) est
devenu l'Institut national supérieur de formation et de recherche pour
l'éducation inclusive (INSEI), même SIREN, siège à Suresnes (92150).

Référentiel : 435 -> **436 SIREN**, 435 programmes.

**Reste non résolu (1 seul)** : Université de Paris (établissement
expérimental), qui continue d'hériter à tort, dans le fichier source, du
SIREN de l'Université Paris VIII (199318270, confirmé) — vrai doublon entre
deux établissements distincts, toujours exclu plutôt que deviné. Tous les
autres opérateurs de l'annexe Jaune 2026 sont désormais résolus.

### v5_25 — Correctif : le CROUS de Corte n'était pas un doublon faux
L'utilisateur a confirmé, et le recoupement officiel indépendant valide
(annuaire-entreprises.data.gouv.fr, societe.com, Wikidata — adresse 22 avenue
Jean Nicoli, Corte, « Crous de Corse ») : **182020107 est bien le SIREN du
CROUS de Corte**. La v5_23 l'avait exclu à tort en le confondant avec un
doublon faux, alors qu'il s'agissait en réalité de DEUX erreurs indépendantes
qui coïncidaient : la recherche avait correctement trouvé le SIREN de Corte,
mais avait aussi recopié à tort cette même valeur sur la ligne Clermont-Ferrand
(déjà corrigée séparément vers 186306973 en v5_23, cette correction reste
valide et confirmée). Une fois Clermont-Ferrand corrigé, 182020107 était donc
libre et correct pour Corte — ajouté au référentiel.

Référentiel : 434 -> **435 SIREN**, 434 programmes.

**Reste non résolu (2)** : l'INSHEA (toujours sans résultat de recherche) et
l'Université de Paris (établissement expérimental), qui continue d'hériter à
tort du SIREN de l'Université Paris VIII (199318270, confirmé) — vrai
doublon celui-là, toujours exclu plutôt que deviné.

### v5_24 — 4 opérateurs supplémentaires résolus (2e passe de recherche)
Sur les 7 opérateurs non résolus de la v5_23, **4 sont désormais intégrés**,
chacun vérifié par recoupement officiel indépendant avant intégration :
- **Casa de Velázquez** (Madrid) -> 180044125
- **ACTIA** (Association de coordination technique pour l'industrie
  agro-alimentaire) -> 338840564
- **HESAM Université** (ComUE) -> 130021447
- **Université Paris-Saclay** -> 130026024

Référentiel : 430 -> **434 SIREN**, 433 programmes.

**Restent non résolus (3)** : l'INSHEA (institut pour l'éducation des jeunes
handicapés, toujours sans résultat de recherche) ; le CROUS de Corte et
l'Université de Paris (établissement expérimental), qui continuent d'hériter
À TORT, dans le fichier source, du SIREN d'une autre entité réelle (CROUS
Clermont-Ferrand et Université Paris VIII respectivement) — toujours
volontairement exclus plutôt que de leur assigner un SIREN incertain.

### v5_23 — Référentiel des opérateurs de l'État étendu à 430 SIREN (annexe Jaune complète)
Le référentiel OPERATEURS_ETAT passe de 70 à **430 SIREN**, à partir de la
réconciliation complète de l'annexe Jaune 2026 (455 lignes, 16 agrégats de
famille jamais recherchés, 438 entités réelles) menée par
`Reconciliation_Operateurs_Etat_Complet.ipynb`. Chaque nouveau SIREN pointe
vers son **programme chef de file** (colonne « Programme chefs de file » du
Jaune), avec la même mécanique que les 69 précédents (`Programme_Operateur`).
Contrôles effectués avant intégration :
- **Cohérence totale** avec les 70 SIREN déjà validés : retrouvés à l'identique
  dans le nouveau fichier, zéro conflit.
- **3 corrections par recoupement officiel** (Luhn invalide ou doublon
  détecté) : musée du quai Branly (185552001 -> 180092140), EPPGHV/la
  Villette (775684014 -> 391406956), CROUS Clermont-Ferrand (182020107,
  partagé à tort avec Corte -> 186306973).
- **2 doublons faux écartés** (SIREN partagé par erreur entre deux entités
  réellement distinctes, aucun remplacement deviné) : CROUS de Corte,
  Université de Paris (établissement expérimental) — restent non résolus.
- **5 opérateurs sans résultat** de recherche, dont Université Paris-Saclay —
  restent non résolus, à traiter séparément.
- 1 SIREN (Génopôle) conservé sans programme (absent de la feuille Jaune
  utilisée pour ce mappage) — cohérent avec le traitement déjà appliqué à
  France Travail.
La reconnaissance par famille (ARS/Agences de l'eau/Parcs nationaux, v5_18)
reste inchangée en repli.

### v5_20 à v5_22 — Section NAF 2025 (rattrapage de changelog)
Ces trois versions, présentes dans le code mais non journalisées à l'époque,
ont ajouté la colonne « Section_NAF » (nomenclature NAF 2025 / NACE Rév. 2.1,
fichier utilisateur), placée juste après « Activite_principale » : la section
(lettre A-V) est déduite de la division (2 premiers chiffres du Code_NAF_APE)
via une table hiérarchique embarquée (`NAF2025_SECTIONS`). Point d'attention
documenté dans le code : les lettres de section 2025 diffèrent du NAF rév.2/
2008 (K, L, M... décalées), et la division 45 (commerce/réparation auto) a
disparu de la nomenclature -> les codes 45.* n'ont pas de section 2025 (case
vide, signalée à l'exécution).
### v5_19 — Documentation intégrale + correctif JSON critique
1. **Correctif critique** : l'ajout manuel du GIP Plateforme de l'inclusion
   (SIREN 130030133) avait cassé le JSON du bloc OPERATEURS_ETAT — deux
   virgules manquantes, le chargement des référentiels échouait entièrement.
   Corrigé. Le SIREN est confirmé officiel (annuaire-entreprises.data.gouv.fr,
   siège 13003013300016, TVA FR61130030133). La valeur du programme, qui
   recopiait le nom de l'opérateur, est remplacée par le vrai programme chef
   de file : « 102 – Accès et retour à l'emploi ». Référentiel : 69 SIREN.
2. **Commentaires** : chaque cellule reçoit un en-tête expliquant son rôle,
   ses entrées/sorties et ses pièges ; chaque fonction non figée reçoit une
   explication de son intention (le « pourquoi », pas seulement le « quoi »).
   Les 5 fonctions FIGÉES sont documentées par un bloc placé AU-DESSUS de
   leur définition : leur corps reste identique bit à bit (garantie de
   non-régression préservée et revérifiée).
3. **README.md** : manuel complet de réexécution et de maintenance.

### v5_18 — Référentiel des opérateurs corrigé + reconnaissance par famille
1. Table des opérateurs de l'État intégralement remplacée par la version
   vérifiée (68 SIREN, puis 69 avec le GIP inclusion) : **30 SIREN étaient
   erronés** dans la table précédente — détecté en comparant deux versions du
   fichier utilisateur, confirmé par recoupement officiel (ex. Société du
   Grand Paris devenue Société des grands projets : 525046017 et non
   838640613). La table des programmes a été reconstruite en cohérence, sinon
   les programmes des entités corrigées se seraient vidés silencieusement.
2. **Reconnaissance par famille** (ARS, Agences de l'eau, Parcs nationaux) :
   repli par motif de nom lorsque l'identification par SIREN échoue, pour
   couvrir les variantes de libellé et les créations futures. Vérification
   faite : 18/18 ARS, 6/6 agences de l'eau et 11/11 parcs nationaux figurent
   déjà nominativement au référentiel ; les lignes « GLOBAL » sont des
   agrégats de famille, jamais des entités (donc sans SIREN, jamais cherchées).

### v5_17 — Programme des opérateurs + libellé NAF + révisions SGAE
1. **« Programme_Operateur »** (bleue, juste après Operateur_Etat) : programme
   chef de file de l'opérateur de l'État (colonne « Programme chefs de file »
   du fichier utilisateur, 74 SIREN) ; « AUTRE » si la ligne n'est pas un
   opérateur. France Travail (130005481, ajouté v5_6) n'a pas de programme
   dans le fichier source -> case vide, à compléter par l'utilisateur.
2. **« Activite_principale »** (bleue, juste après Code_NAF_APE) : libellé
   officiel NAF rév. 2. L'API ne renvoyant que le CODE (vérifié dans son code
   source), la nomenclature INSEE (732 sous-classes, source publique
   SocialGouv/codes-naf validée par contrôles croisés) est embarquée et jointe
   sur le code — uniforme sur toutes les lignes (passe 1, passe 2, alias,
   Étape E).
3. **Référentiel SGAE : 13 catégories révisées** (Classeur1) : SIVOM/SIVU/
   syndicats mixtes/inter-hospitalier -> « Etablissement public », régies ->
   « Collectivité territoriale », 0000/7490/9110 -> « Autre », 3110 ->
   « Entités étrangères », 5195 -> « Association ». ATTENTION : ces nouveaux
   libellés sont au SINGULIER alors que les catégories historiques sont au
   pluriel (« Etablissements publics ») — divergence signalée, harmonisation
   sur demande.

### v5_16 — 2e campagne de corrections (Classeur1.xlsx) + % TVA >= 96
Base : v5_14 (l'utilisateur repart de cette version). Intégration du rapport
annoté « Classeur1.xlsx » (79 lignes) :
• 35 CORRECT + 12 CORRIGE + 3 TROUVE + 1 CORRIGE TVA -> 61 SIREN forcés à 100
  par correspondance EXACTE du nom (Luhn 61/61 ; SIREN nettoyés des \n/espaces ;
  pour les CORRIGE, la colonne SIRET du rapport était l'ancienne valeur : le
  SIRET est repris de la fiche SIRENE du bon SIREN).
• 10 CORRIGE SIRET -> en plus du SIREN (déduit du SIRET fourni), le SIRET de
  l'ÉTABLISSEMENT indiqué est FORCÉ dans le fichier (remplace celui du siège).
• 18 False -> paires nom/SIREN définitivement interdites (fusionnées avec la
  1re campagne ; ces lignes redeviennent NON_TROUVE -> AUTRE ou A CHERCHER).
• Rapport : nouvelle statistique « TVA score >= 96 » (nombre, % des TVA
  trouvées, % des bénéficiaires recherchés), en console et dans l'onglet Resume.
Conflit signalé : le SIREN 753110493 (ADICE) est interdit pour la graphie
« ...*A.D.I.C.E » mais forcé pour la graphie sans suffixe — appliqué tel
qu'annoté, à confirmer.

### v5_14b — Référentiel SGAE : ajout de 5498 (EURL) et 5720 (SASU) -> « Entreprises »
Les deux codes INSEE 5498 (SARL unipersonnelle / EURL) et 5720 (SASU), présents
dans la nomenclature INSEE mais absents du fichier REF_SGAE initial, sont
rattachés à « Entreprises » (comme 5499 SARL et 5710 SAS). Le référentiel passe
de 260 à 262 codes.

### v5_14 — Colonne « Référentiel SGAE » (statut juridique simplifié)
Nouvelle colonne bleue « Référentiel SGAE », placée JUSTE AVANT « Forme_juridique »,
qui simplifie le statut juridique selon le référentiel fourni (REF_SGAE.xlsx,
260 codes). La liaison se fait sur le CODE de catégorie juridique INSEE niveau III
(4 chiffres, extrait des parenthèses de « Forme_juridique ») et non sur le libellé
(jointure fiable). Exemples : université (7383/7389) -> « Etablissements publics »,
région (7230) -> « Collectivités territoriales », SARL/SAS -> « Entreprises »,
association (9220) -> « Associations et Fondations ». Les codes absents du
référentiel laissent la case vide et sont signalés à l'exécution ; à noter que
5498 (SARL unipersonnelle), 5720 (SASU) et les codes de personnes physiques
(1100-1900) n'y figurent pas encore.

### v5_13 — Correction du statut « Cessée » faussé (universités et EPA)
Nouvelle Étape E (additive, hors moteur figé). De nombreuses universités et
établissements publics ont un SIREN historique CESSÉ qui coexiste, sous le même
nom, avec le SIREN actuel toujours actif ; le moteur figé (PENALISER_RADIEES =
False) retient parfois le prédécesseur, d'où un état « Cessée » erroné qui
fausserait les analyses. L'Étape E recherche, pour chaque ligne « Cessée », le
successeur ACTIF de nom EXACTEMENT identique et de même département (filtre
etat_administratif=A de l'API) ; si un successeur actif unique existe, elle
remplace SIREN, TVA, SIRET, forme juridique, niveaux, code NAF, adresse et état
(-> « En activité »). Les entités réellement fermées sans successeur restent
« Cessée ». Recherche mise en cache par (nom, département). Ex. : Université
Clermont Auvergne, Université Grenoble Alpes -> SIREN actif rétabli.

### v5_12 — Rapport : lignes « à chercher à la main » surlignées en orange
Dans le RAPPORT uniquement : toute ligne dont le score est < 96 ET dont le
bénéficiaire a perçu au total >= 300 000 € est surlignée en ORANGE (#FFC000),
en PRIORITÉ sur les couleurs de statut (vert/rouge/gris). Les lignes EXCLU
(personnes physiques, République française…) ne sont pas surlignées : il n'y a
rien à y chercher. Ce sont exactement les lignes marquées « A CHERCHER » dans
le fichier (v5_7), désormais repérables d'un coup d'œil dans le rapport.
Le fichier FTS n'est pas modifié par ce changement.

### v5_11 — Graphie corrigée : « Grand-Est » AVEC tiret (référentiel utilisateur)
Correction de sens sur la v5_10 : la graphie qui fait foi est celle du
référentiel départements->NUTS3/Région FOURNI PAR L'UTILISATEUR — « Grand-Est »
avec tiret. C'est donc la table nuts2_vers_region qui est harmonisée dessus au
chargement (et non l'inverse) : Région_FR affiche « Grand-Est » quel que soit le
chemin de dérivation (CP, CP SIRENE, NUTS2, pays, ville). La classification
Metro/RUP/PTOM reste insensible aux tirets/espaces (v5_10), donc toujours
correcte dans les deux graphies.

### v5_10 — Le référentiel NUTS fait foi pour Région_FR (fix « Grand-Est »)
Bug corrigé : les lignes dont la Région_FR venait du NUTS2 (« Grand Est »,
graphie de la table nuts2_vers_region) n'étaient pas classées Metro/RUP/PTOM,
car le repli comparait à la graphie « Grand-Est » (tiret) de la table
départements. Deux corrections : ① au chargement, la table départements est
HARMONISÉE sur la graphie du référentiel NUTS (le NUTS fait foi — décision
utilisateur) : Région_FR est désormais identique quel que soit le chemin de
dérivation ; ② le repli région de l'Étape C devient insensible aux
tirets/espaces (défense en profondeur), et son repli pays gère les traits
d'union (« Saint-Barthélemy » -> clé « saint barthelemy » -> PTOM, trou repéré
au passage). Aucune donnée nouvelle : uniquement des graphies existantes.

### v5_9 — « Ville (SIRENE) » retirée des fichiers exportés
Comme « Adresse (SIRENE) » et « Code postal (SIRENE) » (v5_4), la colonne
« Ville (SIRENE) » est désormais retirée du fichier final. Les trois colonnes
restent calculées et utilisées EN INTERNE (repli ville du nettoyage géographique,
désambiguïsation 977/978, Metro/RUP/PTOM) — seuls les résultats consolidés
(« NUTS2 corrigé », « Région_FR », « NUTS3 FR », « NUTS3_Numéro »,
« Metro/RUP/PTOM ») apparaissent dans le fichier.

### v5_8 — France Travail localisé par l'adresse + République française sans TVA
1. **Pôle emploi / France Travail** : plus de forçage aveugle du siège. Le
   bénéficiaire est recherché sous « FRANCE TRAVAIL » AVEC l'adresse, la ville et
   le CP de la ligne -> l'ÉTABLISSEMENT local (SIRET de l'agence/direction
   régionale) est retrouvé quand l'adresse le permet ; sinon repli sur le siège.
   Dans tous les cas SIREN = 130005481 (vérifié), TVA FR19130005481, score 100,
   « Bénéficiaire corrigé » = FRANCE TRAVAIL.
2. **Liste des opérateurs de l'État** : le SIREN 348377318 (« Pôle emploi » du
   fichier source — en réalité le Comité d'établissement Pôle emploi Auvergne)
   est RETIRÉ sur confirmation utilisateur ; France Travail (130005481) reste ->
   Operateur_Etat = 1 pour toutes les lignes Pôle emploi/France Travail.
3. **République française** : structure non identifiable (décision utilisateur) —
   AUCUNE TVA ne lui est plus jamais attribuée, aucune recherche lancée (statut
   EXCLU / SANS_TVA_MANUEL au rapport), et jamais « A CHERCHER » malgré ses
   montants (-> AUTRE).

### v5_7 — Corrections du rapport « correction_1 » + seuil fichier 96 + « A CHERCHER »
1. **46 TVA confirmées + 4 SIREN corrigés** (rapport annoté correction_1.xlsx) :
   forcés à 100 via correspondance EXACTE du nom FTS (priorité absolue, SIRET du
   siège récupéré dans SIRENE, « Bénéficiaire corrigé » = nom officiel). La
   correspondance exacte évite tout débordement (« THALES » confirmé ne capte
   pas « THALES ALENIA SPACE »). Clés de Luhn vérifiées : 50/50 valides.
2. **4 faux positifs définitivement écartés** : ces SIREN/TVA ne sont PLUS JAMAIS
   renvoyés pour ces bénéficiaires, quel que soit le chemin (statut NON_TROUVE,
   stratégie FAUX_POSITIF_ECARTE, visible dans le rapport).
3. **Fichier FTS** : seules les TVA de score >= 96 y figurent (SEUIL_FICHIER_TVA
   95 -> 96). Le rapport garde TOUJOURS tous les résultats.
4. **« A CHERCHER »** : une ligne SANS TVA retenue dont le bénéficiaire a perçu
   au total >= 300 000 € (somme « Beneficiary's contracted amount », même règle
   que le rapport) reçoit « A CHERCHER » dans la colonne TVA au lieu de
   « AUTRE » — pour recherche manuelle ciblée. SIREN/SIRET/forme restent AUTRE.

### v5_6 — Pôle emploi corrigé et forcé sur FRANCE TRAVAIL (SIREN 130005481)
Tout bénéficiaire dont le nom contient « POLE EMPLOI » ou « FRANCE TRAVAIL »
est désormais FORCÉ (alias SIREN à priorité absolue, hors moteur figé) sur le
SIREN officiel 130005481 de FRANCE TRAVAIL (ex-Pôle emploi, EPA créé en 2008 —
vérifié sur plusieurs sources concordantes) : TVA calculée depuis ce SIREN,
« Bénéficiaire corrigé » = FRANCE TRAVAIL, fiche SIRENE complète. Le SIREN
130005481 est aussi ajouté à la liste des opérateurs de l'État -> ces lignes
reçoivent Operateur_Etat = 1. Contrôle du fichier source : le SIREN 348377318
libellé « Pôle emploi » y est en réalité le « Comité d'établissement Pôle
emploi Auvergne » (association) — conservé dans la liste par prudence, mais
probablement à retirer (à confirmer).

### v5_5 — Étape D : marquage des opérateurs de l'État (colonne « Operateur_Etat »)
Nouvelle colonne booléenne bleue « Operateur_Etat » (1/0), placée juste APRÈS
« Etat_entreprise » : 1 si le SIREN de la ligne (colonne SIREN, sinon déduit de
la TVA, sinon du SIRET) figure dans la liste des opérateurs de l'État FOURNIE
PAR L'UTILISATEUR (Operateurs_Etat_SIRET_TVA.xlsx), désormais embarquée dans le
notebook (référentiel OPERATEURS_ETAT : 74 SIREN uniques, clés de Luhn
vérifiées). Notes de contrôle : « GIP Plateforme de l'inclusion » est sans
SIREN dans le fichier source (non détectable) ; deux SIREN portent chacun deux
libellés (Agence de l'eau Seine-Normandie, SGP) — sans effet sur le marquage.
Moteur TVA figé inchangé.

### v5_4 — Fichier allégé + téléchargements robustes
1. **« Adresse (SIRENE) » et « Code postal (SIRENE) » retirées du fichier final**
   (demande utilisateur). Elles restent calculées et utilisées EN INTERNE par le
   nettoyage géographique et l'Étape C pour compléter « NUTS2 corrigé »,
   « Région_FR », « NUTS3 FR », « NUTS3_Numéro » et « Metro/RUP/PTOM » ; seule
   « Ville (SIRENE) » (après City) reste visible dans le fichier.
2. **Plus de plantage en fin d'exécution** quand aucune TVA n'était manquante :
   le rapport n'étant pas créé (rapport vide), son téléchargement était tenté
   quand même (FileNotFoundError). Désormais seuls les fichiers réellement créés
   sont téléchargés, avec un message explicite pour les autres.

### v5_3 — « NUTS2 corrigé » complété depuis le département (SIRENE, pays, ville)
Quand le NUTS2 brut est vide ou masqué, la colonne AJOUTÉE « NUTS2 corrigé » est
désormais complétée avec l'ancienne région (nomenclature Eurostat NUTS 2021)
déduite du département de la ligne — lui-même issu, dans l'ordre, du CP brut, du
CP du siège SIRENE (Étape A3), du pays ou de la ville. Un NUTS2 brut non vide
n'est JAMAIS remplacé. Nouveau bloc `nuts2_par_departement` dans GEO_FRANCE :
101 départements croisés avec le fichier officiel Eurostat NUTS_AT_2021.csv
(gisco-services.ec.europa.eu) et validés contre la table départements ET
nuts2_vers_region du référentiel ; les 7 COM hors NUTS (975, 977, 978, 984,
986, 987, 988) suivent la convention du référentiel « NUTS2 = NUTS3 ».
Rien d'autre ne change : Région_FR, NUTS3 FR, NUTS3_Numéro et Metro/RUP/PTOM
étaient déjà complétés par la v5_2 ; le moteur TVA reste figé et intact.

### v5_2 — Complétion SIRENE des champs géographiques + replis Metro/RUP/PTOM
Trois changements, tous ADDITIFS (moteur de recherche TVA figé, intact) :
1. **Étape A3 — Complétion SIRENE** : pour les lignes France dont l'Adresse, la
   Ville ou le Code postal sont vides ou masqués par le FTS (« . », « - »,
   « ***** ») mais qui ont un SIREN identifiable (colonne SIREN, TVA ou SIRET),
   l'adresse du SIÈGE est lue dans SIRENE et versée dans TROIS NOUVELLES colonnes
   bleues : « Adresse (SIRENE) » (après Address), « Ville (SIRENE) » (après City),
   « Code postal (SIRENE) » (après Postal code). Les colonnes d'origine ne sont
   jamais modifiées. Cache global des fiches (`fiche_siren`) partagé avec la
   passe 2 : aucun double appel API.
2. **Région_FR / NUTS3** : nouveau repli « CP SIRENE » (entre le CP brut et le
   pays) ; la « Ville (SIRENE) » alimente aussi le repli ville existant. Les
   vraies personnes physiques (jamais de TVA/SIREN — exclusion volontaire du
   moteur) ne peuvent être complétées QUE par leur ville, via le référentiel
   VILLES_FR déjà intégré (31 950 communes officielles, homonymes ambigus exclus :
   « SAINT-DENIS » reste volontairement non classé).
3. **Metro/RUP/PTOM** : deux replis quand le CP brut est inexploitable —
   ① code postal du siège SIRENE (même logique 97133/97150/977xx-978xx),
   ② Région_FR déjà dérivée (elle-même issue du CP SIRENE, du NUTS2, du pays ou
   de la ville), traduite en territoire par les référentiels existants
   (RUP art. 349 TFUE / PTOM annexe II TFUE) — aucune donnée nouvelle en dur.
   Ex. : « GWADLOUP*REGION » sans CP → siège SIRENE 971xx → RUP ; personne
   physique à « LE TAMPON » → 974 → La Réunion → RUP.

### v3_18 — Classification des projets + rapport enrichi + « AUTRE » sur le fichier
Quatre changements :
1. **Classification des projets** — deux colonnes « N° projet » et « Type de projet »
   juste après « Reference of the Legal Commitment (LC) ». Un projet est MONO s'il
   pointe sur un et un seul bénéficiaire, COLLABORATIF s'il en finance plusieurs,
   INDÉTERMINÉ si la référence LC est vide. Le bénéficiaire est identifié par son
   NOM (« Bénéficiaire corrigé », ramené au nom de base — étoiles et suffixes
   absorbés) ; deux lignes sont EN PLUS reconnues comme le même bénéficiaire si
   elles partagent la même TVA (identifiant le plus fort) ou la même ADRESSE
   COMPLÈTE (pays + rue + CP + ville — cas des noms écrits en plusieurs langues).
   Fusion transitive par projet (union-find). Lignes anonymisées sans adresse :
   chacune compte pour un bénéficiaire (convention FTS 1 ligne = 1 bénéficiaire).
2. **Rapport** — nouvelle colonne « Montant_total_beneficiaire » (somme perçue par le
   bénéficiaire) ; colonnes « Detail_Score » et « Etat_entreprise » (actif) retirées.
   Le rapport contient TOUJOURS toutes les TVA (avec score + Retenu_fichier).
3. **Fichier FTS** — seules les TVA de score ≥ 95 y figurent (les autres, effacées du
   fichier, restent dans le rapport pour vérification/forçage manuel).
4. **« AUTRE » sur le fichier** — dans le fichier FTS, les valeurs vides de TVA, SIREN,
   SIRET et forme juridique deviennent « AUTRE » (le rapport garde les vraies valeurs).

### v3_17b — Référentiel sous-catégories complété (bloc Hors CFP)
Correction des avertissements : les 8 programmes du 3ᵉ bloc « Hors CFP » de la
liste (O.0.1, O.0.OTH, S.0.1, S.0.2, S.0.4, S.0.11, 9.0.2, 9th EDF, + 10th/11th EDF)
sont ajoutés à SOUS_CATEGORIES (185 programmes). O.0.4 - Ukraine Facility, vrai
programme 21-27 malgré son préfixe « O. », est mis en liste blanche
(`prefixe_special_dans_cfp`) pour ne plus déclencher l'avertissement hors-CFP.
Résultat : plus aucun « Sous catégorie = NA » ni fausse alerte de préfixe.

### v3_17 — Géo par la ville, colonnes CFP repositionnées, filtre TVA<95 hors fichier
Quatre changements :
1. **Géo par la VILLE** — quand une ligne n'a ni NUTS2 ni code postal mais a une
   ville (colonne City), la Région_FR, le NUTS3 et le NUTS3_Numéro sont dérivés
   ville → département → région/NUTS3 via un référentiel communes intégré
   (`VILLES_FR`, 31 950 communes officielles). Sur ton Classeur1, 215/222 lignes
   récupèrent leur région. Homonymes non tranchés (pas de CP) et villes étrangères
   restent vides.
2. **Colonnes CFP** — « Sous catégorie » avant « Programme name », puis « Période
   CFP » et « Dépense CFP » juste APRÈS « Programme name ».
3. **Rapport TVA** — inchangé : toutes les TVA trouvées y figurent (+ colonne
   « Retenu_fichier » Oui/Non).
4. **Fichier FTS** — les TVA de score < 95 (`SEUIL_FICHIER_TVA`) sont EFFACÉES du
   fichier (avec SIREN/SIRET/forme/état), car sous ce seuil le risque de faux
   positif est élevé. Elles restent visibles dans le rapport. La v3_16 (nom parfait
   → pénalités géo neutralisées) assure que les vraies TVA au nom exact atteignent 95+.

### v3_16 — Nom parfait : neutralisation de la perte de points due à l'adresse
Quand le nom du bénéficiaire correspond **exactement** à celui de l'annuaire
(exactitude = 100 via `_exactitude`, qui neutralise les mots juridiques et pénalise
les mots en trop), les **pénalités géographiques** du scoring (adresse `-5%`, ville
`-5%`, département `-3%`) ne sont **plus appliquées**. Une adresse divergente (FTS
souvent différent de l'annuaire : déménagement, adresse de projet) ne fait donc plus
chuter une identité de nom certaine — ces vraies TVA gardent un score élevé. Les
bonus géographiques restent actifs, et les noms seulement approchants (ex. « SCI
ACTION CONTRE LA FAIM PATRIMOINE » vs « ACTION CONTRE LA FAIM ») restent pénalisés
normalement. Modification chirurgicale du scoring (aucun autre critère touché) ; ne
peut qu'augmenter ou laisser égal le score, jamais le baisser.

### v3_15b — 6 SIREN manquants trouvés sur l'annuaire des entreprises
Les 6 renommages qui restaient sans SIREN ont été identifiés sur annuaire-entreprises.data.gouv.fr
et ajoutés à la table forcée (désormais 68 SIREN / 163 noms, **couverture 90/90 des corrections
du rapport**) :
- INRAE Transfert → 433960762 ; Autorité Nationale des Jeux (ex-ARJEL) → 130011836 ;
  Département de l'Allier → 220300016 ; Agence de Développement pour la Normandie (AD Normandie)
  → 200006500 ; Grand Est Développement (Grand E-Nov+) → 434049953 ; Opus (ex Union APARE-CME)
  → 316713015.
Pour ces renommages, « Bénéficiaire corrigé » prend le nom officiel de l'annuaire
(INRAE TRANSFERT, AUTORITE NATIONALE DES JEUX, OPUS…) via `_NOM_OFFICIEL_PAR_SIREN`.
0 faux positif vérifié sur les 7 059 lignes France.

### v3_15 — Colonne SIREN, noms officiels annuaire, Région via NUTS2, progression bénéficiaires
Cinq changements :
1. **Table forcée corrigée** — ACTION CONTRE LA FAIM*ACF (et tous les doublons) reçoivent
   enfin le BON SIREN (318990892). 62 SIREN vérifiés / 145 noms, règle par type
   d'annotation (Corriger → variante sans étoile ; Ajouté/Copié → SIREN propre ; Voir/TVA
   copiée → SIREN de la cible).
2. **Progression** — affiche « n/Nombre de bénéficiaires » (uniques) au lieu du nombre de lignes.
3. **Région_FR via NUTS2** — pour les lignes sans code postal (personnes physiques, répliques
   françaises), la Région_FR est dérivée du NUTS2 corrigé (référentiel `nuts2_vers_region`),
   à défaut de la ville ; NUTS3/numéro renseignés quand identifiables (RUP). +25 lignes
   récupérées sur les données réelles.
4. **Noms officiels annuaire** — pour les SIREN forcés (corrections manuelles), « Bénéficiaire
   corrigé » prend le nom officiel de l'annuaire des entreprises (via l'API SIRENE).
5. **Colonne SIREN** ajoutée juste avant SIRET. Forme juridique et état de l'entreprise
   dépendent du SIREN (unité légale, identifiant permanent), l'adresse/SIRET de l'établissement.

### v3_14b — Corrections vérifiées FORCÉES via _ALIAS_SIREN (dans le code)
Les TVA que tu as corrigées à la main dans le rapport (RAPPORT_V4) sont désormais
**forcées de façon déterministe** : 60 SIREN vérifiés → 144 noms FTS, intégrés
dans `_ALIAS_SIREN_CORRECTIONS` (cellule moteur) et fusionnés dans `_ALIAS_SIREN`.
Dans `rechercher_tva_plus`, l'alias SIREN a maintenant la **priorité absolue** :
un bénéficiaire aliasé reçoit son bon SIREN/TVA même si la recherche normale
trouverait autre chose (fini « SCI ACTION CONTRE LA FAIM » — `ACTION CONTRE LA
FAIM*ACF` est forcé sur 318990892). Vérifié : 0 faux positif sur les 7 059 lignes
France, parité intacte pour les non-aliasés. La réconciliation de doublons et le
repli meilleur-candidat restent en place pour les cas non listés.
Restent 9 renommages sans SIREN dans le rapport (INRAE Transfert, Opus, ANJ,
Grand-Est…) — donne-moi leurs SIREN pour les forcer aussi.

### v3_14 — Notebook autonome (données JSON réintégrées dans le code)
Toute la logique de la v3_13 (exclusions restreintes aux personnes physiques,
réconciliation de doublons, repli « meilleur candidat SIRENE » contrôlé par
exactitude) est conservée. Les 6 référentiels (`ETATS`, `SOUS_CATEGORIES`,
`CFP_REGLES`, `PAYS_ALIAS`, `GEO_FRANCE`, `FORMES_JURIDIQUES`) sont désormais
**embarqués dans la cellule 1bis** : le notebook est autonome, aucun fichier
externe à déposer hormis le fichier FTS. Pour modifier un référentiel, édite son
bloc JSON dans la cellule 1bis.

### v3_13b — Repli « meilleur candidat SIRENE existant » (contrôlé par exactitude)
Ajout : pour un bénéficiaire non trouvé (score composite < seuil 80), le moteur
**récupère le meilleur candidat SIRENE qu'il avait déjà trouvé** (au lieu de le
jeter) — MAIS uniquement si le nom du candidat correspond très bien au bénéficiaire
(exactitude ≥ SEUIL_EXACTITUDE_REPLI = 90). On récupère ainsi de vraies entités
sous le seuil sans réintroduire les faux positifs (« SCI ACTION CONTRE LA FAIM »,
au nom divergent, reste rejeté). Additif : ne modifie jamais un résultat déjà
trouvé. Seuil ajustable en tête du moteur.

### v3_13 — Exclusions restreintes + réconciliation de doublons (tout dans le code, aucune donnée en dur)
Deux corrections issues du rapport annoté (RAPPORT_V4), **entièrement algorithmiques,
sans table de correction codée en dur** :
1. **Exclusion = personnes physiques uniquement** (dans le moteur) : les entités
   institutionnelles (Conseil de l'Europe, ESA, ONU, République française…) ne sont
   PLUS exclues et sont désormais recherchées.
2. **Réconciliation de doublons** (étape A2, après la recherche TVA) : deux lignes
   désignant la même structure mais écrites différemment (étoile, suffixe
   « ASSOCIATION »…) sont regroupées par nom de base ; la variante ayant trouvé un
   résultat fiable propage sa TVA/SIREN aux jumelles mal ou non résolues, et
   « Bénéficiaire corrigé » prend le nom de base. Le code CHERCHE lui-même la structure
   correspondante à chaque ligne — rien n'est codé en dur. Garde-fous : pas de fusion
   si le groupe contient plusieurs TVA fiables distinctes (ex. deux établissements
   OCDE), ni si une ligne a déjà une TVA fiable propre.

### v3_6 — Fichier BRUT MONDIAL en entrée, 3 fichiers en sortie
**Entrée** : l'export FTS brut **tous pays**. **Sorties** : ① fichier **GLOBAL enrichi**,
② fichier **FRANCE** (filtré `PAYS_FR`, avec toutes les colonnes du global **plus**
les colonnes France), ③ **rapport TVA**.

**Étape 0 — enrichissement global (toutes lignes, colonnes BDD7EE)** :
« Name of beneficiary (nettoyé) » (étoiles finales retirées) ; « FR/UE/UK/AELE/AUTRE »
et « Etats » après le pays (table ETATS fournie, DOM-TOM/RUP → FR, ISO 3166 strict en
secours, Kosovo = XK) ; « Période CFP » et « Dépense CFP » après « Budget line name » ;
« Sous catégorie » juste avant « Programme name ».

**Définitions CMFE appliquées** — Période CFP = CFP de **conventionnement**, déduite de
l'**année** (2014-2020 → 14-20, 2021-2027 → 21-27) ; Dépense CFP = CFP auquel se
**rattache** la somme : « Hors CFP » pour les instruments hors cadre (O.x / S.x / 9.0.x /
FED), CFP antérieur si le nom de ligne contient « completion of previous », « former »,
« prior to 2021/2014/2007 » ou « (2007 to 2013) », sinon plage de la période.

**Étapes A/B/C (France uniquement)** : TVA (moteur figé, données brutes) ; nettoyage
géographique (CP corrigé, Région_FR, NUTS3) ; « Metro/RUP/PTOM » après le pays
(RUP art. 349 TFUE, PTOM annexe II TFUE, codes postaux La Poste).

### v3_7 — Données séparées du code
Tous les référentiels sont désormais dans **5 fichiers JSON externes** que le
notebook interroge : `ETATS.json`, `SOUS_CATEGORIES.json`, `CFP_REGLES.json`,
`PAYS_ALIAS.json`, `GEO_FRANCE.json`. **À déposer avec le fichier FTS** dans la
cellule de chargement (une seule boîte de dialogue, sélection multiple). Pour
mettre à jour un référentiel (nouveau programme, nouvel instrument hors CFP…),
il suffit d'éditer le JSON — plus aucune donnée dans le code. Seules exceptions,
inchangées : `PAYS_FR` (cellule moteur, figée) et la table des formes
juridiques INSEE (mécanisme hybride téléchargement + secours existant).
La colonne « Metro/RUP/PTOM » est placée juste **avant « Région_FR »**.

### v3_11 — Référentiels par chemin, chargement minimal
Les 6 JSON restent des fichiers séparés (aucune donnée dans le code), lus par
chemin depuis le dossier `DOSSIER_REFERENTIEL` (défaut : `referentiel`, à côté
du notebook). Chargement réduit à une simple boucle. Sur Colab : place le
dossier `referentiel` dans `/content` (ou indique son chemin). Fichiers :
`ETATS.json`, `SOUS_CATEGORIES.json`, `CFP_REGLES.json`, `PAYS_ALIAS.json`,
`GEO_FRANCE.json`, `FORMES_JURIDIQUES.json`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 — INSTALLATION DES DÉPENDANCES
#
# À exécuter EN PREMIER, une fois par session. Colab repart d'une machine
# vierge à chaque connexion : ces paquets ne sont PAS persistants, il faut
# relancer cette cellule après tout redémarrage du runtime.
#
#   rapidfuzz  : similarité floue de chaînes -> c'est le cœur du SCORING.
#                Tolère fautes de frappe, inversions de mots, accents absents.
#   openpyxl   : lecture/écriture .xlsx + couleurs de cellule (fichier final).
#   xlsxwriter : moteur d'écriture du RAPPORT (gère les formats conditionnels,
#                ce qu'openpyxl fait mal) -> les deux sont donc nécessaires.
#   tqdm       : barres de progression (dépendance de certains imports).
#   requests   : appels HTTP vers l'API Recherche d'entreprises.
#   ipywidgets : widget d'upload de fichier sous Colab.
#   pycountry  : table ISO des pays -> zone UE / AELE / UK / AUTRE.
# ═══════════════════════════════════════════════════════════════
%pip install -q rapidfuzz openpyxl xlsxwriter tqdm requests ipywidgets pycountry
print("Dépendances installées")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.8 MB/s eta 0:00:00
Dépendances installées


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 (dite « 1bis ») — RÉFÉRENTIELS EMBARQUÉS : le socle de vérité
#
# POURQUOI EN DUR DANS LE NOTEBOOK ? Parce qu'un référentiel téléchargé au vol
# est un point de panne : si le site source change d'URL ou tombe, le
# traitement s'arrête. Ici tout est figé dans le .ipynb : le notebook est
# AUTONOME et REPRODUCTIBLE à l'identique, même dans six mois. Prix à payer :
# la taille du fichier (~1,7 Mo).
#
# CE QUE CONTIENT CHAQUE BLOC (chaîne JSON stockée dans une variable) :
#
#   GEO_FRANCE ......... 108 départements/territoires -> NUTS3, région,
#                        NUTS2 (ancienne région), Metro/RUP/PTOM. Contient
#                        aussi les tables de désambiguïsation par code postal
#                        (97133 = Saint-Barthélemy -> PTOM ; 97150 =
#                        Saint-Martin -> RUP) et par nom de pays.
#   VILLES_FR .......... 31 950 communes -> département. DERNIER recours quand
#                        une ligne n'a ni code postal ni SIREN. Les homonymes
#                        ambigus (SAINT-DENIS : métropole ET La Réunion) sont
#                        VOLONTAIREMENT absents : mieux vaut ne rien dire que
#                        se tromper de territoire.
#   FORMES_JURIDIQUES .. nomenclature INSEE des catégories juridiques
#                        (cj_septembre_2022, 269 codes) sur 3 niveaux.
#   REF_SGAE ........... 262 codes juridiques -> catégorie simplifiée maison
#                        (« Entreprises », « Etablissements publics »...).
#   NAF_REV2 ........... 732 sous-classes NAF rév. 2 -> libellé d'activité.
#                        Indispensable : l'API renvoie le CODE mais PAS le
#                        libellé (vérifié dans le code source de l'API).
#   OPERATEURS_ETAT .... SIREN des opérateurs de l'État + programme budgétaire
#                        chef de file de chacun.
#
# RÈGLE DE MODIFICATION : ces blocs sont du JSON STRICT. Une virgule oubliée
# entre deux entrées casse tout le chargement (« Expecting ',' delimiter »).
# Après toute édition manuelle, relancez la cellule de chargement : elle
# valide et affiche le nombre d'entrées lues.
# -> Voir README, section « Ajouter une entrée à un référentiel ».
# ═══════════════════════════════════════════════════════════════
# Les référentiels sont embarqués ci-dessous sous forme de blocs JSON. Le
# notebook est AUTONOME : aucun fichier externe à déposer (hormis le fichier FTS
# à traiter). Pour modifier un référentiel, édite le bloc JSON correspondant.
import json

_REFERENTIELS_BLOBS = {
    "ETATS": r'''{
  "rows": [
    {
      "Etats_long_EN": "Austria",
      "Etats_long_FR": "Autriche",
      "Etats_court": "AT",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Belgium",
      "Etats_long_FR": "Belgique",
      "Etats_court": "BE",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Bulgaria",
      "Etats_long_FR": "Bulgarie",
      "Etats_court": "BG",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Cyprus",
      "Etats_long_FR": "Chypre",
      "Etats_court": "CY",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Czech Republic",
      "Etats_long_FR": "Tchéquie",
      "Etats_court": "CZ",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Germany",
      "Etats_long_FR": "Allemagne",
      "Etats_court": "DE",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Denmark",
      "Etats_long_FR": "Danemark",
      "Etats_court": "DK",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Estonia",
      "Etats_long_FR": "Estonie",
      "Etats_court": "EE",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Greece",
      "Etats_long_FR": "Grèce",
      "Etats_court": "EL",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Spain",
      "Etats_long_FR": "Espagne",
      "Etats_court": "ES",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Finland",
      "Etats_long_FR": "Finlande",
      "Etats_court": "FI",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "France",
      "Etats_long_FR": "France",
      "Etats_court": "FR",
      "Etat_Statut": "FR"
    },
    {
      "Etats_long_EN": "Croatia",
      "Etats_long_FR": "Croatie",
      "Etats_court": "HR",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Hungary",
      "Etats_long_FR": "Hongrie",
      "Etats_court": "HU",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Ireland",
      "Etats_long_FR": "Irlande",
      "Etats_court": "IE",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Italy",
      "Etats_long_FR": "Italie",
      "Etats_court": "IT",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Latvia",
      "Etats_long_FR": "Lettonie",
      "Etats_court": "LT",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Luxembourg",
      "Etats_long_FR": "Luxembourg",
      "Etats_court": "LU",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Lithuania",
      "Etats_long_FR": "Lituanie",
      "Etats_court": "LV",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Malta",
      "Etats_long_FR": "Malte",
      "Etats_court": "MT",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Netherlands",
      "Etats_long_FR": "Pays-Bas",
      "Etats_court": "NL",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Poland",
      "Etats_long_FR": "Pologne",
      "Etats_court": "PL",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Portugal",
      "Etats_long_FR": "Portugal",
      "Etats_court": "PT",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Romania",
      "Etats_long_FR": "Roumaie",
      "Etats_court": "RO",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Sweden",
      "Etats_long_FR": "Suède",
      "Etats_court": "SE",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Slovenia",
      "Etats_long_FR": "Slovénie",
      "Etats_court": "SI",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Slovakia",
      "Etats_long_FR": "Slovaquie",
      "Etats_court": "SK",
      "Etat_Statut": "UE"
    },
    {
      "Etats_long_EN": "Switzerland",
      "Etats_long_FR": "Suisse",
      "Etats_court": "CH",
      "Etat_Statut": "AELE"
    },
    {
      "Etats_long_EN": "Norway",
      "Etats_long_FR": "Norvège",
      "Etats_court": "NO",
      "Etat_Statut": "AELE"
    },
    {
      "Etats_long_EN": "Iceland",
      "Etats_long_FR": "Islande",
      "Etats_court": "IS",
      "Etat_Statut": "AELE"
    },
    {
      "Etats_long_EN": "Liechtenstein",
      "Etats_long_FR": "Liechtenstein",
      "Etats_court": "LI",
      "Etat_Statut": "AELE"
    },
    {
      "Etats_long_EN": "United Kingdom",
      "Etats_long_FR": "Royaume-Uni",
      "Etats_court": "UK",
      "Etat_Statut": "UK"
    }
  ]
}''',
    "SOUS_CATEGORIES": r'''{
  "description": "Table Sous_catégories_programmes_CFP fournie par l'utilisateur",
  "rows": [
    {
      "Programme": "1.0.31 - Single Market Programme (incl. SMEs)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.1.3PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "2.1.311 - European Social Fund Plus (ESF+)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.21 - European Recovery and Resilience Facility (incl. Technical Support Instrument)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.22 - Protection of the euro against counterfeiting (the `Pericles IV programme')",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.352 - Rights and Values",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.11 - Asylum, Migration and Integration Fund",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.13 - Nuclear Safety and decommissioning (incl. For Bulgaria and Slovakia)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.23 - Short-term Defence instrument on common procurement",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.111 - Neighbourhood, Development and International Cooperation Instrument - Global Europe ( NDICI - Global Europe )",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.112 - European Instrument for International Nuclear Safety Cooperation (INSC)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.17 - Ukraine Loan Cooperation Mechanism",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "DA/JU/EUI - Decentralised Agencies, Joint Undertakings, EU institutions",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "1.0.11 - Horizon Europe",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.12 - Euratom Research and Training Programme",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.13 - International Thermonuclear Experimental Reactor (ITER)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.1OTH - Other actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.1PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.21 - InvestEU Fund",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.221 - Connecting Europe Facility (CEF) - Transport",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.222 - Connecting Europe Facility (CEF) - Energy",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.223 - Connecting Europe Facility (CEF) - Digital",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.23 - Digital Europe Programme",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.2DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "1.0.2OTH - Other actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.2PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.2SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.31 - Single Market Programme (incl - SMEs)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.32 - EU Anti-Fraud Programme",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.33 - Cooperation in the field of taxation (Fiscalis)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.34 - Cooperation in the field of customs (Customs)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.3DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "1.0.3OTH - Other actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.3PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.41 - European Space Programme",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.0.4DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "1.0.4PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.0.4SC - Union Secure Connectivity",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.1.11 - European Regional Development Fund (ERDF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.1.121 - Cohesion Fund (CF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.1.122 - Cohesion Fund (CF), contribution to the Connecting Europe Facility (CEF) - Transport",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.1.1PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "2.1.311 - European Social Fund (ESF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.13 - Support to the Turkish-Cypriot Community",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.21 - European Recovery and Resilience Facility (incl - Technical Support Instrument)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.22 - Protection of the euro against counterfeiting (the ‘Pericles IV programme')",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.23 - Financing cost of the European Union Recovery Instrument (EURI)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.24 - Union Civil Protection Mechanism (RescEU)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.25 - EU4Health",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.26 - Instrument for emergency support within the Union (ESI)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.2DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "2.2.2SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "2.2.312 - Employment and Social Innovation",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.32 - Erasmus+",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.33 - European Solidarity Corps (ESC)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.34 - Creative Europe",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.351 - Justice",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.352 - Citizens, Equality, Rights and Values",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.2.3DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "2.2.3OTH - Other actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "2.2.3PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "2.2.3SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "3.1.11 - European Agricultural Guarantee Fund (EAGF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.12 - European Agricultural Fund for Rural Development (EAFRD)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.13 - European Maritime, Fisheries and Aquaculture Fund (EMFAF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.14 - Sustainable Fisheries Partnership Agreements (SFPA) and Regional Fisheries Management Organisations (RFMO)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.1DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "3.2.1OTH - Other actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "3.2.1PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "3.2.21 - Programme for Environment and Climate Action (LIFE)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.22 - Just Transition Fund",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.23 - Public sector loan facility under the Just Transition Mechanism (JTM)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.2.2DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "3.2.2PPPA - Pilot projects and preparatory actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "3.2.2SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "4.0.11 - Asylum, Migration and Integration Fund (AMIF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.11 - Guarantee Fund for External Actions",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.1DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "4.0.211 - Integrated Border Management Fund (IBMF) - Instrument for border management and visa (BMVI)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.212 - Integrated Border Management Fund (IBMF) - Instrument for financial support for customs control equipment (CCEi)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.2DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "5.0.11 - Internal Security Fund (ISF)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.12 - Nuclear decommissioning (Lithuania)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.13 - Nuclear Safety and decommissioning (incl - For Bulgaria and Slovakia)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.1DAG - Decentralised Agencies",
      "Période CFP": "21-27",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "5.0.1SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "5.0.211 - European Defence Fund (Research)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.212 - European Defence Fund (Non Research)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.22 - Military Mobility",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.24 - Defence Industrial Reinforcement Instrument",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "5.0.2SC - Union Secure Connectivity",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.111 - Neighbourhood, Development and International Cooperation Instrument - Global Europe (NDICI - Global Europe)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.112 - European Instrument for Internation Nuclear Safety Cooperation (INSC)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.12 - Humanitarian Aid (HUMA)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.13 - Common Foreign and Security Policy (CFSP)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.14 - Overseas Countries and Territories (OCT) (including Greenland)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.15 - MFA+",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.1OTH - Other actions",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "6.0.1SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "6.0.21 - Pre-Accession Assistance (IPA III)",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "6.0.22 - Reform and Growth Facility for Western Balkans",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "7.1.23 - (European schools) Commission",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.312 - Remuneration external staff",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.331 - Recruitment costs",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.333 - Training costs",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.334 - Social and Mobility",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.341 - Information and communication technology",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.351 - Rents and purchases",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.352 - Linked to buildings",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.353 - Security",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.361 - Mission and representation",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.362 - Meetings, committees, conference",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.371 - Official journal",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.372 - Publications",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.373 - Acquisition of information",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.374 - Studies and investigations",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.381 - General equipment, vehicle, furniture",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.382 - Linguistic external services",
      "Période CFP": "21-27",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "7.2.383 - Other administrative expenditure",
      "Période CFP": "21-27",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "0.0.4 - Ukraine Facility",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.10 - European Fund for Strategic Investments (EFSI)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.11 - European satellite navigation systems (EGNOS and Galileo)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.12 - International Thermonuclear Experimental Reactor (ITER)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.13 - European Earth Observation Programme (Copernicus)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.14 - European Solidarity Corps (ESC)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.15 - European Defense Industrial Development Programme",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.2 - Nuclear Safety and Decommissioning",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.31 - Horizon 2020",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.32 - Euratom Research and Training Programme",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.4 - Competitiveness of enterprises and small and medium-sized enterprises (COSME)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.5 - Education, Training and Sport (Erasmus+)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.6 - Employment and Social Innovation (EaSI)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.7 - Customs, Fiscalis and Anti-Fraud",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.81 - Energy",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.82 - Transport",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.83 - Information and Communications Technology (ICT)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.9 - Energy projects to aid economic recovery (EERP)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.1.DAG - Decentralised agencies",
      "Période CFP": "14-20",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "1.1.OTH - Other actions and programmes",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.1.PPPA - Pilot projects and preparatory actions",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.1.SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "1.2.15 - Cohesion fund",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.2.31 - Technical assistance",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.2.4 - European Aid to the Most Deprived (FEAD)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.2.6 - Contribution to the Connecting Europe Facility (CEF)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "1.2.PPPA - Pilot projects and preparatory actions",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "10th European Development Fund (EDF)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "11th European Development Fund (EDF)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.0.10 - European Agricultural Guarantee Fund (EAGF) -  Market related expenditure and direct payments",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.0.20 - European Agricultural Fund for Rural Development (EAFRD)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.0.31 - European Maritime and Fisheries Fund (EMFF)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.0.4 - Environment and climate action (LIFE)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "2.0.DAG - Decentralised agencies",
      "Période CFP": "14-20",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "2.0.PPPA - Pilot projects and preparatory actions",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "2.0.SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "3.0.1 - Asylum, Migration and Integration Fund (AMF)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.10 - Consumer",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.11 - Creative Europe",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.12 - Instrument for Emergency Support within the Union (IES)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.2 - Internal Security Fund",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.3 - IT systems",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.4 - Justice",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.5 - Rights, Equality and Citizenship",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.6 - Union Civil protection Mechanism",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.7 - Europe for Citizens",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.8 - Food and feed",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.9 - Health",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "3.0.DAG - Decentralised agencies",
      "Période CFP": "14-20",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "3.0.PPPA - Pilot projects and preparatory actions",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "3.0.SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "4.0.1 - Instrument for Pre-accession assistance (IPA II)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.10 - Macro-financial Assistance (MFA)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.12 - Union Civil Protection Mechanism",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.13 - EU Aid Volunteers initiative (EUAV)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.2 - European Neighbourhood Instrument (ENI)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.3 - Development Cooperation Instrument (DCI)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.4 - Partnership Instrument (PI)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.5 - European Instrument for Democracy and Human Rights (EIDHR)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.6 - Instrument contributing to Stability and Peace (IcSP)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.7 - Humanitarian aid",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.8 - Common Foreign and Security Policy (CFSP)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.9 - Instrument for Nuclear Safety Cooperation (INSC)",
      "Période CFP": "14-20",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "4.0.DAG - Decentralised agencies",
      "Période CFP": "14-20",
      "Sous catégorie": "Agences décentralisées"
    },
    {
      "Programme": "4.0.OTH - Other actions and programmes",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "4.0.PPPA - Pilot projects and preparatory actions",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "4.0.SPEC - Actions financed under the prerogatives of the Commission and specific competences conferred to the Commission",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "5.2.3PPPA - Pilot projects and preparatory actions",
      "Période CFP": "14-20",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "5.2.3X - Commission administrative expenditure",
      "Période CFP": "14-20",
      "Sous catégorie": "Administration"
    },
    {
      "Programme": "2.2.22 - Protection of the euro against counterfeiting (the 'Pericles IV programme')",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "O.0.4 - Ukraine Facility",
      "Période CFP": "21-27",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "9.0.2 - European Globalisation Adjustment Fund (EGF)",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "9th European Development Fund (EDF)",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "O.0.1 - Innovation Fund (IF)",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "O.0.OTH - Other actions",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "SPEC, PPA, Others"
    },
    {
      "Programme": "S.0.1 - Solidarity and Emergency Aid Reserve (SEAR)",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "S.0.2 - European Globalisation Adjustment Fund (EGF)",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "S.0.4 - Brexit Adjustment Reserve",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    },
    {
      "Programme": "S.0.11 - European Solidarity Reserve",
      "Période CFP": "Hors CFP",
      "Sous catégorie": "Programmes"
    }
  ]
}''',
    "CFP_REGLES": r'''{
  "description": "Instruments dont la Dépense se rattache Hors CFP (issus du référentiel SPILT). periode_hors_cfp_pour_instruments : false = Période selon l'année (définition CMFE), true = Période 'Hors CFP' comme les 12 lignes récentes du référentiel. | prefixe_special_dans_cfp : programmes à préfixe O./S./9.0 qui sont de VRAIS programmes CFP (ex. Ukraine Facility) et ne doivent PAS déclencher l'avertissement hors-CFP.",
  "hors_cfp_programmes": [
    "10th european development fund (edf)",
    "11th european development fund (edf)",
    "9.0.2 - european globalisation adjustment fund (egf)",
    "9th european development fund (edf)",
    "o.0.1 - innovation fund (if)",
    "o.0.oth - other actions",
    "s.0.1 - solidarity and emergency aid reserve (sear)",
    "s.0.11 - european solidarity reserve",
    "s.0.2 - european globalisation adjustment fund (egf)",
    "s.0.4 - brexit adjustment reserve"
  ],
  "periode_hors_cfp_pour_instruments": false,
  "prefixe_special_dans_cfp": [
    "O.0.4 - Ukraine Facility"
  ]
}''',
    "PAYS_ALIAS": r'''{
  "description": "Alias de libellés pays FTS, alias ISO vérifiés (base ISO 3166), territoires rattachés à FR",
  "alias_libelles": {
    "czechia": "czech republic",
    "united kingdom of great britain and northern ireland": "united kingdom",
    "netherlands (the)": "netherlands",
    "korea, republic of": "korea"
  },
  "iso_alias": {
    "falkland islands (malvinas)": "FK",
    "pitcairn": "PN",
    "saint helena, ascension and tristan da cunha": "SH",
    "holy see (vatican city state)": "VA",
    "virgin islands, british": "VG",
    "virgin islands, u.s.": "VI",
    "cote d'ivoire": "CI",
    "democratic republic of the congo": "CD",
    "kosovo (under unscr 1244/99)": "XK",
    "macedonia": "MK",
    "republ": "RS",
    "republic o": "ME",
    "st. lucia": "LC",
    "ivory coast": "CI",
    "congo (democratic republic of)": "CD",
    "macau": "MO",
    "cape verde": "CV",
    "east timor": "TL",
    "swaziland": "SZ",
    "gambia (the)": "GM",
    "faeroe islands": "FO",
    "bahamas(the)": "BS",
    "sao tome and principe": "ST",
    "micronesia": "FM",
    "palestine": "PS",
    "russia": "RU",
    "kosovo": "XK"
  },
  "dom_tom_rup_fr": [
    "french guiana",
    "french polynesia",
    "french southern territories",
    "guadeloupe",
    "guyane",
    "guyane francaise",
    "la reunion",
    "martinique",
    "mayotte",
    "new caledonia",
    "nouvelle caledonie",
    "nouvelle-caledonie",
    "polynesie francaise",
    "reunion",
    "saint barthelemy",
    "saint martin",
    "saint martin (french part)",
    "saint pierre and miquelon",
    "saint pierre et miquelon",
    "saint-barthelemy",
    "saint-martin (french part)",
    "saint-pierre-et-miquelon",
    "taaf",
    "terres australes et antarctiques francaises",
    "wallis and futuna",
    "wallis et futuna",
    "wallis-et-futuna"
  ]
}''',
    "REF_SGAE": r'''{
  "description": "Référentiel SGAE de simplification des statuts juridiques (fourni par l'utilisateur, REF_SGAE.xlsx) : catégorie juridique INSEE niveau III (code 4 chiffres) -> catégorie simplifiée. 262 codes (v5_14b) ; 13 catégories révisées par l'utilisateur en v5_17 (Classeur1 : SIVOM/SIVU/syndicats -> Etablissement public, régies -> Collectivité territoriale, 0000/7490/9110 -> Autre, 3110 -> Entités étrangères, 5195 -> Association).",
  "map": {
      "5498": "Entreprises",
      "5720": "Entreprises",
      "0000": "Autre",
      "1000": "Entreprises",
      "2110": "Autres",
      "2120": "Autres",
      "2210": "Entreprises",
      "2220": "Entreprises",
      "2310": "Entreprises",
      "2320": "Entreprises",
      "2385": "Entreprises",
      "2400": "Autres",
      "2700": "Associations et Fondations",
      "2800": "Autres",
      "2900": "Groupements",
      "3110": "Entités étrangères",
      "3120": "Entreprises",
      "3205": "Organisations internationales",
      "3210": "Entités étrangères",
      "3220": "Entités étrangères",
      "3290": "Entités étrangères",
      "4110": "Etablissements publics",
      "4120": "Etablissements publics",
      "4130": "Etablissements publics",
      "4140": "Etablissements publics",
      "4150": "Collectivité territoriale",
      "4160": "Etat",
      "5191": "Entreprises",
      "5192": "Entreprises",
      "5193": "Entreprises",
      "5194": "Entreprises",
      "5195": "Association",
      "5196": "Entreprises",
      "5202": "Entreprises",
      "5203": "Entreprises",
      "5306": "Entreprises",
      "5307": "Entreprises",
      "5308": "Entreprises",
      "5309": "Entreprises",
      "5310": "Entreprises",
      "5370": "Entreprises",
      "5385": "Entreprises",
      "5410": "Entreprises",
      "5415": "Entreprises",
      "5422": "Entreprises",
      "5426": "Entreprises",
      "5430": "Entreprises",
      "5431": "Entreprises",
      "5432": "Entreprises",
      "5442": "Entreprises",
      "5443": "Entreprises",
      "5451": "Entreprises",
      "5453": "Entreprises",
      "5454": "Entreprises",
      "5455": "Entreprises",
      "5458": "Entreprises",
      "5459": "Entreprises",
      "5460": "Entreprises",
      "5470": "Entreprises",
      "5485": "Entreprises",
      "5499": "Entreprises",
      "5505": "Entreprises",
      "5510": "Entreprises",
      "5515": "Entreprises",
      "5520": "Entreprises",
      "5522": "Entreprises",
      "5525": "Entreprises",
      "5530": "Entreprises",
      "5531": "Entreprises",
      "5532": "Entreprises",
      "5542": "Entreprises",
      "5543": "Entreprises",
      "5546": "Entreprises",
      "5547": "Entreprises",
      "5548": "Entreprises",
      "5551": "Entreprises",
      "5552": "Entreprises",
      "5553": "Entreprises",
      "5554": "Entreprises",
      "5555": "Entreprises",
      "5558": "Entreprises",
      "5559": "Entreprises",
      "5560": "Entreprises",
      "5570": "Entreprises",
      "5585": "Entreprises",
      "5599": "Entreprises",
      "5605": "Entreprises",
      "5610": "Entreprises",
      "5615": "Entreprises",
      "5620": "Entreprises",
      "5622": "Entreprises",
      "5625": "Entreprises",
      "5630": "Entreprises",
      "5631": "Entreprises",
      "5632": "Entreprises",
      "5642": "Entreprises",
      "5643": "Entreprises",
      "5646": "Entreprises",
      "5647": "Entreprises",
      "5648": "Entreprises",
      "5651": "Entreprises",
      "5652": "Entreprises",
      "5653": "Entreprises",
      "5654": "Entreprises",
      "5655": "Entreprises",
      "5658": "Entreprises",
      "5659": "Entreprises",
      "5660": "Entreprises",
      "5670": "Entreprises",
      "5685": "Entreprises",
      "5699": "Entreprises",
      "5710": "Entreprises",
      "5770": "Entreprises",
      "5785": "Entreprises",
      "5800": "Entreprises",
      "6100": "Entreprises",
      "6210": "Groupements",
      "6220": "Groupements",
      "6316": "Entreprises",
      "6317": "Entreprises",
      "6318": "Entreprises",
      "6411": "Entreprises",
      "6511": "Entreprises",
      "6521": "Entreprises",
      "6532": "Entreprises",
      "6533": "Groupements",
      "6534": "Groupements",
      "6535": "Groupements",
      "6536": "Groupements",
      "6537": "Groupements",
      "6538": "Groupements",
      "6539": "Entreprises",
      "6540": "Entreprises",
      "6541": "Entreprises",
      "6542": "Entreprises",
      "6543": "Entreprises",
      "6544": "Entreprises",
      "6551": "Entreprises",
      "6554": "Entreprises",
      "6558": "Entreprises",
      "6560": "Entreprises",
      "6561": "Entreprises",
      "6562": "Entreprises",
      "6563": "Entreprises",
      "6564": "Entreprises",
      "6565": "Entreprises",
      "6566": "Entreprises",
      "6567": "Entreprises",
      "6568": "Entreprises",
      "6569": "Entreprises",
      "6571": "Entreprises",
      "6572": "Entreprises",
      "6573": "Entreprises",
      "6574": "Entreprises",
      "6575": "Entreprises",
      "6576": "Entreprises",
      "6577": "Entreprises",
      "6578": "Entreprises",
      "6585": "Entreprises",
      "6589": "Entreprises",
      "6595": "Entreprises",
      "6596": "Entreprises",
      "6597": "Entreprises",
      "6598": "Entreprises",
      "6599": "Entreprises",
      "6901": "Entreprises",
      "7111": "Etat",
      "7112": "Etat",
      "7113": "Etat",
      "7120": "Etat",
      "7150": "Etat",
      "7160": "Etat",
      "7171": "Etat",
      "7172": "Etat",
      "7179": "Etat",
      "7190": "Etat",
      "7210": "Collectivités territoriales",
      "7220": "Collectivités territoriales",
      "7225": "Collectivités territoriales",
      "7229": "Collectivités territoriales",
      "7230": "Collectivités territoriales",
      "7312": "Collectivités territoriales",
      "7313": "Collectivités territoriales",
      "7314": "Collectivités territoriales",
      "7321": "Associations et Fondations",
      "7322": "Associations et Fondations",
      "7323": "Associations et Fondations",
      "7331": "Etablissements publics",
      "7340": "Collectivités territoriales",
      "7341": "Collectivités territoriales",
      "7342": "Collectivités territoriales",
      "7343": "Collectivités territoriales",
      "7344": "Collectivités territoriales",
      "7345": "Etablissement public",
      "7346": "Collectivités territoriales",
      "7347": "Collectivités territoriales",
      "7348": "Collectivités territoriales",
      "7349": "Etablissements publics",
      "7351": "Etablissements publics",
      "7352": "Etablissements publics",
      "7353": "Etablissement public",
      "7354": "Etablissement public",
      "7355": "Etablissement public",
      "7356": "Etablissement public",
      "7357": "Etablissements publics",
      "7361": "Etablissements publics",
      "7362": "Etablissements publics",
      "7363": "Etablissements publics",
      "7364": "Etablissements publics",
      "7365": "Etablissement public",
      "7366": "Etablissements publics",
      "7367": "Etablissements publics",
      "7371": "Etablissements publics",
      "7372": "Etablissements publics",
      "7373": "Etablissements publics",
      "7378": "Collectivité territoriale",
      "7379": "Etablissements publics",
      "7381": "Etablissements publics",
      "7382": "Etablissements publics",
      "7383": "Etablissements publics",
      "7384": "Etablissements publics",
      "7385": "Etablissements publics",
      "7389": "Etablissements publics",
      "7410": "Groupements",
      "7430": "Etablissements publics",
      "7450": "Etablissements publics",
      "7470": "Groupements",
      "7490": "Autre",
      "8110": "Organismes professionnels et sociaux",
      "8120": "Organismes professionnels et sociaux",
      "8130": "Organismes professionnels et sociaux",
      "8140": "Organismes professionnels et sociaux",
      "8150": "Organismes professionnels et sociaux",
      "8160": "Organismes professionnels et sociaux",
      "8170": "Organismes professionnels et sociaux",
      "8190": "Organismes professionnels et sociaux",
      "8210": "Organismes professionnels et sociaux",
      "8250": "Organismes professionnels et sociaux",
      "8290": "Organismes professionnels et sociaux",
      "8310": "Organismes professionnels et sociaux",
      "8311": "Organismes professionnels et sociaux",
      "8410": "Syndicats",
      "8420": "Syndicats",
      "8450": "Organismes professionnels et sociaux",
      "8470": "Organismes professionnels et sociaux",
      "8490": "Organismes professionnels et sociaux",
      "8510": "Organismes professionnels et sociaux",
      "8520": "Organismes professionnels et sociaux",
      "9110": "Autre",
      "9150": "Associations et Fondations",
      "9210": "Associations et Fondations",
      "9220": "Associations et Fondations",
      "9221": "Associations et Fondations",
      "9222": "Associations et Fondations",
      "9223": "Associations et Fondations",
      "9224": "Associations et Fondations",
      "9230": "Associations et Fondations",
      "9240": "Associations et Fondations",
      "9260": "Associations et Fondations",
      "9300": "Associations et Fondations",
      "9900": "Autres",
      "9970": "Groupements"
  }
}''',

    "NAF_REV2": r'''{
  "description": "Nomenclature d'activités française NAF rév. 2 (INSEE) — 732 sous-classes, code -> libellé. Source : dépôt public SocialGouv/codes-naf (Fabrique numérique des ministères sociaux), validée par contrôles croisés.",
  "map": {
      "01.11Z": "Culture de céréales (à l'exception du riz), de légumineuses et de graines oléagineuses",
      "01.12Z": "Culture du riz",
      "01.13Z": "Culture de légumes, de melons, de racines et de tubercules",
      "01.14Z": "Culture de la canne à sucre",
      "01.15Z": "Culture du tabac",
      "01.16Z": "Culture de plantes à fibres",
      "01.19Z": "Autres cultures non permanentes",
      "01.21Z": "Culture de la vigne",
      "01.22Z": "Culture de fruits tropicaux et subtropicaux",
      "01.23Z": "Culture d'agrumes",
      "01.24Z": "Culture de fruits à pépins et à noyau",
      "01.25Z": "Culture d'autres fruits d'arbres ou d'arbustes et de fruits à coque",
      "01.26Z": "Culture de fruits oléagineux",
      "01.27Z": "Culture de plantes à boissons",
      "01.28Z": "Culture de plantes à épices, aromatiques, médicinales et pharmaceutiques",
      "01.29Z": "Autres cultures permanentes",
      "01.30Z": "Reproduction de plantes",
      "01.41Z": "Élevage de vaches laitières",
      "01.42Z": "Élevage d'autres bovins et de buffles",
      "01.43Z": "Élevage de chevaux et d'autres équidés",
      "01.44Z": "Élevage de chameaux et d'autres camélidés",
      "01.45Z": "Élevage d'ovins et de caprins",
      "01.46Z": "Élevage de porcins",
      "01.47Z": "Élevage de volailles",
      "01.49Z": "Élevage d'autres animaux",
      "01.50Z": "Culture et élevage associés",
      "01.61Z": "Activités de soutien aux cultures",
      "01.62Z": "Activités de soutien à la production animale",
      "01.63Z": "Traitement primaire des récoltes",
      "01.64Z": "Traitement des semences",
      "01.70Z": "Chasse, piégeage et services annexes",
      "02.10Z": "Sylviculture et autres activités forestières",
      "02.20Z": "Exploitation forestière",
      "02.30Z": "Récolte de produits forestiers non ligneux poussant à l'état sauvage",
      "02.40Z": "Services de soutien à l'exploitation forestière",
      "03.11Z": "Pêche en mer",
      "03.12Z": "Pêche en eau douce",
      "03.21Z": "Aquaculture en mer",
      "03.22Z": "Aquaculture en eau douce",
      "05.10Z": "Extraction de houille",
      "05.20Z": "Extraction de lignite",
      "06.10Z": "Extraction de pétrole brut",
      "06.20Z": "Extraction de gaz naturel",
      "07.10Z": "Extraction de minerais de fer",
      "07.21Z": "Extraction de minerais d'uranium et de thorium",
      "07.29Z": "Extraction d'autres minerais de métaux non ferreux",
      "08.11Z": "Extraction de pierres ornementales et de construction, de calcaire industriel, de gypse, de craie et d'ardoise",
      "08.12Z": "Exploitation de gravières et sablières, extraction d’argiles et de kaolin",
      "08.91Z": "Extraction des minéraux chimiques et d'engrais minéraux",
      "08.92Z": "Extraction de tourbe",
      "08.93Z": "Production de sel",
      "08.99Z": "Autres activités extractives n.c.a.",
      "09.10Z": "Activités de soutien à l'extraction d'hydrocarbures",
      "09.90Z": "Activités de soutien aux autres industries extractives",
      "10.11Z": "Transformation et conservation de la viande de boucherie",
      "10.12Z": "Transformation et conservation de la viande de volaille",
      "10.13A": "Préparation industrielle de produits à base de viande",
      "10.13B": "Charcuterie",
      "10.20Z": "Transformation et conservation de poisson, de crustacés et de mollusques",
      "10.31Z": "Transformation et conservation de pommes de terre",
      "10.32Z": "Préparation de jus de fruits et légumes",
      "10.39A": "Autre transformation et conservation de légumes",
      "10.39B": "Transformation et conservation de fruits",
      "10.41A": "Fabrication d'huiles et graisses brutes",
      "10.41B": "Fabrication d'huiles et graisses raffinées",
      "10.42Z": "Fabrication de margarine et graisses comestibles similaires",
      "10.51A": "Fabrication de lait liquide et de produits frais",
      "10.51B": "Fabrication de beurre",
      "10.51C": "Fabrication de fromage",
      "10.51D": "Fabrication d'autres produits laitiers",
      "10.52Z": "Fabrication de glaces et sorbets",
      "10.61A": "Meunerie",
      "10.61B": "Autres activités du travail des grains",
      "10.62Z": "Fabrication de produits amylacés",
      "10.71A": "Fabrication industrielle de pain et de pâtisserie fraîche",
      "10.71B": "Cuisson de produits de boulangerie",
      "10.71C": "Boulangerie et boulangerie-pâtisserie",
      "10.71D": "Pâtisserie",
      "10.72Z": "Fabrication de biscuits, biscottes et pâtisseries de conservation",
      "10.73Z": "Fabrication de pâtes alimentaires",
      "10.81Z": "Fabrication de sucre",
      "10.82Z": "Fabrication de cacao, chocolat et de produits de confiserie",
      "10.83Z": "Transformation du thé et du café",
      "10.84Z": "Fabrication de condiments et assaisonnements",
      "10.85Z": "Fabrication de plats préparés",
      "10.86Z": "Fabrication d'aliments homogénéisés et diététiques",
      "10.89Z": "Fabrication d'autres produits alimentaires n.c.a.",
      "10.91Z": "Fabrication d'aliments pour animaux de ferme",
      "10.92Z": "Fabrication d'aliments pour animaux de compagnie",
      "11.01Z": "Production de boissons alcooliques distillées",
      "11.02A": "Fabrication de vins effervescents",
      "11.02B": "Vinification",
      "11.03Z": "Fabrication de cidre et de vins de fruits",
      "11.04Z": "Production d'autres boissons fermentées non distillées",
      "11.05Z": "Fabrication de bière",
      "11.06Z": "Fabrication de malt",
      "11.07A": "Industrie des eaux de table",
      "11.07B": "Production de boissons rafraîchissantes",
      "12.00Z": "Fabrication de produits à base de tabac",
      "13.10Z": "Préparation de fibres textiles et filature",
      "13.20Z": "Tissage",
      "13.30Z": "Ennoblissement textile",
      "13.91Z": "Fabrication d'étoffes à mailles",
      "13.92Z": "Fabrication d'articles textiles, sauf habillement",
      "13.93Z": "Fabrication de tapis et moquettes",
      "13.94Z": "Fabrication de ficelles, cordes et filets",
      "13.95Z": "Fabrication de non-tissés, sauf habillement",
      "13.96Z": "Fabrication d'autres textiles techniques et industriels",
      "13.99Z": "Fabrication d'autres textiles n.c.a.",
      "14.11Z": "Fabrication de vêtements en cuir",
      "14.12Z": "Fabrication de vêtements de travail",
      "14.13Z": "Fabrication de vêtements de dessus",
      "14.14Z": "Fabrication de vêtements de dessous",
      "14.19Z": "Fabrication d'autres vêtements et accessoires",
      "14.20Z": "Fabrication d'articles en fourrure",
      "14.31Z": "Fabrication d'articles chaussants à mailles",
      "14.39Z": "Fabrication d'autres articles à mailles",
      "15.11Z": "Apprêt et tannage des cuirs ; préparation et teinture des fourrures",
      "15.12Z": "Fabrication d'articles de voyage, de maroquinerie et de sellerie",
      "15.20Z": "Fabrication de chaussures",
      "16.10A": "Sciage et rabotage du bois, hors imprégnation",
      "16.10B": "Imprégnation du bois",
      "16.21Z": "Fabrication de placage et de panneaux de bois",
      "16.22Z": "Fabrication de parquets assemblés",
      "16.23Z": "Fabrication de charpentes et d'autres menuiseries",
      "16.24Z": "Fabrication d'emballages en bois",
      "16.29Z": "Fabrication d'objets divers en bois ; fabrication d'objets en liège, vannerie et sparterie",
      "17.11Z": "Fabrication de pâte à papier",
      "17.12Z": "Fabrication de papier et de carton",
      "17.21A": "Fabrication de carton ondulé",
      "17.21B": "Fabrication de cartonnages",
      "17.21C": "Fabrication d'emballages en papier",
      "17.22Z": "Fabrication d'articles en papier à usage sanitaire ou domestique",
      "17.23Z": "Fabrication d'articles de papeterie",
      "17.24Z": "Fabrication de papiers peints",
      "17.29Z": "Fabrication d'autres articles en papier ou en carton",
      "18.11Z": "Imprimerie de journaux",
      "18.12Z": "Autre imprimerie (labeur)",
      "18.13Z": "Activités de pré-presse",
      "18.14Z": "Reliure et activités connexes",
      "18.20Z": "Reproduction d'enregistrements",
      "19.10Z": "Cokéfaction",
      "19.20Z": "Raffinage du pétrole",
      "20.11Z": "Fabrication de gaz industriels",
      "20.12Z": "Fabrication de colorants et de pigments",
      "20.13A": "Enrichissement et  retraitement de matières nucléaires",
      "20.13B": "Fabrication d'autres produits chimiques inorganiques de base n.c.a.",
      "20.14Z": "Fabrication d'autres produits chimiques organiques de base",
      "20.15Z": "Fabrication de produits azotés et d'engrais",
      "20.16Z": "Fabrication de matières plastiques de base",
      "20.17Z": "Fabrication de caoutchouc synthétique",
      "20.20Z": "Fabrication de pesticides et d’autres produits agrochimiques",
      "20.30Z": "Fabrication de peintures, vernis, encres et mastics",
      "20.41Z": "Fabrication de savons, détergents et produits d'entretien",
      "20.42Z": "Fabrication de parfums et de produits pour la toilette",
      "20.51Z": "Fabrication de produits explosifs",
      "20.52Z": "Fabrication de colles",
      "20.53Z": "Fabrication d'huiles essentielles",
      "20.59Z": "Fabrication d'autres produits chimiques n.c.a.",
      "20.60Z": "Fabrication de fibres artificielles ou synthétiques",
      "21.10Z": "Fabrication de produits pharmaceutiques de base",
      "21.20Z": "Fabrication de préparations pharmaceutiques",
      "22.11Z": "Fabrication et rechapage de pneumatiques",
      "22.19Z": "Fabrication d'autres articles en caoutchouc",
      "22.21Z": "Fabrication de plaques, feuilles, tubes et profilés en matières plastiques",
      "22.22Z": "Fabrication d'emballages en matières plastiques",
      "22.23Z": "Fabrication d'éléments en matières plastiques pour la construction",
      "22.29A": "Fabrication de pièces techniques à base de matières plastiques",
      "22.29B": "Fabrication de produits de consommation courante en matières plastiques",
      "23.11Z": "Fabrication de verre plat",
      "23.12Z": "Façonnage et transformation du verre plat",
      "23.13Z": "Fabrication de verre creux",
      "23.14Z": "Fabrication de fibres de verre",
      "23.19Z": "Fabrication et façonnage d'autres articles en verre, y compris verre technique",
      "23.20Z": "Fabrication de produits réfractaires",
      "23.31Z": "Fabrication de carreaux en céramique",
      "23.32Z": "Fabrication de briques, tuiles et produits de construction, en terre cuite",
      "23.41Z": "Fabrication d'articles céramiques à usage domestique ou ornemental",
      "23.42Z": "Fabrication d'appareils sanitaires en céramique",
      "23.43Z": "Fabrication d'isolateurs et pièces isolantes en céramique",
      "23.44Z": "Fabrication d'autres produits céramiques à usage technique",
      "23.49Z": "Fabrication d'autres produits céramiques",
      "23.51Z": "Fabrication de ciment",
      "23.52Z": "Fabrication de chaux et plâtre",
      "23.61Z": "Fabrication d'éléments en béton pour la construction",
      "23.62Z": "Fabrication d'éléments en plâtre pour la construction",
      "23.63Z": "Fabrication de béton prêt à l'emploi",
      "23.64Z": "Fabrication de mortiers et bétons secs",
      "23.65Z": "Fabrication d'ouvrages en fibre-ciment",
      "23.69Z": "Fabrication d'autres ouvrages en béton, en ciment ou en plâtre",
      "23.70Z": "Taille, façonnage et finissage de pierres",
      "23.91Z": "Fabrication de produits abrasifs",
      "23.99Z": "Fabrication d'autres produits minéraux non métalliques n.c.a.",
      "24.10Z": "Sidérurgie",
      "24.20Z": "Fabrication de tubes, tuyaux, profilés creux et accessoires correspondants en acier",
      "24.31Z": "Étirage à froid de barres",
      "24.32Z": "Laminage à froid de feuillards",
      "24.33Z": "Profilage à froid par formage ou pliage",
      "24.34Z": "Tréfilage à froid",
      "24.41Z": "Production de métaux précieux",
      "24.42Z": "Métallurgie de l'aluminium",
      "24.43Z": "Métallurgie du plomb, du zinc ou de l'étain",
      "24.44Z": "Métallurgie du cuivre",
      "24.45Z": "Métallurgie des autres métaux non ferreux",
      "24.46Z": "Élaboration et transformation de matières nucléaires",
      "24.51Z": "Fonderie de fonte",
      "24.52Z": "Fonderie d'acier",
      "24.53Z": "Fonderie de métaux légers",
      "24.54Z": "Fonderie d'autres métaux non ferreux",
      "25.11Z": "Fabrication de structures métalliques et de parties de structures",
      "25.12Z": "Fabrication de portes et fenêtres en métal",
      "25.21Z": "Fabrication de radiateurs et de chaudières pour le chauffage central",
      "25.29Z": "Fabrication d'autres réservoirs, citernes et conteneurs métalliques",
      "25.30Z": "Fabrication de générateurs de vapeur, à l'exception des chaudières pour le chauffage central",
      "25.40Z": "Fabrication d'armes et de munitions",
      "25.50A": "Forge, estampage, matriçage ; métallurgie des poudres",
      "25.50B": "Découpage, emboutissage",
      "25.61Z": "Traitement et revêtement des métaux",
      "25.62A": "Décolletage",
      "25.62B": "Mécanique industrielle",
      "25.71Z": "Fabrication de coutellerie",
      "25.72Z": "Fabrication de serrures et de ferrures",
      "25.73A": "Fabrication de moules et modèles",
      "25.73B": "Fabrication d'autres outillages",
      "25.91Z": "Fabrication de fûts et emballages métalliques similaires",
      "25.92Z": "Fabrication d'emballages métalliques légers",
      "25.93Z": "Fabrication d'articles en fils métalliques, de chaînes et de ressorts",
      "25.94Z": "Fabrication de vis et de boulons",
      "25.99A": "Fabrication d'articles métalliques ménagers",
      "25.99B": "Fabrication d'autres articles métalliques",
      "26.11Z": "Fabrication de composants électroniques",
      "26.12Z": "Fabrication de cartes électroniques assemblées",
      "26.20Z": "Fabrication d'ordinateurs et d'équipements périphériques",
      "26.30Z": "Fabrication d'équipements de communication",
      "26.40Z": "Fabrication de produits électroniques grand public",
      "26.51A": "Fabrication d'équipements d'aide à la navigation",
      "26.51B": "Fabrication d'instrumentation scientifique et technique",
      "26.52Z": "Horlogerie",
      "26.60Z": "Fabrication d'équipements d'irradiation médicale, d'équipements électromédicaux et électrothérapeutiques",
      "26.70Z": "Fabrication de matériels optique et photographique",
      "26.80Z": "Fabrication de supports magnétiques et optiques",
      "27.11Z": "Fabrication de moteurs, génératrices et transformateurs électriques",
      "27.12Z": "Fabrication de matériel de distribution et de commande électrique",
      "27.20Z": "Fabrication de piles et d'accumulateurs électriques",
      "27.31Z": "Fabrication de câbles de fibres optiques",
      "27.32Z": "Fabrication d'autres fils et câbles électroniques ou électriques",
      "27.33Z": "Fabrication de matériel d'installation électrique",
      "27.40Z": "Fabrication d'appareils d'éclairage électrique",
      "27.51Z": "Fabrication d'appareils électroménagers",
      "27.52Z": "Fabrication d'appareils ménagers non électriques",
      "27.90Z": "Fabrication d'autres matériels électriques",
      "28.11Z": "Fabrication de moteurs et turbines, à l'exception des moteurs d’avions et de véhicules",
      "28.12Z": "Fabrication d'équipements hydrauliques et pneumatiques",
      "28.13Z": "Fabrication d'autres pompes et compresseurs",
      "28.14Z": "Fabrication d'autres articles de robinetterie",
      "28.15Z": "Fabrication d'engrenages et d'organes mécaniques de transmission",
      "28.21Z": "Fabrication de fours et brûleurs",
      "28.22Z": "Fabrication de matériel de levage et de manutention",
      "28.23Z": "Fabrication de machines et d'équipements de bureau (à l'exception des ordinateurs et équipements périphériques)",
      "28.24Z": "Fabrication d'outillage portatif à moteur incorporé",
      "28.25Z": "Fabrication d'équipements aérauliques et frigorifiques industriels",
      "28.29A": "Fabrication d'équipements d'emballage, de conditionnement et de pesage",
      "28.29B": "Fabrication d'autres machines d'usage général",
      "28.30Z": "Fabrication de machines agricoles et forestières",
      "28.41Z": "Fabrication de machines-outils pour le travail des métaux",
      "28.49Z": "Fabrication d'autres machines-outils",
      "28.91Z": "Fabrication de machines pour la métallurgie",
      "28.92Z": "Fabrication de machines pour l'extraction ou la construction",
      "28.93Z": "Fabrication de machines pour l'industrie agro-alimentaire",
      "28.94Z": "Fabrication de machines pour les industries textiles",
      "28.95Z": "Fabrication de machines pour les industries du papier et du carton",
      "28.96Z": "Fabrication de machines pour le travail du caoutchouc ou des plastiques",
      "28.99A": "Fabrication de machines d'imprimerie",
      "28.99B": "Fabrication d'autres machines spécialisées",
      "29.10Z": "Construction de véhicules automobiles",
      "29.20Z": "Fabrication de carrosseries et remorques",
      "29.31Z": "Fabrication d'équipements électriques et électroniques automobiles",
      "29.32Z": "Fabrication d'autres équipements automobiles",
      "30.11Z": "Construction de navires et de structures flottantes",
      "30.12Z": "Construction de bateaux de plaisance",
      "30.20Z": "Construction de locomotives et d'autre matériel ferroviaire roulant",
      "30.30Z": "Construction aéronautique et spatiale",
      "30.40Z": "Construction de véhicules militaires de combat",
      "30.91Z": "Fabrication de motocycles",
      "30.92Z": "Fabrication de bicyclettes et de véhicules pour invalides",
      "30.99Z": "Fabrication d’autres équipements de transport n.c.a.",
      "31.01Z": "Fabrication de meubles de bureau et de magasin",
      "31.02Z": "Fabrication de meubles de cuisine",
      "31.03Z": "Fabrication de matelas",
      "31.09A": "Fabrication de sièges d'ameublement d'intérieur",
      "31.09B": "Fabrication d’autres meubles et industries connexes de l’ameublement",
      "32.11Z": "Frappe de monnaie",
      "32.12Z": "Fabrication d’articles de joaillerie et bijouterie",
      "32.13Z": "Fabrication d’articles de bijouterie fantaisie et articles similaires",
      "32.20Z": "Fabrication d'instruments de musique",
      "32.30Z": "Fabrication d'articles de sport",
      "32.40Z": "Fabrication de jeux et jouets",
      "32.50A": "Fabrication de matériel médico-chirurgical et dentaire",
      "32.50B": "Fabrication de lunettes",
      "32.91Z": "Fabrication d’articles de brosserie",
      "32.99Z": "Autres activités manufacturières n.c.a.",
      "33.11Z": "Réparation d'ouvrages en métaux",
      "33.12Z": "Réparation de machines et équipements mécaniques",
      "33.13Z": "Réparation de matériels électroniques et optiques",
      "33.14Z": "Réparation d'équipements électriques",
      "33.15Z": "Réparation et maintenance navale",
      "33.16Z": "Réparation et maintenance d'aéronefs et d'engins spatiaux",
      "33.17Z": "Réparation et maintenance d'autres équipements de transport",
      "33.19Z": "Réparation d'autres équipements",
      "33.20A": "Installation de structures métalliques, chaudronnées et de tuyauterie",
      "33.20B": "Installation de machines et équipements mécaniques",
      "33.20C": "Conception d'ensemble et assemblage sur site industriel d'équipements de contrôle des processus industriels",
      "33.20D": "Installation d'équipements électriques, de matériels électroniques et optiques ou d'autres matériels",
      "35.11Z": "Production d'électricité",
      "35.12Z": "Transport d'électricité",
      "35.13Z": "Distribution d'électricité",
      "35.14Z": "Commerce d'électricité",
      "35.21Z": "Production de combustibles gazeux",
      "35.22Z": "Distribution de combustibles gazeux par conduites",
      "35.23Z": "Commerce de combustibles gazeux par conduites",
      "35.30Z": "Production et distribution de vapeur et d'air conditionné",
      "36.00Z": "Captage, traitement et distribution d'eau",
      "37.00Z": "Collecte et traitement des eaux usées",
      "38.11Z": "Collecte des déchets non dangereux",
      "38.12Z": "Collecte des déchets dangereux",
      "38.21Z": "Traitement et élimination des déchets non dangereux",
      "38.22Z": "Traitement et élimination des déchets dangereux",
      "38.31Z": "Démantèlement d'épaves",
      "38.32Z": "Récupération de déchets triés",
      "39.00Z": "Dépollution et autres services de gestion des déchets",
      "41.10A": "Promotion immobilière de logements",
      "41.10B": "Promotion immobilière de bureaux",
      "41.10C": "Promotion immobilière d'autres bâtiments",
      "41.10D": "Supports juridiques de programmes",
      "41.20A": "Construction de maisons individuelles",
      "41.20B": "Construction d'autres bâtiments",
      "42.11Z": "Construction de routes et autoroutes",
      "42.12Z": "Construction de voies ferrées de surface et souterraines",
      "42.13A": "Construction d'ouvrages d'art",
      "42.13B": "Construction et entretien de tunnels",
      "42.21Z": "Construction de réseaux pour fluides",
      "42.22Z": "Construction de réseaux électriques et de télécommunications",
      "42.91Z": "Construction d'ouvrages maritimes et fluviaux",
      "42.99Z": "Construction d'autres ouvrages de génie civil n.c.a.",
      "43.11Z": "Travaux de démolition",
      "43.12A": "Travaux de terrassement courants et travaux préparatoires",
      "43.12B": "Travaux de terrassement spécialisés ou de grande masse",
      "43.13Z": "Forages et sondages",
      "43.21A": "Travaux d'installation électrique dans tous locaux",
      "43.21B": "Travaux d'installation électrique sur la voie publique",
      "43.22A": "Travaux d'installation d'eau et de gaz en tous locaux",
      "43.22B": "Travaux d'installation d'équipements thermiques et de climatisation",
      "43.29A": "Travaux d'isolation",
      "43.29B": "Autres travaux d'installation n.c.a.",
      "43.31Z": "Travaux de plâtrerie",
      "43.32A": "Travaux de menuiserie bois et PVC",
      "43.32B": "Travaux de menuiserie métallique et serrurerie",
      "43.32C": "Agencement de lieux de vente",
      "43.33Z": "Travaux de revêtement des sols et des murs",
      "43.34Z": "Travaux de peinture et vitrerie",
      "43.39Z": "Autres travaux de finition",
      "43.91A": "Travaux de charpente",
      "43.91B": "Travaux de couverture par éléments",
      "43.99A": "Travaux d'étanchéification",
      "43.99B": "Travaux de montage de structures métalliques",
      "43.99C": "Travaux de maçonnerie générale et gros œuvre de bâtiment",
      "43.99D": "Autres travaux spécialisés de construction",
      "43.99E": "Location avec opérateur de matériel de construction",
      "45.11Z": "Commerce de voitures et de véhicules automobiles légers",
      "45.19Z": "Commerce d'autres véhicules automobiles",
      "45.20A": "Entretien et réparation de véhicules automobiles légers",
      "45.20B": "Entretien et réparation d'autres véhicules automobiles",
      "45.31Z": "Commerce de gros d'équipements automobiles",
      "45.32Z": "Commerce de détail d'équipements automobiles",
      "45.40Z": "Commerce et réparation de motocycles",
      "46.11Z": "Intermédiaires du commerce en matières premières agricoles, animaux vivants, matières premières textiles et produits semi-finis",
      "46.12A": "Centrales d'achat de carburant",
      "46.12B": "Autres intermédiaires du commerce en combustibles, métaux, minéraux et produits chimiques",
      "46.13Z": "Intermédiaires du commerce en bois et matériaux de construction",
      "46.14Z": "Intermédiaires du commerce en machines, équipements industriels, navires et avions",
      "46.15Z": "Intermédiaires du commerce en meubles, articles de ménage et quincaillerie",
      "46.16Z": "Intermédiaires du commerce en textiles, habillement, fourrures, chaussures et articles en cuir",
      "46.17A": "Centrales d'achat alimentaires",
      "46.17B": "Autres intermédiaires du commerce en denrées, boissons et tabac",
      "46.18Z": "Intermédiaires spécialisés dans le commerce d'autres produits spécifiques",
      "46.19A": "Centrales d'achat non alimentaires",
      "46.19B": "Autres intermédiaires du commerce en produits divers",
      "46.21Z": "Commerce de gros (commerce interentreprises) de céréales, de tabac non manufacturé, de semences et d'aliments pour le bétail",
      "46.22Z": "Commerce de gros (commerce interentreprises) de fleurs et plantes",
      "46.23Z": "Commerce de gros (commerce interentreprises) d'animaux vivants",
      "46.24Z": "Commerce de gros (commerce interentreprises) de cuirs et peaux",
      "46.31Z": "Commerce de gros (commerce interentreprises) de fruits et légumes",
      "46.32A": "Commerce de gros (commerce interentreprises) de viandes de boucherie",
      "46.32B": "Commerce de gros (commerce interentreprises) de produits à base de viande",
      "46.32C": "Commerce de gros (commerce interentreprises) de volailles et gibier",
      "46.33Z": "Commerce de gros (commerce interentreprises) de produits laitiers, œufs, huiles et matières grasses comestibles",
      "46.34Z": "Commerce de gros (commerce interentreprises) de boissons",
      "46.35Z": "Commerce de gros (commerce interentreprises) de produits à base de tabac",
      "46.36Z": "Commerce de gros (commerce interentreprises) de sucre, chocolat et confiserie",
      "46.37Z": "Commerce de gros (commerce interentreprises) de café, thé, cacao et épices",
      "46.38A": "Commerce de gros (commerce interentreprises) de poissons, crustacés et mollusques",
      "46.38B": "Commerce de gros (commerce interentreprises) alimentaire spécialisé divers",
      "46.39A": "Commerce de gros (commerce interentreprises) de produits surgelés",
      "46.39B": "Commerce de gros (commerce interentreprises) alimentaire non spécialisé",
      "46.41Z": "Commerce de gros (commerce interentreprises) de textiles",
      "46.42Z": "Commerce de gros (commerce interentreprises) d'habillement et de chaussures",
      "46.43Z": "Commerce de gros (commerce interentreprises) d'appareils électroménagers",
      "46.44Z": "Commerce de gros (commerce interentreprises) de vaisselle, verrerie et produits d'entretien",
      "46.45Z": "Commerce de gros (commerce interentreprises) de parfumerie et de produits de beauté",
      "46.46Z": "Commerce de gros (commerce interentreprises) de produits pharmaceutiques",
      "46.47Z": "Commerce de gros (commerce interentreprises) de meubles, de tapis et d'appareils d'éclairage",
      "46.48Z": "Commerce de gros (commerce interentreprises) d'articles d'horlogerie et de bijouterie",
      "46.49Z": "Commerce de gros (commerce interentreprises) d'autres biens domestiques",
      "46.51Z": "Commerce de gros (commerce interentreprises) d'ordinateurs, d'équipements informatiques périphériques et de logiciels",
      "46.52Z": "Commerce de gros (commerce interentreprises) de composants et d'équipements électroniques et de télécommunication",
      "46.61Z": "Commerce de gros (commerce interentreprises) de matériel agricole",
      "46.62Z": "Commerce de gros (commerce interentreprises) de machines-outils",
      "46.63Z": "Commerce de gros (commerce interentreprises) de machines pour l'extraction, la construction et le génie civil",
      "46.64Z": "Commerce de gros (commerce interentreprises) de machines pour l'industrie textile et l'habillement",
      "46.65Z": "Commerce de gros (commerce interentreprises) de mobilier de bureau",
      "46.66Z": "Commerce de gros (commerce interentreprises) d'autres machines et équipements de bureau",
      "46.69A": "Commerce de gros (commerce interentreprises) de matériel électrique",
      "46.69B": "Commerce de gros (commerce interentreprises) de fournitures et équipements industriels divers",
      "46.69C": "Commerce de gros (commerce interentreprises) de fournitures et équipements divers pour le commerce et les services",
      "46.71Z": "Commerce de gros (commerce interentreprises) de combustibles et de produits annexes",
      "46.72Z": "Commerce de gros (commerce interentreprises) de minerais et métaux",
      "46.73A": "Commerce de gros (commerce interentreprises) de bois et de matériaux de construction",
      "46.73B": "Commerce de gros (commerce interentreprises) d'appareils sanitaires et de produits de décoration",
      "46.74A": "Commerce de gros (commerce interentreprises) de quincaillerie",
      "46.74B": "Commerce de gros (commerce interentreprises) de fournitures pour la plomberie et le chauffage",
      "46.75Z": "Commerce de gros (commerce interentreprises) de produits chimiques",
      "46.76Z": "Commerce de gros (commerce interentreprises) d'autres produits intermédiaires",
      "46.77Z": "Commerce de gros (commerce interentreprises) de déchets et débris",
      "46.90Z": "Commerce de gros (commerce interentreprises) non spécialisé",
      "47.11A": "Commerce de détail de produits surgelés",
      "47.11B": "Commerce d'alimentation générale",
      "47.11C": "Supérettes",
      "47.11D": "Supermarchés",
      "47.11E": "Magasins multi-commerces",
      "47.11F": "Hypermarchés",
      "47.19A": "Grands magasins",
      "47.19B": "Autres commerces de détail en magasin non spécialisé",
      "47.21Z": "Commerce de détail de fruits et légumes en magasin spécialisé",
      "47.22Z": "Commerce de détail de viandes et de produits à base de viande en magasin spécialisé",
      "47.23Z": "Commerce de détail de poissons, crustacés et mollusques en magasin spécialisé",
      "47.24Z": "Commerce de détail de pain, pâtisserie et confiserie en magasin spécialisé",
      "47.25Z": "Commerce de détail de boissons en magasin spécialisé",
      "47.26Z": "Commerce de détail de produits à base de tabac en magasin spécialisé",
      "47.29Z": "Autres commerces de détail alimentaires en magasin spécialisé",
      "47.30Z": "Commerce de détail de carburants en magasin spécialisé",
      "47.41Z": "Commerce de détail d'ordinateurs, d'unités périphériques et de logiciels en magasin spécialisé",
      "47.42Z": "Commerce de détail de matériels de télécommunication en magasin spécialisé",
      "47.43Z": "Commerce de détail de matériels audio et vidéo en magasin spécialisé",
      "47.51Z": "Commerce de détail de textiles en magasin spécialisé",
      "47.52A": "Commerce de détail de quincaillerie, peintures et verres en petites surfaces (moins de 400 m2)",
      "47.52B": "Commerce de détail de quincaillerie, peintures et verres en grandes surfaces (400 m2et plus)",
      "47.53Z": "Commerce de détail de tapis, moquettes et revêtements de murs et de sols en magasin spécialisé",
      "47.54Z": "Commerce de détail d'appareils électroménagers en magasin spécialisé",
      "47.59A": "Commerce de détail de meubles",
      "47.59B": "Commerce de détail d'autres équipements du foyer",
      "47.61Z": "Commerce de détail de livres en magasin spécialisé",
      "47.62Z": "Commerce de détail de journaux et papeterie en magasin spécialisé",
      "47.63Z": "Commerce de détail d'enregistrements musicaux et vidéo en magasin spécialisé",
      "47.64Z": "Commerce de détail d'articles de sport en magasin spécialisé",
      "47.65Z": "Commerce de détail de jeux et jouets en magasin spécialisé",
      "47.71Z": "Commerce de détail d'habillement en magasin spécialisé",
      "47.72A": "Commerce de détail de la chaussure",
      "47.72B": "Commerce de détail de maroquinerie et d'articles de voyage",
      "47.73Z": "Commerce de détail de produits pharmaceutiques en magasin spécialisé",
      "47.74Z": "Commerce de détail d'articles médicaux et orthopédiques en magasin spécialisé",
      "47.75Z": "Commerce de détail de parfumerie et de produits de beauté en magasin spécialisé",
      "47.76Z": "Commerce de détail de fleurs, plantes, graines, engrais, animaux de compagnie et aliments pour ces animaux en magasin spécialisé",
      "47.77Z": "Commerce de détail d'articles d'horlogerie et de bijouterie en magasin spécialisé",
      "47.78A": "Commerces de détail d'optique",
      "47.78B": "Commerces de détail de charbons et combustibles",
      "47.78C": "Autres commerces de détail spécialisés divers",
      "47.79Z": "Commerce de détail de biens d'occasion en magasin",
      "47.81Z": "Commerce de détail alimentaire sur éventaires et marchés",
      "47.82Z": "Commerce de détail de textiles, d'habillement et de chaussures sur éventaires et marchés",
      "47.89Z": "Autres commerces de détail sur éventaires et marchés",
      "47.91A": "Vente à distance sur catalogue général",
      "47.91B": "Vente à distance sur catalogue spécialisé",
      "47.99A": "Vente à domicile",
      "47.99B": "Vente par automates et autres commerces de détail hors magasin, éventaires ou marchés n.c.a.",
      "49.10Z": "Transport ferroviaire interurbain de voyageurs",
      "49.20Z": "Transports ferroviaires de fret",
      "49.31Z": "Transports urbains et suburbains de voyageurs",
      "49.32Z": "Transports de voyageurs par taxis",
      "49.39A": "Transports routiers réguliers de voyageurs",
      "49.39B": "Autres transports routiers de voyageurs",
      "49.39C": "Téléphériques et remontées mécaniques",
      "49.41A": "Transports routiers de fret interurbains",
      "49.41B": "Transports routiers de fret de proximité",
      "49.41C": "Location de camions avec chauffeur",
      "49.42Z": "Services de déménagement",
      "49.50Z": "Transports par conduites",
      "50.10Z": "Transports maritimes et côtiers de passagers",
      "50.20Z": "Transports maritimes et côtiers de fret",
      "50.30Z": "Transports fluviaux de passagers",
      "50.40Z": "Transports fluviaux de fret",
      "51.10Z": "Transports aériens de passagers",
      "51.21Z": "Transports aériens de fret",
      "51.22Z": "Transports spatiaux",
      "52.10A": "Entreposage et stockage frigorifique",
      "52.10B": "Entreposage et stockage non frigorifique",
      "52.21Z": "Services auxiliaires des transports terrestres",
      "52.22Z": "Services auxiliaires des transports par eau",
      "52.23Z": "Services auxiliaires des transports aériens",
      "52.24A": "Manutention portuaire",
      "52.24B": "Manutention non portuaire",
      "52.29A": "Messagerie, fret express",
      "52.29B": "Affrètement et organisation des transports",
      "53.10Z": "Activités de poste dans le cadre d'une obligation de service universel",
      "53.20Z": "Autres activités de poste et de courrier",
      "55.10Z": "Hôtels et hébergement similaire",
      "55.20Z": "Hébergement touristique et autre hébergement de courte durée",
      "55.30Z": "Terrains de camping et parcs pour caravanes ou véhicules de loisirs",
      "55.90Z": "Autres hébergements",
      "56.10A": "Restauration traditionnelle",
      "56.10B": "Cafétérias et autres libres-services",
      "56.10C": "Restauration de type rapide",
      "56.21Z": "Services des traiteurs",
      "56.29A": "Restauration collective sous contrat",
      "56.29B": "Autres services de restauration n.c.a.",
      "56.30Z": "Débits de boissons",
      "58.11Z": "Édition de livres",
      "58.12Z": "Édition de répertoires et de fichiers d'adresses",
      "58.13Z": "Édition de journaux",
      "58.14Z": "Édition de revues et périodiques",
      "58.19Z": "Autres activités d'édition",
      "58.21Z": "Édition de jeux électroniques",
      "58.29A": "Édition de logiciels système et de réseau",
      "58.29B": "Edition de logiciels outils de développement et de langages",
      "58.29C": "Edition de logiciels applicatifs",
      "59.11A": "Production de films et de programmes pour la télévision",
      "59.11B": "Production de films institutionnels et publicitaires",
      "59.11C": "Production de films pour le cinéma",
      "59.12Z": "Post-production de films cinématographiques, de vidéo et de programmes de télévision",
      "59.13A": "Distribution de films cinématographiques",
      "59.13B": "Edition et distribution vidéo",
      "59.14Z": "Projection de films cinématographiques",
      "59.20Z": "Enregistrement sonore et édition musicale",
      "60.10Z": "Édition et diffusion de programmes radio",
      "60.20A": "Edition de chaînes généralistes",
      "60.20B": "Edition de chaînes thématiques",
      "61.10Z": "Télécommunications filaires",
      "61.20Z": "Télécommunications sans fil",
      "61.30Z": "Télécommunications par satellite",
      "61.90Z": "Autres activités de télécommunication",
      "62.01Z": "Programmation informatique",
      "62.02A": "Conseil en systèmes et logiciels informatiques",
      "62.02B": "Tierce maintenance de systèmes et d’applications informatiques",
      "62.03Z": "Gestion d'installations informatiques",
      "62.09Z": "Autres activités informatiques",
      "63.11Z": "Traitement de données, hébergement et activités connexes",
      "63.12Z": "Portails Internet",
      "63.91Z": "Activités des agences de presse",
      "63.99Z": "Autres services d'information n.c.a.",
      "64.11Z": "Activités de banque centrale",
      "64.19Z": "Autres intermédiations monétaires",
      "64.20Z": "Activités des sociétés holding",
      "64.30Z": "Fonds de placement et entités financières similaires",
      "64.91Z": "Crédit-bail",
      "64.92Z": "Autre distribution de crédit",
      "64.99Z": "Autres activités des services financiers, hors assurance et caisses de retraite, n.c.a.",
      "65.11Z": "Assurance vie",
      "65.12Z": "Autres assurances",
      "65.20Z": "Réassurance",
      "65.30Z": "Caisses de retraite",
      "66.11Z": "Administration de marchés financiers",
      "66.12Z": "Courtage de valeurs mobilières et de marchandises",
      "66.19A": "Supports juridiques de gestion de patrimoine mobilier",
      "66.19B": "Autres activités auxiliaires de services financiers, hors assurance et caisses de retraite, n.c.a.",
      "66.21Z": "Évaluation des risques et dommages",
      "66.22Z": "Activités des agents et courtiers d'assurances",
      "66.29Z": "Autres activités auxiliaires d'assurance et de caisses de retraite",
      "66.30Z": "Gestion de fonds",
      "68.10Z": "Activités des marchands de biens immobiliers",
      "68.20A": "Location de logements",
      "68.20B": "Location de terrains et d'autres biens immobiliers",
      "68.31Z": "Agences immobilières",
      "68.32A": "Administration d'immeubles et autres biens immobiliers",
      "68.32B": "Supports juridiques de gestion de patrimoine immobilier",
      "69.10Z": "Activités juridiques",
      "69.20Z": "Activités comptables",
      "70.10Z": "Activités des sièges sociaux",
      "70.21Z": "Conseil en relations publiques et communication",
      "70.22Z": "Conseil pour les affaires et autres conseils de gestion",
      "71.11Z": "Activités d'architecture",
      "71.12A": "Activité des géomètres",
      "71.12B": "Ingénierie, études techniques",
      "71.20A": "Contrôle technique automobile",
      "71.20B": "Analyses, essais et inspections techniques",
      "72.11Z": "Recherche-développement en biotechnologie",
      "72.19Z": "Recherche-développement en autres sciences physiques et naturelles",
      "72.20Z": "Recherche-développement en sciences humaines et sociales",
      "73.11Z": "Activités des agences de publicité",
      "73.12Z": "Régie publicitaire de médias",
      "73.20Z": "Études de marché et sondages",
      "74.10Z": "Activités spécialisées de design",
      "74.20Z": "Activités photographiques",
      "74.30Z": "Traduction et interprétation",
      "74.90A": "Activité des économistes de la construction",
      "74.90B": "Activités spécialisées, scientifiques et techniques diverses",
      "75.00Z": "Activités vétérinaires",
      "77.11A": "Location de courte durée de voitures et de véhicules automobiles légers",
      "77.11B": "Location de longue durée de voitures et de véhicules automobiles légers",
      "77.12Z": "Location et location-bail de camions",
      "77.21Z": "Location et location-bail d'articles de loisirs et de sport",
      "77.22Z": "Location de vidéocassettes et disques vidéo",
      "77.29Z": "Location et location-bail d'autres biens personnels et domestiques",
      "77.31Z": "Location et location-bail de machines et équipements agricoles",
      "77.32Z": "Location et location-bail de machines et équipements pour la construction",
      "77.33Z": "Location et location-bail de machines de bureau et de matériel informatique",
      "77.34Z": "Location et location-bail de matériels de transport par eau",
      "77.35Z": "Location et location-bail de matériels de transport aérien",
      "77.39Z": "Location et location-bail d'autres machines, équipements et biens matériels n.c.a.",
      "77.40Z": "Location-bail de propriété intellectuelle et de produits similaires, à l'exception des œuvres soumises à copyright",
      "78.10Z": "Activités des agences de placement de main-d'œuvre",
      "78.20Z": "Activités des agences de travail temporaire",
      "78.30Z": "Autre mise à disposition de ressources humaines",
      "79.11Z": "Activités des agences de voyage",
      "79.12Z": "Activités des voyagistes",
      "79.90Z": "Autres services de réservation et activités connexes",
      "80.10Z": "Activités de sécurité privée",
      "80.20Z": "Activités liées aux systèmes de sécurité",
      "80.30Z": "Activités d'enquête",
      "81.10Z": "Activités combinées de soutien lié aux bâtiments",
      "81.21Z": "Nettoyage courant des bâtiments",
      "81.22Z": "Autres activités de nettoyage des bâtiments et nettoyage industriel",
      "81.29A": "Désinfection, désinsectisation, dératisation",
      "81.29B": "Autres activités de nettoyage n.c.a.",
      "81.30Z": "Services d'aménagement paysager",
      "82.11Z": "Services administratifs combinés de bureau",
      "82.19Z": "Photocopie, préparation de documents et autres activités spécialisées de soutien de bureau",
      "82.20Z": "Activités de centres d'appels",
      "82.30Z": "Organisation de foires, salons professionnels et congrès",
      "82.91Z": "Activités des agences de recouvrement de factures et des sociétés d'information financière sur la clientèle",
      "82.92Z": "Activités de conditionnement",
      "82.99Z": "Autres activités de soutien aux entreprises n.c.a.",
      "84.11Z": "Administration publique générale",
      "84.12Z": "Administration publique (tutelle) de la santé, de la formation, de la culture et des services sociaux, autre que sécurité sociale",
      "84.13Z": "Administration publique (tutelle) des activités économiques",
      "84.21Z": "Affaires étrangères",
      "84.22Z": "Défense",
      "84.23Z": "Justice",
      "84.24Z": "Activités d’ordre public et de sécurité",
      "84.25Z": "Services du feu et de secours",
      "84.30A": "Activités générales de sécurité sociale",
      "84.30B": "Gestion des retraites complémentaires",
      "84.30C": "Distribution sociale de revenus",
      "85.10Z": "Enseignement pré-primaire",
      "85.20Z": "Enseignement primaire",
      "85.31Z": "Enseignement secondaire général",
      "85.32Z": "Enseignement secondaire technique ou professionnel",
      "85.41Z": "Enseignement post-secondaire non supérieur",
      "85.42Z": "Enseignement supérieur",
      "85.51Z": "Enseignement de disciplines sportives et d'activités de loisirs",
      "85.52Z": "Enseignement culturel",
      "85.53Z": "Enseignement de la conduite",
      "85.59A": "Formation continue d'adultes",
      "85.59B": "Autres enseignements",
      "85.60Z": "Activités de soutien à l'enseignement",
      "86.10Z": "Activités hospitalières",
      "86.21Z": "Activité des médecins généralistes",
      "86.22A": "Activités de radiodiagnostic et de radiothérapie",
      "86.22B": "Activités chirurgicales",
      "86.22C": "Autres activités des médecins spécialistes",
      "86.23Z": "Pratique dentaire",
      "86.90A": "Ambulances",
      "86.90B": "Laboratoires d'analyses médicales",
      "86.90C": "Centres de collecte et banques d'organes",
      "86.90D": "Activités des infirmiers et des sages-femmes",
      "86.90E": "Activités des professionnels de la rééducation, de l’appareillage et des pédicures-podologues",
      "86.90F": "Activités de santé humaine non classées ailleurs",
      "87.10A": "Hébergement médicalisé pour personnes âgées",
      "87.10B": "Hébergement médicalisé pour enfants handicapés",
      "87.10C": "Hébergement médicalisé pour adultes handicapés et autre hébergement médicalisé",
      "87.20A": "Hébergement social pour handicapés mentaux et malades mentaux",
      "87.20B": "Hébergement social pour toxicomanes",
      "87.30A": "Hébergement social pour personnes âgées",
      "87.30B": "Hébergement social pour handicapés  physiques",
      "87.90A": "Hébergement social pour enfants en difficultés",
      "87.90B": "Hébergement social pour adultes et familles en difficultés et autre hébergement social",
      "88.10A": "Aide à domicile",
      "88.10B": "Accueil ou accompagnement sans hébergement d’adultes handicapés ou de  personnes âgées",
      "88.10C": "Aide par le travail",
      "88.91A": "Accueil de jeunes enfants",
      "88.91B": "Accueil ou accompagnement sans hébergement d’enfants handicapés",
      "88.99A": "Autre accueil ou accompagnement sans hébergement d’enfants et d’adolescents",
      "88.99B": "Action sociale sans hébergement n.c.a.",
      "90.01Z": "Arts du spectacle vivant",
      "90.02Z": "Activités de soutien au spectacle vivant",
      "90.03A": "Création artistique relevant des arts plastiques",
      "90.03B": "Autre création artistique",
      "90.04Z": "Gestion de salles de spectacles",
      "91.01Z": "Gestion des bibliothèques et des archives",
      "91.02Z": "Gestion des musées",
      "91.03Z": "Gestion des sites et monuments historiques et des attractions touristiques similaires",
      "91.04Z": "Gestion des jardins botaniques et zoologiques et des réserves naturelles",
      "92.00Z": "Organisation de jeux de hasard et d'argent",
      "93.11Z": "Gestion d'installations sportives",
      "93.12Z": "Activités de clubs de sports",
      "93.13Z": "Activités des centres de culture physique",
      "93.19Z": "Autres activités liées au sport",
      "93.21Z": "Activités des parcs d'attractions et parcs à thèmes",
      "93.29Z": "Autres activités récréatives et de loisirs",
      "94.11Z": "Activités des organisations patronales et consulaires",
      "94.12Z": "Activités des organisations professionnelles",
      "94.20Z": "Activités des syndicats de salariés",
      "94.91Z": "Activités des organisations religieuses",
      "94.92Z": "Activités des organisations politiques",
      "94.99Z": "Autres organisations fonctionnant par adhésion volontaire",
      "95.11Z": "Réparation d'ordinateurs et d'équipements périphériques",
      "95.12Z": "Réparation d'équipements de communication",
      "95.21Z": "Réparation de produits électroniques grand public",
      "95.22Z": "Réparation d'appareils électroménagers et d'équipements pour la maison et le jardin",
      "95.23Z": "Réparation de chaussures et d'articles en cuir",
      "95.24Z": "Réparation de meubles et d'équipements du foyer",
      "95.25Z": "Réparation d'articles d'horlogerie et de bijouterie",
      "95.29Z": "Réparation d'autres biens personnels et domestiques",
      "96.01A": "Blanchisserie-teinturerie de gros",
      "96.01B": "Blanchisserie-teinturerie de détail",
      "96.02A": "Coiffure",
      "96.02B": "Soins de beauté",
      "96.03Z": "Services funéraires",
      "96.04Z": "Entretien corporel",
      "96.09Z": "Autres services personnels n.c.a.",
      "97.00Z": "Activités des ménages en tant qu'employeurs de personnel domestique",
      "98.10Z": "Activités indifférenciées des ménages en tant que producteurs de biens pour usage propre",
      "98.20Z": "Activités indifférenciées des ménages en tant que producteurs de services pour usage propre",
      "99.00Z": "Activités des organisations et organismes extraterritoriaux"
  }
}''',

    "NAF2025_SECTIONS": r'''{
  "description": "Nomenclature NAF 2025 (NACE Rev. 2.1, INSEE) — fichier utilisateur Structure_NAF_2025_Maj_2024-10-04. 'sections' = lettre A-V -> intitule (22). 'division_section' = division (2 chiffres) -> lettre de section (87), deduite de la feuille hierarchique 'NAF 2025'. La section se deduit de la division du code NAF. ATTENTION : en NAF 2025 les lettres de section different du NAF rev.2/2008 (J,K,L,M... decalees) et la division 45 (commerce/reparation auto) n'existe plus (redistribuee) -> codes 45.* sans section.",
  "sections": {
    "A": "AGRICULTURE, SYLVICULTURE ET PÊCHE",
    "B": "INDUSTRIES EXTRACTIVES",
    "C": "INDUSTRIE MANUFACTURIÈRE",
    "D": "PRODUCTION ET DISTRIBUTION D’ÉLECTRICITÉ, DE GAZ, DE VAPEUR ET D’AIR CONDITIONNÉ",
    "E": "PRODUCTION ET DISTRIBUTION D’EAU ; ASSAINISSEMENT, GESTION DES DÉCHETS ET DÉPOLLUTION",
    "F": "CONSTRUCTION",
    "G": "COMMERCE",
    "H": "TRANSPORTS ET ENTREPOSAGE",
    "I": "HÉBERGEMENT ET RESTAURATION",
    "J": "ÉDITION, DIFFUSION ET ACTIVITÉS DE PRODUCTION ET DE DISTRIBUTION DE CONTENU",
    "K": "TÉLÉCOMMUNICATIONS, PROGRAMMATION INFORMATIQUE, CONSEIL, INFRASTRUCTURE INFORMATIQUE ET AUTRES ACTIVITÉS DE SERVICE INFORMATIQUE",
    "L": "ACTIVITÉS FINANCIÈRES ET D’ASSURANCE",
    "M": "ACTIVITÉS IMMOBILIÈRES",
    "N": "ACTIVITÉS SPÉCIALISÉES, SCIENTIFIQUES ET TECHNIQUES",
    "O": "ACTIVITÉS DE SERVICE ADMINISTRATIF ET DE SOUTIEN",
    "P": "ADMINISTRATION PUBLIQUE ET DÉFENSE ; SÉCURITÉ SOCIALE OBLIGATOIRE",
    "Q": "ENSEIGNEMENT",
    "R": "SANTÉ HUMAINE ET ACTIVITÉS D’ACTION SOCIALE",
    "S": "ARTS, SPORTS ET ACTIVITÉS RÉCRÉATIVES",
    "T": "AUTRES ACTIVITÉS DE SERVICES",
    "U": "ACTIVITÉS DES MÉNAGES EN TANT QU’EMPLOYEURS ; ACTIVITÉS INDIFFÉRENCIÉES DES MÉNAGES EN TANT QUE PRODUCTEURS DE BIENS ET SERVICES POUR USAGE PROPRE",
    "V": "ACTIVITÉS DES ORGANISATIONS ET ORGANISMES EXTRATERRITORIAUX"
  },
  "division_section": {
    "01": "A",
    "02": "A",
    "03": "A",
    "05": "B",
    "06": "B",
    "07": "B",
    "08": "B",
    "09": "B",
    "10": "C",
    "11": "C",
    "12": "C",
    "13": "C",
    "14": "C",
    "15": "C",
    "16": "C",
    "17": "C",
    "18": "C",
    "19": "C",
    "20": "C",
    "21": "C",
    "22": "C",
    "23": "C",
    "24": "C",
    "25": "C",
    "26": "C",
    "27": "C",
    "28": "C",
    "29": "C",
    "30": "C",
    "31": "C",
    "32": "C",
    "33": "C",
    "35": "D",
    "36": "E",
    "37": "E",
    "38": "E",
    "39": "E",
    "41": "F",
    "42": "F",
    "43": "F",
    "46": "G",
    "47": "G",
    "49": "H",
    "50": "H",
    "51": "H",
    "52": "H",
    "53": "H",
    "55": "I",
    "56": "I",
    "58": "J",
    "59": "J",
    "60": "J",
    "61": "K",
    "62": "K",
    "63": "K",
    "64": "L",
    "65": "L",
    "66": "L",
    "68": "M",
    "69": "N",
    "70": "N",
    "71": "N",
    "72": "N",
    "73": "N",
    "74": "N",
    "75": "N",
    "77": "O",
    "78": "O",
    "79": "O",
    "80": "O",
    "81": "O",
    "82": "O",
    "84": "P",
    "85": "Q",
    "86": "R",
    "87": "R",
    "88": "R",
    "90": "S",
    "91": "S",
    "92": "S",
    "93": "S",
    "94": "T",
    "95": "T",
    "96": "T",
    "97": "U",
    "98": "U",
    "99": "V"
  }
}''',

    "OPERATEURS_ETAT": r'''{
  "description": "SIREN des opérateurs de l'État — 443 entités (v5_28), dont 438 avec période(s) de validité (fichier OPETAT.xlsx officiel, mis à jour périodiquement -- format PAR SIREN : liste de [année_début, année_fin], car un même SIREN peut connaître PLUSIEURS périodes successives sous des noms différents, ex. Pôle emploi 2000-2024 -> France Travail 2025-2026, même SIREN 130005481 ; idem ENSTA Paris/ENSTA et Mobilier National/Manufactures nationales). 7 nouveaux SIREN (universités renommées/fusionnées Besançon/Brest/Toulouse/Montpellier, Pass Culture, Solidéo Alpes 2030) ; 5 anciens SIREN d'universités superseded restent dans la liste SANS période (faute de date de fin officielle), à traiter si l'utilisateur confirme la date de bascule. Corrections reprises (déjà vérifiées v5_23/24) : EPMQB, EPPGHV, ACTIA, CROUS Clermont-Ferrand/Corte, Paris VIII/Université de Paris -- ces deux derniers homonymes de SIREN dans OPETAT re-résolus de la même façon qu'avant. EPAURIF : doublon (2024 vs 2026) résolu par la valeur la plus permissive, à confirmer. (v5_26) : 435 de la v5_25 + INSHEA (130000383), confirmé par l'utilisateur et recoupement officiel (Wikipédia, annuaire-entreprises.data.gouv.fr, societe.com) : l'INSHEA (Institut national supérieur de formation et de recherche pour l'éducation des jeunes handicapés et les enseignements adaptés) a été renommé INSEI (éducation inclusive) par décret du 3 mars 2023, même SIREN, siège à Suresnes. TOUS les opérateurs de l'annexe Jaune 2026 sont désormais résolus, à une exception : Université de Paris (établissement expérimental), qui continue d'hériter à tort, dans le fichier source, du SIREN de l'Université Paris VIII (199318270, confirmé) — vrai doublon, reste exclu plutôt que deviné.",
  "periodes": {
      "130000136": [[2000, 2026]],
      "130000383": [[2000, 2026]],
      "130001316": [[2000, 2026]],
      "130002140": [[2000, 2026]],
      "130002504": [[2000, 2026]],
      "130002645": [[2000, 2026]],
      "130002702": [[2000, 2026]],
      "130002728": [[2000, 2026]],
      "130002850": [[2000, 2026]],
      "130002884": [[2000, 2026]],
      "130003221": [[2000, 2026]],
      "130003262": [[2000, 2026]],
      "130003320": [[2000, 2026]],
      "130003643": [[2000, 2026]],
      "130003759": [[2000, 2026]],
      "130003767": [[2000, 2026]],
      "130003981": [[2000, 2026]],
      "130004278": [[2000, 2026]],
      "130005457": [[2000, 2026]],
      "130005481": [[2000, 2024], [2025, 2026]],
      "130006356": [[2000, 2026]],
      "130006364": [[2000, 2026]],
      "130006372": [[2000, 2026]],
      "130006513": [[2000, 2026]],
      "130006547": [[2000, 2026]],
      "130007545": [[2000, 2026]],
      "130007834": [[2000, 2026]],
      "130007842": [[2000, 2026]],
      "130007859": [[2000, 2026]],
      "130007867": [[2000, 2026]],
      "130007883": [[2000, 2026]],
      "130007909": [[2000, 2026]],
      "130007933": [[2000, 2026]],
      "130007966": [[2000, 2026]],
      "130007974": [[2000, 2026]],
      "130007982": [[2000, 2026]],
      "130007990": [[2000, 2026]],
      "130008006": [[2000, 2026]],
      "130008014": [[2000, 2026]],
      "130008030": [[2000, 2026]],
      "130008048": [[2000, 2026]],
      "130008071": [[2000, 2026]],
      "130008121": [[2000, 2026]],
      "130008535": [[2000, 2026]],
      "130008584": [[2000, 2026]],
      "130008857": [[2000, 2025]],
      "130010440": [[2000, 2026]],
      "130010804": [[2000, 2026]],
      "130011844": [[2000, 2026]],
      "130012024": [[2000, 2026]],
      "130012172": [[2000, 2026]],
      "130013097": [[2000, 2026]],
      "130014228": [[2000, 2026]],
      "130014442": [[2000, 2026]],
      "130014541": [[2000, 2026]],
      "130015001": [[2000, 2026]],
      "130015332": [[2000, 2026]],
      "130015506": [[2000, 2026]],
      "130015712": [[2000, 2026]],
      "130016314": [[2000, 2026]],
      "130016371": [[2000, 2026]],
      "130016793": [[2000, 2026]],
      "130017791": [[2000, 2026]],
      "130017890": [[2000, 2026]],
      "130018310": [[2000, 2026]],
      "130018336": [[2000, 2026]],
      "130018351": [[2000, 2026]],
      "130018484": [[2000, 2026]],
      "130020464": [[2000, 2026]],
      "130020597": [[2000, 2026]],
      "130020910": [[2000, 2025]],
      "130021256": [[2000, 2026]],
      "130021330": [[2000, 2026]],
      "130021421": [[2000, 2026]],
      "130021447": [[2000, 2024]],
      "130021454": [[2000, 2026]],
      "130021470": [[2000, 2024]],
      "130021918": [[2000, 2026]],
      "130021959": [[2000, 2026]],
      "130022551": [[2000, 2026]],
      "130023088": [[2000, 2026]],
      "130023385": [[2000, 2026]],
      "130024417": [[2000, 2026]],
      "130024425": [[2000, 2026]],
      "130024433": [[2000, 2026]],
      "130024540": [[2000, 2026]],
      "130024565": [[2000, 2026]],
      "130025281": [[2000, 2026]],
      "130025620": [[2000, 2026]],
      "130025661": [[2000, 2026]],
      "130025745": [[2000, 2026]],
      "130025752": [[2000, 2026]],
      "130025919": [[2000, 2026]],
      "130025927": [[2000, 2026]],
      "130025935": [[2000, 2026]],
      "130025976": [[2000, 2026]],
      "130026024": [[2000, 2026]],
      "130026032": [[2000, 2026]],
      "130026057": [[2000, 2026]],
      "130026081": [[2000, 2026]],
      "130026123": [[2000, 2026]],
      "130026149": [[2000, 2026]],
      "130026222": [[2000, 2026]],
      "130028061": [[2000, 2026]],
      "130029747": [[2000, 2026]],
      "130029754": [[2000, 2026]],
      "130029796": [[2000, 2026]],
      "130030133": [[2000, 2026]],
      "130030190": [[2000, 2026]],
      "130030513": [[2000, 2026]],
      "130030612": [[2000, 2026]],
      "130030620": [[2000, 2026]],
      "130030638": [[2000, 2024]],
      "130030851": [[2000, 2026]],
      "150000966": [[2000, 2026]],
      "180000010": [[2000, 2026]],
      "180000028": [[2000, 2026]],
      "180005019": [[2000, 2026]],
      "180006025": [[2000, 2026]],
      "180006033": [[2000, 2026]],
      "180006082": [[2000, 2026]],
      "180007015": [[2000, 2026]],
      "180007023": [[2000, 2026]],
      "180034027": [[2000, 2026]],
      "180036048": [[2000, 2026]],
      "180036105": [[2000, 2026]],
      "180037012": [[2000, 2026]],
      "180037020": [[2000, 2026]],
      "180043010": [[2000, 2026]],
      "180043028": [[2000, 2026]],
      "180043036": [[2000, 2026]],
      "180043069": [[2000, 2026]],
      "180043093": [[2000, 2026]],
      "180043127": [[2000, 2026]],
      "180044018": [[2000, 2026]],
      "180044034": [[2000, 2026]],
      "180044067": [[2000, 2026]],
      "180044083": [[2000, 2026]],
      "180044091": [[2000, 2026]],
      "180044117": [[2000, 2026]],
      "180044125": [[2000, 2026]],
      "180044174": [[2000, 2026]],
      "180044224": [[2000, 2026]],
      "180044232": [[2000, 2026]],
      "180046013": [[2000, 2026]],
      "180046021": [[2000, 2026]],
      "180046039": [[2000, 2026]],
      "180046054": [[2000, 2026]],
      "180046187": [[2000, 2026]],
      "180046195": [[2000, 2026]],
      "180046237": [[2000, 2026]],
      "180046252": [[2000, 2026]],
      "180046260": [[2000, 2026]],
      "180053027": [[2000, 2026]],
      "180060030": [[2000, 2026]],
      "180065021": [[2000, 2026]],
      "180067019": [[2000, 2026]],
      "180067027": [[2000, 2026]],
      "180070039": [[2000, 2026]],
      "180080012": [[2000, 2026]],
      "180089013": [[2000, 2026]],
      "180089047": [[2000, 2026]],
      "180089369": [[2000, 2026]],
      "180089476": [[2000, 2026]],
      "180089500": [[2000, 2026]],
      "180090011": [[2000, 2026]],
      "180090029": [[2000, 2026]],
      "180090052": [[2000, 2026]],
      "180092025": [[2000, 2026]],
      "180092033": [[2000, 2026]],
      "180092058": [[2000, 2026]],
      "180092082": [[2000, 2026]],
      "180092140": [[2000, 2026]],
      "180092199": [[2000, 2026]],
      "180092207": [[2000, 2026]],
      "180092215": [[2000, 2026]],
      "180092231": [[2000, 2026]],
      "180092256": [[2000, 2026]],
      "180092264": [[2000, 2026]],
      "180092272": [[2000, 2026]],
      "180092371": [[2000, 2026]],
      "180092389": [[2000, 2026]],
      "180092397": [[2000, 2026]],
      "180092405": [[2000, 2026]],
      "180092413": [[2000, 2026]],
      "180092447": [[2000, 2026]],
      "180092538": [[2000, 2026]],
      "180092553": [[2000, 2026]],
      "180503013": [[2000, 2026]],
      "180600041": [[2000, 2026]],
      "180600058": [[2000, 2026]],
      "181300088": [[2000, 2026]],
      "182020107": [[2000, 2026]],
      "183100064": [[2000, 2026]],
      "183100072": [[2000, 2026]],
      "183400084": [[2000, 2026]],
      "183801562": [[2000, 2026]],
      "184401321": [[2000, 2026]],
      "184401339": [[2000, 2026]],
      "184500213": [[2000, 2026]],
      "184800050": [[2000, 2026]],
      "185102001": [[2000, 2026]],
      "185422102": [[2000, 2026]],
      "185703014": [[2000, 2026]],
      "185722949": [[2000, 2026]],
      "185911500": [[2000, 2026]],
      "185911781": [[2000, 2026]],
      "186306973": [[2000, 2026]],
      "186500047": [[2000, 2026]],
      "186706446": [[2000, 2026]],
      "186901559": [[2000, 2026]],
      "186901567": [[2000, 2026]],
      "187300033": [[2000, 2026]],
      "187500061": [[2000, 2026]],
      "187500079": [[2000, 2026]],
      "187500095": [[2000, 2026]],
      "187512512": [[2000, 2026]],
      "187512553": [[2000, 2026]],
      "187512702": [[2000, 2026]],
      "187512777": [[2000, 2026]],
      "187800081": [[2000, 2026]],
      "188002000": [[2000, 2026]],
      "188300057": [[2000, 2026]],
      "188600050": [[2000, 2026]],
      "188719009": [[2000, 2026]],
      "189100142": [[2000, 2026]],
      "189400047": [[2000, 2026]],
      "189710023": [[2000, 2026]],
      "189710080": [[2000, 2026]],
      "189740012": [[2000, 2026]],
      "190608364": [[2000, 2026]],
      "190615633": [[2000, 2026]],
      "191010602": [[2000, 2026]],
      "191302363": [[2000, 2026]],
      "191333400": [[2000, 2026]],
      "191333426": [[2000, 2026]],
      "191333467": [[2000, 2026]],
      "191414085": [[2000, 2026]],
      "191417203": [[2000, 2026]],
      "191700327": [[2000, 2026]],
      "192020907": [[2000, 2026]],
      "192026649": [[2000, 2026]],
      "192500825": [[2000, 2026]],
      "192901197": [[2000, 2026]],
      "192901254": [[2000, 2025]],
      "193101433": [[2000, 2026]],
      "193101508": [[2000, 2026]],
      "193101524": [[2000, 2026]],
      "193101532": [[2000, 2026]],
      "193112562": [[2000, 2026]],
      "193113818": [[2000, 2026]],
      "193113834": [[2000, 2026]],
      "193113867": [[2000, 2026]],
      "193301926": [[2000, 2026]],
      "193301991": [[2000, 2026]],
      "193302031": [[2000, 2026]],
      "193317666": [[2000, 2026]],
      "193322393": [[2000, 2026]],
      "193401122": [[2000, 2026]],
      "193401320": [[2000, 2026]],
      "193415940": [[2000, 2026]],
      "193500774": [[2000, 2026]],
      "193500899": [[2000, 2026]],
      "193500972": [[2000, 2026]],
      "193509379": [[2000, 2026]],
      "193523172": [[2000, 2026]],
      "193708005": [[2000, 2026]],
      "193801347": [[2000, 2026]],
      "193801412": [[2000, 2026]],
      "193819125": [[2000, 2026]],
      "194216149": [[2000, 2026]],
      "194401006": [[2000, 2026]],
      "194401048": [[2000, 2026]],
      "194416186": [[2000, 2026]],
      "194508552": [[2000, 2026]],
      "194909701": [[2000, 2026]],
      "195112966": [[2000, 2026]],
      "195401351": [[2000, 2026]],
      "195600853": [[2000, 2026]],
      "195617188": [[2000, 2026]],
      "195726476": [[2000, 2026]],
      "195903372": [[2000, 2026]],
      "195903380": [[2000, 2026]],
      "195903497": [[2000, 2026]],
      "195936489": [[2000, 2026]],
      "195944038": [[2000, 2026]],
      "195958764": [[2000, 2026]],
      "196012231": [[2000, 2026]],
      "196244016": [[2000, 2026]],
      "196312870": [[2000, 2026]],
      "196402515": [[2000, 2026]],
      "196500482": [[2000, 2026]],
      "196604375": [[2000, 2026]],
      "196701866": [[2000, 2026]],
      "196701890": [[2000, 2026]],
      "196727671": [[2000, 2026]],
      "196811665": [[2000, 2026]],
      "196901730": [[2000, 2026]],
      "196901847": [[2000, 2026]],
      "196901870": [[2000, 2026]],
      "196901896": [[2000, 2026]],
      "196901920": [[2000, 2026]],
      "196917744": [[2000, 2026]],
      "196917751": [[2000, 2026]],
      "196918619": [[2000, 2026]],
      "196924377": [[2000, 2026]],
      "196924591": [[2000, 2026]],
      "197209166": [[2000, 2026]],
      "197308588": [[2000, 2026]],
      "197400682": [[2000, 2026]],
      "197500028": [[2000, 2026]],
      "197500036": [[2000, 2025], [2026, 2026]],
      "197506611": [[2000, 2026]],
      "197507734": [[2000, 2026]],
      "197512346": [[2000, 2026]],
      "197517170": [[2000, 2026]],
      "197517188": [[2000, 2026]],
      "197517196": [[2000, 2026]],
      "197518756": [[2000, 2026]],
      "197518772": [[2000, 2026]],
      "197529050": [[2000, 2026]],
      "197534282": [[2000, 2026]],
      "197534316": [[2000, 2026]],
      "197534597": [[2000, 2026]],
      "197534639": [[2000, 2026]],
      "197534696": [[2000, 2026]],
      "197534704": [[2000, 2026]],
      "197534712": [[2000, 2026]],
      "197534720": [[2000, 2026]],
      "197534787": [[2000, 2026]],
      "197534803": [[2000, 2026]],
      "197534860": [[2000, 2026]],
      "197534886": [[2000, 2026]],
      "197534936": [[2000, 2026]],
      "197534951": [[2000, 2026]],
      "197534969": [[2000, 2026]],
      "197535016": [[2000, 2026]],
      "197536675": [[2000, 2026]],
      "197537426": [[2000, 2026]],
      "197546872": [[2000, 2026]],
      "197546880": [[2000, 2026]],
      "197546922": [[2000, 2026]],
      "197601644": [[2000, 2026]],
      "197601651": [[2000, 2026]],
      "197619042": [[2000, 2026]],
      "197627623": [[2000, 2026]],
      "197804123": [[2000, 2026]],
      "197819444": [[2000, 2026]],
      "197820194": [[2000, 2026]],
      "198013443": [[2000, 2026]],
      "198112013": [[2000, 2026]],
      "198307662": [[2000, 2026]],
      "198406852": [[2000, 2026]],
      "198600736": [[2000, 2026]],
      "198608564": [[2000, 2026]],
      "198706699": [[2000, 2026]],
      "199003567": [[2000, 2026]],
      "199119751": [[2000, 2026]],
      "199212044": [[2000, 2026]],
      "199306036": [[2000, 2026]],
      "199312380": [[2000, 2026]],
      "199318270": [[2000, 2026]],
      "199322306": [[2000, 2026]],
      "199406075": [[2000, 2026]],
      "199406083": [[2000, 2026]],
      "199411117": [[2000, 2026]],
      "199513763": [[2000, 2026]],
      "199715855": [[2000, 2026]],
      "199744780": [[2000, 2026]],
      "199870015": [[2000, 2026]],
      "200008357": [[2000, 2026]],
      "200008431": [[2000, 2026]],
      "200018893": [[2000, 2026]],
      "200090777": [[2000, 2026]],
      "302977145": [[2000, 2026]],
      "303017842": [[2000, 2026]],
      "306664863": [[2000, 2026]],
      "313320244": [[2000, 2026]],
      "322224718": [[2000, 2026]],
      "330715368": [[2000, 2026]],
      "331118760": [[2000, 2026]],
      "331596270": [[2000, 2026]],
      "338840564": [[2000, 2026]],
      "381984921": [[2000, 2026]],
      "385290309": [[2000, 2026]],
      "390199669": [[2000, 2026]],
      "391406956": [[2000, 2026]],
      "391718970": [[2000, 2026]],
      "417822632": [[2000, 2026]],
      "420619439": [[2000, 2026]],
      "421506445": [[2000, 2026]],
      "431959956": [[2000, 2026]],
      "440546018": [[2000, 2025]],
      "441357340": [[2000, 2026]],
      "451930051": [[2000, 2026]],
      "478184906": [[2000, 2026]],
      "488480005": [[2000, 2026]],
      "519587851": [[2000, 2026]],
      "521747444": [[2000, 2026]],
      "524523396": [[2000, 2026]],
      "525046017": [[2000, 2026]],
      "529715922": [[2000, 2026]],
      "534124334": [[2000, 2026]],
      "582056149": [[2000, 2026]],
      "588502310": [[2000, 2026]],
      "662043116": [[2000, 2026]],
      "692039514": [[2000, 2026]],
      "692041585": [[2000, 2026]],
      "752195438": [[2000, 2026]],
      "775664105": [[2000, 2026]],
      "775665912": [[2000, 2026]],
      "775671464": [[2000, 2026]],
      "775685019": [[2000, 2026]],
      "775722879": [[2000, 2026]],
      "775724644": [[2000, 2025]],
      "775729155": [[2000, 2026]],
      "784276180": [[2000, 2026]],
      "784308249": [[2000, 2026]],
      "784396079": [[2000, 2026]],
      "784523318": [[2000, 2026]],
      "784616989": [[2000, 2026]],
      "784804593": [[2000, 2026]],
      "788105245": [[2000, 2026]],
      "824228142": [[2000, 2026]],
      "824544514": [[2000, 2026]],
      "853318459": [[2026, 2026]],
      "882539786": [[2000, 2026]],
      "884439035": [[2000, 2026]],
      "902883057": [[2000, 2026]],
      "910559319": [[2000, 2026]],
      "932490899": [[2000, 2026]],
      "938106564": [[2000, 2026]],
      "938271392": [[2000, 2026]],
      "939106274": [[2000, 2025], [2026, 2026]],
      "941298317": [[2000, 2026]],
      "941636342": [[2026, 2026]],
      "991460858": [[2000, 2026]],
      "999325392": [[2000, 2026]]
  },
  "programmes": {
      "130030513": "150 – Formations supérieures et recherche universitaire",
      "932490899": "150 – Formations supérieures et recherche universitaire",
      "938106564": "150 – Formations supérieures et recherche universitaire",
      "938271392": "150 – Formations supérieures et recherche universitaire",
      "941298317": "150 – Formations supérieures et recherche universitaire",
      "130000136": "155 – soutien des ministères sociaux",
      "130000383": "150 – Formations supérieures et recherche universitaire",
      "130001316": "203 – Infrastructures et services de transports",
      "130002140": "150 – Formations supérieures et recherche universitaire",
      "130002504": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "130002645": "219 – Sport",
      "130002702": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "130002728": "175 – Patrimoines",
      "130002850": "142 – Enseignement supérieur et recherche agricoles",
      "130002884": "217 – Conduite et pilotage des politiques de l'écologie, du développement et de la mobilité durables",
      "130003221": "150 – Formations supérieures et recherche universitaire",
      "130003262": "354 – Administration territoriale de l'État",
      "130003320": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "130003643": "150 – Formations supérieures et recherche universitaire",
      "130003759": "150 – Formations supérieures et recherche universitaire",
      "130003767": "150 – Formations supérieures et recherche universitaire",
      "130003981": "212 – Soutien de la politique de la défense",
      "130004278": "de l'espace",
      "130005457": "150 – Formations supérieures et recherche universitaire",
      "130005481": "102 – Accès et retour à l'emploi",
      "130006356": "150 – Formations supérieures et recherche universitaire",
      "130006364": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "130006372": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "130006513": "175 – Patrimoines",
      "130006547": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "130007545": "150 – Formations supérieures et recherche universitaire",
      "130007834": "155 – soutien des ministères sociaux",
      "130007842": "155 – soutien des ministères sociaux",
      "130007859": "155 – soutien des ministères sociaux",
      "130007867": "155 – soutien des ministères sociaux",
      "130007883": "155 – soutien des ministères sociaux",
      "130007909": "155 – soutien des ministères sociaux",
      "130007933": "155 – soutien des ministères sociaux",
      "130007966": "155 – soutien des ministères sociaux",
      "130007974": "155 – soutien des ministères sociaux",
      "130007982": "155 – soutien des ministères sociaux",
      "130007990": "155 – soutien des ministères sociaux",
      "130008006": "155 – soutien des ministères sociaux",
      "130008014": "155 – soutien des ministères sociaux",
      "130008030": "155 – soutien des ministères sociaux",
      "130008048": "155 – soutien des ministères sociaux",
      "130008071": "155 – soutien des ministères sociaux",
      "130008121": "150 – Formations supérieures et recherche universitaire",
      "130008535": "142 – Enseignement supérieur et recherche agricoles",
      "130008584": "142 – Enseignement supérieur et recherche agricoles",
      "130008857": "131 – Création",
      "130010440": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "130010804": "219 – Sport",
      "130011844": "163 – Jeunesse et vie associative",
      "130012024": "206 – Sécurité et qualité sanitaires de l'alimentation",
      "130012172": "175 – Patrimoines",
      "130013097": "205 – Affaires maritimes, pêche et aquaculture",
      "130014228": "192 – Recherche et enseignement supérieur en matière économique et industrielle",
      "130014442": "310 – Conduite et pilotage de la politique de la justice",
      "130014541": "753 – Contrôle et modernisation de la politique de la circulation et du stationnement routiers",
      "130015001": "113 – Paysages, eau et biodiversité",
      "130015332": "150 – Formations supérieures et recherche universitaire",
      "130015506": "150 – Formations supérieures et recherche universitaire",
      "130015712": "216 – Conduite et pilotage des politiques de l'intérieur",
      "130016314": "150 – Formations supérieures et recherche universitaire",
      "130016371": "150 – Formations supérieures et recherche universitaire",
      "130016793": "113 – Paysages, eau et biodiversité",
      "130017791": "203 – Infrastructures et services de transports",
      "130017890": "175 – Patrimoines",
      "130018310": "159 – Expertise, information géographique et météorologie",
      "130018336": "150 – Formations supérieures et recherche universitaire",
      "130018351": "150 – Formations supérieures et recherche universitaire",
      "130018484": "150 – Formations supérieures et recherche universitaire",
      "130020464": "135 – Urbanisme, territoires et amélioration de l'habitat",
      "130020597": "150 – Formations supérieures et recherche universitaire",
      "130020910": "150 – Formations supérieures et recherche universitaire",
      "130021256": "150 – Formations supérieures et recherche universitaire",
      "130021330": "150 – Formations supérieures et recherche universitaire",
      "130021421": "150 – Formations supérieures et recherche universitaire",
      "130021447": "150 – Formations supérieures et recherche universitaire",
      "130021454": "150 – Formations supérieures et recherche universitaire",
      "130021470": "150 – Formations supérieures et recherche universitaire",
      "130021918": "150 – Formations supérieures et recherche universitaire",
      "130021959": "138 – Emploi outre-mer",
      "130022551": "135 – Urbanisme, territoires et amélioration de l'habitat",
      "130023088": "175 – Patrimoines",
      "130023385": "150 – Formations supérieures et recherche universitaire",
      "130024417": "161 – Sécurité civile",
      "130024425": "231 – Vie étudiante",
      "130024433": "231 – Vie étudiante",
      "130024540": "178 – Préparation et emploi des forces",
      "130024565": "103 – Accompagnement des mutations économiques et développement de l'emploi",
      "130025281": "219 – Sport",
      "130025620": "144 – Environnement et prospective de la politique de défense",
      "130025661": "150 – Formations supérieures et recherche universitaire",
      "130025745": "150 – Formations supérieures et recherche universitaire",
      "130025752": "150 – Formations supérieures et recherche universitaire",
      "130025919": "113 – Paysages, eau et biodiversité",
      "130025927": "155 – soutien des ministères sociaux",
      "130025935": "113 – Paysages, eau et biodiversité",
      "130025976": "150 – Formations supérieures et recherche universitaire",
      "130026024": "150 – Formations supérieures et recherche universitaire",
      "130026032": "112 – Impulsion et coordination de la politique d'aménagement du territoire",
      "130026057": "155 – soutien des ministères sociaux",
      "130026081": "150 – Formations supérieures et recherche universitaire",
      "130026123": "150 – Formations supérieures et recherche universitaire",
      "130026149": "150 – Formations supérieures et recherche universitaire",
      "130026222": "142 – Enseignement supérieur et recherche agricoles",
      "130028061": "150 – Formations supérieures et recherche universitaire",
      "130029747": "150 – Formations supérieures et recherche universitaire",
      "130029754": "150 – Formations supérieures et recherche universitaire",
      "130029796": "150 – Formations supérieures et recherche universitaire",
      "130030133": "102 – Accès et retour à l'emploi",
      "130030190": "103 – Accompagnement des mutations économiques et développement de l'emploi",
      "130030612": "150 – Formations supérieures et recherche universitaire",
      "130030620": "150 – Formations supérieures et recherche universitaire",
      "130030638": "304 – Inclusion sociale et protection des personnes",
      "130030851": "216 – Conduite et pilotage des politiques de l'intérieur",
      "150000966": "178 – Préparation et emploi des forces",
      "180000010": "129 – Coordination du travail gouvernemental",
      "180000028": "169 – Reconnaissance et réparation en faveur du monde combattant, mémoire et liens avec la Nation",
      "180005019": "113 – Paysages, eau et biodiversité",
      "180006025": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180006033": "303 – Immigration et asile",
      "180006082": "185 – Diplomatie culturelle et d'influence",
      "180007015": "169 – Reconnaissance et réparation en faveur du monde combattant, mémoire et liens avec la Nation",
      "180007023": "169 – Reconnaissance et réparation en faveur du monde combattant, mémoire et liens avec la Nation",
      "180034027": "104 – Intégration et accès à la nationalité française",
      "180036048": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180036105": "129 – Coordination du travail gouvernemental",
      "180037012": "111 – Amélioration de la qualité de l'emploi et des relations du travail",
      "180037020": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180043010": "214 – Soutien de la politique de l'éducation nationale",
      "180043028": "214 – Soutien de la politique de l'éducation nationale",
      "180043036": "214 – Soutien de la politique de l'éducation nationale",
      "180043069": "214 – Soutien de la politique de l'éducation nationale",
      "180043093": "334 – Livre et industries culturelles",
      "180043127": "150 – Formations supérieures et recherche universitaire",
      "180044018": "231 – Vie étudiante",
      "180044034": "150 – Formations supérieures et recherche universitaire",
      "180044067": "150 – Formations supérieures et recherche universitaire",
      "180044083": "150 – Formations supérieures et recherche universitaire",
      "180044091": "150 – Formations supérieures et recherche universitaire",
      "180044117": "150 – Formations supérieures et recherche universitaire",
      "180044125": "150 – Formations supérieures et recherche universitaire",
      "180044174": "150 – Formations supérieures et recherche universitaire",
      "180044224": "150 – Formations supérieures et recherche universitaire",
      "180044232": "150 – Formations supérieures et recherche universitaire",
      "180046013": "175 – Patrimoines",
      "180046021": "175 – Patrimoines",
      "180046039": "334 – Livre et industries culturelles",
      "180046054": "131 – Création",
      "180046187": "131 – Création",
      "180046195": "150 – Formations supérieures et recherche universitaire",
      "180046237": "175 – Patrimoines",
      "180046252": "334 – Livre et industries culturelles",
      "180046260": "175 – Patrimoines",
      "180053027": "134 – Développement des entreprises et régulations",
      "180060030": "159 – Expertise, information géographique et météorologie",
      "180065021": "197 – Régimes de retraite et de sécurité sociale des marins",
      "180067019": "159 – Expertise, information géographique et météorologie",
      "180067027": "135 – Urbanisme, territoires et amélioration de l'habitat",
      "180070039": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180080012": "134 – Développement des entreprises et régulations",
      "180089013": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180089047": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180089369": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "180089476": "150 – Formations supérieures et recherche universitaire",
      "180089500": "310 – Conduite et pilotage de la politique de la justice",
      "180090011": "212 – Soutien de la politique de la défense",
      "180090029": "212 – Soutien de la politique de la défense",
      "180090052": "212 – Soutien de la politique de la défense",
      "180092025": "192 – Recherche et enseignement supérieur en matière économique et industrielle",
      "180092033": "215 – Conduite et pilotage des politiques de l'agriculture",
      "180092058": "150 – Formations supérieures et recherche universitaire",
      "180092082": "224 – Soutien aux politiques du ministère de la culture",
      "180092140": "175 – Patrimoines",
      "180092199": "107 – Administration pénitentiaire",
      "180092207": "361 – Transmission des savoirs et démocratisation de la culture",
      "180092215": "361 – Transmission des savoirs et démocratisation de la culture",
      "180092231": "212 – Soutien de la politique de la défense",
      "180092256": "310 – Conduite et pilotage de la politique de la justice",
      "180092264": "175 – Patrimoines",
      "180092272": "135 – Urbanisme, territoires et amélioration de l'habitat",
      "180092371": "365 – Transmission des savoirs et démocratisation de la culture",
      "180092389": "361 – Transmission des savoirs et démocratisation de la culture",
      "180092397": "364 – Transmission des savoirs et démocratisation de la culture",
      "180092405": "362 – Transmission des savoirs et démocratisation de la culture",
      "180092413": "363 – Transmission des savoirs et démocratisation de la culture",
      "180092447": "175 – Patrimoines",
      "180092538": "174 – Énergie, climat et après-mines",
      "180092553": "203 – Infrastructures et services de transports",
      "180503013": "113 – Paysages, eau et biodiversité",
      "180600041": "231 – Vie étudiante",
      "180600058": "113 – Paysages, eau et biodiversité",
      "181300088": "231 – Vie étudiante",
      "182020107": "231 – Vie étudiante",
      "183100064": "113 – Paysages, eau et biodiversité",
      "183100072": "231 – Vie étudiante",
      "183400084": "231 – Vie étudiante",
      "183801562": "231 – Vie étudiante",
      "184401321": "231 – Vie étudiante",
      "184401339": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "184500213": "231 – Vie étudiante",
      "184800050": "113 – Paysages, eau et biodiversité",
      "185102001": "231 – Vie étudiante",
      "185422102": "231 – Vie étudiante",
      "185703014": "113 – Paysages, eau et biodiversité",
      "185722949": "181 – Prévention des risques",
      "185911500": "231 – Vie étudiante",
      "185911781": "113 – Paysages, eau et biodiversité",
      "186306973": "231 – Vie étudiante",
      "186500047": "113 – Paysages, eau et biodiversité",
      "186706446": "231 – Vie étudiante",
      "186901559": "113 – Paysages, eau et biodiversité",
      "186901567": "231 – Vie étudiante",
      "187300033": "113 – Paysages, eau et biodiversité",
      "187500061": "231 – Vie étudiante",
      "187500079": "150 – Formations supérieures et recherche universitaire",
      "187500095": "113 – Paysages, eau et biodiversité",
      "187512512": "150 – Formations supérieures et recherche universitaire",
      "187512553": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "187512702": "150 – Formations supérieures et recherche universitaire",
      "187512777": "204 – Prévention, sécurité sanitaire et offre de soins",
      "187800081": "231 – Vie étudiante",
      "188002000": "231 – Vie étudiante",
      "188300057": "113 – Paysages, eau et biodiversité",
      "188600050": "231 – Vie étudiante",
      "188719009": "231 – Vie étudiante",
      "189400047": "231 – Vie étudiante",
      "189710023": "231 – Vie étudiante",
      "189710080": "113 – Paysages, eau et biodiversité",
      "189740012": "231 – Vie étudiante",
      "190608364": "367 – Transmission des savoirs et démocratisation de la culture",
      "190615633": "150 – Formations supérieures et recherche universitaire",
      "191010602": "150 – Formations supérieures et recherche universitaire",
      "191302363": "361 – Transmission des savoirs et démocratisation de la culture",
      "191333400": "150 – Formations supérieures et recherche universitaire",
      "191333426": "366 – Transmission des savoirs et démocratisation de la culture",
      "191333467": "150 – Formations supérieures et recherche universitaire",
      "191414085": "150 – Formations supérieures et recherche universitaire",
      "191417203": "150 – Formations supérieures et recherche universitaire",
      "191700327": "150 – Formations supérieures et recherche universitaire",
      "192020907": "148 – Fonction publique",
      "192026649": "150 – Formations supérieures et recherche universitaire",
      "192500825": "150 – Formations supérieures et recherche universitaire",
      "192512150": "150 – Formations supérieures et recherche universitaire",
      "192901197": "150 – Formations supérieures et recherche universitaire",
      "192901254": "144 – Environnement et prospective de la politique de défense",
      "192903466": "150 – Formations supérieures et recherche universitaire",
      "193101433": "142 – Enseignement supérieur et recherche agricoles",
      "193101508": "361 – Transmission des savoirs et démocratisation de la culture",
      "193101524": "150 – Formations supérieures et recherche universitaire",
      "193101532": "142 – Enseignement supérieur et recherche agricoles",
      "193112562": "613 – Soutien aux prestations de l'aviation civile",
      "193113818": "150 – Formations supérieures et recherche universitaire",
      "193113834": "150 – Formations supérieures et recherche universitaire",
      "193113842": "150 – Formations supérieures et recherche universitaire",
      "193113867": "150 – Formations supérieures et recherche universitaire",
      "193301926": "150 – Formations supérieures et recherche universitaire",
      "193301991": "361 – Transmission des savoirs et démocratisation de la culture",
      "193302031": "142 – Enseignement supérieur et recherche agricoles",
      "193317666": "150 – Formations supérieures et recherche universitaire",
      "193322393": "166 – Justice judiciaire",
      "193401122": "150 – Formations supérieures et recherche universitaire",
      "193401320": "361 – Transmission des savoirs et démocratisation de la culture",
      "193410891": "150 – Formations supérieures et recherche universitaire",
      "193415940": "150 – Formations supérieures et recherche universitaire",
      "193500774": "150 – Formations supérieures et recherche universitaire",
      "193500899": "361 – Transmission des savoirs et démocratisation de la culture",
      "193500972": "150 – Formations supérieures et recherche universitaire",
      "193509379": "150 – Formations supérieures et recherche universitaire",
      "193523172": "150 – Formations supérieures et recherche universitaire",
      "193708005": "150 – Formations supérieures et recherche universitaire",
      "193801347": "150 – Formations supérieures et recherche universitaire",
      "193801412": "361 – Transmission des savoirs et démocratisation de la culture",
      "193819125": "150 – Formations supérieures et recherche universitaire",
      "194216149": "361 – Transmission des savoirs et démocratisation de la culture",
      "194401006": "150 – Formations supérieures et recherche universitaire",
      "194401048": "361 – Transmission des savoirs et démocratisation de la culture",
      "194416186": "148 – Fonction publique",
      "194508552": "150 – Formations supérieures et recherche universitaire",
      "194909701": "150 – Formations supérieures et recherche universitaire",
      "195112966": "150 – Formations supérieures et recherche universitaire",
      "195401351": "361 – Transmission des savoirs et démocratisation de la culture",
      "195600853": "219 – Sport",
      "195617188": "150 – Formations supérieures et recherche universitaire",
      "195726476": "148 – Fonction publique",
      "195903372": "361 – Transmission des savoirs et démocratisation de la culture",
      "195903380": "150 – Formations supérieures et recherche universitaire",
      "195903497": "150 – Formations supérieures et recherche universitaire",
      "195936489": "148 – Fonction publique",
      "195944038": "150 – Formations supérieures et recherche universitaire",
      "195958764": "150 – Formations supérieures et recherche universitaire",
      "196012231": "150 – Formations supérieures et recherche universitaire",
      "196244016": "150 – Formations supérieures et recherche universitaire",
      "196312870": "361 – Transmission des savoirs et démocratisation de la culture",
      "196402515": "150 – Formations supérieures et recherche universitaire",
      "196500482": "150 – Formations supérieures et recherche universitaire",
      "196604375": "150 – Formations supérieures et recherche universitaire",
      "196701866": "361 – Transmission des savoirs et démocratisation de la culture",
      "196701890": "142 – Enseignement supérieur et recherche agricoles",
      "196727671": "150 – Formations supérieures et recherche universitaire",
      "196811665": "150 – Formations supérieures et recherche universitaire",
      "196901730": "150 – Formations supérieures et recherche universitaire",
      "196901847": "361 – Transmission des savoirs et démocratisation de la culture",
      "196901870": "150 – Formations supérieures et recherche universitaire",
      "196901896": "176 – Police nationale",
      "196901920": "150 – Formations supérieures et recherche universitaire",
      "196917744": "150 – Formations supérieures et recherche universitaire",
      "196917751": "150 – Formations supérieures et recherche universitaire",
      "196918619": "148 – Fonction publique",
      "196924377": "150 – Formations supérieures et recherche universitaire",
      "196924591": "150 – Formations supérieures et recherche universitaire",
      "197209166": "150 – Formations supérieures et recherche universitaire",
      "197308588": "150 – Formations supérieures et recherche universitaire",
      "197400682": "219 – Sport",
      "197500028": "150 – Formations supérieures et recherche universitaire",
      "197500036": "144 – Environnement et prospective de la politique de défense",
      "197506611": "150 – Formations supérieures et recherche universitaire",
      "197507734": "150 – Formations supérieures et recherche universitaire",
      "197512346": "361 – Transmission des savoirs et démocratisation de la culture",
      "197517170": "150 – Formations supérieures et recherche universitaire",
      "197517188": "150 – Formations supérieures et recherche universitaire",
      "197517196": "150 – Formations supérieures et recherche universitaire",
      "197518756": "361 – Transmission des savoirs et démocratisation de la culture",
      "197518772": "361 – Transmission des savoirs et démocratisation de la culture",
      "197529050": "214 – Soutien de la politique de l'éducation nationale",
      "197534282": "150 – Formations supérieures et recherche universitaire",
      "197534316": "150 – Formations supérieures et recherche universitaire",
      "197534597": "150 – Formations supérieures et recherche universitaire",
      "197534639": "129 – Coordination du travail gouvernemental",
      "197534696": "361 – Transmission des savoirs et démocratisation de la culture",
      "197534704": "361 – Transmission des savoirs et démocratisation de la culture",
      "197534712": "150 – Formations supérieures et recherche universitaire",
      "197534720": "150 – Formations supérieures et recherche universitaire",
      "197534787": "150 – Formations supérieures et recherche universitaire",
      "197534803": "150 – Formations supérieures et recherche universitaire",
      "197534860": "150 – Formations supérieures et recherche universitaire",
      "197534886": "150 – Formations supérieures et recherche universitaire",
      "197534936": "192 – Recherche et enseignement supérieur en matière économique et industrielle",
      "197534951": "361 – Transmission des savoirs et démocratisation de la culture",
      "197534969": "150 – Formations supérieures et recherche universitaire",
      "197535016": "217 – Conduite et pilotage des politiques de l'écologie, du développement et de la mobilité durables",
      "197536675": "361 – Transmission des savoirs et démocratisation de la culture",
      "197537426": "150 – Formations supérieures et recherche universitaire",
      "197546872": "361 – Transmission des savoirs et démocratisation de la culture",
      "197546880": "150 – Formations supérieures et recherche universitaire",
      "197546922": "150 – Formations supérieures et recherche universitaire",
      "197601644": "361 – Transmission des savoirs et démocratisation de la culture",
      "197601651": "150 – Formations supérieures et recherche universitaire",
      "197619042": "150 – Formations supérieures et recherche universitaire",
      "197627623": "150 – Formations supérieures et recherche universitaire",
      "197804123": "361 – Transmission des savoirs et démocratisation de la culture",
      "197819444": "150 – Formations supérieures et recherche universitaire",
      "197820194": "142 – Enseignement supérieur et recherche agricoles",
      "198013443": "150 – Formations supérieures et recherche universitaire",
      "198112013": "150 – Formations supérieures et recherche universitaire",
      "198307662": "150 – Formations supérieures et recherche universitaire",
      "198406852": "150 – Formations supérieures et recherche universitaire",
      "198600736": "150 – Formations supérieures et recherche universitaire",
      "198608564": "150 – Formations supérieures et recherche universitaire",
      "198706699": "150 – Formations supérieures et recherche universitaire",
      "199003567": "150 – Formations supérieures et recherche universitaire",
      "199119751": "150 – Formations supérieures et recherche universitaire",
      "199212044": "150 – Formations supérieures et recherche universitaire",
      "199306036": "150 – Formations supérieures et recherche universitaire",
      "199312380": "150 – Formations supérieures et recherche universitaire",
      "199318270": "150 – Formations supérieures et recherche universitaire",
      "199322306": "361 – Transmission des savoirs et démocratisation de la culture",
      "199406075": "150 – Formations supérieures et recherche universitaire",
      "199406083": "142 – Enseignement supérieur et recherche agricoles",
      "199411117": "150 – Formations supérieures et recherche universitaire",
      "199513763": "150 – Formations supérieures et recherche universitaire",
      "199715855": "150 – Formations supérieures et recherche universitaire",
      "199744780": "150 – Formations supérieures et recherche universitaire",
      "199870015": "150 – Formations supérieures et recherche universitaire",
      "200008357": "113 – Paysages, eau et biodiversité",
      "200008431": "113 – Paysages, eau et biodiversité",
      "200018893": "102 – Accès et retour à l'emploi",
      "200090777": "175 – Patrimoines",
      "302977145": "131 – Création",
      "303017842": "175 – Patrimoines",
      "306664863": "131 – Création",
      "313320244": "192 – Recherche et enseignement supérieur en matière économique et industrielle",
      "322224718": "361 – Transmission des savoirs et démocratisation de la culture",
      "330715368": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "331118760": "361 – Transmission des savoirs et démocratisation de la culture",
      "331596270": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "338840564": "142 – Enseignement supérieur et recherche agricoles",
      "381984921": "181 – Prévention des risques",
      "385290309": "181 – Prévention des risques",
      "390199669": "174 – Énergie, climat et après-mines",
      "391406956": "131 – Création",
      "391718970": "131 – Création",
      "417822632": "131 – Création",
      "420619439": "144 – Environnement et prospective de la politique de défense",
      "421506445": "361 – Transmission des savoirs et démocratisation de la culture",
      "431959956": "134 – Développement des entreprises et régulations",
      "440546018": "190 – Recherche dans les domaines de l'énergie, du développement et de la mobilité durables",
      "441357340": "150 – Formations supérieures et recherche universitaire",
      "451930051": "134 – Développement des entreprises et régulations",
      "478184906": "175 – Patrimoines",
      "488480005": "131 – Création",
      "519587851": "361 – Transmission des savoirs et démocratisation de la culture",
      "521747444": "231 – Vie étudiante",
      "524523396": "129 – Coordination du travail gouvernemental",
      "525046017": "203 – Infrastructures et services de transports",
      "529715922": "185 – Diplomatie culturelle et d'influence",
      "534124334": "150 – Formations supérieures et recherche universitaire",
      "582056149": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "588502310": "131 – Création",
      "662043116": "149 – Compétitivité et durabilité de l'agriculture, de l'agroalimentaire  et de la forêt",
      "692039514": "131 – Création",
      "692041585": "175 – Patrimoines",
      "752195438": "185 – Diplomatie culturelle et d'influence",
      "775664105": "150 – Formations supérieures et recherche universitaire",
      "775665912": "193 – Recherche spatiale",
      "775671464": "334 – Livre et industries culturelles",
      "775685019": "172 – Recherches scientifiques et technologiques pluridisciplinaires",
      "775722879": "144 – Environnement et prospective de la politique de défense",
      "775724644": "103 – Accompagnement des mutations économiques et développement de l'emploi",
      "775729155": "190 – Recherche dans les domaines de l'énergie, du développement et de la mobilité durables",
      "784276180": "131 – Création",
      "784308249": "150 – Formations supérieures et recherche universitaire",
      "784396079": "131 – Création",
      "784523318": "142 – Enseignement supérieur et recherche agricoles",
      "784616989": "150 – Formations supérieures et recherche universitaire",
      "784804593": "131 – Création",
      "788105245": "175 – Patrimoines",
      "824228142": "103 – Accompagnement des mutations économiques et développement de l'emploi",
      "824544514": "113 – Paysages, eau et biodiversité",
      "834553729": "350 – Jeux olympiques et paralympiques 2024",
      "882539786": "334 – Livre et industries culturelles",
      "884439035": "175 – Patrimoines",
      "902883057": "334 – Livre et industries culturelles",
      "910559319": "231 – Vie étudiante",
      "939106274": "131 – Création",
      "991460858": "361 – Transmission des savoirs et démocratisation de la culture",
      "999325392": "150 – Formations supérieures et recherche universitaire"
  },
  "sirens": {
      "130030513": "Université Rennes",
      "853318459": "Pass Culture",
      "932490899": "UNIVERSITE MONTPELLIER PAUL VALERY",
      "938106564": "Université de Besançon Marie et Louis Pasteur",
      "938271392": "UNIVERSITE de Toulouse",
      "941298317": "UNIVERSITE BREST",
      "941636342": "Solidéo Alpes 2030",
      "130000136": "INTEFP - Institut national du travail, de l'emploi et de la formation professionnelle",
      "130000383": "INSTITUT NATIONAL SUPERIEUR DE FORMATION ET DE RECHERCHE POUR L'EDUCATION DES JEUNES HANDICAPES ET LES ENSEIGNEMENTS ADAPTES",
      "130001316": "EPSF - Etablissement public de sécurité ferroviaire",
      "130002140": "ECOLE NATIONALE SUPERIEURE D' INFORMATIQUE POUR L' INDUSTRIE ET L' ENTREPRISE EPA",
      "130002504": "ANR - Agence nationale de la recherche",
      "130002645": "MNS - Musée national du sport",
      "130002702": "INAO - Institut national de l'origine et de la qualité",
      "130002728": "EPPD - Etablissement public du palais de la porte Dorée",
      "130002850": "INSTITUT DES SCIENCES ET INDUSTRIES DU VIVANT ET DE L'ENVIRONNEMENT AGRO PARIS TECH",
      "130002884": "ENTPE - Ecole nationale des travaux publics de l'Etat",
      "130003221": "UNIVERSITE DE NOUVELLE CALEDONIE EPSCP",
      "130003262": "ANTS - Agence nationale des titres sécurisés",
      "130003320": "Académie des technologies",
      "130003643": "COMMUNAUTE D'UNIVERSITES ET D'ETABLISSEMENTS UNIVERSITE FEDERALE DE TOULOUSE MIDI-PYRENEES",
      "130003759": "UNIVERSITE DE NIMES",
      "130003767": "COMMUNAUTE D'UNIVERSITES ET ETABLISSEMENTS UNIVERSITE DE LYON",
      "130003981": "SHOM - Service hydrographique et océanographique de la marine",
      "130004278": "ISAE - Institut supérieur de l'aéronautique et de l'espace",
      "130005457": "UNIVERSITE DE STRASBOURG",
      "130005481": "Pôle emploi",
      "130006356": "INSTITUT POLYTECHNIQUE DE BORDEAUX",
      "130006364": "FranceAgriMer",
      "130006372": "ASP - Agence de services et de paiement",
      "130006513": "Etablissement public du château de Fontainebleau",
      "130006547": "ODEADOM - Office de développement de l'économie agricole d'Outre-mer",
      "130007545": "COMMUNAUTE D'UNIVERSITES ET ETABLISSEMENTS UNIVERSITE ANGERS - LE MANS - établissement expérimental",
      "130007834": "AGENCE REGIONALE DE SANTE GRAND EST EPA",
      "130007842": "AGENCE REGIONALE DE SANTE DU CENTRE-VAL DE LOIRE EPA",
      "130007859": "AGENCE REGIONALE DE SANTE DE GUYANE EPA",
      "130007867": "AGENCE REGIONALE DE SANTE NOUVELLE-AQUITAINE EPA",
      "130007883": "AGENCE REGIONALE DE SANTE DE MARTINIQUE EPA",
      "130007909": "AGENCE REGIONALE DE SANTE NORMANDIE EPA",
      "130007933": "AGENCE REGIONALE DE SANTE BOURGOGNE-FRANCHE-COMTE EPA",
      "130007966": "AGENCE REGIONALE DE SANTE DE BRETAGNE EPA",
      "130007974": "AGENCE REGIONALE DE SANTE HAUTS-DE-FRANCE EPA",
      "130007982": "AGENCE REGIONALE DE SANTE DE PROVENCE ALPES COTE D'AZUR EPA",
      "130007990": "AGENCE REGIONALE DE SANTE DE CORSE EPA",
      "130008006": "AGENCE REGIONALE DE SANTE PAYS DE LOIRE",
      "130008014": "AGENCE REGIONALE DE SANTE ILE DE FRANCE EPA",
      "130008030": "AGENCE DE SANTE DE GUADELOUPE, SAINT-MARTIN ET SAINT-BARTHELEMY EPA",
      "130008048": "AGENCE REGIONALE DE SANTE OCCITANIE EPA",
      "130008071": "AGENCE REGIONALE DE SANTE AUVERGNE-RHONE-ALPES EPA",
      "130008121": "ECOLE NORMALE SUPERIEURE DE LYON",
      "130008535": "ECOLE NATIONALE VETERINAIRE AGROALIMENTAIRE ET DE L'ALIMENTATION NANTES ATLANTIQUE",
      "130008584": "INSTITUT D’ENSEIGNEMENT SUPERIEUR ET DE RECHERCHE EN ALIMENTATION, SANTE ANIMALE, SCIENCES AGRONOMIQUES ET DE L’ENVIRONNEMENT",
      "130008857": "EPCCSL - Etablissement public Cité de la céramique - Sèvres et Limoges",
      "130010440": "IFCE - Institut français du cheval et de l'équitation",
      "130010804": "INSEP - Institut national du sport, de l'expertise et de la performance",
      "130011844": "ASC - Agence du service civique",
      "130012024": "ANSES - Agence nationale de sécurité sanitaire, de l'alimentation, de l'environnement et du travail",
      "130012172": "Musée Picasso",
      "130013097": "ENSM - Ecole nationale supérieure maritime",
      "130014228": "GENES - Groupe des écoles nationales d'économie et statistique",
      "130014442": "AGRASC - Agence de gestion et de recouvrement des avoirs saisis et confisqués",
      "130014541": "ANTAI - Agence nationale de traitement automatisé des infractions",
      "130015001": "Etablissement public du Marais poitevin",
      "130015332": "UNIVERSITE D'AIX MARSEILLE",
      "130015506": "UNIVERSITE DE LORRAINE",
      "130015712": "CNAPS - Conseil national des activités privées de sécurité",
      "130016314": "CENTRE UNIVERSITAIRE DE FORMATION ET DE RECHERCHE DE MAYOTTE",
      "130016371": "ETABLISSEMENT PUBLIC CAMPUS CONDORCET",
      "130016793": "PARC NATIONAL LES CALANQUES",
      "130017791": "VNF - Voies navigables de France",
      "130017890": "MuCEM - Musée des civilisations de l'Europe et de la Méditerranée",
      "130018310": "CEREMA - Centre d'études et d'expertise sur les risques, l'environnement, la mobilité et l'aménagement",
      "130018336": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES CENTRE VAL DE LOIRE",
      "130018351": "UNIVERSITE DE BORDEAUX",
      "130018484": "ECOLE NORMALE SUPERIEURE DE RENNES",
      "130020464": "ANCOLS - Agence nationale de contrôle du logement social",
      "130020597": "UNIVERSITE DE LA GUYANE",
      "130020910": "COMMUNAUTE D'UNIVERSITES ET ETABLISSEMENTS UNIVERSITE BOURGOGNE - FRANCHE-COMTE - etablissement expérimental",
      "130021256": "FUN-MOOC",
      "130021330": "COMMUNAUTE D'UNIVERSITES ET ETABLISSEMENTS NORMANDIE UNIVERSITE - établissement expérimental",
      "130021421": "ECOLE NATIONALE SUPERIEURE DE CHIMIE DE PARIS",
      "130021447": "COMMUNAUTE D'UNIVERSITES ET ETABLISSEMENTS HAUTES ETUDES EN SCIENCES ARTS ET METIERS UNIVERSITE",
      "130021454": "COMMUNAUTE D'UNIVERSITES ET ETABLISSEMENTS UNIVERSITE PARIS EST",
      "130021470": "COMMUNAUTE D UNIVERSITES ET ETABLISSEMENTS UNIVERSITE PARIS LUMIERES",
      "130021918": "CLERMONT AUVERGNE INP",
      "130021959": "LADOM - L'agence de l'Outre-mer pour la mobilité",
      "130022551": "FNAP - Fonds national des aides à la pierre",
      "130023088": "Musée Henner-Moreau",
      "130023385": "SORBONNE UNIVERSITE",
      "130024417": "ANSC - Agence nationale du numérique de la sécurité civile",
      "130024425": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - NORMANDIE EPA",
      "130024433": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - BOURGOGNE FRANCHE-COMTE EPA",
      "130024540": "Ecole de l'air et de l'espace",
      "130024565": "France Compétences",
      "130025281": "ANS - Agence nationale du sport",
      "130025620": "IPP - Institut Polytechnique de Paris",
      "130025661": "UNIVERSITE COTE AZUR - établissement expérimental",
      "130025745": "UNIVERSITE POLYTECHNIQUE HAUTS DE France - établissement expérimental",
      "130025752": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES HAUTS-DE-FRANCE EPSCP",
      "130025919": "OFB - Office français de la biodiversité",
      "130025927": "AGENCE REGIONALE DE SANTE DE MAYOTTE EPA",
      "130025935": "PARC NATIONAL DE FORETS",
      "130025976": "CY CERGY PARIS UNIVERSITE - établissement public expérimental EPSCP",
      "130026024": "UNIVERSITE PARIS-SACLAY - établissement exéprimental",
      "130026032": "ANCT - Agence nationale de la cohésion des territoires",
      "130026057": "AGENCE REGIONALE DE SANTE DE LA REUNION EPA",
      "130026081": "UNIVERSITE GRENOBLE ALPES - établissement expérimental",
      "130026123": "UNIVERSITE GUSTAVE EIFFEL - établissement expérimental",
      "130026149": "UNIVERSITE PARIS SCIENCES ET LETTRES (PSL) - établissement expérimental",
      "130026222": "INSTITUT NATIONAL D'ENSEIGNEMENT SUPERIEUR POUR L'AGRICULTURE, L'ALIMENTATION ET L'ENVIRONNEMENT",
      "130028061": "UNIVERSITE CLERMONT AUVERGNE - établissement expérimental",
      "130029747": "NANTES UNIVERSITE - établissement expérimental",
      "130029754": "UNIVERSITE DE LILLE - établissement expérimental",
      "130029796": "UNIVERSITE DE MONTPELLIER - établissement expérimental",
      "130030133": "GIP Plateforme de l’inclusion",
      "130030190": "GIP Les entreprises s'engagent",
      "130030612": "Université Toulouse Capitole – établissement expérimental",
      "130030620": "Ecole d'économie et de sciences quantitatives EPSCP",
      "130030638": "GIP France enfance protégée",
      "130030851": "ACMOSS - Agence des Communications Mobiles Opérationnelles de Sécurité et de Secours",
      "150000966": "Ecole navale",
      "180000010": "Grande Chancellerie de la Légion d'Honneur",
      "180000028": "Conseil national des communes «Compagnon de la Libération »",
      "180005019": "CELRL - Conservatoire de l'espace littoral et des rivages lacustres",
      "180006025": "IRD - Institut de recherche pour le développement",
      "180006033": "OFPRA - Office français de protection des réfugiés et apatrides",
      "180006082": "AEFE - Agence pour l'enseignement français à l'étranger",
      "180007015": "ONAC-VG - Office national des anciens combattants et victimes de guerre",
      "180007023": "INI - Institution nationale des Invalides",
      "180034027": "OFII - Office français de l'immigration et de l'intégration",
      "180036048": "INSERM - Institut national de la santé et de la recherche médicale",
      "180036105": "OFDT - Observatoire Français des Drogues et des Tendances addictives",
      "180037012": "ANACT - Agence nationale pour l'amélioration des conditions de travail",
      "180037020": "INED - Institut national d'études démographiques",
      "180043010": "Réseau Canopé",
      "180043028": "ONISEP - Office national d'information sur les enseignements et les professions",
      "180043036": "CEREQ - Centre d'Etudes et de Recherches sur les Qualifications",
      "180043069": "FEI – France éducation international",
      "180043093": "BPI - Bibliothèque publique d'information",
      "180043127": "AGENCE DE MUTUALISATION DES UNIVERSITES ET DES ETABLISSEMENTS (AMUE)",
      "180044018": "CENTRE NATIONAL DES OEUVRES UNIVERSITAIRES ET SCOLAIRES - CNOUS",
      "180044034": "ACADEMIE DES SCIENCES D'OUTRE MER",
      "180044067": "BIBLIOTHEQUE NATIONALE ET UNIVERSITAIRE DE STRASBOURG",
      "180044083": "ECOLE FRANCAISE D'ATHENES",
      "180044091": "ECOLE FRANCAISE DE ROME",
      "180044117": "ECOLE FRANCAISE D'EXTREME ORIENT",
      "180044125": "CASA DE VELASQUEZ A MADRID",
      "180044174": "MUSEUM NATIONAL D HISTOIRE NATURELLE",
      "180044224": "AGENCE BIBLIOGRAPHIQUE DE L'ENSEIGNEMENT SUPERIEUR",
      "180044232": "CENTRE TECHNIQUE DU LIVRE DE L'ENSEIGNEMENT SUPERIEUR (CTLES)",
      "180046013": "CMN - Centre des monuments nationaux",
      "180046021": "CNAC-GP - Centre national d'art et de culture - Georges Pompidou",
      "180046039": "CNC - Centre national du cinéma et de l'image animée",
      "180046054": "CNAP - Centre national des arts plastiques",
      "180046187": "AFR - Académie de France à Rome EPA",
      "180046195": "INSTITUT FRANCAIS D'ARCHEOLOGIE ORIENTALE EPSCP",
      "180046237": "Musée du Louvre",
      "180046252": "BnF - Bibliothèque nationale de France",
      "180046260": "EPV - Etablissement public du musée et du domaine national de Versailles",
      "180053027": "ANFr - Agence nationale des fréquences",
      "180060030": "Météo-France",
      "180065021": "ENIM - Etablissement national des invalides de la marine",
      "180067019": "IGN - Institut national de l'information géographique et forestière",
      "180067027": "ANAH - Agence nationale de l'habitat",
      "180070039": "INRAE - Institut national pour la recherche en agriculture, alimentation et environnement",
      "180080012": "INPI - Institut national de la propriété industrielle",
      "180089013": "CNRS - Centre national de la recherche scientifique",
      "180089047": "INRIA - Institut national de recherche en informatique et en automatique",
      "180089369": "IPEV - Institut polaire français Paul-Emile Victor",
      "180089476": "RESEAU NATIONAL DE TELECOMMUNICATION POUR LA TECHNOLOGIE D'ENSEIGNEMENT ET LA RECHERCHE",
      "180089500": "IERDJ - Institut des études et de la recherche sur le droit et la justice",
      "180090011": "Musée de l'armée",
      "180090029": "Musée national de la marine",
      "180090052": "Musée de l'air et de l'espace",
      "180092025": "Institut Mines Telecom",
      "180092033": "INFOMA - Institut national de formation des personnels du ministère de l'agriculture",
      "180092058": "EPAURIF - Etablissement public d'aménagement universitaire de la région Ile-de-France",
      "180092082": "OPPIC - Opérateur du patrimoine et des projets immobiliers de la Culture",
      "180092140": "EPMQB - Etablissement public du musée du quai Branly",
      "180092199": "ENAP - Ecole nationale de l'administration pénitentiaire",
      "180092207": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE PARIS VAL DE SEINE EPA",
      "180092215": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE PARIS MALAQUAIS EPA",
      "180092231": "ECPAD - Etablissement de communication et de production audiovisuelle de la défense",
      "180092256": "APIJ - Agence publique pour l'immobilier de la justice",
      "180092264": "INRAP - Institut national de recherches archéologiques préventives",
      "180092272": "CGLLS - Caisse de garantie du logement locatif social",
      "180092371": "ECOLE NATIONALE SUPERIEURE D'ART DE NANCY EPA",
      "180092389": "ECOLE NATIONALE SUPERIEURE D'ART DE BOURGES",
      "180092397": "ECOLE NATIONALE SUPERIEURE D'ART DE LIMOGES-AUBUSSON EPA",
      "180092405": "ECOLE NATIONALE SUPERIEURE D'ART DE CERGY EPA",
      "180092413": "ECOLE NATIONALE SUPERIEURE D'ART DE DIJON EPA",
      "180092447": "Musée d'Orsay et musée de l'Orangerie",
      "180092538": "ANGDM - Agence nationale pour la garantie des droits des mineurs",
      "180092553": "AFITF - Agence de financement des infrastructures de transport de France",
      "180503013": "PARC NATIONAL DES ECRINS",
      "180600041": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - NICE EPA",
      "180600058": "PARC NATIONAL DU MERCANTOUR",
      "181300088": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - AIX - MARSEILLE EPA",
      "182020107": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - CORTE",
      "183100064": "AGENCE DE L'EAU ADOUR GARONNE",
      "183100072": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - TOULOUSE EPA",
      "183400084": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - MONTPELLIER EPA",
      "183801562": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - GRENOBLE EPA",
      "184401321": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - NANTES EPA",
      "184401339": "CNPF - Centre national de la propriété forestière",
      "184500213": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - ORLEANS - TOURS EPA",
      "184800050": "PARC NATIONAL DES CEVENNES",
      "185102001": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - REIMS EPA",
      "185422102": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - NANCY - METZ EPA",
      "185703014": "AGENCE DE L'EAU RHIN MEUSE",
      "185722949": "GEODERIS",
      "185911500": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - LILLE EPA",
      "185911781": "AGENCE DE L EAU ARTOIS PICARDIE",
      "186306973": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - CLERMONT FERRAND EPA",
      "186500047": "PARC NATIONAL DES PYRENEES EPA",
      "186706446": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - STRASBOURG EPA",
      "186901559": "AGENCE DE L'EAU RHONE MEDITERRANEE ET CORSE",
      "186901567": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - LYON EPA",
      "187300033": "PARC NATIONAL LA VANOISE EPA",
      "187500061": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - PARIS EPA",
      "187500079": "CHANCELLERIE DES UNIVERSITES DE PARIS",
      "187500095": "AGENCE DE L'EAU SEINE-NORMANDIE",
      "187512512": "AGENCE ERASMUS+FRANCE/EDUCATION FORMATION",
      "187512553": "GIP - BIO - Agence française pour le développement et la promotion de l'agriculture biologique",
      "187512702": "BIBLIOTHEQUE UNIVERSITAIRE DES LANGUES ET CIVILISATIONS",
      "187512777": "INCa - Institut National du Cancer",
      "187800081": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - VERSAILLES",
      "188002000": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - AMIENS EPA",
      "188300057": "PARC NATIONAL DE PORT CROS",
      "188600050": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - POITIERS EPA",
      "188719009": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - LIMOGES EPA",
      "189100142": "Génopôle",
      "189400047": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - CRETEIL EPA",
      "189710023": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - ANTILLES GUYANE EPA",
      "189710080": "PARC NATIONAL DE LA GUADELOUPE",
      "189740012": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - REUNION EPA",
      "190608364": "EPA VILLA ARSON",
      "190615633": "OBSERVATOIRE DE LA COTE D'AZUR",
      "191010602": "UNIVERSITE DE TECHNOLOGIE DE TROYES",
      "191302363": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE MARSEILLE EPA",
      "191333400": "ECOLE CENTRALE DE MARSEILLE",
      "191333426": "ECOLE NATIONALE SUPERIEURE DE LA PHOTOGRAPHIE EPA",
      "191333467": "INSTITUT D'ETUDES POLITIQUES - AIX EN PROVENCE",
      "191414085": "UNIVERSITE DE CAEN",
      "191417203": "ECOLE NATIONALE SUPERIEURE INGENIEURS DE CAEN",
      "191700327": "UNIVERSITE DE LA ROCHELLE",
      "192020907": "INSTITUT REGIONAL D'ADMINISTRATION - BASTIA EPA",
      "192026649": "UNIVERSITE DE CORSE P PAOLI",
      "192500825": "ECOLE NATIONALE SUPERIEURE DE MECANIQUE ET DES MICROTECHNIQUES",
      "192512150": "UNIVERSITE DE BESANCON",
      "192901197": "ECOLE NATIONALE D'INGENIEURS DE BREST EPA",
      "192901254": "ENSTA Bretagne - Ecole nationale supérieure de techniques avancées Bretagne",
      "192903466": "UNIVERSITE BREST BRETAGNE OCCIDENTALE",
      "193101433": "ECOLE NATIONALE SUPERIEURE DE FORMATION DE L'ENSEIGNEMENT AGRICOLE EPA",
      "193101508": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE TOULOUSE",
      "193101524": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES TOULOUSE",
      "193101532": "ECOLE NATIONALE VETERINAIRE DE TOULOUSE",
      "193112562": "ENAC - Ecole nationale de l'aviation civile",
      "193113818": "INSTITUT NATIONAL POLYTECHNIQUE DE TOULOUSE",
      "193113834": "UNIVERSITE TOULOUSE II",
      "193113842": "UNIVERSITE PAUL SABATIER TOULOUSE III",
      "193113867": "INSTITUT D'ETUDES POLITIQUES - TOULOUSE EPA",
      "193301926": "INSTITUT D'ETUDES POLITIQUES - BORDEAUX",
      "193301991": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE ET DE PAYSAGE DE BORDEAUX EPA",
      "193302031": "ECOLE NATIONALE SUPERIEURE DES SCIENCES AGRONOMIQUES DE BORDEAUX AQUITAINE BORDEAUX SCIENCES AGRO",
      "193317666": "UNIVERSITE BORDEAUX MONTAIGNE BORDEAUX III",
      "193322393": "ENM - Ecole nationale de la magistrature",
      "193401122": "ECOLE NATIONALE SUPERIEURE DE CHIMIE DE MONTPELLIER",
      "193401320": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE MONTPELLIER",
      "193410891": "UNIVERSITE MONTPELLIER III PAUL VALERY",
      "193415940": "CENTRE INFORMATIQUE NATIONAL DE L'ENSEIGNEMENT SUPERIEUR",
      "193500774": "ECOLE NATIONALE SUPERIEURE DE CHIMIE DE RENNES EPA",
      "193500899": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE BRETAGNE EPA",
      "193500972": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES RENNES",
      "193509379": "UNIVERSITE RENNES II HAUTE BRETAGNE",
      "193523172": "INSTITUT D'ETUDES POLITIQUES - RENNES EPA",
      "193708005": "UNIVERSITE DE TOURS FRANCOIS RABELAIS",
      "193801347": "INSTITUT D'ETUDES POLITIQUES - GRENOBLE EPA",
      "193801412": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE GRENOBLE",
      "193819125": "INSTITUT POLYTECHNIQUE DE GRENOBLE",
      "194216149": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE ST-ETIENNE EPA",
      "194401006": "ECOLE CENTRALE DE NANTES",
      "194401048": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE NANTES",
      "194416186": "INSTITUT REGIONAL D'ADMINISTRATION - NANTES",
      "194508552": "UNIVERSITE D'ORLEANS",
      "194909701": "UNIVERSITE D'ANGERS",
      "195112966": "UNIVERSITE DE REIMS CHAMPAGNE-ARDENNE",
      "195401351": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE NANCY EPA",
      "195600853": "ECOLE NATIONALE DE VOILE ET DES SPORTS NAUTIQUES EPA",
      "195617188": "UNIVERSITE DE BRETAGNE SUD",
      "195726476": "INSTITUT REGIONAL D'ADMINISTRATION - METZ EPA",
      "195903372": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE ET DE PAYSAGE DE LILLE",
      "195903380": "ECOLE NATIONALE SUPERIEURE DES ARTS ET INDUSTRIES TEXTILES",
      "195903497": "CENTRALE LILLE INSTITUT",
      "195936489": "INSTITUT REGIONAL D'ADMINISTRATION - LILLE EPA",
      "195944038": "UNIVERSITE DU LITTORAL",
      "195958764": "INSTITUT D ETUDES POLITIQUES - LILLE EPA",
      "196012231": "UNIVERSITE DE TECHNOLOGIE DE COMPIEGNE",
      "196244016": "UNIVERSITE D ARTOIS",
      "196312870": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE CLERMONT-FERRAND EPA",
      "196402515": "UNIVERSITE DE PAU ET DU PAYS DE L'ADOUR",
      "196500482": "ECOLE NATIONALE D INGENIEURS TARBES EPA",
      "196604375": "UNIVERSITE DE PERPIGNAN",
      "196701866": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE STRASBOURG EPA",
      "196701890": "ECOLE NATIONALE DU GENIE DE L'EAU ET DE L'ENVIRONNEMENT DE STRASBOURG EPA",
      "196727671": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES STRASBOURG EPSCP",
      "196811665": "UNIVERSITE DE MULHOUSE",
      "196901730": "INSTITUT D'ETUDES POLITIQUES DE LYON",
      "196901847": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE LYON EPA",
      "196901870": "ECOLE CENTRALE DE LYON",
      "196901896": "ENSPolice - Ecole nationale supérieure de la police",
      "196901920": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES LYON",
      "196917744": "UNIVERSITE CLAUDE BERNARD LYON 1",
      "196917751": "UNIVERSITE LYON II LUMIERE",
      "196918619": "INSTITUT REGIONAL D'ADMINISTRATION - LYON EPA",
      "196924377": "UNIVERSITE LYON 3 JEAN MOULIN",
      "196924591": "ECOLE NATIONALE SUPERIEURE DES SCIENCES DE L'INFORMATION ET DES BIBLIOTHEQUES",
      "197209166": "UNIVERSITE LE MANS",
      "197308588": "UNIVERSITE DE CHAMBERY",
      "197400682": "ECOLE NATIONALE DES SPORTS DE MONTAGNE",
      "197500028": "INSTITUT D'ADMINISTRATION DES ENTREPRISES DE PARIS",
      "197500036": "ENSTA Paris - Ecole nationale supérieure de techniques avancées",
      "197506611": "ECOLE NATIONALE SUPERIEURE LOUIS LUMIERE",
      "197507734": "ECOLE NATIONALE SUPERIEURE DES ARTS ET TECHNIQUES DU THEATRE",
      "197512346": "INP - Institut national du patrimoine",
      "197517170": "UNIVERSITE PARIS 1 PANTHEON-SORBONNE",
      "197517188": "UNIVERSITE PARIS II PANTHEON ASSAS - établissement expérimental",
      "197517196": "UNIVERSITE PARIS III SORBONNE NOUVELLE",
      "197518756": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE PARIS LA VILLETTE EPA",
      "197518772": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE PARIS BELLEVILLE EPA",
      "197529050": "CNED - Centre national d'enseignement à distance",
      "197534282": "INSTITUT DE PHYSIQUE DU GLOBE DE PARIS",
      "197534316": "INSTITUT D'ETUDES POLITIQUES DE PARIS-SCIENCES PO",
      "197534597": "ECOLE NORMALE SUPERIEURE",
      "197534639": "INSP - Institut national du service public",
      "197534696": "CNSAD - Conservatoire national supérieur d'art dramatique",
      "197534704": "ENSAD - Ecole nationale supérieure des arts décoratifs",
      "197534712": "CONSERVATOIRE NATIONAL DES ARTS ET METIERS",
      "197534720": "ECOLE NATIONALE SUPERIEURE DES ARTS ET METIERS (ENSAM)",
      "197534787": "ECOLE NATIONALE DES CHARTES",
      "197534803": "COLLEGE DE France",
      "197534860": "ECOLE PRATIQUE DES HAUTES ETUDES",
      "197534886": "INSTITUT NATIONAL DES LANGUES ET CIVILISATIONS ORIENTALES",
      "197534936": "Mines Paris-Tech",
      "197534951": "CNSMD Paris - Conservatoire national supérieur de musique et de danse de Paris",
      "197534969": "OBSERVATOIRE DE PARIS",
      "197535016": "ENPC - Ecole nationale des Ponts et Chaussées",
      "197536675": "ENSBA - Ecole nationale supérieure des beaux-arts",
      "197537426": "ECOLE DES HAUTES ETUDES EN SCIENCES SOCIALES",
      "197546872": "Ecole du Louvre",
      "197546880": "INSTITUT NATIONAL D' HISTOIRE DE L'ART",
      "197546922": "UNIVERSITE PARIS DAUPHINE",
      "197601644": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE NORMANDIE EPA",
      "197601651": "INSTITUT NATIONAL DES SCIENCES APPLIQUEES ROUEN",
      "197619042": "UNIVERSITE DE ROUEN-NORMANDIE",
      "197627623": "UNIVERSITE DU HAVRE",
      "197804123": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE VERSAILLES EPA",
      "197819444": "UNIVERSITE VERSAILLES ST QUENTIN YVELINE",
      "197820194": "ECOLE NATIONALE SUPERIEURE DU PAYSAGE",
      "198013443": "UNIVERSITE AMIENS PICARDIE JULES VERNE",
      "198112013": "INSTITUT NATIONAL UNIVERSITAIRE JEAN-FRANCOIS CHAMPOLLION",
      "198307662": "UNIVERSITE DE TOULON",
      "198406852": "UNIVERSITE AVIGNON ET DES PAYS DE VAUCLUSE EPSCP",
      "198600736": "ECOLE NATIONALE SUPERIEURE MECANIQUE ET AEROTECHNIQUE",
      "198608564": "UNIVERSITE DE POITIERS",
      "198706699": "UNIVERSITE DE LIMOGES",
      "199003567": "UNIVERSITE TECHNOLOGIE BELFORT MONTBELIARD",
      "199119751": "UNIVERSITE D'EVRY VAL D'ESSONNE",
      "199212044": "UNIVERSITE PARIS X",
      "199306036": "INSTITUT SUPERIEUR DE MECANIQUE DE PARIS",
      "199312380": "UNIVERSITE PARIS XIII PARIS-NORD VILLETANEUSE",
      "199318270": "UNIVERSITE DE PARIS VIII.PARIS VINCENNES",
      "199322306": "ECOLE NATIONALE SUPERIEURE D ARCHITECTURE DE MARNE-VALLEE EPA",
      "199406075": "ECOLE NORMALE SUPERIEURE PARIS-SACLAY",
      "199406083": "ECOLE NATIONALE VETERINAIRE D'ALFORT EPA",
      "199411117": "UNIVERSITE PARIS EST CRETEIL VAL DE MARNE",
      "199513763": "ECOLE NATIONALE SUPERIEURE DE L'ELECTRONIQUE ET DE SES APPLICATIONS",
      "199715855": "UNIVERSITE DES ANTILLES",
      "199744780": "UNIVERSITE DE LA REUNION",
      "199870015": "UNIVERSITE DE LA POLYNESIE FRANCAISE EPSCP",
      "200008357": "PARC NATIONAL DE LA REUNION",
      "200008431": "PARC AMAZONIEN DE GUYANE EPA",
      "200018893": "EPIDE - Etablissement pour l'insertion dans l'emploi",
      "200090777": "EPRNDP - Etablissement public chargé de la conservation et de la restauration de la cathédrale Notre-Dame de Paris",
      "302977145": "Comédie Française",
      "303017842": "Musée Guimet",
      "306664863": "Ensemble intercontemporain",
      "313320244": "LNE - Laboratoire national de métrologie et d'essais",
      "322224718": "CNAC - Centre national des arts du cirque",
      "330715368": "IFREMER - Institut français de recherche pour l'exploitation de la mer",
      "331118760": "ENSCI - Ecole nationale supérieure de création industrielle",
      "331596270": "CIRAD - Centre de coopération internationale en recherche agronomique pour le développement",
      "338840564": "ASSOCIATION DE COORDINATION TECHNIQUE POUR L'INDUSTRIE AGRO ALIMENTAIRE",
      "381984921": "INERIS - Institut national de l'environnement industriel et des risques",
      "385290309": "ADEME - Agence de l'environnement et de la maîtrise de l'énergie",
      "390199669": "ANDRA - Agence nationale pour la gestion des déchets radioactifs",
      "391406956": "EPPGHV - Etablissement public du parc et de la grande halle de la Villette",
      "391718970": "EPCMPP - Etablissement public de la Cité de la musique - Philharmonie de Paris",
      "417822632": "CND - Centre national de la danse",
      "420619439": "X - Ecole polytechnique",
      "421506445": "ENSMIS - Ecole nationale supérieure des métiers de l'image et du son",
      "431959956": "Atout-France",
      "440546018": "IRSN - Institut de radioprotection et de sûreté nucléaire",
      "441357340": "UNIVERSITE DE SAINT ETIENNE",
      "451930051": "Business France",
      "478184906": "CAPA - Cité de l'architecture et du patrimoine",
      "488480005": "Opéra comique",
      "519587851": "Universcience",
      "521747444": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - BORDEAUX EPA",
      "524523396": "IHEDN - Institut des hautes études de Défense nationale",
      "525046017": "SGP - Société des grands projets",
      "529715922": "Institut Français",
      "534124334": "UNIVERSITE BOURGOGNE FRANCHE COMTE",
      "582056149": "BRGM - Bureau de recherches géologiques et minières",
      "588502310": "TNS - Théâtre national de Strasbourg",
      "662043116": "ONF - Office national des forêts",
      "692039514": "Chaillot – Théâtre national de la Danse",
      "692041585": "Rmn-GP - Réunion des musées nationaux - Grand Palais",
      "752195438": "CAMPUS France",
      "775664105": "FONDATION MAISON DES SCIENCES DE L HOMME",
      "775665912": "CNES - Centre national d'études spatiales",
      "775671464": "Cinémathèque française",
      "775685019": "CEA - Commissariat à l'énergie atomique et aux énergies alternatives",
      "775722879": "ONERA - Office national d'études et de recherches aérospatiales",
      "775724644": "Centre info - Centre pour le développement de l'information sur la formation permanente",
      "775729155": "IFPEN - IFP Energies Nouvelles",
      "784276180": "TNO - Théâtre national de l'Odéon",
      "784308249": "FONDATION NATIONALE SCIENCES POLITIQUES",
      "784396079": "Opéra national de Paris",
      "784523318": "ASSOCIATION DE COORDINATION TECHNIQUE AGRICOLE",
      "784616989": "Institut d'optique théorique appliquée",
      "784804593": "TNC - Théâtre national de la Colline",
      "788105245": "Musée des arts décoratifs",
      "824228142": "AFPA - Agence nationale pour la formation professionnelle des adultes",
      "824544514": "AGENCE DE L'EAU LOIRE-BRETAGNE",
      "834553729": "SOLIDEO - Société de livraison des équipements olympiques et paralympiques",
      "882539786": "CNM - Centre national de la musique",
      "884439035": "EPMSM - Etablissement public du Mont-Saint-Michel",
      "902883057": "CNL - Centre national du livre",
      "910559319": "CENTRE REGIONAL OEUVRES UNIV SCOLAIRES - RENNES EPA",
      "939106274": "Etablissement public du Mobilier National",
      "991460858": "CNSMD Lyon - Conservatoire national supérieur de musique et de danse de Lyon",
      "999325392": "CENTRALE SUPELEC"
  }
}''',

    "GEO_FRANCE": r'''{
  "description": "Table départements->NUTS3/Région (fournie par l'utilisateur, NUTS2=NUTS3 pour RUP/COM) + territoires",
  "nuts2_par_departement": {
    "source": "Eurostat NUTS 2021 (NUTS_AT_2021.csv, gisco-services.ec.europa.eu) : NUTS3 (départements) -> NUTS2 (anciennes régions), croisé et validé contre la table départements et nuts2_vers_region ; les 7 COM hors NUTS (975/977/978/984/986/987/988) suivent la convention du référentiel NUTS2=NUTS3",
    "map": {
      "1": "Rhône-Alpes",
      "2": "Picardie",
      "3": "Auvergne",
      "4": "Provence-Alpes-Côte d'Azur",
      "5": "Provence-Alpes-Côte d'Azur",
      "6": "Provence-Alpes-Côte d'Azur",
      "7": "Rhône-Alpes",
      "8": "Champagne-Ardenne",
      "9": "Midi-Pyrénées",
      "10": "Champagne-Ardenne",
      "11": "Languedoc-Roussillon",
      "12": "Midi-Pyrénées",
      "13": "Provence-Alpes-Côte d'Azur",
      "14": "Basse-Normandie",
      "15": "Auvergne",
      "16": "Poitou-Charentes",
      "17": "Poitou-Charentes",
      "18": "Centre-Val de Loire",
      "19": "Limousin",
      "21": "Bourgogne",
      "22": "Bretagne",
      "23": "Limousin",
      "24": "Aquitaine",
      "25": "Franche-Comté",
      "26": "Rhône-Alpes",
      "27": "Haute-Normandie",
      "28": "Centre-Val de Loire",
      "29": "Bretagne",
      "2A": "Corse",
      "2B": "Corse",
      "30": "Languedoc-Roussillon",
      "31": "Midi-Pyrénées",
      "32": "Midi-Pyrénées",
      "33": "Aquitaine",
      "34": "Languedoc-Roussillon",
      "35": "Bretagne",
      "36": "Centre-Val de Loire",
      "37": "Centre-Val de Loire",
      "38": "Rhône-Alpes",
      "39": "Franche-Comté",
      "40": "Aquitaine",
      "41": "Centre-Val de Loire",
      "42": "Rhône-Alpes",
      "43": "Auvergne",
      "44": "Pays de la Loire",
      "45": "Centre-Val de Loire",
      "46": "Midi-Pyrénées",
      "47": "Aquitaine",
      "48": "Languedoc-Roussillon",
      "49": "Pays de la Loire",
      "50": "Basse-Normandie",
      "51": "Champagne-Ardenne",
      "52": "Champagne-Ardenne",
      "53": "Pays de la Loire",
      "54": "Lorraine",
      "55": "Lorraine",
      "56": "Bretagne",
      "57": "Lorraine",
      "58": "Bourgogne",
      "59": "Nord-Pas-de-Calais",
      "60": "Picardie",
      "61": "Basse-Normandie",
      "62": "Nord-Pas-de-Calais",
      "63": "Auvergne",
      "64": "Aquitaine",
      "65": "Midi-Pyrénées",
      "66": "Languedoc-Roussillon",
      "67": "Alsace",
      "68": "Alsace",
      "69": "Rhône-Alpes",
      "70": "Franche-Comté",
      "71": "Bourgogne",
      "72": "Pays de la Loire",
      "73": "Rhône-Alpes",
      "74": "Rhône-Alpes",
      "75": "Île-de-France",
      "76": "Haute-Normandie",
      "77": "Île-de-France",
      "78": "Île-de-France",
      "79": "Poitou-Charentes",
      "80": "Picardie",
      "81": "Midi-Pyrénées",
      "82": "Midi-Pyrénées",
      "83": "Provence-Alpes-Côte d'Azur",
      "84": "Provence-Alpes-Côte d'Azur",
      "85": "Pays de la Loire",
      "86": "Poitou-Charentes",
      "87": "Limousin",
      "88": "Lorraine",
      "89": "Bourgogne",
      "90": "Franche-Comté",
      "91": "Île-de-France",
      "92": "Île-de-France",
      "93": "Île-de-France",
      "94": "Île-de-France",
      "95": "Île-de-France",
      "971": "Guadeloupe",
      "972": "Martinique",
      "973": "Guyane",
      "974": "La Réunion",
      "975": "Saint-Pierre-et-Miquelon",
      "976": "Mayotte",
      "977": "Saint-Barthélemy",
      "978": "Saint-Martin",
      "984": "Terres australes et antarctiques françaises",
      "986": "Wallis-et-Futuna",
      "987": "Polynésie française",
      "988": "Nouvelle-Calédonie"
    }
  },
  "departements": [
    {
      "nuts3": "Ain",
      "code": "1",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Allier",
      "code": "3",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Ardèche",
      "code": "7",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Cantal",
      "code": "15",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Drôme",
      "code": "26",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Isère",
      "code": "38",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Loire",
      "code": "42",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Haute-Loire",
      "code": "43",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Puy-de-Dôme",
      "code": "63",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Rhône",
      "code": "69",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Savoie",
      "code": "73",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Haute-Savoie",
      "code": "74",
      "region": "Auvergne-Rhône-Alpes"
    },
    {
      "nuts3": "Côte-d'Or",
      "code": "21",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Doubs",
      "code": "25",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Jura",
      "code": "39",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Nièvre",
      "code": "58",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Haute-Saône",
      "code": "70",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Saône-et-Loire",
      "code": "71",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Yonne",
      "code": "89",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Territoire-de-Belfort",
      "code": "90",
      "region": "Bourgogne-Franche-Comté"
    },
    {
      "nuts3": "Côtes d'Armor",
      "code": "22",
      "region": "Bretagne"
    },
    {
      "nuts3": "Finistère",
      "code": "29",
      "region": "Bretagne"
    },
    {
      "nuts3": "Ille-et-Vilaine",
      "code": "35",
      "region": "Bretagne"
    },
    {
      "nuts3": "Morbihan",
      "code": "56",
      "region": "Bretagne"
    },
    {
      "nuts3": "Cher",
      "code": "18",
      "region": "Centre-Val de Loire"
    },
    {
      "nuts3": "Eure-et-Loir",
      "code": "28",
      "region": "Centre-Val de Loire"
    },
    {
      "nuts3": "Indre",
      "code": "36",
      "region": "Centre-Val de Loire"
    },
    {
      "nuts3": "Indre-et-Loire",
      "code": "37",
      "region": "Centre-Val de Loire"
    },
    {
      "nuts3": "Loir-et-Cher",
      "code": "41",
      "region": "Centre-Val de Loire"
    },
    {
      "nuts3": "Loiret",
      "code": "45",
      "region": "Centre-Val de Loire"
    },
    {
      "nuts3": "Corse-du-Sud",
      "code": "2A",
      "region": "Corse"
    },
    {
      "nuts3": "Haute-Corse",
      "code": "2B",
      "region": "Corse"
    },
    {
      "nuts3": "Ardennes",
      "code": "8",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Aube",
      "code": "10",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Marne",
      "code": "51",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Haute-Marne",
      "code": "52",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Meurthe-et-Moselle",
      "code": "54",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Meuse",
      "code": "55",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Moselle",
      "code": "57",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Bas-Rhin",
      "code": "67",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Haut-Rhin",
      "code": "68",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Vosges",
      "code": "88",
      "region": "Grand-Est"
    },
    {
      "nuts3": "Aisne",
      "code": "2",
      "region": "Hauts-de-France"
    },
    {
      "nuts3": "Nord",
      "code": "59",
      "region": "Hauts-de-France"
    },
    {
      "nuts3": "Oise",
      "code": "60",
      "region": "Hauts-de-France"
    },
    {
      "nuts3": "Pas-de-Calais",
      "code": "62",
      "region": "Hauts-de-France"
    },
    {
      "nuts3": "Somme",
      "code": "80",
      "region": "Hauts-de-France"
    },
    {
      "nuts3": "Paris",
      "code": "75",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Seine-et-Marne",
      "code": "77",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Yvelines",
      "code": "78",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Essonne",
      "code": "91",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Hauts-de-Seine",
      "code": "92",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Seine-Saint-Denis",
      "code": "93",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Val-de-Marne",
      "code": "94",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Val-D'Oise",
      "code": "95",
      "region": "Île-de-France"
    },
    {
      "nuts3": "Calvados",
      "code": "14",
      "region": "Normandie"
    },
    {
      "nuts3": "Eure",
      "code": "27",
      "region": "Normandie"
    },
    {
      "nuts3": "Manche",
      "code": "50",
      "region": "Normandie"
    },
    {
      "nuts3": "Orne",
      "code": "61",
      "region": "Normandie"
    },
    {
      "nuts3": "Seine-Maritime",
      "code": "76",
      "region": "Normandie"
    },
    {
      "nuts3": "Charente",
      "code": "16",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Charente-Maritime",
      "code": "17",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Corrèze",
      "code": "19",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Creuse",
      "code": "23",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Dordogne",
      "code": "24",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Gironde",
      "code": "33",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Landes",
      "code": "40",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Lot-et-Garonne",
      "code": "47",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Pyrénées-Atlantiques",
      "code": "64",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Deux-Sèvres",
      "code": "79",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Vienne",
      "code": "86",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Haute-Vienne",
      "code": "87",
      "region": "Nouvelle-Aquitaine"
    },
    {
      "nuts3": "Ariège",
      "code": "9",
      "region": "Occitanie"
    },
    {
      "nuts3": "Aude",
      "code": "11",
      "region": "Occitanie"
    },
    {
      "nuts3": "Aveyron",
      "code": "12",
      "region": "Occitanie"
    },
    {
      "nuts3": "Gard",
      "code": "30",
      "region": "Occitanie"
    },
    {
      "nuts3": "Haute-Garonne",
      "code": "31",
      "region": "Occitanie"
    },
    {
      "nuts3": "Gers",
      "code": "32",
      "region": "Occitanie"
    },
    {
      "nuts3": "Hérault",
      "code": "34",
      "region": "Occitanie"
    },
    {
      "nuts3": "Lot",
      "code": "46",
      "region": "Occitanie"
    },
    {
      "nuts3": "Lozère",
      "code": "48",
      "region": "Occitanie"
    },
    {
      "nuts3": "Hautes-Pyrénées",
      "code": "65",
      "region": "Occitanie"
    },
    {
      "nuts3": "Pyrénées-Orientales",
      "code": "66",
      "region": "Occitanie"
    },
    {
      "nuts3": "Tarn",
      "code": "81",
      "region": "Occitanie"
    },
    {
      "nuts3": "Tarn-et-Garonne",
      "code": "82",
      "region": "Occitanie"
    },
    {
      "nuts3": "Loire-Atlantique",
      "code": "44",
      "region": "Pays de la Loire"
    },
    {
      "nuts3": "Maine-et-Loire",
      "code": "49",
      "region": "Pays de la Loire"
    },
    {
      "nuts3": "Mayenne",
      "code": "53",
      "region": "Pays de la Loire"
    },
    {
      "nuts3": "Sarthe",
      "code": "72",
      "region": "Pays de la Loire"
    },
    {
      "nuts3": "Vendée",
      "code": "85",
      "region": "Pays de la Loire"
    },
    {
      "nuts3": "Alpes-de-Haute-Provence",
      "code": "4",
      "region": "Provence-Alpes-Côte d'Azur"
    },
    {
      "nuts3": "Hautes-Alpes",
      "code": "5",
      "region": "Provence-Alpes-Côte d'Azur"
    },
    {
      "nuts3": "Alpes-Maritimes",
      "code": "6",
      "region": "Provence-Alpes-Côte d'Azur"
    },
    {
      "nuts3": "Bouches-du-Rhône",
      "code": "13",
      "region": "Provence-Alpes-Côte d'Azur"
    },
    {
      "nuts3": "Var",
      "code": "83",
      "region": "Provence-Alpes-Côte d'Azur"
    },
    {
      "nuts3": "Vaucluse",
      "code": "84",
      "region": "Provence-Alpes-Côte d'Azur"
    },
    {
      "nuts3": "Guadeloupe",
      "code": "971",
      "region": "Guadeloupe"
    },
    {
      "nuts3": "Martinique",
      "code": "972",
      "region": "Martinique"
    },
    {
      "nuts3": "Guyane",
      "code": "973",
      "region": "Guyane"
    },
    {
      "nuts3": "La Réunion",
      "code": "974",
      "region": "La Réunion"
    },
    {
      "nuts3": "Mayotte",
      "code": "976",
      "region": "Mayotte"
    },
    {
      "nuts3": "Nouvelle-Calédonie",
      "code": "988",
      "region": "Nouvelle-Calédonie"
    },
    {
      "nuts3": "Saint-Pierre-et-Miquelon",
      "code": "975",
      "region": "Saint-Pierre-et-Miquelon"
    },
    {
      "nuts3": "Saint-Barthélemy",
      "code": "977",
      "region": "Saint-Barthélemy"
    },
    {
      "nuts3": "Saint-Martin",
      "code": "978",
      "region": "Saint-Martin"
    },
    {
      "nuts3": "Terres australes et antarctiques françaises",
      "code": "984",
      "region": "Terres australes et antarctiques françaises"
    },
    {
      "nuts3": "Wallis-et-Futuna",
      "code": "986",
      "region": "Wallis-et-Futuna"
    },
    {
      "nuts3": "Polynésie française",
      "code": "987",
      "region": "Polynésie française"
    }
  ],
  "pays_vers_departement": {
    "guadeloupe": "971",
    "martinique": "972",
    "guyane": "973",
    "french guiana": "973",
    "reunion": "974",
    "la reunion": "974",
    "mayotte": "976",
    "saint pierre and miquelon": "975",
    "saint barthelemy": "977",
    "saint martin": "978",
    "french southern territories": "984",
    "wallis and futuna": "986",
    "french polynesia": "987",
    "new caledonia": "988"
  },
  "territoires_par_pays": {
    "guadeloupe": "RUP",
    "martinique": "RUP",
    "guyane": "RUP",
    "french guiana": "RUP",
    "guyane francaise": "RUP",
    "reunion": "RUP",
    "la reunion": "RUP",
    "mayotte": "RUP",
    "saint martin": "RUP",
    "saint martin (french part)": "RUP",
    "saint barthelemy": "PTOM",
    "new caledonia": "PTOM",
    "nouvelle caledonie": "PTOM",
    "french polynesia": "PTOM",
    "polynesie francaise": "PTOM",
    "wallis and futuna": "PTOM",
    "wallis et futuna": "PTOM",
    "saint pierre and miquelon": "PTOM",
    "saint pierre et miquelon": "PTOM",
    "french southern territories": "PTOM",
    "taaf": "PTOM"
  },
  "territoires_par_cp3": {
    "971": "RUP",
    "972": "RUP",
    "973": "RUP",
    "974": "RUP",
    "976": "RUP",
    "975": "PTOM",
    "984": "PTOM",
    "986": "PTOM",
    "987": "PTOM",
    "988": "PTOM"
  },
  "nuts2_canonique": {
    "description": "Orthographe canonique des anciennes régions (colonne NUTS2 d'origine) + alias",
    "canon": [
      "Île-de-France",
      "Rhône-Alpes",
      "Provence-Alpes-Côte d'Azur",
      "Midi-Pyrénées",
      "Aquitaine",
      "Bretagne",
      "Alsace",
      "Nord-Pas-de-Calais",
      "Pays de la Loire",
      "Languedoc-Roussillon",
      "Lorraine",
      "Centre-Val de Loire",
      "Poitou-Charentes",
      "Basse-Normandie",
      "Auvergne",
      "Picardie",
      "Franche-Comté",
      "Haute-Normandie",
      "Bourgogne",
      "Limousin",
      "Champagne-Ardenne",
      "Corse",
      "La Réunion",
      "Guadeloupe",
      "Martinique",
      "Guyane",
      "Mayotte",
      "Nouvelle-Calédonie"
    ],
    "alias": {
      "Centre": "Centre-Val de Loire",
      "Reunion": "La Réunion"
    }
  },
  "nuts2_vers_region": {
    "description": "Ancienne région (valeur NUTS2 corrigée) -> nouvelle Région FR. Pour les RUP, nuts3/numero fournis (NUTS2=NUTS3).",
    "map": {
      "Alsace": "Grand Est",
      "Champagne-Ardenne": "Grand Est",
      "Lorraine": "Grand Est",
      "Aquitaine": "Nouvelle-Aquitaine",
      "Limousin": "Nouvelle-Aquitaine",
      "Poitou-Charentes": "Nouvelle-Aquitaine",
      "Auvergne": "Auvergne-Rhône-Alpes",
      "Rhône-Alpes": "Auvergne-Rhône-Alpes",
      "Bourgogne": "Bourgogne-Franche-Comté",
      "Franche-Comté": "Bourgogne-Franche-Comté",
      "Bretagne": "Bretagne",
      "Centre": "Centre-Val de Loire",
      "Centre-Val de Loire": "Centre-Val de Loire",
      "Corse": "Corse",
      "Île-de-France": "Île-de-France",
      "Languedoc-Roussillon": "Occitanie",
      "Midi-Pyrénées": "Occitanie",
      "Nord-Pas-de-Calais": "Hauts-de-France",
      "Picardie": "Hauts-de-France",
      "Basse-Normandie": "Normandie",
      "Haute-Normandie": "Normandie",
      "Pays de la Loire": "Pays de la Loire",
      "Provence-Alpes-Côte d'Azur": "Provence-Alpes-Côte d'Azur",
      "Guadeloupe": "Guadeloupe",
      "Martinique": "Martinique",
      "Guyane": "Guyane",
      "La Réunion": "La Réunion",
      "Mayotte": "Mayotte"
    },
    "rup_nuts3": {
      "Guadeloupe": {
        "nuts3": "Guadeloupe",
        "numero": "971"
      },
      "Martinique": {
        "nuts3": "Martinique",
        "numero": "972"
      },
      "Guyane": {
        "nuts3": "Guyane",
        "numero": "973"
      },
      "La Réunion": {
        "nuts3": "La Réunion",
        "numero": "974"
      },
      "Mayotte": {
        "nuts3": "Mayotte",
        "numero": "976"
      }
    }
  }
}''',
    "FORMES_JURIDIQUES": r'''{
  "description": "Référentiels juridiques/INSEE (cj_septembre_2022) : codes formes juridiques, niveaux CJ 1/2/3, niveaux INPI/utilisateur, libellés niveaux 1 et 2 INSEE. Chargés par le notebook ; le moteur ne contient plus ces tables en dur.",
  "codes": {
    "0000": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "1000": "Entrepreneur individuel",
    "1100": "Artisan-commerçant",
    "1200": "Commerçant",
    "1300": "Artisan",
    "1400": "Officier public ou ministériel",
    "1500": "Profession libérale",
    "1600": "Exploitant agricole",
    "1700": "Agent commercial",
    "1800": "Associé-gérant de société",
    "1900": "(Autre) Personne physique",
    "2110": "Indivision entre personnes physiques",
    "2120": "Indivision avec personne morale",
    "2210": "Société créée de fait entre personnes physiques",
    "2220": "Société créée de fait avec personne morale",
    "2310": "Société en participation entre personnes physiques",
    "2320": "Société en participation avec personne morale",
    "2385": "Société en participation de professions libérales",
    "2400": "Fiducie",
    "2700": "Paroisse hors zone concordataire",
    "2900": "Autre groupement de droit privé non doté de la personnalité morale",
    "3110": "Représentation ou agence commerciale d'état ou organisme public étranger immatriculé au RCS",
    "3120": "Société commerciale étrangère immatriculée au RCS",
    "3205": "Organisation internationale",
    "3210": "État, collectivité ou établissement public étranger",
    "3220": "Société étrangère non immatriculée au RCS",
    "3290": "Autre personne morale de droit étranger",
    "4110": "Établissement public national à caractère industriel ou commercial doté d'un comptable public",
    "4120": "Établissement public national à caractère industriel ou commercial non doté d'un comptable public",
    "4130": "Exploitant public",
    "4140": "Établissement public local à caractère industriel ou commercial",
    "4150": "Régie d'une collectivité locale à caractère industriel ou commercial",
    "4160": "Institution Banque de France",
    "5191": "Société de caution mutuelle",
    "5192": "Société coopérative de banque populaire",
    "5193": "Caisse de crédit maritime mutuel",
    "5194": "Caisse (fédérale) de crédit mutuel",
    "5195": "Association coopérative inscrite (droit local Alsace Moselle)",
    "5196": "Caisse d'épargne et de prévoyance à forme coopérative",
    "5202": "SNC, Société en nom collectif",
    "5203": "Société en nom collectif coopérative",
    "5306": "SCS, Société en commandite simple",
    "5307": "Société en commandite simple coopérative",
    "5308": "SCA, Société en commandite par actions",
    "5309": "Société en commandite par actions coopérative",
    "5310": "SLP, Société en libre partenariat",
    "5370": "Société de Participations Financières de Profession Libérale Société en commandite par actions (SPFPL SCA)",
    "5385": "Société d'exercice libéral en commandite par actions",
    "5410": "Société nationale à responsabilité limitée",
    "5415": "Société d'économie mixte à responsabilité limitée",
    "5422": "Société immobilière pour le commerce et l'industrie (SICOMI) à responsabilité limitée",
    "5426": "Société immobilière de gestion à responsabilité limitée",
    "5430": "Société d'aménagement foncier et d'équipement rural (SAFER) à responsabilité limitée",
    "5431": "Société mixte d'intérêt agricole (SMIA) à responsabilité limitée",
    "5432": "Société d'intérêt collectif agricole à responsabilité limitée",
    "5442": "Société d'attribution à responsabilité limitée",
    "5443": "Société coopérative de construction à responsabilité limitée",
    "5451": "Société coopérative de consommation à responsabilité limitée",
    "5453": "Société coopérative artisanale à responsabilité limitée",
    "5454": "Société coopérative d'intérêt maritime à responsabilité limitée",
    "5455": "Société coopérative de transport routier à responsabilité limitée",
    "5458": "Société coopérative ouvrière de production (SCOP) à responsabilité limitée",
    "5459": "Union de sociétés coopératives à responsabilité limitée",
    "5460": "Autre SARL coopérative",
    "5470": "Société de Participations Financières de Profession Libérale Société à responsabilité limitée (SPFPL SARL)",
    "5485": "SELARL, Société d'exercice libéral à responsabilité limitée",
    "5498": "SARL unipersonnelle",
    "5499": "SARL, Société à responsabilité limitée (sans autre indication)",
    "5505": "Société anonyme à participation ouvrière à conseil d'administration",
    "5510": "Société anonyme nationale à conseil d'administration",
    "5515": "Société anonyme d'économie mixte à conseil d'administration",
    "5520": "Fonds à forme sociétale à conseil d'administration",
    "5522": "Société anonyme immobilière pour le commerce et l'industrie à conseil d'administration (SICOMI)",
    "5525": "Société anonyme immobilière d'investissement à conseil d'administration",
    "5530": "Société anonyme d'aménagement foncier et d'équipement rural à conseil d'administration (SAFER)",
    "5531": "Société anonyme mixte d'intérêt agricole à conseil d'administration (SMIA)",
    "5532": "Société anonyme d'intérêt collectif agricole à conseil d'administration (SICA)",
    "5542": "Société anonyme d'attribution à conseil d'administration",
    "5543": "Société anonyme coopérative de construction à conseil d'administration",
    "5546": "Société anonyme d'HLM à conseil d'administration",
    "5547": "Société anonyme coopérative de production de HLM à conseil d'administration",
    "5548": "SA de crédit immobilier à conseil d'administration",
    "5551": "Société anonyme coopérative de consommation à conseil d'administration",
    "5552": "Société anonyme coopérative de commerçants-détaillants à conseil d'administration",
    "5553": "Société anonyme coopérative artisanale à conseil d'administration",
    "5554": "Société anonyme coopérative (d'intérêt) maritime à conseil d'administration",
    "5555": "Société anonyme coopérative de transport à conseil d'administration",
    "5558": "Société anonyme coopérative de production à conseil d'administration (SCOP)",
    "5559": "Union de sociétés coopératives à forme anonyme et à conseil d'administration",
    "5560": "Autre société anonyme coopérative à conseil d'administration",
    "5570": "Société de Participations Financières de Profession Libérale Société anonyme à conseil d'administration (SFPL)",
    "5585": "Société d'exercice libéral à forme anonyme à conseil d'administration",
    "5599": "Société anonyme à conseil d'administration (sans autre indication)",
    "5605": "Société anonyme à participation ouvrière à directoire",
    "5610": "Société anonyme nationale à directoire",
    "5615": "Société anonyme d'économie mixte à directoire",
    "5620": "Fonds à forme sociétale à directoire",
    "5622": "Société anonyme immobilière pour le commerce et l'industrie à directoire (SICOMI)",
    "5625": "Société anonyme immobilière d'investissement à directoire",
    "5630": "Société anonyme d'aménagement foncier et d'équipement rural à directoire (SAFER)",
    "5631": "Société anonyme mixte d'intérêt agricole à directoire (SMIA)",
    "5632": "Société anonyme d'intérêt collectif agricole à directoire (SICA)",
    "5642": "Société anonyme d'attribution à directoire",
    "5643": "Société anonyme coopérative de construction à directoire",
    "5646": "Société anonyme de HLM à directoire",
    "5647": "Société anonyme coopérative de production de HLM à directoire",
    "5648": "SA de crédit immobilier à directoire",
    "5651": "Société anonyme coopérative de consommation à directoire",
    "5652": "Société anonyme coopérative de commerçants-détaillants à directoire",
    "5653": "Société anonyme coopérative artisanale à directoire",
    "5654": "Société anonyme coopérative d'intérêt maritime à directoire",
    "5655": "Société anonyme coopérative de transport à directoire",
    "5658": "Société anonyme coopérative de production à directoire (SCOP)",
    "5659": "Union de sociétés coopératives à forme anonyme et à conseil d'administration",
    "5660": "Autre société anonyme coopérative à directoire",
    "5670": "Société de Participations Financières de Profession Libérale Société anonyme à Directoire (SPFPL)",
    "5685": "Société d'exercice libéral à forme anonyme à directoire",
    "5699": "Société anonyme à directoire (sans autre indication)",
    "5710": "SAS, société par actions simplifiée",
    "5720": "Société par actions simplifiées associé unique ou société par actions simplifiées unipersonnelle",
    "5770": "Société de Participations Financières de Profession Libérale Société par actions simplifiée (SPFPL SAS)",
    "5785": "SELAS, Société d'exercice libéral par action simplifiée",
    "5800": "Société européenne",
    "6100": "Caisse d'épargne et de prévoyance",
    "6210": "GEIE, Groupement européen d'intérêt économique",
    "6220": "GIE, Groupement d'intérêt économique",
    "6316": "CUMA, Coopérative d'utilisation de matériel agricole en commun",
    "6317": "Société coopérative agricole",
    "6318": "Union de sociétés coopératives agricoles",
    "6411": "Société d'assurances mutuelles",
    "6511": "Sociétés Interprofessionnelles de Soins Ambulatoires",
    "6521": "Société civile de placement immobilier",
    "6532": "Société civile d'intérêt collectif agricole (SICA)",
    "6533": "GAEC, Groupement agricole d'exploitation en commun",
    "6534": "GFA, Groupement foncier agricole",
    "6535": "GAF, Groupement agricole foncier",
    "6536": "GF, Groupement forestier",
    "6537": "GP, Groupement pastoral",
    "6538": "GFR, Groupement foncier et rural",
    "6539": "Société civile foncière",
    "6540": "Société civile immobilière (SCI)",
    "6541": "Société civile immobilière de construction-vente",
    "6542": "Société civile d'attribution",
    "6543": "Société civile coopérative de construction",
    "6544": "Société civile immobilière d' accession progressive à la propriété",
    "6551": "Société civile coopérative de consommation",
    "6554": "Société civile coopérative d'intérêt maritime",
    "6558": "Société civile coopérative entre médecins",
    "6560": "Autre société civile coopérative",
    "6561": "SCP d'avocats",
    "6562": "SCP d'avocats aux conseils",
    "6563": "SCP d'avoués d'appel",
    "6564": "SCP d'huissiers",
    "6565": "SCP de notaires",
    "6566": "SCP de commissaire-priseur judiciaire",
    "6567": "SCP de greffiers de tribunal de commerce",
    "6568": "SCP de conseils juridiques",
    "6569": "SCP de commissaires aux comptes",
    "6571": "SCP de médecins",
    "6572": "SCP de dentistes",
    "6573": "SCP d'infirmiers",
    "6574": "SCP de masseurs-kinésithérapeutes",
    "6575": "SCP de directeurs de laboratoire d'analyse médicale",
    "6576": "SCP de vétérinaires",
    "6577": "SCP de géomètres experts",
    "6578": "SCP d'architectes",
    "6585": "Autre société civile professionnelle",
    "6588": "Société civile laitière",
    "6589": "Société civile de moyens",
    "6595": "Caisse locale de crédit mutuel",
    "6596": "Caisse de crédit agricole mutuel",
    "6597": "SCEA, Société civile d'exploitation agricole",
    "6598": "EARL, Exploitation agricole à responsabilité limitée pluripersonnelle",
    "6599": "Autre société civile",
    "6901": "Autre personne de droit privé inscrite au registre du commerce et des sociétés",
    "7111": "Autorité constitutionnelle",
    "7112": "Autorité administrative ou publique indépendante",
    "7113": "Ministère",
    "7120": "Service central d'un ministère",
    "7150": "Service du ministère de la Défense",
    "7160": "Service déconcentré à compétence nationale d'un ministère (hors Défense)",
    "7171": "Service déconcentré de l'État à compétence (inter) régionale",
    "7172": "Service déconcentré de l'État à compétence (inter) départementale",
    "7179": "(Autre) Service déconcentré de l'État à compétence territoriale",
    "7190": "Ecole nationale non dotée de la personnalité morale",
    "7210": "Commune et commune nouvelle",
    "7220": "Département",
    "7225": "Collectivité et territoire d'Outre Mer",
    "7229": "(Autre) Collectivité territoriale",
    "7230": "Région",
    "7312": "Commune associée et commune déléguée",
    "7313": "Section de commune",
    "7314": "Ensemble urbain",
    "7321": "Association syndicale autorisée",
    "7322": "Association foncière urbaine",
    "7323": "Association foncière de remembrement",
    "7331": "Établissement public local d'enseignement",
    "7340": "Pôle métropolitain",
    "7341": "Secteur de commune",
    "7342": "District urbain",
    "7343": "Communauté urbaine",
    "7344": "Métropole",
    "7345": "Syndicat intercommunal à vocation multiple (SIVOM)",
    "7346": "Communauté de communes",
    "7347": "Communauté de villes",
    "7348": "Communauté d'agglomération",
    "7349": "Autre établissement public local de coopération non spécialisé ou entente",
    "7351": "Institution interdépartementale ou entente",
    "7352": "Institution interrégionale ou entente",
    "7353": "Syndicat intercommunal à vocation unique (SIVU)",
    "7354": "Syndicat mixte fermé",
    "7355": "Syndicat mixte ouvert",
    "7356": "Commission syndicale pour la gestion des biens indivis des communes",
    "7357": "Pôle d'équilibre territorial et rural (PETR)",
    "7361": "Centre communal d'action sociale",
    "7362": "Caisse des écoles",
    "7363": "Caisse de crédit municipal",
    "7364": "Établissement d'hospitalisation",
    "7365": "Syndicat inter hospitalier",
    "7366": "Établissement public local social et médico-social",
    "7367": "Centre Intercommunal d'action sociale (CIAS)",
    "7371": "Office public d'habitation à loyer modéré (OPHLM)",
    "7372": "Service départemental d'incendie et de secours (SDIS)",
    "7373": "Établissement public local culturel",
    "7378": "Régie d'une collectivité locale à caractère administratif",
    "7379": "(Autre) Établissement public administratif local",
    "7381": "Organisme consulaire",
    "7382": "Établissement public national ayant fonction d'administration centrale",
    "7383": "Établissement public national à caractère scientifique culturel et professionnel",
    "7384": "Autre établissement public national d'enseignement",
    "7385": "Autre établissement public national administratif à compétence territoriale limitée",
    "7389": "Établissement public national à caractère administratif",
    "7410": "Groupement d'intérêt public (GIP)",
    "7430": "Établissement public des cultes d'Alsace-Lorraine",
    "7450": "Etablissement public administratif, cercle et foyer dans les armées",
    "7470": "Groupement de coopération sanitaire à gestion publique",
    "7490": "Autre personne morale de droit administratif",
    "8110": "Régime général de la sécurité sociale",
    "8120": "Régime spécial de sécurité sociale",
    "8130": "Institution de retraite complémentaire",
    "8140": "Mutualité sociale agricole",
    "8150": "Régime maladie des non-salariés non agricoles",
    "8160": "Régime vieillesse ne dépendant pas du régime général de la sécurité sociale",
    "8170": "Régime d'assurance chômage",
    "8190": "Autre régime de prévoyance sociale",
    "8210": "Mutuelle",
    "8250": "Assurance mutuelle agricole inscrite au RCS",
    "8290": "Autre organisme mutualiste",
    "8310": "Comité central d'entreprise",
    "8311": "Comité d'établissement",
    "8410": "Syndicat de salariés",
    "8420": "Syndicat patronal",
    "8450": "Ordre professionnel ou assimilé",
    "8470": "Centre technique industriel ou comité professionnel du développement économique",
    "8490": "Autre organisme professionnel",
    "9110": "Syndicat de copropriété",
    "9150": "Association syndicale libre",
    "9210": "Association non déclarée",
    "9220": "Association déclarée",
    "9221": "Association déclarée \"entreprises d'insertion par l'économique\"",
    "9222": "Association intermédiaire",
    "9223": "Groupement d'employeurs",
    "9224": "AARPI",
    "9230": "Association déclarée reconnue d'utilité publique",
    "9240": "Congrégation",
    "9260": "Association de droit local",
    "9300": "Fondation",
    "9900": "Autre personne morale de droit privé",
    "9970": "Groupement de coopération sanitaire à gestion privée"
  },
  "niveaux_inpi": {
    "0000": [
      "Exploitation en commun",
      "Organisme de placement collectif en valeurs mobilières sans personnalité morale"
    ],
    "1000": [
      "Personne physique",
      "Entrepreneur individuel"
    ],
    "2110": [
      "Exploitation en commun",
      "Indivision entre personnes physiques"
    ],
    "2120": [
      "Exploitation en commun",
      "Indivision avec personne morale"
    ],
    "2210": [
      "Exploitation en commun",
      "Société créée de fait entre personnes physiques"
    ],
    "2220": [
      "Exploitation en commun",
      "Société créée de fait avec personne morale"
    ],
    "2310": [
      "Exploitation en commun",
      "Société en participation entre personnes physiques"
    ],
    "2320": [
      "Exploitation en commun",
      "Société en participation avec personne morale"
    ],
    "2385": [
      "Exploitation en commun",
      "Société en participation de professions libérales"
    ],
    "2900": [
      "Autres structures",
      "Autre groupement de droit privé non doté de la personnalité morale"
    ],
    "3110": [
      "Formes juridiques étrangères",
      "Représentation ou agence commerciale d'état ou organisme public étranger immatriculé au RCS"
    ],
    "3120": [
      "Formes juridiques étrangères",
      "Société commerciale étrangère immatriculée au RCS"
    ],
    "3210": [
      "Formes juridiques étrangères",
      "Formes juridiques étrangères"
    ],
    "3220": [
      "Formes juridiques étrangères",
      "Société étrangère non immatriculée au RCS"
    ],
    "3290": [
      "Formes juridiques étrangères",
      "Formes juridiques étrangères"
    ],
    "4110": [
      "Groupement ou EPIC",
      "Établissements publics industriels et commerciaux"
    ],
    "4120": [
      "Groupement ou EPIC",
      "Établissements publics industriels et commerciaux"
    ],
    "4130": [
      "Etablissement ou organisme public, administration",
      "Etablissement public ou régie à caractère industriel ou commercial"
    ],
    "4140": [
      "Groupement ou EPIC",
      "Établissements publics industriels et commerciaux"
    ],
    "4150": [
      "Groupement ou EPIC",
      "Établissements publics industriels et commerciaux"
    ],
    "4160": [
      "Etablissement ou organisme public, administration",
      "Etablissement public ou régie à caractère industriel ou commercial"
    ],
    "5191": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "5192": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "5193": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "5194": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "5196": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "5202": [
      "Société commerciale pluripersonnelle",
      "SNC, Société en nom collectif"
    ],
    "5203": [
      "Société à forme coopérative",
      "Société en nom collectif coopérative"
    ],
    "5306": [
      "Société commerciale pluripersonnelle",
      "Société en commandite"
    ],
    "5307": [
      "Société à forme coopérative",
      "Société en commandite coopérative"
    ],
    "5308": [
      "Société commerciale pluripersonnelle",
      "Société en commandite"
    ],
    "5309": [
      "Société à forme coopérative",
      "Société en commandite coopérative"
    ],
    "5310": [
      "Société commerciale pluripersonnelle",
      "Société en commandite"
    ],
    "5370": [
      "Société commerciale pluripersonnelle",
      "Société en commandite"
    ],
    "5385": [
      "Société commerciale pluripersonnelle",
      "Société en commandite"
    ],
    "5410": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5415": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5422": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5426": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5430": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5431": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5432": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5442": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5443": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5451": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5453": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5454": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5455": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5458": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5459": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5460": [
      "Société à forme coopérative",
      "Société coopérative à responsabilité limitée"
    ],
    "5470": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5485": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5499": [
      "Société commerciale pluripersonnelle",
      "Société à responsabilité limitée"
    ],
    "5505": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5510": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5515": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5520": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5522": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5525": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5530": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5531": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5532": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5542": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5543": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5546": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5547": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5551": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5552": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5553": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5554": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5555": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5558": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5559": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5560": [
      "Société à forme coopérative",
      "SA coopérative à conseil d'administration"
    ],
    "5570": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5585": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5599": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à conseil d'administration"
    ],
    "5605": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5610": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5615": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5620": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5622": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5625": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5630": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5631": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5632": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5642": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5643": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5646": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5647": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5651": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5652": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5653": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5654": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5655": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5658": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5659": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5660": [
      "Société à forme coopérative",
      "SA coopérative à directoire"
    ],
    "5670": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5685": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5699": [
      "Société commerciale pluripersonnelle",
      "Société anonyme à directoire"
    ],
    "5710": [
      "Société commerciale pluripersonnelle",
      "Société par actions simplifiée"
    ],
    "5770": [
      "Société commerciale pluripersonnelle",
      "Société par actions simplifiée"
    ],
    "5785": [
      "Société commerciale pluripersonnelle",
      "Société par actions simplifiée"
    ],
    "5800": [
      "Formes juridiques étrangères",
      "Société européenne"
    ],
    "6210": [
      "Groupement ou EPIC",
      "GEIE, Groupement européen d'intérêt économique"
    ],
    "6220": [
      "Groupement ou EPIC",
      "GIE, Groupement d'intérêt économique"
    ],
    "6316": [
      "Société à forme coopérative",
      "Société coopérative agricole"
    ],
    "6317": [
      "Société à forme coopérative",
      "Société coopérative agricole"
    ],
    "6318": [
      "Société à forme coopérative",
      "Société coopérative agricole"
    ],
    "6411": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "6511": [
      "Société civile",
      "Autre société civile"
    ],
    "6521": [
      "Société civile",
      "Société immobilière ou foncière"
    ],
    "6532": [
      "Société à forme coopérative",
      "société civile coopérative"
    ],
    "6533": [
      "Société ou groupement agricole",
      "GAEC, Groupement agricole d'exploitation en commun"
    ],
    "6534": [
      "Société ou groupement agricole",
      "GFA, Groupement foncier agricole"
    ],
    "6535": [
      "Société ou groupement agricole",
      "GAF, Groupement agricole foncier"
    ],
    "6536": [
      "Société ou groupement agricole",
      "GF, Groupement forestier"
    ],
    "6537": [
      "Société ou groupement agricole",
      "GP, Groupement pastoral"
    ],
    "6538": [
      "Société ou groupement agricole",
      "GFR, Groupement foncier et rural"
    ],
    "6539": [
      "Société civile",
      "Société immobilière ou foncière"
    ],
    "6540": [
      "Société civile",
      "Société immobilière ou foncière"
    ],
    "6541": [
      "Société civile",
      "Société immobilière ou foncière"
    ],
    "6542": [
      "Société civile",
      "Société immobilière ou foncière"
    ],
    "6543": [
      "Société à forme coopérative",
      "société civile coopérative"
    ],
    "6544": [
      "Société civile",
      "Société immobilière ou foncière"
    ],
    "6551": [
      "Société à forme coopérative",
      "société civile coopérative"
    ],
    "6554": [
      "Société à forme coopérative",
      "société civile coopérative"
    ],
    "6558": [
      "Société à forme coopérative",
      "société civile coopérative"
    ],
    "6560": [
      "Société à forme coopérative",
      "société civile coopérative"
    ],
    "6561": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6562": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6563": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6564": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6565": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6566": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6567": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6568": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6569": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6571": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6572": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6573": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6574": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6575": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6576": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6577": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6578": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6585": [
      "Société civile",
      "société civile professionnelle d'exercice libéral"
    ],
    "6589": [
      "Société civile",
      "Autre société civile"
    ],
    "6595": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "6596": [
      "Société à forme coopérative",
      "Forme coopérative spécifique"
    ],
    "6597": [
      "Société ou groupement agricole",
      "SCEA, Société civile d'exploitation agricole"
    ],
    "6598": [
      "Société ou groupement agricole",
      "EARL, Exploitation agricole à responsabilité limitée pluripersonnelle"
    ],
    "6599": [
      "Société civile",
      "Autre société civile"
    ],
    "6901": [
      "Société civile",
      "Autre société civile"
    ],
    "7111": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7112": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7113": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7120": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7150": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7160": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7171": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7172": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7179": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7190": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7210": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7220": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7225": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7229": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7230": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7312": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7313": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7314": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7321": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7322": [
      "Association & syndicat",
      "Association loi 1901 ou assimilé"
    ],
    "7323": [
      "Association & syndicat",
      "Association loi 1901 ou assimilé"
    ],
    "7331": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7340": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7341": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7342": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7343": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7344": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7345": [
      "Syndicat",
      "Syndicat"
    ],
    "7346": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7347": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7348": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7349": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7351": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7352": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7353": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7354": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7355": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7356": [
      "Association & syndicat",
      "Etablissement public administratif"
    ],
    "7357": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7361": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7362": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7363": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7364": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7365": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7366": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7367": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7371": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7372": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7373": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7378": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7379": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7381": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7382": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7383": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7384": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7385": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7389": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7410": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7430": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7450": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7470": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7490": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "8210": [
      "Autres structures",
      "Organisme mutualiste"
    ],
    "8250": [
      "Autre personne morale",
      "Organisme mutualiste"
    ],
    "9224": [
      "Exploitation en commun",
      "AARPI"
    ]
  },
  "niveaux_utilisateur": {
    "0000": [
      "Exploitation en commun",
      "Organisme de placement collectif en valeurs mobilières sans personnalité morale"
    ],
    "2900": [
      "Autres structures",
      "Autre groupement de droit privé non doté de la personnalité morale"
    ],
    "3210": [
      "Formes juridiques étrangères",
      "Formes juridiques étrangères"
    ],
    "3290": [
      "Formes juridiques étrangères",
      "Formes juridiques étrangères"
    ],
    "4130": [
      "Etablissement ou organisme public, administration",
      "Etablissement public ou régie à caractère industriel ou commercial"
    ],
    "4160": [
      "Etablissement ou organisme public, administration",
      "Etablissement public ou régie à caractère industriel ou commercial"
    ],
    "7111": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7112": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7113": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7120": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7150": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7160": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7171": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7172": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7179": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7190": [
      "Etablissement ou organisme public, administration",
      "Administration de l'état"
    ],
    "7210": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7220": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7225": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7229": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7230": [
      "Etablissement ou organisme public, administration",
      "Collectivité territoriale"
    ],
    "7312": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7313": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7314": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7321": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7322": [
      "Association & syndicat",
      "Association loi 1901 ou assimilé"
    ],
    "7323": [
      "Association & syndicat",
      "Association loi 1901 ou assimilé"
    ],
    "7331": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7340": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7341": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7342": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7343": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7344": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7345": [
      "Syndicat",
      "Syndicat"
    ],
    "7346": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7347": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7348": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7349": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7351": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7352": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7353": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7354": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7355": [
      "Association & syndicat",
      "Syndicat"
    ],
    "7356": [
      "Association & syndicat",
      "Etablissement public administratif"
    ],
    "7357": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7361": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7362": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7363": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7364": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7365": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7366": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7367": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7371": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7372": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7373": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7378": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7379": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7381": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7382": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7383": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7384": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7385": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7389": [
      "Etablissement ou organisme public, administration",
      "Etablissement public administratif"
    ],
    "7410": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7430": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7450": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7470": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "7490": [
      "Etablissement ou organisme public, administration",
      "Autre personne morale de droit public administratif"
    ],
    "8210": [
      "Autres structures",
      "Organisme mutualiste"
    ]
  },
  "cj_niv1": {
    "0": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "1": "Entrepreneur individuel",
    "2": "Groupement de droit privé non doté de la personnalité morale",
    "3": "Personne morale de droit étranger",
    "4": "Personne morale de droit public soumise au droit commercial",
    "5": "Société commerciale",
    "6": "Autre personne morale immatriculée au RCS",
    "7": "Personne morale et organisme soumis au droit administratif",
    "8": "Organisme privé spécialisé",
    "9": "Groupement de droit privé"
  },
  "cj_niv2": {
    "00": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "10": "Entrepreneur individuel",
    "21": "Indivision",
    "22": "Société créée de fait",
    "23": "Société en participation",
    "24": "Fiducie",
    "27": "Paroisse hors zone concordataire",
    "28": "Assujetti unique à la TVA",
    "29": "Autre groupement de droit privé non doté de la personnalité morale",
    "31": "Personne morale de droit étranger, immatriculée au RCS (registre du commerce et des sociétés)",
    "32": "Personne morale de droit étranger, non immatriculée au RCS",
    "41": "Etablissement public ou régie à caractère industriel ou commercial",
    "51": "Société coopérative commerciale particulière",
    "52": "Société en nom collectif",
    "53": "Société en commandite",
    "54": "Société à responsabilité limitée (SARL)",
    "55": "Société anonyme à conseil d'administration",
    "56": "Société anonyme à directoire",
    "57": "Société par actions simplifiée",
    "58": "Société européenne",
    "61": "Caisse d'épargne et de prévoyance",
    "62": "Groupement d'intérêt économique",
    "63": "Société coopérative agricole",
    "64": "Société d'assurance mutuelle",
    "65": "Société civile",
    "69": "Autre personne morale de droit privé inscrite au registre du commerce et des sociétés",
    "71": "Administration de l'état",
    "72": "Collectivité territoriale",
    "73": "Etablissement public administratif",
    "74": "Autre personne morale de droit public administratif",
    "81": "Organisme gérant un régime de protection sociale à adhésion obligatoire",
    "82": "Organisme mutualiste",
    "83": "Comité d'entreprise",
    "84": "Organisme professionnel",
    "85": "Organisme de retraite à adhésion non obligatoire",
    "91": "Syndicat de propriétaires",
    "92": "Association loi 1901 ou assimilé",
    "93": "Fondation",
    "99": "Autre personne morale de droit privé"
  },
  "cj_niv3": {
    "0000": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "1000": "Entrepreneur individuel",
    "2110": "Indivision entre personnes physiques",
    "2120": "Indivision avec personne morale",
    "2210": "Société créée de fait entre personnes physiques",
    "2220": "Société créée de fait avec personne morale",
    "2310": "Société en participation entre personnes physiques",
    "2320": "Société en participation avec personne morale",
    "2385": "Société en participation de professions libérales",
    "2400": "Fiducie",
    "2700": "Paroisse hors zone concordataire",
    "2800": "Assujetti unique à la TVA",
    "2900": "Autre groupement de droit privé non doté de la personnalité morale",
    "3110": "Représentation ou agence commerciale d'état ou organisme public étranger immatriculé au RCS",
    "3120": "Société commerciale étrangère immatriculée au RCS",
    "3205": "Organisation internationale",
    "3210": "État, collectivité ou établissement public étranger",
    "3220": "Société étrangère non immatriculée au RCS",
    "3290": "Autre personne morale de droit étranger",
    "4110": "Établissement public national à caractère industriel ou commercial doté d'un comptable public",
    "4120": "Établissement public national à caractère industriel ou commercial non doté d'un comptable public",
    "4130": "Exploitant public",
    "4140": "Établissement public local à caractère industriel ou commercial",
    "4150": "Régie d'une collectivité locale à caractère industriel ou commercial",
    "4160": "Institution Banque de France",
    "5191": "Société de caution mutuelle",
    "5192": "Société coopérative de banque populaire",
    "5193": "Caisse de crédit maritime mutuel",
    "5194": "Caisse (fédérale) de crédit mutuel",
    "5195": "Association coopérative inscrite (droit local Alsace Moselle)",
    "5196": "Caisse d'épargne et de prévoyance à forme coopérative",
    "5202": "Société en nom collectif",
    "5203": "Société en nom collectif coopérative",
    "5306": "Société en commandite simple",
    "5307": "Société en commandite simple coopérative",
    "5308": "Société en commandite par actions",
    "5309": "Société en commandite par actions coopérative",
    "5310": "Société en libre partenariat (SLP)",
    "5370": "Société de Participations Financières de Profession Libérale Société en commandite par actions (SPFPL SCA)",
    "5385": "Société d'exercice libéral en commandite par actions",
    "5410": "SARL nationale",
    "5415": "SARL d'économie mixte",
    "5422": "SARL immobilière pour le commerce et l'industrie (SICOMI)",
    "5426": "SARL immobilière de gestion",
    "5430": "SARL d'aménagement foncier et d'équipement rural (SAFER)",
    "5431": "SARL mixte d'intérêt agricole (SMIA)",
    "5432": "SARL d'intérêt collectif agricole (SICA)",
    "5442": "SARL d'attribution",
    "5443": "SARL coopérative de construction",
    "5451": "SARL coopérative de consommation",
    "5453": "SARL coopérative artisanale",
    "5454": "SARL coopérative d'intérêt maritime",
    "5455": "SARL coopérative de transport",
    "5458": "SARL coopérative de production (SCOP)",
    "5459": "SARL union de sociétés coopératives",
    "5460": "Autre SARL coopérative",
    "5470": "Société de Participations Financières de Profession Libérale Société à responsabilité limitée (SPFPL SARL)",
    "5485": "Société d'exercice libéral à responsabilité limitée",
    "5499": "Société à responsabilité limitée (sans autre indication)",
    "5505": "SA à participation ouvrière à conseil d'administration",
    "5510": "SA nationale à conseil d'administration",
    "5515": "SA d'économie mixte à conseil d'administration",
    "5520": "Fonds à forme sociétale à conseil d'administration",
    "5522": "SA immobilière pour le commerce et l'industrie (SICOMI) à conseil d'administration",
    "5525": "SA immobilière d'investissement à conseil d'administration",
    "5530": "SA d'aménagement foncier et d'équipement rural (SAFER) à conseil d'administration",
    "5531": "Société anonyme mixte d'intérêt agricole (SMIA) à conseil d'administration",
    "5532": "SA d'intérêt collectif agricole (SICA) à conseil d'administration",
    "5542": "SA d'attribution à conseil d'administration",
    "5543": "SA coopérative de construction à conseil d'administration",
    "5546": "SA de HLM à conseil d'administration",
    "5547": "SA coopérative de production de HLM à conseil d'administration",
    "5548": "SA de crédit immobilier à conseil d'administration",
    "5551": "SA coopérative de consommation à conseil d'administration",
    "5552": "SA coopérative de commerçants-détaillants à conseil d'administration",
    "5553": "SA coopérative artisanale à conseil d'administration",
    "5554": "SA coopérative (d'intérêt) maritime à conseil d'administration",
    "5555": "SA coopérative de transport à conseil d'administration",
    "5558": "SA coopérative de production  (SCOP) à conseil d'administration",
    "5559": "SA union de sociétés coopératives à conseil d'administration",
    "5560": "Autre SA coopérative à conseil d'administration",
    "5570": "Société de Participations Financières de Profession Libérale Société anonyme à conseil d'administration (SPFPL SA à conseil d'administration)",
    "5585": "Société d'exercice libéral à forme anonyme à conseil d'administration",
    "5599": "SA à conseil d'administration (s.a.i.)",
    "5605": "SA à participation ouvrière à directoire",
    "5610": "SA nationale à directoire",
    "5615": "SA d'économie mixte à directoire",
    "5620": "Fonds à forme sociétale à directoire",
    "5622": "SA immobilière pour le commerce et l'industrie (SICOMI) à directoire",
    "5625": "SA immobilière d'investissement à directoire",
    "5630": "Safer anonyme à directoire",
    "5631": "SA mixte d'intérêt agricole (SMIA)",
    "5632": "SA d'intérêt collectif agricole (SICA)",
    "5642": "SA d'attribution à directoire",
    "5643": "SA coopérative de construction à directoire",
    "5646": "SA de HLM à directoire",
    "5647": "Société coopérative de production de HLM anonyme à directoire",
    "5648": "SA de crédit immobilier à directoire",
    "5651": "SA coopérative de consommation à directoire",
    "5652": "SA coopérative de commerçants-détaillants à directoire",
    "5653": "SA coopérative artisanale à directoire",
    "5654": "SA coopérative d'intérêt maritime à directoire",
    "5655": "SA coopérative de transport à directoire",
    "5658": "SA coopérative de production (SCOP) à directoire",
    "5659": "SA union de sociétés coopératives à directoire",
    "5660": "Autre SA coopérative à directoire",
    "5670": "Société de Participations Financières de Profession Libérale Société anonyme à Directoire (SPFPL SA à directoire)",
    "5685": "Société d'exercice libéral à forme anonyme à directoire",
    "5699": "SA à directoire (s.a.i.)",
    "5710": "SAS, société par actions simplifiée",
    "5770": "Société de Participations Financières de Profession Libérale Société par actions simplifiée (SPFPL SAS)",
    "5785": "Société d'exercice libéral par action simplifiée",
    "5800": "Société européenne",
    "6100": "Caisse d'Épargne et de Prévoyance",
    "6210": "Groupement européen d'intérêt économique (GEIE)",
    "6220": "Groupement d'intérêt économique (GIE)",
    "6316": "Coopérative d'utilisation de matériel agricole en commun (CUMA)",
    "6317": "Société coopérative agricole",
    "6318": "Union de sociétés coopératives agricoles",
    "6411": "Société d'assurance à forme mutuelle",
    "6511": "Sociétés Interprofessionnelles de Soins Ambulatoires",
    "6521": "Société civile de placement collectif immobilier (SCPI)",
    "6532": "Société civile d'intérêt collectif agricole (SICA)",
    "6533": "Groupement agricole d'exploitation en commun (GAEC)",
    "6534": "Groupement foncier agricole",
    "6535": "Groupement agricole foncier",
    "6536": "Groupement forestier",
    "6537": "Groupement pastoral",
    "6538": "Groupement foncier et rural",
    "6539": "Société civile foncière",
    "6540": "Société civile immobilière",
    "6541": "Société civile immobilière de construction-vente",
    "6542": "Société civile d'attribution",
    "6543": "Société civile coopérative de construction",
    "6544": "Société civile immobilière d' accession progressive à la propriété",
    "6551": "Société civile coopérative de consommation",
    "6554": "Société civile coopérative d'intérêt maritime",
    "6558": "Société civile coopérative entre médecins",
    "6560": "Autre société civile coopérative",
    "6561": "SCP d'avocats",
    "6562": "SCP d'avocats aux conseils",
    "6563": "SCP d'avoués d'appel",
    "6564": "SCP d'huissiers",
    "6565": "SCP de notaires",
    "6566": "SCP de commissaires-priseurs",
    "6567": "SCP de greffiers de tribunal de commerce",
    "6568": "SCP de conseils juridiques",
    "6569": "SCP de commissaires aux comptes",
    "6571": "SCP de médecins",
    "6572": "SCP de dentistes",
    "6573": "SCP d'infirmiers",
    "6574": "SCP de masseurs-kinésithérapeutes",
    "6575": "SCP de directeurs de laboratoire d'analyse médicale",
    "6576": "SCP de vétérinaires",
    "6577": "SCP de géomètres experts",
    "6578": "SCP d'architectes",
    "6585": "Autre société civile professionnelle",
    "6589": "Société civile de moyens",
    "6595": "Caisse locale de crédit mutuel",
    "6596": "Caisse de crédit agricole mutuel",
    "6597": "Société civile d'exploitation agricole",
    "6598": "Exploitation agricole à responsabilité limitée",
    "6599": "Autre société civile",
    "6901": "Autre personne de droit privé inscrite au registre du commerce et des sociétés",
    "7111": "Autorité constitutionnelle",
    "7112": "Autorité administrative ou publique indépendante",
    "7113": "Ministère",
    "7120": "Service central d'un ministère",
    "7150": "Service du ministère de la Défense",
    "7160": "Service déconcentré à compétence nationale d'un ministère (hors Défense)",
    "7171": "Service déconcentré de l'État à compétence (inter) régionale",
    "7172": "Service déconcentré de l'État à compétence (inter) départementale",
    "7179": "(Autre) Service déconcentré de l'État à compétence territoriale",
    "7190": "Ecole nationale non dotée de la personnalité morale",
    "7210": "Commune et commune nouvelle",
    "7220": "Département",
    "7225": "Collectivité et territoire d'Outre Mer",
    "7229": "(Autre) Collectivité territoriale",
    "7230": "Région",
    "7312": "Commune associée et commune déléguée",
    "7313": "Section de commune",
    "7314": "Ensemble urbain",
    "7321": "Association syndicale autorisée",
    "7322": "Association foncière urbaine",
    "7323": "Association foncière de remembrement",
    "7331": "Établissement public local d'enseignement",
    "7340": "Pôle métropolitain",
    "7341": "Secteur de commune",
    "7342": "District urbain",
    "7343": "Communauté urbaine",
    "7344": "Métropole",
    "7345": "Syndicat intercommunal à vocation multiple (SIVOM)",
    "7346": "Communauté de communes",
    "7347": "Communauté de villes",
    "7348": "Communauté d'agglomération",
    "7349": "Autre établissement public local de coopération non spécialisé ou entente",
    "7351": "Institution interdépartementale ou entente",
    "7352": "Institution interrégionale ou entente",
    "7353": "Syndicat intercommunal à vocation unique (SIVU)",
    "7354": "Syndicat mixte fermé",
    "7355": "Syndicat mixte ouvert",
    "7356": "Commission syndicale pour la gestion des biens indivis des communes",
    "7357": "Pôle d'équilibre territorial et rural (PETR)",
    "7361": "Centre communal d'action sociale",
    "7362": "Caisse des écoles",
    "7363": "Caisse de crédit municipal",
    "7364": "Établissement d'hospitalisation",
    "7365": "Syndicat inter hospitalier",
    "7366": "Établissement public local social et médico-social",
    "7367": "Centre Intercommunal d'action sociale (CIAS)",
    "7371": "Office public d'habitation à loyer modéré (OPHLM)",
    "7372": "Service départemental d'incendie et de secours (SDIS)",
    "7373": "Établissement public local culturel",
    "7378": "Régie d'une collectivité locale à caractère administratif",
    "7379": "(Autre) Établissement public administratif local",
    "7381": "Organisme consulaire",
    "7382": "Établissement public national ayant fonction d'administration centrale",
    "7383": "Établissement public national à caractère scientifique culturel et professionnel",
    "7384": "Autre établissement public national d'enseignement",
    "7385": "Autre établissement public national administratif à compétence territoriale limitée",
    "7389": "Établissement public national à caractère administratif",
    "7410": "Groupement d'intérêt public (GIP)",
    "7430": "Établissement public des cultes d'Alsace-Lorraine",
    "7450": "Etablissement public administratif, cercle et foyer dans les armées",
    "7470": "Groupement de coopération sanitaire à gestion publique",
    "7490": "Autre personne morale de droit administratif",
    "8110": "Régime général de la Sécurité Sociale",
    "8120": "Régime spécial de Sécurité Sociale",
    "8130": "Institution de retraite complémentaire",
    "8140": "Mutualité sociale agricole",
    "8150": "Régime maladie des non-salariés non agricoles",
    "8160": "Régime vieillesse ne dépendant pas du régime général de la Sécurité Sociale",
    "8170": "Régime d'assurance chômage",
    "8190": "Autre régime de prévoyance sociale",
    "8210": "Mutuelle",
    "8250": "Assurance mutuelle agricole",
    "8290": "Autre organisme mutualiste",
    "8310": "Comité social économique d’entreprise",
    "8311": "Comité social économique d'établissement",
    "8410": "Syndicat de salariés",
    "8420": "Syndicat patronal",
    "8450": "Ordre professionnel ou assimilé",
    "8470": "Centre technique industriel ou comité professionnel du développement économique",
    "8490": "Autre organisme professionnel",
    "8510": "Institution de prévoyance",
    "8520": "Institution de retraite supplémentaire",
    "9110": "Syndicat de copropriété",
    "9150": "Association syndicale libre",
    "9210": "Association non déclarée",
    "9220": "Association déclarée",
    "9221": "Association déclarée d'insertion par l'économique",
    "9222": "Association intermédiaire",
    "9223": "Groupement d'employeurs",
    "9224": "Association d'avocats à responsabilité professionnelle individuelle",
    "9230": "Association déclarée, reconnue d'utilité publique",
    "9240": "Congrégation",
    "9260": "Association de droit local (Bas-Rhin, Haut-Rhin et Moselle)",
    "9300": "Fondation",
    "9900": "Autre personne morale de droit privé",
    "9970": "Groupement de coopération sanitaire à gestion privée"
  },
  "niveau1_insee": {
    "1": "Personne physique",
    "2": "Groupement de droit privé non doté de la personnalité morale",
    "3": "Personne morale de droit étranger",
    "4": "Personne morale de droit public soumise au droit commercial",
    "5": "Société commerciale",
    "6": "Autre personne morale immatriculée au RCS",
    "7": "Personne morale et organisme soumis au droit administratif",
    "8": "Organisme privé spécialisé",
    "9": "Groupement de droit privé"
  },
  "niveau2_insee": {
    "11": "Artisan-commerçant",
    "12": "Commerçant",
    "13": "Artisan",
    "14": "Officier public ou ministériel",
    "15": "Profession libérale",
    "16": "Exploitant agricole",
    "17": "Agent commercial",
    "18": "Associé Gérant de société",
    "19": "(Autre) personne physique",
    "21": "Indivision",
    "22": "Société créée de fait",
    "23": "Société en participation",
    "24": "Fiducie",
    "27": "Paroisse hors zone concordataire",
    "29": "Autre groupement de droit privé non doté de la personnalité morale",
    "31": "Personne morale de droit étranger immatriculée au RCS",
    "32": "Personne morale de droit étranger non immatriculée au RCS",
    "41": "Établissement public ou régie à caractère industriel ou commercial",
    "51": "Société coopérative commerciale particulière",
    "52": "Société en nom collectif",
    "53": "Société en commandite",
    "54": "Société à responsabilité limitée (SARL)",
    "55": "Société anonyme à conseil d'administration",
    "56": "Société anonyme à directoire",
    "57": "Société anonyme par actions simplifiées",
    "58": "Société européenne",
    "61": "Caisse d'épargne et de prévoyance",
    "62": "Groupement d'intérêt économique",
    "63": "Société coopérative agricole",
    "64": "Société non commerciale d'assurances",
    "65": "Société civile",
    "69": "Autres personnes de droit privé inscrites au registre du commerce et des sociétés",
    "71": "Administration de l'état",
    "72": "Collectivité territoriale",
    "73": "Établissement public administratif",
    "74": "Autre personne morale de droit public administratif",
    "81": "Organisme gérant un régime de protection sociale à adhésion obligatoire",
    "82": "Organisme mutualiste",
    "83": "Comité d'entreprise",
    "84": "Organisme professionnel",
    "91": "Syndicat de propriétaires",
    "92": "Association loi 1901 ou assimilé",
    "93": "Fondation",
    "99": "Autre personne morale de droit privé"
  }
}''',
    "VILLES_FR": r'''{"MAROLLES SOUS LIGNIERES":"10","LES GRANDES CHAPELLES":"10","MERREY SUR ARCE":"10","JUVANCOURT":"10","LAUBRESSEL":"10","ISLE AUMONT":"10","MERGEY":"10","CRUSCADES":"11","CUBIERES SUR CINOBLE":"11","CUCUGNAN":"11","DONAZAC":"11","FAJAC EN VAL":"11","FESTES ET ST ANDRE":"11","FONTANES DE SAULT":"11","FONTJONCOUSE":"11","GALINAGUES":"11","GENERVILLE":"11","GINESTAS":"11","GREFFEIL":"11","GRUISSAN":"11","ISSEL":"11","JOUCOU":"11","LABASTIDE ESPARBAIRENQUE":"11","LACOMBE":"11","LAGRASSE":"11","LA REDORTE":"11","LASBORDES":"11","LAURAC":"11","LEUCATE":"11","MAILHAC":"11","MAYREVILLE":"11","MERIAL":"11","MONTBRUN DES CORBIERES":"11","MONTGRADAIL":"11","MONTHAUT":"11","MONTREDON DES CORBIERES":"11","MOUTHOUMET":"11","NEVIAN":"11","PADERN":"11","PENNAUTIER":"11","PEYREFITTE SUR L HERS":"11","PEYRIAC DE MER":"11","PEZENS":"11","POMAS":"11","PRADELLES CABARDES":"11","PRADELLES EN VAL":"11","PREIXAN":"11","PUILAURENS":"11","QUILLAN":"11","RIEUX EN VAL":"11","ROQUECOURBE MINERVOIS":"11","ROQUEFEUIL":"11","ROQUEFORT DE SAULT":"11","STE COLOMBE SUR GUETTE":"11","ST COUAT DU RAZES":"11","ST LAURENT DE LA CABRERISSE":"11","ST PAPOUL":"11","SAISSAC":"11","SOUPEX":"11","TREBES":"11","TREZIERS":"11","VILLARDEBELLE":"11","VILLARZEL DU RAZES":"11","VILLEMOUSTAUSSOU":"11","VILLENEUVE LES CORBIERES":"11","VILLENEUVE LES MONTREAL":"11","AGUESSAC":"12","BALAGUIER D OLT":"12","LA BASTIDE SOLAGES":"12","BELMONT SUR RANCE":"12","BOR ET BAR":"12","BROMMAT":"12","BROQUIES":"12","LA CAPELLE BALAGUIER":"12","LE CLAPIER":"12","COMPS LA GRAND VILLE":"12","CONQUES EN ROUERGUE":"12","CREISSELS":"12","ENTRAYGUES SUR TRUYERE":"12","FIRMI":"12","GALGAN":"12","GISSAC":"12","GOLINHAC":"12","LEDERGUES":"12","LESTRADE ET THOUELS":"12","MELJAC":"12","MILLAU":"12","MONTAGNOL":"12","BRUGAIROLLES":"11","CABRESPINE":"11","CASSAIGNES":"11","BOURIGEOLE":"11","CAILHAVEL":"11","BOUTENAC":"11","BOUISSE":"11","CARLIPA":"11","CAILLA":"11","L ABERGEMENT CLEMENCIAT":"01","ARS SUR FORMANS":"01","AMBLEON":"01","BRANCOURT EN LAONNOIS":"02","BUCY LES PIERREPONT":"02","CHATILLON LES SONS":"02","BRUYERES SUR FERE":"02","LA CAPELLE":"02","CERSEUIL":"02","BRENY":"02","BRUYS":"02","HAUT VALROMEY":"01","FRANCHELEINS":"01","DOMPIERRE SUR CHALARONNE":"01","DOMPIERRE SUR VEYLE":"01","GENOUILLEUX":"01","DRUILLAT":"01","GORREVOD":"01","FAREINS":"01","FLAXIEU":"01","GEX":"01","SONTHONNAX LA MONTAGNE":"01","SERRIERES DE BRIORD":"01","ST RAMBERT EN BUGEY":"01","ST PAUL DE VARAX":"01","THEZILLIEU":"01","SERVIGNAT":"01","THOISSEY":"01","SOUCLIN":"01","CHAMPAGNE EN VALROMEY":"01","CHARNOZ SUR AIN":"01","CHAMPFROMIER":"01","CHAZEY BONS":"01","CONFRANCON":"01","COLLONGES":"01","CHALEINS":"01","COLOMIEU":"01","CROTTET":"01","GROSLEE ST BENOIT":"01","PARVES ET NATTAGES":"01","ST ANDRE DE BAGE":"01","ST BENIGNE":"01","REYSSOUZE":"01","RELEVANT":"01","PONCIN":"01","PERON":"01","MATAFELON GRANGES":"01","MALAFRETAZ":"01","JOURNANS":"01","JAYAT":"01","LURCY":"01","LHUIS":"01","LEAZ":"01","BOHAS MEYRIAT RIGNAT":"01","MESSIMY SUR SAONE":"01","MOGNENEINS":"01","ORDONNAZ":"01","MIONNAY":"01","MIJOUX":"01","OZAN":"01","ANIZY LE CHATEAU":"02","ATHIES SOUS LAON":"02","ANCIENVILLE":"02","ANDELAIN":"02","BOURG ST CHRISTOPHE":"01","BELIGNEUX":"01","BOYEUX ST JEROME":"01","BOULIGNEUX":"01","CEYZERIAT":"01","ATTIGNAT":"01","BEAUPONT":"01","BOLOZON":"01","BIRIEUX":"01","BRENAZ":"01","ST GERMAIN LES PAROISSES":"01","ST JULIEN SUR REYSSOUZE":"01","ST JEAN DE THURIGNEUX":"01","ST MAURICE DE BEYNOST":"01","ST JULIEN SUR VEYLE":"01","BANNOST VILLEGAGNON":"77","BARBEY":"77","BAZOCHES LES BRAY":"77","BEAUMONT DU GATINAIS":"77","BLENNES":"77","BOISSETTES":"77","BOISSISE LA BERTRAND":"77","BOUGLIGNY":"77","BOURRON MARLOTTE":"77","BRANSLES":"77","BRAY SUR SEINE":"77","BREAU":"77","BUSSY ST MARTIN":"77","CHALAUTRE LA PETITE":"77","CHAMPDEUIL":"77","LA CHAPELLE RABLAIS":"77","CHEVRY COSSIGNY":"77","COMBS LA VILLE":"77","COMPANS":"77","CONDE STE LIBIAIRE":"77","COULOMMIERS":"77","COURTACON":"77","COUTENCON":"77","CROISSY BEAUBOURG":"77","DARVAULT":"77","FAY LES NEMOURS":"77","FROMONT":"77","FUBLAINES":"77","GIREMOUTIERS":"77","GRANDPUITS BAILLY CARROIS":"77","GRETZ ARMAINVILLIERS":"77","GRISY SUISNES":"77","GUERMANTES":"77","HERICY":"77","HONDEVILLIERS":"77","ISLES LES VILLENOY":"77","JOUARRE":"77","LIZINES":"77","LIZY SUR OURCQ":"77","LORREZ LE BOCAGE PREAUX":"77","LUMIGNY NESLES ORMEAUX":"77","LA MADELEINE SUR LOING":"77","MAINCY":"77","MAISONCELLES EN BRIE":"77","MARY SUR MARNE":"77","MAUPERTHUIS":"77","MAY EN MULTIEN":"77","MEAUX":"77","LE MEE SUR SEINE":"77","MEILLERAY":"77","MOISENAY":"77","MONTCEAUX LES MEAUX":"77","MONTEREAU FAULT YONNE":"77","MONTEREAU SUR LE JARD":"77","MONTEVRAIN":"77","MONTMACHOUX":"77","MORET LOING ET ORVANNE":"77","MORTCERF":"77","NANTEAU SUR ESSONNE":"77","NANTEUIL LES MEAUX":"77","CHAUCONIN NEUFMONTIERS":"77","NOISY RUDIGNON":"77","NOISY SUR ECOLE":"77","OTHIS":"77","PAMFOU":"77","PASSY SUR SEINE":"77","PIERRE LEVEE":"77","PONTAULT COMBAULT":"77","PRECY SUR MARNE":"77","QUINCY VOISINS":"77","REBAIS":"77","RECLOSES":"77","REMAUVILLE":"77","ROZAY EN BRIE":"77","STE AULDE":"77","ST CYR SUR MORIN":"77","ST FARGEAU PONTHIERRY":"77","ST GERMAIN SOUS DOUE":"77","ST OUEN EN BRIE":"77","ST PIERRE LES NEMOURS":"77","ST REMY LA VANNE":"77","ST SAUVEUR SUR ECOLE":"77","SANCY LES PROVINS":"77","SEINE PORT":"77","SIGNY SIGNETS":"77","ARBOYS EN BUGEY":"01","AMBERIEUX EN DOMBES":"01","ANDERT ET CONDON":"01","BAGE LE CHATEL":"01","AMBUTRIX":"01","ARBENT":"01","ARANC":"01","OULCHY LE CHATEAU":"02","OULCHES LA VALLEE FOULON":"02","PRESLES ET THIERNY":"02","PIERREMANDE":"02","PONTAVERT":"02","PITHON":"02","PAVANT":"02","ST CHRISTOPHE A BERRY":"02","SANCY LES CHEMINOTS":"02","SACONIN ET BREUIL":"02","ST ALGIS":"02","SERCHES":"02","DHUYS ET MORIN EN BRIE":"02","MARTIGNY COURPIERRE":"02","MEZIERES SUR OISE":"02","MARCY SOUS MARLE":"02","MERCIN ET VAUX":"02","MARIZY ST MARD":"02","MONAMPTEUIL":"02","MENNESSIS":"02","MEURIVAL":"02","MAIZY":"02","LES SEPTVALLONS":"02","LESQUIELLES ST GERMAIN":"02","LOGNY LES AUBENTON":"02","MAGNY LA FOSSE":"02","LICY CLIGNON":"02","MACHECOURT":"02","LY FONTAINE":"02","LEVERGIES":"02","MACOGNY":"02","LHUYS":"02","LE NOUVION EN THIERACHE":"02","NANTEUIL NOTRE DAME":"02","NAMPCELLES LA COUR":"02","NANTEUIL LA FOSSE":"02","MURET ET CROUTTES":"02","MOY DE L AISNE":"02","NEUFLIEUX":"02","NOGENTEL":"02","NAUROY":"02","MONTIGNY EN ARROUAISE":"02","MONTIGNY LES CONDE":"02","MONTIGNY L ALLIER":"02","MONTCHALONS":"02","MONTBREHAIN":"02","MONTLEVON":"02","MONTBAVIN":"02","LANDIFAY ET BERTAIGNEMONT":"02","LANDOUZY LA VILLE":"02","JOUAIGNES":"02","LA HERIE":"02","JEANTES":"02","HOUSSET":"02","HOURY":"02","LERZY":"02","HARY":"02","NOUVION ET CATILLON":"02","OSTEL":"02","OHIS":"02","ROYAUCOURT ET CHAILVET":"02","QUINCY SOUS LE MONT":"02","ROMENY SUR MARNE":"02","RENNEVAL":"02","RETHEUIL":"02","PROISY":"02","REMIES":"02","PROIX":"02","TAVAUX ET PONTSERICOURT":"02","SERINGES ET NESLES":"02","SOMMETTE EAUCOURT":"02","SILLY LA POTERIE":"02","SURFONTAINE":"02","TANNIERES":"02","TERGNIER":"02","SERVAIS":"02","AMBERIEU EN BUGEY":"01","VINCY REUIL ET MAGNY":"02","WISSIGNICOURT":"02","BELLENAVES":"03","LE BOUCHAUD":"03","ARCHIGNAT":"03","VOHARIES":"02","VORGES":"02","BRAIZE":"03","LA ROBINE SUR GALABRE":"04","PRADS HAUTE BLEONE":"04","HAUTES DUYES":"04","MEOLANS REVEL":"04","REVEST DES BROUSSES":"04","PUIMOISSON":"04","REDORTIERS":"04","CROS DE GEORAND":"07","CHANDOLAS":"07","CHALENCON":"07","CHIROLS":"07","DARBRES":"07","CHAMPIS":"07","BOZAS":"07","BAIRON ET SES ENVIRONS":"08","CHAUMONT PORCIEN":"08","CHEMERY CHEHERY":"08","CHATEAU PORCIEN":"08","BRECY BRIERES":"08","CHALLERANGE":"08","CHEVIERES":"08","CHARDENY":"08","FLAIGNES HAVYS":"08","FROMELENNES":"08","GUIGNICOURT SUR VENCE":"08","LA FRANCHEVILLE":"08","GRIVY LOISY":"08","HERBEUVAL":"08","HAUVINE":"08","FLIGNY":"08","LAIFOUR":"08","ISSANCOURT ET RUMEL":"08","HERPY L ARLESIENNE":"08","MARS SOUS BOURCQ":"08","LAVAL MORENCY":"08","MARQUIGNY":"08","MALANDRY":"08","LUCQUY":"08","ILLY":"08","RETHEL":"08","ROCROI":"08","ST LAMBERT ET MONT DE JEUX":"08","ST LOUP EN CHAMPAGNE":"08","ST REMY LE PETIT":"08","PRIX LES MEZIERES":"08","ST FERGEUX":"08","MONTHERME":"08","NOUART":"08","LA NEUVILLE A MAIRE":"08","NANTEUIL SUR AISNE":"08","MONTIGNY SUR MEUSE":"08","MONTCHEUTIN":"08","LA MONCELLE":"08","OSNES":"08","SIGNY L ABBAYE":"08","SEVIGNY LA FORET":"08","SEMIDE":"08","SEMUY":"08","THENORGUES":"08","THELONNE":"08","LE THOUR":"08","TOURNES":"08","SINGLY":"08","TAIZY":"08","TOGES":"08","TREMBLOIS LES ROCROI":"08","VILLERS LE TOURNEUR":"08","VAUX EN DIEULET":"08","VAUX LES MOURON":"08","VIREUX MOLHAIN":"08","AIGUES JUNTES":"09","ALLIERES":"09","ARIGNAC":"09","MONTROZIER":"12","MUROLS":"12","ONET LE CHATEAU":"12","PALMAS D AVEYRON":"12","PAULHE":"12","PRIVEZAC":"12","QUINS":"12","RULLAC ST CIRQ":"12","ST BEAULIZE":"12","STE EULALIE D OLT":"12","STE EULALIE DE CERNON":"12","ARGENCES EN AUBRAC":"12","ST GEORGES DE LUZENCON":"12","ST IGEST":"12","ST JEAN DELNOUS":"12","ST JUST SUR VIAUR":"12","ST LAURENT DE LEVEZOU":"12","ST PARTHEM":"12","SALLES LA SOURCE":"12","SEBAZAC CONCOURES":"12","SEBRAZAC":"12","TAUSSAC":"12","VEYREAU":"12","VIALA DU PAS DE JAUX":"12","VIMENET":"12","AIX EN PROVENCE":"13","ARLES":"13","BARBENTANE":"13","BERRE L ETANG":"13","BOUC BEL AIR":"13","CABANNES":"13","CEYRESTE":"13","ENSUES LA REDONNE":"13","EYGUIERES":"13","EYRAGUES":"13","GARDANNE":"13","GEMENOS":"13","GIGNAC LA NERTHE":"13","GRAVESON":"13","GREASQUE":"13","ST PIERRE DE MEZOARGUES":"13","MIMET":"13","PEYNIER":"13","PORT ST LOUIS DU RHONE":"13","LA ROQUE D ANTHERON":"13","ROQUEFORT LA BEDOULE":"13","LE ROVE":"13","ST MARTIN DE CRAU":"13","SAUSSET LES PINS":"13","SIMIANE COLLONGUE":"13","TRETS":"13","MARSEILLE 02":"13","MARSEILLE 05":"13","MARSEILLE 09":"13","MARSEILLE 11":"13","MARSEILLE 12":"13","MARSEILLE 13":"13","AIRAN":"14","AMAYE SUR ORNE":"14","ANNEBAULT":"14","LES AUTHIEUX PAPION":"14","AVENAY":"14","BANNEVILLE LA CAMPAGNE":"14","BASLY":"14","BASSENEVILLE":"14","BAYEUX":"14","BEAUMESNIL":"14","BEAUMONT EN AUGE":"14","SOULEUVRE EN BOCAGE":"14","LA BIGNE":"14","BRANVILLE":"14","LA CAMBE":"14","CAMBES EN PLAINE":"14","CAMBREMER":"14","CARDONVILLE":"14","CASTILLON EN AUGE":"14","CASTILLY":"14","CAUMONT L EVENTE":"14","CAUVICOURT":"14","CAUVILLE":"14","CESNY BOIS HALBOUT":"14","CHICHEBOVILLE":"14","CLINCHAMPS SUR ORNE":"14","COMMES":"14","CONDE EN NORMANDIE":"14","ST MARTIN DE BAVEL":"01","AIZY JOUY":"02","AISONVILLE ET BERNOVILLE":"02","VILLETTE SUR AIN":"01","AMIFONTAINE":"02","AGUILCOURT":"02","VILLEBOIS":"01","VARAMBON":"01","TRAMOYES":"01","VIRIAT":"01","AUBIGNY EN LAONNOIS":"02","BARZY EN THIERACHE":"02","AUDIGNICOURT":"02","AZY SUR MARNE":"02","BARENTON CEL":"02","AUDIGNY":"02","BEAUVOIS EN VERMANDOIS":"02","BOSMONT SUR SERRE":"02","BESNY ET LOIZY":"02","BLERANCOURT":"02","BESMONT":"02","BLESMES":"02","BERNOT":"02","BELLEU":"02","BOUE":"02","COLLIGIS CRANDELAIN":"02","CHERMIZY AILLES":"02","CLAIRFONTAINE":"02","CHIVRES VAL":"02","COMMENCHON":"02","CHEVENNES":"02","COLONFAY":"02","CHIERRY":"02","COULONGES COHAN":"02","CUISY EN ALMONT":"02","CRAMAILLE":"02","CUIRIEUX":"02","DOLIGNON":"02","CRAONNE":"02","DHUIZEL":"02","DOMPTIN":"02","DOHIS":"02","DERCY":"02","FLAVIGNY LE GRAND ET BEAURAIN":"02","FONTAINE LES VERVINS":"02","ETAMPES SUR MARNE":"02","LA FERE":"02","EPARCY":"02","GOUDELANCOURT LES BERRIEUX":"02","FRESNES EN TARDENOIS":"02","GRANDLUP ET FAY":"02","GROUGIS":"02","FOSSOY":"02","ST ANDRE D EMBRUN":"05","LA ROCHE DE RAME":"05","LA HAUTE BEAUME":"05","LA GRAVE":"05","ORCIERES":"05","REOTIER":"05","GAP":"05","VILLAR ST PANCRACE":"05","LES VIGNEAUX":"05","TRESCLEOUX":"05","SAVOURNON":"05","SALERANS":"05","VALSERRES":"05","ST VERAN":"05","LAURAC EN VIVARAIS":"07","ORGNAC L AVEN":"07","LAMASTRE":"07","NONIERES":"07","LAVIOLLE":"07","OLIZY PRIMAT":"08","PUILLY ET CHARBEAUX":"08","REMILLY LES POTHEES":"08","REMILLY AILLICOURT":"08","NOVY CHEVRIERES":"08","POURU ST REMY":"08","RENWEZ":"08","SIGY":"77","SOIGNOLLES EN BRIE":"77","SOISY BOUY":"77","SOUPPES SUR LOING":"77","VAIRES SUR MARNE":"77","VALENCE EN BRIE":"77","VENEUX LES SABLONS":"77","VERNEUIL L ETANG":"77","VIEUX CHAMPAGNE":"77","VILLEBEON":"77","VILLENEUVE LES BORDES":"77","VILLIERS SUR MORIN":"77","VOULANGIS":"77","VULAINES LES PROVINS":"77","ANDELU":"78","ANDRESY":"78","AUBERGENVILLE":"78","AUFFARGIS":"78","AUTOUILLET":"78","BAZAINVILLE":"78","BEHOUST":"78","BLARU":"78","BONNELLES":"78","BOUAFLE":"78","CARRIERES SOUS POISSY":"78","CHAUFOUR LES BONNIERES":"78","LES CLAYES SOUS BOIS":"78","COIGNIERES":"78","DAMPIERRE EN YVELINES":"78","FAVRIEUX":"78","FLACOURT":"78","GALLUIS":"78","GARANCIERES":"78","GRESSEY":"78","GUYANCOURT":"78","HARGEVILLE":"78","JOUY MAUVOISIN":"78","JUZIERS":"78","LAINVILLE EN VEXIN":"78","MANTES LA VILLE":"78","MAREIL LE GUYON":"78","LE MESNIL LE ROI":"78","LES MESNULS":"78","MILON LA CHAPELLE":"78","MITTAINVILLE":"78","MONTAINVILLE":"78","MORAINVILLIERS":"78","NEAUPHLE LE VIEUX":"78","NEAUPHLETTE":"78","ORPHIN":"78","LE PERRAY EN YVELINES":"78","POISSY":"78","PORT VILLEZ":"78","PRUNAY EN YVELINES":"78","TACOIGNIERES":"78","LE TARTRE GAUDRAN":"78","TOUSSUS LE NOBLE":"78","TRIEL SUR SEINE":"78","VAUX SUR SEINE":"78","VILLIERS ST FREDERIC":"78","VIROFLAY":"78","VOISINS LE BRETONNEUX":"78","ADILLY":"79","AIGONNAY":"79","AIRVAULT":"79","ARGENTONAY":"79","AZAY LE BRULE":"79","BEAULIEU SOUS PARTHENAY":"79","BEAUSSAIS VITRE":"79","BEAUVOIR SUR NIORT":"79","BOUGON":"79","LE BOURDET":"79","BRESSUIRE":"79","BRETIGNOLLES":"79","LE BREUIL BERNARD":"79","BRIEUIL SUR CHIZE":"79","CELLES SUR BELLE":"79","CHAMPDENIERS ST DENIS":"79","LA CHAPELLE BERTRAND":"79","LA CHAPELLE ST ETIENNE":"79","LA CHAPELLE THIREUIL":"79","PRISSE LA CHARRIERE":"79","MAULEON":"79","CHATILLON SUR THOUET":"79","CHERIGNE":"79","CHERVEUX":"79","VAUCELLES ET BEFFECOURT":"02","TORCY EN VALOIS":"02","VAUX ANDIGNY":"02","LE THUEL":"02","TROESNES":"02","VASSOGNE":"02","TENDE":"06","CHATEAUNEUF DE VERNOUX":"07","ALBA LA ROMAINE":"07","BOUCIEU LE ROI":"07","LES ASSIONS":"07","LE CHAMBON":"07","ASPERJOC":"07","ALISSAS":"07","AILHON":"07","BOREE":"07","LACHAPELLE GRAILLOUSE":"07","LE LAC D ISSARLES":"07","ISSAMOULENC":"07","LALOUVESC":"07","ISSANLAS":"07","LABEGUDE":"07","JAUNAC":"07","JAUJAC":"07","LANAS":"07","LAVAL D AURELLE":"07","LENTILLERES":"07","PAILHARES":"07","MERCUER":"07","MARIAC":"07","MEYRAS":"07","MENIL ANNELLES":"08","MONT LAURENT":"08","NOUZONVILLE":"08","MONTHOIS":"08","OMICOURT":"08","MONDIGNY":"08","NOIRVAL":"08","POURU AUX BOIS":"08","QUATRE CHAMPS":"08","REGNIOWEZ":"08","ST JUVIN":"08","SACHY":"08","PURE":"08","SAPOGNE ET FEUCHERES":"08","SAPOGNE SUR MARCHE":"08","ST PIERRE A ARNES":"08","SORBON":"08","SON":"08","THILAY":"08","VILLERS DEVANT LE THOUR":"08","VILLERS SUR LE MONT":"08","SORCY BAUTHEMONT":"08","VIVIER AU COURT":"08","VIEL ST REMY":"08","TAILLETTE":"08","VERPEL":"08","VOUZIERS":"08","WADELINCOURT":"08","AUDRESSEIN":"09","AUCAZEIN":"09","WASIGNY":"08","ARGEIN":"09","VONCQ":"08","VRIZY":"08","ALZEN":"09","CAZENAVE SERRES ET ALLENS":"09","CASTELNAU DURBAN":"09","CARCANIERES":"09","DREUILHE":"09","CASTERAS":"09","CERIZOLS":"09","ERCE":"09","ORNOLAC USSAT LES BAINS":"09","MONTESQUIEU AVANTES":"09","MONTSERON":"09","MONESPLE":"09","MERIGON":"09","MIGLOS":"09","PECH":"09","ORUS":"09","CAPOULET ET JUNAC":"09","BALAGUERES":"09","AUGIREIN":"09","VANDY":"08","CASTILLON EN COUSERANS":"09","LES BORDES SUR LEZ":"09","CAMPAGNE SUR ARIZE":"09","CAUSSOU":"09","CAYCHAX":"09","CALZAN":"09","BOUAN":"09","BRAGELOGNE BEAUVOIR":"10","BREVIANDES":"10","BARBEREY ST SULPICE":"10","BALNOT SUR LAIGNES":"10","BALNOT LA GRANGE":"10","BOUY SUR ORVIN":"10","BALIGNICOURT":"10","BERTIGNOLLES":"10","BAR SUR AUBE":"10","BOULAGES":"10","CRENEY PRES TROYES":"10","BUCHERES":"10","CELLES SUR OURCE":"10","BRIEL SUR BARSE":"10","COLOMBE LE SEC":"10","CHAVANGES":"10","BREVONNES":"10","CHERVEY":"10","VILLENEUVE D OLMES":"09","BAILLY LE FRANC":"10","VERNIOLLE":"09","AUBETERRE":"10","VENTENAC":"09","ARGANCON":"10","VAYCHIS":"09","URS":"09","ST FELIX DE TOURNEGAT":"09","MONTJOIE EN COUSERANS":"09","ST FELIX DE RIEUTORD":"09","PRAT BONREPAUX":"09","ST BAUZEIL":"09","LES PUJOLS":"09","RIEUCROS":"09","ST AMADOU":"09","LES ISSARDS":"09","ILLARTEIN":"09","COUTENS":"09","GAUDIES":"09","ESCOSSE":"09","LACOURT":"09","ESPLAS":"09","ILHAT":"09","DUN":"09","MAIZIERES LA GRANDE PAROISSE":"10","MAISONS LES CHAOURCE":"10","MESGRIGNY":"10","MATHAUX":"10","SOUEIX ROGALLE":"09","ST MARTIN DE CARALP":"09","ST QUENTIN LA TOUR":"09","LA TOUR DU CRIEU":"09","UCHENTEIN":"09","ST LIZIER":"09","SURBA":"09","TABRE":"09","UNAC":"09","SOR":"09","MONTEGUT PLANTAUREL":"09","MONTFERRIER":"09","MALEGOUDE":"09","LIEURAC":"09","LAPEGE":"09","LANOUX":"09","LARCAT":"09","MASSAT":"09","LERAN":"09","MERAS":"09","MACEY":"10","LONGEVILLE SUR MOGNE":"10","LA LOUPTIERE THENARD":"10","LONGCHAMP SUR AUJON":"10","LAINES AUX BOIS":"10","MAILLY LE CAMP":"10","MAGNY FOUCHARD":"10","LUYERES":"10","FONTENAY DE BOSSERY":"10","FAY LES MARCILLY":"10","FAUX VILLECERF":"10","EAUX PUISEAUX":"10","EPOTHEMONT":"10","JUZANVIGNY":"10","GRANDVILLE":"10","LES GRANGES":"10","COQUAINVILLIERS":"14","CORMOLAIN":"14","COTTUN":"14","COURTONNE LA MEURDRAC":"14","CREULLY":"14","CRICQUEVILLE EN AUGE":"14","CROCY":"14","CROUAY":"14","CULLY":"14","DANVOU LA FERRIERE":"14","DEMOUVILLE":"14","DOZULE":"14","BEAUFOUR DRUVAL":"14","ENGLESQUEVILLE LA PERCEE":"14","EPANEY":"14","ERAINES":"14","ERNES":"14","ESSON":"14","ETERVILLE":"14","LE FAULQ":"14","FONTAINE LE PIN":"14","FOURNEAUX LE VAL":"14","FRESNEY LE PUCEUX":"14","FRESNEY LE VIEUX":"14","LE GAST":"14","GIBERVILLE":"14","GLANVILLE":"14","GOUVIX":"14","GRAINVILLE SUR ODON":"14","GRANDCAMP MAISY":"14","HONFLEUR":"14","HOTOT EN AUGE":"14","HUBERT FOLIE":"14","JURQUES":"14","LANTHEUIL":"14","LESSARD ET LE CHENE":"14","LIVAROT PAYS D AUGE":"14","LES LOGES SAULCES":"14","MAGNY EN BESSIN":"14","MAGNY LE FREULE":"14","MANERBE":"14","LE MARAIS LA CHAPELLE":"14","MARTRAGNY":"14","MERY CORBON":"14","LE MESNIL AU GRAIN":"14","MESNIL CLINCHAMPS":"14","MEZIDON CANON":"14","LES MONCEAUX":"14","MONFREVILLE":"14","MOSLES":"14","MOULT":"14","NONANT":"14","NORON LA POTERIE":"14","NOYERS MISSY":"14","PONT BELLANGER":"14","LE PRE D AUGE":"14","RANVILLE":"14","REPENTIGNY":"14","REUX":"14","ROTS":"14","ROUCAMPS":"14","RUCQUEVILLE":"14","RUSSY":"14","ST ANDRE SUR ORNE":"14","ST CONTEST":"14","VALORBIQUET":"14","ST DENIS DE MERE":"14","ST GABRIEL BRECY":"14","STE HONORINE DES PERTES":"14","ST JEAN DE LIVET":"14","ST LAURENT DE CONDEL":"14","ST MANVIEU NORREY":"14","STE MARGUERITE DE VIETTE":"14","ST MARTIN AUX CHARTRAINS":"14","ST MARTIN DE BIENFAITE LA CRESSONNIERE":"14","ST MARTIN DE MIEUX":"14","ST PIERRE DU BU":"14","PREZ":"08","TREMBLOIS LES CARIGNAN":"08","TERRON SUR AISNE":"08","THUGNY TRUGNY":"08","STONNE":"08","THIS":"08","SY":"08","AIX VILLEMAUR PALIS":"10","THOUARS SUR ARIZE":"09","SERRES SUR ARGET":"09","TROYE D ARIEGE":"09","VERNAJOUL":"09","VERNAUX":"09","SIEURAS":"09","SIGUER":"09","SOULAN":"09","VIREUX WALLERAND":"08","VILLERS SUR BAR":"08","VILLE SUR LUMES":"08","VAUX CHAMPAGNE":"08","VAUX MONTREUIL":"08","WARNECOURT":"08","L AIGUILLON":"09","WIGNICOURT":"08","VENDRESSE":"08","COUFLENS":"09","FERRIERES SUR ARIEGE":"09","DAUMAZAN SUR ARIZE":"09","DURBAN SUR ARIZE":"09","CRAMPAGNA":"09","LE FOSSAT":"09","EYCHEIL":"09","DALOU":"09","LEZAT SUR LEZE":"09","LIMBRASSAC":"09","MONTARDIT":"09","LUZENAC":"09","MALLEON":"09","GESTIES":"09","GANAC":"09","CARLA BAYLE":"09","LE CARLARET":"09","BOUSSENAC":"09","CONTRAZY":"09","CAZAVET":"09","MONTEGUT EN COUSERANS":"09","MONTSEGUR":"09","MONTGAUCH":"09","PRADETTES":"09","LE PUCH":"09","LE PLA":"09","MOULIS":"09","NESCUS":"09","ORGEIX":"09","NIAUX":"09","DOLANCOURT":"10","CUSSANGY":"10","CUNFIN":"10","ORLEANS":"45","OUSSON SUR LOIRE":"45","OUTARVILLE":"45","OUZOUER SOUS BELLEGARDE":"45","OUZOUER SUR TREZEE":"45","PITHIVIERS":"45","RAMOULU":"45","ST DENIS DE L HOTEL":"45","ST DENIS EN VAL":"45","ST HILAIRE ST MESMIN":"45","ST JEAN DE BRAYE":"45","ST PERAVY LA COLOMBE":"45","ST PERE SUR LOIRE":"45","ST PRYVE ST MESMIN":"45","SANTEAU":"45","SARAN":"45","SCEAUX DU GATINAIS":"45","SIGLOY":"45","THIMORY":"45","TIGY":"45","TREILLES EN GATINAIS":"45","VIENNE EN VAL":"45","VILLEMANDEUR":"45","VILLORCEAU":"45","LE CHILLOU":"79","CLAVE":"79","COULONGES THOUARSAIS":"79","FOMPERRON":"79","LA FORET SUR SEVRE":"79","FRONTENAY ROHAN ROHAN":"79","GENNETON":"79","GERMOND ROUVRE":"79","GOURNAY LOIZE":"79","HANC":"79","JUSCORPS":"79","LUCHE SUR BRIOUX":"79","MARNES":"79","MASSAIS":"79","LA MOTHE ST HERAY":"79","NIORT":"79","NUEIL LES AUBIERS":"79","OIRON":"79","PAMPROUX":"79","PAS DE JEU":"79","LA PETITE BOISSIERE":"79","LA PEYRATTE":"79","PIOUSSAY":"79","PRIAIRES":"79","ROM":"79","ST ANDRE SUR SEVRE":"79","VOULMENTIN":"79","ST GELAIS":"79","ST GENEROUX":"79","ST LEGER DE MONTBRUN":"79","ST LOUP LAMAIRE":"79","ST MARTIN LES MELLE":"79","ST MAXIRE":"79","STE OUENNE":"79","ST PIERRE DES ECHAUBROGNES":"79","SAIVRES":"79","SECONDIGNY":"79","SELIGNE":"79","LE TALLUD":"79","THOUARS":"79","TOURTENAY":"79","VANCAIS":"79","VILLEFOLLET":"79","VILLIERS SUR CHIZE":"79","ALLAINES":"80","ALLENAY":"80","AULT":"80","AYENCOURT":"80","BACOUEL SUR SELLE":"80","BAYONVILLERS":"80","BEALCOURT":"80","BEAUCOURT SUR L ANCRE":"80","BECORDEL BECOURT":"80","BEHEN":"80","BERMESNIL":"80","BERTRANCOURT":"80","BETHENCOURT SUR SOMME":"80","BETTEMBOS":"80","BIACHES":"80","BILLANCOURT":"80","BOUCHAVESNES BERGEN":"80","BOUGAINVILLE":"80","BOUILLANCOURT EN SERY":"80","BOUQUEMAISON":"80","BOURDON":"80","BOUSSICOURT":"80","BOUVINCOURT EN VERMANDOIS":"80","BRAY LES MAREUIL":"80","BREILLY":"80","BRUTELLES":"80","BUIGNY L ABBE":"80","BUS LES ARTOIS":"80","BUSSY LES POIX":"80","CAIX":"80","CAPPY":"80","CARNOY":"80","CHAULNES":"80","COIGNEUX":"80","COLINCAMPS":"80","COTTENCHY":"80","COURTEMANCHE":"80","CREUSE":"80","CROUY ST PIERRE":"80","DOMESMONT":"80","DROMESNIL":"80","EAUCOURT SUR SOMME":"80","ESCLAINVILLERS":"80","BENAGUES":"09","CAMARADE":"09","BESTIAC":"09","BURRET":"09","BIERT":"09","ESPLAS DE SEROU":"09","FREYCHENET":"09","ESCLAGNE":"09","GOURBIT":"09","LAPENNE":"09","GARANOU":"09","LASSUR":"09","IGNAUX":"09","LE MAS D AZIL":"09","LESCOUSSE":"09","LEYCHERT":"09","MERCENAC":"09","LESCURE":"09","MADIERE":"09","MANSES":"09","LUDIES":"09","RIEUX DE PELLEPORT":"09","ROUMENGOUX":"09","RIVERENERT":"09","LE PEYRAT":"09","PRAYOLS":"09","ROUZE":"09","QUIE":"09","TARASCON SUR ARIEGE":"09","SENTENAC D OUST":"09","SUC ET SENTENAC":"09","SENTEIN":"09","SORGEAT":"09","SINSAT":"09","SOULA":"09","SEIX":"09","OUZOUER SUR LOIRE":"45","PERS EN GATINAIS":"45","QUIERS SUR BEZONDE":"45","ST AIGNAN DES GUES":"45","ST BRISSON SUR LOIRE":"45","ST FIRMIN DES BOIS":"45","ST GONDON":"45","ST LOUP DES VIGNES":"45","ST MAURICE SUR AVEYRON":"45","ST MAURICE SUR FESSARD":"45","SANDILLON":"45","SEMOY":"45","SULLY SUR LOIRE":"45","THIGNONVILLE":"45","THORAILLES":"45","VANNES SUR COSSON":"45","VIGLAIN":"45","VILLEMURLIN":"45","VIMORY":"45","ANGLARS JUILLAC":"46","ARCAMBAL":"46","LES ARQUES":"46","ASSIER":"46","BELAYE":"46","BIO":"46","BLARS":"46","CALVIGNAC":"46","CAMBURAT":"46","CENEVIERES":"46","CONCOTS":"46","CRESSENSAC":"46","DEGAGNAC":"46","ESCLAUZELS":"46","ESPEDAILLAC":"46","ESPEYROUX":"46","FAJOLES":"46","FIGEAC":"46","ST PAUL FLAUGNAC":"46","GAGNAC SUR CERE":"46","GLANES":"46","GREALOU":"46","COEUR DE CAUSSE":"46","LABATHUDE":"46","LARNAGOL":"46","LASCABANES":"46","LATRONQUIERE":"46","LAUZES":"46","GERAUDOT":"10","MONTIGNY LES MONTS":"10","MESNIL SELLIERES":"10","MOLINS SUR AUBE":"10","MESNIL ST PERE":"10","MESNIL LETTRE":"10","MONTSUZAIN":"10","PLANCY L ABBAYE":"10","NEUVILLE SUR VANNE":"10","PRECY NOTRE DAME":"10","NOGENT SUR AUBE":"10","PONT SUR SEINE":"10","POIVRES":"10","PLANTY":"10","PRUGNY":"10","PINEY":"10","ROUILLY ST LOUP":"10","ST CHRISTOPHE DODINICOURT":"10","ST MARTIN DE BOSSENAY":"10","ST MARDS EN OTHE":"10","ROUILLY SACEY":"10","RACINES":"10","SERMOISE SUR LOIRE":"58","TRACY SUR LOIRE":"58","ABSCON":"59","ARMENTIERES":"59","AUBY":"59","AUCHY LEZ ORCHIES":"59","BACHY":"59","BAISIEUX":"59","LA BASSEE":"59","BAUVIN":"59","BAVAY":"59","BAZUEL":"59","BEAUDIGNIES":"59","BELLIGNIES":"59","BERGUES":"59","BERLAIMONT":"59","BERSEE":"59","BERSILLIES":"59","BERTHEN":"59","BERTRY":"59","BETTRECHIES":"59","BISSEZEELE":"59","BLARINGHEM":"59","BOESCHEPE":"59","BOLLEZEELE":"59","BORRE":"59","BOUCHAIN":"59","BOURBOURG":"59","BOUSBECQUE":"59","BOUSIGNIES SUR ROC":"59","BOUSSIERES SUR SAMBRE":"59","BOUSSOIS":"59","BOUVIGNIES":"59","BRUNEMONT":"59","BRY":"59","BUYSSCHEURE":"59","CAESTRE":"59","CANTIN":"59","CARTIGNIES":"59","CASSEL":"59","CAUDRY":"59","CAULLERY":"59","CYSOING":"59","DECHY":"59","DIMONT":"59","DOUAI":"59","DOURLERS":"59","DRINCHAM":"59","DUNKERQUE":"59","ECUELIN":"59","ENNEVELIN":"59","ESCARMAIN":"59","ESCAUDOEUVRES":"59","ESNES":"59","ESTAIRES":"59","ESTOURMEL":"59","ESWARS":"59","ETROEUNGT":"59","FACHES THUMESNIL":"59","FERON":"59","FLINES LEZ RACHES":"59","FOREST EN CAMBRESIS":"59","FRASNOY":"59","FRELINGHIEN":"59","GHISSIGNIES":"59","GLAGEON":"59","GONDECOURT":"59","GRAND FORT PHILIPPE":"59","GRAVELINES":"59","LA GROISE":"59","ST SAMSON":"14","SEPT VENTS":"14","SOIGNOLLES":"14","SOLIERS":"14","SOULANGY":"14","LE HOM":"14","TILLY SUR SEULLES":"14","LE TORQUESNE":"14","L OUDON":"14","TOURGEVILLE":"14","TOURVILLE EN AUGE":"14","TRACY SUR MER":"14","VALDALLIERE":"14","LE VEY":"14","VICQUES":"14","VIEUX BOURG":"14","VILLERVILLE":"14","VILLONS LES BUISSONS":"14","VILLY BOCAGE":"14","VIRE NORMANDIE":"14","ALLANCHE":"15","ALLEUZE":"15","ANDELAT":"15","ARNAC":"15","AYRENS":"15","BADAILHAC":"15","BASSIGNAC":"15","CARLAT":"15","CHALINARGUES":"15","CHAUDES AIGUES":"15","CHAUSSENAC":"15","LE CLAUX":"15","DRUGEAC":"15","FRIDEFONT":"15","JOU SOUS MONJOU":"15","LABROUSSE":"15","LAPEYRUGUE":"15","LAURIE":"15","VAL D ARCOMIE":"15","MALBO":"15","MARMANHAC":"15","MAURINES":"15","MONTBOUDIF":"15","MOURJOU":"15","NARNHAC":"15","PARLAN":"15","PEYRUSSE":"15","RUYNES EN MARGERIDE":"15","ST AMANDIN":"15","ST CERNIN":"15","ST ILLIDE":"15","ST MARTIN CANTALES":"15","ST MARTIN VALMEROUX":"15","ST SAURY":"15","SOULAGES":"15","TIVIERS":"15","LA TRINITAT":"15","VEBRET":"15","VEDRINES ST LOUP":"15","VEZELS ROUSSY":"15","YTRAC":"15","LE ROUGET PERS":"15","AGRIS":"16","AMBERAC":"16","ANGOULEME":"16","ANVILLE":"16","BALZAC":"16","BEAULIEU SUR SONNETTE":"16","BIOUSSAC":"16","BLANZAC PORCHERESSE":"16","BORS DE MONTMOREAU":"16","BORS DE BAIGNES":"16","BRIE SOUS BARBEZIEUX":"16","BUNZAC":"16","CELLEFROUIN":"16","CHANTILLAC":"16","BOISNE LA TUDE":"16","CHASSENEUIL SUR BONNIEURE":"16","CHASSIECQ":"16","CHENOMMET":"16","CHERVES RICHEMONT":"16","CHIRAC":"16","CONDAC":"16","CONFOLENS":"16","COURBILLAC":"16","HANNOGNE ST REMY":"08","YEVRE LA VILLE":"45","BELMONT STE FOI":"46","BERGANTY":"46","LE BOUYSSOU":"46","BOUZIES":"46","CAJARC":"46","CALAMANE":"46","CAMBOULIT":"46","CARLUCET":"46","CASTELNAU MONTRATIER":"46","CAZILLAC":"46","CORN":"46","CORNAC":"46","COUZOU":"46","CUZAC":"46","DURBANS":"46","FELZINS":"46","FRAYSSINET":"46","FRAYSSINET LE GELAT":"46","GINOUILLAC":"46","GORSES":"46","GOUJOUNAC":"46","LES JUNIES":"46","LACAPELLE CABANAC":"46","LACAPELLE MARIVAL":"46","LALBENQUE":"46","LAMOTHE CASSEL":"46","LARAMIERE":"46","LARROQUE TOIRAC":"46","LAURESSES":"46","LAVAL DE CERE":"46","LEYME":"46","LINAC":"46","LUGAGNAC":"46","MARMINIAC":"46","MONTCUQ EN QUERCY BLANC":"46","MONTVALENT":"46","PONTCIRQ":"46","PRENDEIGNES":"46","PUY L EVEQUE":"46","ST CHAMARAND":"46","ST CIRQ SOUILLAGUET":"46","ST MATRE":"46","ST PAUL DE VERN":"46","ST VINCENT RIVE D OLT":"46","TEYSSIEU":"46","TOUR DE FAURE":"46","ST PIERRE LAFEUILLE":"46","AIGUILLON":"47","ALLEZ ET CAZENEUVE":"47","AMBRUS":"47","ARGENTON":"47","BAJAMONT":"47","BARBASTE":"47","BEAUGAS":"47","BOURNEL":"47","BRUGNAC":"47","CASTELLA":"47","CASTELMORON SUR LOT":"47","CASTELNAU SUR GUPIE":"47","CAUBON ST SAUVEUR":"47","CAUDECOSTE":"47","CAVARC":"47","CLAIRAC":"47","LA CROIX BLANCHE":"47","GAVAUDUN":"47","GRAYSSAS":"47","HAUTESVIGNES":"47","JUSIX":"47","LAFOX":"47","LAMONTJOIE":"47","LAUGNAC":"47","LAUSSOU":"47","LEDAT":"47","LEVIGNAC DE GUYENNE":"47","LEYRITZ MONCASSIN":"47","LOUGRATTE":"47","LUSIGNAN PETIT":"47","MASSOULES":"47","MEILHAN SUR GARONNE":"47","MEZIN":"47","MONTETON":"47","MONTIGNAC DE LAUZUN":"47","ESMERY HALLON":"80","ESTREBOEUF":"80","FEUILLERES":"80","FIGNIERES":"80","FINS":"80","FLAUCOURT":"80","FONCHES FONCHETTE":"80","FONTAINE SOUS MONTDIDIER":"80","FORCEVILLE":"80","FOREST L ABBAYE":"80","FOUQUESCOURT":"80","FOURDRINOY":"80","FRAMICOURT":"80","FRESNOY LES ROYE":"80","FRICOURT":"80","FROHEN SUR AUTHIE":"80","GAMACHES":"80","GINCHY":"80","GREBAULT MESNIL":"80","GRECOURT":"80","GROUCHES LUCHUEL":"80","GUIZANCOURT":"80","HALLOY LES PERNOIS":"80","HARDECOURT AUX BOIS":"80","HAVERNAS":"80","HERISSART":"80","HERLEVILLE":"80","HESCAMPS":"80","HORNOY LE BOURG":"80","HYENCOURT LE GRAND":"80","LAMOTTE WARFUSEE":"80","LEALVILLERS":"80","LIGESCOURT":"80","LONGUEVAL":"80","LOUVRECHY":"80","MACHIEL":"80","MAISON PONTHIEU":"80","MARCELCAVE":"80","MAREUIL CAUBERT":"80","MARIEUX":"80","MARLERS":"80","MEAULTE":"80","MERICOURT L ABBE":"80","MERS LES BAINS":"80","LE MESGE":"80","MEZEROLLES":"80","MIRAUMONT":"80","MOLLIENS AU BOIS":"80","MONCHY LAGACHE":"80","MOUFLERS":"80","MOYENCOURT":"80","MOYENCOURT LES POIX":"80","MUILLE VILLETTE":"80","NAMPS MAISNIL":"80","NESLE":"80","NESLETTE":"80","NEUVILLE AU BOIS":"80","ONEUX":"80","OUST MAREST":"80","OVILLERS LA BOISSELLE":"80","PARGNY":"80","PARVILLERS LE QUESNOY":"80","PENDE":"80","PIENNES ONVILLERS":"80","LE PLESSIER ROZAINVILLERS":"80","POEUILLY":"80","PONT NOYELLES":"80","PROUZEL":"80","REGNIERE ECLUSE":"80","RETHONVILLERS":"80","RIBEMONT SUR ANCRE":"80","RIVERY":"80","ROUVROY EN SANTERRE":"80","RUE":"80","SAILLY LE SEC":"80","ST BLIMONT":"80","ST MAULVIS":"80","ST QUENTIN EN TOURMONT":"80","STE SEGREE":"80","SENTELIE":"80","SOYECOURT":"80","SURCAMPS":"80","THENNES":"80","THIEULLOY L ABBAYE":"80","TILLOY LES CONTY":"80","TREUX":"80","TULLY":"80","VAIRE SOUS CORBIE":"80","VAUCHELLES LES DOMART":"80","VAUCHELLES LES QUESNOY":"80","LENTILLAC DU CAUSSE":"46","LIMOGNE EN QUERCY":"46","LUNEGARDE":"46","MARTEL":"46","MAXOU":"46","MONTDOUMERC":"46","MONTLAUZUN":"46","MONTREDON":"46","NADAILLAC DE ROUGE":"46","PAYRIGNAC":"46","PERN":"46","PEYRILLES":"46","PROMILHANES":"46","PUYBRUN":"46","REILHAGUET":"46","LE ROC":"46","ROUFFILHAC":"46","RUEYRES":"46","ST CHELS":"46","ST CIRQ MADELON":"46","ST JEAN LESPINASSE":"46","ST MARTIN LABOUVAL":"46","ST MEDARD NICOURBY":"46","SALVIAC":"46","SENAILLAC LATRONQUIERE":"46","SENIERGUES":"46","SOUILLAC":"46","SOULOMES":"46","SOUSCEYRAC EN QUERCY":"46","STRENQUELS":"46","THEMINETTES":"46","VARAIRE":"46","VIAZAC":"46","VIRE SUR LOT":"46","MAYRAC":"46","BESSONIES":"46","ST JEAN LAGINESTE":"46","AGME":"47","ANTAGNAC":"47","ASTAFFORT":"47","BALEYSSAGUES":"47","BEAUZIAC":"47","BLAYMONT":"47","BOUGLON":"47","CALIGNAC":"47","CASTELJALOUX":"47","CASTELNAUD DE GRATECAMBE":"47","CAUMONT SUR GARONNE":"47","CAUZAC":"47","CAZIDEROQUE":"47","CONDEZAYGUES":"47","COULX":"47","DAMAZAN":"47","DOLMAYRAC":"47","DURANCE":"47","ENGAYRAC":"47","ESCASSEFORT":"47","ESCLOTTES":"47","ESPIENS":"47","FERRENSAC":"47","FOULAYRONNES":"47","FOURQUES SUR GARONNE":"47","FRANCESCAS":"47","FREGIMONT":"47","GREZET CAVAGNAN":"47","LACAPELLE BIRON":"47","LACAUSSADE":"47","LACEPEDE":"47","LAFITTE SUR LOT":"47","LAGUPIE":"47","LALANDUSSE":"47","LANNES":"47","LAPERCHE":"47","LAROQUE TIMBAUT":"47","MARMANDE":"47","MARMONT PACHAS":"47","MONCAUT":"47","MONCRABEAU":"47","MONGAILLARD":"47","MONHEURT":"47","PAILLOLES":"47","HANTAY":"59","HARDIFORT":"59","HAVELUY":"59","HAVERSKERQUE":"59","HEM":"59","HERGNIES":"59","HON HERGIES":"59","HORDAIN":"59","HOUPLIN ANCOISNE":"59","HOUPLINES":"59","INCHY":"59","JEUMONT":"59","LAMBERSART":"59","LECLUSE":"59","LEDERZEELE":"59","LEERS":"59","LINSELLES":"59","LOOS":"59","LA MADELEINE":"59","MAING":"59","MAIRIEUX":"59","MARETZ":"59","MILLONFOSSE":"59","MONCHAUX SUR ECAILLON":"59","MONCHECOURT":"59","MONTRECOURT":"59","MOUCHIN":"59","MOUVAUX":"59","NEUVILLE EN AVESNOIS":"59","NEUVILLY":"59","NOYELLES LES SECLIN":"59","OXELAERE":"59","PERONNE EN MELANTOIS":"59","RENESCURE":"59","REUMONT":"59","RONCHIN":"59","ROUBAIX":"59","SAILLY LEZ CAMBRAI":"59","SAINGHIN EN MELANTOIS":"59","ST ANDRE LEZ LILLE":"59","ST AUBERT":"59","ST AYBERT":"59","ST WAAST":"59","SARS POTERIES":"59","SEMOUSIES":"59","SEPMERIES":"59","STAPLE":"59","TEMPLEMARS":"59","THUN ST MARTIN":"59","TRESSIN":"59","TRITH ST LEGER":"59","TROISVILLES":"59","VENDEGIES SUR ECAILLON":"59","VERCHAIN MAUGRE":"59","VIEUX BERQUIN":"59","VIEUX CONDE":"59","VILLERS AU TERTRE":"59","VILLERS PLOUICH":"59","VILLERS SIRE NICOLE":"59","WALLERS EN FAGNE":"59","WEMAERS CAPPEL":"59","WEST CAPPEL":"59","WINNEZEELE":"59","WYLDER":"59","ZEGERSCAPPEL":"59","ZERMEZEELE":"59","ACY EN MULTIEN":"60","AMY":"60","ANGY":"60","ANTHEUIL PORTES":"60","AVRIGNY":"60","BABOEUF":"60","BAZICOURT":"60","BEAUGIES SOUS BOIS":"60","BEAUMONT LES NONAINS":"60","BERTHECOURT":"60","BETHANCOURT EN VALOIS":"60","BIENVILLE":"60","BLAINCOURT LES PRECY":"60","BLARGIES":"60","BLICOURT":"60","BLINCOURT":"60","BOISSY FRESNOY":"60","BOUCONVILLERS":"60","BOULLARRE":"60","BOURY EN VEXIN":"60","BRASSEUSE":"60","BULLES":"60","CAMBRONNE LES RIBECOURT":"60","CANLY":"60","CANNECTANCOURT":"60","CARLEPONT":"60","CERNOY":"60","JUSTINE HERBIGNY":"08","LANDRICHAMPS":"08","INAUMONT":"08","HAUDRECY":"08","IMECOURT":"08","LONGWE":"08","LONNY":"08","LIRY":"08","SIGNY LE PETIT":"08","SIGNY MONTLIBERT":"08","TOULIGNY":"08","TETAIGNE":"08","SUGNY":"08","TARZY":"08","SURY":"08","ST CLEMENT A ARNES":"08","RAUCOURT ET FLABA":"08","LA SABOTTERIE":"08","ST MOREL":"08","REVIN":"08","NOYERS PONT MAUGIS":"08","NEUVILLE LES THIS":"08","NOUVION SUR MEUSE":"08","RAILLICOURT":"08","POIX TERRON":"08","RANCENNES":"08","PAUVRES":"08","OCHES":"08","GIRONDELLE":"08","HAM LES MOINES":"08","LA GRANDVILLE":"08","GUE D HOSSUS":"08","FLEIGNEUX":"08","FLEVILLE":"08","GERMONT":"08","GLAIRE":"08","FROMY":"08","FUMAY":"08","NEUVILLE LEZ BEAULIEU":"08","MENIL LEPINOIS":"08","LES MAZURES":"08","MARANWEZ":"08","MAZERNY":"08","MOURON":"08","MANRE":"08","MARBY":"08","TOURCELLES CHAUMONT":"08","VILLERS CERNAY":"08","VAUX VILLAINE":"08","TOURNAVAUX":"08","LES BORDES SUR ARIZE":"09","CADARCET":"09","COUSSA":"09","CANTE":"09","ERP":"09","SENTENAC DE SEROU":"09","TAURIGNAN CASTET":"09","TREMOULET":"09","TOURTROL":"09","VARILHES":"09","SUZAN":"09","L HOSPITALET PRES L ANDORRE":"09","ILLIER ET LARAMADE":"09","MAUVEZIN DE PRAT":"09","MERCUS GARRABET":"09","LAROQUE D OLMES":"09","LOUBIERES":"09","LERCOUL":"09","LORDAT":"09","GALEY":"09","GUDAS":"09","ST JEAN DU CASTILLONNAIS":"09","ST JULIEN DE GRAS CAPOU":"09","ST VICTOR ROUZAUD":"09","LORP SENTARAILLE":"09","QUERIGUT":"09","SENCONAC":"09","RAISSAC":"09","MONTIGNAC TOUPINERIE":"47","NERAC":"47","NOMDIEU":"47","PINEL HAUTERIVE":"47","POMPOGNE":"47","POUDENAS":"47","PUCH D AGENAIS":"47","REAUP LISSE":"47","LA REUNION":"47","STE COLOMBE EN BRUILHOIS":"47","ST MAURIN":"47","ST NICOLAS DE LA BALERME":"47","ST PIERRE DE BUZET":"47","SAMAZAN":"47","SAUMEJAN":"47","SAUMONT":"47","SAUVAGNAS":"47","LA SAUVETAT SUR LEDE":"47","SEMBAS":"47","SOS":"47","SOUMENSAC":"47","LE TEMPLE SUR LOT":"47","TOURLIAC":"47","TRENTELS":"47","VILLEFRANCHE DU QUEYRAN":"47","VILLENEUVE SUR LOT":"47","VIRAZEIL":"47","ARZENC D APCHER":"48","LES MONTS VERTS":"48","PIED DE BORNE":"48","BASSURELS":"48","LE BLEYMARD":"48","LA CANOURGUE":"48","CHANAC":"48","CHAUCHAILLES":"48","LA CHAZE DE PEYRE":"48","CUBIERES":"48","CUBIERETTES":"48","ISPAGNAC":"48","LAVAL DU TARN":"48","MALBOUZON":"48","MARVEJOLS":"48","MENDE":"48","MOISSAC VALLEE FRANCAISE":"48","BOURGS SUR COLAGNE":"48","PONT DE MONTVERT SUD MONT LOZERE":"48","POURCHARESSES":"48","ST ALBAN SUR LIMAGNOLE":"48","MAS ST CHELY":"48","ST FLOUR DE MERCOIRE":"48","ST HILAIRE DE LAVIT":"48","ST LEGER DU MALZIEU":"48","ST MARTIN DE LANSUSCLE":"48","ST MICHEL DE DEZE":"48","ST SAUVEUR DE PEYRE":"48","LES SALCES":"48","SERVERETTE":"48","BAUGE EN ANJOU":"49","BEAUFORT EN ANJOU":"49","BEAUPREAU EN MAUGES":"49","BEHUARD":"49","BOUILLE MENARD":"49","BRIGNE":"49","BRIOLLAY":"49","BRISSARTHE":"49","CERNUSSON":"49","CHALLAIN LA POTHERIE":"49","CHALONNES SUR LOIRE":"49","CHAMBELLAY":"49","CHAMPIGNE":"49","CHENILLE CHAMPTEUSSE":"49","OREE D ANJOU":"49","CHAZE HENRY":"49","CHEMILLE EN ANJOU":"49","VERMANDOVILLERS":"80","VIGNACOURT":"80","VILLECOURT":"80","VILLERS AUX ERABLES":"80","VILLERS BRETONNEUX":"80","VILLERS CAMPSART":"80","VILLERS CARBONNEL":"80","VILLERS LES ROYE":"80","VILLERS TOURNELLE":"80","VISMES":"80","VITZ SUR AUTHIE":"80","WARGNIES":"80","WOINCOURT":"80","ALBAN":"81","ALBI":"81","AMARENS":"81","AMBRES":"81","ANDOUQUE":"81","L ABERGEMENT DE VAREY":"01","ASNIERES SUR SAONE":"01","BELLIGNAT":"01","ARMIX":"01","ST PIERRE ST JEAN":"07","ST MAURICE EN CHALENCON":"07","ST PIERRE LA ROCHE":"07","ST PAUL LE JEUNE":"07","ST MELANY":"07","THORRENC":"07","SECHERAS":"07","ST REMEZE":"07","SILHAC":"07","LES VANS":"07","LES GRANDES ARMOISES":"08","LA VOULTE SUR RHONE":"07","VILLENEUVE DE BERG":"07","TOURNON SUR RHONE":"07","VALVIGNERES":"07","ARNICOURT":"08","VINZIEUX":"07","AOUSTE":"08","VAGNAS":"07","MALARCE SUR LA THINES":"07","LACHAPELLE SOUS AUBENAS":"07","LOUBARESSE":"07","MEZILHAC":"07","PEAUGRES":"07","PEYRAUD":"07","LARNAS":"07","MAUVES":"07","CHARLEVILLE MEZIERES":"08","DOUMELY BEGNY":"08","CONTREUVE":"08","CHARBOGNE":"08","DEVILLE":"08","DOMMERY":"08","CHOOZ":"08","BELLEVILLE ET CHATILLON SUR BAR":"08","BANOGNE RECOUVRANCE":"08","BALAIVES ET BUTZ":"08","BAR LES BUZANCY":"08","BAZEILLES":"08","AUSSONCE":"08","ASFELD":"08","AURE":"08","BELVAL BOIS DES DAMES":"08","BOSSUS LES RUMIGNY":"08","BLANCHEFOSSE ET BAY":"08","BRIEULLES SUR BAR":"08","BOUVELLEMONT":"08","LA BERLIERE":"08","BRIQUENAY":"08","CAUROY":"08","FOISCHES":"08","FEPIN":"08","ST AGREVE":"07","STE MARGUERITE LAFIGERE":"07","ST ETIENNE DE LUGDARES":"07","ST JULIEN EN ST ALBAN":"07","ST MARTIN DE VALAMAS":"07","ST GINEIS EN COIRON":"07","ST JOSEPH DES BANCS":"07","ST JEURE D ANDAURE":"07","ST GENEST LACHAMP":"07","COLOMBIER LE VIEUX":"07","COLOMBIER LE JEUNE":"07","PARDAILLAN":"47","PINDERES":"47","POMPIEY":"47","PONT DU CASSE":"47","PORT STE MARIE":"47","POUSSIGNAC":"47","PUYMICLAN":"47","ROMESTAING":"47","STE COLOMBE DE DURAS":"47","ST JEAN DE DURAS":"47","STE MAURE DE PEYRIAC":"47","ST PIERRE SUR DROPT":"47","ST VITE":"47","SAUVETERRE LA LEMANCE":"47","TONNEINS":"47","TREMONS":"47","VERTEUIL D AGENAIS":"47","VILLEBRAMAR":"47","ALTIER":"48","ANTRENAS":"48","AUMONT AUBRAC":"48","BALSIEGES":"48","BARRE DES CEVENNES":"48","LA BASTIDE PUYLAURENT":"48","CHAMBON LE CHATEAU":"48","CHASTANIER":"48","CHAUDEYRAC":"48","CHAULHAC":"48","ESCLANEDES":"48","LA FAGE MONTIVERNOUX":"48","LA FAGE ST JULIEN":"48","FAU DE PEYRE":"48","FLORAC TROIS RIVIERES":"48","FONTANS":"48","JAVOLS":"48","LAJO":"48","LE MALZIEU VILLE":"48","LE MASSEGROS":"48","MOLEZON":"48","NAUSSAC FONTANES":"48","NOALHAC":"48","LA PANOUSE":"48","LE RECOUX":"48","RIEUTORT DE RANDON":"48","ST DENIS EN MARGERIDE":"48","STE ENIMIE":"48","ST GEORGES DE LEVEJAC":"48","ST GERMAIN DU TEIL":"48","ST JEAN LA FOUILLOUSE":"48","CANS ET CEVENNES":"48","TRELANS":"48","VEBRON":"48","TUFFALUN":"49","AUVERSE":"49","AVIRE":"49","BARACE":"49","BLAISON ST SULPICE":"49","BLOU":"49","BRISSAC QUINCE":"49","CANDE":"49","CANTENAY EPINARD":"49","CHANTELOUP LES BOIS":"49","CHAUDEFONDS SUR LAYON":"49","CORNILLE LES CAVES":"49","CORZE":"49","LE COUDRAY MACOUARD":"49","DOUE LA FONTAINE":"49","DURTAL":"49","LES BOIS D ANJOU":"49","GENNES VAL DE LOIRE":"49","CHAUMONT EN VEXIN":"60","CHOISY AU BAC":"60","CINQUEUX":"60","CORBEIL CERF":"60","CRAMOISY":"60","CRAPEAUMESNIL":"60","CRESSONSACQ":"60","CRISOLLES":"60","CROUTOY":"60","CUIGY EN BRAY":"60","CUISE LA MOTTE":"60","DELINCOURT":"60","DIEUDONNE":"60","DOMFRONT":"60","ELENCOURT":"60","ERAGNY SUR EPTE":"60","ERCUIS":"60","ESPAUBOURG":"60","ESQUENNOY":"60","ETOUY":"60","LE FAYEL":"60","FLEURINES":"60","FONTAINE BONNELEAU":"60","FOUILLEUSE":"60","FOULANGUES":"60","FOUQUENIES":"60","FRESNIERES":"60","LE GALLET":"60","GOURCHELLES":"60","GRANDVILLERS AUX BOIS":"60","HANNACHES":"60","HAUTE EPINE":"60","HERCHIES":"60","LA HERELLE":"60","IVORS":"60","LABOSSE":"60","LAIGNEVILLE":"60","LANNOY CUILLERE":"60","LARBROYE":"60","LATTAINVILLE":"60","LAVERSINES":"60","LAVILLETERTRE":"60","LIANCOURT":"60","LIHUS":"60","LITZ":"60","LUCHY":"60","MACHEMONT":"60","MAREUIL LA MOTTE":"60","MELICOCQ":"60","LE MESNIL EN THELLE":"60","LE MESNIL ST FIRMIN":"60","MONCHY ST ELOI":"60","MONTGERAIN":"60","MONTIERS":"60","MONTREUIL SUR BRECHE":"60","MOUY":"60","NEUVILLE BOSC":"60","LA NEUVILLE GARNIER":"60","LA NEUVILLE SUR OUDEUIL":"60","NIVILLERS":"60","NOURARD LE FRANC":"60","OMECOURT":"60","ORVILLERS SOREL":"60","OUDEUIL":"60","OURSEL MAISON":"60","PIERREFONDS":"60","PISSELEU":"60","LE PLOYRON":"60","PONT STE MAXENCE":"60","LE QUESNEL AUBRY":"60","QUINCAMPOIX FLEUZY":"60","QUINQUEMPOIX":"60","RESSONS SUR MATZ":"60","ROSOY EN MULTIEN":"60","ROTHOIS":"60","ROUSSELOY":"60","ROUVILLERS":"60","ROUVRES EN MULTIEN":"60","ROYAUCOURT":"60","SACY LE GRAND":"60","SACY LE PETIT":"60","ST CREPIN IBOUVILLERS":"60","SAINTINES":"60","ST LEU D ESSERENT":"60","ST MARTIN AUX BOIS":"60","ST MARTIN LE NOEUD":"60","ST MARTIN LONGUEAU":"60","ST PIERRE ES CHAMPS":"60","ST VALERY":"60","PAMIERS":"09","REGAT":"09","LES NOES PRES TROYES":"10","MONTREUIL SUR BARSE":"10","NOE LES MALLETS":"10","MESNIL ST LOUP":"10","MONTAILLOU":"09","MIJANES":"09","ORGIBET":"09","OUST":"09","VILLENEUVE DU PAREAGE":"09","BLAINCOURT SUR AUBE":"10","AVANT LES RAMERUPT":"10","BERCENAY LE HAYER":"10","BERCENAY EN OTHE":"10","ARCIS SUR AUBE":"10","BETIGNICOURT":"10","BOSSANCOURT":"10","BERULLE":"10","AVREUIL":"10","CRESPY LE NEUF":"10","FONTAINE MACON":"10","COURTERANGES":"10","CRESANTIGNES":"10","LES CROUTES":"10","COUVIGNON":"10","FRALIGNES":"10","FONTETTE":"10","FULIGNY":"10","COURCELLES SUR VOIRE":"10","BRIENNE LA VIEILLE":"10","CHAPELLE VALLON":"10","CHAUCHIGNY":"10","COURCEROY":"10","CHAUDREY":"10","CHANNES":"10","COCLOIS":"10","PERTHES LES BRIENNE":"10","PARS LES CHAVANGES":"10","PARS LES ROMILLY":"10","PRECY ST MARTIN":"10","PREMIERFAIT":"10","PRASLIN":"10","VILLE SUR RETOURNE":"08","VRIGNE AUX BOIS":"08","VILLERS SEMEUSE":"08","YVERNAUMONT":"08","ARROUT":"09","AXIAT":"09","ASCOU":"09","ALEU":"09","LA BASTIDE DE BOUSIGNAC":"09","LA BASTIDE DE LORDAT":"09","LA BASTIDE DU SALAT":"09","LA BASTIDE DE SEROU":"09","BONAC IRAZEIN":"09","BETHMALE":"09","BENAIX":"09","BAULOU":"09","BELLOC":"09","RAMERUPT":"10","ST ANDRE LES VERGERS":"10","LA RIVIERE DE CORPS":"10","PUITS ET NUISEMENT":"10","ROSNAY L HOPITAL":"10","STE MAURE":"10","ST LEGER SOUS MARGERIE":"10","ST PARRES AUX TERTRES":"10","ST NABORD SUR AUBE":"10","ST OULPH":"10","TROUANS":"10","VERPILLIERES SUR OURCE":"10","VILLENAUXE LA GRANDE":"10","VILLE SOUS LA FERTE":"10","SOLIGNY LES ETANGS":"10","LA VILLE AUX BOIS":"10","VIAPRES LE PETIT":"10","SOULIGNY":"10","VILLIERS SOUS PRASLIN":"10","CLERE SUR LAYON":"49","COMBREE":"49","COURLEON":"49","DAUMERAY":"49","DENEZE SOUS DOUE":"49","DISTRE":"49","L HOTELLERIE DE FLEE":"49","JARZE VILLAGES":"49","LE LION D ANGERS":"49","LOIRE":"49","MAZE MILON":"49","MONTREUIL JUIGNE":"49","MONTREUIL BELLAY":"49","MONTREUIL SUR MAINE":"49","MONTREVAULT SUR EVRE":"49","MONTSOREAU":"49","MOZE SUR LOUET":"49","NOELLET":"49","NUAILLE":"49","PASSAVANT SUR LAYON":"49","LA PLAINE":"49","MAUGES SUR LOIRE":"49","LES PONTS DE CE":"49","POUANCE":"49","ST AUGUSTIN DES BOIS":"49","ST CLEMENT DE LA PLACE":"49","ST CLEMENT DES LEVEES":"49","ST JEAN DE LA CROIX":"49","ST JEAN DES MAUVRETS":"49","SEVREMOINE":"49","LOIRE AUTHION":"49","SAUMUR":"49","SEICHES SUR LE LOIR":"49","BELLEVIGNE EN LAYON":"49","TOUTLEMONDE":"49","ERDRE EN ANJOU":"49","LYS HAUT LAYON":"49","VILLEBERNIER":"49","VILLEVEQUE":"49","YZERNAY":"49","AGNEAUX":"50","AMIGNY":"50","ANCTEVILLE":"50","ANCTOVILLE SUR BOSCQ":"50","AUCEY LA PLAINE":"50","AUDOUVILLE LA HUBERT":"50","AUXAIS":"50","AZEVILLE":"50","BARENTON":"50","BESLON":"50","BEUZEVILLE LA BASTILLE":"50","BLOSVILLE":"50","BRANVILLE HAGUE":"50","BREHAL":"50","BRICQUEBEC EN COTENTIN":"50","BRICQUEVILLE LA BLOUETTE":"50","BRILLEVAST":"50","CAMETOURS":"50","CARENTAN LES MARAIS":"50","LA CHAPELLE UREE":"50","CHASSEGUEY":"50","CHERBOURG EN COTENTIN":"50","COLOMBY":"50","CONDE SUR VIRE":"50","VICQ SUR MER":"50","LA CROIX AVRANCHIN":"50","CROSVILLE SUR DOUVE":"50","LE DEZERT":"50","DONVILLE LES BAINS":"50","DOVILLE":"50","ECULLEVILLE":"50","FEUGERES":"50","FOLLIGNY":"50","FRESVILLE":"50","GAVRAY":"50","GONNEVILLE LE THEIL":"50","GOUVILLE SUR MER":"50","GENESTELLE":"07","CHAMBONAS":"07","BROSSAINC":"07","DEVESSET":"07","BURZET":"07","INTRES":"07","ETREPIGNY":"08","L ECAILLE":"08","L ECHELLE":"08","ESTREBAY":"08","EXERMONT":"08","ETALLE":"08","DOUZY":"08","ECLY":"08","FRAILLICOURT":"08","HAGNICOURT":"08","GRANDHAM":"08","GRUYERES":"08","LE FRETY":"08","HARCY":"08","LA NEUVILLE EN TOURNE A FUY":"08","LA NEUVILLE AUX JOUTES":"08","MURTIN ET BOGNY":"08","NOVION PORCIEN":"08","NEUVILLE DAY":"08","NEUFMAISON":"08","NEUVIZY":"08","VILLERS DEVANT MOUZON":"08","VILLERS LE TILLEUL":"08","VIEUX LES ASFELD":"08","VAUX LES RUBIGNY":"08","VAUX LES MOUZON":"08","THIN LE MOUTIER":"08","VRIGNE MEUSE":"08","TOURTERON":"08","MATTON ET CLEMENCY":"08","MONTIGNY SUR VENCE":"08","MONTCY NOTRE DAME":"08","MAUBERT FONTAINE":"08","LE MONT DIEU":"08","MONTMEILLANT":"08","MONTGON":"08","MOIRY":"08","ROUVROY SUR AUDRY":"08","POILCOURT SYDNEY":"08","RILLY SUR AISNE":"08","ST MENGES":"08","RUBIGNY":"08","ROIZY":"08","OMONT":"08","ST QUENTIN LE PETIT":"08","SAVIGNY SUR AISNE":"08","SAULT LES RETHEL":"08","SAULT ST REMY":"08","SOMMERANCE":"08","SOMMAUTHE":"08","SECHAULT":"08","TAGNON":"08","SEUIL":"08","SEDAN":"08","CARLA DE ROQUEFORT":"09","ENCOURTIECH":"09","ENGOMER":"09","FORNEX":"09","FOIX":"09","BEDEILHAC ET AYNAT":"09","WILLIERS":"08","BALACET":"09","WAGNON":"08","BAGERT":"09","AUZAT":"09","YONCQ":"08","LEFFINCOURT":"08","HOUDILCOURT":"08","LOGNY BOGNY":"08","HARRICOURT":"08","JONVAL":"08","JANDUN":"08","HAYBES":"08","HAULME":"08","LUMES":"08","CAZALS DES BAYLES":"09","GENNETEIL":"49","GREZ NEUVILLE":"49","GRUGE L HOPITAL":"49","HUILLE":"49","LE LOUROUX BECONNAIS":"49","MEIGNE LE VICOMTE":"49","LONGUENEE EN ANJOU":"49","MEON":"49","MONTIGNE LES RAIRIES":"49","NEUILLE":"49","PARCAY LES PINS":"49","ST CYR EN BOURG":"49","ST LAMBERT LA POTHERIE":"49","ST LEGER SOUS CHOLET":"49","ST REMY LA VARENNE":"49","VERRIERES EN ANJOU":"49","SARRIGNE":"49","SAULGE L HOPITAL":"49","SEGRE":"49","SOEURDRES":"49","SOULAINES SUR AUBANCE":"49","SOULAIRE ET BOURG":"49","LA TESSOUALLE":"49","TRELAZE":"49","TREMENTINES":"49","VARENNES SUR LOIRE":"49","VARRAINS":"49","VERNANTES":"49","VERRIE":"49","VIVY":"49","ANNEVILLE EN SAIRE":"50","ANNOVILLE":"50","AUDERVILLE":"50","BESNEVILLE":"50","BIEVILLE":"50","BINIVILLE":"50","BIVILLE":"50","LA BLOUTIERE":"50","BOISYVON":"50","BRIX":"50","BROUAINS":"50","CAMPROND":"50","CARQUEBUT":"50","CATTEVILLE":"50","CAVIGNY":"50","CATZ":"50","CERENCES":"50","LA CHAISE BAUDOUIN":"50","LES CHAMPS DE LOSQUE":"50","LA CHAPELLE CECELIN":"50","CHERENCE LE HERON":"50","CHERENCE LE ROUSSEL":"50","LA COLOMBE":"50","CONTRIERES":"50","DANGY":"50","DRAGEY RONTHON":"50","DUCEY LES CHERIS":"50","FLOTTEMANVILLE HAGUE":"50","LE FRESNE PORET":"50","GATTEVILLE LE PHARE":"50","GOLLEVILLE":"50","GRANVILLE":"50","GUEHEBERT":"50","HAMELIN":"50","HARDINVAST":"50","LA HAYE BELLEFOND":"50","HELLEVILLE":"50","LE HOMMET D ARTHENAY":"50","HUISNES SUR MER":"50","ISIGNY LE BUAT":"50","LA LANDE D AIROU":"50","LIESVILLE SUR DOUVE":"50","MONTSENELLE":"50","LE LUOT":"50","LE MESNIL ADELEE":"50","SARCUS":"60","SARNOIS":"60","SEMPIGNY":"60","SERY MAGNEVAL":"60","SILLY LE LONG":"60","TALMONTIERS":"60","THERINES":"60","THIEULOY ST ANTOINE":"60","THOUROTTE":"60","TOURLY":"60","TROISSEREUX":"60","VANDELICOURT":"60","VARESNES":"60","VARINFROY":"60","VAUDANCOURT":"60","LE VAUMAIN":"60","VAUMOISE":"60","VENDEUIL CAPLY":"60","VERBERIE":"60","VERDEREL LES SAUQUEUSE":"60","VERNEUIL EN HALATTE":"60","VILLENEUVE LES SABLONS":"60","LA VILLENEUVE SOUS THURY":"60","VILLERS SOUS ST LEU":"60","VILLERS SUR AUCHY":"60","VROCOURT":"60","ALENCON":"61","ALMENECHES":"61","ATHIS VAL DE ROUVRE":"61","AUNOU SUR ORNE":"61","AVERNES SOUS EXMES":"61","BIZOU":"61","BOECE":"61","COUR MAUGIS SUR HUISNE":"61","LE BOUILLON":"61","BURSARD":"61","CAMEMBERT":"61","CEAUCE":"61","CERISE":"61","CHAILLOUE":"61","RIVES D ANDAINE":"61","LA CHAPELLE PRES SEES":"61","LA CHAPELLE SOUEF":"61","LA CHAPELLE VIEL":"61","CIRAL":"61","COMBLOT":"61","COULMER":"61","COURGEON":"61","COURMENIL":"61","CRULAI":"61","DOMFRONT EN POIRAIE":"61","ECHALOU":"61","ECORCEI":"61","LA FERTE EN OUCHE":"61","LA FERTE MACE":"61","LA FRESNAIE FAYEL":"61","GAPREE":"61","GODISSON":"61","LE GRAIS":"61","LE GUE DE LA CHAINE":"61","JUVIGNY VAL D ANDAINE":"61","LA LANDE PATRY":"61","LA LANDE ST SIMEON":"61","LONGNY LES VILLAGES":"61","LONLAY L ABBAYE":"61","MACE":"61","LE MELE SUR SARTHE":"61","LE MENIL DE BRIOUZE":"61","MENIL ERREUX":"61","LE MENIL SCELLEUR":"61","LA MESNIERE":"61","MONTABARD":"61","MONTREUIL AU HOULME":"61","NEUVY AU HOULME":"61","PUTANGES LE LAC":"61","ECOUVES":"61","RANES":"61","REMALARD EN PERCHE":"61","LA ROCHE MABILE":"61","ST AGNAN SUR SARTHE":"61","ST AUBIN DE BONNEVAL":"61","VIVIERS SUR ARTAUT":"10","VILLETTE SUR AUBE":"10","VILLY EN TRODES":"10","VILLE SUR TERRE":"10","VIREY SOUS BAR":"10","ARGELIERS":"11","VOUGREY":"10","VOSNON":"10","AJAC":"11","CARCASSONNE":"11","CAMPAGNA DE SAULT":"11","CAMPS SUR L AGLY":"11","LE BOUSQUET":"11","BARAIGNE":"11","BIZANET":"11","BELPECH":"11","CASCASTEL DES CORBIERES":"11","CAUX ET SAUZENS":"11","CENNE MONESTIES":"11","CASTELNAUDARY":"11","COMIGNE":"11","CASTANS":"11","CITOU":"11","CEPIE":"11","CAVES":"11","CONILHAC DE LA MONTAGNE":"11","COMUS":"11","NANTERRE":"92","SURESNES":"92","VAUCRESSON":"92","VILLE D AVRAY":"92","VILLENEUVE LA GARENNE":"92","BAGNOLET":"93","BOBIGNY":"93","CLICHY SOUS BOIS":"93","LA COURNEUVE":"93","DRANCY":"93","DUGNY":"93","EPINAY SUR SEINE":"93","GOURNAY SUR MARNE":"93","MONTFERMEIL":"93","NEUILLY PLAISANCE":"93","PIERREFITTE SUR SEINE":"93","ROMAINVILLE":"93","STAINS":"93","ARCUEIL":"94","BRY SUR MARNE":"94","CHOISY LE ROI":"94","IVRY SUR SEINE":"94","LE PLESSIS TREVISE":"94","ST MANDE":"94","THIAIS":"94","VALENTON":"94","VILLECRESNES":"94","VILLEJUIF":"94","ARGENTEUIL":"95","ARNOUVILLE":"95","ARRONVILLE":"95","ARTHIES":"95","ATTAINVILLE":"95","BAILLET EN FRANCE":"95","BEAUCHAMP":"95","BEAUMONT SUR OISE":"95","BEZONS":"95","BREANCON":"95","CHAUMONTEL":"95","CHAUVRY":"95","CHERENCE":"95","CORMEILLES EN PARISIS":"95","ERAGNY":"95","EZANVILLE":"95","LA FRETTE SUR SEINE":"95","FROUVILLE":"95","GUIRY EN VEXIN":"95","HEDOUVILLE":"95","HODENT":"95","JAGNY SOUS BOIS":"95","JOUY LE MOUTIER":"95","MARINES":"95","MARLY LA VILLE":"95","LE MESNIL AUBRY":"95","MONTLIGNON":"95","NEUVILLE SUR OISE":"95","LE PERCHAY":"95","PERSAN":"95","PIERRELAYE":"95","GRAIGNES MESNIL ANGOT":"50","HAMBYE":"50","HAUTEVILLE SUR MER":"50","HAUTEVILLE LA GUICHARD":"50","LA HAYE PESNEL":"50","HUDIMESNIL":"50","JUILLEY":"50","LAPENTY":"50","LINGEARD":"50","LA LUZERNE":"50","MAGNEVILLE":"50","MARCEY LES GREVES":"50","MARIGNY LE LOZON":"50","LE MESNIL":"50","LE MESNIL AUBERT":"50","LE MESNILBUS":"50","LE MESNIL GARNIER":"50","LE MESNIL HERMAN":"50","MONTBRAY":"50","MONTJOIE ST MARTIN":"50","MONTPINCHON":"50","MONTSURVENT":"50","MORSALINES":"50","MORTAIN BOCAGE":"50","MUNEVILLE LE BINGARD":"50","NEGREVILLE":"50","NEUFMESNIL":"50","NEUVILLE AU PLAIN":"50","NEUVILLE EN BEAUMONT":"50","NOTRE DAME DE CENILLY":"50","NOUAINVILLE":"50","OCTEVILLE L AVENEL":"50","PERCY EN NORMANDIE":"50","LA PERNELLE":"50","PONTORSON":"50","PRECEY":"50","QUETTREVILLE SUR SIENNE":"50","RAUVILLE LA PLACE":"50","RAVENOVILLE":"50","ST AUBIN DE TERREGATTE":"50","ST AUBIN DU PERRON":"50","ST CLAIR SUR L ELLE":"50","ST CLEMENT RANCOUDRAY":"50","ST FLOXEL":"50","ST FROMOND":"50","ST GERMAIN D ELLE":"50","ST HILAIRE DU HARCOUET":"50","ST JEAN DE DAYE":"50","ST JEAN D ELLE":"50","ST JEAN DES CHAMPS":"50","ST LO D OURVILLE":"50","ST MARTIN D AUDOUVILLE":"50","CHAULIEU":"50","ST MICHEL DE LA PIERRE":"50","ST PIERRE LANGERS":"50","BOURGVALLEES":"50","ST SAUVEUR LENDELIN":"50","ST SEBASTIEN DE RAIDS":"50","ST SENIER DE BEUVRON":"50","SARTILLY BAIE BOCAGE":"50","SIOUVILLE HAGUE":"50","SOTTEVILLE":"50","LE TEILLEUL":"50","TESSY BOCAGE":"50","TONNEVILLE":"50","TORIGNY LES VILLES":"50","TRELLY":"50","TROISGOTS":"50","VALCANVILLE":"50","VAUDREVILLE":"50","VERNIX":"50","LAREE":"32","CHATEAU VERDUN":"09","CAZAUX":"09","BUZAN":"09","BEZAC":"09","COS":"09","FOUGAX ET BARRINEUF":"09","GOULIER":"09","GENAT":"09","GABRE":"09","COURTRIZY ET FUSSIGNY":"02","CLERMONT LES FERMES":"02","LA CROIX SUR OURCQ":"02","CONDE SUR SUIPPE":"02","CONDE EN BRIE":"02","CHEVREGNY":"02","CONNIGIS":"02","CONDREN":"02","COINGT":"02","CHIGNY":"02","CAILLOUEL CREPIGNY":"02","BILLY SUR OURCQ":"02","BOIS LES PARGNY":"02","BUCY LES CERNY":"02","BOURG ET COMIN":"02","BUCY LE LONG":"02","BONNESVALYN":"02","BURELLES":"02","BRENELLE":"02","BRAINE":"02","BETHANCOURT EN VAUX":"02","BILLY SUR AISNE":"02","BERNY RIVIERE":"02","BELLENGLISE":"02","BERTRICOURT":"02","BEUVARDES":"02","BERRIEUX":"02","BERLISE":"02","BEAUME":"02","CAMELIN":"02","CELLES LES CONDE":"02","CERNY LES BUCY":"02","CAULAINCOURT":"02","CHAUDARDES":"02","CHAVIGNON":"02","CHASSEMY":"02","CHAUNY":"02","FONTAINE LES CLERCS":"02","FRESSANCOURT":"02","FIEULAINE":"02","BELMONT LUTHEZIEU":"01","CHATILLON SUR CHALARONNE":"01","CHAVANNES SUR SURAN":"01","CHEIGNIEU LA BALME":"01","CHANOZ CHATENAY":"01","BENONCES":"01","BELLEY":"01","CHALEY":"01","HAUTECOURT ROMANECHE":"01","INJOUX GENISSIAT":"01","JASSANS RIOTTIER":"01","INNIMOND":"01","LABALME":"01","IZENAVE":"01","LAGNIEU":"01","JOYEUX":"01","ILLIAT":"01","IZIEU":"01","MONTLUEL":"01","OYONNAX":"01","NIVOLLET MONTGRIFFON":"01","MONTREAL LA CLUSE":"01","MISERIEUX":"01","MONTCEAUX":"01","NANTUA":"01","NEYRON":"01","ORNEX":"01","CURCIAT DONGALON":"01","CHEZERY FORENS":"01","CORVEISSIAT":"01","CURTAFOND":"01","GROISSIAT":"01","GUEREINS":"01","ECHALLON":"01","CLEYZIEU":"01","EVOSGES":"01","ARMENTIERES SUR OURCQ":"02","ANGUILCOURT LE SART":"02","LE MESNIL AMAND":"50","LE MESNIL AMEY":"50","LE MESNIL AU VAL":"50","LE MESNILLARD":"50","LE MESNIL RAINFRAY":"50","LE MESNIL ROGUES":"50","LES MOITIERS EN BAUPTOIS":"50","MONTEBOURG":"50","MONTMARTIN EN GRAIGNES":"50","MONTMARTIN SUR MER":"50","MONTRABOT":"50","LE MONT ST MICHEL":"50","NOTRE DAME DE LIVOYE":"50","PERRIERS EN BEAUFICEL":"50","LE PERRON":"50","PICAUVILLE":"50","PIROU":"50","QUINEVILLE":"50","RAUVILLE LA BIGOT":"50","REGNEVILLE SUR MER":"50","ROCHEVILLE":"50","ST ANDRE DE BOHON":"50","ST ANDRE DE L EPINE":"50","STE CROIX HAGUE":"50","ST GEORGES DE LA RIVIERE":"50","ST GEORGES DE LIVOYE":"50","ST GEORGES MONTCOCQ":"50","ST HILAIRE PETITVILLE":"50","ST JACQUES DE NEHOU":"50","ST JEAN DE SAVIGNY":"50","ST LO":"50","ST LOUET SUR VIRE":"50","ST MARTIN D AUBIGNY":"50","ST MARTIN DE CENILLY":"50","ST MARTIN DE VARREVILLE":"50","STE MERE EGLISE":"50","ST OVIN":"50","ST PATRICE DE CLAIDS":"50","LE PARC":"50","ST POIS":"50","ST SAUVEUR LE VICOMTE":"50","ST VIGOR DES MONTS":"50","SAUSSEMESNIL":"50","SAVIGNY LE VIEUX":"50","SERVIGNY":"50","SORTOSVILLE EN BEAUMONT":"50","SOTTEVAST":"50","SOURDEVAL":"50","TAILLEPIED":"50","TAMERVILLE":"50","LE TANU":"50","TEURTHEVILLE BOCAGE":"50","THEVILLE":"50","TOURVILLE SUR SIENNE":"50","LE VAL ST PERE":"50","VASTEVILLE":"50","LE VICEL":"50","VIDECOSVILLE":"50","VILLEBAUDON":"50","LES CHAMPEAUX":"61","CHAMPSECRET":"61","CHANU":"61","LA CHAPELLE AU MOINE":"61","CHEMILLI":"61","LA COCHERE":"61","SABLONS SUR HUISNE":"61","COULONGES SUR SARTHE":"61","CROUTTES":"61","DAME MARIE":"61","ECOUCHE LES VALLEES":"61","FEL":"61","LA FERRIERE BECHET":"61","LA FERRIERE BOCHARD":"61","FERRIERES LA VERRERIE":"61","FONTENAI SUR ORNE":"61","FRESNAY LE SAMSON":"61","GANDELAIN":"61","LES GENETTES":"61","HELOUP":"61","ST BRICE SOUS RANES":"61","ST CYR LA ROSIERE":"61","ST DIDIER SOUS ECOUVES":"61","ST GERMAIN DU CORBEIS":"61","ST GERVAIS DES SABLONS":"61","ST JOUIN DE BLAVOU":"61","ST MARD DE RENO":"61","STE MARGUERITE DE CARROUGES":"61","LES ASPRES":"61","ST MARTIN DES LANDES":"61","ST MARTIN DES PEZERITS":"61","ST MARTIN L AIGUILLON":"61","ST PATRICE DU DESERT":"61","SAP EN AUGE":"61","LE SAP ANDRE":"61","SEES":"61","LA SELLE LA FORGE":"61","SENTILLY":"61","SURE":"61","VAL AU PERCHE":"61","TINCHEBRAY BOCAGE":"61","TOUROUVRE AU PERCHE":"61","TRUN":"61","LA VENTROUZE":"61","VIDAI":"61","VILLIERS SOUS MORTAGNE":"61","VIMOUTIERS":"61","VITRAI SOUS LAIGLE":"61","ACQ":"62","AGNEZ LES DUISANS":"62","AIRON ST VAAST":"62","AIX EN ISSART":"62","ALLOUAGNE":"62","AMBLETEUSE":"62","ANGRES":"62","ANNEQUIN":"62","ANNEZIN":"62","ARLEUX EN GOHELLE":"62","ST LOUP DE VARENNES":"71","ST MARTIN BELLE ROCHE":"71","ST MARTIN EN BRESSE":"71","ST MARTIN SOUS MONTAIGU":"71","ST MAURICE DE SATONNAY":"71","ST SERNIN DU BOIS":"71","ST SYMPHORIEN D ANCELLES":"71","ST VALLERIN":"71","ST VINCENT BRAGNY":"71","ST YAN":"71","SAISY":"71","SERRIGNY EN BRESSE":"71","SEVREY":"71","TAVERNAY":"71","TOULON SUR ARROUX":"71","TOURNUS":"71","TRAMAYES":"71","TRAMBLY":"71","LA TRUCHERE":"71","VARENNE L ARCONCE":"71","VARENNES LE GRAND":"71","VERGISSON":"71","VERISSEY":"71","VERJUX":"71","VERZE":"71","CLUX VILLENEUVE":"71","VITRY LES CLUNY":"71","ARDENAY SUR MERIZE":"72","ARTHEZE":"72","BEAUMONT SUR SARTHE":"72","BRAINS SUR GEE":"72","BRIOSNE LES SABLES":"72","CHAHAIGNES":"72","LA CHAPELLE ST AUBIN":"72","LA CHAPELLE ST FRAY":"72","LA CHAPELLE ST REMY":"72","CHASSILLE":"72","CHAUFOUR NOTRE DAME":"72","CHERISAY":"72","CONTILLY":"72","COULAINES":"72","COULONGE":"72","COURCEMONT":"72","COURCIVAL":"72","COURTILLERS":"72","CROSMIERES":"72","DISSE SOUS LE LUDE":"72","VILLENEUVE EN PERSEIGNE":"72","LE PLESSIS GASSOT":"95","PONTOISE":"95","PUISEUX EN FRANCE":"95","LA ROCHE GUYON":"95","ROISSY EN FRANCE":"95","ST OUEN L AUMONE":"95","ST WITZ":"95","VAUREAL":"95","VETHEUIL":"95","LES ABYMES":"971","BAIE MAHAULT":"971","CAPESTERRE BELLE EAU":"971","DESHAIES":"971","MORNE A L EAU":"971","PETIT CANAL":"971","GROS MORNE":"972","LE LAMENTIN":"972","MACOUBA":"972","LE MARIGOT":"972","LE PRECHEUR":"972","RIVIERE SALEE":"972","LE VAUCLIN":"972","CAYENNE":"973","ROURA":"973","ST ELIE":"973","AWALA YALIMAPO":"973","PAPAICHTON":"973","PETITE ILE":"974","LA POSSESSION":"974","ST LEU":"974","ST PHILIPPE":"974","SALAZIE":"974","LE TAMPON":"974","CILAOS":"974","CHICONI":"976","DZAOUDZI":"976","MAMOUDZOU":"976","MTSAMBORO":"976","ANAA":"987","ARUTUA":"987","FANGATAU":"987","HAO":"987","HITIAA O TE RA":"987","HUAHINE":"987","MAKEMO":"987","MAUPITI":"987","MOOREA MAIAO":"987","NUKU HIVA":"987","NUKUTAVAKE":"987","RANGIROA":"987","RAPA":"987","RIMATARA":"987","RURUTU":"987","TAHAA":"987","TAHUATA":"987","TAIARAPU EST":"987","TAKAROA":"987","TAPUTAPUATEA":"987","TEVA I UTA":"987","UA HUKA":"987","UTUROA":"987","BOULOUPARI":"988","HOUAILOU":"988","KOUMAC":"988","LA FOA":"988","LE MONT DORE":"988","OUEGOA":"988","POINDIMIE":"988","POUEBO":"988","YATE":"988","KOUAOUA":"988","AFA":"2A","AJACCIO":"2A","ALTAGENE":"2A","LARRESSINGLE":"32","LASSERADE":"32","LUPIAC":"32","MANENT MONTANE":"32","MARGUESTAU":"32","MASSEUBE":"32","MIRAMONT LATOUR":"32","MONCASSIN":"32","MONFERRAN SAVES":"32","MONFORT":"32","MONGUILHEM":"32","MONLEZUN":"32","MONTAMAT":"32","NOGARO":"32","NOILHAN":"32","ORDAN LARROQUE":"32","PANJAS":"32","PERCHEDE":"32","PESSOULENS":"32","PEYRECAVE":"32","PEYRUSSE VIEILLE":"32","POUY ROQUELAURE":"32","PRENERON":"32","PUJAUDRAN":"32","PUYLAUSIC":"32","RAMOUZENS":"32","REANS":"32","RISCLE":"32","SABAILLAN":"32","SADEILLAN":"32","ST AUNIX LENGROS":"32","ST ELIX":"32","ST GERME":"32","ST LOUBE":"32","ST SAUVY":"32","SAMARAN":"32","SARCOS":"32","SEGOS":"32","SEISSAN":"32","SERE":"32","TOURNAN":"32","TOURNECOUPE":"32","TUDELLE":"32","VILLECOMTAL SUR ARROS":"32","ANGLADE":"33","ARVEYRES":"33","VAL DE VIRVEE":"33","AYGUEMORTE LES GRAVES":"33","BALIZAC":"33","BASSANNE":"33","BELIN BELIET":"33","BIEUJAC":"33","BLAIGNAC":"33","BLAYE":"33","BONZAC":"33","BORDEAUX":"33","BOURDELLES":"33","BRANNENS":"33","BROUQUEYRAN":"33","BRUGES":"33","CADARSAC":"33","CAMIAC ET ST DENIS":"33","CANTENAC":"33","CAPTIEUX":"33","CARIGNAN DE BORDEAUX":"33","CASTELNAU DE MEDOC":"33","CAZAUGITAT":"33","CENON":"33","CESSAC":"33","CESTAS":"33","CIVRAC DE BLAYE":"33","CLEYRAC":"33","COURPIAC":"33","CUDOS":"33","DAIGNAC":"33","ESCOUSSANS":"33","EYRANS":"33","FOSSES ET BALEYSSAC":"33","GARDEGAN ET TOURTIRAC":"33","GAURIAC":"33","GISCOS":"33","GORNAC":"33","GRADIGNAN":"33","GUILLOS":"33","LABESCAU":"33","LADAUX":"33","AUBIGNY AUX KAISNES":"02","AUBENCHEUL AUX BOIS":"02","ARCHON":"02","VILLIEU LOYES MOLLON":"01","VIRIEU LE GRAND":"01","VILLEREVERSURE":"01","AMIGNY ROUY":"02","VESCOURS":"01","VESANCY":"01","VONNAS":"01","VIEU":"01","ACY":"02","VAL REVERMONT":"01","SUTRIEU":"01","VAUX EN BUGEY":"01","LA TRANCLIERE":"01","VERSAILLEUX":"01","TALISSIEU":"01","TOUSSIEUX":"01","TREVOUX":"01","VALEINS":"01","ST ETIENNE SUR REYSSOUZE":"01","ST ANDRE LE BOUCHOUX":"01","ST MARTIN LE CHATEL":"01","ROSSILLON":"01","SAMOGNAT":"01","SANDRANS":"01","SALAVRE":"01","LE POIZAT LALLEYRIAT":"01","MASSIGNIEU DE RIVES":"01","LOMPNIEU":"01","MASSIEUX":"01","MARCHAMP":"01","LEYMENT":"01","MANZIAT":"01","LOMPNAS":"01","PEYZIEUX SUR SAONE":"01","PONT DE VEYLE":"01","PIRAJOUX":"01","REVONNAS":"01","POLLIAT":"01","POLLIEU":"01","PUGIEU":"01","PORT":"01","BARZY SUR MARNE":"02","BARENTON BUGNY":"02","LES AUTELS":"02","CUIRY LES CHAUDARDES":"02","ETAVES ET BOCQUIAUX":"02","LA FERTE CHEVRESIS":"02","ETREAUPONT":"02","DORENGT":"02","CROUY":"02","EPPES":"02","ERLON":"02","HARCIGNY":"02","HARAMONT":"02","GRONARD":"02","HARLY":"02","GERCY":"02","LA NEUVILLE LES DORENGT":"02","NEUVILLE SUR AILETTE":"02","NAMPTEUIL SOUS MURET":"02","NESLES LA MONTAGNE":"02","MONCEAU LES LEUPS":"02","NEUVE MAISON":"02","DIGNE LES BAINS":"04","LA JAVIE":"04","NOYERS SUR JABRON":"04","LA MURE ARGENS":"04","JAUSIERS":"04","CLUMANC":"04","DAUPHIN":"04","ORAISON":"04","MESBRECOURT RICHECOURT":"02","MISSY SUR AISNE":"02","MEZY MOULINS":"02","LISLET":"02","LANDIGOU":"61","LIGNERES":"61","LIGNOU":"61","LONGUENOE":"61","LONLAY LE TESSON":"61","LA MADELEINE BOUVET":"61","MAHERU":"61","MANTILLY":"61","MARDILLY":"61","LE MENIL BERARD":"61","MENIL HERMEI":"61","MENIL HUBERT SUR ORNE":"61","MERRI":"61","MESSEI":"61","MONCY":"61","MONTREUIL LA CAMBE":"61","MORTAGNE AU PERCHE":"61","LA MOTTE FOUQUET":"61","NEAUPHE SOUS ESSAI":"61","NECY":"61","PERCHE EN NOCE":"61","NONANT LE PIN":"61","OCCAGNES":"61","OMMOY":"61","ORIGNY LE BUTIN":"61","ORIGNY LE ROUX":"61","PASSAIS VILLAGES":"61","PERROU":"61","PERVENCHERES":"61","LE PLANTIS":"61","LE RENOUARD":"61","ST ANDRE DE MESSEI":"61","ST AUBIN D APPENAI":"61","ST CHRISTOPHE DE CHAULIEU":"61","ST CLAIR DE HALOUZE":"61","ST ELLIER LES BOIS":"61","STE GAUBURGE STE COLOMBE":"61","ST GEORGES D ANNEBECQ":"61","ST GERMAIN D AUNAY":"61","ST GERMAIN DE MARTIGNY":"61","ST HILAIRE SUR ERRE":"61","ST HILAIRE SUR RISLE":"61","ST LEGER SUR SARTHE":"61","ST LEONARD DES PARCS":"61","STE MARIE LA ROBERT":"61","ST MICHEL TUBOEUF":"61","ST PIERRE LA RIVIERE":"61","TOUQUETTES":"61","UROU ET CRENNES":"61","VIEUX PONT":"61","VILLEDIEU LES BAILLEUL":"61","ABLAIN ST NAZAIRE":"62","ABLAINZEVELLE":"62","ACHEVILLE":"62","ACHICOURT":"62","AIX EN ERGNY":"62","ALETTE":"62","AMBRICOURT":"62","AMETTES":"62","AMPLIER":"62","ANZIN ST AUBIN":"62","AUBIGNY EN ARTOIS":"62","AUBIN ST VAAST":"62","AVESNES LE COMTE":"62","AVONDANCE":"62","AYETTE":"62","BAILLEUL AUX CORNAILLES":"62","BAILLEULMONT":"62","BAPAUME":"62","BARALLE":"62","BAVINCOURT":"62","BEALENCOURT":"62","BEAULENCOURT":"62","BEAUMETZ LES CAMBRAI":"62","BEAUMETZ LES LOGES":"62","BEAURAINS":"62","BEAUVOIS":"62","BEHAGNIES":"62","BELLE ET HOULLEFORT":"62","BERLENCOURT LE CAUROY":"62","BERLES AU BOIS":"62","BERLES MONCHEL":"62","BETHONSART":"62","FRESNAY SUR SARTHE":"72","JOUE L ABBE":"72","JUPILLES":"72","LAMNAY":"72","LAVENAY":"72","LAVERNAT":"72","LOUE":"72","LOUZES":"72","LE LUART":"72","LE LUDE":"72","MARCON":"72","MARESCHE":"72","MAROLLES LES BRAULTS":"72","MEZERAY":"72","LA MILESSE":"72","MOITRON SUR SARTHE":"72","MONHOUDOU":"72","MONTABON":"72","MONTREUIL LE HENRI":"72","NOGENT LE BERNARD":"72","OIZE":"72","NOTRE DAME DU PE":"72","PRECIGNE":"72","RENE":"72","STE CEROTTE":"72","ST DENIS DES COUDRAIS":"72","ST GERVAIS DE VIC":"72","ST GERVAIS EN BELIN":"72","ST JEAN DE LA MOTTE":"72","ST LONGIS":"72","ST OUEN DE MIMBRE":"72","ST PIERRE DES BOIS":"72","ST REMY DU VAL":"72","SARCE":"72","SEMUR EN VALLON":"72","SILLE LE PHILIPPE":"72","LA SUZE SUR SARTHE":"72","TASSE":"72","THELIGNY":"72","TORCE EN VALLEE":"72","VOUVRAY SUR HUISNE":"72","AIME LA PLAGNE":"73","ENTRELACS":"73","ALBIEZ MONTROND":"73","ARITH":"73","LES AVANCHERS VALMOREL":"73","AVRESSIEUX":"73","BESSANS":"73","BILLIEME":"73","BONVILLARD":"73","BOURDEAU":"73","LE BOURGET DU LAC":"73","CHAMOUSSET":"73","LE CHATEL":"73","LE CHATELARD":"73","CHIGNIN":"73","CHINDRIEUX":"73","COHENNOZ":"73","CREST VOLAND":"73","CURIENNE":"73","LES DESERTS":"73","DOUCY EN BAUGES":"73","LES ECHELLES":"73","FEISSONS SUR SALINS":"73","JARRIER":"73","LESCHERAINES":"73","MONTENDRY":"73","MONTGELLAFREY":"73","MONTVALEZAN":"73","MOUXY":"73","NOTRE DAME DE BELLECOMBE":"73","NOTRE DAME DU PRE":"73","PLANCHERINE":"73","PRALOGNAN LA VANOISE":"73","LA RAVOIRE":"73","ROTHERENS":"73","ST ALBAN DE MONTBEL":"73","ST ALBAN LEYSSE":"73","ST AVRE":"73","ST ETIENNE DE CUINES":"73","ST GEORGES D HURTIERES":"73","ST JEAN DE BELLEVILLE":"73","ST JEOIRE PRIEURE":"73","ST MAURICE DE ROTHERENS":"73","ST MICHEL DE MAURIENNE":"73","ST PAUL SUR ISERE":"73","ST SORLIN D ARVES":"73","ST VITAL":"73","SERRIERES EN CHAUTAGNE":"73","SONNAZ":"73","LA TABLE":"73","ARBELLARA":"2A","ARBORI":"2A","AULLENE":"2A","AZZANA":"2A","CARBUCCIA":"2A","CARDO TORGIA":"2A","COGGIA":"2A","CONCA":"2A","ECCICA SUARELLA":"2A","GUARGUALE":"2A","GUITERA LES BAINS":"2A","MARIGNANA":"2A","MONACIA D AULLENE":"2A","OTA":"2A","PASTRICCIOLA":"2A","PIANA":"2A","PIANOTTOLI CALDARELLO":"2A","SARI SOLENZARA":"2A","SOCCIA":"2A","SOTTA":"2A","SANTA MARIA FIGANIELLA":"2A","TASSO":"2A","UCCIANI":"2A","ZONZA":"2A","ALBERTACCE":"2B","ANTISANTI":"2B","BASTIA":"2B","BIGUGLIA":"2B","BRANDO":"2B","CALENZANA":"2B","CAMBIA":"2B","CAMPANA":"2B","CANAVAGGIA":"2B","CASABIANCA":"2B","CERVIONE":"2B","CORSCIA":"2B","CROCE":"2B","ERSA":"2B","FARINOLE":"2B","FAVALELLO":"2B","GHISONACCIA":"2B","ISOLACCIO DI FIUMORBO":"2B","LINGUIZZETTA":"2B","LORETO DI CASINCA":"2B","LURI":"2B","MANSO":"2B","MONTE":"2B","MONTEGROSSO":"2B","MONTICELLO":"2B","NOCARIO":"2B","NONZA":"2B","OLCANI":"2B","OLETTA":"2B","OLMETA DI TUDA":"2B","OMESSA":"2B","PANCHERACCIA":"2B","PERELLI":"2B","PIANELLO":"2B","PIETRALBA":"2B","PIEVE":"2B","PIGNA":"2B","POGGIO DI NAZZA":"2B","POGGIO MARINACCIO":"2B","POPOLASCA":"2B","LA PORTA":"2B","PRUNELLI DI FIUMORBO":"2B","PRUNO":"2B","ROSPIGLIANI":"2B","SORBO OCAGNANO":"2B","SANT ANDREA DI COTONE":"2B","SAN DAMIANO":"2B","SAN MARTINO DI LOTA":"2B","SANTA LUCIA DI MORIANI":"2B","TAGLIO ISOLACCIO":"2B","TALLONE":"2B","TOX":"2B","VALLE D OREZZA":"2B","VILLE DI PARASO":"2B","VILLE DI PIETRABUGNO":"2B","SAN GAVINO DI FIUMORBO":"2B","CHISA":"2B","ARTEMARE":"01","ARANDAS":"01","ARGIS":"01","BENY":"01","GEOVREISSET":"01","MARTIGNAT":"01","LEYSSARD":"01","LAVOURS":"01","GRIEGES":"01","CORMARANCHE EN BUGEY":"01","LAMARQUE":"33","LAMOTHE LANDERRON":"33","LATRESNE":"33","LESTIAC SUR GARONNE":"33","LIGNAN DE BORDEAUX":"33","MARGAUX":"33","MARIMBAULT":"33","MARTILLAC":"33","MASSUGAS":"33","MAZION":"33","MOMBRIER":"33","MORIZES":"33","NAUJAN ET POSTIAC":"33","NEAC":"33","NERIGEAN":"33","NOAILLAC":"33","POMPIGNAC":"33","PONDAURAT":"33","PUYNORMAND":"33","QUEYRAC":"33","RIOCAUD":"33","ST ANDRE DU BOIS":"33","ST ANDRONY":"33","ST ANTOINE DU QUEYRET":"33","ST AUBIN DE BLAYE":"33","ST AUBIN DE MEDOC":"33","ST AVIT ST NAZAIRE":"33","ST CHRISTOPHE DE DOUBLE":"33","ST EMILION":"33","ST ETIENNE DE LISSE":"33","ST FERME":"33","ST GENES DE BLAYE":"33","ST GENES DE CASTILLON":"33","ST LAURENT DU BOIS":"33","ST LEGER DE BALSON":"33","ST MAGNE DE CASTILLON":"33","ST MARTIN LACAUSSADE":"33","ST MEDARD EN JALLES":"33","ST MICHEL DE FRONSAC":"33","ST PARDON DE CONQUES":"33","ST PEY D ARMENS":"33","ST SEURIN SUR L ISLE":"33","SAMONAC":"33","LA SAUVE":"33","SAUVETERRE DE GUYENNE":"33","SEMENS":"33","SILLAS":"33","LE TAILLAN MEDOC":"33","TARGON":"33","LE TEICH":"33","LA TESTE DE BUCH":"33","VALEYRAC":"33","VENDAYS MONTALIVET":"33","VERDELAIS":"33","LE VERDON SUR MER":"33","VILLENAVE D ORNON":"33","VIRELADE":"33","YVRAC":"33","ADISSAN":"34","ARBORAS":"34","BABEAU BOULDOUX":"34","BALARUC LES BAINS":"34","BERLOU":"34","CAUX":"34","CAZEDARNES":"34","CLAPIERS":"34","COLOMBIERES SUR ORB":"34","CORNEILHAN":"34","LE CRES":"34","LE CROS":"34","FERRIERES POUSSAROU":"34","FLORENSAC":"34","FRONTIGNAN":"34","GALARGUES":"34","GIGEAN":"34","GORNIES":"34","GRABELS":"34","GUZARGUES":"34","JUVIGNAC":"34","LAMALOU LES BAINS":"34","LIME":"02","ROCOURT ST MARTIN":"02","PIGNICOURT":"02","RENANSART":"02","ROUGERIES":"02","PERNANT":"02","PLOISY":"02","PINON":"02","ROUPY":"02","LAVAL EN LAONNOIS":"02","LAFFAUX":"02","LAPPION":"02","IVIERS":"02","LAIGNY":"02","LESGES":"02","LEURY":"02","LEUZE":"02","ORIGNY STE BENOITE":"02","PARGNY LA DHUYS":"02","NOGENT L ARTAUD":"02","PASSY EN VALOIS":"02","NIZY LE COMTE":"02","ORAINVILLE":"02","PAPLEUX":"02","OMISSY":"02","PAARS":"02","VILLERS COTTERETS":"02","VILLERS ST CHRISTOPHE":"02","VILLENEUVE SUR FERE":"02","VILLEQUIER AUMONT":"02","VILLEMONTOIRE":"02","LE VERGUIER":"02","PETIT VERLY":"02","VAUXREZIS":"02","VENDIERES":"02","VASSENY":"02","LA CHAPELLE AUX CHASSES":"03","CREUZIER LE VIEUX":"03","DROITURIER":"03","CHOUVIGNY":"03","COULANDON":"03","CHANTELLE":"03","CHEVAGNES":"03","CHARMEIL":"03","CINDRE":"03","BEAUNE D ALLIER":"03","BARBERIER":"03","VIVIERES":"02","CHAMBLET":"03","WATIGNY":"02","AVERMES":"03","AGONGES":"03","BRESNAY":"03","VREGNY":"02","AUDES":"03","ST VINCENT LES FORTS":"04","ST PAUL SUR UBAYE":"04","TOULIS ET ATTENCOURT":"02","VAILLY SUR AISNE":"02","UGNY LE GAY":"02","TERNY SORNY":"02","VARISCOURT":"02","TARTIERS":"02","LE SOURD":"02","THIERNU":"02","TRUCY":"02","VAL DE CHALVAGNE":"04","BAYONS":"04","CASTELLANE":"04","CHATEAU ARNOUX ST AUBAN":"04","CASTELLET LES SAUSSES":"04","CHAUDON NORANTE":"04","CHATEAUREDON":"04","BRUNET":"04","VILLENEUVE SUR ALLIER":"03","VARENNES SUR ALLIER":"03","AUBENAS LES ALPES":"04","USSEL D ALLIER":"03","YGRANDE":"03","VIPLAIX":"03","BARRAS":"04","VENDAT":"03","VENAS":"03","SAINS RICHAUMONT":"02","ROZOY BELLEVALLE":"02","ROZOY SUR SERRE":"02","ROZET ST ALBIN":"02","BEUTIN":"62","BIEFVILLERS LES BAPAUME":"62","BILLY BERCLAU":"62","BLAIRVILLE":"62","BOIRY BECQUERELLE":"62","BOIRY ST MARTIN":"62","BOIRY STE RICTRUDE":"62","BOMY":"62","BOUBERS LES HESMOND":"62","BOUBERS SUR CANCHE":"62","BOURNONVILLE":"62","BOUVIGNY BOYEFFLES":"62","BRUAY LA BUISSIERE":"62","BUS":"62","CALONNE RICOUART":"62","CAMBRIN":"62","CAMIERS":"62","CANETTEMONT":"62","CLENLEU":"62","COQUELLES":"62","COURCELLES LE COMTE":"62","COURRIERES":"62","COURSET":"62","CROIX EN TERNOIS":"62","CUCQ":"62","ECOIVRES":"62","ECQUES":"62","EMBRY":"62","ESCOEUILLES":"62","ESSARS":"62","ESTEVELLES":"62","EVIN MALMAISON":"62","FAUQUEMBERGUES":"62","FERQUES":"62","FIEFS":"62","FOUQUIERES LES BETHUNE":"62","FRESNOY EN GOHELLE":"62","FREVENT":"62","FREVIN CAPELLE":"62","GAUCHIN VERLOINGT":"62","GAUDIEMPRE":"62","GENNES IVERGNY":"62","GIVENCHY LE NOBLE":"62","GOUY EN ARTOIS":"62","GOUY EN TERNOIS":"62","GUEMPS":"62","GUINES":"62","HAUTE AVESNES":"62","HAUTECLOQUE":"62","HENIN BEAUMONT":"62","HERMIN":"62","HESDIGNEUL LES BETHUNE":"62","HEUCHIN":"62","HEURINGHEM":"62","HOUVIN HOUVIGNEUL":"62","HULLUCH":"62","HUMBERCAMPS":"62","HUMEROEUILLE":"62","INGHEM":"62","IVERGNY":"62","IZEL LES EQUERCHIN":"62","LATTRE ST QUENTIN":"62","LEBUCQUIERE":"62","LEFOREST":"62","LIETTRES":"62","LIEVIN":"62","LIGNEREUIL":"62","LINZEUX":"62","LISBOURG":"62","LOOS EN GOHELLE":"62","LOUCHES":"62","MAINTENAY":"62","MAISNIL LES RUITZ":"62","MAISONCELLE":"62","MANINGHEM":"62","MAREST":"62","MARLES SUR CANCHE":"62","MENCAS":"62","MEURCHIN":"62","MORINGHEM":"62","MORY":"62","NABRINGHEN":"62","NEDONCHEL":"62","NEMPONT ST FIRMIN":"62","NEUVILLE VITASSE":"62","NEUVIREUIL":"62","NIELLES LES ARDRES":"62","NORRENT FONTES":"62","NORTKERQUE":"62","NOYELLES LES VERMELLES":"62","NOYELLETTE":"62","NUNCQ HAUTECOTE":"62","LA THUILE":"73","TREVIGNIN":"73","VAL D ISERE":"73","VEREL PRAGONDRAN":"73","VERRENS ARVEY":"73","VILLAREMBERT":"73","VIMINES":"73","ABONDANCE":"74","ALBY SUR CHERAN":"74","ALLONZIER LA CAILLE":"74","ANNECY LE VIEUX":"74","ANTHY SUR LEMAN":"74","ARACHES LA FRASSE":"74","ARBUSIGNY":"74","ARGONAY":"74","LA BALME DE SILLINGY":"74","BLUFFY":"74","BRENTHONNE":"74","BURDIGNIN":"74","CERNEX":"74","CHAMONIX MONT BLANC":"74","CHARVONNEX":"74","CHESSENAZ":"74","CHEVALINE":"74","CHEVRIER":"74","DESINGY":"74","DINGY ST CLAIR":"74","ELOISE":"74","ENTREMONT":"74","ETERCY":"74","ETREMBIERES":"74","GAILLARD":"74","LES GETS":"74","GROISY":"74","HABERE LULLIN":"74","HABERE POCHE":"74","LATHUILE":"74","LESCHAUX":"74","LUCINGES":"74","LUGRIN":"74","MARCELLAZ ALBANAIS":"74","MARGENCEL":"74","MARIGNY ST MARCEL":"74","MAXILLY SUR LEMAN":"74","MUSIEGES":"74","NANGY":"74","NAVES PARMELAN":"74","NERNIER":"74","LES OLLIERES":"74","PEILLONNEX":"74","QUINTAL":"74","REIGNIER ESERY":"74","LA ROCHE SUR FORON":"74","ST ANDRE DE BOEGE":"74","ST FERREOL":"74","ST JORIOZ":"74","ST MARTIN BELLEVUE":"74","ST PAUL EN CHABLAIS":"74","ST PIERRE EN FAUCIGNY":"74","ST SIXT":"74","SERRAVAL":"74","SERVOZ":"74","VAL DE FIER":"74","TANINGES":"74","THUSY":"74","VILLAZ":"74","VILLY LE PELLOUX":"74","VULBENS":"74","PARIS 01":"75","PARIS 02":"75","PARIS 05":"75","PARIS 06":"75","PARIS 08":"75","PARIS 09":"75","PARIS 13":"75","PARIS 20":"75","ANCRETIEVILLE ST VICTOR":"76","ARGUEIL":"76","AUBERMESNIL BEAUMAIS":"76","AUTIGNY":"76","AUZEBOSC":"76","BARDOUVILLE":"76","BAROMESNIL":"76","BAZINVAL":"76","BEAUMONT LE HARENG":"76","BELBEUF":"76","BENNETOT":"76","CONDEISSIAT":"01","CHEVILLARD":"01","CONTREVOZ":"01","COLIGNY":"01","CONZIEU":"01","CORLIER":"01","CHEVRY":"01","ARCY STE RESTITUE":"02","VIEU D IZENAVE":"01","TOSSIAT":"01","VONGNES":"01","VESINES":"01","VERJON":"01","ST GEORGES SUR RENON":"01","ST JEAN DE GONVILLE":"01","ST SORLIN EN BUGEY":"01","SERRIERES SUR AIN":"01","ST JEAN DE NIOST":"01","SURJOUX":"01","TORCIEU":"01","TENAY":"01","CRUZILLES LES MEPILLAT":"01","CRESSIN ROCHEFORT":"01","ECHENEVEX":"01","GARNERANS":"01","FEILLENS":"01","ETREZ":"01","CULOZ":"01","DROM":"01","BOURG EN BRESSE":"01","LA BURBANCHE":"01","BUELLAS":"01","BIZIAT":"01","BOZ":"01","CHAMPDOR CORCELLES":"01","CHATILLON LA PALUD":"01","CHAZEY SUR AIN":"01","CHANEINS":"01","CHALLEX":"01","CHARIX":"01","CESSY":"01","NURIEUX VOLOGNAT":"01","MONTREVEL EN BRESSE":"01","NEUVILLE LES DAMES":"01","PEROUGES":"01","PARCIEUX":"01","PERONNAS":"01","OUTRIAZ":"01","PEYRIAT":"01","ST GENIS SUR MENTHON":"01","ST ANDRE DE CORCY":"01","PONT DE VAUX":"01","STE EUPHEMIE":"01","PREMILLIEU":"01","REPLONGES":"01","RUFFIEU":"01","PEYRIEU":"01","RANCE":"01","BOUCONVILLE VAUCLAIR":"02","BLANZY LES FISMES":"02","BIEUXY":"02","BRUYERES ET MONTBERAULT":"02","BOURGUIGNON SOUS COUCY":"02","BRAYE EN THIERACHE":"02","BRANCOURT LE GRAND":"02","BRISSY HAMEGICOURT":"02","BRAY ST CHRISTOPHE":"02","BRAYE EN LAONNOIS":"02","LA BOUTEILLE":"02","BUCILLY":"02","BRAYE":"02","CERNY EN LAONNOIS":"02","CHAILLEVOIS":"02","CHAMOUILLE":"02","CHAVONNE":"02","CHACRISE":"02","CHAUDUN":"02","BUIRE":"02","ESQUEHERIES":"02","LIAUSSON":"34","LIEURAN CABRIERES":"34","LOUPIAN":"34","LES MATELLES":"34","MERIFONS":"34","MEZE":"34","MINERVE":"34","MONTPELLIER":"34","OLARGUES":"34","PALAVAS LES FLOTS":"34","PINET":"34","POMEROLS":"34","POUZOLS":"34","PRADES SUR VERNAZOBRE":"34","PUECHABON":"34","ROSIS":"34","ROUJAN":"34","ST BAUZILLE DE MONTMEL":"34","ST DREZERY":"34","ST GELY DU FESC":"34","ST JEAN DE CORNIES":"34","ST JEAN DE CUCULLES":"34","ST JEAN DE LA BLAQUIERE":"34","ST JEAN DE MINERVOIS":"34","ST MAURICE NAVACELLES":"34","ST PARGOIRE":"34","ST PONS DE MAUCHIENS":"34","SALASC":"34","SAUVIAN":"34","SERVIAN":"34","SETE":"34","SOUMONT":"34","SUSSARGUES":"34","TAUSSAC LA BILLIERE":"34","LA VACQUERIE ET ST MARTIN DE CASTRIES":"34","VALFLAUNES":"34","ANTRAIN":"35","LA BAUSSAINE":"35","LA BAZOUGE DU DESERT":"35","BEDEE":"35","BETTON":"35","LA BOSSE DE BRETAGNE":"35","LA BOUSSAC":"35","BROUALAN":"35","CANCALE":"35","LA CHAPELLE ERBREE":"35","CHATEAUGIRON":"35","CHAUVIGNE":"35","CHERRUEIX":"35","COMBOURG":"35","CREVIN":"35","LE CROUAIS":"35","DOMAGNE":"35","LA DOMINELAIS":"35","DOURDAIN":"35","EPINIAC":"35","ERCE EN LAMEE":"35","FLEURIGNE":"35","GAEL":"35","GOSNE":"35","GRAND FOUGERAY":"35","HIREL":"35","IRODOUER":"35","JAVENE":"35","LANGOUET":"35","LILLEMER":"35","LOHEAC":"35","LOUTEHEL":"35","LOUVIGNE DU DESERT":"35","MARCILLE ROBERT":"35","MECE":"35","MELESSE":"35","GUIPRY MESSAC":"35","MONTAUBAN DE BRETAGNE":"35","MONTOURS":"35","MONTREUIL DES LANDES":"35","MONTREUIL SOUS PEROUSE":"35","LA NOUAYE":"35","NOYAL CHATILLON SUR SEICHE":"35","LE PERTRE":"35","PIPRIAC":"35","PLERGUER":"35","PLEUMELEUC":"35","RETIERS":"35","LA RICHARDAIS":"35","ROMAZY":"35","ST CHRISTOPHE DE VALAINS":"35","ST DOMINEUC":"35","ST GEORGES DE REINTEMBAULT":"35","ST GENGOULPH":"02","GRAND ROZOY":"02","STE PREUVE":"02","ST QUENTIN":"02","SEPTVAUX":"02","SOISSONS":"02","SERMOISE":"02","SAPONAY":"02","SINCENY":"02","SERAIN":"02","SERVAL":"02","SAVY":"02","LA BATIE VIEILLE":"05","VILLARS COLMARS":"04","VERDACHES":"04","VALERNES":"04","ST POURCAIN SUR SIOULE":"03","ST LEGER SUR VOUZANCE":"03","ST GERMAIN DES FOSSES":"03","ST NICOLAS DES BIEFS":"03","ST LEOPARDIN D AUGY":"03","ST MARTINIEN":"03","ST ENNEMOND":"03","ST MENOUX":"03","ST DESIRE":"03","ST PRIEST EN MURAT":"03","SALIGNY SUR ROUDON":"03","THIEL SUR ACOLIN":"03","STE THERENCE":"03","TAXAT SENAT":"03","SERBANNES":"03","TRETEAU":"03","TARGET":"03","URCAY":"03","ST BONNET DE FOUR":"03","LA PETITE MARCHE":"03","NERIS LES BAINS":"03","NEUILLY LE REAL":"03","POUZY MESANGY":"03","MONTMARAULT":"03","LE MONTET":"03","MONTORD":"03","MOLLES":"03","LA FERTE HAUTERIVE":"03","GANNAY SUR LOIRE":"03","GENNETINES":"03","ETROUSSAT":"03","LALIZOLLE":"03","FLEURIEL":"03","LAFELINE":"03","GIPCY":"03","HYDS":"03","LOUROUX DE BEAUNE":"03","LOUROUX DE BOUBLE":"03","LE MAYET D ECOLE":"03","MEILLARD":"03","LAPRUGNE":"03","MEAULNE":"03","LORIGES":"03","MAZIRAT":"03","LUNEAU":"03","SENEZ":"04","THORAME HAUTE":"04","VALENSOLE":"04","THOARD":"04","CHATEAU VILLE VIEILLE":"05","CHATEAUROUX LES ALPES":"05","CHATEAUNEUF D OZE":"05","CHAMPCELLA":"05","FOUILLOUSE":"05","LES COSTES":"05","CHABOTTES":"05","CHANOUSSE":"05","BREZIERS":"05","LA CHAPELLE EN VALGAUDEMAR":"05","LARDIER ET VALENCA":"05","LA FREISSINOUSE":"05","GUILLESTRE":"05","MONTGARDIN":"05","MANTEYER":"05","NEVACHE":"05","VAL BUECH MEOUGE":"05","PELVOUX":"05","LA ROCHE DES ARNAUDS":"05","PUY ST EUSEBE":"05","PUY ST ANDRE":"05","PUY SANIERES":"05","RISTOLAS":"05","ORPIERRE":"05","OIGNIES":"62","OUVE WIRQUIN":"62","PALLUEL":"62","PIHEM":"62","PIHEN LES GUINES":"62","PITTEFAUX":"62","PLANQUES":"62","LE PORTEL":"62","QUEANT":"62","QUERCAMPS":"62","LE QUESNOY EN ARTOIS":"62","REBERGUES":"62","REBREUVE SUR CANCHE":"62","RECOURT":"62","RECQUES SUR COURSE":"62","RENTY":"62","ROCLINCOURT":"62","RUISSEAUVILLE":"62","RUITZ":"62","SACHIN":"62","ST MARTIN CHOQUEL":"62","ST MARTIN D HARDINGHEM":"62","ST REMY AU BOIS":"62","ST TRICAT":"62","SAPIGNIES":"62","SAUCHY CAUCHY":"62","SAUDEMONT":"62","SERICOURT":"62","SETQUES":"62","SUS ST LEGER":"62","TIGNY NOYELLE":"62","TILLOY LES HERMAVILLE":"62","TORTEQUESNE":"62","TOURNEHEM SUR LA HEM":"62","VENDIN LE VIEIL":"62","VERLINCTHUN":"62","VIEILLE EGLISE":"62","VIEIL MOUTIER":"62","VILLERS AU BOIS":"62","VILLERS CHATEL":"62","WACQUINGHEN":"62","WAILLY":"62","WIERRE AU BOIS":"62","WILLERVAL":"62","WITTERNESSE":"62","ZOTEUX":"62","ZUTKERQUE":"62","ANZAT LE LUGUET":"63","AUGEROLLES":"63","AYAT SUR SIOULE":"63","BEURIERES":"63","BLANZAT":"63","LA BOURBOULE":"63","BOURG LASTIC":"63","BROMONT LAMOTHE":"63","LE BRUGERON":"63","BUSSIERES ET PRUNS":"63","CELLES SUR DUROLLE":"63","CHABRELOCHE":"63","CHAMALIERES":"63","CHATEAUGAY":"63","CHATEAUNEUF LES BAINS":"63","CHATELDON":"63","CHAURIAT":"63","CHAVAROUX":"63","CLERMONT FERRAND":"63","COMBRONDE":"63","COMPAINS":"63","CREVANT LAVEINE":"63","DALLET":"63","DAUZAT SUR VODABLE":"63","ECHANDELYS":"63","ENVAL":"63","ESCOUTOUX":"63","FOURNOLS":"63","GRANDEYROLLES":"63","GRANDVAL":"63","ISSERTEAUX":"63","JOZERAND":"63","LAPS":"63","LEMPDES":"63","LEZOUX":"63","LIMONS":"63","LUDESSE":"63","MALAUZAT":"63","MANZAT":"63","MAREUGHEOL":"63","MAUZUN":"63","MAZAYE":"63","BEUZEVILLE LA GUERARD":"76","BLANGY SUR BRESLE":"76","BOIS GUILBERT":"76","BOLLEVILLE":"76","CALLENGEVILLE":"76","BOSC MESNIL":"76","BRACQUETUIT":"76","BRADIANCOURT":"76","BRAMETOT":"76","CANVILLE LES DEUX EGLISES":"76","RIVES EN SEINE":"76","CLIPONVILLE":"76","COMPAINVILLE":"76","CRIEL SUR MER":"76","CRIQUIERS":"76","ECALLES ALIX":"76","ECRETTEVILLE SUR MER":"76","ELBEUF SUR ANDELLE":"76","ELLECOURT":"76","ERNEMONT LA VILLETTE":"76","ESCLAVELLES":"76","ESTEVILLE":"76","ETALONDES":"76","FERRIERES EN BRAY":"76","LA FERTE ST SAMSON":"76","FONTAINE SOUS PREAUX":"76","LA FONTELAYE":"76","FREAUVILLE":"76","FRESNOY FOLNY":"76","ST MARTIN DE L IF":"76","GONFREVILLE L ORCHER":"76","GONNEVILLE LA MALLET":"76","GOURNAY EN BRAY":"76","GRAIMBOUVILLE":"76","GRAND COURONNE":"76","GREGES":"76","GRIGNEUSEVILLE":"76","GRUCHET LE VALASSE":"76","GRUCHET ST SIMEON":"76","GUEUTTEVILLE LES GRES":"76","LE HAVRE":"76","LE HERON":"76","HOUDETOT":"76","HOUPPEVILLE":"76","HUGLEVILLE EN CAUX":"76","LES IFS":"76","LANDES VIEILLES ET NEUVES":"76","LONGUERUE":"76","MANIQUERVILLE":"76","MANNEVILLE LA GOUPIL":"76","LE MESNIL ESNARD":"76","MONTROTY":"76","MOTTEVILLE":"76","MUCHEDENT":"76","NEUVILLE FERRIERES":"76","OUDALLE":"76","OURVILLE EN CAUX":"76","OUVILLE LA RIVIERE":"76","PLEINE SEVE":"76","QUINCAMPOIX":"76","RAFFETOT":"76","REALCAMP":"76","REUVILLE":"76","ROUEN":"76","ROUTES":"76","RY":"76","SAHURS":"76","ST ANDRE SUR CAILLY":"76","ST AUBIN EPINAY":"76","ST AUBIN LES ELBEUF":"76","ST AUBIN LE CAUF":"76","STE BEUVE EN RIVIERE":"76","ST GILLES DE CRETOT":"76","ST GILLES DE LA NEUVILLE":"76","STE HELENE BONDEVILLE":"76","ST HELLIER":"76","ST JACQUES SUR DARNETAL":"76","ST MACLOU DE FOLLEVILLE":"76","ST MARDS":"76","STE MARGUERITE SUR MER":"76","STE MARGUERITE SUR FAUVILLE":"76","ST MARTIN AUX BUNEAUX":"76","PETIT CAUX":"76","ENGLANCOURT":"02","DEUILLET":"02","ESSISES":"02","ERLOY":"02","CHIVY LES ETOUVELLES":"02","CHEVRESIS MONCEAU":"02","CONDE SUR AISNE":"02","CHEZY SUR MARNE":"02","CLASTRES":"02","CIERGES":"02","CHERET":"02","CHOUY":"02","CILLY":"02","COURCELLES SUR VESLE":"02","CUISSY ET GENY":"02","CROIX FONSOMME":"02","CRAONNELLE":"02","COURBOIN":"02","CRUPILLY":"02","CORBENY":"02","COUPRU":"02","DALLON":"02","FROIDMONT COHARTILLE":"02","FAUCOUCOURT":"02","GERNICOURT":"02","FOLEMBRAY":"02","FESTIEUX":"02","ETREUX":"02","GERGNY":"02","LIESSE NOTRE DAME":"02","MAREST DAMPCOURT":"02","MAREUIL EN DOLE":"02","MARFONTAINE":"02","LOUPEIGNE":"02","LESDINS":"02","LAUNOY":"02","LIZY":"02","GOUSSANCOURT":"02","GIBERCOURT":"02","HOMBLIERES":"02","JUMENCOURT":"02","LANISCOURT":"02","ITANCOURT":"02","HINACOURT":"02","HAUTION":"02","JUMIGNY":"02","HOLNON":"02","NEUFCHATEL SUR AISNE":"02","MONTIGNY SUR CRECY":"02","MONTLOUE":"02","MUSCOURT":"02","MONCEAU LE NEUF ET FAUCOUZY":"02","MERLIEUX ET FOUQUEROLLES":"02","MONTIGNY SOUS MARLE":"02","MONTIGNY LENGRAIN":"02","MAUREGNY EN HAYE":"02","MARLY GOMONT":"02","MONTGOBERT":"02","MENNEVRET":"02","LA NEUVILLE EN BEINE":"02","PROVISEUX ET PLESNOY":"02","QUINCY BASSE":"02","PROUVAIS":"02","PLOMION":"02","RESIGNY":"02","PRISCES":"02","ORIGNY EN THIERACHE":"02","NOUVION LE VINEUX":"02","NOUVION LE COMTE":"02","OULCHY LA VILLE":"02","PASSY SUR MARNE":"02","NOROY SUR OURCQ":"02","OSLY COURTIL":"02","NOIRCOURT":"02","PARGNAN":"02","HAUT BOCAGE":"03","MONETAY SUR ALLIER":"03","MONTILLY":"03","MOLINET":"03","LA CROIX SUR ROUDOULE":"06","VILLAR D ARENE":"05","VAL DES PRES":"05","LE CANNET":"06","ST MALO":"35","ST MALO DE PHILY":"35","ST MEEN LE GRAND":"35","ST MELOIR DES ONDES":"35","ST ONEN LA CHAPELLE":"35","ST OUEN LA ROUERIE":"35","LA SELLE EN COGLES":"35","SERVON SUR VILAINE":"35","TALENSAC":"35","TRANS LA FORET":"35","TREMBLAY":"35","VIGNOC":"35","PONT PEAN":"35","AIGURANDE":"36","ARDENTES":"36","BARAIZE":"36","BRIANTES":"36","CHAILLAC":"36","LA CHAMPENOISE":"36","LA CHAPELLE ST LAURIAN":"36","LA CHATRE LANGLIN":"36","CHITRAY":"36","CIRON":"36","CLERE DU BOIS":"36","FONTGOMBAULT":"36","FONTGUENAND":"36","GARGILESSE DAMPIERRE":"36","GIROUX":"36","LOUROUER ST LAURENT":"36","LYS ST GEORGES":"36","MAILLET":"36","MENETREOLS SOUS VATAN":"36","MERIGNY":"36","MEUNET PLANCHES":"36","MEUNET SUR VATAN":"36","MIGNE":"36","MONTCHEVRIER":"36","MONTIERCHAUME":"36","MONTLEVICQ":"36","NERET":"36","OBTERRE":"36","PALLUAU SUR INDRE":"36","LE PONT CHRETIEN CHABENET":"36","POULIGNY ST MARTIN":"36","POULIGNY ST PIERRE":"36","PREUILLY LA VILLE":"36","ST AOUT":"36","ST DENIS DE JOUHET":"36","ST GAULTIER":"36","ST LACTENCIN":"36","STE LIZAIGNE":"36","STE SEVERE SUR INDRE":"36","SASSIERGES ST GERMAIN":"36","SEGRY":"36","VALENCAY":"36","VAL FOUZON":"36","VIGOUX":"36","BEAUMONT LA RONCE":"37","BENAIS":"37","BOURNAN":"37","BRASLOU":"37","BRAYE SOUS FAYE":"37","LA CELLE GUENAND":"37","CHANCEAUX PRES LOCHES":"37","CHARGE":"37","CHARNIZAY":"37","CHAVEIGNES":"37","CHEDIGNY":"37","CHISSEAUX":"37","ESTEIL":"63","FAYET RONAYE":"63","FERNOEL":"63","LA GODIVELLE":"63","GOUTTIERES":"63","JOB":"63","JOZE":"63","LAQUEUILLE":"63","LARODDE":"63","MARCILLAT":"63","LES MARTRES DE VEYRE":"63","MESSEIX":"63","MOISSAT":"63","MORIAT":"63","CHAMBARON SUR MORGE":"63","RAMBAUD":"05","ST BONNET EN CHAMPSAUR":"05","ST JULIEN EN BEAUCHENE":"05","ST MICHEL DE CHAILLOL":"05","LA SALLE LES ALPES":"05","ST AUBAN D OZE":"05","SAVINES LE LAC":"05","SALEON":"05","CHATEAUNEUF D ENTRAUNES":"06","LA COLLE SUR LOUP":"06","ESCRAGNOLLES":"06","CANTARON":"06","LA GAUDE":"06","GILETTE":"06","DALUIS":"06","ISOLA":"06","UTELLE":"06","VALLAURIS":"06","VILLENEUVE D ENTRAUNES":"06","TOURETTE DU CHATEAU":"06","VALDEROURE":"06","LA TURBIE":"06","ANNONAY":"07","ANDANCE":"07","PEILLE":"06","NICE":"06","LES MUJOULS":"06","LANTOSQUE":"06","MASSOINS":"06","PIERLAS":"06","LUCERAM":"06","RIMPLAS":"06","LEVENS":"06","ST CEZAIRE SUR SIAGNE":"06","ST ANDRE DE LA ROCHE":"06","ROQUEBRUNE CAP MARTIN":"06","ROQUEFORT LES PINS":"06","ST MARTIN DU VAR":"06","ROQUEBILLIERE":"06","THIERY":"06","TOUDON":"06","ANTIBES":"06","CANNES":"06","AURIBEAU SUR SIAGNE":"06","BEAUSOLEIL":"06","CABRIS":"06","ASCROS":"06","LE BEAGE":"07","AUBENAS":"07","BIDON":"07","VILLENEUVE DU LATOU":"09","ALLIBAUDIERES":"10","ARREMBECOURT":"10","AVIREY LINGEY":"10","ARRENTIERES":"10","ARCONVILLE":"10","ARSONVAL":"10","ASSENAY":"10","FRESNOY LE CHATEAU":"10","FONTAINE LES GRES":"10","LA FOSSE CORDUAN":"10","ERVY LE CHATEL":"10","ISLE AUBIGNY":"10","JASSEINES":"10","FRAVAUX":"10","ENGENTE":"10","ETOURVY":"10","ESSOYES":"10","DIERREY ST JULIEN":"10","COLOMBE LA FOSSE":"10","CHARNY LE BACHOT":"10","COURSAN EN OTHE":"10","CHESSY LES PRES":"10","CHAUMESNIL":"10","ECHEMINES":"10","CHESLEY":"10","DAVREY":"10","ST PIERRE DE RIVIERE":"09","STE CROIX VOLVESTRE":"09","ST PAUL DE JARRAT":"09","ST JEAN DU FALGA":"09","TOURTOUSE":"09","ST YBARS":"09","SALSEIN":"09","MEILHAUD":"63","LA MONNERIE LE MONTEL":"63","MONTAIGUT":"63","NEUF EGLISE":"63","NONETTE ORSONNETTE":"63","OLLOIX":"63","PARDINES":"63","PARENT":"63","PASLIERES":"63","PERIGNAT SUR ALLIER":"63","PICHERANDE":"63","POUZOL":"63","ROMAGNAT":"63","ROYAT":"63","ST ALYRE D ARLANC":"63","ST BONNET LE CHASTEL":"63","ST BONNET LES ALLIER":"63","ST DENIS COMBARNAZAT":"63","ST FLORET":"63","ST GERMAIN LEMBRON":"63","ST GERVAIS D AUVERGNE":"63","ST IGNAT":"63","ST JULIEN DE COPPEL":"63","ST MAIGNER":"63","ST PIERRE LE CHASTEL":"63","ST PIERRE ROCHE":"63","ST VICTOR MONTVIANEIX":"63","ST YVOINE":"63","SAUVAGNAT STE MARTHE":"63","TAUVES":"63","TERNANT LES EAUX":"63","THIERS":"63","TREMOUILLE ST LOUP":"63","VALBELEIX":"63","VALZ SOUS CHATEAUNEUF":"63","VASSEL":"63","VERNEUGHEOL":"63","VERNINES":"63","VERTOLAYE":"63","VILLENEUVE LES CERFS":"63","VISCOMTAT":"63","VIVEROLS":"63","VOLLORE MONTAGNE":"63","ABOS":"64","AICIRITS CAMOU SUHAST":"64","AMENDEUIX ONEIX":"64","ANDOINS":"64","ANGOUS":"64","ANHAUX":"64","ANOS":"64","ARBERATS SILLEGUE":"64","ARBONNE":"64","ARETTE":"64","ARNEGUY":"64","ARRICAU BORDES":"64","ARZACQ ARRAZIGUET":"64","ASTE BEON":"64","AURIONS IDERNES":"64","AUTEVIELLE ST MARTIN BIDEREN":"64","BALANSUN":"64","BARZUN":"64","BAYONNE":"64","BEHASQUE LAPISTE":"64","BERNADETS":"64","BERROGAIN LARUNS":"64","BIDARRAY":"64","BILLERE":"64","BOSDARROS":"64","BOUGARBER":"64","ST MARTIN DE COMMUNE":"71","ST MARTIN DU LAC":"71","ST MARTIN DU TARTRE":"71","ST NIZIER SUR ARROUX":"71","ST RACHO":"71","ST SYMPHORIEN DES BOIS":"71","SASSANGY":"71","SAVIGNY SUR GROSNE":"71","SOLUTRE POUILLY":"71","TANCON":"71","TINTRY":"71","TRONCHY":"71","UCHON":"71","VENDENESSE SUR ARROUX":"71","LE VILLARS":"71","VILLEGAUDIN":"71","VILLENEUVE EN MONTAGNE":"71","ST MARTIN LE GAILLARD":"76","ST NICOLAS D ALIERMONT":"76","ST NICOLAS DE LA HAIE":"76","ST PIERRE BENOUVILLE":"76","ST PIERRE DES JONQUIERES":"76","ST RIQUIER ES PLAINS":"76","ST VALERY EN CAUX":"76","SANDOUVILLE":"76","SASSETOT LE MAUCONDUIT":"76","SAUQUEVILLE":"76","SEVIS":"76","SMERMESNIL":"76","SOTTEVILLE LES ROUEN":"76","THEUVILLE AUX MAILLOTS":"76","VAL DE LA HAYE":"76","VARENGEVILLE SUR MER":"76","VASSONVILLE":"76","VATTETOT SUR MER":"76","LA VAUPALIERE":"76","LA VIEUX RUE":"76","VILLAINVILLE":"76","YVETOT":"76","AMILLIS":"77","AUBEPIERRE OZOUER LE REPOS":"77","AUFFERVILLE":"77","SORNEVILLE":"54","SPONVILLE":"54","TELLANCOURT":"54","TUCQUEGNIEUX":"54","UGNY":"54","URUFFE":"54","VALLOIS":"54","VAXAINVILLE":"54","VILLERS LA CHEVRE":"54","VILLERS LA MONTAGNE":"54","VILLERS SOUS PRENY":"54","VILLEY LE SEC":"54","VILLEY ST ETIENNE":"54","XAMMES":"54","XERMAMENIL":"54","AINCREVILLE":"55","AMBLY SUR MEUSE":"55","APREMONT LA FORET":"55","AULNOIS EN PERTHOIS":"55","BAR LE DUC":"55","BEAUCLAIR":"55","BEAUMONT EN VERDUNOIS":"55","BEAUSITE":"55","BILLY SOUS MANGIENNES":"55","BISLEE":"55","BLANZEE":"55","BONZEE":"55","BRABANT EN ARGONNE":"55","BRABANT LE ROI":"55","BRABANT SUR MEUSE":"55","BRAUVILLIERS":"55","BROUENNES":"55","BUZY DARMONT":"55","CESSE":"55","CHARDOGNE":"55","CHATILLON SOUS LES COTES":"55","CHAUVENCY ST HUBERT":"55","LE CLAON":"55","CLERMONT EN ARGONNE":"55","COMBLES EN BARROIS":"55","LES HAUTS DE CHEE":"55","COURCELLES SUR AIRE":"55","COUVONGES":"55","DOMBASLE EN ARGONNE":"55","DUZEY":"55","ECUREY EN VERDUNOIS":"55","EPINONVILLE":"55","ERIZE ST DIZIER":"55","ESNES EN ARGONNE":"55","FROMEREVILLE LES VALLONS":"55","HAN LES JUVIGNY":"55","HAN SUR MEUSE":"55","HAUDIOMONT":"55","GEVILLE":"55","KOEUR LA PETITE":"55","LATOUR EN WOEVRE":"55","LIGNIERES SUR AIRE":"55","LINY DEVANT DUN":"55","LOUPMONT":"55","LOUPPY SUR LOISON":"55","MAIZERAY":"55","MARTINCOURT SUR MEUSE":"55","MAUCOURT SUR ORNE":"55","MECRIN":"55","MENAUCOURT":"55","MILLY SUR BRADON":"55","MONTFAUCON D ARGONNE":"55","MONTIERS SUR SAULX":"55","MONTSEC":"55","MONTZEVILLE":"55","MORANVILLE":"55","MORGEMOULIN":"55","VENTAVON":"05","BAIROLS":"06","AUVARE":"06","UPAIX":"05","MANDELIEU LA NAPOULE":"06","GRASSE":"06","MOUGINS":"06","LIEUCHE":"06","ILONSE":"06","DRAP":"06","GARS":"06","ST VINCENT DE BARRES":"07","ST ROMAIN D AY":"07","SATILLIEU":"07","SANILHAC":"07","SALAVAS":"07","SARRAS":"07","ST MARTIN VESUBIE":"06","TOURRETTES SUR LOUP":"06","LA ROQUETTE SUR VAR":"06","TOURNEFORT":"06","LE TIGNET":"06","LE ROURET":"06","SOSPEL":"06","PUGET THENIERS":"06","PUGET ROSTANG":"06","PIERREFEU":"06","LA PENNE":"06","PEGOMAS":"06","PEILLON":"06","PEONE":"06","ANTRAIGUES SUR VOLANE":"07","VILLEFRANCHE SUR MER":"06","VILLENEUVE LOUBET":"06","ARLEBOSC":"07","VENANSON":"06","ARDOIX":"07","AJOUX":"07","AIZAC":"07","BANNE":"07","BAIX":"07","VALS LES BAINS":"07","TALENCIEUX":"07","LA SOUCHE":"07","SCEAUTRES":"07","TAURIERS":"07","TOULAUD":"07","COLOMBIER LE CARDINAL":"07","BERRIAS ET CASTELJAU":"07","CHARMES SUR RHONE":"07","CELLIER DU LUC":"07","BEAUCHASTEL":"07","COUCOURON":"07","CHOMERAC":"07","BOFFRES":"07","GILHOC SUR ORMEZE":"07","ECLASSAN":"07","ISSARLES":"07","DOMPNAC":"07","ETABLES":"07","DORNAS":"07","GRAS":"07","VERNOSC LES ANNONAY":"07","AMBLY FLEURY":"08","ANCHAMPS":"08","VOCANCE":"07","VEYRAS":"07","AIRE":"08","LES OLLIERES SUR EYRIEUX":"07","MONTSELGUES":"07","PLANZOLLES":"07","MALBOSC":"07","LUSSAS":"07","LYAS":"07","LACHAPELLE SOUS CHANEAC":"07","LAVILLEDIEU":"07","LAVILLATTE":"07","NEBOUZAT":"63","NERONDE SUR DORE":"63","NESCHERS":"63","NOVACELLES":"63","ORCET":"63","ORCIVAL":"63","PLAUZAT":"63","PONT DU CHATEAU":"63","PROMPSAT":"63","PUY ST GULMIER":"63","LE QUARTIER":"63","RANDAN":"63","RENTIERES":"63","RIOM":"63","ROCHE CHARLES LA MAYRAND":"63","ROCHE D AGOUX":"63","LA ROCHE NOIRE":"63","ST ALYRE ES MONTAGNE":"63","ST AMANT TALLENDE":"63","ST ANDRE LE COQ":"63","ST BONNET LE BOURG":"63","ST GENES CHAMPANELLE":"63","ST HILAIRE LA CROIX":"63","ST MARTIN DES PLAINS":"63","ST PRIEST DES CHAMPS":"63","ST QUENTIN SUR SAUXILLANGES":"63","ST SAUVES D AUVERGNE":"63","ST SYLVESTRE PRAGOULIN":"63","ST VICTOR LA RIVIERE":"63","SAUVAGNAT":"63","SAUVESSANGES":"63","TEILHEDE":"63","THIOLIERES":"63","VALCIVIERES":"63","VARENNES SUR USSON":"63","LE VERNET STE MARGUERITE":"63","VERTAIZON":"63","VODABLE":"63","AINCILLE":"64","ALCAY ALCABEHETY SUNHARETTE":"64","AMOROTS SUCCOS":"64","ANOYE":"64","ARAUJUZON":"64","ARBOUET SUSSAUTE":"64","ARGET":"64","ARHANSUS":"64","AROUE ITHOROTS OLHAIBY":"64","ARRAST LARREBIEU":"64","ASCAIN":"64","AUBERTIN":"64","AUSSEVIELLE":"64","BAIGTS DE BEARN":"64","BARCUS":"64","BEGUIOS":"64","BENEJACQ":"64","BESCAT":"64","BIARRITZ":"64","BIDOS":"64","BILHERES":"64","BOUMOURT":"64","BOURDETTES":"64","BRUGES CAPBIS MIFAGET":"64","BUROSSE MENDOUSSE":"64","BUSTINCE IRIBERRY":"64","BUZIET":"64","CASTEIDE DOAT":"64","CASTETNAU CAMBLONG":"64","CASTETNER":"64","CETTE EYGUN":"64","CONCHEZ DE BEARN":"64","CUQUERON":"64","ESLOURENTIES DABAN":"64","ESPELETTE":"64","ESTIALESCQ":"64","ETSAUT":"64","GAMARTHE":"64","GAYON":"64","GOMER":"64","IDAUX MENDY":"64","IHOLDY":"64","ISPOURE":"64","ISTURITS":"64","IZESTE":"64","JASSES":"64","LABASTIDE VILLEFRANCHE":"64","LABATMALE":"64","LACOMMANDE":"64","LACQ":"64","TIGNAC":"09","USSAT":"09","VALS":"09","OSSEY LES TROIS MAISONS":"10","LE PAVILLON STE JULIE":"10","MAROLLES LES BAILLY":"10","POUY SUR VANNES":"10","POLISY":"10","CHARMONT SOUS BARBUISE":"10","BRIENNE LE CHATEAU":"10","CHAMPIGNY SUR AUBE":"10","BUXIERES SUR ARCE":"10","BUCEY EN OTHE":"10","BAR SUR SEINE":"10","CHAMOY":"10","MAIZIERES LES BRIENNE":"10","MAISONS LES SOULAINES":"10","MAISON DES CHAMPS":"10","LUSIGNY SUR BARSE":"10","JULLY SUR SARCE":"10","LONGPRE LE SEC":"10","MAGNICOURT":"10","JESSAINS":"10","LHUITRE":"10","MAGNANT":"10","ORVILLIERS ST JULIEN":"10","MESNIL LA COMTESSE":"10","NOGENT SUR SEINE":"10","MUSSY SUR SEINE":"10","PERIGNY LA ROSE":"10","MARAYE EN OTHE":"10","LA MOTTE TILLY":"10","MONTAULIN":"10","LE MERIOT":"10","MAUVEZIN DE STE CROIX":"09","MERENS LES VALS":"09","LAVELANET":"09","LESPARROU":"09","JUSTINIAC":"09","LOUBAUT":"09","LARBONT":"09","L HERM":"09","LARNAT":"09","ROQUEFORT LES CASCADES":"09","PERLES ET CASTELET":"09","MONTAGAGNE":"09","PRADIERES":"09","PEREILLE":"09","NALZEN":"09","ORLU":"09","ROMILLY SUR SEINE":"10","RIGNY LE FERRON":"10","PRUSY":"10","ST ETIENNE SOUS BARBUISE":"10","ST JULIEN LES VILLAS":"10","ST JEAN DE BONNEVAL":"10","ROUVRES LES VIGNES":"10","RUMILLY LES VAUDES":"10","LA ROTHIERE":"10","ST POUANGE":"10","ST LUPIEN":"10","ST PHAL":"10","VITRY LE CROISE":"10","VILLY LE BOIS":"10","ALAIGNE":"11","VILLERY":"10","ALZONNE":"11","ASSAC":"81","AUSSAC":"81","BEAUVAIS SUR TESCOU":"81","BERTRE":"81","BROZE":"81","CADALEN":"81","CARMAUX":"81","FONTRIEU":"81","COUFOULEUX":"81","COURRIS":"81","ESCROUX":"81","FAUSSERGUES":"81","LE FRAYSSE":"81","LABESSIERE CANDEIL":"81","LABOULBENE":"81","LACAZE":"81","LACROUZETTE":"81","ANCINNES":"72","ASSE LE RIBOUL":"72","AVESNES EN SAOSNOIS":"72","AVESSE":"72","LE BAILLEUL":"72","BEAUFAY":"72","BEAUMONT SUR DEME":"72","LE BREIL SUR MERIZE":"72","CHALLES":"72","CHAMPFLEUR":"72","LA CHAPELLE D ALIGNE":"72","CHATEAU DU LOIR":"72","CHATEAU L HERMITAGE":"72","CHENU":"72","LE CHEVAIN":"72","CHEVILLE":"72","COMMERVEIL":"72","CONFLANS SUR ANILLE":"72","COURCELLES LA FORET":"72","CRANNES EN CHAMPAGNE":"72","CURES":"72","DISSAY SOUS COURCILLON":"72","DISSE SOUS BALLON":"72","DOUCELLES":"72","LA FERTE BERNARD":"72","LAIGNE EN BELIN":"72","LIVET EN SAOSNOIS":"72","LUCEAU":"72","MARIGNE LAILLE":"72","MELLERAY":"72","MEURCE":"72","MONTBIZOT":"72","MOULINS LE CARBONNEL":"72","NEUVILLALAIS":"72","NEUVILLETTE EN CHARNIE":"72","NOYEN SUR SARTHE":"72","PARIGNE L EVEQUE":"72","PIACE":"72","PINCE":"72","PIRMIL":"72","PREVAL":"72","ST CHRISTOPHE DU JAMBET":"72","ST CORNEILLE":"72","ST COSME EN VAIRAIS":"72","STE JAMME SUR SARTHE":"72","ST JEAN D ASSE":"72","ST MAIXENT":"72","ST MARTIN DES MONTS":"72","ST OUEN EN BELIN":"72","ST PATERNE":"72","ST PIERRE DE CHEVILLE":"72","ST ULPHACE":"72","SAVIGNE SOUS LE LUDE":"72","SILLE LE GUILLAUME":"72","SOULIGNE FLACE":"72","SOUVIGNE SUR MEME":"72","THOIGNE":"72","THOIRE SOUS CONTENSOR":"72","TRANGE":"72","TRESSON":"72","VIBRAYE":"72","VILLAINES SOUS LUCE":"72","VIRE EN CHAMPAGNE":"72","AITON":"73","ALLONDAZ":"73","AUSSOIS":"73","AVRIEUX":"73","LA BALME":"73","LA BAUCHE":"73","BELMONT TRAMONET":"73","BETTON BETTONET":"73","LE BOIS":"73","BONVILLARET":"73","CEVINS":"73","CHALLES LES EAUX":"73","LA CHAPELLE DU MONT DU CHAT":"73","LA CHAPELLE ST MARTIN":"73","LA COMPOTE":"73","LA CROIX DE LA ROCHETTE":"73","CRUET":"73","MOULOTTE":"55","MURVAUX":"55","VAL D ORNAIN":"55","MUZERAY":"55","NAIVES ROSIERES":"55","NANCOIS SUR ORNAIN":"55","NANTOIS":"55","NIXEVILLE BLERCOURT":"55","NUBECOURT":"55","PEUVILLERS":"55","POUILLY SUR MEUSE":"55","PRETZ EN ARGONNE":"55","REMOIVILLE":"55","REVILLE AUX BOIS":"55","LES ROISES":"55","RAIVAL":"55","RUPT DEVANT ST MIHIEL":"55","RUPT EN WOEVRE":"55","ST ANDRE EN BARROIS":"55","ST HILAIRE EN WOEVRE":"55","ST JULIEN SOUS LES COTES":"55","SAUDRUPT":"55","SAULX LES CHAMPLON":"55","SAUVIGNY":"55","SEPTSARGES":"55","LES SOUHESMES RAMPONT":"55","TANNOIS":"55","THILLOT":"55","SEUIL D ARGONNE":"55","COUSANCES LES TRICONVILLE":"55","VAUX DEVANT DAMLOUP":"55","VAUX LES PALAMEIX":"55","VERNEUIL GRAND":"55","VERNEUIL PETIT":"55","VIGNEULLES LES HATTONCHATEL":"55","VILLE DEVANT CHAUMONT":"55","VILOSNES HARAUMONT":"55","VITTARVILLE":"55","VOUTHON BAS":"55","WALY":"55","WAVRILLE":"55","WILLERONCOURT":"55","ARZON":"56","BEIGNON":"56","BERNE":"56","BILLIERS":"56","BOHAL":"56","BRECH":"56","CAMORS":"56","CARNAC":"56","CLEGUER":"56","COURNON":"56","EVRIGUET":"56","LA GACILLY":"56","GUER":"56","HENNEBONT":"56","HOEDIC":"56","INZINZAC LOCHRIST":"56","KERGRIST":"56","KERVIGNAC":"56","LANTILLAC":"56","LOCMALO":"56","LOCMARIA":"56","LOCMIQUELIC":"56","LOYAT":"56","MALESTROIT":"56","MESLAN":"56","EVELLYS":"56","NEANT SUR YVEL":"56","NOSTANG":"56","PLOEMEUR":"56","PLOEREN":"56","PLOUAY":"56","PORCARO":"56","RADENAC":"56","ROHAN":"56","LE SAINT":"56","ST GONNERY":"56","ST JEAN BREVELAY":"56","ST PIERRE QUIBERON":"56","SURZUR":"56","TREDION":"56","LA TRINITE SURZUR":"56","ABRESCHVILLER":"57","AJONCOURT":"57","ALBESTROFF":"57","ALSTING":"57","ANGEVILLERS":"57","ASSENONCOURT":"57","AUDUN LE TICHE":"57","LARGENTIERE":"07","LANARCE":"07","JUVINAS":"07","JOANNAS":"07","LABOULE":"07","LIMONY":"07","ARTAISE LE VIVIER":"08","LES AYVELLES":"08","BAYONVILLE":"08","AUBRIVES":"08","ANTHENY":"08","BALLAY":"08","AVAUX":"08","ST CIERGE SOUS LE CHEYLARD":"07","ST ANDEOL DE FOURCHADES":"07","ST CIRGUES EN MONTAGNE":"07","ST ETIENNE DE BOULOGNE":"07","ST BARTHELEMY LE PLAIN":"07","ST ANDRE EN VIVARAIS":"07","ST BARTHELEMY GROZON":"07","ST ANDRE LACHAMP":"07","ST JACQUES D ATTICIEUX":"07","ST LAURENT SOUS COIRON":"07","ST MARTIN SUR LAVEZON":"07","ST GENEST DE BEAUZON":"07","ST MICHEL D AURANCE":"07","ST JULIEN BOUTIERES":"07","ST PIERREVILLE":"07","ST JEAN ROURE":"07","ST ALBAN AURIOLLES":"07","PONT DE LABEAUME":"07","ST ALBAN D AY":"07","ROCHESSAUVE":"07","SABLIERES":"07","PRANLES":"07","PRADONS":"07","LE ROUX":"07","RIBES":"07","BEFFU ET LE MORTHOMME":"08","BOSSEVAL ET BRIANCOURT":"08","BIGNICOURT":"08","BLOMBAY":"08","DOM LE MESNIL":"08","ESCOMBRES ET LE CHESNOIS":"08","COULOMMES ET MARQUENY":"08","LA CROIX AUX BOIS":"08","CONDE LES AUTRY":"08","CHUFFILLY ROCHE":"08","ETEIGNIERES":"08","CLIRON":"08","COUCY":"08","CARIGNAN":"08","BRIENNE SUR AISNE":"08","BOUCONVILLE":"08","CHARNOIS":"08","BULSON":"08","SEVIGNY WALEPPE":"08","SAULCES MONCLIN":"08","SECHEVAL":"08","SENUC":"08","DIGNAC":"16","ECURAS":"16","ETAGNAC":"16","EYMOUTHIERS":"16","LA FAYE":"16","FLEAC":"16","LA FORET DE TESSE":"16","FOUQUEBRUNE":"16","FOUQUEURE":"16","GENAC BIGNAC":"16","GRASSAC":"16","GUIMPS":"16","GURAT":"16","JAULDES":"16","VAL DES VIGNES":"16","LESSAC":"16","LICHERES":"16","LOUZAC ST ANDRE":"16","LA MAGDELEINE":"16","MAGNAC SUR TOUVRE":"16","MANOT":"16","MESNAC":"16","MONTIGNAC CHARENTE":"16","LAGUINGE RESTOUE":"64","LAHOURCADE":"64","LANNEPLAA":"64","LARUNS":"64","LAY LAMIDOU":"64","LESCUN":"64","LICHOS":"64","LICQ ATHEREY":"64","LIMENDOUS":"64","LOUVIE JUZON":"64","LURBE ST CHRISTAU":"64","MENDIVE":"64","MERACQ":"64","MERITEIN":"64","MIOSSENS LANUSSE":"64","MORLAAS":"64","MORLANNE":"64","MOUGUERRE":"64","NOUSTY":"64","OREGUE":"64","ORION":"64","OSTABAT ASME":"64","OUSSE":"64","OZENX MONTESTRUCQ":"64","PAGOLLE":"64","ROQUIAGUE":"64","ST CASTIN":"64","ST GIRONS EN BEARN":"64","ST JEAN DE LUZ":"64","ST JEAN PIED DE PORT":"64","ST MARTIN D ARROSSA":"64","ST PEE SUR NIVELLE":"64","SALIES DE BEARN":"64","SALLES MONGISCARD":"64","SAMSONS LION":"64","SARPOURENX":"64","SAUBOLE":"64","SIROS":"64","TADOUSSE USSAU":"64","URDOS":"64","VIELLESEGURE":"64","ADERVIELLE POUCHERGUES":"65","ANDREST":"65","ANLA":"65","ANSOST":"65","ARCIZANS AVANT":"65","ARCIZANS DESSUS":"65","ARNE":"65","ARRENS MARSOUS":"65","ASQUE":"65","AUBAREDE":"65","BAGNERES DE BIGORRE":"65","BARBAZAN DEBAT":"65","BARRANCOUEU":"65","BARRY":"65","LA BARTHE DE NESTE":"65","BEGOLE":"65","BETPOUEY":"65","BIZOUS":"65","BORDERES SUR L ECHEZ":"65","BOUILH DEVANT":"65","BOURREAC":"65","BUGARD":"65","BULAN":"65","CAMOUS":"65","CAMPAN":"65","CASTELBAJAC":"65","CASTELVIEILH":"65","CIZOS":"65","CLARENS":"65","ESCALA":"65","ESCONDEAUX":"65","ESPECHE":"65","GALAN":"65","GALEZ":"65","GAZAVE":"65","GEZ":"65","GOURGUE":"65","GRUST":"65","GUCHAN":"65","HAGEDET":"65","HIBARETTE":"65","HIIS":"65","IBOS":"65","IZAOURT":"65","IZAUX":"65","LACASSAGNE":"65","LEDAS ET PENTHIES":"81","LOMBERS":"81","MAZAMET":"81","MIOLLES":"81","MONTANS":"81","MONTDURAUSSE":"81","MOULAYRES":"81","PAMPELONNE":"81","PAYRIN AUGMONTEL":"81","POULAN POUZOLS":"81","LE RIALET":"81","ROQUECOURBE":"81","ST GAUZENS":"81","ST SALVY DE LA BALME":"81","SENAUX":"81","SENOUILLAC":"81","SOREZE":"81","TANUS":"81","LE TRAVET":"81","VENES":"81","LE VERDIER":"81","VINDRAC ALAYRAC":"81","ANGEVILLE":"82","LES BARTHES":"82","BRESSOLS":"82","CASTELMAYRAN":"82","CAUSSADE":"82","CAYRAC":"82","CAYRIECH":"82","CORBARIEU":"82","DUNES":"82","FAUROUX":"82","FENEYROLS":"82","GENEBRIERES":"82","GLATENS":"82","GOLFECH":"82","LABASTIDE DE PENNE":"82","LABASTIDE DU TEMPLE":"82","LACOUR":"82","LACOURT ST PIERRE":"82","LAFITTE":"82","LAFRANCAISE":"82","LAMOTHE CUMONT":"82","LAPENCHE":"82","MONBEQUI":"82","MONTAIGU DE QUERCY":"82","MONTRICOUX":"82","POUPAS":"82","PUYGAILLARD DE QUERCY":"82","PUYGAILLARD DE LOMAGNE":"82","ROQUECOR":"82","ST ETIENNE DE TULMONT":"82","ST NAUPHARY":"82","SAVENES":"82","SEPTFONDS":"82","SISTELS":"82","VAISSAC":"82","LES ADRETS DE L ESTEREL":"83","LES ARCS":"83","BANDOL":"83","BARGEME":"83","LE BOURGUET":"83","BRIGNOLES":"83","CALLAS":"83","CARQUEIRANNE":"83","DRAGUIGNAN":"83","FOX AMPHOUX":"83","LA GARDE FREINET":"83","GONFARON":"83","HYERES":"83","MAZAUGUES":"83","MONTAUROUX":"83","MONTMEYAN":"83","PUGET VILLE":"83","REGUSSE":"83","LE REVEST LES EAUX":"83","ROQUEBRUNE SUR ARGENS":"83","EPIERRE":"73","FLUMET":"73","FONTCOUVERTE LA TOUSSUIRE":"73","GERBAIX":"73","GRESY SUR AIX":"73","HAUTELUCE":"73","JARSY":"73","LANSLEBOURG MONT CENIS":"73","LA PLAGNE TARENTAISE":"73","LES MARCHES":"73","MARCIEUX":"73","MARTHOD":"73","LES MOLLETTES":"73","MONTHION":"73","MOTZ":"73","NANCES":"73","LA LECHERE":"73","NOTRE DAME DES MILLIERES":"73","PLANAISE":"73","PONTAMAFREY MONTPASCAL":"73","PRESLE":"73","RANDENS":"73","STE HELENE DU LAC":"73","STE HELENE SUR ISERE":"73","ST OFFENGE":"73","ST OYEN":"73","ST PIERRE D ALBIGNY":"73","ST PIERRE DE BELLEVILLE":"73","ST PIERRE DE GENEBROZ":"73","TRESSERVE":"73","VALMEINIER":"73","LE VERNEIL":"73","VILLARD D HERY":"73","VILLARD LEGER":"73","VILLARODIN BOURGET":"73","VILLAROGER":"73","YENNE":"73","AMANCY":"74","AMBILLY":"74","ANNECY":"74","ANNEMASSE":"74","AVIERNOZ":"74","LE BIOT":"74","BLOYE":"74","BOGEVE":"74","BONNE":"74","CHAINAZ LES FRASSES":"74","CHALLONGES":"74","CHAVANOD":"74","CHENE EN SEMINE":"74","CLARAFOND ARCINE":"74","LES CONTAMINES MONTJOIE":"74","CREMPIGNY BONNEGUETE":"74","CRUSEILLES":"74","CUVAT":"74","DOMANCY":"74","DOUSSARD":"74","EVIAN LES BAINS":"74","EVIRES":"74","FAVERGES SEYTHENEX":"74","FESSY":"74","FETERNES":"74","HAUTEVILLE SUR FIER":"74","LOVAGNY":"74","MANIGOD":"74","MARNAZ":"74","MASSONGY":"74","MEGEVE":"74","MENTHON ST BERNARD":"74","MENTHONNEX SOUS CLERMONT":"74","MESIGNY":"74","MINZIER":"74","MONNETIER MORNEX":"74","NONGLARD":"74","ORCIER":"74","PERRIGNIER":"74","ST GERVAIS LES BAINS":"74","ST JEOIRE":"74","SAMOENS":"74","SCIEZ":"74","SEYNOD":"74","SILLINGY":"74","AUMETZ":"57","BASSING":"57","BENING LES ST AVOLD":"57","BERLING":"57","BETTBORN":"57","BETTING":"57","BETTVILLER":"57","BEZANGE LA PETITE":"57","BELLES FORETS":"57","BREISTROFF LA GRANDE":"57","BROUVILLER":"57","BUHL LORRAINE":"57","CAPPEL":"57","CHATEAU SALINS":"57","CHENOIS":"57","CHIEULLES":"57","COCHEREN":"57","COURCELLES SUR NIED":"57","CUTTING":"57","DALHAIN":"57","DALSTEIN":"57","DANNELBOURG":"57","DOLVING":"57","DOMNOM LES DIEUZE":"57","ELVANGE":"57","ERSTROFF":"57","FAILLY":"57","FALCK":"57","FAMECK":"57","FARSCHVILLER":"57","FENETRANGE":"57","FEY":"57","FLETRANGE":"57","FLOCOURT":"57","FONTOY":"57","FOULIGNY":"57","FRAUENBERG":"57","FREYMING MERLEBACH":"57","GERBECOURT":"57","GOIN":"57","GONDREXANGE":"57","GORZE":"57","GOSSELMING":"57","GUEBENHOUSE":"57","VAL DE BRIDE":"57","GUERTING":"57","GUINKIRCHEN":"57","GUNTZVILLER":"57","HABOUDANGE":"57","HAGONDANGE":"57","BASSE HAM":"57","HANNOCOURT":"57","HAN SUR NIED":"57","HARREBERG":"57","HASPELSCHIEDT":"57","HAUT CLOCHER":"57","HELLERING LES FENETRANGE":"57","HENRIVILLE":"57","HERMELANGE":"57","HILBESHEIM":"57","HOMBOURG HAUT":"57","HONSKIRCH":"57","HOTTVILLER":"57","HULTEHOUSE":"57","IBIGNY":"57","INGLANGE":"57","JOUY AUX ARCHES":"57","KIRVILLER":"57","KNUTANGE":"57","KUNTZIG":"57","LANEUVEVILLE EN SAULNOIS":"57","LEMONCOURT":"57","LESSE":"57","LORRY LES METZ":"57","LOUPERSHOUSE":"57","LUPPY":"57","MANY":"57","MARSAL":"57","MEISENTHAL":"57","MERSCHWEILLER":"57","METZ":"57","MITTELBRONN":"57","MITTERSHEIM":"57","MOLRING":"57","MONTBRONN":"57","MONTOY FLANVILLE":"57","MORVILLE SUR NIED":"57","MOULINS LES METZ":"57","MOUTERHOUSE":"57","MOYEUVRE GRANDE":"57","MULCEY":"57","NELLING":"57","NERCILLAC":"16","NIEUIL":"16","ORIOLLES":"16","PAIZAY NAUDOUIN EMBOURIE":"16","PALLUAUD":"16","LES PINS":"16","PUYREAUX":"16","RIOUX MARTIN":"16","LA ROCHEFOUCAULD":"16","ROUGNAC":"16","ROUZEDE":"16","GRAVES ST AMANT":"16","ST FRAIGNE":"16","ST GOURSON":"16","ST MARY":"16","ST MAURICE DES LIONS":"16","ST MEME LES CARRIERES":"16","ST PROJET ST CONSTANT":"16","ST SIMEUX":"16","SIREUIL":"16","SOUFFRIGNAC":"16","SURIS":"16","TAIZE AIZIE":"16","TURGON":"16","VILHONNEUR":"16","VILLEFAGNAN":"16","VILLOGNON":"16","VOULGEZAC":"16","VOUTHON":"16","XAMBES":"16","YVRAC ET MALLEYRAND":"16","AGUDELLE":"17","ASNIERES LA GIRAUD":"17","BALLON":"17","BEAUVAIS SUR MATHA":"17","BLANZAC LES MATHA":"17","BOUGNEAU":"17","BOURCEFRANC LE CHAPUS":"17","BOUTENAC TOUVENT":"17","BRESDON":"17","BRIVES SUR CHARENTE":"17","CABARIOT":"17","LE CHATEAU D OLERON":"17","CIERZAC":"17","CIRE D AUNIS":"17","LA CLISSE":"17","LA CLOTTE":"17","COIVERT":"17","CORME ROYAL":"17","COURANT":"17","COZES":"17","CRAVANS":"17","DOLUS D OLERON":"17","ECOYEUX":"17","LES EDUTS":"17","EXPIREMONT":"17","FONTAINE CHALENDRAY":"17","FONTENET":"17","FOURAS":"17","LE GICQ":"17","GOURVILLETTE":"17","GUITINIERES":"17","HIERS BROUAGE":"17","LONZAC":"17","LOZAY":"17","LUSSANT":"17","MARSAIS":"17","LES MATHES":"17","MEUX":"17","MONTGUYON":"17","MORNAC SUR SEUDRE":"17","MORTAGNE SUR GIRONDE":"17","NACHAMPS":"17","NIEUL SUR MER":"17","NUAILLE D AUNIS":"17","ORIGNOLLES":"17","OZILLAC":"17","PAILLE":"17","LAMARQUE PONTACQ":"65","LIBAROS":"65","LORTET":"65","LOUCRUP":"65","LOUDERVIELLE":"65","LUBRET ST LUC":"65","LUGAGNAN":"65","LUSTAR":"65","MANSAN":"65","MARQUERIE":"65","MAULEON BAROUSSE":"65","MOULEDOUS":"65","NESTIER":"65","NOUILHAN":"65","OURDON":"65","PEYRAUBE":"65","PIERREFITTE NESTALAS":"65","POUEYFERRE":"65","PUNTOUS":"65","SAILHAN":"65","ST PE DE BIGORRE":"65","SALECHAN":"65","SANOUS":"65","SARP":"65","SARRANCOLIN":"65","SEICH":"65","SENAC":"65","SERE EN LAVEDAN":"65","SERON":"65","SIARROUY":"65","SOREAC":"65","SOST":"65","TAJAN":"65","TOURNOUS DEVANT":"65","VIELLE AURE":"65","VILLEMBITS":"65","VILLENAVE PRES BEARN":"65","L ALBERE":"66","ANSIGNAN":"66","ARBOUSSOLS":"66","ARGELES SUR MER":"66","AYGUATEBIA TALAU":"66","BANYULS DELS ASPRES":"66","LE BARCARES":"66","CAMPOUSSY":"66","CANET EN ROUSSILLON":"66","CANOHES":"66","CASTEIL":"66","CAUDIES DE FENOUILLEDES":"66","EGAT":"66","FONTPEDROUSE":"66","FORMIGUERES":"66","JUJOLS":"66","LATOUR DE FRANCE":"66","MAURY":"66","MOLITG LES BAINS":"66","MONTNER":"66","NEFIACH":"66","NOHEDES":"66","OREILLA":"66","ORTAFFA":"66","PERPIGNAN":"66","PEZILLA DE CONFLENT":"66","RABOUILLET":"66","RIA SIRACH":"66","ST LAURENT DE CERDANS":"66","ST PIERRE DELS FORCATS":"66","SERRALONGUE":"66","URBANYA":"66","VILLELONGUE DELS MONTS":"66","ALTENHEIM":"67","ASCHBACH":"67","BARR":"67","BENFELD":"67","BERNARDSWILLER":"67","BERSTETT":"67","BIBLISHEIM":"67","BISCHWILLER":"67","BOERSCH":"67","BOSSELSHAUSEN":"67","CRASTATT":"67","DALHUNDEN":"67","DAMBACH":"67","DAUENDORF":"67","DEHLINGEN":"67","DIEDENDORF":"67","DIMBSTHAL":"67","DINGSHEIM":"67","DINSHEIM SUR BRUCHE":"67","ST CYR SUR MER":"83","SALERNES":"83","SEILLANS":"83","SIGNES":"83","SILLANS LA CASCADE":"83","SIX FOURS LES PLAGES":"83","SOLLIES TOUCAS":"83","TANNERON":"83","LE THORONET":"83","TOULON":"83","TOURTOUR":"83","LE VAL":"83","VARAGES":"83","LA VERDIERE":"83","VIDAUBAN":"83","AVIGNON":"84","CABRIERES D AVIGNON":"84","CADEROUSSE":"84","CARPENTRAS":"84","L ISLE SUR LA SORGUE":"84","LAFARE":"84","LAGARDE PAREOL":"84","LAPALUD":"84","MAZAN":"84","MERINDOL":"84","MONDRAGON":"84","MONIEUX":"84","LA MOTTE D AIGUES":"84","ORANGE":"84","PERTUIS":"84","PIOLENC":"84","ROAIX":"84","ROBION":"84","SERIGNAN DU COMTAT":"84","TAILLADES":"84","LA TOUR D AIGUES":"84","VIENS":"84","VILLELAURE":"84","VITROLLES EN LUBERON":"84","L AIGUILLON SUR VIE":"85","AUBIGNY LES CLOUZEAUX":"85","BARBATRE":"85","BAZOGES EN PAREDS":"85","BEAUVOIR SUR MER":"85","BENET":"85","LA BERNARDIERE":"85","BOURNEAU":"85","BOURNEZEAU":"85","LA BRETONNIERE LA CLAYE":"85","CHAILLE LES MARAIS":"85","CHAIX":"85","LA CHAIZE LE VICOMTE":"85","CHAMBRETAUD":"85","CHATEAU D OLONNE":"85","CHAUCHE":"85","COMMEQUIERS":"85","FAYMOREAU":"85","FONTENAY LE COMTE":"85","FROIDFOND":"85","LA GARNACHE":"85","L HERBERGEMENT":"85","L HERMENAULT":"85","L ILE D YEU":"85","LE LANGON":"85","MAREUIL SUR LAY DISSAIS":"85","LE MAZEAU":"85","MESNARD LA BAROTIERE":"85","MONSIREIGNE":"85","MOUCHAMPS":"85","MOUTIERS LES MAUXFAITS":"85","PALLUAU":"85","PEAULT":"85","LES PINEAUX":"85","PUY DE SERRE":"85","LA RABATELIERE":"85","MONTREVERD":"85","ST AUBIN DES ORMEAUX":"85","ST GEORGES DE POINTINDOUX":"85","ST GERMAIN DE PRINCAY":"85","ST HILAIRE LA FORET":"85","ST LAURENT SUR SEVRE":"85","BREM SUR MER":"85","ST MICHEL LE CLOUCQ":"85","ST PHILBERT DE BOUAINE":"85","STE RADEGONDE DES NOYERS":"85","ST REVEREND":"85","TALLOIRES MONTMIN":"74","THYEZ":"74","THONON LES BAINS":"74","VETRAZ MONTHOUX":"74","VILLE EN SALLAZ":"74","VIUZ LA CHIESAZ":"74","VIUZ EN SALLAZ":"74","PARIS 14":"75","PARIS 16":"75","PARIS 18":"75","ALLOUVILLE BELLEFOSSE":"76","ANCRETTEVILLE SUR MER":"76","ANGIENS":"76","ANGLESQUEVILLE LA BRAS LONG":"76","ANNEVILLE AMBOURVILLE":"76","ANVEVILLE":"76","ARDOUVAL":"76","AUBERVILLE LA MANUEL":"76","AUBERVILLE LA RENAULT":"76","AUFFAY":"76","AUPPEGARD":"76","LES AUTHIEUX SUR LE PORT ST OUEN":"76","BAILLEUL NEUVILLE":"76","BARENTIN":"76","BEAUVOIR EN LYONS":"76","BELLENCOMBRE":"76","BELMESNIL":"76","BERTHEAUVILLE":"76","BERTREVILLE ST OUEN":"76","BEUZEVILLETTE":"76","BEZANCOURT":"76","BIERVILLE":"76","BOLBEC":"76","BOSC BERENGER":"76","BOSC BORDEL":"76","BOSC GUERARD ST ADRIEN":"76","BOURVILLE":"76","BREMONTIER MERVAL":"76","CALLEVILLE LES DEUX EGLISES":"76","CAMPNEUSEVILLE":"76","CANY BARVILLE":"76","LE CAULE STE BEUVE":"76","LA CHAPELLE ST OUEN":"76","CIDEVILLE":"76","CLEUVILLE":"76","COLMESNIL MANNEVILLE":"76","CRESSY":"76","LA CRIQUE":"76","CRIQUEBEUF EN CAUX":"76","CRITOT":"76","CROIX MARE":"76","DANCOURT":"76","DAUBEUF SERVILLE":"76","ECTOT L AUBER":"76","ELBEUF":"76","ENVERMEU":"76","ERMENOUVILLE":"76","ESTOUTEVILLE ECALLES":"76","ETAINHUS":"76","ETALLEVILLE":"76","ETRETAT":"76","FLAMETS FRETILS":"76","FONTAINE LA MALLET":"76","FONTAINE LE BOURG":"76","FONTAINE LE DUN":"76","FORGES LES EAUX":"76","FRESLES":"76","FRESNE LE PLAN":"76","FRICHEMESNIL":"76","FROBERVILLE":"76","FULTOT":"76","GAINNEVILLE":"76","GANCOURT ST ETIENNE":"76","GODERVILLE":"76","GONFREVILLE CAILLOT":"76","GRAINVILLE LA TEINTURIERE":"76","LES GRANDES VENTES":"76","GREMONVILLE":"76","GUEURES":"76","GUEUTTEVILLE":"76","HATTENVILLE":"76","HAUSSEZ":"76","HERMANVILLE":"76","HEURTEAUVILLE":"76","LA HOUSSAYE BERANGER":"76","INGOUVILLE":"76","LIMESY":"76","NITTING":"57","OGY":"57","PLAINE DE WALSCH":"57","PONTOY":"57","PUTTIGNY":"57","RAHLING":"57","RANGUEVAUX":"57","REDANGE":"57","REMERING LES PUTTELANGE":"57","BASSE RENTGEN":"57","RETONFEY":"57","REYERSVILLER":"57","RICHEVAL":"57","ROCHONVILLERS":"57","RODALBE":"57","ROLBING":"57","RONCOURT":"57","ST AVOLD":"57","ST JEAN KOURTZERODE":"57","ST JULIEN LES METZ":"57","ST QUIRIN":"57","SANRY LES VIGY":"57","SANRY SUR NIED":"57","SARREBOURG":"57","SARREGUEMINES":"57","SARREINSMING":"57","SCHOENECK":"57","SCHORBACH":"57","SCHWERDORFF":"57","SIERCK LES BAINS":"57","SIERSTHAL":"57","STIRING WENDEL":"57","VAHL EBERSING":"57","VARSBERG":"57","VECKRING":"57","VERNEVILLE":"57","VIGY":"57","VILLER":"57","VITTERSBOURG":"57","VULMONT":"57","WALDHOUSE":"57","WALSCHEID":"57","VOELFLING LES BOUZONVILLE":"57","WOUSTVILLER":"57","ZETTING":"57","ZOUFFTGEN":"57","STUCKANGE":"57","ALLUY":"58","ANLEZY":"58","ARTHEL":"58","ARZEMBOUY":"58","AVRIL SUR LOIRE":"58","BEAUMONT LA FERRIERE":"58","BILLY SUR OISY":"58","BREUGNON":"58","LA CELLE SUR NIEVRE":"58","CHAMPVERT":"58","CHANTENAY ST IMBERT":"58","LA CHARITE SUR LOIRE":"58","CHARRIN":"58","CHATILLON EN BAZOIS":"58","CHATIN":"58","CHEVROCHES":"58","CHOUGNY":"58","CORVOL D EMBERNARD":"58","COSNE COURS SUR LOIRE":"58","DAMPIERRE SOUS BOUHY":"58","DEVAY":"58","LA FERMETE":"58","FERTREVE":"58","FLETY":"58","FOURCHAMBAULT":"58","FRASNAY REUGNY":"58","GARCHIZY":"58","GIRY":"58","LAVAULT DE FRETOY":"58","LUCENAY LES AIX":"58","MARS SUR ALLIER":"58","MARZY":"58","MHERE":"58","MILLAY":"58","MONTIGNY SUR CANNE":"58","MONTREUILLON":"58","MONTSAUCHE LES SETTONS":"58","MOULINS ENGILBERT":"58","NEUVILLE LES DECIZE":"58","ONLAY":"58","PLANCHEZ":"58","POUSSEAUX":"58","ST BENIN D AZY":"58","ST FRANCHY":"58","ST GERMAIN CHASSENAY":"58","PESSINES":"17","ESSOUVERT":"17","PORT D ENVAUX":"17","LES PORTES EN RE":"17","PUYROLLAND":"17","RIOUX":"17","ST AGNANT":"17","ST ANDRE DE LIDON":"17","ST CIERS CHAMPAGNE":"17","ST DIZANT DU BOIS":"17","ST FORT SUR GIRONDE":"17","ST GEORGES D OLERON":"17","ST JUST LUZAC":"17","ST MARTIAL DE MIRAMBEAU":"17","ST NAZAIRE SUR CHARENTE":"17","ST OUEN D AUNIS":"17","ST QUANTIN DE RANCANNE":"17","ST ROMAIN SUR GIRONDE":"17","ST SAVINIEN":"17","ST SIGISMOND DE CLERMONT":"17","ST XANDRE":"17","SALIGNAC DE MIRAMBEAU":"17","SOULIGNONNE":"17","SOUSMOULINS":"17","SURGERES":"17","LE THOU":"17","LES TOUCHES DE PERIGNY":"17","VARAIZE":"17","VARZAY":"17","VAUX SUR MER":"17","VILLARS LES BOIS":"17","VILLENEUVE LA COMTESSE":"17","VILLEXAVIER":"17","VIRSON":"17","VOISSAY":"17","LE GRAND VILLAGE PLAGE":"17","AINAY LE VIEIL":"18","ARGENVIERES":"18","ASSIGNY":"18","BERRY BOUY":"18","BLANCAFORT":"18","LA CELETTE":"18","LA CHAPELLE ST URSIN":"18","CHAUMOUX MARCILLY":"18","CONCRESSAULT":"18","CORQUOY":"18","COUST":"18","CREZANCY EN SANCERRE":"18","CROSSES":"18","DUN SUR AURON":"18","FAVERDINES":"18","FEUX":"18","FOECY":"18","GIVARDON":"18","GROISES":"18","HENRICHEMONT":"18","HUMBLIGNY":"18","IDS ST ROCH":"18","LAVERDINES":"18","LURY SUR ARNON":"18","MAISONNAIS":"18","MEREAU":"18","MORLAC":"18","MORNAY SUR ALLIER":"18","NEUVY SUR BARANGEON":"18","ORVAL":"18","PLOU":"18","POISIEUX":"18","PRIMELLES":"18","QUINCY":"18","REZAY":"18","SAGONNE":"18","ST BOUIZE":"18","ST DENIS DE PALIN":"18","ST ELOY DE GY":"18","ST LOUP DES CHAUMES":"18","ST MARTIN D AUXIGNY":"18","ST MICHEL DE VOLANGIS":"18","ST SATUR":"18","STE THORETTE":"18","ST VITTE":"18","SANCERGUES":"18","SANCOINS":"18","DUPPIGHEIM":"67","DURNINGEN":"67","DURSTEL":"67","EBERSMUNSTER":"67","ECKBOLSHEIM":"67","WANGENBOURG ENGENTHAL":"67","ENTZHEIM":"67","ERCKARTSWILLER":"67","ERNOLSHEIM BRUCHE":"67","ERSTEIN":"67","ESCHBACH":"67","ESCHBOURG":"67","FORSTHEIM":"67","FORT LOUIS":"67","FOUDAY":"67","FRIEDOLSHEIM":"67","FURCHHAUSEN":"67","GOTTESHEIM":"67","GOUGENHEIM":"67","GRIESHEIM PRES MOLSHEIM":"67","HESSENHEIM":"67","HINSBOURG":"67","HOCHFELDEN":"67","HOFFEN":"67","HUTTENHEIM":"67","INNENHEIM":"67","ITTERSWILLER":"67","KEFFENACH":"67","KILSTETT":"67","KIRRWILLER":"67","KNOERSHEIM":"67","KURTZENHOUSE":"67","KUTTOLSHEIM":"67","LAMPERTHEIM":"67","LANDERSHEIM":"67","LIMERSHEIM":"67","LIPSHEIM":"67","LIXHAUSEN":"67","LORENTZEN":"67","MACKWILLER":"67","MARCKOLSHEIM":"67","MARMOUTIER":"67","MENCHHOFFEN":"67","MIETESHEIM":"67","MOLSHEIM":"67","MORSBRONN LES BAINS":"67","NATZWILLER":"67","NEUVILLER LA ROCHE":"67","NIEDERBRONN LES BAINS":"67","NIEDERLAUTERBACH":"67","NIEDERROEDERN":"67","NIEDERSTEINBACH":"67","BETSCHDORF":"67","OBERHAUSBERGEN":"67","OBERNAI":"67","OBERROEDERN":"67","TONNOY":"54","VAL ET CHATILLON":"54","VALHEY":"54","VANDELAINVILLE":"54","VANDOEUVRE LES NANCY":"54","VARANGEVILLE":"54","VEHO":"54","VELAINE EN HAYE":"54","VEZELISE":"54","VIEVILLE EN HAYE":"54","VILLE HOUDLEMONT":"54","VILLERS LE ROND":"54","VILLERS LES NANCY":"54","VITREY":"54","VIVIERS SUR CHIERS":"54","WAVILLE":"54","XEUILLEY":"54","XIVRY CIRCOURT":"54","XONVILLE":"54","ABAUCOURT HAUTECOURT":"55","AMANTY":"55","ANCEMONT":"55","ANDERNAY":"55","AUTRECOURT SUR AIRE":"55","BADONVILLIERS GERAUVILLIERS":"55","BELRAIN":"55","BETHINCOURT":"55","BOUREUILLES":"55","BOVEE SUR BARBOURE":"55","BRAS SUR MEUSE":"55","BREUX":"55","BRIZEAUX":"55","CHAMPOUGNY":"55","CHATTANCOURT":"55","CONTRISSON":"55","COURCELLES EN BARROIS":"55","ST SULPICE EN PAREDS":"85","SIGOURNAIS":"85","VOUILLE LES MARAIS":"85","XANTON CHASSENON":"85","ADRIERS":"86","AVANTON":"86","BASSES":"86","BONNEUIL MATOURS":"86","BRIGUEIL LE CHANTRE":"86","BRUX":"86","CEAUX EN LOUDUN":"86","CHAMPAGNE LE SEC":"86","CHAMPIGNY LE SEC":"86","CHASSENEUIL DU POITOU":"86","CHATELLERAULT":"86","CHAUVIGNY":"86","CHERVES":"86","LA ROCHE RIGAULT":"86","DERCE":"86","GLENOUZE":"86","JOUHET":"86","LATILLE":"86","LAVOUX":"86","LENCLOITRE":"86","LIGUGE":"86","LOUDUN":"86","MARIGNY BRIZAY":"86","MAUPREVOIR":"86","MIREBEAU":"86","MONTAMISE":"86","MONTS SUR GUESNES":"86","NIEUIL L ESPOIR":"86","NOUAILLE MAUPERTUIS":"86","PLEUMARTIN":"86","POUANCAY":"86","QUINCAY":"86","RASLAY":"86","ROUILLE":"86","ST JEAN DE SAUVES":"86","ST LEOMER":"86","VALDIVIENNE":"86","ST SAVIOL":"86","ST SECONDIN":"86","SAULGE":"86","SAVIGNE":"86","SEVRES ANXAUMONT":"86","SOMMIERES DU CLAIN":"86","THURAGEAU":"86","THURE":"86","LES TROIS MOUTIERS":"86","VENDEUVRE DU POITOU":"86","VICQ SUR GARTEMPE":"86","AUREIL":"87","BALLEDENT":"87","LA BAZEUGE":"87","BESSINES SUR GARTEMPE":"87","BLOND":"87","BOISSEUIL":"87","BONNAC LA COTE":"87","BREUILAUFA":"87","CIEUX":"87","DROUX":"87","EYMOUTIERS":"87","JABREILLES LES BORDES":"87","LA JONCHERE ST MAURICE":"87","JOUAC":"87","LAVIGNAC":"87","LIMOGES":"87","LINARDS":"87","MAGNAC BOURG":"87","MARVAL":"87","VAL D ISSOIRE":"87","ORADOUR SUR GLANE":"87","PIERRE BUFFIERE":"87","LA PORCHERIE":"87","REMPNAT":"87","ROCHECHOUART":"87","ROZIERS ST GEORGES":"87","STE ANNE ST PRIEST":"87","ST BONNET DE BELLAC":"87","ST DENIS DES MURS":"87","ST GEORGES LES LANDES":"87","ST GERMAIN LES BELLES":"87","ST JUST LE MARTEL":"87","ST MARTIN TERRESSUS":"87","ST PRIEST LIGOURE":"87","LONDINIERES":"76","LONGUEVILLE SUR SCIE":"76","LUNERAY":"76","MARQUES":"76","MAUCOMBLE":"76","MAULEVRIER STE GERTRUDE":"76","MAUNY":"76","MAUQUENCHY":"76","MENTHEVILLE":"76","MESNIERES EN BRAY":"76","LE MESNIL DURDENT":"76","LE MESNIL REAUME":"76","MIRVILLE":"76","MONT CAUVAIRE":"76","LA NEUVILLE CHANT D OISEL":"76","NEVILLE":"76","NOINTOT":"76","NOLLEVAL":"76","PORT JEROME SUR SEINE":"76","OUAINVILLE":"76","OUVILLE L ABBAYE":"76","PIERREFIQUES":"76","PISSY POVILLE":"76","PONTS ET MARAIS":"76","LA POTERIE CAP D ANTIFER":"76","RICARVILLE":"76","ROUVRAY CATILLON":"76","ROYVILLE":"76","ST AUBIN DE CRETOT":"76","ST CLAIR SUR LES MONTS":"76","ST DENIS LE THIBOULT":"76","ST ETIENNE DU ROUVRAY":"76","ST GERMAIN DES ESSOURTS":"76","ST JEAN DU CARDONNAY":"76","ST MACLOU LA BRIERE":"76","MORIENNE":"76","ST MARTIN AU BOSC":"76","ST MARTIN DU MANOIR":"76","ST MARTIN DU VIVIER":"76","ST MARTIN OSMONVILLE":"76","ST MAURICE D ETELAN":"76","ST OUEN SOUS BAILLY":"76","ST PIERRE DE MANNEVILLE":"76","ST PIERRE DE VARENGEVILLE":"76","ST PIERRE EN PORT":"76","ST PIERRE LAVIS":"76","ST REMY BOSCROCOURT":"76","ST SAENS":"76","ST VAAST D EQUIQUEVILLE":"76","SASSEVILLE":"76","SEPT MEULES":"76","SERVAVILLE SALMONVILLE":"76","SIGY EN BRAY":"76","TANCARVILLE":"76","THEROULDEVILLE":"76","TOCQUEVILLE EN CAUX":"76","LE TORP MESNIL":"76","TOTES":"76","LE TRAIT":"76","TREMAUVILLE":"76","VARNEVILLE BRETTEVILLE":"76","VATIERVILLE":"76","VEAUVILLE LES QUELLES":"76","VENTES ST REMY":"76","VILLERS ECALLES":"76","VILLERS SOUS FOUCARMONT":"76","VINNEMERVILLE":"76","YAINVILLE":"76","YERVILLE":"76","YPREVILLE BIVILLE":"76","ANDREZEL":"77","ARBONNE LA FORET":"77","ARGENTIERES":"77","ARMENTIERES EN BRIE":"77","TANCONVILLE":"54","THELOD":"54","THIEBAUMENIL":"54","THOREY LYAUTEY":"54","TRIEUX":"54","VACQUEVILLE":"54","VATHIMENIL":"54","VAUDIGNY":"54","VELAINE SOUS AMANCE":"54","VELLE SUR MOSELLE":"54","VILLACOURT":"54","VILLECEY SUR MAD":"54","VILLERS EN HAYE":"54","ST LAURENT L ABBAYE":"58","ST PARIZE LE CHATEL":"58","ST REVERIEN":"58","ST SAULGE":"58","ST VERAIN":"58","SEMELAY":"58","VILLIERS LE PRE":"50","VILLIERS FOSSARD":"50","ALLIANCELLES":"51","AMBONNAY":"51","ARZILLIERES NEUVILLE":"51","AY CHAMPAGNE":"51","BACONNES":"51","BEAUMONT SUR VESLE":"51","BELVAL EN ARGONNE":"51","BERMERICOURT":"51","BETTANCOURT LA LONGUE":"51","BINSON ET ORQUIGNY":"51","BREUVERY SUR COOLE":"51","BROUILLET":"51","BROUSSY LE PETIT":"51","CERNAY EN DORMOIS":"51","CHAMPIGNEUL CHAMPAGNE":"51","CHANTEMERLE":"51","CHATILLON SUR BROUE":"51","LA CHAUSSEE SUR MARNE":"51","CHAVOT COURCOURT":"51","LE CHEMIN":"51","CHEMINON":"51","CHICHEY":"51","COOLUS":"51","CORMICY":"51","CORMONTREUIL":"51","COUPETZ":"51","COUPEVILLE":"51","COURGIVAUX":"51","COURTEMONT":"51","COURVILLE":"51","COUVROT":"51","CUPERLY":"51","DAMPIERRE AU TEMPLE":"51","DIZY":"51","DORMANS":"51","DROUILLY":"51","ECLAIRES":"51","LES ESSARTS LES SEZANNE":"51","ETREPY":"51","FAUX FRESNAY":"51","FAUX VESIGNEUL":"51","FONTAINE EN DORMOIS":"51","LE GAULT SOIGNY":"51","GERMINON":"51","HAUSSIMONT":"51","JALONS":"51","JANVILLIERS":"51","JOISELLE":"51","LACHY":"51","LENHARREE":"51","LIGNON":"51","LOISY EN BRIE":"51","LOISY SUR MARNE":"51","LUXEMONT ET VILLOTTE":"51","MAISONS EN CHAMPAGNE":"51","MANCY":"51","MARDEUIL":"51","MARGERIE HANCOURT":"51","MAURUPT LE MONTOIS":"51","MECRINGES":"51","MONTGENOST":"51","MONTIGNY SUR VESLE":"51","MONT SUR COURVILLE":"51","MOSLINS":"51","MUIZON":"51","NORROIS":"51","NUISEMENT SUR COOLE":"51","ORBAIS L ABBAYE":"51","OUTINES":"51","PARGNY LES REIMS":"51","LES PETITES LOGES":"51","PLEURS":"51","PLIVOT":"51","PROUILLY":"51","RAPSECOURT":"51","REIMS":"51","LES RIVIERES HENRUEL":"51","ROMIGNY":"51","SAUGY":"18","SAULZAIS LE POTIER":"18","SENS BEAUJEU":"18","SERRUELLES":"18","SIDIAILLES":"18","THAUVENAY":"18","TROUY":"18","VEREAUX":"18","VIGNOUX SUR BARANGEON":"18","VILLABON":"18","VILLEGENON":"18","VORNAY":"18","LES ANGLES SUR CORREZE":"19","ASTAILLAC":"19","BASSIGNAC LE BAS":"19","BENAYES":"19","CAMPS ST MATHURIN LEOBAZEL":"19","CHAMBOULIVE":"19","CHAMPAGNAC LA NOAILLE":"19","CHANAC LES MINES":"19","CHAUFFOUR SUR VELL":"19","CHAUMEIL":"19","CONDAT SUR GANAVEIX":"19","COURTEIX":"19","CUBLAC":"19","DONZENAC":"19","EYGURANDE":"19","FAVARS":"19","GIMEL LES CASCADES":"19","GOULLES":"19","GROS CHASTANG":"19","LE JARDIN":"19","LADIGNAC SUR RONDELLES":"19","LAMAZIERE BASSE":"19","LAMONGERIE":"19","LAVAL SUR LUZEGE":"19","LIGNAREIX":"19","MANSAC":"19","MARCILLAC LA CROISILLE":"19","MAUSSAC":"19","MEYRIGNAC L EGLISE":"19","MONCEAUX SUR DORDOGNE":"19","MONTGIBAUD":"19","NESPOULS":"19","PANDRIGNES":"19","PEYRELEVADE":"19","ROCHE LE PEYROUX":"19","ROSIERS D EGLETONS":"19","ST AULAIRE":"19","ST BONNET L ENFANTIER":"19","ST BONNET PRES BORT":"19","ST CERNIN DE LARCHE":"19","STE FEREOLE":"19","ST FREJOUX":"19","ST HILAIRE LUC":"19","ST JULIEN LE PELERIN":"19","ST MARTIAL DE GIMEL":"19","ST MARTIN LA MEANNE":"19","ST MARTIN SEPERT":"19","ST PANTALEON DE LARCHE":"19","ST PARDOUX L ORTIGIER":"19","ST PRIEST DE GIMEL":"19","ST SETIERS":"19","ST SULPICE LES BOIS":"19","ST VIANCE":"19","ST VICTOUR":"19","SERANDON":"19","SEXCLES":"19","TARNAC":"19","TULLE":"19","UZERCHE":"19","VALIERGUES":"19","VEIX":"19","VOUTEZAC":"19","YSSANDON":"19","AGENCOURT":"21","AISEY SUR SEINE":"21","ALISE STE REINE":"21","ALLEREY":"21","AMPILLY LE SEC":"21","ANCEY":"21","ASNIERES LES DIJON":"21","AUBIGNY LA RONCE":"21","AUVILLARS SUR SAONE":"21","BAULME LA ROCHE":"21","BEAUNE":"21","BELLENEUVE":"21","BENEUVRE":"21","BEURIZOT":"21","BIERRE LES SEMUR":"21","COUVERTPUIS":"55","CUMIERES LE MORT HOMME":"55","CUNEL":"55","DAGONVILLE":"55","DELUT":"55","DEMANGE AUX EAUX":"55","DOMREMY LA CANNE":"55","DOUAUMONT":"55","DOULCON":"55","EPIEZ SUR MEUSE":"55","ERNEVILLE AUX BOIS":"55","EUVILLE":"55","EVRES":"55","FLEURY DEVANT DOUAUMONT":"55","FONTAINES ST CLAIR":"55","FROIDOS":"55","GESNES EN ARGONNE":"55","GIRAUVOISIN":"55","GIVRAUVAL":"55","GREMILLY":"55","GUERPONT":"55","GUSSAINVILLE":"55","HANNONVILLE SOUS LES COTES":"55","HENNEMONT":"55","HOUDELAINCOURT":"55","INOR":"55","LES TROIS DOMAINES":"55","JUVIGNY SUR LOISON":"55","LABEUVILLE":"55","LACROIX SUR MEUSE":"55","LAHAYVILLE":"55","LAIMONT":"55","LAMORVILLE":"55","LAVALLEE":"55","LAVINCOURT":"55","LISLE EN RIGAULT":"55","LONGEAUX":"55","LOUPPY LE CHATEAU":"55","LUZY ST MARTIN":"55","MAIZEY":"55","MARCHEVILLE EN WOEVRE":"55","MARSON SUR BARBOURE":"55","MENIL SUR SAULX":"55","MOIREY FLABAS CREPION":"55","MONTBLAINVILLE":"55","MONTBRAS":"55","MONTIGNY LES VAUCOULEURS":"55","NAIX AUX FORGES":"55","NANCOIS LE GRAND":"55","NANT LE PETIT":"55","NANTILLOIS":"55","NEPVANT":"55","PARFONDRUPT":"55","LES PAROCHES":"55","PONT SUR MEUSE":"55","REMBERCOURT SOMMAISNE":"55","REMENNECOURT":"55","RESSON":"55","REVIGNY SUR ORNAIN":"55","ST GERMAIN SUR MEUSE":"55","ST JOIRE":"55","ST MAURICE SOUS LES COTES":"55","ST PIERREVILLERS":"55","ST REMY LA CALONNE":"55","SALMAGNE":"55","SAULVAUX":"55","SAVONNIERES DEVANT BAR":"55","SAVONNIERES EN PERTHOIS":"55","SEIGNEULLES":"55","SEPVIGNY":"55","SORCY ST MARTIN":"55","STAINVILLE":"55","STENAY":"55","THILLOMBOIS":"55","THONNE LES PRES":"55","THONNELLE":"55","TREMONT SUR SAULX":"55","TROUSSEY":"55","TROYON":"55","UGNY SUR MEUSE":"55","VADONVILLE":"55","VALBOIS":"55","VAUDEVILLE LE HAUT":"55","VIGNEUL SOUS MONTMEDY":"55","VILLEROY SUR MEHOLLE":"55","VILLERS AUX VENTS":"55","VILLERS LES MANGIENNES":"55","VILLERS SUR MEUSE":"55","VILLE SUR COUSANCES":"55","VILLE SUR SAULX":"55","WOIMBEY":"55","ST PRIEST SOUS AIXE":"87","ST SORNIN LEULAC":"87","ST SULPICE LES FEUILLES":"87","ST SYMPHORIEN SUR COUZE":"87","SURDOUX":"87","THORIGNE SUR DUE":"72","VERNEIL LE CHETIF":"72","VILLAINES SOUS MALICORNE":"72","VIVOIN":"72","YVRE L EVEQUE":"72","AILLON LE VIEUX":"73","LES ALLUES":"73","LA BATHIE":"73","BELLECOMBE EN BAUGES":"73","LA BIOLLE":"73","BOURG ST MAURICE":"73","BRAMANS":"73","LA BRIDOIRE":"73","CHAMPAGNEUX":"73","CHAMP LAURENT":"73","CHANAZ":"73","LA CHAVANNE":"73","CORBEL":"73","DRUMETTAZ CLARAFOND":"73","DULLIN":"73","ESSERTS BLAY":"73","ETABLE":"73","FEISSONS SUR ISERE":"73","JACOB BELLECOMBETTE":"73","JONGIEUX":"73","LOISIEUX":"73","MERY":"73","MONTGILBERT":"73","MONTMELIAN":"73","MONTVERNIER":"73","LA MOTTE SERVOLEX":"73","PUGNY CHATENOD":"73","ROGNAIX":"73","ST ALBAN D HURTIERES":"73","ST BON TARENTAISE":"73","ST COLOMBAN DES VILLARDS":"73","ST FRANCOIS LONGCHAMP":"73","ST JEAN D ARVES":"73","ST JEAN D ARVEY":"73","ST JEAN DE COUZ":"73","ST JEAN DE LA PORTE":"73","ST JULIEN MONT DENIS":"73","ST PIERRE DE SOUCY":"73","SALINS FONTAINE":"73","VALLOIRE":"73","VILLARD SALLET":"73","VILLARD SUR DORON":"73","VILLAROUX":"73","VOGLANS":"73","ALLINGES":"74","ARENTHON":"74","BASSY":"74","BELLEVAUX":"74","BOSSEY":"74","BRIZON":"74","CERCIER":"74","CERVENS":"74","CHATEL":"74","CHENS SUR LEMAN":"74","CHOISY":"74","COPPONEX":"74","CORNIER":"74","CRANVES SALES":"74","DOUVAINE":"74","DUINGT":"74","ENTREVERNES":"74","ETAUX":"74","EXCENEVEX":"74","LA FORCLAZ":"74","FRANCLENS":"74","GIEZ":"74","LES HOUCHES":"74","LOISIN":"74","LORNAY":"74","LULLIN":"74","LULLY":"74","MARCELLAZ":"74","MARIGNIER":"74","VILLE SUR YRON":"54","VITTONVILLE":"54","XURES":"54","ARRANCY SUR CRUSNE":"55","AVIOTH":"55","AVOCOURT":"55","AZANNES ET SOUMAZANNES":"55","BANTHEVILLE":"55","BAUDREMONT":"55","BAULNY":"55","BAZINCOURT SUR SAULX":"55","BEHONNE":"55","BENEY EN WOEVRE":"55","BEZONVAUX":"55","BOULIGNY":"55","BRILLON EN BARROIS":"55","BROCOURT EN ARGONNE":"55","BROUSSEY EN BLOIS":"55","BUXIERES SOUS LES COTES":"55","CHAILLON":"55","CHASSEY BEAUPRE":"55","CHONVILLE MALAUMONT":"55","CIERGES SOUS MONTFAUCON":"55","CLERY LE GRAND":"55","COUROUVRE":"55","COUSANCES LES FORGES":"55","DAINVILLE BERTHELEVILLE":"55","DAMVILLERS":"55","DELOUZE ROSIERES":"55","DIEPPE SOUS DOUAUMONT":"55","DOMBRAS":"55","DOMMARTIN LA MONTAGNE":"55","EIX":"55","ERIZE LA PETITE":"55","ETAIN":"55","ETRAYE":"55","FAINS VEEL":"55","FLASSIGNY":"55","FOUCAUCOURT SUR THABAS":"55","FOUCHERES AUX BOIS":"55","FUTEAU":"55","GIMECOURT":"55","GOUSSAINCOURT":"55","HALLES SOUS LES COTES":"55","HAUMONT PRES SAMOGNEUX":"55","HORVILLE EN ORNOIS":"55","IRE LE SEC":"55","JUVIGNY EN PERTHOIS":"55","LACHALADE":"55","LAHEYCOURT":"55","LAMOUILLY":"55","LANDRECOURT LEMPIRE":"55","LANHERES":"55","LISSEY":"55","LONGCHAMPS SUR AIRE":"55","MARRE":"55","MAUVAGES":"55","MELIGNY LE PETIT":"55","MERLES SUR LOISON":"55","LES MONTHAIRONS":"55","CHANTERAINE":"55","NANT LE GRAND":"55","NETTANCOURT":"55","LE NEUFOUR":"55","NEUVILLE EN VERDUNOIS":"55","NEUVILLE SUR ORNAIN":"55","NICEY SUR AIRE":"55","NOYERS AUZECOURT":"55","ORNES":"55","PILLON":"55","PINTHEVILLE":"55","QUINCY LANDZECOURT":"55","RAMBUCOURT":"55","RANCOURT SUR ORNAIN":"55","RECICOURT":"55","RECOURT LE CREUX":"55","REFFROY":"55","RIGNY ST MARTIN":"55","ROMAGNE SOUS MONTFAUCON":"55","RONVAUX":"55","ROUVRES EN WOEVRE":"55","ROUVROIS SUR MEUSE":"55","ROUVROIS SUR OTHAIN":"55","ST JEAN LES BUZY":"55","SAMOGNEUX":"55","ST BRICE COURCELLES":"51","ST EULIEN":"51","ST EUPHRAISE ET CLAIRIZET":"51","ST JEAN SUR TOURBE":"51","ST MARD SUR AUVE":"51","ST MASMES":"51","ST REMY SUR BUSSY":"51","SEPT SAULX":"51","SERMAIZE LES BAINS":"51","SERVON MELZICOURT":"51","SOGNY AUX MOULINS":"51","SOMMESOUS":"51","SOMSOIS":"51","SONGY":"51","SOULANGES":"51","SOULIERES":"51","VAL DE LIVRE":"51","THILLOIS":"51","TOURS SUR MARNE":"51","TRECON":"51","VANAULT LES DAMES":"51","VAUDESINCOURT":"51","VAVRAY LE GRAND":"51","VELYE":"51","VERZY":"51","VILLE EN SELVE":"51","VILLERS FRANQUEUX":"51","VILLESENEUX":"51","VILLIERS AUX CORNEILLES":"51","VIRGINY":"51","VITRY EN PERTHOIS":"51","VITRY LE FRANCOIS":"51","VOIPREUX":"51","WARGEMOULIN HURLUS":"51","ANNONVILLE":"52","ATTANCOURT":"52","AUDELONCOURT":"52","AUJEURRES":"52","AULNOY SUR AUBE":"52","ROCHES BETTAINCOURT":"52","BLAISY":"52","BOLOGNE":"52","BONNECOURT":"52","BOURBONNE LES BAINS":"52","BOURG STE MARIE":"52","BRETHENAY":"52","BUXIERES LES CLEFMONT":"52","CHAMBRONCOURT":"52","CHAMOUILLEY":"52","CHANCENAY":"52","CHANTRAINES":"52","CHATENAY VAUDIN":"52","CHATONRUPT SOMMERMONT":"52","CHEVILLON":"52","CHAMARANDES CHOIGNES":"52","CIREY LES MAREILLES":"52","CIREY SUR BLAISE":"52","COHONS":"52","COLOMBEY LES DEUX EGLISES":"52","COUPRAY":"52","COURCELLES EN MONTAGNE":"52","CUSEY":"52","DAMREMONT":"52","DOMREMY LANDEVILLE":"52","EFFINCOURT":"52","EPIZON":"52","FAYL BILLOT":"52","FERRIERE ET LAFOLIE":"52","FONTAINES SUR MARNE":"52","FORCEY":"52","FRONCLES":"52","GUDMONT VILLIERS":"52","HAUTE AMANCE":"52","HUMBECOURT":"52","HUMES JORQUENAY":"52","ILLOUD":"52","ISOMES":"52","JOINVILLE":"52","JUZENNECOURT":"52","LAFERTE SUR AUBE":"52","LANQUES SUR ROGNON":"52","LAVILLE AUX BOIS":"52","LAVILLENEUVE":"52","LESCHERES SUR LE BLAISERON":"52","MALAINCOURT SUR MEUSE":"52","MARAC":"52","LA PORTE DU DER":"52","VAL DE MEUSE":"52","BISSEY LA COTE":"21","BLIGNY LE SEC":"21","BLIGNY LES BEAUNE":"21","BOUSSENOIS":"21","BRAZEY EN MORVAN":"21","BRAZEY EN PLAINE":"21","BRIANNY":"21","BROIN":"21","BROINDON":"21","BURE LES TEMPLIERS":"21","BUSSEAUT":"21","BUSSEROTTE ET MONTENAILLE":"21","CHAILLY SUR ARMANCON":"21","CHAMBEIRE":"21","CHAMBLANC":"21","CHARENCEY":"21","CHARIGNY":"21","CHATELLENOT":"21","CORBERON":"21","CORGENGOUX":"21","CORMOT LE GRAND":"21","CORROMBLES":"21","CORSAINT":"21","CUISEREY":"21","CURTIL ST SEINE":"21","CUSSY LE CHATEL":"21","DIENAY":"21","ECHALOT":"21","EPERNAY SOUS GEVREY":"21","FAVEROLLES LES LUCEY":"21","FONCEGRIVE":"21","GILLY LES CITEAUX":"21","GISSEY LE VIEIL":"21","GRANCEY LE CHATEAU NEUVELLE":"21","GRESIGNY STE REINE":"21","HAUTEVILLE LES DIJON":"21","IZIER":"21","JUILLENAY":"21","LABERGEMENT LES SEURRE":"21","LAIGNES":"21","LAMARCHE SUR SAONE":"21","LARREY":"21","LEVERNOIS":"21","LIERNAIS":"21","LOUESME":"21","LUCENAY LE DUC":"21","MAGNY LAMBERT":"21","MALAIN":"21","MARCELLOIS":"21","MARCILLY ET DRACY":"21","MENESBLE":"21","MENETREUX LE PITOIS":"21","MIMEURE":"21","MINOT":"21","MIREBEAU SUR BEZE":"21","MOLESME":"21","MOLOY":"21","MONTAGNY LES SEURRE":"21","MONTIGNY ST BARTHELEMY":"21","MONTIGNY MORNAY VILLENEUVE VINGEANNE":"21","MONTLIOT ET COURCELLES":"21","MONTMOYEN":"21","MUSSY LA FOSSE":"21","NICEY":"21","NOIRON SUR SEINE":"21","NORGES LA VILLE":"21","PERRIGNY SUR L OGNON":"21","POMMARD":"21","PONT ET MASSENE":"21","POSANGES":"21","POTHIERES":"21","POUILLY SUR SAONE":"21","PRALON":"21","PRUSLY SUR OURCE":"21","QUINCY LE VICOMTE":"21","SACQUENAY":"21","ST GERMAIN DE MODEON":"21","ST JEAN DE BOEUF":"21","ST MAURICE SUR VINGEANNE":"21","SANTOSSE":"21","SAULX LE DUC":"21","SAUSSY":"21","SAVIGNY LE SEC":"21","LANNUX":"32","LARROQUE ENGALIN":"32","XIVRAY ET MARVOISIN":"55","ALLAIRE":"56","ARZAL":"56","AURAY":"56","BANGOR":"56","BEGANNE":"56","BREHAN":"56","BUBRY":"56","BULEON":"56","CAMPENEAC":"56","LA CHAPELLE GACELINE":"56","CONCORET":"56","LE COURS":"56","CRACH":"56","LA CROIX HELLEAN":"56","DAMGAN":"56","GESTEL":"56","GRAND CHAMP":"56","GUEHENNO":"56","GUELTAS":"56","GUEMENE SUR SCORFF":"56","GUENIN":"56","GUISCRIFF":"56","LE HEZO":"56","LANDEVANT":"56","LANGOELAN":"56","LARMOR PLAGE":"56","LIZIO":"56","LOCMARIAQUER":"56","LOCQUELTAS":"56","MARZAN":"56","MENEAC":"56","MEUCON":"56","MONTENEUF":"56","MONTERREIN":"56","LE PALAIS":"56","PLOEMEL":"56","PLOERDUT":"56","PLOURAY":"56","PLUVIGNER":"56","PONT SCORFF":"56","REGUINY":"56","ST ALLOUESTRE":"56","ST AVE":"56","ST BRIEUC DE MAURON":"56","STE BRIGITTE":"56","ST LAURENT SUR OUST":"56","ST LERY":"56","ST NICOLAS DU TERTRE":"56","ST NOLFF":"56","THEHILLAC":"56","LA TRINITE SUR MER":"56","AMANVILLERS":"57","AMNEVILLE":"57","ARGANCY":"57","ARS SUR MOSELLE":"57","ARZVILLER":"57","AUGNY":"57","BARCHAIN":"57","BECHY":"57","BERG SUR MOSELLE":"57","BERMERING":"57","BINING":"57","BITCHE":"57","BLANCHE EGLISE":"57","BOURGALTROFF":"57","BREIDENBACH":"57","BRONVAUX":"57","BUDLING":"57","BURLIONCOURT":"57","CHARLEVILLE SOUS BOIS":"57","CHARLY ORADOUR":"57","CHATEAU VOUE":"57","CONDE NORTHEN":"57","COUME":"57","DESSELING":"57","DESTRY":"57","DONNELAY":"57","EINCHEVILLE":"57","ENTRANGE":"57","ERNESTVILLER":"57","LES ETANGS":"57","FAULQUEMONT":"57","FIXEM":"57","FLORANGE":"57","FOSSIEUX":"57","FREMESTROFF":"57","FREYBOUSE":"57","MESSERY":"74","MIEUSSY":"74","MORILLON":"74","MORZINE":"74","NANCY SUR CLUSES":"74","NEYDENS":"74","NOVEL":"74","ONNION":"74","LE PETIT BORNAND LES GLIERES":"74","LE REPOSOIR":"74","LA RIVIERE ENVERSE":"74","ST GINGOLPH":"74","ST JEAN DE SIXT":"74","SALLANCHES":"74","LE SAPPEY":"74","SCIONZIER":"74","SEVRIER":"74","SIXT FER A CHEVAL":"74","THONES":"74","VALLORCINE":"74","VERCHAIX":"74","LES VILLARDS SUR THONES":"74","YVOIRE":"74","PARIS 03":"75","PARIS 07":"75","PARIS 19":"75","ALVIMARE":"76","ANCOURTEVILLE SUR HERICOURT":"76","ANGLESQUEVILLE L ESNEVAL":"76","VAL DE SAANE":"76","ANNEVILLE SUR SCIE":"76","ANQUETIERVILLE":"76","AUBERMESNIL AUX ERABLES":"76","AUTHIEUX RATIEVILLE":"76","AVESNES EN BRAY":"76","AVREMESNIL":"76","BELLEVILLE EN CAUX":"76","BENESVILLE":"76","BERMONVILLE":"76","BERNIERES":"76","BERTREVILLE":"76","BEUZEVILLE LA GRENIER":"76","BLAINVILLE CREVON":"76","BOIS GUILLAUME":"76","LE BOIS ROBERT":"76","BORNAMBUSC":"76","BOSC HYONS":"76","BOSC LE HARD":"76","BOSC ROGER SUR BUCHY":"76","BOUELLES":"76","LA BOUILLE":"76","BOURDAINVILLE":"76","BRETTEVILLE DU GRAND CAUX":"76","BUTOT":"76","CAILLEVILLE":"76","CANOUVILLE":"76","CANTELEU":"76","LES CENT ACRES":"76","LA CERLANGUE":"76","CLASVILLE":"76","CLAVILLE MOTTEVILLE":"76","CLEON":"76","CLERES":"76","COLLEVILLE":"76","CROISY SUR ANDELLE":"76","CROPUS":"76","CROSVILLE SUR SCIE":"76","CUVERVILLE SUR YERES":"76","DENESTANVILLE":"76","DIEPPE":"76","DROSAY":"76","EPOUVILLE":"76","EPREVILLE":"76","ERNEMONT SUR BUCHY":"76","ETOUTTEVILLE":"76","FALLENCOURT":"76","FAUVILLE EN CAUX":"76","FECAMP":"76","FONTAINE EN BRAY":"76","FOUCART":"76","FREULLEVILLE":"76","SENON":"55","SIVRY LA PERCHE":"55","SOMMEDIEUE":"55","SPINCOURT":"55","THONNE LA LONG":"55","TILLY SUR MEUSE":"55","TRESAUVAUX":"55","VARENNES EN ARGONNE":"55","VASSINCOURT":"55","VAUCOULEURS":"55","VAUQUOIS":"55","VELAINES":"55","VILLECLOYE":"55","VILLOTTE DEVANT LOUPPY":"55","VILLOTTE SUR AIRE":"55","VOUTHON HAUT":"55","AUGAN":"56","BIGNAN":"56","BRANDERION":"56","CADEN":"56","CARENTOIR":"56","CAUDAN":"56","CLEGUEREC":"56","COLPO":"56","CREDIN":"56","GOURIN":"56","GUEGON":"56","LE GUERNO":"56","GUILLIERS":"56","HELLEAN":"56","ILE D HOUAT":"56","ILE AUX MOINES":"56","INGUINIEL":"56","JOSSELIN":"56","KERFOURN":"56","LANDAUL":"56","LOCOAL MENDON":"56","MOHON":"56","MOUSTOIR AC":"56","MUZILLAC":"56","NEULLIAC":"56","NIVILLAC":"56","PENESTIN":"56","PLESCOP":"56","PLUMELIAU":"56","PLUNERET":"56","PRIZIAC":"56","QUISTINIC":"56","VAL D OUST":"56","ST ABRAHAM":"56","ST CONGARD":"56","ST GRAVE":"56","ST MARTIN SUR OUST":"56","ST PERREUX":"56","ST TUGDUAL":"56","SEGLIEN":"56","TREFFLEAN":"56","TREHORENTEUC":"56","LA TRINITE PORHOET":"56","BONO":"56","KERNASCLEDEN":"56","ADAINCOURT":"57","ADELANGE":"57","ALAINCOURT LA COTE":"57","ALZING":"57","ATTILLONCOURT":"57","AULNOIS SUR SEILLE":"57","BAMBIDERSTROFF":"57","BARONVILLE":"57","BEHREN LES FORBACH":"57","BENESTROFF":"57","BERIG VINTRANGE":"57","BERTHELMING":"57","BETTANGE":"57","BIDESTROFF":"57","BISTEN EN LORRAINE":"57","BLIESBRUCK":"57","BLIES EBERSING":"57","BLIES GUERSVILLER":"57","BOURSCHEID":"57","BOUSSEVILLER":"57","BOUST":"57","CATTENOM":"57","CHATEAU BREHAIN":"57","CHATEAU ROUGE":"57","CHEMERY LES DEUX":"57","NOGENT":"52","NOMECOURT":"52","ORCEVAUX":"52","ORMANCEY":"52","ORQUEVAUX":"52","OUTREMECOURT":"52","OZIERES":"52","PARNOY EN BASSIGNY":"52","PAROY SUR SAULX":"52","PEIGNEY":"52","PERRANCEY LES VIEUX MOULINS":"52","PIERREMONT SUR AMANCE":"52","PISSELOUP":"52","POINSENOT":"52","PONT LA VILLE":"52","LE MONTSAUGEONNAIS":"52","ROCHETAILLEE":"52","ROLAMPONT":"52","ROUGEUX":"52","RUPT":"52","SAINTS GEOSMES":"52","ST LOUP SUR AUJON":"52","SEMOUTIERS MONTSAON":"52","SUZANNECOURT":"52","TERNAT":"52","TORCENAY":"52","TORNAY":"52","TREMILLY":"52","TROISFONTAINES LA VILLE":"52","VAILLANT":"52","VAUXBONS":"52","VECQUEVILLE":"52","VIGNES LA COTE":"52","VILLARS SANTENOGE":"52","VILLEGUSIEN LE LAC":"52","VILLIERS SUR SUIZE":"52","VIOLOT":"52","VITRY EN MONTAGNE":"52","VOUECOURT":"52","AHUILLE":"53","ALEXAIN":"53","AMBRIERES LES VALLEES":"53","BOUESSAY":"53","BOURGON":"53","BRAINS SUR LES MARCHES":"53","CARELLES":"53","CHATEAU GONTIER":"53","CHEMAZE":"53","CHEVAIGNE DU MAINE":"53","COLOMBIERS DU PLESSIS":"53","COMMER":"53","CONTEST":"53","LA DOREE":"53","EPINEUX LE SEGUIN":"53","FOUGEROLLES DU PLESSIS":"53","LE GENEST ST ISLE":"53","GESNES":"53","GESVRES":"53","L HUISSERIE":"53","JAVRON LES CHAPELLES":"53","LARCHAMP":"53","LASSAY LES CHATEAUX":"53","MEZANGERS":"53","MONTAUDIN":"53","PARIGNE SUR BRAYE":"53","PLACE":"53","QUELAINES ST GAULT":"53","RAVIGNY":"53","ST AIGNAN SUR ROE":"53","ST BERTHEVIN":"53","ST ELLIER DU MAINE":"53","ST FORT":"53","ST GEORGES SUR ERVE":"53","ST GERMAIN DE COULAMER":"53","ST LOUP DU DORAT":"53","ST MICHEL DE LA ROE":"53","ST OUEN DES TOITS":"53","ST OUEN DES VALLONS":"53","ST PIERRE DES LANDES":"53","ST PIERRE LA COUR":"53","ST THOMAS DE COURCERIERS":"53","SIMPLE":"53","ALLAMPS":"54","ART SUR MEURTHE":"54","AUDUN LE ROMAN":"54","BAINVILLE AUX MIROIRS":"54","BARBONVILLE":"54","LARROQUE ST SERNIN":"32","LELIN LAPUJOLLE":"32","MAGNAN":"32","MAIGNAUT TAUZIA":"32","MALABAT":"32","MARAMBAT":"32","MARSAN":"32","MONBARDON":"32","MONBLANC":"32","MONCORNEIL GRAZAN":"32","MONLEZUN D ARMAGNAC":"32","MONTADET":"32","MONTAUT LES CRENEAUX":"32","MONT D ASTARAC":"32","MONTIES":"32","MORMES":"32","MOUCHAN":"32","MOUREDE":"32","NOUGAROULET":"32","ORNEZAN":"32","PANASSAC":"32","PEBEES":"32","PERGAIN TAILLAC":"32","PEYRUSSE GRANDE":"32","PEYRUSSE MASSAS":"32","PONSAMPERE":"32","PONSAN SOUBIRAN":"32","POUYLEBON":"32","POUY LOUBRIN":"32","PREIGNAN":"32","PROJAN":"32","RAZENGUES":"32","RICOURT":"32","RIGUEPEU":"32","ROQUEPINE":"32","ROZES":"32","ST ARAILLES":"32","ST ELIX THEUX":"32","ST GRIEDE":"32","STE MERE":"32","ST PAUL DE BAISE":"32","SARRANT":"32","SCIEURAC ET FLOURES":"32","SEMBOUES":"32","SOLOMIAC":"32","THOUX":"32","TOURRENQUETS":"32","TRAVERSERES":"32","URDENS":"32","VALENCE SUR BAISE":"32","VERGOIGNAN":"32","VERLUS":"32","VIC FEZENSAC":"32","AILLAS":"33","ANDERNOS LES BAINS":"33","ARCACHON":"33","ARSAC":"33","BAGAS":"33","BARIE":"33","BAYON SUR GIRONDE":"33","BAZAS":"33","BEAUTIRAN":"33","BEGADAN":"33","BEGLES":"33","BEGUEY":"33","BIGANOS":"33","BLAIGNAN":"33","BLASIMON":"33","BOSSUGAN":"33","BOURIDEYS":"33","BRAUD ET ST LOUIS":"33","CENAC":"33","CISSAC MEDOC":"33","COUBEYRAC":"33","COUQUEQUES":"33","CUBZAC LES PONTS":"33","DAUBEZE":"33","ESPIET":"33","LES ESSEINTES":"33","ETAULIERS":"33","EYNESSE":"33","FONTET":"33","GAJAC":"33","GAURIAGUET":"33","GIRONDE SUR DROPT":"33","GUITRES":"33","GRAVELOTTE":"57","GREMECEY":"57","GRINDORFF BIZING":"57","LE VAL DE GUEBLANGE":"57","GUEBLING":"57","GUERMANGE":"57","GUINGLANGE":"57","HAMPONT":"57","HAYANGE":"57","HEINING LES BOUZONVILLE":"57","HESSE":"57","HILSPRICH":"57","HOLVING":"57","HOMBOURG BUDANGE":"57","L HOPITAL":"57","INSVILLER":"57","IPPLING":"57","KERLING LES SIERCK":"57","HAUTE KONTZ":"57","LACHAMBRE":"57","LAUMESFELD":"57","LELLING":"57","LESSY":"57","LEY":"57","LIEDERSCHIEDT":"57","LIEHON":"57","LIOCOURT":"57","LIXING LES ST AVOLD":"57","MAINVILLERS":"57","MAIZERY":"57","MALAUCOURT SUR SEILLE":"57","MANOM":"57","MARANGE SILVANGE":"57","MARIMONT LES BENESTROFF":"57","LA MAXE":"57","METTING":"57","METZING":"57","MONCHEUX":"57","MONNEREN":"57","MORSBACH":"57","MOYEUVRE PETITE":"57","NEBING":"57","NEUFCHEF":"57","NEUFMOULINS":"57","NIDERVILLER":"57","NOISSEVILLE":"57","NOUSSEVILLER ST NABOR":"57","OMMERAY":"57","PHALSBOURG":"57","PONTPIERRE":"57","PREVOCOURT":"57","RENING":"57","REZONVILLE":"57","ROMBAS":"57","ROSSELANGE":"57","ROUSSY LE VILLAGE":"57","RURANGE LES THIONVILLE":"57","RUSTROFF":"57","STE RUFFINE":"57","SALONNES":"57","SCHWEYEN":"57","SERVIGNY LES RAVILLE":"57","SILLY SUR NIED":"57","SOLGNE":"57","TETING SUR NIED":"57","TREMERY":"57","TRESSANGE":"57","VASPERVILLER":"57","VAUDRECHING":"57","VESCHEIM":"57","VILLING":"57","VILSBERG":"57","VOLMERANGE LES BOULAY":"57","VOLMERANGE LES MINES":"57","VOLMUNSTER":"57","WALDWEISTROFF":"57","WALSCHBRONN":"57","WILLERWALD":"57","ZARBELING":"57","ZIMMING":"57","ALLIGNY COSNE":"58","ALLIGNY EN MORVAN":"58","AMAZY":"58","AUTHIOU":"58","BAZOCHES":"58","BAZOLLES":"58","BEUVRON":"58","BULCY":"58","FRY":"76","GERPONVILLE":"76","GONZEVILLE":"76","GREUVILLE":"76","HAUTOT ST SULPICE":"76","HAUTOT SUR MER":"76","HEBERVILLE":"76","INCHEVILLE":"76","LILLEBONNE":"76","LIMPIVILLE":"76","LINDEBEUF":"76","LINTOT LES BOIS":"76","LONGROY":"76","ARELAUNE EN SEINE":"76","MALLEVILLE LES GRES":"76","MANEGLISE":"76","MANEHOUVILLE":"76","MANNEVILLE ES PLAINS":"76","MAROMME":"76","MATHONVILLE":"76","MELLEVILLE":"76","MENERVAL":"76","LE MESNIL LIEUBRAY":"76","MEULERS":"76","MOLAGNIES":"76","MONTEROLIER":"76","MONT ST AIGNAN":"76","MOULINEAUX":"76","NESLE NORMANDEUSE":"76","NEUF MARCHE":"76","NOTRE DAME DE BLIQUETUIT":"76","NOTRE DAME DE BONDEVILLE":"76","NOTRE DAME DU PARC":"76","NULLEMONT":"76","OCQUEVILLE":"76","PETIT COURONNE":"76","LE PETIT QUEVILLY":"76","PREUSEVILLE":"76","QUEVREVILLE LA POTERIE":"76","RAINFREVILLE":"76","RETONVAL":"76","ROBERTOT":"76","ROLLEVILLE":"76","RONCHEROLLES EN BRAY":"76","RONCHEROLLES SUR LE VIVIER":"76","RONCHOIS":"76","ROUXMESNIL BOUTEILLES":"76","SAANE ST JUST":"76","STE AGATHE D ALIERMONT":"76","ST AUBIN ROUTOT":"76","ST DENIS SUR SCIE":"76","ST GERMAIN SUR EAULNE":"76","ST JEAN DE FOLLEVILLE":"76","ST LAURENT EN CAUX":"76","STE MARIE AU BOSC":"76","STE MARIE DES CHAMPS":"76","ST MARTIN DE BOSCHERVILLE":"76","ST MARTIN DU BEC":"76","ST OUEN DU BREUIL":"76","ST PIERRE LE VIGER":"76","ST SAIRE":"76","ST SAUVEUR D EMALLEVILLE":"76","ST VINCENT CRAMESNIL":"76","SASSETOT LE MALGARDE":"76","SAUCHAY":"76","SAUMONT LA POTERIE":"76","SENNEVILLE SUR FECAMP":"76","SORQUAINVILLE":"76","THIERGEVILLE":"76","THIOUVILLE":"76","LE TILLEUL":"76","TOCQUEVILLE LES MURS":"76","TOURVILLE LA RIVIERE":"76","TOUSSAINT":"76","VATTETOT SOUS BEAUMONT":"76","VENESTANVILLE":"76","BUTOT VENESVILLE":"76","VEULETTES SUR MER":"76","VIEUX MANOIR":"76","VITTEFLEUR":"76","CHERISEY":"57","CHESNY":"57","CHICOURT":"57","CORNY SUR MOSELLE":"57","COURCELLES CHAUSSY":"57","CRAINCOURT":"57","CUVRY":"57","DABO":"57","DALEM":"57","DELME":"57","DENTING":"57","DIEUZE":"57","EGUELSHARDT":"57","ELZANGE":"57","ESCHERANGE":"57","FOLKLING":"57","FONTENY":"57","FORBACH":"57","FRANCALTROFF":"57","FREMERY":"57","GAVISSE":"57","GROS REDERCHING":"57","GUEBESTROFF":"57","GUEBLANGE LES DIEUZE":"57","GUENANGE":"57","GUERSTLING":"57","GUINZELING":"57","HALLERING":"57","HAMBACH":"57","HARGARTEN AUX MINES":"57","HARPRICH":"57","HATTIGNY":"57","HAYES":"57","HERANGE":"57","HOLACOURT":"57","HUNDLING":"57","JALLAUCOURT":"57","JURY":"57","KALHAUSEN":"57","KAPPELKINGER":"57","KEDANGE SUR CANNER":"57","KERBACH":"57","KIRSCHNAUMEN":"57","LAMBACH":"57","LANDROFF":"57","LANING":"57","LAQUENEXY":"57","LEMBERG":"57","LEMUD":"57","LENGELSHEIM":"57","LEZEY":"57","LINDRE BASSE":"57","LINDRE HAUTE":"57","LHOR":"57","LONGEVILLE LES METZ":"57","LORQUIN":"57","LOUDREFING":"57","LOUTZVILLER":"57","LUBECOURT":"57","MACHEREN":"57","MAIZIERES LES METZ":"57","MALLING":"57","MALROY":"57","MENSKIRCH":"57","MERTEN":"57","METZERVISSE":"57","MOMERSTROFF":"57","MONDELANGE":"57","MONDORFF":"57","MONTENACH":"57","MORHANGE":"57","MORVILLE LES VIC":"57","MOYENVIC":"57","NARBEFONTAINE":"57","NEUFGRANGE":"57","NORROY LE VENEUR":"57","NOUSSEVILLER LES BITCHE":"57","OBERDORFF":"57","OBERGAILBACH":"57","OBERSTINZEL":"57","OETING":"57","ORMERSVILLER":"57","OTTANGE":"57","OTTONVILLE":"57","PETIT REDERCHING":"57","PEVANGE":"57","PIBLANGE":"57","PORCELETTE":"57","POURNOY LA CHETIVE":"57","PUTTELANGE AUX LACS":"57","RECHICOURT LE CHATEAU":"57","REDING":"57","REMELFANG":"57","RHODES":"57","RICHE":"57","RICHELING":"57","BARISEY LA COTE":"54","LES BAROCHES":"54","BATTIGNY":"54","BENNEY":"54","BETTAINVILLERS":"54","BORVILLE":"54","BOUXIERES AUX CHENES":"54","BOUZANVILLE":"54","BREHAIN LA VILLE":"54","BRULEY":"54","BULLIGNY":"54","CHAZELLES SUR ALBE":"54","CHOLOY MENILLOT":"54","CLEREY SUR BRENON":"54","COURBESSEAUX":"54","CUSTINES":"54","DIARVILLE":"54","DOMBASLE SUR MEURTHE":"54","DROUVILLE":"54","ESSEY LA COTE":"54","ETREVAL":"54","FAULX":"54","FLAVIGNY SUR MOSELLE":"54","FOUG":"54","FRAISNES EN SAINTOIS":"54","FROUARD":"54","GELAUCOURT":"54","GIBEAUMEIX":"54","GIRIVILLER":"54","GONDREXON":"54","GRIMONVILLER":"54","GUGNEY":"54","HAGEVILLE":"54","HALLOVILLE":"54","HAROUE":"54","HENAMENIL":"54","HERSERANGE":"54","HOUDEMONT":"54","HUSSIGNY GODBRANGE":"54","JUVRECOURT":"54","LAGNEY":"54","LALOEUF":"54","LANEUVELOTTE":"54","LANFROICOURT":"54","LAXOU":"54","LEMENIL MITRY":"54","LEXY":"54","LIMEY REMENAUVILLE":"54","LIVERDUN":"54","LUNEVILLE":"54","MANCIEULLES":"54","MEHONCOURT":"54","MENIL LA TOUR":"54","MERCY LE BAS":"54","MERVILLER":"54","MINORVILLE":"54","MOIVRONS":"54","MONCEL SUR SEILLE":"54","MONTENOY":"54","MONT LE VIGNOBLE":"54","MOUSSON":"54","MURVILLE":"54","NANCY":"54","NORROY LE SEC":"54","OZERAILLES":"54","POMPEY":"54","PORT SUR SEILLE":"54","PULNOY":"54","RAUCOURT":"54","RECLONVILLE":"54","REHAINVILLER":"54","REMBERCOURT SUR MAD":"54","REMEREVILLE":"54","ROSIERES AUX SALINES":"54","ROSIERES EN HAYE":"54","ST AIL":"54","ST BAUSSANT":"54","ST JULIEN LES GORZE":"54","SAIZERAIS":"54","SAULXURES LES VANNES":"54","SAXON SION":"54","SEICHAMPS":"54","SELAINCOURT":"54","SIVRY":"54","CHALAMPE":"68","DIEFMATTEN":"68","DURRENENTZEN":"68","HOSTENS":"33","IZON":"33","LABARDE":"33","LANDIRAS":"33","LARUSCADE":"33","LEGE CAP FERRET":"33","LESPARRE MEDOC":"33","LIGNAN DE BAZAS":"33","LISTRAC MEDOC":"33","LORMONT":"33","LUGAIGNAC":"33","MACAU":"33","MARGUERON":"33","MERIGNAS":"33","MONTUSSAN":"33","MOULIETS ET VILLEMARTIN":"33","MOULIS EN MEDOC":"33","PAILLET":"33","PAREMPUYRE":"33","PAUILLAC":"33","LES PEINTURES":"33","PESSAC":"33","LE PIAN MEDOC":"33","PODENSAC":"33","PORTETS":"33","PRIGNAC ET MARCAMPS":"33","PUYBARBAN":"33","RIONS":"33","ROAILLAN":"33","LA ROQUILLE":"33","SADIRAC":"33","ST CHRISTOLY MEDOC":"33","ST CIBARD":"33","ST CIERS D ABZAC":"33","STE FOY LA GRANDE":"33","ST GENES DE FRONSAC":"33","ST GERMAIN D ESTEUIL":"33","ST LOUBERT":"33","ST MEDARD DE GUIZIERES":"33","ST PEY DE CASTETS":"33","ST PHILIPPE D AIGUILLE":"33","ST PIERRE DE BAT":"33","ST PIERRE DE MONS":"33","ST QUENTIN DE BARON":"33","ST QUENTIN DE CAPLONG":"33","ST SEURIN DE CADOURNE":"33","ST SEVE":"33","ST TROJAN":"33","ST VINCENT DE PERTIGNAS":"33","ST YZAN DE SOUDIAC":"33","SAUGON":"33","SAUMOS":"33","SAVIGNAC DE L ISLE":"33","SOULAC SUR MER":"33","SOULIGNAC":"33","TALENCE":"33","TOULENNE":"33","LE TUZAN":"33","VERAC":"33","ABEILHAN":"34","LES AIRES":"34","ARGELLIERS":"34","ASPIRAN":"34","AUMES":"34","BAILLARGUES":"34","BEZIERS":"34","CANDILLARGUES":"34","CAPESTANG":"34","CEILHES ET ROCOZELS":"34","CESSENON SUR ORB":"34","CESSERAS":"34","COULOBRES":"34","COURNONTERRAL":"34","ESPONDEILHAN":"34","FABREGUES":"34","LAGAMAS":"34","LAVERUNE":"34","LESPIGNAN":"34","LEZIGNAN LA CEBE":"34","LUNEL":"34","LUNEL VIEL":"34","MAS DE LONDRES":"34","LA CELLE SUR LOIRE":"58","CERVON":"58","CHITRY LES MINES":"58","CORBIGNY":"58","CORVOL L ORGUEILLEUX":"58","CUNCY LES VARZY":"58","DIROL":"58","DONZY":"58","DORNES":"58","ENTRAINS SUR NOHAIN":"58","GACOGNE":"58","GARCHY":"58","GERMENAY":"58","ISENAY":"58","LANGERON":"58","LURCY LE BOURG":"58","MENOU":"58","METZ LE COMTE":"58","MOISSY MOULINOT":"58","MONTAPAS":"58","NANNAY":"58","NEVERS":"58","LA NOCLE MAULAIX":"58","OUAGNE":"58","OUDAN":"58","ST ETIENNE ROILAYE":"60","ST LEGER EN BRAY":"60","ST QUENTIN DES PRES":"60","ST VAAST DE LONGMONT":"60","SALENCY":"60","SEREVILLERS":"60","TARTIGNY":"60","THIESCOURT":"60","TRIE CHATEAU":"60","TRUMILLY":"60","ULLY ST GEORGES":"60","VALESCOURT":"60","VAUCHELLES":"60","VIEFVILLERS":"60","VILLERS ST PAUL":"60","VILLERS ST SEPULCRE":"60","VILLERS SUR COUDUN":"60","WELLES PERENNES":"60","LES AUTHIEUX DU PUITS":"61","BEAUFAI":"61","BEAUVAIN":"61","BELLOU EN HOULME":"61","BERD HUIS":"61","BERJOU":"61","BOISSEI LA LANDE":"61","LE BOURG ST LEONARD":"61","BRETONCELLES":"61","BRULLEMAIL":"61","CAHAN":"61","CALIGNY":"61","LE CERCUEIL":"61","CERISY BELLE ETOILE":"61","CETON":"61","CHAMPOSOULT":"61","CHANDAI":"61","LA CHAPELLE BICHE":"61","LA CHAPELLE MONTLIGEON":"61","CISAI ST AUBIN":"61","COMMEAUX":"61","COULIMER":"61","LA COULONCHE":"61","CUISSAI":"61","ECORCHES":"61","ESSAY":"61","LA FERRIERE AU DOYEN":"61","LA FERRIERE AUX ETANGS":"61","FONTAINE LES BASSETS":"61","FONTENAI LES LOUVETS":"61","GACE":"61","GINAI":"61","GUERQUESALLES":"61","L HOME CHAMONDOT":"61","WANCHY CAPVAL":"76","YPORT":"76","YVILLE SUR SEINE":"76","AMPONVILLE":"77","AROZ":"70","ATHESANS ETROITEFONTAINE":"70","AUTOREILLE":"70","AUTREY LES CERRE":"70","AUTREY LE VAY":"70","BEAUJEU ET QUITTEUR":"70","BONNEVENT VELLOREILLE":"70","BOURGUIGNON LES MOREY":"70","BRESILLEY":"70","BROYE AUBIGNEY MONTSEUGNY":"70","LA BRUYERE":"70","BUFFIGNECOURT":"70","CENDRECOURT":"70","CHALONVILLARS":"70","CHAMPLITTE":"70","CHANCEY":"70","CHARCENNE":"70","CHARGEY LES GRAY":"70","CHASSEY LES SCEY":"70","CHAUMERCENNE":"70","COLOMBE LES VESOUL":"70","COMBEAUFONTAINE":"70","COMBERJON":"70","CUGNEY":"70","DENEVRE":"70","ESPRELS":"70","FAUCOGNEY ET LA MER":"70","FAVERNEY":"70","FLEUREY LES ST LOUP":"70","FONTENOIS LA VILLE":"70","FRANCALMONT":"70","FRAMONT":"70","FRESNE ST MAMES":"70","GENEVREUILLE":"70","GRANGES LE BOURG":"70","GRAY":"70","GY":"70","LANTENOT":"70","LIEFFRANS":"70","LOEUILLEY":"70","MAGNONCOURT":"70","MAGNY LES JUSSEY":"70","MALVILLERS":"70","MARAST":"70","MELECEY":"70","MEURCOURT":"70","MIELLIN":"70","MONTCEY":"70","MONTIGNY LES VESOUL":"70","MONTJUSTIN ET VELOTTE":"70","VILLERS CHEMIN ET MONT LES ETRELLES":"70","MONT LE VERNOIS":"70","MONT ST LEGER":"70","LA ROCHE MOREY":"70","MOTEY BESUCHE":"70","MOTEY SUR SAONE":"70","NEUREY EN VAUX":"70","NEUREY LES LA DEMIE":"70","OISELAY ET GRACHAUX":"70","PENNESIERES":"70","PIN":"70","PREIGNEY":"70","PUSEY":"70","QUERS":"70","RAY SUR SAONE":"70","RECOLOGNE LES RIOZ":"70","RENAUCOURT":"70","LA RESIE ST MARTIN":"70","RIGNOVELLE":"70","RIOZ":"70","RUHANS":"70","RUPT SUR SAONE":"70","ST GAND":"70","ST VALBERT":"70","SAUVIGNEY LES PESMES":"70","SAVOYEUX":"70","SECENANS":"70","SENARGENT MIGNAFANS":"70","THEULEY":"70","RODEMACK":"57","ROMELFING":"57","ROPPEVILLER":"57","ROSBRUCK":"57","ROUPELDANGE":"57","RUSSANGE":"57","ST JURE":"57","ST LOUIS LES BITCHE":"57","STE MARIE AUX CHENES":"57","SARRALTROFF":"57","SAULNY":"57","SEREMANGE ERZANGE":"57","SERVIGNY LES STE BARBE":"57","SOUCHT":"57","SPICHEREN":"57","TARQUIMPOL":"57","TENTELING":"57","THICOURT":"57","THIMONVILLE":"57","THIONVILLE":"57","TINCRY":"57","TROISFONTAINES":"57","VAHL LES FAULQUEMONT":"57","VALLERANGE":"57","VANTOUX":"57","VAXY":"57","VELVING":"57","VERGAVILLE":"57","VIEUX LIXHEIM":"57","HAUTE VIGNEULLES":"57","VIRMING":"57","VITTONCOURT":"57","VOYER":"57","WALDWISSE":"57","WIESVILLER":"57","WINTERSBOURG":"57","WITTRING":"57","WOELFLING LES SARREGUEMINES":"57","ARQUIAN":"58","AVREE":"58","BALLERAY":"58","BOUHY":"58","BREVES":"58","BRINON SUR BEUVRON":"58","CERCY LA TOUR":"58","CESSY LES BOIS":"58","CHAMPVOUX":"58","CHAULGNES":"58","LA COLLANCELLE":"58","COLMERY":"58","CRUX LA VILLE":"58","DIENNES AUBIGNY":"58","DUN LES PLACES":"58","EMPURY":"58","GUERIGNY":"58","GUIPY":"58","LANTY":"58","LIMANTON":"58","LA MACHINE":"58","MAGNY COURS":"58","MENESTREAU":"58","MONCEAUX LE COMTE":"58","MOUX EN MORVAN":"58","MURLIN":"58","OUROUER":"58","POUILLY SUR LOIRE":"58","PREMERY":"58","RAVEAU":"58","ROUY":"58","ST AMAND EN PUISAYE":"58","ST AUBIN LES FORGES":"58","ST BONNOT":"58","ST BRISSON":"58","ST HILAIRE EN MORVAN":"58","ST HILAIRE FONTAINE":"58","ST HONORE LES BAINS":"58","ST LEGER DE FOUGERET":"58","ST LEGER DES VIGNES":"58","ST MALO EN DONZIOIS":"58","SAUVIGNY LES BOIS":"58","SAXI BOURDON":"58","ABLANCOURT":"51","ST MARTIN D ABLOIS":"51","ANGLURE":"51","AULNAY SUR MARNE":"51","AVENAY VAL D OR":"51","BASLIEUX SOUS CHATILLON":"51","BEINE NAUROY":"51","EMLINGEN":"68","ETEIMBES":"68","FERRETTE":"68","FRELAND":"68","FROENINGEN":"68","GEISPITZEN":"68","GOLDBACH ALTENBACH":"68","GUEVENATTEN":"68","HAGENTHAL LE HAUT":"68","HEIDWILLER":"68","HESINGUE":"68","HIRTZFELDEN":"68","PORTE DU RIED":"68","HOMBOURG":"68","HORBOURG WIHR":"68","HOUSSEN":"68","ISSENHEIM":"68","KAPPELEN":"68","KEMBS":"68","KOESTLACH":"68","LANDSER":"68","LARGITZEN":"68","LAUTENBACHZELL":"68","LIEBSDORF":"68","LINSDORF":"68","LUCELLE":"68","VALDIEU LUTRAN":"68","MICHELBACH LE HAUT":"68","MITTELWIHR":"68","MITTLACH":"68","MOERNACH":"68","MONTREUX JEUNE":"68","MONTREUX VIEUX":"68","MOOSLARGUE":"68","MORSCHWILLER LE BAS":"68","MUHLBACH SUR MUNSTER":"68","MUNCHHOUSE":"68","NEUF BRISACH":"68","ILLTAL":"68","PFETTERHOUSE":"68","RAEDERSDORF":"68","RAMMERSMATT":"68","RANSPACH LE HAUT":"68","RIESPACH":"68","RODERN":"68","ROMAGNY":"68","ROSENAU":"68","RUELISHEIM":"68","ST COSME":"68","STE CROIX AUX MINES":"68","STE CROIX EN PLAINE":"68","SCHWOBEN":"68","SICKERT":"68","SOULTZBACH LES BAINS":"68","TAGSDORF":"68","THANN":"68","UFFHEIM":"68","UFFHOLTZ":"68","VILLAGE NEUF":"68","VOEGTLINSHOFFEN":"68","WASSERBOURG":"68","WATTWILLER":"68","WETTOLSHEIM":"68","WILLER SUR THUR":"68","WINTZENHEIM":"68","WOLSCHWILLER":"68","WUENHEIM":"68","ZIMMERBACH":"68","AMPLEPUIS":"69","ANCY":"69","ARNAS":"69","BELMONT D AZERGUES":"69","BIBOST":"69","LE BOIS D OINGT":"69","CHAPONOST":"69","CHAUSSAN":"69","LES CHERES":"69","CHIROUBLES":"69","COISE":"69","COLLONGES AU MONT D OR":"69","CORCELLES EN BEAUJOLAIS":"69","CUBLIZE":"69","DAREIZE":"69","FLEURIE":"69","LES HALLES":"69","HAUTE RIVOIRE":"69","JULIENAS":"69","LANTIGNIE":"69","LENTILLY":"69","LETRA":"69","LOIRE SUR RHONE":"69","LUCENAY":"69","MARCY L ETOILE":"69","MONTARNAUD":"34","MONTBAZIN":"34","MONTBLANC":"34","MURVIEL LES MONTPELLIER":"34","PEGAIROLLES DE L ESCALETTE":"34","POILHES":"34","PORTIRAGNES":"34","POUSSAN":"34","PRADES LE LEZ":"34","PUISSALICON":"34","PUISSERGUIER":"34","ROMIGUIERES":"34","ST BAUZILLE DE PUTOIS":"34","ST ETIENNE D ALBAGNAN":"34","ST GENIES DES MOURGUES":"34","ST GERVAIS SUR MARE":"34","ST GUIRAUD":"34","ST HILAIRE DE BEAUVOIR":"34","ST JEAN DE FOS":"34","ST JEAN DE VEDAS":"34","ST MARTIN DE L ARCON":"34","ST MATHIEU DE TREVIERS":"34","LA SALVETAT SUR AGOUT":"34","SAUTEYRARGUES":"34","LA TOUR SUR ORB":"34","USCLAS DU BOSC":"34","VALMASCLE":"34","VALRAS PLAGE":"34","VALROS":"34","VIAS":"34","VIEUSSAN":"34","VILLENEUVE LES BEZIERS":"34","VILLESPASSANS":"34","VILLEVEYRAC":"34","ARBRISSEL":"35","AVAILLES SUR SEICHE":"35","BAILLE":"35","BAIN DE BRETAGNE":"35","BAINS SUR OUST":"35","BAZOUGES LA PEROUSE":"35","BEAUCE":"35","BOVEL":"35","BRUC SUR AFF":"35","BRUZ":"35","CESSON SEVIGNE":"35","CHANTEPIE":"35","LA CHAPELLE JANSON":"35","LA CHAPELLE THOUARAULT":"35","CINTRE":"35","COMBLESSAC":"35","CUGUEN":"35","DOMALAIN":"35","FORGES LA FORET":"35","GENNES SUR SEICHE":"35","GUICHEN":"35","HEDE BAZOUGES":"35","LAILLE":"35","LANDAVRAN":"35","LANDEAN":"35","LECOUSSE":"35","LIFFRE":"35","LE LOROUX":"35","MAXENT":"35","MONTGERMONT":"35","MONTREUIL LE GAST":"35","MOUAZE":"35","MUEL":"35","NOYAL SOUS BAZOUGES":"35","PANCE":"35","PARTHENAY DE BRETAGNE":"35","LE PETIT FOUGERAY":"35","PIRE SUR SEICHE":"35","PLESDER":"35","QUEBRIAC":"35","RENNES":"35","STE ANNE SUR VILAINE":"35","ST AUBIN DES LANDES":"35","ST AUBIN DU PAVAIL":"35","ST BRIEUC DES IFFS":"35","ST BROLADRE":"35","ST CHRISTOPHE DES BOIS":"35","ST COULOMB":"35","ST JEAN SUR COUESNON":"35","ST LEGER DES PRES":"35","ST MARC LE BLANC":"35","ST MARC SUR COUESNON":"35","ST OUEN DES ALLEUX":"35","ST REMY DU PLAIN":"35","LA LANDE DE GOULT":"61","LOISAIL":"61","LONRAI":"61","LOUVIERES EN AUGE":"61","MAGNY LE DESERT":"61","MAUVES SUR HUISNE":"61","MEDAVY":"61","MENIL VIN":"61","LES MENUS":"61","MIEUXCE":"61","MONTCHEVREL":"61","MONTGAUDRY":"61","MONTSECRET CLAIREFOUGERE":"61","NEAUPHE SUR DIVE":"61","NEUVILLE SUR TOUQUES":"61","NORMANDEL":"61","LE PAS ST L HOMER":"61","PLANCHES":"61","RONAI":"61","ST AUBIN DE COURTERAIE":"61","ST BOMER LES FORGES":"61","BOISCHAMPRE":"61","ST DENIS SUR HUISNE":"61","ST EVROULT DE MONTFORT":"61","ST EVROULT NOTRE DAME DU BOIS":"61","ST FULGENT DES ORMES":"61","ST GERMAIN DE CLAIREFEUILLE":"61","ST GERMAIN DE LA COUDRE":"61","ST GERMAIN DES GROIS":"61","ST GERVAIS DU PERRON":"61","ST GILLES DES MARAIS":"61","ST HILAIRE DE BRIOUZE":"61","ST JULIEN SUR SARTHE":"61","ST LANGIS LES MORTAGNE":"61","ST MARTIN D ECUBLEI":"61","ST MAURICE LES CHARENCEY":"61","ST OUEN DE LA COUR":"61","ST PHILBERT SUR ORNE":"61","ST QUENTIN LES CHARDONNETS":"61","ST ROCH SUR EGRENNE":"61","SARCEAUX":"61","SEVIGNY":"61","SURVIE":"61","BAGNOLES DE L ORNE NORMANDIE":"61","TICHEVILLE":"61","TOURNAI SUR DIVE":"61","VAUNOISE":"61","ACQUIN WESTBECOURT":"62","ADINFER":"62","ALINCTHUN":"62","ARDRES":"62","AUCHEL":"62","AUCHY AU BOIS":"62","AUDEMBERT":"62","AUDINCTHUN":"62","AUMERVAL":"62","AVERDOINGT":"62","AVESNES":"62","AZINCOURT":"62","BAILLEUL LES PERNES":"62","BARASTRE":"62","BARLIN":"62","BASSEUX":"62","BAYENGHEM LES SENINGHEM":"62","BAZINGHEN":"62","BENIFONTAINE":"62","BERMICOURT":"62","BERNIEULLES":"62","BIENVILLERS AU BOIS":"62","BLANGERVAL BLANGERMONT":"62","BLANGY SUR TERNOISE":"62","BOIRY NOTRE DAME":"62","BOISDINGHEM":"62","BONNINGUES LES ARDRES":"62","BOURECQ":"62","BOURET SUR CANCHE":"62","BREMES":"62","BRIMEUX":"62","BRUNEMBERT":"62","THIEFFRANS":"70","THIENANS":"70","LE TREMBLOIS":"70","TRESILLEY":"70","TROMAREY":"70","LA VAIVRE":"70","LE VAL DE GOUHENANS":"70","VAUCONCOURT NERVEZAIN":"70","VELLECHEVREUX ET COURBENANS":"70","VELLEFAUX":"70","VELLE LE CHATEL":"70","VELLEXON QUEUTREY ET VAUDEY":"70","VELLOREILLE LES CHOYE":"70","VELORCEY":"70","VENERE":"70","VILLARGENT":"70","VILLEFRANCON":"70","LA VILLENEUVE BELLENOYE LA MAIZE":"70","VILLEPAROIS":"70","VILLERS LES LUXEUIL":"70","VILLERS SUR PORT":"70","VOLON":"70","ALLERIOT":"71","ARTAIX":"71","AUTUN":"71","BARIZEY":"71","BARNAY":"71","BAUDEMONT":"71","BAUDRIERES":"71","BEAUBERY":"71","BOIS STE MARIE":"71","BOUHANS":"71","BURNAND":"71","CHALMOUX":"71","LA CHAPELLE DE GUINCHAY":"71","LA CHAPELLE SOUS BRANCION":"71","LA CHAPELLE SOUS UCHON":"71","CHARBONNAT":"71","CHARDONNAY":"71","CHARETTE VARENNES":"71","CHARRECEY":"71","CHASSEY LE CAMP":"71","CHEVAGNY SUR GUYE":"71","CIEL":"71","CLERMAIN":"71","COLOMBIER EN BRIONNAIS":"71","CONDAL":"71","CRECHES SUR SAONE":"71","CRONAT":"71","CUISEAUX":"71","CUSSY EN MORVAN":"71","CUZY":"71","DENNEVY":"71","DETTEY":"71","DRACY ST LOUP":"71","DYO":"71","ECUISSES":"71","ETANG SUR ARROUX":"71","ETRIGNY":"71","FLACEY EN BRESSE":"71","FLEURY LA MONTAGNE":"71","FRAGNES LA LOYERE":"71","GRANDVAUX":"71","GRANGES":"71","IGORNAY":"71","JALOGNY":"71","JOUDES":"71","LANS":"71","LAYS SUR LE DOUBS":"71","MACON":"71","MAILLY":"71","MELLECEY":"71","MONTCENIS":"71","MONTCHANIN":"71","MONT LES SEURRE":"71","MORLET":"71","MORNAY":"71","MOUTHIER EN BRESSE":"71","OZOLLES":"71","PALINGES":"71","PARIS L HOPITAL":"71","POISSON":"71","POURLANS":"71","PRISSE":"71","PRIZY":"71","RECLESNE":"71","ST AMBREUIL":"71","ST ANDRE LE DESERT":"71","BERGERES SOUS MONTMIRAIL":"51","BERRU":"51","BETHENY":"51","BIGNICOURT SUR MARNE":"51","BOISSY LE REPOS":"51","BOULEUSE":"51","BRANSCOURT":"51","BRAUX ST REMY":"51","BRIMONT":"51","BUSSY LE CHATEAU":"51","BUSSY LETTREE":"51","CHALONS EN CHAMPAGNE":"51","CHAMBRECY":"51","CHAMPGUYON":"51","CHAMPILLON":"51","LES CHARMONTOIS":"51","LE CHATELIER":"51","CHATILLON SUR MARNE":"51","CHATILLON SUR MORIN":"51","COIZARD JOCHES":"51","VAL DES MARAIS":"51","CONDE SUR MARNE":"51","CONGY":"51","CORMOYEUX":"51","COURDEMANGES":"51","COURMAS":"51","CRUGNY":"51","CUCHERY":"51","DAMPIERRE LE CHATEAU":"51","DOMMARTIN DAMPIERRE":"51","DROSNAY":"51","ECURY SUR COOLE":"51","EPENSE":"51","ESCLAVOLLES LUREY":"51","EUVY":"51","FAGNIERES":"51","FAVRESSE":"51","FONTAINE DENIS NUISY":"51","GRANGES SUR AUBE":"51","GRATREUIL":"51","HEILTZ LE MAURUPT":"51","HERPONT":"51","JONCHERY SUR SUIPPE":"51","JONQUERY":"51","LAVANNES":"51","MAILLY CHAMPAGNE":"51","MARCILLY SUR SEINE":"51","MAREUIL EN BRIE":"51","MAREUIL LE PORT":"51","MASSIGES":"51","MATOUGUES":"51","MERY PREMECY":"51","LE MESNIL SUR OGER":"51","MORSAINS":"51","MOURMELON LE GRAND":"51","LA NEUVILLE AUX BOIS":"51","LA NEUVILLE AU PONT":"51","NOIRLIEU":"51","PEVY":"51","PLICHANCOURT":"51","POGNY":"51","POMACLE":"51","PONTFAVERGER MORONVILLIERS":"51","POSSESSE":"51","ROUFFY":"51","ROUVROY RIPONT":"51","ST AMAND SUR FION":"51","ST GERMAIN LA VILLE":"51","ST HILAIRE AU TEMPLE":"51","ST HILAIRE LE PETIT":"51","ST JEAN DEVANT POSSESSE":"51","ST LUMIER LA POPULEUSE":"51","ST MARD LES ROUFFY":"51","ST MARTIN AUX CHAMPS":"51","ST MARTIN SUR LE PRE":"51","ST QUENTIN LE VERGER":"51","ST QUENTIN SUR COOLE":"51","ST REMY SOUS BROYES":"51","SARON SUR AUBE":"51","SCRUPT":"51","SEZANNE":"51","SILLERY":"51","SOGNY EN L ANGLE":"51","SOMME BIONNE":"51","SOMME TOURBE":"51","SOUDE":"51","SOUDRON":"51","TAISSY":"51","THIEBLEMONT FAREMONT":"51","LE THOULT TROSNAY":"51","MEAUX LA MONTAGNE":"69","MORANCE":"69","MORNANT":"69","NEUVILLE SUR SAONE":"69","PIERRE BENITE":"69","POLEYMIEUX AU MONT D OR":"69","POMEYS":"69","POUILLY LE MONIAL":"69","RONTALON":"69","SALLES ARBUISSONNAS EN BEAUJOLAIS":"69","LES SAUVAGES":"69","ST CLEMENT LES PLACES":"69","ST CLEMENT SUR VALSONNE":"69","STE CONSORCE":"69","ST CYR LE CHATOUX":"69","ST DIDIER AU MONT D OR":"69","ST DIDIER SOUS RIVERIE":"69","ST FORGEUX":"69","STE FOY L ARGENTIERE":"69","STE FOY LES LYON":"69","ST GENIS L ARGENTIERE":"69","ST IGNY DE VERS":"69","ST JULIEN SUR BIBOST":"69","ST MARTIN EN HAUT":"69","ST MAURICE SUR DARGOIRE":"69","ST ROMAIN EN GAL":"69","ST SYMPHORIEN SUR COISE":"69","ST VINCENT DE REINS":"69","TALUYERS":"69","TARARE":"69","THIZY LES BOURGS":"69","TRADES":"69","TUPIN ET SEMONS":"69","VALSONNE":"69","VILLEFRANCHE SUR SAONE":"69","VILLE SUR JARNIOUX":"69","YZERON":"69","CHAPONNAY":"69","CORBAS":"69","DECINES CHARPIEU":"69","JONS":"69","MIONS":"69","ST BONNET DE MURE":"69","SATHONAY CAMP":"69","SEREZIN DU RHONE":"69","SOLAIZE":"69","LYON 09":"69","AILLEVANS":"70","ANCIER":"70","ARSANS":"70","AULX LES CROMARY":"70","AUTET":"70","AVRIGNEY VIREY":"70","BASSIGNEY":"70","BEAUMOTTE AUBERTANS":"70","BELONCHAMP":"70","BOULIGNEY":"70","BOURGUIGNON LES CONFLANS":"70","BREUREY LES FAVERNEY":"70","BRUSSEY":"70","BUCEY LES GY":"70","CHAMBORNAY LES BELLEVAUX":"70","LA CHAPELLE ST QUILLAIN":"70","CHARGEY LES PORT":"70","CHAUX LA LOTIERE":"70","CHAVANNE":"70","CORRAVILLERS":"70","LA COTE":"70","COURCUIRE":"70","COUTHENANS":"70","CUVE":"70","DAMBENOIT LES COLOMBE":"70","DAMPVALLEY LES COLOMBE":"70","DAMPVALLEY ST PANCRAS":"70","DELAIN":"70","ECHAVANNE":"70","ESBOZ BREST":"70","ESMOULINS":"70","ETOBON":"70","FALLON":"70","FERRIERES LES SCEY":"70","FLEUREY LES FAVERNEY":"70","FONDREMAND":"70","FONTAINE LES LUXEUIL":"70","FOUVENT ST ANDOCHE":"70","FRAHIER ET CHATEBIER":"70","FRANCHEVELLE":"70","ST SAUVEUR DES LANDES":"35","ST SULPICE LA FORET":"35","ST THURIAL":"35","LE SEL DE BRETAGNE":"35","SIXT SUR AFF":"35","LE THEIL DE BRETAGNE":"35","THORIGNE FOUILLARD":"35","THOURIE":"35","TRESSE":"35","TREVERIEN":"35","VAL D IZE":"35","VIEUX VIEL":"35","VILLAMEE":"35","VITRE":"35","AMBRAULT":"36","BAZAIGES":"36","LA BERTHENOUX":"36","COINGS":"36","CONCREMIERS":"36","DOUADIC":"36","EGUZON CHANTOME":"36","ETRECHET":"36","GOURNAY":"36","LINIEZ":"36","LIZERAY":"36","LOURDOUEIX ST MICHEL":"36","MALICORNAY":"36","MEOBECQ":"36","NEUILLAY LES BOIS":"36","NEUVY PAILLOUX":"36","NURET LE FERRON":"36","PAUDY":"36","PERASSAY":"36","ROUVRES LES BOIS":"36","ST AIGNY":"36","ST AOUSTRILLE":"36","ST HILAIRE SUR BENAIZE":"36","SAZERAY":"36","SELLES SUR NAHON":"36","TRANZAULT":"36","VATAN":"36","VICQ EXEMPLET":"36","VIJON":"36","ARTANNES SUR INDRE":"37","ATHEE SUR CHER":"37","AZAY SUR INDRE":"37","BERTHENAY":"37","BOSSEE":"37","BREHEMONT":"37","CHAMBRAY LES TOURS":"37","CHAMPIGNY SUR VEUDE":"37","CHANCAY":"37","LA CHAPELLE SUR LOIRE":"37","CHAUMUSSAY":"37","CHEILLE":"37","CHEMILLE SUR INDROIS":"37","CINQ MARS LA PILE":"37","COURCOUE":"37","COUZIERS":"37","CROUZILLES":"37","DIERRE":"37","DOLUS LE SEC":"37","ESVRES":"37","FERRIERE LARCON":"37","FERRIERE SUR BEAULIEU":"37","GIZEUX":"37","LES HERMITES":"37","LOCHES":"37","LUBLE":"37","LUYNES":"37","MARCILLY SUR VIENNE":"37","MARIGNY MARMANDE":"37","MAZIERES DE TOURAINE":"37","MONTBAZON":"37","MOSNES":"37","NAZELLES NEGRON":"37","NEUIL":"37","STE OPPORTUNE LA MARE":"27","STE MARIE D ATTEZ":"27","ST PHILBERT SUR BOISSEY":"27","ST PIERRE D AUTILS":"27","ST PIERRE DE BAILLEUL":"27","ST PIERRE DE CERNIERES":"27","ST QUENTIN DES ISLES":"27","ST VINCENT DES BOIS":"27","ST VINCENT DU BOULAY":"27","SEBECOURT":"27","BUISSY":"62","BULLECOURT":"62","BUNEVILLE":"62","CAMBLIGNEUL":"62","CAMPAGNE LES GUINES":"62","CAMPIGNEULLES LES GRANDES":"62","CANTELEUX":"62","CARENCY":"62","CARVIN":"62","CAUCHY A LA TOUR":"62","CLAIRMARAIS":"62","COLLINE BEAUMONT":"62","LA COMTE":"62","CONCHY SUR CANCHE":"62","CONTEVILLE LES BOULOGNE":"62","COULOGNE":"62","COUPELLE VIEILLE":"62","COURCELLES LES LENS":"62","COYECQUES":"62","DELETTES":"62","DENNEBROEUCQ":"62","DOUCHY LES AYETTE":"62","DROUVIN LE MARAIS":"62","DUISANS":"62","ECOURT ST QUENTIN":"62","ELNES":"62","ENQUIN SUR BAILLONS":"62","ESQUERDES":"62","ESTREE WAMIN":"62","FARBUS":"62","FIENNES":"62","FILLIEVRES":"62","FLECHIN":"62","FLEURBAIX":"62","FLORINGHEM":"62","FONCQUEVILLERS":"62","FOSSEUX":"62","FOUFFLIN RICAMETZ":"62","FOUQUIERES LES LENS":"62","GIVENCHY EN GOHELLE":"62","GIVENCHY LES LA BASSEE":"62","GRAINCOURT LES HAVRINCOURT":"62","GUARBECQUE":"62","HALINGHEN":"62","HALLINES":"62","HAMELINCOURT":"62","HAVRINCOURT":"62","HENINEL":"62","HENNEVEUX":"62","HERLIN LE SEC":"62","HERMAVILLE":"62","HERMELINGHEN":"62","HESDIN":"62","HOUCHIN":"62","HUMIERES":"62","ISBERGUES":"62","LAIRES":"62","LEDINGHEM":"62","LESPESSES":"62","LINGHEM":"62","LOCON":"62","LOISON SUR CREQUOISE":"62","LONGUENESSE":"62","LUGY":"62","MARCK":"62","MARENLA":"62","MARESVILLE":"62","MAROEUIL":"62","METZ EN COUTURE":"62","MONCHEL SUR CANCHE":"62","MONCHIET":"62","MONCHY BRETON":"62","MONCHY CAYEUX":"62","MONCHY LE PREUX":"62","MONT BERNANCHON":"62","MONTS EN TERNOIS":"62","MORVAL":"62","MOURIEZ":"62","NESLES":"62","NEUFCHATEL HARDELOT":"62","NEULETTE":"62","NEUVE CHAPELLE":"62","NEUVILLE SOUS MONTREUIL":"62","NIELLES LES CALAIS":"62","NORT LEULINGHEM":"62","NOUVELLE EGLISE":"62","NOYELLES GODAULT":"62","OEUF EN TERNOIS":"62","OFFIN":"62","OUTREAU":"62","PAS EN ARTOIS":"62","PEUPLINGUES":"62","PIERREMONT":"62","PLOUVAIN":"62","LE PONCHEL":"62","ST DESERT":"71","ST HURUGE":"71","ST LEGER DU BOIS":"71","ST LEGER SOUS BEUVRAY":"71","ST MARCELIN DE CRAY":"71","ST MARD DE VAUX":"71","ST MARTIN D AUXY":"71","ST MARTIN DE LIXY":"71","ST MARTIN LA PATROUILLE":"71","ST MAURICE EN RIVIERE":"71","ST MAURICE LES CHATEAUNEUF":"71","ST MAURICE LES COUCHES":"71","ST ROMAIN SOUS GOURDON":"71","ST YTHAIRE":"71","SANVIGNES LES MINES":"71","SASSENAY":"71","SAVIANGES":"71","SENS SUR SEILLE":"71","SIGY LE CHATEL":"71","SIMANDRE":"71","SIVIGNON":"71","LA TAGNIERE":"71","LE TARTRE":"71","THIL SUR ARROUX":"71","TOUTENANT":"71","TRIVY":"71","UCHIZY":"71","VARENNES LES MACON":"71","VARENNE ST GERMAIN":"71","VENDENESSE LES CHAROLLES":"71","VINDECY":"71","LA VINEUSE":"71","VIREY LE GRAND":"71","VITRY EN CHAROLLAIS":"71","VOLESVRES":"71","AUBIGNE RACAN":"72","AVOISE":"72","BAZOUGES SUR LE LOIR":"72","BERFAY":"72","BESSE SUR BRAYE":"72","BLEVES":"72","BONNETABLE":"72","BOUER":"72","LA BRUERE SUR LOIR":"72","CERANS FOULLETOURTE":"72","CHANTENAY VILLEDIEU":"72","LA CHAPELLE GAUGAIN":"72","CLERMONT CREANS":"72","COGNERS":"72","CONNERRE":"72","COUDRECIEUX":"72","DANGEUL":"72","ECORPAIN":"72","ETIVAL LES LE MANS":"72","FATINES":"72","FILLE":"72","FONTENAY SUR VEGRE":"72","GUECELARD":"72","LA GUIERCHE":"72","JOUE EN CHARNIE":"72","LA FLECHE":"72","LHOMME":"72","LIGRON":"72","LOUAILLES":"72","MAMERS":"72","MAREIL EN CHAMPAGNE":"72","MAROLLES LES ST CALAIS":"72","MEZIERES SOUS LAVARDIN":"72","MONTREUIL LE CHETIF":"72","NEUVILLE SUR SARTHE":"72","OISSEAU LE PETIT":"72","PARIGNE LE POLIN":"72","RAHAY":"72","ROUESSE FONTAINE":"72","ROUESSE VASSE":"72","ROUEZ":"72","ROUILLON":"72","ROUPERROUX LE COQUET":"72","ST AUBIN DE LOCQUENAY":"72","ST AUBIN DES COUDRAIS":"72","ST BIEZ EN BELIN":"72","ST CALAIS":"72","ST CELERIN":"72","ST DENIS D ORQUES":"72","TOGNY AUX BOEUFS":"51","TRAMERY":"51","TREPAIL":"51","TRESLON":"51","TRIGNY":"51","TROISSY":"51","VALMY":"51","VANAULT LE CHATEL":"51","VANDEUIL":"51","VENTELAY":"51","VERNANCOURT":"51","VIENNE LE CHATEAU":"51","LA VILLENEUVE LES CHARLEVILLE":"51","VILLENEUVE ST VISTRE ET VILLEVOTTE":"51","VILLERS ALLERAND":"51","VILLERS EN ARGONNE":"51","VILLERS LE CHATEAU":"51","VILLERS MARMERY":"51","VILLEVENARD":"51","VITRY LA VILLE":"51","AINGOULAINCOURT":"52","ARC EN BARROIS":"52","AUTREVILLE SUR LA RENNE":"52","BEURVILLE":"52","BOURDONS SUR ROGNON":"52","BRAUX LE CHATEL":"52","CHAMPSEVRAINE":"52","CEFFONDS":"52","CELLES EN BASSIGNY":"52","CHALINDREY":"52","CHALVRAINES":"52","CHAMPIGNEULLES EN BASSIGNY":"52","CHANOY":"52","CHASSIGNY":"52","CHATEAUVILLAIN":"52","CHAUMONT LA VILLE":"52","CHEZEAUX":"52","CHOILLEY DARDENAY":"52","CIRFONTAINES EN AZOIS":"52","CLEFMONT":"52","COIFFY LE HAUT":"52","CONSIGNY":"52","DARMANNES":"52","DOMMARIEN":"52","DOULAINCOURT SAUCOURT":"52","ECLARON BRAUCOURT STE LIVIERE":"52","ENFONVELLE":"52","LE VAL D ESNOMS":"52","EUFFIGNEIX":"52","FARINCOURT":"52","FOULAIN":"52","GERMAINES":"52","GERMAY":"52","GIEY SUR AUJON":"52","HUILLIECOURT":"52","IS EN BASSIGNY":"52","JONCHERY":"52","LAFERTE SUR AMANCE":"52","LAMOTHE EN BLAISY":"52","BAYARD SUR MARNE":"52","LANGRES":"52","LEVECOURT":"52","LEZEVILLE":"52","MAATZ":"52","MARDOR":"52","MENNOUVEAUX":"52","MONTCHARVOT":"52","MONTREUIL SUR BLAISE":"52","MORIONVILLIERS":"52","NOIDANT CHATENOY":"52","FRESSE":"70","FROIDECONCHE":"70","FROIDETERRE":"70","GEORFANS":"70","GIREFONTAINE":"70","GRAMMONT":"70","GRANDVELLE ET LE PERRENOT":"70","GRANGES LA VILLE":"70","LA LANTERNE ET LES ARMONTS":"70","LUXEUIL LES BAINS":"70","LYOFFANS":"70","MAGNIVRAY":"70","LE MAGNORAY":"70","MALBOUHANS":"70","MANDREVILLARS":"70","MELIN":"70","MELINCOURT":"70","MOLLANS":"70","MONTAGNEY":"70","MONTUREUX ET PRANTIGNY":"70","NAVENNE":"70","LA NEUVELLE LES LURE":"70","ONAY":"70","OYRIERES":"70","PASSAVANT LA ROCHERE":"70","LA PISSEURE":"70","PLANCHER LES MINES":"70","POLAINCOURT ET CLAIREFONTAINE":"70","POMOY":"70","LA ROMAINE":"70","RADDON ET CHAPENDU":"70","RAINCOURT":"70","ST FERJEUX":"70","SAULNOT":"70","TARTECOURT":"70","TERNUAY MELAY ET ST HILAIRE":"70","TINCEY ET PONTREBEAU":"70","VAIVRE ET MONTOILLE":"70","LE VAL ST ELOI":"70","VAUCHOUX":"70","VELESMES ECHEVANNE":"70","VELLECLAIRE":"70","VELLEFREY ET VELLEFRANGE":"70","VENISEY":"70","VEREUX":"70","VERLANS":"70","VILLERS PATER":"70","VILLERS SUR SAULNOT":"70","VISONCOURT":"70","VOUHENANS":"70","VREGILLE":"70","L ABERGEMENT STE COLOMBE":"71","AUTHUMES":"71","BALLORE":"71","BERZE LE CHATEL":"71","BISSY LA MACONNAISE":"71","BISSY SOUS UXELLES":"71","BOSJEAN":"71","BOURGVILAIN":"71","BRANDON":"71","BRANGES":"71","BRUAILLES":"71","BUFFIERES":"71","BURGY":"71","CHAMILLY":"71","CHAMPLECY":"71","CHAPAIZE":"71","LA CHAPELLE SOUS DUN":"71","CHATENOY EN BRESSE":"71","CHAUFFAILLES":"71","CHERIZET":"71","CIRY LE NOBLE":"71","COUCHES":"71","CREOT":"71","CURBIGNY":"71","CURGY":"71","CURTIL SOUS BUFFIERES":"71","DICONNE":"71","DIGOIN":"71","DOMMARTIN LES CUISEAUX":"71","THEILLEMENT":"27","TOURVILLE LA CAMPAGNE":"27","TRIQUEVILLE":"27","VALAILLES":"27","VALLETOT":"27","VANNECROCQ":"27","VASCOEUIL":"27","VAUX SUR EURE":"27","LE VIEIL EVREUX":"27","SYLVAINS LES MOULINS":"27","VILLEZ SUR LE NEUBOURG":"27","VAL DE REUIL":"27","ALLAINES MERVILLIERS":"28","ANET":"28","AUNAY SOUS AUNEAU":"28","AUTHON DU PERCHE":"28","BAILLEAU L EVEQUE":"28","LA BAZOCHE GOUET":"28","BAZOCHES EN DUNOIS":"28","BERCHERES ST GERMAIN":"28","BLANDAINVILLE":"28","BOISSY LES PERCHE":"28","LA BOURDINIERE ST LOUP":"28","BONCE":"28","LE BOULLAY MIVOYE":"28","BRUNELLES":"28","BU":"28","CHAMPSERU":"28","LA CHAPELLE D AUNAINVILLE":"28","CHATEAUDUN":"28","CHATEAUNEUF EN THYMERAIS":"28","CHATILLON EN DUNOIS":"28","CHAUDON":"28","LA CHAUSSEE D IVRY":"28","LES CORVEES LES YYS":"28","LE COUDRAY":"28","COUDRAY AU PERCHE":"28","COUDRECEAU":"28","DAMPIERRE SUR AVRE":"28","DANGEAU":"28","DANGERS":"28","DOUY":"28","ECLUZELLES":"28","ERMENONVILLE LA GRANDE":"28","LA FERTE VILLENEUIL":"28","FONTENAY SUR CONIE":"28","FRAZE":"28","FRESNAY LE GILMERT":"28","FRESNAY L EVEQUE":"28","GASVILLE OISEME":"28","GILLES":"28","LE GUE DE LONGROI":"28","HAVELU":"28","HOUVILLE LA BRANCHE":"28","INTREVILLE":"28","LANNERAY":"28","LAONS":"28","LEVES":"28","LOUVILLIERS LES PERCHE":"28","MARBOUE":"28","MARCHEZAIS":"28","MEAUCE":"28","MONTHARVILLE":"28","NONVILLIERS GRANDHOUX":"28","OINVILLE SOUS AUNEAU":"28","ORGERES EN BEAUCE":"28","PRUDEMANCHE":"28","REVERCOURT":"28","ST ANGE ET TORCAY":"28","ST BOMER":"28","ST CLOUD EN DUNOIS":"28","ST DENIS D AUTHOU":"28","ST LUBIN DES JONCHERETS":"28","SENONCHES":"28","SOULAIRES":"28","LE THIEULIN":"28","THIVARS":"28","TILLAY LE PENEUX":"28","TRIZAY LES BONNEVAL":"28","VERT EN DROUAIS":"28","LES VILLAGES VOVEENS":"28","YMONVILLE":"28","BERRIEN":"29","BEUZEC CAP SIZUN":"29","BOLAZEC":"29","BOTMEUR":"29","BOTSORHEL":"29","BRELES":"29","PRONVILLE":"62","QUERNES":"62","QUIERY LA MOTTE":"62","QUIESTEDE":"62","RANG DU FLIERS":"62","REGNAUVILLE":"62","REMILLY WIRQUIN":"62","RIENCOURT LES CAGNICOURT":"62","RIMBOVAL":"62","ROBECQ":"62","ROELLECOURT":"62","ROUGEFAY":"62","RUMAUCOURT":"62","ST DENOEUX":"62","ST INGLEVERT":"62","ST LAURENT BLANGY":"62","ST MARTIN LEZ TATINGHEM":"62","ST MARTIN BOULOGNE":"62","ST POL SUR TERNOISE":"62","SAMER":"62","SANGATTE":"62","SANGHEN":"62","SAULTY":"62","SERQUES":"62","SOUASTRE":"62","TANGRY":"62","TENEUR":"62","THELUS":"62","THEROUANNE":"62","LA THIEULOYE":"62","TILLOY LES MOFFLAINES":"62","TILLY CAPELLE":"62","TINGRY":"62","TROISVAUX":"62","VALHUON":"62","VERTON":"62","VILLERS BRULIN":"62","VINCLY":"62","WANCOURT":"62","WANQUETIN":"62","BEAUVOIR WAVANS":"62","WAVRANS SUR L AA":"62","WAVRANS SUR TERNOISE":"62","WILLEMAN":"62","WILLENCOURT":"62","WITTES":"62","WIZERNES":"62","ZOUAFQUES":"62","AIX LA FAYETTE":"63","LES ANCIZES COMPS":"63","ANTOINGT":"63","ARCONSAT":"63","ARDES":"63","ARLANC":"63","ARS LES FAVETS":"63","ARTONNE":"63","AUBIAT":"63","AUBIERE":"63","AUTHEZAT":"63","BANSAT":"63","BESSE ET ST ANASTAISE":"63","BILLOM":"63","BOUZEL":"63","BRASSAC LES MINES":"63","BRENAT":"63","CEYRAT":"63","CEYSSAT":"63","CHADELEUF":"63","LA CHAPELLE AGNON":"63","LA CHAPELLE SUR USSON":"63","CHARBONNIERES LES VARENNES":"63","CHASTREIX":"63","CHATEAU SUR CHER":"63","CHAUMONT LE BOURG":"63","COMBRAILLES":"63","CONDAT EN COMBRAILLE":"63","CORENT":"63","COUDES":"63","COURGOUL":"63","CUNLHAT":"63","DAVAYAT":"63","DOMAIZE":"63","DURMIGNAT":"63","DURTOL":"63","EGLISOLLES":"63","VILLEDIEU LES POELES ROUFFIGNY":"50","YVETOT BOCAGE":"50","AIGNY":"51","ST JEAN DES ECHELLES":"72","ST LEONARD DES BOIS":"72","ST MARS LA BRIERE":"72","ST PAVACE":"72","ST PIERRE DU LOROUER":"72","ST REMY DES MONTS":"72","ST VICTEUR":"72","SAOSNES":"72","SARGE LES LE MANS":"72","SCEAUX SUR HUISNE":"72","SOUILLE":"72","SOULIGNE SOUS BALLON":"72","SPAY":"72","SURFONDS":"72","VAULRY":"87","VICQ SUR BREUILH":"87","AOUZE":"88","ATTIGNEVILLE":"88","AULNOIS":"88","BAN DE SAPT":"88","BATTEXEY":"88","BAZEGNEY":"88","BELMONT SUR BUTTANT":"88","BETTEGNEY ST BRICE":"88","LE BEULAY":"88","BIFFONTAINE":"88","BLEURVILLE":"88","BOIS DE CHAMP":"88","BRANTIGNY":"88","BRU":"88","BRUYERES":"88","BULGNEVILLE":"88","BUSSANG":"88","CHAMAGNE":"88","CIRCOURT":"88","CRAINVILLIERS":"88","LA CROIX AUX MINES":"88","DARNEY":"88","DEYCIMONT":"88","DINOZE":"88","DOLAINCOURT":"88","DOMBROT SUR VAIR":"88","DOMFAING":"88","DOMMARTIN LES VALLOIS":"88","DOMVALLIER":"88","DONCIERES":"88","ESSEGNEY":"88","FLOREMONT":"88","FREMIFONTAINE":"88","FREVILLE":"88","GERBAMONT":"88","GIGNEVILLE":"88","GOLBEY":"88","GRAND":"88","GRANDVILLERS":"88","GRANGES AUMONTZEY":"88","HAGNEVILLE ET RONCOURT":"88","HARSAULT":"88","HAUTMOUGEY":"88","LA HOUSSIERE":"88","HYMONT":"88","JEANMENIL":"88","JEUXEY":"88","LANDAVILLE":"88","LAVELINE DEVANT BRUYERES":"88","LEMMECOURT":"88","LIEZEY":"88","LONGCHAMP SOUS CHATENOIS":"88","LUBINE":"88","MARONCOURT":"88","MARTINVELLE":"88","MEMENIL":"88","MENIL DE SENONES":"88","MIRECOURT":"88","MONCEL SUR VAIR":"88","MONT LES NEUFCHATEAU":"88","MOYEMONT":"88","LA NEUVEVILLE DEVANT LEPANGES":"88","NOMEXY":"88","NONZEVILLE":"88","ORTONCOURT":"88","LA PETITE RAON":"88","PLOMBIERES LES BAINS":"88","POMPIERRE":"88","POUSSAY":"88","PROVENCHERES ET COLROY":"88","NULLY":"52","ORMOY LES SEXFONTAINES":"52","OUDINCOURT":"52","PALAISEUL":"52","PLANRUPT":"52","LE CHATELET SUR MEUSE":"52","PRASLAY":"52","RIVES DERVOISES":"52","RACHECOURT SUZEMONT":"52","RANCONNIERES":"52","REYNEL":"52","RIZAUCOURT BUCHEY":"52","ROCHES SUR MARNE":"52","ROMAIN SUR MEUSE":"52","ROUELLES":"52","ROUVROY SUR MARNE":"52","ST BROINGT LE BOIS":"52","ST BROINGT LES FOSSES":"52","ST CIERGUES":"52","ST MARTIN LES LANGRES":"52","SAULLES":"52","SEXFONTAINES":"52","SOULAUCOURT SUR MOUZON":"52","THILLEUX":"52","THONNANCE LES MOULINS":"52","VALCOURT":"52","VALLERET":"52","VIGNORY":"52","VILLIERS EN LIEU":"52","VITRY LES NOGENT":"52","VRONCOURT LA COTE":"52","WASSY":"52","AMPOIGNE":"53","ASSE LE BERENGER":"53","ASTILLE":"53","AVERTON":"53","BALLOTS":"53","LA BAZOUGE DES ALLEUX":"53","LE BIGNON DU MAINE":"53","LA BIGOTTIERE":"53","BOUCHAMPS LES CRAON":"53","BOUERE":"53","BOULAY LES IFS":"53","CHAMPEON":"53","CHAMPFREMONT":"53","LA CHAPELLE CRAONNAISE":"53","CHARCHIGNE":"53","CHATELAIN":"53","CONGRIER":"53","COUPTRAIN":"53","CRENNES SUR FRAUBEE":"53","ENTRAMMES":"53","GORRON":"53","HAMBERS":"53","HERCE":"53","LE HOUSSEAU BRETIGNOLLES":"53","LESBOIS":"53","LEVARE":"53","LIGNIERES ORGERES":"53","LONGUEFUYE":"53","LOUVERNE":"53","MARTIGNE SUR MAYENNE":"53","MONTOURTIER":"53","NEUILLY LE VENDIN":"53","PARNE SUR ROC":"53","PORT BRILLET":"53","LA ROE":"53","ST AIGNAN DE COUPTRAIN":"53","ST BERTHEVIN LA TANNIERE":"53","ST CYR EN PAIL":"53","ST CYR LE GRAVELAIS":"53","ST GEORGES BUTTAVENT":"53","ST JULIEN DU TERROUX":"53","ST MARS SUR COLMONT":"53","ST MARS SUR LA FUTAIE":"53","ST PIERRE SUR ERVE":"53","TORCE VIVIERS EN CHARNIE":"53","VAUTORTE":"53","VIEUVY":"53","VILLEPAIL":"53","VILLIERS CHARLEMAGNE":"53","ABBEVILLE LES CONFLANS":"54","DOMPIERRE SOUS SANVIGNES":"71","FARGES LES CHALON":"71","FARGES LES MACON":"71","FUISSE":"71","GERMOLLES SUR GROSNE":"71","GILLY SUR LOIRE":"71","GUERFAND":"71","HAUTEFOND":"71","IGUERANDE":"71","ISSY L EVEQUE":"71","JAMBLES":"71","JULLY LES BUXY":"71","LEYNES":"71","LIGNY EN BRIONNAIS":"71","LUGNY LES CHAROLLES":"71","MANCEY":"71","MARCIGNY":"71","MARCILLY LES BUXY":"71","MARLY SOUS ISSY":"71","MARLY SUR ARROUX":"71","MARY":"71","MASSILLY":"71","LE MIROIR":"71","MONTAGNY LES BUXY":"71","MONTAGNY PRES LOUHANS":"71","MONTAGNY SUR GROSNE":"71","MONTBELLET":"71","MONTCEAUX RAGNY":"71","MONTCONY":"71","MONTMORT":"71","MONTRET":"71","LA MOTTE ST JEAN":"71","OUDRY":"71","OYE":"71","PARAY LE MONIAL":"71","PERRECY LES FORGES":"71","LA PETITE VERRIERE":"71","PIERRE DE BRESSE":"71","PRETY":"71","LA RACINEUSE":"71","ROYER":"71","ST ALBAIN":"71","ST AMOUR BELLEVUE":"71","ST BERAIN SOUS SANVIGNES":"71","ST BONNET EN BRESSE":"71","ST DENIS DE VAUX":"71","ST DIDIER SUR ARROUX":"71","ST EDMOND":"71","ST FORGEOT":"71","ST GERMAIN DU BOIS":"71","ST GERMAIN EN BRIONNAIS":"71","ST GERMAIN LES BUXY":"71","ST GERVAIS EN VALLIERE":"71","ST LAURENT D ANDENAY":"71","BAILLEUL SIR BERTHOULT":"62","BAINGHEN":"62","BAYENGHEM LES EPERLECQUES":"62","BEAUMETZ LES AIRE":"62","BEAURAINVILLE":"62","BECOURT":"62","BELLEBRUNE":"62","BERGUENEUSE":"62","BERTINCOURT":"62","BETHUNE":"62","BEUGIN":"62","BEUGNY":"62","BEUVREQUEN":"62","BIACHE ST VAAST":"62","BIHUCOURT":"62","BIMONT":"62","BOFFLES":"62","BOISLEUX AU MONT":"62","BOURLON":"62","BOURSIN":"62","BOURTHES":"62","BOUVELINGHEM":"62","BOYELLES":"62","BREBIERES":"62","BREXENT ENOCQ":"62","BULLY LES MINES":"62","CAFFIERS":"62","CALAIS":"62","LA CALOTTERIE":"62","CAMBLAIN L ABBE":"62","CAMPAGNE LES HESDIN":"62","CARLY":"62","CAVRON ST MARTIN":"62","CHELERS":"62","CORBEHEM":"62","COULLEMONT":"62","BRENNILIS":"29","BREST":"29","CAST":"29","COAT MEAL":"29","COMMANA":"29","CROZON":"29","DIRINON":"29","DOUARNENEZ":"29","LE FAOU":"29","GOUESNOU":"29","GOUEZEC":"29","GOULIEN":"29","GOURLIZON":"29","GUILERS":"29","GUIMILIAU":"29","ILE DE BATZ":"29","ILE TUDY":"29","LANDERNEAU":"29","LANGOLEN":"29","LANNEDERN":"29","LESNEVEN":"29","LOCQUENOLE":"29","LOCUNOLE":"29","MORLAIX":"29","NEVEZ":"29","PENMARCH":"29","PLABENNEC":"29","PLEYBEN":"29","PLOEVEN":"29","PLOMELIN":"29","PLOMODIERN":"29","PLONEVEZ PORZAY":"29","PLOUEGAT MOYSAN":"29","PLOUGAR":"29","PLOUGASNOU":"29","PLOUGONVEN":"29","PLOZEVET":"29","PONT AVEN":"29","PONT CROIX":"29","PORSPODER":"29","POULDERGAT":"29","PRIMELIN":"29","QUIMPER":"29","ROSCOFF":"29","ROSNOEN":"29","ST DIVY":"29","ST FREGANT":"29","ST GOAZEC":"29","SCAER":"29","SCRIGNAC":"29","TELGRUC SUR MER":"29","TREGUNC":"29","LE TREHOU":"29","LE TREVOUX":"29","TREZILIDE":"29","AIGALIERS":"30","AIGUEZE":"30","ARRIGAS":"30","ASPERES":"30","BESSEGES":"30","BLAUZAC":"30","BROUZET LES ALES":"30","LA BRUGUIERE":"30","CANAULES ET ARGENTIERES":"30","CAVILLARGUES":"30","CORNILLON":"30","DURFORT ET ST MARTIN DE SOSSENAC":"30","ESTEZARGUES":"30","EUZET":"30","FONS SUR LUSSAN":"30","GARONS":"30","GENERARGUES":"30","GENOLHAC":"30","LE GRAU DU ROI":"30","ISSIRAC":"30","MEYRANNES":"30","MILHAUD":"30","MOULEZAN":"30","NIMES":"30","NOTRE DAME DE LA ROUVIERE":"30","PARIGNARGUES":"30","PONT ST ESPRIT":"30","POULX":"30","POUZILHAC":"30","PUECHREDON":"30","REDESSAN":"30","ROCHEFORT DU GARD":"30","LA ROUVIERE":"30","SABRAN":"30","ST ANDRE DE MAJENCOULES":"30","ST CHAPTES":"30","STE CROIX DE CADERLE":"30","AOUGNY":"51","ARCIS LE PONSART":"51","AUBILLY":"51","AUVE":"51","LE BAIZIL":"51","BASSUET":"51","BILLY LE GRAND":"51","BINARVILLE":"51","BLESME":"51","BOULT SUR SUIPPE":"51","BOURSAULT":"51","BOUZY":"51","BREBAN":"51","BRUGNY VAUDANCOURT":"51","CHALONS SUR VESLE":"51","CHIGNY LES ROSES":"51","CHOUILLY":"51","CONTAULT":"51","CORBEIL":"51","COURCELLES SAPICOURT":"51","COURJEONNET":"51","DOMMARTIN SOUS HANS":"51","ECRIENNES":"51","ELISE DAUCOURT":"51","LES ESSARTS LE VICOMTE":"51","FRIGNICOURT":"51","GAYE":"51","HANS":"51","HAUSSIGNEMONT":"51","HEILTZ L EVEQUE":"51","HUIRON":"51","IGNY COMBLIZY":"51","ISLE SUR MARNE":"51","LES ISTRES ET BURY":"51","JUSSECOURT MINECOURT":"51","MARFAUX":"51","MATIGNICOURT GONCOURT":"51","LE MEIX ST EPOING":"51","MERFY":"51","MOIREMONT":"51","MOURMELON LE PETIT":"51","NANTEUIL LA FORET":"51","NESLE LA REPOSTE":"51","NOGENT L ABBESSE":"51","OGER":"51","OYES":"51","PASSAVANT EN ARGONNE":"51","POCANCY":"51","POIX":"51","PONTHION":"51","PROSNES":"51","PRUNAY":"51","RILLY LA MONTAGNE":"51","ST GIBRIEN":"51","ST JEAN SUR MOIVRE":"51","ST MEMMIE":"51","ST OUEN DOMPROT":"51","ST UTIN":"51","SAPIGNICOURT":"51","SAUDOY":"51","SOIZY AUX BOIS":"51","SOMME VESLE":"51","SOMME YEVRE":"51","SOUAIN PERTHES LES HURLUS":"51","SUIZY LE FRANC":"51","THIBIE":"51","VAL DE VESLE":"51","TINQUEUX":"51","TROIS PUITS":"51","UNCHAIR":"51","VAVRAY LE PETIT":"51","VERT TOULON":"51","LE VEZIER":"51","LE VIEIL DAMPIERRE":"51","VILLE EN TARDENOIS":"51","VILLENEUVE LA LIONNE":"51","VILLENEUVE RENNEVILLE CHEVIGNY":"51","VILLERS AUX NOEUDS":"51","VILLE SUR TOURBE":"51","VINDEY":"51","RAON L ETAPE":"88","REMIREMONT":"88","RENAUVOID":"88","ROCOURT":"88","ROZEROTTE":"88","RUGNEY":"88","RUPT SUR MOSELLE":"88","ST BASLEMONT":"88","ST JEAN D ORMONT":"88","ST MAURICE SUR MOSELLE":"88","SARTES":"88","SAULCY SUR MEURTHE":"88","SAULXURES SUR MOSELOTTE":"88","SEROCOURT":"88","SIONNE":"88","LE SYNDICAT":"88","LE THILLOT":"88","LE THOLY":"88","LES THONS":"88","VAGNEY":"88","VALLEROY AUX SAULES":"88","VENTRON":"88","LE VERMONT":"88","VICHEREY":"88","ACCOLAY":"89","ANDRYES":"89","AUXERRE":"89","AVALLON":"89","BELLECHAUME":"89","BERU":"89","BESSY SUR CURE":"89","CENSY":"89","CHABLIS":"89","CHAMOUX":"89","CHAMVRES":"89","CHARNY OREE DE PUISAYE":"89","CHASSIGNELLES":"89","CHEMILLY SUR SEREIN":"89","CHICHEE":"89","CORNANT":"89","COULANGES LA VINEUSE":"89","CRAIN":"89","CRY":"89","DYE":"89","ETAULE":"89","FLACY":"89","FLEYS":"89","FOISSY LES VEZELAY":"89","FOURONNES":"89","GURGY":"89","GY L EVEQUE":"89","IRANCY":"89","JOUANCY":"89","JUNAY":"89","LEVIS":"89","LIGNORELLES":"89","LINDRY":"89","LOOZE":"89","MALAY LE PETIT":"89","MASSANGIS":"89","MEZILLES":"89","MICHERY":"89","MIGE":"89","MOLOSMES":"89","MONETEAU":"89","MOULINS EN TONNERROIS":"89","MOULINS SUR OUANNE":"89","NITRY":"89","PASILLY":"89","PERCEY":"89","PLESSIS ST JEAN":"89","POILLY SUR SEREIN":"89","AFFRACOURT":"54","AINGERAY":"54","ALLAMONT":"54","ANCERVILLER":"54","ARNAVILLE":"54","ATTON":"54","AUTREPIERRE":"54","BACCARAT":"54","BAINVILLE SUR MADON":"54","BARISEY AU PLAIN":"54","BATILLY":"54","BECHAMPS":"54","BEZANGE LA GRANDE":"54","BONVILLER":"54","MONT BONVILLERS":"54","BOUXIERES AUX DAMES":"54","BUISSONCOURT":"54","CERVILLE":"54","CHAMPEY SUR MOSELLE":"54","CHAMPIGNEULLES":"54","CHARENCY VEZIN":"54","CHAREY":"54","CHAUDENEY SUR MOSELLE":"54","CHENEVIERES":"54","CHENICOURT":"54","CLAYEURES":"54","COINCOURT":"54","COLMEY":"54","CONFLANS EN JARNISY":"54","CONS LA GRANDVILLE":"54","COSNES ET ROMAIN":"54","CRANTENOY":"54","CREVECHAMPS":"54","CREZILLES":"54","DENEUVRE":"54","DONCOURT LES LONGUYON":"54","EINVILLE AU JARD":"54","EPIEZ SUR CHIERS":"54","ERROUVILLE":"54","EUVEZIN":"54","FECOCOURT":"54","FENNEVILLER":"54","FLAINVAL":"54","FRAIMBOIS":"54","FREMENIL":"54","FRESNOIS LA MONTAGNE":"54","GRISCOURT":"54","HAIGNEVILLE":"54","HANNONVILLE SUZEMONT":"54","HAUCOURT MOULAINE":"54","HAUDONVILLE":"54","JARVILLE LA MALGRANGE":"54","JAULNY":"54","JEANDELAINCOURT":"54","JOUAVILLE":"54","LANDECOURT":"54","LANDREMONT":"54","LANEUVEVILLE DERRIERE FOUG":"54","LANEUVEVILLE DEVANT BAYON":"54","LAY ST CHRISTOPHE":"54","LEMAINVILLE":"54","LUBEY":"54","MALZEVILLE":"54","MANONCOURT EN WOEVRE":"54","MARBACHE":"54","MAZERULLES":"54","MERCY LE HAUT":"54","MOINEVILLE":"54","MONTREUX":"54","MOUTROT":"54","NONHIGNY":"54","NOVIANT AUX PRES":"54","OGNEVILLE":"54","OTHE":"54","PAGNY SUR MOSELLE":"54","PARROY":"54","PARUX":"54","PIENNES":"54","PRENY":"54","RECHICOURT LA PETITE":"54","REHON":"54","REILLON":"54","REMENOVILLE":"54","ROYAUMEIX":"54","ST BOINGT":"54","ST JEAN LES LONGUYON":"54","COULOMBY":"62","CREQUY":"62","DENIER":"62","DIEVAL":"62","DOURIEZ":"62","DOUVRIN":"62","ECHINGHEN":"62","ENGUINEGATTE":"62","EQUIRRE":"62","ERIN":"62","ESTREE":"62","ESTREELLES":"62","FAMPOUX":"62","FONTAINE LES BOULANS":"62","FONTAINE LES CROISILLES":"62","FONTAINE L ETALON":"62","FOUQUEREUIL":"62","FRESNES LES MONTAUBAN":"62","FRESSIN":"62","FREVILLERS":"62","GAUCHIN LEGAL":"62","GOMIECOURT":"62","GOSNAY":"62","GOUY SERVINS":"62","GREVILLERS":"62","GRINCOURT LES PAS":"62","GUEMAPPE":"62","GUIGNY":"62","GUINECOURT":"62","HAILLICOURT":"62","HAPLINCOURT":"62","HEBUTERNE":"62","HERMIES":"62","HERNICOURT":"62","HERSIN COUPIGNY":"62","HESDIGNEUL LES BOULOGNE":"62","HESMOND":"62","HOCQUINGHEN":"62","HOUDAIN":"62","HUCLIER":"62","HUCQUELIERS":"62","HUMBERT":"62","INCHY EN ARTOIS":"62","IZEL LES HAMEAU":"62","LABROYE":"62","LAVENTIE":"62","LENS":"62","LEULINGHEM":"62","LICQUES":"62","LIENCOURT":"62","LIERES":"62","LIGNY SUR CANCHE":"62","LIGNY THILLOY":"62","LOISON SOUS LENS":"62","LUMBRES":"62","MAISNIL":"62","MARCONNELLE":"62","MARQUION":"62","MARQUISE":"62","MAZINGHEM":"62","MENTQUE NORTBECOURT":"62","MERLIMONT":"62","MINGOVAL":"62","MONCHEAUX LES FREVENT":"62","MONTIGNY EN GOHELLE":"62","MOULLE":"62","NEUVILLE AU CORNET":"62","NEUVILLE BOURJONVAL":"62","NOYELLES LES HUMIERES":"62","NOYELLES SOUS LENS":"62","NOYELLE VION":"62","OFFEKERQUE":"62","OISY LE VERGER":"62","OPPY":"62","OSTREVILLE":"62","OYE PLAGE":"62","LE PARCQ":"62","PERNES LES BOULOGNE":"62","BOUIN PLUMOISON":"62","POMMIER":"62","PREURES":"62","QUELMES":"62","QUESTRECQUES":"62","QUILEN":"62","QUOEUX HAUT MAINIL":"62","RANSART":"62","REBREUVE RANCHICOURT":"62","ROMBLY":"62","ROQUETOIRE":"62","ROYON":"62","SAILLY AU BOIS":"62","SAINS EN GOHELLE":"62","SAINS LES FRESSIN":"62","SAINS LES PERNES":"62","ST DEZERY":"30","ST HILAIRE D OZILHAN":"30","ST LAURENT DE CARNOLS":"30","ST LAURENT LE MINIER":"30","ST MARCEL DE CAREIRET":"30","ST NAZAIRE DES GARDIES":"30","ST SIFFRET":"30","SALAZAC":"30","SANILHAC SAGRIES":"30","SAZE":"30","THOIRAS":"30","UZES":"30","VALLABREGUES":"30","VALLERARGUES":"30","VALLIGUIERES":"30","VAUVERT":"30","VERFEUIL":"30","RODILHAN":"30","ARGUENOS":"31","ARTIGUE":"31","ASPET":"31","AURIBAIL":"31","AVIGNONET LAURAGAIS":"31","BALESTA":"31","BAZUS":"31","BELESTA EN LAURAGAIS":"31","BENQUE DESSOUS ET DESSUS":"31","BEZINS GARRAUX":"31","BLAJAN":"31","BOISSEDE":"31","BOULOGNE SUR GESSE":"31","BOUTX":"31","CADOURS":"31","CARAGOUDES":"31","CASSAGNABERE TOURNAS":"31","CASSAGNE":"31","CASTANET TOLOSAN":"31","CASTELNAU PICAMPEAU":"31","CATHERVIELLE":"31","CAZAUNOUS":"31","CAZEAUX DE LARBOUST":"31","CAZENEUVE MONTAUT":"31","CIERP GAUD":"31","CIRES":"31","COUEILLES":"31","COURET":"31","CUGURON":"31","LE FAGET":"31","FOLCARDE":"31","FONSORBES":"31","FOURQUEVAUX":"31","FRONTIGNAN DE COMMINGES":"31","GANTIES":"31","GEMIL":"31","GOUAUX DE LARBOUST":"31","GRATENS":"31","JURVIELLE":"31","LABASTIDETTE":"31","LABEGE":"31","LACROIX FALGARDE":"31","LAMASQUERE":"31","LATRAPE":"31","LAVELANET DE COMMINGES":"31","LAVERNOSE LACASSE":"31","LECUSSAN":"31","LESCUNS":"31","LONGAGES":"31","LOURDE":"31","LUNAX":"31","LUSCAN":"31","LA MAGDELAINE SUR TARN":"31","MAILHOLAS":"31","MARTRES DE RIVIERE":"31","MAURAN":"31","MAUREVILLE":"31","MAUVAISIN":"31","MAYREGNE":"31","MAZERES SUR SALAT":"31","MILHAS":"31","MIRAMONT DE COMMINGES":"31","MONDILHAN":"31","MONDOUZIL":"31","MONES":"31","MONT DE GALIE":"31","MONTREJEAU":"31","MONTSAUNES":"31","ORE":"31","PALAMINY":"31","PECHBONNIEU":"31","VOUILLERS":"51","AMBONVILLE":"52","ANDELOT BLANCHEVILLE":"52","ANNEVILLE LA PRAIRIE":"52","ARBOT":"52","AUTIGNY LE PETIT":"52","BAISSEY":"52","BAY SUR AUBE":"52","BETTANCOURT LA FERREE":"52","BIESLES":"52","BRACHAY":"52","BRAINVILLE SUR MEUSE":"52","BRENNES":"52","BREUVANNES EN BASSIGNY":"52","CHALANCEY":"52","VALS DES TILLES":"52","CHAMPIGNY SOUS VARENNES":"52","CHANGEY":"52","COUR L EVEQUE":"52","CULMONT":"52","CURMONT":"52","DAILLANCOURT":"52","DAMMARTIN SUR MEUSE":"52","DANCEVOIR":"52","DOULEVANT LE CHATEAU":"52","ECOT LA COMBE":"52","FRONVILLE":"52","GERMAINVILLIERS":"52","GILLAUME":"52","GRAFFIGNY CHEMIN":"52","HEUILLEY LE GRAND":"52","LAFAUCHE":"52","LANEUVILLE AU PONT":"52","LANTY SUR AUBE":"52","LAVERNOY":"52","LEFFONDS":"52","MANDRES LA COTE":"52","MARANVILLE":"52","MARBEVILLE":"52","MARCILLY EN BASSIGNY":"52","MARNAY SUR MARNE":"52","MATHONS":"52","MONTHERIES":"52","MONTREUIL SUR THONNANCE":"52","NEUILLY L EVEQUE":"52","NIJON":"52","NINVILLE":"52","OCCEY":"52","ORBIGNY AU VAL":"52","OSNE LE VAL":"52","LE PAILLY":"52","PLESNOY":"52","POISEUL":"52","POULANGY":"52","RACHECOURT SUR MARNE":"52","RANGECOURT":"52","ST DIZIER":"52","ST URBAIN MACONCOURT":"52","SILVAROUVRES":"52","SOMMEVOIRE":"52","SONCOURT SUR MARNE":"52","THONNANCE LES JOINVILLE":"52","VARENNES SUR AMANCE":"52","VAUX SUR BLAISE":"52","VERBIESLES":"52","VERSEILLES LE HAUT":"52","VESAIGNES SUR MARNE":"52","VIEVILLE":"52","VILLIERS LES APREY":"52","LA BAZOGE MONTPINCON":"53","LA POSTOLLE":"89","POURRAIN":"89","LE VAL D OCRE":"89","ST AUBIN SUR YONNE":"89","ST GEORGES SUR BAULCHE":"89","ST MARTIN D ORDON":"89","SEPEAUX ST ROMAIN":"89","SERRIGNY":"89","THAROISEAU":"89","LES VALLEES DE LA VANNE":"89","THIZY":"89","THOREY":"89","VAULT DE LUGNY":"89","VAUMORT":"89","VERLIN":"89","VILLENAVOTTE":"89","VILLENEUVE ST SALVES":"89","PERCENEIGE":"89","VILLIERS LES HAUTS":"89","VILLIERS ST BENOIT":"89","ANDELNANS":"90","AUXELLES BAS":"90","BEAUCOURT":"90","BELFORT":"90","BOURG SOUS CHATELET":"90","BOUROGNE":"90","CHAVANNES LES GRANDS":"90","DORANS":"90","EGUENIGUE":"90","ETUEFFONT":"90","FOUSSEMAGNE":"90","GRANDVILLARS":"90","GROSMAGNY":"90","LACOLLONGE":"90","LEPUIX NEUF":"90","MONTBOUTON":"90","PEROUSE":"90","RECHESY":"90","AUTRECHENE":"90","RECOUVRANCE":"90","THIANCOURT":"90","TREVENANS":"90","VAUTHIERMONT":"90","VETRIGNE":"90","VILLARS LE SEC":"90","ATHIS MONS":"91","AUTHON LA PLAINE":"91","AUVERS ST GEORGES":"91","BALLAINVILLIERS":"91","BOISSY LE SEC":"91","BOURAY SUR JUINE":"91","BREUX JOUY":"91","BRIIS SOUS FORGES":"91","BRUNOY":"91","BUNO BONNEVAUX":"91","CERNY":"91","CHAMPCUEIL":"91","CHAMPLAN":"91","COURANCES":"91","DOURDAN":"91","EPINAY SUR ORGE":"91","ETAMPES":"91","FLEURY MEROGIS":"91","FONTENAY LES BRIIS":"91","LA FORET LE ROI":"91","FORGES LES BAINS":"91","JANVILLE SUR JUINE":"91","LINAS":"91","MAISSE":"91","MAROLLES EN BEAUCE":"91","MILLY LA FORET":"91","LA NORVILLE":"91","PALAISEAU":"91","LE PLESSIS PATE":"91","PUSSAY":"91","ROINVILLIERS":"91","SACLAS":"91","SACLAY":"91","ST CYR SOUS DOURDAN":"91","ST ESCOBILLE":"91","ST GERMAIN LES CORBEIL":"91","ST JEAN DE BEAUREGARD":"91","ST SULPICE DE FAVIERES":"91","ST MAURICE AUX FORGES":"54","ST MAX":"54","ST PANCRE":"54","ST SUPPLET":"54","SEXEY AUX FORGES":"54","OHLUNGEN":"67","VAL DE MODER":"67","PLAINE":"67","RAUWILLER":"67","REICHSHOFFEN":"67","REIPERTSWILLER":"67","RIEDSELTZ":"67","ROHR":"67","ROMANSWILLER":"67","ROSENWILLER":"67","ROTHAU":"67","SAALES":"67","ST NABOR":"67","ST PIERRE BOIS":"67","SALMBACH":"67","SARRE UNION":"67","SCHILTIGHEIM":"67","SCHIRMECK":"67","SCHIRRHOFFEN":"67","SELTZ":"67","STOTZHEIM":"67","STRASBOURG":"67","STUTZHEIM OFFENHEIM":"67","SURBOURG":"67","THAL MARMOUTIER":"67","TRIMBACH":"67","TRUCHTERSHEIM":"67","UHLWILLER":"67","URBEIS":"67","VOELLERDINGEN":"67","WALDHAMBACH":"67","WEINBOURG":"67","WEITERSWILLER":"67","WEYER":"67","WILLGOTTHEIM":"67","WIMMENAU":"67","WINGEN SUR MODER":"67","WINTERSHOUSE":"67","WINTZENBACH":"67","WISCHES":"67","WISSEMBOURG":"67","WITTISHEIM":"67","ZOEBERSDORF":"67","ALTENACH":"68","AMMERSCHWIHR":"68","ANDOLSHEIM":"68","APPENWIHR":"68","ARTZENHEIM":"68","ATTENSCHWILLER":"68","BALDERSHEIM":"68","BALTZENHEIM":"68","BENNWIHR":"68","BERGHOLTZ":"68","BERNWILLER":"68","BIEDERTHAL":"68","BREITENBACH HAUT RHIN":"68","CHAVANNES SUR L ETANG":"68","COURTAVON":"68","DESSENHEIM":"68","DURMENACH":"68","ELBACH":"68","ESCHENTZWILLER":"68","FORTSCHWIHR":"68","GEISHOUSE":"68","GEISWASSER":"68","GOMMERSDORF":"68","HOCHSTATT":"68","HUNDSBACH":"68","JUNGHOLTZ":"68","KAYSERSBERG VIGNOBLE":"68","KIFFIS":"68","KNOERINGUE":"68","LABAROCHE":"68","LAUTENBACH":"68","LEIMBACH":"68","LIEBENSWILLER":"68","LIEPVRE":"68","LOGELHEIM":"68","LUTTENBACH PRES MUNSTER":"68","MALMERSPACH":"68","MASEVAUX NIEDERBRUCK":"68","METZERAL":"68","MITZACH":"68","MUESPACH":"68","NEUWILLER":"68","NIEDERHERGHEIM":"68","NIFFER":"68","OBERHERGHEIM":"68","OBERMORSCHWIHR":"68","ST ETIENNE AU MONT":"62","ST FLORIS":"62","STE MARIE KERQUE":"62","ST VENANT":"62","SARTON":"62","SENINGHEM":"62","SIRACOURT":"62","SOMBRIN":"62","LE SOUICH":"62","SURQUES":"62","TARDINGHEN":"62","TERNAS":"62","TOLLENT":"62","VACQUERIE LE BOUCQ":"62","VENDIN LES BETHUNE":"62","VERCHIN":"62","VERQUIN":"62","VILLERS L HOPITAL":"62","WAMIN":"62","WARDRECQUES":"62","WARLINCOURT LES PAS":"62","WARLUZEL":"62","WESTREHEM":"62","WICQUINGHEM":"62","WIRWIGNES":"62","WISMES":"62","WISSANT":"62","APCHAT":"63","AULNAT":"63","BEAUMONT LES RANDAN":"63","BIOLLET":"63","BORT L ETANG":"63","BRIFFONS":"63","BUSSEOL":"63","BUXIERES SOUS MONTAIGUT":"63","LE CENDRE":"63","CHAMBON SUR LAC":"63","CHAMPEIX":"63","CHANAT LA MOUTEYRE":"63","CHAPDES BEAUFORT":"63","LA CHAPELLE MARCOUSSE":"63","CHAPTUZAT":"63","CHARENSAT":"63","CHARNAT":"63","CHAS":"63","CHASSAGNE":"63","CHATEL GUYON":"63","LA CHAULME":"63","CHIDRAC":"63","CLEMENSAT":"63","COURNOLS":"63","COURNON D AUVERGNE":"63","LE CREST":"63","LA CROUZILLE":"63","CULHAT":"63","DORANGES":"63","EGLISENEUVE D ENTRAIGUES":"63","ENNEZAT":"63","ESPIRAT":"63","ESTANDEUIL":"63","GIAT":"63","GIMEAUX":"63","LAMONTGIE":"63","LANDOGNE":"63","LEMPTY":"63","LISSEUIL":"63","MADRIAT":"63","MARINGUES":"63","MAZOIRES":"63","MONT DORE":"63","MONTFERMY":"63","MONTPENSIER":"63","MURAT LE QUAIRE":"63","NOALHAT":"63","NOHANENT":"63","OLBY":"63","OLMET":"63","ORBEIL":"63","PARENTIGNAT":"63","PERIGNAT LES SARLIEVE":"63","PONTAUMUR":"63","PONTGIBAUD":"63","PUY GUILLAUME":"63","QUEUILLE":"63","PEYSSIES":"31","PINSAGUEL":"31","PLAGNOLE":"31","POMPERTUZAT":"31","PORTET D ASPET":"31","POUBEAU":"31","PRADERE LES BOURGUETS":"31","REBIGUE":"31","SABONNERES":"31","STE FOY DE PEYROLIERES":"31","ST JULIA":"31","ST LARY BOUJEAN":"31","ST ORENS DE GAMEVILLE":"31","ST PAUL D OUEIL":"31","ST ROME":"31","SALERM":"31","SALIES DU SALAT":"31","SAUBENS":"31","SAVARTHES":"31","SEPX":"31","SEYRE":"31","SIGNAC":"31","SODE":"31","TERREBASSE":"31","VALLESVILLES":"31","VERNET":"31","VIGNAUX":"31","VIGOULET AUZIL":"31","VILLENOUVELLE":"31","ANSAN":"32","ARDIZAS":"32","ARMENTIEUX":"32","ARMOUS ET CAU":"32","AUCH":"32","AUGNAX":"32","AVERON BERGELLE":"32","AYGUETINTE":"32","BAJONNETTE":"32","BASCOUS":"32","BAZIAN":"32","BEAUMARCHES":"32","BERNEDE":"32","BETPLAN":"32","BEZOLLES":"32","BEZUES BAJON":"32","BLOUSSON SERIAN":"32","BOULAUR":"32","LE BROUILH MONBERT":"32","BRUGNENS":"32","CANNET":"32","CASSAIGNE":"32","CASTELNAU D ARBIEU":"32","CASTELNAU D AUZAN LABARRERE":"32","CASTELNAU SUR L AUVIGNON":"32","CASTET ARROUY":"32","CASTILLON DEBATS":"32","CATONVIELLE":"32","CAUSSENS":"32","CAZAUX D ANGLES":"32","CAZAUX SAVES":"32","CEZAN":"32","CORNEILLAN":"32","CRAVENCERES":"32","DEMU":"32","ENDOUFIELLE":"32","ESTANG":"32","FLEURANCE":"32","GALIAX":"32","IZOTGES":"32","JEGUN":"32","LAHITTE":"32","YQUELON":"50","AMBRIERES":"51","ANTHENAY":"51","ATHIS":"51","AULNAY L AITRE":"51","AUMENANCOURT":"51","AVIZE":"51","BARBONNE FAYEL":"51","BETHENIVILLE":"51","BLAISE SOUS ARZILLIERES":"51","BOURGOGNE":"51","BOUY":"51","BRANDONVILLERS":"51","BRAUX STE COHIERE":"51","LA CAURE":"51","CERNAY LES REIMS":"51","CHAINTRIX BIERGES":"51","CHALTRAIT":"51","BAZOUGERS":"53","BELGEARD":"53","BREE":"53","LA BRULATTE":"53","LE BURET":"53","CHAILLAND":"53","CHANTRIGNE":"53","LA CHAPELLE ANTHENAISE":"53","LA CHAPELLE AU RIBOUL":"53","CHATRES LA FORET":"53","COSSE EN CHAMPAGNE":"53","COURBEVEILLE":"53","CUILLE":"53","EVRON":"53","GASTINES":"53","GRAZAY":"53","LE HORPS":"53","LANDIVY":"53","LAUBRIERES":"53","LOUPFOUGERES":"53","MAISONCELLES DU MAINE":"53","MARCILLE LA VILLE":"53","MAYENNE":"53","MENIL":"53","MERAL":"53","MONTIGNE LE BRILLANT":"53","MONTREUIL POULAY":"53","NEAU":"53","NUILLE SUR VICOIN":"53","OISSEAU":"53","LA PALLU":"53","PRE EN PAIL ST SAMSON":"53","RENNES EN GRENOUILLES":"53","ST AUBIN DU DESERT":"53","ST BAUDELLE":"53","ST DENIS DE GASTINES":"53","ST DENIS DU MAINE":"53","ST GERMAIN LE FOUILLOUX":"53","ST HILAIRE DU MAINE":"53","ST JEAN SUR ERVE":"53","STE MARIE DU BOIS":"53","ST MARTIN DU LIMET":"53","ST PIERRE DES NIDS":"53","ST SATURNIN DU LIMET":"53","LA SELLE CRAONNAISE":"53","SOUCE":"53","SOULGE SUR OUETTE":"53","TRANS":"53","VILLAINES LA JUHEL":"53","VOUTRE":"53","ALLAIN":"54","ARRACOURT":"54","ARRAYE ET HAN":"54","AUBOUE":"54","BADONVILLER":"54","BARBAS":"54","BAYONVILLE SUR MAD":"54","BENAMENIL":"54","BERTRAMBOIS":"54","BERTRICHAMPS":"54","BEY SUR SEILLE":"54","BIENVILLE LA PETITE":"54","BOUILLONVILLE":"54","BRATTE":"54","BREMENIL":"54","BREMONCOURT":"54","BRIEY":"54","BRIN SUR SEILLE":"54","BROUVILLE":"54","BURIVILLE":"54","CEINTREY":"54","CHANTEHEUX":"54","CHARMES LA COTE":"54","CIREY SUR VEZOUZE":"54","COLOMBEY LES BELLES":"54","COYVILLER":"54","CREPEY":"54","CRION":"54","SAULX LES CHARTREUX":"91","SAVIGNY SUR ORGE":"91","VALPUISEAUX":"91","VAUGRIGNEUSE":"91","VAUHALLAN":"91","BOIS COLOMBES":"92","CHATENAY MALABRY":"92","CLAMART":"92","COURBEVOIE":"92","FONTENAY AUX ROSES":"92","LEVALLOIS PERRET":"92","PANTIN":"93","LES PAVILLONS SOUS BOIS":"93","LE PRE ST GERVAIS":"93","VILLETANEUSE":"93","BONNEUIL SUR MARNE":"94","CHENNEVIERES SUR MARNE":"94","CHEVILLY LARUE":"94","FONTENAY SOUS BOIS":"94","ORLY":"94","SANTENY":"94","SUCY EN BRIE":"94","VILLENEUVE LE ROI":"94","VINCENNES":"94","VITRY SUR SEINE":"94","BUHY":"95","CERGY":"95","LA CHAPELLE EN VEXIN":"95","CHARS":"95","CHENNEVIERES LES LOUVRES":"95","ECOUEN":"95","ERMONT":"95","FOSSES":"95","FREPILLON":"95","GENAINVILLE":"95","GENICOURT":"95","GROSLAY":"95","HAUTE ISLE":"95","HERBLAY":"95","L ISLE ADAM":"95","LONGUESSE":"95","LOUVRES":"95","MAUDETOUR EN VEXIN":"95","MERY SUR OISE":"95","RONQUEROLLES":"95","ST LEU LA FORET":"95","SANNOIS":"95","VAUDHERLAND":"95","VEMARS":"95","WY DIT JOLI VILLAGE":"95","GRAND BOURG":"971","LAMENTIN":"971","PETIT BOURG":"971","ST FRANCOIS":"971","TROIS RIVIERES":"971","BASSE POINTE":"972","LE DIAMANT":"972","FONDS ST DENIS":"972","LE MORNE ROUGE":"972","LE ROBERT":"972","ST ESPRIT":"972","SCHOELCHER":"972","LES TROIS ILETS":"972","MACOURIA":"973","MATOURY":"973","ST LAURENT DU MARONI":"973","MONTSINERY TONNEGRANDE":"973","OUANARY":"973","ENTRE DEUX":"974","LA PLAINE DES PALMISTES":"974","MIQUELON LANGLADE":"975","OBERMORSCHWILLER":"68","OBERSAASHEIM":"68","ODEREN":"68","OTTMARSHEIM":"68","REGUISHEIM":"68","RICHWILLER":"68","RIEDISHEIM":"68","RIMBACH PRES GUEBWILLER":"68","RIQUEWIHR":"68","ROGGENHOUSE":"68","RUSTENHART":"68","SAUSHEIM":"68","SCHWEIGHOUSE THANN":"68","SENTHEIM":"68","SEWEN":"68","SOULTZEREN":"68","STEINBRUNN LE HAUT":"68","TRAUBACH LE BAS":"68","TRAUBACH LE HAUT":"68","URBES":"68","VIEUX THANN":"68","WENTZWILLER":"68","WINKEL":"68","WITTELSHEIM":"68","WITTERSDORF":"68","WOLFERSDORF":"68","WOLFGANTZEN":"68","ZELLENBERG":"68","ALBIGNY SUR SAONE":"69","AMBERIEUX":"69","AVEIZE":"69","AVENAS":"69","AZOLETTE":"69","BESSENAY":"69","CHAMBOST LONGESSAIGNE":"69","LA CHAPELLE SUR COISE":"69","CIVRIEUX D AZERGUES":"69","DUERNE":"69","FONTAINES SUR SAONE":"69","FRONTENAS":"69","GIVORS":"69","LES HAIES":"69","JOUX":"69","LEGNY":"69","MOIRE":"69","MONSOLS":"69","MONTMELAS ST SORLIN":"69","OULLINS":"69","RIVERIE":"69","RIVOLET":"69","SOUCIEU EN JARREST":"69","SOUZY":"69","ST APPOLINAIRE":"69","ST BONNET DES BRUYERES":"69","ST ETIENNE DES OULLIERES":"69","ST JEAN LA BUSSIERE":"69","ST LAGER":"69","ST LAURENT D AGNY":"69","ST ROMAIN EN GIER":"69","TERNAND":"69","THEIZE":"69","VAUGNERAY":"69","VERNAY":"69","VILLECHENEVE":"69","COMMUNAY":"69","FEYZIN":"69","GENAS":"69","ST LAURENT DE MURE":"69","ST PIERRE DE CHANDIEU":"69","ST SYMPHORIEN D OZON":"69","COLOMBIER SAUGNIEU":"69","LYON 01":"69","LYON 02":"69","LYON 04":"69","LYON 08":"69","ACHEY":"70","AILLEVILLERS ET LYAUMONT":"70","AMAGE":"70","AMBLANS ET VELOTTE":"70","ANCHENONCOURT ET CHAZEL":"70","AUTREY LES GRAY":"70","AUVET ET LA CHAPELOTTE":"70","BAIGNES":"70","LES BATIES":"70","BAULAY":"70","BAY":"70","BEAUMOTTE LES PIN":"70","BESNANS":"70","BETAUCOURT":"70","LA RENAUDIE":"63","ST AGOULIN":"63","ST CIRGUES SUR COUZE":"63","ST CLEMENT DE VALORGUE":"63","ST CLEMENT DE REGNAT":"63","ST DIER D AUVERGNE":"63","ST ELOY LES MINES":"63","ST GENES CHAMPESPE":"63","ST GENES DU RETZ":"63","ST GEORGES DE MONS":"63","ST HERENT":"63","ST JEAN D HEURS":"63","ST JEAN DES OLLIERES":"63","ST JEAN ST GERVAIS":"63","ST JULIEN LA GENESTE":"63","ST MAURICE PRES PIONSAT":"63","ST MYON":"63","ST NECTAIRE":"63","ST REMY DE CHARGNAT":"63","ST REMY SUR DUROLLE":"63","ST SANDOUX":"63","SARDON":"63","SAURIER":"63","SAUXILLANGES":"63","SINGLES":"63","SUGERES":"63","SURAT":"63","TALLENDE":"63","TORTEBESSE":"63","TOURS SUR MEYMONT":"63","TOURZEL RONZIERES":"63","TREZIOUX":"63","VENSAT":"63","VICHEL":"63","VIC LE COMTE":"63","YOUX":"63","YRONDE ET BURON":"63","ABIDOS":"64","AGNOS":"64","AHETZE":"64","AINHICE MONGELOS":"64","ALOS SIBAS ABENSE":"64","ANCE":"64","ANDREIN":"64","ARCANGUES":"64","ARGAGNON":"64","ARRAUTE CHARRITTE":"64","ARRIEN":"64","ARTHEZ DE BEARN":"64","ARTHEZ D ASSON":"64","ARTIGUELOUTAN":"64","ARUDY":"64","ASASP ARROS":"64","ASSAT":"64","ATHOS ASPIS":"64","AUDAUX":"64","AUTERRIVE":"64","AYHERRE":"64","BALEIX":"64","BALIRACQ MAUMUSSON":"64","BALIROS":"64","BANCA":"64","BASTANES":"64","BASSUSSARRY":"64","BEDOUS":"64","BEHORLEGUY":"64","BEOST":"64","BESINGRAND":"64","BETRACQ":"64","BEYRIE SUR JOYEUSE":"64","BEYRIE EN BEARN":"64","BIRIATOU":"64","BONLOC":"64","BONNUT":"64","BORCE":"64","BOUILLON":"64","BURGARONNE":"64","CADILLON":"64","CARDESSE":"64","CASTEIDE CAMI":"64","CASTERA LOUBIX":"64","CASTETPUGON":"64","CAUBIOS LOOS":"64","CROUSEILLES":"64","DOGNEN":"64","DOUMY":"64","ESPOEY":"64","FEAS":"64","GABASTON":"64","GABAT":"64","GARINDEIN":"64","GELOS":"64","GERDEREST":"64","GERONCE":"64","CHAMPAUBERT":"51","CHAMPVOISY":"51","LA CHAPELLE SOUS ORBAIS":"51","CHATELRAOULD ST LOUVENT":"51","CHATRICES":"51","CHAUMUZY":"51","LA CHEPPE":"51","CLAMANGES":"51","CLESLES":"51","CLOYES SUR MARNE":"51","CONFLANS SUR SEINE":"51","CONNANTRE":"51","COOLE":"51","CORFELIX":"51","CORRIBERT":"51","CORROBERT":"51","CORROY":"51","COULOMMES LA MONTAGNE":"51","COURCEMAIN":"51","COURTAGNON":"51","LA CROIX EN CHAMPAGNE":"51","CUIS":"51","DAMPIERRE SUR MOIVRE":"51","DOMMARTIN LETTREE":"51","DONTRIEN":"51","EPERNAY":"51","ESTERNAY":"51","ETOGES":"51","FEREBRIANGES":"51","FERE CHAMPENOISE":"51","FISMES":"51","FLEURY LA RIVIERE":"51","FONTAINE SUR AY":"51","GIFFAUMONT CHAMPAUBERT":"51","GIGNY BUSSY":"51","GIONGES":"51","GIVRY EN ARGONNE":"51","STE MARIE DU LAC NUISEMENT":"51","HAUTVILLERS":"51","HEUTREGIVILLE":"51","HOURGES":"51","HUMBAUVILLE":"51","ISLES SUR SUIPPE":"51","JONCHERY SUR VESLE":"51","JOUY LES REIMS":"51","LAGERY":"51","LAVAL SUR TOURBE":"51","LINTHELLES":"51","LISSE EN CHAMPAGNE":"51","LIVRY LOUVERCY":"51","LOIVRE":"51","MAFFRECOURT":"51","MARSANGIS":"51","LE MEIX TIERCELIN":"51","MERLAUT":"51","MINAUCOURT LE MESNIL LES HURLUS":"51","MOIVRE":"51","MONCETZ L ABBAYE":"51","MONTBRE":"51","MONTEPREUX":"51","MONTMORT LUCY":"51","MUTIGNY":"51","NESLE LE REPONS":"51","LA NOUE":"51","OLIZY":"51","ORCONTE":"51","PARGNY SUR SAULX":"51","PEAS":"51","PIERRY":"51","POTANGIS":"51","POURCY":"51","PUISIEULX":"51","QUEUDES":"51","RECY":"51","REUIL":"51","ST BON":"51","ST ETIENNE AU TEMPLE":"51","ST IMOGES":"51","ST MARD SUR LE MONT":"51","STE MARIE A PY":"51","ST REMY EN BOUZEMONT ST GENEST ISSON":"51","SAVIGNY SUR ARDRES":"51","SERMIERS":"51","SERZY ET PRIN":"51","SOMPUIS":"51","DAMPVITOUX":"54","DEUXVILLE":"54","DIEULOUARD":"54","DOMGERMAIN":"54","DOMMARIE EULMONT":"54","ECROUVES":"54","EMBERMENIL":"54","EULMONT":"54","FONTENOY LA JOUTE":"54","FORCELLES ST GORGON":"54","GELACOURT":"54","GEMONVILLE":"54","GERBEVILLER":"54","GERMINY":"54","GERMONVILLE":"54","GOGNEY":"54","GONDRECOURT AIX":"54","GRAND FAILLY":"54","GROSROUVRES":"54","HAMMEVILLE":"54","HERIMENIL":"54","HOMECOURT":"54","JEVONCOURT":"54","LAIX":"54","LAMATH":"54","LANEUVEVILLE DEVANT NANCY":"54","LANTEFONTAINE":"54","LEBEUVILLE":"54","LENONCOURT":"54","LETRICOURT":"54","LEYR":"54","LIRONVILLE":"54","LONGUYON":"54","LUDRES":"54","MAIRY MAINVILLE":"54","MAMEY":"54","MANCE":"54","MANGONVILLE":"54","MANONCOURT EN VERMOIS":"54","MANONVILLE":"54","MANONVILLER":"54","MARS LA TOUR":"54","MATTEXEY":"54","MONT L ETROIT":"54","MORIVILLER":"54","MOUACOURT":"54","OCHEY":"54","OMELMONT":"54","ONVILLE":"54","PAREY ST CESAIRE":"54","PETTONVILLE":"54","PEXONNE":"54","PHLIN":"54","PIERRE PERCEE":"54","ROGEVILLE":"54","ST NICOLAS DE PORT":"54","SANZEY":"54","SAULNES":"54","SERANVILLE":"54","SEXEY LES BOIS":"54","SOMMERVILLER":"54","COURCELLES DE TOURAINE":"37","LA CROIX EN TOURAINE":"37","CUSSAY":"37","FONDETTES":"37","HOMMES":"37","JAULNAY":"37","JOUE LES TOURS":"37","LEMERE":"37","LERNE":"37","LE LOUROUX":"37","LUSSAULT SUR LOIRE":"37","MANTHELAN":"37","MARCE SUR ESVES":"37","MONTLOUIS SUR LOIRE":"37","MONTRESOR":"37","MORAND":"37","NEUILLE LE LIERRE":"37","NEUVILLE SUR BRENNE":"37","ORBIGNY":"37","PARCAY SUR VIENNE":"37","POUZAY":"37","RESTIGNE":"37","ROUZIERS DE TOURAINE":"37","ST AUBIN LE DEPEINT":"37","ST ETIENNE DE CHIGNY":"37","ST FLOVIER":"37","BANDRELE":"976","M TSANGAMOUJI":"976","SIGAVE":"986","UVEA":"986","BORA BORA":"987","FAKARAVA":"987","GAMBIER":"987","HIVA OA":"987","MANIHI":"987","RAIVAVAE":"987","REAO":"987","TUBUAI":"987","UA POU":"987","DUMBEA":"988","KAALA GOMEN":"988","LIFOU":"988","MARE":"988","MOINDOU":"988","PAITA":"988","PONERIHOUEN":"988","POUEMBOUT":"988","VOH":"988","MONACO":"99","ALATA":"2A","APPIETTO":"2A","ARRO":"2A","CALCATOGGIO":"2A","CANNELLE":"2A","CARBINI":"2A","CASAGLIONE":"2A","CAURO":"2A","CIAMANNACCE":"2A","COGNOCOLI MONTICCHI":"2A","EVISA":"2A","FIGARI":"2A","FORCIOLO":"2A","GRANACE":"2A","LEVIE":"2A","MURZO":"2A","ORTO":"2A","PALNECA":"2A","PETRETO BICCHISANO":"2A","POGGIOLO":"2A","PROPRIANO":"2A","QUASQUARA":"2A","RENNO":"2A","ROSAZIA":"2A","SALICE":"2A","SARI D ORCINO":"2A","SANT ANDREA D ORCINO":"2A","TAVACO":"2A","TOLLA":"2A","VICO":"2A","BORGO":"2B","BUSTANICO":"2B","CALACUCCIA":"2B","CALVI":"2B","CAMPILE":"2B","CARCHETO BRUSTICO":"2B","CASTELLARE DI CASINCA":"2B","CASTIGLIONE":"2B","CASTIRLA":"2B","CATERI":"2B","CENTURI":"2B","BEULOTTE ST LAURENT":"70","BOREY":"70","BOUGEY":"70","BOUHANS ET FEURG":"70","BOUHANS LES LURE":"70","BOULOT":"70","BOURBEVELLE":"70","BREVILLIERS":"70","BROTTE LES RAY":"70","CALMOUTIER":"70","CEMBOING":"70","CHAMBORNAY LES PIN":"70","CHAMPEY":"70","CHANTES":"70","CHARIEZ":"70","CHASSEY LES MONTBOZON":"70","CIREY":"70","CITERS":"70","CITEY":"70","COGNIERES":"70","COLOMBOTTE":"70","CONFLANS SUR LANTERNE":"70","CONFRACOURT":"70","CORRE":"70","CRESANCEY":"70","CROMARY":"70","CUBRY LES FAVERNEY":"70","LA DEMIE":"70","ECHENANS SOUS MONT VAUDOIS":"70","ECROMAGNY":"70","ECUELLE":"70","EHUNS":"70","ERREVET":"70","ESSERTENNE ET CECEY":"70","ETUZ":"70","FAHY LES AUTREY":"70","FLEUREY LES LAVONCOURT":"70","FROTEY LES LURE":"70","FROTEY LES VESOUL":"70","GRANDECOURT":"70","HAUT DU THEM CHATEAU LAMBERT":"70","JONVELLE":"70","LARIANS ET MUNANS":"70","LARRET":"70","LAVIGNEY":"70","LAVONCOURT":"70","LIEUCOURT":"70","LIEVANS":"70","LONGEVELLE":"70","MAGNY VERNOIS":"70","MAILLEY ET CHAZELOT":"70","LA MALACHERE":"70","MANTOCHE":"70","MEMBREY":"70","MIGNAVILLERS":"70","MONTDORE":"70","MONTIGNY LES CHERLIEU":"70","NEUVELLE LES CROMARY":"70","NOIDANS LE FERROUX":"70","NOIRON":"70","OIGNEY":"70","ORMENANS":"70","OUGE":"70","PONTCEY":"70","PONT SUR L OGNON":"70","PORT SUR SAONE":"70","LA QUARTE":"70","RAZE":"70","RIGNY":"70","ROCHE LINOTTE ET SORANS CORDIERS":"70","RONCHAMP":"70","ST LOUP SUR SEMOUSE":"70","STE MARIE EN CHAUX":"70","TAVEY":"70","VAITE":"70","VANNE":"70","VAROGNE":"70","VAUX LE MONCELOT":"70","VELLEGUINDRY ET LEVRECEY":"70","LA VERGENNE":"70","LA VILLEDIEU EN FONTENETTE":"70","VILLERSEXEL":"70","VILLERS VAUDEY":"70","VITREY SUR MANCE":"70","VORAY SUR L OGNON":"70","VOUGECOURT":"70","GURS":"64","HAGETAUBIN":"64","HIGUERES SOUYE":"64","L HOPITAL ST BLAISE":"64","HOURS":"64","ILHARRE":"64","IROULEGUY":"64","JURANCON":"64","LABASTIDE CEZERACQ":"64","LAGOR":"64","LALONQUETTE":"64","LANNECAUBE":"64","LASSEUBETAT":"64","LEMBEYE":"64","LEREN":"64","LUC ARMAU":"64","LUXE SUMBERRAUTE":"64","MASPARRAUTE":"64","MASPIE LALONQUERE JUILLACQ":"64","MAULEON LICHARRE":"64","MAZERES LEZONS":"64","MIALOS":"64","MOMY":"64","MONASSUT AUDIRACQ":"64","MONTORY":"64","MOURENX":"64","MUSCULDY":"64","NAVAILLES ANGOS":"64","OGEU LES BAINS":"64","ORAAS":"64","ORDIARP":"64","ORIN":"64","ORRIULE":"64","ORTHEZ":"64","PIETS PLASENCE MOUSTROU":"64","PONTIACQ VIELLEPINTE":"64","REBENACQ":"64","RIVEHAUTE":"64","RONTIGNON":"64","ST BOES":"64","STE COLOME":"64","ST JAMMES":"64","ST JEAN POUDGE":"64","ST MARTIN D ARBEROUE":"64","ST PIERRE D IRUBE":"64","SARRANCE":"64","SAULT DE NAVAILLES":"64","SERRES STE MARIE":"64","SEVIGNACQ MEYRACQ":"64","SIMACOURBE":"64","SOUMOULOU":"64","SOURAIDE":"64","TROIS VILLES":"64","UREPEL":"64","URRUGNE":"64","URT":"64","UZAN":"64","VIELLENAVE DE NAVARRENX":"64","VIODOS ABENSE DE BAS":"64","ANGOS":"65","ANTICHAN":"65","ANTIN":"65","ARGELES GAZOST":"65","ARTAGNAN":"65","ASPIN AURE":"65","ASTE":"65","ASTUGUE":"65","AVAJAN":"65","AVENTIGNAN":"65","BARBAZAN DESSUS":"65","BEAUCENS":"65","BERTREN":"65","BETPOUY":"65","BETTES":"65","BORDERES LOURON":"65","BRAMEVAQUE":"65","BURG":"65","CADEAC":"65","CAIXON":"65","CAMPARAN":"65","CAMPISTROUS":"65","CASTELNAU MAGNOAC":"65","CASTERA LOU":"65","CAZAUX FRECHET ANERAN CAMORS":"65","CHEUST":"65","ESQUIEZE SERE":"65","FONTRAILLES":"65","TALUS ST PRIX":"51","TILLOY ET BELLAY":"51","TREFOLS":"51","TROIS FONTAINES L ABBAYE":"51","VADENAY":"51","VASSIMONT ET CHAPELAINE":"51","VAUCLERC":"51","VENTEUIL":"51","VERTUS":"51","LA VEUVE":"51","VIENNE LA VILLE":"51","VILLE DOMMANGE":"51","VILLERS AUX BOIS":"51","VILLERS SOUS CHATILLON":"51","VOUARCES":"51","VRAUX":"51","WARMERIVILLE":"51","WITRY LES REIMS":"51","AGEVILLE":"52","AIZANVILLE":"52","ANDILLY EN BASSIGNY":"52","APREY":"52","AUBEPIERRE SUR AUBE":"52","BAILLY AUX FORGES":"52","BASSONCOURT":"52","BOUZANCOURT":"52","BRICON":"52","BROUSSEVAL":"52","BUSSON":"52","BUXIERES LES VILLIERS":"52","CHAMPIGNY LES LANGRES":"52","CHARMES LA GRANDE":"52","CHAUFFOURT":"52","CHOISEUL":"52","CLINCHAMP":"52","COIFFY LE BAS":"52","DAILLECOURT":"52","DINTEVILLE":"52","DOMMARTIN LE FRANC":"52","DONCOURT SUR MEUSE":"52","DOULEVANT LE PETIT":"52","EURVILLE BIENVILLE":"52","FRAMPAS":"52","FRECOURT":"52","GENEVRIERES":"52","LA GENEVROYE":"52","GILLANCOURT":"52","GRENANT":"52","GUINDRECOURT AUX ORMES":"52","HALLIGNICOURT":"52","LAMANCINE":"52","LANEUVILLE A REMY":"52","LATRECEY ORMOY SUR AUBE":"52","LAVILLENEUVE AU ROI":"52","LEUCHEY":"52","LONGEAU PERCEY":"52","LUZY SUR MARNE":"52","MAIZIERES SUR AMANCE":"52","MERREY":"52","MERTRUD":"52","MEURES":"52","MIRBEL":"52","MORANCOURT":"52","MUSSEY SUR MARNE":"52","NONCOURT SUR LE RONGEANT":"52","PREZ SOUS LAFAUCHE":"52","ST GERMAIN SUR VIENNE":"37","STE MAURE DE TOURAINE":"37","ST REGLE":"37","SORIGNY":"37","TAUXIGNY":"37","TAVANT":"37","THILOUZE":"37","TOURNON ST PIERRE":"37","TOURS":"37","VILLEPERDUE":"37","VILLIERS AU BOUIN":"37","VOU":"37","ALLEMOND":"38","ANJOU":"38","ANNOISIN CHATELANS":"38","ARTAS":"38","BARRAUX":"38","BIOL":"38","BIZONNES":"38","BONNEFAMILLE":"38","LE BOURG D OISANS":"38","BRANGUES":"38","CHALON":"38","CHAMAGNIEU":"38","CHAMPAGNIER":"38","CHARETTE":"38","CHARVIEU CHAVAGNEUX":"38","CHICHILIANNE":"38","CHIMILIN":"38","CHONAS L AMBALLAN":"38","CHUZELLES":"38","CLONAS SUR VAREZE":"38","COUR ET BUIS":"38","CREYS MEPIEU":"38","CROLLES":"38","ECLOSE BADINIERES":"38","LA FORTERESSE":"38","LE FRENEY D OISANS":"38","FRONTONAS":"38","GIERES":"38","GONCELIN":"38","GRANIEU":"38","HEYRIEUX":"38","L ISLE D ABEAU":"38","LAFFREY":"38","LALLEY":"38","LAVARS":"38","LIVET ET GAVET":"38","LUMBIN":"38","MERLAS":"38","MEYSSIEZ":"38","MIZOEN":"38","MOIDIEU DETOURBE":"38","MOIRANS":"38","MONTBONNOT ST MARTIN":"38","MONT DE LANS":"38","MONTEYNARD":"38","MORAS":"38","MORETTE":"38","LA MURETTE":"38","MURIANETTE":"38","NIVOLAS VERMELLE":"38","NOTRE DAME DE VAULX":"38","ORNON":"38","PAJAY":"38","PALADRU":"38","PELLAFOL":"38","PERCY":"38","LA PIERRE":"38","PINSOT":"38","POLIENAS":"38","PONSONNAS":"38","PRIMARETTE":"38","QUET EN BEAUMONT":"38","LES ROCHES DE CONDRIEU":"38","ST AREY":"38","ST BAUDILLE DE LA TOUR":"38","ST CLAIR SUR GALAURE":"38","ST ETIENNE DE CROSSEY":"38","ST GEORGES DE COMMIERS":"38","ST JEAN D AVELANNE":"38","ST JEAN D HERANS":"38","ST JULIEN DE L HERMS":"38","ST LAURENT EN BEAUMONT":"38","ST MARTIN DE CLELLES":"38","ST MAURICE L EXIL":"38","ST MICHEL EN BEAUMONT":"38","ST MICHEL LES PORTES":"38","ST MURY MONTEYMOND":"38","ST NAZAIRE LES EYMES":"38","CHIATRA":"2B","ERBAJOLO":"2B","FOCICCHIA":"2B","LANO":"2B","MATRA":"2B","MAZZOLA":"2B","MOITA":"2B","NESSA":"2B","OLMI CAPPELLA":"2B","PALASCA":"2B","PATRIMONIO":"2B","PERO CASEVECCHIE":"2B","PIEDICORTE DI GAGGIO":"2B","PIEDICROCE":"2B","PIE D OREZZA":"2B","PINO":"2B","POGGIO D OLETTA":"2B","POLVEROSO":"2B","PRATO DI GIOVELLINA":"2B","ROGLIANO":"2B","SCOLCA":"2B","SILVARECCIO":"2B","SANT ANDREA DI BOZIO":"2B","SAN GIOVANNI DI MORIANI":"2B","SANTA MARIA DI LOTA":"2B","SAN NICOLAO":"2B","SANTO PIETRO DI VENACO":"2B","SANTA REPARATA DI BALAGNA":"2B","TARRANO":"2B","VELONE ORNETO":"2B","VENTISERI":"2B","VIVARIO":"2B","ZILIA":"2B","ZUANI":"2B","LA VEZE":"25","VIEILLEY":"25","VILLARS LES BLAMONT":"25","LES VILLEDIEU":"25","VILLERS BUZON":"25","VILLERS GRELOT":"25","VILLERS ST MARTIN":"25","VOIRES":"25","VOUJEAUCOURT":"25","AMBONIL":"26","ANCONE":"26","AUBENASSON":"26","LA REPARA AURIPLES":"26","BARBIERES":"26","LA BATIE DES FONDS":"26","BEAUMONT LES VALENCE":"26","BONLIEU SUR ROUBION":"26","BUIS LES BARONNIES":"26","CHABEUIL":"26","LE CHAFFAL":"26","CHAMARET":"26","CHANTEMERLE LES BLES":"26","CHANTEMERLE LES GRIGNAN":"26","LA CHAPELLE EN VERCORS":"26","CHARENS":"26","CLANSAYES":"26","CLERIEUX":"26","CLIOUSCLAT":"26","COBONNE":"26","COMBOVIN":"26","CRUPIES":"26","ESPELUCHE":"26","ESPENEL":"26","ESTABLET":"26","EYGALIERS":"26","EYMEUX":"26","EYZAHUT":"26","VAL MARAVEL":"26","GEYSSANS":"26","JONCHERES":"26","LABOREL":"26","LENS LESTANG":"26","LUC EN DIOIS":"26","MALATAVERNE":"26","MERCUROL VEAUNES":"26","MOLLANS SUR OUVEZE":"26","MONTAUBAN SUR L OUVEZE":"26","MONTBOUCHER SUR JABRON":"26","MONTELIER":"26","MONTJOYER":"26","MONTREAL LES SOURCES":"26","OMBLEZE":"26","LE POET SIGILLAT":"26","PORTES EN VALDAINE":"26","RECOUBEAU JANSAC":"26","REILHANETTE":"26","ROCHEBAUDIN":"26","ANZY LE DUC":"71","BEAUREPAIRE EN BRESSE":"71","BEAUVERNOIS":"71","BISSEY SOUS CRUCHAUD":"71","BLANZY":"71","BOURBON LANCY":"71","BOUZERON":"71","BRAGNY SUR SAONE":"71","BRIANT":"71","BURZY":"71","BUXY":"71","CHANES":"71","LA CHAPELLE NAUDE":"71","CHARNAY LES CHALON":"71","CHARNAY LES MACON":"71","CHASSELAS":"71","CHENOVES":"71","CLESSY":"71","LA COMELLE":"71","EPERVANS":"71","EPINAC":"71","LE FAY":"71","FRONTENAUD":"71","LES GUERREAUX":"71","LA GUICHE":"71","HURIGNY":"71","JOUVENCON":"71","JUGY":"71","JUIF":"71","LAIZY":"71","LALHEUE":"71","LESSARD LE NATIONAL":"71","LOUHANS":"71","MARTAILLY LES BRANCION":"71","MATOUR":"71","MAZILLE":"71","MERVANS":"71","MESSEY SUR GROSNE":"71","MESVRES":"71","MONTCEAUX L ETOILE":"71","MONTPONT EN BRESSE":"71","NAVILLY":"71","PERREUIL":"71","PIERRECLOS":"71","POUILLOUX":"71","PRESSY SOUS DONDIN":"71","PRUZILLY":"71","RATTE":"71","RIGNY SUR ARROUX":"71","ST ANDRE EN BRESSE":"71","ST AUBIN SUR LOIRE":"71","ST BONNET DE CRAY":"71","ST BONNET DE VIEILLE VIGNE":"71","ST ETIENNE EN BRESSE":"71","ST GENGOUX LE NATIONAL":"71","ST JULIEN DE JONZY":"71","BOURNOS":"64","BUGNEIN":"64","BUROS":"64","CARRERE":"64","CASTETIS":"64","CIBOURE":"64","CLARACQ":"64","COARRAZE":"64","CORBERE ABERES":"64","ESCOU":"64","ESCOUBES":"64","ESCURES":"64","ESPES UNDUREIN":"64","FICHOUS RIUMAYOU":"64","GAN":"64","GERE BELESTEN":"64","GOES":"64","GUETHARY":"64","GUICHE":"64","HALSOU":"64","HENDAYE":"64","HERRERE":"64","IBARROLLE":"64","FRECHENDETS":"65","GARDERES":"65","GAUDENT":"65","GEMBRIE":"65","GEZ EZ ANGLES":"65","GONEZ":"65","GOUAUX":"65","GREZIAN":"65","GUCHEN":"65","HAUBAN":"65","HAUTAGET":"65","HERES":"65","ILHET":"65","LABASSERE":"65","LANNE":"65","LANNEMEZAN":"65","LOUBAJAC":"65","LOUDENVIELLE":"65","LOUIT":"65","LOURDES":"65","LOURES BAROUSSE":"65","LUZ ST SAUVEUR":"65","MINGOT":"65","MONLONG":"65","MONTSERIE":"65","NISTOS":"65","ORGAN":"65","OROIX":"65","OSSEN":"65","OSSUN EZ ANGLES":"65","OUEILLOUX":"65","OURSBELILLE":"65","PEYRET ST ANDRE":"65","PINAS":"65","POUYASTRUC":"65","PUYDARRIEUX":"65","RABASTENS DE BIGORRE":"65","ST LAURENT DE NESTE":"65","SAMURAN":"65","SARRIAC BIGORRE":"65","SAZOS":"65","SIRADAN":"65","SIREIX":"65","SOMBRUN":"65","SOUBLECAUSE":"65","TALAZAC":"65","TARBES":"65","TIBIRAN JAUNAC":"65","TOURNOUS DARRE":"65","TROULEY LABARTHE":"65","UGNOUAS":"65","VIZOS":"65","LE BOULOU":"66","BOURG MADAME":"66","CARAMANY":"66","COLLIOURE":"66","CORBERE LES CABANES":"66","EYNE":"66","FELLUNS":"66","FONTRABIOUSE":"66","LA LLAGONNE":"66","MONTESQUIEU DES ALBERES":"66","OMS":"66","OPOUL PERILLOS":"66","OSSEJA":"66","PALAU DE CERDAGNE":"66","PASSA":"66","PEYRESTORTES":"66","PLANES":"66","PLANEZES":"66","PORTE PUYMORENS":"66","PORT VENDRES":"66","PRUGNANES":"66","PRUNET ET BELPUIG":"66","ST JEAN LASSEILLE":"66","ST LAURENT DE LA SALANQUE":"66","SANSA":"66","SOURNIA":"66","TARGASSONNE":"66","TERRATS":"66","THEZA":"66","THUIR":"66","TRILLA":"66","TROUILLAS":"66","VALCEBOLLERE":"66","VILLEMOLAQUE":"66","RENNEPONT":"52","RIAUCOURT":"52","RIMAUCOURT":"52","RIVIERE LES FOSSES":"52","ROUECOURT":"52","ROUVRES SUR AUBE":"52","ST BLIN":"52","ST VALLIER SUR MARNE":"52","SAUDRON":"52","SEMILLY":"52","SOYERS":"52","THIVET":"52","TREIX":"52","VERSEILLES LE BAS":"52","VILLARS EN AZOIS":"52","VIVEY":"52","VOILLECOMTE":"52","VOISEY":"52","ANDOUILLE":"53","ARON":"53","LA BACONNIERE":"53","BLANDOUET":"53","CHALONS DU MAINE":"53","CHAMPGENETEUX":"53","CHEMERE LE ROI":"53","COSMES":"53","COUESMES VAUCE":"53","DAON":"53","DENAZE":"53","GENNES SUR GLAIZE":"53","JUVIGNE":"53","LIVET":"53","LIVRE LA TOUCHE":"53","LOIRON RUILLE":"53","MADRE":"53","MEE":"53","MESLAY DU MAINE":"53","MONTSURS":"53","PONTMAIN":"53","RENAZE":"53","LE RIBAY":"53","RUILLE FROID FONDS":"53","SACE":"53","ST CENERE":"53","ST CHARLES LA FORET":"53","ST DENIS D ANJOU":"53","ST FRAIMBAULT DE PRIERES":"53","STE GEMMES LE ROBERT":"53","ST GEORGES LE FLECHARD":"53","ST MICHEL DE FEINS":"53","ST PIERRE SUR ORTHE":"53","ST QUENTIN LES ANGES":"53","STE SUZANNE ET CHAMMES":"53","SAULGES":"53","THUBOEUF":"53","AGINCOURT":"54","ANDERNY":"54","ATHIENVILLE":"54","AUTREVILLE SUR MOSELLE":"54","AZERAILLES":"54","BASLIEUX":"54","BAUZEMONT":"54","BAZAILLES":"54","BEUVEZIN":"54","BEZAUMONT":"54","BICQUELEY":"54","BIONVILLE":"54","BOUCQ":"54","BOUXIERES SOUS FROIDMONT":"54","BRALLEVILLE":"54","BURTHECOURT AUX CHENES":"54","CHAMBLEY BUSSIERES":"54","CHENIERES":"54","CROISMARE":"54","DOLCOURT":"54","DOMJEVIN":"54","DOMMARTEMONT":"54","ST ONDRAS":"38","ST PAUL DE VARCES":"38","ST PAUL LES MONESTIER":"38","CRETS EN BELLEDONNE":"38","ST PRIM":"38","ST QUENTIN SUR ISERE":"38","ST SULPICE DES RIVOIRES":"38","ST VINCENT DE MERCUZE":"38","SECHILIENNE":"38","SEMONS":"38","SINARD":"38","TIGNIEU JAMEYZIEU":"38","TREPT":"38","TULLINS":"38","VALJOUFFREY":"38","VAUJANY":"38","VAULNAVEYS LE BAS":"38","VAULNAVEYS LE HAUT":"38","VELANNE":"38","VENERIEU":"38","VENOSC":"38","VILLARD ST CHRISTOPHE":"38","VILLETTE DE VIENNE":"38","VOUREY":"38","ANNOIRE":"39","ARBOIS":"39","LA CHAILLEUSE":"39","AUXANGE":"39","BARESIA SUR L AIN":"39","BAVERANS":"39","BOISSIA":"39","BONLIEU":"39","BONNEFONTAINE":"39","BORNAY":"39","LES BOUCHOUX":"39","BRERY":"39","CERNIEBAUD":"39","LA CHAPELLE SUR FURIEUSE":"39","CHARCIER":"39","LA CHASSAGNE":"39","LA CHATELAINE":"39","CHATEL DE JOUX":"39","CHAUMERGY":"39","CHAUSSENANS":"39","CHEMILLA":"39","CHEVREAUX":"39","CHILLY SUR SALINS":"39","CLAIRVAUX LES LACS":"39","COGNA":"39","COMMENAILLES":"39","CONLIEGE":"39","CONTE":"39","CRESSIA":"39","CUTTURA":"39","CUVIER":"39","DAMMARTIN MARPAIN":"39","LES DEUX FAYS":"39","DOLE":"39","DOMBLANS":"39","DOYE":"39","DRAMELAY":"39","ECRILLE":"39","VAL D EPY":"39","LA FERTE":"39","GENDREY":"39","GERAISE":"39","GERUGE":"39","GRANGE DE VAIVRE":"39","LES HAYS":"39","LAC DES ROUGES TRUITES":"39","LARGILLAY MARSONNAY":"39","LA LATETTE":"39","LAVIGNY":"39","LECT":"39","LONGCOCHON":"39","LOULLE":"39","MAISOD":"39","MANTRY":"39","MARNEZIA":"39","MATHENAY":"39","MENETRU LE VIGNOBLE":"39","MEUSSIA":"39","MIGNOVILLARD":"39","MONAY":"39","MONTBARREY":"39","MONTIGNY SUR L AIN":"39","MONTMIREY LA VILLE":"39","MONTMIREY LE CHATEAU":"39","HAUTS DE BIENNE":"39","LES TROIS CHATEAUX":"39","NANCUISE":"39","NEUBLANS ABERGEMENT":"39","ROCHECHINARD":"26","LA ROCHE DE GLUN":"26","ROUSSIEUX":"26","ST CHRISTOPHE ET LE LARIS":"26","ST DIZIER EN DIOIS":"26","ST DONAT SUR L HERBASSE":"26","STE EULALIE EN ROYANS":"26","ST FERREOL TRENTE PAS":"26","ST GERVAIS SUR ROUBION":"26","STE JALLE":"26","ST LAURENT EN ROYANS":"26","ST MARTIN D AOUT":"26","ST MICHEL SUR SAVASSE":"26","ST NAZAIRE LE DESERT":"26","ST RESTITUT":"26","ST UZE":"26","SAOU":"26","TEYSSIERES":"26","TRESCHENU CREYERS":"26","VALDROME":"26","VALOUSE":"26","VERCLAUSE":"26","GRANGES LES BEAUMONT":"26","ACON":"27","AILLY":"27","AMECOURT":"27","AMFREVILLE SUR ITON":"27","LES ANDELYS":"27","LE VAL D HAZEY":"27","BACQUEVILLE":"27","BAILLEUL LA VALLEE":"27","LES BAUX DE BRETEUIL":"27","BAZINCOURT SUR EPTE":"27","BEAUBRAY":"27","MESNIL EN OUCHE":"27","BERNOUVILLE":"27","BERTHOUVILLE":"27","BERVILLE SUR MER":"27","BOIS ANZERAY":"27","BOISSET LES PREVANCHES":"27","BOISSEY LE CHATEL":"27","LA BONNEVILLE SUR ITON":"27","FLANCOURT CRESCY EN ROUMOIS":"27","BOSC RENOULT EN ROUMOIS":"27","BOSNORMAND":"27","BOULLEVILLE":"27","BOURG ACHARD":"27","BOURNAINVILLE FAVEROLLES":"27","BREUILPONT":"27","CAPELLE LES GRANDS":"27","CAUGE":"27","CHAMBLAC":"27","CHATEAU SUR EPTE":"27","CHAUVINCOURT PROVEMONT":"27","CHERONVILLIERS":"27","MARBOIS":"27","CIERREY":"27","LE CORMIER":"27","CORNEVILLE LA FOUQUETIERE":"27","CRESTOT":"27","CRIQUEBEUF SUR SEINE":"27","LA CROISILLE":"27","CLEF VALLEE D EURE":"27","DOUAINS":"27","DOUDEAUVILLE EN VEXIN":"27","DOUVILLE SUR ANDELLE":"27","ECARDENVILLE LA CAMPAGNE":"27","VEXIN SUR EPTE":"27","ECOUIS":"27","ECQUETOT":"27","EPREVILLE EN LIEUVIN":"27","ETREPAGNY":"27","EVREUX":"27","FAUVILLE":"27","FERRIERES ST HILAIRE":"27","FEUGUEROLLES":"27","FIQUEFLEUR EQUAINVILLE":"27","FLEURY LA FORET":"27","FLIPOU":"27","FONTAINE LA LOUVET":"27","FOUCRAINVILLE":"27","FOURMETOT":"27","IGON":"64","ISSOR":"64","ITXASSOU":"64","LAA MONDRANS":"64","LA BASTIDE CLAIRENCE":"64","LABASTIDE MONREJEAU":"64","LAHONCE":"64","LANTABAT":"64","LAROIN":"64","LARRAU":"64","LARRIBAR SORHAPURU":"64","LASCLAVERIES":"64","LASSEUBE":"64","LEE":"64","LEES ATHAS":"64","LESPIELLE":"64","LOUHOSSOA":"64","LOURENTIES":"64","LOUVIE SOUBIRON":"64","LUCARRE":"64","LUCGARIER":"64","MEHARIN":"64","MIREPEIX":"64","MOMAS":"64","MONTAGUT":"64","NABAS":"64","NARCASTET":"64","NARP":"64","OLORON STE MARIE":"64","OS MARSILLON":"64","OSSES":"64","OUILLON":"64","PARBAYSE":"64","PARDIES":"64","PARDIES PIETAT":"64","POEY DE LESCAR":"64","POEY D OLORON":"64","POMPS":"64","PONTACQ":"64","PORTET":"64","PRECHACQ JOSBAIG":"64","PRECILHON":"64","ST LAURENT BRETAGNE":"64","ST PE DE LEREN":"64","SARE":"64","SAUGUIS ST ETIENNE":"64","SAUVELADE":"64","SEDZERE":"64","SERRES CASTET":"64","SUS":"64","TABAILLE USQUAIN":"64","UHART CIZE":"64","UHART MIXE":"64","VIALER":"64","ALLIER":"65","ANCIZAN":"65","ANERES":"65","ANTIST":"65","ARCIZAC ADOUR":"65","ARCIZAC EZ ANGLES":"65","ARGELES BAGNERES":"65","ARRODETS":"65","ARTALENS SOUIN":"65","AVEZAC PRAT LAHITTE":"65","AYROS ARBOUIX":"65","BANIOS":"65","BAREILLES":"65","BAZET":"65","BAZORDAN":"65","BAZUS AURE":"65","BAZUS NESTE":"65","BERNAC DESSUS":"65","BONNEFONT":"65","BONREPOS":"65","BOUILH PEREUILH":"65","BOURG DE BIGORRE":"65","CASTELNAU RIVIERE BASSE":"65","CASTERETS":"65","CIEUTAT":"65","CRECHETS":"65","DEVEZE":"65","ESCAUNETS":"65","ESCOUBES POUTS":"65","ESPIEILH":"65","ESTAMPURES":"65","ESTARVIELLE":"65","ESTIRAC":"65","FRECHET AURE":"65","FRECHOU FRECHET":"65","VILLENEUVE DE LA RAHO":"66","VIVES":"66","ACHENHEIM":"67","ANDLAU":"67","AUENHEIM":"67","AVOLSHEIM":"67","BALBRONN":"67","BASSEMBERG":"67","BIETLENHEIM":"67","BILWISHEIM":"67","BISCHOFFSHEIM":"67","BITSCHHOFFEN":"67","BLAESHEIM":"67","BOOFZHEIM":"67","BOURG BRUCHE":"67","BOURGHEIM":"67","BREITENBACH":"67","LA BROQUE":"67","CLEEBOURG":"67","DANGOLSHEIM":"67","DAUBENSAND":"67","DETTWILLER":"67","DIEFFENBACH AU VAL":"67","DIEFFENBACH LES WOERTH":"67","DORLISHEIM":"67","DOSSENHEIM KOCHERSBERG":"67","DOSSENHEIM SUR ZINSEL":"67","ELSENHEIM":"67","ENGWILLER":"67","ERGERSHEIM":"67","FEGERSHEIM":"67","FLEXBOURG":"67","GAMBSHEIM":"67","GOERLINGEN":"67","GOERSDORF":"67","GRASSENDORF":"67","GUNDERSHOFFEN":"67","GUNSTETT":"67","HARSKIRCHEN":"67","HATTMATT":"67","HEIDOLSHEIM":"67","HEILIGENSTEIN":"67","HENGWILLER":"67","HERRLISHEIM":"67","HINSINGEN":"67","HIRSCHLAND":"67","HOCHSTETT":"67","HOENHEIM":"67","HOLTZHEIM":"67","HUTTENDORF":"67","JETTERSWILLER":"67","KUTZENHAUSEN":"67","LAMPERTSLOCH":"67","LOCHWILLER":"67","MAENNOLSHEIM":"67","MATZENHEIM":"67","MOMMENHEIM":"67","MONSWILLER":"67","MUNCHHAUSEN":"67","MUTZENHOUSE":"67","NORDHOUSE":"67","NOTHALTEN":"67","OBERHOFFEN SUR MODER":"67","OBERMODERN ZUTZENDORF":"67","SEEBACH":"67","OLWISHEIM":"67","OSTHOFFEN":"67","LA PETITE PIERRE":"67","RHINAU":"67","RIMSDORF":"67","ROPPENHEIM":"67","ROSSFELD":"67","ROTHBACH":"67","SAASENHEIM":"67","SCHALKENDORF":"67","SCHNERSHEIM":"67","SCHOENENBOURG":"67","SCHOPPERTEN":"67","SCHWINDRATZHEIM":"67","SCHWOBSHEIM":"67","SERMERSHEIM":"67","SIEWILLER":"67","DOMMARTIN LES TOUL":"54","DOMMARTIN SOUS AMANCE":"54","DOMPRIX":"54","DOMPTAIL EN L AIR":"54","DONCOURT LES CONFLANS":"54","EINVAUX":"54","ERBEVILLER SUR AMEZULE":"54","ESSEY LES NANCY":"54","FEY EN HAYE":"54","FLIREY":"54","FORCELLES SOUS GUGNEY":"54","FREMONVILLE":"54","GERBECOURT ET HAPLEMONT":"54","GLONVILLE":"54","GORCY":"54","GYE":"54","HABLAINVILLE":"54","HAMONVILLE":"54","HARBOUEY":"54","HEILLECOURT":"54","HOUDREVILLE":"54","HOUSSEVILLE":"54","JAILLON":"54","JARNY":"54","JEANDELIZE":"54","JEZAINVILLE":"54","JOPPECOURT":"54","LAITRE SOUS AMANCE":"54","LANEUVEVILLE AUX BOIS":"54","LESMENILS":"54","LONGWY":"54","LOREY":"54","MAIXE":"54","MALLELOY":"54","MARAINVILLER":"54","MAXEVILLE":"54","MESSEIN":"54","MIGNEVILLE":"54","MONCEL LES LUNEVILLE":"54","MONTIGNY SUR CHIERS":"54","MOUAVILLE":"54","MOYEN":"54","NEUVES MAISONS":"54","NORROY LES PONT A MOUSSON":"54","OLLEY":"54","PAGNEY DERRIERE BARINE":"54","PIERRE LA TREICHE":"54","PONT A MOUSSON":"54","PRAYE":"54","PREUTIN HIGNY":"54","PUXIEUX":"54","QUEVILLONCOURT":"54","ROUVES":"54","ROVILLE DEVANT BAYON":"54","ROZELIEURES":"54","SAFFAIS":"54","STE POLE":"54","ST REMY AUX BOIS":"54","SAULXEROTTE":"54","SEICHEPREY":"54","SIONVILLER":"54","PUSSIGNY":"37","RIGNY USSE":"37","SACHE":"37","ST ANTOINE DU ROCHER":"37","ST BRANCHS":"37","ST CHRISTOPHE SUR LE NAIS":"37","ST GENOUPH":"37","ST MICHEL SUR LOIRE":"37","ST OUEN LES VIGNES":"37","ST PATERNE RACAN":"37","ST PIERRE DES CORPS":"37","SEPMES":"37","VILLAINES LES ROCHERS":"37","LA VILLE AUX DAMES":"37","VILLEDOMER":"37","LES ADRETS":"38","AMBEL":"38","AOSTE":"38","LES AVENIERES VEYRINS THUELLIN":"38","NEUVILLEY":"39","NEVY LES DOLE":"39","NEVY SUR SEILLE":"39","ONGLIERES":"39","ONOZ":"39","OUSSIERES":"39","LE PASQUIER":"39","LES PIARDS":"39","PLAINOISEAU":"39","LES PLANCHES PRES ARBOIS":"39","PLEURE":"39","PLUMONT":"39","PREMANON":"39","PRETIN":"39","PUPILLIN":"39","RAVILLOLES":"39","LES REPOTS":"39","REVIGNY":"39","LA RIXOUSE":"39","ROTHONAY":"39","LES ROUSSES":"39","ST MAURICE CRILLAT":"39","SARROGNA":"39","SELIGNEY":"39","SEPTMONCEL":"39","SERGENON":"39","SIROD":"39","VALFIN SUR VALOUSE":"39","VERGES":"39","VERS EN MONTAGNE":"39","VESCLES":"39","VILLARD ST SAUVEUR":"39","VILLETTE LES DOLE":"39","VINCENT FROIDEVILLE":"39","ANGRESSE":"40","ARBOUCAVE":"40","AUBAGNAN":"40","BAHUS SOUBIRAN":"40","BASCONS":"40","BASTENNES":"40","BERGOUEY":"40","BIAUDOS":"40","BOSTENS":"40","BRETAGNE DE MARSAN":"40","BROCAS":"40","CAMPET ET LAMOLERE":"40","CASTELNAU CHALOSSE":"40","CASTEL SARRAZIN":"40","CAUNA":"40","COMMENSACQ":"40","DAX":"40","DONZACQ":"40","ESCOURCE":"40","GAAS":"40","GAREIN":"40","GEAUNE":"40","GIBRET":"40","HAGETMAU":"40","HASTINGUES":"40","HAURIET":"40","HORSARRIEU":"40","LABENNE":"40","LAGLORIEUSE":"40","LALUQUE":"40","LARBEY":"40","LARRIVIERE ST SAVIN":"40","LAUREDE":"40","LE LEUY":"40","LEVIGNACQ":"40","LIT ET MIXE":"40","MAURRIN":"40","MEES":"40","MOMUY":"40","MONTFORT EN CHALOSSE":"40","ONARD":"40","ONDRES":"40","ONESSE LAHARIE":"40","PECORADE":"40","POYARTIN":"40","ST JULIEN EN BORN":"40","ST PANDELON":"40","ST VINCENT DE TYROSSE":"40","SAUBRIGUES":"40","SEIGNOSSE":"40","SOLFERINO":"40","SOUPROSSE":"40","FRESNE CAUVERVILLE":"27","FRESNE L ARCHEVEQUE":"27","LA BARONNIE":"27","GARENNES SUR EURE":"27","LA GOULAFRIERE":"27","GRAVIGNY":"27","GROSSOEUVRE":"27","LE BOSC DU THEIL":"27","GUISENIERS":"27","HAUVILLE":"27","LA HAYE DE CALLEVILLE":"27","LA HAYE LE COMTE":"27","LA HAYE MALHERBE":"27","HOULBEC COCHEREL":"27","IGOVILLE":"27","ILLIERS L EVEQUE":"27","IVRY LA BATAILLE":"27","JOUY SUR EURE":"27","LA LANDE ST LEGER":"27","LETTEGUIVES":"27","LIEUREY":"27","LORLEAU":"27","MARAIS VERNIER":"27","BUIS SUR DAMVILLE":"27","MORSAN":"27","NAGEL SEEZ MESNIL":"27","LA NEUVE LYRE":"27","LA NEUVILLE DU BOSC":"27","NEUVILLE SUR AUTHOU":"27","NOGENT LE SEC":"27","NOTRE DAME DU HAMEL":"27","PARVILLE":"27","PERRUEL":"27","PITRES":"27","LE PLESSIS HEBERT":"27","PONT DE L ARCHE":"27","POSES":"27","PUCHAY":"27","RADEPONT":"27","ROMAN":"27","LA ROQUETTE":"27","ST AGNAN DE CERNIERES":"27","ST AUBIN DE SCELLON":"27","ST ETIENNE L ALLIER":"27","ST JEAN DU THENNEY":"27","STE MARIE DE VATIMESNIL":"27","ST OUEN DES CHAMPS":"27","ST SAMSON DE LA ROQUE":"27","ST SEBASTIEN DE MORSENT":"27","ST VIGOR":"27","SERQUIGNY":"27","THIBOUVILLE":"27","LE THUIT":"27","LE THUIT DE L OISON":"27","LE TILLEUL OTHON":"27","TILLIERES SUR AVRE":"27","TOUTAINVILLE":"27","LA VACHERIE":"27","VANDRIMARE":"27","VENABLES":"27","VERNEUIL SUR AVRE":"27","VERNEUSSES":"27","LA VIEILLE LYRE":"27","VOISCREVILLE":"27","VRAIVILLE":"27","ARDELLES":"28","AUNEAU BLEURY ST SYMPHORIEN":"28","BAILLEAU ARMENONVILLE":"28","BAUDREVILLE":"28","BAZOCHES LES HAUTES":"28","BEAUMONT LES AUTELS":"28","BEROU LA MULOTIERE":"28","BEVILLE LE COMTE":"28","BOISGASSON":"28","BOISVILLE LA ST PERE":"28","BOUGLAINVAL":"28","BRECHAMPS":"28","BREZOLLES":"28","BROU":"28","LA CHAPELLE FORAINVILLIERS":"28","CHAPELLE ROYALE":"28","CHARPONT":"28","GAYAN":"65","GERDE":"65","GERMS SUR L OUSSOUET":"65","GEU":"65","HACHAN":"65","HECHES":"65","JARRET":"65","JEZEAU":"65","JUILLAN":"65","JULOS":"65","JUNCALAS":"65","LABORDE":"65","ARRAYOU LAHITTE":"65","LAMEAC":"65","LANESPEDE":"65","LAU BALAGNAS":"65","LAYRISSE":"65","LESPOUEY":"65","LHEZ":"65","LIES":"65","LOMBRES":"65","LOMNE":"65","MADIRAN":"65","MAUBOURGUET":"65","MAZERES DE NESTE":"65","MONLEON MAGNOAC":"65","OLEAC DEBAT":"65","ORDIZAN":"65","ORIEUX":"65","ORIGNAC":"65","PEYRIGUERE":"65","PEYRUN":"65","POUMAROUS":"65","PUJO":"65","SADOURNIN":"65","ST LEZER":"65","ST PASTOUS":"65","SALIGOS":"65","SARLABOUS":"65","SARNIGUET":"65","SASSIS":"65","SEGUS":"65","SERE LANSO":"65","TILHOUSE":"65","TUZAGUET":"65","UGLAS":"65","UZ":"65","VIDOU":"65","VIEUZOS":"65","VIGER":"65","VIGNEC":"65","VILLELONGUE":"65","VILLENAVE PRES MARSAC":"65","VISCOS":"65","VISKER":"65","BAREGES":"65","ANGOUSTRINE VILLENEUVE DES ESCALDES":"66","ARLES SUR TECH":"66","BAHO":"66","BAIXAS":"66","BANYULS SUR MER":"66","BOLQUERE":"66","BOULETERNERE":"66","LA CABANASSE":"66","CAIXAS":"66","CAMPOME":"66","CANAVEILLES":"66","CAUDIES DE CONFLENT":"66","CODALET":"66","ELNE":"66","ENVEITG":"66","ESTAVAR":"66","ESTOHER":"66","FILLOLS":"66","FUILLA":"66","ILLE SUR TET":"66","LATOUR DE CAROL":"66","LESQUERDE":"66","LLO":"66","MANTET":"66","MARQUIXANES":"66","MAUREILLAS LAS ILLAS":"66","MONTESCOT":"66","FONT ROMEU ODEILLO VIA":"66","OLETTE":"66","PALAU DEL VIDRE":"66","SOULTZ LES BAINS":"67","STEIGE":"67","STRUTH":"67","STUNDWILLER":"67","SUNDHOUSE":"67","URMATT":"67","UTTENHEIM":"67","VALFF":"67","LA VANCELLE":"67","WALBOURG":"67","WALTENHEIM SUR ZORN":"67","LA WANTZENAU":"67","WASSELONNE":"67","WESTHOUSE":"67","WESTHOUSE MARMOUTIER":"67","WICKERSHEIM WILSHAUSEN":"67","WINGERSHEIM LES QUATRE BANS":"67","WIWERSHEIM":"67","ZITTERSHEIM":"67","BALGAU":"68","BALLERSDORF":"68","BALSCHWILLER":"68","BARTENHEIM":"68","BEBLENHEIM":"68","BELLEMAGNY":"68","BERGHOLTZZELL":"68","BITSCHWILLER LES THANN":"68","BLODELSHEIM":"68","BOLLWILLER":"68","LE BONHOMME":"68","BRUEBACH":"68","BUETHWILLER":"68","BURNHAUPT LE HAUT":"68","GREZIEUX LE FROMENTAL":"42","LEIGNEUX":"42","LEZIGNEUX":"42","MACLAS":"42","MALLEVAL":"42","MARCENOD":"42","NEAUX":"42","NERONDE":"42","NEULISE":"42","PANISSIERES":"42","PERIGNEUX":"42","PINAY":"42","PLANFOY":"42","POUILLY LES FEURS":"42","POUILLY LES NONAINS":"42","POUILLY SOUS CHARLIEU":"42","RIORGES":"42","ROISEY":"42","SAIL LES BAINS":"42","ST ALBAN LES EAUX":"42","ST ANDRE LE PUY":"42","ST BONNET LE COURREAU":"42","ST GENEST LERPT":"42","ST HAON LE VIEUX":"42","ST LAURENT ROCHEFORT":"42","ST MARCEL DE FELINES":"42","ST MARCEL D URFE":"42","ST MAURICE EN GOURGOIS":"42","ST PIERRE LA NOAILLE":"42","ST POLGUES":"42","ST PRIEST LA PRUGNE":"42","ST PRIEST LA VETRE":"42","ST JUST ST RAMBERT":"42","ST REGIS DU COIN":"42","ST ROMAIN LA MOTTE":"42","ST VINCENT DE BOISSET":"42","SEVELINGES":"42","TARENTAISE":"42","LA TOUR EN JAREZ":"42","UNIAS":"42","UNIEUX":"42","VALFLEURY":"42","VEAUCHETTE":"42","VILLEMONTAIS":"42","ARSAC EN VELAY":"43","VISSAC AUTEYRAC":"43","AUTRAC":"43","AUZON":"43","BELLEVUE LA MONTAGNE":"43","BERBEZIT":"43","LE BOUCHET ST NICOLAS":"43","LA CHAISE DIEU":"43","CHAMPAGNAC LE VIEUX":"43","CHANALEILLES":"43","CHANTEUGES":"43","CHASPINHAC":"43","AVIGNONET":"38","BEAUCROISSANT":"38","BEVENAIS":"38","BILIEU":"38","CESSIEU":"38","LE CHAMP PRES FROGES":"38","LA CHAPELLE DE SURIEU":"38","CHARANCIEU":"38","CHARAVINES":"38","CHATEAU BERNARD":"38","CHATTE":"38","CHEYSSIEU":"38","CHOZEAU":"38","ST MARTIN DE LA CLUZE":"38","CORDEAC":"38","CORPS":"38","LA COTE ST ANDRE":"38","DIZIMIEU":"38","DOLOMIEU":"38","ENGINS":"38","EYBENS":"38","FAVERGES DE LA TOUR":"38","LA FLACHERE":"38","FLACHERES":"38","FOUR":"38","FROGES":"38","GRENOBLE":"38","HUEZ":"38","IZEAUX":"38","IZERON":"38","LIEUDIEU":"38","MEYRIEU LES ETANGS":"38","MIRIBEL LES ECHELLES":"38","MONTCARRA":"38","MONTFALCON":"38","MOTTIER":"38","LE MOUTARET":"38","LA MURE":"38","OZ":"38","PASSINS":"38","LE PEAGE DE ROUSSILLON":"38","LE PERIER":"38","PONT DE CHERUY":"38","PRESSINS":"38","QUINCIEU":"38","ROVON":"38","ROYAS":"38","ST ALBIN DE VAULSERRE":"38","ST ANDRE EN ROYANS":"38","ST ANTOINE L ABBAYE":"38","ST AUPRE":"38","ST CHEF":"38","ST CHRISTOPHE EN OISANS":"38","ST DIDIER DE BIZONNES":"38","ST GEOIRE EN VALDAINE":"38","ST GEORGES D ESPERANCHE":"38","ST JUST CHALEYSSIN":"38","ST MARCEL BEL ACCUEIL":"38","ST ROMANS":"38","LA SALLE EN BEAUMONT":"38","LE SAPPEY EN CHARTREUSE":"38","SATOLAS ET BONCE":"38","SEPTEME":"38","SEYSSINET PARISET":"38","SILLANS":"38","SOLEYMIEU":"38","LA SONE":"38","TECHE":"38","TREFFORT":"38","TREMINIS":"38","VASSELIN":"38","VAULX MILIEU":"38","VERNAS":"38","VEUREY VOROIZE":"38","VEYSSILIEU":"38","VIGNIEU":"38","VILLARD RECULAS":"38","VILLARD REYMOND":"38","VILLETTE D ANTHON":"38","VIRIEU":"38","VIRIVILLE":"38","SOUSTONS":"40","TALLER":"40","TARTAS":"40","UZA":"40","VIELLE TURSAN":"40","LE VIGNAU":"40","YGOS ST SATURNIN":"40","AUTAINVILLE":"41","AVERDON":"41","BEAUCHENE":"41","BLOIS":"41","CHAON":"41","LA CHAPELLE MONTMARTIN":"41","CHEMERY":"41","CHISSAY EN TOURAINE":"41","CHOUE":"41","CORMERAY":"41","LA FERTE IMBAULT":"41","FONTAINE RAOUL":"41","FRANCAY":"41","GOMBERGEAN":"41","LAMOTTE BEUVRON":"41","LANCE":"41","LESTIOU":"41","MAVES":"41","MENARS":"41","MEUSNES":"41","MUR DE SOLOGNE":"41","NOURRAY":"41","OISLY":"41","BEAUCE LA ROMAINE":"41","RAHART":"41","ROCE":"41","ST GERVAIS LA FORET":"41","ST LAURENT NOUAN":"41","ST LUBIN EN VERGONNOIS":"41","SALBRIS":"41","SAMBIN":"41","SELOMMES":"41","SOINGS EN SOLOGNE":"41","SOUDAY":"41","SOUESMES":"41","VILLAVARD":"41","VILLEBOUT":"41","VILLEROMAIN":"41","ANDREZIEUX BOUTHEON":"42","BALBIGNY":"42","BARD":"42","BELLEGARDE EN FOREZ":"42","BELLEROCHE":"42","BOISSET ST PRIEST":"42","BRIENNON":"42","BURDIGNES":"42","CHAZELLES SUR LAVIEU":"42","COUTOUVRE":"42","CRAINTILLEUX":"42","DARGOIRE":"42","ESSERTINES EN DONZY":"42","FARNAY":"42","LA FOUILLOUSE":"42","JARNOSSE":"42","JONZIEUX":"42","MABLY":"42","MACHEZAL":"42","MARGERIE CHANTAGRET":"42","MONTVERDUN":"42","NOIRETABLE":"42","OUCHES":"42","LA PACAUDIERE":"42","PELUSSIN":"42","PONCINS":"42","PRECIEUX":"42","ROANNE":"42","STE AGATHE EN DONZY":"42","ST BONNET LE CHATEAU":"42","ST CHAMOND":"42","ST CHRISTO EN JAREZ":"42","STE COLOMBE SUR GAND":"42","ST CYR DE VALORGES":"42","ST ETIENNE":"42","ST GEORGES DE BAROILLE":"42","ST GEORGES HAUTE VILLE":"42","CHARRAY":"28","CINTRAY":"28","CIVRY":"28","CORANCEZ":"28","CRUCEY VILLAGES":"28","DAMMARIE":"28","DAMPIERRE SOUS BROU":"28","DIGNY":"28","ECROSNES":"28","ERMENONVILLE LA PETITE":"28","LES ETILLEUX":"28","FONTAINE LES RIBOUTS":"28","FONTAINE SIMON":"28","LA FRAMBOISIERE":"28","FRANCOURVILLE":"28","FRUNCE":"28","LEVESVILLE LA CHENARD":"28","LOIGNY LA BATAILLE":"28","LORMAYE":"28","LUIGNY":"28","LURAY":"28","LUTZ EN DUNOIS":"28","LA MANCELIERE":"28","MEVOISINS":"28","MIERMAIGNE":"28","MONDONVILLE ST JEAN":"28","MONTIGNY SUR AVRE":"28","MONTLANDON":"28","MORAINVILLE":"28","MORIERS":"28","MOULHARD":"28","NOGENT LE ROI":"28","NOGENT LE ROTROU":"28","OINVILLE ST LIPHARD":"28","LA PUISAYE":"28","ROUVRAY ST DENIS":"28","ST DENIS LES PONTS":"28","ST JEAN DE REBERVILLIERS":"28","ST OUEN MARCHEFROY":"28","ST PIAT":"28","ST PREST":"28","THIMERT GATELLES":"28","TOURY":"28","TRANCRAINVILLE":"28","TREMBLAY LES VILLAGES":"28","EOLE EN BEAUCE":"28","VITRAY EN BEAUCE":"28","YERMENONVILLE":"28","YMERAY":"28","AUDIERNE":"29","CAMARET SUR MER":"29","LE CLOITRE PLEYBEN":"29","COLLOREC":"29","LE CONQUET":"29","CORAY":"29","LA FEUILLEE":"29","LE FOLGOET":"29","LA FOREST LANDERNEAU":"29","FOUESNANT":"29","HENVIC":"29","KERLAZ":"29","KERLOUAN":"29","LANDEVENNEC":"29","LANHOUARNEAU":"29","LANNILIS":"29","LENNON":"29","LOC EGUINER":"29","LOCMARIA PLOUZANE":"29","LOCRONAN":"29","LOGONNA DAOULAS":"29","LOTHEY":"29","LA MARTYRE":"29","OUESSANT":"29","PLOMEUR":"29","PLONEOUR LANVERN":"29","PLONEVEZ DU FAOU":"29","PLOUEZOC H":"29","PLOUGUIN":"29","PLOUNEOUR MENEZ":"29","PLOUNEVEZEL":"29","PLOUNEVEZ LOCHRIST":"29","PLOURIN LES MORLAIX":"29","PLOUZEVEDE":"29","QUERRIEN":"29","ST DERRIEN":"29","ST ELOY":"29","ST JEAN TROLIMON":"29","ST THEGONNEC LOC EGUINER":"29","ST THONAN":"29","TOURCH":"29","LE PERTHUS":"66","PIA":"66","PONTEILLA":"66","PUYVALADOR":"66","PY":"66","RIGARDA":"66","STE COLOMBE DE LA COMMANDERIE":"66","ST MARTIN DE FENOUILLET":"66","SALEILLES":"66","SALSES LE CHATEAU":"66","SAUTO":"66","SOUANYAS":"66","TAULIS":"66","TOULOUGES":"66","TRESSERRE":"66","UR":"66","VALMANYA":"66","VILLELONGUE DE LA SALANQUE":"66","VILLENEUVE LA RIVIERE":"66","VINCA":"66","VINGRAU":"66","ADAMSWILLER":"67","SOMMERAU":"67","ALTECKENDORF":"67","ALTORF":"67","ARTOLSHEIM":"67","ASSWILLER":"67","BAERENDORF":"67","BERG":"67","BERGBIETEN":"67","BERSTHEIM":"67","BISSERT":"67","BLANCHERUPT":"67","BOSSENDORF":"67","BREITENAU":"67","BRUMATH":"67","BUSWILLER":"67","BUST":"67","CLIMBACH":"67","COLROY LA ROCHE":"67","DIEBOLSHEIM":"67","DIEMERINGEN":"67","DRACHENBRONN BIRLENBACH":"67","DUNTZENHEIM":"67","DURRENBACH":"67","DUTTLENHEIM":"67","ECKARTSWILLER":"67","EICHHOFFEN":"67","ESCHWILLER":"67","EYWILLER":"67","FOUCHY":"67","GRIES":"67","HAGUENAU":"67","HANGENBIETEN":"67","HEGENEY":"67","HERBSHEIM":"67","HUNSPACH":"67","INGOLSHEIM":"67","ITTENHEIM":"67","NEUGARTHEIM ITTLENHEIM":"67","KALTENHOUSE":"67","KERTZFELD":"67","KESSELDORF":"67","KIENHEIM":"67","KINTZHEIM":"67","KOGENHEIM":"67","KRAUTERGERSHEIM":"67","KRAUTWILLER":"67","LALAYE":"67","LAUBACH":"67","LEMBACH":"67","LOHR":"67","LUPSTEIN":"67","MAISONSGOUTTE":"67","MEMMELSHOFFEN":"67","MERKWILLER PECHELBRONN":"67","MITTELBERGHEIM":"67","MOLLKIRCH":"67","MORSCHWILLER":"67","MOTHERN":"67","MUHLBACH SUR BRUCHE":"67","MUTTERSHOLTZ":"67","NEEWILLER PRES LAUTERBOURG":"67","NEUWILLER LES SAVERNE":"67","NIEDERHASLACH":"67","NIEDERMODERN":"67","NIEDERSOULTZBACH":"67","OBENHEIM":"67","OBERDORF SPACHBACH":"67","OBERHOFFEN LES WISSEMBOURG":"67","CHASSAGNES":"43","CHAVANIAC LAFAYETTE":"43","CHOMELIX":"43","LA CHOMETTE":"43","CRAPONNE SUR ARZON":"43","CRONCE":"43","CUSSAC SUR LOIRE":"43","ESPALEM":"43","ESPLANTAS VAZEILLES":"43","FREYCENET LA TOUR":"43","LANGEAC":"43","LEOTOING":"43","LORLANGES":"43","MAZERAT AUROUZE":"43","MAZEYRAT D ALLIER":"43","MONISTROL D ALLIER":"43","PONT SALOMON":"43","PRESAILLES":"43","ST CHRISTOPHE SUR DOLAISON":"43","ST FERREOL D AUROURE":"43","ST GENEYS PRES ST PAULIEN":"43","ST GEORGES D AURAC":"43","ST GERMAIN LAPRADE":"43","ST HAON":"43","ST JULIEN CHAPTEUIL":"43","ST MAURICE DE LIGNON":"43","ST PAULIEN":"43","ST PIERRE EYNAC":"43","ST PRIVAT D ALLIER":"43","ST PRIVAT DU DRAGON":"43","STE SIGOLENE":"43","ST VENERAND":"43","ST VERT":"43","ST VICTOR MALESCOURS":"43","SALZUIT":"43","SAUGUES":"43","SOLIGNAC SUR LOIRE":"43","TENCE":"43","VALS LE CHASTEL":"43","LES VILLETTES":"43","CHAUMES EN RETZ":"44","AVESSAC":"44","BLAIN":"44","BOUEE":"44","BRAINS":"44","LE CELLIER":"44","CHATEAUBRIANT":"44","COUERON":"44","DREFFEAC":"44","GUERANDE":"44","LA HAIE FOUASSIERE":"44","HERBIGNAC":"44","INDRE":"44","JUIGNE DES MOUTIERS":"44","MAUVES SUR LOIRE":"44","LA MEILLERAYE DE BRETAGNE":"44","NANTES":"44","NOYAL SUR BRUTZ":"44","ORVAULT":"44","PANNECE":"44","PAULX":"44","PETIT MARS":"44","PIRIAC SUR MER":"44","PLESSE":"44","PONTCHATEAU":"44","PORNIC":"44","PUCEUL":"44","RIAILLE":"44","RUFFIGNE":"44","ST AIGNAN GRANDLIEU":"44","STE ANNE SUR BRIVET":"44","ST AUBIN DES CHATEAUX":"44","CORCOUE SUR LOGNE":"44","ST HILAIRE DE CLISSON":"44","ST JULIEN DE CONCELLES":"44","ST LEGER LES VIGNES":"44","ST MALO DE GUERSAC":"44","STE PAZANNE":"44","SION LES MINES":"44","TOUVOIS":"44","VIGNEUX DE BRETAGNE":"44","LA CHEVALLERAIS":"44","ADON":"45","AUGERVILLE LA RIVIERE":"45","BAZOCHES LES GALLERANDES":"45","BAZOCHES SUR LE BETZ":"45","BEAULIEU SUR LOIRE":"45","BOISMORAND":"45","VOREPPE":"38","ALIEZE":"39","ANDELOT MORVAL":"39","AUGERANS":"39","AUMUR":"39","BALANOD":"39","LA BALME D EPY":"39","BAUME LES MESSIEURS":"39","BERSAILLIN":"39","BIEF DES MAISONS":"39","BILLECUL":"39","BLETTERANS":"39","BLYE":"39","BOIS DE GAND":"39","BONNAUD":"39","CERNANS":"39","CEZIA":"39","CHAMBLAY":"39","CHAMPAGNE SUR LOUE":"39","CHAMPROUGIER":"39","CHANCIA":"39","LA CHARME":"39","CHASSAL":"39","NANCHEZ":"39","CHAUX CHAMPAGNY":"39","CHENE BERNARD":"39","CHEVIGNY":"39","CLUCY":"39","COISERETTE":"39","COISIA":"39","COURLANS":"39","LE DESCHAUX":"39","ENTRE DEUX MONTS":"39","EQUEVILLON":"39","ETIVAL":"39","FAY EN MONTAGNE":"39","FOULENAY":"39","FRONTENAY":"39","GATEY":"39","GEVINGEY":"39","GILLOIS":"39","GIZIA":"39","GRANDE RIVIERE":"39","GROZON":"39","JOUHE":"39","LADOYE SUR SEILLE":"39","LAMOURA":"39","LAVANS LES DOLE":"39","LAVANS LES ST CLAUDE":"39","LEGNA":"39","LOUVATANGE":"39","LOUVENNE":"39","MARNOZ":"39","MENOTEY":"39","MESNOIS":"39","MIEGES":"39","MIERY":"39","MONNETAY":"39","MONTAGNA LE TEMPLIER":"39","MONTCUSEL":"39","MONTMOROT":"39","MONT SOUS VAUDREY":"39","MORBIER":"39","MOUCHARD":"39","NOZEROY":"39","OFFLANGES":"39","ORGELET":"39","PAGNEY":"39","PANNESSIERES":"39","PEINTRE":"39","LA PESSE":"39","PIMORIN":"39","PLASNE":"39","PLENISETTE":"39","PORT LESNEY":"39","ST HILAIRE SOUS CHARLIEU":"42","ST JEAN ST MAURICE SUR LOIRE":"42","ST JULIEN D ODDES":"42","ST JULIEN MOLIN MOLETTE":"42","ST MARCELLIN EN FOREZ":"42","ST MARTIN LA PLAINE":"42","ST MEDARD EN FOREZ":"42","ST PAUL DE VEZELIN":"42","ST PIERRE DE BOEUF":"42","ST PRIEST LA ROCHE":"42","ST RIRAND":"42","ST ROMAIN EN JAREZ":"42","ST THOMAS LA GARDE":"42","SURY LE COMTAL":"42","LA TOURETTE":"42","LA TUILIERE":"42","VEAUCHE":"42","VENDRANGES":"42","LA VERSANNE":"42","VILLEREST":"42","CHAUSSETERRE":"42","AIGUILHE":"43","ALLEYRAC":"43","ARLET":"43","AUBAZAT":"43","BAS EN BASSET":"43","BOURNONCLE ST PIERRE":"43","BRIVES CHARENSAC":"43","CERZAT":"43","CHADRON":"43","CHAMBEZON":"43","CHANIAT":"43","LA CHAPELLE BERTIN":"43","CHASTEL":"43","COHADE":"43","COLLAT":"43","COSTAROS":"43","COUBON":"43","CUBELLES":"43","FAY SUR LIGNON":"43","FERRUSSAC":"43","JAVAUGUES":"43","LEMPDES SUR ALLAGNON":"43","MALREVERS":"43","MALVALETTE":"43","LE MAS DE TENCE":"43","MAZET ST VOY":"43","MONTUSCLAT":"43","PEBRAC":"43","RAURET":"43","RETOURNAC":"43","RIOTORD":"43","ST DIDIER D ALLIER":"43","ST ETIENNE SUR BLESLE":"43","ST HOSTIEN":"43","ST JULIEN D ANCE":"43","ST JULIEN MOLHESABATE":"43","ST JUST PRES BRIOUDE":"43","ST PAL DE SENOUIRE":"43","ST PREJET D ALLIER":"43","ST ROMAIN LACHALM":"43","ST VICTOR SUR ARLANC":"43","ST VIDAL":"43","SANSSAC L EGLISE":"43","LA SEAUVE SUR SEMENE":"43","LES VASTRES":"43","VERGONGHEON":"43","VERNASSAL":"43","ABBARETZ":"44","CAMPBON":"44","DIVATTE SUR LOIRE":"44","LA CHAPELLE DES MARAIS":"44","CHAUVE":"44","CLISSON":"44","LE CROISIC":"44","DERVAL":"44","DONGES":"44","LA BAULE ESCOUBLAC":"44","FEGREAC":"44","LE GAVRE":"44","GETIGNE":"44","GRAND AUVERNE":"44","GUEMENE PENFAO":"44","LA LIMOUZINIERE":"44","MACHECOUL ST MEME":"44","MALVILLE":"44","MESQUER":"44","TREGLONOU":"29","TREMEOC":"29","TREOGAT":"29","TREOUERGAT":"29","ALLEGRE LES FUMADES":"30","AUBAIS":"30","AUJARGUES":"30","BERNIS":"30","BOUCOIRAN ET NOZIERES":"30","BOURDIC":"30","BRAGASSARGUES":"30","CAISSARGUES":"30","CHAMBORIGAUD":"30","CHUSCLAN":"30","CODOGNAN":"30","COLLIAS":"30","COLLORGUES":"30","CONGENIES":"30","CRESPIAN":"30","SEIGNY":"21","SEMUR EN AUXOIS":"21","LADOIX SERRIGNY":"21","SOIRANS":"21","SUSSEY":"21","TERREFONDREE":"21","TROUHANS":"21","TRUGNY":"21","VELARS SUR OUCHE":"21","VESVRES":"21","VEUXHAULLES SUR AUBE":"21","VILLARS FONTAINE":"21","VILLEBICHOT":"21","LA VILLENEUVE LES CONVERS":"21","VILLERS LES POTS":"21","VILLOTTE ST SEINE":"21","VILLOTTE SUR OURCE":"21","ANDEL":"22","BEGARD":"22","BERHET":"22","BOBITAL":"22","LE BODEO":"22","COADOUT":"22","COETLOGON":"22","LE MENE":"22","BINIC ETABLES SUR MER":"22","GLOMEL":"22","GUINGAMP":"22","HILLION":"22","KERMOROC H":"22","LANFAINS":"22","LANGOAT":"22","LANGUEUX":"22","LANLEFF":"22","LANMODEZ":"22","LANVALLAY":"22","LANVELLEC":"22","LEHON":"22","LOGUIVY PLOUGRAS":"22","MANTALLOT":"22","MELLIONNEC":"22","MERILLAC":"22","MOUSTERU":"22","MUR DE BRETAGNE":"22","PAIMPOL":"22","PLEBOULLE":"22","PLEDRAN":"22","PLEGUIEN":"22","FREHEL":"22","PLELO":"22","PLESSIX BALISSON":"22","PLESTAN":"22","PLEUBIAN":"22","PLOUAGAT":"22","PLOUARET":"22","PLOUBALAY":"22","PLOUBAZLANEC":"22","PLOUBEZRE":"22","PLOUER SUR RANCE":"22","PLOUHA":"22","PLOUMAGOAR":"22","PLOUMILLIAU":"22","PLOUNEVEZ MOEDEC":"22","PLOUVARA":"22","PLURIEN":"22","POMMERET":"22","POMMERIT JAUDY":"22","PORDIC":"22","LE QUIOU":"22","ST BRIEUC":"22","ST CARREUC":"22","ST CAST LE GUILDO":"22","OBERSCHAEFFOLSHEIM":"67","TACONNAY":"58","TINTURY":"58","TRESNAY":"58","TRONSANGES":"58","URZY":"58","VARENNES LES NARCY":"58","VIGNOL":"58","VILLAPOURCON":"58","VILLE LANGY":"58","AIBES":"59","ALLENNES LES MARAIS":"59","ANNEUX":"59","ARTRES":"59","ASSEVENT":"59","AULNOY LEZ VALENCIENNES":"59","AVELIN":"59","AVESNES LES AUBERT":"59","BANTIGNY":"59","BAS LIEU":"59","BEAUVOIS EN CAMBRESIS":"59","BETHENCOURT":"59","BEUGNIES":"59","BEUVRAGES":"59","BOURGHELLES":"59","BOUSSIERES EN CAMBRESIS":"59","BOUVINES":"59","BRILLON":"59","BUGNICOURT":"59","BUSIGNY":"59","CAGNONCLES":"59","CAPINGHEM":"59","CLAIRFAYTS":"59","CLARY":"59","COLLERET":"59","COMINES":"59","CRAYWICK":"59","DAMOUSIES":"59","DIMECHAUX":"59","DOIGNIES":"59","ECLAIBES":"59","EMERCHICOURT":"59","EPPE SAUVAGE":"59","ERQUINGHEM LE SEC":"59","ERQUINGHEM LYS":"59","FERIN":"59","FLINES LES MORTAGNE":"59","FONTAINE AU PIRE":"59","FRESNES SUR ESCAUT":"59","GHYVELDE":"59","GOMMEGNIES":"59","GRAND FAYT":"59","HASNON":"59","HASPRES":"59","HAUSSY":"59","HAUT LIEU":"59","HAYNECOURT":"59","HAZEBROUCK":"59","HELESMES":"59","HERIN":"59","HERLIES":"59","HOLQUE":"59","HONDEGHEM":"59","HONNECHY":"59","HOUTKERQUE":"59","LANDRECIES":"59","LECELLES":"59","LEFFRINCKOUCKE":"59","LEWARDE":"59","LEZENNES":"59","LIESSIES":"59","LILLE":"59","LOMPRET":"59","LOOBERGHE":"59","LOON PLAGE":"59","LYS LEZ LANNOY":"59","MALINCOURT":"59","MARCHIENNES":"59","MARCOING":"59","MARQUETTE EN OSTREVANT":"59","MASNY":"59","MAUROIS":"59","METEREN":"59","MONS EN PEVELE":"59","MONTIGNY EN OSTREVENT":"59","NEUF MESNIL":"59","BOISSEAUX":"45","BONDAROY":"45","BONNY SUR LOIRE":"45","BOUZONVILLE AUX BOIS":"45","BRAY EN VAL":"45","BUCY LE ROI":"45","CERCOTTES":"45","CHAINGY":"45","CHALETTE SUR LOING":"45","LA CHAPELLE ST SEPULCRE":"45","CHARSONVILLE":"45","CHATEAU RENARD":"45","CHATILLON COLIGNY":"45","CHATILLON LE ROI":"45","CHEVILLON SUR HUILLARD":"45","CHEVRY SOUS LE BIGNON":"45","COMBLEUX":"45","CORBEILLES":"45","COULLONS":"45","LA COUR MARIGNY":"45","COURTEMPIERRE":"45","CROTTES EN PITHIVERAIS":"45","DADONVILLE":"45","DAMMARIE EN PUISAYE":"45","DOUCHY MONTCORBON":"45","DRY":"45","EGRY":"45","ERCEVILLE":"45","ERVAUVILLE":"45","ESCRENNES":"45","FAVERELLES":"45","FEROLLES":"45","FLEURY LES AUBRAIS":"45","FOUCHEROLLES":"45","GAUBERTIN":"45","GIEN":"45","GRANGERMONT":"45","GY LES NONAINS":"45","INGRE":"45","LANGESSE":"45","LION EN BEAUCE":"45","LORCY":"45","LE MALESHERBOIS":"45","MARCILLY EN VILLETTE":"45","MERINVILLE":"45","MEZIERES EN GATINAIS":"45","MONTBARROIS":"45","MONTCRESSON":"45","LE MOULINET SUR SOLIN":"45","LA NEUVILLE SUR ESSONNE":"45","NEUVY EN SULLIAS":"45","OISON":"45","OUZOUER DES CHAMPS":"45","PREFONTAINES":"45","ST CYR EN VAL":"45","SOUGY":"45","TRAINOU":"45","VARENNES CHANGY":"45","VILLENEUVE SUR CONIE":"45","ALVIGNAC":"46","AUJOLS":"46","AUTOIRE":"46","AYNAC":"46","BAGNAC SUR CELE":"46","LE BASTIT":"46","BEDUER":"46","LE BOULVE":"46","CABRERETS":"46","CAHORS":"46","CANIAC DU CAUSSE":"46","CARNAC ROUFFIAC":"46","CASTELFRANC":"46","CONCORES":"46","CREGOLS":"46","FLAUJAC GARE":"46","FOURMAGNAC":"46","GINDOU":"46","GRAMAT":"46","GREZELS":"46","LABASTIDE DU HAUT MONT":"46","LANZAC":"46","LEOBARD":"46","MARCILHAC SUR CELE":"46","MASCLAT":"46","MECHMONT":"46","MERCUES":"46","MIERS":"46","NADILLAC":"46","RYE":"39","ST CYR MONTMALIN":"39","ST LAMAIN":"39","ST LUPICIN":"39","SAIZENAY":"39","SALANS":"39","SALIGNEY":"39","SAVIGNA":"39","SERMANGE":"39","SERRE LES MOULIERES":"39","SUPT":"39","SYAM":"39","VAUX LES ST CLAUDE":"39","VAUX SUR POLIGNY":"39","LE VERNOIS":"39","VERTAMBOZ":"39","VILLECHANTRIA":"39","VILLERSERINE":"39","VILLETTE LES ARBOIS":"39","VITREUX":"39","VRIANGE":"39","AIRE SUR L ADOUR":"40","ANGOUME":"40","ARTHEZ D ARMAGNAC":"40","ARX":"40","AUDIGNON":"40","AURICE":"40","BAS MAUCO":"40","BEGAAR":"40","BEYRIES":"40","BISCARROSSE":"40","BOURDALAT":"40","BOURRIOT BERGONCE":"40","CAPBRETON":"40","CASTELNAU TURSAN":"40","CLEDES":"40","DOAZIT":"40","GABARRET":"40","GAMARDE LES BAINS":"40","HAUT MAUCO":"40","HONTANX":"40","LACAJUNTE":"40","LUCBARDEZ ET BARGUES":"40","LUE":"40","LUSSAGNET":"40","MAYLIS":"40","MIMBASTE":"40","MONGET":"40","MORGANX":"40","MOUSTEY":"40","NERBIS":"40","ORIST":"40","OUSSE SUZAN":"40","PARENTIS EN BORN":"40","PERQUIE":"40","PEYREHORADE":"40","PIMBO":"40","PISSOS":"40","RENUNG":"40","RIMBEZ ET BAUDIETS":"40","SABRES":"40","ST ANDRE DE SEIGNANX":"40","ST ETIENNE D ORTHE":"40","STE EULALIE EN BORN":"40","ST YAGUEN":"40","SARRAZIET":"40","SAUBION":"40","SAUBUSSE":"40","SAUGNACQ ET MURET":"40","SERRESLOUS ET ARRIBANS":"40","SIEST":"40","TARNOS":"40","TERCIS LES BAINS":"40","TETHIEU":"40","TILH":"40","TOULOUZETTE":"40","YCHOUX":"40","YZOSSE":"40","ARTINS":"41","AVARAY":"41","BAUZY":"41","BRACIEUX":"41","BREVAINVILLE":"41","BUSLOUP":"41","CHAMPIGNY EN BEAUCE":"41","LA CHAPELLE VENDOMOISE":"41","MOISDON LA RIVIERE":"44","MONTRELAIS":"44","PAIMBOEUF":"44","LA PLANCHE":"44","PORNICHET":"44","PORT ST PERE":"44","PREFAILLES":"44","LA REGRIPPIERE":"44","REMOUILLE":"44","REZE":"44","ST BREVIN LES PINS":"44","ST ETIENNE DE MER MORTE":"44","ST JEAN DE BOISEAU":"44","STE LUCE SUR LOIRE":"44","ST LUMINE DE COUTAIS":"44","ST LYPHARD":"44","ST MARS LA JAILLE":"44","ST MICHEL CHEF CHEF":"44","SOULVACHE":"44","LE TEMPLE DE BRETAGNE":"44","LA TURBALLE":"44","VUE":"44","LA GRIGONNAIS":"44","AILLANT SUR MILLERON":"45","ARTENAY":"45","ASCHERES LE MARCHE":"45","BACCON":"45","BEAUGENCY":"45","LE BIGNON MIRABEAU":"45","BOISCOMMUN":"45","BORDEAUX EN GATINAIS":"45","BRICY":"45","CESARVILLE DOSSAINVILLE":"45","LA CHAPELLE SUR AVEYRON":"45","LE CHARME":"45","CORTRAT":"45","COUDROY":"45","DAMPIERRE EN BURLY":"45","DARVOY":"45","FAY AUX LOGES":"45","FONTENAY SUR LOING":"45","GUIGNEVILLE":"45","JARGEAU":"45","JOUY LE POTIER":"45","LAILLY EN VAL":"45","LORRIS":"45","LOURY":"45","MARDIE":"45","MAREAU AUX PRES":"45","MELLEROY":"45","MESSAS":"45","MEZIERES LEZ CLERY":"45","MIGNERES":"45","MIGNERETTE":"45","NANCRAY SUR RIMARDE":"45","OUGNY":"58","OULON":"58","PARIGNY LA ROSE":"58","PARIGNY LES VAUX":"58","POISEUX":"58","POUGUES LES EAUX":"58","RUAGES":"58","ST BENIN DES BOIS":"58","STE COLOMBE DES BOIS":"58","ST JEAN AUX AMOGNES":"58","ST OUEN SUR LOIRE":"58","ST PARIZE EN VIRY":"58","ST PIERRE LE MOUTIER":"58","ST QUENTIN SUR NOHAIN":"58","ST SEINE":"58","SOUGY SUR LOIRE":"58","SURGY":"58","TALON":"58","TOURY LURCY":"58","TOURY SUR JOUR":"58","ST HELEN":"22","ST HERVE":"22","ST JOUAN DE L ISLE":"22","ST MAYEUX":"22","ST MELOIR DES BOIS":"22","ST MICHEL EN GREVE":"22","ST RIEUL":"22","ST SAMSON SUR RANCE":"22","TRAMAIN":"22","TREBRIVAN":"22","TREDIAS":"22","TREGUEUX":"22","TREGUIER":"22","TRELIVAN":"22","TREMEUR":"22","TREVE":"22","TREZENY":"22","YVIGNAC LA TOUR":"22","AZERABLES":"23","BANIZE":"23","BELLEGARDE EN MARCHE":"23","LA CELLE DUNOISE":"23","CHAVANAT":"23","CHENERAILLES":"23","CLAIRAVAUX":"23","LE DONZEIL":"23","FELLETIN":"23","FONTANIERES":"23","GARTEMPE":"23","JOUILLAT":"23","LEPAUD":"23","LEYRAT":"23","LINARD":"23","LIZIERES":"23","MAISON FEYNE":"23","MALLERET":"23","LE MAS D ARTIGE":"23","MERINCHAL":"23","MOURIOUX VIEILLEVILLE":"23","MOUTIER ROZEILLE":"23","NOTH":"23","LA NOUAILLE":"23","PONTARION":"23","LA POUGE":"23","RETERRE":"23","SARDENT":"23","SERMUR":"23","SOUBREBOST":"23","ST AVIT DE TARDES":"23","ST BARD":"23","STE FEYRE":"23","ST GEORGES LA POUGE":"23","ST GERMAIN BEAUPRE":"23","ST HILAIRE LA PLAINE":"23","ST HILAIRE LE CHATEAU":"23","ST JULIEN LA GENETE":"23","ST MARC A FRONGIER":"23","ST MAURICE LA SOUTERRAINE":"23","ST MEDARD LA ROCHETTE":"23","ST MERD LA BREUILLE":"23","ST PRIEST LA FEUILLE":"23","ST PRIEST LA PLAINE":"23","ST SILVAIN BELLEGARDE":"23","ST SULPICE LE DUNOIS":"23","VALLIERE":"23","LA VILLENEUVE":"23","ALLES SUR DORDOGNE":"24","ALLEMANS":"24","ANLHIAC":"24","BASSILLAC":"24","BEAUMONTOIS EN PERIGORD":"24","BEAUREGARD ET BASSAC":"24","PAYS DE BELVES":"24","BERGERAC":"24","BEYNAC ET CAZENAC":"24","BIRAS":"24","BOISSE":"24","BOISSEUILH":"24","BORREZE":"24","BOUTEILLES ST SEBASTIEN":"24","CASTELNAUD LA CHAPELLE":"24","CAZOULES":"24","CENDRIEUX":"24","CHANCELADE":"24","LA CHAPELLE GRESIGNAC":"24","CHERVEIX CUBAS":"24","CONDAT SUR TRINCOU":"24","LA COQUILLE":"24","COURSAC":"24","COURS DE PILE":"24","CREYSSAC":"24","CREYSSENSAC ET PISSOT":"24","DOUCHAPT":"24","EXCIDEUIL":"24","EYMET":"24","NOMAIN":"59","NOYELLES SUR ESCAUT":"59","NOYELLES SUR SELLE":"59","OBIES":"59","OOST CAPPEL":"59","ORCHIES":"59","ORSINVAL":"59","OUDEZEELE":"59","PITGAM":"59","POIX DU NORD":"59","POMMEREUIL":"59","PONT SUR SAMBRE":"59","PREMESQUES":"59","PROVILLE":"59","QUAEDYPRE":"59","QUAROUBLE":"59","QUIEVRECHAIN":"59","QUIEVY":"59","RACHES":"59","RAMILLIES":"59","REJET DE BEAULIEU":"59","RIBECOURT LA TOUR":"59","RIEULAY":"59","ROEULX":"59","ROUCOURT":"59","ROUSIES":"59","ROUVIGNIES":"59","SAINGHIN EN WEPPES":"59","ST HILAIRE SUR HELPE":"59","ST JANS CAPPEL":"59","ST PYTHON":"59","ST REMY CHAUSSEE":"59","ST SOUPLET":"59","ST SYLVESTRE CAPPEL":"59","SARS ET ROSIERES":"59","SAULTAIN":"59","SEMERIES":"59","SEQUEDIN":"59","SOLRINNES":"59","SPYCKER":"59","STEENE":"59","STEENWERCK":"59","TAISNIERES EN THIERACHE":"59","THIANT":"59","THUN ST AMAND":"59","TILLOY LEZ CAMBRAI":"59","TOUFFLERS":"59","TRELON":"59","VILLERS OUTREAUX":"59","VILLERS POL":"59","WALINCOURT SELVIGNY":"59","WANDIGNIES HAMAGE":"59","WASNES AU BAC":"59","WATTIGNIES LA VICTOIRE":"59","WAVRECHAIN SOUS DENAIN":"59","WAVRECHAIN SOUS FAULX":"59","WAZIERS":"59","ABBEVILLE ST LUCIEN":"60","AIRION":"60","AMBLAINVILLE":"60","ANGICOURT":"60","ATTICHY":"60","AUNEUIL":"60","BAILLEUL LE SOC":"60","BEHERICOURT":"60","BETHISY ST MARTIN":"60","BETHISY ST PIERRE":"60","BETZ":"60","BLANCFOSSE":"60","BOISSY LE BOIS":"60","BONLIER":"60","BONNEUIL EN VALOIS":"60","BORNEL":"60","BOUTAVENT":"60","BOUTENCOURT":"60","BRESLES":"60","BREUIL LE SEC":"60","BROMBOS":"60","BROQUIERS":"60","BRUNVILLERS LA MOTTE":"60","BUCAMPS":"60","CANDOR":"60","CANNY SUR THERAIN":"60","CHAMANT":"60","CHEVREVILLE":"60","CHIRY OURSCAMP":"60","CHOISY LA VICTOIRE":"60","CIRES LES MELLO":"60","COUDUN":"60","PINSAC":"46","POMAREDE":"46","RUDELLE":"46","STE ALAUZIE":"46","LES PECHS DU VERS":"46","ST CIRQ LAPOPIE":"46","ST GERMAIN DU BEL AIR":"46","ST LAURENT LES TOURS":"46","ST MARTIN LE REDON":"46","ST MAURICE EN QUERCY":"46","ST MICHEL LOUBEJOU":"46","SAULIAC SUR CELE":"46","SENAILLAC LAUZES":"46","SOTURAC":"46","TERROU":"46","VILLESEQUE":"46","ALLEMANS DU DROPT":"47","ANDIRAN":"47","ANTHE":"47","BOUDY DE BEAUREGARD":"47","BRUCH":"47","CANCON":"47","CASTELCULIER":"47","COCUMONT":"47","COLAYRAC ST CIRQ":"47","CUZORN":"47","DONDAS":"47","FAUILLET":"47","FIEUX":"47","FONGRAVE":"47","FRECHOU":"47","FRESPECH":"47","GRANGES SUR LOT":"47","LABRETONIE":"47","LAPLUME":"47","LAYRAC":"47","LE MAS D AGENAIS":"47","MAZIERES NARESSE":"47","MIRAMONT DE GUYENNE":"47","MONSEMPRON LIBOS":"47","MONTAGNAC SUR AUVIGNON":"47","MONTAGNAC SUR LEDE":"47","PENNE D AGENAIS":"47","PEYRIERE":"47","PRAYSSAS":"47","RAZIMET":"47","ROUMAGNE":"47","ST BARTHELEMY D AGENAIS":"47","STE BAZEILLE":"47","ST HILAIRE DE LUSIGNAN":"47","ST MARTIN CURTON":"47","ST PE ST SIMON":"47","ST PIERRE DE CLAIRAC":"47","ST SAUVEUR DE MEILHAN":"47","ST SYLVESTRE SUR LOT":"47","LA SAUVETAT DE SAVERES":"47","LA SAUVETAT DU DROPT":"47","SAVIGNAC DE DURAS":"47","SERIGNAC PEBOUDOU":"47","SEYCHES":"47","ALBARET LE COMTAL":"48","ALBARET STE MARIE":"48","ALLENC":"48","AUROUX":"48","BADAROUX":"48","LES BONDONS":"48","CULTURES":"48","ESTABLES":"48","FRAISSINET DE FOURQUES":"48","GABRIAS":"48","GRANDRIEU":"48","LES HERMAUX":"48","JULIANGES":"48","MAS D ORCIERES":"48","MONTRODAT":"48","RECOULES DE FUMAS":"48","LE ROZIER":"48","ST ANDRE CAPCEZE":"48","ST BONNET DE MONTAUROUX":"48","ST CHELY D APCHER":"48","STE COLOMBE DE PEYRE":"48","ST GAL":"48","CHATILLON SUR CHER":"41","CHAUMONT SUR THARONNE":"41","CONAN":"41","CONCRIERS":"41","COUTURE SUR LOIR":"41","CRUCHERAY":"41","LES ESSARTS":"41","FONTAINES EN SOLOGNE":"41","FOUGERES SUR BIEVRE":"41","LES HAYES":"41","HUISSEAU EN BEAUCE":"41","HUISSEAU SUR COSSON":"41","LANCOME":"41","LANDES LE GAULOIS":"41","LOREUX":"41","MASLIVES":"41","MENNETOU SUR CHER":"41","VALENCISSE":"41","MONDOUBLEAU":"41","MONTRICHARD VAL DE CHER":"41","MULSANS":"41","NOUAN LE FUZELIER":"41","ONZAIN":"41","LE PLESSIS L ECHELLE":"41","ROMILLY":"41","ST CLAUDE DE DIRAY":"41","ST DYE SUR LOIRE":"41","SASSAY":"41","SEIGY":"41","THORE LA ROCHETTE":"41","VENDOME":"41","VILLEBAROU":"41","VILLEMARDY":"41","VILLETRUN":"41","VILLIERS SUR LOIR":"41","ABOEN":"42","AILLEUX":"42","AMIONS":"42","APINAC":"42","LA BENISSON DIEU":"42","BESSEY":"42","LA CHAMBA":"42","CHERIER":"42","CIVENS":"42","CLEPPE":"42","CORDELLE":"42","LE CROZET":"42","DEBATS RIVIERE D ORPRA":"42","ESSERTINES EN CHATELNEUF":"42","FEURS":"42","SAVOISY":"21","SOMBERNON":"21","SOUSSEY SUR BRIONNE":"21","TALANT":"21","TART LE HAUT":"21","TELLECEY":"21","THENISSEY":"21","VARANGES":"21","VENAREY LES LAUMES":"21","VERONNES":"21","VERREY SOUS DREE":"21","VERREY SOUS SALMAISE":"21","VIEILMOULIN":"21","VILLARGOIX":"21","VILLEBERNY":"21","VILLENEUVE SOUS CHARIGNY":"21","VILLERS LA FAYE":"21","VILLY LE MOUTIER":"21","VOSNE ROMANEE":"21","ALLINEUC":"22","AUCALEUC":"22","BOURSEUL":"22","BULAT PESTIVIEN":"22","CANIHUEL":"22","LES CHAMPS GERAUX":"22","CORSEUL":"22","CREHEN":"22","ERQUY":"22","GOMENE":"22","VARZY":"58","VILLENEUVE D ASCQ":"59","ANOR":"59","ARNEKE":"59","AUBRY DU HAINAUT":"59","AUDIGNIES":"59","AVESNES SUR HELPE":"59","AVESNES LE SEC":"59","BAIVES":"59","BAMBECQUE":"59","BANTOUZELLE":"59","BAVINCHOVE":"59","BELLAING":"59","BEVILLERS":"59","BONDUES":"59","BRAY DUNES":"59","BRIASTRE":"59","BRUAY SUR L ESCAUT":"59","CAMBRAI":"59","CAPELLE":"59","CAPPELLE BROUCK":"59","CARNIN":"59","CATTENIERES":"59","CERFONTAINE":"59","CHATEAU L ABBAYE":"59","CHEMY":"59","COURCHELETTES":"59","CURGIES":"59","CUVILLERS":"59","DENAIN":"59","EBBLINGHEM":"59","ELESMES":"59","ELINCOURT":"59","ESCOBECQUES":"59","ESTREUX":"59","ESTRUN":"59","FLOURSIES":"59","FONTAINE AU BOIS":"59","FRETIN":"59","FROMELLES":"59","GODEWAERSVELDE":"59","GONNELIEU":"59","GRANDE SYNTHE":"59","HAMEL":"59","HAUTMONT":"59","HONDSCHOOTE":"59","KILLEM":"59","LALLAING":"59","LAROUILLIES":"59","LIGNY EN CAMBRESIS":"59","LIMONT FONTAINE":"59","LOFFRE":"59","LA LONGUEVILLE":"59","LOUVIGNIES QUESNOY":"59","MARCQ EN BAROEUL":"59","MARESCHES":"59","MARQUETTE LEZ LILLE":"59","MERCKEGHEM":"59","MERIGNIES":"59","MERRIS":"59","MONS EN BAROEUL":"59","MONTAY":"59","MONTIGNY EN CAMBRESIS":"59","MORBECQUE":"59","MORTAGNE DU NORD":"59","NEUF BERQUIN":"59","NIEPPE":"59","NIEURLET":"59","ODOMEZ":"59","OHAIN":"59","ONNAING":"59","OSTRICOURT":"59","PECQUENCOURT":"59","PETITE FORET":"59","PETIT FAYT":"59","PHALEMPIN":"59","PONT A MARCQ":"59","QUIEVELON":"59","RAISMES":"59","RECQUIGNIES":"59","REXPOEDE":"59","RIEUX EN CAMBRESIS":"59","ROOST WARENDIN":"59","RUBROUCK":"59","LES RUES DES VIGNES":"59","RUMILLY EN CAMBRESIS":"59","SAILLY LEZ LANNOY":"59","SAINS DU NORD":"59","ST AMAND LES EAUX":"59","ST BENIN":"59","GABILLOU":"24","GAGEAC ET ROUILLAC":"24","GINESTET":"24","GOUT ROSSIGNOL":"24","GRAND BRASSAC":"24","GRIVES":"24","ISSIGEAC":"24","JAYAC":"24","LA JEMAYE":"24","LACROPTE":"24","LALINDE":"24","LES LECHES":"24","LEGUILLAC DE CERCLES":"24","LEGUILLAC DE L AUCHE":"24","MARCILLAC ST QUENTIN":"24","MARSANEIX":"24","MEYRALS":"24","MILHAC D AUBEROCHE":"24","MONTAZEAU":"24","MONTPON MENESTEROL":"24","NANTHEUIL":"24","PERIGUEUX":"24","PEYZAC LE MOUSTIER":"24","PLAZAC":"24","PRIGONRIEUX":"24","PUYRENIER":"24","ROUFFIGNAC ST CERNIN DE REILHAC":"24","SADILLAC":"24","ST AMAND DE COLY":"24","ST AUBIN DE LANQUAIS":"24","ST AVIT DE VIALARD":"24","ST AVIT SENIEUR":"24","ST CREPIN D AUBEROCHE":"24","STE EULALIE D EYMET":"24","ST FELIX DE VILLADEIX":"24","ST FRONT DE PRADOUX":"24","ST FRONT LA RIVIERE":"24","ST LAURENT DES HOMMES":"24","ST LAURENT DES VIGNES":"24","ST LEON SUR VEZERE":"24","ST LOUIS EN L ISLE":"24","ST MARCORY":"24","ST MARTIAL VIVEYROL":"24","ST MARTIN DE RIBERAC":"24","ST MARTIN L ASTIER":"24","ST MEARD DE DRONE":"24","ST PARDOUX ET VIELVIC":"24","ST PRIEST LES FOUGERES":"24","ST SEURIN DE PRATS":"24","ST VINCENT DE CONNEZAC":"24","ST VINCENT LE PALUEL":"24","ST VINCENT SUR L ISLE":"24","SAVIGNAC LES EGLISES":"24","SIMEYROLS":"24","SOURZAC":"24","TAMNIES":"24","TEILLOTS":"24","TEYJAT":"24","LA TOUR BLANCHE":"24","VANXAINS":"24","VARAIGNES":"24","VAUNAC":"24","VERGT":"24","VILLEFRANCHE DU PERIGORD":"24","ALLENJOIE":"25","AMAGNEY":"25","AMANCEY":"25","ARBOUANS":"25","ARC SOUS MONTENOT":"25","AUBONNE":"25","AVANNE AVENEY":"25","AVOUDREY":"25","BANNANS":"25","BATTENANS VARIN":"25","BELFAYS":"25","BESANCON":"25","BEURE":"25","BONNAL":"25","BOUJAILLES":"25","BOURGUIGNON":"25","BOUSSIERES":"25","BOUVERANS":"25","BRAILLANS":"25","LES BRESEUX":"25","BROGNARD":"25","COURCELLES LES GISORS":"60","COYE LA FORET":"60","CREVECOEUR LE GRAND":"60","LE CROCQ":"60","CUTS":"60","LE DELUGE":"60","DIVES":"60","ECUVILLY":"60","ERQUERY":"60","ESCLES ST PIERRE":"60","FEUQUIERES":"60","FITZ JAMES":"60","FRESNOY EN THELLE":"60","LE FRESTOY VAUX":"60","FRETOY LE CHATEAU":"60","GANNES":"60","GREMEVILLERS":"60","GUIGNECOURT":"60","HARDIVILLERS EN VEXIN":"60","HEMEVILLERS":"60","HERICOURT SUR THERAIN":"60","HERMES":"60","LA HOUSSOYE":"60","JUVIGNIES":"60","LACHAPELLE SOUS GERBEROY":"60","LACHAUSSEE DU BOIS D ECU":"60","LAGNY":"60","LAMECOURT":"60","LATAULE":"60","LOUEUSE":"60","MAIGNELAY MONTIGNY":"60","MARQUEGLISE":"60","MENEVILLERS":"60","MERY LA BATAILLE":"60","LE MESNIL THERIBUS":"60","MONCEAUX L ABBAYE":"60","MONTAGNY STE FELICITE":"60","MONT L EVEQUE":"60","MONTMARTIN":"60","MORVILLERS":"60","MOUCHY LE CHATEL":"60","MUREAUMONT":"60","NERY":"60","LA NEUVILLE ST PIERRE":"60","NOGENT SUR OISE":"60","NOROY":"60","OGNOLLES":"60","ONS EN BRAY":"60","ORMOY LE DAVIEN":"60","PIMPREZ":"60","LE PLESSIS BRION":"60","LE PLESSIS PATTE D OIE":"60","PRONLEROY":"60","RAVENEL":"60","REMERANGLES":"60","RHUIS":"60","RIBECOURT DRESLINCOURT":"60","ROTANGY":"60","ST ANDRE FARIVILLERS":"60","ST DENISCOURT":"60","ST GERMAIN LA POTERIE":"60","ST GERMER DE FLY":"60","ST OMER EN CHAUSSEE":"60","ST REMY EN L EAU":"60","SENOTS":"60","SILLY TILLARD":"60","SOMMEREUX":"60","THIERS SUR THEVE":"60","THURY SOUS CLERMONT":"60","TILLE":"60","TROSLY BREUIL":"60","LE VAUROUX":"60","VEZ":"60","VILLENEUVE SUR VERBERIE":"60","VILLERS SUR BONNIERES":"60","VILLOTRAN":"60","WACQUEMOULIN":"60","ARGENTAN":"61","AUGUAISE":"61","ST JULIEN DES POINTS":"48","ST JULIEN DU TOURNEL":"48","ST LAURENT DE MURET":"48","ST LAURENT DE VEYRES":"48","ST PRIVAT DE VALLONGUE":"48","ST ROME DE DOLAN":"48","SERVIERES":"48","VIALAS":"48","ANGERS":"49","ANGRIE":"49","BEAUCOUZE":"49","BECON LES GRANITS":"49","BOUCHEMAINE":"49","LE BOURG D IRE":"49","LA BREILLE LES PINS":"49","CARBAY":"49","LES CERQUEUX":"49","CHAMPTOCE SUR LOIRE":"49","CHATEAUNEUF SUR SARTHE":"49","CHAVAGNES":"49","CHEFFES":"49","CHEMELLIER":"49","CHIGNE":"49","CHOLET":"49","DENEE":"49","LA FERRIERE DE FLEE":"49","JUIGNE SUR LOIRE":"49","JUVARDEIL":"49","LONGUE JUMELLES":"49","LOURESSE ROCHEMENIER":"49","LUIGNE":"49","MAULEVRIER":"49","MIRE":"49","MONTILLIERS":"49","NOYANT":"49","NOYANT LA GRAVOYERE":"49","LA POSSONNIERE":"49","QUERRE":"49","LES RAIRIES":"49","ROU MARSON":"49","STE GEMMES SUR LOIRE":"49","ST GEORGES SUR LOIRE":"49","ST JEAN DE LINIERES":"49","ST JUST SUR DIVE":"49","VAL DU LAYON":"49","ST LEGER DES BOIS":"49","ST MACAIRE DU BOIS":"49","ST PAUL DU BOIS":"49","SCEAUX D ANJOU":"49","SOMLOIRE":"49","SOUCELLES":"49","THORIGNE D ANJOU":"49","LE TREMBLAY":"49","VAUCHRETIEN":"49","LES VERCHERS SUR LAYON":"49","VERGONNES":"49","VERNOIL LE FOURRIER":"49","APPEVILLE":"50","AVRANCHES":"50","BARNEVILLE CARTERET":"50","LA BARRE DE SEMILLY":"50","BLAINVILLE SUR MER":"50","JULLOUVILLE":"50","BRETTEVILLE SUR AY":"50","BREUVILLE":"50","BREVILLE SUR MER":"50","BRICQUEVILLE SUR MER":"50","GOMMENEC H":"22","GRACES":"22","KERFOT":"22","KERGRIST MOELOU":"22","KERIEN":"22","LANDEBAERON":"22","LANDEBIA":"22","LA LANDEC":"22","LANDEHEN":"22","LANGROLAY SUR RANCE":"22","LANNION":"22","LANRODEC":"22","LANTIC":"22","LAURENAN":"22","MAEL CARHAIX":"22","LA MALHOURE":"22","MATIGNON":"22","LA MEAUGON":"22","MERDRIGNAC":"22","MERLEAC":"22","MINIHY TREGUIER":"22","PEDERNEC":"22","PENVENAN":"22","PERROS GUIREC":"22","PEUMERIT QUINTIN":"22","PLAINTEL":"22","PLEUDIHEN SUR RANCE":"22","PLOUISY":"22","PLOUNERIN":"22","PLOURAC H":"22","PLOURHAN":"22","PLOUZELAMBRE":"22","PLUDUAL":"22","PLUZUNET":"22","POULDOURAN":"22","LA PRENESSAYE":"22","QUEMPER GUEZENNEC":"22","QUESSOY":"22","QUEVERT":"22","ROSTRENEN":"22","ST AGATHON":"22","ST CLET":"22","ST CONNAN":"22","ST CONNEC":"22","ST DENOUAL":"22","ST GELVEN":"22","ST GILLES PLIGEAUX":"22","ST GUEN":"22","ST JACUT DE LA MER":"22","ST NICODEME":"22","ST PEVER":"22","SENVEN LEHART":"22","SEVIGNAC":"22","SQUIFFIEC":"22","TREDARZEC":"22","TREDREZ LOCQUEMEAU":"22","TREDUDER":"22","TREFUMEL":"22","TREGASTEL":"22","TREGUIDEL":"22","TRELEVERN":"22","TREMEL":"22","TRESSIGNAUX":"22","TROGUERY":"22","BEISSAT":"23","BLAUDEIX":"23","BOSROGER":"23","BOURGANEUF":"23","BOUSSAC BOURG":"23","BUSSIERE ST GEORGES":"23","LA CELLE SOUS GOUZON":"23","CHAMBON STE CROIX":"23","CHAMBON SUR VOUEIZE":"23","LA CHAPELLE ST MARTIAL":"23","LA CHAPELLE TAILLEFERT":"23","COLONDANNES":"23","GENTIOUX PIGEROLLES":"23","GLENIC":"23","LE GRAND BOURG":"23","LAFAT":"23","LAVAUFRANCHE":"23","LIOUX LES MONGES":"23","MAINSAT":"23","MAISONNISSES":"23","MALLERET BOUSSAC":"23","STE MARIE CAPPEL":"59","ST MOMELIN":"59","ST VAAST EN CAMBRESIS":"59","SAMEON":"59","SANTES":"59","SASSEGNIES":"59","SEBOURG":"59","SERANVILLERS FORENVILLE":"59","SERCUS":"59","SIN LE NOBLE":"59","SOLRE LE CHATEAU":"59","TETEGHEM COUDEKERQUE VILLAGE":"59","THIENNES":"59","THIVENCELLE":"59","TOURMIGNIES":"59","VALENCIENNES":"59","VENDEGIES AU BOIS":"59","VIESLY":"59","VIEUX MESNIL":"59","VILLERS GUISLAIN":"59","VOLCKERINCKHOVE":"59","WALLERS":"59","WANNEHAIN":"59","WARGNIES LE PETIT":"59","WARLAING":"59","WILLEMS":"59","WILLIES":"59","ZUYDCOOTE":"59","DON":"59","ANDEVILLE":"60","ANGIVILLERS":"60","AUGER ST VINCENT":"60","BAILLEVAL":"60","BALAGNY SUR THERAIN":"60","BEAUDEDUIT":"60","BEAULIEU LES FONTAINES":"60","BEAURAINS LES NOYON":"60","BELLOY":"60","BIERMONT":"60","BLACOURT":"60","BOUILLANCY":"60","BOUVRESSE":"60","BREGY":"60","BREUIL LE VERT":"60","CATIGNY":"60","CATILLON FUMECHON":"60","CHAMBORS":"60","LA CHAPELLE EN SERVAL":"60","CHAVENCON":"60","CHEPOIX":"60","CHOQUEUSE LES BENARDS":"60","COMPIEGNE":"60","LE COUDRAY ST GERMER":"60","LE COUDRAY SUR THELLE":"60","COULOISY":"60","COURTIEUX":"60","CREIL":"60","CROISSY SUR CELLE":"60","CUIGNIERES":"60","CUVILLY":"60","DUVY":"60","EPINEUSE":"60","ESCHES":"60","EVRICOURT":"60","FAY LES ETANGS":"60","FEIGNEUX":"60","FLAVY LE MELDEUX":"60","FORMERIE":"60","FOURNIVAL":"60","GAUDECHART":"60","GILOCOURT":"60","GOUY LES GROSEILLERS":"60","GREZ":"60","HAINVILLERS":"60","HARDIVILLERS":"60","HENONVILLE":"60","HETOMESNIL":"60","HODENC L EVEQUE":"60","HONDAINVILLE":"60","JAMERICOURT":"60","JAUX":"60","LABOISSIERE EN THELLE":"60","LACHAPELLE AUX POTS":"60","LACHAPELLE ST PIERRE":"60","LAFRAYE":"60","LAMORLAYE":"60","LAVACQUERIE":"60","LAVERRIERE":"60","LEVIGNEN":"60","LOCONVILLE":"60","LONGUEIL ANNEL":"60","LONGUEIL STE MARIE":"60","MAISONCELLE TUILERIE":"60","MARGNY AUX CERISES":"60","MAULERS":"60","BUFFARD":"25","BUGNY":"25","BURGILLE":"25","BUSY":"25","CENDREY":"25","CHALEZE":"25","CHANTRANS":"25","CHARQUEMONT":"25","CHATILLON GUYOTTE":"25","CHAUCENNE":"25","CHAY":"25","LA CHENALOTTE":"25","CHENECEY BUILLON":"25","LA CLUSE ET MIJOUX":"25","COLOMBIER FONTAINE":"25","COTEBRUNE":"25","DAMBENOIS":"25","DAMMARTIN LES TEMPLIERS":"25","DASLE":"25","DOMPIERRE LES TILLEULS":"25","LES ECORCES":"25","EMAGNY":"25","ETERNOZ":"25","ETOUVANS":"25","ETRABONNE":"25","EXINCOURT":"25","FERTANS":"25","FLANGEBOUCHE":"25","FONTAIN":"25","FOURCATIER ET MAISON NEUVE":"25","FRASNE":"25","GELLIN":"25","GENEUILLE":"25","GERMEFONTAINE":"25","GERMONDANS":"25","GLAY":"25","GONSANS":"25","GRAND COMBE CHATELEU":"25","LES GRAS":"25","LES HOPITAUX VIEUX":"25","HUANNE MONTMARTIN":"25","HYEVRE PAROISSE":"25","JOUGNE":"25","LAISSEY":"25","LANANS":"25","LANTHENANS":"25","LAVANS VUILLAFANS":"25","LEVIER":"25","LIESLE":"25","LONGEVELLE SUR DOUBS":"25","LOUGRES":"25","LE LUHIER":"25","MANCENANS LIZERNE":"25","MANDEURE":"25","MARCHAUX":"25","MARVELISE":"25","MERCEY LE GRAND":"25","MEREY SOUS MONTROND":"25","MONTAGNEY SERVIGNEY":"25","MONTBELIARDOT":"25","MONTECHEROUX":"25","MONTJOIE LE CHATEAU":"25","MYON":"25","NANCRAY":"25","NARBIEF":"25","NEUCHATEL URTIERE":"25","LES PREMIERS SAPINS":"25","NOIRONTE":"25","ORGEANS BLANCHEFONTAINE":"25","ORNANS":"25","OSSELLE ROUTELLE":"25","PASSAVANT":"25","PONT DE ROIDE VERMONDANS":"25","POULIGNEY LUSANS":"25","PRESENTEVILLERS":"25","ROGNON":"25","ST JUAN":"25","ST JULIEN LES RUSSEY":"25","ST MAURICE COLOMBIER":"25","SAUVAGNEY":"25","SECHIN":"25","SILLEY BLEFOND":"25","SOLEMONT":"25","SOULCE CERNAY":"25","TALLENAY":"25","THULAY":"25","THUREY LE MONT":"25","VAL DE ROULANS":"25","VAUX ET CHANTEGRUE":"25","VELLEROT LES VERCEL":"25","VERCEL VILLEDIEU LE CAMP":"25","AUNAY LES BOIS":"61","BAZOCHES SUR HOENE":"61","BELFONDS":"61","BELLOU LE TRICHARD":"61","BONNEFOI":"61","BRETHEL":"61","BRIEUX":"61","ST CYR DU GAULT":"41","ST DENIS SUR LOIRE":"41","ST ETIENNE DES GUERETS":"41","STE GEMMES":"41","ST MARTIN DES BOIS":"41","ST RIMAY":"41","SARGE SUR BRAYE":"41","SASNIERES":"41","SEILLAC":"41","SEUR":"41","THEILLAY":"41","TROO":"41","VALAIRE":"41","VIEVY LE RAYE":"41","LA VILLE AUX CLERCS":"41","VILLEFRANCHE SUR CHER":"41","VILLEHERVIERS":"41","VILLENY":"41","VILLEPORCHER":"41","VILLERABLE":"41","VILLERMAIN":"41","ARTHUN":"42","AVEIZIEUX":"42","BOURG ARGENTAL":"42","LE CERGNE":"42","CEZAY":"42","CHALMAZEL JEANSAGNIERE":"42","CHAMBEON":"42","CHAMPDIEU":"42","LA CHAPELLE VILLARS":"42","CHAZELLES SUR LYON":"42","COMBRE":"42","CREMEAUX":"42","CUINZIER":"42","DANCE":"42","DOIZIEUX":"42","ECOCHE":"42","ECOTAY L OLME":"42","L ETRAT":"42","FRAISSES":"42","LA GIMOND":"42","GRAMMOND":"42","GREZOLLES":"42","LAVIEU":"42","LAY":"42","LORETTE":"42","MARCILLY LE CHATEL":"42","MARCLOPT":"42","MARINGES":"42","MIZERIEUX":"42","MONTBRISON":"42","NERVIEUX":"42","PARIGNY":"42","ROZIER EN DONZY":"42","SAIL SOUS COUZAN":"42","ST BARTHELEMY LESTRA":"42","ST DENIS SUR COISE":"42","ST DIDIER SUR ROCHEFORT":"42","ST GALMIER":"42","ST GENEST MALIFAUX":"42","GENILAC":"42","ST JODARD":"42","ST JUST EN CHEVALET":"42","ST NIZIER DE FORNAS":"42","ST NIZIER SOUS CHARLIEU":"42","ST PAUL D UZORE":"42","ST PAUL EN CORNILLON":"42","ST PAUL EN JAREZ":"42","LES SALLES":"42","SOLEYMIEUX":"42","URBISE":"42","VERANNE":"42","VERIN":"42","VERRIERES EN FOREZ":"42","ALLEYRAS":"43","AUREC SUR LOIRE":"43","BLESLE":"43","BRUCHEVILLE":"50","CAMBERNON":"50","LE GRIPPON":"50","CHAVOY":"50","CLITOURPS":"50","COUDEVILLE SUR MER":"50","COULOUVRAY BOISBENATRE":"50","COURTILS":"50","COUVAINS":"50","DENNEVILLE":"50","DIGULLEVILLE":"50","DOMJEAN":"50","EQUILLY":"50","L ETANG BERTRAND":"50","FERMANVILLE":"50","GATHEMO":"50","LA HAYE D ECTOT":"50","HEAUVILLE":"50","THEREVAL":"50","HIESVILLE":"50","HOCQUIGNY":"50","HUBERVILLE":"50","LAULNE":"50","LESTRE":"50","LES LOGES SUR BRECEY":"50","MARCHESIEUX":"50","MARTINVAST":"50","MEAUTIS":"50","LA MEURDRAQUIERE":"50","MONTAIGU LES BOIS":"50","MONTFARVILLE":"50","MOON SUR ELLE":"50","LA MOUCHE":"50","MOYON VILLAGES":"50","LE NEUFBOURG":"50","OMONVILLE LA ROGUE":"50","ORVAL SUR SIENNE":"50","LE PETIT CELLAND":"50","LES PIEUX":"50","PLACY MONTAIGU":"50","LE PLESSIS LASTELLE":"50","PONT HEBERT":"50","PONTS":"50","PORTBAIL":"50","QUETTEHOU":"50","RAIDS":"50","RAMPAN":"50","REMILLY SUR LOZON":"50","ROMAGNY FONTENAY":"50","ST CHRISTOPHE DU FOC":"50","ST DENIS LE GAST":"50","ST GERMAIN DE VARREVILLE":"50","ST GERMAIN SUR SEVES":"50","ST JEAN DU CORAIL DES BOIS":"50","ST MARTIN LE GREARD":"50","ST NICOLAS DE PIERREPONT":"50","ST PIERRE DE SEMILLY":"50","ST PLANCHERS":"50","ST SENIER SOUS AVRANCHES":"50","ST VAAST LA HOUGUE":"50","TERRE ET MARAIS":"50","SOULLES":"50","SURTAINVILLE":"50","TEURTHEVILLE HAGUE":"50","TIREPIED":"50","MASBARAUD MERIGNAT":"23","MOUTIER MALCARD":"23","PUY MALSIGNAT":"23","ST AGNANT DE VERSILLAT":"23","ST ALPINIEN":"23","ST AMAND JARTOUDEIX":"23","ST AVIT LE PAUVRE":"23","STE FEYRE LA MONTAGNE":"23","ST GEORGES NIGREMONT":"23","ST LEGER BRIDEREIX":"23","ST MARTIN STE CATHERINE":"23","ST MAURICE PRES CROCQ":"23","ST ORADOUX DE CHIROUZE":"23","ST ORADOUX PRES CROCQ":"23","ST SULPICE LE GUERETOIS":"23","TERCILLAT":"23","TROIS FONDS":"23","AGONAC":"24","AJAT":"24","ANTONNE ET TRIGONANT":"24","BADEFOLS SUR DORDOGNE":"24","BEAUPOUYET":"24","BEAUREGARD DE TERRASSON":"24","BELEYMAS":"24","BERBIGUIERES":"24","BLIS ET BORN":"24","BOULAZAC ISLE MANOIRE":"24","LE BOURDEIX":"24","BROUCHAUD":"24","LE BUISSON DE CADOUIN":"24","BUSSAC":"24","BUSSEROLLES":"24","CALVIAC EN PERIGORD":"24","CAMPAGNAC LES QUERCY":"24","CAMPSEGRET":"24","CANTILLAC":"24","CERCLES":"24","CHAMPEAUX ET LA CHAPELLE POMMIER":"24","CHAMPS ROMAIN":"24","CHANTERAC":"24","CHAPDEUIL":"24","LA CHAPELLE FAUCHER":"24","LA CHAPELLE GONAGUET":"24","LA CHAPELLE MONTABOURLET":"24","CLERMONT D EXCIDEUIL":"24","COLY":"24","CONNEZAC":"24","CUBJAC":"24","DOMME":"24","DOUVILLE":"24","LA DOUZE":"24","DOUZILLAC":"24","DUSSAC":"24","EGLISE NEUVE DE VERGT":"24","ESCOIRE":"24","EYGURANDE ET GARDEDEUIL":"24","EYZERAC":"24","FAURILLES":"24","FIRBEIX":"24","FLORIMONT GAUMIER":"24","FRAISSE":"24","GENIS":"24","GRUN BORDAS":"24","HAUTEFORT":"24","LAMONZIE ST MARTIN":"24","LANOUAILLE":"24","LEMBRAS":"24","LEMPZOURS":"24","LIMEUIL":"24","LOUBEJAC":"24","MANZAC SUR VERN":"24","MARSAC SUR L ISLE":"24","MENESPLET":"24","MILHAC DE NONTRON":"24","MINZAC":"24","MONMARVES":"24","MONTFERRAND DU PERIGORD":"24","NADAILLAC":"24","NAILHAC":"24","NANTEUIL AURIAC DE BOURZAC":"24","MAYSEL":"60","LE MEUX":"60","MONNEVILLE":"60","MONTAGNY EN VEXIN":"60","MONTEPILLOY":"60","MONTMACQ":"60","MONTREUIL SUR THERAIN":"60","MORIENVAL":"60","MOULIN SOUS TOUVENT":"60","MOYVILLERS":"60","NEUFVY SUR ARONDE":"60","LA NEUVILLE SUR RESSONS":"60","NOIREMONT":"60","ORMOY VILLERS":"60","ORRY LA VILLE":"60","PASSEL":"60","PLAILLY":"60","LE PLESSIER SUR BULLES":"60","PLESSIS DE ROYE":"60","PONTPOINT":"60","PORQUERICOURT":"60","PUISEUX EN BRAY":"60","PUITS LA VALLEE":"60","RAINVILLERS":"60","RANTIGNY":"60","RARAY":"60","REILLY":"60","REUIL SUR BRECHE":"60","RICQUEBOURG":"60","RIVECOURT":"60","ROCHY CONDE":"60","ROY BOISSY":"60","ROYE SUR MATZ":"60","RUSSY BEMONT":"60","ST AUBIN EN BRAY":"60","LANDOUZY LA COUR":"02","LUCY LE BOCAGE":"02","LESCHELLE":"02","LIERVAL":"02","LOUATRE":"02","LEMPIRE":"02","LANCHY":"02","LAON":"02","LOR":"02","MORGNY EN THIERACHE":"02","MONTREUIL AUX LIONS":"02","MONTGRU ST HILAIRE":"02","MONT NOTRE DAME":"02","MONT D ORIGNY":"02","MONT ST PERE":"02","MONTHENAULT":"02","MONDREPUIS":"02","MONTHUREL":"02","MORSAIN":"02","LE HERIE LA VIEVILLE":"02","JUVINCOURT ET DAMARY":"02","HARTENNES ET TAUX":"02","LEHAUCOURT":"02","JEANCOURT":"02","GUIVRY":"02","HIRSON":"02","GUISE":"02","GUNY":"02","LA NEUVILLE BOSMONT":"02","LE PLESSIER HULEU":"02","NOYANT ET ACONIN":"02","OIGNY EN VALOIS":"02","OLLEZY":"02","PAISSY":"02","PASLY":"02","MARIZY STE GENEVIEVE":"02","MAAST ET VIOLAINE":"02","MARIGNY EN ORXOIS":"02","MISSY AUX BOIS":"02","LA MALMAISON":"02","MACQUIGNY":"02","MAISSEMY":"02","MARGIVAL":"02","MANICAMP":"02","MALZY":"02","PRESLES ET BOVES":"02","PUISEUX EN RETZ":"02","PREMONTRE":"02","ROGECOURT":"02","PONT ARCY":"02","PONTRUET":"02","PRIEZ":"02","ST PIERRE LES FRANQUEVILLE":"02","ST ERME OUTRE ET RAMECOURT":"02","ST NICOLAS AUX BOIS":"02","ROZIERES SUR CRISE":"02","ROUVROY SUR SERRE":"02","ST MARTIN RIVIERE":"02","ST PIERRE AIGLE":"02","VERNE":"25","CRESSAC ST GENIS":"16","DIRAC":"16","ECHALLAT":"16","FEUILLADE":"16","FONTENILLE":"16","FOUSSIGNAC":"16","GONDEVILLE":"16","GOND PONTOUVRE":"16","LE GRAND MADIEU":"16","HIESSE":"16","HOULETTE":"16","L ISLE D ESPAGNAC":"16","JAVREZAC":"16","JUILLAC LE COQ":"16","LACHAISE":"16","LAGARDE SUR LE NE":"16","LESIGNAC DURAND":"16","LIGNIERES SONNEVILLE":"16","LUPSAULT":"16","LUXE":"16","MAINE DE BOIXE":"16","MALAVILLE":"16","MARCILLAC LANVILLE":"16","MARILLAC LE FRANC":"16","MASSIGNAC":"16","MONTIGNE":"16","MONTMOREAU ST CYBARD":"16","MOUTHIERS SUR BOEME":"16","PARZAC":"16","PILLAC":"16","PLASSAC ROUFFIAC":"16","PLEUVILLE":"16","RANCOGNE":"16","RUELLE SUR TOUVRE":"16","ST ANGEAU":"16","ST AULAIS LA CHAPELLE":"16","ST CLAUD":"16","ST LAURENT DE BELZAGOT":"16","ST PREUIL":"16","ST QUENTIN SUR CHARENTE":"16","SALLES D ANGLES":"16","SALLES DE BARBEZIEUX":"16","SUAUX":"16","LA TACHE":"16","LE TATRE":"16","TORSAC":"16","TUZIE":"16","VAUX ROUILLAC":"16","VENTOUSE":"16","VERDILLE":"16","VERTEUIL SUR CHARENTE":"16","LE VIEUX CERIER":"16","VIEUX RUFFEC":"16","VILLEJESUS":"16","VILLEJOUBERT":"16","ALLAS CHAMPAGNE":"17","ANNEPONT":"17","ARCHIAC":"17","BALANZAC":"17","BEURLAY":"17","BOIS":"17","LE BOIS PLAGE EN RE":"17","BOISREDON":"17","BOSCAMNANT":"17","CHAILLEVETTE":"17","LE CHAY":"17","CHENAC ST SEURIN D UZET":"17","CHEPNIERS":"17","CHERMIGNAC":"17","CHEVANCEAUX":"17","CLERAC":"17","CONSAC":"17","CORME ECLUSE":"17","CRAZANNES":"17","CROIX CHAPEAU":"17","DOMPIERRE SUR MER":"17","ECHILLAIS":"17","L EGUILLE":"17","FLEAC SUR SEUGNE":"17","CAYRES":"43","CHAMALIERES SUR LOIRE":"43","CHARRAIX":"43","CHAUDEYROLLES":"43","CONNANGLES":"43","DUNIERES":"43","ESPALY ST MARCEL":"43","FIX ST GENEYS":"43","GRENIER MONTGON":"43","JAX":"43","JULLIANGES":"43","LANDOS":"43","LANTRIAC":"43","LAUSSONNE":"43","LAVAUDIEU":"43","LAVOUTE SUR LOIRE":"43","MALVIERES":"43","MONISTROL SUR LOIRE":"43","MONTFAUCON EN VELAY":"43","MONTREGARD":"43","PAULHAGUET":"43","LE PUY EN VELAY":"43","RAUCOULES":"43","ROCHE EN REGNIER":"43","ST ARCONS DE BARGES":"43","ST BONNET LE FROID":"43","ST DIDIER SUR DOULON":"43","ST ETIENNE DU VIGAN":"43","STE EUGENIE DE VILLENEUVE":"43","ST GEORGES LAGRICOL":"43","ST JULIEN DU PINET":"43","ST LAURENT CHABREUGES":"43","ST PAL DE MONS":"43","ST PAUL DE TARTAS":"43","ST PREJET ARMANDON":"43","SEMBADEL":"43","SIAUGUES STE MARIE":"43","SOLIGNAC SOUS ROCHE":"43","TAILHAC":"43","THORAS":"43","TORSIAC":"43","VALPRIVAS":"43","VEZEZOUX":"43","VIEILLE BRIOUDE":"43","VIELPRAT":"43","VILLENEUVE D ALLIER":"43","BATZ SUR MER":"44","LA BERNERIE EN RETZ":"44","BESNE":"44","LE BIGNON":"44","LA BOISSIERE DU DORE":"44","BOUGUENAIS":"44","VILLENEUVE EN RETZ":"44","CARQUEFOU":"44","LA CHAPELLE LAUNAY":"44","CHATEAU THEBAUD":"44","LA CHEVROLIERE":"44","CORDEMAIS":"44","CROSSAC":"44","ERBRAY":"44","GUENROUET":"44","HERIC":"44","JOUE SUR ERDRE":"44","LA MARNE":"44","MISSILLAC":"44","NORT SUR ERDRE":"44","NOTRE DAME DES LANDES":"44","PIERRIC":"44","LA PLAINE SUR MER":"44","PONT ST MARTIN":"44","POUILLE LES COTEAUX":"44","LE POULIGUEN":"44","LA REMAUDIERE":"44","ROUANS":"44","ROUGE":"44","SAFFRE":"44","ST ETIENNE DE MONTLUC":"44","ST GEREON":"44","VAIR SUR LOIRE":"44","ST JULIEN DE VOUVANTES":"44","STE REINE DE BRETAGNE":"44","ST VINCENT DES LANDES":"44","TREILLIERES":"44","VARENGUEBEC":"50","VAROUVILLE":"50","VAUDRIMESNIL":"50","LES VEYS":"50","LARGNY SUR AUTOMNE":"02","LEUILLY SOUS COUCY":"02","LAVAQUERESSE":"02","LAVERSINE":"02","LONGPONT":"02","JONCOURT":"02","LATILLY":"02","LA BASTIDE SUR L HERS":"09","LA BASTIDE DE BESPLAS":"09","AULUS LES BAINS":"09","AX LES THERMES":"09","BETCHAT":"09","ARVIGNA":"09","BESSET":"09","ASTON":"09","AULOS":"09","ST MARTIN LA CAMPAGNE":"27","STE OPPORTUNE DU BOSC":"27","ST OUEN DE PONTCHEUIL":"27","ST OUEN DU TILLEUL":"27","ST PAUL DE FOURQUES":"27","ST PIERRE DE CORMEILLES":"27","ST PIERRE DES FLEURS":"27","ST PIERRE DU BOSGUERARD":"27","ST PIERRE DU VAUVRAY":"27","ST VICTOR DE CHRETIENVILLE":"27","ST VICTOR D EPINE":"27","SASSEY":"27","LA SAUSSAYE":"27","THIERVILLE":"27","LE THIL":"27","LES THILLIERS EN VEXIN":"27","TOURVILLE SUR PONT AUDEMER":"27","LE TREMBLAY OMONVILLE":"27","LE TRONCQ":"27","TROUVILLE LA HAULE":"27","VEZILLON":"27","VILLERS EN VEXIN":"27","VILLETTES":"27","VILLIERS EN DESOEUVRE":"27","ARDELU":"28","ARGENVILLIERS":"28","LES AUTELS VILLEVILLON":"28","BARMAINVILLE":"28","BEAUCHE":"28","BOISSY EN DROUAIS":"28","BOUTIGNY PROUAIS":"28","BROUE":"28","BULLAINVILLE":"28","BULLOU":"28","CHAMPROND EN GATINE":"28","LA CHAPELLE FORTIN":"28","CHAPELLE GUILLAUME":"28","CHATAINCOURT":"28","CHAUFFOURS":"28","CLOYES SUR LE LOIR":"28","COMBRES":"28","CORMAINVILLE":"28","LA CROIX DU PERCHE":"28","DANCY":"28","DONNEMAIN ST MAMES":"28","DROUE SUR DROUETTE":"28","ESCORPAIN":"28","LA FERTE VIDAME":"28","FRIAIZE":"28","GALLARDON":"28","GARANCIERES EN BEAUCE":"28","LA GAUDAINE":"28","GOHORY":"28","GUILLONVILLE":"28","HANCHES":"28","HOUX":"28","JAUDRAIS":"28","LA LOUPE":"28","MAINTENON":"28","MANOU":"28","MARVILLE MOUTIERS BRULE":"28","MEROUVILLE":"28","MIGNIERES":"28","MONTIGNY LE CHARTIF":"28","MORANCEZ":"28","MOTTEREAU":"28","NERON":"28","NEUVY EN BEAUCE":"28","NANTHIAT":"24","NASTRINGUES":"24","NAUSSANNES":"24","LE PIZOU":"24","PONTEYRAUD":"24","PRATS DE CARLUX":"24","PRATS DU PERIGORD":"24","QUEYSSAC":"24","RAZAC D EYMET":"24","RAZAC DE SAUSSIGNAC":"24","RAZAC SUR L ISLE":"24","LA ROQUE GAGEAC":"24","SAGELAT":"24","STE ALVERE ST LAURENT LES BATONS":"24","ST AUBIN DE NABIRAT":"24","ST AULAYE PUYMANGOU":"24","ST BARTHELEMY DE BELLEGARDE":"24","STE CROIX DE MAREUIL":"24","ST CYR LES CHAMPAGNES":"24","STE EULALIE D ANS":"24","ST GEORGES BLANCANEIX":"24","ST GEORGES DE MONTCLARD":"24","ST GERMAIN DE BELVES":"24","ST JEAN D ATAUX":"24","ST JEAN DE COLE":"24","ST JEAN D ESTISSAC":"24","ST JEAN D EYRAUD":"24","ST JORY LAS BLOUX":"24","ST LAURENT LA VALLEE":"24","ST MARTIN DES COMBES":"24","ST MARTIN LE PIN":"24","ST MEDARD D EXCIDEUIL":"24","ST MICHEL DE MONTAIGNE":"24","ST PAUL DE SERRE":"24","ST PAUL LA ROCHE":"24","ST PIERRE D EYRAUD":"24","ST POMPONT":"24","ST ROMAIN ET ST CLEMENT":"24","ST SULPICE D EXCIDEUIL":"24","ST VINCENT JALMOUTIERS":"24","SARLIAC SUR L ISLE":"24","SAVIGNAC DE NONTRON":"24","SENCENAC PUY DE FOURCHES":"24","SINGLEYRAC":"24","SORGES ET LIGUEUX EN PERIGORD":"24","SOULAURES":"24","THONAC":"24","TREMOLAT":"24","URVAL":"24","VALLEREUIL":"24","VELINES":"24","VERGT DE BIRON":"24","VERTEILLAC":"24","VILLAC":"24","ABBEVILLERS":"25","ADAM LES PASSAVANT":"25","ALLONDANS":"25","AMATHAY VESIGNEUX":"25","AMONDANS":"25","ARC ET SENANS":"25","ARC SOUS CICON":"25","AUTECHAUX ROIDE":"25","AVILLEY":"25","BAUME LES DAMES":"25","LE BELIEU":"25","BELLEHERBE":"25","BERCHE":"25","BETHONCOURT":"25","LE BIZOT":"25","BONNETAGE":"25","BOURNOIS":"25","BRETIGNEY":"25","BRETIGNEY NOTRE DAME":"25","BRETONVILLERS":"25","BYANS SUR DOUBS":"25","GUMIERES":"42","L HOPITAL LE GRAND":"42","LUPE":"42","LURIECQ":"42","MAGNEUX HAUTE RIVE":"42","MARLHES":"42","MORNAND EN FOREZ":"42","LES NOES":"42","NOTRE DAME DE BOISSET":"42","PALOGNEUX":"42","PAVEZIN":"42","ST THIBAUT":"02","SAULCHERY":"02","MONTCOMBROUX LES MINES":"03","PIERREFITTE SUR LOIRE":"03","NEUILLY EN DONJON":"03","NASSIGNY":"03","MAZERIER":"03","POEZAT":"03","ST MARTIN DU TILLEUL":"27","ST MARTIN ST FIRMIN":"27","ST MESLIN DU BOSC":"27","ST OUEN DE THOUBERVILLE":"27","ST PHILBERT SUR RISLE":"27","ST PIERRE DU VAL":"27","ST SYLVESTRE DE CORMEILLES":"27","SAUSSAY LA CAMPAGNE":"27","SEREZ":"27","SURTAUVILLE":"27","TILLEUL DAME AGNES":"27","TOURNEDOS SUR SEINE":"27","TOURNEVILLE":"27","LA TRINITE DE REVILLE":"27","LES VENTES":"27","VILLERS SUR LE ROULE":"27","VITOT":"27","ARROU":"28","AUNAY SOUS CRECY":"28","AUTHEUIL":"28","BAILLEAU LE PIN":"28","BERCHERES SUR VESGRE":"28","BILLANCELLES":"28","LE BOULLAY LES DEUX EGLISES":"28","CHAMPHOL":"28","LA CHAPELLE DU NOYER":"28","CHARONVILLE":"28","CHARTAINVILLIERS":"28","CHARTRES":"28","CHASSANT":"28","LES CHATELETS":"28","COLTAINVILLE":"28","COURBEHAYE":"28","COURTALAIN":"28","COURVILLE SUR EURE":"28","CRECY COUVE":"28","DENONVILLE":"28","DREUX":"28","EPERNON":"28","FESSANVILLIERS MATTANVILLIERS":"28","FONTAINE LA GUYON":"28","FONTENAY SUR EURE":"28","GARNAY":"28","GAS":"28","LE GAULT ST DENIS":"28","GELLAINVILLE":"28","GOUILLONS":"28","GUILLEVILLE":"28","HAPPONVILLIERS":"28","JALLANS":"28","LETHUIN":"28","LEVAINVILLE":"28","LOUVILLIERS EN DROUAIS":"28","LUCE":"28","LUMEAU":"28","LUPLANTE":"28","MAILLEBOIS":"28","MAINVILLIERS":"28","MARCHEVILLE":"28","MEREGLISE":"28","LE MESNIL THOMAS":"28","MEZIERES AU PERCHE":"28","MEZIERES EN DROUAIS":"28","MOLEANS":"28","MONTBOISSIER":"28","NOGENT LE PHAYE":"28","NOTTONVILLE":"28","OUARVILLE":"28","OZOIR LE BREUIL":"28","POISVILLIERS":"28","PRASVILLE":"28","GEMOZAC":"17","LA JARNE":"17","JAZENNES":"17","LAGORD":"17","LANDRAIS":"17","LOIRE SUR NIE":"17","MIGRON":"17","MONTENDRE":"17","MONTILS":"17","MONTPELLIER DE MEDILLAN":"17","NANCRAS":"17","NERE":"17","NIEUL LES SAINTES":"17","NIEUL LE VIROUIL":"17","PISANY":"17","PREGUILLAC":"17","RIVEDOUX PLAGE":"17","ST BRIS DES BOIS":"17","ST CYR DU DORET":"17","ST DENIS D OLERON":"17","ST FROULT":"17","ST GENIS DE SAINTONGE":"17","ST GEORGES ANTIGNAC":"17","ST GEORGES DES COTEAUX":"17","ST GERMAIN DU SEUDRE":"17","ST HILAIRE DE VILLEFRANCHE":"17","ST JEAN D ANGELY":"17","ST MAIGRIN":"17","ST MANDE SUR BREDOIRE":"17","STE MARIE DE RE":"17","ST PALAIS DE PHIOLIN":"17","ST ROGATIEN":"17","ST SEVER DE SAINTONGE":"17","ST SEVERIN SUR BOUTONNE":"17","ST SIMON DE BORDES":"17","STE SOULLE":"17","SALEIGNES":"17","SALLES SUR MER":"17","SAUJON":"17","SEMILLAC":"17","SEMUSSAC":"17","TAILLANT":"17","LA TREMBLADE":"17","TRIZAY":"17","LA VALLEE":"17","VANDRE":"17","LA VERGNE":"17","VILLARS EN PONS":"17","VILLIERS COUTURE":"17","VINAX":"17","YVES":"17","LES AIX D ANGILLON":"18","BANNEGON":"18","BEFFES":"18","BOULLERET":"18","LA CHAPELLE HUGON":"18","LA CHAPELOTTE":"18","CREZANCAY SUR CHER":"18","CUFFY":"18","CULAN":"18","DAMPIERRE EN CROT":"18","FARGES EN SEPTAINE":"18","FUSSY":"18","GARIGNY":"18","LA GUERCHE SUR L AUBOIS":"18","LERE":"18","LISSAY LOCHY":"18","LUGNY BOURBONNAIS":"18","MERY ES BOIS":"18","NOHANT EN GOUT":"18","OIZON":"18","ORCENAIS":"18","LA PERCHE":"18","ST AIGNAN DES NOYERS":"18","ST BAUDEL":"18","ST CEOLS":"18","ST CHRISTOPHE LE CHAUDRY":"18","ST DOULCHARD":"18","ST GEORGES DE POISIEUX":"18","TRIGNAC":"44","VALLET":"44","LOIREAUXENCE":"44","VERTOU":"44","VILLEPOT":"44","ANDONVILLE":"45","AUDEVILLE":"45","AULNAY LA RIVIERE":"45","BARVILLE EN GATINAIS":"45","BATILLY EN GATINAIS":"45","BAULE":"45","BEAUCHAMPS SUR HUILLARD":"45","BONNEE":"45","BOUILLY EN GATINAIS":"45","BRETEAU":"45","BRIARRES SUR ESSONNE":"45","CEPOY":"45","CERNOY EN BERRY":"45","CHAMBON LA FORET":"45","CHANTEAU":"45","CHANTECOQ":"45","LA CHAPELLE ST MESMIN":"45","CHATEAUNEUF SUR LOIRE":"45","COULMIERS":"45","COURTEMAUX":"45","DONNERY":"45","DORDIVES":"45","ECHILLEUSES":"45","ENGENVILLE":"45","ESTOUY":"45","FERRIERES EN GATINAIS":"45","GEMIGNY":"45","GIVRAINES":"45","GRENEVILLE EN BEAUCE":"45","ISDES":"45","JOUY EN PITHIVERAIS":"45","LOMBREUIL":"45","MENESTREAU EN VILLETTE":"45","MEUNG SUR LOIRE":"45","MONTEREAU":"45","NEVOY":"45","NIBELLE":"45","NOGENT SUR VERNISSON":"45","BOURGUIGNON SOUS MONTBAVIN":"02","BOHAIN EN VERMANDOIS":"02","BOUFFIGNEREUX":"02","BICHANCOURT":"02","BOURESCHES":"02","BRUMETZ":"02","BONNEIL":"02","ESSOMES SUR MARNE":"02","ESSIGNY LE PETIT":"02","DIZY LE GROS":"02","EPAUX BEZU":"02","EBOULEAU":"02","DRAVEGNY":"02","DANIZY":"02","DOUCHY":"02","EFFRY":"02","LA CHAPELLE SUR CHEZY":"02","CHIVRES EN LAONNOIS":"02","CHATILLON SUR OISE":"02","CLACY ET THIERRET":"02","CELLES SUR AISNE":"02","CHERY CHARTREUVE":"02","CIRY SALSOGNE":"02","LE CHARMEL":"02","COUVRON ET AUMENCOURT":"02","COEUVRES ET VALSERY":"02","CRECY SUR SERRE":"02","COUCY LES EPPES":"02","DAGNY LAMBERCY":"02","CYS LA COMMUNE":"02","CONCEVREUX":"02","COYOLLES":"02","CUGNY":"02","BAZOCHES SUR VESLES":"02","BEAUMONT EN BEINE":"02","BASSOLES AULERS":"02","BEZU ST GERMAIN":"02","BEAUREVOIR":"02","BANCIGNY":"02","BEAUTOR":"02","NOGENT SUR EURE":"28","ORROUER":"28","OUERRE":"28","OYSONVILLE":"28","PIERRES":"28","LES PINTHIERES":"28","POINVILLE":"28","PRUNAY LE GILLON":"28","RECLAINVILLE":"28","ST ARNOULT DES BOIS":"28","STE GEMME MORONVAL":"28","ST ELIPH":"28","ST MARTIN DE NIGELLES":"28","ST MAUR SUR LE LOIR":"28","ST MAURICE ST GERMAIN":"28","ST SAUVEUR MARVILLE":"28","ST VICTOR DE BUTHON":"28","SAINVILLE":"28","SANCHEVILLE":"28","SANDARVILLE":"28","LA SAUCELLE":"28","SERAZEREUX":"28","SOREL MOUSSEL":"28","SOUANCE AU PERCHE":"28","THIRON GARDAIS":"28","THIVILLE":"28","TREON":"28","UMPEAU":"28","VAUPILLON":"28","VER LES CHARTRES":"28","VILLEBON":"28","VILLIERS ST ORIEN":"28","YEVRES":"28","BODILIS":"29","BRIEC":"29","CHATEAUNEUF DU FAOU":"29","CLOHARS CARNOET":"29","ELLIANT":"29","GARLAN":"29","GUILVINEC":"29","GUIMAEC":"29","HANVEC":"29","HUELGOAT":"29","ILE DE SEIN":"29","KERNOUES":"29","LAMPAUL GUIMILIAU":"29","LAMPAUL PLOUDALMEZEAU":"29","LANARVILY":"29","LANDELEAU":"29","LANDIVISIAU":"29","LANDUDAL":"29","LANMEUR":"29","LAZ":"29","LOCQUIREC":"29","LOPEREC":"29","LOPERHET":"29","MELLAC":"29","MESPAUL":"29","MILIZAC":"29","MOTREFF":"29","PLEUVEN":"29","PLOBANNALEC LESCONIL":"29","PLOGASTEL ST GERMAIN":"29","PLOGOFF":"29","PLOUARZEL":"29","PLOUDIRY":"29","PLOUEDERN":"29","PLOUENAN":"29","PLOUGONVELIN":"29","PLOUGOULM":"29","PLOUIDER":"29","PLOUMOGUER":"29","PLOUNEVENTER":"29","PLOUYE":"29","PLOUZANE":"29","PONT L ABBE":"29","POULLAN SUR MER":"29","POULLAOUEN":"29","LE RELECQ KERHUON":"29","RIEC SUR BELON":"29","LA ROCHE MAURICE":"29","ST POL DE LEON":"29","STE SEVE":"29","ST VOUGAY":"29","RENAISON":"42","RIVAS":"42","ROZIER COTES D AUREC":"42","STE AGATHE LA BOUTERESSE":"42","ST BONNET DES QUARTS":"42","ST BONNET LES OULES":"42","STE CROIX EN JAREZ":"42","ST CYR DE FAVIERES":"42","ST ETIENNE LE MOLARD":"42","ST FORGEUX LESPINASSE":"42","STE FOY ST SULPICE":"42","ST GEORGES EN COUZAN":"42","ST GERMAIN LA MONTAGNE":"42","ST GERMAIN LESPINASSE":"42","ST HAON LE CHATEL":"42","ST HEAND":"42","ST JEAN BONNEFONDS":"42","ST JEAN SOLEYMIEUX":"42","ST JUST EN BAS":"42","ST LAURENT LA CONCHE":"42","ST MARTIN D ESTREAUX":"42","ST MICHEL SUR RHONE":"42","ST ROMAIN D URFE":"42","ST ROMAIN LE PUY":"42","ST SAUVEUR EN RUE":"42","ST VICTOR SUR RHINS":"42","SALVIZINET":"42","SAUVAIN":"42","SOUTERNON":"42","LA TERRASSE SUR DORLAY":"42","THELIS LA COMBE":"42","VALEILLE":"42","LA VALLA SUR ROCHEFORT":"42","VIOLAY":"42","VIRICELLES":"42","AGNAT":"43","ARAULES":"43","BAINS":"43","BLASSAC":"43","LE BRIGNON":"43","BRIOUDE":"43","LE CHAMBON SUR LIGNON":"43","CHAMPCLAUSE":"43","CISTRIERES":"43","COUTEUGES":"43","DESGES":"43","DOMEYRAT":"43","FONTANNES":"43","FREYCENET LA CUCHE":"43","FRUGERES LES MINES":"43","GOUDET":"43","JOSAT":"43","LAVAL SUR DOULON":"43","LOUDES":"43","LUBILHAC":"43","MEZERES":"43","MONTCLARD":"43","MOUDEYRES":"43","OUIDES":"43","PINOLS":"43","ST BERAIN":"43","ST CHRISTOPHE D ALLIER":"43","STE FLORINE":"43","ST JEAN D AUBRIGOUX":"43","ST JEAN DE NAY":"43","ST JEAN LACHALM":"43","ST JULIEN DES CHAZES":"43","ST MARTIN DE FUGERES":"43","ST PAL DE CHALENCON":"43","ST PIERRE DU CHAMP":"43","VAZEILLES LIMANDRE":"43","VENTEUGES":"43","VERGEZAC":"43","VOREY":"43","YSSINGEAUX":"43","AIGREFEUILLE SUR MAINE":"44","BOUAYE":"44","LA CHAPELLE GLAIN":"44","LA CHAPELLE HEULIN":"44","LA CHAPELLE SUR ERDRE":"44","CHEIX EN RETZ":"44","CORSEPT":"44","FAY DE BRETAGNE":"44","GRANDCHAMPS DES FONTAINES":"44","HAUTE GOULAINE":"44","JANS":"44","PRE ST EVROULT":"28","PRE ST MARTIN":"28","LE PUISET":"28","LES RESSUINTES":"28","ROHAIRE":"28","ROMILLY SUR AIGRE":"28","RUEIL LA GADELIERE":"28","ST DENIS DES PUITS":"28","ST GEORGES SUR EURE":"28","ST LUBIN DE CRAVANT":"28","ST LUBIN DE LA HAYE":"28","ST MAIXME HAUTERIVE":"28","SOURS":"28","TERMINIERS":"28","UNVERRE":"28","VICHERES":"28","VILLEMEUX SUR EURE":"28","ARZANO":"29","BANNALEC":"29","BOHARS":"29","BRASPARTS":"29","CARHAIX PLOUGUER":"29","CHATEAULIN":"29","CLEDEN CAP SIZUN":"29","CLEDEN POHER":"29","CLEDER":"29","LE CLOITRE ST THEGONNEC":"29","CONCARNEAU":"29","DINEAULT":"29","EDERN":"29","GOUESNACH":"29","GUERLESQUIN":"29","GUILER SUR GOYEN":"29","GUIPRONVEL":"29","GUISSENY":"29","IRVILLAC":"29","LE JUCH":"29","KERGLOFF":"29","KERSAINT PLABENNEC":"29","LAMPAUL PLOUARZEL":"29","LANDEDA":"29","LANDREVARZEC":"29","LANDUNVEZ":"29","LANNEANOU":"29","LANRIVOARE":"29","LEUHAN":"29","LOCMELAR":"29","LOCTUDY":"29","LOQUEFFRET":"29","MAHALON":"29","MOELAN SUR MER":"29","PEUMERIT":"29","PLOUDANIEL":"29","PLOUEGAT GUERAND":"29","PLOUESCAT":"29","PLOURIN":"29","PLOUVORN":"29","QUIMPERLE":"29","REDENE":"29","ROSCANVEL":"29","ST PABU":"29","ST RENAN":"29","ST SEGAL":"29","ST THOIS":"29","SANTEC":"29","SIBIRIL":"29","SPEZET":"29","TAULE":"29","TREBABU":"29","TREGARVAN":"29","PONT DE BUIS LES QUIMERCH":"29","AIGUES MORTES":"30","AIMARGUES":"30","ARAMON":"30","ARRE":"30","AUBUSSARGUES":"30","AULAS":"30","BEZOUCE":"30","BORDEZAC":"30","BRIGNON":"30","ST LEGER LE PETIT":"18","STE MONTAINE":"18","ST PIERRE LES BOIS":"18","ST PIERRE LES ETIEUX":"18","SENNECAY":"18","SEVRY":"18","UZAY LE VENON":"18","VAILLY SUR SAULDRE":"18","VESDUN":"18","AFFIEUX":"19","ALBIGNAC":"19","ALLASSAC":"19","AUBAZINES":"19","BELLECHASSAGNE":"19","BEYNAT":"19","BEYSSAC":"19","BILHAC":"19","BORT LES ORGUES":"19","BRIVE LA GAILLARDE":"19","LE CHASTANG":"19","CHASTEAUX":"19","CHAVANAC":"19","CORNIL":"19","CUREMONTE":"19","DARAZAC":"19","LACELLE":"19","LAFAGE SUR SOMBRE":"19","LAGLEYGEOLLE":"19","LAROCHE PRES FEYT":"19","LASCAUX":"19","LE LONZAC":"19","LUBERSAC":"19","MALEMORT":"19","MESTES":"19","MONTAIGNAC ST HIPPOLYTE":"19","MOUSTIER VENTADOUR":"19","NONARDS":"19","OBJAT":"19","PERPEZAC LE NOIR":"19","LE PESCHER":"19","PUY D ARNAC":"19","QUEYSSAC LES VIGNES":"19","LA ROCHE CANILLAC":"19","ST BONNET AVALOUZE":"19","ST BONNET ELVERT":"19","ST CYR LA ROCHE":"19","ST ELOY LES TUILERIES":"19","STE FORTUNADE":"19","ST HILAIRE FOISSAC":"19","ST JAL":"19","ST JULIEN AUX BOIS":"19","ST JULIEN LE VENDOMOIS":"19","ST JULIEN PRES BORT":"19","ST MARTIAL ENTRAYGUES":"19","ST MERD DE LAPLEAU":"19","ST PANTALEON DE LAPLEAU":"19","ST PARDOUX CORBIER":"19","SOUDAINE LAVINADIERE":"19","TOY VIAM":"19","TREIGNAC":"19","VIGEOIS":"19","AGEY":"21","AISY SOUS THIL":"21","ANTHEUIL":"21","ANTIGNY LA VILLE":"21","ARRANS":"21","AUBIGNY LES SOMBERNON":"21","AUTRICOURT":"21","BAIGNEUX LES JUIFS":"21","BEIRE LE CHATEL":"21","BELLENOD SUR SEINE":"21","BESSEY EN CHAUME":"21","BESSEY LES CITEAUX":"21","BEUREY BAUGUAY":"21","BINGES":"21","BLAGNY SUR VINGEANNE":"21","BOUSSEY":"21","BUNCEY":"21","CENSEREY":"21","CHAMBOLLE MUSIGNY":"21","CHAMP D OISEAU":"21","CHAMPRENAULT":"21","CHARREY SUR SEINE":"21","CHAUDENAY LE CHATEAU":"21","LA FERTE MILON":"02","FESMY LE SART":"02","FLAVY LE MARTEL":"02","ETOUVELLES":"02","FLUQUIERES":"02","GOUDELANCOURT LES PIERREPONT":"02","FRANCILLY SELENCY":"02","FONTAINE UTERTE":"02","GRICOURT":"02","GRUGIES":"02","GIZY":"02","SIMIANE LA ROTONDE":"04","ST MARTIN LES SEYNE":"04","SOURRIBES":"04","ST LIONS":"04","ST JURS":"04","SIGONCE":"04","LA MOTTE EN CHAMPSAUR":"05","NOSSAGE ET BENEVENT":"05","PELLEAUTIER":"05","MOYDANS":"05","NEFFES":"05","OZE":"05","LE CASTELLARD MELAN":"04","LE CHAFFAUT ST JURSON":"04","LE BRUSQUET":"04","LA BREOLE":"04","ARCHAIL":"04","BLIEUX":"04","BEVONS":"04","BANON":"04","ALLOS":"04","ALLEMAGNE EN PROVENCE":"04","VARENNES SUR TECHE":"03","VALLON EN SULLY":"03","VALIGNAT":"03","TRONGET":"03","VALIGNY":"03","VOUSSAC":"03","SUSSAT":"03","ST GERMAIN DE SALLES":"03","ST DIDIER EN DONJON":"03","ST MARTIN DES LAIS":"03","ST FARGEOL":"03","SERVILLY":"03","SEUILLET":"03","SAULZET":"03","ST PONT":"03","LE MAYET DE MONTAGNE":"03","ST AUBIN LE MONIAL":"03","MONETAY SUR LOIRE":"03","MONTAIGU LE BLIN":"03","NIZEROLLES":"03","MEILLERS":"03","RONNET":"03","LA ROCHEGIRON":"04","PEYROULES":"04","PEYRUIS":"04","PONTIS":"04","PEIPIN":"04","RIEZ":"04","MARCILLAT EN COMBRAILLE":"03","FERRIERES SUR SICHON":"03","ESPINASSE VOZELLE":"03","LAVAULT STE ANNE":"03","HERISSON":"03","MARIOL":"03","GOUISE":"03","HURIEL":"03","LANGY":"03","BUXIERES LES MINES":"03","CHATEL DE NEUVRE":"03","CHAREIL CINTRAT":"03","BIZENEUILLE":"03","CHAMBERAT":"03","BRUGHEAS":"03","BLOMARD":"03","COSNE D ALLIER":"03","ST YVI":"29","TREFLEVENEZ":"29","TREGARANTEC":"29","ALZON":"30","ARPAILLARGUES ET AUREILLAC":"30","BAGNOLS SUR CEZE":"30","LA BASTIDE D ENGRAS":"30","BEZ ET ESPARON":"30","BOISSET ET GAUJAC":"30","BRANOUX LES TAILLADES":"30","BREAU ET SALAGOSSE":"30","BROUZET LES QUISSAC":"30","LE CAILAR":"30","CAMPESTRE ET LUC":"30","CANNES ET CLAIRAN":"30","CENDRAS":"30","COLOGNAC":"30","CONCOULES":"30","CONNAUX":"30","CONQUEYRAC":"30","CRUVIERS LASCOURS":"30","DIONS":"30","GOUDARGUES":"30","JONQUIERES ST VINCENT":"30","LANGLADE":"30","LASALLE":"30","LAUDUN L ARDOISE":"30","LIRAC":"30","LOGRIAN FLORIAN":"30","MALONS ET ELZE":"30","MARTIGNARGUES":"30","MASSANES":"30","MEYNES":"30","MOLIERES CAVAILLAC":"30","MONOBLET":"30","MONTMIRAT":"30","REMOULINS":"30","ST ANDRE D OLERARGUES":"30","ST BONNET DE SALENDRINQUE":"30","STE CECILE D ANDORGE":"30","ST ETIENNE DE L OLM":"30","ST FLORENT SUR AUZONNET":"30","ST GENIES DE MALGOIRES":"30","ST GERVASY":"30","ST JEAN DE CRIEULON":"30","ST JULIEN DE CASSAGNAS":"30","ST PAUL LA COSTE":"30","ST VICTOR LA COSTE":"30","SERNHAC":"30","SOUVIGNARGUES":"30","SUMENE":"30","VALLABRIX":"30","VALLERAUGUE":"30","VERGEZE":"30","VIC LE FESQ":"30","VILLENEUVE LES AVIGNON":"30","VISSEC":"30","AGASSAC":"31","ALAN":"31","AMBAX":"31","ANTICHAN DE FRONTIGNES":"31","ASPRET SARRAT":"31","AUREVILLE":"31","AURIGNAC":"31","AZAS":"31","BACHAS":"31","BALMA":"31","BAZIEGE":"31","BEAUMONT SUR LEZE":"31","BELLEGARDE STE MARIE":"31","BELLESSERRE":"31","BESSIERES":"31","BLAGNAC":"31","LE LANDREAU":"44","LOUISFERT":"44","LUSANGER":"44","MARSAC SUR DON":"44","MASSERAC":"44","MONTBERT":"44","MONTOIR DE BRETAGNE":"44","MOUZEIL":"44","LE PALLET":"44","LE PELLERIN":"44","ST COLOMBAN":"44","ST HILAIRE DE CHALEONS":"44","ST JOACHIM":"44","ST MARS DE COUTAIS":"44","ST PHILBERT DE GRAND LIEU":"44","SAVENAY":"44","LES SORINIERES":"44","SUCE SUR ERDRE":"44","THOUARE SUR LOIRE":"44","TREFFIEUX":"44","VRITZ":"44","GENESTON":"44","ATTRAY":"45","AUTRY LE CHATEL":"45","BOIGNY SUR BIONNE":"45","BOU":"45","BOUGY LEZ NEUVILLE":"45","BOUZY LA FORET":"45","BROMEILLES":"45","CHAILLY EN GATINAIS":"45","CHAMPOULET":"45","CHARMONT EN BEAUCE":"45","CHEVILLY":"45","CHILLEURS AUX BOIS":"45","LES CHOUX":"45","CHUELLES":"45","DAMMARIE SUR LOING":"45","EPIEDS EN BEAUCE":"45","ESCRIGNELLES":"45","FEINS EN GATINAIS":"45","INGRANNES":"45","JURANVILLE":"45","LADON":"45","LEOUVILLE":"45","LIGNY LE RIBAULT":"45","LOUZOUER":"45","MARSAINVILLIERS":"45","MONTARGIS":"45","MONTLIARD":"45","MORVILLE EN BEAUCE":"45","NARGIS":"45","NEUVILLE AUX BOIS":"45","ONDREVILLE SUR ESSONNE":"45","PAUCOURT":"45","PIERREFITTE ES BOIS":"45","PITHIVIERS LE VIEIL":"45","PRESNOY":"45","ROUVRAY STE CROIX":"45","ROZIERES EN BEAUCE":"45","ROZOY LE VIEIL":"45","ST BENOIT SUR LOIRE":"45","ST HILAIRE LES ANDRESIS":"45","ST HILAIRE SUR PUISEAUX":"45","ST JEAN DE LA RUELLE":"45","ST LOUP DE GONOIS":"45","ST LYE LA FORET":"45","ST MARTIN SUR OCRE":"45","LA SELLE SUR LE BIED":"45","SENNELY":"45","TAVERS":"45","TRIGUERES":"45","VILLAMBLAIN":"45","LA CADIERE ET CAMBO":"30","LA CALMETTE":"30","CALVISSON":"30","CARDET":"30","CASTILLON DU GARD":"30","CAVEIRAC":"30","CLARENSAC":"30","COMBAS":"30","DEAUX":"30","L ESTRECHURE":"30","FOURNES":"30","GARRIGUES STE EULALIE":"30","LA GRAND COMBE":"30","LECQUES":"30","LEDIGNAN":"30","LEZAN":"30","LIOUC":"30","LES MAGES":"30","MARGUERITTES":"30","MAURESSARGUES":"30","MEJANNES LES ALES":"30","MONTDARDIER":"30","MONTIGNARGUES":"30","NAVACELLES":"30","NERS":"30","ORSAN":"30","POTELIERES":"30","PUJAUT":"30","ST ALEXANDRE":"30","ST ANDRE DE ROQUEPERTUIS":"30","ST ANDRE DE VALBORGNE":"30","ST BONNET DU GARD":"30","ST CESAIRE DE GAUZIGNAN":"30","ST ETIENNE DES SORTS":"30","ST FELIX DE PALLIERES":"30","ST HILAIRE DE BRETHMAS":"30","ST HIPPOLYTE DE CATON":"30","ST JEAN DE CEYRARGUES":"30","ST JEAN DE MARUEJOLS ET AVEJAN":"30","ST JEAN DE SERRES":"30","ST JEAN DU GARD":"30","ST JULIEN DE LA NEF":"30","ST LAURENT DES ARBRES":"30","ST LAURENT LA VERNEDE":"30","ST MARTIN DE VALGALGUES":"30","ST PAULET DE CAISSON":"30","ST PRIVAT DE CHAMPCLOS":"30","ST PRIVAT DES VIEUX":"30","ST ROMAN DE CODIERES":"30","ST SEBASTIEN D AIGREFEUILLE":"30","ST VICTOR DES OULES":"30","SALINDRES":"30","SAVIGNARGUES":"30","SOUDORGUES":"30","TAVEL":"30","VENEJAN":"30","VERS PONT DU GARD":"30","VESTRIC ET CANDIAC":"30","AIGNES":"31","AIGREFEUILLE":"31","ANAN":"31","ARBON":"31","ARGUT DESSOUS":"31","ARLOS":"31","AURAGNE":"31","AURIAC SUR VENDINELLE":"31","AUSSEING":"31","AUZAS":"31","AUZEVILLE TOLOSANE":"31","BACHOS":"31","BAGNERES DE LUCHON":"31","BARBAZAN":"31","BEAUCHALOT":"31","BEAUTEVILLE":"31","BELBEZE DE LAURAGAIS":"31","BERAT":"31","BONDIGOUX":"31","BONREPOS RIQUET":"31","BOUSSAN":"31","BOUZIN":"31","BRUGUIERES":"31","BURGALAYS":"31","LE BURGAUD":"31","CARAMAN":"31","CARBONNE":"31","CASTAGNAC":"31","CASTELMAUROU":"31","LE CASTERA":"31","CHAUME ET COURCHAMP":"21","LA CHAUME":"21","CHENOVE":"21","CHEVIGNY EN VALIERE":"21","CHIVRES":"21","CHOREY LES BEAUNE":"21","CIREY LES PONTAILLER":"21","CORGOLOIN":"21","CORPEAU":"21","COUCHEY":"21","COURCELLES FREMOY":"21","COURCELLES LES MONTBARD":"21","COURLON":"21","COURTIVRON":"21","CREANCEY":"21","CREPAND":"21","CUSSEY LES FORGES":"21","DAMPIERRE EN MONTAGNE":"21","DAROIS":"21","DIANCEY":"21","DIJON":"21","DOMPIERRE EN MORVAN":"21","DREE":"21","ECHENON":"21","ERINGES":"21","ETAIS":"21","L ETANG VERGY":"21","ETORMAY":"21","FAIN LES MONTBARD":"21","FENAY":"21","LE FETE":"21","FLAGEY ECHEZEAUX":"21","FLAGEY LES AUXONNE":"21","FOISSY":"21","FONTAINES LES SECHES":"21","FRANXAULT":"21","GRANCEY SUR OURCE":"21","GRENANT LES SOMBERNON":"21","JANCIGNY":"21","JOURS LES BAIGNEUX":"21","LACANCHE":"21","LAMARGELLE":"21","LANTHES":"21","LECHATELET":"21","LICEY SUR VINGEANNE":"21","LONGEAULT":"21","MAGNY ST MEDARD":"21","LES MAILLYS":"21","MAISEY LE DUC":"21","MARCIGNY SOUS THIL":"21","MASSINGY LES SEMUR":"21","MAUVILLY":"21","MELOISEY":"21","MERCEUIL":"21","MEURSANGES":"21","MONTCEAU ET ECHARNANT":"21","LA MOTTE TERNANT":"21","NOIRON SUR BEZE":"21","ORRET":"21","PANGES":"21","PASQUES":"21","POISEUL LA VILLE ET LAPERRIERE":"21","PONTAILLER SUR SAONE":"21","POUILLENAY":"21","PRECY SOUS THIL":"21","PREMEAUX PRISSEY":"21","PREMIERES":"21","PUITS":"21","QUEMIGNY SUR SEINE":"21","REMILLY EN MONTAGNE":"21","RUFFEY LES ECHIREY":"21","SAFFRES":"21","ST MARC SUR SEINE":"21","ST NICOLAS LES CITEAUX":"21","ST PIERRE EN VAUX":"21","STE SABINE":"21","SAMEREY":"21","SAULON LA CHAPELLE":"21","DOMESSARGUES":"30","FLAUX":"30","FRESSAC":"30","GAGNIERES":"30","LAMELOUZE":"30","CHATELPERRON":"03","CHAVROCHES":"03","COMMENTRY":"03","COULEUVRE":"03","CHAZEMAIS":"03","CONTIGNY":"03","CHAVENON":"03","COURCAIS":"03","EBREUIL":"03","VAL D ORONAYE":"04","MONTAGNAC MONTPEZAT":"04","MONTSALIER":"04","MELVE":"04","ESPARRON DE VERDON":"04","FAUCON DU CAIRE":"04","CHAMPTERCIER":"04","MALLEMOISSON":"04","LAMBRUISSE":"04","MAJASTRES":"04","ENTREVAUX":"04","MANOSQUE":"04","ARPHEUILLES ST PRIEST":"03","BELLERIVE SUR ALLIER":"03","AUTRY ISSARDS":"03","BILLEZOIS":"03","ARRONNES":"03","ABREST":"03","BIOZAT":"03","WIMY":"02","VILLERS LES GUISE":"02","VIRY NOUREUIL":"02","GRAND VERLY":"02","VENDELLES":"02","VIEL ARCY":"02","VERDILLY":"02","VENIZEL":"02","VOYENNE":"02","VIFFORT":"02","ST ANDRE LES ALPES":"04","ST JACQUES":"04","ROUMOULES":"04","ROUGON":"04","BARCILLONNETTE":"05","LA BATIE NEUVE":"05","CHAUFFAYER":"05","BUISSARD":"05","VOLONNE":"04","UBRAYE":"04","LE MONETIER LES BAINS":"05","ETOILE ST CYRICE":"05","FREISSINIERES":"05","MONT DAUPHIN":"05","MONTGENEVRE":"05","LA FAURIE":"05","EYGLIERS":"05","CHORGES":"05","EMBRUN":"05","LE DEVOLUY":"05","ST JACQUES EN VALGODEMARD":"05","ST EUSEBE EN CHAMPSAUR":"05","ST LEGER LES MELEZES":"05","PUY ST VINCENT":"05","CHAMPIGNOL LEZ MONDEVILLE":"10","BRILLECOURT":"10","BOURDENAY":"10","BOURANTON":"10","BERGERES":"10","BEUREY":"10","BAYEL":"10","BESSY":"10","FOULEIX":"24","GAUGEAC":"24","GROLEJAC":"24","JAURE":"24","JAVERLHAC ET LA CHAPELLE ST ROBERT":"24","RUDEAU LADOSSE":"24","LAMONZIE MONTASTRUC":"24","LAMOTHE MONTRAVEL":"24","LUSIGNAC":"24","LUSSAS ET NONTRONNEAU":"24","BONREPOS SUR AUSSONNELLE":"31","BOURG D OUEIL":"31","BOURG ST BERNARD":"31","BRETX":"31","BUZET SUR TARN":"31","CABANAC CAZAUX":"31","CABANAC SEGUENVILLE":"31","CASTELBIAGUE":"31","CASTELNAU D ESTRETEFONDS":"31","CASTILLON DE LARBOUST":"31","CAUBIAC":"31","CAZARIL TAMBOURES":"31","CAZERES":"31","CHARLAS":"31","CHEIN DESSUS":"31","CIER DE RIVIERE":"31","CINTEGABELLE":"31","COLOMIERS":"31","CORNEBARRIEU":"31","DONNEVILLE":"31","DREMIL LAFAGE":"31","EAUNES":"31","EOUX":"31","ESCANECRABE":"31","ESTENOS":"31","FIGAROL":"31","FRANQUEVIELLE":"31","FRONTON":"31","FROUZINS":"31","FUSTIGNAC":"31","GARDOUCH":"31","GAURE":"31","GOUAUX DE LUCHON":"31","GOUTEVERNISSE":"31","GOYRANS":"31","GURAN":"31","HUOS":"31","LABROQUERE":"31","LAHAGE":"31","LANTA":"31","LAPEYRERE":"31","LATOUR":"31","LAUNAC":"31","LEGUEVIN":"31","LESTELLE DE ST MARTORY":"31","LOUBENS LAURAGAIS":"31","MARIGNAC LASPEYRES":"31","MARTRES TOLOSANE":"31","MASCARVILLE":"31","MAUREMONT":"31","MAUZAC":"31","MELLES":"31","MENVILLE":"31","MONESTROL":"31","MONTAIGUT SUR SAVE":"31","MONTASTRUC DE SALIES":"31","MONTBERON":"31","MONTBRUN BOCAGE":"31","MONTESQUIEU GUITTAUT":"31","MONTGEARD":"31","MONTJOIRE":"31","MONTOUSSIN":"31","MOURVILLES HAUTES":"31","NIZAN GESSE":"31","NOGARET":"31","OO":"31","PECHABOU":"31","PELLEPORT":"31","PONLAT TAILLEBOURG":"31","PORTET DE LUCHON":"31","RIEUCAZE":"31","ROUMENS":"31","ST ARAILLE":"31","ST AVENTIN":"31","ST BEAT":"31","ST FERREOL DE COMMINGES":"31","STE FOY D AIGREFEUILLE":"31","ST JULIEN SUR GARONNE":"31","ST LYS":"31","ST MARCEL PAULEL":"31","SAMOUILLAN":"31","SAUVETERRE DE COMMINGES":"31","SAVERES":"31","SEILH":"31","VILLEVOQUES":"45","VITRY AUX LOGES":"45","ANGLARS NOZAC":"46","BALADOU":"46","BELFORT DU QUERCY":"46","BIARS SUR CERE":"46","LE BOURG":"46","CADRIEU":"46","CAHUS":"46","CARENNAC":"46","CATUS":"46","CAVAGNAC":"46","CREMPS":"46","CUZANCE":"46","ESPAGNAC STE EULALIE":"46","FAYCELLES":"46","FRANCOULES":"46","FRAYSSINHES":"46","GIRAC":"46","ISSENDOLUS":"46","ISSEPTS":"46","LABASTIDE MARNHAC":"46","LABURGADE":"46","LACHAPELLE AUZAC":"46","LADIRAT":"46","LAMAGDELAINE":"46","LAMOTHE FENELON":"46","LAROQUE DES ARCS":"46","LATOUILLE LENTILLAC":"46","LAVERCANTIERE":"46","LENTILLAC ST BLAISE":"46","LUNAN":"46","MONTET ET BOUXAL":"46","MONTGESTY":"46","NUZEJOULS":"46","PAYRAC":"46","PESCADOIRES":"46","PLANIOLES":"46","PRUDHOMAT":"46","RAMPOUX":"46","SABADEL LATRONQUIERE":"46","SABADEL LAUZES":"46","ST CERE":"46","ST DAUNES":"46","ST DENIS CATUS":"46","ST DENIS LES MARTEL":"46","ST JEAN DE LAUR":"46","ST LAURENT LOLMIE":"46","ST PIERRE TOIRAC":"46","ST SOZY":"46","ST VINCENT DU PENDIT":"46","SAUX":"46","SONAC":"46","SOUCIRAC":"46","UZECH":"46","AGEN":"47","AGNAC":"47","AURADOU":"47","BIRAC SUR TREC":"47","BLANQUEFORT SUR BRIOLANCE":"47","BOE":"47","BON ENCONTRE":"47","BOURLENS":"47","BOUSSES":"47","CHAMESEY":"25","CHAMPOUX":"25","CHAPELLE DES BOIS":"25","CHASSAGNE ST DENIS":"25","CHAUX LES PASSAVANT":"25","LA CHEVILLOTTE":"25","CONSOLATION MAISONNETTES":"25","CORCELLE MIESLOT":"25","COURCHAPON":"25","COUR ST MAURICE":"25","COURVIERES":"25","CROSEY LE PETIT":"25","CASTILLON DE ST MARTORY":"31","CAZARILH LASPENES":"31","CHAUM":"31","CORRONSAC":"31","ENCAUSSE LES THERMES":"31","ESTADENS":"31","FALGA":"31","LE FAUGA":"31","FOUGARON":"31","FRANCAZAL":"31","GAILLAC TOULZA":"31","GALIE":"31","GARIN":"31","GENSAC DE BOULOGNE":"31","GIBEL":"31","GOURDAN POLIGNAN":"31","GREPIAC":"31","LE GRES":"31","HERRAN":"31","JUZET DE LUCHON":"31","JUZET D IZAUT":"31","LAGRAULET ST NICOLAS":"31","LAPEYROUSE FOSSAT":"31","LARCAN":"31","LAREOLE":"31","LAUNAGUET":"31","LAUZERVILLE":"31","LAYRAC SUR TARN":"31","LEZ":"31","LILHAC":"31","LODES":"31","MARQUEFAVE":"31","MARTISSERRE":"31","MAURESSAC":"31","MONTASTRUC SAVES":"31","MONTAUBAN DE LUCHON":"31","MONTBERNARD":"31","MONTCLAR LAURAGAIS":"31","MONTGAILLARD LAURAGAIS":"31","MONTGAZIN":"31","MOUSTAJON":"31","NENIGAN":"31","PIN BALMA":"31","POINTIS DE RIVIERE":"31","POINTIS INARD":"31","POUZE":"31","RAZECUEILLE":"31","RIEUMAJOU":"31","ROUEDE":"31","ROUFFIAC TOLOSAN":"31","ST BERTRAND DE COMMINGES":"31","ST CLAR DE RIVIERE":"31","ST FRAJOU":"31","ST IGNAN":"31","ST JEAN":"31","ST JEAN LHERM":"31","ST JORY":"31","ST LOUP CAMMAS":"31","ST LOUP EN COMMINGES":"31","ST MARCET":"31","ST MARTORY":"31","ST PE DELBOSC":"31","ST RUSTICE":"31","SAJAS":"31","SAMAN":"31","SEILHAN":"31","SOUEICH":"31","TOUILLE":"31","TOULOUSE":"31","L UNION":"31","VACQUIERS":"31","VENERQUE":"31","VILLARIES":"31","VILLAUDRIC":"31","VILLEMATIER":"31","VILLENEUVE LES BOULOC":"31","VILLENEUVE TOLOSANE":"31","ESCOULIS":"31","CAZAC":"31","AIGNAN":"32","AUBIET":"32","AURADE":"32","AVEZAN":"32","BARCUGNAN":"32","BECCAS":"32","BEDECHAN":"32","LE MARTINET":"30","MASSILLARGUES ATTUECH":"30","MEJANNES LE CLAP":"30","MONTAREN ET ST MEDIERS":"30","ORTHOUX SERIGNAC QUILHAN":"30","POUGNADORESSE":"30","RIBAUTE LES TAVERNES":"30","ROBIAC ROCHESSADOULE":"30","LA ROQUE SUR CEZE":"30","ST BAUZELY":"30","ST CHRISTOL DE RODIERES":"30","ST GENIES DE COMOLAS":"30","ST HIPPOLYTE DE MONTAIGU":"30","ST HIPPOLYTE DU FORT":"30","ST PONS LA CALM":"30","ST THEODORIT":"30","ST VICTOR DE MALCAP":"30","LES SALLES DU GARDON":"30","SARDAN":"30","SAUVE":"30","SEYNES":"30","TRESQUES":"30","UCHAUD":"30","LA VERNAREDE":"30","ARNAUD GUILHEM":"31","AUSSON":"31","BAREN":"31","BAX":"31","BOIS DE LA PIERRE":"31","BORDES DE RIVIERE":"31","BOUDRAC":"31","BRIGNEMONT":"31","LE CABANIAL":"31","CAIGNAC":"31","CAMBERNARD":"31","CAPENS":"31","CASTELGAILLARD":"31","CASTIES LABRANDE":"31","CEPET":"31","CESSALES":"31","CIER DE LUCHON":"31","CUGNAUX":"31","LE CUING":"31","DAUX":"31","DEYME":"31","DRUDAS":"31","ESCALQUENS":"31","ESPANES":"31","FLOURENS":"31","FORGUES":"31","FRANCARVILLE":"31","FRANCON":"31","LE FRECHET":"31","FRONTIGNAN SAVES":"31","GAGNAC SUR GARONNE":"31","GENSAC SUR GARONNE":"31","GOUDEX":"31","GOUZENS":"31","GRAGNAGUE":"31","GRATENTOUR":"31","GRENADE":"31","L ISLE EN DODON":"31","ISSUS":"31","IZAUT DE L HOTEL":"31","LABASTIDE BEAUVOIR":"31","LACAUGNE":"31","LAFFITE TOUPIERE":"31","LAFITTE VIGORDANE":"31","LAUTIGNAC":"31","LUSSAN ADEILHAC":"31","MALVEZIE":"31","MARSOULAS":"31","MASSABRAC":"31","MERENVIELLE":"31","MONDAVEZAN":"31","MONDONVILLE":"31","MONTEGUT BOURJAC":"31","MONTEGUT LAURAGAIS":"31","MONTESPAN":"31","MANAURIE":"24","MAUZAC ET GRAND CASTANG":"24","MAUZENS ET MIREMONT":"24","MAZEYROLLES":"24","MENSIGNAC":"24","MESCOULES":"24","MONBAZILLAC":"24","MONTAGNAC LA CREMPSE":"24","MONTCARET":"24","MONPLAISANT":"24","MOULEYDIER":"24","NEGRONDES":"24","ORLIAC":"24","PARCOUL CHENAUD":"24","PAUSSAC ET ST VIVIEN":"24","PEYRIGNAC":"24","PIEGUT PLUVIERS":"24","PORT STE FOY ET PONCHAPT":"24","PROISSANS":"24","RIBAGNAC":"24","LA ROCHEBEAUCOURT ET ARGENTINE":"24","LA ROCHE CHALAIS":"24","ST AGNE":"24","ST ANTOINE DE BREUILH":"24","ST AVIT RIVIERE":"24","ST BARTHELEMY DE BUSSIERE":"24","ST CAPRAISE DE LALINDE":"24","ST CERNIN DE LABARDE":"24","ST CERNIN DE L HERM":"24","ST CREPIN DE RICHEMONT":"24","ST CREPIN ET CARLUCET":"24","ST ETIENNE DE PUYCORBIER":"24","ST FRONT D ALEMPS":"24","ST GERAUD DE CORPS":"24","STE INNOCENCE":"24","ST JULIEN DE CREMPSE":"24","ST JULIEN DE LAMPON":"24","ST LEON D ISSIGEAC":"24","ST LEON SUR L ISLE":"24","ST MARTIAL D ALBAREDE":"24","ST MARTIAL DE VALETTE":"24","ST MARTIN DE GURSON":"24","ST MEARD DE GURCON":"24","ST MICHEL DE DOUBLE":"24","STE NATHALENE":"24","ST NEXANS":"24","ST PANTALY D EXCIDEUIL":"24","ST PARDOUX DE DRONE":"24","ST PARDOUX LA RIVIERE":"24","ST RABIER":"24","ST SULPICE DE MAREUIL":"24","ST SULPICE DE ROUMAGNAC":"24","STE TRIE":"24","ST VINCENT DE COSSE":"24","SALIGNAC EYVIGUES":"24","SAUSSIGNAC":"24","SAVIGNAC LEDRIER":"24","SERVANCHES":"24","SIGOULES":"24","SIORAC EN PERIGORD":"24","TEMPLE LAGUYON":"24","TOCANE ST APRE":"24","VILLAMBLARD":"24","VILLEFRANCHE DE LONCHAT":"24","ADAM LES VERCEL":"25","AIBRE":"25","AUDINCOURT":"25","BADEVEL":"25","BAVANS":"25","BERTHELANGE":"25","BIANS LES USIERS":"25","BIEF":"25","BOLANDOZ":"25","BOUCLANS":"25","BRERES":"25","CADEMENE":"25","CHAFFOIS":"25","CHALEZEULE":"25","CHARBONNIERES LES SAPINS":"25","CHATELBLANC":"25","CHATILLON LE DUC":"25","CHATILLON SUR LISON":"25","LES TERRES DE CHAUX":"25","CHAUX LES CLERVAL":"25","TOUTENS":"31","URAU":"31","VALLEGUE":"31","VAUDREUILLE":"31","ARBLADE LE BAS":"32","ARROUEDE":"32","AVENSAC":"32","BARCELONNE DU GERS":"32","BARRAN":"32","BASSOUES":"32","BAZUGUES":"32","BERRAC":"32","BOUCAGNERES":"32","BOUZON GELLENAVE":"32","CADEILHAN":"32","CADEILLAN":"32","CAHUZAC SUR ADOUR":"32","CAILLAVET":"32","CASTELNAU D ANGLES":"32","CASTERON":"32","CLERMONT POUYGUILLES":"32","CONDOM":"32","COULOUME MONDEBAT":"32","COURTIES":"32","CRASTES":"32","CUELAS":"32","DURAN":"32","EAUZE":"32","FAGET ABBATIAL":"32","FOURCES":"32","GAZAUPOUY":"32","GOUX":"32","JUSTIAN":"32","LAGUIAN MAZOUS":"32","LAHAS":"32","LANNEMAIGNAN":"32","LASSERAN":"32","LAVARDENS":"32","LAVERAET":"32","LIGARDES":"32","LOUSLITGES":"32","MAGNAS":"32","MANAS BASTANOUS":"32","MANSEMPUY":"32","MANSENCOME":"32","MARESTAING":"32","MARGOUET MEYMES":"32","MERENS":"32","MIRANDE":"32","MIRANNES":"32","MONT DE MARRAST":"32","NOULENS":"32","PAUILHAC":"32","POMPIAC":"32","POUYDRAGUIN":"32","PRECHAC SUR ADOUR":"32","PUYCASQUIER":"32","STE CHRISTIE D ARMAGNAC":"32","ST MARTIN D ARMAGNAC":"32","ST MARTIN GIMOIS":"32","ST ORENS":"32","ST ORENS POUY PETIT":"32","ST OST":"32","ST PUY":"32","ST SOULAN":"32","SANSAN":"32","SARRAGACHIES":"32","SARRAGUZAN":"32","SAUVIMONT":"32","SEAILLES":"32","SEMPESSERRE":"32","SION":"32","SIRAC":"32","TARSAC":"32","TASQUE":"32","TERMES D ARMAGNAC":"32","TIESTE URAGNOUX":"32","TILLAC":"32","TOUJOUSE":"32","TOURDUN":"32","URGOSSE":"32","ARES":"33","LES ARTIGUES DE LUSSAC":"33","LE CROUZET":"25","DAMPJOUX":"25","DESANDANS":"25","DESERVILLERS":"25","DUNG":"25","ECOT":"25","L ECOUVOTTE":"25","EPENOUSE":"25","ESNANS":"25","ETRAPPE":"25","FLAGEY RIGNEY":"25","FONTENELLE MONTBY":"25","FRANEY":"25","FROIDEVAUX":"25","GEMONVAL":"25","GOUMOIS":"25","GRAND COMBE DES BOIS":"25","LES HOPITAUX NEUFS":"25","HOUTAUD":"25","ISSANS":"25","LABERGEMENT STE MARIE":"25","LAVAL LE PRIEURE":"25","LONGEVELLE LES RUSSEY":"25","LONGEVILLE":"25","MAGNY CHATELARD":"25","MAISONS DU BOIS LIEVREMONT":"25","MALBRANS":"25","MATHAY":"25","MESMAY":"25","METABIEF":"25","MONCLEY":"25","MONDON":"25","MONTANCY":"25","MONTANDON":"25","MONTBELIARD":"25","MONT DE LAVAL":"25","MONTLEBON":"25","MONTROND LE CHATEAU":"25","NOVILLARS":"25","ONANS":"25","OUGNEY DOUVOT":"25","PELOUSEY":"25","PETITE CHAUX":"25","PIERREFONTAINE LES BLAMONT":"25","PLAIMBOIS VENNES":"25","LA PLANEE":"25","LES PONTETS":"25","PONT LES MOULINS":"25","RANG":"25","RECULFOZ":"25","REMONDANS VAIVRE":"25","RIGNEY":"25","RIGNOSOT":"25","LA RIVIERE DRUGEON":"25","ROSET FLUANS":"25","RUFFEY LE CHATEAU":"25","ST VIT":"25","SANTOCHE":"25","SAONE":"25","SARRAGEOIS":"25","SCEY MAISIERES":"25","SEMONDANS":"25","SERVIN":"25","SOURANS":"25","TAILLECOURT":"25","TALLANS":"25","THISE":"25","TOUILLON ET LOUTELET":"25","TREPOT":"25","URTIERE":"25","VAIRE ARCIER":"25","VALOREILLE":"25","VERGRANNE":"25","LE VERNOY":"25","VOILLANS":"25","VUILLECIN":"25","VYT LES BELVOIR":"25","SOLAURE EN DIOIS":"26","ALEYRAC":"26","AOUSTE SUR SYE":"26","ARNAYON":"26","BATHERNAY":"26","LA BAUME DE TRANSIT":"26","LA BAUME D HOSTUN":"26","LA BEGUDE DE MAZENC":"26","BELLECOMBE TARENDOL":"26","BELLOC ST CLAMENS":"32","BETCAVE AGUIN":"32","BEZERIL":"32","BIRAN":"32","BOURROUILLAN":"32","BRETAGNE D ARMAGNAC":"32","CASTERA LECTOUROIS":"32","CASTERA VERDUZAN":"32","CAZAUBON":"32","CAZENEUVE":"32","DUFFORT":"32","DURBAN":"32","ENCAUSSE":"32","ESCORNEBOEUF":"32","ESTIPOUY":"32","ESTRAMIAC":"32","FLAMARENS":"32","GARRAVET":"32","GAUDONVILLE":"32","GONDRIN":"32","GOUTZ":"32","HAULIES":"32","L ISLE BOUZON":"32","LAGARDE HACHAN":"32","SERY LES MEZIERES":"02","TROSLY LOIRE":"02","SEBONCOURT":"02","SOMMELANS":"02","SEQUEHART":"02","SISSONNE":"02","SOUPIR":"02","SISSY":"02","VAUX EN VERMANDOIS":"02","VEUILLY LA POTERIE":"02","LA VALLEE AU BLE":"02","VIGNEUX HOCQUET":"02","VAUXAILLON":"02","VAUDESSON":"02","VEZAPONIN":"02","VEZILLY":"02","TUPIGNY":"02","CHATEAU SUR ALLIER":"03","BESSAY SUR ALLIER":"03","BROUT VERNET":"03","LA CHABANNE":"03","BEZENET":"03","CESSET":"03","BUSSET":"03","BEGUES":"03","VILLIERS ST DENIS":"02","BARRAIS BUSSOLLES":"03","AINAY LE CHATEAU":"03","VILLERS HELON":"02","ANDELAROCHE":"03","ARFEUILLES":"03","WIEGE FATY":"02","VOULPAIX":"02","BEAULON":"03","BAYET":"03","ST MARCEL EN MARCILLAT":"03","ST PRIEST D ANDELOT":"03","ST GERAND LE PUY":"03","SAZERET":"03","DOMPIERRE SUR BESBRE":"03","GARNAT SUR ENGIEVRE":"03","DURDAT LAREQUILLE":"03","ISLE ET BARDAIS":"03","CHATEL MONTAGNE":"03","DEUX CHAISES":"03","COGNAT LYONNE":"03","LE DONJON":"03","GANNAT":"03","JALIGNY SUR BESBRE":"03","LAPALISSE":"03","ISSERPENT":"03","LAVOINE":"03","LIMOISE":"03","LAMAIDS":"03","JENZAT":"03","LODDES":"03","MAGNET":"03","VILLEFRANCHE D ALLIER":"03","LE VEURDRE":"03","SOUVIGNY":"03","MONTESQUIEU LAURAGAIS":"31","MONTGAILLARD DE SALIES":"31","MONTGISCARD":"31","MONTGRAS":"31","MONTMAURIN":"31","MONTPITOL":"31","MOURVILLES BASSES":"31","PAYSSOUS":"31","PEGUILHAN":"31","PIBRAC":"31","PLAISANCE DU TOUCH":"31","PORTET SUR GARONNE":"31","POUCHARRAMET":"31","QUINT FONSEGRIVES":"31","REGADES":"31","RIEUX VOLVESTRE":"31","ROQUEFORT SUR GARONNE":"31","SACCOURVIELLE":"31","SAIGUEDE":"31","ST FELIX LAURAGAIS":"31","STE LIVRADE":"31","ST MAMET":"31","ST PAUL SUR SAVE":"31","ST PE D ARDET":"31","ST PLANCARD":"31","ST SULPICE SUR LEZE":"31","SALLES ET PRATVIEL":"31","SALLES SUR GARONNE":"31","LA SALVETAT ST GILLES":"31","SANA":"31","SAUX ET POMAREDE":"31","SEDEILHAC":"31","SEGREVILLE":"31","SENGOUAGNET":"31","TOURNEFEUILLE":"31","TREBONS DE LUCHON":"31","TREBONS SUR LA GRASSE":"31","VALCABRERE":"31","VALENTINE":"31","VIEILLE TOULOUSE":"31","VILLATE":"31","VILLEFRANCHE DE LAURAGAIS":"31","VILLEMUR SUR TARN":"31","VILLENEUVE DE RIVIERE":"31","VILLENEUVE LECUSSAN":"31","BINOS":"31","AYZIEU":"32","BERAUT":"32","BETOUS":"32","BIVES":"32","BLAZIERT":"32","CABAS LOUMASSES":"32","CAMPAGNE D ARMAGNAC":"32","CASTELNAU BARBARENS":"32","CASTILLON MASSAS":"32","CHELAN":"32","COLOGNE":"32","ESPAON":"32","FREGOUVILLE":"32","FUSTEROUAU":"32","GAZAX ET BACCARISSE":"32","IDRAC RESPAILLES":"32","L ISLE ARNE":"32","JUILLES":"32","LABARTHETE":"32","LABASTIDE SAVES":"32","LAGRAULET DU GERS":"32","LAMAGUERE":"32","LAMAZERE":"32","LANNEPAX":"32","LARROQUE SUR L OSSE":"32","LASSEUBE PROPRE":"32","LAURAET":"32","LAYMONT":"32","LOMBEZ":"32","LOUBEDAT":"32","LUPPE VIOLLES":"32","MANCIET":"32","MARAVAT":"32","MARCIAC":"32","MAULEON D ARMAGNAC":"32","MAULICHERES":"32","MIELAN":"32","MIRADOUX":"32","MONFERRAN PLAVES":"32","CHAUX NEUVE":"25","CHEVIGNEY LES VERCEL":"25","CLERON":"25","LES COMBES":"25","COURCELLES LES MONTBELIARD":"25","CUBRIAL":"25","CUSANCE":"25","DAMPIERRE SUR LE DOUBS":"25","DAMPRICHARD":"25","DANNEMARIE SUR CRETE":"25","ECOLE VALENTIN":"25","EPEUGNEY":"25","ETRAY":"25","ETUPES":"25","EYSSON":"25","FLEUREY":"25","FONTENOTTE":"25","FOURBANNE":"25","FRANOIS":"25","GENEY":"25","GEVRESIN":"25","GLAMONDANS":"25","GLERE":"25","GONDENANS LES MOULINS":"25","GRAND CHARMONT":"25","FOURNETS LUISANS":"25","LES GRANGETTES":"25","GROSBOIS":"25","L ISLE SUR LE DOUBS":"25","JALLERANGE":"25","VILLERS LE LAC":"25","LANDRESSE":"25","LARNOD":"25","LIZINE":"25","LOMONT SUR CRETE":"25","LONGECHAUX":"25","LONGEVILLES MONT D OR":"25","MONCEY":"25","MONTFERRAND LE CHATEAU":"25","MONTFLOVIN":"25","MONTIVERNAGE":"25","MORRE":"25","NANS":"25","ORCHAMPS VENNES":"25","OUHANS":"25","PASSONFONTAINE":"25","PESSANS":"25","PIREY":"25","PLAIMBOIS DU MIROIR":"25","LES PLAINS ET GRANDS ESSARTS":"25","PUESSANS":"25","PUGEY":"25","QUINGEY":"25","RANDEVILLERS":"25","RAYNANS":"25","REMORAY BOUJEONS":"25","RENEDALE":"25","ROCHE LES CLERVAL":"25","RONCHAUX":"25","RONDEFONTAINE":"25","ROULANS":"25","SAMSON":"25","SANCEY":"25","SELONCOURT":"25","SOCHAUX":"25","LA SOMMETTE":"25","SOYE":"25","THORAISE":"25","LA TOUR DE SCAY":"25","TOURNANS":"25","TREVILLERS":"25","VAIRE LE PETIT":"25","VAUCLUSE":"25","VAUCLUSOTTE":"25","VAUFREY":"25","VERRIERES DU GROSBOIS":"25","VIETHOREY":"25","VILLERS CHIEF":"25","VILLERS LA COMBE":"25","VUILLAFANS":"25","BERNOS BEAULAC":"33","BERSON":"33","BERTHEZ":"33","BEYCHAC ET CAILLAU":"33","BLESIGNAC":"33","BONNETAN":"33","CABANAC ET VILLAGRAINS":"33","CADILLAC EN FRONSADAIS":"33","CAMBLANES ET MEYNAC":"33","CAMIRAN":"33","CAPLONG":"33","CARS":"33","CARTELEGUE":"33","CASSEUIL":"33","CASTELVIEL":"33","CASTILLON DE CASTETS":"33","CASTILLON LA BATAILLE":"33","CAUDROT":"33","CAUVIGNAC":"33","CAVIGNAC":"33","CAZATS":"33","CERONS":"33","CIVRAC SUR DORDOGNE":"33","COIRAC":"33","COURS DE MONSEGUR":"33","COURS LES BAINS":"33","COUTRAS":"33","DONNEZAC":"33","FLAUJAGUES":"33","FLOUDES":"33","FRANCS":"33","GALGON":"33","GENISSAC":"33","GRAYAN ET L HOPITAL":"33","GUJAN MESTRAS":"33","HURE":"33","ISLE ST GEORGES":"33","LADOS":"33","LALANDE DE POMEROL":"33","LANTON":"33","LAVAZAN":"33","LEOGNAN":"33","LIBOURNE":"33","LIGUEUX":"33","LISTRAC DE DUREZE":"33","MARCENAIS":"33","MIOS":"33","MONTAGOUDIN":"33","NEUFFONS":"33","LE NIZAN":"33","OMET":"33","PESSAC SUR DORDOGNE":"33","LE PIAN SUR GARONNE":"33","PINEUILH":"33","PUGNAC":"33","ST ANDRE DE CUBZAC":"33","ST ANDRE ET APPELLES":"33","ST CIERS SUR GIRONDE":"33","STE CROIX DU MONT":"33","STE FLORENCE":"33","ST GERMAIN DE GRAVE":"33","ST JEAN DE BLAIGNAC":"33","ST JEAN D ILLAC":"33","ST MACAIRE":"33","ST MARIENS":"33","ST MARTIN DE LAYE":"33","ST MEDARD D EYRANS":"33","ST MICHEL DE RIEUFRET":"33","ST ROMAIN LA VIRVEE":"33","ST SAUVEUR DE PUYNORMAND":"33","ST SELVE":"33","ST SEURIN DE BOURG":"33","ST SEURIN DE CURSAC":"33","STE TERRE":"33","ST VIVIEN DE BLAYE":"33","SALAUNES":"33","SAUCATS":"33","SIGALENS":"33","BESAYES":"26","BESIGNAN":"26","BEZAUDUN SUR BINE":"26","BOULC":"26","BOURG DE PEAGE":"26","BRETTE":"26","LA CHARCE":"26","CHAROLS":"26","CHATEAUNEUF DE BORDETTE":"26","CHATILLON ST JEAN":"26","CHAUVAC LAUX MONTAUX":"26","CONDILLAC":"26","CORNILLON SUR L OULE":"26","LA COUCOURDE":"26","CREPOL":"26","CREST":"26","DIEULEFIT":"26","ETOILE SUR RHONE":"26","EYROLES":"26","FAY LE CLOS":"26","HAUTERIVES":"26","LAPEYROUSE MORNAY":"26","LAVAL D AIX":"26","LAVEYRON":"26","LUS LA CROIX HAUTE":"26","MALISSARD":"26","MARIGNAC EN DIOIS":"26","MONTCHENU":"26","MONTCLAR SUR GERVANNE":"26","MONTFERRAND LA FARE":"26","MONTVENDRE":"26","LA MOTTE CHALANCON":"26","LA MOTTE FANJAS":"26","MOURS ST EUSEBE":"26","ORCINAS":"26","OURCHES":"26","PARNANS":"26","PENNES LE SEC":"26","LA PENNE SUR L OUVEZE":"26","PEYRUS":"26","PIEGROS LA CLASTRE":"26","PLAISIANS":"26","PONTAIX":"26","PROPIAC":"26","PUYGIRON":"26","ROCHEFORT EN VALDAINE":"26","ROCHE ST SECRET BECONNE":"26","LA ROCHE SUR LE BUIS":"26","ROMEYER":"26","ST JULIEN EN VERCORS":"26","ST MARTIN LE COLONEL":"26","SAVASSE":"26","SEDERON":"26","LES TOURRETTES":"26","UPIE":"26","VACHERES EN QUINT":"26","VAUNAVEYS LA ROCHETTE":"26","VERCHENY":"26","VERONNE":"26","VILLEBOIS LES PINS":"26","VILLEPERDRIX":"26","ANGERVILLE LA CAMPAGNE":"27","ASNIERES":"27","BALINES":"27","BEAUMONTEL":"27","BEAUMONT LE ROGER":"27","BEMECOURT":"27","BERNIERES SUR SEINE":"27","BERVILLE LA CAMPAGNE":"27","LE BOSC ROGER EN ROUMOIS":"27","BOSGOUET":"27","BOSGUERARD DE MARCOUVILLE":"27","LES BOTTEREAUX":"27","BOUAFLES":"27","BOUQUELON":"27","BOURG BEAUDOUIN":"27","GRAND BOURGTHEROULDE":"27","BRETAGNOLLES":"27","BRIONNE":"27","BUEIL":"27","BUREY":"27","YZEURE":"03","LA PALUD SUR VERDON":"04","NIOZELLES":"04","REILLANNE":"04","QUINSON":"04","NIBLES":"04","LIMANS":"04","PIEGUT":"04","CLAMENSANE":"04","BEAUVEZER":"04","LE CAIRE":"04","CERESTE":"04","GREOUX LES BAINS":"04","LE FUGERET":"04","DEMANDOLX":"04","ESTOUBLON":"04","GANAGOBIE":"04","L ESCALE":"04","ST LAURENT DU VERDON":"04","STE CROIX DU VERDON":"04","ST VINCENT SUR JABRON":"04","ST MARTIN DE BROMES":"04","SEYNE":"04","SOLEILHAS":"04","TARTONNE":"04","SELONNET":"04","ST MAIME":"04","SAUSSES":"04","GARDE COLOMBE":"05","LA FARE EN CHAMPSAUR":"05","BARRET SUR MEOUGE":"05","ESPINASSES":"05","LA BEAUME":"05","LE BERSAC":"05","CEILLAC":"05","ASPRES SUR BUECH":"05","ASPRES LES CORPS":"05","THORAME BASSE":"04","LES THUILES":"04","TURRIERS":"04","VAUMEILH":"04","VALBELLE":"04","VERGONS":"04","ARVIEUX":"05","VOLX":"04","ST MAURICE EN VALGODEMARD":"05","ST JULIEN EN CHAMPSAUR":"05","ST ANDRE DE ROSANS":"05","ST CHAFFREY":"05","ANDON":"06","LE BAR SUR LOUP":"06","BREIL SUR ROYA":"06","CAGNES SUR MER":"06","AMIRAT":"06","CAILLE":"06","BEUIL":"06","CONSEGUDES":"06","CAP D AIL":"06","COLOMARS":"06","CAUSSOLS":"06","COURMES":"06","DURANUS":"06","FONTAN":"06","ST ETIENNE DE FONTBELLON":"07","ST ANDRE DE CRUZIERES":"07","ST ALBAN EN MONTAGNE":"07","ST CIRGUES DE PRADES":"07","ROCHECOLOMBE":"07","ST DESIRAT":"07","MONPARDIAC":"32","MONTEGUT ARROS":"32","MONTEGUT SAVES":"32","MONTESQUIOU":"32","MONTESTRUC SUR GERS":"32","PAVIE":"32","PELLEFIGUE":"32","PLIEUX":"32","LA ROMIEU":"32","ROQUELAURE ST AUBIN":"32","SABAZAN":"32","STE AURENCE CAZAUX":"32","ST BLANCARD":"32","ST MEZARD":"32","ST MONT":"32","ST PIERRE D AUBEZIES":"32","SARAMON":"32","SEGOUFIELLE":"32","SEREMPUY":"32","TACHOIRES":"32","TERRAUBE":"32","TOUGET":"32","TRONCENS":"32","VILLEFRANCHE":"32","VIOZAN":"32","AMBARES ET LAGRAVE":"33","AMBES":"33","ARBIS":"33","ARCINS":"33","ARTIGUES PRES BORDEAUX":"33","AUDENGE":"33","AURIOLLES":"33","AUROS":"33","BAYAS":"33","BELLEBAT":"33","LES BILLAUX":"33","LE BOUSCAT":"33","FRAISSE DES CORBIERES":"11","LABASTIDE D ANJOU":"11","FONTIERS CABARDES":"11","FOURNES CABARDES":"11","LABASTIDE EN VAL":"11","GAJA LA SELVE":"11","GOURVIEILLE":"11","LASTOURS":"11","LA PALME":"11","GRANES":"11","CONQUES SUR ORBIEL":"11","ARGENS MINERVOIS":"11","CASTELNAU D AUDE":"11","CAMPLONG D AUDE":"11","LA CASSAIGNE":"11","CAUDEBRONDE":"11","COUSTOUGE":"11","LE CLAT":"11","CAMURAC":"11","EMBRES ET CASTELMAURE":"11","DURBAN CORBIERES":"11","CUXAC D AUDE":"11","ESCOULOUBRE":"11","FANJEAUX":"11","FABREZAN":"11","DOUZENS":"11","ESCALES":"11","CUMIES":"11","VERNONVILLIERS":"10","VILLEMEREUIL":"10","VAUPOISSON":"10","THIEFFRAIN":"10","SAVIERES":"10","TRANNES":"10","VANLAY":"10","BUGARACH":"11","BERRIAC":"11","BELVIS":"11","ARZENS":"11","ALLAN":"26","ARPAVON":"26","BARCELONNE":"26","BARNAVE":"26","BARRET DE LIOURE":"26","LA BAUME CORNILLANE":"26","BEAUFORT SUR GERVANNE":"26","BEAUREGARD BARET":"26","BEAURIERES":"26","BEAUSEMBLANT":"26","BOURG LES VALENCE":"26","BOUVANTE":"26","CHABRILLAN":"26","CHARMES SUR L HERBASSE":"26","CHATEAUNEUF DE GALAURE":"26","CHAUDEBONNE":"26","LA CHAUDIERE":"26","CLAVEYSON":"26","CLEON D ANDRAN":"26","CONDORCET":"26","CURNIER":"26","DIE":"26","DIVAJEU":"26","FRANCILLON SUR ROUBION":"26","LA GARDE ADHEMAR":"26","GENISSIEUX":"26","GIGORS ET LOZERON":"26","HOSTUN":"26","IZON LA BRUISSE":"26","LORIOL SUR DROME":"26","MANAS":"26","MARSAZ":"26","MENGLON":"26","MERINDOL LES OLIVIERS":"26","MEVOUILLON":"26","MIRABEL ET BLACONS":"26","MONTBRISON SUR LEZ":"26","MONTBRUN LES BAINS":"26","MONTGUERS":"26","MONTLAUR EN DIOIS":"26","MONTMIRAL":"26","LA MOTTE DE GALAURE":"26","LE PEGUE":"26","PELONNE":"26","PIEGON":"26","PIERRELATTE":"26","LES PILLES":"26","LE POET EN PERCIP":"26","PONET ET ST AUBAN":"26","PONT DE BARRET":"26","PONT DE L ISERE":"26","POYOLS":"26","PUY ST MARTIN":"26","RATIERES":"26","RIMON ET SAVEL":"26","RIOMS":"26","ROCHEFOURCHAT":"26","ROMANS SUR ISERE":"26","ROUSSET LES VIGNES":"26","ST AGNAN EN VERCORS":"26","ST BARTHELEMY DE VALS":"26","ST BENOIT EN DIOIS":"26","STE EUPHEMIE SUR OUVEZE":"26","ST JULIEN EN QUINT":"26","ST MARCEL LES SAUZET":"26","ST MAURICE SUR EYGUES":"26","ST PAUL LES ROMANS":"26","ST RAMBERT D ALBON":"26","ST SAUVEUR EN DIOIS":"26","ST SAUVEUR GOUVERNET":"26","ST SORLIN EN VALLOIRE":"26","ST THOMAS EN ROYANS":"26","SALLES SOUS BOIS":"26","SAULCE SUR RHONE":"26","SERVES SUR RHONE":"26","LA TOUCHE":"26","TRUINAS":"26","VERCOIRAN":"26","VESC":"26","VINSOBRES":"26","ACQUIGNY":"27","AIGLEVILLE":"27","AIZIER":"27","AMFREVILLE ST AMAND":"27","APPEVILLE ANNEBAULT":"27","ARMENTIERES SUR AVRE":"27","AUTHEUIL AUTHOUILLET":"27","LES AUTHIEUX":"27","AUTHOU":"27","LES BARILS":"27","BARNEVILLE SUR SEINE":"27","SOUSSANS":"33","TAILLECAVAT":"33","TARNES":"33","TAYAC":"33","TIZAC DE CURTON":"33","LE TOURNE":"33","TRESSES":"33","VILLENAVE DE RIONS":"33","ANIANE":"34","AUMELAS":"34","AUTIGNAC":"34","AVENE":"34","BRENAS":"34","BUZIGNARGUES":"34","CAUSSES ET VEYRAN":"34","COURNIOU":"34","FERRALS LES MONTAGNES":"34","FRAISSE SUR AGOUT":"34","GANGES":"34","JACOU":"34","LANSARGUES":"34","LODEVE":"34","MAGALAS":"34","MOULES ET BAUCELS":"34","MOUREZE":"34","MUDAISON":"34","MURVIEL LES BEZIERS":"34","NEBIAN":"34","NOTRE DAME DE LONDRES":"34","OLONZAC":"34","OUPIA":"34","PEROLS":"34","PEZENES LES MINES":"34","PIGNAN":"34","LE POUGET":"34","LE PRADAL":"34","LE PUECH":"34","PUILACHER":"34","LES RIVES":"34","ROQUEBRUN":"34","ROUET":"34","ST AUNES":"34","ST CLEMENT DE RIVIERE":"34","ST ETIENNE DE GOURGAS":"34","ST FELIX DE LODEZ":"34","ST GUILHEM LE DESERT":"34","ST JEAN DE BUEGES":"34","ST MARTIN DE LONDRES":"34","ST NAZAIRE DE LADAREZ":"34","ST SATURNIN DE LUCIAN":"34","SERIGNAN":"34","SOUBES":"34","TRESSAN":"34","VENDARGUES":"34","VIOLS LE FORT":"34","ACIGNE":"35","AMANLIS":"35","ANDOUILLE NEUVILLE":"35","BAULON":"35","BILLE":"35","BLERUAIS":"35","BOISGERVILLY":"35","BONNEMAIN":"35","LA BOUEXIERE":"35","BREAL SOUS MONTFORT":"35","BREAL SOUS VITRE":"35","BRETEIL":"35","CAMPEL":"35","CHANCE":"35","LA CHAPELLE DU LOU DU LAC":"35","LA CHAPELLE ST AUBERT":"35","CHASNE SUR ILLET":"35","CHAVAGNE":"35","COGLES":"35","COMBOURTILLE":"35","LA COUYERE":"35","DINGE":"35","DOMLOUP":"35","FEINS":"35","LE FERRE":"35","GAHARD":"35","LA GOUESNIERE":"35","GOVEN":"35","L HERMITAGE":"35","LIEURON":"35","LIVRE SUR CHANGEON":"35","CAORCHES ST NICOLAS":"27","CHAMPENARD":"27","CHAMPIGNY LA FUTELAYE":"27","COLLANDRES QUINCARNON":"27","CORNEVILLE SUR RISLE":"27","COUDRES":"27","COURBEPINE":"27","CROSVILLE LA VIEILLE":"27","LES DAMPS":"27","MESNILS SUR ITON":"27","ECAQUELON":"27","ECAUVILLE":"27","ETURQUERAYE":"27","FATOUVILLE GRESTAIN":"27","FLEURY SUR ANDELLE":"27","FONTAINE BELLENGER":"27","FONTAINE L ABBE":"27","FONTAINE SOUS JOUY":"27","GADENCOURT":"27","GAILLON":"27","GAMACHES EN VEXIN":"27","GASNY":"27","GAUDREVILLE LA RIVIERE":"27","GAUVILLE LA CAMPAGNE":"27","GOURNAY LE GUERIN":"27","GRAINVILLE":"27","GROSLEY SUR RISLE":"27","HARQUENCY":"27","LA HAYE AUBREE":"27","LA HAYE DU THEIL":"27","HEUBECOURT HARICOURT":"27","HEUDEBOUVILLE":"27","HEUDREVILLE EN LIEUVIN":"27","HONDOUVILLE":"27","HOUVILLE EN VEXIN":"27","IVILLE":"27","LOUYE":"27","MANDEVILLE":"27","MARTAGNY":"27","MEREY":"27","LE MESNIL FUGUET":"27","MISEREY":"27","MONTFORT SUR RISLE":"27","MONTREUIL L ARGILLE":"27","MORAINVILLE JOUVEAUX":"27","MOUETTES":"27","MUIDS":"27","NEAUFLES AUVERGNY":"27","LA NEUVE GRANGE":"27","NOJEON EN VEXIN":"27","NOTRE DAME DE L ISLE":"27","NOTRE DAME D EPINE":"27","PISEUX":"27","LES PLACES":"27","LE PLANQUAY":"27","LA PYLE":"27","QUATREMARE":"27","QUILLEBEUF SUR SEINE":"27","ST AQUILIN DE PACY":"27","ST DENIS LE FERMENT":"27","ST DIDIER DES BOIS":"27","ST ELIER":"27","ST ELOI DE FOURQUES":"27","ST GERMAIN VILLAGE":"27","ST GREGOIRE DU VIEVRE":"27","ST JEAN DE LA LEQUERAYE":"27","ST MACLOU":"27","ST MARDS DE FRESNE":"27","LE LESME":"27","CALONGES":"47","CASSENEUIL":"47","CASSIGNAS":"47","CAUBEYRES":"47","DOUZAINS":"47","DURAS":"47","ESTILLAC":"47","FALS":"47","FARGUES SUR OURBISE":"47","FUMEL":"47","GALAPIAN":"47","GONTAUD DE NOGARET":"47","GRATELOUP ST GAYRAND":"47","ST SAUVEUR SUR TINEE":"06","TOUET DE L ESCARENE":"06","ST VALLIER DE THIEY":"06","ST PAUL DE VENCE":"06","TOUET SUR VAR":"06","SALLAGRIFFON":"06","ACCONS":"07","VENCE":"06","LA ROQUE EN PROVENCE":"06","GREOLIERES":"06","GUILLAUMES":"06","PEYMEINADE":"06","MALAUSSENE":"06","LE MAS":"06","MENTON":"06","OPIO":"06","BOULIEU LES ANNONAY":"07","ARRAS SUR RHONE":"07","BOURG ST ANDEOL":"07","ALBOUSSIERE":"07","CHASSIERS":"07","BALAZUC":"07","CHANEAC":"07","CHARNAS":"07","ARCENS":"07","DUNIERE SUR EYRIEUX":"07","LE CRESTET":"07","DESAIGNES":"07","CHEMINAS":"07","CHAZEAUX":"07","FLAVIAC":"07","CORNAS":"07","FABRAS":"07","CRUAS":"07","GLUN":"07","ST MARCEL LES ANNONAY":"07","ST LAURENT LES BAINS":"07","ST MARTIN D ARDECHE":"07","ST JUST D ARDECHE":"07","ST JEAN CHAMBRE":"07","LABASTIDE SUR BESORGUES":"07","LALEVADE D ARDECHE":"07","GUILHERAND GRANGES":"07","LABASTIDE DE VIRAC":"07","LABATIE D ANDAURE":"07","GROSPIERRES":"07","LABLACHERE":"07","GRAVIERES":"07","LABEAUME":"07","JOYEUSE":"07","ST MICHEL DE CHABRILLANOUX":"07","ST SAUVEUR DE CRUZIERES":"07","ST PIERRE DE COLOMBIER":"07","ST VINCENT DE DURFORT":"07","ST SAUVEUR DE MONTAGUT":"07","ST ROMAIN DE LERPS":"07","ST MAURICE D IBIE":"07","ST MONTAN":"07","VANOSC":"07","LE CHATELET SUR SORMONNE":"08","CHALANDRY ELAIRE":"08","BOULT AUX BOIS":"08","BOURG FIDELE":"08","LA BESACE":"08","CERNION":"08","LES PETITES ARMOISES":"08","AUVILLERS LES FORGES":"08","AUBONCOURT VAUZELLES":"08","VILLEVOCANCE":"07","VAUDEVANT":"07","AIGLEMONT":"08","AUFLANCE":"08","VESSEAUX":"07","AMAGNE":"08","VOGUE":"07","CONDE LES HERPY":"08","CLAVY WARBY":"08","RUBECOURT ET LAMECOURT":"08","ST ETIENNE A ARNES":"08","ST LOUP TERRIER":"08","ST GERMAINMONT":"08","RIMOGNE":"08","VINETS":"10","AIROUX":"11","AUNAT":"11","BRAM":"11","BOURISP":"65","CABANAC":"65","CADEILHAN TRACHERE":"65","CAMALES":"65","CAUTERETS":"65","CAZAUX DEBAT":"65","CHIS":"65","DOURS":"65","ENS":"65","ESCONNETS":"65","ESCOTS":"65","ESPARROS":"65","ESTERRE":"65","FRECHEDE":"65","GAVARNIE GEDRE":"65","GOUDON":"65","GRAILHEN":"65","HOUEYDETS":"65","HOURC":"65","ILHEU":"65","LABASTIDE":"65","LAFITOLE":"65","LAHITTE TOUPIERE":"65","LALANNE TRIE":"65","LALOUBERE":"65","LAMARQUE RUSTAING":"65","LARAN":"65","LASCAZERES":"65","LASLADES":"65","LESCURRY":"65","MOLERE":"65","MOMERES":"65","MONTOUSSE":"65","MOUMOULOUS":"65","MUN":"65","NEUILH":"65","ODOS":"65","ORINCLES":"65","OSMETS":"65","OURDIS COTDOUSSAN":"65","OUSTE":"65","OUZOUS":"65","PAILHAC":"65","PEYROUSE":"65","RECURT":"65","SABALOS":"65","SACOUE":"65","ST LARY SOULAN":"65","ST SEVER DE RUSTAN":"65","SALLES ADOUR":"65","SARIAC MAGNOAC":"65","SEMEAC":"65","SOUYEAUX":"65","TARASTEIX":"65","THEBE":"65","THERMES MAGNOAC":"65","TOURNAY":"65","TRAMEZAIGUES":"65","TRIE SUR BAISE":"65","VIC EN BIGORRE":"65","VIEY":"65","BAILLESTAVY":"66","BOULE D AMONT":"66","BROUILLA":"66","CABESTANY":"66","CALCE":"66","CASES DE PENE":"66","CASTELNOU":"66","CLARA":"66","CONAT":"66","CORBERE":"66","CORNEILLA DE CONFLENT":"66","CORNEILLA DEL VERCOL":"66","CORSAVY":"66","COUSTOUGES":"66","DORRES":"66","LES CLUSES":"66","LES BAUX STE CROIX":"27","BAZOQUES":"27","LE BEC THOMAS":"27","BERNAY":"27","BEZU ST ELOI":"27","BOIS ARNAULT":"27","BOISNEY":"27","BOISSY LAMBERVILLE":"27","BOUCHEVILLIERS":"27","BOUQUETOT":"27","BOURNEVILLE STE CROIX":"27","BOURTH":"27","BROSVILLE":"27","CAILLOUET ORGEVILLE":"27","CHAISE DIEU DU THEIL":"27","CHAMBRAY":"27","CHAMP DOLENT":"27","LA CHAPELLE HARENG":"27","CHAVIGNY BAILLEUL":"27","CHENNEBRUN":"27","COLLETOT":"27","COMBON":"27","CONCHES EN OUCHE":"27","LA COUTURE BOUSSEY":"27","CRIQUEBEUF LA CAMPAGNE":"27","CROISY SUR EURE":"27","CROTH":"27","DANGU":"27","DAUBEUF PRES VATTEVILLE":"27","DRUCOURT":"27","DURANVILLE":"27","EPAIGNES":"27","EPEGARD":"27","FERRIERES HAUT CLOCHER":"27","FONTAINE LA SORET":"27","FOUQUEVILLE":"27","FRESNEY":"27","GLISOLLES":"27","HACQUEVILLE":"27","HARDENCOURT COCHEREL":"27","LA HARENGERE":"27","LA HAYE ST SYLVESTRE":"27","HEUDREVILLE SUR EURE":"27","HONGUEMARE GUENOUVILLE":"27","HOUETTEVILLE":"27","LA HOUSSAYE":"27","ILLEVILLE SUR MONTFORT":"27","JUMELLES":"27","LAUNAY":"27","LILLY":"27","LISORS":"27","LIVET SUR AUTHOU":"27","LONGCHAMPS":"27","LOUVIERS":"27","LA MADELEINE DE NONANCOURT":"27","MANDRES":"27","MANNEVILLE SUR RISLE":"27","MARCILLY LA CAMPAGNE":"27","MENESQUEVILLE":"27","MERCEY":"27","MESNIL ROUSSET":"27","MORGNY":"27","MOUSSEAUX NEUVILLE":"27","MUZY":"27","NASSANDRES":"27","NEAUFLES ST MARTIN":"27","LE NEUBOURG":"27","PACY SUR EURE":"27","LE PLESSIS GROHAN":"27","LE PLESSIS STE OPPORTUNE":"27","LOUVIGNE DE BAIS":"35","LUITRE":"35","MARCILLE RAOUL":"35","MARPIRE":"35","LA MEZIERE":"35","MINIAC MORVAN":"35","MONDEVERT":"35","MONTFORT SUR MEU":"35","NOUVOITOU":"35","NOYAL SUR VILAINE":"35","PARCE":"35","PARIGNE":"35","PLECHATEL":"35","PLEINE FOUGERES":"35","POCE LES BOIS":"35","POLIGNE":"35","QUEDILLAC":"35","RANNEE":"35","LE RHEU":"35","ROZ LANDRIEUX":"35","SAINS":"35","ST AUBIN DU CORMIER":"35","ST BRIAC SUR MER":"35","ST ETIENNE EN COGLES":"35","ST GANTON":"35","ST GEORGES DE GREHAIGNE":"35","ST GERMAIN DU PINEL":"35","ST GERMAIN SUR ILLE":"35","ST GONDRAN":"35","ST GONLAY":"35","ST GUINOUX":"35","ST MARCAN":"35","ST SENOUX":"35","ST SULIAC":"35","TAILLIS":"35","TEILLAY":"35","LE TIERCENT":"35","TINTENIAC":"35","LE VERGER":"35","VERN SUR SEICHE":"35","VEZIN LE COQUET":"35","VIEUX VY SUR COUESNON":"35","LA VILLE ES NONAIS":"35","ARGENTON SUR CREUSE":"36","ARTHON":"36","AZAY LE FERRON":"36","BAUDRES":"36","BELABRE":"36","BOUESSE":"36","BRIVES":"36","BUXIERES D AILLAC":"36","BUZANCAIS":"36","CHAMPILLET":"36","CHASSENEUIL":"36","LA CHATRE":"36","CHAZELET":"36","CHOUDAY":"36","CREVANT":"36","CROZON SUR VAUVRE":"36","DIORS":"36","ECUEILLE":"36","GEHEE":"36","HEUGNES":"36","JEU LES BOIS":"36","JEU MALOCHES":"36","LANGE":"36","LIGNAC":"36","LUANT":"36","LURAIS":"36","LYE":"36","MIGNY":"36","MONTIPOURET":"36","MOUHET":"36","MOULINS SUR CEPHONS":"36","PAULNAY":"36","BADECON LE PIN":"36","REBOURSIN":"36","ST CHARTIER":"36","ST GENOU":"36","SARZAY":"36","SAULNAY":"36","GUERIN":"47","HOUEILLES":"47","LAGRUERE":"47","LAPARADE":"47","LAUZUN":"47","LAVARDAC":"47","LOUBES BERNAC":"47","MOIRAX":"47","MONBAHUS":"47","MONBALEN":"47","MONTAYRAL":"47","MONVIEL":"47","MOUSTIER":"47","PARRANQUET":"47","PAULHIAC":"47","PUYSSERAMPION":"47","ST ANTOINE DE FICALBA":"47","ST CAPRAIS DE LERM":"47","ST COLOMB DE LAUZUN":"47","STE COLOMBE DE VILLENEUVE":"47","STE GEMME MARTAILLAC":"47","ST GERAUD":"47","ST MARTIN DE BEAUVILLE":"47","ST MARTIN DE VILLEREAL":"47","ST MARTIN PETIT":"47","ST PARDOUX ISAAC":"47","ST QUENTIN DU DROPT":"47","ST ROMAIN LE NOBLE":"47","ST VINCENT DE LAMONTJOIE":"47","THOUARS SUR GARONNE":"47","TOMBEBOEUF":"47","TOURTRES":"47","VIANNE":"47","VILLENEUVE DE DURAS":"47","VILLEREAL":"47","BAGNOLS LES BAINS":"48","BANASSAC CANILHAC":"48","LES BESSONS":"48","BLAVIGNAC":"48","CASSAGNAS":"48","CHASTEL NOUVEL":"48","CHATEAUNEUF DE RANDON":"48","CHEYLARD L EVEQUE":"48","BEDOUES COCURES":"48","LE COLLET DE DEZE":"48","GATUZIERES":"48","GRANDVALS":"48","HURES LA PARADE":"48","LANGOGNE":"48","LAUBERT":"48","LAVAL ATGER":"48","PAULHAC EN MARGERIDE":"48","PREVENCHERES":"48","RECOULES D AUBRAC":"48","RIMEIZE":"48","ST ETIENNE DU VALDONNEZ":"48","VENTALON EN CEVENNES":"48","ST MARTIN DE BOUBAUX":"48","ST PIERRE DE NOGARET":"48","ST PRIVAT DU FAU":"48","ST SAUVEUR DE GINESTOUX":"48","LA TIEULE":"48","ARMAILLE":"49","BOURG L EVEQUE":"49","BREZE":"49","BROC":"49","CHALONNES SOUS LE LUDE":"49","DENEZE SOUS LE LUDE":"49","ECOUFLANT":"49","FONTEVRAUD L ABBAYE":"49","LA NEUVILLE LES WASIGNY":"08","MAISONCELLE ET VILLERS":"08","MONT ST REMY":"08","MESSINCOURT":"08","NEUFMANIL":"08","NEUFLIZE":"08","MARGUT":"08","MOGUES":"08","LANDRES ET ST GEORGES":"08","LES HAUTES RIVIERES":"08","LEPRON LES VALLEES":"08","HANNOGNE ST MARTIN":"08","HOULDIZY":"08","HIERGES":"08","LETANNE":"08","LALOBBE":"08","LIART":"08","LINAY":"08","GESPUNSART":"08","GUINCOURT":"08","HANNAPPES":"08","GERNELLE":"08","ECORDAL":"08","GIVONNE":"08","DRAIZE":"08","GIVRON":"08","GOMONT":"08","SAULCES CHAMPENOISES":"08","ST PIERRE SUR VENCE":"08","STE VAUBOURG":"08","SORMONNE":"08","ARRIEN EN BETHMALE":"09","ARABAUX":"09","ARTIGAT":"09","ALLIAT":"09","ARNAVE":"09","ALBIES":"09","APPY":"09","VAL D AUZON":"10","AVANT LES MARCILLY":"10","LES BORDES AUMONT":"10","BAGNEUX LA FOSSE":"10","BOUY LUXEMBOURG":"10","BOURGUIGNONS":"10","BLIGNICOURT":"10","BERNON":"10","LA CHAPELLE ST LUC":"10","CHALETTE SUR VOIRE":"10","CHAMP SUR BARSE":"10","COURTAOULT":"10","COURTERON":"10","LA CHAISE":"10","CHAOURCE":"10","CHACENAY":"10","RABAT LES TROIS SEIGNEURS":"09","SAVIGNAC LES ORMEAUX":"09","ST JEAN DE VERGES":"09","ST MARTIN D OYDES":"09","ROQUEFIXADE":"09","SAVERDUN":"09","STE FOI":"09","RIMONT":"09","SEM":"09","EGUILLY SOUS BOIS":"10","DROUPT STE MARIE":"10","DROUPT ST BASLE":"10","DONNEMENT":"10","ECLANCE":"10","DOSCHES":"10","ROSIERES PRES TROYES":"10","PONT STE MARIE":"10","RILLY STE SYRE":"10","RADONVILLIERS":"10","PETIT MESNIL":"10","PROVERVILLE":"10","PEL ET DER":"10","RONCENAY":"10","POLISOT":"10","RHEGES":"10","LA LOGE AUX CHEVRES":"10","MONTCEAUX LES VAUDES":"10","LES LOGES MARGUERON":"10","MARCILLY LE HAYER":"10","LOCHES SUR OURCE":"10","LENTILLES":"10","MONTGUEUX":"10","LESMONT":"10","ERR":"66","ESCARO":"66","ESPIRA DE L AGLY":"66","ESPIRA DE CONFLENT":"66","EUS":"66","LAROQUE DES ALBERES":"66","LLAURO":"66","MILLAS":"66","MONTBOLO":"66","MONTFERRER":"66","REAL":"66","REYNES":"66","RODES":"66","SAHORRE":"66","ST ARNAC":"66","ST ESTEVE":"66","ST GENIS DES FONTAINES":"66","ST JEAN PLA DE CORTS":"66","STE LEOCADIE":"66","ST MARSAL":"66","ST MICHEL DE LLOTES":"66","SERDINYA":"66","LE SOLER":"66","TARERACH":"66","THUES ENTRE VALLS":"66","TORDERES":"66","TORREILLES":"66","TREVILLACH":"66","VILLEFRANCHE DE CONFLENT":"66","LE VIVIER":"66","ALTWILLER":"67","BAREMBACH":"67","BEINHEIM":"67","BELLEFOSSE":"67","BERNARDVILLE":"67","BETTWILLER":"67","BINDERNHEIM":"67","BISCHHEIM":"67","BLIENSCHWILLER":"67","BOOTZHEIM":"67","COSSWILLER":"67","DAHLENHEIM":"67","EBERSHEIM":"67","ERNOLSHEIM LES SAVERNE":"67","FORSTFELD":"67","FROHMUHL":"67","GEISPOLSHEIM":"67","GERSTHEIM":"67","GOTTENHOUSE":"67","GRENDELBRUCH":"67","GRESSWILLER":"67","GRIESHEIM SUR SOUFFEL":"67","HAEGEN":"67","HANDSCHUHEIM":"67","HEILIGENBERG":"67","HILSENHEIM":"67","HIPSHEIM":"67","HOHENGOEFT":"67","HOHFRANKENHEIM":"67","HURTIGHEIM":"67","ILLKIRCH GRAFFENSTADEN":"67","INGENHEIM":"67","ISSENHAUSEN":"67","KESKASTEL":"67","KLEINGOEFT":"67","KOLBSHEIM":"67","LAUTERBOURG":"67","LEUTENHEIM":"67","LITTENHEIM":"67","LOBSANN":"67","MEISTRATZHEIM":"67","MELSHEIM":"67","MERTZWILLER":"67","MINVERSHEIM":"67","MITTELHAUSBERGEN":"67","MITTELSCHAEFFOLSHEIM":"67","MUSSIG":"67","NEUHAEUSEL":"67","PONT AUDEMER":"27","PONT AUTHOU":"27","PONT ST PIERRE":"27","LES PREAUX":"27","QUITTEBEUF":"27","RICHEVILLE":"27","ROSAY SUR LIEURE":"27","ROUGE PERRIERS":"27","ST ANTONIN DE SOMMAIRE":"27","ST AUBIN D ECROSVILLE":"27","ST BENOIT DES OMBRES":"27","ST CYR DE SALERNE":"27","LE VAUDREUIL":"27","ST CYR LA CAMPAGNE":"27","ST DENIS DES MONTS":"27","ST ETIENNE SOUS BAILLEUL":"27","ST LAURENT DU TENCEMENT":"27","ST LEGER DE ROTES":"27","ST PIERRE DE SALERNE":"27","ST PIERRE LA GARENNE":"27","ST SULPICE DE GRIMBOUVILLE":"27","ST VICTOR SUR AVRE":"27","SUZAY":"27","LE THEIL NOLENT":"27","THIBERVILLE":"27","LE TILLEUL LAMBERT":"27","LE TORPT":"27","TOSNY":"27","TOSTES":"27","TOURNEDOS BOIS HUBERT":"27","TOUVILLE":"27","LA TRINITE DE THOUBERVILLE":"27","LE VAL DAVID":"27","VATTEVILLE":"27","VIEUX PORT":"27","VILLEZ SOUS BAILLEUL":"27","VIRONVAY":"27","ABONDANT":"28","ALLUYES":"28","BARJOUVILLE":"28","BELHOMERT GUEHOUVILLE":"28","BERCHERES LES PIERRES":"28","LE BOULLAY THIERRY":"28","BRICONVILLE":"28","CHALLET":"28","CHAMPROND EN PERCHET":"28","LES CHATELLIERS NOTRE DAME":"28","CHUISNES":"28","CLEVILLIERS":"28","CONIE MOLITARD":"28","DAMBRON":"28","EPEAUTROLLES":"28","FRESNAY LE COMTE":"28","FRETIGNY":"28","GARANCIERES EN DROUAIS":"28","GERMAINVILLE":"28","GUAINVILLE":"28","ILLIERS COMBRAY":"28","LAMBLORE":"28","LANDELLES":"28","LANGEY":"28","LOGRON":"28","LOUVILLE LA CHENARD":"28","LUISANT":"28","MAROLLES LES BUIS":"28","LE MEE":"28","MESLAY LE GRENET":"28","MESLAY LE VIDAME":"28","MITTAINVILLIERS VERIGNY":"28","MOINVILLE LA JEULIN":"28","MONTIGNY LE GANNELON":"28","MONTIREAU":"28","NEUVY EN DUNOIS":"28","OLLE":"28","OULINS":"28","PERONVILLE":"28","PONTGOUIN":"28","POUPRY":"28","ST AVIT LES GUESPIERES":"28","ST EMAN":"28","ST HILAIRE SUR YERRE":"28","SAUZELLES":"36","LE TRANGER":"36","URCIERS":"36","VEUIL":"36","VIGOULANT":"36","VOUILLON":"36","AUTRECHE":"37","AVON LES ROCHES":"37","AVRILLE LES PONCEAUX":"37","AZAY LE RIDEAU":"37","BARROU":"37","BETZ LE CHATEAU":"37","BOSSAY SUR CLAISE":"37","CANDES ST MARTIN":"37","CANGEY":"37","CERE LA RONDE":"37","CHAMBOURG SUR INDRE":"37","CHANCEAUX SUR CHOISILLE":"37","CHANNAY SUR LATHAN":"37","LA CHAPELLE BLANCHE ST MARTIN":"37","CHATEAU LA VALLIERE":"37","CHENONCEAUX":"37","CLERE LES PINS":"37","CONTINVOIR":"37","CRISSAY SUR MANSE":"37","CROTELLES":"37","DAME MARIE LES BOIS":"37","FAYE LA VINEUSE":"37","LA GUERCHE":"37","DESCARTES":"37","HUISMES":"37","LARCAY":"37","LOCHE SUR INDROIS":"37","LUZILLE":"37","METTRAY":"37","MONTHODON":"37","NOTRE DAME D OE":"37","NOUANS LES FONTAINES":"37","NOYANT DE TOURAINE":"37","PARCAY MESLAY":"37","RICHELIEU":"37","RILLY SUR VIENNE":"37","ST BAULD":"37","ST LAURENT DE LIN":"37","ST NICOLAS DE BOURGUEIL":"37","ST NICOLAS DES MOTETS":"37","ST SENOCH":"37","SAUNAY":"37","SEMBLANCAY":"37","SUBLAINES":"37","LA TOUR ST GELIN":"37","TROGUES":"37","VEIGNE":"37","VERETZ":"37","VERNOU SUR BRENNE":"37","VILLEBOURG":"37","VILLELOIN COULANGE":"37","VOUVRAY":"37","AGNIN":"38","ALLEVARD":"38","APPRIEU":"38","BEAUVOIR EN ROYANS":"38","BIVIERS":"38","BOSSIEU":"38","BOUGE CHAMBALUD":"38","BRESSIEUX":"38","BREZINS":"38","BRIE ET ANGONNES":"38","LA BUISSIERE":"38","BURCIN":"38","CHANTESSE":"38","CHARANTONNAY":"38","CHASSE SUR RHONE":"38","CHAVANOZ":"38","CHELIEU":"38","CHORANCHE":"38","CLAVANS EN HAUT OISANS":"38","COGNET":"38","LA COMBE DE LANCEY":"38","COMMELLE":"38","CORENC":"38","CRACHIER":"38","CULIN":"38","DOMARIN":"38","DOMENE":"38","LEZIGNE":"49","LINIERES BOUTON":"49","MEIGNE":"49","LA MENITRE":"49","MOULIHERNE":"49","MURS ERIGNE":"49","NOTRE DAME D ALLENCON":"49","NYOISEAU":"49","LE PLESSIS GRAMMOIRE":"49","LA PREVIERE":"49","LE PUY NOTRE DAME":"49","LES ROSIERS SUR LOIRE":"49","ST BARTHELEMY D ANJOU":"49","ST CHRISTOPHE DU BOIS":"49","ST MICHEL ET CHANVEAUX":"49","ST SATURNIN SUR LOIRE":"49","LA SEGUINIERE":"49","SOUZAY CHAMPIGNY":"49","LES ULMES":"49","VAUDELNAY":"49","VEZINS":"49","VILLEMOISAN":"49","AIREL":"50","BACILLY":"50","BAUDRE":"50","BEAUCOUDRAY":"50","BEAUFICEL":"50","BEAUMONT HAGUE":"50","BENOITVILLE":"50","BEUVRIGNY":"50","LA BONNEVILLE":"50","BOURGUENOLLES":"50","BOUTTEVILLE":"50","BRETTEVILLE":"50","BREVANDS":"50","BUAIS LES MONTS":"50","CANISY":"50","CEAUX":"50","COUTANCES":"50","CREANCES":"50","CROLLON":"50","ECAUSSEVILLE":"50","EMONDEVILLE":"50","FLOTTEMANVILLE":"50","FONTENAY SUR MER":"50","GEFFOSSES":"50","LA GOHANNIERE":"50","GRATOT":"50","HAUTTEVILLE BOCAGE":"50","HEUGUEVILLE SUR SIENNE":"50","JUVIGNY LE TERTRE":"50","LESSAY":"50","LINGREVILLE":"50","LE LOREUR":"50","LA LUCERNE D OUTREMER":"50","MARGUERAY":"50","MAUPERTUIS":"50","LA MEAUFFE":"50","LE MESNIL EURY":"50","LE MESNIL OZENNE":"50","LE MESNIL ROUXELIN":"50","LE MESNIL TOVE":"50","LE MESNIL VENERON":"50","LE MESNIL VIGOT":"50","LEVIGNY":"10","NEUVILLE SUR SEINE":"10","MONTIER EN L ISLE":"10","NOGENT EN OTHE":"10","ORIGNY LE SEC":"10","MONTIERAMEY":"10","MONTPOTHIER":"10","MOREMBERT":"10","PARGUES":"10","ONJON":"10","ESTISSAC":"10","JONCREUIL":"10","JAVERNANT":"10","HAMPIGNY":"10","LANTAGES":"10","FRESNAY":"10","LAGESSE":"10","CAUNETTE SUR LAUQUET":"11","CLERMONT SUR LAUQUET":"11","COUFFOULENS":"11","CAVANAC":"11","COUDONS":"11","LE PONDY":"18","RAYMOND":"18","ST AMAND MONTROND":"18","STE GEMME EN SANCERROIS":"18","ST GEORGES SUR LA PREE":"18","STE LUNAISE":"18","ST OUTRILLE":"18","ST PRIEST LA MARCHE":"18","SAVIGNY EN SANCERRE":"18","SOYE EN SEPTAINE":"18","SURY ES BOIS":"18","THAUMIERS":"18","VALLENAY":"18","VASSELAY":"18","VIERZON":"18","VINON":"18","ALBUSSAC":"19","ALTILLAC":"19","AMBRUGEAT":"19","ARNAC POMPADOUR":"19","BAR":"19","BASSIGNAC LE HAUT":"19","BEAULIEU SUR DORDOGNE":"19","BEYSSENAC":"19","BONNEFOND":"19","CHABRIGNAC":"19","CHAMPAGNAC LA PRUNE":"19","CHANTEIX":"19","CHIRAC BELLEVUE":"19","COLLONGES LA ROUGE":"19","CONCEZE":"19","DARNETS":"19","EGLETONS":"19","ESTIVALS":"19","EYBURIE":"19","GOURDON MURAT":"19","GRANDSAIGNE":"19","LAGARDE ENVAL":"19","LAMAZIERE HAUTE":"19","LANTEUIL":"19","MADRANGES":"19","MEILHARDS":"19","MONESTIER PORT DIEU":"19","ORLIAC DE BAR":"19","PALISSE":"19","REYGADE":"19","RILHAC XAINTRIE":"19","ST BAZILE DE MEYSSAC":"19","ST EXUPERY LES ROCHES":"19","ST GERMAIN LAVOLPS":"19","ST HILAIRE LES COURBES":"19","ST MEXANT":"19","ST SALVADOUR":"19","SALON LA TOUR":"19","SARROUX":"19","USSAC":"19","VEGENNES":"19","VIAM":"19","VIGNOLS":"19","AISEREY":"21","ARCENANT":"21","ARNAY SOUS VITTEAUX":"21","AUBAINE":"21","AVELANGES":"21","BALOT":"21","BARBIREY SUR OUCHE":"21","BARJON":"21","BEAUMONT SUR VINGEANNE":"21","NEUVE EGLISE":"67","NIEDERNAI":"67","OBERSOULTZBACH":"67","OHNENHEIM":"67","OSTHOUSE":"67","PFALZWEYER":"67","PLOBSHEIM":"67","RATZWILLER":"67","REICHSFELD":"67","RETSCHWILLER":"67","RINGENDORF":"67","ROHRWILLER":"67","ROSTEIG":"67","ROTTELSHEIM":"67","ROUNTZENHEIM":"67","SARREWERDEN":"67","SAVERNE":"67","SCHARRACHBERGHEIM IRMSTETT":"67","SCHERWILLER":"67","SCHILLERSDORF":"67","SCHIRRHEIN":"67","SCHOENAU":"67","SOUFFELWEYERSHEIM":"67","SOUFFLENHEIM":"67","SOULTZ SOUS FORETS":"67","SPARSBACH":"67","STATTMATTEN":"67","VENDENHEIM":"67","VOLKSBERG":"67","WAHLENHEIM":"67","WEYERSHEIM":"67","WILDERSBACH":"67","WILWISHEIM":"67","WINDSTEIN":"67","WOERTH":"67","WOLFISHEIM":"67","WOLSCHHEIM":"67","ZINSWILLER":"67","BATTENHEIM":"68","BETTLACH":"68","BOURBACH LE HAUT":"68","BRUNSTATT DIDENHEIM":"68","DIETWILLER":"68","DOLLEREN":"68","EGLINGEN":"68","ESCHBACH AU VAL":"68","FELDBACH":"68","FELLERING":"68","FRIESEN":"68","FULLEREN":"68","GILDWILLER":"68","GUEBERSCHWIHR":"68","GUEWENHEIM":"68","GUNSBACH":"68","HAGENTHAL LE BAS":"68","HARTMANNSWILLER":"68","HEIMERSDORF":"68","HETTENSCHLAG":"68","HUSSEREN LES CHATEAUX":"68","ILLFURTH":"68","ILLZACH":"68","JETTINGEN":"68","KATZENTHAL":"68","KINGERSHEIM":"68","LAPOUTROIE":"68","LUEMSCHWILLER":"68","LUTTERBACH":"68","MERTZEN":"68","MERXHEIM":"68","MICHELBACH LE BAS":"68","MOOSCH":"68","LE HAUT SOULTZBACH":"68","MUESPACH LE HAUT":"68","MULHOUSE":"68","ST JEAN PIERRE FIXTE":"28","ST LAURENT LA GATINE":"28","ST LEGER DES AUBEES":"28","ST LUCIEN":"28","ST LUPERCE":"28","ST REMY SUR AVRE":"28","SAUMERAY":"28","SERVILLE":"28","TRIZAY COUTRETOT ST SERGE":"28","VIEUVICQ":"28","VILLAMPUY":"28","VILLEAU":"28","VILLIERS LE MORHIER":"28","VOISE":"28","ARGOL":"29","BENODET":"29","BOURG BLANC":"29","BRIGNOGAN PLAGE":"29","CARANTEC":"29","CLOHARS FOUESNANT":"29","COMBRIT":"29","DAOULAS":"29","LE DRENNEC":"29","ERGUE GABERIC":"29","LA FORET FOUESNANT":"29","GOULVEN":"29","GUENGAT":"29","GUICLAN":"29","GUILLIGOMARC H":"29","GUIPAVAS":"29","HOPITAL CAMFROUT":"29","ILE MOLENE":"29","KERNILIS":"29","LANDUDEC":"29","LANILDUT":"29","LANNEUFFRET":"29","LANVEOC":"29","LOC BREVALAIRE":"29","LOCMARIA BERRIEN":"29","CONFORT MEILARS":"29","MELGVEN":"29","PENCRAN":"29","PLEYBER CHRIST":"29","PLOGONNEC":"29","PLONEIS":"29","PLOUDALMEZEAU":"29","PLOUGASTEL DAOULAS":"29","PLOUGOURVEST":"29","PLOUGUERNEAU":"29","PLOUIGNEAU":"29","PLOUNEOUR TREZ":"29","PLOUVIEN":"29","PLOVAN":"29","PLUGUFFAN":"29","LE PONTHOU":"29","PORT LAUNAY":"29","POULDREUZIC":"29","QUEMENEVEN":"29","ROSPORDEN":"29","ST COULITZ":"29","ST EVARZEC":"29","ST HERNIN":"29","ST JEAN DU DOIGT":"29","ST MEEN":"29","ST NIC":"29","ST RIVOAL":"29","SIZUN":"29","TREFFIAGAT":"29","TREFLAOUENAN":"29","TREFLEZ":"29","TREGOUREZ":"29","TREGUENNEC":"29","TREMAOUEZAN":"29","ALES":"30","ANDUZE":"30","ARGILLIERS":"30","ARPHY":"30","AUBORD":"30","AUMESSAS":"30","BAGARD":"30","ECHIROLLES":"38","GILLONNAY":"38","LE GRAND LEMPS":"38","HURTIERES":"38","JARCIEU":"38","JARDIN":"38","JARRIE":"38","LEYRIEU":"38","MARCOLLIN":"38","MARNANS":"38","MAYRES SAVEL":"38","AUTRANS MEAUDRE EN VERCORS":"38","MENS":"38","MEYRIE":"38","MIRIBEL LANCHATRE":"38","MORESTEL":"38","LA MOTTE D AVEILLANS":"38","MURINAIS":"38","NANTES EN RATIER":"38","NOTRE DAME DE COMMIERS":"38","NOTRE DAME DE MESAGE":"38","OULLES":"38","OYTIER ST OBLAS":"38","PANOSSAS":"38","PARMILIEU":"38","PIERRE CHATEL":"38","PONTCHARRA":"38","PORCIEU AMBLAGNIEU":"38","PREBOIS":"38","QUAIX EN CHARTREUSE":"38","REVEL TOURDAN":"38","ROCHETOIRIN":"38","ROMAGNIEU":"38","ROYBON":"38","ST ALBAN DE ROCHE":"38","ST ANDRE LE GAZ":"38","ST BAUDILLE ET PIPET":"38","ST BLAISE DU BUIS":"38","ST BONNET DE CHAVAGNE":"38","ST CLAIR DE LA TOUR":"38","ST GUILLAUME":"38","ST HILAIRE DE LA COTE":"38","ST JEAN DE BOURNAY":"38","ST LAURENT DU PONT":"38","ST MARTIN D HERES":"38","ST MARTIN LE VINOUX":"38","ST MAURICE EN TRIEVES":"38","ST PIERRE DE BRESSIEUX":"38","ST PIERRE DE MESAGE":"38","ST ROMAIN DE JALIONAS":"38","SALAISE SUR SANNE":"38","SEREZIN DE LA TOUR":"38","SERMERIEU":"38","SERPAIZE":"38","SEYSSINS":"38","SIEVOZ":"38","TENCIN":"38","LA TERRASSE":"38","THODURE":"38","LA TRONCHE":"38","VALBONNAIS":"38","VALENCIN":"38","VERNIOZ":"38","LA VERPILLIERE":"38","VEZERONCE CURTIN":"38","VIENNE":"38","VILLARD BONNOT":"38","VILLEMOIRIEU":"38","VILLENEUVE DE MARC":"38","VILLE SOUS ANJOU":"38","VIZILLE":"38","VOISSANT":"38","CHAMROUSSE":"38","ARINTHOD":"39","ARLAY":"39","LES ARSURES":"39","ARSURE ARSURETTE":"39","AUGEA":"39","BALAISEAUX":"39","BANS":"39","LES MOITIERS D ALLONNE":"50","MONTHUCHON":"50","NEHOU":"50","NICORPS":"50","OZEVILLE":"50","GRANDPARIGNY":"50","REFFUVEILLE":"50","REVILLE":"50","RONCEY":"50","LA RONDE HAYE":"50","LE ROZEL":"50","ST AUBIN DES PREAUX":"50","ST DENIS LE VETU":"50","ST GEORGES D ELLE":"50","ST GERMAIN DE TOURNEBUT":"50","ST JEAN LE THOMAS":"50","ST LAURENT DE CUVES":"50","ST LAURENT DE TERREGATTE":"50","ST MALO DE LA LANDE":"50","ST MAUR DES BOIS":"50","ST MICHEL DE MONTJOIE":"50","ST PAIR SUR MER":"50","ST PIERRE D ARTHEGLISE":"50","ST PIERRE DE COUTANCES":"50","ST QUENTIN SUR LE HOMME":"50","ST SAUVEUR DE PIERREPONT":"50","SIDEVILLE":"50","SORTOSVILLE":"50","SOURDEVAL LES BOIS":"50","TOLLEVAST":"50","TREAUVILLE":"50","TRIBEHOU":"50","URVILLE NACQUEVILLE":"50","VAINS":"50","LE VAST":"50","VER":"50","DENEUILLE LES MINES":"03","ECHASSIERES":"03","ESCUROLLES":"03","LIERNOLLES":"03","DOMERAT":"03","DOYET":"03","LA VILLE AUX BOIS LES DIZY":"02","VILLENEUVE ST GERMAIN":"02","VILLERS AGRON AIGUIZY":"02","VERNEUIL SUR SERRE":"02","VICHEL NANTEUIL":"02","VIC SUR AISNE":"02","VERVINS":"02","VIERZY":"02","VESLUD":"02","LA VALLEE MULATRE":"02","SONS ET RONCHERES":"02","TRELOU SUR MARNE":"02","TAILLEFONTAINE":"02","VENEROLLES":"02","THENELLES":"02","VENDHUILE":"02","VENDEUIL":"02","VAUXTIN":"02","SUZY":"02","DENEUILLE LES CHANTELLE":"03","BOURBON L ARCHAMBAULT":"03","CHIRAT L EGLISE":"03","LA CHAPELAUDE":"03","COUTANSOUZE":"03","CRECHY":"03","SERAUCOURT LE GRAND":"02","ST PAUL AUX BOIS":"02","SEPTMONTS":"02","ST GOBAIN":"02","BEZE":"21","BEZOUOTTE":"21","BLAISY BAS":"21","BLAISY HAUT":"21","BONCOURT LE BOIS":"21","BOUDREVILLE":"21","BOUILLAND":"21","BOUIX":"21","BREMUR ET VAUROIS":"21","BRETENIERE":"21","BUFFON":"21","LA BUSSIERE SUR OUCHE":"21","BUSSY LE GRAND":"21","CESSEY SUR TILLE":"21","CHAMPAGNE SUR VINGEANNE":"21","CHAMPAGNY":"21","CHAMPDOTRE":"21","CHATILLON SUR SEINE":"21","CHAUDENAY LA VILLE":"21","CHAUGEY":"21","CHEUGE":"21","CHEVANNAY":"21","CIVRY EN MONTAGNE":"21","CLEMENCEY":"21","COMBERTAULT":"21","CORCELLES LES CITEAUX":"21","COULMIER LE SEC":"21","COUTERNON":"21","CURLEY":"21","DAIX":"21","DARCEY":"21","ECHANNAY":"21","ECUTIGNY":"21","ESSEY":"21","FIXIN":"21","FLAVIGNEROT":"21","FONTAINE LES DIJON":"21","FONTANGY":"21","FRAIGNOT ET VESVROTTE":"21","GEMEAUX":"21","GENLIS":"21","GEVREY CHAMBERTIN":"21","LES GOULLES":"21","GURGY LE CHATEAU":"21","JAILLY LES MOULINS":"21","JALLANGES":"21","JEUX LES BARD":"21","VAL MONT":"21","LANTILLY":"21","LEUGLAY":"21","LONGECOURT EN PLAINE":"21","LONGECOURT LES CULETRE":"21","LUSIGNY SUR OUCHE":"21","MAGNIEN":"21","MAGNY LES AUBIGNY":"21","MAGNY MONTARLOT":"21","MAGNY LES VILLERS":"21","MARCHESEUIL":"21","MARCILLY OGNY":"21","MARLIENS":"21","MARSANNAY LA COTE":"21","MARSANNAY LE BOIS":"21","MASSINGY LES VITTEAUX":"21","MAXILLY SUR SAONE":"21","LE MEIX":"21","MEULSON":"21","MOITRON":"21","MONTAGNY LES BEAUNE":"21","MONTLAY EN AUXOIS":"21","MONTMANCON":"21","MOSSON":"21","MUSIGNY":"21","NESLE ET MASSOULT":"21","NOIRON SOUS GEVREY":"21","NORMIER":"21","OBTREE":"21","ORAIN":"21","ORGEUX":"21","PAGNY LE CHATEAU":"21","PLOMBIERES LES DIJON":"21","PLUVAULT":"21","PLUVET":"21","POISEUL LA GRANGE":"21","POISEUL LES SAULX":"21","POUILLY EN AUXOIS":"21","PULIGNY MONTRACHET":"21","LA ROCHE EN BRENIL":"21","STE COLOMBE EN AUXOIS":"21","MURBACH":"68","NIEDERENTZEN":"68","OBERLARG":"68","OSENBACH":"68","OSTHEIM":"68","PETIT LANDAU":"68","RANSPACH":"68","RETZWILLER":"68","RODEREN":"68","RORSCHWIHR":"68","RUEDERBACH":"68","ST AMARIN":"68","ST ULRICH":"68","SEPPOIS LE HAUT":"68","SIERENTZ":"68","SOPPE LE BAS":"68","SOULTZ HAUT RHIN":"68","SOULTZMATT":"68","STAFFELFELDEN":"68","STERNENBERG":"68","STETTEN":"68","STRUETH":"68","SUNDHOFFEN":"68","THANNENKIRCH":"68","UNGERSHEIM":"68","URSCHENHEIM":"68","VIEUX FERRETTE":"68","VOGELGRUN":"68","VOLGELSHEIM":"68","WALBACH":"68","WALHEIM":"68","WECKOLSHEIM":"68","WERENTZHOUSE":"68","WIHR AU VAL":"68","WITTENHEIM":"68","ZIMMERSHEIM":"68","ANSE":"69","L ARBRESLE":"69","BLACE":"69","BRIGNAIS":"69","BRULLIOLES":"69","CALUIRE ET CUIRE":"69","CONDRIEU":"69","CRAPONNE":"69","DARDILLY":"69","DIEME":"69","ECHALAS":"69","EMERINGES":"69","FLEURIEU SUR SAONE":"69","GRANDRIS":"69","GREZIEU LE MARCHE":"69","JARNIOUX":"69","LACENAS":"69","LACHASSAGNE":"69","LIERGUES":"69","LOZANNE":"69","MARCHAMPT":"69","MESSIMY":"69","MONTROMANT":"69","ORLIENAS":"69","PONTCHARRA SUR TURDINE":"69","POULE LES ECHARMEAUX":"69","PROPIERES":"69","QUINCIE EN BEAUJOLAIS":"69","QUINCIEUX":"69","REGNIE DURETTE":"69","ST DIDIER SUR BEAUJEU":"69","ST FONS":"69","ST GERMAIN NUELLES":"69","ST JACQUES DES ARRETS":"69","ST JEAN D ARDIERES":"69","ST JEAN DES VIGNES":"69","ST LAURENT DE CHAMOUSSET":"69","ST LAURENT D OINGT":"69","ST MAMERT":"69","ST NIZIER D AZERGUES":"69","STE PAULE":"69","ST PIERRE LA PALUD":"69","ST SORLIN":"69","BLANDAS":"30","BOUILLARGUES":"30","BOUQUET":"30","LA CAPELLE ET MASMOLENE":"30","CARNAS":"30","CARSAN":"30","CASTELNAU VALENCE":"30","CAUSSE BEGON":"30","CODOLET":"30","CORBES":"30","CORCONNE":"30","COURRY":"30","DOMAZAN":"30","DOURBIES":"30","FONTARECHES":"30","GAILHAN":"30","GALLARGUES LE MONTUEUX":"30","LE GARN":"30","JUNAS":"30","LAVAL PRADEL":"30","LAVAL ST ROMAN":"30","LEDENON":"30","MANDAGOUT":"30","MANDUEL":"30","MARUEJOLS LES GARDON":"30","MOLIERES SUR CEZE":"30","MONTFRIN":"30","MUS":"30","NAGES ET SOLORGUES":"30","PEYREMALE":"30","LES PLANTIERS":"30","PONTEILS ET BRESIS":"30","REVENS":"30","ROGUES":"30","ROQUEDUR":"30","ST BENEZET":"30","ST CHRISTOL LES ALES":"30","ST COME ET MARUEJOLS":"30","ST DIONISY":"30","ST JEAN DE VALERISCLE":"30","ST JEAN DU PIN":"30","ST JULIEN DE PEYROLAS":"30","ST JULIEN LES ROSIERS":"30","ST JUST ET VACQUIERES":"30","ST LAURENT D AIGOUZE":"30","ST MAMERT DU GARD":"30","ST MAURICE DE CAZEVIEILLE":"30","ST MICHEL D EUZET":"30","ST QUENTIN LA POTERIE":"30","ST SAUVEUR CAMPRIEU":"30","SALINELLES":"30","SENECHAS":"30","SERVIERS ET LABAUME":"30","SOMMIERES":"30","SOUSTELLE":"30","THARAUX":"30","THEZIERS":"30","TORNAC":"30","VEZENOBRES":"30","VILLEVIEILLE":"30","ST PAUL LES FONTS":"30","AYGUESVIVES":"31","ARBAS":"31","ARDIEGE":"31","AURIN":"31","AUSSONNE":"31","AUZIELLE":"31","BAGIRY":"31","BEAUZELLE":"31","BELBERAUD":"31","BELBEZE EN COMMINGES":"31","BILLIERE":"31","BOUSSENS":"31","BRAGAYRAC":"31","CAMBIAC":"31","CANENS":"31","CARDEILHAC":"31","CASTELGINEST":"31","CASTERA VIGNOLES":"31","CAUJAC":"31","CAZAUX LAYRISSE":"31","BEFFIA":"39","BELLECOMBE":"39","BESAIN":"39","BIEF DU FOURG":"39","BOIS D AMONT":"39","BRETENIERES":"39","BREVANS":"39","BUVILLY":"39","CHAINEE DES COUPIS":"39","CHAMBERIA":"39","CHARNOD":"39","CHATELAY":"39","LA CHAUMUSSE":"39","CHAUSSIN":"39","LA CHAUX DU DOMBIEF":"39","CHILLE":"39","LES CROZETS":"39","CUISIA":"39","DAMPARIS":"39","DOUCIER":"39","DOURNON":"39","ECLANS NENON":"39","LES ESSARDS TAIGNEVAUX":"39","ESSERVAL TARTRE":"39","ETREPIGNEY":"39","FALLETANS":"39","LE FIED":"39","FONTENU":"39","FORT DU PLASNE":"39","FRAROZ":"39","GEVRY":"39","GRAYE ET CHARNAY":"39","GRUSSE":"39","LARNAUD":"39","LE LATET":"39","LAVANCIA EPERCY":"39","LAVANS SUR VALOUSE":"39","LESCHERES":"39","LOISIA":"39","LONS LE SAUNIER":"39","LE LOUVEROT":"39","LA LOYE":"39","MARTIGNA":"39","MAYNAL":"39","MENETRUX EN JOUX":"39","MOIRANS EN MONTAGNE":"39","MONNET LA VILLE":"39","MONTEPLAIN":"39","MONTIGNY LES ARSURES":"39","NANCE":"39","OUR":"39","PAGNOZ":"39","PARCEY":"39","PASSENANS":"39","PATORNAY":"39","LE PETIT MERCEY":"39","PETIT NOIR":"39","PILLEMOINE":"39","LES PLANCHES EN MONTAGNE":"39","POINTRE":"39","PONT D HERY":"39","PONT DU NAVOY":"39","PUBLY":"39","RAINANS":"39","RECANOZ":"39","ROGNA":"39","ROUFFANGE":"39","ST HYMETIERE":"39","ST JEAN D ETREUX":"39","SALINS LES BAINS":"39","SANTANS":"39","ST GOBERT":"02","SAMOUSSY":"02","SOMMERON":"02","VILLERS SUR FERE":"02","VILLE SAVOYE":"02","VUILLERY":"02","WASSIGNY":"02","AUROUER":"03","VIVAISE":"02","BESSON":"03","BOST":"03","BERT":"03","NOYANT D ALLIER":"03","PARAY LE FRESIL":"03","NEURE":"03","LA CONDAMINE CHATELARD":"04","ENCHASTRAYES":"04","ENTRAGES":"04","CHATEAUNEUF VAL ST DONAT":"04","CHATEAUNEUF MIRAVAIL":"04","COLMARS":"04","ENTREPIERRES":"04","ST POURCAIN SUR BESBRE":"03","ST GERAND DE VAUX":"03","ST REMY EN ROLLAT":"03","ST ELOY D ALLIER":"03","ST PIERRE LAVAL":"03","ST YORRE":"03","AUBIGNOSC":"04","BARREME":"04","VITRAY":"03","BARLES":"04","AUZET":"04","TOULON SUR ALLIER":"03","TORTEZAIS":"03","VILLEBRET":"03","LE THEIL":"03","SANSSAT":"03","THIONNE":"03","VIEURE":"03","VAUMAS":"03","LA MOTTE DU CAIRE":"04","LES OMERGUES":"04","MONTLAUX":"04","OPPEDETTE":"04","ONGLES":"04","FAUCON DE BARCELONNETTE":"04","LE LAUZET UBAYE":"04","ENTREVENNES":"04","FORCALQUIER":"04","MEAILLES":"04","MALIJAI":"04","GIGORS":"04","LURS":"04","ST ETIENNE LES ORGUES":"04","STE CROIX A LAUZE":"04","REVEST ST MARTIN":"04","REVEST DU BION":"04","PUIMICHEL":"04","LARAGNE MONTEGLIN":"05","FOREST ST JULIEN":"05","LE GLAIZIL":"05","MONTBRAND":"05","LES ORRES":"05","LETTRET":"05","CROTS":"05","ST ETIENNE LE LAUS":"05","PUY ST PIERRE":"05","LA PIARRE":"05","REMOLLON":"05","RIBEYRET":"05","STE COLOMBE SUR SEINE":"21","ST EUPHRONE":"21","ST JEAN DE LOSNE":"21","ST LEGER TRIEY":"21","STE MARIE LA BLANCHE":"21","ST PRIX LES ARNAY":"21","ST VICTOR SUR OUCHE":"21","SALIVES":"21","SELONGEY":"21","SENNECEY LES DIJON":"21","SOISSONS SUR NACEY":"21","SOUHEY":"21","TANAY":"21","TART L ABBAYE":"21","THOISY LE DESERT":"21","TIL CHATEL":"21","TILLENAY":"21","TRECLUN":"21","UNCEY LE FRANC":"21","VAROIS ET CHAIGNOT":"21","VAUX SAULES":"21","VERDONNET":"21","VIANGES":"21","VIC DE CHASSENAY":"21","VIC DES PRES":"21","VIELVERGE":"21","VIEUX CHATEAU":"21","VILLERS PATRAS":"21","VILLERS ROTIN":"21","VILLEY SUR TILLE":"21","VILLIERS EN MORVAN":"21","VISERNY":"21","VONGES":"21","BELLE ISLE EN TERRE":"22","BOQUEHO":"22","LA BOUILLIE":"22","BREHAND":"22","CALLAC":"22","CALORGUEN":"22","CAMLEZ":"22","CAOUENNEC LANVEZEAC":"22","CAULNES":"22","CORLAY":"22","EREAC":"22","GOUDELIN":"22","HEMONSTOIR":"22","HENON":"22","JUGON LES LACS COMMUNE NOUVELLE":"22","KERMARIA SULARD":"22","LAMBALLE":"22","LANCIEUX":"22","LANGUEDIAS":"22","LANLOUP":"22","LANMERIN":"22","LANVOLLON":"22","LESCOUET GOUAREC":"22","LEZARDRIEUX":"22","LOCARN":"22","LOHUEC":"22","LOUANNEC":"22","MAGOAR":"22","LE MERZER":"22","MORIEUX":"22","LE MOUSTOIR":"22","NOYAL":"22","PLANCOET":"22","PLANGUENOUAL":"22","PLEDELIAC":"22","PLEHEDEL":"22","LES MOULINS":"22","PLEMY":"22","PLENEE JUGON":"22","PLERIN":"22","PLEUDANIEL":"22","PLEUMEUR BODOU":"22","PLEVIN":"22","PLOUEZEC":"22","PLOUGONVER":"22","PLOUNEVEZ QUINTIN":"22","PLUMAUDAN":"22","PONT MELVEZ":"22","LE QUILLIO":"22","TASSIN LA DEMI LUNE":"69","VAULX EN VELIN":"69","VAUX EN BEAUJOLAIS":"69","VENISSIEUX":"69","VERNAISON":"69","CHASSIEU":"69","LYON 05":"69","LYON 07":"69","ADELANS ET LE VAL DE BITHAINE":"70","ANDORNAY":"70","ANJEUX":"70","LAURAGUEL":"11","LIGNAIROLLES":"11","LIMOUX":"11","MAGRIE":"11","MARCORIGNAN":"11","MARSA":"11","MAZEROLLES DU RAZES":"11","MAZUBY":"11","MIRAVAL CABARDES":"11","MIREVAL LAURAGAIS":"11","MONTJARDIN":"11","MONTOLIEU":"11","MOUSSAN":"11","MOUX":"11","PARAZA":"11","PAZIOLS":"11","PECHARIC ET LE PY":"11","PEYRIAC MINERVOIS":"11","PLAVILLA":"11","PORTEL DES CORBIERES":"11","PUICHERIC":"11","PUIVERT":"11","RAISSAC D AUDE":"11","RIEUX MINERVOIS":"11","RIVEL":"11","ROULLENS":"11","STE CAMELLE":"11","ST COUAT D AUDE":"11","ST FERRIOL":"11","ST GAUDERIC":"11","ST JULIEN DE BRIOLA":"11","ST LOUIS ET PARAHOU":"11","ST MARCEL SUR AUDE":"11","ST MARTIN DES PUITS":"11","ST MARTIN LALANDE":"11","ST MARTIN LYS":"11","SALLELES CABARDES":"11","SALLES SUR L HERS":"11","SALZA":"11","SIGEAN":"11","SOUGRAIGNE":"11","TRASSANEL":"11","VENTENAC EN MINERVOIS":"11","VILLANIERE":"11","VILLAR ST ANSELME":"11","VILLARZEL CABARDES":"11","VILLAUTOU":"11","VILLEFLOURE":"11","VILLEMAGNE":"11","VILLENEUVE LA COMPTAL":"11","VILLENEUVE MINERVOIS":"11","VILLEROUGE TERMENES":"11","VILLESPY":"11","VILLETRITOULS":"11","VINASSAN":"11","ALMONT LES JUNIES":"12","ANGLARS ST FELIX":"12","AYSSENES":"12","BALSAC":"12","LE BAS SEGALA":"12","BRANDONNET":"12","BRASC":"12","CALMELS ET LE VIALA":"12","LA CAPELLE BONANCE":"12","CASTELNAU DE MANDAILLES":"12","CASTELNAU PEGAYROLS":"12","CLAIRVAUX D AVEYRON":"12","COLOMBIES":"12","COMBRET":"12","CONDOM D AUBRAC":"12","CONNAC":"12","CURIERES":"12","DRULHE":"12","DURENQUE":"12","GAILLAC D AVEYRON":"12","CIADOUX":"31","CLERMONT LE FORT":"31","COULADERE":"31","COX":"31","EMPEAUX":"31","ESPERCE":"31","ESTANCARBON":"31","EUP":"31","FONBEAUZARD":"31","FONTENILLES":"31","LE FOUSSERET":"31","GARAC":"31","GARIDECH":"31","HIS":"31","JUZES":"31","LABARTHE INARD":"31","LABARTHE RIVIERE":"31","LABARTHE SUR LEZE":"31","LABASTIDE CLERMONT":"31","LABASTIDE PAUMES":"31","LABASTIDE ST SERNIN":"31","LABRUYERE DORSA":"31","LAGARDELLE SUR LEZE":"31","LAGRACE DIEU":"31","LAHITERE":"31","LALOURET LAFFITEAU":"31","LANDORTHE":"31","LATOUE":"31","LESPINASSE":"31","LESPITEAU":"31","LESPUGUE":"31","LEVIGNAC":"31","LIEOUX":"31","LOUDET":"31","MANCIOUX":"31","MARIGNAC LASCLARES":"31","MARLIAC":"31","MERVILLA":"31","MIREPOIX SUR TARN":"31","MOLAS":"31","MONTASTRUC LA CONSEILLERE":"31","MONTBERAUD":"31","MONTBRUN LAURAGAIS":"31","MONTCLAR DE COMMINGES":"31","MONTESQUIEU VOLVESTRE":"31","MONTGAILLARD SUR SAVE":"31","MONTOULIEU ST BERNARD":"31","MONTRABE":"31","MURET":"31","NAILLOUX":"31","NOUEILLES":"31","ODARS":"31","ONDES":"31","PECHBUSQUE":"31","PEYRISSAS":"31","PEYROUZET":"31","LE PIN MURELET":"31","PINS JUSTARET":"31","LE PLAN":"31","POUY DE TOUGES":"31","PRESERVILLE":"31","PROUPIARY":"31","PUYDANIEL":"31","PUYMAURIN":"31","PUYSSEGUR":"31","RAMONVILLE ST AGNE":"31","RIEUMES":"31","RIOLAS":"31","ROQUESERIERE":"31","ROQUETTES":"31","ST CEZERT":"31","ST ELIX LE CHATEAU":"31","ST ELIX SEGLAN":"31","ST GAUDENS":"31","ST GENIES BELLEVUE":"31","ST PIERRE DE LAGES":"31","SALEICH":"31","LA SALVETAT LAURAGAIS":"31","SARRECAVE":"31","SARREMEZAN":"31","SAUSSENS":"31","SENARENS":"31","SEYSSES":"31","TARABEL":"31","LES TOURREILLES":"31","VENDINE":"31","LARRA":"31","ARBLADE LE HAUT":"32","SAUGEOT":"39","SERGENAUX":"39","SONGESON":"39","TAXENNE":"39","THOIRETTE":"39","THOIRIA":"39","TOULOUSE LE CHATEAU":"39","TOURMONT":"39","UXELLES":"39","VALEMPOULIERES":"39","VANNOZ":"39","VEVY":"39","LA VIEILLE LOYE":"39","VILLENEUVE SOUS PYMONT":"39","VILLERS LES BOIS":"39","VULVOZ":"39","ARENGOSSE":"40","ARGELOUSE":"40","ARJUZANX":"40","BASSERCLES":"40","BATS":"40","BENQUET":"40","BETBEZER D ARMAGNAC":"40","BEYLONGUE":"40","BONNEGARDE":"40","BOUGUE":"40","CACHEN":"40","CALLEN":"40","CANDRESSE":"40","CASSEN":"40","CASTANDET":"40","CASTELNER":"40","CASTETS":"40","CAUPENNE":"40","CERE":"40","DUHORT BACHEN":"40","DUMES":"40","ESCALANS":"40","GARROSSE":"40","GAUJACQ":"40","GOURBERA":"40","GOUTS":"40","LACQUY":"40","LOSSE":"40","LUGLON":"40","MAILLERES":"40","MAUVEZIN D ARMAGNAC":"40","MIRAMONT SENSACQ":"40","MORCENX":"40","NASSIET":"40","NOUSSE":"40","ORTHEVIELLE":"40","OZOURT":"40","PARLEBOSCQ":"40","POMAREZ":"40","PONTENX LES FORGES":"40","PORT DE LANNE":"40","POUDENX":"40","POUYDESSEAUX":"40","POYANNE":"40","ST AGNET":"40","ST GEOURS D AURIBAT":"40","ST GOR":"40","ST JEAN DE LIER":"40","ST JEAN DE MARSACQ":"40","ST JULIEN D ARMAGNAC":"40","ST LAURENT DE GOSSE":"40","ST LON LES MINES":"40","STE MARIE DE GOSSE":"40","ST MICHEL ESCALUS":"40","ST PAUL EN BORN":"40","ST PERDON":"40","SARBAZAN":"40","SARRON":"40","SAUGNAC ET CAMBRAN":"40","LE SEN":"40","TOSSE":"40","UCHACQ ET PARENTIS":"40","VICQ D AURIBAT":"40","VIELLE ST GIRONS":"40","BONNEVEAU":"41","CHAILLES":"41","LA CHAPELLE ENCHERIE":"41","CHATRES SUR CHER":"41","CHITENAY":"41","CHOUZY SUR CISSE":"41","COUR SUR LOIRE":"41","EPUISAY":"41","LA FERTE BEAUHARNAIS":"41","GIEVRES":"41","HERBAULT":"41","LUNAY":"41","LA MADELEINE VILLEFROUIN":"41","RISOUL":"05","ROSANS":"05","ST MICHEL L OBSERVATOIRE":"04","UVERNET FOURS":"04","ST JULIEN D ASSE":"04","ST GENIEZ":"04","SISTERON":"04","VACHERES":"04","CHAMPOLEON":"05","CHABESTAN":"05","AIGUILLES":"05","ANCELLE":"05","CREVOUX":"05","BRUIS":"05","ST JEAN ST NICOLAS":"05","ST MARTIN DE QUEYRIERES":"05","ST LAURENT DU CROS":"05","TANTONVILLE":"54","THUMEREVILLE":"54","TOMBLAINE":"54","TOUL":"54","TRAMONT EMY":"54","TRONDES":"54","TRONVILLE":"54","VANNES LE CHATEL":"54","VAUCOURT":"54","VENEY":"54","VENNEZEY":"54","VERDENAL":"54","VIGNEULLES":"54","VILLE AU MONTOIS":"54","VILLE AU VAL":"54","VILLERS LES MOIVRONS":"54","VILLERUPT":"54","VIRECOURT":"54","VITERNE":"54","VRONCOURT":"54","XIROCOURT":"54","XOUSSE":"54","AUBREVILLE":"55","BAALON":"55","BEAUFORT EN ARGONNE":"55","BEAULIEU EN ARGONNE":"55","BELLEVILLE SUR MEUSE":"55","BELRUPT EN VERDUNOIS":"55","BETHELAINVILLE":"55","BEUREY SUR SAULX":"55","BIENCOURT SUR ORGE":"55","BONCOURT SUR MEUSE":"55","BOUCONVILLE SUR MADT":"55","BOVIOLLES":"55","BRIEULLES SUR MEUSE":"55","BRIXEY AUX CHANOINES":"55","BUREY LA COTE":"55","CHAMPNEUVILLE":"55","CHAUMONT DEVANT DAMVILLERS":"55","CHAUMONT SUR AIRE":"55","CLERY LE PETIT":"55","CONSENVOYE":"55","DAMLOUP":"55","DAMMARIE SUR SAULX":"55","DIEUE SUR MEUSE":"55","DOMMARY BARONCOURT":"55","DOMPCEVRIN":"55","DOMPIERRE AUX BOIS":"55","DONCOURT AUX TEMPLIERS":"55","DUGNY SUR MEUSE":"55","LES EPARGES":"55","ERIZE LA BRULEE":"55","ETON":"55","FOAMEIX ORNEL":"55","FROMEZEY":"55","GERCOURT ET DRILLANCOURT":"55","GINCREY":"55","GONDRECOURT LE CHATEAU":"55","GOURAINCOURT":"55","HAIRONVILLE":"55","HARVILLE":"55","HAUDAINVILLE":"55","HERBEUVILLE":"55","RUNAN":"22","ST BRANDAN":"22","ST GILDAS":"22","ST GILLES LES BOIS":"22","ST GILLES VIEUX MARCHE":"22","ST GLEN":"22","ST JEAN KERDANIEL":"22","ST JUDOCE":"22","ST MAUDAN":"22","ST MICHEL DE PLELAN":"22","ST QUAY PERROS":"22","ST THELO":"22","ST VRAN":"22","TONQUEDEC":"22","TREBEURDEN":"22","TREBRY":"22","TREFFRIN":"22","TREGLAMUS":"22","TREGON":"22","AHUN":"23","AJAIN":"23","ANZEME":"23","AURIAT":"23","AZAT CHATENET":"23","BETETE":"23","BORD ST GEORGES":"23","BOSMOREAU LES MINES":"23","CHAMBONCHARD":"23","CHAMBORAND":"23","CHAMPSANGLARD":"23","CHARD":"23","CHATELARD":"23","CHATELUS LE MARCHEIX":"23","CHATELUS MALVALEIX":"23","LE COMPAS":"23","CRESSAT":"23","CROZE":"23","DONTREIX":"23","DUN LE PALESTEL":"23","EVAUX LES BAINS":"23","FAUX MAZURAS":"23","GOUZON":"23","GUERET":"23","ISSOUDUN LETRIEIX":"23","JALESCHES":"23","JANAILLAT":"23","JARNAGES":"23","LAVAVEIX LES MINES":"23","MAGNAT L ETRANGE":"23","MOUTIER D AHUN":"23","NEOUX":"23","NOUHANT":"23","NOUZEROLLES":"23","PARSAC RIMONDEIX":"23","POUSSANGES":"23","ROUGNAT":"23","ROYERE DE VASSIVIERE":"23","SANNAT":"23","LA SAUNIERE":"23","ST CHABRAIS":"23","ST FIEL":"23","ST FRION":"23","ST JULIEN LE CHATEL":"23","ST MARIEN":"23","ST MARTIAL LE VIEUX":"23","ST MICHEL DE VEISSE":"23","ST PIERRE BELLEVUE":"23","VIGEVILLE":"23","LA VILLETELLE":"23","ALLAS LES MINES":"24","ARCHIGNAC":"24","AURIAC DU PERIGORD":"24","BADEFOLS D ANS":"24","BEAURONNE":"24","BEAUSSAC":"24","BEZENAC":"24","BOUNIAGUES":"24","BOURGNAC":"24","BOURNIQUEL":"24","BOURROU":"24","BREUILH":"24","CARSAC AILLAC":"24","LA CASSAGNE":"24","CAUSE DE CLERANS":"24","CHAMPAGNAC DE BELAIR":"24","CHAMPAGNE ET FONTAINE":"24","CHASSAIGNES":"24","CHATEAU L EVEQUE":"24","CHERVAL":"24","CORGNAC SUR L ISLE":"24","GRAMOND":"12","LAPANOUSE DE CERNON":"12","LUC LA PRIMAUBE":"12","MALEVILLE":"12","MANHAC":"12","MARCILLAC VALLON":"12","MARTIEL":"12","MAYRAN":"12","MELAGUES":"12","MONTEZIC":"12","MONTJAUX":"12","FONDAMENTE":"12","MORLHON LE HAUT":"12","NAJAC":"12","NAUVIALE":"12","PONT DE SALARS":"12","PRADES D AUBRAC":"12","MOUNES PROHENCOUX":"12","RODEZ":"12","ROQUEFORT SUR SOULZON":"12","LA ROQUE STE MARGUERITE":"12","LA ROUQUETTE":"12","ST AFFRIQUE":"12","ST ANDRE DE NAJAC":"12","ST ANDRE DE VEZINES":"12","ST CHELY D AUBRAC":"12","ST ROME DE TARN":"12","ST SATURNIN DE LENNE":"12","ST SERNIN SUR RANCE":"12","SALMIECH":"12","SAUJAC":"12","SEVERAC D AVEYRON":"12","SOULAGES BONNEVAL":"12","THERONDELS":"12","VALADY":"12","LE VIBAL":"12","VILLECOMTAL":"12","CURAN":"12","AURONS":"13","BEAURECUEIL":"13","CABRIES":"13","CADOLIVE":"13","CARRY LE ROUET":"13","CHATEAUNEUF LE ROUGE":"13","CHATEAURENARD":"13","FOS SUR MER":"13","FUVEAU":"13","ISTRES":"13","JOUQUES":"13","LANCON PROVENCE":"13","MARTIGUES":"13","MAS BLANC DES ALPILLES":"13","MAUSSANE LES ALPILLES":"13","MEYRARGUES":"13","MOLLEGES":"13","ORGON":"13","PEYPIN":"13","PLAN DE CUQUES":"13","LE PUY STE REPARADE":"13","ST CANNAT":"13","ST MARC JAUMEGARDE":"13","ST REMY DE PROVENCE":"13","ST VICTORET":"13","SALON DE PROVENCE":"13","VAUVENARGUES":"13","VENELLES":"13","VENTABREN":"13","MARSEILLE 03":"13","MARSEILLE 06":"13","MARSEILLE 08":"13","ABLON":"14","COLOMBY ANGUERNY":"14","ARGANCHY":"14","AUDRIEU":"14","AUNAY SUR ODON":"14","MALHERBE SUR AJON":"14","BARNEVILLE LA BERTRAN":"14","BARON SUR ODON":"14","BAZENVILLE":"14","BERNIERES SUR MER":"14","BLAINVILLE SUR ORNE":"14","BONNEMAISON":"14","BONNEVILLE SUR TOUQUES":"14","BONNOEIL":"14","BOULON":"14","AUJAN MOURNEDE":"32","AURIMONT":"32","AUX AUSSAT":"32","BERDOUES":"32","BONAS":"32","CASTELNAVET":"32","CASTEX D ARMAGNAC":"32","CASTILLON SAVES":"32","CASTIN":"32","CAUPENNE D ARMAGNAC":"32","CAZAUX VILLECOMTAL":"32","CERAN":"32","CLERMONT SAVES":"32","COURRENSAN":"32","ESCLASSAN LABASTIDE":"32","ESPAS":"32","ESTAMPES":"32","GAUJAN":"32","GAVARRET SUR AULOUSTE":"32","GEE RIVIERE":"32","GIMBREDE":"32","GIMONT":"32","GISCARO":"32","HAGET":"32","LE HOUGA":"32","L ISLE DE NOE":"32","JU BELLOC":"32","LABEJAN":"32","LABRIHE":"32","LADEVEZE RIVIERE":"32","LADEVEZE VILLE":"32","LAGARDERE":"32","LALANNE ARQUE":"32","LAMOTHE GOAS":"32","LANNE SOUBIRAN":"32","LAUJUZAN":"32","LEBOULIN":"32","LECTOURE":"32","LIAS":"32","LIAS D ARMAGNAC":"32","LOUBERSAN":"32","LOURTIES MONBRUN":"32","LOUSSOUS DEBAT":"32","MARSOLAN":"32","MAS D AUVIGNON":"32","MAUMUSSON LAGUIAN":"32","MIRAMONT D ASTARAC":"32","MONBRUN":"32","MONCLAR SUR LOSSE":"32","MONGAUSY":"32","MONLAUR BERNET":"32","MONTIRON":"32","MOUCHES":"32","ORBESSAN":"32","PALLANNE":"32","PESSAN":"32","PIS":"32","PUYSEGUR":"32","ROQUELAURE":"32","ST AVIT FRANDAT":"32","STE CHRISTIE":"32","ST CLAR":"32","ST CRICQ":"32","STE DODE":"32","ST JEAN LE COMTAL":"32","ST JEAN POUTGE":"32","ST LIZIER DU PLANTE":"32","ST MARTIN DE GOYNE":"32","SALLES D ARMAGNAC":"32","SAMATAN":"32","SAVIGNAC MONA":"32","SEMEZIES CACHAN":"32","SEYSSES SAVES":"32","SIMORRE":"32","TAYBOSC":"32","TIRENT PONTEJAC":"32","AUSSOS":"32","ARBANATS":"33","AVENSAN":"33","LE BARP":"33","BAURECH":"33","BELVES DE CASTILLON":"33","BOMMES":"33","MAREUIL SUR CHER":"41","MAZANGE":"41","MER":"41","MONTHOU SUR BIEVRE":"41","LES MONTILS":"41","MONTOIRE SUR LE LOIR":"41","MOREE":"41","NAVEIL":"41","NEUNG SUR BEUVRON":"41","ORCAY":"41","PIERREFITTE SUR SAULDRE":"41","PONTLEVOY":"41","PRUNIERS EN SOLOGNE":"41","LES ROCHES L EVEQUE":"41","ROUGEOU":"41","ST AGIL":"41","ST BOHAIRE":"41","ST FIRMIN DES PRES":"41","ST GEORGES SUR CHER":"41","ST HILAIRE LA GRAVELLE":"41","ST JEAN FROIDMENTEL":"41","ST JULIEN DE CHEDON":"41","ST LEONARD EN BEAUCE":"41","ST SULPICE DE POMMERAY":"41","SAVIGNY SUR BRAYE":"41","SERIS":"41","SUEVRES":"41","THESEE":"41","TOURAILLES":"41","TREHET":"41","VEUVES":"41","VILLEXANTON":"41","VILLIERSFAUX":"41","AMBIERLE":"42","BELMONT DE LA LOIRE":"42","LE BESSAT":"42","BUSSY ALBIEUX":"42","CALOIRE":"42","CELLIEU":"42","CHALAIN LE COMTAL":"42","CHAMBLES":"42","LE CHAMBON FEUGEROLLES":"42","LA CHAMBONIE":"42","CHARLIEU":"42","LE COTEAU":"42","LA COTE EN COUZAN":"42","GRAIX":"42","LA GRAND CROIX":"42","COUSTAUSSA":"11","LA DIGNE D AMONT":"11","DUILHAC SOUS PEYREPERTUSE":"11","ESPEZEL":"11","LA FAJOLLE":"11","FELINES TERMENES":"11","FENDEILLE":"11","FEUILLA":"11","FITOU":"11","LADERN SUR LAUQUET":"11","LANET":"11","LASSERRE DE PROUILLE":"11","LAURE MINERVOIS":"11","LIMOUSIS":"11","MARQUEIN":"11","MAYRONNES":"11","MEZERVILLE":"11","MIREPEISSET":"11","MOLANDIER":"11","MONTFORT SUR BOULZANE":"11","MONZE":"11","NARBONNE":"11","NEBIAS":"11","PALAIRAC":"11","PAYRA SUR L HERS":"11","PEPIEUX":"11","PEYRENS":"11","PIEUSSE":"11","PLAIGNE":"11","RAISSAC SUR LAMPY":"11","RENNES LE CHATEAU":"11","ROQUEFERE":"11","HERMEVILLE EN WOEVRE":"55","HEUDICOURT SOUS LES COTES":"55","IPPECOURT":"55","LES ISLETTES":"55","JOUY EN ARGONNE":"55","JULVECOURT":"55","KOEUR LA GRANDE":"55","LACHAUSSEE":"55","LANEUVILLE AU RUPT":"55","LAVOYE":"55","LIGNY EN BARROIS":"55","LISLE EN BARROIS":"55","LOISON":"55","LONGEVILLE EN BARROIS":"55","LOUVEMONT COTE DU POIVRE":"55","MANGIENNES":"55","MELIGNY LE GRAND":"55","MENIL AUX BOIS":"55","MONT DEVANT SASSEY":"55","MORLEY":"55","MOULAINVILLE":"55","NAIVES EN BLOIS":"55","NEUVILLY EN ARGONNE":"55","NONSARD LAMARCHE":"55","NOUILLONPONT":"55","OLIZY SUR CHIERS":"55","OSCHES":"55","OURCHES SUR MEUSE":"55","PAGNY SUR MEUSE":"55","PAREID":"55","RANZIERES":"55","REGNEVILLE SUR MEUSE":"55","ST LAURENT SUR OTHAIN":"55","SAMPIGNY":"55","SAULMORY ET VILLEFRANCHE":"55","SAUVOY":"55","SENONCOURT LES MAUJOUY":"55","SEUZEY":"55","SILMONT":"55","SOMMEILLES":"55","SOUILLY":"55","THIERVILLE SUR MEUSE":"55","TREVERAY":"55","TRONVILLE EN BARROIS":"55","VARNEVILLE":"55","VAVINCOURT":"55","VERY":"55","VIGNOT":"55","VILLERS DEVANT DUN":"55","VILLERS SOUS PAREID":"55","VOID VACON":"55","WISEPPE":"55","BAUD":"56","BELZ":"56","BIEUZY":"56","BILLIO":"56","LE CROISTY":"56","CRUGUEL":"56","ELVEN":"56","ERDEVEN":"56","LES FOUGERETS":"56","GAVRES":"56","GLENAC":"56","LA GREE ST LAURENT":"56","GROIX":"56","ILE D ARZ":"56","LANESTER":"56","LANGUIDIC":"56","LANOUEE":"56","LANVAUDAN":"56","LANVENEGEN":"56","LAUZACH":"56","LIGNOL":"56","LIMERZEL":"56","LOCMARIA GRAND CHAMP":"56","LORIENT":"56","MALGUENAC":"56","MAURON":"56","MERLEVENEZ":"56","MISSIRIAC":"56","MOLAC":"56","MOREAC":"56","NOYAL MUZILLAC":"56","COULAURES":"24","COULOUNIEIX CHAMIERS":"24","COUX ET BIGAROQUE MOUZENS":"24","CUNEGES":"24","ECHOURGNAC":"24","ETOUARS":"24","EYVIRAT":"24","LES FARGES":"24","FLAUGEAC":"24","LE FLEIX":"24","FOSSEMAGNE":"24","FOUGUEYROLLES":"24","GRANGES D ANS":"24","LES GRAULGES":"24","HAUTEFAYE":"24","LARZAC":"24","LAVEYSSIERE":"24","LIORAC SUR LOUYRE":"24","LOLME":"24","MARNAC":"24","MARSALES":"24","MAYAC":"24","MONMADALES":"24","NABIRAT":"24","NONTRON":"24","ORLIAGUET":"24","PAZAYAC":"24","PEYRILLAC ET MILLAC":"24","PEZULS":"24","POMPORT":"24","PRESSIGNAC VICQ":"24","ST AMAND DE VERGT":"24","ST ANDRE D ALLAS":"24","ST ANDRE DE DOUBLE":"24","ST ANTOINE CUMOND":"24","ST AQUILIN":"24","ST CHAMASSY":"24","ST FELIX DE BOURDEILLES":"24","ST FELIX DE REILLAC ET MORTEMART":"24","ST GENIES":"24","ST JULIEN D EYMET":"24","STE MARIE DE CHIGNAC":"24","ST MARTIAL D ARTENSET":"24","ST MARTIN DE FRESSENGEAS":"24","ST MEDARD DE MUSSIDAN":"24","STE MONDANE":"24","STE ORSE":"24","ST PAUL LIZONNE":"24","ST PIERRE DE COLE":"24","ST PRIVAT DES PRES":"24","ST ROMAIN DE MONPAZIER":"24","ST SAUD LACOUSSIERE":"24","ST SAUVEUR LALANDE":"24","SALAGNAC":"24","SARLANDE":"24","SARLAT LA CANEDA":"24","SAVIGNAC DE MIREMONT":"24","SCEAU ST ANGEL":"24","SIORAC DE RIBERAC":"24","TERRASSON LAVILLEDIEU":"24","THENON":"24","THIVIERS":"24","TOURTOIRAC":"24","TRELISSAC":"24","TURSAC":"24","VALOJOULX":"24","VENDOIRE":"24","VEYRINES DE VERGT":"24","VILLETOUREIX":"24","ABBANS DESSOUS":"25","ABBENANS":"25","ACCOLANS":"25","LES ALLIES":"25","ANTEUIL":"25","AUTECHAUX":"25","LE BARBOUX":"25","BARTHERANS":"25","BELVOIR":"25","BEUTAL":"25","BLARIANS":"25","BLUSSANGEAUX":"25","BULLE":"25","BURNEVILLERS":"25","BOURGEAUVILLE":"14","BRETTEVILLE L ORGUEILLEUSE":"14","BRETTEVILLE SUR DIVES":"14","LE BREVEDENT":"14","BRUCOURT":"14","CARPIQUET":"14","CINTHEAUX":"14","CLARBEC":"14","COLLEVILLE MONTGOMERY":"14","COLOMBIERS SUR SEULLES":"14","CORDEY":"14","COURSEULLES SUR MER":"14","CREPON":"14","CRICQUEVILLE EN BESSIN":"14","DAMBLAINVILLE":"14","DANESTAL":"14","DEAUVILLE":"14","LE DETROIT":"14","DEUX JUMEAUX":"14","DIVES SUR MER":"14","DUCY STE MARGUERITE":"14","ENGLESQUEVILLE EN AUGE":"14","EPRON":"14","EQUEMAUVILLE":"14","ESQUAY NOTRE DAME":"14","ESQUAY SUR SEULLES":"14","FIERVILLE BRAY":"14","FIRFOL":"14","FONTENAY LE MARMION":"14","FRESNE LA MERE":"14","GARCELLES SECQUEVILLE":"14","GLOS":"14","GONNEVILLE SUR MER":"14","GRANDCHAMP LE CHATEAU":"14","GRANGUES":"14","HERMIVAL LES VAUX":"14","HIEVILLE":"14","LA HOUBLONNIERE":"14","ISIGNY SUR MER":"14","LES ISLES BARDEL":"14","JORT":"14","JUVIGNY SUR SEULLES":"14","LAIZE LA VILLE":"14","LANDELLES ET COUPIGNY":"14","MAISONCELLES PELVEY":"14","MAISONCELLES SUR AJON":"14","MANVIEUX":"14","LE MESNIL ROBERT":"14","MEUVAINES":"14","MONTEILLE":"14","MONTVIETTE":"14","MORTEAUX COULIBOEUF":"14","LES MOUTIERS EN AUGE":"14","MUTRECY":"14","NEUILLY LA FORET":"14","NOTRE DAME D ESTREES CORBON":"14","OSMANVILLE":"14","OUEZY":"14","OUILLY LE TESSON":"14","LA POMMERAYE":"14","PONT FARCY":"14","PORT EN BESSIN HUPPAIN":"14","RANCHY":"14","RAPILLY":"14","RUBERCY":"14","RUMESNIL":"14","RYES":"14","ST BENOIT D HEBERTOT":"14","VAL DE VIE":"14","ST GERMAIN LANGOT":"14","STE HONORINE DU FAY":"14","ST JULIEN SUR CALONNE":"14","ST LOUET SUR SEULLES":"14","STE MARGUERITE D ELLE":"14","BOULIAC":"33","BRACH":"33","BUDOS":"33","CABARA":"33","CADAUJAC":"33","CADILLAC":"33","CAMARSAC":"33","CAMPS SUR L ISLE":"33","CANEJAN":"33","CARCANS":"33","CARDAN":"33","CASTELMORON D ALBRET":"33","CASTETS EN DORTHE":"33","CASTRES GIRONDE":"33","CIVRAC EN MEDOC":"33","COIMERES":"33","CREON":"33","CROIGNON":"33","CUBNEZAIS":"33","CUSSAC FORT MEDOC":"33","DARDENAC":"33","DOULEZON":"33","EYSINES":"33","LE FIEU":"33","GABARNAC":"33","GANS":"33","GREZILLAC":"33","LE HAILLAN":"33","ILLATS":"33","JUGAZAN":"33","LA BREDE":"33","LA LANDE DE FRONSAC":"33","LAPOUYADE":"33","LEOGEATS":"33","LES LEVES ET THOUMEYRAGUES":"33","LOUCHATS":"33","LOUPES":"33","LUCMAU":"33","LUDON MEDOC":"33","LUGASSON":"33","MARCILLAC":"33","MARTIGNAS SUR JALLE":"33","MESTERRIEUX":"33","MONPRIMBLANC":"33","MOURENS":"33","NOAILLAN":"33","PETIT PALAIS ET CORNEMPS":"33","POMEROL":"33","POMPEJAC":"33","PORCHERES":"33","LE POUT":"33","PRIGNAC EN MEDOC":"33","PUISSEGUIN":"33","RIMONS":"33","RUCH":"33","ST AUBIN DE BRANNE":"33","ST AVIT DE SOULEGE":"33","ST CAPRAIS DE BORDEAUX":"33","ST CHRISTOLY DE BLAYE":"33","ST CHRISTOPHE DES BARDES":"33","ST CIERS DE CANESSE":"33","ST DENIS DE PILE":"33","ST GERMAIN DU PUCH":"33","ST JULIEN BEYCHEVELLE":"33","ST LAURENT MEDOC":"33","ST LAURENT D ARCE":"33","ST LAURENT DU PLAN":"33","ST LOUBES":"33","ST MARTIN DE LERM":"33","ST MARTIN DE SESCAS":"33","ST MICHEL DE LAPUJADE":"33","ST MORILLON":"33","ST PHILIPPE DU SEIGNAL":"33","ST PIERRE D AURILLAC":"33","ST SULPICE DE FALEYRENS":"33","ST SULPICE DE GUILLERAGUES":"33","ST SULPICE DE POMMIERS":"33","ST SULPICE ET CAMEYRAC":"33","ST VIVIEN DE MEDOC":"33","ST YZANS DE MEDOC":"33","SALLEBOEUF":"33","LES SALLES DE CASTILLON":"33","SAUTERNES":"33","TABANAC":"33","TEUILLAC":"33","ROQUETAILLADE":"11","ROUBIA":"11","ROUVENAC":"11","ST JEAN DE BARROU":"11","ST JEAN DE PARACOL":"11","ST PIERRE DES CHAMPS":"11","STE VALIERE":"11","SALLES D AUDE":"11","SEIGNALENS":"11","SOUILHANELS":"11","TALAIRAN":"11","LA TOURETTE CABARDES":"11","TOURREILLES":"11","TRAUSSE":"11","TUCHAN":"11","VALMIGERE":"11","VERZEILLE":"11","VILLALIER":"11","AGEN D AVEYRON":"12","AURIAC LAGAST":"12","AUZITS":"12","BROUSSE LE CHATEAU":"12","CAMPOURIEZ":"12","CAMPUAC":"12","BARAQUEVILLE":"12","CASTELMARY":"12","LE CAYROL":"12","COMPOLIBAT":"12","COMPREGNAC":"12","COUPIAC":"12","DRUELLE":"12","LE FEL":"12","ESPEYRAC":"12","FLAVIN":"12","FLORENTIN LA CAPELLE":"12","LA FOUILLADE":"12","GOUTRENS":"12","LANUEJOULS":"12","LESCURE JAOUL":"12","LIVINHAC LE HAUT":"12","MARNHAGUES ET LATOUR":"12","MARTRIN":"12","MONTFRANC":"12","MONTSALES":"12","MOSTUEJOULS":"12","MOURET":"12","NAUSSAC":"12","OLEMPS":"12","PEYRELEAU":"12","PEYRUSSE LE ROC":"12","POMAYROLS":"12","POUSTHOMY":"12","PRADES SALARS":"12","PREVINQUIERES":"12","REQUISTA":"12","ST AMANS DES COTS":"12","ST BEAUZELY":"12","ST CHRISTOPHE VALLON":"12","ST COME D OLT":"12","STE JULIETTE SUR VIAUR":"12","ST SEVER DU MOUSTIER":"12","ST SYMPHORIEN DE THENIERES":"12","ST VICTOR ET MELVIEU":"12","SAUVETERRE DE ROUERGUE":"12","SEGUR":"12","SENERGUES":"12","TAURIAC DE CAMARES":"12","TAURIAC DE NAUCELLE":"12","TREMOUILLES":"12","VALZERGUES":"12","VILLEFRANCHE DE PANAT":"12","ALLEINS":"13","AUBAGNE":"13","LA BARBEN":"13","BELCODENE":"13","LA BOUILLADISSE":"13","PEAULE":"56","PEILLAC":"56","PLAUDREN":"56","PLEUCADEUC":"56","PLEUGRIFFET":"56","PLOUHARNEL":"56","PLUHERLIN":"56","PLUMELEC":"56","QUIBERON":"56","ST GERAND":"56","ST GILDAS DE RHUYS":"56","ST GUYOMARD":"56","ST JACUT LES PINS":"56","ST MALO DE BEIGNON":"56","SERENT":"56","LE SOURN":"56","SULNIAC":"56","TAUPONT":"56","LE TOUR DU PARC":"56","TREAL":"56","ACHAIN":"57","ALTRIPPE":"57","ANZELING":"57","ARRAINCOURT":"57","ARS LAQUENEXY":"57","AY SUR MOSELLE":"57","BAERENTHAL":"57","BERTRANGE":"57","BERVILLER EN MOSELLE":"57","BETTELAINVILLE":"57","BEYREN LES SIERCK":"57","BIBICHE":"57","BICKENHOLTZ":"57","BIDING":"57","BIONCOURT":"57","BIONVILLE SUR NIED":"57","BISTROFF":"57","BOUCHEPORN":"57","BOULANGE":"57","BOULAY MOSELLE":"57","BOURDONNAY":"57","BOUSBACH":"57","BROUCK":"57","BUDING":"57","CARLING":"57","CHAILLY LES ENNERY":"57","CHAMBREY":"57","CHATEL ST GERMAIN":"57","CLOUANGE":"57","COLMEN":"57","CREUTZWALD":"57","DIANE CAPELLE":"57","DIEBLING":"57","EBLANGE":"57","ENCHENBERG":"57","ERCHING":"57","ETZLING":"57","FLEISHEIM":"57","FLEVY":"57","FOVILLE":"57","FRESNES EN SAULNOIS":"57","GOETZENBRUCK":"57","GOMELANGE":"57","GROSTENQUIN":"57","GRUNDVILLER":"57","GUENVILLER":"57","GUESSLING HEMERING":"57","HALSTROFF":"57","HARAUCOURT SUR SEILLE":"57","HASELBOURG":"57","HAVANGE":"57","HAZEMBOURG":"57","HELLIMER":"57","HELSTROFF":"57","HEMILLY":"57","HENRIDORFF":"57","HESTROFF":"57","HINCKANGE":"57","HOMMERT":"57","CERNAY L EGLISE":"25","CHAMPLIVE":"25","CHATEAUVIEUX LES FOSSES":"25","CHEMAUDIN":"25","CHEVROZ":"25","CLERVAL":"25","CORCELLES FERRIERES":"25","CORCONDRAY":"25","COURTETAIN ET SALANS":"25","CUSE ET ADRISANS":"25","DAMBELIN":"25","DOUBS":"25","DURNES":"25","ECHAY":"25","ECHENANS":"25","ETALANS":"25","FAIMBE":"25","FESCHES LE CHATEL":"25","FESSEVILLERS":"25","FONTAINE LES CLERVAL":"25","LES FONTENELLES":"25","FOURG":"25","FOURNET BLANCHEROCHE":"25","FUANS":"25","GONDENANS MONTBY":"25","GOUX LES DAMBELIN":"25","GOUX LES USIERS":"25","GRANDFONTAINE SUR CREUSE":"25","GUYANS DURNES":"25","GUYANS VENNES":"25","HYEMONDANS":"25","INDEVILLERS":"25","LABERGEMENT DU NAVOIS":"25","LAIRE":"25","LAVIRON":"25","LIEBVILLERS":"25","LODS":"25","LONGEMAISON":"25","LUXIOL":"25","MALBUISSON":"25","MALPAS":"25","MAZEROLLES LE SALIN":"25","MEDIERE":"25","MEREY VIEILLEY":"25","MISEREY SALINES":"25","MONT DE VOUGNEY":"25","MONTENOIS":"25","MORTEAU":"25","MOUTHE":"25","LE MOUTHEROT":"25","NAISEY LES GRANGES":"25","NOEL CERNEUX":"25","ORVE":"25","OUVANS":"25","OYE ET PALLET":"25","PALANTINE":"25","PALISE":"25","PLACEY":"25","POINTVILLERS":"25","POUILLEY FRANCAIS":"25","POUILLEY LES VIGNES":"25","LA PRETIERE":"25","RANCENAY":"25","REUGNEY":"25","RILLANS":"25","ROCHEJEAN":"25","ROCHES LES BLAMONT":"25","ROSUREUX":"25","ROUGEMONTOT":"25","ROUHE":"25","RUREY":"25","ST JULIEN LES MONTBELIARD":"25","SEPTFONTAINES":"25","SERRE LES SAPINS":"25","SILLEY AMANCEY":"25","SURMONT":"25","THIEBOUHANS":"25","UZELLE":"25","VALDAHON":"25","VALONNE":"25","VANDONCOURT":"25","VAUDRIVILLERS":"25","VAUX LES PRES":"25","VELLEROT LES BELVOIR":"25","VENNES":"25","VERNIERFONTAINE":"25","VILLARS ST GEORGES":"25","ST MARTIN DE FONTENAY":"14","ST MARTIN DE LA LIEUE":"14","ST MARTIN DE MAILLOC":"14","ST PIERRE CANIVET":"14","ST PIERRE SUR DIVES":"14","ST VAAST EN AUGE":"14","SALLEN":"14","SUBLES":"14","THIEVILLE":"14","TORTEVAL QUESNAY":"14","TOURNIERES":"14","TOURVILLE SUR ODON":"14","TROUVILLE SUR MER":"14","USSY":"14","VENDES":"14","VENDEUVRE":"14","VER SUR MER":"14","LA VESPIERE FRIARDEL":"14","VIERVILLE SUR MER":"14","VOUILLY":"14","ANGLARDS DE ST FLOUR":"15","AUZERS":"15","ALBEPIERRE BREDONS":"15","CALVINET":"15","CEZENS":"15","CHARMENSAC":"15","CHEYLADE":"15","COREN":"15","CROS DE MONTVERT":"15","CROS DE RONESQUE":"15","FONTANGES":"15","GIOU DE MAMOU":"15","JOURSAC":"15","JUSSAC":"15","LAFEUILLADE EN VEZIE":"15","LANOBRE":"15","LAROQUEVIEILLE":"15","LAVIGERIE":"15","LEYNHAC":"15","LUGARDE":"15","MASSIAC":"15","MENTIERES":"15","MONTCHAMP":"15","MOUSSAGES":"15","NIEUDAN":"15","PAULHENC":"15","PLEAUX":"15","ST BONNET DE CONDAT":"15","ST CIRGUES DE JORDANNE":"15","ST ETIENNE DE CARLAT":"15","ST ETIENNE DE CHOMEIL":"15","ST GERONS":"15","ST PAUL DE SALERS":"15","ST REMY DE CHAUDES AIGUES":"15","ST SANTIN CANTALES":"15","ST SANTIN DE MAURS":"15","ST VINCENT DE SALERS":"15","SANSAC VEINAZES":"15","SEGUR LES VILLAS":"15","SENEZERGUES":"15","LE TRIOULOU":"15","VALJOUZE":"15","VALUEJOLS":"15","YDES":"15","AIGRE":"16","ANGEDUC":"16","TIZAC DE LAPOUYADE":"33","VENSAC":"33","VERTHEUIL":"33","AGDE":"34","AGEL":"34","ALIGNAN DU VENT":"34","ASSIGNAN":"34","BASSAN":"34","BELARGA":"34","BESSAN":"34","BOISSERON":"34","BOUJAN SUR LIBRON":"34","LE BOUSQUET D ORB":"34","BOUZIGUES":"34","CAMPAGNAN":"34","CASTELNAU DE GUERS":"34","CASTRIES":"34","CAZOULS LES BEZIERS":"34","CEBAZAN":"34","COMBAILLAUX":"34","COMBES":"34","CREISSAN":"34","CRUZY":"34","DIO ET VALQUIERES":"34","FERRIERES LES VERRERIES":"34","FONTES":"34","FOUZILHON":"34","HEREPIAN":"34","JONCELS":"34","LAURENS":"34","LIEURAN LES BEZIERS":"34","MARSILLARGUES":"34","MAUREILHAN":"34","MIREVAL":"34","MONTFERRIER SUR LEZ":"34","MONTOULIERS":"34","MURLES":"34","OLMET ET VILLECUN":"34","PARDAILHAN":"34","PEGAIROLLES DE BUEGES":"34","PERET":"34","PLAISSAN":"34","LE POUJOL SUR ORB":"34","PREMIAN":"34","PUIMISSON":"34","QUARANTE":"34","RESTINCLIERES":"34","RIEUSSEC":"34","STE CROIX DE QUINTILLARGUES":"34","ST FELIX DE L HERAS":"34","ST GENIES DE VARENSAL":"34","ST GEORGES D ORQUES":"34","ST PONS DE THOMIERES":"34","ST THIBERY":"34","ST VINCENT DE BARBEYRARGUES":"34","ST VINCENT D OLARGUES":"34","SAUSSAN":"34","SAUSSINES":"34","LE SOULIE":"34","TEYRAN":"34","THEZAN LES BEZIERS":"34","VAILHAN":"34","VAILHAUQUES":"34","VELIEUX":"34","VENDEMIAN":"34","VILLENEUVETTE":"34","VILLETELLE":"34","ARGENTRE DU PLESSIS":"35","BAGUER MORVAN":"35","BALAZE":"35","BECHEREL":"35","BOURGBARRE":"35","BRIELLES":"35","CARDROC":"35","LA CHAPELLE BOUEXIC":"35","LA CHAPELLE DES FOUGERETZ":"35","LA CHAPELLE DE BRAIN":"35","CHARTRES DE BRETAGNE":"35","CHATEAUNEUF D ILLE ET VILAINE":"35","CHATILLON EN VENDELAIS":"35","CHATEAUNEUF LES MARTIGUES":"13","LA CIOTAT":"13","CORNILLON CONFOUX":"13","CUGES LES PINS":"13","EGUILLES":"13","EYGALIERES":"13","LA FARE LES OLIVIERS":"13","FONTVIEILLE":"13","MIRAMAS":"13","NOVES":"13","PELISSANNE":"13","LA PENNE SUR HUVEAUNE":"13","ROGNAC":"13","ROQUEVAIRE":"13","SAINTES MARIES DE LA MER":"13","LE THOLONET":"13","COUDOUX":"13","MARSEILLE 15":"13","AMAYE SUR SEULLES":"14","AMFREVILLE":"14","ANCTOVILLE":"14","ASNIERES EN BESSIN":"14","LES AUTHIEUX SUR CALONNE":"14","BALLEROY SUR DROME":"14","BAROU EN AUGE":"14","BAVENT":"14","BERNESQ":"14","BIEVILLE BEUVILLE":"14","BEUVRON EN AUGE":"14","BLAY":"14","LE BO":"14","BONNEVILLE LA LOUVET":"14","BOUGY":"14","BREMOY":"14","BRETTEVILLE LE RABET":"14","LE BREUIL EN AUGE":"14","LE BREUIL EN BESSIN":"14","BREVILLE LES MONTS":"14","BRICQUEVILLE":"14","CAEN":"14","CAHAGNES":"14","LA CAINE":"14","CAIRON":"14","CAMPAGNOLLES":"14","CESNY AUX VIGNES":"14","CHOUAIN":"14","CLECY":"14","COLOMBIERES":"14","COMBRAY":"14","CORDEBUGLE":"14","COURTONNE LES DEUX EGLISES":"14","CRESSEVEUILLE":"14","CRICQUEBOEUF":"14","CRISTOT":"14","CROISSANVILLE":"14","CULEY LE PATRY":"14","DONNAY":"14","DOUVILLE EN AUGE":"14","DOUVRES LA DELIVRANDE":"14","ELLON":"14","EMIEVILLE":"14","ESCOVILLE":"14","ESPINS":"14","ETREHAM":"14","FLEURY SUR ORNE":"14","LA FOLIE":"14","FORMIGNY":"14","LE FOURNET":"14","FUMICHON":"14","GONNEVILLE SUR HONFLEUR":"14","GONNEVILLE EN AUGE":"14","HEROUVILLE ST CLAIR":"14","HOSTE":"57","INSMING":"57","JUVELIZE":"57","KERPRICH AUX BOIS":"57","KLANG":"57","KOENIGSMACKER":"57","LAFRIMBOLLE":"57","LANGUIMBERG":"57","LAUDREFANG":"57","LAUNSTROFF":"57","LEYVILLER":"57","LIXING LES ROUHLING":"57","LORRY MARDIGNY":"57","LUTTANGE":"57","LUTZELBOURG":"57","MAIZIERES LES VIC":"57","MARANGE ZONDRANGE":"57","MARIEULLES":"57","MECLEUVES":"57","MEGANGE":"57","METAIRIES ST QUIRIN":"57","MONTIGNY LES METZ":"57","NEUNKIRCHEN LES BOUZONVILLE":"57","NIEDERSTINZEL":"57","NIEDERVISSE":"57","NOUILLY":"57","OBERVISSE":"57","OBRECK":"57","ORIOCOURT":"57","ORNY":"57","ORON":"57","PANGE":"57","PETIT TENQUIN":"57","PETITE ROSSELLE":"57","PETTONCOURT":"57","PHILIPPSBOURG":"57","PLESNOIS":"57","PUTTELANGE LES THIONVILLE":"57","REMELFING":"57","REMELING":"57","REMERING":"57","RIMLING":"57","RORBACH LES DIEUZE":"57","ROZERIEULLES":"57","ST HUBERT":"57","ST JEAN ROHRBACH":"57","SARRALBE":"57","SCHALBACH":"57","SCY CHAZELLES":"57","SECOURT":"57","SEMECOURT":"57","SOTZELING":"57","STURZELBRONN":"57","SUISSE":"57","TETERCHEN":"57","THEDING":"57","TORCHEVILLE":"57","TROMBORN":"57","TURQUESTEIN BLANCRUPT":"57","UCKANGE":"57","VALMESTROFF":"57","VANY":"57","VATIMONT":"57","VECKERSVILLER":"57","VERNY":"57","VILLERS SUR NIED":"57","VIONVILLE":"57","VITRY SUR ORNE":"57","VOLSTROFF":"57","VRY":"57","XANREY":"57","XOCOURT":"57","YUTZ":"57","ACHUN":"58","ANTHIEN":"58","ARLEUF":"58","ARMES":"58","BEAUMONT SARDOLLES":"58","BICHES":"58","BONA":"58","CHALLUY":"58","CHAMPALLEMENT":"58","CHASNAY":"58","CHATEAU CHINON VILLE":"58","CHEVENON":"58","VILLARS SOUS DAMPJOUX":"25","VILLARS SOUS ECOT":"25","VILLE DU PONT":"25","ALBON":"26","ALIXAN":"26","ANDANCETTE":"26","ANNEYRON":"26","ARTHEMONAY":"26","AUBRES":"26","BALLONS":"26","BEAUMONT EN DIOIS":"26","BEAUVALLON":"26","BELLEGARDE EN DIOIS":"26","BENIVAY OLLON":"26","BOUCHET":"26","BOURDEAUX":"26","BOUVIERES":"26","BREN":"26","CHARPEY":"26","CHATEAUNEUF SUR ISERE":"26","CHATILLON EN DIOIS":"26","COLONZELLE":"26","CORNILLAC":"26","DONZERE":"26","EPINOUZE":"26","EROME":"26","EURRE":"26","EYGALAYES":"26","EYGLUY ESCOULIN":"26","FELINES SUR RIMANDOULE":"26","GRANE":"26","LES GRANGES GONTARDES":"26","LARNAGE":"26","LA LAUPIE":"26","LIVRON SUR DROME":"26","MIRMANDE":"26","MONTAULIEU":"26","MONTELIMAR":"26","MONTMEYRAN":"26","MONTSEGUR SUR LAUZON":"26","MORAS EN VALLOIRE":"26","NYONS":"26","ORIOL EN ROYANS":"26","PLAN DE BAIX":"26","LE POET LAVAL":"26","PONSAS":"26","PORTES LES VALENCE":"26","PRADELLE":"26","LES PRES":"26","REMUZAT":"26","ROCHEFORT SAMSON":"26","LA ROCHE SUR GRANE":"26","ROYNAC":"26","SAHUNE":"26","ST AUBAN SUR L OUVEZE":"26","ST BARDOUX":"26","ST JEAN EN ROYANS":"26","ST MARTIN EN VERCORS":"26","ST NAZAIRE EN ROYANS":"26","ST PANTALEON LES VIGNES":"26","ST ROMAN":"26","SOUSPIERRE":"26","SUZE":"26","TAIN L HERMITAGE":"26","TERSANNE":"26","TULETTE":"26","VALAURIE":"26","VASSIEUX EN VERCORS":"26","VILLEFRANCHE LE CHATEAU":"26","VOLVENT":"26","JAILLANS":"26","ST VINCENT LA COMMANDERIE":"26","ACLOU":"27","AMBENAY":"27","ARNIERES SUR ITON":"27","AVIRON":"27","BARC":"27","BARQUET":"27","BEAUFICEL EN LYONS":"27","LE BEC HELLOUIN":"27","BERENGEVILLE LA CAMPAGNE":"27","BERNIENVILLE":"27","BERVILLE EN ROUMOIS":"27","BEUZEVILLE":"27","BEZU LA FORET":"27","ASNIERES SUR NOUERE":"16","AUBETERRE SUR DRONNE":"16","BAIGNES STE RADEGONDE":"16","BARDENAC":"16","BASSAC":"16","BLANZAGUET ST CYBARD":"16","BRILLAC":"16","CHAMPMILLON":"16","CHASSORS":"16","CHATIGNAC":"16","CHERVES CHATELARS":"16","LA CHEVRERIE":"16","CONDEON":"16","COULGENS":"16","L HOPITAL SOUS ROCHEFORT":"42","L HORME":"42","JAS":"42","JURE":"42","LENTIGNY":"42","LERIGNEUX":"42","MAIZILLY":"42","MAROLS":"42","MERLE LEIGNEC":"42","MONTARCHER":"42","MONTCHAL":"42","MONTROND LES BAINS":"42","NANDAX":"42","NOAILLY":"42","NOLLIEUX":"42","PERREUX":"42","PRALONG":"42","LA RICAMARIE":"42","RIVE DE GIER":"42","ROCHE LA MOLIERE":"42","ST ANDRE D APCHON":"42","ST CYR LES VIGNES":"42","ST DENIS DE CABANNE":"42","ST HILAIRE CUSSON LA VALMITTE":"42","ST JEAN LA VETRE":"42","ST JULIEN LA VETRE":"42","ST JUST LA PENDUE":"42","ST LEGER SUR ROANNE":"42","ST MARTIN LA SAUVETE":"42","ST MARTIN LESTRA":"42","ST PRIEST EN JAREZ":"42","ST ROMAIN LES ATHEUX":"42","ST SYMPHORIEN DE LAY":"42","ST THURIN":"42","SALT EN DONZY":"42","LA TALAUDIERE":"42","TARTARAS":"42","TRELINS":"42","USSON EN FOREZ":"42","LA VALLA EN GIER":"42","VIRIGNEUX":"42","VIVANS":"42","ALLEGRE":"43","ARLEMPDES":"43","BEAUNE SUR ARZON":"43","BEAUX":"43","BEAUZAC":"43","BESSAMOREL":"43","LA BESSEYRE ST MARY":"43","BLAVOZY":"43","CEAUX D ALLEGRE":"43","CEYSSAC":"43","CHADRAC":"43","LA CHAPELLE D AUREC":"43","LA CHAPELLE GENESTE":"43","CHASPUZAC":"43","CHILHAC":"43","LES ESTABLES":"43","FRUGIERES LE PIN":"43","LAPTE":"43","LAVOUTE CHILHAC":"43","LE MONASTIER SUR GAZEILLE":"43","MONLET":"43","LE PERTUIS":"43","QUEYRIERES":"43","ST ANDRE DE CHALENCON":"43","ST ARCONS D ALLIER":"43","ST AUSTREMOINE":"43","CHELUN":"35","CHEVAIGNE":"35","COESMES":"35","DINARD":"35","DROUGES":"35","ERBREE":"35","LA FRESNAIS":"35","LA GUERCHE DE BRETAGNE":"35","LES IFFS":"35","JANZE":"35","LALLEU":"35","LANGAN":"35","LANHELIN":"35","LANRIGAN":"35","LONGAULNAY":"35","LOURMAIS":"35","MEILLAC":"35","MEZIERES SUR COUESNON":"35","MINIAC SOUS BECHEREL":"35","LE MINIHIC SUR RANCE":"35","MONT DOL":"35","MONTERFIL":"35","MONTREUIL SUR ILLE":"35","MOUSSE":"35","LA NOE BLANCHE":"35","PRINCE":"35","RENAC":"35","ROMILLE":"35","ST BRICE EN COGLES":"35","ST GEORGES DE CHESNE":"35","ST JEAN SUR VILAINE":"35","ST JOUAN DES GUERETS":"35","ST LUNAIRE":"35","ST PERAN":"35","ST PERN":"35","LA SELLE EN LUITRE":"35","LA SELLE GUERCHAISE":"35","SENS DE BRETAGNE":"35","SOUGEAL":"35","TREFFENDEL":"35","VENDEL":"35","VERGEAL":"35","ANJOUIN":"36","ARGY":"36","BOMMIERS":"36","LA BUXERETTE":"36","CELON":"36","CHABRIS":"36","LA CHAPELLE ORTHEMALE":"36","CHATEAUROUX":"36","CHAVIN":"36","CUZION":"36","DEOLS":"36","FRANCILLON":"36","FREDILLE":"36","ISSOUDUN":"36","LACS":"36","LEVROUX":"36","LUCAY LE MALE":"36","MARTIZAY":"36","MAUVIERES":"36","MERS SUR INDRE":"36","LA MOTTE FEUILLY":"36","MOUHERS":"36","NEONS SUR CREUSE":"36","NOHANT VIC":"36","ORSENNES":"36","OULCHES":"36","LE POINCONNET":"36","PRISSAC":"36","ST BENOIT DU SAULT":"36","ST CHRISTOPHE EN BAZELLE":"36","ST CHRISTOPHE EN BOUCHERIE":"36","ST CYRAN DU JAMBOT":"36","STE FAUSTE":"36","ST GEORGES SUR ARNON":"36","ST PIERRE DE LAMPS":"36","ST VALENTIN":"36","SEMBLECAY":"36","TENDU":"36","THEVET ST JULIEN":"36","TOURNON ST MARTIN":"36","VENDOEUVRES":"36","LA VERNELLE":"36","HEROUVILLETTE":"14","HEULAND":"14","LA HOGUETTE":"14","HOULGATE":"14","IFS":"14","JUAYE MONDAYE":"14","LANDES SUR AJON":"14","LEAUPARTIE":"14","LOUVAGNY":"14","MAGNY LA CAMPAGNE":"14","MALTOT":"14","MANNEVILLE LA PIPARD":"14","MARTIGNY SUR L ANTE":"14","MATHIEU":"14","LE MESNIL BENOIST":"14","LE MESNIL CAUSSOIS":"14","LE MESNIL EUDES":"14","MITTOIS":"14","MONDRAINVILLE":"14","MONTFIQUET":"14","MONTREUIL EN AUGE":"14","NORREY EN AUGE":"14","OUILLY LE VICOMTE":"14","OUISTREHAM":"14","PENNEDEPIE":"14","PERCY EN AUGE":"14","PERIERS SUR LE DAN":"14","PIERREFITTE EN AUGE":"14","PIERREFITTE EN CINGLAIS":"14","PLUMETOT":"14","PONTECOULANT":"14","POUSSY LA CAMPAGNE":"14","PREAUX BOCAGE":"14","PUTOT EN AUGE":"14","PUTOT EN BESSIN":"14","BIEVILLE QUETIEVILLE":"14","REVIERS":"14","ROCQUANCOURT":"14","LA ROQUE BAIGNARD":"14","ST AIGNAN DE CRAMESNIL":"14","SEULLINE":"14","ST GEORGES EN AUGE":"14","ST GERMAIN DU PERT":"14","STE MARIE OUTRE L EAU":"14","ST MARTIN DES ENTREES":"14","ST OUEN DU MESNIL OGER":"14","ST PAUL DU VERNAY":"14","ST PIERRE DU FRESNE":"14","ST PIERRE DU JONQUET":"14","ST VAAST SUR SEULLES":"14","ST VIGOR LE GRAND":"14","SAON":"14","SASSY":"14","SOUMONT ST QUENTIN":"14","TESSEL":"14","TILLY LA CAMPAGNE":"14","TOUQUES":"14","TOURNEBU":"14","TREPREL":"14","TROIS MONTS":"14","TRUNGY":"14","VACOGNES NEUILLY":"14","VAUCELLES":"14","VAUX SUR SEULLES":"14","VERSON":"14","VICTOT PONTFOL":"14","VIENNE EN BESSIN":"14","VIGNATS":"14","VILLERS CANIVET":"14","LA VILLETTE":"14","VILLY LEZ FALAISE":"14","VIMONT":"14","BREZONS":"15","CAYROLS":"15","CELOUX":"15","CHALVIGNAC":"15","CHAMPS SUR TARENTAINE MARCHAL":"15","CHASTEL SUR MURAT":"15","COLLANDRES":"15","CIEZ":"58","CIZELY":"58","DOMPIERRE SUR NIEVRE":"58","DORNECY":"58","EPIRY":"58","GERMIGNY SUR LOIRE":"58","GIMOUILLE":"58","GRENOIS":"58","LAMENAY SUR LOIRE":"58","LAROCHEMILLAY":"58","LORMES":"58","LUTHENAY UXELOUP":"58","LUZY":"58","MAGNY LORMES":"58","LA MARCHE":"58","MARIGNY L EGLISE":"58","MARIGNY SUR YONNE":"58","MAUX":"58","MONTAMBERT":"58","MONTENOISON":"58","MONT ET MARRE":"58","MONTIGNY AUX AMOGNES":"58","MOURON SUR YONNE":"58","NEUFFONTAINES":"58","NEUVY SUR LOIRE":"58","NUARS":"58","OUROUX EN MORVAN":"58","PAZY":"58","PERROY":"58","POIL":"58","PREPORCHE":"58","SAINCAIZE MEAUCE":"58","ST ANDRE EN MORVAN":"58","ST GRATIEN SAVIGNY":"58","ST PEREUSE":"58","SAIZY":"58","SARDY LES EPIRY":"58","SAVIGNY POIL FOL":"58","ST LOUP GEANGES":"71","ST MARTIN EN GATINOIS":"71","ST MAURICE DES CHAMPS":"71","ST MICAUD":"71","ST POINT":"71","ST ROMAIN SOUS VERSIGNY":"71","ST SYMPHORIEN DE MARMAGNE":"71","ST USUGE":"71","ST VINCENT EN BRESSE":"71","SALORNAY SUR GUYE":"71","SAMPIGNY LES MARANGES":"71","SANCE":"71","SEMUR EN BRIONNAIS":"71","SENOZAN":"71","SERCY":"71","SERLEY":"71","SERMESSE":"71","SUIN":"71","THUREY":"71","UXEAU":"71","VAUBAN":"71","VAUDEBARRIER":"71","VERSAUGUES":"71","VIRE":"71","VITRY SUR LOIRE":"71","FLEURVILLE":"71","AILLIERES BEAUVOIR":"72","AMNE":"72","ASSE LE BOISNE":"72","LES AULNEAUX":"72","BALLON ST MARS":"72","BEILLE":"72","BERUS":"72","BOULOIRE":"72","BOURG LE ROI":"72","LA CHAPELLE AUX CHOUX":"72","LA CHAPELLE DU BOIS":"72","LA CHARTRE SUR LE LOIR":"72","CHEMIRE EN CHARNIE":"72","CHEMIRE LE GAUDIN":"72","CHERREAU":"72","COULANS SUR GEE":"72","COURGENARD":"72","CRE SUR LOIR":"72","CRISSE":"72","DEGRE":"72","DEHAULT":"72","DOLLON":"72","BOIS NORMAND PRES LYRE":"27","BONNEVILLE APTOT":"27","BOSQUENTIN":"27","BOSROBERT":"27","LE BOULAY MORIN":"27","BRESTOT":"27","BROGLIE":"27","CAILLY SUR EURE":"27","CARSIX":"27","CESSEVILLE":"27","CHAIGNES":"27","DAUBEUF LA CAMPAGNE":"27","EPREVILLE PRES LE NEUBOURG":"27","ETREVILLE":"27","FARCEAUX":"27","FAVEROLLES LA CAMPAGNE":"27","LA FERRIERE SUR RISLE":"27","LA FORET DU PARC":"27","GISORS":"27","GIVERVILLE":"27","GUICHAINVILLE":"27","HECMANVILLE":"27","HENNEZIS":"27","LES HOGUES":"27","L HOSMES":"27","HOULBEC PRES LE GROS THEIL":"27","JUIGNETTES":"27","LE LANDIN":"27","LOUVERSEY":"27","MALLEVILLE SUR LE BEC":"27","MALOUY":"27","MARBEUF":"27","MARTOT":"27","MENNEVAL":"27","LE MESNIL HARDRAY":"27","MESNIL SUR L ESTREE":"27","MESNIL VERCLIVES":"27","MEZIERES EN VEXIN":"27","MONTAURE":"27","LA NOE POULAIN":"27","ORVAUX":"27","PERRIERS LA CAMPAGNE":"27","PERRIERS SUR ANDELLE":"27","PINTERVILLE":"27","PLASNES":"27","PRESSAGNY L ORGUEILLEUX":"27","ROMILLY LA PUTHENAYE":"27","ROMILLY SUR ANDELLE":"27","ROUGEMONTIERS":"27","SACQUENVILLE":"27","ST ANDRE DE L EURE":"27","ST AUBIN DU THENNEY":"27","ST AUBIN LE VERTUEUX":"27","ST CHRISTOPHE SUR CONDE":"27","STE COLOMBE LA COMMANDERIE":"27","ST DENIS D AUGERONS":"27","ST ETIENNE DU VAUVRAY":"27","STE GENEVIEVE LES GASNY":"27","ST GERMAIN DE FRESNEY":"27","ST GERMAIN LA CAMPAGNE":"27","ST GERMAIN SUR AVRE":"27","ST JULIEN DE LA LIEGUE":"27","ST LEGER DU GENNETEY":"27","ST LUC":"27","ST MARDS DE BLACARVILLE":"27","COURSAN":"11","CUXAC CABARDES":"11","DAVEJEAN":"11","DERNACUEILLETTE":"11","ESCUEILLENS ET ST JUST":"11","ESPERAZA":"11","FERRALS LES CORBIERES":"11","FONTERS DU RAZES":"11","GAJA ET VILLEDIEU":"11","GINCLA":"11","GRAMAZIE":"11","HOUNOUX":"11","ST DIDIER EN VELAY":"43","ST ETIENNE LARDEYROL":"43","ST GERON":"43","ST ILPIZE":"43","ST JEURES":"43","ST JUST MALMONT":"43","SENEUJOLS":"43","TIRANGES":"43","VALS PRES LE PUY":"43","VARENNES ST HONORAT":"43","ANCENIS":"44","ASSERAC":"44","BASSE GOULAINE":"44","BONNOEUVRE":"44","CASSON":"44","CONQUEREUIL":"44","COUFFE":"44","FERCE":"44","FROSSAY":"44","LAVAU SUR LOIRE":"44","LE LOROUX BOTTEREAU":"44","MAISDON SUR SEVRE":"44","MESANGER":"44","MOUAIS":"44","LES MOUTIERS EN RETZ":"44","MOUZILLON":"44","OUDON":"44","PETIT AUVERNE":"44","PRINQUIAU":"44","ST FIACRE SUR MAINE":"44","ST GILDAS DES BOIS":"44","ST HERBLAIN":"44","ST LUMINE DE CLISSON":"44","ST MOLF":"44","ST NICOLAS DE REDON":"44","ST PERE EN RETZ":"44","ST SEBASTIEN SUR LOIRE":"44","ST VIAUD":"44","SAUTRON":"44","SEVERAC":"44","LES TOUCHES":"44","TRANS SUR ERDRE":"44","VAY":"44","ASCOUX":"45","AUTRUY SUR JUINE":"45","AUVILLIERS EN GATINAIS":"45","LE BARDON":"45","BATILLY EN PUISAYE":"45","BEAUNE LA ROLANDE":"45","BOESSES":"45","BOULAY LES BARRES":"45","BOYNES":"45","BRIARE":"45","BUCY ST LIPHARD":"45","LA CHAPELLE ONZERAIN":"45","CHAPELON":"45","CHATILLON SUR LOIRE":"45","CHECY":"45","CLERY ST ANDRE":"45","COINCES":"45","COMBREUX":"45","CONFLANS SUR LOING":"45","CORQUILLEROY":"45","COURCY AUX LOGES":"45","DESMONTS":"45","DIMANCHEVILLE":"45","LA FERTE ST AUBIN":"45","FREVILLE DU GATINAIS":"45","GERMIGNY DES PRES":"45","GIDY":"45","HUETRE":"45","HUISSEAU SUR MAUVES":"45","INTVILLE LA GUETARD":"45","LION EN SULLIAS":"45","VICQ SUR NAHON":"36","VILLEGOUIN":"36","ABILLY":"37","AMBILLOU":"37","AMBOISE":"37","ANTOGNY LE TILLAC":"37","AUZOUER EN TOURAINE":"37","BEAULIEU LES LOCHES":"37","BEAUMONT EN VERON":"37","BEAUMONT VILLAGE":"37","BLERE":"37","LE BOULAY":"37","BOURGUEIL":"37","BRECHES":"37","BUEIL EN TOURAINE":"37","CERELLES":"37","CHARENTILLY":"37","CHATEAU RENAULT":"37","CHOUZE SUR LOIRE":"37","CIGOGNE":"37","CIVRAY SUR ESVES":"37","COUESMES":"37","CRAVANT LES COTEAUX":"37","DRACHE":"37","DRUYE":"37","EPEIGNE SUR DEME":"37","ESVES LE MOUTIER":"37","GENILLE":"37","LE GRAND PRESSIGNY":"37","LE LIEGE":"37","LIGRE":"37","LIMERAY":"37","LOUANS":"37","LOUESTAULT":"37","MARRAY":"37","MONNAIE":"37","MONTREUIL EN TOURAINE":"37","NEUILLY LE BRIGNON":"37","PERRUSSON":"37","LE PETIT PRESSIGNY":"37","PREUILLY SUR CLAISE":"37","RAZINES":"37","RILLE":"37","LA ROCHE CLERMAULT":"37","STE CATHERINE DE FIERBOIS":"37","ST EPAIN":"37","ST JEAN ST GERMAIN":"37","ST LAURENT EN GATINES":"37","ST PATRICE":"37","ST QUENTIN SUR INDROIS":"37","SAVIGNE SUR LATHAN":"37","SAVONNIERES":"37","SENNEVIERES":"37","SONZAY":"37","VILLEDOMAIN":"37","YZEURES SUR CREUSE":"37","LES ABRETS EN DAUPHINE":"38","ARZAY":"38","ASSIEU":"38","AUBERIVES EN ROYANS":"38","AUBERIVES SUR VAREZE":"38","BALBINS":"38","LA BALME LES GROTTES":"38","LA BATIE MONTGASCON":"38","BEAUFIN":"38","BERNIN":"38","BESSINS":"38","BOURGOIN JALLIEU":"38","CHABONS":"38","CHANTELOUVE":"38","CHAPAREILLAN":"38","LA CHAPELLE DE LA TOUR":"38","CHARNECLES":"38","CHASSIGNIEU":"38","CLELLES":"38","COGNIN LES GORGES":"38","COLOMBE":"38","CORBELIN":"38","LES COTES D AREY":"38","LES COTES DE CORPS":"38","COUBLEVIE":"38","CREMIEU":"38","DOISSIN":"38","FERRIERES ST MARY":"15","GIRGOLS":"15","GLENAT":"15","GOURDIEGES":"15","JUNHAC":"15","LACAPELLE BARRES":"15","LADINHAC":"15","LAVEISSIERE":"15","MANDAILLES ST JULIEN":"15","MARCOLES":"15","MAURS":"15","MEALLET":"15","MOLEDES":"15","OMPS":"15","PAILHEROLS":"15","POLMINHAC":"15","PRADIERS":"15","RIOM ES MONTAGNES":"15","ST MARY LE PLAIN":"15","ST PAUL DES LANDES":"15","SANSAC DE MARMIESSE":"15","SERIERS":"15","SOURNIAC":"15","VELZIC":"15","VEZE":"15","VIEILLESPESSE":"15","LE VIGEAN":"15","AIGNES ET PUYPEROUX":"16","AMBERNAC":"16","ANGEAC CHAMPAGNE":"16","ANGEAC CHARENTE":"16","AUNAC":"16","AUSSAC VADALLE":"16","BARBEZIEUX ST HILAIRE":"16","BECHERESSE":"16","CHABRAC":"16","CHADURIE":"16","CHALLIGNAC":"16","CHAMPAGNE VIGNY":"16","CHARRAS":"16","CHATEAUNEUF SUR CHARENTE":"16","CHILLAC":"16","COURGEAC":"16","COUTURE":"16","CURAC":"16","DEVIAT":"16","EDON":"16","ETRIAC":"16","GENSAC LA PALLUE":"16","GENTE":"16","LES GOURS":"16","GOURVILLE":"16","GUIZENGEARD":"16","JARNAC":"16","LONGRE":"16","MAINZAC":"16","MANSLE":"16","MARTHON":"16","MAZIERES":"16","MONTEMBOEUF":"16","MONTROLLET":"16","MOUTON":"16","NABINAUD":"16","NANCLARS":"16","NANTEUIL EN VALLEE":"16","NERSAC":"16","NONAC":"16","PRANZAC":"16","PUYMOYEN":"16","RAIX":"16","ROULLET ST ESTEPHE":"16","ST AMANT DE MONTMOREAU":"16","ST AMANT DE BOIXE":"16","ST AMANT DE NOUERE":"16","DOMFRONT EN CHAMPAGNE":"72","DUNEAU":"72","DUREIL":"72","ECOMMOY":"72","EPINEU LE CHEVREUIL":"72","LA FONTAINE ST MARTIN":"72","FYE":"72","GESNES LE GANDELIN":"72","LE GRAND LUCE":"72","GREEZ SUR ROC":"72","JAUZE":"72","LAVARE":"72","LOMBRON":"72","LOUPLANDE":"72","LUCE SOUS BALLON":"72","LUCHE PRINGE":"72","LE MANS":"72","MANSIGNE":"72","MONCE EN SAOSNOIS":"72","MONTAILLE":"72","MULSANNE":"72","NEUFCHATEL EN SAOSNOIS":"72","NUILLE LE JALAIS":"72","PARCE SUR SARTHE":"72","PARENNES":"72","PEZE LE ROBERT":"72","POILLE SUR VEGRE":"72","PONCE SUR LE LOIR":"72","PONTVALLAIN":"72","PRUILLE L EGUILLE":"72","LA QUINTE":"72","REQUEIL":"72","RUILLE SUR LOIR":"72","SABLE SUR SARTHE":"72","ST CALEZ EN SAOSNOIS":"72","ST GEORGES DE LA COUEE":"72","ST GEORGES LE GAULTIER":"72","ST GERMAIN SUR SARTHE":"72","ST JEAN DU BOIS":"72","STE OSMANE":"72","ST PAUL LE GAULTIER":"72","ST REMY DE SILLE":"72","STE SABINE SUR LONGEVE":"72","SEGRIE":"72","SOULITRE":"72","TELOCHE":"72","TERREHAULT":"72","THOIRE SUR DINAN":"72","TUFFE VAL DE LA CHERONNE":"72","VALENNES":"72","VALLON SUR GEE":"72","VANCE":"72","VERNIE":"72","VILLAINES LA GONAIS":"72","VOIVRES LES LE MANS":"72","YVRE LE POLIN":"72","ARBIN":"73","ATTIGNAT ONCIN":"73","BARBERAZ":"73","BONNEVAL SUR ARC":"73","BOZEL":"73","LA CHAMBRE":"73","CHAMOUX SUR GELON":"73","CHAMPAGNY EN VANOISE":"73","COGNIN":"73","CONJUX":"73","DETRIER":"73","DOMESSIN":"73","ECOLE":"73","FRETERIVE":"73","LA GIETTAZ":"73","GILLY SUR ISERE":"73","HERMILLON":"73","LAISSAUD":"73","LANDRY":"73","LANSLEVILLARD":"73","LEPIN LE LAC":"73","MONTAIMONT":"73","MONTRICHER ALBANNE":"73","LA MOTTE EN BAUGES":"73","LAURABUC":"11","LESPINASSIERE":"11","LEUC":"11","LOUPIA":"11","LA LOUVIERE LAURAGAIS":"11","MALRAS":"11","MARSEILLETTE":"11","LES MARTYS":"11","MAS CABARDES":"11","MAS SAINTES PUELLES":"11","MISSEGRE":"11","MOLLEVILLE":"11","MONTAZELS":"11","MONTSERET":"11","MOUSSOULENS":"11","PAULIGNE":"11","PECH LUNA":"11","PEYREFITTE DU RAZES":"11","LA POMAREDE":"11","POMY":"11","POUZOLS MINERVOIS":"11","QUINTILLAN":"11","QUIRBAJOU":"11","RODOME":"11","ROQUEFORT DES CORBIERES":"11","ROUFFIAC D AUDE":"11","ROUFFIAC DES CORBIERES":"11","RUSTIQUES":"11","ST ANDRE DE ROQUELONGUE":"11","STE COLOMBE SUR L HERS":"11","ST JULIA DE BEC":"11","ST JUST ET LE BEZU":"11","ST MICHEL DE LANES":"11","ST PAULET":"11","ST POLYCARPE":"11","SALLELES D AUDE":"11","SALSIGNE":"11","LA SERPENT":"11","SERVIES EN VAL":"11","SOUILHE":"11","SOULATGE":"11","THEZAN DES CORBIERES":"11","TOURNISSAN":"11","VERDUN EN LAURAGAIS":"11","VIGNEVIEILLE":"11","VILLARDONNEL":"11","VILLAR EN VAL":"11","VILLASAVARY":"11","VILLEDAIGNE":"11","VILLEDUBERT":"11","VILLEGAILHENC":"11","VILLEGLY":"11","VILLELONGUE D AUDE":"11","LES ALBRES":"12","AMBEYRAC":"12","ASPRIERES":"12","BALAGUIER SUR RANCE":"12","LA BASTIDE PRADINES":"12","CAMARES":"12","CAMJAC":"12","CANTOIN":"12","CAPDENAC GARE":"12","LA CAPELLE BLEYS":"12","CASSUEJOULS":"12","LA CAVALERIE":"12","CENTRES":"12","LES COSTES GOZON":"12","DECAZEVILLE":"12","ESCANDOLIERES":"12","LACROIX BARREZ":"12","LAGUIOLE":"12","LAISSAC SEVERAC L EGLISE":"12","LASSOUTS":"12","LUNAC":"12","LE MONASTERE":"12","MONTBAZENS":"12","MOYRAZES":"12","LE NAYRAC":"12","OLS ET RINHODES":"12","PEUX ET COUFFOULEUX":"12","MAREAU AUX BOIS":"45","MARIGNY LES USAGES":"45","MONTBOUY":"45","MORMANT SUR VERNISSON":"45","NESPLOY":"45","OUSSOY EN GATINAIS":"45","OUVROUER LES CHAMPS":"45","PANNECIERES":"45","PATAY":"45","POILLY LEZ GIEN":"45","PRESSIGNY LES PINS":"45","PUISEAUX":"45","REBRECHIEN":"45","ROUVRES ST JEAN":"45","RUAN":"45","ST AIGNAN LE JAILLARD":"45","ST AY":"45","ST FIRMIN SUR LOIRE":"45","ST MARTIN D ABBAT":"45","SEICHEBRIERES":"45","LA SELLE EN HERMOY":"45","SERMAISES":"45","SOLTERRE":"45","SULLY LA CHAPELLE":"45","SURY AUX BOIS":"45","TIVERNON":"45","TOURNOISIS":"45","TRINAY":"45","VENNECY":"45","VIEILLES MAISONS SUR JOUDRY":"45","VILLEMOUTIERS":"45","ANGLARS":"46","BACH":"46","BAGAT EN QUERCY":"46","BELMONT BRETENOUX":"46","BETAILLE":"46","BRETENOUX":"46","BRENGUES":"46","CAILLAC":"46","CAMBAYRAC":"46","CAPDENAC":"46","CARAYAC":"46","CARDAILLAC":"46","CIEURAC":"46","CRAYSSAC":"46","DOUELLE":"46","DURAVEL":"46","ESPERE":"46","ESTAL":"46","FLAUJAC POUJOLS":"46","FLORESSAS":"46","GIGOUZAC":"46","GINTRAC":"46","LABASTIDE DU VERT":"46","LAGARDELLE":"46","LHOSPITALET":"46","LISSAC ET MOURET":"46","LIVERNON":"46","LOUBRESSAC":"46","LUZECH":"46","MAYRINHAC LENTOUR":"46","MEYRONNE":"46","MILHAC":"46","MONTAMEL":"46","LE MONTAT":"46","MONTCLERA":"46","ORNIAC":"46","PADIRAC":"46","PRAYSSAC":"46","PUYJOURDES":"46","LES QUATRE ROUTES DU LOT":"46","REYREVIGNES":"46","ROCAMADOUR":"46","ST BRESSOU":"46","ST JEAN MIRABEL":"46","ST MEDARD DE PRESQUE":"46","ST MICHEL DE BANNIERES":"46","THEDIRAC":"46","THEGRA":"46","THEMINES":"46","TRESPOUX RASSIELS":"46","VALROUFIE":"46","VAYLATS":"46","VAYRAC":"46","VIDAILLAC":"46","ESTRABLIN":"38","EYDOCHE":"38","FONTANIL CORNILLON":"38","HERBEYS":"38","LANS EN VERCORS":"38","LUZINAY":"38","MALLEVAL EN VERCORS":"38","MARCIEU":"38","MASSIEU":"38","MOISSIEU SUR DOLON":"38","MONESTIER D AMBEL":"38","MONESTIER DE CLERMONT":"38","MONSTEROUX MILIEU":"38","MONTALIEU VERCIEU":"38","MONTCHABOUD":"38","SERRE NERPOL":"38","ORIS EN RATTIER":"38","OYEU":"38","PACT":"38","PANISSAGE":"38","PENOL":"38","PLAN":"38","LE PONT DE CLAIX":"38","PONT EVEQUE":"38","RUY":"38","ST AGNIN SUR BION":"38","ST ALBAN DU RHONE":"38","STE ANNE SUR GERVONDE":"38","ST BARTHELEMY DE SECHILIENNE":"38","ST CHRISTOPHE SUR GUIERS":"38","ST EGREVE":"38","ST ETIENNE DE ST GEOIRS":"38","ST GEOIRS":"38","ST HILAIRE DE BRENS":"38","ST ISMIER":"38","ST JEAN DE MOIRANS":"38","ST JEAN DE SOUDAIN":"38","ST JOSEPH DE RIVIERE":"38","ST JUST DE CLAIX":"38","ST MARCELLIN":"38","STE MARIE D ALLOIX":"38","ST MARTIN D URIAGE":"38","ST MICHEL DE ST GEOIRS":"38","ST NIZIER DU MOUCHEROTTE":"38","ST PAUL D IZEAUX":"38","ST PIERRE DE CHARTREUSE":"38","ST PIERRE DE CHERENNES":"38","ST QUENTIN FALLAVIER":"38","ST ROMAIN DE SURIEU":"38","ST SIMEON DE BRESSIEUX":"38","ST SORLIN DE MORESTEL":"38","ST VICTOR DE CESSIEU":"38","SALAGNON":"38","LA SALETTE FALLAVAUX":"38","SARDIEU":"38","SEYSSUEL":"38","SONNAY":"38","SOUSVILLE":"38","TORCHEFELON":"38","LE TOUVET":"38","VARACIEUX":"38","VARCES ALLIERES ET RISSET":"38","VERTRIEU":"38","VILLARD DE LANS":"38","VOIRON":"38","AMANGE":"39","ANDELOT EN MONTAGNE":"39","AUDELANGE":"39","BARRETAINE":"39","BIARNE":"39","BIEFMORIN":"39","BOURG DE SIROD":"39","BRACON":"39","ST CIERS SUR BONNIEURE":"16","ST GENIS D HIERSAC":"16","AUGE ST MEDARD":"16","ST PALAIS DU NE":"16","ST QUENTIN DE CHALAIS":"16","STE SEVERE":"16","ST SEVERIN":"16","STE SOULINE":"16","ST SULPICE DE COGNAC":"16","TOUVRE":"16","VOEUIL ET GIGET":"16","VOUHARTE":"16","ANTEZANT LA CHAPELLE":"17","ARCES":"17","ARDILLIERES":"17","ARS EN RE":"17","ARVERT":"17","AVY":"17","BALLANS":"17","BARZAN":"17","BEAUGEAY":"17","BERCLOUX":"17","BERNAY ST MARTIN":"17","BIGNAY":"17","BORESSE ET MARTRON":"17","BREUIL LA REORTE":"17","BRIE SOUS MATHA":"17","BURIE":"17","BUSSAC FORET":"17","CHADENAC":"17","CHAMOUILLAC":"17","CHAMPDOLENT":"17","CHANIERS":"17","CHANTEMERLE SUR LA SOIE":"17","LA CHAPELLE DES POTS":"17","CHATELAILLON PLAGE":"17","LA COUARDE SUR MER":"17","COURCERAC":"17","COURPIGNAC":"17","CRAMCHABAN":"17","DAMPIERRE SUR BOUTONNE":"17","DOMPIERRE SUR CHARENTE":"17","LA FLOTTE":"17","LE FOUILLOUX":"17","LES GONDS":"17","GRANDJEAN":"17","LA GRIPPERIE ST SYMPHORIEN":"17","LE GUE D ALLERE":"17","LA JARRIE":"17","LA JARRIE AUDOUIN":"17","JONZAC":"17","JUICQ":"17","LA LAIGNE":"17","LEOVILLE":"17","LOIRE LES MARAIS":"17","LORIGNAC":"17","LOULAY":"17","LOUZIGNAC":"17","MATHA":"17","MEDIS":"17","MESCHERS SUR GIRONDE":"17","MEURSAC":"17","MOEZE":"17","MORAGNE":"17","MURON":"17","NANTILLE":"17","NEUILLAC":"17","NEUVICQ":"17","NEUVICQ LE CHATEAU":"17","LES NOUILLERS":"17","POMMIERS MOULONS":"17","POUILLAC":"17","REAUX SUR TREFLE":"17","ROMAZIERES":"17","ROMEGOUX":"17","ROYAN":"17","SABLONCEAUX":"17","ST GEORGES DE DIDONNE":"17","PEISEY NANCROIX":"73","RUFFIEUX":"73","ST BALDOPH":"73","ST BERON":"73","ST CASSIN":"73","ST FRANC":"73","ST FRANCOIS DE SALES":"73","ST GENIX SUR GUIERS":"73","ST JEAN DE MAURIENNE":"73","STE MARIE D ALVEY":"73","LES BELLEVILLE":"73","ST MARTIN SUR LA CHAMBRE":"73","ST PIERRE D ALVEY":"73","ST PIERRE DE CURTILLE":"73","TERMIGNON":"73","THENESOL":"73","TIGNES":"73","TOURNON":"73","VERTHEMEX":"73","VIVIERS DU LAC":"73","ALLEVES":"74","LE BOUCHET":"74","CHAMPANGES":"74","LA CHAPELLE D ABONDANCE":"74","LA CHAPELLE ST MAURICE":"74","CHAPEIRY":"74","CHAVANNAZ":"74","CHENEX":"74","CHEVENOZ":"74","LES CLEFS":"74","LA CLUSAZ":"74","CLUSES":"74","COLLONGES SOUS SALEVE":"74","CONTAMINE SUR ARVE":"74","LA COTE D ARBROZ":"74","CRAN GEVRIER":"74","DEMI QUARTIER":"74","ESSERT ROMAND":"74","FAUCIGNY":"74","FILLINGES":"74","FRANGY":"74","HERY SUR ALBY":"74","JONZIER EPAGNY":"74","LARRINGES":"74","LYAUD":"74","VAL DE CHAISE":"74","MARLIOZ":"74","MEILLERIE":"74","MENTHONNEX EN BORNES":"74","MONTAGNY LES LANCHES":"74","MONTRIOND":"74","NEUVECELLE":"74","PRAZ SUR ARLY":"74","PUBLIER":"74","ST GERMAIN SUR RHONE":"74","SALES":"74","SALLENOVES":"74","SAXEL":"74","SEYTROUX":"74","THORENS GLIERES":"74","VALLEIRY":"74","LA VERNAZ":"74","VINZIER":"74","VOVRAY EN BORNES":"74","PARIS 11":"75","PARIS 12":"75","PARIS 15":"75","AMFREVILLE LA MI VOIE":"76","ANCOURT":"76","ANGERVILLE BAILLEUL":"76","ANGERVILLE LA MARTEL":"76","ANGERVILLE L ORCHER":"76","ANNOUVILLE VILMESNIL":"76","ARQUES LA BATAILLE":"76","AUBEGUIMONT":"76","RIEUPEYROUX":"12","RIVIERE SUR TARN":"12","RODELLE":"12","ST GENIEZ D OLT ET D AUBRAC":"12","ST IZAIRE":"12","ST JEAN D ALCAPIES":"12","ST JEAN ET ST PAUL":"12","ST LAURENT D OLT":"12","ST LEONS":"12","ST MARTIN DE LENNE":"12","ST ROME DE CERNON":"12","ST SANTIN":"12","CAUSSE ET DIEGE":"12","TOULONJAC":"12","VABRES L ABBAYE":"12","VAILHOURLES":"12","VAUREILLES":"12","VERSOLS ET LAPEYRE":"12","VIALA DU TARN":"12","ALLAUCH":"13","AUREILLE":"13","AURIOL":"13","LES BAUX DE PROVENCE":"13","CASSIS":"13","LA DESTROUSSE":"13","LAMBESC":"13","MAILLANE":"13","MOURIES":"13","LES PENNES MIRABEAU":"13","PEYROLLES EN PROVENCE":"13","PLAN D ORGON":"13","PORT DE BOUC":"13","PUYLOUBIER":"13","ROGNES":"13","ROGNONAS":"13","ST CHAMAS":"13","ST MITRE LES REMPARTS":"13","TARASCON":"13","VERNEGUES":"13","VERQUIERES":"13","CARNOUX EN PROVENCE":"13","MARSEILLE 01":"13","MARSEILLE 07":"13","MARSEILLE 16":"13","ANGOVILLE":"14","ANISY":"14","ARROMANCHES LES BAINS":"14","ASNELLES":"14","AUVILLARS":"14","BANVILLE":"14","BEAUMAIS":"14","BENERVILLE SUR MER":"14","BENY SUR MER":"14","BERNIERES D AILLY":"14","BLONVILLE SUR MER":"14","BONNEBOSQ":"14","BRETTEVILLE SUR ODON":"14","BROUAY":"14","CABOURG":"14","CAHAGNOLLES":"14","CAMPANDRE VALCONGRAIN":"14","ANZEX":"47","ARMILLAC":"47","AURIAC SUR DROPT":"47","BAZENS":"47","BOURGOUGNAGUE":"47","BOURRAN":"47","BUZET SUR BAISE":"47","CASTILLONNES":"47","CLERMONT DESSOUS":"47","CLERMONT SOUBIRAN":"47","COURBIAC":"47","COUTHURES SUR GARONNE":"47","DAUSSE":"47","DEVILLAC":"47","DOUDRAC":"47","FAUGUEROLLES":"47","FEUGAROLLES":"47","HAUTEFAGE LA TOUR":"47","LABASTIDE CASTEL AMOUROUX":"47","MADAILLAN":"47","MARCELLUS":"47","MASQUIERES":"47","MASSELS":"47","MAUVEZIN SUR GUPIE":"47","MONFLANQUIN":"47","MONTPOUILLAN":"47","NICOLE":"47","PUYMIROL":"47","RAYET":"47","ST ETIENNE DE FOUGERES":"47","ST ETIENNE DE VILLEREAL":"47","ST EUTROPE DE BORN":"47","ST FRONT SUR LEMANCE":"47","ST JEAN DE THURAC":"47","STE LIVRADE SUR LOT":"47","ST MAURICE DE LESTAPEL":"47","ST PARDOUX DU BREUIL":"47","ST PASTOUR":"47","ST SALVY":"47","SAUVETERRE ST DENIS":"47","SAVIGNAC SUR LEYZE":"47","SENESTIS":"47","SERIGNAC SUR GARONNE":"47","TOURNON D AGENAIS":"47","VARES":"47","VILLETON":"47","XAINTRAILLES":"47","ARZENC DE RANDON":"48","BRENOUX":"48","CHADENET":"48","CHASSERADES":"48","FOURNELS":"48","LACHAMP":"48","LES LAUBIES":"48","LA MALENE":"48","LE MALZIEU FORAIN":"48","MEYRUEIS":"48","NASBINALS":"48","PALHERS":"48","PELOUSE":"48","LE POMPIDOU":"48","PRINSUEJOLS":"48","RIBENNES":"48","ROUSSES":"48","ST ANDRE DE LANCIZE":"48","ST BONNET DE CHIRAC":"48","STE CROIX VALLEE FRANCAISE":"48","ST ETIENNE VALLEE FRANCAISE":"48","ST FREZAL D ALBUGES":"48","ST GERMAIN DE CALBERTE":"48","ST LEGER DE PEYRE":"48","ST PAUL LE FROID":"48","ST PIERRE DES TRIPIERS":"48","LES VIGNES":"48","BRIOD":"39","CENSEAU":"39","CESANCEY":"39","LES CHALESMES":"39","CHAMPAGNOLE":"39","CHARCHILLA":"39","CHARENCY":"39","CHATEAU CHALON":"39","LE CHATELEY":"39","CHAUX DES CROTENAY":"39","LA CHAUX EN BRESSE":"39","CHEMENOT":"39","CHEVROTAINE":"39","CHOUX":"39","COSGES":"39","COURBETTE":"39","COUSANCE":"39","COYRIERE":"39","CRENANS":"39","DARBONNAY":"39","DENEZIERES":"39","DESNES":"39","ECLEUX":"39","EVANS":"39","LA FAVIERE":"39","FETIGNY":"39","FONCINE LE BAS":"39","FRAISANS":"39","FRASNE LES MEULIERES":"39","LE FRASNOIS":"39","GENOD":"39","LAINS":"39","LE LARDERET":"39","LARRIVOIRE":"39","LONGWY SUR LE DOUBS":"39","MACORNAY":"39","MALANGE":"39","MALLEREY":"39","MARIGNA SUR VALOUSE":"39","MESSIA SUR SORNE":"39","MOLAMBOZ":"39","MOLINGES":"39","LES MOLUNES":"39","MONTHOLIER":"39","MONT SUR MONNET":"39","LES MOUSSIERES":"39","MOUTOUX":"39","NEY":"39","ORCHAMPS":"39","OUGNEY":"39","PICARREAU":"39","PLAISIA":"39","PONT DE POITTE":"39","QUINTIGNY":"39","RUFFEY SUR SEILLE":"39","ST AMOUR":"39","SAMPANS":"39","SOUCIA":"39","TASSENIERES":"39","THERVAY":"39","THESY":"39","THOISSIA":"39","TRENAL":"39","VAUDREY":"39","VERCIA":"39","VERIA":"39","VERS SOUS SELLIERES":"39","VILLARDS D HERIA":"39","VILLARD SUR BIENNE":"39","VILLERS ROBERT":"39","VOITEUR":"39","ST GEORGES DES AGOUTS":"17","ST GERMAIN DE VIBRAC":"17","ST GREGOIRE D ARDENNES":"17","ST JEAN DE LIVERSAY":"17","ST LAURENT DE LA BARRIERE":"17","ST LAURENT DE LA PREE":"17","ST MARTIAL SUR NE":"17","ST MARTIN DE COUX":"17","ST MARTIN DE JUILLERS":"17","ST MARTIN DE RE":"17","STE MEME":"17","ST OUEN LA THENE":"17","ST PARDOULT":"17","ST PIERRE DE JUILLERS":"17","ST PIERRE D OLERON":"17","ST PORCHAIRE":"17","ST SAUVEUR D AUNIS":"17","ST VAIZE":"17","THAIRE":"17","TONNAY CHARENTE":"17","TORXE":"17","TUGERAS ST MAURICE":"17","VILLEDOUX":"17","ALLOGNY":"18","ANNOIX":"18","ARCOMPS":"18","AZY":"18","BESSAIS LE FROMENTAL":"18","BOUZAIS":"18","BRUERE ALLICHAMPS":"18","LA CHAPELLE D ANGILLON":"18","CHARENTONNAY":"18","LE CHATELET":"18","CLEMONT":"18","COUARGUES":"18","COURS LES BARRES":"18","DAMPIERRE EN GRACAY":"18","LA GROUTTE":"18","IVOY LE PRE":"18","JARS":"18","JOUET SUR L AUBOIS":"18","LAZENAY":"18","MASSAY":"18","MEILLANT":"18","MENETOU SALON":"18","MENETREOL SOUS SANCERRE":"18","MERY SUR CHER":"18","MORNAY BERRY":"18","NEUILLY EN DUN":"18","NEUILLY EN SANCERRE":"18","PIGNY":"18","PLAIMPIED GIVAUDINS":"18","PRECY":"18","PRESLY":"18","QUANTILLY":"18","ST FLORENT SUR CHER":"18","ST GERMAIN DU PUY":"18","ST HILAIRE DE GONDILLY":"18","ST HILAIRE EN LIGNIERES":"18","SANCERRE":"18","SAVIGNY EN SEPTAINE":"18","SOULANGIS":"18","TENDRON":"18","TOUCHAY":"18","VEAUGUES":"18","VERNAIS":"18","VIGNOUX SOUS LES AIX":"18","VILLEQUIERS":"18","VOUZERON":"18","BRIVEZAC":"19","BUGEAT":"19","CHAMBERET":"19","CHAMEYRAT":"19","LA CHAPELLE AUX SAINTS":"19","CHAPELLE SPINASSE":"19","CLERGOUX":"19","COMBRESSOL":"19","CORREZE":"19","DAMPNIAT":"19","DAVIGNAC":"19","L EGLISE AUX BOIS":"19","ESPARTIGNAC":"19","EYREIN":"19","GUMOND":"19","AUZOUVILLE SUR RY":"76","AUZOUVILLE SUR SAANE":"76","BAILLOLET":"76","BAONS LE COMTE":"76","BEAUBEC LA ROSIERE":"76","BEAUSSAULT":"76","BEC DE MORTAGNE":"76","BENARVILLE":"76","BERVILLE SUR SEINE":"76","BIVILLE LA BAIGNARDE":"76","BIVILLE LA RIVIERE":"76","BLACQUEVILLE":"76","BONSECOURS":"76","BOIS D ENNEBOURG":"76","BOIS HEROULT":"76","BOIS HIMONT":"76","BORDEAUX ST CLAIR":"76","BOSVILLE":"76","LE BOURG DUN":"76","BRACHY":"76","BURES EN BRAY":"76","CANEHAN":"76","CARVILLE LA FOLLETIERE":"76","CARVILLE POT DE FER":"76","LE CATELIER":"76","CATENAY":"76","CAUVILLE SUR MER":"76","LA CHAPELLE DU BOURGAY":"76","LA CHAPELLE SUR DUN":"76","CLAIS":"76","CRIQUETOT LE MAUCONDUIT":"76","CRIQUETOT SUR LONGUEVILLE":"76","CROIXDALLE":"76","DAMPIERRE EN BRAY":"76","DARNETAL":"76","DOUVREND":"76","DUCLAIR":"76","ECRETTEVILLE LES BAONS":"76","ELETOT":"76","ENVRONVILLE":"76","EPINAY SUR DUCLAIR":"76","EPRETOT":"76","ESLETTES":"76","EU":"76","FESQUES":"76","FONGUEUSEMARE":"76","GANZEVILLE":"76","GERVILLE":"76","GONNEVILLE SUR SCIE":"76","GRAINVILLE SUR RY":"76","LE GRAND QUEVILLY":"76","GRAVAL":"76","GRUGNY":"76","GRUMESNIL":"76","LA HALLOTIERE":"76","HARCANVILLE":"76","HARFLEUR":"76","HAUDRICOURT":"76","HAUTOT LE VATOIS":"76","HERMEVILLE":"76","HEUGLEVILLE SUR SCIE":"76","HODENG HODENGER":"76","LE HOULME":"76","ILLOIS":"76","LESTANVILLE":"76","LINTOT":"76","LONGUEIL":"76","MALAUNAY":"76","MARTAINVILLE EPREVILLE":"76","MELAMARE":"76","MESANGUEVILLE":"76","MESNIL MAUGER":"76","MESNIL RAOUL":"76","LE MESNIL SOUS JUMIEGES":"76","MONCHY SUR EU":"76","MONTIVILLIERS":"76","MONTVILLE":"76","MORGNY LA POMMERAYE":"76","NOTRE DAME D ALIERMONT":"76","OSMOY ST VALERY":"76","POMMEREVAL":"76","PUISENVAL":"76","QUIBERVILLE":"76","QUIEVRECOURT":"76","REBETS":"76","CARCAGNY":"14","CARTIGNY L EPINAY":"14","CHEUX":"14","COLLEVILLE SUR MER":"14","COLOMBELLES":"14","CORMELLES LE ROYAL":"14","COSSESSEVILLE":"14","COUDRAY RABUT":"14","COURSON":"14","COURVAUDON":"14","CREVECOEUR EN AUGE":"14","CUSSY":"14","ECRAMMEVILLE":"14","EPINAY SUR ODON":"14","ESTREES LA CAMPAGNE":"14","FAUGUERNON":"14","FEUGUEROLLES BULLY":"14","LA FOLLETIERE ABENON":"14","FOULOGNES":"14","FOURCHES":"14","FRENOUVILLE":"14","LE FRESNE CAMILLY":"14","GAVRUS":"14","GRIMBOSQ":"14","HERMANVILLE SUR MER":"14","L HOTELLERIE":"14","LANGRUNE SUR MER":"14","LECAUDE":"14","LION SUR MER":"14","LISIEUX":"14","LISON":"14","LOUCELLES":"14","MAIZET":"14","LE MESNIL MAUGER":"14","MONTS EN BESSIN":"14","MOUEN":"14","NOROLLES":"14","NORON L ABBAYE":"14","OLENDON":"14","ONDEFONTAINE":"14","ORBEC":"14","OUFFIERES":"14","OUILLY DU HOULEY":"14","PARFOURU SUR ODON":"14","PERRIERES":"14","PERTHEVILLE NERS":"14","PLANQUERY":"14","LE PLESSIS GRIMOULT":"14","POTIGNY":"14","QUETTEVILLE":"14","LA RIVIERE ST SAUVEUR":"14","ST ANDRE D HEBERTOT":"14","ST COME DE FRESNE":"14","ST ETIENNE LA THILLAYE":"14","ST GERMAIN DE LIVET":"14","ST GERMAIN LE VASSON":"14","ST LAURENT DU MONT":"14","ST LAURENT SUR MER":"14","ST LOUP HORS":"14","ST OUEN LE PIN":"14","ST PAIR":"14","ST PIERRE AZIF":"14","ST SEVER CALVADOS":"14","ST VIGOR DES MEZERETS":"14","SANNERVILLE":"14","SAONNET":"14","SURRAIN":"14","THAON":"14","LE THEIL EN AUGE":"14","TIERCEVILLE":"14","TRACY BOCAGE":"14","LA VACQUERIE":"14","VARAVILLE":"14","VAUDELOGES":"14","VAUX SUR AURE":"14","ANTOIGNE":"49","ARTANNES SUR THOUET":"49","AUBIGNE SUR LAYON":"49","ST CYR SUR LOIRE":"37","ST MARTIN LE BEAU":"37","ST ROCH":"37","SAVIGNY EN VERON":"37","SAZILLY":"37","SEUILLY":"37","SOUVIGNY DE TOURAINE":"37","THENEUIL":"37","TRUYES":"37","VALLERES":"37","VERNEUIL LE CHATEAU":"37","VERNEUIL SUR INDRE":"37","VILLANDRY":"37","L ALBENC":"38","ANTHON":"38","ARANDON":"38","AURIS":"38","BEAUVOIR DE MARC":"38","BELLEGARDE POUSSIEU":"38","BLANDIN":"38","BOUVESSE QUIRIEU":"38","BRESSON":"38","LA BUISSE":"38","CHAMPIER":"38","CHAMP SUR DRAC":"38","CHANAS":"38","LA CHAPELLE DU BARD":"38","CHATEAUVILAIN":"38","LE CHEYLAS":"38","CHEZENEUVE":"38","CHIRENS":"38","CHOLONGE":"38","CORNILLON EN TRIEVES":"38","CORRENCON EN VERCORS":"38","DIEMOZ":"38","ENTRE DEUX GUIERS":"38","LES EPARRES":"38","EYZIN PINET":"38","GRESSE EN VERCORS":"38","HIERES SUR AMBY":"38","JANNEYRIAS":"38","LAVALDENS":"38","LENTIOL":"38","LONGECHENAL":"38","MARCILLOLES":"38","MEYLAN":"38","LE MONESTIER DU PERCY":"38","MONTSEVEROUX":"38","LA MORTE":"38","LA MOTTE ST MARTIN":"38","NANTOIN":"38","NOTRE DAME DE L OSIER":"38","NOYAREY":"38","OPTEVOZ":"38","ORNACIEUX":"38","PISIEU":"38","POISAT":"38","POMMIER DE BEAUREPAIRE":"38","POMMIERS LA PLACETTE":"38","PONT EN ROYANS":"38","PROVEYSIEUX":"38","REAUMONT":"38","RENAGE":"38","RENCUREL":"38","REVENTIN VAUGRIS":"38","ROISSARD":"38","ST BUEIL":"38","ST CLAIR DU RHONE":"38","ST DIDIER DE LA TOUR":"38","ST HILAIRE DU ROSIER":"38","ST JEAN DE VAULX":"38","ST JULIEN DE RAZ":"38","ST LATTIER":"38","ST MARTIN DE VAULSERRE":"38","ST NICOLAS DE MACHERIN":"38","ST PANCRASSE":"38","VOSBLES":"39","ARESCHES":"39","AMOU":"40","ARSAGUE":"40","AUDON":"40","AZUR":"40","BELHADE":"40","BIARROTTE":"40","BORDERES ET LAMENSANS":"40","BRASSEMPOUY":"40","CAGNOTTE":"40","CARCARES STE CROIX":"40","CARCEN PONSON":"40","CAUNEILLE":"40","CAZERES SUR L ADOUR":"40","COUDURES":"40","ESTIBEAUX":"40","ESTIGARDE":"40","GARREY":"40","GASTES":"40","GELOUX":"40","GOOS":"40","GOUSSE":"40","GRENADE SUR L ADOUR":"40","HEUGAS":"40","HINX":"40","JOSSE":"40","LABOUHEYRE":"40","LEON":"40","LINXE":"40","LIPOSTHEY":"40","LUBBON":"40","MAGESCQ":"40","MANO":"40","MANT":"40","MARPAPS":"40","MOLIETS ET MAA":"40","MONTSOUE":"40","MUGRON":"40","NARROSSE":"40","PAYROS CAZAUTETS":"40","PHILONDENX":"40","PONTONX SUR L ADOUR":"40","PRECHACQ LES BAINS":"40","PUYOL CAZALET":"40","ST CRICQ CHALOSSE":"40","ST CRICQ DU GAVE":"40","ST CRICQ VILLENEUVE":"40","ST GEIN":"40","ST GEOURS DE MAREMNE":"40","ST MARTIN DE SEIGNANX":"40","ST MARTIN D ONEY":"40","ST PAUL LES DAX":"40","ST SEVER":"40","SAMADET":"40","SINDERES":"40","TRENSACQ":"40","URGONS":"40","VIEUX BOUCAU LES BAINS":"40","VILLENAVE":"40","VILLENEUVE DE MARSAN":"40","AMBLOY":"41","ANGE":"41","AREINES":"41","BOISSEAU":"41","CELLE":"41","LA CHAPELLE ST MARTIN EN PLAINE":"41","CHAUMONT SUR LOIRE":"41","CHOUSSY":"41","CORMENON":"41","COUFFY":"41","COULOMMIERS LA TOUR":"41","DHUIZON":"41","EPIAIS":"41","FAVEROLLES SUR CHER":"41","FONTAINE LES COTEAUX":"41","GY EN SOLOGNE":"41","LORGES":"41","MARCHENOIR":"41","MARCILLY EN GAULT":"41","LA MAROLLE EN SOLOGNE":"41","MESLAND":"41","MILLANCAY":"41","MOISY":"41","MONTHOU SUR CHER":"41","MONTLIVAULT":"41","OUCQUES":"41","OUZOUER LE DOYEN":"41","HAUTEFAGE":"19","LAGRAULIERE":"19","LAGUENNE":"19","LARCHE":"19","LATRONCHE":"19","LESTARDS":"19","LIGNEYRAC":"19","LOSTANGES":"19","MARCILLAC LA CROZE":"19","MARC LA TOUR":"19","MASSERET":"19","MERLINES":"19","MILLEVACHES":"19","PERPEZAC LE BLANC":"19","PEYRISSAC":"19","CONFOLENT PORT DIEU":"19","RILHAC TREIGNAC":"19","SADROC":"19","ST BAZILE DE LA ROCHE":"19","ST BONNET LA RIVIERE":"19","ST BONNET LES TOURS DE MERLE":"19","ST ETIENNE AUX CLOS":"19","ST GENIEZ O MERLE":"19","ST PARDOUX LE VIEUX":"19","ST SORNIN LAVOLPS":"19","ST YRIEIX LE DEJALAT":"19","SARRAN":"19","SEGUR LE CHATEAU":"19","SEILHAC":"19","SIONIAC":"19","SORNAC":"19","SOURSAC":"19","TUDEILS":"19","VARETZ":"19","ARCEAU":"21","AUXANT":"21","AUXEY DURESSES":"21","AUXONNE":"21","AVOSNES":"21","BARD LES EPOISSES":"21","BEIRE LE FORT":"21","BELLENOT SOUS POUILLY":"21","BENOISEY":"21","BESSEY LA COUR":"21","BEVY":"21","BILLEY":"21","SOURCE SEINE":"21","BOURBERAIN":"21","BOUX SOUS SALMAISE":"21","BOUZE LES BEAUNE":"21","BRION SUR OURCE":"21","BROCHON":"21","CHAIGNAY":"21","CHAMBAIN":"21","CHAMESSON":"21","CHAMPEAU EN MORVAN":"21","CHANCEAUX":"21","CHANNAY":"21","CHARREY SUR SAONE":"21","CHAZILLY":"21","CLAMEREY":"21","COLLONGES LES BEVY":"21","COMMARIN":"21","CORCELLES LES MONTS":"21","COURCELLES LES SEMUR":"21","CRUGEY":"21","CULETRE":"21","DETAIN ET BRUANT":"21","DUESME":"21","EBATY":"21","ESBARRES":"21","ESSAROIS":"21","FAUVERNEY":"21","FONTAINE FRANCAISE":"21","FUSSEY":"21","GERGUEIL":"21","GERLAND":"21","GEVROLLES":"21","GISSEY SUR OUCHE":"21","GOMMEVILLE":"21","GROSBOIS EN MONTAGNE":"21","GROSBOIS LES TICHEY":"21","GURGY LA VILLE":"21","HEUILLEY SUR SAONE":"21","LABERGEMENT LES AUXONNE":"21","LA REMUEE":"76","RIVILLE":"76","ROGERVILLE":"76","ROUMARE":"76","STE ADRESSE":"76","ST AIGNAN SUR RY":"76","ST ANTOINE LA FORET":"76","ST CRESPIN":"76","ST EUSTACHE LA FORET":"76","ST JACQUES D ALIERMONT":"76","ST JOUIN BRUNEVAL":"76","ST LAURENT DE BREVEDENT":"76","STE MARGUERITE SUR DUCLAIR":"76","ST MICHEL D HALESCOURT":"76","ST NICOLAS DE LA TAILLE":"76","ST OUEN LE MAUGER":"76","ST PIERRE EN VAL":"76","ST PIERRE LES ELBEUF":"76","ST ROMAIN DE COLBOSC":"76","ST VAAST DU VAL":"76","ST VICTOR L ABBAYE":"76","ST VIGOR D YMONVILLE":"76","SAUSSEUZEMARE EN CAUX":"76","LE THIL RIBERPRE":"76","TOUFFREVILLE SUR EU":"76","TOURVILLE LES IFS":"76","LE TREPORT":"76","LA TRINITE DU MONT":"76","VALLIQUERVILLE":"76","VATTEVILLE LA RUE":"76","VEAUVILLE LES BAONS":"76","VEULES LES ROSES":"76","VIBEUF":"76","VIEUX ROUEN SUR BRESLE":"76","VILLY SUR YERES":"76","VIRVILLE":"76","YEBLERON":"76","YQUEBEUF":"76","YVECRIQUE":"76","ANNET SUR MARNE":"77","AUGERS EN BRIE":"77","AULNOY":"77","BABY":"77","BAGNEAUX SUR LOING":"77","MOREILLES":"85","MOUILLERON LE CAPTIF":"85","PETOSSE":"85","POIROUX":"85","POUZAUGES":"85","LA REORTHE":"85","LES SABLES D OLONNE":"85","ST ANDRE GOULE D OIE":"85","ST AVAUGOURD DES LANDES":"85","RIVES DE L YON":"85","ST FULGENT":"85","ST GEORGES DE MONTAIGU":"85","STE HERMINE":"85","ST HILAIRE DE LOULAY":"85","ST HILAIRE DES LOGES":"85","ST JULIEN DES LANDES":"85","ST MALO DU BOIS":"85","ST MARTIN DE FRAIGNEAU":"85","ST MARTIN DES FONTAINES":"85","ST MARTIN DES NOYERS":"85","ST MARTIN DES TILLEULS":"85","ST MAURICE LE GIRARD":"85","ST MICHEL EN L HERM":"85","ST PAUL MONT PENIT":"85","ST PROUANT":"85","ST VINCENT SUR GRAON":"85","ST VINCENT SUR JARD":"85","SOULLANS":"85","LE TABLIER":"85","TRIAIZE":"85","VENANSAULT":"85","ANTRAN":"86","ASNIERES SUR BLOUR":"86","BERTHEGON":"86","BERUGES":"86","BEUXES":"86","BLANZAY":"86","BLASLAY":"86","BOURESSE":"86","BOURG ARCHAMBAULT":"86","VERSAINVILLE":"14","VIEUX FUME":"14","VIEUX PONT EN AUGE":"14","VILLERS SUR MER":"14","ANTERRIEUX":"15","ARPAJON SUR CERE":"15","AURIAC L EGLISE":"15","CASSANIOUZE":"15","CHANTERELLE":"15","LA CHAPELLE LAURENT":"15","CLAVIERES":"15","COLTINES":"15","CRANDELLES":"15","DEUX VERGES":"15","LE FAU":"15","FREIX ANGLARDS":"15","JALEYRAC":"15","LANDEYRAT":"15","LAROQUEBROU":"15","LAVEISSENET":"15","LEYVAUX":"15","LORCIERES":"15","MOLOMPIZE":"15","MONTSALVY":"15","NAUCELLES":"15","NEUSSARGUES MOISSAC":"15","NEUVEGLISE":"15","RAGEADE":"15","RAULHAC":"15","ST CONSTANT FOURNOULES":"15","ST ETIENNE DE MAURS":"15","ST MARTIN SOUS VIGOUROUX":"15","ST PROJET DE SALERS":"15","ST URCIZE":"15","SALERS":"15","SAUVAT":"15","LA SEGALASSIERE":"15","TANAVELLE":"15","TEISSIERES LES BOULIES":"15","LES TERNES":"15","VALETTE":"15","VERNOLS":"15","VIEILLEVIE":"15","YOLET":"15","LES ADJOTS":"16","ALLOUE":"16","BARBEZIERES":"16","BESSAC":"16","BOISBRETEAU":"16","BOUEX":"16","BRETTES":"16","BREVILLE":"16","BRIGUEUIL":"16","CHAMPAGNE MOUTON":"16","CHARME":"16","COURLAC":"16","DOUZAT":"16","EBREON":"16","EMPURE":"16","FONTCLAIREAU":"16","GIMEUX":"16","JULIENNE":"16","LADIVILLE":"16","LE LINDOIS":"16","ROUMAZIERES LOUBERT":"16","MAGNAC LAVALETTE VILLARS":"16","MEDILLAC":"16","MONTBRON":"16","MONTIGNAC LE COQ":"16","MORNAC":"16","ST PIERRE DE MEAROZ":"38","ST SORLIN DE VIENNE":"38","ST THEOFFREY":"38","ST VICTOR DE MORESTEL":"38","SARCENAS":"38","SASSENAGE":"38","SAVAS MEPIN":"38","SICCIEU ST JULIEN ET CARISIEU":"38","SUCCIEU":"38","SUSVILLE":"38","THEYS":"38","LA TOUR DU PIN":"38","TRAMOLE":"38","VALENCOGNE":"38","LA VALETTE":"38","VATILIEU":"38","LE VERSOUD":"38","VIF":"38","VILLARD NOTRE DAME":"38","VILLEFONTAINE":"38","ABERGEMENT LA RONCE":"39","ABERGEMENT LE GRAND":"39","ABERGEMENT LE PETIT":"39","ABERGEMENT LES THESY":"39","AIGLEPIERRE":"39","ARCHELANGE":"39","AROMAS":"39","ASNANS BEAUVOISIN":"39","AUGISEY":"39","AUTHUME":"39","AVIGNON LES ST CLAUDE":"39","BLOIS SUR SEILLE":"39","BOURCIA":"39","BRAINANS":"39","BRANS":"39","BROISSIA":"39","CHAMOLE":"39","CHAMPDIVERS":"39","CHAPELLE VOLAND":"39","CHAPOIS":"39","CHAREZIER":"39","CHATEAU DES PRES":"39","CHAVERIA":"39","CHEMIN":"39","CHENE SEC":"39","CHILLY LE VIGNOBLE":"39","CHISSERIA":"39","CHISSEY SUR LOUE":"39","CHOISEY":"39","COLONNE":"39","CORNOD":"39","COURLAOUX":"39","COYRON":"39","CRAMANS":"39","CROTENAY":"39","DESSIA":"39","DIGNA":"39","DOMPIERRE SUR MONT":"39","FONCINE LE HAUT":"39","FONTAINEBRUX":"39","LA FRASNEE":"39","FREBUANS":"39","GREDISANS":"39","IVORY":"39","IVREY":"39","JEURRE":"39","LAJOUX":"39","LAVANGEOT":"39","LEMUY":"39","LONGCHAUMOIS":"39","LA MARRE":"39","MERONA":"39","MESNAY":"39","MOIRON":"39","MOISSEY":"39","MONTAGNA LE RECONDUIT":"39","MONTFLEUR":"39","MONTMARLON":"39","MOURNANS CHARBONNY":"39","MOUTONNE":"39","MUTIGNEY":"39","LES NANS":"39","LE PLESSIS DORIN":"41","PRUNAY CASSEREAU":"41","RENAY":"41","RUAN SUR EGVONNE":"41","ST AMAND LONGPRE":"41","AURILLAC":"15","BARRIAC LES BOSQUETS":"15","BRAGEAC":"15","CHALIERS":"15","LA CHAPELLE D ALAGNON":"15","ESCORAILLES":"15","LE FALGOUX":"15","JABRUN":"15","LABESSERETTE":"15","LACAPELLE DEL FRAISSE":"15","LACAPELLE VIESCAMP":"15","LASCELLE":"15","LAVASTRIE":"15","LEUCAMP":"15","LIEUTADES":"15","MADIC":"15","MENET":"15","LA MONSELIE":"15","MONTGRELEIX":"15","MONTMURAT":"15","MONTVERT":"15","PIERREFORT":"15","REZENTIERES":"15","ROANNES ST MARY":"15","ROFFIAC":"15","ROUZIERS":"15","ST BONNET DE SALERS":"15","ST CIRGUES DE MALBERT":"15","ST ETIENNE CANTALES":"15","ST JACQUES DES BLATS":"15","ST JULIEN DE TOURSAC":"15","ST MAMET LA SALVETAT":"15","ST PONCY":"15","TALIZAT":"15","TEISSIERES DE CORNET":"15","THIEZAC":"15","TREMOUILLE":"15","TRIZAC":"15","LE VAULMIER":"15","VIC SUR CERE":"15","VIRARGUES":"15","ANSAC SUR VIENNE":"16","BARRET":"16","BARRO":"16","BAYERS":"16","BAZAC":"16","BELLON":"16","BENEST":"16","BOURG CHARENTE":"16","BOUTEVILLE":"16","BOUTIERS ST TROJAN":"16","BRIE SOUS CHALAIS":"16","BROSSAC":"16","CHABANAIS":"16","CHASSENON":"16","CHATEAUBERNARD":"16","CHENON":"16","COGNAC":"16","COMBIERS":"16","COURCOME":"16","LA COURONNE":"16","CRITEUIL LA MAGDELEINE":"16","EPENEDE":"16","ERAVILLE":"16","EXIDEUIL":"16","LONGVIC":"21","LOSNE":"21","MAGNY LA VILLE":"21","MANLAY":"21","MARANDEUIL":"21","MARCILLY SUR TILLE":"21","MAREY LES FUSSEY":"21","MAREY SUR TILLE":"21","MARIGNY LE CAHOUET":"21","MAVILLY MANDELOT":"21","MENESSAIRE":"21","MESSIGNY ET VANTOUX":"21","MEURSAULT":"21","MOLINOT":"21","MONTHELIE":"21","MONTIGNY SUR ARMANCON":"21","MONTIGNY SUR AUBE":"21","MOUTIERS ST JEAN":"21","NAN SOUS THIL":"21","NANTOUX":"21","NEUILLY LES DIJON":"21","NOIDAN":"21","OISILLY":"21","ORIGNY":"21","OUGES":"21","PAGNY LA VILLE":"21","PAINBLANC":"21","POINCON LES LARREY":"21","PONCEY LES ATHEE":"21","PONCEY SUR L IGNON":"21","PONT":"21","POUILLY SUR VINGEANNE":"21","RECEY SUR OURCE":"21","REMILLY SUR TILLE":"21","RENEVE":"21","REULLE VERGY":"21","ROCHEFORT SUR BREVON":"21","ROUVRES SOUS MEILLY":"21","RUFFEY LES BEAUNE":"21","ST ANDEUX":"21","ST GERMAIN LE ROCHEUX":"21","ST GERMAIN LES SENAILLY":"21","STE MARIE SUR OUCHE":"21","ST SEINE L ABBAYE":"21","ST SEINE SUR VINGEANNE":"21","ST SYMPHORIEN SUR SAONE":"21","SAULIEU":"21","SAULON LA RUE":"21","SAVIGNY SOUS MALAIN":"21","SAVOLLES":"21","SEMAREY":"21","SEMOND":"21","SINCEY LES ROUVRAY":"21","TALMAY":"21","TARSUL":"21","THOIRES":"21","THOISY LA BERCHERE":"21","THOMIREY":"21","THOREY EN PLAINE":"21","THOREY SOUS CHARNY":"21","THOREY SUR OUCHE":"21","TICHEY":"21","TOUTRY":"21","TROUHAUT":"21","VAL SUZON":"21","VANVEY":"21","VAUCHIGNON":"21","VEILLY":"21","VELOGNY":"21","VERNOIS LES VESVRES":"21","VEUVEY SUR OUCHE":"21","VIEVIGNE":"21","VIEVY":"21","VIGNOLES":"21","VILLARS ET VILLENOTTE":"21","VILLECOMTE":"21","VILLIERS LE DUC":"21","VILLY EN AUXOIS":"21","VITTEAUX":"21","VOUDENAY":"21","VOULAINES LES TEMPLIERS":"21","BOURBRIAC":"22","ILE DE BREHAT":"22","BRELIDY":"22","CARNOET":"22","COATASCORN":"22","BOURNAND":"86","CEAUX EN COUHE":"86","CHAPELLE VIVIERS":"86","CHATAIN":"86","CHATEAU GARNIER":"86","CHIRE EN MONTREUIL":"86","CIVAUX":"86","CLOUE":"86","COUSSAY LES BOIS":"86","CROUTELLE":"86","CURZAY SUR VONNE":"86","DISSAY":"86","HAIMS":"86","JAZENEUIL":"86","JOURNET":"86","LUSSAC LES CHATEAUX":"86","MARIGNY CHEMEREAU":"86","MOUTERRE SUR BLOURDE":"86","NEUVILLE DE POITOU":"86","OYRE":"86","PINDRAY":"86","PORT DE PILES":"86","POUANT":"86","QUEAUX":"86","LA ROCHE POSAY":"86","ST GERVAIS LES TROIS CLOCHERS":"86","ST MARTIN L ARS":"86","ST REMY SUR CREUSE":"86","SAIRES":"86","SAVIGNY SOUS FAYE":"86","SILLARS":"86","TERCE":"86","LA TRIMOUILLE":"86","VERRUE":"86","LE VIGEANT":"86","LA VILLEDIEU DU CLAIN":"86","VOUNEUIL SOUS BIARD":"86","AMBAZAC":"87","BEAUMONT DU LAC":"87","BELLAC":"87","BERSAC SUR RIVALIER":"87","BURGNAC":"87","BUSSIERE GALANT":"87","CHAILLAC SUR VIENNE":"87","LE CHALARD":"87","CHATEAUPONSAC":"87","CHEISSOUX":"87","CHERONNAC":"87","CONDAT SUR VIENNE":"87","LA CROISILLE SUR BRIANCE":"87","DINSAC":"87","GAJOUBERT":"87","LA GENEYTOUSE":"87","GLANGES":"87","ISLE":"87","JOURGNAC":"87","MAILHAC SUR BENAIZE":"87","MASLEON":"87","MEUZAC":"87","NEDDE":"87","NEUVIC ENTIER":"87","ORADOUR ST GENEST":"87","ORADOUR SUR VAYRES":"87","PAGEAS":"87","PENSOL":"87","PEYRAT DE BELLAC":"87","PEYRAT LE CHATEAU":"87","PEYRILHAC":"87","RILHAC RANCON":"87","ST AMAND MAGNAZEIX":"87","ST BAZILE":"87","ST JUNIEN":"87","ST LAURENT LES EGLISES":"87","ST LEONARD DE NOBLAT":"87","ST MARTIN LE MAULT":"87","ST OUEN SUR GARTEMPE":"87","ST VICTURNIEN":"87","LES SALLES LAVAUGUYON":"87","SOLIGNAC":"87","AHEVILLE":"88","NONAVILLE":"16","ORGEDEUIL":"16","POURSAC":"16","ST ADJUTORY":"16","ST AMANT DE BONNIEURE":"16","ST EUTROPE":"16","ST GROUX":"16","ST LAURENT DE CERIS":"16","ST LAURENT DE COGNAC":"16","ST MARTIN DU CLOCHER":"16","ST SULPICE DE RUFFEC":"16","ST YRIEIX SUR CHARENTE":"16","SAULGOND":"16","TRIAC LAUTRAIT":"16","VILLIERS LE ROUX":"16","VITRAC ST VINCENT":"16","YVIERS":"16","AIGREFEUILLE D AUNIS":"17","ANGOULINS":"17","ANNEZAY":"17","ARCHINGEAY":"17","AUMAGNE":"17","BEDENAC":"17","BENON":"17","BRIE SOUS ARCHIAC":"17","BRIE SOUS MORTAGNE":"17","BRIZAMBOURG":"17","LA BROUSSE":"17","BUSSAC SUR CHARENTE":"17","CHARTUZAC":"17","CHATENET":"17","CHAUNAC":"17","CHERAC":"17","CHERBONNIERES":"17","CHIVES":"17","CORIGNAC":"17","COURCOURY":"17","LA CROIX COMTESSE":"17","LE DOUHET":"17","ECHEBRUNE":"17","ECURAT":"17","EPARGNES":"17","FONTAINES D OZILLAC":"17","GIBOURNE":"17","HAIMPS":"17","LOIX":"17","MAZERAY":"17","MIGRE":"17","MONTLIEU LA GARDE":"17","LE MUNG":"17","NIEULLE SUR SEUDRE":"17","PLASSAY":"17","PONS":"17","POURSAY GARNAUD":"17","PRIGNAC":"17","PUILBOREAU":"17","PUY DU LAC":"17","RETAUD":"17","ST CLEMENT DES BALEINES":"17","ST COUTANT LE GRAND":"17","ST GEORGES DE LONGUEPIERRE":"17","ST GERMAIN DE LUSIGNAN":"17","ST MARTIN D ARY":"17","NOGNA":"39","ORBAGNA":"39","OUNANS":"39","PLENISE":"39","POIDS DE FIOLE":"39","PRATZ":"39","RANCHOT":"39","RANS":"39","REITHOUSE":"39","RELANS":"39","ROCHEFORT SUR NENON":"39","ROMANGE":"39","ROTALIER":"39","SAFFLOZ":"39","ST BARAING":"39","ST GERMAIN EN MONTAGNE":"39","ST LAURENT EN GRANDVAUX":"39","ST LOTHAIN":"39","ST THIEBAUD":"39","SELLIERES":"39","SOUVANS":"39","TAVAUX":"39","LA TOUR DU MEIX":"39","LE VAUDIOUX":"39","VERNANTOIS":"39","VILLENEUVE D AVAL":"39","VILLENEUVE LES CHARNOD":"39","VILLERS FARLAY":"39","VILLEVIEUX":"39","LE VILLEY":"39","ARTASSENX":"40","BAIGTS":"40","BANOS":"40","BAUDIGNAN":"40","BELIS":"40","BELUS":"40","BENESSE LES DAX":"40","BENESSE MAREMNE":"40","BUANES":"40","CANENX ET REAUT":"40","CASTAIGNOS SOUSLENS":"40","CLASSUN":"40","CREON D ARMAGNAC":"40","EUGENIE LES BAINS":"40","EYRES MONCUBE":"40","LE FRECHE":"40","GAILLERES":"40","HABAS":"40","HERM":"40","HERRE":"40","LABASTIDE CHALOSSE":"40","LABASTIDE D ARMAGNAC":"40","LABRIT":"40","LACRABE":"40","LAHOSSE":"40","LATRILLE":"40","LENCOUACQ":"40","LESGOR":"40","LOUER":"40","LOURQUEN":"40","RETJONS":"40","LUXEY":"40","MAILLAS":"40","MAURIES":"40","MEZOS":"40","MIMIZAN":"40","MISSON":"40","MONT DE MARSAN":"40","MOUSCARDES":"40","OEYREGAVE":"40","OEYRELUY":"40","ORX":"40","OSSAGES":"40","PEY":"40","PEYRE":"40","PUJO LE PLAN":"40","RION DES LANDES":"40","RIVIERE SAAS ET GOURBY":"40","ST LOUBOUER":"40","GARAT":"16","GARDES LE PONTAROUX":"16","HIERSAC":"16","JUIGNAC":"16","LESTERPS":"16","LINARS":"16","LONDIGNY":"16","LONNES":"16","MAINXE":"16","MERPINS":"16","LES METAIRIES":"16","MONTBOYER":"16","MONTMERAC":"16","MOULIDARS":"16","MOUTONNEAU":"16","ORADOUR FANAIS":"16","PASSIRAC":"16","LA PERUSE":"16","POULLIGNAC":"16","PRESSIGNAC":"16","RANVILLE BREUILLAUD":"16","REPARSAC":"16","RONSENAC":"16","ST BONNET":"16","ST CYBARDEAUX":"16","ST FORT SUR LE NE":"16","ST GERMAIN DE MONTBRON":"16","SALLES DE VILLEFAGNAN":"16","SALLES LAVALETTE":"16","SAUVAGNAC":"16","SAUVIGNAC":"16","SIGOGNE":"16","SOYAUX":"16","TAPONNAT FLEURIGNAC":"16","THEIL RABIER":"16","TOURRIERS":"16","TOUVERAC":"16","TROIS PALIS":"16","TUSSON":"16","VAUX LAVALETTE":"16","VIGNOLLES":"16","VILLEBOIS LAVALETTE":"16","VINDELLE":"16","VIVILLE":"16","VOUZAN":"16","ILE D AIX":"17","ALLAS BOCAGE":"17","ARTHENAC":"17","AUTHON EBEON":"17","AYTRE":"17","BAGNIZEAU":"17","LA BARDE":"17","BAZAUGES":"17","BELLUIRE":"17","BLANZAY SUR BOUTONNE":"17","BORDS":"17","BOUHET":"17","BRAN":"17","BREUIL MAGNE":"17","CERCOUX":"17","CHAMPAGNOLLES":"17","CHERVETTES":"17","CLAM":"17","CLAVETTE":"17","COURCON":"17","CRESSE":"17","DOEUIL SUR LE MIGNON":"17","LES EGLISES D ARGENTEUIL":"17","ESNANDES":"17","LA FREDIERE":"17","GERMIGNAC":"17","GIVREZAC":"17","COATREVEN":"22","COETMIEUX":"22","DINAN":"22","DUAULT":"22","LE FOEIL":"22","GOUAREC":"22","GRACE UZEL":"22","LE HAUT CORLAY":"22","HENANBIHEN":"22","HENANSAL":"22","LE HINGLE":"22","KERBORS":"22","KERPERT":"22","LANISCAT":"22","LANRIVAIN":"22","LOUARGAT":"22","LOUDEAC":"22","MAEL PESTIVIEN":"22","PAULE":"22","PERRET":"22","PLELAN LE PETIT":"22","PLELAUFF":"22","PLESIDY":"22","PLESLIN TRIGAVOU":"22","PLESTIN LES GREVES":"22","PLEVEN":"22","PLOEUC L HERMITAGE":"22","PLOUASNE":"22","PLOUEC DU TRIEUX":"22","PLOUGRAS":"22","PLOUGRESCANT":"22","PLOUGUERNEVEL":"22","PLOUGUIEL":"22","PLOULEC H":"22","PLUDUNO":"22","PLUFUR":"22","PLUMAUGAT":"22","PLUMIEUX":"22","POMMERIT LE VICOMTE":"22","QUINTIN":"22","RUCA":"22","ST BARNABE":"22","ST BIHY":"22","ST ETIENNE DU GUE DE L ISLE":"22","ST JUVAT":"22","ST LORMEL":"22","ST QUAY PORTRIEUX":"22","ST IGEAUX":"22","TADEN":"22","TREGROM":"22","TREMARGAT":"22","TREMEREUC":"22","TREMOREL":"22","TREMUSON":"22","TREVEREC":"22","TREVRON":"22","ARRENES":"23","AUGERES":"23","BASVILLE":"23","LE BOURG D HEM":"23","LA CHAPELLE BALOUE":"23","LE CHAUCHET":"23","LA COURTINE":"23","CROCQ":"23","DOMEYROT":"23","FAUX LA MONTAGNE":"23","FENIERS":"23","FLAYAT":"23","FLEURAT":"23","LA FORET DU TEMPLE":"23","FRESSELINES":"23","LADAPEYRE":"23","LEPINAS":"23","LOURDOUEIX ST PIERRE":"23","LUPERSAT":"23","MAUTES":"23","MAZEIRAT":"23","ARCHETTES":"88","BAINVILLE AUX SAULES":"88","BAZOILLES ET MENIL":"88","BEGNECOURT":"88","BELMONT SUR VAIR":"88","BIECOURT":"88","BOUZEMONT":"88","CERTILLEUX":"88","CHANTRAINE":"88","CHARMOIS DEVANT BRUYERES":"88","CHAUMOUSEY":"88","CIRCOURT SUR MOUZON":"88","BAN SUR MEURTHE CLEFCY":"88","CLEZENTAINE":"88","COINCHES":"88","CORNIMONT":"88","COURCELLES SOUS CHATENOIS":"88","DAMAS ET BETTEGNEY":"88","DARNEY AUX CHENES":"88","DERBAMONT":"88","DESTORD":"88","DEYVILLERS":"88","DOMEVRE SOUS MONTFORT":"88","DOMJULIEN":"88","DOMMARTIN AUX BOIS":"88","FERDRUPT":"88","LA FORGE":"88","FRAIN":"88","FREBECOURT":"88","FRENELLE LA PETITE":"88","FRIZON":"88","GIRECOURT SUR DURBION":"88","LA GRANDE FOSSE":"88","GRANDRUPT DE BAINS":"88","GREUX":"88","GUGNECOURT":"88","HARCHECHAMP":"88","HARMONVILLE":"88","HENNEZEL":"88","HERPELMONT":"88","HOUECOURT":"88","ISCHES":"88","LAVELINE DU HOUX":"88","LESSEUX":"88","LIRONCOURT":"88","LUVIGNY":"88","MADONNE ET LAMEREY":"88","MAREY":"88","MATTAINCOURT":"88","MAXEY SUR MEUSE":"88","MAZELEY":"88","MENIL SUR BELVITTE":"88","MONT LES LAMARCHE":"88","MONTHUREUX LE SEC":"88","MONTMOTIER":"88","MORIZECOURT":"88","NEUFCHATEAU":"88","NOSSONCOURT":"88","PAIR ET GRANDRUPT":"88","PAREY SOUS MONTFORT":"88","LA PETITE FOSSE":"88","PLEUVEZAIN":"88","PONT LES BONFAYS":"88","PONT SUR MADON":"88","LES POULIERES":"88","PROVENCHERES LES DARNEY":"88","RAINVILLE":"88","RAON SUR PLAINE":"88","RAPEY":"88","RAVES":"88","REGNEVELLE":"88","REGNEY":"88","RUPPES":"88","ST MICHEL SUR MEURTHE":"88","SONCOURT":"88","TAINTRUX":"88","TENDON":"88","CAPAVENIR VOSGES":"88","TOTAINVILLE":"88","UBEXY":"88","LA VACHERESSE ET LA ROUILLIE":"88","VERVEZELLE":"88","VEXAINCOURT":"88","VIENVILLE":"88","VIMENIL":"88","LES VOIVRES":"88","WISEMBACH":"88","XARONVAL":"88","AILLANT SUR THOLON":"89","ANGELY":"89","ARCES DILO":"89","ST PALAIS DE NEGRIGNAC":"17","ST PIERRE DU PALAIS":"17","ST SEURIN DE PALENNE":"17","SAINTES":"17","SALIGNAC SUR CHARENTE":"17","SEIGNE":"17","SOUMERAS":"17","TALMONT SUR GIRONDE":"17","TANZAC":"17","TAUGON":"17","TESSON":"17","THAIMS":"17","TONNAY BOUTONNE":"17","VENERAND":"17","VERGNE":"17","VERINES":"17","VILLEMORIN":"17","VIROLLET":"17","PORT DES BARQUES":"17","LA BREE LES BAINS":"17","ALLOUIS":"18","APREMONT SUR ALLIER":"18","AUBIGNY SUR NERE":"18","BEDDES":"18","BENGY SUR CRAON":"18","BLET":"18","BOURGES":"18","BRINON SUR SAULDRE":"18","LA CHAPELLE MONTLINARD":"18","CHAROST":"18","CHATEAUMEILLANT":"18","LE CHAUTAY":"18","CORNUSSE":"18","CROISY":"18","EPINEUIL LE FLEURIEL":"18","GARDEFORT":"18","GRACAY":"18","GROSSOUVRE":"18","IGNOL":"18","JUSSY LE CHAUDRIER":"18","LOYE SUR ARNON":"18","LUGNY CHAMPAGNE":"18","MARCAIS":"18","MARSEILLES LES AUBIGNY":"18","MEHUN SUR YEVRE":"18","MENETOU COUTURE":"18","MENETREOL SUR SAULDRE":"18","MONTLOUIS":"18","MORTHOMIERS":"18","NANCAY":"18","NEUVY DEUX CLOCHERS":"18","OUROUER LES BOURDELINS":"18","ST LYE":"10","ST LEGER SOUS BRIENNE":"10","ST BENOIST SUR VANNE":"10","ST BENOIT SUR SEINE":"10","SOULAINES DHUYS":"10","STE SAVINE":"10","ST FLAVY":"10","SAULCY":"10","LA VILLENEUVE AU CHENE":"10","BELVIANES ET CAVIRAC":"11","BELFORT SUR REBENTY":"11","BESSEDE DE SAULT":"11","ARQUETTES EN VAL":"11","BELFLOU":"11","ARAGON":"11","VOUE":"10","VILLENEUVE AU CHEMIN":"10","VILLEMOIRON EN OTHE":"10","VALLENTIGNY":"10","VAUCHASSIS":"10","VERRICOURT":"10","VILLADIN":"10","VAUCOGNE":"10","TRAINEL":"10","TURGY":"10","OBERSTEINBACH":"67","OFFENDORF":"67","OFFWILLER":"67","OTTERSTHAL":"67","ST MARTIN DE HINX":"40","ST MAURICE SUR ADOUR":"40","SANGUINET":"40","SERRES GASTON":"40","SEYRESSE":"40","SOORTS HOSSEGOR":"40","SORDE L ABBAYE":"40","SORE":"40","SORT EN CHALOSSE":"40","VIELLE SOUBIRAN":"40","BAILLOU":"41","BINAS":"41","BOUFFRY":"41","BOURSAY":"41","BRIOU":"41","CANDE SUR BEUVRON":"41","CHAMBON SUR CISSE":"41","LA CHAPELLE VICOMTESSE":"41","LA CHAUSSEE ST VICTOR":"41","CHAUVIGNY DU PERCHE":"41","CHEVERNY":"41","COUDDES":"41","COUR CHEVERNY":"41","COURMEMIN":"41","CROUY SUR COSSON":"41","DANZE":"41","DROUE":"41","FAYE":"41","LA FERTE ST CYR":"41","FORTAN":"41","FRETEVAL":"41","LE GAULT PERCHE":"41","JOSNES":"41","LASSAY SUR CROISNE":"41","MARAY":"41","MARCILLY EN BEAUCE":"41","MEHERS":"41","MONTEAUX":"41","MONT PRES CHAMBORD":"41","MONTRIEUX EN SOLOGNE":"41","MONTROUVEAU":"41","MUIDES SUR LOIRE":"41","NOYERS SUR CHER":"41","OUCHAMPS":"41","PEZOU":"41","LE POISLAY":"41","PRAY":"41","RHODON":"41","RILLY SUR LOIRE":"41","ROMORANTIN LANTHENAY":"41","ST GOURGON":"41","ST JACQUES DES GUERETS":"41","ST JULIEN SUR CHER":"41","ST MARC DU COR":"41","ST ROMAIN SUR CHER":"41","ST VIATRE":"41","SELLES ST DENIS":"41","SELLES SUR CHER":"41","SOUVIGNY EN SOLOGNE":"41","THOURY":"41","TOUR EN SOLOGNE":"41","VALLIERES LES GRANDES":"41","VEILLEINS":"41","VERNOU EN SOLOGNE":"41","VILLECHAUVE":"41","VILLEDIEU LE CHATEAU":"41","VILLEFRANCOEUR":"41","VILLENEUVE FROUVILLE":"41","VILLERBON":"41","VOUZON":"41","YVOY LE MARRON":"41","ARCINGES":"42","BOEN SUR LIGNON":"42","BOISSET LES MONTROND":"42","CHAGNON":"42","CHALAIN D UZORE":"42","CHAMPOLY":"42","CHANDON":"42","LA CHAPELLE EN LAFAYE":"42","CHAVANAY":"42","CHIRASSIMONT":"42","CHUYER":"42","COMMELLE VERNAY":"42","LA GREVE SUR MIGNON":"17","GREZAC":"17","L HOUMEAU":"17","LA JARD":"17","JARNAC CHAMPAGNE":"17","JUSSAS":"17","LANDES":"17","LUCHAT":"17","MACQUEVILLE":"17","MESSAC":"17","MONTROY":"17","NEULLES":"17","NUAILLE SUR BOUTONNE":"17","PONT L ABBE D ARNOULT":"17","LA RONDE":"17","ROUFFIGNAC":"17","ST AIGULIN":"17","ST BONNET SUR GIRONDE":"17","ST CESAIRE":"17","ST CIERS DU TAILLON":"17","ST DIZANT DU GUA":"17","ST GERMAIN DE MARENCENNES":"17","ST JEAN D ANGLE":"17","ST JULIEN DE L ESCAP":"17","STE LHEURINE":"17","ST MARTIAL DE VITATERNE":"17","ST MEDARD D AUNIS":"17","ST PALAIS SUR MER":"17","ST PIERRE D AMILLY":"17","ST PIERRE DE L ISLE":"17","STE RAMEE":"17","ST ROMAIN DE BENET":"17","ST SATURNIN DU BOIS":"17","ST SIMON DE PELLOUAILLE":"17","ST SORLIN DE CONAC":"17","ST SULPICE D ARNOULT":"17","ST SULPICE DE ROYAN":"17","ST THOMAS DE CONAC":"17","ST TROJAN LES BAINS":"17","SEMOUSSAC":"17","LE SEURE":"17","SIECQ":"17","SOUBISE":"17","SOUBRAN":"17","VANZAC":"17","VERGEROUX":"17","ARDENAIS":"18","ARGENT SUR SAULDRE":"18","AUBINGES":"18","AUGY SUR AUBOIS":"18","AVORD":"18","BARLIEU":"18","BELLEVILLE SUR LOIRE":"18","BUE":"18","LA CELLE CONDE":"18","CERBOIS":"18","CHALIVOY MILON":"18","CHARENTON DU CHER":"18","CHATEAUNEUF SUR CHER":"18","CHERY":"18","CHEZAL BENOIT":"18","COUY":"18","DREVANT":"18","ENNORDRES":"18","FARGES ALLICHAMPS":"18","GERMIGNY L EXEMPT":"18","HERRY":"18","INEUIL":"18","JALOGNES":"18","JUSSY CHAMPAGNE":"18","LANTAN":"18","LAPAN":"18","LEVET":"18","LUNERY":"18","MAREUIL SUR ARNON":"18","MENETOU RATEL":"18","MEASNES":"23","MONTBOUCHER":"23","NAILLAT":"23","NOUZERINES":"23","NOUZIERS":"23","PEYRABOUT":"23","PONTCHARRAUD":"23","SAGNAT":"23","LA SERRE BUSSIERE VIEILLE":"23","SOUMANS":"23","SOUS PARSAT":"23","LA SOUTERRAINE":"23","ST AGNANT PRES CROCQ":"23","ST DIZIER LA TOUR":"23","ST DIZIER LES DOMAINES":"23","ST DIZIER LEYRENNE":"23","ST DOMET":"23","ST ETIENNE DE FURSAC":"23","ST GOUSSAUD":"23","ST JUNIEN LA BREGERE":"23","ST LEGER LE GUERETOIS":"23","ST MARC A LOUBAUD":"23","ST PARDOUX D ARNET":"23","ST PIERRE CHERIGNAT":"23","ST PIERRE LE BOST":"23","ST SILVAIN MONTAIGUT":"23","ST SILVAIN SOUS TOULX":"23","ST SULPICE LES CHAMPS":"23","ST VAURY":"23","ST VICTOR EN MARCHE":"23","ST YRIEIX LES BOIS":"23","TARDES":"23","VIDAILLAT":"23","ANGOISSE":"24","AUGIGNAC":"24","LA BACHELLERIE":"24","BAYAC":"24","BONNEVILLE ET ST AVIT DE FUMADIERES":"24","BOURDEILLES":"24","BOURG DES MAISONS":"24","BOURG DU BOST":"24","BOUZIC":"24","CARLUX":"24","CARVES":"24","CASTELS":"24","CENAC ET ST JULIEN":"24","LE CHANGE":"24","LA CHAPELLE AUBAREIL":"24","CLERMONT DE BEAUREGARD":"24","CONDAT SUR VEZERE":"24","COUBJOURS":"24","COUZE ET ST FRONT":"24","DAGLAN":"24","LES EYZIES DE TAYAC SIREUIL":"24","FANLAC":"24","FESTALEMPS":"24","LA FEUILLADE":"24","FONROQUE":"24","GARDONNE":"24","LA GONTERIE BOULOUNEIX":"24","ISSAC":"24","JOURNIAC":"24","JUMILHAC LE GRAND":"24","LANQUAIS":"24","LE LARDIN ST LAZARE":"24","LAVALADE":"24","LIMEYRAT":"24","MONPAZIER":"24","MONSAC":"24","MONSAGUEL":"24","MONSEC":"24","MONTAGNAC D AUBEROCHE":"24","MONTAGRIER":"24","MONTREM":"24","MUSSIDAN":"24","NOTRE DAME DE SANILHAC":"24","PAULIN":"24","PAUNAT":"24","PETIT BERSAC":"24","PONTOURS":"24","PREYSSAC D EXCIDEUIL":"24","RAMPIEUX":"24","ARMEAU":"89","ASQUINS":"89","BEUGNON":"89","BLANNAY":"89","BONNARD":"89","BUTTEAUX":"89","CARISEY":"89","LA CELLE ST CYR":"89","CEZY":"89","CHAMPVALLON":"89","CHARBUY":"89","CHASTELLUX SUR CURE":"89","CHEMILLY SUR YONNE":"89","CHENEY":"89","CHICHERY":"89","CHITRY":"89","CISERY":"89","COLLEMIERS":"89","COULANGERON":"89","DOMECY SUR LE VAULT":"89","ESCOLIVES STE CAMILLE":"89","FOISSY SUR VANNE":"89","FONTENAY SOUS FOURONNES":"89","VALRAVILLON":"89","LAINSECQ":"89","LALANDE":"89","LAROCHE ST CYDROINE":"89","LASSON":"89","LUCY SUR CURE":"89","MERRY SEC":"89","MERRY SUR YONNE":"89","MOLESMES":"89","MOUFFY":"89","NEUVY SAUTOUR":"89","PAROY SUR THOLON":"89","PIFFONDS":"89","PIMELLES":"89","POILLY SUR THOLON":"89","PONTAUBERT":"89","PONT SUR VANNE":"89","PONT SUR YONNE":"89","PROVENCY":"89","QUARRE LES TOMBES":"89","ST DENIS LES SENS":"89","ST JULIEN DU SAULT":"89","ST LEGER VAUBAN":"89","ST MAURICE AUX RICHES HOMMES":"89","SALIGNY":"89","SANTIGNY":"89","SAUVIGNY LE BEUREAL":"89","SENNEVOY LE BAS":"89","SERBONNES":"89","SOUMAINTRAIN":"89","STIGNY":"89","TONNERRE":"89","TOUCY":"89","TRONCHOY":"89","VAL DE MERCY":"89","VERGIGNY":"89","VERMENTON":"89","VILLEBLEVIN":"89","VILLEBOUGIS":"89","VILLECHETIVE":"89","VILLEFARGEAU":"89","VILLENEUVE LA DONDAGRE":"89","VILLENEUVE LA GUYARD":"89","VILLENEUVE LES GENETS":"89","VILLENEUVE SUR YONNE":"89","VILLEVALLIER":"89","VILLIERS SUR THOLON":"89","YROUERRE":"89","AUXELLES HAUT":"90","BORON":"90","OTTWILLER":"67","PETERSBACH":"67","PRINTZHEIM":"67","RANRUPT":"67","REUTENBOURG":"67","RITTERSHOFFEN":"67","ROESCHWOOG":"67","ROSHEIM":"67","RUSS":"67","ST BLAISE LA ROCHE":"67","ST JEAN SAVERNE":"67","SCHAEFFERSHEIM":"67","SCHAFFHOUSE SUR ZORN":"67","SCHEIBENHARD":"67","SCHLEITHAL":"67","SCHOENBOURG":"67","SCHWEIGHOUSE SUR MODER":"67","SOLBACH":"67","STEINBOURG":"67","THAL DRULINGEN":"67","TRAENHEIM":"67","WALDOLWISHEIM":"67","WANGEN":"67","WESTHOFFEN":"67","WINGEN":"67","WITTERNHEIM":"67","WITTERSHEIM":"67","ZEINHEIM":"67","ALGOLSHEIM":"68","ASPACH LE BAS":"68","ASPACH MICHELBACH":"68","AUBURE":"68","BERRWILLER":"68","BIESHEIM":"68","BISCHWIHR":"68","BISEL":"68","BOURBACH LE BAS":"68","BRECHAUMONT":"68","BRINCKHEIM":"68","BURNHAUPT LE BAS":"68","BUSCHWILLER":"68","CARSPACH":"68","FELDKIRCH":"68","FISLIS":"68","FLAXLANDEN":"68","FOLGENSBOURG":"68","FRANKEN":"68","GUEBWILLER":"68","GUEMAR":"68","GUNDOLSHEIM":"68","HABSHEIM":"68","HAGENBACH":"68","HATTSTATT":"68","HEITEREN":"68","HEIWILLER":"68","HELFRANTZKIRCH":"68","HINDLINGEN":"68","HIRSINGUE":"68","HIRTZBACH":"68","HUNINGUE":"68","HUSSEREN WESSERLING":"68","ILLHAEUSERN":"68","INGERSHEIM":"68","KIRCHBERG":"68","KRUTH":"68","LAUW":"68","LEYMEN":"68","LIGSDORF":"68","LINTHAL":"68","MAGSTATT LE BAS":"68","MAGSTATT LE HAUT":"68","MANSPACH":"68","MEYENHEIM":"68","MUNTZENHEIM":"68","MUNWILLER":"68","NAMBSHEIM":"68","OBERENTZEN":"68","ORBEY":"68","ORSCHWIHR":"68","PFAFFENHEIM":"68","PFASTATT":"68","PULVERSHEIM":"68","RANSPACH LE BAS":"68","COTTANCE":"42","CROIZET SUR GAND":"42","EPERCIEUX ST PAUL":"42","FIRMINY":"42","LA GRESLE":"42","SICHAMPS":"58","TAMNAY EN BAZOIS":"58","TEIGNY":"58","THAIX":"58","THIANGES":"58","VARENNES VAUZELLES":"58","VIELMANAY":"58","AMFROIPRET":"59","ANHIERS":"59","ANNOEULLIN":"59","ANSTAING":"59","ARLEUX":"59","ATTICHES":"59","AUBERS":"59","AUBIGNY AU BAC":"59","AVESNELLES":"59","BACHANT":"59","BETTIGNIES":"59","BOESEGHEM":"59","BOIS GRENIER":"59","BOULOGNE SUR HELPE":"59","BOURSIES":"59","BOUSIES":"59","BOUSIGNIES":"59","BROUCKERQUE":"59","BROXEELE":"59","CAMPHIN EN PEVELE":"59","CAPPELLE EN PEVELE":"59","CARNIERES":"59","LE CATEAU CAMBRESIS":"59","CAUROIR":"59","LA CHAPELLE D ARMENTIERES":"59","CHERENG":"59","CHOISIES":"59","CONDE SUR L ESCAUT":"59","COUSOLRE":"59","COUTICHES":"59","CROIX CALUYAU":"59","DEULEMONT":"59","DOMPIERRE SUR HELPE":"59","DOUCHY LES MINES":"59","EECKE":"59","EMMERIN":"59","ERINGHEM":"59","ESCAUTPONT":"59","ESQUELBECQ":"59","FAMARS":"59","FAUMONT":"59","FEIGNIES":"59","FELLERIES":"59","FLAUMONT WAUDRECHIES":"59","FLERS EN ESCREBIEUX":"59","FLETRE":"59","FOREST SUR MARQUE":"59","FOURNES EN WEPPES":"59","FRESSAIN":"59","FRESSIES":"59","GOEULZIN":"59","LA GORGUE":"59","GOUZEAUCOURT":"59","GUESNAIN":"59","GUSSIGNIES":"59","HALLENNES LEZ HAUBOURDIN":"59","HAUBOURDIN":"59","HAUCOURT EN CAMBRESIS":"59","HECQ":"59","HERRIN":"59","HESTRUD":"59","HONNECOURT SUR ESCAUT":"59","JOLIMETZ":"59","LAMBRES LEZ DOUAI":"59","LANDAS":"59","LEDRINGHEM":"59","LESDAIN":"59","LEZ FONTAINE":"59","LOCQUIGNOL":"59","LYNDE":"59","LE MAISNIL":"59","MARCQ EN OSTREVENT":"59","MARQUILLIES":"59","MAUBEUGE":"59","MAULDE":"59","MAZINGHIEN":"59","MOROGUES":"18","MOULINS SUR YEVRE":"18","NERONDES":"18","NEUVY LE BARROIS":"18","NOHANT EN GRACAY":"18","OSMERY":"18","PARASSY":"18","PREUILLY":"18","PREVERANGES":"18","REIGNY":"18","ST GEORGES SUR MOULON":"18","ST HILAIRE DE COURT":"18","ST JEANVRIN":"18","STE SOLANGE":"18","SALIGNY LE VIF":"18","SANTRANGES":"18","LE SUBDRAY":"18","SURY PRES LERE":"18","SURY EN VAUX":"18","THENIOUX":"18","TORTERON":"18","VENESMES":"18","VERDIGNY":"18","VILLECELIN":"18","VILLENEUVE SUR CHER":"18","VORLY":"18","ARGENTAT":"19","AYEN":"19","BRANCEILLES":"19","BRIGNAC LA PLAINE":"19","LA CHAPELLE AUX BROCS":"19","LA CHAPELLE ST GERAUD":"19","CHARTRIER FERRIERE":"19","CHAVEROCHE":"19","CHENAILLER MASCHEIX":"19","COSNAC":"19","COUFFY SUR SARSONNE":"19","ESPAGNAC":"19","ESTIVAUX":"19","FEYT":"19","JUGEALS NAZARETH":"19","LAPLEAU":"19","LIGINIAC":"19","LIOURDRES":"19","LISSAC SUR COUZE":"19","LOUIGNAC":"19","MARGERIDES":"19","MENOIRE":"19","MEYMAC":"19","MEYSSAC":"19","MONESTIER MERLINES":"19","ORGNAC SUR VEZERE":"19","PALAZINGES":"19","PERET BEL AIR":"19","PEROLS SUR VEZERE":"19","ROSIERS DE JUILLAC":"19","ST CIRGUES LA LOUTRE":"19","ST ETIENNE LA GENESTE":"19","ST GERMAIN LES VERGNES":"19","ST HILAIRE PEYROUX":"19","ST HILAIRE TAURIEUX":"19","ST JULIEN MAUMONT":"19","STE MARIE LAPANOUZE":"19","ST MERD LES OUSSINES":"19","ST PARDOUX LA CROISILLE":"19","ST SOLVE":"19","ST YBARD":"19","SERILHAC":"19","SERVIERES LE CHATEAU":"19","SOUDEILLES":"19","THALAMY":"19","TROCHE":"19","TURENNE":"19","VARS SUR ROSEIX":"19","VITRAC SUR MONTANE":"19","AHUY":"21","AIGNAY LE DUC":"21","ALOXE CORTON":"21","AMPILLY LES BORDES":"21","ARCONCEY":"21","ARC SUR TILLE":"21","ARGILLY":"21","ARNAY LE DUC":"21","ASNIERES EN MONTAGNE":"21","AUBIGNY EN PLAINE":"21","AVOT":"21","BAGNOT":"21","RIBERAC":"24","ROUFFIGNAC DE SIGOULES":"24","ST ANTOINE D AUBEROCHE":"24","ST AUBIN DE CADELECH":"24","ST CAPRAISE D EYMET":"24","ST CYBRANET":"24","STE FOY DE BELVES":"24","STE FOY DE LONGAS":"24","ST FRONT SUR NIZONNE":"24","ST GERMAIN DU SALEMBRE":"24","ST GERMAIN ET MONS":"24","ST GEYRAC":"24","ST HILAIRE D ESTISSAC":"24","ST JORY DE CHALAIS":"24","ST MARCEL DU PERIGORD":"24","ST MARTIAL DE NABIRAT":"24","ST MAIME DE PEREYROL":"24","ST MICHEL DE VILLADEIX":"24","ST PANTALY D ANS":"24","ST PIERRE DE CHIGNAC":"24","ST PIERRE DE FRUGIE":"24","ST SEVERIN D ESTISSAC":"24","SALLES DE BELVES":"24","SERGEAC":"24","SERRES ET MONTGUYARD":"24","SOUDAT":"24","VALEUIL":"24","VEYRIGNAC":"24","VEYRINES DE DOMME":"24","VIEUX MAREUIL":"24","ABBANS DESSUS":"25","AISSEY":"25","APPENANS":"25","AUDEUX":"25","LES AUXONS":"25","BART":"25","BATTENANS LES MINES":"25","BLUSSANS":"25","BONDEVAL":"25","BRECONCHAUX":"25","BREMONDANS":"25","BREY ET MAISON DU BOIS":"25","BY":"25","CESSEY":"25","CHAMESOL":"25","CHAMPVANS LES MOULINS":"25","CHAPELLE D HUIN":"25","CHARMAUVILLERS":"25","CHAZOT":"25","CHEVIGNEY SUR L OGNON":"25","CHOUZELOT":"25","CROSEY LE GRAND":"25","CROUZET MIGETTE":"25","CUBRY":"25","CUSSEY SUR LISON":"25","CUSSEY SUR L OGNON":"25","DAMPIERRE LES BOIS":"25","DELUZ":"25","DEVECEY":"25","DOMPREL":"25","ECURCEY":"25","EPENOY":"25","EVILLERS":"25","FALLERANS":"25","FERRIERES LE LAC":"25","FERRIERES LES BOIS":"25","FEULE":"25","LES FINS":"25","LES FOURGS":"25","FRAMBOUHANS":"25","GENNES":"25","GOUHELANS":"25","GOUX SOUS LANDET":"25","LA GRANGE":"25","GRANGES NARBOZ":"25","LE GRATTERIS":"25","GUILLON LES BAINS":"25","HAUTERIVE LA FRESSE":"25","HERIMONCOURT":"25","L HOPITAL DU GROSBOIS":"25","L HOPITAL ST LIEFFROY":"25","HYEVRE MAGNY":"25","BREBOTTE":"90","CHATENOIS LES FORGES":"90","CRAVANCHE":"90","CUNELIERES":"90","DANJOUTIN":"90","DELLE":"90","DENNEY":"90","EVETTE SALBERT":"90","FELON":"90","LACHAPELLE SOUS ROUGEMONT":"90","MONTREUX CHATEAU":"90","PETITEFONTAINE":"90","REPPE":"90","ROMAGNY SOUS ROUGEMONT":"90","ROUGEGOUTTE":"90","VALDOIE":"90","ARPAJON":"91","BOIS HERPIN":"91","BOISSY LE CUTTE":"91","BONDOUFLE":"91","BOUSSY ST ANTOINE":"91","BRIERES LES SCELLES":"91","BURES SUR YVETTE":"91","CHILLY MAZARIN":"91","CORBEIL ESSONNES":"91","LE COUDRAY MONTCEAUX":"91","COURDIMANCHE SUR ESSONNE":"91","DANNEMOIS":"91","DRAVEIL":"91","ECHARCON":"91","FONTENAY LE VICOMTE":"91","ITTEVILLE":"91","LIMOURS":"91","LONGJUMEAU":"91","MORIGNY CHAMPIGNY":"91","ORVEAU":"91","PARAY VIEILLE POSTE":"91","PRUNAY SUR ESSONNE":"91","PUISELET LE MARAIS":"91","RIS ORANGIS":"91","ST PIERRE DU PERRAY":"91","CONGERVILLE THIONVILLE":"91","LE VAL ST GERMAIN":"91","VAYRES SUR ESSONNE":"91","VILLABE":"91","YERRES":"91","SERMAGES":"58","SUILLY LA TOUR":"58","TAZILLY":"58","TROIS VEVRES":"58","TRUCY L ORGUEILLEUX":"58","VANDENESSE":"58","VAUCLAIX":"58","VILLIERS SUR YONNE":"58","VITRY LACHE":"58","ANICHE":"59","ANZIN":"59","ARMBOUTS CAPPEL":"59","AUBENCHEUL AU BAC":"59","AUBERCHICOURT":"59","AULNOYE AYMERIES":"59","AWOINGT":"59","BANTEUX":"59","BEAUCAMPS LIGNY":"59","BEAUMONT EN CAMBRESIS":"59","BEAURAIN":"59","BEAUREPAIRE SUR SAMBRE":"59","BERELLES":"59","BERMERAIN":"59","BERMERIES":"59","BEUVRY LA FORET":"59","BRUILLE LEZ MARCHIENNES":"59","BRUILLE ST AMAND":"59","CAMPHIN EN CAREMBAULT":"59","CANTAING SUR ESCAUT":"59","CAPPELLE LA GRANDE":"59","CATILLON SUR SAMBRE":"59","COBRIEUX":"59","COUDEKERQUE BRANCHE":"59","CREVECOEUR SUR L ESCAUT":"59","CROCHTE":"59","CUINCY":"59","DEHERIES":"59","LE DOULIEU":"59","RANTZWILLER":"68","REININGUE":"68","RIMBACH PRES MASEVAUX":"68","RIXHEIM":"68","ROPPENTZWILLER":"68","STE MARIE AUX MINES":"68","SEPPOIS LE BAS":"68","STEINBRUNN LE BAS":"68","STEINSOULTZ":"68","STORCKENSOHN":"68","STOSSWIHR":"68","TURCKHEIM":"68","UEBERSTRASS":"68","WAHLBACH":"68","WALDIGHOFEN":"68","WALTENHEIM":"68","WEGSCHEID":"68","WESTHALTEN":"68","WICKERSCHWIHR":"68","WIDENSOLEN":"68","WILDENSTEIN":"68","WILLER":"68","AFFOUX":"69","LES ARDILLATS":"69","BRINDAS":"69","BRON":"69","BRUSSIEU":"69","CAILLOUX SUR FONTAINES":"69","CENVES":"69","CHAMBOST ALLIERES":"69","CHAMPAGNE AU MONT D OR":"69","CHASSAGNY":"69","CHAZAY D AZERGUES":"69","CHENAS":"69","CHENELETTE":"69","CLAVEISOLLES":"69","COURZIEU":"69","COUZON AU MONT D OR":"69","CURIS AU MONT D OR":"69","DRACE":"69","FLEURIEUX SUR L ARBRESLE":"69","IRIGNY":"69","LANCIE":"69","LARAJASSE":"69","LIMAS":"69","LONGESSAIGNE":"69","MONTROTTIER":"69","ODENAS":"69","LES OLMES":"69","OUROUX":"69","LE PERREON":"69","ROCHETAILLEE SUR SAONE":"69","RONNO":"69","SAIN BEL":"69","SOURCIEUX LES MINES":"69","ST ANDEOL LE CHATEAU":"69","ST ANDRE LA COTE":"69","ST CLEMENT DE VERS":"69","ST CYR SUR LE RHONE":"69","ST GENIS LES OLLIERES":"69","ST GERMAIN AU MONT D OR":"69","ST MARCEL L ECLAIRE":"69","ST ROMAIN AU MONT D OR":"69","TAPONAS":"69","THURINS":"69","LA TOUR DE SALVAGNY":"69","VILLIE MORGON":"69","VOURLES":"69","JONAGE":"69","MEYZIEU":"69","MONTANAY":"69","RILLIEUX LA PAPE":"69","SIMANDRES":"69","LYON 03":"69","ABONCOURT GESINCOURT":"70","AILLONCOURT":"70","AISEY ET RICHECOURT":"70","AMBIEVILLERS":"70","AMONCOURT":"70","MILLAM":"59","MONCHEAUX":"59","LA NEUVILLE":"59","NIERGNIES":"59","NIVELLE":"59","NOORDPEENE":"59","NOYELLES SUR SAMBRE":"59","PERENCHIES":"59","POTELLE":"59","PRESEAU":"59","PREUX AU SART":"59","PROUVY":"59","RAIMBEAUCOURT":"59","RONCQ":"59","ROSULT":"59","RUESNES":"59","RUMEGIES":"59","ST HILAIRE LEZ CAMBRAI":"59","ST PIERRE BROUCK":"59","SALESCHES":"59","SALOME":"59","SAULZOIR":"59","SOMAIN":"59","STEENVOORDE":"59","TEMPLEUVE EN PEVELE":"59","VERTAIN":"59","VRED":"59","WALLON CAPPEL":"59","WAMBAIX":"59","WAMBRECHIES":"59","WATTEN":"59","WERVICQ SUD":"59","WICRES":"59","WULVERDINGHE":"59","ACHY":"60","LES AGEUX":"60","ANSACQ":"60","ANSAUVILLERS":"60","APPILLY":"60","ARSY":"60","AUCHY LA MONTAGNE":"60","AVILLY ST LEONARD":"60","BAILLEUL SUR THERAIN":"60","BARGNY":"60","BEAUVAIS":"60","BONVILLERS":"60","BORAN SUR OISE":"60","BOREST":"60","BOUBIERS":"60","BOURSONNE":"60","BRAISNES SUR ARONDE":"60","BRIOT":"60","BURY":"60","CAISNES":"60","CAMPEAUX":"60","CAMPREMY":"60","CANNY SUR MATZ":"60","CATENOY":"60","CATHEUX":"60","CHAMBLY":"60","COIVREL":"60","CONCHY LES POTS":"60","COURTEUIL":"60","CREPY EN VALOIS":"60","CREVECOEUR LE PETIT":"60","CRILLON":"60","CUVERGNON":"60","DAMERAUCOURT":"60","DOMELIERS":"60","ENENCOURT LEAGE":"60","ENENCOURT LE SEC":"60","ERMENONVILLE":"60","ERNEMONT BOUTAVENT":"60","ESCAMES":"60","ESTREES ST DENIS":"60","LE FAY ST QUENTIN":"60","FLECHY":"60","FONTAINE ST LUCIEN":"60","FONTENAY TORCY":"60","FRESNEAUX MONTCHEVREUIL":"60","FRESNE LEGUILLON":"60","FRESNOY LE LUAT":"60","GLAIGNES":"60","GODENVILLERS":"60","GOINCOURT":"60","BARD LE REGULIER":"21","BEAUNOTTE":"21","BELAN SUR OURCE":"21","BILLY LES CHANCEAUX":"21","BISSEY LA PIERRE":"21","BLANCEY":"21","BLIGNY SUR OUCHE":"21","BONNENCONTRE":"21","BOUHEY":"21","BOUSSELANGE":"21","BRAIN":"21","BRESSEY SUR TILLE":"21","CHASSAGNE MONTRACHET":"21","CHASSEY":"21","CHAUME LES BAIGNEUX":"21","CHAUMONT LE BOIS":"21","CHEMIN D AISEY":"21","CHEVIGNY ST SAUVEUR":"21","CLENAY":"21","CLOMOT":"21","COLLONGES LES PREMIERES":"21","COMBLANCHIEN":"21","CORCELLES LES ARTS":"21","CORPOYER LA CHAPELLE":"21","COURBAN":"21","CRECEY SUR TILLE":"21","CRIMOLOIS":"21","CURTIL VERGY":"21","CUSSY LA COLONNE":"21","DAMPIERRE ET FLEE":"21","DRAMBON":"21","ECHEVRONNE":"21","ECHIGEY":"21","EGUILLY":"21","EPOISSES":"21","ETALANTE":"21","ETEVAUX":"21","ETROCHEY":"21","FAIN LES MOUTIERS":"21","FLAMMERANS":"21","FLAVIGNY SUR OZERAIN":"21","FLEUREY SUR OUCHE":"21","FONTAINES EN DUESMOIS":"21","FORLEANS":"21","GISSEY SOUS FLAVIGNY":"21","GLANON":"21","IS SUR TILLE":"21","IZEURE":"21","JOUEY":"21","LABERGEMENT FOIGNEY":"21","LACOUR D ARCENAY":"21","LAPERRIERE SUR SAONE":"21","MACONGE":"21","MAGNY SUR TILLE":"21","MARCENAY":"21","MARIGNY LES REULLEE":"21","MARTROIS":"21","MEILLY SUR ROUVRES":"21","MEUILLEY":"21","MISSERY":"21","MOLPHEY":"21","MONTBARD":"21","MONTBERTHAULT":"21","MONTIGNY MONTFORT":"21","MONTOILLOT":"21","MOREY ST DENIS":"21","NOD SUR SEINE":"21","NOGENT LES MONTBARD":"21","NUITS ST GEORGES":"21","PELLEREY":"21","PERNAND VERGELESSES":"21","PERRIGNY LES DIJON":"21","PICHANGES":"21","PRENOIS":"21","QUEMIGNY POISOT":"21","QUETIGNY":"21","RIEL LES EAUX":"21","LA ROCHEPOT":"21","LA ROCHE VANNEAU":"21","ROILLY":"21","ROUVRES EN PLAINE":"21","ST ANTHOT":"21","LANTENNE VERTIERE":"25","LAVANS QUINGEY":"25","LAVERNAY":"25","LA LONGEVILLE":"25","LORAY":"25","MAICHE":"25","MAMIROLLE":"25","MANCENANS":"25","LE MEMONT":"25","MESANDANS":"25","MESLIERES":"25","MONTBENOIT":"25","MONTGESOYE":"25","MONTMAHOUX":"25","MONTPERREUX":"25","MONTUSSAINT":"25","MOUTHIER HAUTE PIERRE":"25","NANS SOUS STE ANNE":"25","NOIREFONTAINE":"25","NOMMAY":"25","OLLANS":"25","PIERREFONTAINE LES VARANS":"25","POMPIERRE SUR DOUBS":"25","PONTARLIER":"25","RENNES SUR LOUE":"25","ROCHE LEZ BEAUPRE":"25","ROSIERES SUR BARBECHE":"25","LE RUSSEY":"25","ST GEORGES ARMONT":"25","ST GORGON MAIN":"25","ST POINT LAC":"25","SARAZ":"25","SOMBACOUR":"25","TARCENAY":"25","TRESSANDANS":"25","TROUVANS":"25","VALENTIGNEY":"25","VELESMES ESSARTS":"25","VELLEVANS":"25","VENISE":"25","VENNANS":"25","VERNOIS LES BELVOIR":"25","VERRIERES DE JOUX":"25","VIEUX CHARMONT":"25","VILLENEUVE D AMONT":"25","VILLERS SOUS CHALAMONT":"25","VILLERS SOUS MONTROND":"25","VORGES LES PINS":"25","ALLEX":"26","AUCELON":"26","AULAN":"26","AUTICHAMP":"26","LA BATIE ROLLAND":"26","BEAUMONT MONTEUX":"26","CHALANCON":"26","LE CHALON":"26","CHAMALOC":"26","CHANOS CURSON":"26","CHASTEL ARNAUD":"26","CHATEAUNEUF DU RHONE":"26","CHATUZANGE LE GOUBET":"26","CROZES HERMITAGE":"26","ECHEVIS":"26","FERRASSIERES":"26","GLANDAGE":"26","LE GRAND SERRE":"26","GRIGNAN":"26","GUMIANE":"26","LACHAU":"26","LEONCEL":"26","LESCHES EN DIOIS":"26","MANTHES":"26","MARCHES":"26","MARGES":"26","MARSANNE":"26","MIRABEL AUX BARONNIES":"26","MISCON":"26","MONTELEGER":"26","ECAILLON":"59","ECCLES":"59","ENGLEFONTAINE":"59","ENGLOS":"59","ENNETIERES EN WEPPES":"59","ERCHIN":"59","ERRE":"59","ESCAUDAIN":"59","ESQUERCHIN":"59","ETH":"59","FECHAIN":"59","FENAIN":"59","FERRIERE LA GRANDE":"59","FERRIERE LA PETITE":"59","FLESQUIERES":"59","FLOYON":"59","FOURMIES":"59","GENECH":"59","GOGNIES CHAUSSEE":"59","GRUSON":"59","HALLUIN":"59","HAULCHIN":"59","HEM LENGLET":"59","HERZEELE":"59","HORNAING":"59","HOUDAIN LEZ BAVAY":"59","HOYMILLE":"59","ILLIES":"59","IWUY":"59","JENLAIN":"59","LANNOY":"59","LAUWIN PLANQUE":"59","LESQUIN":"59","LIEU ST AMAND":"59","LOURCHES":"59","LOUVIL":"59","LOUVROIL":"59","MARBAIX":"59","MAROILLES":"59","MARPENT":"59","MASNIERES":"59","MASTAING":"59","MECQUIGNIES":"59","MOEUVRES":"59","MONCEAU ST WAAST":"59","MOUSTIER EN FAGNE":"59","NEUVILLE EN FERRAIN":"59","NEUVILLE ST REMY":"59","NEUVILLE SUR ESCAUT":"59","OBRECHIES":"59","OCHTEZEELE":"59","ORS":"59","PAILLENCOURT":"59","PREUX AU BOIS":"59","PRISCHES":"59","PROVIN":"59","QUERENAING":"59","LE QUESNOY":"59","QUESNOY SUR DEULE":"59","RADINGHEM EN WEPPES":"59","RAILLENCOURT STE OLLE":"59","RAINSARS":"59","RAMOUSIES":"59","RAUCOURT AU BOIS":"59","ROBERSART":"59","ROMBIES ET MARCHIPONT":"59","ROMERIES":"59","ST GEORGES SUR L AA":"59","ST MARTIN SUR ECAILLON":"59","ST REMY DU NORD":"59","ST SAULVE":"59","SECLIN":"59","LA SENTINELLE":"59","SOCX":"59","SOMMAING":"59","STEENBECQUE":"59","STRAZEELE":"59","TAISNIERES SUR HON":"59","TERDEGHEM":"59","THUMERIES":"59","THUN L EVEQUE":"59","TILLOY LEZ MARCHIENNES":"59","TOURCOING":"59","UXEM":"59","VENDEVILLE":"59","VERLINGHEM":"59","VIEUX RENG":"59","VILLERS EN CAUCHIES":"59","WAHAGNIES":"59","AMONT ET EFFRENEY":"70","ARBECEY":"70","ARC LES GRAY":"70","ARGILLIERES":"70","ARPENANS":"70","BARD LES PESMES":"70","BATTRANS":"70","BAUDONCOURT":"70","BETONCOURT ST PANCRAS":"70","BETONCOURT SUR MANCE":"70","BOUHANS LES MONTBOZON":"70","BOULT":"70","BOURSIERES":"70","BOUSSERAUCOURT":"70","BREUCHOTTE":"70","BROTTE LES LUXEUIL":"70","CENANS":"70","CERRE LES NOROY":"70","CHAMPTONNAY":"70","LA CHAPELLE LES LUXEUIL":"70","CHATENEY":"70","CHAUVIREY LE CHATEL":"70","CHAUVIREY LE VIEIL":"70","CHEVIGNEY":"70","CHOYE":"70","CINTREY":"70","COISEVAUX":"70","CONFLANDEY":"70","CORBENAY":"70","LA CORBIERE":"70","CORDONNET":"70","COULEVON":"70","COURTESOULT ET GATEY":"70","LA CREUSE":"70","CREVENEY":"70","CULT":"70","DAMPIERRE SUR LINOTTE":"70","DAMPIERRE SUR SALON":"70","DEMANGEVELLE":"70","ECHENOZ LA MELINE":"70","ECHENOZ LE SEC":"70","EQUEVILLEY":"70","ESMOULIERES":"70","ETRELLES ET LA MONTBLEUSE":"70","FEDRY":"70","FERRIERES LES RAY":"70","LES FESSEY":"70","FONTENOIS LES MONTBOZON":"70","FRANCOURT":"70","GOUHENANS":"70","GRAY LA VILLE":"70","HAUTEVELLE":"70","JASNEY":"70","JUSSEY":"70","LINEXERT":"70","LOMONT":"70","LA LONGINE":"70","LOULANS VERCHAMP":"70","MAGNY DANIGON":"70","MAILLERONCOURT CHARETTE":"70","MAILLERONCOURT ST PANCRAS":"70","MENOUX":"70","MERSUAY":"70","MOFFANS ET VACHERESSE":"70","MOIMAY":"70","MONTUREUX LES BAULAY":"70","NANTILLY":"70","NEUVELLE LES LA CHARITE":"70","LA NEUVELLE LES SCEY":"70","NOIDANS LES VESOUL":"70","OPPENANS":"70","ORMOICHE":"70","PALANTE":"70","PERCEY LE GRAND":"70","PESMES":"70","PLAINEMONT":"70","PLANCHER BAS":"70","PONT DU BOIS":"70","POYANS":"70","LA PROISELIERE ET LANGLE":"70","PURGEROT":"70","QUENOCHE":"70","LA GRANDE RESIE":"70","ROCHE ET RAUCOURT":"70","ST BROING":"70","GOLANCOURT":"60","GOURNAY SUR ARONDE":"60","GRANDFRESNOY":"60","GRANDRU":"60","HANVOILE":"60","HAUDIVILLERS":"60","HEILLES":"60","HODENC EN BRAY":"60","JAULZY":"60","JOUY SOUS THELLE":"60","LABERLIERE":"60","LAGNY LE SEC":"60","LALANDE EN SON":"60","LALANDELLE":"60","LEGLANTIERS":"60","LIANCOURT ST PIERRE":"60","MAREST SUR MATZ":"60","MAREUIL SUR OURCQ":"60","MARGNY LES COMPIEGNE":"60","MARGNY SUR MATZ":"60","MERU":"60","LE MESNIL CONTEVILLE":"60","MILLY SUR THERAIN":"60","MONCEAUX":"60","MONCHY HUMIERES":"60","MONDESCOURT":"60","MONTATAIRE":"60","MORLINCOURT":"60","MORTEFONTAINE EN THELLE":"60","MUIDORGE":"60","NAMPCEL":"60","NANTEUIL LE HAUDOUIN":"60","NEUILLY EN THELLE":"60","NOVILLERS":"60","OGNON":"60","OROER":"60","PARNES":"60","PIERREFITTE EN BEAUVAISIS":"60","LE PLESSIER SUR ST JUST":"60","LE PLESSIS BELLEVILLE":"60","PONCHON":"60","PRECY SUR OISE":"60","PREVILLERS":"60","RESSONS L ABBAYE":"60","RETHONDES":"60","ST AUBIN SOUS ERQUERY":"60","ST CREPIN AUX BOIS":"60","STE EUSOYE":"60","ST PIERRE LES BITRY":"60","ST SAMSON LA POTERIE":"60","ST VAAST LES MELLO":"60","LE SAULCHOY":"60","SAVIGNIES":"60","SERANS":"60","SOLENTE":"60","SONGEONS":"60","THERDONNE":"60","THIBIVILLERS":"60","THURY EN VALOIS":"60","TRACY LE MONT":"60","TRICOT":"60","TRIE LA VILLE":"60","TROUSSENCOURT":"60","VALDAMPIERRE":"60","VENETTE":"60","VER SUR LAUNETTE":"60","VIGNEMONT":"60","VILLEMBRAY":"60","VILLERS ST FRAMBOURG":"60","VILLERS VICOMTE":"60","VILLESELVE":"60","VINEUIL ST FIRMIN":"60","AUX MARAIS":"60","APPENAI SOUS BELLEME":"61","AUBRY EN EXMES":"61","AUNOU LE FAUCON":"61","BANVOU":"61","BELLAVILLIERS":"61","BELLEME":"61","BONSMOULINS":"61","LE BOSC RENOULT":"61","CARROUGES":"61","CHAHAINS":"61","LE CHALANGE":"61","CHAMPEAUX SUR SARTHE":"61","CHAMP HAUT":"61","ST BROING LES MOINES":"21","ST HELIER":"21","ST MARTIN DE LA MER":"21","ST SEINE EN BACHE":"21","SALMAISE":"21","SAVIGNY LES BEAUNE":"21","SAVILLY":"21","SAVOUGES":"21","SEGROIS":"21","SEMEZANGES":"21","SENAILLY":"21","SEURRE":"21","TART LE BAS":"21","THOSTE":"21","TORCY ET POULIGNY":"21","TOUILLON":"21","TROCHERES":"21","TURCEY":"21","URCY":"21","VANDENESSE EN AUXOIS":"21","VANNAIRE":"21","VERNOT":"21","VERTAULT":"21","VIC SOUS THIL":"21","VILLAINES EN DUESMOIS":"21","VILLAINES LES PREVOTES":"21","VILLEFERRY":"21","VOUGEOT":"21","BRINGOLO":"22","BROONS":"22","BRUSVILY":"22","CALANHEL":"22","LE CAMBOUT":"22","CAVAN":"22","CHATELAUDREN":"22","LA CHEZE":"22","COHINIAC":"22","EVRAN":"22","GAUSSON":"22","GUENROC":"22","GUITTE":"22","GURUNHUEL":"22","LA HARMOYE":"22","HENGOAT":"22","ILLIFAUT":"22","LANGAST":"22","LANGUENAN":"22","LANNEBERT":"22","LANRELAS":"22","LE LESLAY":"22","LOC ENVEL":"22","LOSCOUET SUR MEU":"22","MEGRIT":"22","PABU":"22","PENGUILY":"22","PLAINE HAUTE":"22","PLENEUF VAL ANDRE":"22","PLERNEUF":"22","PLEUMEUR GAUTIER":"22","PLEVENON":"22","PLOEZAL":"22","PLOREC SUR ARGUENON":"22","PLOUFRAGAN":"22","PLOUGUENAST":"22","PLOURIVO":"22","PLUSQUELLEC":"22","PLUSSULIEN":"22","PONTRIEUX":"22","PRAT":"22","QUEMPERVEN":"22","QUINTENIC":"22","MONTFROC":"26","MONTJOUX":"26","MONTMAUR EN DIOIS":"26","MONTOISON":"26","MONTRIGAUD":"26","MORNANS":"26","MUREILS":"26","PEYRINS":"26","PIERRELONGUE":"26","LE POET CELARD":"26","POMMEROL":"26","REAUVILLE":"26","LA ROCHETTE DU BUIS":"26","ROTTIER":"26","ROUSSAS":"26","ST BONNET DE VALCLERIEUX":"26","ST LAURENT D ONAY":"26","ST MARCEL LES VALENCE":"26","ST MAY":"26","ST PAUL TROIS CHATEAUX":"26","SOLERIEUX":"26","SOYANS":"26","SUZE LA ROUSSE":"26","TAULIGNAN":"26","LES TONILS":"26","TRIORS":"26","VERS SUR MEOUGE":"26","GERVANS":"26","ALIZAY":"27","AMFREVILLE SOUS LES MONTS":"27","ANDE":"27","AULNAY SUR ITON":"27","AUTHEVERNES":"27","BACQUEPUIS":"27","LE BOIS HELLAIN":"27","BOIS JEROME ST OUEN":"27","BREUX SUR AVRE":"27","CALLEVILLE":"27","CANAPPEVILLE":"27","CAUVERVILLE EN ROUMOIS":"27","LA CHAPELLE BAYVEL":"27","LA CHAPELLE DU BOIS DES FAULX":"27","LA CHAPELLE REANVILLE":"27","CLAVILLE":"27","CONDE SUR RISLE":"27","CONNELLES":"27","CORNY":"27","COURCELLES SUR SEINE":"27","COURTEILLES":"27","DARDEZ":"27","EMALLEVILLE":"27","EZY SUR EURE":"27","FAINS":"27","LE FIDELAIRE":"27","FORT MOVILLE":"27","FOULBEC":"27","FRENEUSE SUR RISLE":"27","GAILLARDBOIS CRESSENVILLE":"27","GAUCIEL":"27","GIVERNY":"27","GLOS SUR RISLE":"27","GRAVERON SEMERVILLE":"27","GUERNY":"27","L HABIT":"27","HARCOURT":"27","LA HAYE DE ROUTOT":"27","HECTOMARE":"27","LA HEUNIERE":"27","HUEST":"27","INCARVILLE":"27","IRREVILLE":"27","LYONS LA FORET":"27","WARGNIES LE GRAND":"59","WARHEM":"59","WARNETON":"59","WASQUEHAL":"59","WATTIGNIES":"59","WATTRELOS":"59","WAVRIN":"59","WIGNEHIES":"59","WORMHOUT":"59","ZUYTPEENE":"59","AGNETZ":"60","AUMONT EN HALATTE":"60","AUTHEUIL EN VALOIS":"60","AUTRECHES":"60","AVRECHY":"60","BACHIVILLERS":"60","BACOUEL":"60","BELLE EGLISE":"60","BERNEUIL EN BRAY":"60","BERNEUIL SUR AISNE":"60","BONNEUIL LES EAUX":"60","BOULOGNE LA GRASSE":"60","BRENOUILLE":"60","BUICOURT":"60","CAMBRONNE LES CLERMONT":"60","CAUFFRY":"60","CAUVIGNY":"60","CEMPUIS":"60","CHANTILLY":"60","CHEVINCOURT":"60","CLAIROIX":"60","COURCELLES EPAYELLES":"60","CROUY EN THELLE":"60","DARGIES":"60","ELINCOURT STE MARGUERITE":"60","EMEVILLE":"60","ERQUINVILLERS":"60","ESSUILES":"60","ETAVIGNY":"60","EVE":"60","FLAVACOURT":"60","FONTAINE CHAALIS":"60","FONTAINE LAVAGANNE":"60","FOUQUEROLLES":"60","FRANCASTEL":"60","FRENICHES":"60","FRESNOY LA RIVIERE":"60","FROCOURT":"60","FROISSY":"60","GENVRY":"60","GERBEROY":"60","GOUVIEUX":"60","GUISCARD":"60","GURY":"60","HADANCOURT LE HAUT CLOCHER":"60","HAUTBOS":"60","HAUTEFONTAINE":"60","HOUDANCOURT":"60","IVRY LE TEMPLE":"60","LACHELLE":"60","LACROIX ST OUEN":"60","LASSIGNY":"60","LHERAULE":"60","LIBERMONT":"60","LIERVILLE":"60","LIEUVILLERS":"60","LORMAISON":"60","MAIMBEVILLE":"60","MAISONCELLE ST PIERRE":"60","MARSEILLE EN BEAUVAISIS":"60","MELLO":"60","LE MESNIL SUR BULLES":"60","MOLIENS":"60","MONTJAVOULT":"60","MONTLOGNON":"60","LE MONT ST ADRIEN":"60","MORANGLES":"60","MORY MONTCRUX":"60","MUIRANCOURT":"60","NEUFCHELLES":"60","NEUILLY SOUS CLERMONT":"60","LA NEUVILLE D AUMONT":"60","LA NEUVILLE EN HEZ":"60","LA NEUVILLE ROY":"60","LA NEUVILLE VAULT":"60","ST LOUP NANTOUARD":"70","SAULX":"70","SAUVIGNEY LES GRAY":"70","SCYE":"70","SERVIGNEY":"70","SEVEUX":"70","VALLEROIS LE BOIS":"70","VELLEFRIE":"70","VELLEMINFROY":"70","VELLEMOZ":"70","VESOUL":"70","VILLAFANS":"70","VILLERS LA VILLE":"70","VY LES FILAIN":"70","L ABERGEMENT DE CUISERY":"71","ALLEREY SUR SAONE":"71","AMANZE":"71","AMEUGNY":"71","ANGLURE SOUS DUN":"71","ANOST":"71","ANTULLY":"71","BEAUMONT SUR GROSNE":"71","LES BIZOTS":"71","LA BOULAYE":"71","BRESSE SUR GROSNE":"71","BROYE":"71","CERSOT":"71","CHALON SUR SAONE":"71","CHAMPFORGEUIL":"71","LA CHAPELLE AU MANS":"71","LA CHAPELLE DE BRAGNY":"71","LA CHAPELLE DU MONT DE FRANCE":"71","CHASSIGNY SOUS DUN":"71","CHATEAU":"71","CHEILLY LES MARANGES":"71","COLLONGE EN CHAROLLAIS":"71","COLLONGE LA MADELEINE":"71","CORMATIN":"71","CORTAMBERT":"71","CURTIL SOUS BURNAND":"71","DEVROUZE":"71","ESSERTENNE":"71","FRETTERANS":"71","GENELARD":"71","LA GENETE":"71","GIBLES":"71","LA GRANDE VERRIERE":"71","GRURY":"71","GUEUGNON":"71","L HOPITAL LE MERCIER":"71","JONCY":"71","LAIVES":"71","LAIZE":"71","LESME":"71","LESSARD EN BRESSE":"71","LONGEPIERRE":"71","LOURNAND":"71","LUCENAY L EVEQUE":"71","MALAY":"71","MARCILLY LA GUEURCE":"71","LE ROUSSET MARIZY":"71","MENETREUIL":"71","MERCUREY":"71","MILLY LAMARTINE":"71","MONTCOY":"71","MOREY":"71","NOCHIZE":"71","OUROUX SOUS LE BOIS STE MARIE":"71","PALLEAU":"71","PERRIGNY SUR LOIRE":"71","LE PLANOIS":"71","PLOTTES":"71","RANCY":"71","ROMANECHE THORINS":"71","ROMENAY":"71","SAILLENARD":"71","CONDE SUR SARTHE":"61","COUDEHARD":"61","DAMIGNY":"61","DURCET":"61","ECHAUFFOUR":"61","EXMES":"61","LA GONFRIERE":"61","GOULET":"61","GUEPREI":"61","HABLOVILLE":"61","IRAI":"61","JOUE DU BOIS":"61","JUVIGNY SUR ORNE":"61","LALACELLE":"61","LANDISACQ":"61","LIVAIE":"61","LOUGE SUR MAIRE":"61","MARCHEMAISONS":"61","MEHOUDIN":"61","LE MENIL BROUT":"61","LE MENIL CIBOULT":"61","MENIL GONDOUIN":"61","MONTGAROULT":"61","MONTILLY SUR NOIREAU":"61","MONT ORMEL":"61","MOUTIERS AU PERCHE":"61","NEUILLY LE BISSON":"61","LE PIN AU HARAS":"61","RI":"61","ROIVILLE":"61","ST CENERI LE GEREI":"61","STE CERONNE LES MORTAGNE":"61","ST FRAIMBAULT":"61","ST GERMAIN LE VIEUX":"61","STE HONORINE LA CHARDONNE":"61","ST LAMBERT SUR DIVE":"61","ST NICOLAS DE SOMMAIRE":"61","ST OUEN LE BRISOULT":"61","ST PIERRE DES LOGES":"61","ST PIERRE LA BRUYERE":"61","ST QUENTIN DE BLAVOU":"61","ST SAUVEUR DE CARROUGES":"61","ST SULPICE SUR RISLE":"61","ST SYMPHORIEN DES BRUYERES":"61","LES MONTS D ANDAINE":"61","SEMALLE":"61","SILLY EN GOUFFERN":"61","TANQUES":"61","TELLIERES LE PLESSIS":"61","TREMONT":"61","LES VENTES DE BOURSE":"61","VILLEBADIN":"61","LES YVETEAUX":"61","ACHIET LE GRAND":"62","ACHIET LE PETIT":"62","AIRE SUR LA LYS":"62","ALEMBON":"62","ALQUINES":"62","AMBRINES":"62","ANVIN":"62","AUBROMETZ":"62","AUCHY LES HESDIN":"62","AUDREHEM":"62","AUDRUICQ":"62","AUTINGUES":"62","AUXI LE CHATEAU":"62","AVESNES LES BAPAUME":"62","BANCOURT":"62","BEAUFORT BLAVINCOURT":"62","BERCK":"62","BEUGNATRE":"62","BEUSSENT":"62","BEUVRY":"62","BEZINGHEM":"62","BILLY MONTIGNY":"62","LA ROCHE DERRIEN":"22","ROSPEZ":"22","ST ADRIEN":"22","ST CARADEC":"22","ST CARNE":"22","ST DONAN":"22","ST LAUNEUC":"22","ST MADEN":"22","ST MARTIN DES PRES":"22","ST MAUDEZ":"22","ST NICOLAS DU PELEM":"22","ST POTAN":"22","STE TREPHINE":"22","ST TRIMOEL":"22","TREBEDAN":"22","TREDANIEL":"22","TREGOMEUR":"22","TREGONNEAU":"22","TREOGAN":"22","TREVENEUC":"22","TREVOU TREGUIGNEC":"22","UZEL":"22","LA VICOMTE SUR RANCE":"22","LE VIEUX BOURG":"22","LE VIEUX MARCHE":"22","VILDE GUINGALAN":"22","YFFINIAC":"22","YVIAS":"22","ARFEUILLE CHATAIN":"23","AUZANCES":"23","BAZELAT":"23","BENEVENT L ABBAYE":"23","BLESSAC":"23","BONNAT":"23","LA BRIONNE":"23","BUDELIERE":"23","BUSSIERE DUNOISE":"23","BUSSIERE NOUVELLE":"23","CEYROUX":"23","CHAMBERAUD":"23","LA CHAUSSADE":"23","CLUGNAT":"23","CROZANT":"23","FRANSECHES":"23","GIOUX":"23","MALVAL":"23","MANSAT LA COURRIERE":"23","LES MARS":"23","LA MAZIERE AUX BONS HOMMES":"23","LE MONTEIL AU VICOMTE":"23","MORTROUX":"23","PEYRAT LA NONIERE":"23","PIONNAT":"23","ST MARTIAL LE MONT":"23","ST MARTIN CHATEAU":"23","ST MOREIL":"23","ST PARDOUX MORTEROLLES":"23","ST PARDOUX LES CARDS":"23","ST PIERRE DE FURSAC":"23","ST PRIEST PALUS":"23","ST QUENTIN LA CHABANNE":"23","ST SILVAIN BAS LE ROC":"23","ST YRIEIX LA MONTAGNE":"23","THAURON":"23","TOULX STE CROIX":"23","VERNEIGES":"23","VIERSAT":"23","ABJAT SUR BANDIAT":"24","ANNESSE ET BEAULIEU":"24","AUBAS":"24","AUDRIX":"24","BANEUIL":"24","BARDOU":"24","BERTRIC BUREE":"24","LA BOISSIERE D ANS":"24","BOSSET":"24","BRANTOME EN PERIGORD":"24","LE BUGUE":"24","BUSSIERE BADIL":"24","CAPDROT":"24","CARSAC DE GURSON":"24","CHALAGNAC":"24","CHAMPCEVINEL":"24","CHAMPNIERS ET REILHAC":"24","LA CHAPELLE MONTMOREAU":"24","MAINNEVILLE":"27","MANNEVILLE LA RAOULT":"27","MARCILLY SUR EURE":"27","MELICOURT":"27","MENILLES":"27","LE MESNIL JOURDAIN":"27","MESNIL SOUS VIENNE":"27","MOISVILLE":"27","MOUFLAINES":"27","NOARDS":"27","NONANCOURT":"27","LE NOYER EN OUCHE":"27","PIENCOURT":"27","PORTE JOIE":"27","PORT MORT":"27","LA POTERIE MATHIEU":"27","PULLAY":"27","ROUTOT":"27","RUGLES":"27","ST AUBIN SUR GAILLON":"27","ST AUBIN SUR QUILLEBEUF":"27","ST CHRISTOPHE SUR AVRE":"27","ST CLAIR D ARCEY":"27","STE COLOMBE PRES VERNON":"27","ST GEORGES DU MESNIL":"27","ST GEORGES DU VIEVRE":"27","ST GEORGES MOTEL":"27","ST GERMAIN DE PASQUIER":"27","ST GERMAIN DES ANGLES":"27","VILLECHETIF":"10","LA VILLENEUVE AU CHATELOT":"10","VENDEUVRE SUR BARSE":"10","VILLIERS HERBISSE":"10","LA VENDUE MIGNOT":"10","VAUCHONVILLIERS":"10","VILLE SUR ARCE":"10","VILLELOUP":"10","VILLACERF":"10","CAMPAGNE SUR AUDE":"11","BELVEZE DU RAZES":"11","BELCASTEL ET BUC":"11","BIZE MINERVOIS":"11","BOUILHONNAC":"11","LES CASSES":"11","LA BEZOLE":"11","BOURIEGE":"11","CAILHAU":"11","BLOMAC":"11","VILLIERS LE BOIS":"10","ALBIERES":"11","BELCAIRE":"11","VULAINES":"10","BARBAIRA":"11","ARMISSAN":"11","ALAIRAC":"11","BADENS":"11","VAL DE LAMBRONNE":"11","CAUNETTES EN VAL":"11","CAUNES MINERVOIS":"11","CAZALRENOUX":"11","COUNOZOULS":"11","CASTELRENG":"11","COUIZA":"11","VALLANT ST GEORGES":"10","THENNELIERES":"10","LA SAULSOTTE":"10","UNIENVILLE":"10","TRANCAULT":"10","SOMMEVAL":"10","TROYES":"10","MORTAGNE SUR SEVRE":"85","MOUILLERON ST GERMAIN":"85","MOUZEUIL ST MARTIN":"85","NIEUL LE DOLENT":"85","NIEUL SUR L AUTISE":"85","NOIRMOUTIER EN L ILE":"85","NOTRE DAME DE MONTS":"85","PISSOTTE":"85","REAUMUR":"85","NOTRE DAME DE RIEZ":"85","LA ROCHE SUR YON":"85","ROCHETREJOUX":"85","ST AUBIN LA PLAINE":"85","ST DENIS LA CHEVASSE":"85","ST ETIENNE DE BRILLOUET":"85","ST HILAIRE DE RIEZ":"85","ST HILAIRE LE VOUHIS":"85","NOYERS ST MARTIN":"60","NOYON":"60","ORROUY":"60","PAILLART":"60","PEROY LES GOMBRIES":"60","PLAINVAL":"60","PONTARME":"60","PONTOISE LES NOYON":"60","PORCHEUX":"60","PUISEUX LE HAUBERGER":"60","QUESMY":"60","REEZ FOSSE MARTIN":"60","REMECOURT":"60","ROBERVAL":"60","ROMESCAMPS":"60","ROUVROY LES MERLES":"60","SAINS MORAINVILLERS":"60","ST JUST EN CHAUSSEE":"60","SERIFONTAINE":"60","SERMAIZE":"60","SUZOY":"60","THIVERNY":"60","TRACY LE VAL":"60","TROUSSURES":"60","VERDERONNE":"60","VILLERS ST BARTHELEMY":"60","VILLERS ST GENEST":"60","VILLERS SUR TRIE":"60","VILLERS VERMONT":"60","WAMBEZ":"60","WARLUIS":"60","WAVIGNIES":"60","AUBRY LE PANTHOU":"61","AVERNES ST GOURGON":"61","BAZOCHES AU HOULME":"61","BRIOUZE":"61","CHAMPCERIE":"61","LE CHAMP DE LA PIERRE":"61","LE CHATEAU D ALMENECHES":"61","CORBON":"61","COULONCES":"61","COURGEOUT":"61","CRAMENIL":"61","EPERRAIS":"61","LA GENEVRAIE":"61","GIEL COURTEILLES":"61","JOUE DU PLAIN":"61","L AIGLE":"61","LA LANDE DE LOUGE":"61","LE MAGE":"61","MENIL FROGER":"61","LE MENIL GUYON":"61","MENIL HUBERT EN EXMES":"61","LE MENIL VICOMTE":"61","LE MERLERAULT":"61","MONTMERREI":"61","MORTREE":"61","MOULINS LA MARCHE":"61","MOULINS SUR ORNE":"61","MOUSSONVILLIERS":"61","OMMEEL":"61","LE PIN LA GARENNE":"61","POINTEL":"61","PONTCHARDON":"61","POUVRAI":"61","RAI":"61","RESENLIEU":"61","ROUPERROUX":"61","SAI":"61","ST ANDRE DE BRIOUZE":"61","ST AQUILIN DE CORBION":"61","ST AUBIN EN CHAROLLAIS":"71","ST BERAIN SUR DHEUNE":"71","ST BOIL":"71","ST BONNET DE JOUX":"71","ST EMILAND":"71","ST GENGOUX DE SCISSE":"71","ST GERMAIN DU PLAIN":"71","ST IGNY DE ROCHE":"71","ST JEAN DE TREZY":"71","ST JULIEN SUR DHEUNE":"71","ST LEGER LES PARAY":"71","ST LEGER SOUS LA BUSSIERE":"71","MAZIERES EN GATINE":"79","MESSE":"79","MONTRAVERS":"79","PRAHECQ":"79","PRAILLES":"79","LE RETAIL":"79","ST AMAND SUR SEVRE":"79","ST CHRISTOPHE SUR ROC":"79","ST HILAIRE LA PALUD":"79","ST LIN":"79","STE NEOMAYE":"79","ST POMPAIN":"79","ST ROMANS LES MELLE":"79","THORIGNE":"79","VALLANS":"79","LE VERT":"79","VIENNAY":"79","AIRAINES":"80","AIZECOURT LE BAS":"80","ANDECHY":"80","ARREST":"80","ASSAINVILLERS":"80","AUCHONVILLERS":"80","AUMATRE":"80","AUTHIEULE":"80","AVELUY":"80","BAIZIEUX":"80","BEAUCAMPS LE JEUNE":"80","BEAUCAMPS LE VIEUX":"80","BEAUMETZ":"80","BEAUMONT HAMEL":"80","BEAUQUESNE":"80","BELLOY EN SANTERRE":"80","BERNATRE":"80","BERNAY EN PONTHIEU":"80","BERTEAUCOURT LES DAMES":"80","BIENCOURT":"80","BLANGY SOUS POIX":"80","BOSQUEL":"80","BOUCHOIR":"80","BOUILLANCOURT LA BATAILLE":"80","BOURSEVILLE":"80","BOUTTENCOURT":"80","BOVELLES":"80","BRACHES":"80","BRAILLY CORNEHOTTE":"80","BRESLE":"80","BUIGNY LES GAMACHES":"80","CHAMPIEN":"80","CHIRMONT":"80","CONTY":"80","COULONVILLERS":"80","COURCELETTE":"80","CREMERY":"80","CROIXRAULT":"80","LE CROTOY":"80","CURCHY":"80","DANCOURT POPINCOURT":"80","DOMART SUR LA LUCE":"80","DOMINOIS":"80","DOMLEGER LONGVILLERS":"80","DOMPIERRE BECQUINCOURT":"80","DOMQUEUR":"80","DOULLENS":"80","ENNEMAIN":"80","EPAGNE EPAGNETTE":"80","EPLESSIER":"80","EQUENNES ERAMECOURT":"80","ERCHES":"80","ERCHEU":"80","ESTREES SUR NOYE":"80","BLESSY":"62","BLINGEL":"62","BOULOGNE SUR MER":"62","BOYAVAL":"62","BRIAS":"62","BUCQUOY":"62","BUIRE AU BOIS":"62","BUIRE LE SEC":"62","BURBURE":"62","BUSNES":"62","CAGNICOURT":"62","CAMBLAIN CHATELAIN":"62","CAMPAGNE LES WARDRECQUES":"62","LA CAUCHIE":"62","CHERIENNES":"62","CLERQUES":"62","CLETY":"62","COLEMBERT":"62","CONDETTE":"62","CONTEVILLE EN TERNOIS":"62","COUIN":"62","COUPELLE NEUVE":"62","CREMAREST":"62","CROISETTE":"62","DANNES":"62","DIVION":"62","DOHEM":"62","DOURGES":"62","ECLIMEUX":"62","ECQUEDECQUES":"62","ECURIE":"62","EPERLECQUES":"62","EQUIHEN PLAGE":"62","ERVILLERS":"62","ETRUN":"62","FAVREUIL":"62","FEBVIN PALFART":"62","FESTUBERT":"62","FICHEUX":"62","FRAMECOURT":"62","FRETHUN":"62","GOUVES":"62","GOUY ST ANDRE":"62","GOUY SOUS BELLONNE":"62","GROFFLIERS":"62","HABARCQ":"62","HAISNES":"62","HAMBLAIN LES PRES":"62","HARNES":"62","HENU":"62","HERBINGHEN":"62","LA HERLIERE":"62","HESTRUS":"62","HEZECQUES":"62","HOULLE":"62","LACRES":"62","LEBIEZ":"62","LESTREM":"62","LEUBRINGHEN":"62","LIGNY LES AIRE":"62","LILLERS":"62","LA LOGE":"62","LA MADELAINE SOUS MONTREUIL":"62","MAGNICOURT EN COMTE":"62","MARCONNE":"62","MARESQUEL ECQUEMICOURT":"62","MARLES LES MINES":"62","MARTINPUICH":"62","MATRINGHEM":"62","MONCHY AU BOIS":"62","MONDICOURT":"62","MONTENESCOURT":"62","NEUVILLE ST VAAST":"62","NIELLES LES BLEQUIN":"62","NORDAUSQUES":"62","OBLINGHEM":"62","PENIN":"62","PERNES":"62","POLINCOVE":"62","POMMERA":"62","PONT A VENDIN":"62","QUESQUES":"62","RECLINGHEM":"62","RIENCOURT LES BAPAUME":"62","RINXENT":"62","ROEUX":"62","ROLLANCOURT":"62","LA CHAPELLE ST JEAN":"24","CHOURGNAC":"24","CLADECH":"24","COMBERANCHE ET EPELUCHE":"24","CONNE DE LABARDE":"24","DOISSAT":"24","LA DORNAC":"24","EGLISE NEUVE D ISSAC":"24","EYLIAC":"24","COURTAULY":"11","LA COURTETE":"11","LA DIGNE D AVAL":"11","FA":"11","FAJAC LA RELENQUE":"11","FENOUILLET DU RAZES":"11","FERRAN":"11","FLOURE":"11","FONTIES D AUDE":"11","FOURTOU":"11","FRAISSE CABARDES":"11","GARDIE":"11","GINOLES":"11","LES ILHES":"11","LABECEDE LAURAGAIS":"11","LAFAGE":"11","LAIRIERE":"11","LAROQUE DE FA":"11","LEZIGNAN CORBIERES":"11","LUC SUR AUDE":"11","LUC SUR ORBIEU":"11","MALVES EN MINERVOIS":"11","MALVIES":"11","MAS DES COURS":"11","MONTFERRAND":"11","NIORT DE SAULT":"11","PORT LA NOUVELLE":"11","ORNAISONS":"11","OUVEILLAN":"11","PALAJA":"11","PEXIORA":"11","PUGINIER":"11","RENNES LES BAINS":"11","RIBAUTE":"11","RIBOUISSE":"11","ROUTIER":"11","ST FRICHOUX":"11","ST MARTIN DE VILLEREGLAN":"11","ST MARTIN LE VIEIL":"11","ST NAZAIRE D AUDE":"11","SALVEZINES":"11","SONNAC SUR L HERS":"11","TAURIZE":"11","TERROLES":"11","TOUROUZELLE":"11","TREILLES":"11","TREVILLE":"11","VENTENAC CABARDES":"11","VERAZA":"11","VILLEBAZY":"11","VILLESEQUE DES CORBIERES":"11","VILLESEQUELANDE":"11","VILLESISCLE":"11","ALRANCE":"12","ARNAC SUR DOURDOU":"12","ARVIEU":"12","BERTHOLENE":"12","BESSUEJOULS":"12","BOISSE PENCHOT":"12","BOZOULS":"12","BRUSQUE":"12","CAMBOULAZET":"12","CANET DE SALARS":"12","CASSAGNES BEGONHES":"12","COMPEYRE":"12","CORNUS":"12","COUBISOU":"12","LA COUVERTOIRADE":"12","CRANSAC":"12","LA CRESSE":"12","ESPALION":"12","FLAGNAC":"12","L HOSPITALET DU LARZAC":"12","HUPARLAC":"12","ST JUIRE CHAMPGILLON":"85","ST PAUL EN PAREDS":"85","STE PEXINE":"85","ST VINCENT STERLANGES":"85","SALLERTAINE":"85","TALLUD STE GEMME":"85","THIRE":"85","THOUARSAIS BOUILDROUX":"85","TIFFAUGES":"85","VELLUIRE":"85","LA VERRIE":"85","VOUVANT":"85","ASLONNES":"86","AVAILLES EN CHATELLERAULT":"86","AVAILLES LIMOUZINE":"86","BELLEFONDS":"86","CELLE LEVESCAULT":"86","CHABOURNAY":"86","CHALANDRAY":"86","CHATEAU LARCHER":"86","CISSE":"86","COUSSAY":"86","CUHON":"86","LA GRIMAUDIERE":"86","ITEUIL":"86","JARDRES":"86","JAUNAY CLAN":"86","JOUSSE":"86","LATHUS ST REMY":"86","LEIGNE SUR USSEAU":"86","MAIRE":"86","MAISONNEUVE":"86","MASSOGNES":"86","MAZEUIL":"86","MESSEME":"86","MILLAC":"86","MONTREUIL BONNIN":"86","MOUTERRE SILLY":"86","NAINTRE":"86","ORCHES":"86","OUZILLY":"86","PAIZAY LE SEC":"86","PAYROUX":"86","RANTON":"86","ROCHES PREMARIE ANDILLE":"86","ROIFFE":"86","ST GAUDENT":"86","ST GEORGES LES BAILLARGEAUX":"86","ST PIERRE DE MAILLE":"86","ST PIERRE D EXIDEUIL":"86","SAMMARCOLLES":"86","USSON DU POITOU":"86","VELLECHES":"86","VOULON":"86","VOUNEUIL SUR VIENNE":"86","AIXE SUR VIENNE":"87","AUGNE":"87","AZAT LE RIS":"87","BEYNAC":"87","BOSMIE L AIGUILLE":"87","LE BUIS":"87","BUJALEUF":"87","LES CARS":"87","CHAMPSAC":"87","LA CHAPELLE MONTBRANDEIX":"87","LE CHATENET EN DOGNON":"87","COMPREIGNAC":"87","LA CROIX SUR GARTEMPE":"87","DOMPS":"87","LE DORAT":"87","EYBOULEUF":"87","EYJEAUX":"87","LADIGNAC LE LONG":"87","LUSSAC LES EGLISES":"87","MONTROL SENARD":"87","MORTEMART":"87","LE PALAIS SUR VIENNE":"87","PANAZOL":"87","ST DENIS SUR SARTHON":"61","ST GEORGES DES GROSEILLERS":"61","ST HILAIRE LA GERARD":"61","ST HILAIRE LE CHATEL":"61","STE HONORINE LA GUILLAUME":"61","ST MARS D EGRENNE":"61","ST MARTIN DU VIEUX BELLEME":"61","STE OPPORTUNE":"61","ST OUEN DE SECHEROUVRE":"61","ST OUEN SUR ITON":"61","ST PIERRE DU REGARD":"61","STE SCOLASSE SUR SARTHE":"61","SAIRES LA VERRERIE":"61","SEVRAI":"61","SOLIGNY LA TRAPPE":"61","TANVILLE":"61","TESSE FROULAY":"61","TORCHAMP":"61","LA TRINITE DES LAITIERS":"61","VALFRAMBERT":"61","AFFRINGUES":"62","AGNIERES":"62","AGNY":"62","AIRON NOTRE DAME":"62","AIX NOULETTE":"62","AMES":"62","ANDRES":"62","ARRAS":"62","LES ATTAQUES":"62","ATTIN":"62","AUCHY LES MINES":"62","AUDINGHEN":"62","AUDRESSELLES":"62","AVION":"62","AVROULT":"62","BAILLEULVAL":"62","BAINCTHUN":"62","BAJUS":"62","BALINGHEM":"62","BEAUDRICOURT":"62","BEAUMERIE ST MARTIN":"62","BELLONNE":"62","BERNEVILLE":"62","BLENDECQUES":"62","BLEQUIN":"62","BOIS BERNARD":"62","BOISJEAN":"62","BOISLEUX ST MARC":"62","BONNINGUES LES CALAIS":"62","BOUQUEHAULT":"62","CALONNE SUR LA LYS":"62","CAMPAGNE LES BOULONNAIS":"62","CAMPIGNEULLES LES PETITES":"62","CANLERS":"62","CAPELLE FERMONT":"62","CAPELLE LES HESDIN":"62","CAUCOURT":"62","CHOCQUES":"62","CONCHIL LE TEMPLE":"62","CORMONT":"62","COUTURELLE":"62","CUINCHY":"62","DAINVILLE":"62","DESVRES":"62","ECOUST ST MEIN":"62","ECUIRES":"62","ELEU DIT LEAUWETTE":"62","ENQUIN LES MINES":"62","EPINOY":"62","EPS":"62","ERGNY":"62","ERNY ST JULIEN":"62","ESCALLES":"62","ESTREE BLANCHE":"62","ESTREE CAUCHY":"62","ETAING":"62","ETAPLES":"62","FERFAY":"62","FEUCHY":"62","FONTAINE LES HERMANS":"62","FORTEL EN ARTOIS":"62","FREMICOURT":"62","FRENCQ":"62","FRESNICOURT LE DOLMEN":"62","FRESNOY":"62","FRUGES":"62","FOLIES":"80","FORCEVILLE EN VIMEU":"80","FOUCAUCOURT EN SANTERRE":"80","FOUCAUCOURT HORS NESLE":"80","FRANVILLERS":"80","FREMONTIERS":"80","GOYENCOURT":"80","GRIVILLERS":"80","GUYENCOURT SUR NOYE":"80","GUYENCOURT SAULCOURT":"80","HAILLES":"80","HALLENCOURT":"80","HAMELET":"80","HANGEST EN SANTERRE":"80","HANGEST SUR SOMME":"80","HATTENCOURT":"80","HAUTVILLERS OUVILLE":"80","HEDAUVILLE":"80","HEUZECOURT":"80","HIERMONT":"80","HOMBLEUX":"80","LAFRESGUIMONT ST MARTIN":"80","LAHOUSSOYE":"80","LAMOTTE BREBIERE":"80","LIGNIERES CHATELAIN":"80","LIGNIERES EN VIMEU":"80","LIHONS":"80","LIOMER":"80","LONGAVESNES":"80","LONGUEVILLETTE":"80","MAISNIERES":"80","MAISON ROLAND":"80","MALPART":"80","MARQUAIX":"80","LE MEILLARD":"80","MEREAUCOURT":"80","MESNIL DOMQUEUR":"80","MESNIL ST NICAISE":"80","METIGNY":"80","MEZIERES EN SANTERRE":"80","MIANNAY":"80","MILLENCOURT":"80","MIRVAUX":"80","MISERY":"80","MOISLAINS":"80","ESTREES MONS":"80","MONTONVILLERS":"80","MORCHAIN":"80","MORLANCOURT":"80","NEUFMOULIN":"80","NEUILLY LE DIEN":"80","LA NEUVILLE LES BRAY":"80","OCHANCOURT":"80","OFFIGNIES":"80","OISEMONT":"80","PERTAIN":"80","POIX DE PICARDIE":"80","PONCHES ESTRUVAL":"80","PROUVILLE":"80","PUCHEVILLERS":"80","PUNCHY":"80","PUZEAUX":"80","QUEND":"80","LE QUESNEL":"80","QUIRY LE SEC":"80","RAMBURES":"80","REMAISNIL":"80","REVELLES":"80","ROIGLISE":"80","ROLLOT":"80","ROUVREL":"80","RUBESCOURT":"80","ST LEGER LES DOMART":"80","ST LEGER SUR BRESLE":"80","ST SAUFLIEU":"80","SALOUEL":"80","SAUVILLERS MONGIVAL":"80","SENARPONT":"80","TALMAS":"80","TEMPLEUX LE GUERARD":"80","TERRAMESNIL":"80","THOIX":"80","TILLOLOY":"80","VAUX EN AMIENOIS":"80","VAUX MARQUENNEVILLE":"80","VERCOURT":"80","VILLE LE MARCLET":"80","VILLERS FAUCON":"80","VRAIGNES EN VERMANDOIS":"80","VRAIGNES LES HORNOY":"80","WARVILLERS":"80","SAILLY LABOURSE":"62","ST FOLQUIN":"62","ST HILAIRE COTTES":"62","ST MICHEL SOUS BOIS":"62","SALPERWICK":"62","SARS LE BOIS":"62","SAVY BERLETTE":"62","SEMPY":"62","SERVINS":"62","SIBIVILLE":"62","SIMENCOURT":"62","SOUCHEZ":"62","TORTEFONTAINE":"62","LE TOUQUET PARIS PLAGE":"62","TRAMECOURT":"62","TRESCAULT":"62","VAUDRINGHEM":"62","VAULX VRAUCOURT":"62","VELU":"62","VERCHOCQ":"62","VIEIL HESDIN":"62","VIMY":"62","VIS EN ARTOIS":"62","WAIL":"62","WAMBERCOURT":"62","WARLENCOURT EAUCOURT":"62","WIDEHEM":"62","WIMEREUX":"62","WINGLES":"62","WISQUES":"62","ZUDAUSQUES":"62","LIBERCOURT":"62","YTRES":"62","AMBERT":"63","AUBUSSON D AUVERGNE":"63","AUGNAT":"63","AURIERES":"63","AUZAT LA COMBELLE":"63","AUZELLES":"63","BEAUREGARD VENDON":"63","BERGONNE":"63","BERTIGNAT":"63","BLOT L EGLISE":"63","BONGHEAT":"63","BOUDES":"63","BULHON":"63","CEILLOUX":"63","CHAMBON SUR DOLORE":"63","CHAMEANE":"63","CLERLANDE":"63","COLLANGES":"63","COURPIERE":"63","DORAT":"63","EFFIAT":"63","ESPINCHAL":"63","FAYET LE CHATEAU":"63","AULHAT FLAT":"63","GERZAT":"63","GIGNAT":"63","GLAINE MONTAIGUT":"63","LA GOUTELLE":"63","GRANDRIF":"63","HEUME L EGLISE":"63","ISSOIRE":"63","LABESSETTE":"63","LUZILLAT":"63","MALINTRAT":"63","MANGLIEU":"63","MARSAC EN LIVRADOIS":"63","LES MARTRES D ARTIERE":"63","MARTRES SUR MORGE":"63","MEDEYROLLES":"63","MENETROL":"63","MUROL":"63","ORCINES":"63","ORLEAT":"63","PALLADUC":"63","PERPEZAT":"63","PESCHADOIRES":"63","PESSAT VILLENEUVE":"63","PIGNOLS":"63","PULVERIERES":"63","RAVEL":"63","REIGNAT":"63","STE AGATHE":"63","ST AMANT ROCHE SAVINE":"63","ST ANTHEME":"63","LAVAL ROQUECEZIERE":"12","LA LOUBIERE":"12","MURASSON":"12","MUR DE BARREZ":"12","MURET LE CHATEAU":"12","NANT":"12","NAUCELLE":"12","PRADINAS":"12","PRUINES":"12","REBOURGUIL":"12","ROUSSENNAC":"12","ST FELIX DE LUNEL":"12","ST FELIX DE SORGUES":"12","ST JEAN DU BRUEL":"12","SALLES COURBATIES":"12","SALLES CURAN":"12","SALVAGNAC CAJARC":"12","LA SALVETAT PEYRALES":"12","SANVENSA":"12","SAUCLIERES":"12","LA SERRE":"12","SYLVANES":"12","LE TRUEL":"12","VEZINS DE LEVEZOU":"12","VILLEFRANCHE DE ROUERGUE":"12","VIVIEZ":"12","BOULBON":"13","GRANS":"13","LAMANON":"13","MALLEMORT":"13","MARIGNANE":"13","MEYREUIL":"13","PARADOU":"13","ST ANDIOL":"13","ST ANTONIN SUR BAYON":"13","ST ESTEVE JANSON":"13","ST ETIENNE DU GRES":"13","ST PAUL LES DURANCE":"13","ST SAVOURNIN":"13","SENAS":"13","SEPTEMES LES VALLONS":"13","VELAUX":"13","MARSEILLE 04":"13","MARSEILLE 10":"13","MARSEILLE 14":"13","AGY":"14","AIGNERVILLE":"14","AMBLIE":"14","ARGENCES":"14","AUBERVILLE":"14","BARBEVILLE":"14","BAUQUAY":"14","BISSIERES":"14","BLANGY LE CHATEAU":"14","BONS TASSILLY":"14","BOURGUEBUS":"14","RANCON":"87","RILHAC LASTOURS":"87","ST BARBANT":"87","ST HILAIRE LA TREILLE":"87","ST HILAIRE LES PLACES":"87","ST LEGER LA MONTAGNE":"87","ST SULPICE LAURIERE":"87","THOURON":"87","VERNEUIL SUR VIENNE":"87","VILLEFAVARD":"87","ALLARMONT":"88","AMEUVELLE":"88","AUTIGNY LA TOUR":"88","AYDOILLES":"88","BADMENIL AUX BOIS":"88","BAN DE LAVELINE":"88","BASSE SUR LE RUPT":"88","BAYECOURT":"88","BAZOILLES SUR MEUSE":"88","BELMONT LES DARNEY":"88","BERTRIMOUTIER":"88","BETTONCOURT":"88","BOCQUEGNEY":"88","LA BOURGONCE":"88","BOUXIERES AUX BOIS":"88","BROUVELIEURES":"88","CHAMPDRAY":"88","LA CHAPELLE AUX BOIS":"88","CHARMOIS L ORGUEILLEUX":"88","CHATILLON SUR SAONE":"88","CHAUFFECOURT":"88","LE CLERJUS":"88","CONTREXEVILLE":"88","DEINVILLERS":"88","DOCELLES":"88","DOMMARTIN LES REMIREMONT":"88","DOMPAIRE":"88","DOMREMY LA PUCELLE":"88","ETIVAL CLAIREFONTAINE":"88","FONTENOY LE CHATEAU":"88","FRAIZE":"88","GELVECOURT ET ADOMPT":"88","GEMAINGOUTTE":"88","GERARDMER":"88","GIRANCOURT":"88","GIRCOURT LES VIEVILLE":"88","GODONCOURT":"88","GORHEY":"88","GRANDRUPT":"88","GUGNEY AUX AULX":"88","HADOL":"88","JARMENIL":"88","JUBAINVILLE":"88","JUSSARUPT":"88","JUVAINCOURT":"88","LAMARCHE":"88","LAVAL SUR VOLOGNE":"88","LEPANGES SUR VOLOGNE":"88","LIFFOL LE GRAND":"88","MADECOURT":"88","MALAINCOURT":"88","MARTIGNY LES BAINS":"88","MARTIGNY LES GERBONVAUX":"88","MEDONVILLE":"88","MENARMONT":"88","MORELMAISON":"88","NAYEMONT LES FOSSES":"88","LA NEUVEVILLE SOUS CHATENOIS":"88","NEUVILLERS SUR FAVE":"88","NORROY":"88","OFFROICOURT":"88","PADOUX":"88","PIERREPONT SUR L ARENTELE":"88","PLAINFAING":"88","PORTIEUX":"88","LE PUID":"88","RAMBERVILLERS":"88","RAMONCHAMP":"88","RAON AUX BOIS":"88","REHAUPAL":"88","RELANGES":"88","REMOVILLE":"88","REPEL":"88","ROCHESSON":"88","ROLLAINVILLE":"88","ROMAIN AUX BOIS":"88","LE ROULIER":"88","ROUVRES EN XAINTOIS":"88","GALAMETZ":"62","GAVRELLE":"62","GONNEHEM":"62","GRAND RULLECOURT":"62","GUISY":"62","HAM EN ARTOIS":"62","HAMES BOUCRES":"62","HANNESCAMPS":"62","HARAVESNES":"62","HARDINGHEN":"62","HAUT LOQUIN":"62","HELFAUT":"62","HENDECOURT LES CAGNICOURT":"62","HENDECOURT LES RANSART":"62","HENIN SUR COJEUL":"62","HERBELLES":"62","HERLINCOURT":"62","HERVELINGHEN":"62","HESDIN L ABBE":"62","HINGES":"62","HUBERSENT":"62","HUBY ST LEU":"62","INCOURT":"62","INXENT":"62","ISQUES":"62","JOURNY":"62","LABEUVRIERE":"62","LABOURSE":"62","LAGNICOURT MARCEL":"62","LAMBRES":"62","LANDRETHUN LE NORD":"62","LANDRETHUN LES ARDRES":"62","LAPUGNOY":"62","LEFAUX":"62","LEPINE":"62","LESPINOY":"62","LEULINGHEN BERNES":"62","LIGNY ST FLOCHEL":"62","LONGFOSSE":"62","LORGIES":"62","LOTTINGHEN":"62","LOZINGHEM":"62","MAGNICOURT SUR CANCHE":"62","MANIN":"62","MANINGHEN HENNE":"62","MARANT":"62","MAZINGARBE":"62","MERCATEL":"62","MERCK ST LIEVIN":"62","MONTCAVREL":"62","MONT ST ELOI":"62","MORCHIES":"62","MUNCQ NIEURLET":"62","NEDON":"62","NOEUX LES AUXI":"62","NOEUX LES MINES":"62","NOREUIL":"62","NOYELLES SOUS BELLONNE":"62","OFFRETHUN":"62","OURTON":"62","PARENTY":"62","PELVES":"62","PREDEFIN":"62","PRESSY":"62","RACQUINGHEM":"62","RADINGHEM":"62","RAYE SUR AUTHIE":"62","REBREUVIETTE":"62","RECQUES SUR HEM":"62","RELY":"62","RETY":"62","RODELINGHEM":"62","ROUSSENT":"62","RUMINGHEM":"62","RUYAULCOURT":"62","SAILLY EN OSTREVENT":"62","SAILLY SUR LA LYS":"62","SAINS LES MARQUION":"62","ST JOSSE":"62","ST MARTIN SUR COJEUL":"62","ST MICHEL SUR TERNOISE":"62","ST NICOLAS":"62","ST OMER CAPELLE":"62","SALLAUMINES":"62","LE SARS":"62","SAUCHY LESTREE":"62","SAULCHOY":"62","SENLECQUES":"62","WOIGNARUE":"80","Y":"80","YVRENCHEUX":"80","YZEUX":"80","YONVAL":"80","ALMAYRAC":"81","ARTHES":"81","BELLESERRE":"81","BRIATEXTE":"81","BURLATS":"81","CADIX":"81","CAMBON LES LAVAUR":"81","CASTELNAU DE LEVIS":"81","COMBEFA":"81","CUNAC":"81","CUQ TOULZA":"81","ESCOUSSENS":"81","ESPERAUSSES":"81","FRAISSINES":"81","FRAUSSEILLES":"81","GIJOUNET":"81","LABARTHE BLEYS":"81","LABASTIDE ST GEORGES":"81","LABRUGUIERE":"81","GUITALENS L ALBAREDE":"81","LAMILLARIE":"81","LAPARROUQUIAL":"81","MAGRIN":"81","MARNAVES":"81","MARZENS":"81","MASSALS":"81","MAURENS SCOPONT":"81","MONTREDON LABESSONNIE":"81","PECHAUDIER":"81","PEYREGOUX":"81","PUYBEGON":"81","PUYCALVEL":"81","PUYGOUZON":"81","REALMONT":"81","LE RIOLS":"81","ST AFFRIQUE LES MONTAGNES":"81","ST AMANS SOULT":"81","ST AMANS VALTORET":"81","ST ANTONIN DE LACALM":"81","ST BEAUZILE":"81","STE CECILE DU CAYROU":"81","ST GENEST DE CONTEST":"81","ST LIEUX LAFENASSE":"81","ST MARTIN LAGUEPIE":"81","ST SERNIN LES LAVAUR":"81","SALIES":"81","SALVAGNAC":"81","SAUSSENAC":"81","SEMALENS":"81","SERENAC":"81","SOUAL":"81","TERSSAC":"81","TEULAT":"81","TONNAC":"81","TREBAS":"81","VAOUR":"81","LE VINTROU":"81","VIRAC":"81","VIVIERS LES LAVAUR":"81","VIVIERS LES MONTAGNES":"81","ALBEFEUILLE LAGARDE":"82","ALBIAS":"82","AUTY":"82","BARDIGUES":"82","BARRY D ISLEMADE":"82","BELBEZE EN LOMAGNE":"82","BOUDOU":"82","BRUNIQUEL":"82","CORDES TOLOSANNES":"82","CUMONT":"82","ESCAZEAUX":"82","GASQUES":"82","GRAMONT":"82","LABASTIDE ST PIERRE":"82","LABOURGADE":"82","ST BONNET PRES ORCIVAL":"63","ST BONNET PRES RIOM":"63","ST DIERY":"63","ST ELOY LA GLACIERE":"63","ST ETIENNE DES CHAMPS":"63","ST ETIENNE SUR USSON":"63","ST FERREOL DES COTES":"63","ST GAL SUR SIOULE":"63","ST GENES LA TOURETTE":"63","ST GERMAIN PRES HERMENT":"63","ST GERMAIN L HERM":"63","ST HILAIRE LES MONGES":"63","ST JULIEN PUY LAVEZE":"63","ST LAURE":"63","ST MARTIN D OLLIERES":"63","ST PIERRE COLAMINE":"63","ST QUINTIN SUR SIOULE":"63","ST REMY DE BLOT":"63","SALLEDES":"63","TRALAIGUES":"63","USSON":"63","VIRLET":"63","VOINGT":"63","VOLLORE VILLE":"63","YSSAC LA TOURETTE":"63","AAST":"64","AINHARP":"64","AINHOA":"64","ALDUDES":"64","ANGLET":"64","ARANCOU":"64","ARAUX":"64","ARBUS":"64","AREN":"64","ARESSY":"64","ARMENDARITS":"64","ARROS DE NAY":"64","ARROSES":"64","ARTIGUELOUVE":"64","ASTIS":"64","AUBOUS":"64","AUGA":"64","AYDIUS":"64","BARDOS":"64","BARINQUE":"64","BARRAUTE CAMU":"64","BASSILLON VAUZE":"64","BERGOUEY VIELLENAVE":"64","BIDACHE":"64","BIELLE":"64","BIZANOS":"64","BUNUS":"64","BUSSUNARITS SARRASQUETTE":"64","BUZY":"64","CAMBO LES BAINS":"64","CARRESSE CASSABER":"64","CASTILLON D ARTHEZ":"64","CHARRE":"64","CHARRITTE DE BAS":"64","CHERAUTE":"64","COSLEDAA LUBE BOAST":"64","COUBLUCQ":"64","DENGUIN":"64","EAUX BONNES":"64","ESCOT":"64","ESCOUT":"64","ESPIUTE":"64","ESTERENCUBY":"64","ESTOS":"64","GARLIN":"64","GESTAS":"64","GEUS D OLORON":"64","GOTEIN LIBARRENX":"64","GUINARTHE PARENTIES":"64","GURMENCON":"64","HASPARREN":"64","HAUT DE BOSDARROS":"64","L HOPITAL D ORION":"64","HOSTA":"64","IDRON":"64","LABEYRIE":"64","LACARRE":"64","LACARRY ARHAN CHARRITTE DE HAUT":"64","LAGOS":"64","LAMAYOU":"64","BRETTEVILLE SUR LAIZE":"14","BUCEELS":"14","LE BU SUR ROUVRES":"14","CHAMP DU BOULT":"14","CONDE SUR IFS":"14","CONDE SUR SEULLES":"14","COUPESARTE":"14","CRESSERONS":"14","DRUBEC":"14","EVRECY":"14","FIERVILLE LES PARCS":"14","FONTAINE ETOUPEFOUR":"14","FONTAINE HENRY":"14","FONTENAY LE PESNEL":"14","FONTENERMONT":"14","FORMENTIN":"14","FOURNEVILLE":"14","GEFOSSE FONTENAY":"14","GENNEVILLE":"14","GERROTS":"14","GOUSTRANVILLE":"14","GRAINVILLE LANGANNERIE":"14","GRAYE SUR MER":"14","GRENTHEVILLE":"14","GUERON":"14","HOTTOT LES BAGUES":"14","LA LANDE SUR DROME":"14","LEFFARD":"14","LINGEVRES":"14","LISORES":"14","LITTEAU":"14","LE MOLAY LITTRY":"14","LE LOCHEUR":"14","LONGRAYE":"14","LONGUES SUR MER":"14","LONGVILLERS":"14","LUC SUR MER":"14","MANDEVILLE EN BESSIN":"14","MAY SUR ORNE":"14","MERVILLE FRANCEVILLE PLAGE":"14","LE MESNIL AUZOUF":"14","LE MESNIL GUILLAUME":"14","LE MESNIL PATRY":"14","LE MESNIL SUR BLANGY":"14","LE MESNIL VILLEMENT":"14","MONCEAUX EN BESSIN":"14","LES MOUTIERS EN CINGLAIS":"14","MOYAUX":"14","NOTRE DAME DE LIVAYE":"14","LES OUBEAUX":"14","OUVILLE LA BIEN TOURNEE":"14","PERIERS EN AUGE":"14","PLACY":"14","PRETREVILLE":"14","ROCQUES":"14","ROSEL":"14","ST AUBIN D ARQUENAY":"14","STE CROIX GRAND TONNE":"14","STE CROIX SUR MER":"14","ST DENIS DE MAILLOC":"14","ST DESIR":"14","ST GATIEN DES BOIS":"14","ST GERMAIN D ECTOT":"14","ST GERMAIN LA BLANCHE HERBE":"14","STE HONORINE DE DUCY":"14","ST HYMER":"14","ST JEAN DES ESSARTIERS":"14","ST JOUIN":"14","ST JULIEN LE FAUCON":"14","ST LEGER DUBOSQ":"14","ST LOUP DE FRIBOIS":"14","ST MANVIEU BOCAGE":"14","ST MARTIN DE BLAGNY":"14","ST PHILBERT DES CHAMPS":"14","SALLENELLES":"14","SEPT FRERES":"14","SOMMERVIEU":"14","ST MAURICE SUR MORTAGNE":"88","ST NABORD":"88","ST STAIL":"88","SANCHEY":"88","SANS VALLOIS":"88","LE SAULCY":"88","SENONES":"88","SERAUMONT":"88","SERCOEUR":"88","URIMENIL":"88","VECOUX":"88","VELOTTE ET TATIGNECOURT":"88","VINCEY":"88","VOUXEY":"88","XERTIGNY":"88","XONRUPT LONGEMER":"88","AISY SUR ARMANCON":"89","ANCY LE FRANC":"89","ANNAY SUR SEREIN":"89","ARCY SUR CURE":"89","BERNOUIL":"89","BLEIGNY LE CARREAU":"89","BLENEAU":"89","BRANNAY":"89","BRIENON SUR ARMANCON":"89","BROSSES":"89","CERISIERS":"89","CHAILLEY":"89","CHAMPIGNELLES":"89","CHAMPLAY":"89","CHAMPLOST":"89","CHAMPS SUR YONNE":"89","CHATEL CENSOIR":"89","CHENY":"89","CHEU":"89","COLLAN":"89","COMPIGNY":"89","COUTARNOUX":"89","CUSSY LES FORGES":"89","DANNEMOINE":"89","DOMATS":"89","DRACY":"89","EPINEAU LES VOVES":"89","FLEURY LA VALLEE":"89","FLOGNY LA CHAPELLE":"89","FULVY":"89","GRIMAULT":"89","ISLAND":"89","L ISLE SUR SEREIN":"89","JULLY":"89","LEZINNES":"89","LUCY LE BOIS":"89","MAILLOT":"89","MALAY LE GRAND":"89","MONTACHER VILLEGARDIN":"89","MONTILLOT":"89","NAILLY":"89","OUANNE":"89","PACY SUR ARMANCON":"89","PARLY":"89","PONTIGNY":"89","PRECY LE SEC":"89","RUGNY":"89","ST CYR LES COLONS":"89","ST GERMAIN DES CHAMPS":"89","SAMBOURG":"89","SERGINES":"89","SOUGERES EN PUISAYE":"89","TANLAY":"89","THORIGNY SUR OREUSE":"89","TREIGNY":"89","TREVILLY":"89","VALLAN":"89","SORRUS":"62","THIEMBRONNE":"62","TILQUES":"62","TINCQUES":"62","LE TRANSLOY":"62","TUBERSENT":"62","VACQUERIETTE ERQUIERES":"62","VERMELLES":"62","VERQUIGNEUL":"62","VIEILLE CHAPELLE":"62","VILLERS AU FLOS":"62","VILLERS LES CAGNICOURT":"62","VILLERS SIR SIMON":"62","VIOLAINES":"62","VITRY EN ARTOIS":"62","WABEN":"62","WAILLY BEAUCAMP":"62","LE WAST":"62","WIERRE EFFROY":"62","WIMILLE":"62","LA CAPELLE LES BOULOGNE":"62","AYDAT":"63","BAFFIE":"63","BAS ET LEZAT":"63","BEAUREGARD L EVEQUE":"63","LE BREUIL SUR COUZE":"63","CEBAZAT":"63","CHAMPAGNAT LE JEUNE":"63","CHAMPETIERES":"63","CHANONAT":"63","CHARBONNIER LES MINES":"63","CHARBONNIERES LES VIEILLES":"63","LE CHEIX":"63","CISTERNES LA FORET":"63","CONDAT LES MONTBOISSIER":"63","CRESTE":"63","DORE L EGLISE":"63","EGLISENEUVE DES LIARDS":"63","EGLISENEUVE PRES BILLOM":"63","LA FORIE":"63","GELLES":"63","HERMENT":"63","JUMEAUX":"63","LACHAUX":"63","LA TOUR D AUVERGNE":"63","LOUBEYRAT":"63","MARAT":"63","MARSAT":"63","MENAT":"63","MIREFLEURS":"63","LE MONESTIER":"63","MONTEL DE GELAT":"63","MOUREUILLE":"63","MOZAC":"63","OLLIERGUES":"63","PERRIER":"63","PESLIERES":"63","PIONSAT":"63","LES PRADEAUX":"63","PRONDINES":"63","ROCHEFORT MONTAGNE":"63","SAILLANT":"63","ST BABEL":"63","STE CHRISTINE":"63","ST DONAT":"63","ST GEORGES SUR ALLIER":"63","ST GERVAIS SOUS MEYMONT":"63","ST GERVAZY":"63","ST JACQUES D AMBUR":"63","ST JEAN EN VAL":"63","ST MARTIN DES OLMES":"63","ST PIERRE LA BOURLHONNE":"63","ST PRIEST BRAMEFANT":"63","ST SAUVEUR LA SAGNE":"63","SAULZET LE FROID":"63","SAURET BESSERVE":"63","SAUVIAT":"63","SAYAT":"63","SERMENTIZON":"63","SERVANT":"63","SEYCHALLES":"63","SOLIGNAT":"63","THURET":"63","VARENNES SUR MORGE":"63","VERGHEAS":"63","LAMAGISTERE":"82","LAUZERTE":"82","LAVIT":"82","MONTAUBAN":"82","MONTECH":"82","MONTFERMIER":"82","ST ARROUMEX":"82","ST NAZAIRE DE VALENTANE":"82","LA SALVETAT BELMONTET":"82","VERLHAC TESCOU":"82","VILLEMADE":"82","AIGUINES":"83","BELGENTIER":"83","BRAS":"83","BRENON":"83","LA CADIERE D AZUR":"83","CAMPS LA SOURCE":"83","CORRENS":"83","LA CRAU":"83","CUERS":"83","FLASSANS SUR ISSOLE":"83","FREJUS":"83","GAREOULT":"83","GRIMAUD":"83","LE LAVANDOU":"83","LORGUES":"83","LA MOLE":"83","MONTFORT SUR ARGENS":"83","NANS LES PINS":"83","LE PLAN DE LA TOUR":"83","POURRIERES":"83","LA ROQUEBRUSSANNE":"83","ROUGIERS":"83","ST MARTIN DE PALLIERES":"83","ST MAXIMIN LA STE BAUME":"83","ST TROPEZ":"83","LES SALLES SUR VERDON":"83","TAVERNES":"83","TRIGANCE":"83","LA VALETTE DU VAR":"83","VERIGNON":"83","VILLECROZE":"83","VINON SUR VERDON":"83","ANSOUIS":"84","AURIBEAU":"84","LE BARROUX":"84","LA BASTIDE DES JOURDANS":"84","LE BEAUCET":"84","BEAUMES DE VENISE":"84","BEDARRIDES":"84","BOLLENE":"84","BUOUX":"84","CABRIERES D AIGUES":"84","CRILLON LE BRAVE":"84","ENTRAIGUES SUR LA SORGUE":"84","GIGONDAS":"84","GORDES":"84","LAMOTTE DU RHONE":"84","LORIOL DU COMTAT":"84","LOURMARIN":"84","MORIERES LES AVIGNON":"84","MORMOIRON":"84","PUGET":"84","PUYMERAS":"84","ST LEGER DU VENTOUX":"84","ST ROMAIN EN VIENNOIS":"84","ST ROMAN DE MALEGARDE":"84","ST SATURNIN LES AVIGNON":"84","SARRIANS":"84","SAVOILLAN":"84","SIVERGUES":"84","SORGUES":"84","SUZETTE":"84","LE THOR":"84","VAISON LA ROMAINE":"84","VELLERON":"84","VILLES SUR AUZON":"84","LANNE EN BARETOUS":"64","LARCEVEAU ARROS CIBITS":"64","LARRESSORE":"64","LECUMBERRY":"64","LEDEUIX":"64","LESCAR":"64","LESPOURCY":"64","LIVRON":"64","LOHITZUN OYHERCQ":"64","LOMBIA":"64","LONCON":"64","LUCQ DE BEARN":"64","LUSSAGNET LUSSON":"64","MASCARAAS HARON":"64","MAURE":"64","MEILLON":"64","MENDITTE":"64","MONCAYOLLE LARRORY MENDIBIEU":"64","MONPEZAT":"64","MONTANER":"64","MONTARDON":"64","MOUHOUS":"64","OGENNE CAMPTORT":"64","OSSE EN ASPE":"64","OSSENX":"64","OSSERAIN RIVAREYTE":"64","PEYRELONGUE ABOS":"64","PONSON DEBAT POUTS":"64","POULIACQ":"64","POURSIUGUES BOUCOUE":"64","PRECHACQ NAVARRENX":"64","PUYOO":"64","RAMOUS":"64","ST ABIT":"64","ST ESTEBEN":"64","ST JUST IBARRE":"64","SALLESPISSE":"64","SAMES":"64","SAUCEDE":"64","SEBY":"64","SEDZE MAUBECQ":"64","SEVIGNACQ":"64","SUSMIOU":"64","TARON SADIRAC VIELLENAVE":"64","URDES":"64","USTARITZ":"64","UZOS":"64","VERDETS":"64","VIVEN":"64","ADAST":"65","ADE":"65","ARAGNOUET":"65","ARREAU":"65","AVERAN":"65","AVEUX":"65","BARBACHEN":"65","BARLEST":"65","BARTHE":"65","BARTRES":"65","BERBERUST LIAS":"65","BERNAC DEBAT":"65","BERNADETS DEBAT":"65","BERNADETS DESSUS":"65","BONNEMAZON":"65","ST LUMIER EN CHAMPAGNE":"51","ST MARTIN L HEUREUX":"51","STE MENEHOULD":"51","ST QUENTIN LES MARAIS":"51","ST SOUPLET SUR PY":"51","ST THIERRY":"51","ST THOMAS EN ARGONNE":"51","SARCY":"51","SIVRY ANTE":"51","SOMMEPY TAHURE":"51","SOMME SUIPPE":"51","SUIPPES":"51","THAAS":"51","VATRY":"51","VAUDEMANGE":"51","VERZENAY":"51","VESIGNEUL SUR MARNE":"51","LA VILLE SOUS ORBAIS":"51","VOILEMONT":"51","VOUZY":"51","VROIL":"51","MAGENTA":"51","TOUR EN BESSIN":"14","TOURNAY SUR ODON":"14","TREVIERES":"14","TROARN":"14","VALSEME":"14","PONT D OUILLY":"14","ANGLARDS DE SALERS":"15","APCHON":"15","VILLY LE MARECHAL":"10","YEVRES LE PETIT":"10","ALET LES BAINS":"11","VILLEMOYENNE":"10","VILLEMORIEN":"10","ANTUGNAC":"11","VOIGNY":"10","AXAT":"11","MONTMORENCY BEAUFORT":"10","MONTMARTIN LE HAUT":"10","POUAN LES VALLEES":"10","PLAINES ST LANGE":"10","PLESSIS BARBUISE":"10","PAISY COSDON":"10","ORTILLON":"10","MONTFEY":"10","PAYNS":"10","BROUSSES ET VILLARET":"11","BELLEGARDE DU RAZES":"11","LES BRUNELS":"11","CAMBIEURE":"11","BREZILHAC":"11","BAGNOLES":"11","CAPENDU":"11","AZILLE":"11","ST HILAIRE SOUS ROMILLY":"10","ST LEGER PRES TROYES":"10","RIGNY LA NONNEUSE":"10","PRUNAY BELLEVILLE":"10","LES RICEYS":"10","RUVIGNY":"10","RANCES":"10","POUGY":"10","ST REMY SOUS BARBUISE":"10","ST PARRES LES VAUDES":"10","ST LOUP DE BUFFIGNY":"10","TORVILLIERS":"10","SEMOINE":"10","VAUDES":"10","LIGNOL LE CHATEAU":"10","LA LOGE POMBLIN":"10","LANDREVILLE":"10","LASSICOURT":"10","JAUCOURT":"10","HERBISSE":"10","JUVANZE":"10","JEUGNY":"10","LIREY":"10","CONILHAC CORBIERES":"11","COURNANEL":"11","CHALABRE":"11","LONGUEVILLE SUR AUBE":"10","MARIGNY LE CHATEL":"10","MARNAY SUR SEINE":"10","MERY SUR SEINE":"10","METZ ROBERT":"10","MEURVILLE":"10","LONGSOLS":"10","MESSON":"10","LA MOTHE ACHARD":"85","VEZINNES":"89","VILLEMANOCHE":"89","VILLETHIERRY":"89","BERMONT":"90","CHAVANATTE":"90","ELOIE":"90","FRAIS":"90","FROIDEFONTAINE":"90","GIROMAGNY":"90","NOVILLARD":"90","OFFEMONT":"90","PETITMAGNY":"90","PHAFFANS":"90","ROPPE":"90","ROUGEMONT LE CHATEAU":"90","SERMAMAGNY":"90","SEVENANS":"90","URCEREY":"90","ABBEVILLE LA RIVIERE":"91","ANGERVILLIERS":"91","BOUTERVILLIERS":"91","BRETIGNY SUR ORGE":"91","CHALO ST MARS":"91","COURCOURONNES":"91","CROSNE":"91","ETIOLLES":"91","GOMETZ LE CHATEL":"91","LES GRANGES LE ROI":"91","GUIBEVILLE":"91","LARDY":"91","LEUVILLE SUR ORGE":"91","MARCOUSSIS":"91","MEROBERT":"91","LES MOLIERES":"91","MORSANG SUR ORGE":"91","ONCY SUR ECOLE":"91","ORSAY":"91","RICHARVILLE":"91","ST CYR LA RIVIERE":"91","ST GERMAIN LES ARPAJON":"91","ST MICHEL SUR ORGE":"91","SAINTRY SUR SEINE":"91","SOISY SUR ECOLE":"91","VERT LE GRAND":"91","VERT LE PETIT":"91","VILLECONIN":"91","VILLEJUST":"91","LES ULIS":"91","ANTONY":"92","CHAVILLE":"92","BAILLY ROMAINVILLIERS":"77","BALLOY":"77","BARBIZON":"77","BEAUTHEIL":"77","BEZALLES":"77","BOISSISE LE ROI":"77","BOISSY AUX CAILLES":"77","BOISSY LE CHATEL":"77","BOMBON":"77","BOUTIGNY":"77","BRIE COMTE ROBERT":"77","CESSON":"77","CHAINTREAUX":"77","CHAMPCENEST":"77","CHAMPS SUR MARNE":"77","LA CHAPELLE MOUTILS":"77","CHATEAUBLEAU":"77","CHATEAU LANDON":"77","CHEVRY EN SEREINE":"77","CONCHES SUR GONDOIRE":"77","COUPVRAY":"77","CRECY LA CHAPELLE":"77","CREVECOEUR EN BRIE":"77","DAGNY":"77","DAMPMART":"77","DHUISY":"77","DOUE":"77","EGREVILLE":"77","ESBLY":"77","FONTAINEBLEAU":"77","FONTENAY TRESIGNY":"77","FORFRY":"77","FOUJU":"77","VERNET LA VARENNE":"63","VEYRE MONTON":"63","VILLOSANGES":"63","VOLVIC":"63","ABERE":"64","ABITAIN":"64","ACCOUS":"64","AHAXE ALCIETTE BASCASSAN":"64","ANGAIS":"64","ARAMITS":"64","ARNOS":"64","ASCARAT":"64","ASSON":"64","AUSSURUCQ":"64","AYDIE":"64","BAUDREIX":"64","BELLOCQ":"64","BENTAYOU SEREE":"64","BERENX":"64","BEUSTE":"64","BIDART":"64","BOEIL BEZING":"64","BORDERES":"64","BOUCAU":"64","BOUEILH BOUEILHO LASQUE":"64","BRISCOUS":"64","CABIDOS":"64","CAME":"64","CAMOU CIHIGUE":"64","CASTEIDE CANDAU":"64","CASTET":"64","CASTETBON":"64","CASTILLON DE LEMBEYE":"64","DIUSSE":"64","DOAZON":"64","DOMEZAIN BERRAUTE":"64","ESCOS":"64","ESPECHEDE":"64","ESQUIULE":"64","ETCHARRY":"64","ETCHEBAR":"64","EYSUS":"64","GARLEDE MONDEBAT":"64","GAROS":"64","GARRIS":"64","GEUS D ARZACQ":"64","HELETTE":"64","IRISSARRY":"64","JATXOU":"64","JAXU":"64","JUXUE":"64","LABETS BISCAY":"64","LACADEE":"64","LAHONTAN":"64","LALONGUE":"64","LESTELLE BETHARRAM":"64","LICHANS SUNHAR":"64","LONS":"64","LOUBIENG":"64","LOURDIOS ICHERE":"64","MACAYE":"64","MALAUSSANNE":"64","MASLACQ":"64","MAUCOR":"64","MENDIONDE":"64","MESPLEDE":"64","MONCLA":"64","MONEIN":"64","MONT DISSE":"64","MOUMOUR":"64","NAVARRENX":"64","NOGUERES":"64","ORSANCO":"64","OSSAS SUHARE":"64","PAU":"64","PONSON DESSUS":"64","RIBARROUY":"64","RIUPEYROUS":"64","ST ARMOU":"64","ST DOS":"64","STE ENGRACE":"64","ST ETIENNE DE BAIGORRY":"64","ST FAUST":"64","ST GLADIE ARRIVE MUNEIN":"64","WOLFSKIRCHEN":"67","WOLXHEIM":"67","ZEHNACKER":"67","LE BERNARD":"85","BOIS DE CENE":"85","LE BOUPERE":"85","BRETIGNOLLES SUR MER":"85","BREUIL BARRET":"85","CHANTONNAY":"85","LA CHAPELLE ACHARD":"85","LA CHAPELLE PALLUAU":"85","DAMVIX":"85","DOIX LES FONTAINES":"85","ESSARTS EN BOCAGE":"85","SEVREMONT":"85","FOUGERE":"85","LA GAUBRETIERE":"85","LE GIVRE":"85","GRAND LANDES":"85","JARD SUR MER":"85","LANDEVIEILLE":"85","LOGE FOUGEREUSE":"85","LONGEVILLE SUR MER":"85","MACHE":"85","LA MERLATIERE":"85","MONTOURNAIS":"85","MOUTIERS SUR LE LAY":"85","NESMY":"85","L ORBRIE":"85","LE POIRE SUR VELLUIRE":"85","LE POIRE SUR VIE":"85","ST BENOIST SUR MER":"85","ST CYR DES GATS":"85","ST DENIS DU PAYRE":"85","STE FLAIVE DES LOUPS":"85","ST GILLES CROIX DE VIE":"85","ST HILAIRE DE VOUST":"85","ST MARS LA REORTHE":"85","ST MARTIN LARS EN STE HERMINE":"85","ST MATHURIN":"85","ST MAURICE DES NOUES":"85","SERIGNE":"85","LA TAILLEE":"85","LA TARDIERE":"85","THORIGNY":"85","TREIZE VENTS":"85","VAIRE":"85","VENDRENNES":"85","AMBERRE":"86","ANGLES SUR L ANGLIN":"86","ARCHIGNY":"86","BENASSAY":"86","BIARD":"86","CENON SUR VIENNE":"86","CHAMPAGNE ST HILAIRE":"86","LA CHAPELLE MOULIERE":"86","CHENEVELLES":"86","CURCAY SUR DIVE":"86","GIZAY":"86","GOUEX":"86","GUESNES":"86","LAVAUSSEAU":"86","LEIGNE LES BOIS":"86","LHOMMAIZE":"86","LIGLET":"86","LINAZAY":"86","LINIERS":"86","LUCHAPT":"86","MARTAIZE":"86","MONDION":"86","MONTMORILLON":"86","MORTON":"86","NERIGNAC":"86","NUEIL SOUS FAYE":"86","PAYRE":"86","PERSAC":"86","POITIERS":"86","AILLIANVILLE":"52","ALLICHAMPS":"52","ANROSEY":"52","ARBIGNY SOUS VARENNES":"52","ARNANCOURT":"52","AUTIGNY LE GRAND":"52","AVRECOURT":"52","BEAUCHEMIN":"52","BLESSONVILLE":"52","BLUMERAY":"52","BOURMONT":"52","BUGNIERES":"52","CELSOY":"52","CERISIERES":"52","CHARMES EN L ANGLE":"52","CHATENAY MACHERON":"52","CIRFONTAINES EN ORNOIS":"52","COLMIER LE BAS":"52","COLMIER LE HAUT":"52","COURCELLES SUR BLAISE":"52","DOMBLAIN":"52","DOMMARTIN LE ST PERE":"52","ECHENAY":"52","ESNOUVEAUX":"52","FLAMMERECOURT":"52","FRESNES SUR APANCE":"52","GERMISAY":"52","GONCOURT":"52","GUINDRECOURT SUR BLAISE":"52","GUYONVELLE":"52","HACOURT":"52","HARREVILLE LES CHANTEURS":"52","HUMBERVILLE":"52","LACHAPELLE EN BLAISY":"52","LANEUVELLE":"52","LARIVIERE ARNONCOURT":"52","LECEY":"52","LEURVILLE":"52","LIFFOL LE PETIT":"52","LOUVEMONT":"52","MANOIS":"52","MAREILLES":"52","MOESLAINS":"52","MONTOT SUR ROGNON":"52","MOUILLERON":"52","NEUILLY SUR SUIZE":"52","NEUVELLE LES VOISEY":"52","NOIDANT LE ROCHEUX":"52","ORBIGNY AU MONT":"52","ORGES":"52","PANSEY":"52","PERROGNEY LES FONTAINES":"52","PERRUSSE":"52","POINSON LES FAYL":"52","POINSON LES GRANCEY":"52","POINSON LES NOGENT":"52","POISSONS":"52","RIVIERES LE BOIS":"52","OLONNE SUR MER":"85","OULMES":"85","LE PERRIER":"85","ROCHESERVIERE":"85","ST CHRISTOPHE DU LIGNERON":"85","ST CYR EN TALMONDAIS":"85","STE GEMME LA PLAINE":"85","ST JEAN DE BEUGNE":"85","ST JEAN DE MONTS":"85","ST LAURENT DE LA SALLE":"85","ST MAIXENT SUR VIE":"85","ST PIERRE DU CHEMIN":"85","TALMONT ST HILAIRE":"85","LA TRANCHE SUR MER":"85","TREIZE SEPTIERS":"85","LA FAUTE SUR MER":"85","AYRON":"86","BERRIE":"86","BETHINES":"86","BIGNOUX":"86","LA CHAPELLE MONTREUIL":"86","CHARRAIS":"86","CHAUNAY":"86","CHENECHE":"86","CHOUPPES":"86","COUHE":"86","DANGE ST ROMAIN":"86","DOUSSAY":"86","LA FERRIERE AIROUX":"86","FLEIX":"86","FONTAINE LE COMTE":"86","FROZES":"86","GENCAY":"86","LAUTHIERS":"86","LEIGNES SUR FONTAINE":"86","LIZANT":"86","LUSIGNAN":"86","MAULAY":"86","MIGNALOUX BEAUVOIR":"86","MIGNE AUXANCES":"86","MONTHOIRON":"86","MOULISMES":"86","PRESSAC":"86","LE ROCHEREAU":"86","ST GENEST D AMBIERE":"86","ST LEGER DE MONTBRILLAIS":"86","ST MACOUX":"86","ST MAURICE LA CLOUERE":"86","SENILLE ST SAUVEUR":"86","SAVIGNY LEVESCAULT":"86","SMARVES":"86","SOSSAIS":"86","VEZIERES":"86","VILLEMORT":"86","YVERSAY":"86","LES BILLANGES":"87","CHAMPAGNAC LA RIVIERE":"87","CHAMPNETERY":"87","CHAPTELAT":"87","CHATEAUNEUF LA FORET":"87","COUZEIX":"87","DOMPIERRE LES EGLISES":"87","FOLLES":"87","GLANDON":"87","GORRE":"87","JAVERDAT":"87","LAURIERE":"87","MAISONNAIS SUR TARDOIRE":"87","MEILHAC":"87","MOISSANNES":"87","NANTIAT":"87","NEXON":"87","SAILLAT SUR VIENNE":"87","ST AMAND LE PETIT":"87","ST AUVENT":"87","ST BONNET BRIANCE":"87","ST BRICE SUR VIENNE":"87","ST GENEST SUR ROSELLE":"87","ST GILLES LES FORETS":"87","FRESNES SUR MARNE":"77","GRESSY":"77","GUERCHEVILLE":"77","HAUTEFEUILLE":"77","LA HOUSSAYE EN BRIE":"77","ICHY":"77","ISLES LES MELDEUSES":"77","JAULNES":"77","JOUY SUR MORIN":"77","LAGNY SUR MARNE":"77","LAVAL EN BRIE":"77","LEUDON EN BRIE":"77","LONGPERRIER":"77","LOUAN VILLEGRUIS FONTAINE":"77","MAISONCELLES EN GATINAIS":"77","MARCHEMORET":"77","MAUREGARD":"77","LE MESNIL AMELOT":"77","MITRY MORY":"77","MONTIGNY LENCOUP":"77","MORMANT":"77","MORTERY":"77","MOUROUX":"77","MOUY SUR SEINE":"77","NANTOUILLET":"77","NOISIEL":"77","LES ORMES SUR VOULZIE":"77","OZOUER LE VOULGIS":"77","PENCHARD":"77","PEZARCHES":"77","LE PLESSIS AUX BOIS":"77","LE PLESSIS FEU AUSSOUX":"77","LE PLESSIS L EVEQUE":"77","POIGNY":"77","POMPONNE":"77","SAACY SUR MARNE":"77","ST MARS VIEUX MAISONS":"77","SAMMERON":"77","SOLERS":"77","THOURY FEROTTES":"77","TOUQUIN":"77","TOURNAN EN BRIE":"77","TOUSSON":"77","USSY SUR MARNE":"77","VANVILLE":"77","VAUX SUR LUNAIN":"77","VERNOU LA CELLE SUR SEINE":"77","VILLENEUVE SOUS DAMMARTIN":"77","VILLIERS SUR SEINE":"77","VINANTES":"77","ADAINVILLE":"78","ARNOUVILLE LES MANTES":"78","BAZOCHES SUR GUYONNE":"78","BOINVILLE EN MANTOIS":"78","BOINVILLE LE GAILLARD":"78","BOISSETS":"78","BOURDONNE":"78","LES BREVIAIRES":"78","BUCHELAY":"78","BULLION":"78","CARRIERES SUR SEINE":"78","CHANTELOUP LES VIGNES":"78","CHAPET":"78","LE CHESNAY":"78","CONDE SUR VESGRE":"78","EPONE":"78","L ETANG LA VILLE":"78","LA FALAISE":"78","FOLLAINVILLE DENNEMONT":"78","FONTENAY LE FLEURY":"78","FONTENAY MAUVOISIN":"78","FOURQUEUX":"78","HARDRICOURT":"78","HERMERAY":"78","LIMETZ VILLEZ":"78","LOUVECIENNES":"78","MAGNY LES HAMEAUX":"78","MAREIL MARLY":"78","MARLY LE ROI":"78","ZELLWILLER":"67","ALTKIRCH":"68","AMMERTZWILLER":"68","BANTZENHEIM":"68","BENDORF":"68","BERENTZWILLER":"68","BERGHEIM":"68","BETTENDORF":"68","BILTZHEIM":"68","BLOTZHEIM":"68","BRETTEN":"68","COLMAR":"68","DURLINSDORF":"68","EGUISHEIM":"68","ENSISHEIM":"68","FALKWILLER":"68","FESSENHEIM":"68","GALFINGUE":"68","GRIESBACH AU VAL":"68","GRUSSENHEIM":"68","HAUSGAUEN":"68","HECKEN":"68","HEGENHEIM":"68","HEIMSBRUNN":"68","HERRLISHEIM PRES COLMAR":"68","HOHROD":"68","HUNAWIHR":"68","JEBSHEIM":"68","KOETZINGUE":"68","KUNHEIM":"68","LUTTER":"68","MOLLAU":"68","NIEDERMORSCHWIHR":"68","OBERBRUCK":"68","OLTINGUE":"68","RAEDERSHEIM":"68","RIMBACHZELL":"68","ROMBACH LE FRANC":"68","ROUFFACH":"68","RUMERSHEIM LE HAUT":"68","SCHLIERBACH":"68","SONDERNACH":"68","SONDERSDORF":"68","SPECHBACH":"68","STEINBACH":"68","TAGOLSHEIM":"68","ZAESSINGUE":"68","ZILLISHEIM":"68","ALIX":"69","AMPUIS":"69","CERCIE":"69","CHAMELET":"69","CHARBONNIERES LES BAINS":"69","CHARENTAY":"69","CHEVINAY":"69","DENICE":"69","ECULLY":"69","EVEUX":"69","FONTAINES ST MARTIN":"69","GLEIZE":"69","GREZIEU LA VARENNE":"69","JULLIE":"69","LAMURE SUR AZERGUES":"69","LIMONEST":"69","LISSIEU":"69","LONGES":"69","MARCILLY D AZERGUES":"69","MEYS":"69","LA MULATIERE":"69","OINGT":"69","POLLIONNAY":"69","RANCHAL":"69","ST BONNET LE TRONCY":"69","ST CYR AU MONT D OR":"69","PRINCAY":"86","LA PUYE":"86","ST JULIEN L ARS":"86","ST LAON":"86","ST LAURENT DE JOURDES":"86","SANXAY":"86","SCORBE CLAIRVAUX":"86","THOLLET":"86","VAUX SUR VIENNE":"86","VIVONNE":"86","VOULEME":"86","VOUZAILLES":"86","ARNAC LA POSTE":"87","BUSSIERE POITEVINE":"87","CHAMBORET":"87","CHATEAU CHERVIX":"87","COGNAC LA FORET":"87","COUSSAC BONNEVAL":"87","CROMAC":"87","DARNAC":"87","DOURNAZAC":"87","FEYTIAT":"87","FLAVIGNAC":"87","FROMENTAL":"87","LES GRANDS CHEZEAUX":"87","JANAILHAC":"87","MAGNAC LAVAL":"87","LA MEYZE":"87","NIEUL":"87","NOUIC":"87","RAZES":"87","LA ROCHE L ABEILLE":"87","ROUSSAC":"87","ROYERES":"87","ST GENCE":"87","ST HILAIRE BONNEVAL":"87","ST JULIEN LE PETIT":"87","ST JUNIEN LES COMBES":"87","ST LAURENT SUR GORRE":"87","ST LEGER MAGNAZEIX":"87","ST MARTIN DE JUSSAC":"87","ST MATHIEU":"87","ST PRIEST TAURION":"87","ST SORNIN LA MARCHE":"87","ST YRIEIX SOUS AIXE":"87","SAUVIAT SUR VIGE":"87","SUSSAC":"87","VEYRAC":"87","VIDEIX":"87","LE VIGEN":"87","AMBACOURT":"88","AVRANVILLE":"88","BEAUFREMONT":"88","BELRUPT":"88","BLEVAINCOURT":"88","BRECHAINVILLE":"88","BULT":"88","CELLES SUR PLAINE":"88","LA CHAPELLE DEVANT BRUYERES":"88","CHATAS":"88","CHATEL SUR MOSELLE":"88","CHEF HAUT":"88","CLAUDON":"88","CLEREY LA COTE":"88","CORCIEUX":"88","COUSSEY":"88","DAMAS AUX BOIS":"88","DAMBLAIN":"88","DARNIEULLES":"88","DOMBROT LE SEC":"88","DOMPTAIL":"88","DOUNOUX":"88","ELOYES":"88","EPINAL":"88","FAUCOMPIERRE":"88","FIMENIL":"88","FRAPELLE":"88","FRENELLE LA GRANDE":"88","FRESSE SUR MOSELLE":"88","GENDREVILLE":"88","GERBEPAL":"88","GIRONCOURT SUR VRAINE":"88","ROCHEFORT SUR LA COTE":"52","ST THIEBAULT":"52","SARREY":"52","SIGNEVILLE":"52","SOMMANCOURT":"52","SOMMERECOURT":"52","THOL LES MILLIERES":"52","VAUDRECOURT":"52","VAUDREMONT":"52","VAUX SUR ST URBAIN":"52","VESAIGNES SOUS LAFAUCHE":"52","VESVRES SOUS CHALANCEY":"52","VILLE EN BLAISOIS":"52","VONCOURT":"52","VRAINCOURT":"52","ARGENTON NOTRE DAME":"53","ARGENTRE":"53","ARQUENAY":"53","BALLEE":"53","LA BAZOUGE DE CHEMERE":"53","BEAULIEU SUR OUDON":"53","BONCHAMP LES LAVAL":"53","LE BOURGNEUF LA FORET":"53","LA CHAPELLE RAINSOUIN":"53","CHATILLON SUR COLMONT":"53","COSSE LE VIVIEN":"53","COURCITE":"53","LA CROIXILLE":"53","LA CROPTE":"53","DEUX EVAILLES":"53","ERNEE":"53","FONTAINE COUVERTE":"53","FORCE":"53","LA GRAVELLE":"53","GREZ EN BOUERE":"53","LA HAIE TRAVERSAINE":"53","HARDANGES":"53","IZE":"53","JUBLAINS":"53","LAIGNE":"53","LAUNAY VILLIERS":"53","LOIGNE SUR MAYENNE":"53","LOUVIGNE":"53","MARIGNE PEUTON":"53","MONTENAY":"53","MONTFLOURS":"53","MOULAY":"53","NIAFLES":"53","LE PAS":"53","PEUTON":"53","LA ROUAUDIERE":"53","ST AUBIN FOSSE LOUVAIN":"53","ST CALAIS DU DESERT":"53","ST CHRISTOPHE DU LUAT":"53","ST GERMAIN D ANXURE":"53","ST GERMAIN LE GUILLAUME":"53","ST JEAN SUR MAYENNE":"53","ST LAURENT DES MORTIERS":"53","ST LOUP DU GAST":"53","ST MARTIN DE CONNEE":"53","ST POIX":"53","SENONNES":"53","THORIGNE EN CHARNIE":"53","VAIGES":"53","VIMARCE":"53","ABAUCOURT":"54","AFFLEVILLE":"54","ALLONDRELLE LA MALMAISON":"54","AMENONCOURT":"54","ANGOMONT":"54","ANOUX":"54","ANSAUVILLE":"54","ANTHELUPT":"54","ARMAUCOURT":"54","AVRIL":"54","AZELOT":"54","BATHELEMONT":"54","ST JEAN LIGOURE":"87","ST JOUVENT":"87","STE MARIE DE VAUX":"87","ST MARTIAL SUR ISOP":"87","ST MARTIN LE VIEUX":"87","ST MAURICE LES BROUSSES":"87","ST MEARD":"87","ST VITTE SUR BRIANCE":"87","ST YRIEIX LA PERCHE":"87","SEREILHAC":"87","TERSANNES":"87","THIAT":"87","VERNEUIL MOUSTIERS":"87","LES ABLEUVENETTES":"88","AINGEVILLE":"88","ANGLEMONT":"88","ANOULD":"88","AROFFE":"88","ARRENTES DE CORCIEUX":"88","AUZAINVILLIERS":"88","LA BAFFE":"88","BAINS LES BAINS":"88","BALLEVILLE":"88","BARBEY SEROUX":"88","BAUDRICOURT":"88","BAZIEN":"88","BEAUMENIL":"88","BONVILLET":"88","BOULAINCOURT":"88","BOUXURULLES":"88","LA BRESSE":"88","CHAMP LE DUC":"88","CHAVELOT":"88","CHENIMENIL":"88","CHERMISEY":"88","CLEURIE":"88","COMBRIMONT":"88","DENIPAIRE":"88","DIGNONVILLE":"88","DOGNEVILLE":"88","DOMBASLE DEVANT DARNEY":"88","DOMBASLE EN XAINTOIS":"88","DOMEVRE SUR AVIERE":"88","DOMEVRE SUR DURBION":"88","DOMMARTIN SUR VRAINE":"88","ENTRE DEUX EAUX":"88","ESCLES":"88","ESLEY":"88","ESTRENNES":"88","EVAUX ET MENIL":"88","FAUCONCOURT":"88","FIGNEVELLE":"88","FOMEREY":"88","GEMMELAINCOURT":"88","GIGNEY":"88","GIRMONT VAL D AJOL":"88","HADIGNY LES VERRIERES":"88","HAILLAINVILLE":"88","HARDANCOURT":"88","HAREVILLE":"88","HAROL":"88","HENNECOURT":"88","HURBACHE":"88","JESONVILLE":"88","LERRAIN":"88","LIGNEVILLE":"88","LUSSE":"88","MACONCOURT":"88","MADEGNEY":"88","MANDRAY":"88","MANDRES SUR VAIR":"88","MAZIROT":"88","MIDREVAUX":"88","MONTHUREUX SUR SAONE":"88","MOYENMOUTIER":"88","LA NEUVEVILLE SOUS MONTFORT":"88","PARGNY SOUS MUREAU":"88","PUNEROT":"88","REBEUVILLE":"88","REHAINCOURT":"88","ROBECOURT":"88","ROUVRES LA CHETIVE":"88","ST AME":"88","ST ETIENNE LES REMIREMONT":"88","ST MENGE":"88","MEDAN":"78","MEZIERES SUR SEINE":"78","MILLEMONT":"78","MONTALET LE BOIS":"78","MONTIGNY LE BRETONNEUX":"78","ORCEMONT":"78","ORGERUS":"78","PERDREAUVILLE":"78","ST ILLIERS LA VILLE":"78","ST ILLIERS LE BOIS":"78","ST LEGER EN YVELINES":"78","SARTROUVILLE":"78","TRAPPES":"78","LE TREMBLAY SUR MAULDRE":"78","LA VERRIERE":"78","VERSAILLES":"78","LA VILLENEUVE EN CHEVRIE":"78","L ABSIE":"79","ARDIN":"79","BOISME":"79","BOISSEROLLES":"79","LA BOISSIERE EN GATINE":"79","BOUILLE ST PAUL":"79","BOUSSAIS":"79","BRIOUX SUR BOUTONNE":"79","LE BUSSEAU":"79","CERIZAY":"79","CERSAY":"79","CHANTECORPS":"79","LA CHAPELLE ST LAURENT":"79","COMBRAND":"79","COURLAY":"79","CREZIERES":"79","ENSIGNE":"79","FAYE L ABBESSE":"79","LA FERRIERE EN PARTHENAY":"79","GOURGE":"79","GRANZAY GRIPT":"79","LAGEON":"79","LARGEASSE":"79","MAISONNAY":"79","MAUZE THOUARSAIS":"79","MELLERAN":"79","MISSE":"79","MONTALEMBERT":"79","MOUTIERS SOUS CHANTEMERLE":"79","POMPAIRE":"79","POUFFONDS":"79","POUGNE HERISSON":"79","PRIN DEYRANCON":"79","ST AUBIN DU PLAIN":"79","ST JEAN DE THOUARS":"79","ST JOUIN DE MARNES":"79","ST MAIXENT L ECOLE":"79","ST MARC LA LANDE":"79","ST MARTIN DE BERNEGOUE":"79","ST MARTIN DE MACON":"79","ST MAURICE ETUSSON":"79","SANSAIS":"79","SAURAIS":"79","SCILLE":"79","SEPVRET":"79","THENEZAY":"79","VANZAY":"79","VAUTEBIS":"79","VERRUYES":"79","VILLIERS EN BOIS":"79","VILLIERS EN PLAINE":"79","XAINTRAY":"79","ABLAINCOURT PRESSOIR":"80","ACHEUX EN VIMEU":"80","AIGNEVILLE":"80","AIZECOURT LE HAUT":"80","ALLONVILLE":"80","ST ETIENNE LA VARENNE":"69","ST GENIS LAVAL":"69","ST GEORGES DE RENEINS":"69","ST JEAN DE TOUSLAS":"69","ST JUST D AVRAY":"69","ST ROMAIN DE POPEY":"69","VAUXRENARD":"69","VILLEURBANNE":"69","PUSIGNAN":"69","SATHONAY VILLAGE":"69","TOUSSIEU":"69","LYON 06":"69","ABELCOURT":"70","ANDELARRE":"70","ANDELARROT":"70","ANGIREY":"70","ATTRICOURT":"70","AUGICOURT":"70","AUTHOISON":"70","LES AYNANS":"70","LA BASSE VAIVRE":"70","BELFAHY":"70","BELVERNE":"70","BETONCOURT LES BROTTE":"70","BEVEUGE":"70","BLONDEFONTAINE":"70","BONBOILLON":"70","BOUGNON":"70","BOURGUIGNON LES LA CHARITE":"70","BREUCHES":"70","BROYE LES LOUPS ET VERFONTAINE":"70","BUCEY LES TRAVES":"70","CHAGEY":"70","CHARMES ST VALBERT":"70","CHAUX LES PORT":"70","CHENEBIER":"70","CHENEVREY ET MOROGNE":"70","CLAIREGOUTTE":"70","CONTREGLISE":"70","CORNOT":"70","COURCHATON":"70","CREVANS ET LA CHAPELLE LES GRANGES":"70","DAMPIERRE LES CONFLANS":"70","FAYMONT":"70","FRASNE LE CHATEAU":"70","FREDERIC FONTAINE":"70","FRETIGNEY ET VELLOREILLE":"70","GENEVREY":"70","GEVIGNEY ET MERCEY":"70","GEZIER ET FONTENELAY":"70","GOURGEON":"70","GRATTERY":"70","HUGIER":"70","HURECOURT":"70","HYET":"70","LAMBREY":"70","LES MAGNY":"70","MAGNY JOBERT":"70","MAUSSANS":"70","MERCEY SUR SAONE":"70","MONTARLOT LES RIOZ":"70","MONTBOILLON":"70","MONTBOZON":"70","MONTCOURT":"70","MONTESSAUX":"70","NOROY LE BOURG":"70","ORICOURT":"70","OVANCHES":"70","PERROUSE":"70","PUSY ET EPENOUX":"70","RANZEVELLE":"70","LA ROSIERE":"70","GRIGNONCOURT":"88","GRUEY LES SURANCE":"88","HAGECOURT":"88","HERGUGNEY":"88","HOUEVILLE":"88","HOUSSERAS":"88","JAINVILLOTTE":"88","JORXEY":"88","LANGLEY":"88","LEGEVILLE ET BONFAYS":"88","MARAINVILLE SUR MADON":"88","MENIL EN XAINTOIS":"88","LE MENIL":"88","LE MONT":"88","MORIVILLE":"88","MORTAGNE":"88","NOMPATELIZE":"88","OELLEVILLE":"88","PALLEGNEY":"88","POUXEUX":"88","RACECOURT":"88","REMOMEIX":"88","ROMONT":"88","LES ROUGES EAUX":"88","ROVILLE AUX CHENES":"88","ROZIERES SUR MOUZON":"88","ST BENOIT LA CHIPOTTE":"88","ST DIE DES VOSGES":"88","ST OUEN LES PAREY":"88","ST PRANCHER":"88","SAULXURES LES BULGNEVILLE":"88","SENAIDE":"88","SOCOURT":"88","THEY SOUS MONTFORT":"88","TIGNECOURT":"88","TOLLAINCOURT":"88","TREMONZEY":"88","UXEGNEY":"88","UZEMAIN":"88","LE VAL D AJOL":"88","LES VALLOIS":"88","LE VALTIN":"88","VARMONZEY":"88","VAXONCOURT":"88","VILLONCOURT":"88","VILLOTTE":"88","VITTEL":"88","VIVIERS LE GRAS":"88","VRECOURT":"88","VROVILLE":"88","XAFFEVILLERS":"88","APPOIGNY":"89","ARGENTENAY":"89","ARTHONNAY":"89","BAGNEAUX":"89","BASSOU":"89","BAZARNES":"89","BEINE":"89","LA BELLIOLE":"89","BUSSY EN OTHE":"89","CHAMPCEVRAIS":"89","LA CHAPELLE VAUPELTEIGNE":"89","CHARENTENAY":"89","CHATEL GERARD":"89","CHEROY":"89","COULANGES SUR YONNE":"89","COURGIS":"89","COURLON SUR YONNE":"89","COURSON LES CARRIERES":"89","COURTOIN":"89","CRUZY LE CHATEL":"89","CUDOT":"89","DIGES":"89","EGLENY":"89","EGRISELLES LE BOCAGE":"89","EPINEUIL":"89","ESNON":"89","ETAIS LA SAUVIN":"89","ETIVEY":"89","LA FERTE LOUPIERE":"89","FONTAINE LA GAILLARDE":"89","BAYON":"54","BERNECOURT":"54","BEUVEILLE":"54","BLAINVILLE SUR L EAU":"54","BLENOD LES PONT A MOUSSON":"54","BLENOD LES TOUL":"54","BRUVILLE":"54","CHALIGNY":"54","CHAMPENOUX":"54","CHAOUILLEY":"54","CLEMERY":"54","CREVIC":"54","CRUSNES":"54","DAMELEVIERES":"54","DOMEVRE EN HAYE":"54","DOMEVRE SUR VEZOUZE":"54","DOMMARTIN LA CHAUSSEE":"54","EPLY":"54","ESSEY ET MAIZERAIS":"54","FILLIERES":"54","FLEVILLE DEVANT NANCY":"54","FLEVILLE LIXIERES":"54","FLIN":"54","FONTENOY SUR MOSELLE":"54","FRIAUVILLE":"54","FROVILLE":"54","GELLENONCOURT":"54","GEZONCOURT":"54","GOVILLER":"54","GRIPPORT":"54","HATRIZE":"54","HAUSSONVILLE":"54","HERBEVILLER":"54","HOEVILLE":"54","HOUDELMONT":"54","HUDIVILLER":"54","JOEUF":"54","JOLIVET":"54","JOUDREVILLE":"54","LABRY":"54","LANDRES":"54","LARONXE":"54","LAY ST REMY":"54","LEINTREY":"54","LONGLAVILLE":"54","LOROMONTZEY":"54","LUPCOURT":"54","MAGNIERES":"54","MAIDIERES":"54","MAILLY SUR SEILLE":"54","MALAVILLERS":"54","MANDRES AUX QUATRE TOURS":"54","MARTHEMONT":"54","MEXY":"54","MONTAUVILLE":"54","MONT SUR MEURTHE":"54","MORFONTAINE":"54","MORVILLE SUR SEILLE":"54","NEUFMAISONS":"54","NEUVILLER LES BADONVILLER":"54","NEUVILLER SUR MOSELLE":"54","NOMENY":"54","OGEVILLER":"54","ORMES ET VILLE":"54","PETIT FAILLY":"54","PETITMONT":"54","PONT ST VINCENT":"54","PULLIGNY":"54","PULNEY":"54","PUXE":"54","RAON LES LEAU":"54","RAVILLE SUR SANON":"54","REHERREY":"54","REPAIX":"54","RICHARDMENIL":"54","SAULXURES LES NANCY":"54","SERROUVILLE":"54","THEY SOUS VAUDEMONT":"54","THEZEY ST MARTIN":"54","THIAUCOURT REGNIEVILLE":"54","THIAVILLE SUR MEURTHE":"54","THUILLEY AUX GROSEILLES":"54","TIERCELET":"54","TRAMONT LASSUS":"54","SANDAUCOURT":"88","SENONGES":"88","SERECOURT":"88","SOULOSSE SOUS ST ELOPHE":"88","SURIAUVILLE":"88","THIEFOSSE":"88","THIRAUCOURT":"88","THUILLIERES":"88","TILLEUX":"88","TRAMPOT":"88","TRANQUEVILLE GRAUX":"88","VALFROICOURT":"88","VALLEROY LE SEC":"88","VAUBEXY":"88","VILLE SUR ILLON":"88","VILLOUXEL":"88","VIOCOURT":"88","VIOMENIL":"88","VIVIERS LES OFFROICOURT":"88","VOMECOURT":"88","VOMECOURT SUR MADON":"88","XAMONTARUPT":"88","ZINCOURT":"88","ANCY LE LIBRE":"89","ANNAY LA COTE":"89","ANNEOT":"89","ANNOUX":"89","ARGENTEUIL SUR ARMANCON":"89","ASNIERES SOUS BOIS":"89","BAON":"89","BIERRY LES BELLES FONTAINES":"89","BOEURS EN OTHE":"89","BRANCHES":"89","LA CHAPELLE SUR OREUSE":"89","LES CLERIMOIS":"89","COULOURS":"89","COURGENAY":"89","COURTOIS SUR YONNE":"89","DISSANGIS":"89","DIXMONT":"89","DOLLOT":"89","DOMECY SUR CURE":"89","DRUYES LES BELLES FONTAINES":"89","ETIGNY":"89","FOURNAUDIN":"89","GISY LES NOBLES":"89","GUILLON":"89","JOIGNY":"89","JOUX LA VILLE":"89","LAILLY":"89","LICHERES PRES AIGREMONT":"89","LICHERES SUR YONNE":"89","LIXY":"89","LUCY SUR YONNE":"89","MARMEAUX":"89","MIGENNES":"89","MOLINONS":"89","MONTIGNY LA RESLE":"89","MONT ST SULPICE":"89","MOUTIERS EN PUISAYE":"89","NUITS":"89","PRECY SUR VRIN":"89","QUENNE":"89","RAVIERES":"89","ROFFEY":"89","SAINPUITS":"89","ST ANDRE EN TERRE PLAINE":"89","ST BRANCHER":"89","ST BRIS LE VINEUX":"89","ST FARGEAU":"89","ST MAURICE LE VIEIL":"89","SAUVIGNY LE BOIS":"89","SAVIGNY SUR CLAIRIS":"89","SENAN":"89","SENNEVOY LE HAUT":"89","SENS":"89","SERMIZELLES":"89","LES SIEGES":"89","SOMMECAISE":"89","AMIENS":"80","ANDAINVILLE":"80","AUBERCOURT":"80","AUTHEUX":"80","AUTHUILLE":"80","AVELESGES":"80","BAZENTIN":"80","BEAUCOURT EN SANTERRE":"80","BEAUCOURT SUR L HALLUE":"80","BELLANCOURT":"80","BELLOY SUR SOMME":"80","BERNES":"80","BERNY EN SANTERRE":"80","BOUVAINCOURT SUR BRESLE":"80","BRAY SUR SOMME":"80","BRIQUEMESNIL FLOXICOURT":"80","BROUCHY":"80","BRUCAMPS":"80","BUIGNY ST MACLOU":"80","BUIRE SUR L ANCRE":"80","BUSSU":"80","BUVERCHY":"80","CAMBRON":"80","CAMPS EN AMIENOIS":"80","CANNESSIERES":"80","CANTIGNY":"80","CAOURS":"80","CARREPUIS":"80","CARTIGNY":"80","CAVILLON":"80","CAYEUX SUR MER":"80","CERISY BULEUX":"80","CERISY":"80","LA CHAVATTE":"80","CHIPILLY":"80","CLAIRY SAULCHOIX":"80","COMBLES":"80","CONTOIRE":"80","COULLEMELLE":"80","COURCELLES AU BOIS":"80","CRAMONT":"80","CRECY EN PONTHIEU":"80","CRESSY OMENCOURT":"80","CROIX MOLIGNEAUX":"80","DERNANCOURT":"80","DEVISE":"80","DRUCAT":"80","ENGLEBELMER":"80","EPPEVILLE":"80","ESSERTAUX":"80","ESTREES LES CRECY":"80","ETRICOURT MANANCOURT":"80","LA FALOISE":"80","FEUQUIERES EN VIMEU":"80","FLERS SUR NOYE":"80","FLESSELLES":"80","FONTAINE LES CAPPY":"80","FONTAINE SUR MAYE":"80","FONTAINE SUR SOMME":"80","FOREST MONTIERS":"80","FORT MAHON PLAGE":"80","FOSSEMANANT":"80","FRAMERVILLE RAINECOURT":"80","FRANLEU":"80","FRANSART":"80","FRECHENCOURT":"80","FRIAUCOURT":"80","FROYELLES":"80","GAPENNES":"80","GORENFLOS":"80","GRAND LAVIERS":"80","GRATIBUS":"80","GUERBIGNY":"80","GUESCHART":"80","GUILLEMONT":"80","HALLIVILLERS":"80","HANGARD":"80","HARBONNIERES":"80","HEILLY":"80","HEM HARDINVAL":"80","HEM MONACU":"80","HENENCOURT":"80","HUCHENNEVILLE":"80","HUPPY":"80","IGNAUCOURT":"80","IRLES":"80","ROSIERES SUR MANCE":"70","STE MARIE EN CHANOIS":"70","SAPONCOURT":"70","SCEY SUR SAONE ET ST ALBIN":"70","SEMMADON":"70","SENONCOURT":"70","SERVANCE":"70","SOING CUBRY CHARENTENAY":"70","SORANS LES BREUREY":"70","TRAITIEFONTAINE":"70","TRAVES":"70","TREMOINS":"70","VALAY":"70","VALLEROIS LORIOZ":"70","VANDELANS":"70","VANTOUX ET LONGEVELLE":"70","VELET":"70","VERNOIS SUR MANCE":"70","LA VERNOTTE":"70","VILLARS LE PAUTEL":"70","VILLERS BOUTON":"70","VILORY":"70","VYANS LE VAL":"70","VY LE FERROUX":"70","VY LES LURE":"70","VY LES RUPT":"70","ALUZE":"71","BANTANGES":"71","BELLEVESVRE":"71","BERGESSERIN":"71","BERZE LA VILLE":"71","BISSY SUR FLEY":"71","BOURG LE COMTE":"71","BRIENNE":"71","CERON":"71","CHAINTRE":"71","CHAMBILLY":"71","CHAMPAGNY SOUS UXELLES":"71","LA CHAPELLE ST SAUVEUR":"71","LA CHAPELLE THECLE":"71","LA CHARMEE":"71","CHAROLLES":"71","CHATEL MORON":"71","CHATENOY LE ROYAL":"71","CHENAY LE CHATEL":"71","CHEVAGNY LES CHEVRIERES":"71","CHISSEY EN MORVAN":"71","CHISSEY LES MACON":"71","LA CLAYETTE":"71","CLUNY":"71","CORDESSE":"71","CORTEVAIX":"71","CRESSY SUR SOMME":"71","LE CREUSOT":"71","CRUZILLE":"71","CUISERY":"71","CULLES LES ROCHES":"71","CURDIN":"71","DAMEREY":"71","DAMPIERRE EN BRESSE":"71","DAVAYE":"71","DEMIGNY":"71","DEZIZE LES MARANGES":"71","DOMPIERRE LES ORMES":"71","DONZY LE NATIONAL":"71","DONZY LE PERTUIS":"71","DRACY LE FORT":"71","DRACY LES COUCHES":"71","ECUELLES":"71","EPERTULLY":"71","FLEY":"71","FRANGY EN BRESSE":"71","FRONTENARD":"71","GERGY":"71","GERMAGNY":"71","GIGNY SUR SAONE":"71","GREVILLY":"71","HUILLY SUR SEILLE":"71","LACROST":"71","MALTAT":"71","MARTIGNY LE COMTE":"71","MONTCEAU LES MINES":"71","MONTMELARD":"71","MONT ST VINCENT":"71","MOROGES":"71","FONTENAY PRES CHABLIS":"89","FONTENAY PRES VEZELAY":"89","JAULGES":"89","LAIN":"89","LIGNY LE CHATEL":"89","MAILLY LA VILLE":"89","MAILLY LE CHATEAU":"89","MARSANGY":"89","MENADES":"89","MERRY LA VALLEE":"89","PAILLY":"89","PARON":"89","PAROY EN OTHE":"89","PERRIGNY SUR ARMANCON":"89","PIERRE PERTHUIS":"89","PISY":"89","PREGILBERT":"89","PREHY":"89","ROGNY LES SEPT ECLUSES":"89","STE COLOMBE SUR LOING":"89","ST LOUP D ORDON":"89","STE MAGNANCE":"89","ST MARTIN SUR ARMANCON":"89","ST MAURICE THIZOUAILLE":"89","ST MORE":"89","STE PALLAYE":"89","SAINTS EN PUISAYE":"89","ST SAUVEUR EN PUISAYE":"89","ST SEROTIN":"89","STE VERTU":"89","SAVIGNY EN TERRE PLAINE":"89","SEIGNELAY":"89","SEMENTRON":"89","SORMERY":"89","TAINGY":"89","TANNERRE EN PUISAYE":"89","THAROT":"89","TISSEY":"89","TRICHEY":"89","TRUCY SUR YONNE":"89","TURNY":"89","VASSY SOUS PISY":"89","VAUDEURS":"89","VENOUSE":"89","VERON":"89","VEZANNES":"89","VILLECIEN":"89","VILLENEUVE L ARCHEVEQUE":"89","VILLIERS VINEUX":"89","VILLON":"89","VIREAUX":"89","VOLGRE":"89","ANGEOT":"90","ANJOUTEY":"90","BANVILLARS":"90","BAVILLIERS":"90","BESSONCOURT":"90","BOTANS":"90","CHEVREMONT":"90","FAVEROIS":"90","GROSNE":"90","LAMADELEINE VAL DES ANGES":"90","LARIVIERE":"90","MORVILLARS":"90","MOVAL":"90","RIERVESCEMONT":"90","ST DIZIER L EVEQUE":"90","SUARCE":"90","VESCEMONT":"90","VEZELOIS":"90","ARRANCOURT":"91","AUVERNAUX":"91","BAULNE":"91","BOIGNEVILLE":"91","BOISSY SOUS ST YON":"91","BOULLAY LES TROUX":"91","TRAMONT ST ANDRE":"54","TREMBLECOURT":"54","VANDELEVILLE":"54","VAUDEMONT":"54","VILCEY SUR TREY":"54","VILLE EN VERMOIS":"54","VITRIMONT":"54","VOINEMONT":"54","HAN DEVANT PIERREPONT":"54","ABAINVILLE":"55","AMEL SUR L ETANG":"55","AUTREVILLE ST LAMBERT":"55","AVILLERS STE CROIX":"55","BANNONCOURT":"55","BAUDIGNECOURT":"55","BAUDONVILLIERS":"55","BAZEILLES SUR OTHAIN":"55","BELLERAY":"55","BOINVILLE EN WOEVRE":"55","BONNET":"55","LE BOUCHON SUR SAULX":"55","BOUQUEMONT":"55","BRANDEVILLE":"55","BRAQUIS":"55","BREHEVILLE":"55","BROUSSEY RAULECOURT":"55","BUREY EN VAUX":"55","CHALAINES":"55","CHARNY SUR MEUSE":"55","CHARPENTRY":"55","CHAUVENCY LE CHATEAU":"55","CHAUVONCOURT":"55","CHEPPY":"55","COMBRES SOUS LES COTES":"55","COMMERCY":"55","CULEY":"55","DANNEVOUX":"55","DUN SUR MEUSE":"55","ECOUVIEZ":"55","FORGES SUR MEUSE":"55","FREMEREVILLE SOUS LES COTES":"55","FRESNES AU MONT":"55","FRESNES EN WOEVRE":"55","GENICOURT SUR MEUSE":"55","GERY":"55","GRIMAUCOURT EN WOEVRE":"55","GRIMAUCOURT PRES SAMPIGNY":"55","HEIPPES":"55","HEVILLIERS":"55","JAMETZ":"55","JONVILLE EN WOEVRE":"55","LAHAYMEIX":"55","LANEUVILLE SUR MEUSE":"55","LEMMES":"55","LEROUVILLE":"55","LION DEVANT DUN":"55","LOISEY":"55","MALANCOURT":"55","MANDRES EN BARROIS":"55","MANHEULLES":"55","MARVILLE":"55","MAULAN":"55","MAXEY SUR VAISE":"55","MENIL LA HORGNE":"55","MOGEVILLE":"55","MONTIGNY DEVANT SASSEY":"55","MONTMEDY":"55","MONTPLONNE":"55","MOUILLY":"55","MOULINS ST HUBERT":"55","NEUVILLE LES VAUCOULEURS":"55","PAGNY LA BLANCHE COTE":"55","PIERREFITTE SUR AIRE":"55","RAMBLUZIN ET BENOITE VAUX":"55","RARECOURT":"55","RIAVILLE":"55","RICHECOURT":"55","RIGNY LA SALLE":"55","ROBERT ESPAGNE":"55","ROMAGNE SOUS LES COTES":"55","RUPT AUX NONAINS":"55","VALLERY":"89","VENIZY":"89","VENOY":"89","VERNOY":"89","VEZELAY":"89","VILLEPERROT":"89","VILLIERS LOUIS":"89","VINCELOTTES":"89","VINNEUF":"89","VOUTENAY SUR CURE":"89","ARGIESANS":"90","COURTELEVANT":"90","ESSERT":"90","FECHE L EGLISE":"90","FLORIMONT":"90","JONCHEREY":"90","LACHAPELLE SOUS CHAUX":"90","LEBETAIN":"90","LEPUIX":"90","MENONCOURT":"90","MEROUX":"90","MEZIRE":"90","PETIT CROIX":"90","ST GERMAIN LE CHATELET":"90","VELLESCOT":"90","BALLANCOURT SUR ESSONNE":"91","BOISSY LA RIVIERE":"91","BOUTIGNY SUR ESSONNE":"91","BROUY":"91","CHALOU MOULINEUX":"91","CHAMARANDE":"91","CORBREUSE":"91","EGLY":"91","EPINAY SOUS SENART":"91","LA FORET STE CROIX":"91","GIRONVILLE SUR ESSONNE":"91","GOMETZ LA VILLE":"91","GUIGNEVILLE SUR ESSONNE":"91","LEUDEVILLE":"91","LISSES":"91","LONGPONT SUR ORGE":"91","MESPUITS":"91","MONNERVILLE":"91","MONTGERON":"91","MONTLHERY":"91","MORSANG SUR SEINE":"91","PLESSIS ST BENOIST":"91","QUINCY SOUS SENART":"91","ST MAURICE MONTCOURONNE":"91","SOISY SUR SEINE":"91","SOUZY LA BRICHE":"91","TIGERY":"91","VARENNES JARCY":"91","VERRIERES LE BUISSON":"91","VIDELLES":"91","LA VILLE DU BOIS":"91","VILLIERS SUR ORGE":"91","ASNIERES SUR SEINE":"92","BOULOGNE BILLANCOURT":"92","BOURG LA REINE":"92","CLICHY":"92","LA GARENNE COLOMBES":"92","GENNEVILLIERS":"92","HAUTEVILLE LOMPNES":"01","FERNEY VOLTAIRE":"01","GERMAGNAT":"01","FOISSIAT":"01","IZERNORE":"01","DOUVRES":"01","HOSTIAZ":"01","GRILLY":"01","CORMORANCHE SUR SAONE":"01","CRAS SUR REYSSOUZE":"01","DIVONNE LES BAINS":"01","COURMANGOUX":"01","CHAVORNAY":"01","CHEVROUX":"01","JUMEL":"80","LAMOTTE BULEUX":"80","LANCHES ST HILAIRE":"80","LAVIEVILLE":"80","LIANCOURT FOSSE":"80","LIERAMONT":"80","LONG":"80","LONGPRE LES CORPS SAINTS":"80","LONGUEAU":"80","LE MAZIS":"80","MEHARICOURT":"80","MESNIL MARTINSART":"80","MOLLIENS DREUIL":"80","MONTIGNY LES JONGLEURS":"80","MOREUIL":"80","NAMPTY":"80","NEUILLY L HOPITAL":"80","NOUVION":"80","NOYELLES SUR MER":"80","OMIECOURT":"80","OUTREBOIS":"80","PERNOIS":"80","PIERREPONT SUR AVRE":"80","PLACHY BUYON":"80","PONT DE METZ":"80","PONT REMY":"80","PORT LE GRAND":"80","POTTE":"80","QUERRIEU":"80","QUESNOY SUR AIRAINES":"80","ROISEL":"80","ROUY LE GRAND":"80","ROUY LE PETIT":"80","SAILLY FLIBEAUCOURT":"80","SAINS EN AMIENOIS":"80","ST ACHEUL":"80","ST AUBIN MONTENOY":"80","ST AUBIN RIVIERE":"80","ST GERMAIN SUR BRESLE":"80","ST MAXENT":"80","ST QUENTIN LA MOTTE CROIX AU BAILLY":"80","SALEUX":"80","SAULCHOY SOUS POIX":"80","SEUX":"80","SOREL EN VIMEU":"80","SOURDON":"80","THEZY GLIMONT":"80","THIEPVAL":"80","THIEULLOY LA VILLE":"80","TILLOY FLORIVILLE":"80","LE TITRE":"80","TOURS EN VIMEU":"80","TOUTENCOURT":"80","LE TRANSLAY":"80","UGNY L EQUIPEE":"80","VALINES":"80","VECQUEMONT":"80","VERGIES":"80","LA VICOGNE":"80","VILLE SUR ANCRE":"80","VOYENNES":"80","VRELY":"80","WARLOY BAILLON":"80","YVRENCH":"80","AIGUEFONDE":"81","ALGANS":"81","ANDILLAC":"81","AUSSILLON":"81","BELLEGARDE MARSAL":"81","BOISSEZON":"81","CAGNAC LES MINES":"81","CAMBON":"81","DENAT":"81","FENOLS":"81","FLORENTIN":"81","FREJAIROLLES":"81","GIROUSSENS":"81","GRAULHET":"81","LABASTIDE DE LEVIS":"81","LABASTIDE DENAT":"81","LABASTIDE GABAUSSE":"81","LACABAREDE":"81","LACAPELLE PINET":"81","MUSSY SOUS DUN":"71","NANTON":"71","NEUVY GRANDCHAMP":"71","OSLON":"71","OUROUX SUR SAONE":"71","OZENAY":"71","PONTOUX":"71","LE PULEY":"71","RATENELLE":"71","LA ROCHE VINEUSE":"71","ROUSSILLON EN MORVAN":"71","ST CHRISTOPHE EN BRESSE":"71","ST CHRISTOPHE EN BRIONNAIS":"71","ST CLEMENT SUR GUYE":"71","ST DIDIER EN BRESSE":"71","ST DIDIER EN BRIONNAIS":"71","ST GERVAIS SUR COUCHES":"71","ST JEAN DE VAUX":"71","ST JULIEN DE CIVRY":"71","ST LAURENT EN BRIONNAIS":"71","FRUCOURT":"80","GLISY":"80","GRATTEPANCHE":"80","GRIVESNES":"80","GRUNY":"80","GUEUDECOURT":"80","GUIGNEMICOURT":"80","GUILLAUCOURT":"80","HAM":"80","HARPONVILLE":"80","HERBECOURT":"80","HUMBERCOURT":"80","LAMARONDE":"80","LESBOEUFS":"80","LIERCOURT":"80","LUCHEUX":"80","MAILLY MAILLET":"80","MAIZICOURT":"80","MARCHE ALLOUARDE":"80","MARESTMONTIERS":"80","MARQUIVILLERS":"80","MATIGNY":"80","MERICOURT EN VIMEU":"80","MESNIL EN ARROUAISE":"80","MILLENCOURT EN PONTHIEU":"80","MONSURES":"80","MONTAGNE FAYEL":"80","FIEFFES MONTRELET":"80","MORVILLERS ST SATURNIN":"80","MOUFLIERES":"80","NAMPONT":"80","NAOURS":"80","NOYELLES EN CHAUSSEE":"80","OCCOCHES":"80","PICQUIGNY":"80","PISSY":"80","PYS":"80","QUESNOY LE MONTANT":"80","QUIVIERES":"80","RAINCHEVAL":"80","RAINNEVILLE":"80","RIENCOURT":"80","ROGY":"80","RONSSOY":"80","ROSIERES EN SANTERRE":"80","RUBEMPRE":"80","ST LEGER LES AUTHIE":"80","ST VALERY SUR SOMME":"80","ST VAAST EN CHAUSSEE":"80","SAISSEVAL":"80","SENLIS LE SEC":"80","SOREL":"80","TEMPLEUX LA FOSSE":"80","TERTRY":"80","VAUCHELLES LES AUTHIE":"80","VILLERS SOUS AILLY":"80","VILLERS SUR AUTHIE":"80","VIRONCHAUX":"80","WARSY":"80","WIRY AU MONT":"80","WOIREL":"80","YAUCOURT BUSSUS":"80","BRUYERES LE CHATEL":"91","CHAMPMOTTEUX":"91","CHATIGNONVILLE":"91","CHAUFFOUR LES ETRECHY":"91","CHEPTAINVILLE":"91","COURSON MONTELOUP":"91","D HUISON LONGUEVILLE":"91","ESTOUCHES":"91","LA FERTE ALAIS":"91","FONTAINE LA RIVIERE":"91","GIF SUR YVETTE":"91","GUILLERVAL":"91","JUVISY SUR ORGE":"91","MAROLLES EN HUREPOIX":"91","MAUCHAMPS":"91","MENNECY":"91","MOIGNY SUR ECOLE":"91","NAINVILLE LES ROCHES":"91","ORMOY LA RIVIERE":"91","PECQUEUSE":"91","ST YON":"91","TORFOU":"91","VIGNEUX SUR SEINE":"91","VILLEBON SUR YVETTE":"91","VILLEMOISSON SUR ORGE":"91","VILLENEUVE SUR AUVERS":"91","VILLIERS LE BACLE":"91","VIRY CHATILLON":"91","WISSOUS":"91","COLOMBES":"92","GARCHES":"92","ISSY LES MOULINEAUX":"92","ST LEGER SUR DHEUNE":"71","ST MARTIN DE SALENCEY":"71","ST PIERRE DE VARENNES":"71","ST SERNIN DU PLAIN":"71","SAUNIERES":"71","SAVIGNY EN REVERMONT":"71","SAVIGNY SUR SEILLE":"71","LA CELLE EN MORVAN":"71","SENNECEY LE GRAND":"71","SIMARD":"71","SOLOGNY":"71","SOMMANT":"71","VARENNES ST SAUVEUR":"71","VARENNES SOUS DUN":"71","VAUX EN PRE":"71","VERDUN SUR LE DOUBS":"71","VEROSVRES":"71","ARCONNAY":"72","ARNAGE":"72","ASNIERES SUR VEGRE":"72","AUVERS LE HAMON":"72","AUVERS SOUS MONTFAUCON":"72","BERNAY EN CHAMPAGNE":"72","BOESSE LE SEC":"72","BRETTE LES PINS":"72","BRULON":"72","CHAMPROND":"72","LA CHAPELLE HUON":"72","CONGE SUR ORNE":"72","CONLIE":"72","CORMES":"72","COURCEBOEUFS":"72","COURGAINS":"72","DOUILLET":"72","EVAILLE":"72","FERCE SUR SARTHE":"72","LE GREZ":"72","JUIGNE SUR SARTHE":"72","MAIGNE":"72","MALICORNE SUR SARTHE":"72","MAREIL SUR LOIR":"72","MAROLLETTE":"72","MAYET":"72","MEZIERES SUR PONTHOUIN":"72","MONCE EN BELIN":"72","NAUVAY":"72","NEUVY EN CHAMPAGNE":"72","NOGENT SUR LOIR":"72","RUPT SUR OTHAIN":"55","ST AMAND SUR ORNAIN":"55","ST AUBIN SUR AIRE":"55","ST MIHIEL":"55","SASSEY SUR MEUSE":"55","SIVRY SUR MEUSE":"55","SOMMELONNE":"55","TAILLANCOURT":"55","THONNE LE THIL":"55","VACHERAUVILLE":"55","VADELAINCOURT":"55","VAUBECOURT":"55","VELOSNES":"55","VILLE DEVANT BELRAIN":"55","VILLE EN WOEVRE":"55","WATRONVILLE":"55","WOEL":"55","AMBON":"56","ARRADON":"56","BADEN":"56","BERRIC":"56","BRANDIVY":"56","CALAN":"56","CAMOEL":"56","CROIXANVEC":"56","ETEL":"56","FEREL":"56","GOURHEL":"56","GUERN":"56","GUIDEL":"56","LANGONNET":"56","LARMOR BADEN":"56","LOCMINE":"56","MALANSAC":"56","MELRAND":"56","MONTERBLANC":"56","MONTERTELOT":"56","NOYAL PONTIVY":"56","PERSQUEN":"56","PLOERMEL":"56","PLOUGOUMELEN":"56","PLUMELIN":"56","PLUMERGAT":"56","PONTIVY":"56","QUELNEUC":"56","QUESTEMBERT":"56","QUEVEN":"56","REMINIAC":"56","RIANTEC":"56","LA ROCHE BERNARD":"56","ROCHEFORT EN TERRE":"56","ROUDOUALLEC":"56","ST CARADEC TREGOMEL":"56","ST DOLAY":"56","ST JEAN LA POTERIE":"56","ST MALO DES TROIS FONTAINES":"56","ST SERVANT":"56","ST THURIAU":"56","ST VINCENT SUR OUST":"56","SARZEAU":"56","SAUZON":"56","SENE":"56","SILFIAC":"56","THEIX NOYALO":"56","VANNES":"56","LA VRAIE CROIX":"56","STE ANNE D AURAY":"56","ABONCOURT SUR SEILLE":"57","ACHEN":"57","ALGRANGE":"57","ALTVILLER":"57","AMELECOURT":"57","ANCY DORNOT":"57","APACH":"57","ARRIANCE":"57","AZOUDANGE":"57","BACOURT":"57","LE BAN ST MARTIN":"57","BARST":"57","BAZONCOURT":"57","DAGNEUX":"01","CONFORT":"01","COURTES":"01","DORTAN":"01","MANTENAY MONTLIN":"01","MEILLONNAS":"01","MARSONNAS":"01","JUJURIEUX":"01","MARIGNIEU":"01","LANCRANS":"01","JASSERON":"01","MAILLAT":"01","LELEX":"01","LAIZ":"01","ST TRIVIER SUR MOIGNANS":"01","ST TRIVIER DE COURTES":"01","SULIGNAT":"01","VERNOUX":"01","MURS ET GELIGNIEUX":"01","LES NEYROLLES":"01","PONT D AIN":"01","MONTANGES":"01","MEXIMIEUX":"01","MERIGNAT":"01","MONTCET":"01","NIEVROZ":"01","ONCIEU":"01","BELLEGARDE SUR VALSERINE":"01","BREGNIER CORDON":"01","BELLEYDOUX":"01","BETTANT":"01","BLYES":"01","ST MAURICE DE GOURDANS":"01","ST NIZIER LE BOUCHOUX":"01","ST MAURICE DE REMENS":"01","ST NIZIER LE DESERT":"01","ST MARTIN DU FRENE":"01","SIMANDRE SUR SURAN":"01","STE JULIE":"01","SAUVERNY":"01","SEGNY":"01","CHATILLON EN MICHAILLE":"01","LA CHAPELLE DU CHATELARD":"01","CHALLES LA MONTAGNE":"01","CHAVEYRIAT":"01","CEYZERIEU":"01","CHALAMONT":"01","CEIGNES":"01","CHANAY":"01","ST ANDRE SUR VIEUX JONC":"01","ST GERMAIN SUR RENON":"01","ST DIDIER D AUSSIAT":"01","ST DENIS EN BUGEY":"01","ST ANDRE D HUIRIAT":"01","PREMEYZEL":"01","POUILLAT":"01","ST CHAMP":"01","PRIAY":"01","VILLEMOTIER":"01","VILLES":"01","ACHERY":"02","AGNICOURT ET SECHELLES":"02","AULNOIS SOUS LAON":"02","ANY MARTIN RIEUX":"02","ASSIS SUR SERRE":"02","AUBENTON":"02","AIZELLES":"02","ARRANCY":"02","ATTILLY":"02","ST PIERRE D ARGENCON":"05","LE SAUZE DU LAC":"05","VALLOUISE":"05","BENDEJUN":"06","VEYNES":"05","THEUS":"05","ST MARTIN D ENTRAUNES":"06","ST DALMAS LE SELVAGE":"06","ST ETIENNE DE TINEE":"06","ST JEAN CAP FERRAT":"06","ST LAURENT DU VAR":"06","ROQUESTERON":"06","RIGAUD":"06","MARIE":"06","LACOUGOTTE CADOUL":"81","LASGRAISSES":"81","LAUTREC":"81","LEMPAUT":"81","LESCOUT":"81","LESCURE D ALBIGEOIS":"81","MARSSAC SUR TARN":"81","LE MASNAU MASSUGUIES":"81","MASSAC SERAN":"81","MISSECLE":"81","MONESTIES":"81","MONTDRAGON":"81","MONT ROC":"81","MONTROSIER":"81","MOULIN MAGE":"81","MOUZENS":"81","MOUZIEYS TEULET":"81","NAGES":"81","ORBAN":"81","PALLEVILLE":"81","PAULINET":"81","PEYROLE":"81","PONT DE LARN":"81","PRATVIEL":"81","RAYSSAC":"81","ROUSSAYROLLES":"81","ST AMANCET":"81","ST BENOIT DE CARMAUX":"81","ST JEAN DE MARCEL":"81","ST JEAN DE VALS":"81","ST JULIEN GAULENE":"81","ST PAUL CAP DE JOUX":"81","ST PIERRE DE TRIVISY":"81","ST SALVI DE CARCAVES":"81","LE SEQUESTRE":"81","SERVIES":"81","TAIX":"81","TERRE CLAPIER":"81","VERDALLE":"81","VILLEFRANCHE D ALBIGEOIS":"81","VILLENEUVE LES LAVAUR":"81","VILLENEUVE SUR VERE":"81","AUVILLAR":"82","BEAUMONT DE LOMAGNE":"82","BOURG DE VISA":"82","CASTELSAGRAT":"82","CASTERA BOUZET":"82","CAZES MONDENARD":"82","COMBEROUGER":"82","DIEUPENTALE":"82","ESCATALENS":"82","FAUDOAS":"82","GOAS":"82","LARRAZET":"82","MIRAMONT DE QUERCY":"82","MONTBARLA":"82","ST ANTONIN NOBLE VAL":"82","ST CIRICE":"82","ST NICOLAS DE LA GRAVE":"82","ST VINCENT D AUTEJAC":"82","ST VINCENT LESPINASSE":"82","VERDUN SUR GARONNE":"82","VILLEBRUMIER":"82","ARTIGNOSC SUR VERDON":"83","BAGNOLS EN FORET":"83","BARGEMON":"83","BAUDINARD SUR VERDON":"83","CABASSE":"83","LE CANNET DES MAURES":"83","CLAVIERS":"83","COGOLIN":"83","COMPS SUR ARTUBY":"83","EVENOS":"83","FAYENCE":"83","LA LONDE LES MAURES":"83","OLLIOULES":"83","PIERREFEU DU VAR":"83","PUGET SUR ARGENS":"83","ROCBARON":"83","YZENGREMER":"80","BERLATS":"81","BLAYE LES MINES":"81","CAMBOUNES":"81","CAMBOUNET SUR LE SOR":"81","LES CAMMAZES":"81","CARBES":"81","CASTELNAU DE MONTMIRAL":"81","CRESPINET":"81","CURVALLE":"81","FAYSSAC":"81","FAUCH":"81","FIAC":"81","GAILLAC":"81","GARREVAQUES":"81","ITZAC":"81","JOUQUEVIEL":"81","LABASTIDE ROUAIROUX":"81","LABOUTARIE":"81","LACAUNE":"81","LAGARDIOLLE":"81","LAGRAVE":"81","LASFAILLADES":"81","LISLE SUR TARN":"81","MAILHOC":"81","MASSAGUEL":"81","MILHAVET":"81","MONTPINIER":"81","POUDIS":"81","PUECHOURSI":"81","RABASTENS":"81","ST JULIEN DU PUY":"81","ST MICHEL DE VAX":"81","LA SAUZIERE ST JEAN":"81","LE SEGUR":"81","SIEURAC":"81","SOUEL":"81","TEILLET":"81","TREVIEN":"81","VABRE":"81","VALENCE D ALBIGEOIS":"81","BELVEZE":"82","BESSENS":"82","BIOULE":"82","CASTELSARRASIN":"82","LE CAUSE":"82","ESPALAIS":"82","GIMAT":"82","GINALS":"82","GOUDOURVILLE":"82","L HONOR DE COS":"82","LAGUEPIE":"82","LAMOTHE CAPDEVILLE":"82","LAVAURETTE":"82","LA VILLE DIEU DU TEMPLE":"82","LEOJAC":"82","LIZAC":"82","LOZE":"82","MALAUSE":"82","MANSONVILLE":"82","MAS GRENIER":"82","MEAUZAC":"82","MERLES":"82","MONCLAR DE QUERCY":"82","MONTAGUDET":"82","MONTBARTIER":"82","NEGREPELISSE":"82","NOHIC":"82","PUYCORNET":"82","PUYLAGARDE":"82","PUYLAROQUE":"82","ST JEAN DU BOUZET":"82","STE JULIETTE":"82","TOUFFAILLES":"82","TREJOULS":"82","VAREN":"82","VAZERAC":"82","VIGUERON":"82","AUPS":"83","BARJOLS":"83","NOUANS":"72","PANON":"72","PERAY":"72","PIZIEUX":"72","MONTFORT LE GESNOIS":"72","PREVELLES":"72","PRUILLE LE CHETIF":"72","ROEZE SUR SARTHE":"72","RUAUDIN":"72","RUILLE EN CHAMPAGNE":"72","ST CHRISTOPHE EN CHAMPAGNE":"72","ST GEORGES DU ROSAY":"72","ST GERMAIN D ARCE":"72","ST MARS DE LOCQUENAY":"72","ST MARS D OUTILLE":"72","ST MICHEL DE CHAVAIGNES":"72","ST OUEN EN CHAMPAGNE":"72","ST PIERRE DES ORMES":"72","ST VINCENT DU LOROUER":"72","SAVIGNE L EVEQUE":"72","SOUGE LE GANELON":"72","SOUVIGNE SUR SARTHE":"72","TASSILLE":"72","TENNIE":"72","THOREE LES PINS":"72","VAAS":"72","VEZOT":"72","VILLAINES LA CARELLE":"72","VOUVRAY SUR LOIR":"72","AIGUEBELETTE LE LAC":"73","AIGUEBELLE":"73","AIGUEBLANCHE":"73","AILLON LE JEUNE":"73","AIX LES BAINS":"73","ALBERTVILLE":"73","ALBIEZ LE JEUNE":"73","ARGENTINE":"73","ARVILLARD":"73","AYN":"73","BOURGET EN HUILE":"73","BRIDES LES BAINS":"73","BRISON ST INNOCENT":"73","CESARCHES":"73","CHAMBERY":"73","LES CHAPELLES":"73","LES CHAVANNES EN MAURIENNE":"73","COISE ST JEAN PIED GAUTHIER":"73","ENTREMONT LE VIEUX":"73","FRANCIN":"73","FRENEY":"73","FRONTENEX":"73","GRESIN":"73","GRESY SUR ISERE":"73","MERCURY":"73","MEYRIEUX TROUET":"73","MODANE":"73","MONTAGNOLE":"73","MONTAILLEUR":"73","MONTSAPEY":"73","MYANS":"73","NOTRE DAME DU CRUET":"73","NOVALAISE":"73","ONTEX":"73","ORELLE":"73","PALLUD":"73","PUYGROS":"73","QUEIGE":"73","ST ALBAN DES VILLARDS":"73","STE FOY TARENTAISE":"73","ST JEAN DE CHEVELU":"73","STE MARIE DE CUINES":"73","ST MARTIN D ARC":"73","ST MARTIN DE LA PORTE":"73","ST REMY DE MAURIENNE":"73","ST THIBAUD DE COUZ":"73","SEEZ":"73","BEBING":"57","BELLANGE":"57","BEUX":"57","BOUSTROFF":"57","BOUZONVILLE":"57","BREHAIN":"57","BRETTNACH":"57","BROUDERDORFF":"57","BRULANGE":"57","BURTONCOURT":"57","CHANVILLE":"57","CHEMINOT":"57","COIN LES CUVRY":"57","COIN SUR SEILLE":"57","COLLIGNY":"57","CONTHIL":"57","CONTZ LES BAINS":"57","CREHANGE":"57","DANNE ET QUATRE VENTS":"57","DIFFEMBACH LES HELLIMER":"57","DISTROFF":"57","EBERSVILLER":"57","EPPING":"57","ETTING":"57","EVRANGE":"57","FAREBERSVILLER":"57","FEVES":"57","FILSTROFF":"57","FLASTROFF":"57","FOLSCHVILLER":"57","FOULCREY":"57","FRAQUELFING":"57","FREISTROFF":"57","FRIBOURG":"57","GANDRANGE":"57","GARREBOURG":"57","GELUCOURT":"57","GIVRYCOURT":"57","GRENING":"57","GROSBLIEDERSTROFF":"57","HAGEN":"57","HAM SOUS VARSBERG":"57","HANGVILLER":"57","HANVILLER":"57","HARTZVILLER":"57","HAUCONCOURT":"57","HEMING":"57","HERNY":"57","HERTZING":"57","HETTANGE GRANDE":"57","HOLLING":"57","HOMMARTING":"57","HUNTING":"57","ILLANGE":"57","IMLING":"57","JUVILLE":"57","KANFEN":"57","KEMPLICH":"57","KIRSCH LES SIERCK":"57","LANDANGE":"57","LANEUVEVILLE LES LORQUIN":"57","LANGATTE":"57","LENING":"57","LIDREZING":"57","LIXHEIM":"57","LOMMERANGE":"57","LONGEVILLE LES ST AVOLD":"57","LOSTROFF":"57","MAIZEROY":"57","MANDEREN":"57","MANHOUE":"57","MARTHILLE":"57","MAXSTADT":"57","METZERESCHE":"57","MEY":"57","MONCOURT":"57","MONTOIS LA MONTAGNE":"57","NEUFVILLAGE":"57","NIDERHOFF":"57","NILVANGE":"57","NOVEANT SUR MOSELLE":"57","OUDRENNE":"57","PAGNY LES GOIN":"57","PELTRE":"57","PIERREVILLERS":"57","PLAPPEVILLE":"57","POSTROFF":"57","POURNOY LA GRASSE":"57","RACRANGE":"57","RAVILLE":"57","RETTEL":"57","RITZING":"57","ROHRBACH LES BITCHE":"57","ROURE":"06","CHATEAUNEUF GRASSE":"06","BEZAUDUN LES ALPES":"06","BRIANCONNET":"06","BLAUSASC":"06","BOUYON":"06","BIOT":"06","ENTRAUNES":"06","CHATEAUNEUF VILLEVIEILLE":"06","COURSEGOULES":"06","L ESCARENE":"06","GATTIERES":"06","CUEBRIS":"06","THEOULE SUR MER":"06","SERANON":"06","TOURRETTE LEVENS":"06","VILLARS SUR VAR":"06","LA BRIGUE":"06","SAUZE":"06","CREYSSEILLES":"07","FREYSSENET":"07","LE CHEYLARD":"07","DAVEZIEUX":"07","CHAUZON":"07","GLUIRAS":"07","ST SYMPHORIEN SOUS CHOMERAC":"07","ARDEUIL ET MONTFAUXELLES":"08","ALLAND HUY ET SAUSSEUIL":"08","VALLON PONT D ARC":"07","ANNELLES":"08","SOYONS":"07","AUTHE":"08","ST DIDIER SOUS AUBENAS":"07","ST MICHEL DE BOULOGNE":"07","ST ETIENNE DE VALOUX":"07","ST MARCEL D ARDECHE":"07","ST JULIEN LABROUSSE":"07","ST JULIEN DU GUA":"07","ST PERAY":"07","SAGNES ET GOUDOULET":"07","ST CIERGE LA SERRE":"07","ST ANDEOL DE VALS":"07","ST ANDEOL DE BERG":"07","POURCHERES":"07","QUINTENAS":"07","PRIVAS":"07","ROMPON":"07","RUOMS":"07","LE CHATELET SUR RETOURNE":"08","CHESNOIS AUBONCOURT":"08","EUILLY ET LOMBUT":"08","CHEVEUGES":"08","DONCHERY":"08","DAMOUZY":"08","EVIGNY":"08","FAGNON":"08","LA FERTE SUR CHIERS":"08","FRANCHEVAL":"08","FAISSAULT":"08","LA FEREE":"08","FLOING":"08","FLIZE":"08","CHAMPIGNEUL SUR VENCE":"08","BEAUMONT EN ARGONNE":"08","CHATEL CHEHERY":"08","CHAMPIGNEULLE":"08","BOULZICOURT":"08","BREVILLY":"08","BARBAISE":"08","BLAGNY":"08","AUTRY":"08","GRANDPRE":"08","HAM SUR MEUSE":"08","GIVET":"08","ST PAUL EN FORET":"83","SEILLONS SOURCE D ARGENS":"83","LA SEYNE SUR MER":"83","TOURRETTES":"83","TOURVES":"83","RAYOL CANADEL SUR MER":"83","ST MANDRIER SUR MER":"83","ALTHEN DES PALUDS":"84","LA BASTIDONNE":"84","BEAUMONT DE PERTUIS":"84","BLAUVAC":"84","BONNIEUX":"84","BRANTES":"84","CAMARET SUR AIGUES":"84","CASTELLET":"84","FAUCON":"84","LAGARDE D APT":"84","MALEMORT DU COMTAT":"84","METHAMIS":"84","MODENE":"84","LA ROQUE ALRIC":"84","LA ROQUE SUR PERNES":"84","SAIGNON":"84","UCHAUX":"84","VALREAS":"84","VAUGINES":"84","VEDENE":"84","BAZOGES EN PAILLERS":"85","BEAULIEU SOUS LA ROCHE":"85","BESSAY":"85","BOUILLE COURDAULT":"85","LES BROUZILS":"85","CHALLANS":"85","CHASNAIS":"85","CHEFFOIS":"85","CURZON":"85","LE GIROUARD":"85","LA GUYONNIERE":"85","L ILE D ELLE":"85","LAIROUX":"85","LES LUCS SUR BOULOGNE":"85","MENOMBLET":"85","MERVENT":"85","CHERY LES POUILLY":"02","CHARLY SUR MARNE":"02","CHEZY EN ORXOIS":"02","CHERY LES ROZOY":"02","CHATEAU THIERRY":"02","CONTESCOURT":"02","CHARTEVES":"02","CHAOURSE":"02","BEARD GEOVREISSIAT":"01","GRAND CORENT":"01","LESCHEROUX":"01","LOYETTES":"01","LHOPITAL":"01","LOCHIEU":"01","FRANS":"01","GIRON":"01","BAGE LA VILLE":"01","ANGLEFORT":"01","BEREZIAT":"01","AMBRONAY":"01","ARBIGNY":"01","BANEINS":"01","CHAVANNES SUR REYSSOUZE":"01","CHATEAU GAILLARD":"01","LA BOISSE":"01","CERTINES":"01","CIVRIEUX":"01","BEYNOST":"01","BILLIAT":"01","BRIORD":"01","BRENOD":"01","PREVESSIN MOENS":"01","LE PLANTAY":"01","REYRIEUX":"01","RAMASSE":"01","PERREX":"01","BAUDUEN":"83","CHATEAUVERT":"83","COTIGNAC":"83","LA CROIX VALMER":"83","ENTRECASTEAUX":"83","LA FARLEDE":"83","FIGANIERES":"83","FLAYOSC":"83","FORCALQUEIRET":"83","LE LUC":"83","LES MAYONS":"83","MEOUNES LES MONTRIEUX":"83","PLAN D AUPS STE BAUME":"83","POURCIEUX":"83","LE PRADET":"83","LA ROQUE ESCLAPON":"83","STE ANASTASIE SUR ISSOLE":"83","SANARY SUR MER":"83","SOLLIES PONT":"83","TARADEAU":"83","VINS SUR CARAMY":"83","ST ANTONIN DU VAR":"83","APT":"84","AUBIGNAN":"84","BEAUMETTES":"84","BEAUMONT DU VENTOUX":"84","BUISSON":"84","CADENET":"84","CASENEUVE":"84","CAUMONT SUR DURANCE":"84","CHATEAUNEUF DE GADAGNE":"84","CHATEAUNEUF DU PAPE":"84","COURTHEZON":"84","CUCURON":"84","ENTRECHAUX":"84","GRAMBOIS":"84","GRILLON":"84","LAURIS":"84","MALAUCENE":"84","MENERBES":"84","MONTEUX":"84","PUYVERT":"84","RASTEAU":"84","RICHERENCHES":"84","RUSTREL":"84","STE CECILE LES VIGNES":"84","ST HIPPOLYTE LE GRAVEYRON":"84","ST MARCELLIN LES VAISON":"84","ST MARTIN DE LA BRASQUE":"84","ST PIERRE DE VASSOLS":"84","ST TRINIT":"84","SANNES":"84","SAULT":"84","SEGURET":"84","TRAVAILLAN":"84","VACQUEYRAS":"84","VENASQUE":"84","VIOLES":"84","VISAN":"84","AIZENAY":"85","LA BARRE DE MONTS":"85","BEAUFOU":"85","LA BOISSIERE DE MONTAIGU":"85","LA BOISSIERE DES LANDES":"85","LA BRUFFIERE":"85","LA CAILLERE ST HILAIRE":"85","CEZAIS":"85","LA CHAIZE GIRAUD":"85","CHAMPAGNE LES MARAIS":"85","LA CHAPELLE AUX LYS":"85","LA CHAPELLE HERMIER":"85","LA CHAPELLE THEMER":"85","LA CHATAIGNERAIE":"85","CHATEAU GUIBERT":"85","CHAVAGNES EN PAILLERS":"85","LA COPECHAGNIERE":"85","CUGAND":"85","LES EPESSES":"85","FALLERON":"85","FOUSSAIS PAYRE":"85","GIVRAND":"85","GROSBREUIL":"85","GRUES":"85","L ILE D OLONNE":"85","SOLLIERES SARDIERES":"73","TOURS EN SAVOIE":"73","TRAIZE":"73","UGINE":"73","VENTHON":"73","VEREL DE MONTBEL":"73","VILLARGONDRAN":"73","VIONS":"73","ALEX":"74","ARCHAMPS":"74","ARMOY":"74","ARTHAZ PONT NOTRE DAME":"74","AYSE":"74","BALLAISON":"74","LA BALME DE THUY":"74","LA BAUME":"74","BERNEX":"74","BOEGE":"74","BONS EN CHABLAIS":"74","BOUSSY":"74","LA CHAPELLE RAMBAUD":"74","CHATILLON SUR CLUSES":"74","COMBLOUX":"74","CONTAMINE SARZIN":"74","CORDON":"74","CUSY":"74","DINGY EN VUACHE":"74","DRAILLANT":"74","EPAGNY METZ TESSY":"74","FEIGERES":"74","LE GRAND BORNAND":"74","GRUFFY":"74","MACHILLY":"74","MAGLAND":"74","MARIN":"74","MEGEVETTE":"74","MEYTHET":"74","MONT SAXONNEX":"74","MOYE":"74","LA MURAZ":"74","MURES":"74","PERS JUSSY":"74","POISY":"74","REYVROZ":"74","ST CERGUES":"74","ST EUSTACHE":"74","ST JEAN D AULPS":"74","ST JEAN DE THOLOME":"74","ST JULIEN EN GENEVOIS":"74","SCIENTRIER":"74","THOLLON LES MEMISES":"74","USINENS":"74","VACHERESSE":"74","VANZY":"74","VEIGY FONCENEX":"74","VEYRIER DU LAC":"74","VILLE LA GRAND":"74","VILLY LE BOUVERET":"74","PARIS 04":"75","PARIS 10":"75","PARIS 17":"75","AMBRUMESNIL":"76","ANCEAUMEVILLE":"76","AUMALE":"76","AUTRETOT":"76","AUVILLIERS":"76","AUZOUVILLE AUBERBOSC":"76","AUZOUVILLE L ESNEVAL":"76","AVESNES EN VAL":"76","BACQUEVILLE EN CAUX":"76","BAILLY EN RIVIERE":"76","BEAUVAL EN CAUX":"76","BEAUTOT":"76","BERTRIMONT":"76","BIHOREL":"76","BLOSSEVILLE":"76","LE BOCASSE":"76","BOIS L EVEQUE":"76","BOISSAY":"76","BOSC EDELINE":"76","BOUDEVILLE":"76","BREAUTE":"76","ROUHLING":"57","SAILLY ACHATEL":"57","ST EPVRE":"57","ST FRANCOIS LACROIX":"57","ST JEAN DE BASSEL":"57","ST PRIVAT LA MONTAGNE":"57","SCHMITTVILLER":"57","SCHNECKENBUSCH":"57","SEINGBOUSE":"57","SILLEGNY":"57","SILLY EN SAULNOIS":"57","TALANGE":"57","TERVILLE":"57","THONVILLE":"57","TRAGNY":"57","TRITTELING REDLACH":"57","VAHL LES BENESTROFF":"57","VALMUNSTER":"57","VANNECOURT":"57","VIBERSVILLER":"57","VIC SUR SEILLE":"57","VILLERS STONCOURT":"57","VOIMHAUT":"57","WALTEMBOURG":"57","WOIPPY":"57","WUISSE":"57","XOUAXANGE":"57","ZILLING":"57","ZOMMANGE":"57","DIESEN":"57","ARBOURSE":"58","ASNAN":"58","AUNAY EN BAZOIS":"58","AZY LE VIF":"58","BEARD":"58","BILLY CHEVANNES":"58","BLISMES":"58","CHALAUX":"58","CHALLEMENT":"58","CHAMPLEMY":"58","LA CHAPELLE ST ANDRE":"58","CHATEAU CHINON CAMPAGNE":"58","CHATEAUNEUF VAL DE BARGIS":"58","CHAUMARD":"58","CHEVANNES CHANGY":"58","CORANCY":"58","COSSAYE":"58","COULANGES LES NEVERS":"58","COULOUTRE":"58","DECIZE":"58","DRUY PARIGNY":"58","DUN SUR GRANDRY":"58","FACHIN":"58","FLEURY SUR LOIRE":"58","FLEZ CUZY":"58","GIEN SUR CURE":"58","GLUX EN GLENNE":"58","GOULOUX":"58","IMPHY":"58","JAILLY":"58","LIMON":"58","LA MAISON DIEU":"58","MESVES SUR LOIRE":"58","MONTARON":"58","MONTIGNY EN MORVAN":"58","MORACHES":"58","MYENNES":"58","POUQUES LORMES":"58","ST ANDELAIN":"58","ST AUBIN DES CHAUMES":"58","ST MARTIN D HEUILLE":"58","ST MARTIN SUR NOHAIN":"58","CAMPUGNAN":"33","CANTOIS":"33","CAPIAN":"33","CARBON BLANC":"33","LAUNOIS SUR VENCE":"08","JOIGNY SUR MEUSE":"08","MARVAUX VIEUX":"08","JUNIVILLE":"08","MARLEMONT":"08","LA HORGNE":"08","LAMETZ":"08","SAURAT":"09","ST JEAN D AIGUES VIVES":"09","ST QUIRC":"09","ST GIRONS":"09","SABARAT":"09","SAUTEL":"09","SEGURA":"09","USTOU":"09","TAURIGNAN VIEUX":"09","AILLEVILLE":"10","VICDESSOS":"09","VIVIES":"09","UNZENT":"09","VEBRE":"09","BARBUISE":"10","AVON LA PEZE":"10","ASSENCIERES":"10","BAROVILLE":"10","ARRELLES":"10","CHAUFFOUR LES BAILLY":"10","CHASEREY":"10","LE CHENE":"10","CHENNEGY":"10","CORMOST":"10","CLEREY":"10","ETRELLES SUR AUBE":"10","DIERREY ST PIERRE":"10","FAYS LA CHAPELLE":"10","DOMMARTIN LE COQ":"10","COUSSEGREY":"10","DIENVILLE":"10","COURTENOT":"10","CRANCEY":"10","EPAGNE":"10","DOSNON":"10","FERREUX QUINCEY":"10","GYE SUR SEINE":"10","FONTVANNES":"10","GELANNES":"10","FEUGES":"10","GUMERY":"10","ST CLOUD":"92","VANVES":"92","L ILE ST DENIS":"93","LIVRY GARGAN":"93","NOISY LE GRAND":"93","SEVRAN":"93","ALFORTVILLE":"94","JOINVILLE LE PONT":"94","LE KREMLIN BICETRE":"94","LIMEIL BREVANNES":"94","NOGENT SUR MARNE":"94","NOISEAU":"94","ORMESSON SUR MARNE":"94","AINCOURT":"95","AUVERS SUR OISE":"95","BESSANCOURT":"95","BONNEUIL EN FRANCE":"95","BOUFFEMONT":"95","BOUQUEVAL":"95","BRIGNANCOURT":"95","CHATENAY EN FRANCE":"95","DEUIL LA BARRE":"95","ENGHIEN LES BAINS":"95","EPIAIS LES LOUVRES":"95","FREMAINVILLE":"95","GADANCOURT":"95","GARGES LES GONESSE":"95","GRISY LES PLATRES":"95","LE HEAULME":"95","LIVILLIERS":"95","LUZARCHES":"95","PIZAY":"01","MONTMERLE SUR SAONE":"01","NEUVILLE SUR AIN":"01","LE MONTELLIER":"01","MONTHIEUX":"01","MONTRACOL":"01","MONTAGNAT":"01","MARLIEUX":"01","MEZERIAT":"01","MAGNIEU":"01","MARBOZ":"01","CORBONOD":"01","DOMSURE":"01","CROZET":"01","FARGES":"01","CORMOZ":"01","CONAND":"01","VILLARS LES DOMBES":"01","AUTREMENCOURT":"02","AUTREPPES":"02","ST ETIENNE SUR CHALARONNE":"01","ST DIDIER SUR CHALARONNE":"01","ST JEAN SUR REYSSOUZE":"01","ST DIDIER DE FORMANS":"01","ST DENIS LES BOURG":"01","ST CYR SUR MENTHON":"01","RIGNIEUX LE FRANC":"01","ST GERMAIN DE JOUX":"01","ST GENIS POUILLY":"01","ST LAURENT SUR SAONE":"01","ST JEAN SUR VEYLE":"01","SAULT BRENAZ":"01","SEILLONNAZ":"01","ST VULBAS":"01","STE OLIVE":"01","SERMOYER":"01","VIRIEU LE PETIT":"01","VANDEINS":"01","VIRIGNIN":"01","ARTEMPS":"02","AMBLENY":"02","AMBRIEF":"02","ANNOIS":"02","VALLEES EN CHAMPAGNE":"02","BARENTON SUR SERRE":"02","BARISIS AUX BOIS":"02","BELLICOURT":"02","BERTAUCOURT EPOURDON":"02","BERGUES SUR SAMBRE":"02","BERTHENICOURT":"02","BERZY LE SEC":"02","BEZU LE GUERY":"02","BERRY AU BAC":"02","BEUGNEUX":"02","BENAY":"02","BESME":"02","BONY":"02","BRISSAY CHOIGNY":"02","BUIRONFOSSE":"02","BRUNEHAMEL":"02","LE CATELET":"02","CESSIERES":"02","BUSSIARES":"02","CHALANDRY":"02","BRASLES":"02","CERIZY":"02","COUCY LE CHATEAU AUFFRIQUE":"02","COURTEMONT VARENNES":"02","COUCY LA VILLE":"02","COURMELLES":"02","COURBES":"02","CORCY":"02","CROUTTES SUR MARNE":"02","CUIRY LES IVIERS":"02","CRECY AU MONT":"02","CUIRY HOUSSE":"02","COUVRELLES":"02","CREZANCY":"02","CUFFIES":"02","DAMMARD":"02","LA JAUDONNIERE":"85","LA JONCHERE":"85","LANDERONDE":"85","LES LANDES GENUSSON":"85","MARSAIS STE RADEGONDE":"85","MARTINET":"85","BASSEVELLE":"77","BELLOT":"77","BOULANCOURT":"77","BUSSY ST GEORGES":"77","LA CELLE SUR MORIN":"77","CESSOY EN MONTOIS":"77","CHAMPAGNE SUR SEINE":"77","CHANGIS SUR MARNE":"77","LA CHAPELLE LA REINE":"77","LA CHAPELLE ST SULPICE":"77","CHARTRETTES":"77","CHARTRONGES":"77","CHAUMES EN BRIE":"77","CHENOU":"77","CHOISY EN BRIE":"77","CITRY":"77","CLOS FONTAINE":"77","COCHEREL":"77","COLLEGIEN":"77","CONGIS SUR THEROUANNE":"77","COULOMBS EN VALOIS":"77","COURCELLES EN BASSEE":"77","COURTRY":"77","COUTEVROULT":"77","LA CROIX EN BRIE":"77","DAMMARTIN SUR TIGEAUX":"77","DORMELLES":"77","LES ECRENNES":"77","EVRY GREGY SUR YERRE":"77","LA GENEVRAYE":"77","GESVRES LE CHAPITRE":"77","GOUAIX":"77","GOUVERNES":"77","GUIGNES":"77","GURCY LE CHATEL":"77","JAIGNES":"77","LESCHEROLLES":"77","LIVRY SUR SEINE":"77","MAROLLES SUR SEINE":"77","MERY SUR MARNE":"77","MISY SUR YONNE":"77","MONTCOURT FROMONVILLE":"77","MONTIGNY LE GUESDIER":"77","MOUSSEAUX LES BRAY":"77","MOUSSY LE NEUF":"77","NANTEAU SUR LUNAIN":"77","NANTEUIL SUR MARNE":"77","NEUFMOUTIERS EN BRIE":"77","NOYEN SUR SEINE":"77","OISSERY":"77","PECY":"77","LE PLESSIS PLACY":"77","POMMEUSE":"77","RAMPILLON":"77","RUBELLES":"77","RUPEREUX":"77","ST GERMAIN LAXIS":"77","ST GERMAIN SUR ECOLE":"77","ST HILLIERS":"77","ST MARTIN DU BOSCHET":"77","ST MERY":"77","ST THIBAULT DES VIGNES":"77","SAMOIS SUR SEINE":"77","SIVRY COURTRY":"77","SOURDUN":"77","THOMERY":"77","THORIGNY SUR MARNE":"77","URY":"77","VAUCOURTOIS":"77","VERT ST DENIS":"77","VILLEMARECHAL":"77","VILLEMAREUIL":"77","VILLEMER":"77","VILLENEUVE ST DENIS":"77","VILLENOY":"77","VILLEVAUDE":"77","BRETTEVILLE ST LAURENT":"76","CAILLY":"76","CAUDEBEC LES ELBEUF":"76","CONTREMOULINS":"76","COTTEVRARD":"76","CRASVILLE LA MALLET":"76","CRASVILLE LA ROCQUEFORT":"76","CRIQUETOT L ESNEVAL":"76","CRIQUETOT SUR OUVILLE":"76","CUY ST FIACRE":"76","DAMPIERRE ST NICOLAS":"76","DEVILLE LES ROUEN":"76","DOUDEVILLE":"76","ECRAINVILLE":"76","ECTOT LES BAONS":"76","ELBEUF EN BRAY":"76","ETAIMPUIS":"76","FLOCQUES":"76","FOUCARMONT":"76","LA FRENAYE":"76","FRESNAY LE LONG":"76","FRESQUIENNES":"76","LA GAILLARDE":"76","GAILLEFONTAINE":"76","GONNETOT":"76","GRAINVILLE YMAUVILLE":"76","LE HANOUARD":"76","HAUTOT L AUVRAY":"76","HAUTOT SUR SEINE":"76","HENOUVILLE":"76","HERICOURT EN CAUX":"76","HERONCHELLES":"76","HODENG AU BOSC":"76","HOUQUETOT":"76","IMBLEVILLE":"76","ISNEAUVILLE":"76","JUMIEGES":"76","LAMMERVILLE":"76","LANQUETOT":"76","LA LONDE":"76","LONGMESNIL":"76","LOUVETOT":"76","MANNEVILLETTE":"76","MARTIN EGLISE":"76","MENONVAL":"76","MESNIL FOLLEMPRISE":"76","MESNIL PANNEVILLE":"76","MILLEBOSC":"76","MONCHAUX SORENG":"76","MONTREUIL EN CAUX":"76","MORVILLE SUR ANDELLE":"76","NESLE HODENG":"76","NEUFBOSC":"76","NEUFCHATEL EN BRAY":"76","NORVILLE":"76","FRANQUEVILLE ST PIERRE":"76","NOTRE DAME DU BEC":"76","OCTEVILLE SUR MER":"76","OFFRANVILLE":"76","OHERVILLE":"76","OISSEL":"76","OMONVILLE":"76","PALUEL":"76","PARC D ANXTOT":"76","PAVILLY":"76","PIERREVAL":"76","POMMEREUX":"76","PRETOT VICQUEMARE":"76","QUEVILLON":"76","RICARVILLE DU VAL":"76","ROCQUEFORT":"76","SAINNEVILLE":"76","ST AUBIN CELLOVILLE":"76","ST AUBIN SUR SCIE":"76","STE CROIX SUR BUCHY":"76","ST DENIS D ACLON":"76","ST GEORGES SUR FONTAINE":"76","ST GERMAIN D ETABLES":"76","ST GERMAIN SOUS CAILLY":"76","ST JEAN DE LA NEUVILLE":"76","ST LEGER DU BOURG DENIS":"76","ST MARTIN AUX ARBRES":"76","ST MARTIN L HORTIER":"76","ST PAER":"76","CHAMADELLE":"33","CURSAN":"33","DIEULIVOL":"33","LES EGLISOTTES ET CHALAURES":"33","ESCAUDES":"33","FALEYRAS":"33","FARGUES ST HILAIRE":"33","GAILLAN EN MEDOC":"33","GOUALADE":"33","GOURS":"33","HOURTIN":"33","JAU DIGNAC ET LOIRAC":"33","LACANAU":"33","LANDERROUAT":"33","LANDERROUET SUR SEGUR":"33","LANGOIRAN":"33","LERM ET MUSSET":"33","LOUPIAC DE LA REOLE":"33","LUGON ET L ILE DU CARNAY":"33","LUGOS":"33","MADIRAC":"33","MARANSIN":"33","MARIONS":"33","MARTRES":"33","MASSEILLES":"33","MONGAUZY":"33","NAUJAC SUR MER":"33","ORDONNAC":"33","PELLEGRUE":"33","PERISSAC":"33","PEUJARD":"33","LE PORGE":"33","PREIGNAC":"33","PUJOLS SUR CIRON":"33","RAUZAN":"33","LA REOLE":"33","ST ANTOINE SUR L ISLE":"33","ST CAPRAIS DE BLAYE":"33","ST COME":"33","ST EXUPERY":"33","ST FELIX DE FONCAUDE":"33","STE FOY LA LONGUE":"33","ST GENES DE LOMBAUD":"33","ST GENIS DU BOIS":"33","ST GERMAIN DE LA RIVIERE":"33","ST GIRONS D AIGUEVIVES":"33","ST HILAIRE DE LA NOAILLE":"33","ST LOUIS DE MONTFERRAND":"33","ST MAGNE":"33","ST MICHEL DE CASTELNAU":"33","ST VIVIEN DE MONSEGUR":"33","SOUSSAC":"33","TALAIS":"33","UZESTE":"33","VIGNONET":"33","VILLANDRAUT":"33","VILLEGOUGE":"33","VIRSAC":"33","MARCHEPRIME":"33","AGONES":"34","ASSAS":"34","AZILLANET":"34","BALARUC LE VIEUX":"34","BEDARIEUX":"34","BRISSAC":"34","CABREROLLES":"34","CAMBON ET SALVERGUES":"34","CAMPLONG":"34","CARLENCAS ET LEVAS":"34","CASTANET LE HAUT":"34","CASTELNAU LE LEZ":"34","LA CAUNETTE":"34","CAUSSE DE LA SELLE":"34","CAUSSINIOJOULS":"34","MAFFLIERS":"95","MAREIL EN FRANCE":"95","MARGENCY":"95","MENUCOURT":"95","MONTMORENCY":"95","MONTREUIL SUR EPTE":"95","NESLES LA VALLEE":"95","NOISY SUR OISE":"95","OMERVILLE":"95","ST BRICE SOUS FORET":"95","SEUGY":"95","SURVILLIERS":"95","LE THILLAY":"95","VALLANGOUJARD":"95","VILLERS EN ARTHIES":"95","VILLIERS ADAM":"95","VILLIERS LE BEL":"95","ANSE BERTRAND":"971","POINTE A PITRE":"971","POINTE NOIRE":"971","TERRE DE BAS":"971","LES ANSES D ARLET":"972","FORT DE FRANCE":"972","LE LORRAIN":"972","RIVIERE PILOTE":"972","REGINA":"973","GRAND SANTI":"973","KANI KELI":"976","TSINGONI":"976","ALO":"986","MAHINA":"987","PAEA":"987","PAPARA":"987","PUNAAUIA":"987","MESNIL ST LAURENT":"02","MARCHAIS":"02","LUZOIR":"02","MAYOT":"02","MARLE":"02","FERE EN TARDENOIS":"02","ESSIGNY LE GRAND":"02","L EPINE AUX BOIS":"02","EVERGNICOURT":"02","ETREILLERS":"02","DOMMIERS":"02","DAMPLEUX":"02","DROIZY":"02","FRIERES FAILLOUEL":"02","FRESNOY LE GRAND":"02","FROIDESTREES":"02","FOURDRAIN":"02","FONSOMME":"02","FORESTE":"02","HAPPENCOURT":"02","HAUTEVESNES":"02","GUIGNICOURT":"02","GRANDRIEUX":"02","GUYENCOURT":"02","JAULGONNE":"02","HANNAPES":"02","GANDELU":"02","GAUCHY":"02","IRON":"02","MISSY LES PIERREPONT":"02","MONCEAU SUR OISE":"02","MONCEAU LE WAAST":"02","MONS EN LAONNOIS":"02","MOLINCHART":"02","MONNES":"02","MONTESCOURT LIZEROLLES":"02","LA NEUVILLE HOUSSET":"02","MONTIGNY LE FRANC":"02","NEUVILLE ST AMAND":"02","NEUILLY ST FRONT":"02","MOUSSY VERNEUIL":"02","MONTHIERS":"02","NEUVILLE SUR MARGIVAL":"02","PANCY COURTECON":"02","PARCY ET TIGNY":"02","NOUVRON VINGRE":"02","PARGNY FILAIN":"02","PARFONDRU":"02","NOYALES":"02","CHASSENARD":"03","LE BRETHON":"03","BRANSAT":"03","CHAPEAU":"03","LA VILLE AUX BOIS LES PONTAVERT":"02","VERNEUIL SOUS COUCY":"02","VESLES ET CAUMONT":"02","VENDRESSE BEAULNE":"02","VIELS MAISONS":"02","URVILLERS":"02","VAUXBUIN":"02","VERMAND":"02","VASSENS":"02","ST REMY BLANZY":"02","TUGNY ET PONT":"02","THENAILLES":"02","ST BANDRY":"02","TREFCON":"02","SORBAIS":"02","TRAVECY":"02","SELENS":"02","URCEL":"02","LOUROUX BOURBONNAIS":"03","LOUCHY MONTFAND":"03","LA GUILLERMIE":"03","LURCY LEVIS":"03","LETELON":"03","LUSIGNY":"03","LENAX":"03","PLOYART ET VAURSEINE":"02","PUISIEUX ET CLANLIEU":"02","POUILLY SUR SERRE":"02","VILLIERS EN BIERE":"77","VILLIERS ST GEORGES":"77","VILLUIS":"77","VOULTON":"77","VOULX":"77","VULAINES SUR SEINE":"77","YEBLES":"77","ABLIS":"78","AUFFREVILLE BRASSEUIL":"78","LA BOISSIERE ECOLE":"78","LA CELLE LES BORDES":"78","CERNAY LA VILLE":"78","CLAIREFONTAINE EN YVELINES":"78","CRAVENT":"78","CROISSY SUR SEINE":"78","ELANCOURT":"78","EMANCE":"78","LES ESSARTS LE ROI":"78","EVECQUEMONT":"78","FEUCHEROLLES":"78","FLINS SUR SEINE":"78","FONTENAY ST PERE":"78","GAZERAN":"78","GOUSSONVILLE":"78","GROSROUVRE":"78","GUERNES":"78","LA HAUTEVILLE":"78","HOUDAN":"78","JOUY EN JOSAS":"78","LOMMOYE":"78","MAREIL SUR MAULDRE":"78","MAULE":"78","MENERVILLE":"78","MONTESSON":"78","MULCENT":"78","NEZEL":"78","PONTHEVRARD":"78","PORCHEVILLE":"78","LE PORT MARLY":"78","LA QUEUE LES YVELINES":"78","ROLLEBOISE":"78","ROSNY SUR SEINE":"78","ST CYR L ECOLE":"78","ST GERMAIN DE LA GRANGE":"78","ST NOM LA BRETECHE":"78","ST REMY L HONORE":"78","AIFFRES":"79","ARGENTON L EGLISE":"79","LE BEUGNON":"79","BOUILLE LORETZ":"79","BRION PRES THOUET":"79","CAUNAY":"79","CHAIL":"79","LA COUARDE":"79","COULON":"79","ECHIRE":"79","EPANNES":"79","EXIREUIL":"79","FAYE SUR ARDIN":"79","FONTENILLE ST MARTIN D ENTRAIGUES":"79","GLENAY":"79","LES GROSEILLERS":"79","LEZAY":"79","LOUBILLE":"79","LOUZY":"79","LUZAY":"79","MAUZE SUR LE MIGNON":"79","MAZIERES SUR BERONNE":"79","NEUVY BOUIN":"79","PLIBOUX":"79","PUGNY":"79","ST RIQUIER EN RIVIERE":"76","ST VAAST DIEPPEDALLE":"76","SIERVILLE":"76","SOMMERY":"76","SOMMESNIL":"76","SOTTEVILLE SOUS LE VAL":"76","SOTTEVILLE SUR MER":"76","THIETREVILLE":"76","THIL MANNEVILLE":"76","TOUFFREVILLE LA CORBELINE":"76","TOURVILLE SUR ARQUES":"76","LES TROIS PIERRES":"76","TROUVILLE":"76","TURRETOT":"76","VERGETOT":"76","YMARE":"76","ACHERES LA FORET":"77","BEAUCHERY ST MARTIN":"77","BERNAY VILBERT":"77","BETON BAZOCHES":"77","BOULEURS":"77","LA BROSSE MONTCEAUX":"77","BROU SUR CHANTEREINE":"77","BURCY":"77","CHAILLY EN BIERE":"77","CHAILLY EN BRIE":"77","CHALAUTRE LA GRANDE":"77","CHALIFERT":"77","CHALMAISON":"77","CHANTELOUP EN BRIE":"77","LES CHAPELLES BOURBON":"77","CHAUFFRY":"77","CHEVRU":"77","CLAYE SOUILLY":"77","COUBERT":"77","COUILLY PONT AUX DAMES":"77","COULOMMES":"77","COURCHAMP":"77","COURPALAY":"77","CROUY SUR OURCQ":"77","CUCHARMOY":"77","DAMMARIE LES LYS":"77","DIANT":"77","DOUY LA RAMEE":"77","ECHOUBOULAINS":"77","EGLIGNY":"77","EMERAINVILLE":"77","ESMANS":"77","FERICY":"77","FEROLLES ATTILLY":"77","FERRIERES EN BRIE":"77","LA FERTE GAUCHER":"77","LA FERTE SOUS JOUARRE":"77","FONTAINE FOURCHES":"77","FONTAINE LE PORT":"77","FONTAINS":"77","FRETOY":"77","GARENTREVILLE":"77","GERMIGNY L EVEQUE":"77","GERMIGNY SOUS COULOMBS":"77","GIRONVILLE":"77","LA GRANDE PAROISSE":"77","GRISY SUR SEINE":"77","HERME":"77","JABLINES":"77","JUTIGNY":"77","LIMOGES FOURCHES":"77","LISSY":"77","LES MARETS":"77","MAREUIL LES MEAUX":"77","MARLES EN BRIE":"77","MELUN":"77","MELZ SUR SEINE":"77","MONTENILS":"77","MONTIGNY SUR LOING":"77","MONTOLIVET":"77","MONTRY":"77","NANDY":"77","LE CAYLAR":"34","CAZEVIEILLE":"34","CAZOULS D HERAULT":"34","CERS":"34","CEYRAS":"34","CLERMONT L HERAULT":"34","COURNONSEC":"34","FELINES MINERVOIS":"34","FOZIERES":"34","GABIAN":"34","GRAISSESSAC":"34","LATTES":"34","LAUROUX":"34","LIGNAN SUR ORB":"34","LA LIVINIERE":"34","MARAUSSAN":"34","MAUGUIO":"34","MONTADY":"34","NEFFIES":"34","NEZIGNAN L EVEQUE":"34","NISSAN LEZ ENSERUNE":"34","OCTON":"34","PAULHAN":"34","PEZENAS":"34","POPIAN":"34","POUJOLS":"34","POUZOLLES":"34","RIOLS":"34","ROQUEREDONDE":"34","ROQUESSELS":"34","ST ANDRE DE BUEGES":"34","ST ANDRE DE SANGONIS":"34","ST BAUZILLE DE LA SYLVE":"34","ST CHINIAN":"34","ST ETIENNE ESTRECHOUX":"34","ST GENIES DE FONTEDIT":"34","ST NAZAIRE DE PEZAN":"34","ST PAUL ET VALMALLE":"34","ST PIERRE DE LA FAGE":"34","ST SERIES":"34","SATURARGUES":"34","SORBS":"34","TOURBES":"34","LE TRIADOU":"34","USCLAS D HERAULT":"34","VACQUIERES":"34","VALERGUES":"34","VENDRES":"34","VERARGUES":"34","VERRERIES DE MOUSSANS":"34","VIC LA GARDIOLE":"34","VILLEMAGNE L ARGENTIERE":"34","VILLENEUVE LES MAGUELONE":"34","VIOLS EN LAVAL":"34","LA GRANDE MOTTE":"34","BAGUER PICAN":"35","BOISTRUDAN":"35","BOURG DES COMPTES":"35","LES BRULAIS":"35","LA CHAPELLE AUX FILTZMEENS":"35","LA CHAPELLE CHAUSSEE":"35","CLAYES":"35","CORPS NUDS":"35","DOL DE BRETAGNE":"35","DOMPIERRE DU CHEMIN":"35","EANCE":"35","ERCE PRES LIFFRE":"35","ETRELLES":"35","FOUGERES":"35","GEVEZE":"35","GUIGNEN":"35","GUIPEL":"35","IFFENDIC":"35","LAIGNELET":"35","LANDUJAN":"35","MARTIGNE FERCHAUD":"35","MAURE DE BRETAGNE":"35","MEDREAC":"35","MERNEL":"35","MONTAUTOUR":"35","MONTHAULT":"35","MORDELLES":"35","PAIMPONT":"35","TAIARAPU OUEST":"987","TATAKOTO":"987","TUMARAA":"987","BELEP":"988","BOURAIL":"988","FARINO":"988","HIENGHENE":"988","L ILE DES PINS":"988","KONE":"988","THIO":"988","ILE DE CLIPPERTON":"989","ALBITRECCIA":"2A","AMBIEGNA":"2A","BALOGNA":"2A","BILIA":"2A","CAMPO":"2A","CARGESE":"2A","CARGIACA":"2A","CASALABRIVA":"2A","CORRANO":"2A","COTI CHIAVARI":"2A","CRISTINACCE":"2A","GIUNCHETO":"2A","GROSSA":"2A","GROSSETO PRUGNA":"2A","LECCI":"2A","LOPIGNA":"2A","LORETO DI TALLANO":"2A","OCANA":"2A","OLIVESE":"2A","OLMETO":"2A","PIETROSELLA":"2A","QUENZA":"2A","REZZA":"2A","SARROLA CARCOPINO":"2A","SERRIERA":"2A","SOLLACARO":"2A","SORBOLLANO":"2A","SAN GAVINO DI CARBINI":"2A","SANTA MARIA SICHE":"2A","TAVERA":"2A","URBALACONE":"2A","ALANDO":"2B","ALERIA":"2B","AREGNO":"2B","BARBAGGIO":"2B","BELGODERE":"2B","CANARI":"2B","CARPINETO":"2B","CARTICASI":"2B","CASTELLO DI ROSTINO":"2B","CORBARA":"2B","CROCICCHIA":"2B","FELCE":"2B","FICAJA":"2B","GAVIGNANO":"2B","LENTO":"2B","LUCCIANA":"2B","MONCALE":"2B","MOROSAGLIA":"2B","MORSIGLIA":"2B","MURACCIOLE":"2B","MURO":"2B","NOVALE":"2B","OCCHIATANA":"2B","OGLIASTRO":"2B","OLMETA DI CAPOCORSO":"2B","OLMO":"2B","PARATA":"2B","PENTA ACQUATELLA":"2B","PENTA DI CASINCA":"2B","PIOBETTA":"2B","POGGIO DI VENACO":"2B","PORRI":"2B","RAPAGGIO":"2B","RIVENTOSA":"2B","SALICETO":"2B","SOLARO":"2B","SORIO":"2B","SANTO PIETRO DI TENDA":"2B","TALASANI":"2B","PARGNY LES BOIS":"02","PONT ST MARD":"02","PARPEVILLE":"02","PREMONT":"02","QUIERZY":"02","PONTRU":"02","REUILLY SAUVIGNY":"02","RESSONS LE LONG":"02","RAILLIMONT":"02","RAMICOURT":"02","RIBEMONT":"02","ROGNY":"02","ROUCY":"02","CREUZIER LE NEUF":"03","CRESSANGES":"03","FRANCHESSE":"03","FOURILLES":"03","CHEZELLE":"03","COUZON":"03","CUSSET":"03","CHEZY":"03","BARCELONNETTE":"04","LA BRILLANNE":"04","BELLAFFAIRE":"04","BRAS D ASSE":"04","LE VILHAIN":"03","ANNOT":"04","MONTEIGNET SUR L ANDELOT":"03","PARAY SOUS BRIAILLES":"03","MONTAIGUET EN FOREZ":"03","MONTBEUGNY":"03","MALICORNE":"03","MONTOLDRE":"03","MONTLUCON":"03","MONTVICQ":"03","MESPLES":"03","NADES":"03","ST BONNET DE ROCHEFORT":"03","ST BONNET TRONCAIS":"03","ST ETIENNE DE VICQ":"03","ST DIDIER LA FORET":"03","QUINSSAINES":"03","PREMILHAT":"03","RONGERES":"03","ST MARCEL EN MURAT":"03","TEILLET ARGENTY":"03","ST PLAISIR":"03","ST SAUVIER":"03","SAUVAGNY":"03","SAULCET":"03","ST VOIR":"03","SORBIER":"03","TERJAT":"03","VERNEUIL EN BOURBONNAIS":"03","THENEUILLE":"03","TREZELLES":"03","VERNUSSE":"03","TREIGNAT":"03","VERNEIX":"03","TREVOL":"03","VEAUCE":"03","VICHY":"03","CURBANS":"04","CRUIS":"04","DRAIX":"04","L HOSPITALET":"04","FONTIENNE":"04","LARDIERS":"04","L ARGENTIERE LA BESSEE":"05","LA BATIE MONTSALEON":"05","FURMEYER":"05","BRIANCON":"05","BARATIER":"05","JARJAYES":"05","EOURRES":"05","ABRIES":"05","LAYE":"05","MALLEFOUGASSE AUGES":"04","MOUSTIERS STE MARIE":"04","PIERREVERT":"04","LA ROCHENARD":"79","ST AUBIN LE CLOUD":"79","STE EANNE":"79","ST GENARD":"79","ST JACQUES DE THOUARS":"79","ST LAURS":"79","ST LEGER DE LA MARTINIERE":"79","ST PAUL EN GATINE":"79","ST VARENT":"79","ST VINCENT LA CHATRE":"79","SAUZE VAUSSAIS":"79","SECONDIGNE SUR BELLE":"79","VASLES":"79","VAUSSEROUX":"79","ACHEUX EN AMIENOIS":"80","AGENVILLERS":"80","AILLY LE HAUT CLOCHER":"80","AILLY SUR NOYE":"80","AILLY SUR SOMME":"80","ALLERY":"80","ARGOEUVES":"80","ARVILLERS":"80","ASSEVILLERS":"80","AUBVILLERS":"80","BALATRE":"80","BAVELINCOURT":"80","BEAUFORT EN SANTERRE":"80","BEAUVAL":"80","BELLEUSE":"80","BELLOY ST LEONARD":"80","BERNAVILLE":"80","BETTENCOURT RIVIERE":"80","BEUVRAIGNES":"80","BLANGY TRONVILLE":"80","LE BOISLE":"80","BOVES":"80","BROCOURT":"80","BUSSUS BUSSUEL":"80","CAHON":"80","CANDAS":"80","CAYEUX EN SANTERRE":"80","CHUIGNOLLES":"80","CITERNE":"80","COISY":"80","CONTAY":"80","CORBIE":"80","DEMUIN":"80","DOINGT":"80","DOUILLY":"80","DREUIL LES AMIENS":"80","DRIENCOURT":"80","L ECHELLE ST AURIN":"80","ECLUSIER VAUX":"80","EMBREVILLE":"80","EPEHY":"80","EPENANCOURT":"80","EQUANCOURT":"80","ERGNIES":"80","ESTREES DENIECOURT":"80","ETALON":"80","ETELFAY":"80","FALVY":"80","FLUY":"80","FONTAINE LE SEC":"80","FOURCIGNY":"80","FRANSURES":"80","FRESNES MAZANCOURT":"80","FRESNEVILLE":"80","FRESNOY ANDAINVILLE":"80","FRESSENNEVILLE":"80","FRETTECUISSE":"80","FRICAMPS":"80","FRISE":"80","FRIVILLE ESCARBOTIN":"80","ST GOIN":"64","SAUVAGNON":"64","SAUVETERRE DE BEARN":"64","SEMEACQ BLACHON":"64","SERRES MORLAAS":"64","SUHESCUN":"64","TARDETS SORHOLUS":"64","TARSACQ":"64","URCUIT":"64","NEMOURS":"77","ORLY SUR MORIN":"77","ORMESSON":"77","PALEY":"77","POINCY":"77","PRESLES EN BRIE":"77","REAU":"77","ROISSY EN BRIE":"77","ST GERMAIN SUR MORIN":"77","ST JEAN LES DEUX JUMEAUX":"77","ST JUST EN BRIE":"77","ST MESMES":"77","ST OUEN SUR MORIN":"77","ST PATHUS":"77","SAINTS":"77","ST SAUVEUR LES BRAY":"77","ST SOUPPLETS":"77","SAVIGNY LE TEMPLE":"77","SAVINS":"77","SEPT SORTS":"77","SERRIS":"77","THENISY":"77","TIGEAUX":"77","LA TRETOIRE":"77","TREUZY LEVELAY":"77","TRILPORT":"77","TROCY EN MULTIEN":"77","VARENNES SUR SEINE":"77","VARREDDES":"77","VAUDOY EN BRIE":"77","VENDREST":"77","VERDELOT":"77","VILLECERF":"77","VILLENAUXE LA PETITE":"77","VILLENEUVE SUR BELLOT":"77","VILLIERS SOUS GREZ":"77","VIMPELLES":"77","VOINSLES":"77","BAZEMONT":"78","BENNECOURT":"78","BOINVILLIERS":"78","BOISSY MAUVOISIN":"78","BONNIERES SUR SEINE":"78","BOUGIVAL":"78","BREVAL":"78","CHAMBOURCY":"78","CHEVREUSE":"78","CIVRY LA FORET":"78","COURGENT":"78","ECQUEVILLY":"78","GAILLON SUR MONTCIENT":"78","GAMBAISEUIL":"78","GUITRANCOURT":"78","ISSOU":"78","JAMBVILLE":"78","JEUFOSSE":"78","JOUARS PONTCHARTRAIN":"78","JUMEAUVILLE":"78","LEVIS ST NOM":"78","LIMAY":"78","LES LOGES EN JOSAS":"78","MAGNANVILLE":"78","MAISONS LAFFITTE":"78","MANTES LA JOLIE":"78","MAULETTE":"78","MAURECOURT":"78","LE MESNIL ST DENIS":"78","MEZY SUR SEINE":"78","MOISSON":"78","NOISY LE ROI":"78","OINVILLE SUR MONTCIENT":"78","ORSONVILLE":"78","ORVILLIERS":"78","LE PECQ":"78","PRUNAY LE TEMPLE":"78","ROCHEFORT EN YVELINES":"78","ST ARNOULT EN YVELINES":"78","ST HILARION":"78","SONCHAMP":"78","LE TERTRE ST DENIS":"78","THIVERVAL GRIGNON":"78","LE VESINET":"78","VIEILLE EGLISE EN YVELINES":"78","PLELAN LE GRAND":"35","PLEUGUENEUC":"35","PLEURTUIT":"35","REDON":"35","RIMOU":"35","ROZ SUR COUESNON":"35","ST AUBIN D AUBIGNE":"35","ST BENOIT DES ONDES":"35","ST GERMAIN EN COGLES":"35","ST HILAIRE DES LANDES":"35","ST JACQUES DE LA LANDE":"35","ST MALON SUR MEL":"35","ST MAUGAN":"35","ST MEDARD SUR ILLE":"35","ST M HERVE":"35","ST M HERVON":"35","ST PIERRE DE PLESGUEN":"35","ST SEGLIN":"35","ST THUAL":"35","ST UNIAC":"35","TORCE":"35","TREMEHEUC":"35","TRESBOEUF":"35","TRIMER":"35","VISSEICHE":"35","LE VIVIER SUR MER":"35","AIZE":"36","LE BLANC":"36","BOUGES LE CHATEAU":"36","CEAULMONT":"36","CHATILLON SUR INDRE":"36","CLUIS":"36","CONDE":"36","DUNET":"36","DUN LE POELIER":"36","FEUSINES":"36","FLERE LA RIVIERE":"36","LINGE":"36","LUCAY LE LIBRE":"36","LUREUIL":"36","LUZERET":"36","LE MAGNY":"36","MENETOU SUR NAHON":"36","LE MENOUX":"36","MEZIERES EN BRENNE":"36","MONTGIVRAY":"36","MOSNAY":"36","NEUVY ST SEPULCHRE":"36","NIHERNE":"36","LE PECHEREAU":"36","PELLEVOISIN":"36","LA PEROUILLE":"36","POULAINES":"36","POULIGNY NOTRE DAME":"36","PRUNIERS":"36","SACIERGES ST MARTIN":"36","ST CIVRAN":"36","ST MICHEL EN BRENNE":"36","ST PIERRE DE JARDS":"36","ST PLANTAIRE":"36","VERNEUIL SUR IGNERAIE":"36","VILLEDIEU SUR INDRE":"36","VILLEGONGIS":"36","VILLENTROIS":"36","ASSAY":"37","AZAY SUR CHER":"37","BALLAN MIRE":"37","BRAYE SUR MAULNE":"37","BRIDORE":"37","BRIZAY":"37","LA CELLE ST AVANT":"37","LA CHAPELLE AUX NAUX":"37","CHEMILLE SUR DEME":"37","CHINON":"37","CINAIS":"37","CIRAN":"37","CIVRAY DE TOURAINE":"37","CORMERY":"37","COURCAY":"37","EPEIGNE LES BOIS":"37","FRANCUEIL":"37","L ILE BOUCHARD":"37","VALLE D ALESANI":"2B","VALLE DI ROSTINO":"2B","VENZOLASCA":"2B","VESCOVATO":"2B","VIGNALE":"2B","CHAUSSOY EPAGNY":"80","CHUIGNES":"80","CIZANCOURT":"80","CLERY SUR SOMME":"80","COCQUEREL":"80","CONDE FOLIE":"80","CONTALMAISON":"80","COURCELLES SOUS MOYENCOURT":"80","COURCELLES SOUS THOIX":"80","CURLU":"80","DAOURS":"80","DARGNIES":"80","DAVENESCOURT":"80","DOMART EN PONTHIEU":"80","DOMPIERRE SUR AUTHIE":"80","DOMVAST":"80","DOUDELAINVILLE":"80","EPAUMESNIL":"80","EPECAMPS":"80","ERCOURT":"80","ERONDELLE":"80","ETINEHEM":"80","ETREJUST":"80","FESCAMPS":"80","FIENVILLERS":"80","FLIXECOURT":"80","FOUENCAMPS":"80","FRANSU":"80","FRESNES TILLOLOY":"80","FRESNOY AU VAL":"80","FRESNOY EN CHAUSSEE":"80","FRETTEMEULE":"80","GAUVILLE":"80","GENTELLES":"80","GEZAINCOURT":"80","HALLU":"80","HANCOURT":"80","HERVILLY":"80","HESBECOURT":"80","HEUCOURT CROQUOISON":"80","INVAL BOIRON":"80","LABOISSIERE EN SANTERRE":"80","LANCHERES":"80","LANGUEVOISIN QUIQUERY":"80","LAUCOURT":"80","LAWARDE MAUGER L HORTOY":"80","LICOURT":"80","LOEUILLY":"80","LOUVENCOURT":"80","MAILLY RAINEVAL":"80","MARCHELEPOT":"80","MARICOURT":"80","MARTAINNEVILLE":"80","MENESLIES":"80","MERELESSART":"80","MERICOURT SUR SOMME":"80","MESNIL BRUNTEL":"80","MESNIL ST GEORGES":"80","MONS BOUBERT":"80","MONTAUBAN DE PICARDIE":"80","MONTIGNY SUR L HALLUE":"80","MORISEL":"80","NESLE L HOPITAL":"80","NEUVILLE COPPEGUEULE":"80","NEUVILLE LES LOEUILLY":"80","LA NEUVILLE SIRE BERNARD":"80","NIBAS":"80","NURLU":"80","OISSY":"80","ORESMAUX":"80","PIERREGOT":"80","MONTJUSTIN":"04","MONTFURON":"04","MORIEZ":"04","MISON":"04","MOLINES EN QUEYRAS":"05","MONETIER ALLEMONT":"05","MEREUIL":"05","LE POET":"05","RABOU":"05","LAZER":"05","ST JULIEN DU VERDON":"04","ST MARTIN LES EAUX":"04","VALAVOIRE":"04","STE TULLE":"04","SALIGNAC":"04","VILLEMUS":"04","ST CLEMENT SUR DURANCE":"05","REALLON":"05","BEAULIEU SUR MER":"06","VILLAR LOUBIERE":"05","ST PIERRE AVEZ":"05","SIGOTTIER":"05","BELVEDERE":"06","LA SAULCE":"05","LE SAIX":"05","TALLARD":"05","LA BOLLENE VESUBIE":"06","BERRE LES ALPES":"06","CASTAGNIERS":"06","CASTELLAR":"06","CIPIERES":"06","COARAZE":"06","CARROS":"06","LA ROQUETTE SUR SIAGNE":"06","REVEST LES ROCHES":"06","MOUANS SARTOUX":"06","LES FERRES":"06","FALICON":"06","ROUBION":"06","GORBIO":"06","EZE":"06","GILHAC ET BRUZAC":"07","MARCOLS LES EAUX":"07","LACHAMP RAPHAEL":"07","EMPURANY":"07","LAVEYRUNE":"07","ST APOLLINAIRE DE RIAS":"07","ST BARTHELEMY LE MEIL":"07","ROCHEPAULE":"07","ROCHEMAURE":"07","ROIFFIEUX":"07","ST BASILE":"07","ROCHER":"07","ST FORTUNAT SUR EYRIEUX":"07","ST JEAN LE CENTENIER":"07","ST GEORGES LES BAINS":"07","ST ETIENNE DE SERRE":"07","ST JULIEN DU SERRE":"07","ST JEAN DE MUZOLS":"07","ST JEURE D AY":"07","ST FELICIEN":"07","ST SYMPHORIEN DE MAHUN":"07","ST MAURICE D ARDECHE":"07","ST PIERRE SUR DOUX":"07","ST LAURENT DU PAPE":"07","ST JULIEN LE ROUX":"07","ST JULIEN VOCANCE":"07","ST LAGER BRESSAC":"07","USCLADES ET RIEUTORD":"07","VERNOUX EN VIVARAIS":"07","ST THOME":"07","UROST":"64","UZEIN":"64","VIELLENAVE D ARTHEZ":"64","AGOS VIDALOS":"65","ARBEOST":"65","ARDENGOST":"65","ARIES ESPENAN":"65","ARRAS EN LAVEDAN":"65","ARRODETS EZ ANGLES":"65","ARTIGUEMY":"65","ASPIN EN LAVEDAN":"65","AUCUN":"65","AURIEBAT":"65","AYZAC OST":"65","AZEREIX":"65","AZET":"65","BATSERE":"65","BAZILLAC":"65","BEAUDEAN":"65","BETBEZE":"65","BEYREDE JUMET":"65","BOO SILHEN":"65","BOULIN":"65","BUN":"65","BUZON":"65","CAHARET":"65","CALAVANTE":"65","CAMPUZAN":"65","CAPVERN":"65","CASTERA LANUSSE":"65","CAUSSADE RIVIERE":"65","CAZARILH":"65","CHELLE DEBAT":"65","CHELLE SPOU":"65","CHEZE":"65","COUSSAN":"65","ESBAREICH":"65","ESTENSAN":"65","FERRERE":"65","GAILLAGOS":"65","GAUSSAN":"65","GAZOST":"65","GENEREST":"65","GERM":"65","GUIZERIX":"65","HITTE":"65","HORGUES":"65","JACQUE":"65","LABATUT RIVIERE":"65","LAPEYRE":"65","LASSALES":"65","LEZIGNAN":"65","LIAC":"65","LIZOS":"65","LOUEY":"65","LUBY BETMONT":"65","LUQUET":"65","LUTILHOUS":"65","MAZOUAU":"65","MERILHEU":"65","OLEAC DESSUS":"65","OMEX":"65","ORLEIX":"65","OSSUN":"65","OURDE":"65","PAREAC":"65","PINTAC":"65","POUY":"65","POUZAC":"65","SABARROS":"65","ST LANNE":"65","SARROUILLES":"65","SENTOUS":"65","SERE RUSTAING":"65","SINZOS":"65","SOULOM":"65","THUY":"65","TOSTAT":"65","TREBONS":"65","TROUBAT":"65","VILLEPREUX":"78","VILLIERS LE MAHIEU":"78","ASNIERES EN POITOU":"79","ASSAIS LES JUMEAUX":"79","AZAY SUR THOUET":"79","LA BATAILLE":"79","BESSINES":"79","CHIZE":"79","FORS":"79","LES FOSSES":"79","LA FOYE MONJAULT":"79","FRANCOIS":"79","LHOUMOIS":"79","LORIGNE":"79","LOUBIGNE":"79","LOUIN":"79","LUCHE THOUARSAIS":"79","LUSSERAY":"79","MAIRE LEVESCAULT":"79","MONTROUGE":"92","PUTEAUX":"92","RUEIL MALMAISON":"92","AUBERVILLIERS":"93","LE BOURGET":"93","COUBRON":"93","LES LILAS":"93","NOISY LE SEC":"93","TREMBLAY EN FRANCE":"93","BOISSY ST LEGER":"94","CACHAN":"94","CHAMPIGNY SUR MARNE":"94","CHARENTON LE PONT":"94","L HAY LES ROSES":"94","MANDRES LES ROSES":"94","LA QUEUE EN BRIE":"94","ST MAUR DES FOSSES":"94","AVERNES":"95","LE BELLAY EN VEXIN":"95","BELLOY EN FRANCE":"95","BERNES SUR OISE":"95","BETHEMONT LA FORET":"95","BOISSY L AILLERIE":"95","BRAY ET LU":"95","EAUBONNE":"95","EPIAIS RHUS":"95","EPINAY CHAMPLATREUX":"95","GONESSE":"95","GOUZANGREZ":"95","HARAVILLIERS":"95","LABBEVILLE":"95","MAGNY EN VEXIN":"95","MERIEL":"95","MOISSELLES":"95","MONTGEROULT":"95","MOURS":"95","NERVILLE LA FORET":"95","NEUILLY EN VEXIN":"95","PARMAIN":"95","LE PLESSIS BOUCHARD":"95","SOISY SOUS MONTMORENCY":"95","US":"95","VALMONDOIS":"95","VIENNE EN ARTHIES":"95","VILLAINES SOUS BOIS":"95","BASSE TERRE":"971","GOURBEYRE":"971","LA DESIRADE":"971","LE GOSIER":"971","GOYAVE":"971","INGRANDES DE TOURAINE":"37","LANGEAIS":"37","LIGNIERES DE TOURAINE":"37","LIGUEIL":"37","MARCILLY SUR MAULNE":"37","LA MEMBROLLE SUR CHOISILLE":"37","NEUILLE PONT PIERRE":"37","NEUVY LE ROI":"37","NOIZAY":"37","NOUATRE":"37","NOUZILLY":"37","PANZOULT":"37","PAULMY":"37","PERNAY":"37","POCE SUR CISSE":"37","PONT DE RUAN":"37","PORTS":"37","REIGNAC SUR INDRE":"37","LA RICHE":"37","ROCHECORBON":"37","ST AVERTIN":"37","ST BENOIT LA FORET":"37","BEAULIEU SUR LAYON":"49","BEGROLLES EN MAUGES":"49","BRAIN SUR ALLONNES":"49","BREIL":"49","BROSSAY":"49","CHACE":"49","LA CHAPELLE HULLIN":"49","LA CHAPELLE ST LAUD":"49","LA CHAPELLE SUR OUDON":"49","CHARCE ST ELLIER SUR AUBANCE":"49","CHATELAIS":"49","CHAVAIGNES":"49","CHAZE SUR ARGOS":"49","CIZAY LA MADELEINE":"49","CONCOURSON SUR LAYON":"49","CONTIGNE":"49","LA CORNUAILLE":"49","CORON":"49","ECUILLE":"49","ETRICHE":"49","FENEU":"49","FREIGNE":"49","INGRANDES LE FRESNE SUR LOIRE":"49","LA JAILLE YVON":"49","LA LANDE CHASLES":"49","LOUVAINES":"49","MARCE":"49","MARIGNE":"49","MARTIGNE BRIAND":"49","LE MAY SUR EVRE":"49","MAZIERES EN MAUGES":"49","MONTGUILLON":"49","MONTREUIL SUR LOIR":"49","MORANNES SUR SARTHE":"49","ROCHEFORT SUR LOIRE":"49","STE GEMMES D ANDIGNE":"49","ST GEORGES SUR LAYON":"49","ST MARTIN DE LA PLACE":"49","ST MELAINE SUR AUBANCE":"49","ST PHILBERT DU PEUPLE":"49","ST SAUVEUR DE FLEE":"49","SAVENNIERES":"49","PONTHOILE":"80","POULAINVILLE":"80","POZIERES":"80","PROYART":"80","LE QUESNE":"80","QUEVAUVILLERS":"80","RAMBURELLES":"80","REMAUGIES":"80","REMIENCOURT":"80","SAIGNEVILLE":"80","SAILLY LAURETTE":"80","SAILLY SAILLISEL":"80","ST CHRIST BRIOST":"80","ST FUSCIEN":"80","ST RIQUIER":"80","SAVEUSE":"80","TINCOURT BOUCLY":"80","TOEUFLES":"80","VAUX SUR SOMME":"80","VERPILLIERES":"80","VERS SUR SELLES":"80","VRON":"80","WIENCOURT L EQUIPEE":"80","AGUTS":"81","ALBINE":"81","AMBIALET":"81","APPELLE":"81","ARFONS":"81","ARIFAT":"81","BANNIERES":"81","BARRE":"81","LE BEZ":"81","BLAN":"81","BOUT DU PONT DE LARN":"81","BUSQUE":"81","CAHUZAC SUR VERE":"81","CARLUS":"81","CAUCALIERES":"81","CESTAYROLS":"81","CORDES SUR CIEL":"81","DAMIATTE":"81","DONNAZAC":"81","DOURGNE":"81","LE DOURN":"81","FREJEVILLE":"81","LE GARRIC":"81","LACAPELLE SEGALAR":"81","LACROISILLE":"81","LAMONTELARIE":"81","LIVERS CAZELLES":"81","LOUBERS":"81","MEZENS":"81","MILHARS":"81","MIRANDOL BOURGNOUNAC":"81","MONTGEY":"81","MONTVALEN":"81","MOULARES":"81","MOUZIEYS PANENS":"81","MURAT SUR VEBRE":"81","PADIES":"81","PENNE":"81","PUYCELSI":"81","PUYLAURENS":"81","RONEL":"81","ROQUEVIDAL":"81","ROUAIROUX":"81","ST CIRGUE":"81","ST JEAN DE RIVES":"81","ST LIEUX LES LAVAUR":"81","ST MARCEL CAMPES":"81","ST MICHEL LABADIE":"81","ST SULPICE LA POINTE":"81","TECOU":"81","TEYSSODE":"81","VALDERIES":"81","VALDURENQUE":"81","VEILHES":"81","VIANE":"81","VIELMUR SUR AGOUT":"81","VITERBE":"81","VALGORGE":"07","SAMPZON":"07","THUEYTS":"07","LE TEIL":"07","SAVAS":"07","UCEL":"07","AUTRECOURT ET POURRON":"08","AUTRUCHE":"08","BAALONS":"08","BALHAM":"08","VALBONNE":"06","SPERACEDES":"06","VALDEBLORE":"06","ST AUBAN":"06","SIGALE":"06","SAORGE":"06","ALBON D ARDECHE":"07","BEAUVENE":"07","AUBIGNAS":"07","BERZEME":"07","BARNAS":"07","BESSAS":"07","ASTET":"07","BOGY":"07","MONTPEZAT SOUS BAUZON":"07","MAZAN L ABBAYE":"07","LE PLAGNAL":"07","PEREYRES":"07","LE POUZIN":"07","MEYSSE":"07","PLATS":"07","AUBIGNY LES POTHEES":"08","ACY ROMANCE":"08","ALINCOURT":"08","ANGECOURT":"08","VINEZAC":"07","ARREUX":"08","BLANZY LA SALONNAISE":"08","BOGNY SUR MEUSE":"08","BERGNICOURT":"08","BERTONCOURT":"08","BOUTANCOURT":"08","BIERMES":"08","BOURCQ":"08","CORNY MACHEROMENIL":"08","LES DEUX VILLES":"08","DRICOURT":"08","CORNAY":"08","DAIGNY":"08","ELAN":"08","MALAKOFF":"92","MARNES LA COQUETTE":"92","MEUDON":"92","NEUILLY SUR SEINE":"92","LE PLESSIS ROBINSON":"92","SEVRES":"92","AULNAY SOUS BOIS":"93","LE BLANC MESNIL":"93","BONDY":"93","GAGNY":"93","NEUILLY SUR MARNE":"93","LE RAINCY":"93","ROSNY SOUS BOIS":"93","VAUJOURS":"93","VILLEMOMBLE":"93","ABLON SUR SEINE":"94","CRETEIL":"94","GENTILLY":"94","MAISONS ALFORT":"94","LE PERREUX SUR MARNE":"94","RUNGIS":"94","VILLENEUVE ST GEORGES":"94","VILLIERS SUR MARNE":"94","ABLEIGES":"95","AMENUCOURT":"95","ASNIERES SUR OISE":"95","BANTHELU":"95","BRUYERES SUR OISE":"95","VIDOUZE":"65","VIELLE ADOUR":"65","VIELLE LOURON":"65","VIER BORDES":"65","VILLEMUR":"65","CANTAOUS":"65","ALENYA":"66","AMELIE LES BAINS PALALDA":"66","CALMEILLES":"66","CAMELAS":"66","CASEFABRE":"66","CATLLAR":"66","CERBERE":"66","CERET":"66","CLAIRA":"66","CORNEILLA LA RIVIERE":"66","ESTAGEL":"66","FINESTRET":"66","GLORIANES":"66","JOCH":"66","LAMANERE":"66","LATOUR BAS ELNE":"66","LLUPIA":"66","LOS MASOS":"66","MATEMALE":"66","MONTALBA LE CHATEAU":"66","MONT LOUIS":"66","MOSSET":"66","NAHUJA":"66","NYER":"66","PEZILLA LA RIVIERE":"66","POLLESTRES":"66","PORTA":"66","PRATS DE MOLLO LA PRESTE":"66","PRATS DE SOURNIA":"66","RAILLEU":"66","RASIGUERES":"66","RIVESALTES":"66","SAILLAGOUSE":"66","ST FELIU D AMONT":"66","ST FELIU D AVALL":"66","ST PAUL DE FENOUILLET":"66","SOREDE":"66","TAILLET":"66","TAURINYA":"66","TAUTAVEL":"66","LE TECH":"66","VERNET LES BAINS":"66","ALBE":"67","BALDENHEIM":"67","BATZENDORF":"67","BERNOLSHEIM":"67","BISCHHOLTZ":"67","BOESENBIESEN":"67","BOLSENHEIM":"67","BREUSCHWICKERSHEIM":"67","BURBACH":"67","BUTTEN":"67","CROETTWILLER":"67","DACHSTEIN":"67","DAMBACH LA VILLE":"67","DIEFFENTHAL":"67","DOMFESSEL":"67","DONNENHEIM":"67","DRULINGEN":"67","DRUSENHEIM":"67","EBERBACH SELTZ":"67","ECKWERSHEIM":"67","EPFIG":"67","ESCHAU":"67","ETTENDORF":"67","FESSENHEIM LE BAS":"67","FRIESENHEIM":"67","FROESCHWILLER":"67","FURDENHEIM":"67","GEISWILLER":"67","GERTWILLER":"67","GEUDERTHEIM":"67","GOXWILLER":"67","GUMBRECHTSHOFFEN":"67","GUNGWILLER":"67","HATTEN":"67","HERBITZHEIM":"67","HINDISHEIM":"67","VIEUX FORT":"971","L AJOUPA BOUILLON":"972","LE FRANCOIS":"972","LE MARIN":"972","IRACOUBO":"973","KOUROU":"973","SAUL":"973","L ETANG SALE":"974","LES TROIS BASSINS":"974","BANDRABOUA":"976","OUANGANI":"976","PAMANDZI":"976","SADA":"976","FAAA":"987","FATU HIVA":"987","HIKUERU":"987","PIRAE":"987","CANALA":"988","OUVEA":"988","POUM":"988","POYA":"988","TOUHO":"988","AZILONE AMPAZA":"2A","BELVEDERE CAMPOMORO":"2A","BOCOGNANO":"2A","BONIFACIO":"2A","TIERCE":"49","TURQUANT":"49","AGON COUTAINVILLE":"50","ANNEVILLE SUR MER":"50","ARGOUGES":"50","AUMEVILLE LESTRE":"50","LA BALEINE":"50","BARFLEUR":"50","BAUPTE":"50","BERIGNY":"50","BRECEY":"50","BRICQUEBOSQ":"50","CANVILLE LA ROCQUE":"50","CARANTILLY":"50","CARNET":"50","CARNEVILLE":"50","CAROLLES":"50","CERISY LA FORET":"50","CERISY LA SALLE":"50","CHAMPREPUS":"50","COUVILLE":"50","LES CRESNAYS":"50","DIGOSVILLE":"50","EROUDEVILLE":"50","ETIENVILLE":"50","FIERVILLE LES MINES":"50","GENETS":"50","LA GODEFROY":"50","GONFREVILLE":"50","GOUVETS":"50","LE GRAND CELLAND":"50","GREVILLE HAGUE":"50","GRIMESNIL":"50","GROSVILLE":"50","LE GUISLAIN":"50","HEMEVEZ":"50","HERENGUERVILLE":"50","JOBOURG":"50","JOGANVILLE":"50","LENGRONNE":"50","LES LOGES MARCHIS":"50","LOLIF":"50","LE LOREY":"50","MAUPERTUS SUR MER":"50","LE MESNIL GILBERT":"50","LE MESNIL VILLEMAN":"50","MONTABOT":"50","MONTAIGU LA BRISETTE":"50","MONTANEL":"50","MONTCUIT":"50","MONTREUIL SUR LOZON":"50","MORIGNY":"50","MUNEVILLE SUR MER":"50","OMONVILLE LA PETITE":"50","ORGLANDES":"50","OUVILLE":"50","PERIERS":"50","PONTAUBAULT":"50","QUIBOU":"50","REIGNEVILLE BOCAGE":"50","SACEY":"50","BALIGNAC":"82","BOURRET":"82","CAMPSAS":"82","CANALS":"82","CASTELFERRUS":"82","CAYLUS":"82","DURFORT LACAPELETTE":"82","ESPARSAC":"82","ESPINAS":"82","FAJOLLES":"82","FINHAN":"82","GARGANVILLAR":"82","GARIES":"82","LACAPELLE LIVRON":"82","MOISSAC":"82","MONTALZAT":"82","MONTBETON":"82","MONTPEZAT DE QUERCY":"82","ORGUEIL":"82","PERVILLE":"82","PIQUECOS":"82","POMMEVIC":"82","REALVILLE":"82","REYNIES":"82","ST AMANS DU PECH":"82","ST AMANS DE PELLAGAL":"82","ST BEAUZEIL":"82","ST PAUL D ESPIS":"82","ST PORQUIER":"82","VALEILLES":"82","AMPUS":"83","LE BEAUSSET":"83","BESSE SUR ISSOLE":"83","BORMES LES MIMOSAS":"83","BRUE AURIAC":"83","CARCES":"83","CARNOULES":"83","CAVALAIRE SUR MER":"83","COLLOBRIERES":"83","GASSIN":"83","GINASSERVIS":"83","LA MARTRE":"83","MOISSAC BELLEVUE":"83","LE MUY":"83","NEOULES":"83","OLLIERES":"83","PIGNANS":"83","PONTEVES":"83","RAMATUELLE":"83","RIBOUX":"83","STE MAXIME":"83","ST ZACHARIE":"83","SOLLIES VILLE":"83","TRANS EN PROVENCE":"83","BEDOIN":"84","CAIRANNE":"84","CAROMB":"84","CAVAILLON":"84","CHEVAL BLANC":"84","CRESTET":"84","FLASSAN":"84","GOULT":"84","JONQUERETTES":"84","JOUCAS":"84","LAGNES":"84","LIOUX":"84","MORNAS":"84","OPPEDE":"84","PERNES LES FONTAINES":"84","PEYPIN D AIGUES":"84","SABLET":"84","ST MARTIN DE CASTILLON":"84","ST SATURNIN LES APT":"84","SAUMANE DE VAUCLUSE":"84","FONTAINE DE VAUCLUSE":"84","BUTRY SUR OISE":"95","CHAMPAGNE SUR OISE":"95","CLERY EN VEXIN":"95","COMMENY":"95","CONDECOURT":"95","CORMEILLES EN VEXIN":"95","COURCELLES SUR VIOSNE":"95","COURDIMANCHE":"95","DOMONT":"95","FONTENAY EN PARISIS":"95","FREMECOURT":"95","HEROUVILLE":"95","MENOUVILLE":"95","MONTIGNY LES CORMEILLES":"95","MONTMAGNY":"95","MONTSOULT":"95","NUCOURT":"95","OSNY":"95","PISCOP":"95","LE PLESSIS LUZARCHES":"95","PUISEUX PONTOISE":"95","ST CLAIR SUR EPTE":"95","ST CYR EN ARTHIES":"95","SARCELLES":"95","TAVERNY":"95","THEMERICOURT":"95","VIARMES":"95","VILLERON":"95","BAILLIF":"971","BOUILLANTE":"971","CAPESTERRE DE MARIE GALANTE":"971","LE MOULE":"971","TERRE DE HAUT":"971","VIEUX HABITANTS":"971","LE CARBET":"972","CASE PILOTE":"972","DUCOS":"972","GRAND RIVIERE":"972","LE MORNE VERT":"972","MANA":"973","REMIRE MONTJOLY":"973","SINNAMARY":"973","MARIPASOULA":"973","CAMOPI":"973","APATOU":"973","LES AVIRONS":"974","BRAS PANON":"974","ACOUA":"976","BOUENI":"976","CHIRONGUI":"976","DEMBENI":"976","KOUNGOU":"976","HOERDT":"67","LE HOHWALD":"67","ICHTRATZHEIM":"67","INGWILLER":"67","KAUFFENHEIM":"67","KINDWILLER":"67","KIRCHHEIM":"67","KIRRBERG":"67","KRIEGSHEIM":"67","LANGENSOULTZBACH":"67","LICHTENBERG":"67","LINGOLSHEIM":"67","LUTZELHOUSE":"67","MACKENHEIM":"67","MARLENHEIM":"67","MULHAUSEN":"67","MUNDOLSHEIM":"67","MUTZIG":"67","NEUBOIS":"67","NIEDERHAUSBERGEN":"67","NIEDERSCHAEFFOLSHEIM":"67","NORDHEIM":"67","OBERBRONN":"67","OBERHASLACH":"67","OBERLAUTERBACH":"67","ODRATZHEIM":"67","OERMINGEN":"67","ORSCHWILLER":"67","OSTWALD":"67","OTTERSWILLER":"67","OTTROTT":"67","PFULGRIESHEIM":"67","PREUSCHDORF":"67","PUBERG":"67","QUATZENHEIM":"67","RANGEN":"67","REICHSTETT":"67","REINHARDSMUNSTER":"67","REXINGEN":"67","RICHTOLSHEIM":"67","RINGELDORF":"67","ROTT":"67","SAESSOLSHEIM":"67","SAND":"67","SCHAFFHOUSE PRES SELTZ":"67","SCHERLENHEIM":"67","SCHWENHEIM":"67","SELESTAT":"67","SESSENHEIM":"67","SIEGEN":"67","SILTZHEIM":"67","STEINSELTZ":"67","STILL":"67","THANVILLE":"67","TIEFFENBACH":"67","TRIEMBACH AU VAL":"67","UHRWILLER":"67","UTTENHOFFEN":"67","UTTWILLER":"67","WALDERSBACH":"67","WEISLINGEN":"67","WEITBRUCH":"67","WINTZENHEIM KOCHERSBERG":"67","CUTTOLI CORTICCHIATO":"2A","FOZZANO":"2A","MOCA CROCE":"2A","OSANI":"2A","PARTINELLO":"2A","PERI":"2A","PILA CANALE":"2A","PORTO VECCHIO":"2A","SARTENE":"2A","SERRA DI FERRO":"2A","SERRA DI SCOPAMENE":"2A","STE LUCIE DE TALLANO":"2A","ZEVACO":"2A","ZOZA":"2A","AGHIONE":"2B","AITI":"2B","ALGAJOLA":"2B","AMPRIANI":"2B","BARRETTALI":"2B","CAMPI":"2B","CANALE DI VERDE":"2B","CASALTA":"2B","CASTIFAO":"2B","CORTE":"2B","COSTA":"2B","ERONE":"2B","FELICETO":"2B","GALERIA":"2B","GIOCATOJO":"2B","GIUNCAGGIO":"2B","LAMA":"2B","LAVATOGGIO":"2B","LUGO DI NAZZA":"2B","LUMIO":"2B","MERIA":"2B","MOLTIFAO":"2B","MURATO":"2B","NOCETA":"2B","PIAZZALI":"2B","PIAZZOLE":"2B","PIETRICAGGIO":"2B","PIETROSO":"2B","PIOGGIOLA":"2B","POGGIO MEZZANA":"2B","RAPALE":"2B","RUTALI":"2B","SCATA":"2B","SERMANO":"2B","SERRA DI FIUMORBO":"2B","SISCO":"2B","SOVERIA":"2B","SANT ANTONINO":"2B","SAN GIULIANO":"2B","SANTA REPARATA DI MORIANI":"2B","VALLICA":"2B","VOLPAJOLA":"2B","ZALANA":"2B","ST BRICE DE LANDELLES":"50","ST CYR DU BAILLEUL":"50","ST EBREMOND DE BONFOSSE":"50","ST GEORGES DE ROUELLEY":"50","ST GERMAIN DES VAUX":"50","ST GERMAIN SUR AY":"50","ST JAMES":"50","ST JEAN DE LA HAIZE":"50","ST JEAN DE LA RIVIERE":"50","ST MARTIN DE BONFOSSE":"50","ST MARTIN LE BOUILLANT":"50","ST MAURICE EN COTENTIN":"50","ST PIERRE EGLISE":"50","ST SAUVEUR LA POMMERAYE":"50","STE SUZANNE SUR VIRE":"50","SEBEVILLE":"50","SENOVILLE":"50","TANIS":"50","TURQUEVILLE":"50","VALOGNES":"50","LA VENDELEE":"50","VERGONCEY":"50","VIRANDEVILLE":"50","ALLEMANCHE LAUNAY ET SOYER":"51","ANGLUZELLES ET COURCELLES":"51","ARGERS":"51","ARRIGNY":"51","BASLIEUX LES FISMES":"51","BASSU":"51","BAUDEMENT":"51","BEAUNAY":"51","BELVAL SOUS CHATILLON":"51","BERGERES LES VERTUS":"51","BERZIEUX":"51","BEZANNES":"51","BIGNICOURT SUR SAULX":"51","BOUCHY ST GENEST":"51","BOUVANCOURT":"51","BROUSSY LE GRAND":"51","BRUSSON":"51","CAUROY LES HERMONVILLE":"51","LA CELLE SOUS CHANTEMERLE":"51","CHAMERY":"51","CHAMPLAT ET BOUJACOURT":"51","CHAPELAINE":"51","LA CHAPELLE FELCOURT":"51","LA CHAPELLE LASSON":"51","CHARLEVILLE":"51","CHEPPES LA PRAIRIE":"51","CHERVILLE":"51","COMPERTRIX":"51","CONNANTRAY VAUREFROY":"51","COURLANDON":"51","COURTHIEZY":"51","COURTISOLS":"51","CRAMANT":"51","CUISLES":"51","CUMIERES":"51","DOMMARTIN VARIMONT":"51","DOMPREMY":"51","VAL DE VIERE":"51","ECOLLEMONT":"51","ECUEIL":"51","ECURY LE REPOS":"51","EPOYE":"51","ESCARDES":"51","FAVEROLLES ET COEMY":"51","FLORENT EN ARGONNE":"51","L AIGUILLON SUR MER":"85","AUZAY":"85","BELLEVIGNY":"85","BOUFFERE":"85","LE CHAMP ST PERE":"85","CHAVAGNES LES REDOUX":"85","COEX":"85","CORPE":"85","DOMPIERRE SUR YON":"85","LE FENOUILLER":"85","LE GUE DE VELLUIRE":"85","LA GUERINIERE":"85","LES HERBIERS":"85","LUCON":"85","LES MAGNILS REIGNIERS":"85","MAILLEZAIS":"85","MALLIEVRE":"85","MARILLET":"85","LA MEILLERAIE TILLAY":"85","BARCY":"77","BOISDON":"77","CANNES ECLUSE":"77","CARNETIN":"77","CELY":"77","CERNEUX":"77","CHAMIGNY":"77","LA CHAPELLE IGER":"77","CHARMENTRAY":"77","LE CHATELET EN BRIE":"77","CHATENAY SUR SEINE":"77","CHATILLON LA BORDE":"77","CHENOISE":"77","CHEVRAINVILLIERS":"77","COURQUETAINE":"77","CREGY LES MEAUX":"77","CRISENOY":"77","DAMMARTIN EN GOELE":"77","DONNEMARIE DONTILLY":"77","EVERLY":"77","FAREMOUTIERS":"77","FLEURY EN BIERE":"77","GASTINS":"77","GRAVON":"77","GREZ SUR LOING":"77","GUERARD":"77","LA HAUTE MAISON":"77","IVERNY":"77","JOSSIGNY":"77","JOUY LE CHATEL":"77","LARCHANT":"77","LESCHES":"77","LIVERDY EN BRIE":"77","LOGNES":"77","LUISETAINES":"77","LUZANCY":"77","MAGNY LE HONGRE":"77","MAISON ROUGE":"77","MESSY":"77","MOISSY CRAMAYEL":"77","MONS EN MONTOIS":"77","MONTCEAUX LES PROVINS":"77","MONTDAUPHIN":"77","MONTGE EN GOELE":"77","MONTHYON":"77","MOUSSY LE VIEUX":"77","NANGIS":"77","OBSONVILLE":"77","OCQUERRE":"77","OZOIR LA FERRIERE":"77","PONTCARRE":"77","PROVINS":"77","QUIERS":"77","REUIL EN BRIE":"77","NAPUKA":"987","PAPEETE":"987","PUKAPUKA":"987","TUREIA":"987","NOUMEA":"988","SARRAMEA":"988","ARGIUSTA MORICCIO":"2A","BASTELICA":"2A","BASTELICACCIA":"2A","COZZANO":"2A","FOCE":"2A","FRASSETO":"2A","GUAGNO":"2A","LETIA":"2A","MELA":"2A","OLMICCIA":"2A","SAMPOLO":"2A","VALLE DI MEZZANA":"2A","VERO":"2A","VIGGIANELLO":"2A","VILLANOVA":"2A","ZERUBIA":"2A","ZICAVO":"2A","ZIGLIARA":"2A","ALTIANI":"2B","ALZI":"2B","ASCO":"2B","AVAPESSA":"2B","BIGORNO":"2B","BISINCHI":"2B","CAGNANO":"2B","CAMPITELLO":"2B","CASAMACCIOLI":"2B","CASANOVA":"2B","CASEVECCHIE":"2B","CASTELLARE DI MERCURIO":"2B","CASTINETA":"2B","FURIANI":"2B","GHISONI":"2B","L ILE ROUSSE":"2B","LOZZI":"2B","MAUSOLEO":"2B","MONACIA D OREZZA":"2B","NOVELLA":"2B","ORTALE":"2B","ORTIPORIO":"2B","PIANO":"2B","PIEDIGRIGGIO":"2B","PIEDIPARTINO":"2B","PIETRACORBARA":"2B","PIETRA DI VERDE":"2B","PIETRASERENA":"2B","PRUNELLI DI CASACCONI":"2B","QUERCITELLO":"2B","RUSIO":"2B","SPELONCATO":"2B","STAZZONA":"2B","SAN GAVINO D AMPUGNANI":"2B","SAN GAVINO DI TENDA":"2B","SAN LORENZO":"2B","SANTA LUCIA DI MERCURIO":"2B","SANTA MARIA POGGIO":"2B","TOMINO":"2B","TRALONCA":"2B","URTACA":"2B","LA FORESTIERE":"51","FRESNE LES REIMS":"51","GIVRY LES LOISY":"51","GIZAUCOURT":"51","GLANNES":"51","GOURGANCON":"51","LES GRANDES LOGES":"51","GRAUVES":"51","GUEUX":"51","HEILTZ LE HUTIER":"51","HERMONVILLE":"51","LARZICOURT":"51","LEUVRIGNY":"51","LHERY":"51","LINTHES":"51","LUDES":"51","MAIRY SUR MARNE":"51","MALMY":"51","MARSON":"51","LES MESNEUX":"51","MOEURS VERDEY":"51","MONCETZ LONGEVAS":"51","MONDEMENT MONTGIVROUX":"51","LA NEUVILLE AUX LARRIS":"51","OIRY":"51","OMEY":"51","OUTREPONT":"51","PASSY GRIGNY":"51","PIERRE MORAINS":"51","POILLY":"51","REIMS LA BRULEE":"51","REUVES":"51","SACY":"51","ST ETIENNE SUR SUIPPE":"51","ST HILAIRE LE GRAND":"51","ST JUST SAUVAGE":"51","ROUILLY":"77","SABLONNIERES":"77","ST ANGE LE VIEL":"77","ST DENIS LES REBAIS":"77","ST LOUP DE NAUD":"77","ST MAMMES":"77","ST MARTIN EN BIERE":"77","SAMOREAU":"77","SOGNOLLES EN MONTOIS":"77","TANCROU":"77","LA TOMBE":"77","TRILBARDOU":"77","LE VAUDOUE":"77","VAUX LE PENIL":"77","VIGNELY":"77","VILLENEUVE LE COMTE":"77","VILLEPARISIS":"77","VILLE ST JACQUES":"77","VINCY MANOEUVRE":"77","VOISENON":"77","LES ALLUETS LE ROI":"78","AULNAY SUR MAULDRE":"78","BOISSY SANS AVOIR":"78","BREUIL BOIS ROBERT":"78","BRUEIL EN VEXIN":"78","LA CELLE ST CLOUD":"78","CHATOU":"78","CHAVENAY":"78","CHOISEL":"78","CONFLANS STE HONORINE":"78","CRESPIERES":"78","DAMMARTIN EN SERVE":"78","DAVRON":"78","FLEXANVILLE":"78","FLINS NEUVE EGLISE":"78","GAMBAIS":"78","GARGENVILLE":"78","HERBEVILLE":"78","HOUILLES":"78","MEULAN EN YVELINES":"78","MONTCHAUVET":"78","MONTFORT L AMAURY":"78","MOUSSEAUX SUR SEINE":"78","LES MUREAUX":"78","NEAUPHLE LE CHATEAU":"78","PARAY DOUAVILLE":"78","PLAISIR":"78","POIGNY LA FORET":"78","RAIZEUX":"78","RAMBOUILLET":"78","RENNEMOULIN":"78","ST FORGET":"78","ST GERMAIN EN LAYE":"78","ST MARTIN DE BRETHENCOURT":"78","ST MARTIN LA GARENNE":"78","STE MESME":"78","ST REMY LES CHEVREUSE":"78","SAULX MARCHAIS":"78","SENLISSE":"78","SEPTEUIL":"78","SOINDRES":"78","TESSANCOURT SUR AUBETTE":"78","VELIZY VILLACOUBLAY":"78","VERNEUIL SUR SEINE":"78","VILLENNES SUR SEINE":"78","AMAILLOUX":"79","AMURE":"79","ARCAIS":"79","ARDILLEUX":"79","AVAILLES THOUARSAIS":"79","BECELEUF":"79","LA CRECHE":"79","VALLECALLE":"2B","VALLE DI CAMPOLORO":"2B","VENACO":"2B","VERDESE":"2B","VEZZANI":"2B","BRULAIN":"79","LA CHAPELLE POUILLOUX":"79","CHAURAY":"79","CHEF BOUTONNE":"79","CHEY":"79","CHICHE":"79","CIRIERES":"79","CLUSSAIS LA POMMERAIE":"79","COULONGES SUR L AUTIZE":"79","COUTIERES":"79","COUTURE D ARGENSON":"79","EXOUDUN":"79","FENERY":"79","FRESSINES":"79","IRAIS":"79","LIMALONGES":"79","MAISONTIERS":"79","MENIGOUTE":"79","MONCOUTANT":"79","MOUGON":"79","NANTEUIL":"79","OROUX":"79","PAIZAY LE CHAPT":"79","PAIZAY LE TORT":"79","PAMPLIE":"79","PARTHENAY":"79","PERIGNE":"79","PERS":"79","PUIHARDY":"79","REFFANNES":"79","ST CYR LA LANDE":"79","ST ETIENNE LA CIGOGNE":"79","ST GEORGES DE NOISNE":"79","ST GEORGES DE REX":"79","ST GERMAIN DE LONGUE CHAUME":"79","ST JOUIN DE MILLY":"79","ST MAIXENT DE BEUGNE":"79","ST MARTIN DE ST MAIXENT":"79","ST MARTIN DE SANZAY":"79","ST ROMANS DES CHAMPS":"79","STE SOLINE":"79","STE VERGE":"79","SCIECQ":"79","SOMPT":"79","SOUTIERS":"79","TESSONNIERE":"79","THORIGNY SUR LE MIGNON":"79","TILLOU":"79","TRAYES":"79","LE VANNEAU IRLEAU":"79","VERNOUX EN GATINE":"79","VERNOUX SUR BOUTONNE":"79","VILLEMAIN":"79","ABBEVILLE":"80","AGENVILLE":"80","ALBERT":"80","ARGOULES":"80","ARQUEVES":"80","AVESNES CHAUSSOY":"80","BARLEUX":"80","BAYENCOURT":"80","BEHENCOURT":"80","BERGICOURT":"80","BERTANGLES":"80","BERTEAUCOURT LES THENNES":"80","BETHENCOURT SUR MER":"80","BETTENCOURT ST OUEN":"80","BIARRE":"80","BOISBERGUES":"80","BOUCHON":"80","BOUFFLERS":"80","BOUZINCOURT":"80","BUIRE COURCELLES":"80","BUS LA MESIERE":"80","BUSSY LES DAOURS":"80","CACHY":"80","CANAPLES":"80","CARDONNETTE":"80","LE CARDONNOIS":"80","CAULIERES":"80","LA CHAUSSEE TIRANCOURT":"80","PARIS":"75","LYON":"69","MARSEILLE":"13","SAINT ETIENNE":"42","KERST PLABENNEC":"29"}''',
}

def localiser_referentiels():
    """Renvoie les référentiels (JSON embarqués parsés) — aucune lecture de fichier."""
    refs = {cle: json.loads(blob) for cle, blob in _REFERENTIELS_BLOBS.items()}
    print("Référentiels intégrés chargés : " + ", ".join(refs))
    return refs


In [ ]:
# NOTE : cet import isolé en tête de cellule est un reliquat historique ;
# unicodedata est réimporté proprement dans la ligne d'imports ci-dessous.
# Il est conservé tel quel pour ne rien casser (règle : on ne touche pas à ce
# qui fonctionne sans raison impérieuse).
import unicodedata
# ═══════════════════════════════════════════════════════════════
# CELLULE 3 — IMPORTS, PARAMÈTRES GLOBAUX, DÉTECTION DE L'ENVIRONNEMENT
#
# Un notebook partage UN SEUL espace de noms : tout ce qui est défini ici reste
# disponible dans toutes les cellules suivantes. Si vous voyez un « NameError »
# plus bas, c'est presque toujours que cette cellule n'a pas été réexécutée
# après un redémarrage du runtime.
#
# ATTENTION : les paramètres de scoring définis ici (SEUIL_SCORE,
# SEUIL_HAUTE_CONF, PENALISER_RADIEES) sont INTANGIBLES — ils ont été calibrés
# par comparaison de versions successives. Les modifier change silencieusement
# le nombre de TVA trouvées. Voir le README, section « Règles intangibles ».
# ═══════════════════════════════════════════════════════════════
import io, re, time, math, unicodedata
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from rapidfuzz import fuzz
from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

# ── Détection de l'environnement ───────────────────────────────
try:
    from google.colab import files as _colab_files   # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print(f"Environnement : {'Google Colab' if IN_COLAB else 'Jupyter / local'}")

# ── Paramètres ajustables ──────────────────────────────────────
# Score minimal (0-100) pour qu'un candidat de l'API soit ACCEPTÉ comme étant
# le bon bénéficiaire. Calibré empiriquement : la V1 utilisait 82 (trop strict,
# on perdait des entités valides), la V2 utilisait 78 (trop permissif, faux
# positifs). 80 est le compromis retenu et VALIDÉ — ne pas le bouger.
SEUIL_SCORE       = 80    # score minimal pour accepter (V1=82, V2=78 → 80 équilibré)
# Au-dessus de ce score, la ligne est considérée « haute confiance » et
# colorée en VERT dans le fichier ; entre SEUIL_SCORE et ce seuil, elle est
# JAUNE (trouvée mais à l'œil humain de confirmer si besoin).
SEUIL_HAUTE_CONF  = 92    # score >= ce seuil → vert (sinon jaune)
# Pause entre deux appels API. L'API publique tolère ~7 requêtes/seconde ;
# en dessous de 0.15 s on déclenche des HTTP 429 (rate limit) et le traitement
# ralentit au lieu d'accélérer. C'est le facteur qui DÉTERMINE la durée totale
# du traitement : ni Colab ni une machine plus puissante n'y changeront rien.
DELAI_API         = 0.15  # secondes entre appels API (rate limit ≈ 7 req/s)
MAX_PAR_REQUETE   = 8      # nombre de résultats API examinés par requête
# INTANGIBLE. Une entreprise « radiée » (cessée) n'est PAS pénalisée au
# scoring. Raison : beaucoup d'universités et d'EPA ont un SIREN historique
# cessé qui coexiste avec l'actuel ; pénaliser ferait perdre des appariements
# corrects. L'état erroné est corrigé APRÈS coup, par l'Étape E
# (reconcilier_etats_cesses), jamais par le score.
PENALISER_RADIEES = False  # pénalité radiée DÉSACTIVÉE : l'état radié n'influence jamais le score
CONFIRMER_TROUVES = False  # False = la confirmation par exactitude ne touche JAMAIS un résultat trouvé
                           # (récupération pure sur les non-trouvés ; aucun risque de perte)

API_URL = "https://recherche-entreprises.api.gouv.fr/search"

# ── Couleurs Excel (hex sans #) ────────────────────────────────
COULEUR_LIGNE_VERTE = "C6EFCE"
COULEUR_LIGNE_JAUNE = "FFEB9C"
COULEUR_CELL_VERTE  = "00B050"
COULEUR_CELL_ORANGE = "FFC000"

print("Configuration chargée :")
print(f"  Seuil acceptation : {SEUIL_SCORE}   |  Haute confiance : {SEUIL_HAUTE_CONF}")
print(f"  Pénalité radiées  : {'oui (-10%)' if PENALISER_RADIEES else 'non'}")
ENRICHIR_TVA_EXISTANTES = True

# Bleu clair : signature visuelle de TOUTE colonne AJOUTÉE par le pipeline.
# Règle absolue du projet : on ne modifie ni ne colorie jamais une colonne
# d'origine du FTS ; toute information calculée va dans une colonne nouvelle,
# peinte de ce bleu, placée juste à côté de sa colonne source.
C_VERT_AJOUT = "BDD7EE"   # bleu clair : valeur corrigée / colonne ajoutée (étape nettoyage)
print("Config pipeline OK")
VERBOSE = True

Environnement : Google Colab
Configuration chargée :
  Seuil acceptation : 80   |  Haute confiance : 92
  Pénalité radiées  : non
Config pipeline OK


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 3 — MOTEUR DE RECHERCHE TVA v3 (FUSION V1 + V2)
#   • requêtes = stratégies par type (V2) + génériques (V1) en repli
#   • scoring  = adresse granulaire (V2) + pénalité radiée douce (V1)
#   • TVA      = DGFiP réelle si dispo, sinon calculée depuis le SIREN
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# BOÎTE À OUTILS DE NORMALISATION
# Tout le pipeline compare des noms écrits par des humains, dans 27 pays, avec
# des conventions différentes. Ces fonctions ramènent tout à une forme
# canonique comparable. Elles sont utilisées PARTOUT : ne pas en changer le
# comportement sans mesurer l'impact sur le nombre de TVA trouvées.
# ─────────────────────────────────────────────────────────────────────────────

def _normaliser(texte: str) -> str:
    if not texte:
        return ""
    t = unicodedata.normalize("NFD", str(texte).upper())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())


# Le numéro de TVA intracommunautaire français se CALCULE à partir du SIREN :
#   clé = (12 + 3 × (SIREN mod 97)) mod 97,  puis  TVA = "FR" + clé(2 chiffres) + SIREN
# C'est une formule officielle DGFiP, pas une approximation : dès qu'on connaît
# le SIREN, on connaît la TVA avec certitude. Inversement, on retrouve le SIREN
# en retirant les 4 premiers caractères d'une TVA française.
def _siren_vers_tva(siren: str) -> str:
    n = int(str(siren).strip())
    cle = (12 + 3 * (n % 97)) % 97
    return f"FR{cle:02d}{siren}"


# Une case « TVA » du FTS peut être vide de mille façons : chaîne vide, NaN
# pandas, « - », « . », « N/A », ou le mot « AUTRE » que NOUS écrivons en
# sortie. Cette fonction centralise la question « cette TVA est-elle absente ? »
# pour que tout le pipeline réponde de la même manière.
def _tva_manquante(val) -> bool:
    if val is None:
        return True
    try:
        import math
        if isinstance(val, float) and math.isnan(val):
            return True
    except Exception:
        pass
    return str(val).strip() in {"", "-", "nan", "NaN", "N/A", "n/a", "NA", "none", "None"}


PAYS_FR = frozenset({
    "france", "french polynesia", "martinique", "new caledonia",
    "saint pierre and miquelon", "wallis and futuna",
    "guadeloupe", "guyane", "french guiana",
    "mayotte", "reunion", "saint martin", "saint barthelemy",
})

# EXCLUSION : UNIQUEMENT les personnes physiques (correction v3_13). Les entités
# institutionnelles (Conseil de l'Europe, ESA, ONU, République française…) ont un
# SIREN/TVA et NE DOIVENT PLUS être exclues — elles sont désormais recherchées.
PATTERNS_EXCLUSION = [
    "NATURAL PERSON", "PERSONNE PRIVEE", "PERSONNE PHYSIQUE",
    "PRIVATPERSON", "PRIVATE PERSON",
    "ART. 38(7)", "ART 38(7)", "ART. 38 (7)",
]

# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  FONCTION FIGÉE nº1/5 — _est_exclu                                        ║
# ║  NE PAS MODIFIER : le corps de cette fonction est vérifié bit à bit à      ║
# ║  chaque livraison. Toute retouche (même un espace) casse la garantie de    ║
# ║  non-régression et fait varier le nombre de TVA trouvées.                  ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
# RÔLE : écarter d'emblée les lignes qu'on ne DOIT pas chercher — les personnes
# physiques anonymisées par la Commission (« NATURAL PERSON », « PERSONNE
# PRIVÉE », « Art. 38(7) »...). Ce sont des particuliers : ils n'ont ni SIREN
# ni TVA, les chercher produirait des faux positifs et gaspillerait des appels.
def _est_exclu(nom: str) -> bool:
    nom_up = str(nom).upper()
    return any(pat in nom_up for pat in PATTERNS_EXCLUSION)


PAYS_BLACKLIST = {
    "FRANCE", "BELGIQUE", "BELGIUM", "ALLEMAGNE", "GERMANY",
    "ESPAGNE", "SPAIN", "ITALIE", "ITALY", "LUXEMBOURG",
    "AUTRICHE", "AUSTRIA", "SUISSE", "SWITZERLAND",
    "PAYS BAS", "NETHERLANDS", "PORTUGAL", "POLOGNE", "POLAND",
    "GRECE", "GREECE", "DANEMARK", "DENMARK", "FINLANDE", "FINLAND",
    "SUEDE", "SWEDEN", "IRLANDE", "IRELAND", "UK", "ROYAUME UNI",
    "UNITED KINGDOM", "UNITED STATES", "ETATS UNIS", "CANADA",
    "MAROC", "TUNISIE", "TCHEQUE", "CZECH", "ROUMANIE", "ROMANIA",
}

SUFFIXES_REGEX = [
    r"\bS\.?A\.?S\.?\b", r"\bS\.?A\.?R\.?L\.?\b", r"\bE\.?U\.?R\.?L\.?\b",
    r"\bS\.?N\.?C\.?\b",  r"\bS\.?C\.?A\.?\b",    r"\bS\.?C\.?S\.?\b",
    r"\bS\.?C\.?O\.?P\.?\b", r"\bS\.?C\.?I\.?C\.?\b",
    r"\bG\.?I\.?E\.?\b",  r"\bG\.?I\.?P\.?\b",
    r"\bE\.?P\.?I\.?C\.?\b", r"\bE\.?P\.?A\.?\b",
    r"\bASSOCIATION\b",   r"\bASSOC\.?\b",
    r"\bFONDATION\b",     r"\bFOND\.?\b",
    r"\bSTe?\.?\b",
    r"\bLTD\.?\b",        r"\bGMBH\b",
]
# NB : "COMMUNE" a été VOLONTAIREMENT retiré des suffixes.
#      On ne supprime jamais "COMMUNE" / "COMMUNE DE" d'un nom (consigne métier).

def _supprimer_suffixes(nom: str) -> str:
    r = str(nom).upper()
    for pat in SUFFIXES_REGEX:
        r = re.sub(pat, "", r, flags=re.IGNORECASE)
    return " ".join(r.split()).strip("-").strip(",").strip()


STOP_WORDS = {
    "DE", "DU", "DES", "LE", "LA", "LES", "ET", "EN", "AU", "AUX",
    "L", "D", "UN", "UNE", "POUR", "PAR", "SUR", "DANS", "A",
    "THE", "OF", "AND", "FOR", "IN", "TO", "DI", "DEL",
}

def _mots_cles(nom: str, n: int = 4) -> str:
    mots = re.sub(r"[^\w\s]", " ", str(nom).upper()).split()
    return " ".join([m for m in mots if m not in STOP_WORDS and len(m) > 1][:n])


def _extraire_core(nom: str) -> str:
    # IMPORTANT : on ne retire JAMAIS "COMMUNE"/"COMMUNE DE" (consigne métier).
    # _extraire_core ne sert plus que pour les entités AUTRE (suppression des
    # suffixes juridiques uniquement).
    return _supprimer_suffixes(nom)


# ── Troncature conditionnelle du mot d'en-tête (ASSOCIATION, FEDERATION…) ─────
# Règle métier :
#   • On retire le mot d'en-tête UNIQUEMENT s'il est suivi DIRECTEMENT d'un mot plein.
#   • On le CONSERVE s'il est suivi d'une préposition/article (DE, DU, DES, D',
#     POUR, LA, LE, …) : dans ce cas le mot d'en-tête fait partie du nom réel.
# Listes confirmées sur les 66 000 lignes du fichier FTS.

ENTETES_ASSO = (
    "ASSOCIATION", "ASSOC", "FEDERATION", "FÉDÉRATION", "CONFEDERATION", "CONFÉDÉRATION",
    "LIGUE", "AMICALE", "COMITE", "COMITÉ", "UNION", "COLLECTIF",
    "GROUPEMENT", "MOUVEMENT", "SYNDICAT", "FONDATION",
)

PREPOSITIONS_BLOQUANTES = {
    "DE", "DU", "DES", "D", "LA", "LE", "LES", "L",
    "POUR", "A", "AU", "AUX", "EN", "ET", "SUR", "AVEC",
}

_RE_ENTETE = re.compile(
    r"^(" + "|".join(ENTETES_ASSO) + r")\.?\s+(.*)$",
    re.IGNORECASE,
)

def _tronquer_entete(nom: str):
    """
    Retourne le nom SANS le mot d'en-tête si — et seulement si — ce mot est
    suivi directement d'un mot plein. Sinon retourne None (on garde le nom complet).
    """
    m = _RE_ENTETE.match(nom.strip())
    if not m:
        return None
    reste = m.group(2).strip()
    if not reste:
        return None
    premier = reste.split()[0]
    # normaliser : enlever apostrophe finale ("D'" -> "D", "L'" -> "L")
    premier_norm = re.sub(r"['’].*$", "", premier.upper()).strip(".")
    if premier_norm in PREPOSITIONS_BLOQUANTES:
        return None          # suivi d'une préposition/article → NE PAS tronquer
    reste = re.sub(r"\s*\(.*?\)", "", reste).strip()  # enlever (DEPT) éventuel
    return reste if len(reste) > 3 else None


def _detecter_type(nom: str) -> str:
    nom_up = nom.upper()
    if re.match(r"^(COMMUNE|VILLE DE|MAIRIE|COMMUNAUTE|COMMUNAUTÉ)\b", nom_up):
        return "COMMUNE"
    if re.match(r"^(ASSOCIATION|ASSOC\.?|FEDERATION|FÉDÉRATION|LIGUE|AMICALE|"
                r"COMITE|COMITÉ|UNION|COLLECTIF|GROUPEMENT|MOUVEMENT)\b", nom_up):
        return "ASSOCIATION"
    return "AUTRE"


# ── Garde-fou COMMUNE : le résultat API doit être une vraie collectivité ──────
# Une commune/EPCI est toujours enregistrée sous un nom qui COMMENCE par un
# marqueur de collectivité. Sinon le match (club, comité des fêtes, caisse des
# écoles… de la même ville) est un FAUX POSITIF et doit être rejeté.
_MARQUEURS_COLLECTIVITE = (
    "COMMUNE", "MAIRIE", "VILLE DE", "COMMUNAUTE", "METROPOLE", "EUROMETROPOLE",
    "AGGLOMERATION", "DEPARTEMENT", "REGION ", "CONSEIL DEPARTEMENTAL",
    "CONSEIL REGIONAL", "SYNDICAT", "SIVOM", "SIVU", "SMICTOM", "SMIRTOM",
    "SDIS", "PETR", "POLE METROPOLITAIN",
)

def _est_collectivite(nom_api_norm: str) -> bool:
    """True si le nom (déjà normalisé) commence par un marqueur de collectivité."""
    n = (nom_api_norm or "").strip()
    return any(n == m.strip() or n.startswith(m.strip() + " ") or n.startswith(m)
               for m in _MARQUEURS_COLLECTIVITE)


def _dedup(reqs: list) -> list:
    vus, result = set(), []
    for r in reqs:
        cle = (r["q"].strip().lower(), r.get("cp"))
        if cle not in vus and r["q"].strip():
            vus.add(cle)
            result.append(r)
    return result


_GEO_QUALIFIERS = re.compile(
    r"\b(NATIONALE?S?|DE FRANCE|FRAN[CÇ]AISE?S?|R[EÉ]GIONALE?S?|D[EÉ]PARTEMENTALE?S?|"
    r"INTERNATIONALE?S?|EUROP[EÉ]ENNE?S?|F[EÉ]D[EÉ]RALE?S?|METROPOLITAINE?S?)\b",
    re.IGNORECASE,
)

def _supprimer_geo(nom: str):
    result = _GEO_QUALIFIERS.sub("", nom)
    result = re.sub(r"\s{2,}", " ", result).strip(" ,-")
    if _normaliser(result) != _normaliser(nom) and len(result) > 4:
        return result
    return None


def _sigle(nom: str, reqs: list, cp) -> list:
    if "*" not in nom:
        return reqs
    parties = [p.strip() for p in nom.split("*") if p.strip()]
    sg = min(parties, key=len)
    cle = _normaliser(sg)
    if (len(cle.replace(" ", "")) >= 4 and cle not in PAYS_BLACKLIST
            and cle != _normaliser(nom)
            and cle not in {_normaliser(r["q"]) for r in reqs}):
        reqs.append({"q": sg, "cp": cp, "s": "SIGLE"})
    return reqs


# ══════════════════════════════════════════════════════════════════════════════
# GÉNÉRATION DES REQUÊTES — FUSION
#   1) stratégies par TYPE (issu de V2, avec adresse)
#   2) stratégies GÉNÉRIQUES (issu de V1) ajoutées en repli (préfixe G_)
# ══════════════════════════════════════════════════════════════════════════════

def _requetes_par_type(nom: str, adresse=None, cp=None, ville=None) -> list:
    """Stratégies spécialisées par type d'entité (logique V2)."""
    nom      = str(nom).strip()
    nom_base = nom.split("*")[0].strip() if "*" in nom else nom
    core     = _extraire_core(nom_base)
    type_e   = _detecter_type(nom)
    reqs     = []

    adr_clean = ""
    if adresse and str(adresse).strip():
        adr_clean = re.sub(r"[^\w\s]", " ", str(adresse).upper()).strip()
        adr_clean = re.sub(r"\b\d{5}\b", "", adr_clean).strip()

    def avec_adr(q: str) -> str:
        return f"{q} {adr_clean}".strip() if adr_clean else q

    if type_e == "COMMUNE":
        # On garde TOUJOURS le nom complet "COMMUNE DE X" — jamais de noyau.
        nom_propre = re.sub(r"\s*\(.*?\)", "", nom_base).strip()  # retire "(ISERE)" etc.
        q_adr = avec_adr(nom_propre)
        if q_adr != nom_propre:
            reqs.append({"q": q_adr, "cp": cp, "s": "C1_NOM_ADR_CP"})
        reqs.append({"q": nom_propre, "cp": cp, "s": "C2_NOM_CP"})
        if cp:
            reqs.append({"q": nom_propre, "cp": None, "s": "C3_NOM"})
        # si parenthèses retirées, tenter aussi le nom_base tel quel
        if _normaliser(nom_propre) != _normaliser(nom_base):
            reqs.append({"q": nom_base, "cp": cp, "s": "C4_NOM_BRUT_CP"})
        reqs = _sigle(nom, reqs, cp)

    elif type_e == "ASSOCIATION":
        # Troncature CONDITIONNELLE du mot d'en-tête (voir _tronquer_entete) :
        #   - retirée seulement si suivie d'un mot plein
        #   - conservée si suivie d'une préposition/article (DE, DU, POUR, LA…)
        sans_asso = _tronquer_entete(nom_base)
        if sans_asso:
            sans_asso = sans_asso.split("*")[0].strip() if "*" in sans_asso else sans_asso

        if sans_asso and len(sans_asso) > 3 and _normaliser(sans_asso) != _normaliser(nom_base):
            q_sa = avec_adr(sans_asso)
            if q_sa != sans_asso:
                reqs.append({"q": q_sa, "cp": cp, "s": "A1_SANS_ENTETE_ADR_CP"})
            reqs.append({"q": sans_asso, "cp": cp, "s": "A2_SANS_ENTETE_CP"})
            if cp:
                reqs.append({"q": sans_asso, "cp": None, "s": "A3_SANS_ENTETE"})

        q_adr = avec_adr(nom_base)
        if q_adr != nom_base:
            reqs.append({"q": q_adr, "cp": cp, "s": "A4_NOM_ADR_CP"})
        reqs.append({"q": nom_base, "cp": cp, "s": "A5_NOM_CP"})
        if cp:
            reqs.append({"q": nom_base, "cp": None, "s": "A6_NOM"})

        mc = _mots_cles(nom_base, 4)
        if mc and len(mc) > 5 and _normaliser(mc) not in {_normaliser(r["q"]) for r in reqs}:
            reqs.append({"q": mc, "cp": cp, "s": "A7_MOTS_CLES"})

        if ville and str(ville).strip() and _normaliser(ville) not in PAYS_BLACKLIST:
            premiers = _mots_cles(nom_base, 2)
            if premiers:
                reqs.append({"q": f"{premiers} {str(ville).strip()}", "cp": None, "s": "A8_NOM_VILLE"})

        base_geo = _supprimer_geo(sans_asso) if sans_asso else None
        if base_geo and _normaliser(base_geo) not in {_normaliser(r["q"]) for r in reqs}:
            reqs.append({"q": base_geo, "cp": None, "s": "A9_SANS_GEO"})
        else:
            nom_geo = _supprimer_geo(nom_base)
            if nom_geo and _normaliser(nom_geo) not in {_normaliser(r["q"]) for r in reqs}:
                reqs.append({"q": nom_geo, "cp": None, "s": "A9_SANS_GEO"})
        reqs = _sigle(nom, reqs, cp)

    else:  # AUTRE
        q1 = avec_adr(nom_base)
        if q1 != nom_base:
            reqs.append({"q": q1, "cp": cp, "s": "E1_NOM_ADR_CP"})
        # E2_CORE_ADR_CP SUPPRIMÉE : le découpage en "noyau" génère des faux positifs.
        reqs.append({"q": nom_base, "cp": cp, "s": "E3_NOM_CP"})
        if cp:
            reqs.append({"q": nom_base, "cp": None, "s": "E4_NOM"})
        ss = _supprimer_suffixes(nom_base)
        if ss and len(ss) > 4 and _normaliser(ss) not in {_normaliser(r["q"]) for r in reqs}:
            reqs.append({"q": ss, "cp": cp, "s": "E6_SANS_SUFFIXE"})
        mc = _mots_cles(nom_base, 4)
        if mc and len(mc) > 5 and _normaliser(mc) not in {_normaliser(r["q"]) for r in reqs}:
            reqs.append({"q": mc, "cp": cp, "s": "E7_MOTS_CLES"})
        nn = _normaliser(nom_base)
        if nn and nn not in {_normaliser(r["q"]) for r in reqs}:
            reqs.append({"q": nn, "cp": cp, "s": "E8_NOM_NORMALISE"})
        if len(nom_base) > 55:
            tronque = nom_base[:50].rsplit(" ", 1)[0]
            if len(tronque) > 12 and _normaliser(tronque) not in {_normaliser(r["q"]) for r in reqs}:
                reqs.append({"q": tronque, "cp": None, "s": "E9_NOM_TRONQUE"})
        nom_geo = _supprimer_geo(nom_base)
        if nom_geo and _normaliser(nom_geo) not in {_normaliser(r["q"]) for r in reqs}:
            reqs.append({"q": nom_geo, "cp": None, "s": "E10_SANS_GEO"})
        reqs = _sigle(nom, reqs, cp)

    return reqs


def _requetes_generiques(nom_brut: str, ville=None, cp=None) -> list:
    """Stratégies génériques de V1 — ajoutées en repli (préfixe G_)."""
    nom = str(nom_brut).strip()
    reqs = []
    reqs.append({"q": nom, "cp": cp, "s": "G1_NOM_COMPLET_CP"})
    if cp:
        reqs.append({"q": nom, "cp": None, "s": "G2_NOM_COMPLET"})
    if "*" in nom:
        parties = [p.strip() for p in nom.split("*") if p.strip()]
        nom_principal = max(parties, key=len)
        if nom_principal.upper() != nom.upper():
            reqs.append({"q": nom_principal, "cp": cp, "s": "G3_NOM_PRINCIPAL"})
    sans_suf = _supprimer_suffixes(nom)
    if sans_suf and len(sans_suf) > 4 and sans_suf.upper() != nom.upper():
        reqs.append({"q": sans_suf, "cp": cp, "s": "G5_SANS_SUFFIXE"})
    mots_cles = _mots_cles(nom, 4)
    if mots_cles and len(mots_cles) > 5:
        reqs.append({"q": mots_cles, "cp": cp, "s": "G6_MOTS_CLES"})
    if len(nom) > 45:
        tronque = nom[:42].rsplit(" ", 1)[0]
        if len(tronque) > 10:
            reqs.append({"q": tronque, "cp": None, "s": "G8_NOM_TRONQUE"})
    if ville and str(ville).strip():
        premiers = _mots_cles(nom, 2)
        if premiers:
            reqs.append({"q": f"{premiers} {str(ville).strip()}", "cp": None, "s": "G9_NOM_VILLE"})
    return reqs


def _nom_base_sans_star(nom: str) -> str:
    """Partie avant le premier '*' (le nom principal)."""
    return nom.split("*")[0].strip() if "*" in nom else nom


def _requete_sans_parentheses(nom: str, cp=None) -> list:
    """Stratégie SANS_PAR : nom complet débarrassé des '(...)'. Pour TOUS les types."""
    base = _nom_base_sans_star(str(nom).strip())
    sans_par = re.sub(r"\s*\(.*?\)", "", base).strip()
    if sans_par and _normaliser(sans_par) != _normaliser(base) and len(sans_par) > 3:
        return [
            {"q": sans_par, "cp": cp, "s": "P1_SANS_PAR_CP"},
            {"q": sans_par, "cp": None, "s": "P2_SANS_PAR"},
        ]
    return []


def _requetes_apres_star(nom: str, cp=None) -> list:
    """
    Stratégie STAR : cherche le(s) SEGMENT(S) COMPLET(S) après '*'.
    Utile quand la partie après '*' est un nom alternatif réel : une collectivité
    doublon (MAIRIE DE X), un comité, une école, une structure rattachée…
    (Le sigle court <= 6 caractères reste géré séparément par _sigle.)
    Ces requêtes ne sont PAS soumises au garde-fou collectivité (marquées _star).
    """
    if "*" not in str(nom):
        return []
    out = []
    parties = [p.strip() for p in str(nom).split("*") if p.strip()]
    for seg in parties[1:]:
        seg = re.sub(r"\s*\(.*?\)", "", seg).strip()
        if len(seg.replace(" ", "")) > 6:        # segment "long" = nom alternatif
            out.append({"q": seg, "cp": cp,   "s": "STAR_APRES_CP", "_star": True})
            out.append({"q": seg, "cp": None, "s": "STAR_APRES",    "_star": True})
    return out


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  FONCTION FIGÉE nº2/5 — _generer_requetes                                 ║
# ║  NE PAS MODIFIER (vérifiée bit à bit).                                    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
# RÔLE : à partir d'UN nom FTS, fabriquer la LISTE ORDONNÉE des requêtes à
# tenter contre l'API. C'est ici que vivent les 33 stratégies du moteur :
# nom complet, nom sans suffixe juridique, cœur du nom, sigle, mots-clés,
# variantes avec/sans code postal, etc. — de la plus précise à la plus large.
# L'ordre compte : la première requête qui produit un candidat au-dessus du
# seuil gagne, on n'essaie pas les suivantes (économie d'appels API).
# Le détail stratégie par stratégie est documenté dans le README.
def _generer_requetes(nom: str, adresse=None, cp=None, ville=None) -> list:
    """
    FUSION des stratégies, dédupliquée :
      • par type (V2) — SANS les anciennes stratégies CORE (faux positifs)
      • génériques (V1) en repli
      • nom sans parenthèses (P1/P2)
      • segment après '*' (STAR) — peut viser un comité/structure rattaché
    Garde-fou collectivité : actif sur les requêtes basées sur le nom de la
    commune, désactivé sur les requêtes STAR (qui ciblent une autre entité).
    """
    type_e = _detecter_type(nom)
    reqs  = _requetes_par_type(nom, adresse, cp, ville)
    reqs += _requete_sans_parentheses(nom, cp)
    reqs += _requetes_generiques(nom, ville, cp)
    reqs += _requetes_apres_star(nom, cp)

    if type_e == "COMMUNE":
        # toutes les requêtes "nom de commune" exigent une collectivité,
        # sauf les requêtes STAR qui cherchent une structure rattachée distincte
        for r in reqs:
            if not r.get("_star"):
                r["col"] = True
    return _dedup(reqs)


# ══════════════════════════════════════════════════════════════════════════════
# SCORING v4+ (adresse granulaire V2 + pénalité radiée douce V1)
# ══════════════════════════════════════════════════════════════════════════════

def _extraire_rue(adresse: str) -> str:
    s = _normaliser(str(adresse))
    s = re.sub(r"\b\d{5}\b", "", s)
    return " ".join(s.split())


def _construire_rue_api(siege: dict) -> str:
    parties = []
    num = str(siege.get("numero_voie") or "").strip()
    indice = str(siege.get("indice_repetition") or "").strip()
    if num:
        parties.append(num + indice)
    type_v = str(siege.get("type_voie") or "").strip()
    lib_v = str(siege.get("libelle_voie") or "").strip()
    if type_v: parties.append(type_v)
    if lib_v:  parties.append(lib_v)
    if parties:
        return _normaliser(" ".join(parties))
    return _extraire_rue(str(siege.get("adresse") or ""))


def _score_adresse(ref_adr, siege):
    if not ref_adr or not siege:
        return 0, ""
    ref_rue = _extraire_rue(str(ref_adr))
    if not ref_rue or len(ref_rue) < 4:
        return 0, ""
    api_rue = _construire_rue_api(siege)
    if not api_rue or len(api_rue) < 4:
        return 0, ""
    sim = fuzz.token_sort_ratio(ref_rue, api_rue)
    bonus = 0
    comp = str(siege.get("complement_adresse") or "").strip()
    if comp and fuzz.token_sort_ratio(ref_rue, _normaliser(comp)) >= 75:
        bonus = 2
    if sim >= 85:
        pts = min(6 + bonus, 8)
        return pts, f"adr+{pts}({api_rue[:26]})"
    if sim >= 70:
        pts = min(3 + bonus, 5)
        return pts, f"adr+{pts}({api_rue[:26]})"
    if sim < 30 and len(ref_rue) >= 8 and len(api_rue) >= 8:
        return -5, f"adr!=({api_rue[:26]})"
    return 0, f"adr~{sim}%"


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  FONCTION FIGÉE nº4/5 — _scorer                                           ║
# ║  NE PAS MODIFIER (vérifiée bit à bit). C'est LE cœur du système.          ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
# RÔLE : donner une note 0-100 à un candidat renvoyé par l'API, en combinant
# la similarité du NOM (rapidfuzz), la concordance de l'ADRESSE, de la VILLE
# et du CODE POSTAL, plus des bonus/malus métier (établissement actif, type
# d'entité cohérent...). C'est cette note qui est comparée à SEUIL_SCORE (80)
# pour accepter ou rejeter, puis à SEUIL_FICHIER_TVA (96) pour décider si la
# TVA entre dans le fichier final.
def _scorer(ref_nom, api_nom, ref_adr, ref_ville, ref_cp, api_res):
    if not api_nom:
        return 0, "nom_vide"
    audit = []

    # ① Similarité textuelle
    s_sort = fuzz.token_sort_ratio(ref_nom, api_nom)
    s_set  = fuzz.token_set_ratio(ref_nom, api_nom)
    audit += [f"sort={s_sort}", f"set={s_set}"]

    # ② Ratio de longueur + partial conditionnel
    l_ref = max(len(ref_nom.replace(" ", "")), 1)
    l_api = max(len(api_nom.replace(" ", "")), 1)
    ratio = min(l_ref, l_api) / max(l_ref, l_api)
    if ratio >= 0.60:
        sp = fuzz.partial_ratio(ref_nom, api_nom)
        score = max(s_sort, int(s_set * 0.97), int(sp * 0.88))
        audit.append(f"partial={sp}->{int(sp*0.88)}")
    else:
        score = int(max(s_sort, int(s_set * 0.97)) * 0.90)
        audit.append(f"partial=ignore(ratio={ratio:.2f},-10%)")

    # ★ NOM PARFAIT : si le nom du bénéficiaire correspond EXACTEMENT à celui de
    #   l'API (exactitude 100 via _exactitude, qui neutralise mots juridiques et
    #   pénalise les mots en trop), on NEUTRALISE les pénalités géographiques
    #   (adresse/ville/département) : une adresse divergente ne doit pas faire
    #   chuter une identité de nom certaine. Les BONUS géo restent appliqués.
    nom_parfait = _exactitude(ref_nom, api_nom) >= 100
    if nom_parfait:
        audit.append("nom_exact=100(pénalités géo neutralisées)")

    # ③ Adresse granulaire (V2)
    siege = api_res.get("siege") or {}
    delta, msg = _score_adresse(ref_adr, siege)
    if delta > 0:
        score = min(100, score + delta); audit.append(msg)
    elif delta < 0 and score < 95 and not nom_parfait:
        score = max(0, int(score * 0.95)); audit.append(msg)
    elif msg:
        audit.append(msg)

    # ④ Ville
    api_cp = str(siege.get("code_postal", "") or "").strip()
    api_commune = _normaliser(siege.get("libelle_commune", "") or "")
    if ref_ville and api_commune:
        sv = fuzz.token_sort_ratio(_normaliser(ref_ville), api_commune)
        if sv >= 80:
            score = min(100, score + 5); audit.append(f"ville+5({api_commune})")
        elif sv < 35 and score < 95 and not nom_parfait:
            score = max(0, int(score * 0.95)); audit.append(f"ville-5%({api_commune})")

    # ⑤ Département
    api_dept = api_cp[:2] if len(api_cp) >= 2 else ""
    ref_dept = ref_cp[:2] if ref_cp and len(ref_cp) >= 2 else ""
    if ref_dept and api_dept:
        if ref_dept == api_dept:
            score = min(100, score + 3); audit.append(f"dept={ref_dept}+3")
        elif score < 93 and not nom_parfait:
            score = max(0, int(score * 0.97)); audit.append(f"dept!=({ref_dept}!={api_dept})-3%")

    # ⑥ État administratif — pénalité DOUCE (V1 réintégré, assoupli)
    if PENALISER_RADIEES:
        etat = (api_res.get("etat_administratif") or "").strip().upper()
        if etat == "C" and score < 90:
            score = max(0, int(score * 0.90)); audit.append("radiee-10%")

    return min(100, max(0, score)), " | ".join(audit)




# ── AJOUTS (n'affectent ni la recherche ni le scoring) ───────────────────────
_FORMES_JURIDIQUES_SECOURS = {
    "0000": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "1000": "Entrepreneur individuel",
    "1100": "Artisan-commerçant",
    "1200": "Commerçant",
    "1300": "Artisan",
    "1400": "Officier public ou ministériel",
    "1500": "Profession libérale",
    "1600": "Exploitant agricole",
    "1700": "Agent commercial",
    "1800": "Associé-gérant de société",
    "1900": "(Autre) Personne physique",
    "2110": "Indivision entre personnes physiques",
    "2120": "Indivision avec personne morale",
    "2210": "Société créée de fait entre personnes physiques",
    "2220": "Société créée de fait avec personne morale",
    "2310": "Société en participation entre personnes physiques",
    "2320": "Société en participation avec personne morale",
    "2385": "Société en participation de professions libérales",
    "2400": "Fiducie",
    "2700": "Paroisse hors zone concordataire",
    "2900": "Autre groupement de droit privé non doté de la personnalité morale",
    "3110": "Représentation ou agence commerciale d'état ou organisme public étranger immatriculé au RCS",
    "3120": "Société commerciale étrangère immatriculée au RCS",
    "3205": "Organisation internationale",
    "3210": "État, collectivité ou établissement public étranger",
    "3220": "Société étrangère non immatriculée au RCS",
    "3290": "Autre personne morale de droit étranger",
    "4110": "Établissement public national à caractère industriel ou commercial doté d'un comptable public",
    "4120": "Établissement public national à caractère industriel ou commercial non doté d'un comptable public",
    "4130": "Exploitant public",
    "4140": "Établissement public local à caractère industriel ou commercial",
    "4150": "Régie d'une collectivité locale à caractère industriel ou commercial",
    "4160": "Institution Banque de France",
    "5191": "Société de caution mutuelle",
    "5192": "Société coopérative de banque populaire",
    "5193": "Caisse de crédit maritime mutuel",
    "5194": "Caisse (fédérale) de crédit mutuel",
    "5195": "Association coopérative inscrite (droit local Alsace Moselle)",
    "5196": "Caisse d'épargne et de prévoyance à forme coopérative",
    "5202": "SNC, Société en nom collectif",
    "5203": "Société en nom collectif coopérative",
    "5306": "SCS, Société en commandite simple",
    "5307": "Société en commandite simple coopérative",
    "5308": "SCA, Société en commandite par actions",
    "5309": "Société en commandite par actions coopérative",
    "5310": "SLP, Société en libre partenariat",
    "5370": "Société de Participations Financières de Profession Libérale Société en commandite par actions (SPFPL SCA)",
    "5385": "Société d'exercice libéral en commandite par actions",
    "5410": "Société nationale à responsabilité limitée",
    "5415": "Société d'économie mixte à responsabilité limitée",
    "5422": "Société immobilière pour le commerce et l'industrie (SICOMI) à responsabilité limitée",
    "5426": "Société immobilière de gestion à responsabilité limitée",
    "5430": "Société d'aménagement foncier et d'équipement rural (SAFER) à responsabilité limitée",
    "5431": "Société mixte d'intérêt agricole (SMIA) à responsabilité limitée",
    "5432": "Société d'intérêt collectif agricole à responsabilité limitée",
    "5442": "Société d'attribution à responsabilité limitée",
    "5443": "Société coopérative de construction à responsabilité limitée",
    "5451": "Société coopérative de consommation à responsabilité limitée",
    "5453": "Société coopérative artisanale à responsabilité limitée",
    "5454": "Société coopérative d'intérêt maritime à responsabilité limitée",
    "5455": "Société coopérative de transport routier à responsabilité limitée",
    "5458": "Société coopérative ouvrière de production (SCOP) à responsabilité limitée",
    "5459": "Union de sociétés coopératives à responsabilité limitée",
    "5460": "Autre SARL coopérative",
    "5470": "Société de Participations Financières de Profession Libérale Société à responsabilité limitée (SPFPL SARL)",
    "5485": "SELARL, Société d'exercice libéral à responsabilité limitée",
    "5498": "SARL unipersonnelle",
    "5499": "SARL, Société à responsabilité limitée (sans autre indication)",
    "5505": "Société anonyme à participation ouvrière à conseil d'administration",
    "5510": "Société anonyme nationale à conseil d'administration",
    "5515": "Société anonyme d'économie mixte à conseil d'administration",
    "5520": "Fonds à forme sociétale à conseil d'administration",
    "5522": "Société anonyme immobilière pour le commerce et l'industrie à conseil d'administration (SICOMI)",
    "5525": "Société anonyme immobilière d'investissement à conseil d'administration",
    "5530": "Société anonyme d'aménagement foncier et d'équipement rural à conseil d'administration (SAFER)",
    "5531": "Société anonyme mixte d'intérêt agricole à conseil d'administration (SMIA)",
    "5532": "Société anonyme d'intérêt collectif agricole à conseil d'administration (SICA)",
    "5542": "Société anonyme d'attribution à conseil d'administration",
    "5543": "Société anonyme coopérative de construction à conseil d'administration",
    "5546": "Société anonyme d'HLM à conseil d'administration",
    "5547": "Société anonyme coopérative de production de HLM à conseil d'administration",
    "5548": "SA de crédit immobilier à conseil d'administration",
    "5551": "Société anonyme coopérative de consommation à conseil d'administration",
    "5552": "Société anonyme coopérative de commerçants-détaillants à conseil d'administration",
    "5553": "Société anonyme coopérative artisanale à conseil d'administration",
    "5554": "Société anonyme coopérative (d'intérêt) maritime à conseil d'administration",
    "5555": "Société anonyme coopérative de transport à conseil d'administration",
    "5558": "Société anonyme coopérative de production à conseil d'administration (SCOP)",
    "5559": "Union de sociétés coopératives à forme anonyme et à conseil d'administration",
    "5560": "Autre société anonyme coopérative à conseil d'administration",
    "5570": "Société de Participations Financières de Profession Libérale Société anonyme à conseil d'administration (SFPL)",
    "5585": "Société d'exercice libéral à forme anonyme à conseil d'administration",
    "5599": "Société anonyme à conseil d'administration (sans autre indication)",
    "5605": "Société anonyme à participation ouvrière à directoire",
    "5610": "Société anonyme nationale à directoire",
    "5615": "Société anonyme d'économie mixte à directoire",
    "5620": "Fonds à forme sociétale à directoire",
    "5622": "Société anonyme immobilière pour le commerce et l'industrie à directoire (SICOMI)",
    "5625": "Société anonyme immobilière d'investissement à directoire",
    "5630": "Société anonyme d'aménagement foncier et d'équipement rural à directoire (SAFER)",
    "5631": "Société anonyme mixte d'intérêt agricole à directoire (SMIA)",
    "5632": "Société anonyme d'intérêt collectif agricole à directoire (SICA)",
    "5642": "Société anonyme d'attribution à directoire",
    "5643": "Société anonyme coopérative de construction à directoire",
    "5646": "Société anonyme de HLM à directoire",
    "5647": "Société anonyme coopérative de production de HLM à directoire",
    "5648": "SA de crédit immobilier à directoire",
    "5651": "Société anonyme coopérative de consommation à directoire",
    "5652": "Société anonyme coopérative de commerçants-détaillants à directoire",
    "5653": "Société anonyme coopérative artisanale à directoire",
    "5654": "Société anonyme coopérative d'intérêt maritime à directoire",
    "5655": "Société anonyme coopérative de transport à directoire",
    "5658": "Société anonyme coopérative de production à directoire (SCOP)",
    "5659": "Union de sociétés coopératives à forme anonyme et à conseil d'administration",
    "5660": "Autre société anonyme coopérative à directoire",
    "5670": "Société de Participations Financières de Profession Libérale Société anonyme à Directoire (SPFPL)",
    "5685": "Société d'exercice libéral à forme anonyme à directoire",
    "5699": "Société anonyme à directoire (sans autre indication)",
    "5710": "SAS, société par actions simplifiée",
    "5720": "Société par actions simplifiées associé unique ou société par actions simplifiées unipersonnelle",
    "5770": "Société de Participations Financières de Profession Libérale Société par actions simplifiée (SPFPL SAS)",
    "5785": "SELAS, Société d'exercice libéral par action simplifiée",
    "5800": "Société européenne",
    "6100": "Caisse d'épargne et de prévoyance",
    "6210": "GEIE, Groupement européen d'intérêt économique",
    "6220": "GIE, Groupement d'intérêt économique",
    "6316": "CUMA, Coopérative d'utilisation de matériel agricole en commun",
    "6317": "Société coopérative agricole",
    "6318": "Union de sociétés coopératives agricoles",
    "6411": "Société d'assurances mutuelles",
    "6511": "Sociétés Interprofessionnelles de Soins Ambulatoires",
    "6521": "Société civile de placement immobilier",
    "6532": "Société civile d'intérêt collectif agricole (SICA)",
    "6533": "GAEC, Groupement agricole d'exploitation en commun",
    "6534": "GFA, Groupement foncier agricole",
    "6535": "GAF, Groupement agricole foncier",
    "6536": "GF, Groupement forestier",
    "6537": "GP, Groupement pastoral",
    "6538": "GFR, Groupement foncier et rural",
    "6539": "Société civile foncière",
    "6540": "Société civile immobilière (SCI)",
    "6541": "Société civile immobilière de construction-vente",
    "6542": "Société civile d'attribution",
    "6543": "Société civile coopérative de construction",
    "6544": "Société civile immobilière d' accession progressive à la propriété",
    "6551": "Société civile coopérative de consommation",
    "6554": "Société civile coopérative d'intérêt maritime",
    "6558": "Société civile coopérative entre médecins",
    "6560": "Autre société civile coopérative",
    "6561": "SCP d'avocats",
    "6562": "SCP d'avocats aux conseils",
    "6563": "SCP d'avoués d'appel",
    "6564": "SCP d'huissiers",
    "6565": "SCP de notaires",
    "6566": "SCP de commissaire-priseur judiciaire",
    "6567": "SCP de greffiers de tribunal de commerce",
    "6568": "SCP de conseils juridiques",
    "6569": "SCP de commissaires aux comptes",
    "6571": "SCP de médecins",
    "6572": "SCP de dentistes",
    "6573": "SCP d'infirmiers",
    "6574": "SCP de masseurs-kinésithérapeutes",
    "6575": "SCP de directeurs de laboratoire d'analyse médicale",
    "6576": "SCP de vétérinaires",
    "6577": "SCP de géomètres experts",
    "6578": "SCP d'architectes",
    "6585": "Autre société civile professionnelle",
    "6588": "Société civile laitière",
    "6589": "Société civile de moyens",
    "6595": "Caisse locale de crédit mutuel",
    "6596": "Caisse de crédit agricole mutuel",
    "6597": "SCEA, Société civile d'exploitation agricole",
    "6598": "EARL, Exploitation agricole à responsabilité limitée pluripersonnelle",
    "6599": "Autre société civile",
    "6901": "Autre personne de droit privé inscrite au registre du commerce et des sociétés",
    "7111": "Autorité constitutionnelle",
    "7112": "Autorité administrative ou publique indépendante",
    "7113": "Ministère",
    "7120": "Service central d'un ministère",
    "7150": "Service du ministère de la Défense",
    "7160": "Service déconcentré à compétence nationale d'un ministère (hors Défense)",
    "7171": "Service déconcentré de l'État à compétence (inter) régionale",
    "7172": "Service déconcentré de l'État à compétence (inter) départementale",
    "7179": "(Autre) Service déconcentré de l'État à compétence territoriale",
    "7190": "Ecole nationale non dotée de la personnalité morale",
    "7210": "Commune et commune nouvelle",
    "7220": "Département",
    "7225": "Collectivité et territoire d'Outre Mer",
    "7229": "(Autre) Collectivité territoriale",
    "7230": "Région",
    "7312": "Commune associée et commune déléguée",
    "7313": "Section de commune",
    "7314": "Ensemble urbain",
    "7321": "Association syndicale autorisée",
    "7322": "Association foncière urbaine",
    "7323": "Association foncière de remembrement",
    "7331": "Établissement public local d'enseignement",
    "7340": "Pôle métropolitain",
    "7341": "Secteur de commune",
    "7342": "District urbain",
    "7343": "Communauté urbaine",
    "7344": "Métropole",
    "7345": "Syndicat intercommunal à vocation multiple (SIVOM)",
    "7346": "Communauté de communes",
    "7347": "Communauté de villes",
    "7348": "Communauté d'agglomération",
    "7349": "Autre établissement public local de coopération non spécialisé ou entente",
    "7351": "Institution interdépartementale ou entente",
    "7352": "Institution interrégionale ou entente",
    "7353": "Syndicat intercommunal à vocation unique (SIVU)",
    "7354": "Syndicat mixte fermé",
    "7355": "Syndicat mixte ouvert",
    "7356": "Commission syndicale pour la gestion des biens indivis des communes",
    "7357": "Pôle d'équilibre territorial et rural (PETR)",
    "7361": "Centre communal d'action sociale",
    "7362": "Caisse des écoles",
    "7363": "Caisse de crédit municipal",
    "7364": "Établissement d'hospitalisation",
    "7365": "Syndicat inter hospitalier",
    "7366": "Établissement public local social et médico-social",
    "7367": "Centre Intercommunal d'action sociale (CIAS)",
    "7371": "Office public d'habitation à loyer modéré (OPHLM)",
    "7372": "Service départemental d'incendie et de secours (SDIS)",
    "7373": "Établissement public local culturel",
    "7378": "Régie d'une collectivité locale à caractère administratif",
    "7379": "(Autre) Établissement public administratif local",
    "7381": "Organisme consulaire",
    "7382": "Établissement public national ayant fonction d'administration centrale",
    "7383": "Établissement public national à caractère scientifique culturel et professionnel",
    "7384": "Autre établissement public national d'enseignement",
    "7385": "Autre établissement public national administratif à compétence territoriale limitée",
    "7389": "Établissement public national à caractère administratif",
    "7410": "Groupement d'intérêt public (GIP)",
    "7430": "Établissement public des cultes d'Alsace-Lorraine",
    "7450": "Etablissement public administratif, cercle et foyer dans les armées",
    "7470": "Groupement de coopération sanitaire à gestion publique",
    "7490": "Autre personne morale de droit administratif",
    "8110": "Régime général de la sécurité sociale",
    "8120": "Régime spécial de sécurité sociale",
    "8130": "Institution de retraite complémentaire",
    "8140": "Mutualité sociale agricole",
    "8150": "Régime maladie des non-salariés non agricoles",
    "8160": "Régime vieillesse ne dépendant pas du régime général de la sécurité sociale",
    "8170": "Régime d'assurance chômage",
    "8190": "Autre régime de prévoyance sociale",
    "8210": "Mutuelle",
    "8250": "Assurance mutuelle agricole inscrite au RCS",
    "8290": "Autre organisme mutualiste",
    "8310": "Comité central d'entreprise",
    "8311": "Comité d'établissement",
    "8410": "Syndicat de salariés",
    "8420": "Syndicat patronal",
    "8450": "Ordre professionnel ou assimilé",
    "8470": "Centre technique industriel ou comité professionnel du développement économique",
    "8490": "Autre organisme professionnel",
    "9110": "Syndicat de copropriété",
    "9150": "Association syndicale libre",
    "9210": "Association non déclarée",
    "9220": "Association déclarée",
    "9221": "Association déclarée \"entreprises d'insertion par l'économique\"",
    "9222": "Association intermédiaire",
    "9223": "Groupement d'employeurs",
    "9224": "AARPI",
    "9230": "Association déclarée reconnue d'utilité publique",
    "9240": "Congrégation",
    "9260": "Association de droit local",
    "9300": "Fondation",
    "9900": "Autre personne morale de droit privé",
    "9970": "Groupement de coopération sanitaire à gestion privée",
}
# ── Formes juridiques : nomenclature INSEE complète (hybride téléchargement + secours) ──
def _charger_formes_juridiques():
    """Tente de télécharger la nomenclature INSEE des catégories juridiques (254 codes).
    Retombe sur la table de secours intégrée si le réseau échoue. Purement descriptif."""
    urls = [
        # export CSV Opendatasoft (code, libellé) — plusieurs miroirs
        "https://data.iledefrance.fr/explore/dataset/categories-juridiques-insee/download/?format=csv&use_labels_for_header=false&csv_separator=%3B",
        "https://public.opendatasoft.com/explore/dataset/categories-juridiques-insee/download/?format=csv&use_labels_for_header=false&csv_separator=%3B",
        "https://ODS.backoffice.smartidf.services/explore/dataset/categories-juridiques-insee/download/?format=csv&use_labels_for_header=false&csv_separator=%3B",
    ]
    import csv, io as _io
    for url in urls:
        try:
            r = requests.get(url, timeout=15)
            r.raise_for_status()
            txt = r.content.decode("utf-8", errors="replace")
            sep = ";" if txt.count(";") >= txt.count(",") else ","
            table = {}
            for row in csv.reader(_io.StringIO(txt), delimiter=sep):
                if len(row) < 2:
                    continue
                code = re.sub(r"\D", "", str(row[0]))
                lib = str(row[1]).strip()
                if re.fullmatch(r"\d{4}", code) and lib and not lib[0].isdigit():
                    table[code] = lib
            if len(table) >= 200:   # nomenclature complète chargée
                print(f"  Formes juridiques : {len(table)} codes chargés depuis l'INSEE/data.gouv")
                return table
        except Exception:
            continue
    print("  Formes juridiques : téléchargement indisponible -> table de secours intégrée")
    return dict(_FORMES_JURIDIQUES_SECOURS)

_FORMES_JURIDIQUES = _charger_formes_juridiques()



# ── Niveaux I/II par code INSEE, source officielle INPI 2026 (prioritaire) ──
_NIVEAUX_INPI = {
    "0000": ("Exploitation en commun", "Organisme de placement collectif en valeurs mobilières sans personnalité morale"),
    "1000": ("Personne physique", "Entrepreneur individuel"),
    "2110": ("Exploitation en commun", "Indivision entre personnes physiques"),
    "2120": ("Exploitation en commun", "Indivision avec personne morale"),
    "2210": ("Exploitation en commun", "Société créée de fait entre personnes physiques"),
    "2220": ("Exploitation en commun", "Société créée de fait avec personne morale"),
    "2310": ("Exploitation en commun", "Société en participation entre personnes physiques"),
    "2320": ("Exploitation en commun", "Société en participation avec personne morale"),
    "2385": ("Exploitation en commun", "Société en participation de professions libérales"),
    "2900": ("Autres structures", "Autre groupement de droit privé non doté de la personnalité morale"),
    "3110": ("Formes juridiques étrangères", "Représentation ou agence commerciale d'état ou organisme public étranger immatriculé au RCS"),
    "3120": ("Formes juridiques étrangères", "Société commerciale étrangère immatriculée au RCS"),
    "3210": ("Formes juridiques étrangères", "Formes juridiques étrangères"),
    "3220": ("Formes juridiques étrangères", "Société étrangère non immatriculée au RCS"),
    "3290": ("Formes juridiques étrangères", "Formes juridiques étrangères"),
    "4110": ("Groupement ou EPIC", "Établissements publics industriels et commerciaux"),
    "4120": ("Groupement ou EPIC", "Établissements publics industriels et commerciaux"),
    "4130": ("Etablissement ou organisme public, administration", "Etablissement public ou régie à caractère industriel ou commercial"),
    "4140": ("Groupement ou EPIC", "Établissements publics industriels et commerciaux"),
    "4150": ("Groupement ou EPIC", "Établissements publics industriels et commerciaux"),
    "4160": ("Etablissement ou organisme public, administration", "Etablissement public ou régie à caractère industriel ou commercial"),
    "5191": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "5192": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "5193": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "5194": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "5196": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "5202": ("Société commerciale pluripersonnelle", "SNC, Société en nom collectif"),
    "5203": ("Société à forme coopérative", "Société en nom collectif coopérative"),
    "5306": ("Société commerciale pluripersonnelle", "Société en commandite"),
    "5307": ("Société à forme coopérative", "Société en commandite coopérative"),
    "5308": ("Société commerciale pluripersonnelle", "Société en commandite"),
    "5309": ("Société à forme coopérative", "Société en commandite coopérative"),
    "5310": ("Société commerciale pluripersonnelle", "Société en commandite"),
    "5370": ("Société commerciale pluripersonnelle", "Société en commandite"),
    "5385": ("Société commerciale pluripersonnelle", "Société en commandite"),
    "5410": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5415": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5422": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5426": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5430": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5431": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5432": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5442": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5443": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5451": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5453": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5454": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5455": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5458": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5459": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5460": ("Société à forme coopérative", "Société coopérative à responsabilité limitée"),
    "5470": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5485": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5499": ("Société commerciale pluripersonnelle", "Société à responsabilité limitée"),
    "5505": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5510": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5515": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5520": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5522": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5525": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5530": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5531": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5532": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5542": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5543": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5546": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5547": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5551": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5552": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5553": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5554": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5555": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5558": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5559": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5560": ("Société à forme coopérative", "SA coopérative à conseil d'administration"),
    "5570": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5585": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5599": ("Société commerciale pluripersonnelle", "Société anonyme à conseil d'administration"),
    "5605": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5610": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5615": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5620": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5622": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5625": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5630": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5631": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5632": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5642": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5643": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5646": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5647": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5651": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5652": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5653": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5654": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5655": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5658": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5659": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5660": ("Société à forme coopérative", "SA coopérative à directoire"),
    "5670": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5685": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5699": ("Société commerciale pluripersonnelle", "Société anonyme à directoire"),
    "5710": ("Société commerciale pluripersonnelle", "Société par actions simplifiée"),
    "5770": ("Société commerciale pluripersonnelle", "Société par actions simplifiée"),
    "5785": ("Société commerciale pluripersonnelle", "Société par actions simplifiée"),
    "5800": ("Formes juridiques étrangères", "Société européenne"),
    "6210": ("Groupement ou EPIC", "GEIE, Groupement européen d'intérêt économique"),
    "6220": ("Groupement ou EPIC", "GIE, Groupement d'intérêt économique"),
    "6316": ("Société à forme coopérative", "Société coopérative agricole"),
    "6317": ("Société à forme coopérative", "Société coopérative agricole"),
    "6318": ("Société à forme coopérative", "Société coopérative agricole"),
    "6411": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "6511": ("Société civile", "Autre société civile"),
    "6521": ("Société civile", "Société immobilière ou foncière"),
    "6532": ("Société à forme coopérative", "société civile coopérative"),
    "6533": ("Société ou groupement agricole", "GAEC, Groupement agricole d'exploitation en commun"),
    "6534": ("Société ou groupement agricole", "GFA, Groupement foncier agricole"),
    "6535": ("Société ou groupement agricole", "GAF, Groupement agricole foncier"),
    "6536": ("Société ou groupement agricole", "GF, Groupement forestier"),
    "6537": ("Société ou groupement agricole", "GP, Groupement pastoral"),
    "6538": ("Société ou groupement agricole", "GFR, Groupement foncier et rural"),
    "6539": ("Société civile", "Société immobilière ou foncière"),
    "6540": ("Société civile", "Société immobilière ou foncière"),
    "6541": ("Société civile", "Société immobilière ou foncière"),
    "6542": ("Société civile", "Société immobilière ou foncière"),
    "6543": ("Société à forme coopérative", "société civile coopérative"),
    "6544": ("Société civile", "Société immobilière ou foncière"),
    "6551": ("Société à forme coopérative", "société civile coopérative"),
    "6554": ("Société à forme coopérative", "société civile coopérative"),
    "6558": ("Société à forme coopérative", "société civile coopérative"),
    "6560": ("Société à forme coopérative", "société civile coopérative"),
    "6561": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6562": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6563": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6564": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6565": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6566": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6567": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6568": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6569": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6571": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6572": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6573": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6574": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6575": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6576": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6577": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6578": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6585": ("Société civile", "société civile professionnelle d'exercice libéral"),
    "6589": ("Société civile", "Autre société civile"),
    "6595": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "6596": ("Société à forme coopérative", "Forme coopérative spécifique"),
    "6597": ("Société ou groupement agricole", "SCEA, Société civile d'exploitation agricole"),
    "6598": ("Société ou groupement agricole", "EARL, Exploitation agricole à responsabilité limitée pluripersonnelle"),
    "6599": ("Société civile", "Autre société civile"),
    "6901": ("Société civile", "Autre société civile"),
    "7111": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7112": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7113": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7120": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7150": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7160": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7171": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7172": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7179": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7190": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7210": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7220": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7225": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7229": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7230": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7312": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7313": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7314": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7321": ("Association & syndicat", "Syndicat"),
    "7322": ("Association & syndicat", "Association loi 1901 ou assimilé"),
    "7323": ("Association & syndicat", "Association loi 1901 ou assimilé"),
    "7331": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7340": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7341": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7342": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7343": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7344": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7345": ("Syndicat", "Syndicat"),
    "7346": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7347": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7348": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7349": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7351": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7352": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7353": ("Association & syndicat", "Syndicat"),
    "7354": ("Association & syndicat", "Syndicat"),
    "7355": ("Association & syndicat", "Syndicat"),
    "7356": ("Association & syndicat", "Etablissement public administratif"),
    "7357": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7361": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7362": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7363": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7364": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7365": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7366": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7367": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7371": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7372": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7373": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7378": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7379": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7381": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7382": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7383": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7384": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7385": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7389": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7410": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7430": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7450": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7470": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7490": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "8210": ("Autres structures", "Organisme mutualiste"),
    "8250": ("Autre personne morale", "Organisme mutualiste"),
    "9224": ("Exploitation en commun", "AARPI"),
}

# ── Niveaux I et II des catégories juridiques (regroupements) ──
# Table utilisateur (regroupements 'métier') ; repli sur les niveaux officiels INSEE.
_NIVEAUX_UTILISATEUR = {
    "0000": ("Exploitation en commun", "Organisme de placement collectif en valeurs mobilières sans personnalité morale"),
    "2900": ("Autres structures", "Autre groupement de droit privé non doté de la personnalité morale"),
    "3210": ("Formes juridiques étrangères", "Formes juridiques étrangères"),
    "3290": ("Formes juridiques étrangères", "Formes juridiques étrangères"),
    "4130": ("Etablissement ou organisme public, administration", "Etablissement public ou régie à caractère industriel ou commercial"),
    "4160": ("Etablissement ou organisme public, administration", "Etablissement public ou régie à caractère industriel ou commercial"),
    "7111": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7112": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7113": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7120": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7150": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7160": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7171": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7172": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7179": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7190": ("Etablissement ou organisme public, administration", "Administration de l'état"),
    "7210": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7220": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7225": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7229": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7230": ("Etablissement ou organisme public, administration", "Collectivité territoriale"),
    "7312": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7313": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7314": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7321": ("Association & syndicat", "Syndicat"),
    "7322": ("Association & syndicat", "Association loi 1901 ou assimilé"),
    "7323": ("Association & syndicat", "Association loi 1901 ou assimilé"),
    "7331": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7340": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7341": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7342": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7343": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7344": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7345": ("Syndicat", "Syndicat"),
    "7346": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7347": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7348": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7349": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7351": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7352": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7353": ("Association & syndicat", "Syndicat"),
    "7354": ("Association & syndicat", "Syndicat"),
    "7355": ("Association & syndicat", "Syndicat"),
    "7356": ("Association & syndicat", "Etablissement public administratif"),
    "7357": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7361": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7362": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7363": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7364": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7365": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7366": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7367": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7371": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7372": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7373": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7378": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7379": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7381": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7382": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7383": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7384": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7385": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7389": ("Etablissement ou organisme public, administration", "Etablissement public administratif"),
    "7410": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7430": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7450": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7470": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "7490": ("Etablissement ou organisme public, administration", "Autre personne morale de droit public administratif"),
    "8210": ("Autres structures", "Organisme mutualiste"),
}

_NIVEAU1_INSEE = {
    "1": "Personne physique",
    "2": "Groupement de droit privé non doté de la personnalité morale",
    "3": "Personne morale de droit étranger",
    "4": "Personne morale de droit public soumise au droit commercial",
    "5": "Société commerciale",
    "6": "Autre personne morale immatriculée au RCS",
    "7": "Personne morale et organisme soumis au droit administratif",
    "8": "Organisme privé spécialisé",
    "9": "Groupement de droit privé",
}

_NIVEAU2_INSEE = {
    "11": "Artisan-commerçant",
    "12": "Commerçant",
    "13": "Artisan",
    "14": "Officier public ou ministériel",
    "15": "Profession libérale",
    "16": "Exploitant agricole",
    "17": "Agent commercial",
    "18": "Associé Gérant de société",
    "19": "(Autre) personne physique",
    "21": "Indivision",
    "22": "Société créée de fait",
    "23": "Société en participation",
    "24": "Fiducie",
    "27": "Paroisse hors zone concordataire",
    "29": "Autre groupement de droit privé non doté de la personnalité morale",
    "31": "Personne morale de droit étranger immatriculée au RCS",
    "32": "Personne morale de droit étranger non immatriculée au RCS",
    "41": "Établissement public ou régie à caractère industriel ou commercial",
    "51": "Société coopérative commerciale particulière",
    "52": "Société en nom collectif",
    "53": "Société en commandite",
    "54": "Société à responsabilité limitée (SARL)",
    "55": "Société anonyme à conseil d'administration",
    "56": "Société anonyme à directoire",
    "57": "Société anonyme par actions simplifiées",
    "58": "Société européenne",
    "61": "Caisse d'épargne et de prévoyance",
    "62": "Groupement d'intérêt économique",
    "63": "Société coopérative agricole",
    "64": "Société non commerciale d'assurances",
    "65": "Société civile",
    "69": "Autres personnes de droit privé inscrites au registre du commerce et des sociétés",
    "71": "Administration de l'état",
    "72": "Collectivité territoriale",
    "73": "Établissement public administratif",
    "74": "Autre personne morale de droit public administratif",
    "81": "Organisme gérant un régime de protection sociale à adhésion obligatoire",
    "82": "Organisme mutualiste",
    "83": "Comité d'entreprise",
    "84": "Organisme professionnel",
    "91": "Syndicat de propriétaires",
    "92": "Association loi 1901 ou assimilé",
    "93": "Fondation",
    "99": "Autre personne morale de droit privé",
}

# ── Catégories juridiques INSEE (cj_septembre_2022) : 3 niveaux officiels ──
_CJ_NIV1 = {
    "0": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "1": "Entrepreneur individuel",
    "2": "Groupement de droit privé non doté de la personnalité morale",
    "3": "Personne morale de droit étranger",
    "4": "Personne morale de droit public soumise au droit commercial",
    "5": "Société commerciale",
    "6": "Autre personne morale immatriculée au RCS",
    "7": "Personne morale et organisme soumis au droit administratif",
    "8": "Organisme privé spécialisé",
    "9": "Groupement de droit privé",
}

_CJ_NIV2 = {
    "00": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "10": "Entrepreneur individuel",
    "21": "Indivision",
    "22": "Société créée de fait",
    "23": "Société en participation",
    "24": "Fiducie",
    "27": "Paroisse hors zone concordataire",
    "28": "Assujetti unique à la TVA",
    "29": "Autre groupement de droit privé non doté de la personnalité morale",
    "31": "Personne morale de droit étranger, immatriculée au RCS (registre du commerce et des sociétés)",
    "32": "Personne morale de droit étranger, non immatriculée au RCS",
    "41": "Etablissement public ou régie à caractère industriel ou commercial",
    "51": "Société coopérative commerciale particulière",
    "52": "Société en nom collectif",
    "53": "Société en commandite",
    "54": "Société à responsabilité limitée (SARL)",
    "55": "Société anonyme à conseil d'administration",
    "56": "Société anonyme à directoire",
    "57": "Société par actions simplifiée",
    "58": "Société européenne",
    "61": "Caisse d'épargne et de prévoyance",
    "62": "Groupement d'intérêt économique",
    "63": "Société coopérative agricole",
    "64": "Société d'assurance mutuelle",
    "65": "Société civile",
    "69": "Autre personne morale de droit privé inscrite au registre du commerce et des sociétés",
    "71": "Administration de l'état",
    "72": "Collectivité territoriale",
    "73": "Etablissement public administratif",
    "74": "Autre personne morale de droit public administratif",
    "81": "Organisme gérant un régime de protection sociale à adhésion obligatoire",
    "82": "Organisme mutualiste",
    "83": "Comité d'entreprise",
    "84": "Organisme professionnel",
    "85": "Organisme de retraite à adhésion non obligatoire",
    "91": "Syndicat de propriétaires",
    "92": "Association loi 1901 ou assimilé",
    "93": "Fondation",
    "99": "Autre personne morale de droit privé",
}

_CJ_NIV3 = {
    "0000": "Organisme de placement collectif en valeurs mobilières sans personnalité morale",
    "1000": "Entrepreneur individuel",
    "2110": "Indivision entre personnes physiques",
    "2120": "Indivision avec personne morale",
    "2210": "Société créée de fait entre personnes physiques",
    "2220": "Société créée de fait avec personne morale",
    "2310": "Société en participation entre personnes physiques",
    "2320": "Société en participation avec personne morale",
    "2385": "Société en participation de professions libérales",
    "2400": "Fiducie",
    "2700": "Paroisse hors zone concordataire",
    "2800": "Assujetti unique à la TVA",
    "2900": "Autre groupement de droit privé non doté de la personnalité morale",
    "3110": "Représentation ou agence commerciale d'état ou organisme public étranger immatriculé au RCS",
    "3120": "Société commerciale étrangère immatriculée au RCS",
    "3205": "Organisation internationale",
    "3210": "État, collectivité ou établissement public étranger",
    "3220": "Société étrangère non immatriculée au RCS",
    "3290": "Autre personne morale de droit étranger",
    "4110": "Établissement public national à caractère industriel ou commercial doté d'un comptable public",
    "4120": "Établissement public national à caractère industriel ou commercial non doté d'un comptable public",
    "4130": "Exploitant public",
    "4140": "Établissement public local à caractère industriel ou commercial",
    "4150": "Régie d'une collectivité locale à caractère industriel ou commercial",
    "4160": "Institution Banque de France",
    "5191": "Société de caution mutuelle",
    "5192": "Société coopérative de banque populaire",
    "5193": "Caisse de crédit maritime mutuel",
    "5194": "Caisse (fédérale) de crédit mutuel",
    "5195": "Association coopérative inscrite (droit local Alsace Moselle)",
    "5196": "Caisse d'épargne et de prévoyance à forme coopérative",
    "5202": "Société en nom collectif",
    "5203": "Société en nom collectif coopérative",
    "5306": "Société en commandite simple",
    "5307": "Société en commandite simple coopérative",
    "5308": "Société en commandite par actions",
    "5309": "Société en commandite par actions coopérative",
    "5310": "Société en libre partenariat (SLP)",
    "5370": "Société de Participations Financières de Profession Libérale Société en commandite par actions (SPFPL SCA)",
    "5385": "Société d'exercice libéral en commandite par actions",
    "5410": "SARL nationale",
    "5415": "SARL d'économie mixte",
    "5422": "SARL immobilière pour le commerce et l'industrie (SICOMI)",
    "5426": "SARL immobilière de gestion",
    "5430": "SARL d'aménagement foncier et d'équipement rural (SAFER)",
    "5431": "SARL mixte d'intérêt agricole (SMIA)",
    "5432": "SARL d'intérêt collectif agricole (SICA)",
    "5442": "SARL d'attribution",
    "5443": "SARL coopérative de construction",
    "5451": "SARL coopérative de consommation",
    "5453": "SARL coopérative artisanale",
    "5454": "SARL coopérative d'intérêt maritime",
    "5455": "SARL coopérative de transport",
    "5458": "SARL coopérative de production (SCOP)",
    "5459": "SARL union de sociétés coopératives",
    "5460": "Autre SARL coopérative",
    "5470": "Société de Participations Financières de Profession Libérale Société à responsabilité limitée (SPFPL SARL)",
    "5485": "Société d'exercice libéral à responsabilité limitée",
    "5499": "Société à responsabilité limitée (sans autre indication)",
    "5505": "SA à participation ouvrière à conseil d'administration",
    "5510": "SA nationale à conseil d'administration",
    "5515": "SA d'économie mixte à conseil d'administration",
    "5520": "Fonds à forme sociétale à conseil d'administration",
    "5522": "SA immobilière pour le commerce et l'industrie (SICOMI) à conseil d'administration",
    "5525": "SA immobilière d'investissement à conseil d'administration",
    "5530": "SA d'aménagement foncier et d'équipement rural (SAFER) à conseil d'administration",
    "5531": "Société anonyme mixte d'intérêt agricole (SMIA) à conseil d'administration",
    "5532": "SA d'intérêt collectif agricole (SICA) à conseil d'administration",
    "5542": "SA d'attribution à conseil d'administration",
    "5543": "SA coopérative de construction à conseil d'administration",
    "5546": "SA de HLM à conseil d'administration",
    "5547": "SA coopérative de production de HLM à conseil d'administration",
    "5548": "SA de crédit immobilier à conseil d'administration",
    "5551": "SA coopérative de consommation à conseil d'administration",
    "5552": "SA coopérative de commerçants-détaillants à conseil d'administration",
    "5553": "SA coopérative artisanale à conseil d'administration",
    "5554": "SA coopérative (d'intérêt) maritime à conseil d'administration",
    "5555": "SA coopérative de transport à conseil d'administration",
    "5558": "SA coopérative de production  (SCOP) à conseil d'administration",
    "5559": "SA union de sociétés coopératives à conseil d'administration",
    "5560": "Autre SA coopérative à conseil d'administration",
    "5570": "Société de Participations Financières de Profession Libérale Société anonyme à conseil d'administration (SPFPL SA à conseil d'administration)",
    "5585": "Société d'exercice libéral à forme anonyme à conseil d'administration",
    "5599": "SA à conseil d'administration (s.a.i.)",
    "5605": "SA à participation ouvrière à directoire",
    "5610": "SA nationale à directoire",
    "5615": "SA d'économie mixte à directoire",
    "5620": "Fonds à forme sociétale à directoire",
    "5622": "SA immobilière pour le commerce et l'industrie (SICOMI) à directoire",
    "5625": "SA immobilière d'investissement à directoire",
    "5630": "Safer anonyme à directoire",
    "5631": "SA mixte d'intérêt agricole (SMIA)",
    "5632": "SA d'intérêt collectif agricole (SICA)",
    "5642": "SA d'attribution à directoire",
    "5643": "SA coopérative de construction à directoire",
    "5646": "SA de HLM à directoire",
    "5647": "Société coopérative de production de HLM anonyme à directoire",
    "5648": "SA de crédit immobilier à directoire",
    "5651": "SA coopérative de consommation à directoire",
    "5652": "SA coopérative de commerçants-détaillants à directoire",
    "5653": "SA coopérative artisanale à directoire",
    "5654": "SA coopérative d'intérêt maritime à directoire",
    "5655": "SA coopérative de transport à directoire",
    "5658": "SA coopérative de production (SCOP) à directoire",
    "5659": "SA union de sociétés coopératives à directoire",
    "5660": "Autre SA coopérative à directoire",
    "5670": "Société de Participations Financières de Profession Libérale Société anonyme à Directoire (SPFPL SA à directoire)",
    "5685": "Société d'exercice libéral à forme anonyme à directoire",
    "5699": "SA à directoire (s.a.i.)",
    "5710": "SAS, société par actions simplifiée",
    "5770": "Société de Participations Financières de Profession Libérale Société par actions simplifiée (SPFPL SAS)",
    "5785": "Société d'exercice libéral par action simplifiée",
    "5800": "Société européenne",
    "6100": "Caisse d'Épargne et de Prévoyance",
    "6210": "Groupement européen d'intérêt économique (GEIE)",
    "6220": "Groupement d'intérêt économique (GIE)",
    "6316": "Coopérative d'utilisation de matériel agricole en commun (CUMA)",
    "6317": "Société coopérative agricole",
    "6318": "Union de sociétés coopératives agricoles",
    "6411": "Société d'assurance à forme mutuelle",
    "6511": "Sociétés Interprofessionnelles de Soins Ambulatoires",
    "6521": "Société civile de placement collectif immobilier (SCPI)",
    "6532": "Société civile d'intérêt collectif agricole (SICA)",
    "6533": "Groupement agricole d'exploitation en commun (GAEC)",
    "6534": "Groupement foncier agricole",
    "6535": "Groupement agricole foncier",
    "6536": "Groupement forestier",
    "6537": "Groupement pastoral",
    "6538": "Groupement foncier et rural",
    "6539": "Société civile foncière",
    "6540": "Société civile immobilière",
    "6541": "Société civile immobilière de construction-vente",
    "6542": "Société civile d'attribution",
    "6543": "Société civile coopérative de construction",
    "6544": "Société civile immobilière d' accession progressive à la propriété",
    "6551": "Société civile coopérative de consommation",
    "6554": "Société civile coopérative d'intérêt maritime",
    "6558": "Société civile coopérative entre médecins",
    "6560": "Autre société civile coopérative",
    "6561": "SCP d'avocats",
    "6562": "SCP d'avocats aux conseils",
    "6563": "SCP d'avoués d'appel",
    "6564": "SCP d'huissiers",
    "6565": "SCP de notaires",
    "6566": "SCP de commissaires-priseurs",
    "6567": "SCP de greffiers de tribunal de commerce",
    "6568": "SCP de conseils juridiques",
    "6569": "SCP de commissaires aux comptes",
    "6571": "SCP de médecins",
    "6572": "SCP de dentistes",
    "6573": "SCP d'infirmiers",
    "6574": "SCP de masseurs-kinésithérapeutes",
    "6575": "SCP de directeurs de laboratoire d'analyse médicale",
    "6576": "SCP de vétérinaires",
    "6577": "SCP de géomètres experts",
    "6578": "SCP d'architectes",
    "6585": "Autre société civile professionnelle",
    "6589": "Société civile de moyens",
    "6595": "Caisse locale de crédit mutuel",
    "6596": "Caisse de crédit agricole mutuel",
    "6597": "Société civile d'exploitation agricole",
    "6598": "Exploitation agricole à responsabilité limitée",
    "6599": "Autre société civile",
    "6901": "Autre personne de droit privé inscrite au registre du commerce et des sociétés",
    "7111": "Autorité constitutionnelle",
    "7112": "Autorité administrative ou publique indépendante",
    "7113": "Ministère",
    "7120": "Service central d'un ministère",
    "7150": "Service du ministère de la Défense",
    "7160": "Service déconcentré à compétence nationale d'un ministère (hors Défense)",
    "7171": "Service déconcentré de l'État à compétence (inter) régionale",
    "7172": "Service déconcentré de l'État à compétence (inter) départementale",
    "7179": "(Autre) Service déconcentré de l'État à compétence territoriale",
    "7190": "Ecole nationale non dotée de la personnalité morale",
    "7210": "Commune et commune nouvelle",
    "7220": "Département",
    "7225": "Collectivité et territoire d'Outre Mer",
    "7229": "(Autre) Collectivité territoriale",
    "7230": "Région",
    "7312": "Commune associée et commune déléguée",
    "7313": "Section de commune",
    "7314": "Ensemble urbain",
    "7321": "Association syndicale autorisée",
    "7322": "Association foncière urbaine",
    "7323": "Association foncière de remembrement",
    "7331": "Établissement public local d'enseignement",
    "7340": "Pôle métropolitain",
    "7341": "Secteur de commune",
    "7342": "District urbain",
    "7343": "Communauté urbaine",
    "7344": "Métropole",
    "7345": "Syndicat intercommunal à vocation multiple (SIVOM)",
    "7346": "Communauté de communes",
    "7347": "Communauté de villes",
    "7348": "Communauté d'agglomération",
    "7349": "Autre établissement public local de coopération non spécialisé ou entente",
    "7351": "Institution interdépartementale ou entente",
    "7352": "Institution interrégionale ou entente",
    "7353": "Syndicat intercommunal à vocation unique (SIVU)",
    "7354": "Syndicat mixte fermé",
    "7355": "Syndicat mixte ouvert",
    "7356": "Commission syndicale pour la gestion des biens indivis des communes",
    "7357": "Pôle d'équilibre territorial et rural (PETR)",
    "7361": "Centre communal d'action sociale",
    "7362": "Caisse des écoles",
    "7363": "Caisse de crédit municipal",
    "7364": "Établissement d'hospitalisation",
    "7365": "Syndicat inter hospitalier",
    "7366": "Établissement public local social et médico-social",
    "7367": "Centre Intercommunal d'action sociale (CIAS)",
    "7371": "Office public d'habitation à loyer modéré (OPHLM)",
    "7372": "Service départemental d'incendie et de secours (SDIS)",
    "7373": "Établissement public local culturel",
    "7378": "Régie d'une collectivité locale à caractère administratif",
    "7379": "(Autre) Établissement public administratif local",
    "7381": "Organisme consulaire",
    "7382": "Établissement public national ayant fonction d'administration centrale",
    "7383": "Établissement public national à caractère scientifique culturel et professionnel",
    "7384": "Autre établissement public national d'enseignement",
    "7385": "Autre établissement public national administratif à compétence territoriale limitée",
    "7389": "Établissement public national à caractère administratif",
    "7410": "Groupement d'intérêt public (GIP)",
    "7430": "Établissement public des cultes d'Alsace-Lorraine",
    "7450": "Etablissement public administratif, cercle et foyer dans les armées",
    "7470": "Groupement de coopération sanitaire à gestion publique",
    "7490": "Autre personne morale de droit administratif",
    "8110": "Régime général de la Sécurité Sociale",
    "8120": "Régime spécial de Sécurité Sociale",
    "8130": "Institution de retraite complémentaire",
    "8140": "Mutualité sociale agricole",
    "8150": "Régime maladie des non-salariés non agricoles",
    "8160": "Régime vieillesse ne dépendant pas du régime général de la Sécurité Sociale",
    "8170": "Régime d'assurance chômage",
    "8190": "Autre régime de prévoyance sociale",
    "8210": "Mutuelle",
    "8250": "Assurance mutuelle agricole",
    "8290": "Autre organisme mutualiste",
    "8310": "Comité social économique d’entreprise",
    "8311": "Comité social économique d'établissement",
    "8410": "Syndicat de salariés",
    "8420": "Syndicat patronal",
    "8450": "Ordre professionnel ou assimilé",
    "8470": "Centre technique industriel ou comité professionnel du développement économique",
    "8490": "Autre organisme professionnel",
    "8510": "Institution de prévoyance",
    "8520": "Institution de retraite supplémentaire",
    "9110": "Syndicat de copropriété",
    "9150": "Association syndicale libre",
    "9210": "Association non déclarée",
    "9220": "Association déclarée",
    "9221": "Association déclarée d'insertion par l'économique",
    "9222": "Association intermédiaire",
    "9223": "Groupement d'employeurs",
    "9224": "Association d'avocats à responsabilité professionnelle individuelle",
    "9230": "Association déclarée, reconnue d'utilité publique",
    "9240": "Congrégation",
    "9260": "Association de droit local (Bas-Rhin, Haut-Rhin et Moselle)",
    "9300": "Fondation",
    "9900": "Autre personne morale de droit privé",
    "9970": "Groupement de coopération sanitaire à gestion privée",
}


def _niveaux_jur(code):
    """Retourne (Niveau_I, Niveau_II, Niveau_III) selon la nomenclature INSEE
    des catégories juridiques (cj_septembre_2022) : 1er chiffre / 2 premiers / code complet."""
    code = str(code).strip()
    if not code:
        return ("", "", "")
    n1 = _CJ_NIV1.get(code[:1], "")
    n2 = _CJ_NIV2.get(code[:2], "") if len(code) >= 2 else ""
    n3 = _CJ_NIV3.get(code, "") if len(code) >= 4 else ""
    return (n1, n2, n3)

def _infos_entreprise(res):
    siege = res.get("siege") or {}
    nj = str(res.get("nature_juridique") or siege.get("nature_juridique") or "").strip()
    # 1) libellé fourni directement par l'API (forme_juridique.libelle) si présent
    fj = res.get("forme_juridique") or siege.get("forme_juridique") or {}
    lib = ""
    if isinstance(fj, dict):
        lib = str(fj.get("libelle") or "").strip()
        nj = str(fj.get("code") or nj).strip()
    # 2) sinon, traduction par la nomenclature INSEE (complète ou secours)
    if not lib:
        lib = _FORMES_JURIDIQUES.get(nj, "")
    forme = f"{lib} ({nj})" if lib else nj
    naf = str(res.get("activite_principale") or siege.get("activite_principale") or "").strip()
    libnaf = str(res.get("libelle_activite_principale") or siege.get("libelle_activite_principale") or "").strip()
    code_naf = f"{naf} – {libnaf}" if (naf and libnaf) else naf
    etat_code = str(res.get("etat_administratif") or siege.get("etat_administratif") or "").strip().upper()
    etat = {"A":"En activité","C":"Cessée"}.get(etat_code, "")
    siret = str(siege.get("siret") or res.get("siret") or "").strip()
    adr_api = str(siege.get("adresse") or res.get("adresse") or "").strip()
    cp_api    = str(siege.get("code_postal") or "").strip()        # v5_2 : lecture seule
    ville_api = str(siege.get("libelle_commune") or "").strip()    # v5_2 : lecture seule
    if not adr_api:
        adr_api = " ".join(x for x in (cp_api, ville_api) if x)
    _n1, _n2, _n3 = _niveaux_jur(nj)
    siren_ul = str(res.get("siren") or siege.get("siren") or (siret[:9] if siret else "")).strip()
    return {"siren":siren_ul, "siret":siret, "forme_juridique":forme, "code_naf":code_naf, "etat":etat,
            "adresse_api":adr_api, "cp_api":cp_api, "ville_api":ville_api,
            "niveau_i":_n1, "niveau_ii":_n2, "niveau_iii":_n3}

def siren_depuis_tva(tva):
    t = re.sub(r"[^0-9A-Za-z]", "", str(tva).upper())
    m = re.match(r"^FR[0-9A-Z]{2}(\d{9})", t)   # FR + clé + SIREN (tolère .0 / SIRET / espaces)
    if m:
        return m.group(1)
    d = re.sub(r"\D", "", t)
    if not t.startswith("FR"):
        if len(d) in (9, 14) and not re.search(r"[A-Z]", t):
            return d[:9]                          # SIREN (9) ou SIRET (14) nu
        return ""                                 # TVA étrangère : on ne devine pas
    if len(d) >= 11:
        return d[2:11]                            # FR + 2 (clé) + 9 (SIREN)
    if len(d) >= 9:
        return d[:9]
    return ""

def _fiche_avec_nom(res):
    """v5_39 — enveloppe LECTURE SEULE de _infos_entreprise pour les fiches par
    SIREN (Passe 0, Passe mémoire, passe 2) : ajoute le nom officiel de
    l'annuaire (« nom_complet », sinon « nom_raison_sociale » — même extraction
    que le moteur), que _infos_entreprise ne renvoie pas. _infos_entreprise
    elle-même n'est PAS modifiée : elle alimente aussi _appeler_api (moteur
    figé, best.update), où un champ nom_api ajouté écraserait celui du candidat
    retenu par le moteur."""
    d = dict(_infos_entreprise(res))
    if not d.get("nom_api"):
        d["nom_api"] = str(res.get("nom_complet") or res.get("nom_raison_sociale") or "").strip()
    return d

def infos_par_siren(siren, session, essais=4):
    """Fiche par SIREN avec réessais (le site web utilise la même API : si la
    fiche existe en ligne, un échec ici est presque toujours un 429/timeout)."""
    if not siren: return {}
    attente = 1.0
    for tentative in range(essais):
        try:
            time.sleep(DELAI_API)
            r = session.get(API_URL, params={"q": siren, "page":1, "per_page":5}, timeout=12)
            if r.status_code == 429:            # limite de débit : attendre puis retenter
                time.sleep(attente); attente *= 2
                continue
            r.raise_for_status()
            results = r.json().get("results", [])
        except Exception:
            time.sleep(attente); attente *= 2
            continue
        if results:
            for res in results:
                if str(res.get("siren","")).strip() == str(siren).strip():
                    return _fiche_avec_nom(res)      # v5_39 : fiche + nom officiel
            return _fiche_avec_nom(results[0])       # v5_39 : fiche + nom officiel
        # résultats vides : un seul re-essai (raté ponctuel de l'API)
        if tentative >= 1:
            return {}
        time.sleep(attente); attente *= 2
    return {}



CACHE_FICHES_SIREN = {}   # v5_2 : cache global des fiches SIRENE (évite tout double appel)
def fiche_siren(siren, session):
    """Fiche SIRENE par SIREN, avec cache global (lecture seule, HORS moteur figé)."""
    s = str(siren or "").strip()
    if not s:
        return {}
    if s not in CACHE_FICHES_SIREN:
        CACHE_FICHES_SIREN[s] = infos_par_siren(s, session)
    return CACHE_FICHES_SIREN[s]

# ── Appel API + sélection du meilleur résultat (TVA DGFiP prioritaire) ────────
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  FONCTION FIGÉE nº3/5 — _appeler_api                                      ║
# ║  NE PAS MODIFIER (vérifiée bit à bit).                                    ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
# RÔLE : lancer UNE requête HTTP, récupérer jusqu'à MAX_PAR_REQUETE candidats,
# les faire noter par _scorer, et renvoyer le meilleur s'il dépasse le seuil.
# Gère aussi les réessais (HTTP 429 = trop de requêtes) et la priorité donnée
# aux entités dont la TVA est confirmée côté DGFiP.
# SIGNATURE EXACTE (utile pour écrire des tests hors-ligne) :
#   _appeler_api(query, cp, nom_ref, adr_ref, ville_ref, session,
#                exige_collectivite=False)
def _appeler_api(query, cp, nom_ref, adr_ref, ville_ref, session, exige_collectivite=False):
    params = {"q": query, "page": 1, "per_page": MAX_PAR_REQUETE}
    if cp and str(cp).strip():
        dept = str(cp).strip()
        dept = dept[:3] if dept[:2] in ("97", "98") else dept[:2]
        if dept.isdigit():
            params["departement"] = dept
    try:
        time.sleep(DELAI_API)
        r = session.get(API_URL, params=params, timeout=12)
        r.raise_for_status()
        resultats = r.json().get("results", [])
    except Exception:
        return None
    if not resultats:
        return None

    ref_n = _normaliser(nom_ref)
    ref_v = _normaliser(ville_ref or "")
    best, best_score = None, -1
    for res in resultats:
        siren = str(res.get("siren", "")).strip()
        if not siren:
            continue
        noms = [res.get("nom_complet") or ""]
        for champ in ("nom_raison_sociale", "sigle"):
            if res.get(champ):
                noms.append(res[champ])
        for nom_api in noms:
            if not nom_api:
                continue
            api_norm = _normaliser(nom_api)
            score, detail = _scorer(ref_n, api_norm,
                                    adr_ref or "", ref_v, cp or "", res)
            # Garde-fou : pour une requête 'nom de commune', n'accepter que des
            # collectivités (rejette club / comité / caisse des écoles de la ville).
            if exige_collectivite and not _est_collectivite(api_norm):
                score = min(score, 40)
                detail += " | NON_COLLECTIVITE(rejet)"
            if score > best_score:
                best_score = score
                best = {"siren": siren, "nom_api": nom_api,
                        "score": score, "detail": detail, "_r": res}
    if not best:
        return None

    tva = _tva_dgfip(best["_r"])
    if tva:
        best["tva"]    = tva
        best["detail"] += " | TVA=DGFiP"
    else:
        best["tva"] = _siren_vers_tva(best["siren"])
    best.update(_infos_entreprise(best["_r"]))   # lecture seule, post-sélection
    del best["_r"]
    return best


def _tva_dgfip(api_res):
    lst = (api_res.get("complements") or {}).get("tva", None)
    if not lst:
        return None
    if isinstance(lst, str):
        lst = [lst]
    for t in lst:
        t = str(t).strip().upper().replace(" ", "")
        if t.startswith("FR") and len(t) == 13 and t[2:].isdigit():
            return t
    return None


# ── Point d'entrée : cascade fusionnée ────────────────────────────────────────
# ── Décollage des mots collés par troncature (DUMOTOCYCLE -> DU MOTOCYCLE) ─────
_PREFIXES_COLLES = ["DU","DES","DE","LES","LA","LE","ET","EN","AUX","AU"]
_RE_PREFIXES = re.compile(r"\b(" + "|".join(_PREFIXES_COLLES) + r")([A-ZÀ-Ÿ]{3,})")
_BLANCHE_COLLES = {"DESERT","DESIR","DESSIN","DESSERT","DESASTRE","DETAIL","DEVELOPPEMENT",
    "DEVELOPPER","DELEGATION","DELEGUE","DEMOCRATIE","DEMOCRATIQUE","DEPARTEMENT","DEPARTEMENTAL",
    "DEPENSE","DEFENSE","DECISION","DECHET","DECOUVERTE","LABORATOIRE","LABEL","LANGUE","LANGAGE",
    "LECTURE","LEGUME","LEGISLATION","LIBERTE","LOGEMENT","ETUDE","ETUDES","ETABLISSEMENT","ETAT",
    "ETRANGER","ENFANCE","ENFANT","ENERGIE","ENSEIGNEMENT","ENTREPRISE","ENVIRONNEMENT","EDUCATION",
    "DESORMAIS","LEADER","LEADERSHIP","AUTOMOBILE","AUTONOMIE","AUTORITE","AUVERGNE","DESIGN",
    "LEGENDE","LECON","DELICE","DELAI","DEMARCHE","DUREE","DURABLE"}
_ARTICLES_DBL = {"DELA":"DE LA","DELE":"DE LE","DELES":"DE LES"}
def _decoller_un(m):
    pref,suite=m.group(1),m.group(2)
    return pref+suite if (pref+suite) in _BLANCHE_COLLES else f"{pref} {suite}"
def decoller_mots(nom):
    s=re.sub(r"\b(DELA|DELE|DELES)\b(?=\s+[A-ZÀ-Ÿ])", lambda m:_ARTICLES_DBL[m.group(1)], str(nom))
    return _RE_PREFIXES.sub(_decoller_un, s)


# ── Corrections de noms (coquilles sûres + renommages) — stratégie additionnelle ──
_CORRECTIONS_NOMS = [
    (r"\bCOMUNAUTE\b", "COMMUNAUTE"),
    (r"\bINTERANTIONALE?\b", "INTERNATIONALE"),
    (r"\bSLEROSE\b", "SCLEROSE"),
    (r"\bPROFESSINNELLE\b", "PROFESSIONNELLE"),
    (r"\bACCEUIL\b", "ACCUEIL"),
    (r"\bAVENCEMENT\b", "AVANCEMENT"),
    (r"\bOSSERVATION\b", "OBSERVATION"),
    (r"\bMEDICINE\b", "MEDECINE"),
    (r"\bINTERDEPARTMENTAL\b", "INTERDEPARTEMENTAL"),
    (r"\bDEPARTAMENTAL\b", "DEPARTEMENTAL"),
    (r"\bENVIRONEMENT\b", "ENVIRONNEMENT"),
    (r"\bMID-PYRENEES\b", "MIDI-PYRENEES"),
    (r"\bVALEES\b", "VALLEES"),
    (r"\bISTITUTE\b", "INSTITUTE"),
    (r"\bRRESTAURATIVE\b", "RESTAURATIVE"),
    (r"\bFERROVIAIIRES\b", "FERROVIAIRES"),
    (r"\bINSDUSTRIE\b", "INDUSTRIE"),
    (r"\bLACHAUSSURE\b", "LA CHAUSSURE"),
    (r"\bDEES\s+LANDES\b", "DES LANDES"),
    (r"\bDULIVRADOIS\b", "DU LIVRADOIS"),
    (r"\bDUMOTOCYCLE\b", "DU MOTOCYCLE"),
    (r"\bDESUCRE\b", "DE SUCRE"),
    (r"\bDUSKI\b", "DU SKI"),
    (r"\bDELA\b", "DE LA"),
    (r"\bETDES\b", "ET DES"),
    (r"\bPETITESET\b", "PETITES ET"),
    (r"\bEDUCATIONFORMATION\b", "EDUCATION FORMATION"),
    (r"\bINNOVATIONASSOCIATION\b", "INNOVATION ASSOCIATION"),
    # — mots collés par troncature (ciblés, non ambigus) —
    (r"\bDUMOTOCYCLE\b", "DU MOTOCYCLE"),
    (r"\bDELORRAINE\b", "DE LORRAINE"),
    (r"\bDESCOMMUNICATIONS\b", "DES COMMUNICATIONS"),
    (r"\bDELA\b", "DE LA"),
    (r"\bPETITESET\b", "PETITES ET"),
    (r"\bEDUCATIONFORMATION\b", "EDUCATION FORMATION"),
    (r"\bACCUEILCARREFOUR\b", "ACCUEIL CARREFOUR"),
    (r"\bUTILITEPUBLIQUE\b", "UTILITE PUBLIQUE"),
    (r"\bTROISFRONTIERES\b", "TROIS FRONTIERES"),
    (r"\bDUSKI\b", "DU SKI"),
    (r"\bDUPAYS\b", "DU PAYS"),
    (r"\bDUBASSIN\b", "DU BASSIN"),
    (r"\bETDES\b", "ET DES"),
    (r"\bETDE\b", "ET DE"),
    (r"\bVALDE\b", "VAL DE"),
    (r"\bDESES\b", "DE SES"),
    (r"\bPETITESET\b", "PETITES ET"),
    (r"\bP[ÔO]LE\s+EMPLOI\b", "FRANCE TRAVAIL"),   # renommage officiel
    # Décollages de mots collés par troncature (aucun n'est un vrai mot français)
    (r"\bDUMOTOCYCLE\b", "DU MOTOCYCLE"),
    (r"\bDELORRAINE\b", "DE LORRAINE"),
    (r"\bDESCOMMUNICATIONS\b", "DES COMMUNICATIONS"),
    (r"\bDELA\b", "DE LA"),
    (r"\bPETITESET\b", "PETITES ET"),
    (r"\bACCUEILCARREFOUR\b", "ACCUEIL CARREFOUR"),
    (r"\bEDUCATIONFORMATION\b", "EDUCATION FORMATION"),
    (r"\bDEGESTION\b", "DE GESTION"),
    (r"\bDEPARIS\b", "DE PARIS"),
    (r"\bDELIMOGES\b", "DE LIMOGES"),
    (r"\bDUSKI\b", "DU SKI"),
    (r"\bDINDUSTRIEDE\b", "D INDUSTRIE DE"),
    (r"\bETDES\b", "ET DES"),
    (r"\bETDE\b", "ET DE"),
    (r"\bETMETROPOLES\b", "ET METROPOLES"),
    (r"\bDESES\b", "DE SES"),
    (r"\bSAINTPPIERRE\b", "SAINT PIERRE"),
    (r"\bDESPORTS\b", "DES SPORTS"),
    (r"\bDEMORT\b", "DE MORT"),
    (r"\bDEFRANCE\b", "DE FRANCE"),
]

_VOCAB = set()   # rempli au chargement du fichier (mots fiables du jeu de données)
_PREFIXES_COLLES = ["DES", "DU", "DE", "LES", "LA", "LE", "ET", "EN", "AUX", "AU"]
def decoller_mots(nom):
    """Décolle un petit mot grammatical collé devant un mot, SEULEMENT si le
    reste est un vrai mot vu ailleurs dans le fichier (évite LANDES->LA NDES)."""
    if not _VOCAB:
        return str(nom)
    out = []
    for m in str(nom).split():
        if any(c in m for c in "-'."):
            out.append(m); continue
        fait = False
        for p in sorted(_PREFIXES_COLLES, key=len, reverse=True):
            mu = m.upper()
            if mu.startswith(p) and len(mu) > len(p) + 3 and mu[len(p):] in _VOCAB:
                out.append(f"{m[:len(p)]} {m[len(p):]}"); fait = True; break
        if not fait:
            out.append(m)
    return " ".join(out)
def corriger_nom(nom):
    """Corrige des coquilles sûres et applique les renommages. Additif : un nom
    sans coquille connue est renvoyé inchangé (aucun effet sur les 87% trouvés)."""
    s = decoller_mots(str(nom))   # 1) décolle les mots collés par troncature
    for pat, rep in _CORRECTIONS_NOMS:   # 2) coquilles + renommages
        s = re.sub(pat, rep, s, flags=re.IGNORECASE)
    s = decoller_mots(s)   # décollage piloté par le vocabulaire
    return s


# ── Alias par SIREN : organisations internationales vérifiées (data.gouv) ─────
# Stratégie de DERNIER RECOURS : n'agit que si la recherche normale a échoué.
# SIREN vérifiés un par un -> aucun faux positif possible.
_SIREN_CONNUS = [
    # (motif sur le nom FTS, SIREN, libellé officiel)
    (r"ORGANISATION FOR ECONOMIC CO|COOPERATION ET DE DEVELOPPEMENT ECONOMIQUE|\bOCDE\b|\bOECD\b",
        "775687957", "ORGANIS COOPERATION DEVELOPP ECONOMIQUE (OCDE)"),
    (r"\bUNESCO\b|UNITED NATIONS EDUCATIONAL|NATIONS UNIES POUR L.EDUCATION",
        "775665789", "UNITED NATION EDUCA SCIENT CULTUR ORGANI (UNESCO)"),
    (r"ORGANISATION INTERNATIONALE DE LA FRANCOPHONIE",
        "784314858", "ORGANISATION INTERNATIONALE DE LA FRANCOPHONIE"),
    (r"INTERNATIONAL CRIMINAL POLICE|POLICE CRIMINELLE|\bINTERPOL\b",
        "785447467", "ORGANISATION INTERNAT POLICE CRIMINELLE (OIPC INTERPOL)"),
    (r"RECHERCHE SUR LE CANCER|RESEARCH ON CANCER",
        "779925684", "CENTRE INTERNAT RECH CANCER (CIRC IARC)"),
]
def _siren_connu(nom):
    s = str(nom)
    for pat, siren, lib in _SIREN_CONNUS:
        if re.search(pat, s, flags=re.IGNORECASE):
            return siren, lib
    return None, None


# ── Organisations internationales : alias nom FTS -> SIREN (vérifiés annuaire) ──
# Ces entités ont un nom légal SIRENE très différent du nom FTS ; on cible le SIREN.
_ALIAS_SIREN = {
    "775687957": ["ORGANISATION DE COOPERATION ET DE DEVELOPPEMENT ECONOMIQUES", "OCDE",
                  "ORGANISATION FOR ECONOMIC CO-OPERATION AND DEVELOPMENT", "OECD"],
    "775665789": ["ORGANISATION DES NATIONS UNIES POUR L'EDUCATION", "UNESCO",
                  "UNITED NATION EDUCA SCIENT CULTUR ORGANI"],
    "784314858": ["ORGANISATION INTERNATIONALE DE LA FRANCOPHONIE", "OIF"],
    "785447467": ["INTERNATIONAL CRIMINAL POLICE ORGANIZATION", "INTERPOL",
                  "ORGANISATION INTERNATIONALE POLICE CRIMINELLE", "OIPC"],
    "779925684": ["CENTRE INTERNATIONAL DE RECHERCHE SUR LE CANCER", "CIRC",
                  "INTERNATIONAL AGENCY FOR RESEARCH ON CANCER", "IARC"],
}
def _norm_alias(s):
    s = unicodedata.normalize("NFD", str(s).upper())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[^A-Z0-9]", " ", s)
# ── Corrections manuelles vérifiées (rapport RAPPORT_V4) : SIREN forcé → noms FTS.
#    68 SIREN / 163 noms. Complète _ALIAS_SIREN.

# ── v5_7 : corrections vérifiées du rapport « correction_1.xlsx » ─────────────
# 46 « CORRECT » (TVA confirmées) + 4 « CORRIGE » (SIREN corrigé, TVA effacée) :
# forcées à 100 par correspondance EXACTE du nom FTS (pas de sous-chaîne, pour ne
# jamais capter des variantes non confirmées, ex. THALES vs THALES ALENIA SPACE).
_CORRECTIONS_RAPPORT_V1 = {
    "AGENCE DE DEVELOPPEMENT ECONOMIQUEDE LA NOUVELLE CALEDONIE ASSOCIATION*ADECAL": "993389543",
    "ALLIANCE FOR INTERNATIONAL MEDICALACTION ASSOCIATION*ALIMA": "932808918",
    "ARTTIC SAS*": "880362439",
    "ASSOCIATION 4D": "899353726",
    "ASSOCIATION APOLLONIA*EUROPEAN ARTEXCHANGES ECHANGES ARTITIQUES EUROPEENS": "421007436",
    "ASSOCIATION BRETAGNE POLOGNE*": "378251920",
    "ASSOCIATION CENTRE DON BOSCO*ETABLISSEMENT INSERTION PAR LA FORMATIONEIPF DON BOSCO": "775577950",
    "ASSOCIATION FSD France": "490944998",
    "ASSOCIATION INTERNATIONALE DES MAIRES ET RESPONSABLES DES CAPITALES ETMETROPOLES PARTIELLEMENT OU ENTIEREMENT FRANCOPHONES*AIMF": "439247040",
    "ASSOCIATION PARCOURS D EXIL*": "444001804",
    "BRUITPARIF ASSOCIATION*OBSERVATOIRE REGIONAL DU BRUIT EN ILE DE FRANCE": "483921219",
    "BUREAU INTERNATIONAL CATHOLIQUE DEL ENFANCE ASSOCIATION*INTERNATIONAL CATHOLIC CHILD BUREAU BICE ICCB": "313821316",
    "CENTRE D INFORMATION JEUNESSE SEINEET MARNE ASSOCIATION*CIJ 77": "352945174",
    "COMITE DE COOPERATION AVEC LE LAOSASSOCIATION*CCL": "381757814",
    "COMITE DE JUMELAGE D'AIXE SUR VIENNE ASSOCIATION*": "421122235",
    "COMITE DE JUMELAGE DE LA VILLE DE SAINT-BRIEUC ASSOCIATION": "389042383",
    "COMITE DE JUMELAGE DE MONTAIGUT ENCOMBRAILLE ASSOCIATION*": "523309870",
    "COMITE REGIONAL U S E P*COMITE REGIONAL L UNION SPORTIVE DE L ENSEIGNEMENT DU PREMIER DEGRE LANGUEDOC ROUSSILLON": "478632672",
    "COMMUNE D'AMIENS*ECOLE SUPERIEURE ART ET DESIGN": "200027084",
    "CONFEDERATION EUROPEENNE DES VIGNERONS INDEPENDANTS ASSOCIATION*CEVI": "480960715",
    "CONGES INTEMPERIES BTP - UNION DESCAISSES DE FRANCE": "784621344",
    "CONSEIL DEPARTEMENTAL DES BOUCHES DU RHONE": "820015147",
    "CONSEIL INTERNATIONAL MUSEES*INTERNATIONAL COUNCIL OF MUSEUMS": "784617813",
    "CONSERVATOIRE BOTANIQUE NATIONAL DES PYRENEES ET DES MIDI-PYRENEESSYNDICAT MIXTE CONSERVATOIRE BOTANIQUE PYRENEEN*CBNPMP": "256502154",
    "CONSERVATOIRE D'ESPACES NATURELS DEPICARDIE": "381226406",
    "COORDINATION EUROPEENNE DES PRODUCTEURS INDEPENDANTS - CEPI*EUROPEAN COORDINATION OF INDEPENDENT PRODUCERS": "803779917",
    "DELPHIS DEVELOPPEMENT ETUDES POUR LE LOGEMENT LA PROMOTION DE L'HABITAT L'INNOVATION ET LE SOCIAL ASSOCIATION*DELPHIS": "352244107",
    "FEDERATION EUROPEENNE DU SPORT D'ENTREPRISE-EFCS*EUROPEAN FEDERATION OF COMPANY SPORT": "842605255",
    "FEDERATION FRANCAISE DE LA COOPERATION FRUITIERE LEGUMIERE ET HORTICOLE SYNDICAT PATRONAL*FELCOOP": "391487493",
    "FONDATION FONDAMENTAL*": "499010379",
    "FORUM CIVIQUE EUROPEEN ASSOCIATION*FCE EUROPEAN CIVIC FORUM": "389571498",
    "FORUM DE PARIS SUR LA PAIX*PARIS PEACE FORUM": "838383081",
    "GRAPHISTES DE L'OMBRE ASSOCIATION*GO": "493115703",
    "INSTITUT FRANCAIS DES DROITS ET LIBERTES*FRENCH INSTITUE OF RIGHTS AND FREEEDOMS": "482698982",
    "ITALIA - SPORT INSIEME*ITALIE - SPORT ENSEMBLE": "822794772",
    "LA CHAINE DE L'ESPOIR ASSOCIATION*LCDE": "399818418",
    "LE CERCLE POLAIRE ASSOCIATION*LCP": "500798244",
    "LES ENFANTS DE CINEMA*": "397724717",
    "MOUVEMENT DES ENTREPRISES DE FRANCE ASSOCIATION*MEDEF ALSACE": "481894988",
    "PERSE CIRCUS ECOLE DE CIRQUE DE THIERVILLE SUR MEUSE ASSOCIATION*": "441452950",
    "PISTES SOLIDAIRES ASSOCIATION*": "448858704",
    "PLAN-SEQUENCE ASSOCIATION*": "414469726",
    "PRODISS ORGANISME PROFESSIONNEL*SNPS": "407988120",
    "SOLIDARITE THERAPEUTIQUE INITIATIVE SANTE": "453058679",
    "SYNDICAT MIXTE DU PARC INTERREGIONAL DU MARAIS POITEVIN*": "257902205",
    "SYNDICAT NATIONAL DIRECTEUR DES ENTREPRISES ARTISTIQUES ET CULTURELLES*SYNDEAC": "316344076",
    "TANDEM PLUS": "537419418",
    "THALES": "910082114",
    "UNION NATIONALE DES INDUSTRIES FRANCAISES DE L'AMEUBLEMENT ASSOCIATION*UNIFA": "784522559",
    "UNION SYNDICALE PRODUCT AUDIOVISUELLE SYNDICAT PATRONAL*USPA UNION SYNDICALE DE LA PRODUCTION AUDIOVISUELLE": "343224754"
}
# 4 « False » : faux positifs confirmés — ces SIREN/TVA ne doivent PLUS JAMAIS
# être renvoyés pour CES bénéficiaires (paires nom -> SIREN interdits).
_TVA_INTERDITES_BRUT = {
    "ASSOCIATION POUR LE DEVELOPPEMENT DE LA FORMATION PROFESSIONNELLE DANS LES TRANSPORTS": ["504123886"],
    "BIRIBIN LIMOUSINES SAS*SEBG": ["414199141"],
    "ETUDIANTS ET DEVELOPPEMENT": ["792272981"],
    "L'UNION EUROPEENNE DES ASSOCIATIONSDE JOURNALISTES SCIENTIFIQUES ASSOCIATION* EUROPEAN UNION OF SCIENCE JOURNALISTS ASSOCIATIONS EUSJA": ["808334478"]
}
_ALIAS_SIREN_EXACT = {}
_TVA_INTERDITES = {}

_ALIAS_SIREN_CORRECTIONS = {
    "130011836": ["AUTORITE NATIONALE DES JEUX", "AUTORITE REGULATION JEUX EN LIGNE"],
    "200006500": ["AD NORMANDIE", "AGENCE DE DEVELOPPEMENT POUR LA NORMANDIE", "AGENCE DE L'INNOVATION EN REGION HAUTE NORMANDIE"],
    "200008357": ["PARC NATIONAL DE LA REUNION", "PARC NATIONAL DE LA REUNION*"],
    "220300016": ["CONSEIL DEPARTEMENTAL DE L ALLIER", "CONSEIL GENERAL DE L ALLIER", "DEPARTEMENT DE L ALLIER"],
    "229750013": ["COLLECTIVITE TERRITORIALE DE SAINTPPIERRE ET MIQUELON", "COLLECTIVITE TERRITORIALE DE SAINTPPIERRE ET MIQUELON*TERRITORIAL COLLECTIVITY OF SAINT PIERRE AND MIQUELON"],
    "249730045": ["COMMUNAUTE D'AGGLOMERATION DU CENTRE LITTORAL"],
    "256802752": ["PARC NATURAL REGIONAL DES BALLONS DES VOSGES"],
    "267411080": ["CHI DES HOPITAUX DU PAYS DU MONT BLANC", "CHI DES HOPITAUX DU PAYS DU MONT BLANC*"],
    "312409030": ["INSTITUT DES HAUTES ETUDES ECONOMIQUES ET COMMERCIALES", "INSTITUT DES HAUTES ETUDES ECONOMIQUES ET COMMERCIALES ASSOCIATION", "INSTITUT DES HAUTES ETUDES ECONOMIQUES ET COMMERCIALES ASSOCIATION*INSEEC"],
    "313255747": ["INTERNATIONAL ENERGY AGENCY", "INTERNATIONAL ENERGY AGENCY*IEA AGENCE INTERNATIONALE DE L'ENERGIE AIE"],
    "316713015": ["OPUS", "UNION ASSOCIATION POUR LA PARTICIPATION ET L'ACTION GENERALE", "UNION ASSOCIATION POUR LA PARTICIPATION ET L'ACTION REGIONALE"],
    "317425973": ["CENTRE INFORMATION JEUNESSE DU VALD'OISE ASSOCIATION", "CENTRE INFORMATION JEUNESSE DU VALD'OISE ASSOCIATION*CIJ95"],
    "318990892": ["ACTION CONTRE LA FAIM", "ACTION CONTRE LA FAIM*ACF"],
    "325933703": ["FONDATION TOUR DU VALAT", "FONDATION TOUR DU VALAT*TDV"],
    "331942672": ["CENTRE INTERNATIONAL DE HAUTES ETUDES AGRONOMIQUES MEDITERRANEENNES", "CENTRE INTERNATIONAL DE HAUTES ETUDES AGRONOMIQUES MEDITERRANEENNES*", "CENTRE INTERNATIONAL DES HAUTES ETUDES AGRONOMIQUES MEDITERRANEENNES INSTITUT AGRONOMIQUE MEDITERRANEEN DE MONTPELLIER CIHEAM IAMM"],
    "335232831": ["ACTING FOR LIFE, LA VIE, PAS LA SURVIE", "ACTING FOR LIFE, LA VIE, PAS LA SURVIE ASSOCIATION", "ACTING FOR LIFE, LA VIE, PAS LA SURVIE ASSOCIATION*"],
    "347403156": ["ASMAE ASSOCIATION SOEUR EMMANUELLEASSOCIATION", "ASMAE ASSOCIATION SOEUR EMMANUELLEASSOCIATION*", "ASMAE-ASSOCIATION SOEUR EMMANUELLE"],
    "384431821": ["EURISY-PROMOTION DE L'ENSEIGNEMENTET DE L'INFORMATION SUR L'AVENCEMENT DE LA TECHNOLOGIE SPATIALE ET DESES APPLICATIONS EN EUROPE", "EURISY-PROMOTION DE L'ENSEIGNEMENTET DE L'INFORMATION SUR L'AVENCEMENT DE LA TECHNOLOGIE SPATIALE ET DESES APPLICATIONS EN EUROPE*"],
    "390370377": ["NORD FRANCE INNOVATION DEVELOPPEMENT", "NORD FRANCE INNOVATION DEVELOPPEMENT ASSOCIATION", "NORD FRANCE INNOVATION DEVELOPPEMENT ASSOCIATION*NFID"],
    "392113445": ["INFORMATION MUSIQUE ANIMATION JEUNESSE", "INFORMATION MUSIQUE ANIMATION JEUNESSE IMAJ ASSOCIATION", "INFORMATION MUSIQUE ANIMATION JEUNESSE IMAJ ASSOCIATION*"],
    "393531280": ["CENTRE DE RESSOURCES ET DE DOCUMENTATION DES EURES TRANSFONTALIERS DELORRAINE ASSOCIATION", "CENTRE DE RESSOURCES ET DE DOCUMENTATION DES EURES TRANSFONTALIERS DELORRAINE ASSOCIATION*CRD EURES LORRAINE", "CENTRE DE RESSOURCES ET DE DOCUMENTATION EURES/ FRONTALIERS GRAND EST"],
    "401335104": ["CENTRE INTERNATIONAL DE FORMATION EUROPEENNE", "CENTRE INTERNATIONAL DE FORMATIONEUROPEENNE CIFE ASSOCIATION", "CENTRE INTERNATIONAL DE FORMATIONEUROPEENNE CIFE ASSOCIATION*"],
    "407526508": ["EUROMONTANA AISBL", "EUROMONTANA AISBL*"],
    "407566934": ["ASSOCIATION DES CHAMBRES D'AGRICULTURE DE L'ARC ATLANTIQUE", "ASSOCIATION DES CHAMBRES D'AGRICULTURE DE L'ARC ATLANTIQUE*CHAMBERS OFAGRICULTURE ATLANTIC AREA"],
    "407571306": ["CENTRE D'INFORMATION SUR LES INSTITUTIONS EUROPEENNES STRASBOURG", "CENTRE D'INFORMATION SUR LES INSTITUTIONS EUROPEENNES STRASBOURG ASSOCIATION", "CENTRE D'INFORMATION SUR LES INSTITUTIONS EUROPEENNES STRASBOURG ASSOCIATION*CIIES"],
    "412884454": ["FOUNDATION POUR LA NATURE  ET LHOMME", "FOUNDATION POUR LA NATURE  ET LHOMME*FOUNDATION POUR LA NATURE ET LHOME"],
    "413459066": ["EURORDIS - EUROPEAN ORGANISATION FOR RARE DISEASES ASSOCIATION", "EURORDIS - EUROPEAN ORGANISATION FOR RARE DISEASES ASSOCIATION*"],
    "422611905": ["EURODOC", "EURODOC ASSOCIATION", "EURODOC ASSOCIATION*"],
    "422915116": ["ASSOCIATION MIGRATION SOLIDARITE &ECHANGE POUR LE DEVELOPPEMENT", "ASSOCIATION MIGRATION SOLIDARITE &ECHANGE POUR LE DEVELOPPEMENT*AMSED"],
    "424901916": ["PL4Y INTERNATIONAL", "PLAY INTERNATIONAL"],
    "429565252": ["MRCA/RELIEF INTERNATIONAL - FRANCE", "RELIEF INTERNATIONAL FRANCE"],
    "433228681": ["END CHILD PROSTITUTION AND TRAFFICKING (ECPAT FRANCE)", "END CHILD PROSTITUTION AND TRAFFICKING (ECPAT FRANCE)*", "END CHILD PROSTITUTION, CHILD PORNOGRAPHY AND TRAFFICKING OF CHILDRENFOR SEXUAL PURPOSES"],
    "433960762": ["INRA TRANSFERT SA", "INRA TRANSFERT SAS", "INRAE TRANSFERT"],
    "434049953": ["GRAND E-NOV", "GRAND E-NOV +", "GRAND E-NOV PLUS", "GRAND EST DEVELOPPEMENT"],
    "434082756": ["CENTRE REGIONAL INTER-ASSOCIATIF ETDE SOUTIEN TECHNIQUE POUR LES ECHANGES EUROPEENS  EN LORRAINE (C.R.I.ST.E.E.L.)", "CENTRE REGIONAL INTER-ASSOCIATIF ETDE SOUTIEN TECHNIQUE POUR LES ECHANGES EUROPEENS  EN LORRAINE (C.R.I.ST.E.E.L.) ASSOCIATION", "CENTRE REGIONAL INTER-ASSOCIATIF ETDE SOUTIEN TECHNIQUE POUR LES ECHANGES EUROPEENS  EN LORRAINE (C.R.I.ST.E.E.L.) ASSOCIATION*"],
    "437527013": ["EUROCHIPS ASSOCIATION", "EUROCHIPS ASSOCIATION*EUROPEAN NETWORK FOR CHILDREN OF IMPRISONED PARENTS"],
    "439833252": ["ASSEMBLEE DES REGIONS EUROPEENNES FRUITIERES LEGUMIERES ET HORTICOLESASSOCIATION", "ASSEMBLEE DES REGIONS EUROPEENNES FRUITIERES LEGUMIERES ET HORTICOLESASSOCIATION*AREFLH"],
    "440819027": ["ASSOCIATION DES RESIDENCES ROYALESEUROPEENNES"],
    "442144002": ["ALDA - ASSOCIATION EUROPEENNE POURLA DEMOCRATIE LOCALE", "ASSOCIATION DES AGENCES DE LA DEMOCRATIE LOCALE", "ASSOCIATION DES AGENCES DE LA DEMOCRATIE LOCALE*ASSOCIATION OF LOCAL DEMOCRACY AGENCIES AADL/ALDA"],
    "445205099": ["ASSOCIATION LUCI LIGHTING URBAN COMMUNITY INTERNATIONAL", "ASSOCIATION LUCI LIGHTING URBAN COMMUNITY INTERNATIONAL*"],
    "450654595": ["EUROPEAN OBSERVATOIRE OF SPORT ANDEMPLOYMENT", "EUROPEAN OBSERVATOIRE OF SPORT ANDEMPLOYMENT ASSOCIATION INTERNATIONALE", "EUROPEAN OBSERVATOIRE OF SPORT ANDEMPLOYMENT ASSOCIATION INTERNATIONALE*OBSERVATOIRE EUROPEEN DE L EMPLOI ET DU SPORT EOSE", "EUROPEAN OBSERVATOIRE OF SPORT ANDEMPLOYMENT*OBSERVATOIRE EUROPEEN DEL EMPLOI ET DU SPORT EOSE"],
    "480105261": ["ASSOCIATION EUROPEENNE DES EMPLOYEURS DU SPORT", "ASSOCIATION EUROPEENNE DES EMPLOYEURS DU SPORT* EUROPEAN ASSOCIATION OF SPORT EMPLOYERS EASE", "ASSOCIATION EUROPEENNE DES EMPLOYEURS DU SPORT*EUROPEAN ASSOCIATION OF SPORT EMPLOYERS", "ASSOCIATION EUROPEENNE DES EMPLOYEURS DU SPORT*EUROPEAN ASSOCIATION OFSPORT EMPLOYERS"],
    "481910040": ["ASSOCIATION DES REGIONS EUROPEENNESDES PRODUITS D'ORIGINE"],
    "483730289": ["COORDINATING COMMITTEE FOR INTERNATIONAL VOLUNTARY SERVICE", "COORDINATING COMMITTEE FOR INTERNATIONAL VOLUNTARY SERVICE*COMITE DE COORDINATION DU SERVICE VOLONTAIRE INTERNATIONAL CCIVS CCSVI"],
    "485118392": ["ASSOCIATION INTERNATIONALE YAHAD INUNUM", "ASSOCIATION INTERNATIONALE YAHAD-INUNUM", "ASSOCIATION INTERNATIONALE YAHAD-INUNUM*"],
    "498140029": ["CONFEDERATION NATIONALE ARTISANALEDES INSTITUTS DE BEAUTE ORGANISATION PROFESSIONNELLE", "CONFEDERATION NATIONALE ARTISANALEDES INSTITUTS DE BEAUTE ORGANISATION PROFESSIONNELLE*CNAIB"],
    "499722643": ["COMITE CONSULTATIF REGIONAL DES EAUX OCCIDENTALES SUD ASSOCIATION", "COMITE CONSULTATIF REGIONAL DES EAUX OCCIDENTALES SUD ASSOCIATION*CCRSUD"],
    "509851754": ["INTERNATIONAL ENERGY AGENCY", "INTERNATIONAL ENERGY AGENCY*AGENCEINTERNATIONALE DE L'ENERGIE"],
    "534435052": ["BONDY INNOVATION"],
    "538857962": ["AQUITAINE DEVELOPPEMENT INNOVATIONASSOCIATION", "AQUITAINE DEVELOPPEMENT INNOVATIONASSOCIATION*"],
    "750031056": ["CONFERENCE DES VILLES DE L ARC ATLANTIQUE ASSOCIATION", "CONFERENCE DES VILLES DE L ARC ATLANTIQUE ASSOCIATION*CONFERENCE OF ATLANTIC ARC CITIES CONFERENCIA DE LACIUDADES ATLANTICAS CONFERENCIA DAS", "CONFERENCE DES VILLES DE L'ARC ATLANTIQUE", "CONFERENCE DES VILLES DE L'ARC ATLANTIQUE*CONFERENCE OF ATLANTIC ARC CITIES CONFERENCIA DE LA CIUDADES ATLANTICAS CONFERENCIA DAS CIDADES"],
    "775660103": ["UNION NATIONALE DES MAISONS FAMILIALES ET RURALES D'EDUCATION ET D'ORIENTATION ASSOCIATION", "UNION NATIONALE DES MAISONS FAMILIALES ET RURALES D'EDUCATION ET D'ORIENTATION ASSOCIATION*UNMFREO"],
    "775665789": ["UNITED NATIONS EDUCATIONAL SCIENTIFIC AND CULTURAL ORGANIZATION", "UNITED NATIONS EDUCATIONAL SCIENTIFIC AND CULTURAL ORGANIZATION*ORGANISATION DES NATIONES UNIES POUR L'EDUCATION LA SCIENCE ET LA CULTURE", "UNITED NATIONS EDUCATIONAL SCIENTIFIC AND CULTURAL ORGANIZATION*UNESCO ORGANISATION DES NATIONES UNIES L'EDUCATION LA SCIENCE ET LA CULTURE"],
    "775691736": ["COMITE NATIONAL DES PECHES MARITIMES ET DES ELEVAGES MARINS CNPMEM ORGANISME PROFESSIONNEL", "COMITE NATIONAL DES PECHES MARITIMES ET DES ELEVAGES MARINS CNPMEM ORGANISME PROFESSIONNEL*"],
    "778832675": ["COMMISSION CENTRALE POUR LA NAVIGATION DU RHIN", "COMMISSION CENTRALE POUR LA NAVIGATION DU RHIN*CCNR"],
    "778860080": ["CONSEIL DE L' EUROPE", "CONSEIL DE L' EUROPE*COUNCIL OF EUROPE"],
    "781112891": ["CENTRE REGIONAL DE LUTTE CONTRE LECANCER HENRI BECQUEREL ROUEN", "CENTRE REGIONAL DE LUTTE CONTRE LECANCER HENRI BECQUEREL ROUEN*"],
    "781626817": ["ACCOMPAGNEMENT LIEUX D'ACCUEIL CARREFOUR EDUCATIF ET SOCIAL", "ACCOMPAGNEMENT LIEUX D'ACCUEILCARREFOUR EDUCATIF ET SOCIALASSOCIATION RECONNUE D UTILITEPUBLIQUE", "ACCOMPAGNEMENT LIEUX D'ACCUEILCARREFOUR EDUCATIF ET SOCIALASSOCIATION RECONNUE D UTILITEPUBLIQUE*ALC"],
    "783707060": ["ECOLE DES HAUTES ETUDES COMMERCIALES DU NORD"],
    "784412546": ["CENTRE INTERNATIONAL DE FORMATIONEUROPEENNE CIFE ASSOCIATION", "CENTRE INTERNATIONAL DE FORMATIONEUROPEENNE CIFE ASSOCIATION*"],
    "784579641": ["COMITE POUR LES RELATIONS NATIONALES ET INTERNATIONALES DES ASSOCIATIONS DE JEUNESSE ET D EDUCATION POPULAIRE CNAJEP ASSOCIATION", "COMITE POUR LES RELATIONS NATIONALES ET INTERNATIONALES DES ASSOCIATIONS DE JEUNESSE ET D EDUCATION POPULAIRE CNAJEP ASSOCIATION*"],
    "785423997": ["EUROPEAN SPACE AGENCY", "EUROPEAN SPACE AGENCY*AGENCE SPATIALE EUROPEENNE", "EUROPEAN SPACE AGENCY*ESA AGENCE SPATIALE EUROPEENNE ASE"],
    "802547125": ["ASSOCIATIONS FRANCAISE DES POLES DE COMPETITIVITE", "ASSOCIATIONS FRANCAISE DES POLES DE COMPETITIVITE*THE FRENCH COMPETITIVENESS CLUSTERS ALLIANCE"],
    "847817798": ["EUROPEAN BANKING AUTHORITY", "EUROPEAN BANKING AUTHORITY*AUTHORITE BANCAIRE EUROPEENNE", "EUROPEAN BANKING AUTHORITY*EBA"],
    "850995002": ["LA MAISON DE L'EUROPE EN MAYENNE ASSOCIATION", "LA MAISON DE L'EUROPE EN MAYENNE ASSOCIATION*MEM53"],
    "913098802": ["AGENCE FERROVIAIRE EUROPEENNE", "AGENCE FERROVIAIRE EUROPEENNE*EUROPEAN RAILWAY AGENCY ERA", "EUROPEAN UNION AGENCY FOR RAILWAYS"],
    "922352216": ["SOLIDARITES INTERNATIONAL", "SOLIDARITES INTERNATIONAL ASSOCIATION", "SOLIDARITES INTERNATIONAL ASSOCIATION*"],
    "938263019": ["FRANCE TERRE D'ASILE", "FRANCE TERRE D'ASILE-ASSOCIATION", "FRANCE TERRE D'ASILE-ASSOCIATION*"],
}

# Nom officiel (annuaire des entreprises) à forcer dans « Bénéficiaire corrigé »
# pour les SIREN dont le nom FTS diffère fortement (renommages vérifiés à la main).
_NOM_OFFICIEL_PAR_SIREN = {
    "130005481": "FRANCE TRAVAIL",   # v5_6 : nom officiel (ex-Pôle emploi)
    "130011836": "AUTORITE NATIONALE DES JEUX",
    "200006500": "AGENCE DE DEVELOPPEMENT POUR LA NORMANDIE",
    "220300016": "DEPARTEMENT DE L ALLIER",
    "316713015": "OPUS",
    "433960762": "INRAE TRANSFERT",
    "434049953": "GRAND EST DEVELOPPEMENT",
}

# Fusion des corrections vérifiées dans la table d'alias (le nom FTS le plus
# spécifique gagne ; en cas de SIREN déjà présent, on complète la liste de noms).
for _siren_c, _noms_c in _ALIAS_SIREN_CORRECTIONS.items():
    _ALIAS_SIREN.setdefault(_siren_c, [])
    for _n in _noms_c:
        if _n not in _ALIAS_SIREN[_siren_c]:
            _ALIAS_SIREN[_siren_c].append(_n)

# ── v5_16 : corrections vérifiées du rapport « Classeur1.xlsx » (2e campagne) ──
# 35 CORRECT + 12 CORRIGE + 3 TROUVE + 1 CORRIGE TVA : SIREN forcé à 100 par
# correspondance EXACTE du nom FTS. Clés de Luhn vérifiées : 61/61 valides.
_CORRECTIONS_RAPPORT_V2 = {
    "ACCORD RELATIF AUX PECHES DANS LE SUD DE L'OCEAN INDIEN*SOUTHERN INDIAN OCEAN FISHERIES AGREEMENT (SIOFA)": "823504279",
    "AGRISUD INTERNATIONAL INSTITUT INTERNATIONAL POUR APPUI AU DEVELOPPEMENT ASSOCIATION*": "390364776",
    "AGROPOLIS FONDATION": "404087439",
    "ALLIANCE POUR UNE MINE RESPONSABLEEUROPE - ARM EUROPE": "843233610",
    "ASSOCIATION DU FESTIVAL INTERNATIONAL DE PROGRAMMES AUDIOVISUELS*": "828470609",
    "ASSOCIATION FRANCAISE POUR L'ETUDEDU SOL": "785152042",
    "ASSOCIATION FSD FRANCE*": "490944998",
    "ASSOCIATION MEDECINS DU MONDE*FRANCE": "321018749",
    "ASSOCIATION POUR LE DEVELOPPEMENT DES INITIATIVES CITOYENNES ET EUROPEENNES": "753110493",
    "ASSOCIATION PREMIERS PLANS*FESTIVAL PREMIERS PLANS": "420134389",
    "ASSOCIATION SOLIDARITES JEUNESSES MCP*SJMCP": "877480269",
    "BANQUE DE DEVELOPPEMENT DU CONSEILDE L'EUROPE": "784656167",
    "BANQUE DE DEVELOPPEMENT DU CONSEILDE L'EUROPE 9*CEB": "784656167",
    "CALAIS PROMOTION, ASSOCIATION POURLE DEVELOPPEMENT ECONOMIQUE DU PAYS DU CALAISIS": "512472416",
    "CARE FRANCE ASSOCIATION*": "334805801",
    "CENTRE DE RESSOURCES D'EXPERTISE ET DE PERFORMANCE SPORTIVES DE RHONE-ALPES": "130018914",
    "CENTRE HOSPITALIER REG UNIVERSITAIRE DIJON*CHU DIJON HOPITAL DU BOCAGE": "265906719",
    "CENTRE HOSPITALIER ROUEN*HOPITAL CHARLES NICOLLE": "267601680",
    "COALITION MONDIALE CONTRE LA PEINEDE MORT ASSOCIATION*WORLD COALITION AGAINST THE DEATH PENALTY WCADP": "519878698",
    "COATEX SAS*": "971509070",
    "COMITE FRANCAIS POUR L'UICN*UNION INTERNATIONALE POUR LA CONSERVATIONDE LA NATURE": "424211704",
    "COMITE INTERPROFESSIONNEL DU POULETCLASSIQUE ET CERTIFIE": "504352998",
    "CONSEIL INTERPROFESSIONNEL DES VINS DU ROUSSILLON": "434341103",
    "CONSERVATOIRE D ESPACE NATURELS DELORRAINE": "333915569",
    "CONSERVATOIRE D ESPACES NATURELS DECHAMPAGNE ARDENNE": "344896998",
    "CONSERVATOIRE D'ESPACES NATURELS CORSE": "390752202",
    "CONSERVATOIRE D'ESPACES NATURELS DE MID-PYRENEES*": "390717999",
    "E-SENIORS: INITIATION DES SENIORS AUX NTIC ASSOCIATION*E-SENIORS": "491282364",
    "ECOLE DE FORMATION PROFESSIONNELLEDES AVOCATS DES BARREAUX DU RESSORT DE LA COUR D'APPEL DE PARIS": "300227279",
    "ECOLE DES INGENIEURS DE LA VILLE DEPARIS*EIVP": "200000693",
    "ECOLE SUPERIEURE DES SCIENCES COMMERCIALES D ANGERS ASSOCIATION*": "442262119",
    "FORUM EUROPEEN POUR LA SECURITE URBAINE ASSOCIATION*EUROPEAN FORUM FORURBAN SECURITY FESU EFUS": "380253740",
    "GRANDIR DIGNEMENT ASSOCIATION*": "792085136",
    "GWADLOUP* GUADELOUPE REGION": "239710015",
    "HESPUL ASSOCIATION*": "402178701",
    "JEUNES TALENTS CIRQUE": "393118088",
    "JUSTICE COOPERATION INTERNATIONALEGIP": "130016462",
    "JUSTICE COOPERATION INTERNATIONALEGIP*JCI": "130016462",
    "LE GROUPE OUEST*": "539831420",
    "LIGUE FRANCAISE DE L'ENSEIGNEMENT ET DE L'EDUCATION PERMANENTE ASSOCIATION*": "315872184",
    "MAISON DE L'EUROPE TOURS CENTRE VALDE LOIRE ASSOCIATION*": "379046782",
    "NOE CONSERVATION ASSOCIATION*": "504862780",
    "NORD-PAS-DE-CALAIS*REGION": "235900016",
    "ORGANISATION EUROPEENNE ET MEDITERRANEENNE POUR LA PROTECTION DES PLANTES*EUROPEAN AND MEDITERRANEAN PLANT PROTECTION ORGANIZATION": "784669269",
    "PEPINIERES EUROPEENNES POUR JEUNESARTISTES ASSOCIATION*PEJA": "387926405",
    "PLANETE URGENCE ASSOCIATION*": "433095718",
    "REGIE REUNION THD": "899470330",
    "RESEAU DES ACHETEURS HOSPITALIERS IDF*RESEAU DES ACHETEURS HOSPITALIERS D'ILE-DE-FRANCE GIP RESAH-IDF": "130005010",
    "RESEAU EXPRESS JEUNES*YOUTH EXPRESS NETWORK": "417564614",
    "RESEAU SEMENCES PAYSANNES - ASSOCIATION POUR LA BIODIVERSITE DES SEMENCES ET PLANTS DANS LES FERMES*RSP": "453127250",
    "SYNDICAT MIXTE ATLANPOLE*": "254401839",
    "SYNDICAT MIXTE BAIE DE SOMME GRANDLITTORAL PICARD": "200039311",
    "SYNDICAT MIXTE INTERDEPARTEMENTAL DE LA VALLEE DE LA LEZE (SMIVAL)": "250900479",
    "SYNDICAT MIXTE POUR LA GESTION DESPORTS DE SUD ALSACE": "200076206",
    "SYNDICAT NATIONAL DES INDUSTRIELSDE OEUFS": "433078987",
    "TERRITOIRE DES ILES WALLIS ET FUTUNA* TERRITORY OF THE WALLIS AND FUTUNA SLANDS": "229860010",
    "UNION INTERNATIONALE DES HUISSIERSDE JUSTICE ET OFFICIERS JUDICIAIRES": "511895377",
    "UNION REGIONALE DES ACTEURS LOCAUXDE L'EUROPE EN AUVERGNE-RHONE-ALPES": "843665795",
    "UNION TECHNIQUE DE L'AUTOMOBILE DUMOTOCYCLE ET DU CYCLE SAS*": "438725723",
    "UNIVERSITE DE NICE SOPHIA ANTIPOLIS*SCES CENTRAUX NICE": "819024589",
    "UNIVERSITE DIJON BOURGOGNE*": "938230612"
}
# 10 « CORRIGE SIRET » : en plus du SIREN, le SIRET de l'ÉTABLISSEMENT indiqué
# par l'utilisateur est FORCÉ (remplace le SIRET du siège renvoyé par l'API).
_SIRET_FORCE_BRUT = {
    "ASSOCIATION DU FESTIVAL INTERNATIONAL DE PROGRAMMES AUDIOVISUELS*": "82847060900029",
    "ASSOCIATION POUR LE DEVELOPPEMENT DES INITIATIVES CITOYENNES ET EUROPEENNES": "75311049300016",
    "ASSOCIATION PREMIERS PLANS*FESTIVAL PREMIERS PLANS": "42013438900014",
    "ASSOCIATION SOLIDARITES JEUNESSES MCP*SJMCP": "87748026900019",
    "CENTRE HOSPITALIER REG UNIVERSITAIRE DIJON*CHU DIJON HOPITAL DU BOCAGE": "26590671900017",
    "CENTRE HOSPITALIER ROUEN*HOPITAL CHARLES NICOLLE": "26760168000015",
    "CONSERVATOIRE D ESPACES NATURELS DECHAMPAGNE ARDENNE": "34489699800046",
    "HESPUL ASSOCIATION*": "40217870100015",
    "JEUNES TALENTS CIRQUE": "39311808800011",
    "LIGUE FRANCAISE DE L'ENSEIGNEMENT ET DE L'EDUCATION PERMANENTE ASSOCIATION*": "31587218400017"
}
# 18 « False » : faux positifs confirmés (2e campagne) — paires nom -> SIREN interdits.
_TVA_INTERDITES_BRUT_V2 = {
    "AGENCE REGIONALE POUR L'INNOVATIONET L'INTERNATIONALISATION DES ENTREPRISES DE PROVENCE ALPES COTE D'AZUR": ["130007982"],
    "AGENCE REGIONALE POUR L'INNOVATIONET L'INTERNATIONALISATION DES ENTREPRISES DE PROVENCE ALPES COTE D'AZUR*": ["130007982"],
    "ASSOCIATION INTERNATIONALE DE SIGNALISATION MARITIME* AISM-IALA INTERNATIONAL ASSOCIATION OF LIGHT HOUSEMARINE AIDS TO NAVIGATION AUTHORITI": ["393096094"],
    "ASSOCIATION MAISON DE L'EUROPE DE RENNES ET HAUTE BRETAGNE*": ["511540304"],
    "ASSOCIATION POUR LE DEVELOPPEMENT DES INITIATIVES CITOYENNES ET EUROPEENNES*A.D.I.C.E": ["753110493"],
    "BRETAGNE DEVELOPPEMENT INNOVATION ASSOCIATION*BDI": ["418006367"],
    "CENTRE D'ETUDES PROSPECTIVES ET D'INFORMATIONS INTERNATIONALES*CEPII": ["377673702"],
    "CONSERVATOIRE D'ESPACES NATURELS DU NORD-PAS-DE-CALAIS": ["384643938"],
    "CORSE*REGION": ["913869350"],
    "ECOLE NATIONALE D'ADMINISTRATION*ENA": ["485241327"],
    "ECOLE NATIONALE SUPERIEURE DES TECHNIQUES INDUSTRIELLES ET DES MINES DE NANTES*": ["197500036"],
    "FONDATION DE COOPERATION SCIENTIFIQUE MALADIE D'ALZHEIMER ET MALADIESAPPARENTEES*": ["538405952"],
    "FONDATION DE COOPERATION SCIENTIFIQUE VOIR ET ENTENDRE*FOUNDATION OF SCIENTIFIC COOPERATION FOR HEARING AND SEEING FSCHS": ["538405952"],
    "LE PARTENARIAT ASSOCIATION*": ["793599663"],
    "NOUVELLE CALEDONIE*NEW CALEDONIA": ["882249824"],
    "PARIS REGION ENTREPRISES*": ["327716205"],
    "POLYNESIE FRANCAISE*FRENCH POLYNESIA": ["802391433"],
    "UNION DES SYNDICATS DE L INDUSTRIEROUTIERE FRANCAISE*": ["512238668"]
}
_SIRET_FORCE = {}

# v5_7 + v5_16 : normalisation des corrections exactes + interdits (au chargement)
for _n_c, _s_c in _CORRECTIONS_RAPPORT_V1.items():
    _ALIAS_SIREN_EXACT[_norm_alias(_n_c)] = _s_c
for _n_c, _s_c in _CORRECTIONS_RAPPORT_V2.items():
    _ALIAS_SIREN_EXACT[_norm_alias(_n_c)] = _s_c
for _n_c, _s_c in _SIRET_FORCE_BRUT.items():
    _SIRET_FORCE[_norm_alias(_n_c)] = _s_c
for _n_i, _l_i in _TVA_INTERDITES_BRUT.items():
    _TVA_INTERDITES[_norm_alias(_n_i)] = set(_l_i)
for _n_i, _l_i in _TVA_INTERDITES_BRUT_V2.items():
    _TVA_INTERDITES.setdefault(_norm_alias(_n_i), set()).update(_l_i)

def siren_alias(nom):
    """Renvoie le SIREN si le nom FTS correspond à une organisation internationale connue."""
    n = _norm_alias(nom)
    if n in _ALIAS_SIREN_EXACT:          # v5_7 : correction exacte du rapport (priorité)
        return _ALIAS_SIREN_EXACT[n]
    for siren, variantes in _ALIAS_SIREN.items():
        for v in variantes:
            vn = _norm_alias(v)
            if len(vn) >= 6 and vn in n:
                return siren
            if len(vn) < 6 and re.search(r"\b" + re.escape(vn) + r"\b", n):
                return siren
    return None


# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  FONCTION FIGÉE nº5/5 — rechercher_tva                                    ║
# ║  NE PAS MODIFIER (vérifiée bit à bit). Point d'entrée du moteur.          ║
# ╚═══════════════════════════════════════════════════════════════════════════╝
# RÔLE : orchestrer la cascade complète pour UN bénéficiaire :
#   exclusion éventuelle -> génération des requêtes -> appels successifs ->
#   sélection du meilleur candidat -> renvoi d'un dictionnaire de résultat.
# Toute évolution du comportement doit se faire dans l'ENVELOPPE
# rechercher_tva_plus (définie plus bas, NON figée), jamais ici.
def rechercher_tva(nom, adresse, ville, cp, session, seuil):
    vide = {"statut": "", "strategie": "", "siren": "",
            "tva": "", "nom_api": "", "score": 0, "detail": "",
            "siret": "", "forme_juridique": "", "code_naf": "", "etat": "", "adresse_api": "",
            "niveau_i": "", "niveau_ii": "", "niveau_iii": ""}
    if _est_exclu(nom):
        return {**vide, "statut": "EXCLU"}

    requetes = _generer_requetes(nom, adresse, cp, ville)
    meilleur = None
    for req in requetes:
        res = _appeler_api(req["q"], req["cp"], nom, adresse, ville, session,
                           req.get("col", False))
        if not res:
            continue
        if meilleur is None or res["score"] > meilleur["score"]:
            meilleur = {**res, "strategie": req["s"]}
        if res["score"] >= seuil:
            return {"statut": "TROUVE", "strategie": req["s"],
                    "siren": res["siren"], "tva": res["tva"],
                    "nom_api": res["nom_api"], "score": res["score"],
                    "detail": res["detail"],
                    "siret": res.get("siret", ""), "forme_juridique": res.get("forme_juridique", ""),
                    "code_naf": res.get("code_naf", ""), "etat": res.get("etat", ""),
                    "adresse_api": res.get("adresse_api", ""),
                    "niveau_i": res.get("niveau_i", ""), "niveau_ii": res.get("niveau_ii", ""),
                    "niveau_iii": res.get("niveau_iii", "")}
    if meilleur:
        # NON_TROUVE : on ne pose PAS la TVA (score < seuil), mais on CONSERVE le
        # meilleur candidat existant (siren/tva/infos) dans des champs « _cand_* »
        # pour un repli contrôlé par exactitude du nom (voir _repli_meilleur_candidat).
        return {"statut": "NON_TROUVE", "strategie": meilleur["strategie"],
                "siren": "", "tva": "", "nom_api": meilleur["nom_api"],
                "score": meilleur["score"], "detail": meilleur["detail"],
                "siret": "", "forme_juridique": "", "code_naf": "", "etat": "", "adresse_api": "",
                "niveau_i": "", "niveau_ii": "", "niveau_iii": "",
                "_cand_siren": meilleur.get("siren", ""), "_cand_tva": meilleur.get("tva", ""),
                "_cand_nom_api": meilleur.get("nom_api", ""), "_cand_score": meilleur.get("score", 0),
                "_cand_siret": meilleur.get("siret", ""),
                "_cand_forme_juridique": meilleur.get("forme_juridique", ""),
                "_cand_code_naf": meilleur.get("code_naf", ""), "_cand_etat": meilleur.get("etat", ""),
                "_cand_adresse_api": meilleur.get("adresse_api", ""),
                "_cand_niveau_i": meilleur.get("niveau_i", ""),
                "_cand_niveau_ii": meilleur.get("niveau_ii", ""),
                "_cand_niveau_iii": meilleur.get("niveau_iii", "")}
    return {**vide, "statut": "NON_TROUVE"}


# ── Confirmation par EXACTITUDE (anti-« satellites » : CSE, fonds de dotation, SCI…) ──
# Le moteur peut accepter une structure rattachée (comité d'entreprise, amicale,
# fondation partenariale…) dont le nom CONTIENT le nom cherché. Cette surcouche,
# ADDITIVE (le moteur reste intact), re-vérifie tout résultat non strictement
# exact en interrogeant l'API sur le nom seul (force de V1) et garde le candidat
# le plus exact. Elle ne remplace un résultat que par strictement mieux.
_MARQUEURS_SATELLITE = [
    "CSE", "CE", "COMITE SOCIAL ET ECONOMIQUE", "COMITE D ENTREPRISE",
    "FONDS DE DOTATION", "FONDATION PARTENARIALE", "AMICALE", "ASSOCIATION SPORTIVE",
    "BUREAU DES ELEVES", "BUREAU DES ARTS", "ORCHESTRE", "SYNDICAT", "SYND",
    "CGT", "CFDT", "FO", "FSU", "UNSA", "SNAP", "CONSEIL LOCAL", "PARENTS D ELEVES",
    "FCPE", "SCI", "AMIS", "PREFECTURE", "GENDARMERIE", "SECTION", "PROMOTION",
    "CLUB", "ALUMNI", "JUNIOR", "GRETA", "APEL", "COOPERATIVE SCOLAIRE",
]

def _norm_exact(s):
    s = unicodedata.normalize("NFD", str(s).upper())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

_SUFFIXES_FIN = [
    "ORGANISME PROFESSIONNEL", "ORGANISME PROFESSIONEL", "SYNDICAT DE SALARIES",
    "ASSOCIATION", "SYNDICAT", "SA", "SAS", "SASU", "SARL", "EURL", "SNC",
    "SCA", "SCOP", "SCIC", "SEM", "GIE", "GEIE", "GIP",
]

def _variantes_nom(nom):
    s = re.sub(r"\*+\s*$", "", str(nom)).strip()
    out = [s]
    if "*" in s:
        avant, _, apres = s.partition("*")
        if avant.strip():
            out.append(avant.strip())
        if len(apres.strip()) >= 4:
            out.append(apres.strip())
    # variantes sans suffixe juridique final (SIRENE omet souvent SA/SAS/ASSOCIATION…)
    plus = []
    for v in list(out):
        u = " " + re.sub(r"\s+", " ", str(v).upper().strip()) + " "
        for suf in _SUFFIXES_FIN:
            if u.endswith(" " + suf + " "):
                t = u[: -len(suf) - 2].strip()
                if len(t) >= 3 and t not in (x.upper() for x in out):
                    plus.append(t)
                break
    return out + plus

def _mots_satellites(extra):
    e = " " + " ".join(sorted(extra)) + " "
    return any((" " + m + " ") in e for m in _MARQUEURS_SATELLITE)

def _exactitude(nom_ref, nom_api):
    b = _norm_exact(nom_api)
    if not b:
        return 0
    tb = set(b.split())
    best = 0
    for v in _variantes_nom(nom_ref):
        a = _norm_exact(v)
        if not a:
            continue
        if a == b:
            return 100
        sc = fuzz.token_sort_ratio(a, b)
        extra = tb - set(a.split())
        if extra:
            sc -= min(24, 6 * len(extra))
            if _mots_satellites(extra):
                sc -= 30
        best = max(best, sc)
    return max(0, int(best))

def _candidats_bruts(query, session):
    try:
        time.sleep(DELAI_API)
        r = session.get(API_URL, params={"q": query, "page": 1, "per_page": 8}, timeout=12)
        r.raise_for_status()
        return r.json().get("results", [])
    except Exception:
        return []

def _resultat_depuis_candidat(res_api, strategie, score):
    siren = str(res_api.get("siren", "")).strip()
    base = {"statut": "TROUVE", "strategie": strategie, "siren": siren,
            "tva": _siren_vers_tva(siren),
            "nom_api": str(res_api.get("nom_complet") or res_api.get("nom_raison_sociale") or "").strip(),
            "score": score, "detail": "confirmation par exactitude du nom"}
    base.update(_infos_entreprise(res_api))
    return base

def _confirmer_exactitude(nom, res, cp, session, marge=2):
    ex_res = _exactitude(nom, res.get("nom_api", "")) if res.get("statut") == "TROUVE" else 0
    if ex_res >= 98:
        return res
    meilleur, ex_best = None, max(ex_res, 0)
    vus = set()
    for v in _variantes_nom(nom):
        q = v.strip()
        if not q or q.upper() in vus:
            continue
        vus.add(q.upper())
        for cand in _candidats_bruts(q, session):
            nom_c = str(cand.get("nom_complet") or cand.get("nom_raison_sociale") or "")
            e = _exactitude(nom, nom_c)
            if e > ex_best + marge and e >= 90:
                meilleur, ex_best = cand, e
    if meilleur is not None:
        return _resultat_depuis_candidat(meilleur, "CONFIRMATION_EXACTE", ex_best)
    return res


def rechercher_tva_plus(nom, adresse, ville, cp, session, seuil):
    """Enrobage ADDITIF. PRIORITÉ ABSOLUE : alias SIREN vérifié (organisations
    internationales + corrections manuelles du rapport) — force le bon SIREN/TVA
    même si la recherche normale trouverait autre chose (ex. « SCI ACTION CONTRE
    LA FAIM »). Sinon, recherche normale sur le nom d'origine (parité 87%), puis
    replis."""
    # 0) PRIORITÉ : alias SIREN vérifié (force le bon résultat)
    _sa0 = siren_alias(nom)
    if _sa0:
        try:
            _info0 = infos_par_siren(_sa0, session)
        except Exception:
            _info0 = {}
        _nom_officiel0 = _NOM_OFFICIEL_PAR_SIREN.get(_sa0) or _info0.get("nom_api") or _ALIAS_SIREN.get(_sa0, [nom])[0]
        return {"statut": "TROUVE", "strategie": "ALIAS_SIREN", "siren": _sa0,
                "tva": _siren_vers_tva(_sa0), "nom_api": _nom_officiel0,
                "score": 100, "detail": "SIREN vérifié (alias / correction manuelle)",
                "siret": _info0.get("siret", ""), "forme_juridique": _info0.get("forme_juridique", ""),
                "code_naf": _info0.get("code_naf", ""), "etat": _info0.get("etat", ""),
                "adresse_api": _info0.get("adresse_api", ""),
                "niveau_i": _info0.get("niveau_i", ""), "niveau_ii": _info0.get("niveau_ii", ""),
                "niveau_iii": _info0.get("niveau_iii", "")}
    # 1) Recherche normale sur le nom d'origine (identique au moteur 87%)
    res = rechercher_tva(nom, adresse, ville, cp, session, seuil)
    if res.get("statut") == "TROUVE" and res.get("tva"):
        return _confirmer_exactitude(nom, res, cp, session) if CONFIRMER_TROUVES else res
    # 2) Repli : organisation internationale à SIREN vérifié
    _sa = siren_alias(nom)
    if _sa:
        try:
            _info = infos_par_siren(_sa, session)
        except Exception:
            _info = {}
        return {"statut": "TROUVE", "strategie": "ALIAS_SIREN", "siren": _sa,
                "tva": _siren_vers_tva(_sa), "nom_api": _NOM_OFFICIEL_PAR_SIREN.get(_sa) or _info.get("nom_api") or _ALIAS_SIREN.get(_sa, [nom])[0], "score": 100,
                "detail": "alias organisation internationale (SIREN vérifié)",
                "siret": _info.get("siret", ""), "forme_juridique": _info.get("forme_juridique", ""),
                "code_naf": _info.get("code_naf", ""), "etat": _info.get("etat", ""),
                "adresse_api": _info.get("adresse_api", ""),
                "niveau_i": _info.get("niveau_i", ""), "niveau_ii": _info.get("niveau_ii", ""),
                "niveau_iii": _info.get("niveau_iii", "")}
    # 3) Repli : nom corrigé (décollage + coquilles) si différent de l'original
    nom2 = corriger_nom(nom)
    if nom2 != nom:
        res2 = rechercher_tva(nom2, adresse, ville, cp, session, seuil)
        if res2.get("statut") == "TROUVE" and res2.get("tva"):
            res2["detail"] = (res2.get("detail", "") + " | via nom corrigé").strip(" |")
            return _confirmer_exactitude(nom, res2, cp, session) if CONFIRMER_TROUVES else res2
    # 4) Repli contrôlé : reprendre le MEILLEUR candidat SIRENE déjà trouvé
    #    (mais rejeté car score composite < seuil) UNIQUEMENT si son nom
    #    correspond très bien au bénéficiaire -> récupère les vraies entités
    #    sans réintroduire les faux positifs type « SCI ACTION CONTRE LA FAIM ».
    res = _confirmer_exactitude(nom, res, cp, session)
    if res.get("statut") == "TROUVE" and res.get("tva"):
        return res
    rep = _repli_meilleur_candidat(nom, res)
    if rep is not None:
        return rep
    return res


_rechercher_tva_plus_base = rechercher_tva_plus   # v5_7

SIREN_FRANCE_TRAVAIL = "130005481"   # v5_8 : SIREN officiel vérifié (EPA, ex-Pôle emploi)

def _est_france_travail(nom):
    _n = re.sub(r"\s+", "", _norm_alias(nom))   # _norm_alias garde des espaces
    return "POLEEMPLOI" in _n or "FRANCETRAVAIL" in _n

# v5_8 : bénéficiaires pour lesquels AUCUNE TVA ne doit jamais être attribuée
# (structure non identifiable — décision utilisateur). Jamais « A CHERCHER ».
_BENEFICIAIRES_SANS_TVA_PREFIXES = ("REPUBLIQUEFRANCAISE",)

def _beneficiaire_sans_tva(nom):
    _n = re.sub(r"\s+", "", _norm_alias(nom))   # _norm_alias garde des espaces
    return any(_n.startswith(_p) for _p in _BENEFICIAIRES_SANS_TVA_PREFIXES)

def rechercher_tva_plus(nom, adresse, ville, cp, session, seuil):
    """v5_7/v5_8 — enveloppe finale :
    1. bénéficiaires « sans TVA » (République française…) : jamais de TVA, aucune
       recherche (statut EXCLU documenté au rapport) ;
    2. Pôle emploi / France Travail : recherché sous « FRANCE TRAVAIL » AVEC
       l'adresse/ville/CP de la ligne -> retrouve l'ÉTABLISSEMENT local (SIRET) ;
       SIREN garanti 130005481 ; repli = siège France Travail ;
    3. faux positifs confirmés (correction_1) définitivement écartés."""
    # 1) v5_8 : structures non identifiables -> jamais de TVA, pas d'appel API
    if _beneficiaire_sans_tva(nom):
        return {"statut": "EXCLU", "strategie": "SANS_TVA_MANUEL", "siren": "",
                "tva": "", "nom_api": "", "score": 0, "siret": "", "forme_juridique": "",
                "code_naf": "", "etat": "", "adresse_api": "",
                "niveau_i": "", "niveau_ii": "", "niveau_iii": "",
                "detail": "structure non identifiable (décision utilisateur) : TVA jamais attribuée"}
    # 2) v5_8 : Pôle emploi -> FRANCE TRAVAIL, localisé par l'adresse de la ligne
    if _est_france_travail(nom):
        res_ft = _rechercher_tva_plus_base("FRANCE TRAVAIL", adresse, ville, cp, session, seuil)
        if res_ft.get("statut") == "TROUVE" and str(res_ft.get("siren", "")) == SIREN_FRANCE_TRAVAIL:
            return {**res_ft, "strategie": "ALIAS_SIREN", "score": 100,
                    "nom_api": "FRANCE TRAVAIL",
                    "detail": "Pôle emploi -> FRANCE TRAVAIL (établissement localisé par l'adresse)"}
        try:
            _ift = fiche_siren(SIREN_FRANCE_TRAVAIL, session)
        except Exception:
            _ift = {}
        return {"statut": "TROUVE", "strategie": "ALIAS_SIREN", "siren": SIREN_FRANCE_TRAVAIL,
                "tva": _siren_vers_tva(SIREN_FRANCE_TRAVAIL), "nom_api": "FRANCE TRAVAIL",
                "score": 100, "detail": "Pôle emploi -> FRANCE TRAVAIL (repli : siège, SIREN vérifié)",
                "siret": _ift.get("siret", ""), "forme_juridique": _ift.get("forme_juridique", ""),
                "code_naf": _ift.get("code_naf", ""), "etat": _ift.get("etat", ""),
                "adresse_api": _ift.get("adresse_api", ""),
                "niveau_i": _ift.get("niveau_i", ""), "niveau_ii": _ift.get("niveau_ii", ""),
                "niveau_iii": _ift.get("niveau_iii", "")}
    # 3) v5_7 : recherche normale + faux positifs confirmés écartés
    res = _rechercher_tva_plus_base(nom, adresse, ville, cp, session, seuil)
    _int = _TVA_INTERDITES.get(_norm_alias(nom))
    if _int and str(res.get("siren", "")) in _int:
        return {**res, "statut": "NON_TROUVE", "strategie": "FAUX_POSITIF_ECARTE",
                "siren": "", "tva": "", "siret": "", "nom_api": "", "score": 0,
                "forme_juridique": "", "code_naf": "", "etat": "", "adresse_api": "",
                "niveau_i": "", "niveau_ii": "", "niveau_iii": "",
                "detail": "TVA écartée définitivement : faux positif confirmé (corrections rapport)"}
    # v5_16 : SIRET d'établissement FORCÉ (corrections « CORRIGE SIRET ») — remplace
    # le SIRET du siège quand le SIREN correspond bien à celui vérifié.
    _sf = _SIRET_FORCE.get(_norm_alias(nom))
    if _sf and str(res.get("siren", "")) == _sf[:9]:
        res = {**res, "siret": _sf,
               "detail": (res.get("detail", "") + " | SIRET établissement forcé (correction vérifiée)").strip(" |")}
    return res

# Seuil d'exactitude du nom pour accepter le meilleur candidat sous le seuil global.
SEUIL_EXACTITUDE_REPLI = 90

def _repli_meilleur_candidat(nom, res):
    """Accepte le meilleur candidat SIRENE conservé (_cand_*) si l'exactitude
    du nom >= SEUIL_EXACTITUDE_REPLI. Renvoie un résultat TROUVE, sinon None."""
    if not res or not res.get("_cand_siren") or not res.get("_cand_tva"):
        return None
    ex = _exactitude(nom, res.get("_cand_nom_api", ""))
    if ex < SEUIL_EXACTITUDE_REPLI:
        return None
    return {"statut": "TROUVE", "strategie": "REPLI_EXACT",
            "siren": res["_cand_siren"], "tva": res["_cand_tva"],
            "nom_api": res.get("_cand_nom_api", ""), "score": res.get("_cand_score", 0),
            "detail": f"meilleur candidat SIRENE existant, nom exact {ex}% (repli)",
            "siret": res.get("_cand_siret", ""),
            "forme_juridique": res.get("_cand_forme_juridique", ""),
            "code_naf": res.get("_cand_code_naf", ""), "etat": res.get("_cand_etat", ""),
            "adresse_api": res.get("_cand_adresse_api", ""),
            "niveau_i": res.get("_cand_niveau_i", ""), "niveau_ii": res.get("_cand_niveau_ii", ""),
            "niveau_iii": res.get("_cand_niveau_iii", "")}


print("Moteur v3 (fusion V1+V2) chargé")
print("  Requetes : type-based (C/A/E) + generiques (G) en repli")
print("  Scoring  : 5 criteres + adresse granulaire + radiee douce")
_demo = _generer_requetes("FEDERATION NATIONALE DE NATATION*FNN", cp="31000", ville="TOULOUSE")
print(f"  Demo : {len(_demo)} requetes generees pour une federation")


  Formes juridiques : téléchargement indisponible -> table de secours intégrée
Moteur v3 (fusion V1+V2) chargé
  Requetes : type-based (C/A/E) + generiques (G) en repli
  Scoring  : 5 criteres + adresse granulaire + radiee douce
  Demo : 10 requetes generees pour une federation


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5 — ÉTAPE A : L'ENRICHISSEMENT TVA (cœur opérationnel)
# ═══════════════════════════════════════════════════════════════
# LES DEUX PASSES, À BIEN DISTINGUER :
#   • PASSE 1 — lignes SANS TVA dans le FTS : on CHERCHE (appels API, moteur
#     figé). C'est la partie longue et coûteuse du traitement.
#   • PASSE 2 — lignes AVEC une TVA déjà fournie par la Commission : on ne
#     cherche RIEN. On déduit le SIREN de cette TVA et on va simplement
#     DOCUMENTER l'entité (SIRET, forme juridique, NAF, état).
# Principe : on ne remet jamais en cause une TVA fournie par la Commission,
# on l'enrichit seulement (non-destruction de la donnée source).

# Seuil de rétention des TVA DANS LE FICHIER (le rapport garde TOUT).
# Une TVA trouvée avec un score < SEUIL_FICHIER_TVA est CONSERVÉE dans le rapport
# mais EFFACÉE du fichier FTS (avec ses colonnes entreprise), car sous ce seuil le
# risque de faux positif est élevé. La v3_16 (nom parfait -> pénalités géo
# neutralisées) garantit que les vraies TVA au nom exact atteignent 95+.
SEUIL_FICHIER_TVA = 96   # v5_7 : seules les TVA de score >= 96 vont dans le fichier

# ═══════════════════════════════════════════════════════════════
# CELLULE 4 — Détection des colonnes + enrichissement France
# ═══════════════════════════════════════════════════════════════

# v5_30 -- Valeurs de la colonne « Main registration number of beneficiary »
# à traiter comme VIDES (mêmes conventions que le FTS pour les champs masqués,
# + les libellés spécifiques à cette colonne observés dans l'export 2025).
_MAIN_REG_VIDE = {"", "-", ".", "N/A", "N/A - NOT APPLICABLE", "NONE", "NAN"}

def _main_registration_valide(v):
    """Nettoie et valide la valeur de « Main registration number of beneficiary ».
    Renvoie un SIREN à 9 chiffres si -- et SEULEMENT si -- la valeur ressemble à
    un SIREN français ET passe la clé de Luhn ; chaîne vide sinon.

    ATTENTION -- cette colonne est un champ GÉNÉRIQUE européen : chaque pays y
    inscrit son propre numéro de registre national, dans SON PROPRE format
    (Allemagne : « VR7795 » -- Vereinsregister : Belgique : numéro d'entreprise
    à 10 chiffres ; Italie/Espagne : codes provinciaux hétérogènes). Elle n'est
    donc exploitable QUE pour les lignes dont le pays est la France -- cette
    fonction ne fait que le nettoyage/la validation du format ; c'est à
    l'appelant de restreindre l'usage aux lignes françaises (mask_fr).
    """
    s = str(v or "").strip().upper()
    if s in _MAIN_REG_VIDE:
        return ""
    chiffres = re.sub(r"\D", "", s)
    if len(chiffres) == 14:       # SIRET fourni par erreur -> les 9 premiers = SIREN
        chiffres = chiffres[:9]
    if len(chiffres) != 9:
        return ""
    # clé de Luhn (même contrôle que partout ailleurs dans le pipeline)
    total = 0
    for i, ch in enumerate(reversed(chiffres)):
        d = int(ch)
        if i % 2 == 1:
            d *= 2
            if d > 9: d -= 9
        total += d
    return chiffres if total % 10 == 0 else ""


def detecter_colonnes(df: pd.DataFrame) -> dict:
    """Retrouve les colonnes utiles du FTS par MOTS-CLÉS, pas par nom exact.

    Pourquoi : d'un export à l'autre, les en-têtes changent légèrement
    (« VAT number of beneficiary », « VAT_number », « Numéro TVA »...).
    Chercher une sous-chaîne insensible à la casse évite de casser le pipeline
    à chaque nouvelle livraison de la Commission.

    Renvoie {rôle: nom réel de la colonne}. Une valeur peut être None si la
    colonne est absente : chaque étape en aval teste donc toujours la présence
    avant de s'en servir (`if col_x:`).
    """
    def trouver(*mots):
        for m in mots:
            for c in df.columns:
                if m.lower() in str(c).lower():
                    return c
        return None

    cols = {
        "tva":     trouver("vat number", "vat_number", "tva", "numero tva"),
        "nom":     trouver("name of beneficiary", "beneficiary name", "nom benefi", "nom_benef", "nom unique"),
        "pays":    trouver("beneficiary country", "country", "pays"),
        "adresse": trouver("address", "adresse", "street", "rue", "beneficiary address"),
        "cp":      trouver("postal code", "code postal", "zip", "postcode"),
        "ville":   trouver("city", "ville", "locality", "localite"),
        # v5_30 : nouvelle colonne de l'export FTS 2025 -- le SIREN (ou
        # équivalent national pour les autres pays) est parfois DÉJÀ fourni
        # par la Commission. Absente des exports antérieurs -> None, ignorée
        # proprement partout où elle est utilisée (compatibilité ascendante).
        "main_reg": trouver("main registration"),
    }
    manquantes = [k for k in ("tva", "nom", "pays") if not cols[k]]
    if manquantes:
        raise ValueError(
            f"Colonnes obligatoires introuvables : {manquantes}\n"
            f"Colonnes disponibles : {list(df.columns)}"
        )
    print("Colonnes détectées :")
    for k, v in cols.items():
        print(f"  {k:10s} : {v if v else '(absent — optionnel)'}")
    return cols


def _montant_num(v):
    """Convertit un montant FTS en nombre exploitable.

    Le FTS mélange les conventions : « 1 234,56 », « 1,234.56 », « 1234.56 »,
    parfois avec des espaces insécables. On ramène tout au float Python.
    Renvoie 0.0 si la valeur est illisible — JAMAIS d'exception : un montant
    mal formé ne doit pas interrompre un traitement de 100 000 lignes.
    """
    """Parse un montant FTS en float ('653837.93', '1 234,56'...)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 0.0
    s = str(v).replace("\xa0", "").replace(" ", "")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif "," in s:
        s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    try:
        return float(s)
    except ValueError:
        return 0.0

# ═══════════════════════════════════════════════════════════════
# v5_31 — MÉMOIRE DE CORRESPONDANCE inter-millésimes (nom + département -> SIREN)
# ═══════════════════════════════════════════════════════════════
# Le fichier FTS d'une année donnée ne contient QUE cette année (vérifié :
# l'export 2025 ne contient aucune autre valeur de "Year"). Le SIREN fourni
# par la Commission via "Main registration number of beneficiary" (Passe 0)
# n'existe que pour 2025+ : cette mémoire permet de RÉUTILISER ces SIREN
# certains pour les MILLÉSIMES ANTÉRIEURS (2014-2024), qui ne contiennent pas
# cette colonne -- sans refaire un appel de recherche pour un bénéficiaire
# déjà résolu de façon fiable une autre année.
#
# Clé de correspondance = (nom normalisé, DÉPARTEMENT déduit du code postal),
# jamais le nom seul : un même nom peut désigner plusieurs implantations
# RÉELLEMENT DISTINCTES (ex. une chambre de commerce régionale) -- le
# département sert de désambiguïsation minimale, cohérente avec le reste du
# pipeline (aucune promesse d'unicité au-delà : en cas d'ambiguïté résiduelle,
# on préfère ne rien affirmer et laisser la ligne repartir en recherche).
CORRESPONDANCE_SIREN_MEMOIRE = {}   # (nom_norm, dept) -> siren

# v5_37 — SOCLE EMBARQUÉ (mécanisme hybride, comme les autres tables du pipeline) :
# les correspondances FIABLES déjà établies (SIREN Commission « Main Registration »
# + TVA fournies, millésime 2025) sont intégrées EN DUR ci-dessous — le notebook
# fonctionne donc même sans déposer correspondance_siren.json. Si le fichier est
# déposé malgré tout, il se FUSIONNE par-dessus (ses entrées gagnent en cas de
# doublon : il est plus récent que ce socle). Le socle est à régénérer à chaque
# livraison depuis le dernier correspondance_siren.json fourni par l'utilisateur.
_CORRESPONDANCE_SIREN_SOCLE = r'''{"GIP FCIP ACADEMIE D AIX MARSEILLE||13":"181337130","GIP FORMATION ET INSERTION PROFESSI NNELLE DE L ACADEMIE DE NICE||06":"180619199","LYCEE POLYVALENT VAUVENARGUES||13":"191332063","3IS||13":"818428104","ACCORD RELATIF AUX PECHES DANS LE S UD DE L OCEAN INDIEN||974":"823504279","ACTED||75":"402886816","ACTION CONTRE LA FAIM||93":"318990892","AGENCE BRETAGNE NEXT||35":"532239472","AGENCE DE DEVELOPPEMENT ECONOMIQUE DE LA CORSE||2A":"392175568","AGENCE DE DEVELOPPEMENT ET D INNOVA TION AQUITAINE LIMOUSIN POITOU CHAR||33":"820336733","AGENCE DU SERVICE CIVIQUE||75":"130011844","AGENCE ERASMUS FRANCE EDUCATION FORMATION||33":"187512512","AGENCE REGIONALE DE DEVELOPPEMENT E T D INNOVATION HAUTS DE FRANCE||59":"390370377","AGIR ENSEMBLE POUR LES DROITS HUMAI NS||69":"393347380","ALCIME ASSOCIATION||13":"428789986","ALDA ASSOCIATION EUROPEENNE POUR LA DEMOCRATIE LOCALE||67":"442144002","ALTERNATIVES EUROPEENNES ASSOCIATIO N||75":"532871449","ARTCENA||75":"388948309","ASSOCIATION D INFORMATION SUR LE LO GEMENT DU PUY DE DOME||63":"325634715","ASSOCIATION DEPARTEMENTALE POUR L I NFORMATION SUR LE LOGEMENT ADIL 9||92":"387758824","ASSOCIATION DU FESTIVAL INTERNATION AL DES SERIES DE LILLE HAUTS DE FRA||59":"833393044","ASSOCIATION DU LYCEE SAINTE MARIE D E NEVERS||31":"776944282","ASSOCIATION DU RESEAU DES PRESIDENT S DES COURS SUPREMES JUDICIAIRES DE||75":"454019779","ASSOCIATION EUROPA CINEMAS||75":"383893443","ASSOCIATION FEDERATION HANDICAP INT ERNATIONAL||69":"519655997","ASSOCIATION GROUPE URGENCE REHABILI TATION DEVELOPPEMENT||26":"424079622","ASSOCIATION MEDECINS DU MONDE||93":"321018749","ASSOCIATION MIGRATION SOLIDARITE ECHANGE POUR LE DEVELOPPEMENT||67":"422915116","ASSOCIATION POUR LE DEVELOPPEMENT D ES INITIATIVES CITOYENNES ET EUROPE||59":"424867067","ASSOCIATION RESEAU EUROPEEN DE MUSI QUE ANCIENNE||01":"442782041","CARE FRANCE ASSOCIATION||93":"334805801","CENTRE FOR ECONOMIC POLICY RESEARCH||75":"898411806","CENTRE HOSPITALIER DE VALENCIENNES||59":"265906735","CENTRE HOSPITALIER NATIONAL D OPHTA LMOLOGIE DES QUINZE VINGTS||75":"180036014","CENTRE INTERNAT RECH CANCER||69":"779925684","CENTRE INTERNATIONAL DE FORMATION E UROPEENNE||":"784412546","CENTRE INTERNATIONAL DE RECHERCHE A UX FRONTIERES DE LA CHIMIE FONDATIO||67":"499246601","CHAMBRE DE COMMERCE ET D INDUSTRIE DE LA REUNION||974":"189742117","CHAMBRE DE COMMERCE ET D INDUSTRIE DE REGION DES ILES DE GUADELOUPE||971":"130014087","CHAMBRE DE COMMERCE ET D INDUSTRIE DE LA MARTINIQUE||":"189720022","CHAMBRE REGIONALE D AGRICULTURE OCC ITANIE||31":"130021603","CHILDREN OF PRISONERS EUROPE||75":"437527013","CIRCUSNEXT||75":"495387813","CIVISME ET DEMOCRATIE OMO EDUC CIV SOC CIVISME DEMOCRAT||75":"401916622","COLLECTIVITE DE SAINT BARTHELEMY||977":"219711231","COLLEGE DEPARTEMENTAL ALBERT THOMAS||19":"191900190","COLLEGE LE PETIT MAIRAT||16":"191600311","COMITE CONSULTATIF REGIONAL DES EAU X OCCIDENTALES SUD ASSOCIATION||56":"499722643","COMITE DE JUMELAGE LES ANCIZES COMP S SAINT GEORGES DE MONS||63":"539401398","COMITE POUR LES RELATIONS NATIONALE S ET INTERNATIONALES DES ASSOCIATIO||75":"784579641","COMMUNAUTE DE COMMUNES SAONE BEAUJO LAIS||69":"200067817","COMMUNE DE PLOUER SUR RANCE||22":"212202139","CONFEDERATION INTERNATIONALE DES C INEMAS D ART ET D ESSAI ASSOCIATION||75":"380516120","CONSEIL INTERNATIONAL MUSEES ATIONAL COUNCIL OF MUSEUMS||75":"784617813","CONSERVATOIRE RHONE ALPES DES ESPAC ES NATURELS ASSOCIATION||69":"398534222","ENERGIE PARTAGEE||75":"529402406","ENOC RESEAU EUROPEEN DES OMBUDSMANS POUR ENFANTS||67":"508460151","EURODOC||75":"422611905","EUROPEAN SYNCHROTRON RADIATION FACI LITY||38":"338723919","EXPERTISE FRANCE||75":"808734792","FEDERATION DEPARTEMENTALE DES CENTR E D INITIATIVES POUR VALORISER L AG||":"401534672","FEDORA THE EUROPEAN CIRCLE OF PHI LANTHROPISTS OF OPERA BALLET||75":"808924708","FONDATION FRANCAISE POUR LA RECHERC HE SUR LA BIODIVERSITE||75":"508175627","FORUM CIVIQUE EUROPEEN ASSOCIATION FCE EUROPEAN CIVIC FORUM||75":"502004377","FRANCE EDUCATION INTERNATIONAL||92":"180043069","GRAND EST DEVELOPPEMENT||68":"434049953","GROUPE DES ECOLES NATIONALES D ECON OMIE ET STATISTIQUE||91":"130014228","GROUPE SOS PULSE||75":"500176896","HABITAT ET HUMANISME AUVERGNE||63":"508742202","INCUBATEUR D ENTREPRISES INNOVANTES INIZIA||2B":"798482097","INITIATIVE DEVELOPPEMENT||86":"394598023","INSTITUT DES HAUTES ETUDES SCIENTIF IQUES FONDATION||91":"775688252","INSTITUT NATIONAL DE LA STATISTIQUE ET DES ETUDES ECONOMIQUES||92":"120027016","INTERNATIONAL CRIMINAL POLICE ORGAN IZATION||69":"785447467","LA CHAINE DE L ESPOIR||75":"399818418","LA COMMUNAUTE D UNIVERSITES ET ETABLISSE MENTS DE TOULOUSE||31":"130021322","LA POUDRIERE ECOLE DU FILM D ANIM ATION||26":"417767753","LA REUNION CONNECTEE||974":"842430878","LA REUNION INNOVATION||974":"924151111","LE GROUPE OUEST||29":"490274594","LIGUE POUR LA PROTECTION DES OISEAU X AUVERGNE RHONE ALPES||69":"301125100","LINKING INITIATIVES AND VENUES IN E UROPE DEVELOPING MUSICAL ACTIONS||44":"789708385","LYCEE PROFESSIONNEL AGRICOLE HAUTE SOMME||80":"198013286","LYCEE PROFESSIONNEL MADELEINE VIONN ET LYCEE DES METIERS DE LA GESTIO||93":"199301292","LYCEE PROFESSIONNEL SONIA DELAUNAY||59":"195901111","MAIRIE DE L ETANG LA VILLE||78":"217802248","MAIRIE DE MARCOUSSIS||91":"219103637","MAISON FAMILIALE RURALE DE L OUEST LYONNAIS||69":"302931431","MAISON NATURE ENVIRONNEMENT||30":"401259056","MONTLOUIS SUR LOIRE||37":"213701568","OGEC COLLEGE SAINT JOSEPH PLOUESCAT||29":"450136734","OGEC DE LE GENEST ISLE||53":"308856202","OGEC ENSEMBLE SCOLAIRE NIORTAIS||79":"399741321","OGEC SAINT DOMINIQUE||60":"780557997","ORGANISATION EUROPEENNE POUR L EQUI PEMENT DE L AVIATION CIVILE EUROCAE||93":"397944596","ORGANISATION INTERNATIONALE DE LA V IGNE ET DU VIN||75":"784354615","ORGANISME DE GESTION DU LYCEE PRIVE SAINT JACQUES DE COMPOSTELLE AGEEF||86":"300726312","PISTES SOLIDAIRES||64":"448858704","PREMIERE URGENCE INTERNATIONALE||92":"531199974","PROMEDIATION||75":"811590298","PROMOTION DU POISSON DES ETANGS DE LA DOMBES||":"452030372","RELIEF INTERNATIONAL FRANCE||92":"429565252","RESEAU DES PROCUREURS GENERAUX OU I NSTITUTIONS EQUIVALENTES PRES LES C||75":"929242071","RESEAU EXPRESS JEUNES NETWORK||67":"417564614","RESEAU FRANCAIS DES INSTITUTS D ETU DES AVANCEES FONDATION||69":"500383997","RISINGSUD||13":"799551189","SDSN ASSOCIATION PARIS||75":"821261237","SOLIDARITES INTERNATIONAL||92":"389515180","SOLIHA LOIRE PUY DE DOME SOLIDAIR ES POUR L HABITAT||":"776398737","SYNDICAT DES RIVIERES DOMBES CHAL ARONNE BORDS DE SAONE||":"200013290","SYNDICAT MIXTE DES GORGES DU GARDON||30":"253002489","SYNDICAT MIXTE VEYLE VIVANTE||":"250102431","THALES||92":"552059024","THE ALLIANCE FOR INTERNATIONAL MEDI CAL ACTION||75":"831620398","THE INTERNATIONAL HUMAN FRONTIER SCIENCE PROGRAM ORGANISATION||67":"378265136","TOURNONS LA PAGE TLP||75":"883775405","TRIANGLE GENERATION HUMANITAIRE||69":"408856672","UNION EUROPEENNE DES AVEUGLES EAN BLIND UNION||75":"390171460","UNIVERSITE RENNES II||35":"193509379","UNIVERSITE SAVOIE MONT BLANC||73":"197308588","VISION DU MONDE WORLD VISION ASSOCI ATION||75":"449349026","VOIES NAVIGABLES DE FRANCE||":"130017791","MINISTERE DE L EUROPE ET DES AFFAIRES ETRANGERES||75":"110006012","EUROPEAN UNION AGENCY FOR RAILWAYS||59":"913098802","UNIVERSITE DES ANTILLES||971":"199715855","AD NORMANDIE||14":"200006500","CHAMBRE DE COMMERCE ET D INDUSTRIE DE NOUVELLE CALEDONIE||988":"130029838","DEV UP CENTRE VAL DE LOIRE||45":"405047572","UN MONDE MIGRANT||":"889057196","ECOLE NATIONALE SUPERIEURE DES MINE S DE PARIS||75":"197534936","AGENCE DE L ENVIRONNEMENT ET DE LA MAITRISE DE L ENERGIE||49":"385290309","ECOLE CENTRALE DE LYON||69":"196901870","DEPARTEMENT DE L AUDE||11":"221100019","ASSOCIATION MEDITATION LAIQUE POUR L EDUCATION||31":"811573179","BREST METROPOLE||29":"242900314","AGENCE NATIONALE DE LA RECHERCHE||75":"130002504","FONDATION EUROPEENNE DE LA SCIENCE ASSOCIATION FES||67":"302493986","PIANOCEAN||26":"799554696","MARENA SERVICES ET FORMATION||13":"999389323","VIRAGE ENERGIE||59":"494529613","COMMUNE DE BESANCON||25":"212500565","GCS HOPITAUX UNIVERSITAIRES GRAND O UEST||49":"130018328","UNIVERSITE DE LA REUNION||977":"199744780","INSTITUT NATIONAL DES SCIENCES APPL IQUEES DE TOULOUSE||31":"193101524","COMMUNE DE LUZ SAINT SAUVEUR||":"216502955","THE GREEN ROOM ARTS ECO RESPONSABIL ITE ET PARTICIPATION||50":"841746670","EUROPEAN CENTER FOR HUMAN RIGHTS CE NTRE EUROPEEN DES DROITS DE L HOMME||67":"890339807","LYCEE GENERAL ET TECHNOLOGIQUE LAFA YETTE||63":"196300214","CHAMBRE DE COMMERCE ET D INDUSTRIE DU GERS||32":"183200013","ASSOCIATION STIVALIENNE ET RAONNAIS E DE TENNIS DE TABLE||88":"752162305","CENTRE D ACTION ET DE PREVENTION CO NTRE LA RADICALISATION DES INDIVIDU||33":"813653052","FORUM REFUGIES||69":"326922879","FEDERATION INTERNATIONALE DES COMMU NAUTES DE L ARCHE||60":"434914370","COLLECTIVITE TERRITORIALE DE MARTIN IQUE||972":"200055507","JB RIDE||46":"753953546","ASSOCIATION INTERCULTURA||22":"440645745","UNION REGIONALE DES ACTEURS LOCAUX DE L EUROPE EN AUVERGNE RHONE ALPES||":"843665795","TEMPLARS ROUTE EUROPEAN FEDERATION||10":"828091942","LYON TANGO FESTIVAL||69":"909337032","IFP ENERGIES NOUVELLES||92":"775729155","ASSOCIATION NATIONALE INTERPROFESSI ONNELLE DU BETAIL ET DES VIANDES||75":"378355929","FEDERATION FRANCAISE DE SKI||74":"775691603","ORGANISME DE GESTION DE L ENSEIGNEM ENT CATHOLIQUE D ARGENTAN||61":"440621811","ESCRIME PAYS DE LUNEL||34":"824307060","GROUPEMENT D INTERET PUBLIC FORMATI ON TOUT AU LONG DE LA VIE||54":"185422136","ASSOCIATION POUR LE SOUTIEN A LA CI TOYENNETE EUROPEENNE||972":"991822628","CHU DE NANTES||44":"264400136","MAISON DE LEUROPE DES LANDES WIPSEE||40":"852307032","KULTUROVA||64":"932723976","CONFERENCE DES DIRECTEURS DES ECOLE S FRANCAISES DINGENIEURS COMMISSION||75":"518147202","HAUT CONSEIL DE L EVALUATION DE LA RECHERCHE ET DE L ENSEIGNEMENT SUPE||75":"130003494","EURASIA NET||13":"804808947","FONDATION JEAN JACQUES LAFFONT TOUL OUSE SCIENCES ECONOMIQUES||31":"494737976","COMMUNE D OLETTA||2B":"212001853","POMPIERS DE L URGENCE INTERNATIONAL E||87":"485014518","COMMUNE DE PAU||64":"216404459","REGION REUNION LA REUNION||978":"239740012","NEW NET RESEAU POUR LA FETE LE POUVOIR D AGIR ET LE BIEN ETRE||75":"795290840","CTRE INVESTIGAT PREVENT CLINIQUE ETOILE||75":"508741824","INSTITUT SINANO ASSOCIATION||38":"503562878","GIP CNFM GROUPEMENT POUR LA COORDINATION NATIONALE DE LA FORMATION ENMICROELECTRONIQUE ET EN NANOTECHNOLOGIES||38":"183830132","OBSERVATOIRE FRANCAIS DES DROGUES ET DES TOXICOMANIES GIP OFDT||93":"180036105","EPICENTRE||75":"340494905","FONDATION RAOUL FOLLEREAU||75":"784719510","INSTITUT FRANCAIS DES DROITS ET LIBERTES FRENCH INSTITUE OF RIGHTS AND FREEEDOMS||75":"482698982","SOITEC SA||38":"384711909","INTERNATIONAL CRIMINAL POLICE ORGANIZATION ORGANISATION INTERNATIONALE POLICE CRIMINELLE||69":"785447467","EUROPEAN ALLIANCE FOR THE SOCIAL SC IENCES AND THE HUMANITIES EASSH||75":"830189650","ARTICLE 1||75":"499381812","PARLAMENT EUROPEEN DES JEUNES FRANC E||75":"432681336","COMITE DES DONNEES SCIENTIFIQUES ET TECHNOLOGIQUES ASSOCIATION||75":"301272910","REUNIWATT||974":"518919345","CONSERVATOIRE DE L ESPACE LITTORAL ET DES RIVAGES LACUSTRES||17":"180005019","GROUPE ETUDE PROTECTION OISEAUX GUY ANE||973":"391711181","SOCIETE D ETUDES ORNITHOLOGIQUES DE LA REUNION||974":"412632184","ASS NALE ETUDIANT SCIENC TECH ACTIV ITE PHYS SPORT||92":"495207334","CONFERENCE DES DIRECTEURS D UNITES OU DE DEPARTEMENTS DE FORMATION ET||94":"433094323","COORDINATION FRANCAISE POUR LE LOBB Y EUROPEEN DES FEMMES||75":"411223779","ACOUCITE ASSOCIATION||69":"410118434","AGENCE DU CLIMAT LE GUICHET DES SO LUTIONS||67":"899818827","COMMUNAUTE D AGGLOMERATION DU NORD GRANDE TERRE||971":"200044691","ENERGIES 2050||06":"539215889","EUROMETROPOLE DE STRASBOURG||67":"246700488","I4CE INSTITUTE FOR CLIMATE ECONOM ICS||75":"500201983","GROUPE D INFORMATION ET DE SOUTIEN AUX TRAVAILLEURS IMMIGRES||75":"315131573","EUROPEAN NETWORK OF OUTDOOR SPORTS||":"844591941","FEDE FRANC RANDONNE PEDESTRE||94":"303588164","FEDERATION INTERNATIONALE DE TOURIS ME EQUESTRE||92":"819010240","GROUPE KEDGE BUSINESS SCHOOL||33":"514005123","ALLIANCE POUR LES TECHNOLOGIES DES LANGUES ATL EDIC||02":"929059178","MAISON DE L EUROPE DROME ARDECHE||26":"750956401","MAISON DES EUROPEENS LYON||69":"419875166","ASSOCIATION MILITANTS DES SAVOIRS DS||31":"752792978","FORUM EUROPEEN POUR LA SECURITE URB AINE ASSOCIATION||75":"380253740","AGENCE D URBANISME DE L AGGLOMERATI ON MARSEILLAISE||13":"782897607","LIEBHERR AEROSPACE TOULOUSE SAS||31":"552016834","ENTENTE POUR LA FORET MEDITERRANEEN NE||13":"200016012","FONDATION PARALYSIE CEREBRALE||75":"492500087","FIAT CANTUS||92":"800264905","FORUM VOIX ETOUFFEES||67":"481394328","LIGUE INTERNATIONALE CONTRE LE RACI SME ET L ANTISEMITISME SECTION DU B||67":"801214420","ORCHESTRE LES METAMORPHOSES||41":"889901518","SAFRAN POWER UNITS||31":"630800084","FONDS MONDIAL POUR LA NATURE FRANCE||93":"302518667","AGENCE DE L EAU RHIN MEUSE||57":"185703014","AGENCE DE L EAU SEINE NORMANDIE||92":"187500095","DES HOMMES ET DES ARBRES LES RACIN ES DE DEMAIN DHDA||54":"881631741","POUR UNE HYDROLOGIE REGENERATIVE||26":"920310976","SYNDICAT MIXTE DE L EAU DE L ASSAI NISSEMENT COLLECTIF DE L ASSAINISS||10":"200062107","REGION GRAND EST||67":"200052264","COORDINATION EUROPEENNE DES PRODUCT EURS INDEPENDANTS||75":"803779917","FEDERATION FRANCAISE DE HOCKEY SUR GAZON||92":"784406100","PRINCESSE MARGOT||94":"803390798","CONSEIL INTERNATIONAL DE LA MUSIQUE ASSOCIATION||75":"314246935","FUTBOL MAS FRANCE||75":"831522362","CONFEDERATION GENERALE DU TRAVAIL FORCE OUVRIERE||75":"784578247","COLLECTIF POUR UN SERVICE CIVIQUE EUROPEEN||75":"848873394","PLAY INTERNATIONAL||75":"424901916","SAFRAN LANDING SYSTEMS||78":"712019538","SYNDICAT NATIONAL DES BASKETTEURS||75":"352118186","ASSOCIATION PARTICIP ACTION||75":"803322395","OW2||75":"499409712","PLACE NETWORK||75":"825373897","INRAE TRANSFERT SAS||75":"433960762","INTERNATIONAL FRUIT AND VEGETABLE J UICE ASSOCIATION IFU||75":"389343971","AGENCE KARKADE||13":"924835010","FEDERATION FRANCAISE DE VOLLEY BALL||94":"784406126","GRAND ACCELERATEUR NATIONAL D IONS LOURDS GIE||14":"997888300","SMALL ISLANDS ORGANISATION||13":"825327117","FEDERATION NATIONALE SOLIDARITE FEM MES||75":"325347045","FAKEOFF||75":"838114551","ASSOCIATION DE LA PLATEFORME EUROPE NNE NERIS||92":"791372998","ATHLETIC CLUB BOULOGNE BILLANCOURT||92":"785307299","CERCLE DE L AVIRON DE LYON||69":"779672468","INTERPROFESSIONNELLE RHONE ALPES||69":"400276986","CONFEDERATION NATIONALE HANDICAP EMPLOI DES ORGANISMES DE PLACEMENT||75":"751984972","CONSEIL INTERPROFESSIONNEL DES VINS DU ROUSSILLON||66":"434341103","UNICANCER||75":"532834090","ELECTROCYCLE L ASSO D3E||75":"828961656","LYCEE PROF HOTELIER DE LARGENTIERE||07":"190700161","LYCEE PROFESSIONNEL GABRIEL PERI||94":"199401324","LYCEE PROFESSIONNEL PIERRE MENDES F RANCE||88":"198800138","FEDERATION FRANCAISE DE DANSE||75":"784622367","CARBONE FERTILE CENTRE NATIONAL D A GROECOLOGIE||32":"902240985","FEMMES ENTRAIDE ET AUTONOMIE||75":"751613092","CENTRE INTERNATIONAL DE HAUTES ETUD ES AGRONOMIQUES MEDITERRANEENNES||75":"331942672","AMICALE DU CAMP DE GURS||64":"448775213","MEMOIRES MUSICALES SANS FRONTIERE||64":"887835270","SUSTAINABLE FINANCE OBSERVATORY||75":"789179363","URBAMONDE FRANCE||75":"819110768","COMITE CONTRE L ESCLAVAGE MODERNE||75":"419367909","CONSEIL NATIONAL DES BARREAUX||75":"391576964","CONFERENCE DES DIRECTEURS DE SERVIC E UNIVERSITAIRE DE FORMATION CONTIN||75":"480994508","COMPAGNIE MORADI||44":"877510388","OFFICE DE TOURISME DU PAYS DE FALAI SE||":"852664374","SPORTLYANZ||75":"988352027","PARIS BASKET 18E||75":"447965948","ACCENTONIC||93":"539578674","ELISFA||94":"348963307","SAVOIR DEVENIR||75":"827497876","ETHNOART||75":"449920636","LES ANNEAUX DE LA MEMOIRE||44":"383446952","PAU CANOE KAYAK CLUB UNIVERSITAIRE||64":"379497944","RIGHT CLICK||75":"917647125","EURORDIS RARE DISEASES EUROPE||75":"413459066","EUROPEAN YOUTH CIRCUS ORGANISATION||93":"800714941","OFFICE NATIONAL DES ANCIENS COMBATT ANTS ET VICTIMES DE GUERRE||75":"180007015","RESEAU ACTION CLIMAT FRANCE||93":"422466201","GHETT UP||93":"822555397","ECOLE DE FORMATION PROFESSIONNELLE DES AVOCATS DES BARREAUX DU RESSORT||75":"300227279","ASSOCIATION SPORTIVE BANQUE DE FRAN CE||75":"302983796","ASS NUMERICAL FRANCOPHONE UNIVERSITE GLO BAL||75":"490492600","RESEAU FRANCAIS REGISTRES CANCER FR ANCIM||31":"449715556","ASSOCIATION KAMPOS SAINT DENIS ACAD EMIE FOOTBALL||93":"832374219","FONDATION INSTITUT DE RECHERCHE POU R LE DEVELOPPEMENT DURABLE ET LES R||75":"484781133","GROUPEMENT DES INDUSTRIELS FRANCAIS DE L ENERGIE NUCLEAIRE GIFEN||75":"842871220","ASSOCIATION EUROPEENNE DES EMPLOYEU RS DU SPORT||94":"480105261","COLLECTIF 50 50||75":"791441165","EUROPEAN WOMEN S AUDIOVISUAL NETWOR K||67":"792229783","SOLUTION SOLIDARITE INCLUSION||75":"850590357","FIFTY FIFTY||26":"852856889","SPORT ET CITOYENNETE 3S||49":"500180732","DELPHIS DEVELOPPEMENT ETUDES POUR L E LOGEMENT LA PROMOTION DE L HABITA||75":"352244107","AGENCE DES COMMUNICATIONS MOBILES O PERATIONNELLES DE SECURITE ET DE SE||92":"130030851","UNIS CITE||75":"398191569","COOLIVING||91":"890512338","DROP DE BETON 93||93":"790839351","CENTRES D ENTRAINEMENT AUX METHODES D EDUCATION ACTIVE ASSOCIATION||75":"775664634","ASSOCIATION POUR LA SANTE DES FEMME S D AFRIQUE DU MAGHREB ET DU MOYEN||93":"852279827","EONA X||75":"922984513","ETHIC OCEAN||93":"491411419","IEEE FRANCE SECTION ASSOCIATION||75":"443739792","ASSOCIATION POUR LA FORMATION LA P REVENTION ET L ACCES AU DROIT||93":"435121041","D2 DYNAMIQUE ET DEVELOPPEMENT||22":"893604850","FEDERATION DES TRAVAILLEURS DE LA M ETALLURGIE CGT||93":"784358962","CAMPUS DES METIERS ET DES QUALIFICA TIONS DES INDUSTRIES DE LA MER EN B||29":"831132618","INSTITUT TEXTILE ET CHIMIQUE DE LYO N||69":"320140551","EUROPEAN IMPLEMENTATION NETWORK||67":"827822750","DIGITAL RESEARCH INFRASTRUCTURE FOR THE ARTS AND HUMANITIES||75":"804996361","AFEJI HAUTS DE FRANCE||59":"304576218","LA FABULERIE||13":"528704117","ASSOCIATION EUROPEENNE POUR LA DEFE NSE DES DROITS ET DES LIBERTES||67":"897495370","PAYE TON PINARD||69":"985256858","SOLIDARITE FEMMES BEAUJOLAIS||69":"888723103","AGENCE NATIONALE POUR LA GESTION DE S DECHETS RADIOACTIFS ANDRA||92":"390199669","CHANTIERS ACTIONS CITOYENNETE ET IN CLUSION AURA PROJET ETUDES ET CHAN||63":"835331539","AUTORITE DE REGULATION DES TRANSPOR TS||75":"130012321","EPLEFPA LE VALENTIN||26":"192607653","ARCENCIELFRANCE||69":"512615659","CONFEDERATION DES PETITES ET MOYENN ES ENTREPRISES||92":"303215016","WINGS FOR PEACE||35":"923171573","ECOLE SUPERIEURE DES TECHNOLOGIES I NDUSTRIELLES AVANCEES ESTIA||64":"824457675","LAMINES MARCHANDS EUROPEENS SA LME||59":"568801013","ASSOCIATION PANAKUH||973":"800875130","SERVICE CIVIL INTERNATIONAL BRANCHE FRANCAISE||59":"784412462","INSTITUT NATIONAL DU CANCER GIP A||92":"187512777","CENTRE DE RESSOURCES D EXPERTISE ET DE PERFORMANCE SPORTIVES DE RHONE||":"130018914","PARCOURS LE MONDE SUD EST||":"817759004"}'''
for _cle_s, _siren_s in json.loads(_CORRESPONDANCE_SIREN_SOCLE).items():
    _nom_s, _dept_s = _cle_s.rsplit("||", 1)
    if _nom_s:   # jamais de clé au nom vide (v5_36)
        CORRESPONDANCE_SIREN_MEMOIRE[(_nom_s, _dept_s)] = _siren_s
print(f"Mémoire de correspondance : socle embarqué chargé "
      f"({len(CORRESPONDANCE_SIREN_MEMOIRE):,} entrées, millésime 2025)")

def _cle_correspondance(nom, cp):
    return (_normaliser(str(nom)), dept_depuis_cp(cp))

def _siren_memoire_nom_seul(nom_norm):
    """v5_36 — repli : si la clé exacte (nom, dept) est absente mais que le nom,
    à lui seul, ne correspond qu'à UN SEUL SIREN dans toute la mémoire, ce SIREN
    est réutilisé (cas réel observé sur le millésime 2023 : déménagement du
    siège entre deux années, ex. CARE FRANCE 93 -> 75 : 14 lignes perdues pour
    cette seule raison). S'il correspond à PLUSIEURS SIREN distincts (homonymes
    réels, ex. chambres consulaires régionales), on ne tranche pas : chaîne
    vide, la ligne repart en recherche normale — cohérent avec la philosophie
    du pipeline (en cas d'ambiguïté, ne rien affirmer)."""
    if not nom_norm:
        return ""
    sirens = {s for (n, _d), s in CORRESPONDANCE_SIREN_MEMOIRE.items() if n == nom_norm}
    return sirens.pop() if len(sirens) == 1 else ""

def construire_correspondance_siren(df_final, rapport, colonnes):
    """Construit la mémoire à partir d'un fichier déjà enrichi de façon FIABLE
    (Main Registration -- stratégie MAIN_REGISTRATION -- ou TVA déjà fournie
    par la Commission, passe 2) : jamais depuis un résultat de RECHERCHE
    (qui reste faillible), pour que cette mémoire ne propage aucune erreur."""
    col_nom, col_cp = colonnes["nom"], colonnes.get("cp")
    strategies_sures = {"MAIN_REGISTRATION", "TVA_EXISTANTE"}
    par_nom = {r["Nom_FTS"]: r for r in rapport if r.get("Strategie") in strategies_sures and r.get("SIREN")}
    n_ajoutes = 0
    for idx in df_final.index:
        nom = str(df_final.at[idx, col_nom]).strip()
        r = par_nom.get(nom)
        if not r:
            continue
        cp = str(df_final.at[idx, col_cp]).strip() if col_cp else ""
        cle = _cle_correspondance(nom, cp)
        if not cle[0]:
            continue   # v5_36 : jamais de clé au nom vide (bénéficiaire masqué/illisible)
        if cle not in CORRESPONDANCE_SIREN_MEMOIRE:
            CORRESPONDANCE_SIREN_MEMOIRE[cle] = r["SIREN"]
            n_ajoutes += 1
    print(f"Mémoire de correspondance : {n_ajoutes:,} nouvelle(s) entrée(s) "
          f"({len(CORRESPONDANCE_SIREN_MEMOIRE):,} au total en mémoire)")
    return CORRESPONDANCE_SIREN_MEMOIRE

def exporter_correspondance_siren(nom_fichier="correspondance_siren.json"):
    """Sauvegarde la mémoire pour la réutiliser lors d'un traitement ultérieur
    (un autre millésime, une autre session Colab)."""
    payload = {f"{nom}||{dept}": siren for (nom, dept), siren in CORRESPONDANCE_SIREN_MEMOIRE.items()}
    with open(nom_fichier, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    print(f"Mémoire exportée : {nom_fichier} ({len(payload):,} entrées) "
          f"-- à réutiliser via charger_correspondance_siren() sur un autre millésime.")

def charger_correspondance_siren(nom_fichier="correspondance_siren.json"):
    """Recharge une mémoire exportée précédemment (ex. issue du traitement du
    millésime 2025) avant de traiter un AUTRE millésime (2014-2024...)."""
    try:
        with open(nom_fichier, encoding="utf-8") as f:
            payload = json.load(f)
    except FileNotFoundError:
        print(f"Aucun fichier de mémoire trouvé ({nom_fichier}) -- démarrage à vide.")
        return CORRESPONDANCE_SIREN_MEMOIRE
    n_ignorees = 0
    for cle, siren in payload.items():
        nom, dept = cle.rsplit("||", 1)
        if not nom:
            n_ignorees += 1   # v5_36 : clé au nom vide (bénéficiaire masqué) -- jamais rechargée
            continue
        CORRESPONDANCE_SIREN_MEMOIRE[(nom, dept)] = siren
    print(f"Mémoire de correspondance rechargée : {len(payload) - n_ignorees:,} entrée(s) "
          f"({len(CORRESPONDANCE_SIREN_MEMOIRE):,} au total en mémoire"
          + (f" ; {n_ignorees} clé(s) au nom vide ignorée(s)" if n_ignorees else "") + ")")
    return CORRESPONDANCE_SIREN_MEMOIRE


# v5_32 -- DÉSAMBIGUÏSATION D'ÉTABLISSEMENT (SIRET) PAR CODE POSTAL, best-effort.
#
# LIMITE HONNÊTE À CONNAÎTRE : l'API recherche-entreprises.api.gouv.fr ne
# permet PAS de lister tous les établissements d'un SIREN (confirmé :
# combiner un SIREN/SIRET avec un filtre géographique fait IGNORER ce filtre
# silencieusement, d'après la documentation officielle de l'API elle-même ;
# et le champ matching_etablissements reste souvent vide même quand
# plusieurs établissements existent réellement). Il n'existe donc PAS de
# méthode garantie, via cette API gratuite et sans clé, pour retrouver à
# coup sûr le bon établissement d'un grand groupe multi-sites.
#
# Le contournement ci-dessous reste néanmoins légitime et documenté : un
# filtre géographique combiné à une recherche TEXTUELLE (nom), lui, est
# respecté (déjà utilisé par le moteur figé lui-même, voir _appeler_api). On
# cherche donc par nom + département, on ne retient QUE les candidats dont le
# SIREN correspond exactement à celui déjà connu, et on regarde si le siège
# ou un établissement listé correspond au code postal de la ligne FTS.
# Repli sûr et systématique sur le siège si rien de probant n'est trouvé --
# jamais d'établissement deviné.
def _etablissement_par_cp(siren, nom, cp, session):
    if not cp or not str(cp).strip() or not nom:
        return None
    dept = str(cp).strip()[:2]
    if dept[:2] in ("97", "98"):
        dept = str(cp).strip()[:3]
    if not dept.isdigit():
        return None
    try:
        time.sleep(DELAI_API)
        r = session.get(API_URL, params={"q": nom, "departement": dept, "page": 1, "per_page": 5}, timeout=12)
        r.raise_for_status()
        resultats = r.json().get("results", [])
    except Exception:
        return None
    cp_cible = re.sub(r"\D", "", str(cp))
    for res in resultats:
        if str(res.get("siren", "")).strip() != siren:
            continue   # ce n'est pas la bonne unité légale -- on ignore
        candidats = [res.get("siege") or {}] + list(res.get("matching_etablissements") or [])
        for etab in candidats:
            if re.sub(r"\D", "", str(etab.get("code_postal", ""))) == cp_cible and etab.get("siret"):
                return {"siret": etab["siret"],
                        "adresse_api": f"{etab.get('code_postal','')} {etab.get('libelle_commune','')}".strip()}
    return None   # rien de probant -- l'appelant garde le SIRET du siège (comportement inchangé)


def enrichir_france(df: pd.DataFrame, colonnes: dict, seuil: int = SEUIL_SCORE):
    global CACHE_RECHERCHE_TVA
    col_tva  = colonnes["tva"]
    # Montant total perçu par bénéficiaire (somme de « Beneficiary's contracted amount »)
    col_montant = next((c for c in df.columns
                        if "contracted amount" in str(c).lower()
                        and "beneficiary" in str(c).lower()
                        and "estimated" not in str(c).lower()
                        and "commitment" not in str(c).lower()), None)
    montant_par_benef = {}
    if col_montant:
        _t = df[[colonnes["nom"], col_montant]].copy()
        _t["_m"] = _t[col_montant].map(_montant_num)
        montant_par_benef = _t.groupby(_t[colonnes["nom"]].astype(str).str.strip())["_m"].sum().to_dict()
    col_nom  = colonnes["nom"]
    col_pays = colonnes["pays"]
    col_adr  = colonnes.get("adresse")
    col_cp   = colonnes.get("cp")
    col_vl   = colonnes.get("ville")

    mask_fr   = df[col_pays].astype(str).str.strip().str.lower().isin(PAYS_FR)
    mask_tvam = mask_fr & df[col_tva].apply(_tva_manquante)
    n_total   = int(mask_tvam.sum())

    # v5_30 -- PASSE 0 : le SIREN est parfois déjà fourni par la Commission
    # (colonne « Main registration number of beneficiary », export FTS 2025+).
    # Ces lignes ne passent PAS par le moteur de recherche : on documente
    # directement l'entité depuis son SIREN (comme la Passe 2), sans appel de
    # recherche. `mask_recherche` retire ces lignes du périmètre de la Passe 1.
    col_main_reg = colonnes.get("main_reg")
    if col_main_reg:
        mask_main_reg = (mask_tvam
                         & df[col_main_reg].apply(_main_registration_valide).astype(bool)
                         # règle intangible : les bénéficiaires "sans TVA possible" (ex.
                         # République française) n'en reçoivent JAMAIS, même si Main
                         # Registration contenait une valeur exploitable pour cette ligne.
                         & ~df[col_nom].apply(_beneficiaire_sans_tva))
    else:
        mask_main_reg = pd.Series(False, index=df.index)
    # v5_34 -- PROPAGATION INTRA-FICHIER : si un bénéficiaire a une ligne AVEC
    # Main Registration et d'autres lignes SANS (même fichier, même année --
    # ex. plusieurs projets 2025 pour le même CNRS), la mémoire est alimentée
    # IMMÉDIATEMENT depuis les lignes Passe 0 de CE traitement -- avant même que
    # la Passe 0 ne s'exécute réellement -- pour que les lignes sœurs profitent
    # de la Passe mémoire (ci-dessous) plutôt que de repartir inutilement en
    # recherche. Coût nul : aucun appel API, seule une lecture de colonne.
    if col_main_reg:
        for idx in df.index[mask_main_reg]:
            _s_intra = _main_registration_valide(df.at[idx, col_main_reg])
            _cp_intra = str(df.at[idx, col_cp]).strip() if col_cp else ""
            CORRESPONDANCE_SIREN_MEMOIRE.setdefault(
                _cle_correspondance(df.at[idx, col_nom], _cp_intra), _s_intra)

    # v5_31 -- lignes résolues par la MÉMOIRE de correspondance (nom+dept
    # -> SIREN, construite lors d'un traitement antérieur -- typiquement le
    # millésime 2025 -- OU, depuis v5_34, lors de CE MÊME traitement, cf.
    # ci-dessus). Seulement pour ce qui n'est PAS déjà résolu par le Main
    # Registration de la ligne elle-même (mask_main_reg, priorité 1).
    def _en_memoire(idx):
        if col_cp is None:
            return False
        nom_l = str(df.at[idx, col_nom]).strip()
        cp_l  = str(df.at[idx, col_cp]).strip()
        cle_l = _cle_correspondance(nom_l, cp_l)
        if not cle_l[0]:
            return False   # v5_36 : nom vide -> jamais de correspondance
        return (cle_l in CORRESPONDANCE_SIREN_MEMOIRE
                or bool(_siren_memoire_nom_seul(cle_l[0])))   # v5_36 : repli nom seul
    mask_memoire = pd.Series(False, index=df.index)
    if CORRESPONDANCE_SIREN_MEMOIRE:
        _idx_a_verifier = df.index[mask_tvam & ~mask_main_reg]
        # v5_35 -- garde-fou : affecter une LISTE VIDE à une sélection .loc[]
        # VIDE lève TypeError sous pandas (Invalid value '[]' for dtype
        # 'bool') alors que ce devrait être un no-op ; on ne fait l'affectation
        # que si la sélection n'est pas vide.
        if len(_idx_a_verifier):
            mask_memoire.loc[_idx_a_verifier] = [_en_memoire(i) for i in _idx_a_verifier]

    mask_recherche = mask_tvam & ~mask_main_reg & ~mask_memoire

    print("Pays couverts :")
    for p, n in df.loc[mask_fr, col_pays].astype(str).str.strip().str.lower().value_counts().items():
        print(f"  {p:35s} : {n:,}")
    print(f"\nTotal France : {int(mask_fr.sum()):,} lignes  |  {n_total:,} TVA manquants")

    if n_total == 0:
        print("Aucune TVA manquante.")
        _d = df.copy()
        for _c in ("SIRET","Forme_juridique","Code_NAF_APE","Etat_entreprise"):
            if _c not in _d.columns: _d[_c] = ""
        return _d, [], {}

    cols_uniq = [col_nom] + [c for c in (col_adr, col_cp, col_vl) if c]
    benef_uniq = (df.loc[mask_recherche, cols_uniq]
                    .drop_duplicates(subset=[col_nom])
                    .reset_index(drop=True))
    n_uniq = len(benef_uniq)
    if col_main_reg:
        print(f"  dont {int(mask_main_reg.sum()):,} avec SIREN déjà fourni par la Commission "
              f"(Passe 0, sans appel de recherche)")
    if CORRESPONDANCE_SIREN_MEMOIRE:
        print(f"  dont {int(mask_memoire.sum()):,} résolu(s) via la mémoire de correspondance "
              f"(SIREN connu d'un autre millésime, sans appel de recherche)")
    print(f"  -> {n_uniq:,} bénéficiaires uniques restent à chercher (recherche normale)")

    print(f"\n{n_uniq:,} bénéficiaires uniques à rechercher")
    print(f"Adresse utilisée : {'oui' if col_adr else 'non (colonne absente)'}")
    print(f"Durée estimée    : {int(n_uniq * DELAI_API * 2)}s – {int(n_uniq * DELAI_API * 9)}s "
          f"()")

    session = requests.Session()
    session.headers.update({"User-Agent": "FTS-TVA-Recherche/3.0-fusion"})

    cache, rapport = {}, []
    for _, row in benef_uniq.iterrows():
        nom     = str(row[col_nom]).strip()
        adresse = str(row[col_adr]).strip() if col_adr and pd.notna(row.get(col_adr)) else None
        cp      = str(row[col_cp]).strip()  if col_cp  and pd.notna(row.get(col_cp))  else None
        ville   = str(row[col_vl]).strip()  if col_vl  and pd.notna(row.get(col_vl))  else None

        res = rechercher_tva_plus(nom, adresse, ville, cp, session, seuil)
        cache[nom] = res
        # Cas SIREN forcé (correction manuelle vérifiée) : « Bénéficiaire corrigé »
        # prend le VRAI nom officiel de l'annuaire des entreprises (nom_api).
        if res.get("strategie") == "ALIAS_SIREN" and res.get("nom_api"):
            NOM_CORRIGE_PAR_FTS[nom] = res["nom_api"]
        if globals().get('VERBOSE', False):
            _n=len(cache); _p=_n/max(n_uniq,1)*100
            if res['statut']=='TROUVE' and res['tva']:
                _v='\u2705 TROUVE (score '+str(res['score'])+') -> '+str(res['nom_api'])[:45]+' ['+str(res['tva'])+']'
            else:
                _v='\u274c non trouve'
            print('['+str(_n)+'/'+str(n_uniq)+' bénéficiaires '+format(_p,'.1f')+'%] '+str(nom)[:45]+'  '+_v)
        rapport.append({
            "Nom_FTS":      nom,
            "Statut":       res["statut"],
            "Strategie":    res["strategie"],
            "SIREN":        res["siren"],
            "TVA_Trouvee":  res["tva"],
            "Nom_API":      res["nom_api"],
            "Adresse_API":  res.get("adresse_api", ""),
            "Score":        res["score"],
            "Montant_total_beneficiaire": round(montant_par_benef.get(str(nom).strip(), 0.0), 2),
            "SIRET":           res.get("siret",""),
            "Forme_juridique": res.get("forme_juridique",""),
            "Niveau_I":         res.get("niveau_i",""),
            "Niveau_II":        res.get("niveau_ii",""),
            "Niveau_III":       res.get("niveau_iii",""),
            "Code_NAF_APE":    res.get("code_naf",""),
            "Retenu_fichier":  ("Oui" if (res["statut"]=="TROUVE" and res["tva"]
                                          and res.get("score",0) >= SEUIL_FICHIER_TVA) else "Non"),
        })

    df_out, rows_api = df.copy(), {}
    for _c in ("SIREN","SIRET","Forme_juridique","Code_NAF_APE","Etat_entreprise"):
        if _c not in df_out.columns: df_out[_c] = ""
    for idx in df_out[mask_recherche].index:
        nom = str(df_out.at[idx, col_nom]).strip()
        res = cache.get(nom)
        if res and res["statut"] == "TROUVE" and res["tva"] and res.get("score", 0) >= SEUIL_FICHIER_TVA:
            df_out.at[idx, col_tva] = res["tva"]
            df_out.at[idx, "SIREN"] = res.get("siren","")   # identifiant PERMANENT (ne change jamais)
            df_out.at[idx, "SIRET"] = res.get("siret","")   # établissement (change avec l'adresse)
            df_out.at[idx, "Forme_juridique"] = res.get("forme_juridique","")
            df_out.at[idx, "Niveau_I"]   = res.get("niveau_i","")
            df_out.at[idx, "Niveau_II"]  = res.get("niveau_ii","")
            df_out.at[idx, "Niveau_III"] = res.get("niveau_iii","")
            df_out.at[idx, "Code_NAF_APE"]    = res.get("code_naf","")
            df_out.at[idx, "Etat_entreprise"] = res.get("etat","")
            rows_api[idx] = res["score"]

    # v5_30 -- PASSE 0 : documentation directe depuis le SIREN fourni par la
    # Commission (Main Registration). Un seul appel API par SIREN UNIQUE
    # (cache fiche_siren partagé avec l'étape A3 et la Passe 2) -- jamais deux
    # fois le même. Score 100 : fourni directement par la Commission, validé
    # par la clé de Luhn -- confiance maximale, comme une correction ALIAS_SIREN.
    if col_main_reg and mask_main_reg.any():
        idx_main_reg = df_out[mask_main_reg].index
        print(f"\nPasse 0 — SIREN déjà fourni (Main Registration) : {len(idx_main_reg):,} ligne(s)")
        session0 = requests.Session(); session0.headers.update({"User-Agent": "FTS-main-registration/1.0"})
        siren_par_idx = {idx: _main_registration_valide(df_out.at[idx, col_main_reg]) for idx in idx_main_reg}
        sirens_uniques = sorted(set(siren_par_idx.values()))
        print(f"  {len(sirens_uniques):,} SIREN unique(s) à documenter (0 appel de recherche)")
        for _i, _siren in enumerate(sirens_uniques, 1):
            fiche_siren(_siren, session0)   # peuple le cache global, réutilisé ci-dessous
            if globals().get("VERBOSE", False):
                _f = CACHE_FICHES_SIREN.get(_siren, {})
                print(f"  [{_i}/{len(sirens_uniques)}] SIREN {_siren} -> "
                      f"{'\u2705 ' + _f.get('nom_api','') if _f else '\u274c introuvable'}")
        n_p0_ok = 0
        for idx in idx_main_reg:
            _siren = siren_par_idx[idx]
            _fiche = dict(CACHE_FICHES_SIREN.get(_siren, {}))
            _tva_p0 = _siren_vers_tva(_siren)
            nom_p0 = str(df_out.at[idx, col_nom]).strip()
            # v5_32 -- si le CP de la ligne diffère de celui du siège, tenter de
            # retrouver le BON établissement (best-effort, cf. limite documentée
            # ci-dessus) avant de se rabattre sur le SIRET du siège.
            _cp_ligne = str(df_out.at[idx, col_cp]).strip() if col_cp else ""
            if _cp_ligne and re.sub(r"\D", "", _cp_ligne) != re.sub(r"\D", "", str(_fiche.get("cp_api", ""))):
                _etab = _etablissement_par_cp(_siren, _fiche.get("nom_api") or nom_p0, _cp_ligne, session0)
                if _etab:
                    _fiche["siret"] = _etab["siret"]
            if _fiche.get("nom_api"):   # cohérent avec le traitement des corrections ALIAS_SIREN
                NOM_CORRIGE_PAR_FTS[nom_p0] = _fiche["nom_api"]
            df_out.at[idx, col_tva] = _tva_p0
            df_out.at[idx, "SIREN"] = _siren
            df_out.at[idx, "SIRET"] = _fiche.get("siret", "")
            df_out.at[idx, "Forme_juridique"] = _fiche.get("forme_juridique", "")
            df_out.at[idx, "Niveau_I"]   = _fiche.get("niveau_i", "")
            df_out.at[idx, "Niveau_II"]  = _fiche.get("niveau_ii", "")
            df_out.at[idx, "Niveau_III"] = _fiche.get("niveau_iii", "")
            df_out.at[idx, "Code_NAF_APE"]    = _fiche.get("code_naf", "")
            df_out.at[idx, "Etat_entreprise"] = _fiche.get("etat", "")
            rows_api[idx] = 100
            n_p0_ok += 1
            rapport.append({
                "Nom_FTS": nom_p0, "Statut": "TROUVE", "Strategie": "MAIN_REGISTRATION",
                "SIREN": _siren, "TVA_Trouvee": _tva_p0, "Nom_API": _fiche.get("nom_api", ""),
                "Adresse_API": _fiche.get("adresse_api", ""), "Score": 100,
                "Montant_total_beneficiaire": round(montant_par_benef.get(nom_p0, 0.0), 2),
                "SIRET": _fiche.get("siret", ""), "Forme_juridique": _fiche.get("forme_juridique", ""),
                "Niveau_I": _fiche.get("niveau_i", ""), "Niveau_II": _fiche.get("niveau_ii", ""),
                "Niveau_III": _fiche.get("niveau_iii", ""), "Code_NAF_APE": _fiche.get("code_naf", ""),
                "Retenu_fichier": "Oui",
            })
        print(f"  {n_p0_ok:,} ligne(s) documentée(s) via Passe 0 -- score 100, aucun appel de recherche")

    # v5_31 -- Passe MÉMOIRE : documentation directe depuis le SIREN retrouvé
    # dans la mémoire de correspondance (nom+dept), construite lors du
    # traitement d'un AUTRE millésime (typiquement 2025, où Main Registration
    # a fourni un SIREN certain). Même logique que la Passe 0, même niveau de
    # confiance (score 100) : ce SIREN a déjà été validé par la Commission
    # elle-même l'année où on l'a obtenu, on ne le redemande pas au moteur de
    # recherche pour une autre année du même bénéficiaire.
    if mask_memoire.any():
        idx_memoire = df_out[mask_memoire].index
        print(f"\nPasse mémoire — SIREN connu d'un autre millésime : {len(idx_memoire):,} ligne(s)")
        session_m = requests.Session(); session_m.headers.update({"User-Agent": "FTS-memoire-correspondance/1.0"})
        siren_par_idx_m = {}
        strat_par_idx_m = {}   # v5_36 : trace d'audit -- clé exacte ou repli nom seul
        for idx in idx_memoire:
            nom_l = str(df_out.at[idx, col_nom]).strip()
            cp_l  = str(df_out.at[idx, col_cp]).strip() if col_cp else ""
            cle_l = _cle_correspondance(nom_l, cp_l)
            if cle_l in CORRESPONDANCE_SIREN_MEMOIRE:
                siren_par_idx_m[idx] = CORRESPONDANCE_SIREN_MEMOIRE[cle_l]
                strat_par_idx_m[idx] = "MEMOIRE_CORRESPONDANCE"
            else:
                siren_par_idx_m[idx] = _siren_memoire_nom_seul(cle_l[0])
                strat_par_idx_m[idx] = "MEMOIRE_NOM_SEUL"   # v5_36 : déménagement probable
        for _siren in sorted(set(siren_par_idx_m.values())):
            fiche_siren(_siren, session_m)
        n_mem_ok = 0
        for idx in idx_memoire:
            _siren = siren_par_idx_m[idx]
            _fiche = dict(CACHE_FICHES_SIREN.get(_siren, {}))
            _tva_m = _siren_vers_tva(_siren)
            nom_m = str(df_out.at[idx, col_nom]).strip()
            # v5_32 -- même désambiguïsation d'établissement que la Passe 0 (best-effort)
            _cp_ligne_m = str(df_out.at[idx, col_cp]).strip() if col_cp else ""
            if _cp_ligne_m and re.sub(r"\D", "", _cp_ligne_m) != re.sub(r"\D", "", str(_fiche.get("cp_api", ""))):
                _etab_m = _etablissement_par_cp(_siren, _fiche.get("nom_api") or nom_m, _cp_ligne_m, session_m)
                if _etab_m:
                    _fiche["siret"] = _etab_m["siret"]
            if _fiche.get("nom_api"):
                NOM_CORRIGE_PAR_FTS[nom_m] = _fiche["nom_api"]
            df_out.at[idx, col_tva] = _tva_m
            df_out.at[idx, "SIREN"] = _siren
            df_out.at[idx, "SIRET"] = _fiche.get("siret", "")
            df_out.at[idx, "Forme_juridique"] = _fiche.get("forme_juridique", "")
            df_out.at[idx, "Niveau_I"]   = _fiche.get("niveau_i", "")
            df_out.at[idx, "Niveau_II"]  = _fiche.get("niveau_ii", "")
            df_out.at[idx, "Niveau_III"] = _fiche.get("niveau_iii", "")
            df_out.at[idx, "Code_NAF_APE"]    = _fiche.get("code_naf", "")
            df_out.at[idx, "Etat_entreprise"] = _fiche.get("etat", "")
            rows_api[idx] = 100
            n_mem_ok += 1
            rapport.append({
                "Nom_FTS": nom_m, "Statut": "TROUVE", "Strategie": strat_par_idx_m[idx],
                "SIREN": _siren, "TVA_Trouvee": _tva_m, "Nom_API": _fiche.get("nom_api", ""),
                "Adresse_API": _fiche.get("adresse_api", ""), "Score": 100,
                "Montant_total_beneficiaire": round(montant_par_benef.get(nom_m, 0.0), 2),
                "SIRET": _fiche.get("siret", ""), "Forme_juridique": _fiche.get("forme_juridique", ""),
                "Niveau_I": _fiche.get("niveau_i", ""), "Niveau_II": _fiche.get("niveau_ii", ""),
                "Niveau_III": _fiche.get("niveau_iii", ""), "Code_NAF_APE": _fiche.get("code_naf", ""),
                "Retenu_fichier": "Oui",
            })
        print(f"  {n_mem_ok:,} ligne(s) documentée(s) via la mémoire -- score 100, aucun appel de recherche")

    # Passe 2 : documenter les TVA DÉJÀ présentes (via SIREN) — n'affecte pas la recherche
    try:
        ENRICHIR_TVA_EXISTANTES
    except NameError:
        ENRICHIR_TVA_EXISTANTES = True
    if ENRICHIR_TVA_EXISTANTES:
        mask_exist = mask_fr & ~df[col_tva].apply(_tva_manquante)
        exist_uniq = df.loc[mask_exist, [col_nom, col_tva]].drop_duplicates(subset=[col_nom])
        if len(exist_uniq):
            print(f"\nPasse 2 — TVA déjà présentes à documenter : {len(exist_uniq):,}")
            session2 = requests.Session(); session2.headers.update({"User-Agent":"FTS-infos/1.0"})
            cache_siren, infos_nom = {}, {}
            _n2c, _t2 = 0, len(exist_uniq)
            for _, row in exist_uniq.iterrows():
                _n2c += 1
                nom_e = str(row[col_nom]).strip(); siren = siren_depuis_tva(row[col_tva])
                _pfx = f"[{_n2c:>5}/{_t2} {_n2c/_t2*100:5.1f}%] {nom_e[:45]:45s}"
                if not siren:
                    print(f"{_pfx} \u274c SIREN non deductible de la TVA ({row[col_tva]})")
                    continue
                if siren not in cache_siren:
                    cache_siren[siren] = fiche_siren(siren, session2)   # v5_2 : cache global partagé avec A3
                infos_nom[nom_e] = cache_siren[siren]
                _inf = cache_siren[siren]
                # v5_40 : remonter le nom officiel de l'annuaire pour la colonne
                # « Nom API » (lecture seule de la fiche déjà obtenue — la passe 2
                # n'alimentait pas le rapport, la colonne restait donc vide pour
                # toutes les TVA déjà fournies par la Commission).
                if _inf and _inf.get("nom_api"):
                    NOM_API_PAR_FTS[nom_e] = _inf["nom_api"]
                if _inf and _inf.get("siret"):
                    print(f"{_pfx} \u2705 documente [SIRET {_inf.get('siret','')}]")
                elif _inf:
                    print(f"{_pfx} \u26a0\ufe0f trouve sans SIRET (SIREN {siren})")
                else:
                    print(f"{_pfx} \u274c API sans resultat pour SIREN {siren}")
            for idx in df_out[mask_exist].index:
                nom_e2 = str(df_out.at[idx, col_nom]).strip()
                info = infos_nom.get(nom_e2)
                if info:
                    df_out.at[idx, "SIREN"]           = siren_depuis_tva(df_out.at[idx, col_tva]) or info.get("siren","")
                    df_out.at[idx, "SIRET"]           = info.get("siret","")
                    df_out.at[idx, "Forme_juridique"] = info.get("forme_juridique","")
                    df_out.at[idx, "Niveau_I"]   = info.get("niveau_i","")
                    df_out.at[idx, "Niveau_II"]  = info.get("niveau_ii","")
                    df_out.at[idx, "Niveau_III"] = info.get("niveau_iii","")
                    df_out.at[idx, "Code_NAF_APE"]    = info.get("code_naf","")
                    df_out.at[idx, "Etat_entreprise"] = info.get("etat","")

    # Replacer les colonnes ajoutées juste après la TVA
    _aj = ["SIRET","Forme_juridique","Niveau_I","Niveau_II","Niveau_III","Code_NAF_APE","Etat_entreprise"]
    _cols = [c for c in df_out.columns if c not in _aj]
    _i = _cols.index(col_tva) + 1
    df_out = df_out[_cols[:_i] + _aj + _cols[_i:]]

    n_trouve = len(rows_api)
    n_haute  = sum(1 for s in rows_api.values() if s >= SEUIL_HAUTE_CONF)
    print("\n" + "=" * 48)
    print(f"RÉSULTAT : {n_trouve:,} / {n_total:,} TVA trouvés "
          f"({n_trouve / max(n_total, 1) * 100:.1f}%)")
    print(f"  Haute confiance (vert) : {n_haute:,}")
    print(f"  Conf. moyenne  (jaune) : {n_trouve - n_haute:,}")
    print(f"  Non trouvés            : {n_total - n_trouve:,}")
    print("=" * 48)
        # Placer la colonne SIREN juste avant SIRET (SIREN = identifiant permanent)
    if "SIREN" in df_out.columns and "SIRET" in df_out.columns:
        _cols = [c for c in df_out.columns if c != "SIREN"]
        _cols.insert(_cols.index("SIRET"), "SIREN")
        df_out = df_out[_cols]
    CACHE_RECHERCHE_TVA = cache
    return df_out, rapport, rows_api


print("Fonctions d'enrichissement chargées")
print(f"  PAYS_FR : {len(PAYS_FR)} valeurs (France + RUP/COM)")

Mémoire de correspondance : socle embarqué chargé (338 entrées, millésime 2025)
Fonctions d'enrichissement chargées
  PAYS_FR : 13 valeurs (France + RUP/COM)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5 — Export : dataset enrichi coloré + rapport détaillé
# ═══════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════
# CELLULE 6 — FONCTIONS D'EXPORT (fichier enrichi + rapport)
# ═══════════════════════════════════════════════════════════════
# Deux sorties très différentes :
#   • le FICHIER : la donnée destinée à MicroStrategy. Il ne contient QUE les
#     TVA fiables (score >= 96). Colonnes ajoutées peintes en bleu.
#   • le RAPPORT : la trace d'audit. Il contient TOUT, y compris les échecs et
#     les scores faibles, avec un code couleur pour guider la relecture
#     humaine. C'est lui que l'on annote pour produire les corrections.
# Ne jamais « simplifier » en ne produisant que le fichier : le rapport est ce
# qui rend le traitement auditable et améliorable.

def exporter_excel_colore(df: pd.DataFrame, rows_api: dict, nom_fichier: str) -> None:
    print(f"Export : {nom_fichier} ({len(df):,} lignes)…")
    df.to_excel(nom_fichier, index=False, sheet_name="DATASET", engine="openpyxl")
    wb = load_workbook(nom_fichier)
    ws = wb["DATASET"]

    col_tva_nom = next((c for c in df.columns if "vat" in c.lower() or "tva" in c.lower()), None)
    col_tva_idx = list(df.columns).index(col_tva_nom) + 1 if col_tva_nom else None
    n_cols = len(df.columns)

    fill_v  = PatternFill(start_color=COULEUR_LIGNE_VERTE, fill_type="solid")
    fill_j  = PatternFill(start_color=COULEUR_LIGNE_JAUNE, fill_type="solid")
    fill_cv = PatternFill(start_color=COULEUR_CELL_VERTE,  fill_type="solid")
    fill_co = PatternFill(start_color=COULEUR_CELL_ORANGE, fill_type="solid")
    font_b  = Font(bold=True, color="FFFFFF")

    for idx, score in rows_api.items():
        row_excel = int(idx) + 2
        haute = score >= SEUIL_HAUTE_CONF
        for c in range(1, n_cols + 1):
            ws.cell(row=row_excel, column=c).fill = fill_v if haute else fill_j
        if col_tva_idx:
            cell = ws.cell(row=row_excel, column=col_tva_idx)
            cell.fill = fill_cv if haute else fill_co
            cell.font = font_b

    if col_tva_idx:
        hdr = ws.cell(row=1, column=col_tva_idx)
        hdr.fill = PatternFill(start_color="4472C4", fill_type="solid")
        hdr.font = Font(bold=True, color="FFFFFF")

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    wb.save(nom_fichier)
    n_haute = sum(1 for s in rows_api.values() if s >= SEUIL_HAUTE_CONF)
    print(f"  {len(rows_api):,} lignes colorées : {n_haute:,} vert + {len(rows_api) - n_haute:,} jaune")


# Convertit un numéro de colonne (1, 2, 3...) en lettre Excel (A, B, C... Z,
# AA, AB...). Nécessaire pour écrire des formules de mise en forme
# conditionnelle, qui s'expriment en références de type « $H2 ».
def _col_lettre(n: int) -> str:
    s = ""
    while n > 0:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s


# Produit le RAPPORT d'audit (.xlsx). Points clés :
#   • il contient TOUTES les lignes cherchées, quel que soit le résultat ;
#   • code couleur par statut : vert TROUVE, rouge NON_TROUVE, gris EXCLU ;
#   • SURLIGNAGE ORANGE PRIORITAIRE des lignes « à chercher à la main » :
#     score < 96 ET montant cumulé du bénéficiaire >= 300 000 €. Ce sont les
#     cas où l'enjeu financier justifie une vérification humaine ;
#   • onglet « Resume » avec les taux, dont le % de TVA au score >= 96.
# Si aucune TVA n'était manquante, le rapport est vide et AUCUN fichier n'est
# créé : c'est normal, l'orchestration teste l'existence avant de télécharger.
def generer_rapport(rapport: list, nom_fichier: str) -> None:
    if not rapport:
        print("Rapport vide.")
        return
    print(f"Rapport : {nom_fichier}…")
    df_r = pd.DataFrame(rapport)
    with pd.ExcelWriter(nom_fichier, engine="xlsxwriter") as writer:
        df_r.to_excel(writer, index=False, sheet_name="Resultats_TVA")
        wb, ws, n = writer.book, writer.sheets["Resultats_TVA"], len(df_r)
        fmt_hdr  = wb.add_format({"bold": True, "bg_color": "#4472C4", "font_color": "#FFFFFF",
                                  "border": 1, "text_wrap": True, "valign": "vcenter"})
        fmt_ok   = wb.add_format({"bg_color": "#C6EFCE"})
        fmt_non  = wb.add_format({"bg_color": "#FFC7CE"})
        fmt_excl = wb.add_format({"bg_color": "#D9D9D9"})
        for i, col in enumerate(df_r.columns):
            ws.write(0, i, col, fmt_hdr)
        let_s  = _col_lettre(list(df_r.columns).index("Statut") + 1)
        n_cols = len(df_r.columns) - 1
        # v5_12 : mise en évidence ORANGE, PRIORITAIRE et UNIQUEMENT AU RAPPORT —
        # « à chercher à la main » : score < 96 ET bénéficiaire >= 300 000 €
        # (hors lignes EXCLU : personnes physiques, République française…).
        if "Score" in df_r.columns and "Montant_total_beneficiaire" in df_r.columns:
            fmt_ac = wb.add_format({"bg_color": "#FFC000"})
            let_sc = _col_lettre(list(df_r.columns).index("Score") + 1)
            let_mt = _col_lettre(list(df_r.columns).index("Montant_total_beneficiaire") + 1)
            ws.conditional_format(1, 0, n, n_cols, {
                "type": "formula",
                "criteria": (f'=AND(ISNUMBER(${let_sc}2), ${let_sc}2<96, '
                             f'ISNUMBER(${let_mt}2), ${let_mt}2>=300000, ${let_s}2<>"EXCLU")'),
                "format": fmt_ac, "stop_if_true": True})
        for statut, fmt in [("TROUVE", fmt_ok), ("NON_TROUVE", fmt_non), ("EXCLU", fmt_excl)]:
            ws.conditional_format(1, 0, n, n_cols, {
                "type": "formula",
                "criteria": f'=${let_s}2="{statut}"',
                "format": fmt})
        ws.set_row(0, 30)
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, n, n_cols)
        for i, w in enumerate([60, 14, 28, 14, 18, 50, 8, 90]):
            ws.set_column(i, i, w)

        resume = df_r["Statut"].value_counts().reset_index()
        resume.columns = ["Statut", "Nombre"]
        resume["Taux (%)"] = (resume["Nombre"] / len(df_r) * 100).round(1)
        # v5_16 : ligne « TVA score >= seuil » ajoutée au résumé
        _sc = pd.to_numeric(df_r.get("Score"), errors="coerce")
        _nt = int((df_r["Statut"] == "TROUVE").sum())
        _n96 = int(((df_r["Statut"] == "TROUVE") & (_sc >= SEUIL_FICHIER_TVA)).sum())
        _pct_t = round(_n96 / _nt * 100, 1) if _nt else 0.0
        _pct_g = round(_n96 / len(df_r) * 100, 1) if len(df_r) else 0.0
        resume = pd.concat([resume, pd.DataFrame([{
            "Statut": f"TVA score >= {SEUIL_FICHIER_TVA} (retenues fichier)",
            "Nombre": _n96, "Taux (%)": _pct_t}])], ignore_index=True)
        resume.to_excel(writer, index=False, sheet_name="Resume")
        strat = df_r[df_r["Statut"] == "TROUVE"]["Strategie"].value_counts().reset_index()
        if not strat.empty:
            strat.columns = ["Stratégie gagnante", "Nombre"]
            strat.to_excel(writer, index=False, sheet_name="Resume", startrow=len(resume) + 3)
        writer.sheets["Resume"].set_column(0, 0, 30)
        writer.sheets["Resume"].set_column(1, 2, 15)

    print(f"  {len(df_r)} entrées")
    for _, row in resume.iterrows():
        print(f"  {row['Statut']:15s} : {int(row['Nombre']):5d}  ({row['Taux (%)']:.1f}%)")
    # v5_16 : pourcentage des TVA au score >= SEUIL_FICHIER_TVA (retenues au fichier)
    _scores = pd.to_numeric(df_r.get("Score"), errors="coerce")
    _n_trouve = int((df_r["Statut"] == "TROUVE").sum())
    _n_96 = int(((df_r["Statut"] == "TROUVE") & (_scores >= SEUIL_FICHIER_TVA)).sum())
    if _n_trouve:
        print(f"  TVA score >= {SEUIL_FICHIER_TVA} : {_n_96:,}  "
              f"({_n_96 / _n_trouve * 100:.1f}% des TVA trouvées | "
              f"{_n_96 / len(df_r) * 100:.1f}% des bénéficiaires recherchés)")


print("Fonctions d'export chargées")

Fonctions d'export chargées


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5-A3 — Étape A3 : COMPLÉTION SIRENE des champs géographiques (v5_2)
# ═══════════════════════════════════════════════════════════════
# Pour les lignes FRANCE dont l'Adresse / la Ville / le Code postal sont vides
# ou MASQUÉS par le FTS (« . », « - », « ***** »...), on lit l'adresse du SIÈGE
# dans SIRENE via le SIREN (colonne SIREN, sinon déduit de la TVA, sinon du
# SIRET) et on remplit TROIS NOUVELLES colonnes, placées après leurs sources :
#   « Adresse (SIRENE) »     après Address
#   « Ville (SIRENE) »       après City
#   « Code postal (SIRENE) » après Postal code
# Les colonnes d'origine ne sont JAMAIS modifiées (règle pipeline). Les valeurs
# viennent exclusivement de la fiche SIRENE (lecture seule, rien d'inventé).
# En aval : Région_FR / NUTS3 / Metro-RUP-PTOM utilisent ces colonnes en REPLI.
# Les vraies personnes physiques (exclues du moteur : pas de TVA/SIREN) ne
# peuvent pas être complétées ici — seule leur ville (repli VILLES_FR) les sert.

_PLACEHOLDERS_GEO = {"", "-", "--", ".", "..", "*", "**", "***", "****", "*****",
                     "nan", "none", "n/a", "na", "null"}

# ═══════════════════════════════════════════════════════════════
# CELLULE 7 — ÉTAPE A3 : COMPLÉTION GÉOGRAPHIQUE PAR SIRENE
# ═══════════════════════════════════════════════════════════════
# PROBLÈME RÉSOLU ICI : la Commission masque souvent l'adresse des
# bénéficiaires (« . », « - », « ***** »). Résultat : impossible de déduire la
# région, donc impossible de ventiler les fonds par territoire.
# SOLUTION : quand on connaît le SIREN (trouvé en passe 1 ou déduit de la TVA
# existante), on va chercher l'adresse du SIÈGE dans SIRENE et on la met dans
# des colonnes de TRAVAIL — « Adresse (SIRENE) », « Ville (SIRENE) »,
# « Code postal (SIRENE) ».
# IMPORTANT : ces trois colonnes servent uniquement aux calculs en aval
# (région, NUTS, Metro/RUP/PTOM) puis sont RETIRÉES du fichier final. Elles
# n'apparaissent pas dans la livraison : ce sont des échafaudages.

def _geo_vide(v):
    """Vrai si la valeur est vide ou masquée par le FTS (« . », « - », « ***** »)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return True
    return str(v).strip().lower() in _PLACEHOLDERS_GEO

# Récupère le SIREN d'une ligne en essayant trois sources, dans cet ordre :
#   1. la colonne SIREN (si l'enrichissement l'a déjà remplie) ;
#   2. le SIREN déduit de la TVA (les 9 derniers chiffres d'une TVA FR) ;
#   3. les 9 premiers chiffres du SIRET.
# Cette cascade est reprise à l'identique dans plusieurs étapes : c'est la
# façon canonique d'obtenir « l'identifiant de cette ligne » dans ce pipeline.
def _siren_de_ligne(row, col_tva):
    """SIREN exploitable d'une ligne : colonne SIREN, sinon TVA, sinon SIRET."""
    v = str(row.get("SIREN", "") or "").strip()
    if v.isdigit() and len(v) == 9:
        return v
    tva = row.get(col_tva, "") if col_tva else ""
    if not _tva_manquante(tva) and str(tva).strip().upper() != "AUTRE":
        s = siren_depuis_tva(tva)
        if s:
            return s
    v = re.sub(r"\D", "", str(row.get("SIRET", "") or ""))
    if len(v) == 14:
        return v[:9]
    return ""

def completer_geo_sirene(df, colonnes, session=None):
    """Étape A3 (ADDITIVE, hors moteur figé) : complète Adresse/Ville/CP manquants
    depuis la fiche SIRENE du siège. Ne touche à AUCUNE colonne d'origine."""
    df = df.copy()
    col_nom  = colonnes["nom"]
    col_tva  = colonnes.get("tva")
    col_pays = colonnes.get("pays")
    cibles = [(colonnes.get("adresse"), "Adresse (SIRENE)",     "adresse_api"),
              (colonnes.get("ville"),   "Ville (SIRENE)",       "ville_api"),
              (colonnes.get("cp"),      "Code postal (SIRENE)", "cp_api")]
    cibles = [t for t in cibles if t[0]]
    if not cibles:
        print("Complétion SIRENE : aucune colonne Adresse/Ville/CP détectée — étape ignorée.")
        return df
    for _, nc, _ in cibles:
        if nc in df.columns:
            df = df.drop(columns=[nc])
        df[nc] = ""

    # Lignes concernées : France + au moins un champ géo vide + SIREN identifiable
    if col_pays:
        mask_fr = df[col_pays].astype(str).str.strip().str.lower().isin(PAYS_FR)
    else:
        mask_fr = pd.Series(True, index=df.index)
    a_completer = {}          # siren -> [index de lignes]
    n_sans_siren = 0
    for idx in df.index[mask_fr]:
        if not any(_geo_vide(df.at[idx, src]) for src, _, _ in cibles):
            continue
        s = _siren_de_ligne(df.loc[idx], col_tva)
        if s:
            a_completer.setdefault(s, []).append(idx)
        else:
            n_sans_siren += 1

    n_tot = len(a_completer)
    print(f"Lignes France avec géo manquante : SIREN identifiable pour {sum(len(v) for v in a_completer.values()):,} "
          f"ligne(s) ({n_tot:,} SIREN uniques) | sans SIREN (rien à faire) : {n_sans_siren:,}")
    if n_tot:
        deja = sum(1 for s in a_completer if s in CACHE_FICHES_SIREN)
        print(f"Fiches déjà en cache : {deja:,}/{n_tot:,} | "
              f"durée estimée des appels restants : ~{int((n_tot - deja) * DELAI_API * 4)}s")
        if session is None:
            session = requests.Session()
            session.headers.update({"User-Agent": "FTS-geo-completion/1.0"})
        n_ok = n_rempli = 0
        for i, (s, idxs) in enumerate(a_completer.items(), 1):
            nom_aff = str(df.at[idxs[0], col_nom])[:45]
            fiche = fiche_siren(s, session)
            if not fiche:
                print(f"[{i}/{n_tot} {i/n_tot*100:5.1f}%] {nom_aff:45s} \u274c fiche SIRENE introuvable (SIREN {s})")
                continue
            rempli_ici = 0
            for idx in idxs:
                for src, nc, cle in cibles:
                    val = str(fiche.get(cle, "") or "").strip()
                    if val and _geo_vide(df.at[idx, src]):
                        df.at[idx, nc] = val
                        rempli_ici += 1
            n_ok += 1
            n_rempli += rempli_ici
            print(f"[{i}/{n_tot} {i/n_tot*100:5.1f}%] {nom_aff:45s} \u2705 "
                  f"{fiche.get('cp_api','')} {fiche.get('ville_api','')} "
                  f"({rempli_ici} case(s) sur {len(idxs)} ligne(s))")
        print(f"Complétion SIRENE : {n_ok:,}/{n_tot:,} SIREN documentés | {n_rempli:,} cases complétées")

    # Placement : chaque nouvelle colonne juste APRÈS sa colonne source
    noms_nc = [nc for _, nc, _ in cibles]
    ordre = [c for c in df.columns if c not in noms_nc]
    for src, nc, _ in cibles:
        if src in ordre:
            ordre.insert(ordre.index(src) + 1, nc)
        else:
            ordre.append(nc)
    return df[ordre]

print("Étape A3 chargée : complétion SIRENE des champs géographiques prête")


Étape A3 chargée : complétion SIRENE des champs géographiques prête


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5 — Nettoyage géographique : adresses, code postal, Région_FR, NUTS3
# ═══════════════════════════════════════════════════════════════
# NUTS2 d'origine (anciennes régions) CONSERVÉ. On AJOUTE Région_FR (nouvelles
# régions) et NUTS3 FR (département), déduits du code postal, sur TOUTES les lignes.
import unicodedata, re

# Tables géographiques chargées depuis GEO_FRANCE.json à l'étape de chargement
# (AUCUNE donnée en dur dans le code — demande utilisateur v3_7).
DEPT_NUTS3_FR  = {}   # code département -> nom du département (NUTS3)
DEPT_REGION_FR = {}   # code département -> région (NUTS2 = NUTS3 pour RUP/COM)
DEPT_NUMERO    = {}   # code département -> numéro (1, 2A, 971…)
PAYS_VERS_DEPARTEMENT = {}   # pays/territoire -> code département INSEE
NUTS2_PAR_DEPT = {}          # code département -> ancienne région (NUTS2 Eurostat 2021) — v5_3
OPERATEURS_ETAT_SIRENS = {}  # SIREN -> nom d'opérateur de l'État (liste utilisateur) — v5_5
REF_SGAE_PAR_CODE = {}       # code juridique niveau III (4 ch.) -> catégorie SGAE — v5_14
OPERATEURS_ETAT_PROG = {}    # SIREN opérateur -> programme chef de file — v5_17
OPERATEURS_ETAT_PERIODES = {}  # SIREN -> (annee_debut, annee_fin) validité — v5_27
NAF_LIBELLE_PAR_CODE = {}    # code NAF rév.2 -> libellé d'activité principale — v5_17
SECTION_LIBELLE_PAR_LETTRE = {}  # NAF 2025 : lettre de section (A-V) -> intitulé — v5_22
SECTION_PAR_DIVISION       = {}  # NAF 2025 : division (2 chiffres) -> lettre de section — v5_22
_NUTS2_TABLE = {}            # clé normalisée -> orthographe canonique (GEO_FRANCE.json)

# ═══════════════════════════════════════════════════════════════
# CELLULE 8 — NETTOYAGE GÉOGRAPHIQUE : NUTS2, région, NUTS3
# ═══════════════════════════════════════════════════════════════
# Objectif : donner à chaque ligne française sa position dans la nomenclature
# territoriale européenne, même quand le FTS est lacunaire.
#
# CHAÎNE DE REPLI (l'ordre est capital — du plus fiable au moins fiable) :
#   1. code postal BRUT du FTS ;
#   2. code postal du SIÈGE SIRENE (colonne de travail de l'Étape A3) ;
#   3. pays (utile pour les collectivités d'outre-mer) ;
#   4. NUTS2 brut déjà présent dans le FTS ;
#   5. VILLE, via le référentiel des 31 950 communes (dernier recours).
# Dès qu'un maillon donne un résultat, on s'arrête. Si aucun ne donne rien, la
# case reste VIDE : on n'invente jamais une région.
#
# PIÈGE CONNU : les graphies de région divergent entre référentiels
# (« Grand-Est » vs « Grand Est »). Toutes les comparaisons de région passent
# donc par une clé normalisée sans tiret ni espace ni accent.

def _cle_nuts2(s):
    s = unicodedata.normalize("NFD", str(s).upper())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"[^A-Z]", "", s)

def corriger_nuts2(v):
    v = str(v).strip()
    if not v or v in ("-", "nan", "None"):
        return ""
    return _NUTS2_TABLE.get(_cle_nuts2(v), v)

def _cle_dept(num):
    return num if (num in ("2A","2B") or len(num) >= 3) else num.zfill(2)

def dept_depuis_cp(cp):
    cp = str(cp).strip()
    if len(cp) != 5 or not cp.isdigit():
        return ""
    if cp == "97133":                        # Saint-Barthélemy (code postal La Poste)
        return "977"
    if cp == "97150":                        # Saint-Martin (code postal La Poste)
        return "978"
    if cp[:2] in ("97", "98"):
        return cp[:3]
    if cp[:2] == "20":                       # Corse : 200/201 = 2A, 202+ = 2B
        return "2A" if cp < "20200" else "2B"
    return cp[:2]

def infos_geo_depuis_cp(cp, ville=""):
    d = dept_depuis_cp(cp)
    # 977xx/978xx : en codes POSTAUX ce sont des CEDEX de La Réunion (les vrais
    # codes postaux de Saint-Barthélemy et Saint-Martin sont 97133 et 97150,
    # traités ci-dessus). On ne bascule vers les COM que si la ville l'indique.
    if d in ("977", "978") and str(cp).strip() not in ("97133", "97150"):
        v = str(ville or "").lower()
        v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode()
        if "barth" in v or "gustavia" in v:
            d = "977"
        elif ("martin" in v or "marigot" in v) and "reunion" not in v:
            d = "978"
        else:
            d = "974"                        # La Réunion (CEDEX)
    # numéro : d'abord la table, sinon le code département brut issu du CP (91, 977, 978...)
    numero = DEPT_NUMERO.get(d, "")
    if not numero and d:
        numero = d.lstrip("0") if d.isdigit() else d   # "91"->"91", "07"->"7", "2A"->"2A", "977"->"977"
    return {"dept": d, "nuts3": DEPT_NUTS3_FR.get(d, ""),
            "region_fr": DEPT_REGION_FR.get(d, ""), "numero": numero}


def reordonner_adresse(adr):
    """Remet l'adresse au format 'numéro puis voie'. Retourne (adresse, modifiee?)."""
    s = " ".join(str(adr).split())
    if not s or s.lower() in ("nan", "none", "-"):
        return s, False
    if re.match(r"^\d", s):
        return s, False
    m = re.search(r"^(.*?)[\s,]+(\d+(?:[/-]\d+)?)(\s+(?:BIS|TER|QUATER))?$", s, re.IGNORECASE)
    if m and m.group(1).strip(" ,"):
        rue = m.group(1).strip(" ,"); num = m.group(2) + (m.group(3) or "")
        return f"{num} {rue}", True
    return s, False

# Répare les codes postaux abîmés par Excel : « 1000 » au lieu de « 01000 »
# (le zéro initial saute quand la colonne est lue comme un nombre), espaces
# parasites, caractères non numériques. Renvoie aussi un drapeau indiquant si
# une correction a eu lieu, pour la tracer dans le fichier.
def corriger_cp(cp):
    """Nettoie un code postal (espaces, zéro initial). Retourne (cp, corrige?)."""
    brut = str(cp); s = re.sub(r"\D", "", brut)
    if not s:
        return "", False
    if len(s) < 5: s = s.zfill(5)
    elif len(s) > 5: s = s[:5]
    return s, (s != re.sub(r"\s", "", brut))

def _dept_depuis_pays(p):
    t = unicodedata.normalize("NFKD", str(p or "")).encode("ascii", "ignore").decode()
    return PAYS_VERS_DEPARTEMENT.get(" ".join(t.split()).strip().lower(), "")

# DERNIER RECOURS géographique : deviner le département à partir du seul nom
# de commune. Utilisé surtout pour les personnes physiques (exclues du moteur,
# donc sans SIREN) dont on ne connaît que la ville.
# Les communes homonymes sont volontairement ABSENTES du référentiel : si le
# nom est ambigu, on préfère ne rien conclure plutôt que d'affecter la ligne
# au mauvais territoire (ce qui fausserait les analyses Metro/RUP/PTOM).
def _geo_depuis_ville(ville):
    """Ville -> (region_fr, nuts3, numero) via VILLE_VERS_DEPT + GEO_FRANCE.
    ('','','') si la ville est inconnue ou ambiguë."""
    if not ville:
        return "", "", ""
    v = unicodedata.normalize("NFKD", str(ville)).encode("ascii","ignore").decode()
    v = re.sub(r"[^A-Za-z0-9]+", " ", v).strip().upper()
    if not v or v in ("-", "NAN"):
        return "", "", ""
    dept = VILLE_VERS_DEPT.get(v)
    if not dept:
        v2 = v.replace("SAINTE ", "STE ").replace("SAINT ", "ST ")
        dept = VILLE_VERS_DEPT.get(v2)
    if not dept:
        return "", "", ""
    dcle = _cle_dept(str(dept))
    return (DEPT_REGION_FR.get(dcle, ""), DEPT_NUTS3_FR.get(dcle, ""), DEPT_NUMERO.get(dcle, ""))

def _region_depuis_nuts2(nuts2_corrige):
    """Nouvelle Région FR à partir d'une ancienne région (valeur NUTS2 corrigée),
    via le référentiel NUTS2_VERS_REGION. '' si inconnue."""
    if not nuts2_corrige:
        return ""
    return NUTS2_VERS_REGION.get(str(nuts2_corrige).strip(), "")

def nettoyer_geo(df, col_adresse, col_cp, col_nuts2):
    """Colonnes AJOUTÉES : Adresse réordonnée (v5_40 — l'adresse d'origine de la
    Commission n'est PLUS modifiée), Code postal corrigé, Région_FR, NUTS3 FR.
    NUTS2 d'origine CONSERVÉ. S'applique à TOUTES les lignes."""
    if not DEPT_NUTS3_FR:
        raise RuntimeError("Référentiels géographiques non chargés — exécute la cellule "
                           "de chargement (fichier GEO_FRANCE.json requis).")
    df = df.copy()
    adr_mod = set()

    # 1) Adresses -> NOUVELLE colonne « Adresse réordonnée » (v5_40). La colonne
    #    d'origine (Commission) n'est plus touchée. La nouvelle colonne contient
    #    l'adresse réordonnée quand une correction s'applique, sinon l'adresse
    #    d'origine TELLE QUELLE — elle est donc utilisable seule par tous les
    #    traitements d'affichage en aval. adr_mod repère les lignes réellement
    #    réordonnées (surlignage dans l'export). Le moteur de recherche, lui,
    #    a déjà tourné sur l'adresse BRUTE (règle intangible), en amont.
    if col_adresse:
        _adr_reord = []
        for idx in df.index:
            val = df.at[idx, col_adresse]
            if pd.isna(val):
                _adr_reord.append("")
                continue
            new, ch = reordonner_adresse(val)
            if ch:
                _adr_reord.append(new)
                adr_mod.add(idx)
            else:
                _adr_reord.append(str(val))
        df["Adresse réordonnée"] = _adr_reord

    # 2) Code postal corrigé + 3) Région_FR + 4) NUTS3 FR (déduits du CP), TOUTES lignes
    col_ville_geo = next((c for c in df.columns
                          if "city" in str(c).lower() or "ville" in str(c).lower()), None)
    col_pays_geo = next((c for c in df.columns
                         if "beneficiary country" in str(c).lower()
                         or str(c).strip().lower() in ("country", "pays")), None)
    cp_corr, region_fr, nuts3_vals, nuts3_num = [], [], [], []
    for idx in df.index:
        c, _ = corriger_cp(df.at[idx, col_cp]) if col_cp else ("", False)
        cp_corr.append(c)
        _v = df.at[idx, col_ville_geo] if col_ville_geo is not None else ""
        g = infos_geo_depuis_cp(c, "" if pd.isna(_v) else str(_v))
        # 2bis) REPLI CP SIRENE (Étape A3, v5_2) : CP brut inexploitable mais code
        #       postal du SIÈGE SIRENE connu -> même logique CP -> région/NUTS3.
        if not g["region_fr"] and "Code postal (SIRENE)" in df.columns:
            _cs = df.at[idx, "Code postal (SIRENE)"]
            if not _geo_vide(_cs):
                _vs = _v
                if _geo_vide(_vs) and "Ville (SIRENE)" in df.columns:
                    _vs = df.at[idx, "Ville (SIRENE)"]
                _c2, _ = corriger_cp(_cs)
                _g2 = infos_geo_depuis_cp(_c2, "" if pd.isna(_vs) else str(_vs))
                if _g2["region_fr"]:
                    g = _g2
        if not g["region_fr"] and col_pays_geo is not None:
            _d = _dept_depuis_pays(df.at[idx, col_pays_geo])
            if _d:   # CP inexploitable mais territoire identifié par le pays
                g = {"dept": _d, "nuts3": DEPT_NUTS3_FR.get(_d, ""),
                     "region_fr": DEPT_REGION_FR.get(_d, ""),
                     "numero": DEPT_NUMERO.get(_d, _d)}
        # 3bis) REPLI sans code postal (personnes physiques, répliques françaises) :
        #   dériver la Région_FR depuis le NUTS2 corrigé (référentiel actuel), puis
        #   à défaut depuis la ville. NUTS3/numéro seulement si identifiables (RUP).
        if not g["region_fr"] and col_nuts2:
            _n2c = corriger_nuts2(df.at[idx, col_nuts2])
            _reg = _region_depuis_nuts2(_n2c)
            if not _reg and col_ville_geo is not None:      # la ville vaut parfois le NUTS2
                _reg = _region_depuis_nuts2(corriger_nuts2(_v))
            if _reg:
                g = {"dept": g["dept"], "region_fr": _reg,
                     "nuts3": RUP_NUTS3_PAR_REGION.get(_reg, {}).get("nuts3", g["nuts3"]),
                     "numero": RUP_NUTS3_PAR_REGION.get(_reg, {}).get("numero", g["numero"])}
        # 3ter) REPLI par la VILLE — brute, puis « Ville (SIRENE) » (v5_2) :
        #   ville -> département -> Région_FR + NUTS3 + numéro (référentiel communes).
        if not g["region_fr"]:
            _v_eff = _v if (col_ville_geo is not None and not _geo_vide(_v)) else ""
            if _geo_vide(_v_eff) and "Ville (SIRENE)" in df.columns:
                _vsir = df.at[idx, "Ville (SIRENE)"]
                _v_eff = "" if _geo_vide(_vsir) else _vsir
            if not _geo_vide(_v_eff):
                _rg, _n3, _num = _geo_depuis_ville(_v_eff)
                if _rg:
                    g = {"dept": _num or g["dept"], "region_fr": _rg, "nuts3": _n3, "numero": _num}
        region_fr.append(g["region_fr"])
        nuts3_vals.append(g["nuts3"])
        nuts3_num.append(g["numero"])
    df["Code postal corrigé"] = cp_corr
    df["Région_FR"] = region_fr      # nouvelles régions (NUTS2 d'origine conservé)
    df["NUTS3 FR"]  = nuts3_vals
    df["NUTS3_Numéro"] = nuts3_num
    # v5_3 : « NUTS2 corrigé » COMPLÉTÉ quand le NUTS2 brut est vide — ancienne
    # région (nomenclature Eurostat NUTS 2021) déduite du département identifié
    # dans la boucle ci-dessus (CP brut -> CP SIRENE -> pays -> ville) via
    # NUTS2_PAR_DEPT. Un NUTS2 brut non vide n'est JAMAIS remplacé.
    nuts2_corr, n_nuts2_complete = [], 0
    for _pos, _i in enumerate(df.index):
        _v = corriger_nuts2(df.at[_i, col_nuts2]) if col_nuts2 else ""
        if not _v and nuts3_num[_pos]:
            _v2 = NUTS2_PAR_DEPT.get(_cle_dept(str(nuts3_num[_pos])), "")
            if _v2:
                _v = _v2
                n_nuts2_complete += 1
        nuts2_corr.append(_v)
    df["NUTS2 corrigé"] = nuts2_corr

    # Placement : Adresse réordonnée après l'adresse ; CP corrigé après le CP ;
    # Région_FR après NUTS2 ; NUTS3 FR après Région_FR
    ajout = ["Code postal corrigé", "NUTS2 corrigé", "Région_FR", "NUTS3 FR", "NUTS3_Numéro"]
    if "Adresse réordonnée" in df.columns:
        ajout = ["Adresse réordonnée"] + ajout
    cols = [c for c in df.columns if c not in ajout]
    if "Adresse réordonnée" in df.columns:
        if col_adresse and col_adresse in cols:
            cols.insert(cols.index(col_adresse) + 1, "Adresse réordonnée")
        else:
            cols.append("Adresse réordonnée")
    if col_cp and col_cp in cols:
        cols.insert(cols.index(col_cp) + 1, "Code postal corrigé")
    else:
        cols.append("Code postal corrigé")
    if col_nuts2 and col_nuts2 in cols:
        cols.insert(cols.index(col_nuts2) + 1, "NUTS2 corrigé")
        cols.insert(cols.index("NUTS2 corrigé") + 1, "Région_FR")
        cols.insert(cols.index("Région_FR") + 1, "NUTS3 FR")
        cols.insert(cols.index("NUTS3 FR") + 1, "NUTS3_Numéro")
    else:
        cols += ["NUTS2 corrigé", "Région_FR", "NUTS3 FR", "NUTS3_Numéro"]
    df = df[cols]

    print(f"  Adresses réordonnées   : {len(adr_mod):,}")
    print(f"  NUTS2 corrigé complétés: {n_nuts2_complete:,} (depuis le département)")
    print(f"  Région_FR renseignées  : {sum(1 for v in region_fr if v):,}")
    print(f"  NUTS3 FR renseignés    : {sum(1 for v in nuts3_vals if v):,}")
    return df, adr_mod, set()   # 3e valeur (compat signature) : plus de modif NUTS2

print("Fonctions de nettoyage géographique chargées (NUTS2 conservé + Région_FR ajoutée)")


Fonctions de nettoyage géographique chargées (NUTS2 conservé + Région_FR ajoutée)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5-0 — Étape 0 : ENRICHISSEMENT GLOBAL (tous pays)
# ═══════════════════════════════════════════════════════════════
# v3_7 : AUCUNE donnée dans le code. Tous les référentiels sont chargés depuis
# 5 fichiers JSON déposés au lancement (cellule de chargement) :
#   ETATS.json, SOUS_CATEGORIES.json, CFP_REGLES.json, PAYS_ALIAS.json,
#   GEO_FRANCE.json
#
# Colonnes ajoutées (aucune colonne d'origine modifiée, coloriage BDD7EE) :
#  - « Bénéficiaire corrigé »  après le nom (étoiles finales retirées)
#  - « FR/UE/UK/AELE/AUTRE » + « Etats » après « Beneficiary country »
#  - « Période CFP » + « Dépense CFP »   après « Budget line name »
#  - « Sous catégorie »                  juste AVANT « Programme name »
#
# Définitions CMFE (méthode utilisateur) :
#  * Période CFP  = CFP durant lequel le projet a été CONVENTIONNÉ
#                   → déduite de l'ANNÉE (colonne Year) : 2014-2020 → « 14-20 »,
#                     2021-2027 → « 21-27 » (2007-2013 → « 07-13 », signalé).
#  * Dépense CFP  = CFP auquel se RATTACHE la somme, indépendamment de l'année
#                   d'engagement → « Hors CFP » pour les instruments listés dans
#                   CFP_REGLES.json ; sinon CFP antérieur si le nom de ligne
#                   contient « completion of… », « former », « prior to
#                   2021/2014/2007 » ou « (2007 to 2013) » ; sinon plage de la
#                   Période. (Les règles textuelles sont de la LOGIQUE et
#                   restent dans le code ; les LISTES sont dans les JSON.)

def _norm_g(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().lower()

# ── Structures remplies par charger_referentiels() — vides tant que les JSON
#    ne sont pas chargés ─────────────────────────────────────────────────────
CORRESPONDANCE_ETATS = {}   # nom normalisé -> (code Etat, statut FR/UE/UK/AELE)
ALIAS_PAYS = {}             # libellé FTS -> libellé de la table ETATS
DOM_TOM_RUP_FR = set()      # territoires rattachés à FR (zone ET code)
ISO_ALIAS = {}              # libellé -> code ISO vérifié (secours)
SCAT_PAR_PROG = {}          # programme normalisé -> (sous-catégorie, période)
HORS_CFP_PROGRAMMES = set() # programmes dont la Dépense est « Hors CFP »
PREFIXE_SPECIAL_CFP = set() # programmes préfixe O./S./9.0 mais VRAIS programmes CFP
NOM_CORRIGE_PAR_FTS = {}
NOM_API_PAR_FTS = {}   # v5_40 : nom officiel annuaire par nom FTS (rempli par la passe 2 — TVA existantes)
NUTS2_VERS_REGION = {}
VILLE_VERS_DEPT = {}   # ville normalisée -> code département (référentiel communes)   # ancienne région (NUTS2 corrigé) -> nouvelle Région FR
RUP_NUTS3_PAR_REGION = {}    # nom FTS -> « Bénéficiaire corrigé » (réconciliation)
CACHE_RECHERCHE_TVA = {}    # nom FTS -> résultat de recherche (rempli par enrichir_france)
PERIODE_HORS_CFP_POUR_INSTRUMENTS = False
_REFERENTIELS_OK = False

FICHIERS_REFERENTIELS = ("ETATS.json", "SOUS_CATEGORIES.json", "CFP_REGLES.json",
                         "PAYS_ALIAS.json", "GEO_FRANCE.json", "FORMES_JURIDIQUES.json")

# Lit les blocs JSON de la cellule 2 et remplit les DICTIONNAIRES GLOBAUX
# utilisés partout ensuite (DEPT_REGION_FR, VILLE_VERS_DEPT, NUTS2_PAR_DEPT,
# REF_SGAE_PAR_CODE, NAF_LIBELLE_PAR_CODE, OPERATEURS_ETAT_SIRENS...).
#
# C'est ici qu'a lieu l'HARMONISATION DES GRAPHIES de région : la table
# départements fournie par l'utilisateur FAIT FOI (« Grand-Est » avec tiret),
# et la table NUTS2->région est alignée dessus. Sans cela, une même région
# s'écrirait de deux façons selon le chemin de calcul, et le classement
# Metro/RUP/PTOM échouerait silencieusement sur ces lignes.
#
# La fonction AFFICHE le nombre d'entrées lues par référentiel : c'est votre
# premier contrôle de bonne santé. Un compteur à 0 = un bloc JSON cassé.
def charger_referentiels(jsons):
    """Construit tous les index depuis les 5 JSON (clé -> objet déjà parsé).
    Clés attendues : ETATS, SOUS_CATEGORIES, CFP_REGLES, PAYS_ALIAS, GEO_FRANCE."""
    global PERIODE_HORS_CFP_POUR_INSTRUMENTS, _REFERENTIELS_OK
    manquants = [k for k in ("ETATS", "SOUS_CATEGORIES", "CFP_REGLES",
                             "PAYS_ALIAS", "GEO_FRANCE") if k not in jsons]
    if manquants:
        raise ValueError(
            "Fichier(s) JSON de référence manquant(s) : "
            + ", ".join(f"{m}.json" for m in manquants)
            + f"\nDépose les 5 fichiers requis : {', '.join(FICHIERS_REFERENTIELS)}")

    # 1) ETATS — zone géographique + code Etat
    CORRESPONDANCE_ETATS.clear()
    _rows = jsons["ETATS"]["rows"]
    for _r in _rows:
        _code, _statut = _r.get("Etats_court"), _r.get("Etat_Statut")
        for _cle in (_r.get("Etats_long_EN"), _r.get("Etats_long_FR"), _code):
            if _cle:
                CORRESPONDANCE_ETATS[_norm_g(_cle)] = (_code, _statut)
    print(f"  [ok] ETATS.json            : {len(_rows)} pays")

    # 2) SOUS_CATEGORIES — programme -> (sous-catégorie, période)
    SCAT_PAR_PROG.clear()
    _sc = jsons["SOUS_CATEGORIES"]["rows"]
    for _r in _sc:
        SCAT_PAR_PROG[_norm_g(_r["Programme"])] = (_r["Sous catégorie"], _r["Période CFP"])
    print(f"  [ok] SOUS_CATEGORIES.json  : {len(_sc)} programmes")

    # 3) CFP_REGLES — instruments hors CFP + commutateur Période
    HORS_CFP_PROGRAMMES.clear()
    HORS_CFP_PROGRAMMES.update(_norm_g(p) for p in jsons["CFP_REGLES"]["hors_cfp_programmes"])
    PREFIXE_SPECIAL_CFP.clear()
    PREFIXE_SPECIAL_CFP.update(_norm_g(p) for p in jsons["CFP_REGLES"].get("prefixe_special_dans_cfp", []))
    PERIODE_HORS_CFP_POUR_INSTRUMENTS = bool(
        jsons["CFP_REGLES"].get("periode_hors_cfp_pour_instruments", False))
    print(f"  [ok] CFP_REGLES.json       : {len(HORS_CFP_PROGRAMMES)} instruments hors CFP | "
          f"Période instruments = {'Hors CFP' if PERIODE_HORS_CFP_POUR_INSTRUMENTS else 'année'}")

    # 4) PAYS_ALIAS — alias libellés, alias ISO, DOM-TOM/RUP
    ALIAS_PAYS.clear();  ALIAS_PAYS.update(jsons["PAYS_ALIAS"]["alias_libelles"])
    ISO_ALIAS.clear();   ISO_ALIAS.update({_norm_g(k): v for k, v in jsons["PAYS_ALIAS"]["iso_alias"].items()})
    DOM_TOM_RUP_FR.clear()
    DOM_TOM_RUP_FR.update(_norm_g(x) for x in jsons["PAYS_ALIAS"]["dom_tom_rup_fr"])
    print(f"  [ok] PAYS_ALIAS.json       : {len(ALIAS_PAYS)} alias libellés | "
          f"{len(ISO_ALIAS)} alias ISO | {len(DOM_TOM_RUP_FR)} territoires FR")

    # 5) GEO_FRANCE — départements + territoires (tables des cellules géo)
    DEPT_NUTS3_FR.clear(); DEPT_REGION_FR.clear(); DEPT_NUMERO.clear()
    for _d in jsons["GEO_FRANCE"]["departements"]:
        _k = _cle_dept(str(_d["code"]))
        DEPT_NUTS3_FR[_k]  = _d["nuts3"]
        DEPT_REGION_FR[_k] = _d["region"]
        DEPT_NUMERO[_k]    = str(_d["code"])
    PAYS_VERS_DEPARTEMENT.clear()
    PAYS_VERS_DEPARTEMENT.update({_norm_g(k): str(v) for k, v in
                                  jsons["GEO_FRANCE"]["pays_vers_departement"].items()})
    TERRITOIRE_PAR_PAYS.clear()
    TERRITOIRE_PAR_PAYS.update({_norm_g(k): v for k, v in
                                jsons["GEO_FRANCE"]["territoires_par_pays"].items()})
    TERRITOIRE_PAR_CP3.clear()
    TERRITOIRE_PAR_CP3.update({str(k): v for k, v in
                               jsons["GEO_FRANCE"]["territoires_par_cp3"].items()})
    VILLE_VERS_DEPT.clear()
    if "VILLES_FR" in jsons:
        VILLE_VERS_DEPT.update(jsons["VILLES_FR"])
    NUTS2_PAR_DEPT.clear()   # v5_3 : département -> ancienne région (NUTS2)
    _n2d = jsons["GEO_FRANCE"].get("nuts2_par_departement", {}).get("map", {})
    NUTS2_PAR_DEPT.update({_cle_dept(str(_k)): _v for _k, _v in _n2d.items()})
    REF_SGAE_PAR_CODE.clear()   # v5_14 : statut juridique simplifié (référentiel SGAE)
    for _k, _v in jsons.get("REF_SGAE", {}).get("map", {}).items():
        _c = re.sub(r"\D", "", str(_k))
        if _c:
            REF_SGAE_PAR_CODE[_c] = _v
    if REF_SGAE_PAR_CODE:
        print(f"  [ok] REF_SGAE.json         : {len(REF_SGAE_PAR_CODE)} codes -> catégorie SGAE")
    OPERATEURS_ETAT_PERIODES.clear()   # v5_28 : périodes de validité, LISTE par SIREN
    # (un SIREN peut connaître plusieurs périodes successives sous des noms
    # différents, ex. Pôle emploi 2000-2024 -> France Travail 2025-2026 : même
    # SIREN 130005481, deux couples [début, fin] dans la liste.)
    for _k, _v in jsons.get("OPERATEURS_ETAT", {}).get("periodes", {}).items():
        _s = re.sub(r"\D", "", str(_k))
        if len(_s) != 9 or not isinstance(_v, list) or not _v:
            continue
        _couples = _v if isinstance(_v[0], list) else [_v]   # rétro-compat v5_27 (couple unique)
        OPERATEURS_ETAT_PERIODES[_s] = [(int(a), int(b)) for a, b in _couples]
    OPERATEURS_ETAT_PROG.clear()   # v5_17 : programmes chefs de file des opérateurs
    for _k, _v in jsons.get("OPERATEURS_ETAT", {}).get("programmes", {}).items():
        _s = re.sub(r"\D", "", str(_k))
        if len(_s) == 9:
            OPERATEURS_ETAT_PROG[_s] = str(_v).strip()
    NAF_LIBELLE_PAR_CODE.clear()   # v5_17 : NAF rév.2 -> libellé
    NAF_LIBELLE_PAR_CODE.update(jsons.get("NAF_REV2", {}).get("map", {}))
    if NAF_LIBELLE_PAR_CODE:
        print(f"  [ok] NAF_REV2.json         : {len(NAF_LIBELLE_PAR_CODE)} sous-classes | "
              f"{len(OPERATEURS_ETAT_PROG)} programmes d'opérateurs | "
              f"{len(OPERATEURS_ETAT_PERIODES)} période(s) de validité")
    SECTION_LIBELLE_PAR_LETTRE.clear()   # v5_22 : sections NAF 2025 (fichier utilisateur)
    SECTION_LIBELLE_PAR_LETTRE.update(jsons.get("NAF2025_SECTIONS", {}).get("sections", {}))
    SECTION_PAR_DIVISION.clear()
    SECTION_PAR_DIVISION.update(jsons.get("NAF2025_SECTIONS", {}).get("division_section", {}))
    if SECTION_PAR_DIVISION:
        print(f"  [ok] NAF2025_SECTIONS.json : {len(SECTION_LIBELLE_PAR_LETTRE)} sections | "
              f"{len(SECTION_PAR_DIVISION)} divisions")
    OPERATEURS_ETAT_SIRENS.clear()   # v5_5 : opérateurs de l'État (liste utilisateur)
    for _k, _v in jsons.get("OPERATEURS_ETAT", {}).get("sirens", {}).items():
        _s = re.sub(r"\D", "", str(_k))
        if len(_s) == 9:
            OPERATEURS_ETAT_SIRENS[_s] = _v
    if OPERATEURS_ETAT_SIRENS:
        print(f"  [ok] OPERATEURS_ETAT.json  : {len(OPERATEURS_ETAT_SIRENS)} SIREN d'opérateurs de l'État")
    NUTS2_VERS_REGION.clear(); RUP_NUTS3_PAR_REGION.clear()
    _n2r = jsons["GEO_FRANCE"].get("nuts2_vers_region", {})
    NUTS2_VERS_REGION.update(_n2r.get("map", {}))
    RUP_NUTS3_PAR_REGION.update(_n2r.get("rup_nuts3", {}))
    # v5_11 : LA GRAPHIE DU RÉFÉRENTIEL DÉPARTEMENTS->NUTS3/RÉGION (fourni par
    # l'utilisateur) FAIT FOI : « Grand-Est » AVEC tiret. La table nuts2_vers_region
    # est harmonisée dessus, pour que Région_FR soit identique quel que soit le
    # chemin de dérivation (CP, CP SIRENE, NUTS2, pays, ville).
    import unicodedata as _ucd
    def _cle_region_h(_s):
        _t = _ucd.normalize("NFKD", str(_s)).encode("ascii", "ignore").decode().lower()
        return re.sub(r"[^a-z0-9]", "", _t)
    _regs_ref = {}
    for _r_h in DEPT_REGION_FR.values():
        _regs_ref.setdefault(_cle_region_h(_r_h), _r_h)
    _n_harmo = 0
    for _k_h, _v_h in list(NUTS2_VERS_REGION.items()):
        _c_h = _regs_ref.get(_cle_region_h(_v_h))
        if _c_h and _c_h != _v_h:
            NUTS2_VERS_REGION[_k_h] = _c_h
            _n_harmo += 1
    if _n_harmo:
        print(f"  [ok] Graphie du référentiel départements appliquée : {_n_harmo} entrée(s) NUTS2->région harmonisée(s) (ex. « Grand-Est »)")
    _NUTS2_TABLE.clear()
    _n2 = jsons["GEO_FRANCE"].get("nuts2_canonique", {})
    for _c in _n2.get("canon", []):
        _NUTS2_TABLE[_cle_nuts2(_c)] = _c
    for _k, _v in (_n2.get("alias", {}) or {}).items():
        _NUTS2_TABLE[_cle_nuts2(_k)] = _v
    print(f"  [ok] GEO_FRANCE.json       : {len(DEPT_NUTS3_FR)} départements/territoires | "
          f"{len(TERRITOIRE_PAR_PAYS)} pays->RUP/PTOM | {len(TERRITOIRE_PAR_CP3)} CP3 | "
          f"{len(_NUTS2_TABLE)} NUTS2 canoniques | {len(NUTS2_PAR_DEPT)} dept->NUTS2")

    # 6) FORMES_JURIDIQUES — codes INSEE + niveaux (le moteur garde son mécanisme
    #    hybride téléchargement + secours, INTACT ; le JSON, éditable, prime
    #    simplement pour les codes qu'il liste — données identiques par défaut)
    if "FORMES_JURIDIQUES" in jsons:
        _fj = jsons["FORMES_JURIDIQUES"]
        try:
            _FORMES_JURIDIQUES.update({str(k): str(v) for k, v in _fj.get("codes", {}).items()})
            _NIVEAUX_INPI.update({str(k): tuple(v) for k, v in _fj.get("niveaux_inpi", {}).items()})
            _NIVEAUX_UTILISATEUR.update({str(k): (tuple(v) if isinstance(v, list) else v)
                                         for k, v in _fj.get("niveaux_utilisateur", {}).items()})
            _CJ_NIV1.update({str(k): str(v) for k, v in _fj.get("cj_niv1", {}).items()})
            _CJ_NIV2.update({str(k): str(v) for k, v in _fj.get("cj_niv2", {}).items()})
            _CJ_NIV3.update({str(k): str(v) for k, v in _fj.get("cj_niv3", {}).items()})
            print(f"  [ok] FORMES_JURIDIQUES.json: {len(_fj.get('codes', {}))} codes | "
                  f"{len(_fj.get('cj_niv3', {}))} CJ niv.3 | {len(_fj.get('niveaux_inpi', {}))} INPI")
        except NameError:
            print("  [!] FORMES_JURIDIQUES.json ignoré (moteur non encore chargé)")
    else:
        print("  [!] FORMES_JURIDIQUES.json absent — tables hybrides du moteur conservées")

    _REFERENTIELS_OK = True
    print(f"  [ok] VILLES_FR            : {len(VILLE_VERS_DEPT):,} communes (ville->département)")
    print("Référentiels chargés.")

def _verifier_referentiels():
    if not _REFERENTIELS_OK:
        raise RuntimeError("Référentiels non chargés — exécute la cellule de chargement "
                           f"en déposant : {', '.join(FICHIERS_REFERENTIELS)}")

# ── 1) Zone géographique + code Etat ────────────────────────────────────────
def _code_iso_fallback(nom_pays):
    """Recherche ISO stricte uniquement (pas de flou : 'Niger'→'Nigeria' est
    un faux positif classique). None si aucune correspondance fiable."""
    try:
        import pycountry
    except ImportError:
        return None
    try:
        return pycountry.countries.lookup(str(nom_pays)).alpha_2
    except LookupError:
        return ISO_ALIAS.get(_norm_g(nom_pays))

# Classe un pays en zone : FR / UE / UK / AELE / AUTRE, et donne son code ISO.
# Sert à distinguer les bénéficiaires français (à enrichir) du reste du monde
# (conservés tels quels dans le fichier GLOBAL).
def zone_et_code_pays(pays):
    """-> (zone FR/UE/UK/AELE/AUTRE, code Etat) ; ('', '') si pays vide."""
    if pays is None or (isinstance(pays, float) and pd.isna(pays)) or str(pays).strip() in ("", "-"):
        return "", ""
    cle = _norm_g(pays)
    cle = _norm_g(ALIAS_PAYS.get(cle, cle))
    if cle in DOM_TOM_RUP_FR:
        return "FR", "FR"
    m = CORRESPONDANCE_ETATS.get(cle)
    if m:
        code, statut = m
        zone = statut if statut in ("FR", "UE", "UK", "AELE") else "AUTRE"
        return zone, code
    return "AUTRE", (_code_iso_fallback(pays) or "")

# ── 2) Nom bénéficiaire nettoyé ─────────────────────────────────────────────
# Nettoie un nom pour l'AFFICHAGE uniquement (colonne « Bénéficiaire corrigé »).
# ATTENTION — RÈGLE INTANGIBLE : ce nom nettoyé n'est JAMAIS utilisé pour la
# recherche. Le moteur cherche toujours sur le nom BRUT du FTS. Nettoyer avant
# de chercher dégradait les résultats (perte de sigles discriminants).
def nettoyer_nom_beneficiaire(nom):
    """Retire UNIQUEMENT les étoiles finales non suivies de texte
    (« NAME* » → « NAME »). Les étoiles internes séparent parfois deux entités
    réellement distinctes → conservées. Noms tout-étoiles (anonymisés) inchangés."""
    if nom is None or (isinstance(nom, float) and pd.isna(nom)):
        return nom
    s = str(nom)
    if re.fullmatch(r"\*+", s.strip()):
        return s
    return s.rstrip("*").rstrip()

# ── 3) Période CFP / Dépense CFP (définitions CMFE) ────────────────────────
_RE_PREFIXE_SPECIAL = re.compile(r"^\s*(o\.|s\.|9\.0\.)", re.IGNORECASE)

def est_programme_hors_cfp(prog):
    t = _norm_g(prog)
    if not t:
        return False
    return t in HORS_CFP_PROGRAMMES or "european development fund" in t

# Rattache une année au Cadre Financier Pluriannuel correspondant
# (2014-2020, 2021-2027, 2028-2034). Sert aux analyses par période de
# programmation budgétaire européenne.
def periode_cfp_depuis_annee(annee):
    m = re.search(r"(19|20)\d{2}", str(annee or ""))
    if not m:
        return ""
    y = int(m.group(0))
    if 2014 <= y <= 2020:
        return "14-20"
    if 2021 <= y <= 2027:
        return "21-27"
    if 2007 <= y <= 2013:
        return "07-13"
    return ""

def depense_cfp(budget_line_name, periode, prog):
    if est_programme_hors_cfp(prog):
        return "Hors CFP"
    t = _norm_g(budget_line_name)
    if periode not in ("14-20", "21-27", "07-13"):
        return ""
    if re.search(r"prior to 2007|before 2007", t):
        return "Avant 2007"
    if re.search(r"2007\s*(?:to|-|\u00e0)\s*2013", t):
        return "2007-2013"
    prev = ("completion" in t and ("previous" in t or "former" in t)) or "former" in t
    if periode == "21-27":
        if "prior to 2014" in t:
            return "2007-2013"
        # sur les lignes 21-27, « completion of … » désigne l'achèvement d'un
        # programme antérieur même sans « previous » (23/24 cas du référentiel)
        if "prior to 2021" in t or prev or "completion" in t:
            return "2014-2020"
        return "2021-2027"
    if periode == "14-20":
        if "prior to 2014" in t or prev:
            return "2007-2013"
        return "2014-2020"
    # 07-13
    if prev or "prior to 2007" in t:
        return "Avant 2007"
    return "2007-2013"


# ═══════════════════════════════════════════════════════════════
# RÉCONCILIATION DE DOUBLONS (v3_13) — 100 % algorithmique, aucune donnée en dur
# ═══════════════════════════════════════════════════════════════
# Problème corrigé : deux lignes qui désignent la MÊME structure mais s'écrivent
# différemment (souvent à cause d'une étoile ou d'un suffixe « ASSOCIATION »)
# obtiennent des TVA différentes — l'une correcte, l'autre non
# (ex. « PARC NATIONAL DE LA REUNION » = bon ; « PARC NATIONAL DE LA REUNION* »
#  attrape par erreur « AMICALE DU PARC… »). Ici, quand une variante a trouvé un
# résultat FIABLE et que la variante de base (nom le plus court, sans étoile)
# existe aussi dans le fichier, on propage le meilleur résultat aux jumelles.
_SUFFIXES_ENTITE = ["ASSOCIATION", "ASSO", "AISBL", "ASBL", "FONDATION",
                    "SA", "SAS", "SARL", "SCIC", "SCOP", "GIE", "EPIC", "EPA"]

def _cle_base_recon(nom):
    """Nom normalisé, tronqué au 1er « * », suffixe d'entité final retiré."""
    s = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode()
    s = s.split("*")[0]
    s = re.sub(r"[^A-Za-z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().upper()
    mots = s.split(" ")
    while len(mots) > 1 and mots[-1] in _SUFFIXES_ENTITE:
        mots.pop()
    return " ".join(mots)

def appliquer_noms_corriges(df, mapping=None):
    """Écrit « Bénéficiaire corrigé » depuis un mapping nom_fts -> nom_corrige."""
    if "Bénéficiaire corrigé" not in df.columns or not mapping:
        return df, 0
    col_nom = next((c for c in df.columns if str(c).strip().lower() == "name of beneficiary"), None)
    if col_nom is None:
        return df, 0
    df = df.copy()
    vals = df["Bénéficiaire corrigé"].tolist()
    noms = df[col_nom].astype(str).tolist()
    n = 0
    for i, nom in enumerate(noms):
        nc = mapping.get(nom) or mapping.get(nom.strip())
        if nc and vals[i] != nc:
            vals[i] = nc; n += 1
    df["Bénéficiaire corrigé"] = vals
    return df, n

def ajouter_nom_api(df, rapport=None):
    """v5_38/v5_40 — Colonne « Nom API » : écriture OFFICIELLE du bénéficiaire
    telle que retournée par l'annuaire des entreprises
    (recherche-entreprises.api.gouv.fr), placée juste APRÈS « Bénéficiaire
    corrigé ». Vide quand aucune fiche API n'a été obtenue pour ce nom.
    v5_40 — TROIS sources fusionnées, par priorité décroissante :
      1. rapport avec TVA RETENUE (passes 0, mémoire et 1) ;
      2. passe 2 — TVA déjà fournies par la Commission (NOM_API_PAR_FTS),
         la grande majorité du fichier, absente du rapport jusqu'ici ;
      3. autres lignes du rapport (fiche API obtenue mais candidat écarté).
    Jamais une variante FTS, contrairement à « Bénéficiaire corrigé »."""
    df = df.copy()
    if "Nom API" in df.columns:
        df = df.drop(columns=["Nom API"])
    col_nom = next((c for c in df.columns if str(c).strip().lower() == "name of beneficiary"), None)
    mapping = {}
    for r in (rapport or []):                       # 1) rapport, TVA retenue
        na = str(r.get("Nom_API") or "").strip()
        if na and str(r.get("Retenu_fichier") or "").strip().lower() == "oui":
            mapping.setdefault(str(r.get("Nom_FTS") or "").strip(), na)
    for _n, _na in NOM_API_PAR_FTS.items():          # 2) passe 2 (TVA existantes)
        _na = str(_na or "").strip()
        if _na:
            mapping.setdefault(str(_n).strip(), _na)
    for r in (rapport or []):                       # 3) reste du rapport
        na = str(r.get("Nom_API") or "").strip()
        if na:
            mapping.setdefault(str(r.get("Nom_FTS") or "").strip(), na)
    if col_nom is None:
        df["Nom API"] = ""
        print("⚠️ Colonne « Name of beneficiary » introuvable — « Nom API » vide.")
    else:
        noms = df[col_nom].astype(str).str.strip()
        df["Nom API"] = [mapping.get(n, "") for n in noms]
    cols = [c for c in df.columns if c != "Nom API"]
    if "Bénéficiaire corrigé" in cols:
        cols.insert(cols.index("Bénéficiaire corrigé") + 1, "Nom API")
    elif col_nom in cols:
        cols.insert(cols.index(col_nom) + 1, "Nom API")
    else:
        cols.append("Nom API")
    df = df[cols]
    n_ok = int((df["Nom API"] != "").sum())
    print(f"Nom API : {n_ok:,} ligne(s) renseignée(s) depuis l'annuaire")
    return df

# ÉTAPE A2 — RÉCONCILIATION DES DOUBLONS
# Un même bénéficiaire peut apparaître sous plusieurs graphies (avec/sans
# étoile, accents, ponctuation). Si l'une des graphies a trouvé une TVA et
# l'autre non, on ALIGNE la seconde sur la première : c'est la même structure.
# Le rapprochement se fait sur une clé normalisée, jamais sur du « à peu près ».
def reconcilier_doublons(df_out, cache, col_nom, col_tva, rows_api, seuil_fiable=90):
    """Regroupe les bénéficiaires par nom de base ; propage la meilleure TVA
    fiable du groupe aux variantes non trouvées ou moins sûres. Ne fusionne
    QUE si une variante « de base » (la plus courte du groupe) partage la clé —
    ce qui évite d'unir deux entités réellement distinctes.
    Retourne (df_out, nb_lignes_corrigées, journal)."""
    # 1) meilleur résultat par nom (depuis le cache de recherche)
    best_par_nom = {}
    for nom, res in cache.items():
        if res and res.get("statut") == "TROUVE" and res.get("tva"):
            best_par_nom[str(nom).strip()] = res

    # 2) grouper les noms présents dans la sous-partie France par clé de base
    from collections import defaultdict
    groupes = defaultdict(list)
    noms_france = df_out.loc[df_out[col_nom].notna(), col_nom].astype(str).str.strip().unique()
    for nom in noms_france:
        groupes[_cle_base_recon(nom)].append(nom)

    # 3) pour chaque groupe multi-variantes, choisir le résultat canonique
    canonique = {}           # clé_base -> res à propager
    for cle, noms in groupes.items():
        if len(noms) < 2 or not cle:
            continue
        candidats = [(n, best_par_nom[n]) for n in noms if n in best_par_nom]
        if not candidats:
            continue
        # entités distinctes : au moins 2 TVA FIABLES différentes dans le groupe
        tvas_fiables = {r.get("tva") for n, r in candidats
                        if r.get("score", 0) >= seuil_fiable and r.get("tva")}
        if len(tvas_fiables) >= 2:
            continue    # ex. deux établissements OCDE distincts -> pas de fusion
        def _rang(item):
            n, r = item
            a_etoile = "*" in n
            score = r.get("score", 0)
            # priorité : score fiable, PUIS nom sans étoile, PUIS nom le plus court
            return (score >= seuil_fiable, not a_etoile, -len(n), score)
        n_best, r_best = max(candidats, key=_rang)
        # n'intervenir que si le meilleur est fiable
        if r_best.get("score", 0) >= seuil_fiable or len(candidats) >= 1:
            canonique[cle] = (n_best, r_best)

    # 4) appliquer : toute ligne dont la variante diverge du canonique est corrigée
    n_corr, journal = 0, []
    mask = df_out[col_nom].notna()
    for idx in df_out[mask].index:
        nom = str(df_out.at[idx, col_nom]).strip()
        cle = _cle_base_recon(nom)
        canon = canonique.get(cle)
        if not canon:
            continue
        n_best, r_best = canon
        if nom == n_best:
            continue
        res_actuel = cache.get(nom)
        tva_actuelle = str(df_out.at[idx, col_tva] or "").strip()
        tva_canon = r_best.get("tva", "")
        if tva_actuelle and tva_actuelle == tva_canon:
            continue
        # SÉCURITÉ 1 : la ligne a déjà une TVA propre trouvée de façon fiable
        # (score >= seuil_fiable) et DIFFÉRENTE -> entité distincte (ex. deux
        # établissements OCDE « *IEA » vs « *AIE »). On NE fusionne PAS.
        if res_actuel and res_actuel.get("statut") == "TROUVE" \
           and res_actuel.get("tva") and res_actuel.get("score", 0) >= seuil_fiable:
            continue
        # SÉCURITÉ 2 : ne pas écraser une variante sans étoile déjà trouvée
        if res_actuel and res_actuel.get("statut") == "TROUVE" \
           and res_actuel.get("score", 0) >= r_best.get("score", 0) \
           and "*" not in nom:
            continue
        # Cohérence avec le filtre fichier : ne propager la TVA dans le FICHIER
        # que si le jumeau canonique atteint le seuil de rétention.
        if r_best.get("score", 0) < globals().get("SEUIL_FICHIER_TVA", 95):
            continue
        df_out.at[idx, col_tva] = tva_canon
        df_out.at[idx, "SIREN"]           = r_best.get("siren", "")
        df_out.at[idx, "SIRET"]           = r_best.get("siret", "")
        df_out.at[idx, "Forme_juridique"] = r_best.get("forme_juridique", "")
        df_out.at[idx, "Niveau_I"]   = r_best.get("niveau_i", "")
        df_out.at[idx, "Niveau_II"]  = r_best.get("niveau_ii", "")
        df_out.at[idx, "Niveau_III"] = r_best.get("niveau_iii", "")
        df_out.at[idx, "Code_NAF_APE"]    = r_best.get("code_naf", "")
        df_out.at[idx, "Etat_entreprise"] = r_best.get("etat", "")
        rows_api[idx] = r_best.get("score", 0)
        NOM_CORRIGE_PAR_FTS[nom] = n_best     # « Bénéficiaire corrigé » = variante de base
        n_corr += 1
        journal.append((nom, n_best, tva_canon))
    return df_out, n_corr, journal


# ── 3bis) Référence projet dérivée du libellé de l'objet ────────────────────
# v5_41 — « Project / Contract Reference » n'est renseignée par la Commission
# qu'à partir du millésime 2025 ; avant, elle vaut « N/A - Not applicable »
# alors que l'information est PRÉSENTE en tête de « Subject of grant or
# contract » :
#     « 101177660 - MAKE-A-THEK - MODULAR LIBRARY MAKERSPACES… »
#      └ la référence du projet
# RÈGLE (utilisateur) : une référence de projet est TOUJOURS un NOMBRE. Tout ce
# qui n'est pas numérique (codes TEN-T « 2012-DE-17022-S », codes à barres
# obliques « VS/2014/0582 », codes à tirets « TA-1117 »…) n'est PAS une
# référence et n'est donc jamais capturé.
# On crée UNE colonne « Référence projet complétée », placée juste après sa
# source (règle pipeline : la colonne d'origine n'est jamais modifiée) :
#     • valeur de la Commission si elle existe (recopiée telle quelle) ;
#     • sinon le nombre trouvé en tête du libellé ;
#     • sinon « NA » (rien n'est inventé).

COL_REF_PROJ_C = "Référence projet complétée"

# Longueur admise pour une référence numérique : 6 chiffres (FP7 : 613960) à
# 9 chiffres (Horizon Europe : 101177660). En dessous de 6, on tomberait sur des
# années ou des numéros d'ordre (« 2013 11 02 OPTTEST… ») -> jamais capturés.

# 1) Le libellé COMMENCE par le nombre : « 643271 - ERACoSysMed - … »,
#    « 605159 ENETRAP III », « 101177660 - MAKE-A-THEK - … ».
_RE_REF_NUM = re.compile(r'^\s*"?\s*(\d{6,9})(?!\d)')
# 2) Le libellé commence par un CODE D'APPEL (lettres + tirets) suivi du nombre :
#    « KBBE-2013 613960 SMARTBEES » -> 613960
#    « FP7-SEC-2013 PANDHUB N 607433 … » -> 607433
#    Le code de tête désigne l'appel à propositions, pas le projet ; on ne
#    remonte que le NOMBRE, et uniquement s'il apparaît dans les 60 premiers
#    caractères (au-delà, ce n'est plus un identifiant mais du texte libre).
_RE_REF_CODE_PUIS_NUM = re.compile(
    r'^\s*"?\s*[A-Z][A-Z0-9]{1,9}(?:-[A-Z0-9]{1,6}){1,3}\b(?:.{0,60}?)\b(\d{6,9})(?!\d)', re.S)

def extraire_reference_objet(libelle):
    """Libellé « Subject of grant or contract » -> (référence, méthode).
    Chaînes vides si aucun nombre de référence n'est identifiable."""
    if libelle is None or (isinstance(libelle, float) and pd.isna(libelle)):
        return "", ""
    t = str(libelle).strip()
    if not t:
        return "", ""
    m = _RE_REF_NUM.match(t)
    if m:
        return m.group(1), "NUM_EN_TETE"
    m = _RE_REF_CODE_PUIS_NUM.match(t)
    if m:
        return m.group(1), "NUM_APRES_CODE_APPEL"
    return "", ""


# Valeurs de la Commission considérées comme « à compléter ».
_VIDES_REF = {"", "-", "--", "NA", "N/A", "N/A - NOT APPLICABLE", "NOT APPLICABLE",
              "NON APPLICABLE", "NAN", "NONE", "NULL"}

def _ref_vide(v):
    """Vrai si la cellule est vide ou porte un « N/A » de la Commission."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return True
    return str(v).strip().upper() in _VIDES_REF


def deriver_reference_projet(df):
    """Ajoute « Référence projet complétée » juste après « Project / Contract
    Reference » : valeur de la Commission, sinon nombre extrait du libellé,
    sinon « NA ». Retourne (df, colonnes_ajoutées)."""
    def _tr(*fragments):
        for c in df.columns:
            bas = str(c).strip().lower()
            if all(f in bas for f in fragments):
                return c
        return None

    col_sujet = _tr("subject of grant")
    col_proj  = _tr("project", "reference")

    if not col_sujet:
        print("⚠️ Colonne « Subject of grant or contract » introuvable — "
              "référence projet non dérivée.")
        return df, []

    df = df.drop(columns=[c for c in (COL_REF_PROJ_C,) if c in df.columns])

    origine = (df[col_proj] if (col_proj and col_proj in df.columns)
               else pd.Series([""] * len(df), index=df.index))
    valeurs, n_comm, n_deriv, n_na = [], 0, 0, 0
    stats_meth, non_reconnus = {}, []
    for _o, _s in zip(origine, df[col_sujet]):
        if not _ref_vide(_o):                     # la Commission l'a renseignée
            valeurs.append(str(_o).strip()); n_comm += 1
            continue
        _r, _m = extraire_reference_objet(_s)
        if _r:                                    # nombre extrait du libellé
            valeurs.append(_r); n_deriv += 1
            stats_meth[_m] = stats_meth.get(_m, 0) + 1
        else:                                     # aucun nombre -> NA
            valeurs.append("NA"); n_na += 1
            _t = str(_s).strip()
            if _t and _t.lower() != "nan":
                non_reconnus.append(_t)
    df[COL_REF_PROJ_C] = valeurs

    # Placement : juste après « Project / Contract Reference »
    if col_proj and col_proj in df.columns:
        cols = [c for c in df.columns if c != COL_REF_PROJ_C]
        cols.insert(cols.index(col_proj) + 1, COL_REF_PROJ_C)
        df = df[cols]

    # Bilan (liste, pas de barre de progression)
    print(f"Référence projet complétée (source : « {col_sujet} ») :")
    print(f"   - lignes totales                      : {len(df):,}")
    print(f"   - reprises de la Commission           : {n_comm:,}")
    print(f"   - dérivées du libellé                 : {n_deriv:,}")
    print(f"   - « NA » (aucun nombre identifiable)  : {n_na:,}")
    for _m, _c in sorted(stats_meth.items(), key=lambda x: -x[1]):
        print(f"       [{_m:<22}] {_c:,}")
    if non_reconnus:
        formes = list(dict.fromkeys(non_reconnus))
        print(f"   Libellés sans référence numérique ({len(formes):,} formes distinctes) — "
              f"15 premiers :")
        for _i, _s in enumerate(formes[:15], 1):
            print(f"       [{_i}/15] {_s[:100]}")
    return df, [COL_REF_PROJ_C]


# ── 4) Enrichissement global ────────────────────────────────────────────────
def enrichir_global(df):
    """Ajoute les colonnes globales sur TOUTES les lignes. Retourne
    (df, liste_colonnes_ajoutées)."""
    _verifier_referentiels()
    df = df.copy()

    def _tr(nom_exact):
        for c in df.columns:
            if str(c).strip().lower() == nom_exact:
                return c
        for c in df.columns:
            if nom_exact in str(c).lower():
                return c
        return None

    col_nom   = _tr("name of beneficiary")
    col_pays  = _tr("beneficiary country")
    col_annee = _tr("year")
    col_bln   = _tr("budget line name")
    col_prog  = _tr("programme name")

    ajout = ["Bénéficiaire corrigé", "FR/UE/UK/AELE/AUTRE", "Etats",
             "Période CFP", "Dépense CFP", "Sous catégorie"]
    for _c in ajout:
        if _c in df.columns:
            df = df.drop(columns=[_c])

    n = len(df)

    # Nom nettoyé
    if col_nom:
        df["Bénéficiaire corrigé"] = [nettoyer_nom_beneficiaire(v) for v in df[col_nom]]
    else:
        df["Bénéficiaire corrigé"] = ""
        print("⚠️ Colonne « Name of beneficiary » introuvable — nom nettoyé vide.")

    # Zone + Etats
    pays_sans_code = {}
    zones, codes = [], []
    if col_pays:
        cache_p = {}
        for v in df[col_pays]:
            k = _norm_g(v)
            if k not in cache_p:
                cache_p[k] = zone_et_code_pays(v)
            z, c = cache_p[k]
            zones.append(z); codes.append(c)
            if z and not c:
                pays_sans_code[str(v)] = pays_sans_code.get(str(v), 0) + 1
    else:
        zones = [""] * n; codes = [""] * n
        print("⚠️ Colonne pays introuvable — zone/Etats vides.")
    df["FR/UE/UK/AELE/AUTRE"] = zones
    df["Etats"] = codes

    # Période / Dépense CFP
    annees_hors_plage = {}
    periodes, depenses = [], []
    vals_annee = df[col_annee] if col_annee else pd.Series([""] * n, index=df.index)
    vals_bln   = df[col_bln]   if col_bln   else pd.Series([""] * n, index=df.index)
    vals_prog  = df[col_prog]  if col_prog  else pd.Series([""] * n, index=df.index)
    if not col_annee:
        print("⚠️ Colonne « Year » introuvable — Période CFP vide.")
    lignes_hors_cfp = {}
    cache_cfp = {}
    for an, bln, prog in zip(vals_annee, vals_bln, vals_prog):
        ck = (str(an), _norm_g(bln), _norm_g(prog))
        if ck not in cache_cfp:
            if PERIODE_HORS_CFP_POUR_INSTRUMENTS and est_programme_hors_cfp(prog):
                per = "Hors CFP"
            else:
                per = periode_cfp_depuis_annee(an)
            dep = depense_cfp(bln, per, prog)
            cache_cfp[ck] = (per, dep)
        per, dep = cache_cfp[ck]
        periodes.append(per); depenses.append(dep)
        if per in ("07-13", "") and col_annee:
            annees_hors_plage[str(an)] = annees_hors_plage.get(str(an), 0) + 1
        if dep == "Hors CFP":
            lignes_hors_cfp[str(prog)] = lignes_hors_cfp.get(str(prog), 0) + 1
    df["Période CFP"] = periodes
    df["Dépense CFP"] = depenses

    # Sous catégorie
    progs_sans_scat = {}
    sous_cats = []
    if col_prog:
        for prog in vals_prog:
            sc = SCAT_PAR_PROG.get(_norm_g(prog))
            if sc:
                sous_cats.append(sc[0])
            else:
                sous_cats.append("NA")
                if str(prog).strip() and str(prog).lower() != "nan":
                    progs_sans_scat[str(prog)] = progs_sans_scat.get(str(prog), 0) + 1
    else:
        sous_cats = [""] * n
        print("⚠️ Colonne « Programme name » introuvable — Sous catégorie vide.")
    df["Sous catégorie"] = sous_cats

    # Placement des colonnes
    cols = [c for c in df.columns if c not in ajout]
    if col_nom and col_nom in cols:
        cols.insert(cols.index(col_nom) + 1, "Bénéficiaire corrigé")
    else:
        cols.append("Bénéficiaire corrigé")
    if col_pays and col_pays in cols:
        _i = cols.index(col_pays)
        cols[_i + 1:_i + 1] = ["FR/UE/UK/AELE/AUTRE", "Etats"]
    else:
        cols += ["FR/UE/UK/AELE/AUTRE", "Etats"]
    # « Sous catégorie » juste AVANT « Programme name », puis « Période CFP » et
    # « Dépense CFP » juste APRÈS « Programme name » (demande utilisateur v3_17).
    if col_prog and col_prog in cols:
        cols.insert(cols.index(col_prog), "Sous catégorie")
        _ip = cols.index(col_prog)
        cols[_ip + 1:_ip + 1] = ["Période CFP", "Dépense CFP"]
    else:
        cols += ["Sous catégorie", "Période CFP", "Dépense CFP"]
    df = df[cols]

    # v5_41 — Référence projet complétée (dérivée du libellé de l'objet quand la
    # Commission ne la fournit pas — cas de tous les millésimes avant 2025).
    df, _cols_ref = deriver_reference_projet(df)
    ajout = ajout + _cols_ref

    # ── Classification des projets (Mono / Collaboratif / Indéterminé) ──
    # Un projet = une « Reference of the Legal Commitment (LC) ». On compte les
    # bénéficiaires DISTINCTS par projet (sur le nom de base : avant étoile,
    # suffixe d'entité retiré) pour absorber les variantes d'écriture. 1 -> Mono,
    # >=2 -> Collaboratif, LC vide -> Indéterminé.
    col_lc = next((c for c in df.columns
                   if "legal commitment" in str(c).lower()
                   or str(c).strip().lower() == "lc"), None)
    col_nom_b = next((c for c in df.columns if str(c).strip().lower() == "name of beneficiary"), None)
    col_pays_b = next((c for c in df.columns
                       if "beneficiary country" in str(c).lower()
                       or str(c).strip().lower() in ("country", "pays")), None)
    col_ville_b = next((c for c in df.columns if str(c).strip().lower() == "city"), None)
    col_adr_b = next((c for c in df.columns if "address" in str(c).lower() or "adresse" in str(c).lower()), None)
    if col_lc and col_nom_b:
        def _norm_txt(x):
            t = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
            return re.sub(r"[^A-Za-z0-9]+", " ", t).strip().upper()
        def _est_anonyme(x):
            s = str(x).strip()
            return (s == "" or bool(re.fullmatch(r"\*+", s)))
        def _base_benef(n):
            b = _norm_txt(str(n).split("*")[0])
            for _suf in ("ASSOCIATION","ASSO","AISBL","ASBL","GMBH","LTD","BV","SPA","SRL",
                         "SA","SAS","SARL","GROUP","GROUPE"):
                if b.endswith(" " + _suf):
                    b = b[:-len(_suf)-1]
            return b.strip()
        def _lc_vide(v):
            return (v is None or (isinstance(v, float) and pd.isna(v))
                    or str(v).strip() in ("", "-", "nan"))
        # ── Identité d'un bénéficiaire (comptage des distincts par projet) ──
        # Le bénéficiaire est identifié par son NOM (« Bénéficiaire corrigé »,
        # ramené au nom de base : étoile et suffixes d'entité absorbés). Deux
        # lignes sont EN PLUS reconnues comme le MÊME bénéficiaire si elles
        # partagent la même TVA (identifiant le plus fort) ou la même ADRESSE
        # COMPLÈTE (pays + rue + CP + ville) — cas des noms écrits en plusieurs
        # langues. Fusion transitive (union-find) par projet. Les lignes
        # anonymisées sans adresse comptent chacune pour un bénéficiaire
        # (convention FTS : 1 ligne = 1 bénéficiaire par engagement).
        col_src_nom = "Bénéficiaire corrigé" if "Bénéficiaire corrigé" in df.columns else col_nom_b
        col_tva_b = next((c for c in df.columns
                          if "vat" in str(c).lower() and "beneficiary" in str(c).lower()), None)
        _c_cp_b = next((c for c in df.columns if "postal" in str(c).lower()
                        and "corrig" not in str(c).lower()), None)
        def _tva_cle(v):
            s = str(v or "").strip().upper().replace(" ", "")
            return s if (len(s) >= 8 and s not in ("-", "NAN", "AUTRE") and s != "") else ""
        def _adr_cle(_ix):
            _ad = df.at[_ix, col_adr_b] if col_adr_b else ""
            if _est_anonyme(_ad):
                return ""
            _adn = _norm_txt(_ad)
            if len(_adn.replace(" ", "")) < 8:      # adresse trop courte = non discriminante
                return ""
            _pa  = _norm_txt(df.at[_ix, col_pays_b]) if col_pays_b else ""
            _vi  = df.at[_ix, col_ville_b] if col_ville_b else ""
            _cpv = df.at[_ix, _c_cp_b] if _c_cp_b else ""
            _vin = "" if _est_anonyme(_vi) else _norm_txt(_vi)
            _cpn = "" if _est_anonyme(_cpv) else str(_cpv).strip().upper()
            return _pa + "|" + _adn + "|" + _cpn + "|" + _vin
        # clés d'identité de chaque ligne (position -> liste de clés)
        _keys_list = []
        for _ix, _nv in zip(df.index, df[col_src_nom]):
            _ks = []
            _b = _base_benef(_nv)
            if _b:
                _ks.append(("N", _b))
            if col_tva_b:
                _t = _tva_cle(df.at[_ix, col_tva_b])
                if _t:
                    _ks.append(("T", _t))
            _a = _adr_cle(_ix)
            if _a:
                _ks.append(("A", _a))
            _keys_list.append(_ks)
        _tmp = pd.DataFrame({"lc": df[col_lc].astype(str).str.strip(),
                             "vide": df[col_lc].map(_lc_vide)}, index=df.index)
        # union-find PAR PROJET : lignes partageant nom OU TVA OU adresse fusionnent
        from collections import defaultdict as _dd
        _pos_par_lc = _dd(list)
        for _p, (_lcv, _vd) in enumerate(zip(_tmp["lc"], _tmp["vide"])):
            if not _vd:
                _pos_par_lc[_lcv].append(_p)
        _nb = {}
        for _lcv, _ps in _pos_par_lc.items():
            _par = list(range(len(_ps)))
            def _find(x):
                while _par[x] != x:
                    _par[x] = _par[_par[x]]; x = _par[x]
                return x
            _proprietaire = {}
            for _j, _p in enumerate(_ps):
                for _k in _keys_list[_p]:
                    if _k in _proprietaire:
                        _ra, _rb = _find(_j), _find(_proprietaire[_k])
                        if _ra != _rb:
                            _par[_ra] = _rb
                    else:
                        _proprietaire[_k] = _j
            _nb[_lcv] = len({_find(_j) for _j in range(len(_ps))})
        # numéro de projet séquentiel par LC (ordre d'apparition)
        _num_lc, _cpt = {}, 0
        _num_col, _type_col = [], []
        for _lcv, _vd in zip(_tmp["lc"], _tmp["vide"]):
            if _vd:
                _num_col.append(""); _type_col.append("Indéterminé"); continue
            if _lcv not in _num_lc:
                _cpt += 1; _num_lc[_lcv] = _cpt
            _num_col.append(_num_lc[_lcv])
            _type_col.append("Mono" if _nb.get(_lcv, 0) <= 1 else "Collaboratif")
        df["N° projet"] = _num_col
        df["Type de projet"] = _type_col
        _cols2 = [c for c in df.columns if c not in ("N° projet", "Type de projet")]
        _il = _cols2.index(col_lc)
        _cols2[_il + 1:_il + 1] = ["N° projet", "Type de projet"]
        df = df[_cols2]
        _nmono = _type_col.count("Mono"); _ncol = _type_col.count("Collaboratif"); _nind = _type_col.count("Indéterminé")
        print(f"Projets : {len(_num_lc):,} distincts | lignes Mono {_nmono:,} | "
              f"Collaboratif {_ncol:,} | Indéterminé {_nind:,}")
    else:
        print("⚠️ Colonne « Reference of the Legal Commitment (LC) » introuvable — "
              "classification des projets ignorée.")

    # Bilan (liste, pas de barre)
    print(f"Zone géographique : " + " | ".join(f"{z} {zones.count(z):,}" for z in ("FR", "UE", "UK", "AELE", "AUTRE")))
    print("Période CFP : " + " | ".join(f"{p or '(vide)'} {periodes.count(p):,}" for p in sorted(set(periodes))))
    print("Dépense CFP : " + " | ".join(f"{d or '(vide)'} {depenses.count(d):,}" for d in sorted(set(depenses))))
    suspects = {}
    for prog in vals_prog:
        t = _norm_g(prog)
        if t and _RE_PREFIXE_SPECIAL.match(t) and not est_programme_hors_cfp(prog) \
           and t not in PREFIXE_SPECIAL_CFP:
            suspects[str(prog)] = suspects.get(str(prog), 0) + 1
    if suspects:
        print("⚠️ Programme(s) à préfixe O./S./9.0. ABSENTS de CFP_REGLES.json "
              "(Dépense laissée selon année/texte — à confirmer) :")
        for p, nn in sorted(suspects.items()):
            print(f"   - {p} ({nn:,} lignes)")
    if lignes_hors_cfp:
        print("Instruments hors CFP (Dépense = Hors CFP) :")
        for p, nn in sorted(lignes_hors_cfp.items()):
            print(f"   - {p} ({nn:,} lignes)")
    if pays_sans_code:
        print(f"⚠️ {len(pays_sans_code)} pays sans code Etat (absents d'ETATS.json + ISO introuvable) :")
        for p, nn in sorted(pays_sans_code.items()):
            print(f"   - {p} ({nn:,} lignes)")
    if annees_hors_plage:
        print(f"⚠️ Années hors 2014-2027 (Période « 07-13 » ou vide) :")
        for a, nn in sorted(annees_hors_plage.items()):
            print(f"   - {a} ({nn:,} lignes)")
    if progs_sans_scat:
        print(f"⚠️ {len(progs_sans_scat)} programme(s) absents de SOUS_CATEGORIES.json (Sous catégorie = NA) :")
        for p, nn in sorted(progs_sans_scat.items()):
            print(f"   - {p} ({nn:,} lignes)")
    return df, ajout

def exporter_global(df, colonnes_ajoutees, nom_fichier):
    """Export du fichier GLOBAL en UNE passe (xlsxwriter — indispensable sur
    ~100 000+ lignes) : colonnes ajoutées teintées BDD7EE, volet figé, filtre."""
    print(f"Export : {nom_fichier} ({len(df):,} lignes)…")
    with pd.ExcelWriter(nom_fichier, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name="DATASET")
        wb, ws = writer.book, writer.sheets["DATASET"]
        fmt_ajout   = wb.add_format({"bg_color": "#" + C_VERT_AJOUT})
        fmt_entete  = wb.add_format({"bg_color": "#" + C_VERT_AJOUT, "bold": True})
        cols = list(df.columns)
        for nomcol in colonnes_ajoutees:
            if nomcol not in cols:
                continue
            ic = cols.index(nomcol)
            ws.set_column(ic, ic, None, fmt_ajout)      # teinte toute la colonne
            ws.write(0, ic, nomcol, fmt_entete)         # ré-écrit l'en-tête en gras
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(df), len(cols) - 1)

print("Étape 0 chargée : fonctions prêtes — référentiels intégrés au notebook (cellule 1bis)")


Étape 0 chargée : fonctions prêtes — référentiels intégrés au notebook (cellule 1bis)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5ter — Étape C : Metro/RUP/PTOM (France uniquement)
# ═══════════════════════════════════════════════════════════════
# RUP : art. 349 TFUE (Guadeloupe, Guyane, Martinique, Mayotte, La Réunion,
# Saint-Martin) ; PTOM : annexe II TFUE (Nouvelle-Calédonie, Polynésie
# française, Wallis-et-Futuna, Saint-Pierre-et-Miquelon, Saint-Barthélemy,
# TAAF). Déduit du pays, puis du code postal corrigé (codes POSTAUX La Poste :
# Saint-Barthélemy = 97133, Saint-Martin = 97150 ; 977xx/978xx = CEDEX de
# La Réunion, désambiguïsés par la ville).

def _norm_t(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().lower()

# Tables chargées depuis GEO_FRANCE.json (cellule de chargement) —
# aucune donnée en dur (v3_7).
TERRITOIRE_PAR_PAYS = {}   # pays/territoire -> RUP / PTOM
TERRITOIRE_PAR_CP3  = {}   # préfixe CP (3 chiffres) -> RUP / PTOM

# ═══════════════════════════════════════════════════════════════
# CELLULE 10 — ÉTAPES C, D, E, SGAE, NAF (qualification des entités)
# ═══════════════════════════════════════════════════════════════
# Cinq enrichissements successifs, tous ADDITIFS (aucune colonne d'origine
# touchée) :
#   C     Metro / RUP / PTOM       — statut territorial européen
#   D     Operateur_Etat + Programme_Operateur
#   E     réconciliation des états « Cessée » faussés
#   SGAE  statut juridique simplifié
#   NAF   libellé de l'activité principale
#
# ÉTAPE C — Metro / RUP / PTOM. Distinction JURIDIQUE européenne, pas
# géographique : les RUP (régions ultrapériphériques, art. 349 TFUE :
# Guadeloupe, Martinique, Guyane, La Réunion, Mayotte, Saint-Martin) font
# partie de l'UE et sont éligibles aux fonds structurels ; les PTOM (annexe II
# TFUE : Nouvelle-Calédonie, Polynésie, Saint-Barthélemy, Wallis-et-Futuna...)
# n'en font PAS partie et relèvent d'un régime d'association distinct.
# Se tromper ici fausse toute analyse d'éligibilité.
# Piège classique : Saint-Barthélemy (97133) = PTOM, Saint-Martin (97150) =
# RUP — deux îles voisines, deux régimes opposés, distingués au code postal.

def _classer_metro_rup_ptom(pays, cp, ville=""):
    p = _norm_t(pays)
    if p in TERRITOIRE_PAR_PAYS:
        return TERRITOIRE_PAR_PAYS[p]
    if p != "france":
        return ""
    c = re.sub(r"\D", "", str(cp or ""))
    if len(c) != 5:
        return ""
    if c == "97133":
        return "PTOM"                 # Saint-Barthélemy
    if c == "97150":
        return "RUP"                  # Saint-Martin
    if c[:3] in ("977", "978"):
        # codes POSTAUX 977xx/978xx = CEDEX de La Réunion (RUP), sauf mention
        # explicite de Saint-Barthélemy (PTOM) / Saint-Martin (RUP) dans la ville
        v = _norm_t(ville)
        if "barth" in v or "gustavia" in v:
            return "PTOM"
        return "RUP"
    if c[:3] in TERRITOIRE_PAR_CP3:
        return TERRITOIRE_PAR_CP3[c[:3]]
    if c[:2] in ("97", "98"):
        return ""
    return "Metro"

def enrichir_metro_rup_ptom(df, col_pays):
    """Ajoute « Metro/RUP/PTOM » juste AVANT « Région_FR » (toutes lignes)."""
    _verifier_referentiels()
    df = df.copy()

    def _tr(nom_exact):
        for c in df.columns:
            if str(c).strip().lower() == nom_exact:
                return c
        for c in df.columns:
            if nom_exact in str(c).lower():
                return c
        return None

    col_cp_corr = "Code postal corrigé" if "Code postal corrigé" in df.columns else _tr("postal code")
    col_ville = _tr("city")
    if "Metro/RUP/PTOM" in df.columns:
        df = df.drop(columns=["Metro/RUP/PTOM"])

    n = len(df)
    vals_pays  = df[col_pays] if col_pays else pd.Series([""] * n, index=df.index)
    vals_cp    = df[col_cp_corr] if col_cp_corr else pd.Series([""] * n, index=df.index)
    vals_ville = df[col_ville] if col_ville else pd.Series([""] * n, index=df.index)

    # v5_2 — colonnes de repli (Étape A3 + nettoyage géographique)
    vals_cps = df["Code postal (SIRENE)"] if "Code postal (SIRENE)" in df.columns else pd.Series([""] * n, index=df.index)
    vals_vls = df["Ville (SIRENE)"] if "Ville (SIRENE)" in df.columns else pd.Series([""] * n, index=df.index)
    vals_reg = df["Région_FR"] if "Région_FR" in df.columns else pd.Series([""] * n, index=df.index)

    # v5_2 — Région -> Metro/RUP/PTOM, DÉRIVÉ des référentiels existants (aucune
    # donnée nouvelle en dur) : territoire du département de la région (RUP art. 349
    # TFUE / PTOM annexe II TFUE via GEO_FRANCE) ; région métropolitaine -> Metro.
    def _cle_region(_s):
        return re.sub(r"[^a-z0-9]", "", _norm_t(_s))   # v5_10 : insensible tirets/espaces

    _terr_region = {}
    for _d, _r in DEPT_REGION_FR.items():
        _t = (TERRITOIRE_PAR_PAYS.get(_norm_t(str(_r).replace("-", " ")))   # v5_10 : « Saint-Barthélemy » -> « saint barthelemy »
              or TERRITOIRE_PAR_CP3.get(str(_d))
              or ("Metro" if not str(_d).startswith(("97", "98")) else ""))
        if not _t:
            continue
        _k = _cle_region(_r)
        if _k in _terr_region and _terr_region[_k] != _t:
            _terr_region[_k] = ""      # conflit (improbable) : on ne classe pas
        else:
            _terr_region[_k] = _t

    metro, n_fr_nonclasse = [], 0
    n_sirene = n_region = 0
    for pays, cp, ville, cps, vls, reg in zip(vals_pays, vals_cp, vals_ville,
                                              vals_cps, vals_vls, vals_reg):
        v = _classer_metro_rup_ptom(pays, cp, ville)
        if v == "" and _norm_t(pays) == "france":
            # Repli 1 (v5_2) : code postal du siège SIRENE (Étape A3) — conserve la
            # logique 97133 / 97150 / 977xx-978xx désambiguïsée par la ville.
            if not _geo_vide(cps):
                _vil = ville if not _geo_vide(ville) else vls
                v = _classer_metro_rup_ptom(pays, cps, _vil)
                if v:
                    n_sirene += 1
            # Repli 2 (v5_2) : Région_FR déjà dérivée (CP SIRENE, NUTS2, pays, ville).
            if v == "" and not _geo_vide(reg):
                v = _terr_region.get(_cle_region(reg), "")   # v5_10 : « Grand Est » = « Grand-Est »
                if v:
                    n_region += 1
        metro.append(v)
        if v == "" and _norm_t(pays) == "france":
            n_fr_nonclasse += 1
    df["Metro/RUP/PTOM"] = metro

    # Placement : juste AVANT « Région_FR » (demande utilisateur v3_7) ;
    # à défaut, après la colonne pays.
    cols = [c for c in df.columns if c != "Metro/RUP/PTOM"]
    if "Région_FR" in cols:
        cols.insert(cols.index("Région_FR"), "Metro/RUP/PTOM")
    elif col_pays and col_pays in cols:
        cols.insert(cols.index(col_pays) + 1, "Metro/RUP/PTOM")
    else:
        cols.append("Metro/RUP/PTOM")
    df = df[cols]

    print(f"Metro/RUP/PTOM : Metro {metro.count('Metro'):,} | RUP {metro.count('RUP'):,} | "
          f"PTOM {metro.count('PTOM'):,} | via CP SIRENE {n_sirene:,} | via Région_FR {n_region:,} | "
          f"non classé (France, géo inexploitable) {n_fr_nonclasse:,}")
    return df


# v5_18 — Reconnaissance par FAMILLE (ARS, Agences de l'eau, Parcs nationaux) :
# ces trois familles comptent chacune de nombreux établissements RÉGIONAUX
# (19 ARS, 6 agences de l'eau, 12 parcs) qui ne seront jamais tous listés par
# SIREN de façon exhaustive et durable (renommages, nouveaux parcs...). On les
# reconnaît donc aussi par un PATTERN de nom, en repli SEULEMENT si le SIREN
# de la ligne n'est déjà identifié dans OPERATEURS_ETAT_SIRENS.
_PATTERNS_FAMILLE_ETAT = [
    (re.compile(r"AGENCE\s+REGIONALE\s+DE\s+SANTE|AGENCE\s+DE\s+SANTE\s+DE|^ARS\b", re.I),
     "155 – soutien des ministères sociaux"),
    (re.compile(r"AGENCE\s+DE\s+L\s*'?\s*EAU", re.I),
     "113 – Paysages, eau et biodiversité"),
    (re.compile(r"PARC\s+NATIONAL|PARC\s+AMAZONIEN", re.I),
     "113 – Paysages, eau et biodiversité"),
]

# ÉTAPE D (complément) — RECONNAISSANCE PAR FAMILLE
# La liste des opérateurs de l'État est nominative (par SIREN). Problème : si
# le FTS écrit « ARS OCCITANIE » au lieu de « AGENCE REGIONALE DE SANTE
# OCCITANIE EPA », et que la recherche TVA a échoué sur cette ligne, l'opérateur
# n'est pas reconnu alors qu'il figure bien au référentiel.
# Ce repli tague la ligne à partir d'un MOTIF DE NOM, pour trois familles dont
# les membres sont nombreux et les libellés instables :
#   • les Agences régionales de santé (18)
#   • les Agences de l'eau (6)
#   • les Parcs nationaux (11)
# Il ne s'applique QUE si l'identification par SIREN a échoué : il ne peut
# donc jamais contredire une donnée sûre.
def _famille_etat(nom):
    """Renvoie le programme chef de file si le nom correspond à une famille
    d'opérateurs de l'État connue (ARS / Agences de l'eau / Parcs nationaux),
    sinon None. Lecture seule, n'écrase jamais un SIREN déjà identifié."""
    for motif, programme in _PATTERNS_FAMILLE_ETAT:
        if motif.search(str(nom or "")):
            return programme
    return None

def marquer_operateurs_etat(df):
    """Étape D (v5_5, révisée v5_27, multi-périodes v5_28) — Colonne
    « Operateur_Etat » (« oui »/« non »), placée juste APRÈS « Etat_entreprise » :
    « oui » si le SIREN de la ligne (colonne SIREN, sinon déduit de la TVA, sinon
    du SIRET) figure dans la liste des opérateurs de l'État ET que l'ANNÉE de la
    ligne (colonne FTS « Year ») tombe dans AU MOINS UNE de ses périodes de
    validité connues (source OPETAT.xlsx, mise à jour périodiquement -- un même
    SIREN peut avoir plusieurs périodes successives sous des noms différents,
    ex. Pôle emploi 2000-2024 puis France Travail 2025-2026) ; « non » sinon.
    Un SIREN opérateur SANS période connue reste considéré valide sur toute la
    plage — comportement documenté et non silencieux (compteur en fin d'exécution).
    Colonne coloriée en bleu à l'export."""
    df = df.copy()
    if not OPERATEURS_ETAT_SIRENS:
        print("Opérateurs de l'État : référentiel vide — étape ignorée.")
        return df
    col_tva = next((c for c in df.columns if "vat" in str(c).lower()), None)
    col_nom_op = next((c for c in df.columns if any(
        m in str(c).lower() for m in ("name of beneficiary", "beneficiary name", "nom benefi", "nom_benef"))), None)
    # v5_27 : même détection de la colonne "Year" que enrichir_global (Étape 0),
    # pour rester cohérent d'une étape à l'autre du pipeline.
    col_annee = next((c for c in df.columns if str(c).strip().lower() == "year"), None)
    if col_annee is None:
        col_annee = next((c for c in df.columns if "year" in str(c).lower()), None)

    vals, progs = [], []
    n_op = n_famille = n_hors_periode = n_annee_illisible = 0
    for idx in df.index:
        s = ""
        v = str(df.at[idx, "SIREN"]).strip() if "SIREN" in df.columns else ""
        if v.isdigit() and len(v) == 9:
            s = v
        elif col_tva is not None:
            _t = df.at[idx, col_tva]
            if not _tva_manquante(_t) and str(_t).strip().upper() != "AUTRE":
                s = siren_depuis_tva(_t) or ""
        if not s and "SIRET" in df.columns:
            v = re.sub(r"\D", "", str(df.at[idx, "SIRET"] or ""))
            if len(v) == 14:
                s = v[:9]
        est_op = s in OPERATEURS_ETAT_SIRENS
        prog_ligne = OPERATEURS_ETAT_PROG.get(s, "") if est_op else ""
        via_famille = False
        if not est_op:
            # repli par FAMILLE (ARS/Agences de l'eau/Parcs nationaux) — seulement
            # si le SIREN n'a pas déjà permis de conclure. Sans SIREN identifié,
            # aucune période individuelle n'est disponible : non restreint par date.
            _prog_fam = _famille_etat(df.at[idx, col_nom_op]) if col_nom_op else None
            if _prog_fam:
                est_op = True
                prog_ligne = _prog_fam
                via_famille = True
                n_famille += 1

        # v5_27 : restriction par période de validité, UNIQUEMENT si le SIREN a
        # une période connue (21/436 à ce jour) ET que l'année de la ligne est
        # lisible. Sinon, comportement inchangé (pas de restriction).
        if est_op and not via_famille and s in OPERATEURS_ETAT_PERIODES and col_annee is not None:
            _brut = df.at[idx, col_annee]
            try:
                _an = int(re.sub(r"\D", "", str(_brut))[:4])
                # v5_28 : la LISTE de périodes (un SIREN peut avoir été opérateur
                # sous plusieurs identités successives -- Pôle emploi/France
                # Travail, ENSTA Paris/ENSTA...) ; "oui" si l'année tombe dans
                # N'IMPORTE LAQUELLE des périodes connues pour ce SIREN.
                # v5_29 : DÉCALAGE N-1 sur la borne de DÉBUT uniquement. Le
                # référentiel OPETAT reconnaît un opérateur à compter de l'année
                # N ; mais l'engagement FTS correspondant est fréquemment
                # enregistré dès l'année N-1 (décalage administratif entre
                # l'engagement et sa reconnaissance officielle). Une ligne FTS
                # de l'année (début-1) est donc acceptée -- la borne de FIN,
                # elle, n'est PAS assouplie (aucune indication en ce sens).
                if not any((_deb - 1) <= _an <= _fin for _deb, _fin in OPERATEURS_ETAT_PERIODES[s]):
                    est_op = False
                    prog_ligne = ""
                    n_hors_periode += 1
            except (ValueError, IndexError):
                n_annee_illisible += 1   # année illisible -> pas de restriction (prudence)

        n_op += est_op
        vals.append("oui" if est_op else "non")
        progs.append(prog_ligne if est_op else "AUTRE")
    for _c in ("Operateur_Etat", "Programme_Operateur"):
        if _c in df.columns:
            df = df.drop(columns=[_c])
    df["Operateur_Etat"] = vals
    df["Programme_Operateur"] = progs
    ordre = [c for c in df.columns if c not in ("Operateur_Etat", "Programme_Operateur")]
    if "Etat_entreprise" in ordre:
        _p = ordre.index("Etat_entreprise") + 1
        ordre.insert(_p, "Operateur_Etat"); ordre.insert(_p + 1, "Programme_Operateur")
    else:
        ordre += ["Operateur_Etat", "Programme_Operateur"]
    df = df[ordre]
    print(f"Opérateurs de l'État : {n_op:,} ligne(s) « oui » sur {len(df):,} "
          f"(référentiel : {len(OPERATEURS_ETAT_SIRENS)} SIREN, {len(OPERATEURS_ETAT_PERIODES)} "
          f"avec période vérifiée | {n_famille:,} via famille | {n_hors_periode:,} exclu(s) car "
          f"hors période de validité | {n_annee_illisible:,} année illisible, non restreint par prudence)")
    if col_annee is None:
        print("  ⚠️ Colonne « Year » introuvable : aucune restriction par période appliquée "
              "(comportement identique à avant v5_27).")
    return df

print("Étape C chargée : Metro/RUP/PTOM prêt + Étape D opérateurs de l'État")

# ═══════════════════════════════════════════════════════════════
# ÉTAPE E — RÉCONCILIATION DES ÉTATS « CESSÉE » (v5_13)
# ═══════════════════════════════════════════════════════════════
# Beaucoup d'universités et d'établissements publics ont un SIREN historique
# CESSÉ (fusion/réorganisation) coexistant, sous le MÊME nom, avec le SIREN
# ACTUEL toujours actif. Le moteur figé (PENALISER_RADIEES = False) ne pénalise
# pas les entités cessées et retient parfois le prédécesseur -> l'état affiché
# est « Cessée » alors que l'entité fonctionne. Un état faux fausse les analyses.
#
# Cette étape est ADDITIVE et HORS moteur figé : pour chaque ligne dont l'état
# est « Cessée », elle recherche l'unité légale ACTIVE de nom EXACTEMENT identique
# (normalisé) et de MÊME département, via le filtre etat_administratif=A de l'API.
# Si un successeur actif UNIQUE est trouvé, elle remplace SIREN/TVA/SIRET/forme/
# niveaux/NAF/adresse/état par l'entité active. Sinon la ligne reste inchangée
# (une entité réellement fermée sans successeur reste « Cessée » — c'est correct).

_CACHE_SUCCESSEUR_ACTIF = {}   # (nom_normalisé, dept) -> fiche active ou None

def _dept_de_cp(cp):
    d = re.sub(r"\D", "", str(cp or ""))
    if not d:
        return ""
    return d[:3] if d[:2] in ("97", "98") else d[:2]

# ÉTAPE E — LE PROBLÈME DES ÉTATS « CESSÉE » FAUSSÉS
# Beaucoup d'universités et d'EPA ont fusionné : leur ancien SIREN est CESSÉ
# mais porte le MÊME NOM que le nouveau, toujours actif. Le moteur (qui ne
# pénalise pas les entités radiées, cf. PENALISER_RADIEES = False) retient
# parfois l'ancien -> le fichier affiche « Cessée » pour une université qui
# fonctionne, ce qui fausserait toute analyse.
# Correctif : pour chaque ligne « Cessée », on cherche un successeur ACTIF
# portant EXACTEMENT le même nom, dans le MÊME département (filtre
# etat_administratif=A de l'API). Deux garde-fous :
#   • nom exactement identique (pas d'à-peu-près) et même département ;
#   • un seul candidat actif, sinon on ne tranche pas.
# Une entité réellement fermée, sans successeur, reste « Cessée » : c'est
# correct et voulu.
def _chercher_successeur_actif(nom, cp, ville, session):
    """Unité légale ACTIVE de nom EXACTEMENT identique (successeur probable d'une
    entité cessée). Exact-match normalisé + même département -> zéro faux
    rapprochement. Réutilise _normaliser/_infos_entreprise/_siren_vers_tva en
    lecture seule ; ne touche pas au moteur figé."""
    ref = _normaliser(nom)
    dept = _dept_de_cp(cp)
    cle = (ref, dept)
    if cle in _CACHE_SUCCESSEUR_ACTIF:
        return _CACHE_SUCCESSEUR_ACTIF[cle]
    params = {"q": nom, "page": 1, "per_page": 10, "etat_administratif": "A"}
    if dept.isdigit():
        params["departement"] = dept
    resultats = None
    attente = 1.0
    for _ in range(3):
        try:
            time.sleep(DELAI_API)
            r = session.get(API_URL, params=params, timeout=12)
            if r.status_code == 429:
                time.sleep(attente); attente *= 2; continue
            r.raise_for_status()
            resultats = r.json().get("results", [])
            break
        except Exception:
            time.sleep(attente); attente *= 2
    if not resultats:
        _CACHE_SUCCESSEUR_ACTIF[cle] = None
        return None
    # candidats ACTIFS dont le nom normalisé est EXACTEMENT celui recherché
    actifs = {}
    for res in resultats:
        if str(res.get("etat_administratif", "")).strip().upper() != "A":
            continue
        noms = [res.get("nom_complet"), res.get("nom_raison_sociale"), res.get("sigle")]
        if any(champ and _normaliser(champ) == ref for champ in noms):
            s = str(res.get("siren", "")).strip()
            if s:
                actifs[s] = res
    fiche = None
    if len(actifs) == 1:
        res = next(iter(actifs.values()))
        info = _infos_entreprise(res)
        info["tva"] = _siren_vers_tva(info.get("siren", ""))
        fiche = info
    # si plusieurs actifs homonymes : ambigu -> on ne tranche pas (None)
    _CACHE_SUCCESSEUR_ACTIF[cle] = fiche
    return fiche

def reconcilier_etats_cesses(df, colonnes, session=None):
    """Étape E (v5_13) — remplace les entités marquées « Cessée » par leur
    successeur ACTIF quand il existe (même nom exact, même département)."""
    df = df.copy()
    if "Etat_entreprise" not in df.columns:
        print("Réconciliation des états : colonne Etat_entreprise absente — étape ignorée.")
        return df
    col_nom = colonnes["nom"]; col_tva = colonnes.get("tva")
    col_cp = colonnes.get("cp"); col_vl = colonnes.get("ville")
    def _cessee(v):
        return _norm_t(v) in ("cessee", "cessée", "cesse", "radiee", "radiée")
    idx_cesses = [i for i in df.index if _cessee(df.at[i, "Etat_entreprise"])]
    if not idx_cesses:
        print("Réconciliation des états : aucune ligne « Cessée » — rien à faire.")
        return df
    if session is None:
        session = requests.Session()
        session.headers.update({"User-Agent": "FTS-reconciliation-etat/1.0"})
    # regrouper par (nom, dept) pour ne chercher qu'une fois
    groupes = {}
    for i in idx_cesses:
        nom = str(df.at[i, col_nom]).strip()
        cp = df.at[i, col_cp] if col_cp else ""
        groupes.setdefault((nom, _dept_de_cp(cp)), []).append(i)
    print(f"Réconciliation des états « Cessée » : {len(idx_cesses):,} ligne(s), "
          f"{len(groupes):,} entité(s) unique(s) à vérifier…")
    n_corr = n_lignes = 0
    for k, (nom, dept) in enumerate(groupes.keys(), 1):
        idxs = groupes[(nom, dept)]
        cp = df.at[idxs[0], col_cp] if col_cp else ""
        vl = df.at[idxs[0], col_vl] if col_vl else ""
        fiche = _chercher_successeur_actif(nom, cp, vl, session)
        if not fiche or not fiche.get("siren"):
            continue
        for i in idxs:
            df.at[i, "SIREN"] = fiche.get("siren", "")
            if col_tva:
                df.at[i, col_tva] = fiche.get("tva", "") or df.at[i, col_tva]
            df.at[i, "SIRET"] = fiche.get("siret", "")
            df.at[i, "Forme_juridique"] = fiche.get("forme_juridique", "")
            df.at[i, "Niveau_I"]   = fiche.get("niveau_i", "")
            df.at[i, "Niveau_II"]  = fiche.get("niveau_ii", "")
            df.at[i, "Niveau_III"] = fiche.get("niveau_iii", "")
            df.at[i, "Code_NAF_APE"] = fiche.get("code_naf", "")
            df.at[i, "Etat_entreprise"] = "En activité"
            n_lignes += 1
        n_corr += 1
        print(f"  [{k}/{len(groupes)}] {nom[:45]:45s} -> SIREN actif {fiche.get('siren')} "
              f"({len(idxs)} ligne(s))")
    print(f"Réconciliation des états : {n_corr:,} entité(s) corrigée(s) « Cessée » -> « En activité » "
          f"| {n_lignes:,} ligne(s) | {len(groupes)-n_corr:,} sans successeur actif (inchangées)")
    return df

print("Étape E chargée : réconciliation des états « Cessée » -> successeur actif")

# ÉTAPE SGAE — STATUT JURIDIQUE SIMPLIFIÉ
# La nomenclature INSEE compte 269 catégories juridiques : beaucoup trop fin
# pour un tableau de bord. Le SGAE a défini une grille simplifiée
# (« Entreprises », « Etablissements publics », « Collectivités
# territoriales », « Associations et Fondations »...).
# JOINTURE SUR LE CODE, PAS SUR LE LIBELLÉ : le code à 4 chiffres est extrait
# des parenthèses de « Forme_juridique » (ex. « ... (7383) » -> 7383). Le
# libellé n'est fiable qu'à 66 % entre les deux référentiels (abréviations
# « SA » vs « Société anonyme »), le code l'est à 100 %.
def ajouter_referentiel_sgae(df):
    """v5_14 — Ajoute « Référentiel SGAE » (statut juridique simplifié), placée
    JUSTE AVANT « Forme_juridique ». La catégorie est déduite du code juridique
    INSEE niveau III (4 chiffres) présent entre parenthèses dans « Forme_juridique »
    (ex. « ...(7383) » -> « Etablissements publics »). Jointure sur le CODE (clé
    fiable), pas sur le libellé. Codes absents du référentiel -> case vide."""
    df = df.copy()
    if "Forme_juridique" not in df.columns:
        print("Référentiel SGAE : colonne Forme_juridique absente — étape ignorée.")
        return df
    if not REF_SGAE_PAR_CODE:
        print("Référentiel SGAE : référentiel vide — étape ignorée.")
        return df
    def _code_juridique(forme):
        s = str(forme or "")
        m = re.search(r"\((\d{4})\)", s)          # code entre parenthèses
        if m:
            return m.group(1)
        s2 = s.strip()
        return s2 if re.fullmatch(r"\d{4}", s2) else ""   # forme = code nu
    cats, n_ok, n_sanscode, codes_absents = [], 0, 0, {}
    for f in df["Forme_juridique"]:
        code = _code_juridique(f)
        if not code:
            cats.append(""); n_sanscode += 1; continue
        cat = REF_SGAE_PAR_CODE.get(code, "")
        if cat:
            n_ok += 1
        else:
            codes_absents[code] = codes_absents.get(code, 0) + 1
        cats.append(cat)
    if "Référentiel SGAE" in df.columns:
        df = df.drop(columns=["Référentiel SGAE"])
    df["Référentiel SGAE"] = cats
    # placement : juste AVANT « Forme_juridique »
    ordre = [c for c in df.columns if c != "Référentiel SGAE"]
    ordre.insert(ordre.index("Forme_juridique"), "Référentiel SGAE")
    df = df[ordre]
    print(f"Référentiel SGAE : {n_ok:,} ligne(s) catégorisée(s) | {n_sanscode:,} sans code juridique")
    if codes_absents:
        _tot = sum(codes_absents.values())
        _ex = ", ".join(f"{c} (x{n})" for c, n in sorted(codes_absents.items(), key=lambda x:-x[1])[:6])
        print(f"  codes hors référentiel SGAE : {_tot:,} ligne(s) — {_ex}")
    return df

# ÉTAPE NAF — LIBELLÉ DE L'ACTIVITÉ PRINCIPALE
# L'API Recherche d'entreprises renvoie le CODE NAF (« 85.42Z ») mais PAS son
# libellé — vérifié dans le code source de l'API. On joint donc sur le code
# avec la nomenclature NAF rév. 2 embarquée (732 sous-classes).
# Avantage de joindre sur le code plutôt que de lire un libellé API : le
# résultat est uniforme sur TOUTES les lignes, quelle que soit la façon dont
# elles ont été identifiées (passe 1, passe 2, alias, Étape E).
def ajouter_activite_principale(df):
    """v5_17 — Colonne « Activite_principale » (libellé NAF rév. 2 officiel), placée
    JUSTE APRÈS « Code_NAF_APE ». L'API ne renvoie que le CODE (vérifié dans le
    code source de l'API) : le libellé vient de la nomenclature INSEE embarquée
    (732 sous-classes). Code vide/inconnu -> case vide (signalé)."""
    df = df.copy()
    if "Code_NAF_APE" not in df.columns:
        print("Activité principale : colonne Code_NAF_APE absente — étape ignorée.")
        return df
    if not NAF_LIBELLE_PAR_CODE:
        print("Activité principale : nomenclature NAF vide — étape ignorée.")
        return df
    libs, n_ok, inconnus = [], 0, {}
    for c in df["Code_NAF_APE"]:
        code = str(c or "").strip().upper()
        lib = NAF_LIBELLE_PAR_CODE.get(code, "")
        if lib:
            n_ok += 1
        elif code and code not in ("", "AUTRE", "NAN", "-"):
            inconnus[code] = inconnus.get(code, 0) + 1
        libs.append(lib)
    if "Activite_principale" in df.columns:
        df = df.drop(columns=["Activite_principale"])
    df["Activite_principale"] = libs
    ordre = [c for c in df.columns if c != "Activite_principale"]
    ordre.insert(ordre.index("Code_NAF_APE") + 1, "Activite_principale")
    df = df[ordre]
    print(f"Activité principale : {n_ok:,} libellé(s) NAF renseigné(s)")
    if inconnus:
        _ex = ", ".join(f"{k} (x{v})" for k, v in sorted(inconnus.items(), key=lambda x: -x[1])[:5])
        print(f"  codes NAF hors nomenclature : {sum(inconnus.values()):,} ligne(s) — {_ex}")
    return df

# ÉTAPE NAF (suite) — SECTION NAF 2025 (NACE Rev. 2.1)
# La section (A-V) se déduit de la DIVISION (2 premiers chiffres) du code NAF,
# via la table hiérarchique du fichier INSEE NAF 2025 fourni par l'utilisateur.
# ATTENTION : en NAF 2025 les LETTRES de section diffèrent du NAF rév.2/2008
# (ex. K=Télécom/info, L=Finance en 2025) et la division 45 (commerce/réparation
# auto) a disparu -> les codes 45.* n'ont pas de section 2025 (laissés vides).
def ajouter_section_naf(df):
    """v5_22 — Colonne « Section_NAF » (section NAF 2025 « Lettre – Intitulé »),
    placée JUSTE APRÈS « Activite_principale ». Déduite de la division (2 chiffres)
    du Code_NAF_APE via la table INSEE NAF 2025 embarquée. Division absente de la
    table (ex. 45) ou code vide -> case vide (signalé)."""
    df = df.copy()
    if "Code_NAF_APE" not in df.columns:
        print("Section NAF : colonne Code_NAF_APE absente — étape ignorée.")
        return df
    if not SECTION_PAR_DIVISION or not SECTION_LIBELLE_PAR_LETTRE:
        print("Section NAF : table NAF 2025 vide — étape ignorée.")
        return df
    vals, n_ok, sans = [], 0, {}
    for c in df["Code_NAF_APE"]:
        code = str(c or "").strip().upper()
        div = re.sub(r"\D", "", code)[:2]
        lettre = SECTION_PAR_DIVISION.get(div, "")
        if lettre:
            vals.append(f"{lettre} – {SECTION_LIBELLE_PAR_LETTRE.get(lettre, '')}")
            n_ok += 1
        else:
            vals.append("")
            if code and code not in ("", "AUTRE", "NAN", "-") and div:
                sans[div] = sans.get(div, 0) + 1
    if "Section_NAF" in df.columns:
        df = df.drop(columns=["Section_NAF"])
    df["Section_NAF"] = vals
    ordre = [c for c in df.columns if c != "Section_NAF"]
    _anchor = "Activite_principale" if "Activite_principale" in ordre else "Code_NAF_APE"
    ordre.insert(ordre.index(_anchor) + 1, "Section_NAF")
    df = df[ordre]
    print(f"Section NAF 2025 : {n_ok:,} section(s) renseignée(s)")
    if sans:
        _ex = ", ".join(f"division {k} (x{v})" for k, v in sorted(sans.items(), key=lambda x: -x[1])[:5])
        print(f"  divisions hors table NAF 2025 : {sum(sans.values()):,} ligne(s) — {_ex}")
    return df

print("Étape SGAE chargée : colonne « Référentiel SGAE » (statut juridique simplifié)")


Étape C chargée : Metro/RUP/PTOM prêt + Étape D opérateurs de l'État
Étape E chargée : réconciliation des états « Cessée » -> successeur actif
Étape SGAE chargée : colonne « Référentiel SGAE » (statut juridique simplifié)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 6 — Export colorié : TVA (vert/jaune) + nettoyage (vert)
# ═══════════════════════════════════════════════════════════════
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

def exporter_pipeline(df, rows_api, adr_mod, nuts2_mod, col_adresse, col_nuts2,
                      col_cp_corr, col_nuts3, nom_fichier):
    # v5_32 -- SIREN replacé juste APRÈS la colonne Main Registration (le SIREN
    # fourni par la Commission), pour comparer d'un coup d'œil le SIREN de la
    # Commission et celui retenu par le pipeline (identiques la plupart du
    # temps -- Passe 0/mémoire ; utile à l'audit quand ils divergent). Ignoré
    # proprement sur un fichier sans cette colonne (millésimes antérieurs).
    col_main_reg_export = next((c for c in df.columns if "main registration" in str(c).lower()), None)
    if col_main_reg_export and "SIREN" in df.columns:
        ordre = [c for c in df.columns if c != "SIREN"]
        ordre.insert(ordre.index(col_main_reg_export) + 1, "SIREN")
        df = df[ordre]
    print(f"Export : {nom_fichier} ({len(df):,} lignes)…")
    df.to_excel(nom_fichier, index=False, sheet_name="DATASET", engine="openpyxl")
    wb = load_workbook(nom_fichier); ws = wb["DATASET"]
    entetes = [c.value for c in ws[1]]
    def idx_col(nom):
        return (entetes.index(nom) + 1) if nom in entetes else None

    fill_vert  = PatternFill(start_color=COULEUR_LIGNE_VERTE, fill_type="solid")
    fill_jaune = PatternFill(start_color=COULEUR_LIGNE_JAUNE, fill_type="solid")
    fill_cv_v  = PatternFill(start_color=COULEUR_CELL_VERTE,  fill_type="solid")
    fill_cv_o  = PatternFill(start_color=COULEUR_CELL_ORANGE, fill_type="solid")
    fill_ajout = PatternFill(start_color=C_VERT_AJOUT, fill_type="solid")
    font_blanc = Font(bold=True, color="FFFFFF")

    # 1) Lignes TVA trouvées (vert haute conf / jaune moyenne) + cellule TVA
    i_tva = idx_col(next((c for c in entetes if "vat" in str(c).lower() or "tva" in str(c).lower()), ""))
    n_cols = len(entetes)
    for idx, score in rows_api.items():
        r = int(idx) + 2; haute = score >= SEUIL_HAUTE_CONF
        for c in range(1, n_cols + 1):
            ws.cell(row=r, column=c).fill = fill_vert if haute else fill_jaune
        if i_tva:
            cell = ws.cell(row=r, column=i_tva)
            cell.fill = fill_cv_v if haute else fill_cv_o; cell.font = font_blanc

    # 2) Colonne « Adresse réordonnée » (v5_40) : entête en bleu ajout + cellules
    #    RÉELLEMENT réordonnées surlignées. La colonne Address de la Commission
    #    n'est plus touchée (ni modifiée, ni coloriée). Repli : anciens fichiers
    #    sans la nouvelle colonne -> comportement d'avant (colonne d'origine).
    i_adr = idx_col("Adresse réordonnée") or idx_col(col_adresse)
    if idx_col("Adresse réordonnée"):
        ws.cell(row=1, column=i_adr).fill = fill_ajout
        ws.cell(row=1, column=i_adr).font = Font(bold=True)
    if i_adr:
        for idx in adr_mod:
            ws.cell(row=int(idx) + 2, column=i_adr).fill = fill_ajout

    # 3) Cellules NUTS2 corrigées → vert
    i_n2 = idx_col(col_nuts2)
    if i_n2:
        for idx in nuts2_mod:
            ws.cell(row=int(idx) + 2, column=i_n2).fill = fill_ajout

    # 4) Colonnes AJOUTÉES (CP corrigé, NUTS3 FR) → entête + cellules en vert
    for nomcol in (col_cp_corr, col_nuts3):
        ic = idx_col(nomcol)
        if not ic:
            continue
        ws.cell(row=1, column=ic).fill = fill_ajout
        ws.cell(row=1, column=ic).font = Font(bold=True)
        for r in range(2, ws.max_row + 1):
            if ws.cell(row=r, column=ic).value not in (None, ""):
                ws.cell(row=r, column=ic).fill = fill_ajout

    for _nc in ('Bénéficiaire corrigé','Nom API','Référence projet complétée','NUTS2 corrigé','Région_FR','NUTS3_Numéro','FR/UE/UK/AELE/AUTRE','Etats','Metro/RUP/PTOM','Période CFP','Dépense CFP','Sous catégorie','SIREN','SIRET','Forme_juridique','Niveau_I','Niveau_II','Niveau_III','Code_NAF_APE','Activite_principale','Section_NAF','Etat_entreprise','Operateur_Etat','Programme_Operateur','Référentiel SGAE'):
        _ic=idx_col(_nc)
        if not _ic:
            continue
        ws.cell(row=1,column=_ic).fill=fill_ajout
        ws.cell(row=1,column=_ic).font=Font(bold=True)
        for _r in range(2,ws.max_row+1):
            if ws.cell(row=_r,column=_ic).value not in (None,''):
                ws.cell(row=_r,column=_ic).fill=fill_ajout

    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    wb.save(nom_fichier)
    print(f"  Vert/jaune TVA : {len(rows_api):,} | adresses vertes : {len(adr_mod):,} | "
          f"NUTS2 verts : {len(nuts2_mod):,}")

SEUIL_MONTANT_A_CHERCHER = 300_000   # v5_7 : bénéficiaires >= 300 000 € sans TVA retenue

# v5_40 — Colonnes ENTREPRISE (INSEE + référentiel SGAE) : « A CHERCHER » si le
# bénéficiaire pèse >= 300 000 € et n'est ni « République française » ni une
# personne physique anonymisée ; sinon « AUTRE ». (Demande utilisateur : le
# statut « A CHERCHER » vaut aussi pour SIREN, SIRET, les colonnes INSEE et le
# référentiel SGAE, plus seulement la TVA.)
_COLS_FINITION_ENTREPRISE = ("SIREN", "SIRET", "Forme_juridique",
                             "Niveau_I", "Niveau_II", "Niveau_III",
                             "Code_NAF_APE", "Activite_principale",
                             "Section_NAF", "Etat_entreprise",
                             "Référentiel SGAE")

# v5_40 — Toutes les AUTRES colonnes AJOUTÉES par le pipeline : case vide ->
# « AUTRE » (demande utilisateur : plus aucune case vide dans les colonnes
# ajoutées ; les colonnes d'origine de la Commission ne sont jamais touchées).
_COLS_FINITION_AUTRE_SEUL = ("Bénéficiaire corrigé", "Nom API",
                             "Adresse réordonnée", "Code postal corrigé",
                             "NUTS2 corrigé", "Région_FR", "NUTS3 FR",
                             "NUTS3_Numéro", "FR/UE/UK/AELE/AUTRE", "Etats",
                             "Metro/RUP/PTOM", "Période CFP", "Dépense CFP",
                             "Sous catégorie", "Operateur_Etat",
                             "Programme_Operateur")

def appliquer_autre_et_a_chercher(df_final, colonnes):
    """v5_40 — Remplissage final du FICHIER (le rapport garde tout) :
    - TVA + colonnes ENTREPRISE (_COLS_FINITION_ENTREPRISE) : case vide ->
      « A CHERCHER » si le bénéficiaire a perçu au total >= 300 000 € (somme de
      « Beneficiary's contracted amount ») ET n'est ni « République française »
      (décision utilisateur v5_8) ni une personne physique anonymisée (NATURAL
      PERSON, PERSONNE PRIVÉE, Art. 38(7)… — lecture seule de _est_exclu,
      fonction figée) ; sinon « AUTRE ».
    - Autres colonnes AJOUTÉES (_COLS_FINITION_AUTRE_SEUL) : case vide ->
      « AUTRE ».
    Les colonnes d'origine de la Commission ne sont jamais modifiées."""
    df_final = df_final.copy()
    col_tva, col_nom = colonnes["tva"], colonnes["nom"]
    def _vide_cell(v):
        return v is None or (isinstance(v, float) and pd.isna(v)) or str(v).strip() in ("", "-", "nan")
    def _benef_autre(n):
        # jamais « A CHERCHER » pour ces bénéficiaires : tout vide -> « AUTRE »
        return _beneficiaire_sans_tva(n) or _est_exclu(n)
    _col_montant = next((c for c in df_final.columns
                         if "contracted amount" in str(c).lower()
                         and "beneficiary" in str(c).lower()
                         and "estimated" not in str(c).lower()
                         and "commitment" not in str(c).lower()), None)
    _mt_benef = {}
    if _col_montant:
        _tm = df_final[[col_nom, _col_montant]].copy()
        _tm["_m"] = _tm[_col_montant].map(_montant_num)
        _mt_benef = _tm.groupby(_tm[col_nom].astype(str).str.strip())["_m"].sum().to_dict()
    _noms = df_final[col_nom].astype(str).str.strip().tolist()
    n_ac, n_autre = 0, 0
    for _colc in (col_tva,) + _COLS_FINITION_ENTREPRISE:
        if _colc not in df_final.columns:
            continue
        _nv = []
        for _v, _n in zip(df_final[_colc], _noms):
            if not _vide_cell(_v):
                _nv.append(_v)
            elif _mt_benef.get(_n, 0.0) >= SEUIL_MONTANT_A_CHERCHER and not _benef_autre(_n):
                _nv.append("A CHERCHER"); n_ac += 1
            else:
                _nv.append("AUTRE"); n_autre += 1
        df_final[_colc] = _nv
    for _colc in _COLS_FINITION_AUTRE_SEUL:
        if _colc not in df_final.columns:
            continue
        _nv = [("AUTRE" if _vide_cell(_v) else _v) for _v in df_final[_colc]]
        n_autre += sum(1 for _a, _b in zip(_nv, df_final[_colc]) if _a == "AUTRE" and _vide_cell(_b))
        df_final[_colc] = _nv
    print(f"  « A CHERCHER » : {n_ac:,} case(s) (bénéficiaire >= "
          f"{SEUIL_MONTANT_A_CHERCHER:,} €, TVA + colonnes entreprise/INSEE/SGAE) "
          f"| « AUTRE » : {n_autre:,} case(s) vide(s) des colonnes ajoutées")
    return df_final

print("Export pipeline chargé")

Export pipeline chargé


---
## Étape 1 — Charger le fichier brut France (.xlsx/.csv)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 13 — CHARGEMENT : RÉFÉRENTIELS PUIS FICHIER FTS À TRAITER
#
# Deux opérations, dans cet ordre obligatoire :
#   1. charger_referentiels() lit les blocs JSON de la cellule 2 et remplit
#      les dictionnaires globaux. Les compteurs affichés (« 108 départements »,
#      « 31 950 communes », « 732 sous-classes NAF »...) sont votre contrôle
#      de bonne santé : un zéro signale un bloc JSON cassé.
#   2. Le fichier FTS est ensuite chargé dans `df_brut` — sous Colab via le
#      sélecteur de fichier, en local via un chemin.
#
# Le fichier attendu est l'export FTS (.xlsx ou .csv) avec au minimum le nom
# du bénéficiaire et le pays ; TVA, adresse, ville, code postal et montant
# améliorent fortement le résultat quand ils sont présents.
# ═══════════════════════════════════════════════════════════════
REFERENTIELS = localiser_referentiels()
charger_referentiels(REFERENTIELS)

nom_input, contenu = None, None
if IN_COLAB:
    print("\nDépose maintenant le fichier FTS (.xlsx/.csv) :")
    up = _colab_files.upload()
    for _nom, _cont in up.items():
        if _nom.lower().endswith((".xlsx", ".xls", ".csv")):
            nom_input, contenu = _nom, _cont
else:
    ch = input("Chemin du fichier FTS : ").strip().strip('"').strip("'")
    p = Path(ch); nom_input = p.name; contenu = p.read_bytes()

assert contenu is not None, "Aucun fichier FTS (.xlsx/.csv) déposé."

xl = pd.ExcelFile(io.BytesIO(contenu))
sh = next((s for s in xl.sheet_names if "DATASET" in s.upper()), xl.sheet_names[0])
df_brut = pd.read_excel(io.BytesIO(contenu), sheet_name=sh, dtype=str)
print(f"Fichier : {nom_input} — {len(df_brut):,} lignes (onglet '{sh}')")

col_adresse = next((c for c in df_brut.columns if "address" in c.lower() or "adresse" in c.lower()), None)
col_cp      = next((c for c in df_brut.columns if "postal" in c.lower() and "corrig" not in c.lower()), None)
col_nuts2   = next((c for c in df_brut.columns if c.strip().upper() == "NUTS2"), None)
print(f"Colonnes — adresse: '{col_adresse}' | CP: '{col_cp}' | NUTS2: '{col_nuts2}'")


Référentiels intégrés chargés : ETATS, SOUS_CATEGORIES, CFP_REGLES, PAYS_ALIAS, REF_SGAE, NAF_REV2, NAF2025_SECTIONS, OPERATEURS_ETAT, GEO_FRANCE, FORMES_JURIDIQUES, VILLES_FR
  [ok] ETATS.json            : 32 pays
  [ok] SOUS_CATEGORIES.json  : 198 programmes
  [ok] CFP_REGLES.json       : 10 instruments hors CFP | Période instruments = année
  [ok] PAYS_ALIAS.json       : 4 alias libellés | 27 alias ISO | 27 territoires FR
  [ok] REF_SGAE.json         : 262 codes -> catégorie SGAE
  [ok] NAF_REV2.json         : 732 sous-classes | 440 programmes d'opérateurs | 438 période(s) de validité
  [ok] NAF2025_SECTIONS.json : 22 sections | 87 divisions
  [ok] OPERATEURS_ETAT.json  : 443 SIREN d'opérateurs de l'État
  [ok] Graphie du référentiel départements appliquée : 3 entrée(s) NUTS2->région harmonisée(s) (ex. « Grand-Est »)
  [ok] GEO_FRANCE.json       : 108 départements/territoires | 21 pays->RUP/PTOM | 10 CP3 | 30 NUTS2 canoniques | 108 dept->NUTS2
  [ok] FORMES_JURIDIQUES.json: 269 code

Saving FTS 14-25 BRUT.xlsx to FTS 14-25 BRUT.xlsx
Fichier : FTS 14-25 BRUT.xlsx — 1,034,556 lignes (onglet 'Sheet1')
Colonnes — adresse: 'Address' | CP: 'Postal code' | NUTS2: 'NUTS2'


---
## Étape 2 — Nettoyage puis enrichissement TVA

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 15 — ORCHESTRATION : LE DÉROULÉ COMPLET DU TRAITEMENT
# ═══════════════════════════════════════════════════════════════
# C'est LA cellule à lancer une fois toutes les autres exécutées. Elle
# enchaîne les étapes dans un ordre non négociable, chacune dépendant des
# précédentes :
#
#   ÉTAPE 0    Enrichissement GLOBAL (tous pays) : zone UE/AELE/UK, période
#              CFP, nom nettoyé -> produit le fichier GLOBAL.
#   FILTRE     On isole les lignes France : seules elles sont rapprochables
#              de SIRENE.
#   ÉTAPE A    Enrichissement TVA en deux passes (recherche + documentation).
#   ÉTAPE A2   Réconciliation des doublons de graphie.
#   ÉTAPE A3   Complétion géographique par le siège SIRENE.
#   ÉTAPE E    Correction des états « Cessée » faussés.
#   NETTOYAGE  Géographie : CP corrigé, NUTS2, Région_FR, NUTS3.
#   ÉTAPE C    Metro / RUP / PTOM.
#   ÉTAPE D    Opérateurs de l'État + programme chef de file.
#   ÉTAPE SGAE Statut juridique simplifié.
#   ÉTAPE NAF  Libellé d'activité principale.
#   FINITION   « AUTRE » / « A CHERCHER », retrait des colonnes de travail,
#              export du fichier + du rapport, téléchargement.
#
# DURÉE : dominée par les appels API de l'étape A (~0,15 s par appel, souvent
# plusieurs requêtes par bénéficiaire). Compter plusieurs heures sur un gros
# fichier. Ce n'est PAS un problème de puissance machine : c'est le débit
# autorisé par l'API publique qui fixe la limite.
#
# REPRISE APRÈS COUPURE : Colab peut déconnecter une session longue, et les
# caches vivent en mémoire (donc perdus au redémarrage). Pour un très gros
# fichier, découpez-le et traitez-le par lots.
# ═══════════════════════════════════════════════════════════════
# CELLULE 10 - GLOBAL (etape 0) -> filtre France -> TVA (brut) -> nettoyage -> exports
base = Path(nom_input).stem
nom_global  = f"{base}_GLOBAL_ENRICHI.xlsx"
nom_sortie  = f"{base}_FRANCE_PREPARE_ENRICHI.xlsx"
nom_rapport = f"{base}_RAPPORT_TVA.xlsx"

print("ETAPE 0 - Enrichissement GLOBAL (tous pays)")
df_global, cols_globales = enrichir_global(df_brut)
exporter_global(df_global, cols_globales, nom_global)

print("\nFiltrage France (PAYS_FR : France + RUP/COM)")
cols = detecter_colonnes(df_global)
_mask_fr = df_global[cols["pays"]].astype(str).str.strip().str.lower().isin(PAYS_FR)
df_france = df_global[_mask_fr].reset_index(drop=True)
print(f"  {int(_mask_fr.sum()):,} lignes France sur {len(df_global):,}")

# v5_31 -- reprise d'une mémoire de correspondance exportée lors d'un traitement
# antérieur (ex. le millésime 2025) : déposez le fichier ici si vous en avez
# un, sinon laissez tel quel (démarrage à vide, comportement inchangé).
try:
    charger_correspondance_siren("correspondance_siren.json")
except Exception as _e:
    print(f"Mémoire de correspondance non chargée ({_e}) -- démarrage à vide.")

print("\nETAPE A - Enrichissement TVA (sur donnees brutes)")
df_enrichi, rapport, rows_api = enrichir_france(df_france, cols, seuil=SEUIL_SCORE)

# v5_31 -- alimente/complète la mémoire avec les résolutions FIABLES de cette
# exécution (Main Registration + TVA déjà fournie), pour un usage sur un
# AUTRE millésime traité plus tard. Exportée automatiquement en fin de
# traitement (voir finition).
construire_correspondance_siren(df_enrichi, rapport, cols)

print("\nETAPE A2 - Reconciliation des doublons (meme structure, ecritures differentes)")
df_enrichi, _nb_recon, _journal = reconcilier_doublons(
    df_enrichi, CACHE_RECHERCHE_TVA, cols["nom"], cols["tva"], rows_api)
print(f"  {_nb_recon:,} ligne(s) alignee(s) sur leur structure jumelle")
for _av, _ap, _tva in _journal[:20]:
    print(f"    - {_av[:45]:45s} -> {_ap[:35]:35s} [{_tva}]")
if len(_journal) > 20:
    print(f"    ... (+{len(_journal)-20} autres)")
df_enrichi, _nb_nomcorr = appliquer_noms_corriges(df_enrichi, NOM_CORRIGE_PAR_FTS)
df_enrichi = ajouter_nom_api(df_enrichi, rapport)   # v5_38 : écriture officielle annuaire

print("\nETAPE E - Reconciliation des etats CESSEE -> successeur actif (v5_13)")
df_enrichi = reconcilier_etats_cesses(df_enrichi, cols)

print("\nETAPE A3 - Complétion SIRENE des champs géographiques manquants (v5_2)")
df_enrichi = completer_geo_sirene(df_enrichi, cols)

print("\nETAPE B - Nettoyage geographique (affichage)")
df_final, adr_mod, _ = nettoyer_geo(df_enrichi, col_adresse, col_cp, col_nuts2)

print("\nETAPE C - Metro/RUP/PTOM")
df_final = enrichir_metro_rup_ptom(df_final, cols["pays"])

print("\nETAPE D - Marquage des opérateurs de l'État (v5_5)")
df_final = marquer_operateurs_etat(df_final)

print("\nETAPE SGAE - Statut juridique simplifie (referentiel SGAE, v5_14)")
df_final = ajouter_referentiel_sgae(df_final)

print("\nETAPE NAF - Libelle d'activite principale (nomenclature INSEE, v5_17)")
df_final = ajouter_activite_principale(df_final)
df_final = ajouter_section_naf(df_final)   # v5_22 : section NAF 2025

# FINITION v5_40 (fichier seulement, le rapport garde tout) : les cases vides de
# TOUTES les colonnes ajoutées par le pipeline sont remplies. TVA + colonnes
# entreprise/INSEE/référentiel SGAE : « A CHERCHER » (bénéficiaire >= 300 000 €,
# hors République française et personnes physiques anonymisées) sinon « AUTRE » ;
# autres colonnes ajoutées : « AUTRE ». Fonction définie avec l'export.
df_final = appliquer_autre_et_a_chercher(df_final, cols)
# v5_4/v5_9 : les TROIS colonnes de travail SIRENE sont retirées du FICHIER
# (demande utilisateur). Elles ont déjà servi, ci-dessus, à compléter
# NUTS2 corrigé / Région_FR / NUTS3 / Metro-RUP-PTOM.
df_final = df_final.drop(columns=[_c for _c in ("Adresse (SIRENE)", "Ville (SIRENE)", "Code postal (SIRENE)")
                                  if _c in df_final.columns])

exporter_pipeline(df_final, rows_api, adr_mod, set(), col_adresse, col_nuts2,
                  "Code postal corrigé", "NUTS3 FR", nom_sortie)
generer_rapport(rapport, nom_rapport)

# v5_31 -- export de la mémoire de correspondance (nom+dept -> SIREN), à
# réutiliser telle quelle lors du traitement d'un AUTRE millésime : redéposer
# ce fichier au même endroit (correspondance_siren.json) la prochaine fois.
nom_memoire = "correspondance_siren.json"
exporter_correspondance_siren(nom_memoire)

# v5_4 : ne télécharger que les fichiers réellement créés (le rapport n'existe
# pas quand aucune TVA n'était manquante — plus de FileNotFoundError).
for _f in (nom_global, nom_sortie, nom_rapport, nom_memoire):
    if Path(_f).exists():
        if IN_COLAB:
            _colab_files.download(_f)
        else:
            print(f"  - {Path(_f).resolve()}")
    else:
        print(f"  (fichier non créé, rien à télécharger : {_f} — rapport vide, aucune TVA manquante)")
print("\nTERMINE - 4 fichiers generes (fichier + rapport + GLOBAL + mémoire de correspondance)")


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
[ 1737/6572  26.4%] AGORALAB                                      ✅ documente [SIRET 81896818200019]
[ 1738/6572  26.4%] AGRINATURA EEIG*THE EUROPEAN ALLIANCE AGRICUL ✅ documente [SIRET 48026283100013]
[ 1739/6572  26.5%] AIR MARINE                                    ✅ documente [SIRET 38136506300043]
[ 1740/6572  26.5%] AKIANI                                        ✅ documente [SIRET 79506866700034]
[ 1741/6572  26.5%] ALCEA FRANCE EURL A CAPITAL VARIABLE*         ✅ documente [SIRET 51008254800023]
[ 1742/6572  26.5%] ALERION                                       ✅ documente [SIRET 81216334300047]
[ 1743/6572  26.5%] ALGAIA                                        ✅ documente [SIRET 47808014600069]
[ 1744/6572  26.5%] ALLIANCE FORETS BOIS *AGENCE FORESTARN        ✅ documente [SIRET 53477026800017]
[ 1745/6572  26.6%] ALPHAGEOMEGA                                  ✅ documente [SIRET 80125198400010]
[ 1746/6572  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


TERMINE - 4 fichiers generes (fichier + rapport + GLOBAL + mémoire de correspondance)
